# Databricks Inspire AI

**Infuse your Data with Databricks Genie Code AI capabilities to deliver business value.**

Inspire AI reads your Unity Catalog metadata (schemas, table and column names) and uses a 7-phase multi-model AI pipeline to discover, score, and generate production-ready use cases — complete with Genie Code instruction notebooks, documents, and presentations.

---

## What You Get

- **Genie Code Notebooks**: One Databricks notebook per use case. A use case without a Genie Code instruction gets a skeleton (its details and a regeneration marker). A Genie notebook has three cells written by Inspire: the title with a short use-case card, a one-line disclaimer and "Copy the cell below into Genie Code without the surrounding quotes (the `r"""` and `"""` lines), and let Genie Code implement it."; the Genie Code instruction as a code cell; and Run controls, the last cell, whose `main()` runs the `run_use_case()` that Genie Code builds, here or as the notebook's tagged Databricks job (`dbx_inspire_ai_*` tags, so the Inspire dashboard registers the run). Copy the instruction into Genie Code and let it run: Genie Code adds its cells between the instruction and Run controls and runs the notebook on serverless compute. The notebook writes only the business output table and, for families A-D, the Unity Catalog model, and `notebooks/AGENTS.md` gives Genie Code the rules for every notebook.
- **PDF Catalog** — Professional PDF document cataloging all discovered use cases with executive summary, domain grouping, and scoring.
- **PowerPoint Presentation** — Executive-ready slides summarizing domains, priorities, and top use cases.
- **Excel Use Cases Catalog** — Spreadsheet with all use cases, scores, metadata, and prioritization.

**23 Languages**: English, French, German, Spanish, Hindi, Chinese (Mandarin), Japanese, Arabic, Portuguese, Russian, Swedish, Danish, Norwegian, Finnish, Italian, Polish, Romanian, Ukrainian, Dutch, Korean, Indonesian, Malay, Tamil.

---

## Quick Start

1. **Configure**: Fill in **Business Name** and **UC Metadata** (required). Optionally set **Inspire Database** for tracking.
2. **Run**: Click **Run All** to start the engine.
3. **Explore**: Find all generated artifacts in your **Generation Path**.

---

## Configuration Guide

### Operation
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 00 | **Operation** | Controls the top-level workflow: | `Discover Use Cases` |
| | | `Discover Use Cases` — Runs the full discovery pipeline (phases 1-7). Each use case gets a skeleton notebook containing the full metadata and a marker cell with `generate_genie_code_instruction = False`; Genie Code instructions are then generated for the top-N use cases by final rank (widget **15. Generate Genie Code Instruction For**, default 0, so Discover writes skeleton notebooks only in every quality mode and a use case gets its instruction when its PoC/MVP or its notebook is started from the App) or, in `Ultra High` when the widget is not 0, for every elite use case and only those, and their Genie notebooks replace the skeletons. The `__inspire_usecases` table persists every field needed to later regenerate the Genie Code Instruction (directly-involved schema, FK relationships, full-catalog listing, technique guidance). The `__inspire_session` table persists per-session prompt context (enriched business context, strategic goals, priorities, strategic initiative, value chain, revenue model, generation instructions section). | |
| | | `Generate Use Cases` — **Skips discovery entirely** and only regenerates Genie Code Instructions for use cases selected via one of two routes. <br/>**Route 1 (preferred, DB flag):** Flag rows in Inspire's tracking table via SQL (the Inspire UI also exposes this as a toggle): `UPDATE <inspire_database>.__inspire_usecases SET generate_genie_code_instruction = 'Yes' WHERE id = '<use_case_id>' AND session_id = <session_id>;` then run this notebook. After generation, flagged rows are reset back to `'No'`. <br/>**Route 2 (fallback, notebook flag):** If no DB rows are flagged, Inspire scans notebooks under the **Generation Path** and picks up any whose marker cell sets `generate_genie_code_instruction = True`. The marker is cross-referenced with `__inspire_usecases` to rehydrate the cached prompt fields; regenerating the notebook replaces the marker cell in-place. <br/>**Note:** PDF Catalog and Presentation selections are ignored in this mode. | |

### Required Inputs
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 01 | **Business Name** | Your organization or project name (used for titles, context, and AI prompts). | *(Empty)* |
| 02 | **UC Metadata** | Comma-separated Unity Catalog paths: Catalogs, Schemas, or individual Tables (e.g., `main.finance, sales_catalog, main.hr.employees`). You can mix all three levels. Alternatively, provide an absolute path to a JSON file to run in **Docs-Only mode** (skip discovery, generate artifacts from existing data). | *(Empty)* |

### Optional Tracking
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 03 | **Inspire Database** | A `catalog.schema` path where Inspire stores its tracking tables and result tables (e.g., `my_catalog.inspire_results`). The schema will be created if it does not exist. If left empty, defaults to `main._inspire` as a placeholder. Result table names are quoted three-level names, `` `catalog`.`schema`.`_inspire_<id>_<name>` ``. | *(Empty — optional)* |

### Table Election
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 05 | **Table Election** | Controls which tables are used for use case generation: | `Let Inspire Decides` |
| | | `Let Inspire Decides` — AI automatically classifies tables as transactional vs. dimensional and selects the optimal set based on volume thresholds. | |
| | | `All Tables` — Use every business table (no filtering). | |
| | | `Transactional Only` — Only use tables classified as transactional. | |

### Use Cases Quality
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 06 | **Use Cases Quality** | Sets the quality bar applied after scoring. All three modes share one broad generation pass and differ only in which scored use cases are selected: | `Good` |
| | | `Good`: Default. Keeps use cases scored Medium, High, Very High, or Ultra High. | |
| | | `High`: Keeps only use cases scored High, Very High, or Ultra High. | |
| | | `Ultra High`: Elite selection. High-or-better candidates go through an elite stage that keeps only use cases with substantial business value, strong metadata evidence, and a proven build path, judged against an absolute bar (see **Ultra High: Elite Selection** below). | |
| | | Legacy labels are still accepted and mapped automatically: `Coverage Mode (All)`, `Balanced`, `Good Quality` and `High Quality` run as `Good`; `Strict Quality` and `Very High Quality` run as `High`. `Ultra High` is used only when you choose it. Any other value stops the run with a validation error that lists the valid options. | |

### Strategic Configuration
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 07 | **Business Domains** | Comma-separated business domains to focus on (e.g., `Risk, Finance, Operations`). If provided, use cases will be aligned **only** to these domains. If left empty, the AI will auto-detect and cluster domains from your data. | *(Empty — auto-detected)* |
| 08 | **Business Priorities** | Multi-select strategic priorities. Use cases aligned to selected priorities will score higher: `Increase Revenue`, `Reduce Cost`, `Optimize Operations`, `Mitigate Risk`, `Empower Talent`, `Enhance Experience`, `Drive Innovation`, `Achieve ESG`, `Protect Revenue`, `Execute Strategy`. | `Increase Revenue` |
| 09 | **Generation Instructions** | Free-text instructions for use case generation (e.g., `join table X with Y on column Z, focus on fraud detection, use XGBoost, ignore tables starting with __log`). When provided, these are interpreted by AI and applied as non-negotiable instructions across all generation phases. | *(Empty)* |

### Generation Options
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 10 | **Generation Options** | Multi-select artifacts: | `PDF Catalog` |
| | | `PDF Catalog` — Generate professional PDF documentation per language. | |
| | | `Presentation` — Generate executive-ready PowerPoint slides per language. | |
| | | *(Note: Genie Code Instruction generation is controlled by the **Operation** widget and widget 15 below, not by this multi-select.)* | |
| 12 | **Generation Path** | Workspace folder where all outputs are saved (notebooks, PDFs, PPTX, Excel). | `./../demos/inspire_gen/` |
| 13 | **Documents Languages** | Target language(s) for PDF and Presentation artifacts (multi-select). Required when PDF or Presentation is selected. | `English` |
| 15 | **Generate Genie Code Instruction For** | How many use cases get a Genie Code instruction during Discover, taken in final-rank order: a non-negative integer or `all` (`0` turns it off in every quality mode). In `Ultra High` any other value gives every elite use case and only those one, whatever the number; domain coverage picks keep their skeleton notebook. | `0` |

### Advanced Options
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 14 | **Inspire Session ID** | Optional. Provide a fixed session ID to resume or correlate runs. If left empty, a unique ID is auto-generated. The session ID is used as the merge key in the tracking table. | *(Auto-generated)* |

---

## How It Works (v0.7.3 production release)

Inspire runs a 7-phase AI pipeline using a multi-model cascade (Claude Opus 4.6 / Sonnet 4.6 / Opus 4.5 / Sonnet 4.5 / GPT-OSS 120B / 20B with automatic fallback):

| Phase | What Happens |
|:------|:-------------|
| **1. Initialization** | Extracts business context, interprets the 7-category generation-instruction extraction (see below), filters out technical/system tables aggressively. Context inferred from the business name alone is labelled `[ASSUMPTION ...]` (the industry is taken from that labelled context, never from the business name itself), strategic initiative and revenue model stay `UNKNOWN_FROM_METADATA` because no input carries them, and every scoring and generation prompt treats assumption text as a hypothesis, never as evidence. Tables whose metadata cannot be loaded are named in a warning step and in `results_json.metadata_unavailable`. |
| **2. Use Case Generation** | Two passes cover structured analytics, AI/ML, and statistical techniques. All 19 quality gates (D1–D19) are instructed at generation time so the LLM tries to satisfy them as it writes each UC. |
| **3. Domain Clustering** | Clusters use cases into business domains, detects subdomains, merges overlapping domains, deduplicates subdomains across domains, and splits oversized mega-domains. |
| **4. Scoring, Dedup & Selection** | (a) LLM Semantic Dedup removes near-duplicates; the global pass starts on the thinker model and cascades to the next model when that one is slow or fails. (b) Combined Value + Quality scoring: one LLM pass scores every UC on an absolute scale and is the only Priority source (there is no separate per-domain value pass); each scoring and Best-of-Best batch sees only the schema of its own tables and their foreign-key partners; any HARD VETO caps quality at 2.0; a UC without a valid score is never selected. (c) **Quality Bar** of the selected mode: `Good` keeps Medium or better, `High` keeps High or better. (d) `Good` / `High`: **Best-of-Best ranking** drops hard-vetoed UCs, and only when more than 75 survive a stratified-by-idea-theme trim caps the set (effective cap default range [91, 110] = `90 + randint(1, 20)`). `Ultra High`: the **Elite Selection** stage replaces the trim (no quota, no cap). (e) Each user-specified Business Domain left empty receives one domain coverage pick, labelled below bar. There is no recovery, top-up or padding; and if nothing clears the bar the run does not ship empty: it returns the closest candidates labelled below the selected mode's bar, with their real scores and the check each missed, under a plain "0 use cases met the bar" notice. (f) Every mode renumbers the final list in final-rank order (`N01` is the domain of the best use case, `U01` its best use case); earlier ids are provisional. |
| **5. Genie Code Instructions** | Writes a 7-section Genie Code instruction for the top-N use cases by final rank (widget 15), or for every elite use case in `Ultra High`, from the decision block, live Unity Catalog metadata, the foreign-key paths and the execution family of its technique (one technique registry maps every technique to a family A-H). Every cited table and column is checked against live metadata and invented figures are rejected, with one uncached repair; in `Ultra High` an instruction that still fails is not emitted. The instruction is the brief Genie Code builds and runs the whole notebook from. |
| **6. Summary & Artifacts** | Executive summary, per-use-case Genie Code notebooks (organized by domain/subdomain), PDF catalog, PPTX presentation, Excel catalog with BoB scorecard columns. |
| **7. Translation** | Translates all artifacts into selected languages. |

### The 19 Quality Gates — Ruthless but Fair

Every use case must survive 19 gates. **Every gate is LLM-evaluated — zero regex, zero pattern matching, pure judgment.** Full spec: the **The 19 Quality Gates** section in [`readme.md`](readme.md#the-19-quality-gates).

In v0.7.2 the gates operate in **two modes**:

- **Binary (generation + Combined Scoring)**: each gate passes or fails. HARD VETO caps quality_score at 2.0 so the quality bar of every mode drops the UC.
- **0–10 fine-grained (Best-of-Best ranking, when count > 75)**: each gate scored 0–10 per UC. Produces a ranked shortlist that feeds the stratified trim.

**Tier 1 — Technical Grounding (D1–D11)**

| # | Gate | Question |
|---|------|----------|
| D1 | Causal Signal | Do the data columns directly cause or indicate the outcome being predicted? |
| D2 | Cause-Effect Validity | Is the cause-effect relationship logical, proven, industry-recognized? |
| D3 | Data Granularity | Is the data at the right level of detail for the analysis? |
| D4 | Critical Dimensions | Are all required columns actually present in the schema? |
| D5 | Logical Possibility | Can the operation be performed on the given types? |
| D6 | Metric Validity | Can the claimed metric actually be calculated from available fields? |
| D7 | Design–Schema Match | Does the technical design cite columns that exist? |
| D8 | Semantic Uniqueness | After verb / noun / entity / metric normalization — is this a duplicate? |
| D9 | Analytical Depth (Strip Test) | Remove ML-technique mentions — is there real substance left? |
| D10 | Activation Quality (Deliverable Test) | Does the "with …" suffix name a business deliverable, not just a technique? |
| D11 | Domain & Technique Balance | Does the portfolio cluster too heavily on one domain+technique? |

**Tier 2 — Business Value (D12–D14)** — *"would you actually fund this?"*

| # | Gate | Persona | Question |
|---|------|---------|----------|
| D12 | Business Relevance & Opportunity | Stakeholder | Is it relevant day-to-day? Does it deliver tangible value? Can you name the team or role that consumes it? Is it a current pain or a future opportunity? |
| D13 | Sponsor Test | Sponsor / CFO | Would you fund it with a real budget line? Could you pitch it to the CEO without caveats? |
| D14 | Engineering Test | Production Engineer | Can it go to production on existing Databricks capabilities? Can you support and maintain it long-term? Can you fix issues without bespoke infrastructure? |

**Tier 3 — Principal-Business-Analyst (D15–D19)** — the questions a 20-year PBA asks before sign-off

| # | Gate | Question |
|---|------|----------|
| D15 | Decision Cadence Match | Does data refresh speed match decision speed? |
| D16 | The Monday Test | Would a stakeholder's Monday look different? Must name a concrete action. |
| D17 | Explainability & Audit Defensibility | Can you explain the output in 5 minutes without proprietary magic? |
| D18 | 18-Month Longevity | In 18 months, is this still running or abandoned? |
| D19 | Attributable Impact | Can the team prove 6 months later that this moved a measurable dollar amount? |

**Portfolio size after all 19 gates**: there is no target count, quota, or floor. The portfolio holds every use case that clears the bar of the selected mode, and the count is reported, never targeted. `Good` and `High` cap a very large survivor set with the stratified trim below; `Ultra High` has no cap.

### Best-of-Best Ranking + Stratified Trim (NEW in v0.7.2)

This section describes the `Good` and `High` modes; `Ultra High` replaces the trim with the Elite Selection stage below. Whenever the portfolio that cleared the quality bar has at least `bob_min_threshold` UCs (default 20), Inspire scores every UC with the Best-of-Best LLM pass so `bob_score` is populated. When the count additionally exceeds `aggressive_prune_threshold` (default 75), the same flow continues into the Stratified Trim. (v0.8.7: BoB scoring decoupled from trim gate — earlier releases skipped scoring for portfolios in the 20–75 zone, leaving `bob_score = NULL`.) The flow is:

**Step 1 — Best-of-Best scoring**
- Prompt: `BEST_OF_BEST_SCORING_PROMPT`
- Batched (25 UCs per batch), parallel
- For each UC the LLM returns: 19 per-gate scores 0–10 + a 2–4 word `idea_theme` label + short justification

**Step 2 — Tier-weighted composite**

```
tier1 = mean(D1..D11)  # technical grounding
tier2 = mean(D12..D14) # business value
tier3 = mean(D15..D19) # PBA sign-off

bob_score = 0.30*tier1 + 0.35*tier2 + 0.35*tier3

# Hard-veto floor
if any gate D1..D19 < 3:
    bob_score = min(bob_score, 2.0)
```

Weight rationale: technical grounding is a pass/fail check (survivors already sound); business fundability and PBA actionability are the differentiators, equally weighted.

**Step 3 — Stratified trim by idea theme**

```
group UCs by idea_theme
cap_per_theme = target_max * bob_theme_max_share   # default 15% → 15 of 100

FLOOR rule:   every theme keeps ≥ 1 UC            # coverage guarantee
CAP rule:     no theme exceeds cap_per_theme       # no single theme dominates
BUDGET rule:  remaining slots allocated proportional to theme size

within each theme:
    sort by bob_score DESC
    take top-K
```

**Why stratified beats flat top-N**: a flat "keep top 100 by bob_score" would silently wipe out narrow-but-valuable ideas (e.g. 3 strong pricing-optimization UCs beaten by 20 mediocre anomaly-detection UCs). Stratified preserves the discovery breadth — the whole point of Inspire — while ruthlessly culling redundancy within each idea.

**Step 4 — Persisted scorecard**

5 new columns land in `__inspire_usecases` and the Excel/CSV export:

| Column | Type | Values |
|---|---|---|
| `idea_theme` | STRING | e.g. "Anomaly Detection", "Pricing Optimization" |
| `bob_score` | DOUBLE | 0–10 composite (2.0 = hard-veto floor hit) |
| `bob_tier1_score` | DOUBLE | mean of D1–D11, 0–10 |
| `bob_tier2_score` | DOUBLE | mean of D12–D14, 0–10 |
| `bob_tier3_score` | DOUBLE | mean of D15–D19, 0–10 |

Migration is idempotent — safe on existing tables.

**Validation evidence** (tpcds_sf1 Samples database, session `8041728042000030`):

| Metric | Value |
|---|---|
| Phase 2 raw UCs | 378 |
| After dedup + Combined Scoring | 308 |
| After Balanced filter (now `Good`) | 244 |
| **After BoB + Stratified Trim** | **100** |
| Distinct idea themes | 75 |
| Largest theme | Anomaly Detection — 11 UCs |
| Avg bob_score | 6.7 |
| bob_score range | 2.0 → 8.23 |
| Tier averages | t1 7.23 · t2 6.92 · t3 6.23 |
| Phase L final-gate | 16/19 passed |
| Run duration | 68 min |

### Ultra High: Elite Selection (v1.2.0)

`Ultra High` keeps only use cases with substantial business value, strong metadata evidence, and a proven build path. It shares the broad generation pass with `Good` and `High` and differs only in selection:

1. **Candidates**: use cases scored High or better, plus a second look at Medium use cases with a high value score, so a high-value idea with a noisy quality label is not lost. When Business Domains are set, candidates are assigned to those domains first.
2. **Best-of-Best prescreen**: hard-vetoed candidates and candidates below the prescreen minimum are dropped.
3. **Elite judge** (`ULTRA_HIGH_ELITE_JUDGE_PROMPT`): for every candidate it records the decision the use case changes and who owns it, the KPI, the value driver tied to named columns (no invented figures), the signal, outcome, time and entity-key columns, the join path, the Databricks build path and its blockers, the data shapes the use case depends on that metadata cannot prove (runtime assumptions), every reason that defeats it as a fatal flaw from a fixed list (outcome derived from its own inputs, no decision changed, duplicate of another use case, unverifiable data shape, infeasible with the available columns, value overclaimed), a red-team "why this fails" note, and absolute 1–10 scores for value, evidence, implementability and actionability. The red-team verdict is derived from the fatal flaws, and an assessment that contradicts itself is discarded and judged again.
4. **Metadata verification**: every cited table and column must exist, each technique's required columns must be present with compatible types (a temporal technique may use a date key that joins a dated table; an entity key is a declared key, an identifier-like name such as `*_id`, `*_key`, `*_sk`, `*_code`, `*_no` or `*_number`, or a text column named as the entity the decision acts on), and a multi-table use case needs a foreign-key path. A supervised use case whose outcome is a rule over its own signal columns in the same period is rejected (it is a ranking or a rule, not a model), a co-occurrence use case grouped by its table's own row key is rejected, and a use case built on Databricks AI text functions without model training is verified and built as AI text processing. The evidence is metadata only: names, types, comments, constraints, tags and table properties. When verification rejects only the judge's own citations (for example a mistyped column or a text column cited as the time column), the use case gets one re-judge with the cache bypassed, the exact failures and the real column lists of its tables, and is verified again; data that lacks the needed column stays rejected.
5. **Absolute bar**: a use case is elite only when it has no fatal flaw, the judge rates it elite, all four scores reach the minimum (8 of 10 by default), its value tier is Transformational or Material, and verification passes. There is no quota, no cap and no padding. A use case just below the bar gets one independent second opinion and is promoted only when the averaged scores of both opinions reach the minimum; every other miss is recorded in a near-miss ledger with the gate it failed.
6. **Overlap, coverage and ranking**: of two overlapping elites the stronger is kept, and elites that change the same decision on different channel tables collapse to the one with the higher elite score; each user-specified Business Domain without an elite receives one domain coverage pick labelled "Domain coverage pick, below bar"; the final set is clustered into domains without merging small domains (its near misses are clustered with it, so the near-miss ledger shows their domains) and ranked by elite score (0.40 value, 0.25 evidence, 0.20 implementability, 0.15 actionability), with domain coverage picks after every elite use case.
7. **Genie Code**: every elite use case, and only elite use cases, receives its Genie Code instruction; domain coverage picks keep their skeleton notebook. The notebook's readiness gate checks the use case's runtime assumptions before it builds anything and stops with a plain message when one does not hold.

If no use case clears the bar, the run still ends successfully and does not ship empty: it returns the near misses rejected without a fatal flaw, ranked by elite score and labelled "below the Ultra High bar" with the check each missed, then High-tier candidates, under a plain "0 use cases met the Ultra High bar" notice. The scores shown are the real elite scores; nothing is invented.

**Run-to-run stability.** `Ultra High` keeps only use cases that clear an absolute bar, so the elite set is not a fixed-size list and is not guaranteed identical when you re-run the same inputs. The highest-scored ideas are the most stable between runs, but both the size of the elite set and its borderline members vary run to run: the candidates come from an LLM, and the elite judge itself samples the candidates it accepts (its judge endpoint rejects the `temperature` parameter, so it cannot be made deterministic). Across the release pairs the judge accepted 15% to 38% of candidates, and the same `tpcds_sf1` corpus returned 13 and 11 elites on one build, 35 and 30 on another, and 8 on the final build. Measured idea-level overlap between repeat runs (same decision and outcome, titles ignored) was about 17-18%: the two `bakehouse` runs shared 2 ideas of 7 and 7, and the two `tpcds_sf1` reruns shared 10 of 35 and 30. When a stable shortlist matters, run `Ultra High` twice and keep the use cases that appear in both runs.

### Selection Outcome and Tracking (v1.2.0)

- **Pipeline steps** under the `Use Case Prioritization` stage: `Quality Bar` (a warning step carrying the below-bar fallback list when 0 use cases met the bar), `Selection Funnel` (one ledger entry per stage: count in, count out, drops by reason) and `Elite Selection` (`Ultra High` only).
- **Session results** in `__inspire_session.results_json`: `funnel_ledger` (the stage ledger), `selection_outcome` (present when 0 use cases met the bar) and `elite_outcome` (`Ultra High` only: elite count, coverage picks and the near-miss ledger).
- **Run outcome**: when an artifact step fails or a translated document keeps text in English, `results_json` also carries `failed_steps` (one entry per failed step: `step`, `language`, `error`) and `translation_gaps` (per language: `use_cases` with text shown in English, `untranslated_use_cases` and a count per field). Every run also carries `metadata_unavailable` (`count` and `tables`: each table whose metadata could not be loaded, with the reason) and `llm_endpoint_health` (`unusable` endpoints with the reason, `temperature_unsupported`, `probed`). Lost tables are also named in an `ended_warning` `Metadata Load` step under the `Data Preparation` stage and in the honesty report. The run still completes: its closing banner reads "COMPLETED WITH WARNINGS" and lists the failed steps, translation gaps and lost tables, and the final `Inspire pipeline completed` step is `ended_warning` with the message `Completed with warnings: ...` instead of `ended_success` / `Completed`.
- **New `__inspire_usecases` columns**: `quality` (quality label), `quality_mode` (the run's canonical mode), `elite_status`, `elite_score`, `elite_assessment` (JSON) and `selection_flags` (JSON); `code_structure_mandate` is now filled for every row. A session's rows are replaced in one Delta transaction (staged, then one `replaceWhere` overwrite), so readers never see a partly written session; if the replace fails, the previous rows stay and `failed_steps` records it. Table names are quoted, so an Inspire Database whose catalog or schema name needs quoting (for example one with a hyphen) works.
- **Artifacts**: a score that was never computed reads `Not scored` in the Excel, CSV and Markdown catalogs, never a default. Per-language file names use ISO 639-1 codes: Swedish, Polish, Dutch, Indonesian, Malay and Chinese (Mandarin) now get `sv`, `pl`, `nl`, `id`, `ms` and `zh` (previously `sw`, `po`, `du`, `in`, `ma` and `ch`).
- **LLM CSV answers**: every CSV answer passes one row guard. A row that is short of its columns, carries unexplained extra values, or holds a copied id, table name, technique, quality label or enum value in another column is rejected, logged as `[CSV-ROW misaligned]` with a count per reason, and treated as missing, so the stage retries it instead of reading shifted values. A translation answer counts as cut off only when it ends inside an open quoted field, its last row stops mid-row, or its closing rows never started; a complete answer that lacks a row is retried once and split only if the retry is incomplete too. A deduplication keep-list that was cut off, or that names a use case outside its id column, is refused instead of dropping use cases.

### What changed in v0.7.3

1. ✅ **Hard-veto pre-filter (fix)** — v0.7.2 let hard-veto UCs (`bob_score = 2.0`) survive the trim when alone in their theme. v0.7.3 drops UCs with `bob_hard_veto_gate` set BEFORE grouping by theme. Validated on tpcds_sf1 (session 8042073003073003): 13 hard-veto UCs dropped, 0 survivors in final portfolio.
2. ✅ **Target-max jitter (new)** — v0.7.2 always produced exactly 100 UCs. v0.7.3 computes `effective_cap = target_max_use_cases + randint(1, target_max_jitter)` → default range [91, 110]. Portfolio sizes are now non-round (tpcds v0.7.3 = 105).
3. ✅ **Multi-language filename fix** — Excel + Markdown were overwriting each other across languages. v0.7.3 uses `_{lang_abbr}` suffix when multiple languages are selected (e.g., `readme_en.md`, `readme_de.md`).
4. ✅ **Parallel Route 1/2 Genie regeneration** — v0.7.2 processed UCs sequentially (100 × 3 min = 5 hours). v0.7.3 uses a ThreadPoolExecutor with `genie_regen_parallelism = 10` default → ~30 min for 100 UCs.

### Remaining (non-blocking) issues

1. **Idea themes still too granular** — the BoB prompt asks for 15–25 themes but produces 60–75. Future work: second-pass theme canonicalizer.
2. **Phase L `L12_q3_drop_under_50pct`** flags when >50% of scored UCs are trimmed. Behaviourally correct (we intended the trim) but the gate may need its threshold tuned.

### `Generation Instructions` — 7 Categories (v0.7.0+)

The free-text **09 Generation Instructions** widget is interpreted by an LLM into seven categories. Write naturally — no special syntax. If a category is not mentioned, it is left empty and the pipeline runs as if not provided.

| # | Category | Example Phrase | Effect |
|---|---|---|---|
| 1 | Table Joins | "join orders with customers on customer_id" | Genie Code treats tables as pre-joined |
| 2 | Goal Setting | "focus on fraud detection" | Goal Alignment Filter removes non-aligned UCs; aligned UCs are never penalized for domain or technique balance but still face the quality bar of the selected mode |
| 3 | Forced Techniques | "use XGBoost for all classification" | Generator prefers the named technique |
| 4 | Table Ignores | "ignore tables starting with __log" | Filter treats matching tables as TECHNICAL |
| 5 | **Construct Exclusions** (v0.7.0) | "we don't have a loyalty program" | UCs referencing excluded constructs are rejected |
| 6 | **Change-Management Posture** (v0.7.0) | "we're conservative about production changes" | Conservative → rollback mandate; experimental → pilot-scope mandate |
| 7 | **Data Characteristics** (v0.7.0) | "claims table has no labeled fraud outcomes" | Technique selection respects these — no supervised ML on un-labeled tables |

### Runtime config keys (v0.7.2)

| Key | Default | Purpose |
|---|---|---|
| `aggressive_prune_threshold` | 75 | Trigger Stratified Trim when portfolio count exceeds this. (v0.8.7: BoB scoring is no longer gated on this — see `bob_min_threshold`.) |
| `bob_min_threshold` | **20** | v0.8.7: Run Best-of-Best LLM scoring whenever portfolio has at least this many UCs, regardless of trim. Ensures `bob_score` is populated in the tracking table even for small portfolios that don't need trimming. |
| `target_max_use_cases` | **90** | Base for the effective cap (v0.7.3 semantics; was the fixed ceiling pre-v0.7.3) |
| `target_max_jitter` | **20** | Jitter amount: `effective_cap = target_max_use_cases + randint(1, jitter)`; default (90, 20) → [91, 110] |
| `bob_scoring_batch_size` | 25 | UCs per BoB LLM call (smaller than Combined Scoring because 19-score JSON is heavier) |
| `bob_theme_max_share` | 0.15 | Max fraction of target_max per idea theme (default → ~15 UCs per theme) |
| `genie_regen_parallelism` | **10** | Concurrent Genie Code generations per wave: Discover auto-Genie and Route 1/2 regeneration (v0.7.3; all Genie waves from v1.2.0) — ~5h → ~30min for 100 UCs |

### What's new since v0.7.0

- **v1.2.1**: A scoring batch that times out is retried instead of dropped: only the first attempt is cut by the straggler budget, every retry gets the full per-task timeout, and a timed-out batch is retried in halves, so `Ultra High` no longer loses candidates to slow judge calls (v1.2.0 runs lost 2 to 50 per run); a discarded elite-judge record is re-judged with the exact reason, and the judge prompt now names every key it must return, including `verdict`; a verdict still above its own scores after that re-judge is capped at what they support. An unusable dedup keep-list is asked for once more before global dedup falls back to per-domain dedup or a domain keeps all of its use cases; a failed global dedup over a single group is not resent; a trailing note can no longer wipe the keep-list, and its id column follows the majority of rows. Logs report by cause: recovered retries, cascades and cancellations are warnings (a worker cancelled by its own straggler retry ends `ended_warning`, not `ended_error`), and errors are kept for outcomes that degraded.
- **v1.2.0**: Quality modes `Good`, `High` and `Ultra High` (legacy labels still accepted); one broad generation pass shared by every mode; the `Ultra High` elite selection; no recovery, top-up or target-zone padding in any mode (when nothing clears the bar the run ships the closest candidates labelled below the bar instead of an empty result); domain coverage picks for every user-specified Business Domain; selection funnel ledger; failed artifact steps and translation gaps reported in the run banner, the final pipeline step and `results_json` (`failed_steps`, `translation_gaps`); atomic tracking replace; ISO 639-1 language codes in file names; combined scoring as the single Priority source (the per-domain value pass and its two prompts are removed); final-rank renumbering in every mode; per-batch schema subsets for scoring and Best-of-Best; the global dedup cascades past a slow thinker; lost tables and the LLM endpoint health in `results_json`; quoted result table names.
- **v0.7.3** — Hard-veto pre-filter (resolves the v0.7.2 hard-veto-through-theme-floor slip-through); target-max jitter for non-round portfolio sizes (default [91, 110]); per-language filename suffix fix (Excel + Markdown now get `_en` / `_de` suffixes in multi-language runs); parallel Route 1/2 Genie regeneration (100 UCs regenerated in ~30 min instead of ~5h).
- **v0.7.2** — Best-of-Best ranking + stratified-by-idea-theme trim; new `BEST_OF_BEST_SCORING_PROMPT`; 5 persisted BoB columns; tier-weighted composite + hard-veto floor.
- **v0.7.1** — 19 quality gates (D1–D19) fully documented in readme; D12–D14 business/sponsor/engineering gates and D15–D19 principal-analyst gates enforced at Combined Scoring; unified MD style across all docs; NameError fix in aggressive vetting.
- **v0.7.0** — Context-extraction upgrade: 7 categories extracted from the `09 Generation Instructions` widget (construct exclusions, change-management posture, data characteristics); `has_genie_code` column added to tracking table.

---

## Docs-Only Mode (JSON Path)

If you provide an absolute file path (starting with `/`) in **UC Metadata** instead of catalog/schema names, Inspire enters **Docs-Only mode**: it loads use cases from the JSON file and generates notebooks, PDFs, and presentations without running discovery or scoring.

---

## Privacy & Trust

<div style="background-color:#f0f7ed; color:#1b5e20; padding:15px; border-radius:8px; border: 1px solid #c3e6cb;">
  <b>Metadata Only</b>: Inspire AI reads <b>only</b> your metadata (catalog, schema, table, and column names, column types, comments, key constraints, tags, and table properties) to generate and verify use cases. It does <b>NOT</b> access, read, or sample your actual data at any point during the pipeline, including the <code>Ultra High</code> elite verification.
</div>

---

## Tips

- **Business Priorities vs. Generation Instructions**: Business Priorities are pre-defined categories (multi-select) that influence scoring weights. Generation Instructions are free-text and are parsed by AI into structured directives (table joins, goals, forced algorithms, table ignores) that are applied as non-negotiable rules across all generation phases.
- **Table Election**: For large catalogs (hundreds of tables), `Let Inspire Decides` typically yields the best results by focusing on transactional tables that drive the most valuable analytics.
- **Use Cases Quality**: Every mode generates the same broad candidate set. The default `Good` keeps Medium-or-better use cases; `High` keeps only High-or-better; `Ultra High` keeps only the elite use cases that clear an absolute bar, which can be a short list or none at all.
- **Inspire Database**: This is where Inspire persists tracking tables (`__inspire_session`, `__inspire_step`, `__inspire_usecases`) with all generated use cases, scores, and session metadata. Use it to track runs over time or resume work.
- **Session ID**: Provide the same Session ID across runs to update existing use cases in the tracking table rather than creating new entries.
- **Technical Table Filtering**: Inspire always aggressively filters out system tables, audit logs, and technical metadata to focus on business-relevant use cases.
- **Genie Code Instructions**: Each Genie notebook holds one self-contained instruction for this use case and this data: the goal, the data, the readiness check, the build steps, the method and acceptance, the business output and the definition of done. A readiness stop is a valid outcome, and the notebook then writes nothing; when no model beats its baseline, the notebook writes the baseline's rows and registers no model.



In [0]:
DATABRICKS_INSPIRE_BANNER = r"""
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    ____        _        _          _      _                             ┃
┃   |  _ \  __ _| |_ __ _| |__  _ __(_) ___| | _____                      ┃
┃   | | | |/ _` | __/ _` | '_ \| '__| |/ __| |/ / __|                     ┃
┃   | |_| | (_| | || (_| | |_) | |  | | (__|   <\__ \                     ┃
┃   |____/ \__,_|\__\__,_|_.__/|_|  |_|\___|_|\_\___/                     ┃
┃       ___                      _                  _    ___              ┃
┃      |_ _| _ __   ___  _ __   (_) _ __  ___      / \  |_ _|             ┃
┃       | | | '_ \ / __|| '_ \  | || '__|/ _ \    / _ \  | |              ┃
┃       | | | | | |\__ \| |_) | | || |  |  __/   / ___ \ | |              ┃
┃      |___||_| |_||___/| .__/  |_||_|   \___|  /_/   \_\___|             ┃
┃                       |_|                                               ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
"""

import logging

# ==============================================================================
# TECHNICAL CONTEXT - Prompt to Model Type Cascade Configuration
# ==============================================================================
# Each prompt is mapped to a model_type + model_size. At runtime, the system
# resolves (type, size) to an ordered list of enabled models (sorted by "order").
# The lowest-order enabled model is tried first; on failure, the next one is used.
# To change which model bucket a prompt uses, modify "model_type"/"model_size" below.
# To add/remove/reorder models, modify the "models" section.
# ==============================================================================
DEFAULT_MODEL_SIZE_BY_TYPE = {
    "worker": "large",
    "thinker": "large",
}

__AGENT_VERSION__ = "v1.3.0"

TECHNICAL_CONTEXT = {
    "runtime": {
        "llm_timeout_seconds": 300,
        "max_retry_attempts": 1,
        "llm_endpoint_demote_after_timeouts": 3,
        "scan_parallelism": 8,
        "metadata_parallelism": 8,
        "global_min_parallelism": 4,
        "global_max_parallelism": 20,
        "default_max_parallelism": 16,
        "cluster_memory_gb": 32,
        "cluster_worker_count": 2,
        "quality_scoring_batch_size": 30,
        "quality_scoring_max_workers": 10,
        "batch_processing_timeout": 900,
        "batch_processing_max_attempts": 3,
        "table_election_threshold": 5,
        "table_election_transactional_tables_min": 3,
        "aggressive_prune_threshold": 75,  # Trigger Stratified Trim when portfolio > this; LLM aims to settle around 75 (50-100 zone).
        "bob_min_threshold": 20,  # v0.8.7: Run Best-of-Best LLM scoring (without trim) whenever portfolio >= this. Decoupled from trim gate so bob_score is populated for portfolios in the 20-75 zone too.
        "target_max_use_cases": 90,   # Base for effective cap; actual cap = target_max_use_cases + randint(1, target_max_jitter).
        "target_max_jitter": 20,      # Jitter amount; default (90, 20) → effective cap in [91, 110] to avoid round-number artifacts.
        "bob_scoring_batch_size": 25,
        "bob_theme_max_share": 0.15,
        "genie_regen_parallelism": 10,
        "scoring_split_retry_rounds": 2,
        "pass1_temperature": 0.25,
        "pass2_temperature": 0.1,  # B12: lowered from 0.5 (creative gap-fill) to 0.1 (table-grounded gap fill)
        "inspire_score_quality_weight": 0.65,
        "inspire_score_priority_weight": 0.35,
        "ultra_high_second_look_min_value": 8.0,
        "ultra_high_second_look_max": 25,
        "ultra_high_bob_prescreen_min": 5.0,
        "ultra_high_judge_batch_size": 5,
        "ultra_high_min_dimension_score": 8,
        "ultra_high_near_miss_band": 1,
        "ultra_high_allowed_value_tiers": ["Transformational", "Material"],
        "elite_judge_preferred_endpoints": ["databricks-claude-opus-4-8"],
        "ultra_high_score_weights": {"business_value": 0.40, "evidence_strength": 0.25,
                                     "implementability": 0.20, "actionability": 0.15},
        "deep_evidence_budget_seconds": 900,
        "deep_statement_timeout_seconds": 120,
        "deep_max_tables": 80,
        "deep_max_columns_per_table": 80,
        "deep_max_value_columns_per_table": 8,
        "deep_max_joins": 40,
        "deep_sample_rows": 1000000,
        "deep_history_days": 30,
        "deep_history_rows": 5000,
        "deep_min_join_match": 0.05,
        "deep_mostly_null_share": 0.95,
        "deep_min_signal_rows": 30,
        "deep_max_kpi_baselines": 300,
        "usage_signal_days": 90,
        "usage_election_keep_share": 0.25,
        "strategy_alignment_batch_size": 8,
        "ai_call_context_fill_max": 0.45,
        "ai_concurrency_floor": 4,
        "model_fallback_chain": [
            {"type": "thinker", "size": "large"},
            {"type": "worker", "size": "large"},
            {"type": "thinker", "size": "small"},
            {"type": "worker", "size": "small"},
            {"type": "thinker", "size": "tiny"},
            {"type": "worker", "size": "tiny"},
        ],
    },
    "prompts_models": [
        # === PHASE 1: INITIALIZATION & CONTEXT EXTRACTION ===
        # Temperature: 0.3-0.4 for accurate extraction of business context
        {"prompt_name": "BUSINESS_CONTEXT_WORKER_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.3, "stage_name": "Business Context", "step_name": "Business Context Worker", "progress_increment": 2.0},      # Step 1: Extract business context, goals, priorities
        {"prompt_name": "GENERAL_INSTRUCTION_INTERPRETER_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.0, "stage_name": "Initialization", "step_name": "Generation Instructions Interpreter", "progress_increment": 1.0},
        {"prompt_name": "FILTER_BUSINESS_TABLES_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.2, "stage_name": "Table Filtering", "step_name": "Business Table Filter", "progress_increment": 2.0},       # Step 3: Filter business vs technical tables (precision needed)
        
        # === PHASE 2: USE CASE GENERATION (PARALLEL) ===
        # Temperature: 0.25 for Pass 1 (near-deterministic exhaustive discovery), 0.5 for Pass 2 (creative gap-filling)
        {"prompt_name": "BASE_USE_CASE_GEN_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.25, "stage_name": "Use Cases Generation", "step_name": "Use Cases", "progress_increment": 4.0},
        
        # === PHASE 3: DOMAIN CLUSTERING ===
        # Temperature: 0.4-0.5 for balanced clustering decisions
        {"prompt_name": "DOMAIN_FINDER_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.5, "stage_name": "Business Domains Clustering", "step_name": "Domain Finder", "progress_increment": 2.0},                # Step 5: Cluster use cases into business domains
        {"prompt_name": "SUBDOMAIN_DETECTOR_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.4, "stage_name": "Business Domain Clustering", "step_name": "Subdomain Detector", "progress_increment": 2.0},           # Step 6: Detect subdomains within each domain
        {"prompt_name": "DOMAINS_MERGER_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.4, "stage_name": "Business Domains Clustering", "step_name": "Domain Merger", "progress_increment": 1.0},               # Step 7: Merge similar domains (optional)
        {"prompt_name": "SUBDOMAINS_MERGER_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.3, "stage_name": "Business Domain Clustering", "step_name": "Subdomain Merger", "progress_increment": 1.0},            # Step 7.5: Merge duplicate subdomains across domains
        {"prompt_name": "DOMAIN_SPLITTER_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.3, "stage_name": "Business Domain Clustering", "step_name": "Domain Splitter", "progress_increment": 1.0},                # Step 7.6: Split oversized mega-domains
        
        # === PHASE 4: SCORING & DEDUPLICATION (PARALLEL SCORING) ===
        # Temperature: 0.2 for consistent, accurate scoring
        {"prompt_name": "COMBINED_VALUE_QUALITY_SCORE_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.0, "stage_name": "Use Cases Scoring", "step_name": "Combined Value Quality Scoring", "progress_increment": 3.0}, # Step 8c: Combined value+quality scoring in one pass
        {"prompt_name": "BEST_OF_BEST_SCORING_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.0, "stage_name": "Use Case Prioritization", "step_name": "Best of Best Ranking", "progress_increment": 1.0}, # Step 8d: Per-gate 0-10 scoring for aggressive vetting
        {"prompt_name": "ULTRA_HIGH_ELITE_JUDGE_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.0, "stage_name": "Use Case Prioritization", "step_name": "Elite Selection", "progress_increment": 1.0},
        {"prompt_name": "REVIEW_USE_CASES_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.0, "stage_name": "Review", "step_name": "Use Cases Review", "progress_increment": 2.0},            # Step 9: Deduplication (conservative - true duplicates only, low temp for consistency)
        {"prompt_name": "GOAL_ALIGNMENT_FILTER_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.0, "stage_name": "Goal Filtering", "step_name": "Goal Alignment Check", "progress_increment": 1.0},
        {"prompt_name": "THEME_ASSIGNMENT_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.0, "stage_name": "Use Case Prioritization", "step_name": "Idea Themes", "progress_increment": 0.0},
        {"prompt_name": "STRATEGY_ALIGNMENT_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.0, "stage_name": "Strategy Alignment", "step_name": "Strategy Alignment", "progress_increment": 2.0},
        {"prompt_name": "Assign_User_Domains", "model_type": "worker", "model_size": "large", "temperature": 0.0, "stage_name": "Business Domains Clustering", "step_name": "User Domain Assignment", "progress_increment": 0.0},
        {"prompt_name": "canonicalize_sponsor_beneficiary_roles", "model_type": "worker", "model_size": "large", "temperature": 0.3, "stage_name": "Review", "step_name": "Sponsor Canonicalization", "progress_increment": 0.0},
        {"prompt_name": "PlaceholderDomainRename", "model_type": "worker", "model_size": "large", "temperature": 0.3, "stage_name": "Business Domains Clustering", "step_name": "Placeholder Domain Rename", "progress_increment": 0.0},
        {"prompt_name": "ENDPOINT_PROBE", "model_type": "worker", "model_size": "large", "temperature": 0.0, "stage_name": "Initialization", "step_name": "Endpoint Probe", "progress_increment": 0.0},
        
        
        # === PHASE 5b: GENIE CODE INSTRUCTIONS GENERATION ===
        # Temperature: 0.3 for balanced, detailed implementation plans
        {"prompt_name": "USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.3, "stage_name": "Genie Code Instructions", "step_name": "Genie Instruction Generator", "progress_increment": 4.0},
        
        # === PHASE 6: SUMMARY & ARTIFACTS ===
        # Temperature: 0.5 for engaging summaries
        {"prompt_name": "SUMMARY_GEN_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.5, "stage_name": "Summary", "step_name": "Executive Summary", "progress_increment": 1.0},                  # Step 12: Generate executive summary
        
        # === PHASE 7: TRANSLATION (MULTI-LANGUAGE) ===
        # Temperature: 0.2-0.3 for accurate translation
        {"prompt_name": "KEYWORDS_TRANSLATE_PROMPT", "model_type": "worker", "model_size": "tiny", "temperature": 0.2, "stage_name": "Translation", "step_name": "Keywords Translation", "progress_increment": 1.0},           # Step 14: Translate keywords (accuracy)
        {"prompt_name": "USE_CASE_TRANSLATE_PROMPT", "model_type": "worker", "model_size": "tiny", "temperature": 0.3, "stage_name": "Translation", "step_name": "Use Cases Translation", "progress_increment": 1.0},           # Step 15: Translate use cases
    ],
    "models": [
        {
            "name": "claude-opus-4-6",
            "order": 1,
            "type": "thinker",
            "size": "large",
            "enabled": True,
            "llm_endpoint_name": "databricks-claude-opus-4-6",
            "llm_input_context_tokens_count": 200000,
            "llm_output_context_tokens_count": 128000
        },
        {
            "name": "claude-sonnet-4-6",
            "order": 2,
            "type": "worker",
            "size": "large",
            "enabled": True,
            "llm_endpoint_name": "databricks-claude-sonnet-4-6",
            "llm_input_context_tokens_count": 200000,
            "llm_output_context_tokens_count": 64000
        },
        {
            "name": "claude-opus-4-5",
            "order": 3,
            "type": "thinker",
            "size": "large",
            "enabled": True,
            "llm_endpoint_name": "databricks-claude-opus-4-5",
            "llm_input_context_tokens_count": 200000,
            "llm_output_context_tokens_count": 64000
        },
        {
            "name": "claude-sonnet-4-5",
            "order": 4,
            "type": "worker",
            "size": "large",
            "enabled": True,
            "llm_endpoint_name": "databricks-claude-sonnet-4-5",
            "llm_input_context_tokens_count": 200000,
            "llm_output_context_tokens_count": 64000
        },
        {
            "name": "gpt-oss-120b",
            "order": 5,
            "type": "worker",
            "size": "small",
            "enabled": True,
            "llm_endpoint_name": "databricks-gpt-oss-120b",
            "llm_input_context_tokens_count": 131072,
            "llm_output_context_tokens_count": 25000
        },
        {
            "name": "gpt-oss-20b",
            "order": 6,
            "type": "worker",
            "size": "tiny",
            "enabled": True,
            "llm_endpoint_name": "databricks-gpt-oss-20b",
            "llm_input_context_tokens_count": 131072,
            "llm_output_context_tokens_count": 25000
        }
    ]
}

MULTI_ENTITY_PROGRESS_PROMPTS = {
    "COMBINED_VALUE_QUALITY_SCORE_PROMPT",
    "BEST_OF_BEST_SCORING_PROMPT",
    "ULTRA_HIGH_ELITE_JUDGE_PROMPT",
    "REVIEW_USE_CASES_PROMPT",
    "STRATEGY_ALIGNMENT_PROMPT",
    "KEYWORDS_TRANSLATE_PROMPT",
    "USE_CASE_TRANSLATE_PROMPT",
}

PROMPT_STORY_TITLES = {
    "BUSINESS_CONTEXT_WORKER_PROMPT": ("Business Understanding", "Understanding Business Priorities"),
    "GENERAL_INSTRUCTION_INTERPRETER_PROMPT": ("Business Understanding", "Interpreting Generation Instructions"),
    "FILTER_BUSINESS_TABLES_PROMPT": ("Data Preparation", "Selecting Business Tables"),
    "BASE_USE_CASE_GEN_PROMPT": ("Use Case Design", "Generating Use Cases"),
    "DOMAIN_FINDER_PROMPT": ("Domain Mapping", "Mapping Business Domains"),
    "SUBDOMAIN_DETECTOR_PROMPT": ("Domain Mapping", "Refining Business Subdomains"),
    "DOMAINS_MERGER_PROMPT": ("Domain Mapping", "Consolidating Overlapping Domains"),
    "SUBDOMAINS_MERGER_PROMPT": ("Domain Mapping", "Deduplicating Subdomains Across Domains"),
    "DOMAIN_SPLITTER_PROMPT": ("Domain Mapping", "Rebalancing Oversized Domains"),
    "COMBINED_VALUE_QUALITY_SCORE_PROMPT": ("Use Case Prioritization", "Ranking Use Cases"),
    "BEST_OF_BEST_SCORING_PROMPT": ("Use Case Prioritization", "Best-of-Best Ranking"),
    "ULTRA_HIGH_ELITE_JUDGE_PROMPT": ("Use Case Prioritization", "Elite Selection"),
    "REVIEW_USE_CASES_PROMPT": ("Use Case Prioritization", "Reviewing and Removing Duplicates"),
    "GOAL_ALIGNMENT_FILTER_PROMPT": ("Goal Filtering", "Filtering Use Cases by Goal Alignment"),
    "THEME_ASSIGNMENT_PROMPT": ("Use Case Prioritization", "Naming Idea Themes"),
    "STRATEGY_ALIGNMENT_PROMPT": ("Strategy Alignment", "Aligning Use Cases to the Strategy"),
    "SUMMARY_GEN_PROMPT": ("Executive Readout", "Building Executive Summary"),
    "KEYWORDS_TRANSLATE_PROMPT": ("Localization", "Translating Business Keywords"),
    "USE_CASE_TRANSLATE_PROMPT": ("Localization", "Translating Use Cases"),
}

PROMPT_STORY_STARTED_MESSAGES = {
    "BUSINESS_CONTEXT_WORKER_PROMPT": "Inspire has started understanding business priorities and strategic goals",
    "GENERAL_INSTRUCTION_INTERPRETER_PROMPT": "Inspire has started interpreting generation instructions",
    "FILTER_BUSINESS_TABLES_PROMPT": "Inspire has started inspecting business tables for relevance",
    "BASE_USE_CASE_GEN_PROMPT": "Inspire has started generating business use cases",
    "DOMAIN_FINDER_PROMPT": "Inspire has started organizing use cases into business domains",
    "SUBDOMAIN_DETECTOR_PROMPT": "Inspire has started refining domains into subdomains",
    "DOMAINS_MERGER_PROMPT": "Inspire has started consolidating overlapping domains",
    "SUBDOMAINS_MERGER_PROMPT": "Inspire has started deduplicating subdomains across domains",
    "DOMAIN_SPLITTER_PROMPT": "Inspire has started rebalancing oversized domains",
    "COMBINED_VALUE_QUALITY_SCORE_PROMPT": "Inspire has started ranking use cases by combined value and quality",
    "BEST_OF_BEST_SCORING_PROMPT": "Inspire has started the Best-of-Best ranking with 19-gate per-UC scoring",
    "ULTRA_HIGH_ELITE_JUDGE_PROMPT": "Inspire has started the Ultra High elite selection",
    "REVIEW_USE_CASES_PROMPT": "Inspire has started reviewing use cases to remove overlap and improve quality",
    "GOAL_ALIGNMENT_FILTER_PROMPT": "Inspire has started filtering use cases that do not align with user-specified goal",
    "THEME_ASSIGNMENT_PROMPT": "Inspire has started naming the idea themes of the whole portfolio",
    "STRATEGY_ALIGNMENT_PROMPT": "Inspire has started aligning the selected use cases to the strategy and defining their KPIs and benefits",
    "SUMMARY_GEN_PROMPT": "Inspire has started creating the executive summary",
    "KEYWORDS_TRANSLATE_PROMPT": "Inspire has started translating business terminology",
    "USE_CASE_TRANSLATE_PROMPT": "Inspire has started translating use case outcomes",
}
BEST_OF_BEST_WORKER_PREFIX = "BestOfBest"
BEST_OF_BEST_STORY_TITLE = ("Use Case Prioritization", "Best of Best")
BEST_OF_BEST_STORY_STARTED = "Inspire has started checking which use cases pass every Best of Best test"

INLINE_PROMPT_NAMES = frozenset({"Assign_User_Domains", "canonicalize_sponsor_beneficiary_roles",
                                 "PlaceholderDomainRename", "ENDPOINT_PROBE"})

ULTRA_HIGH_DIMENSIONS = ("business_value", "evidence_strength", "implementability", "actionability")
ULTRA_HIGH_VALUE_TIERS = ("Transformational", "Material", "Incremental", "Marginal")
ULTRA_HIGH_FATAL_FLAW_CATEGORIES = ("OUTCOME_DERIVED_FROM_INPUTS", "NO_DECISION_CHANGED", "DUPLICATE_OF_ANOTHER_USE_CASE",
                                    "DATA_SHAPE_ASSUMPTION_UNVERIFIABLE", "INFEASIBLE_WITH_AVAILABLE_COLUMNS", "VALUE_OVERCLAIMED")
ULTRA_HIGH_CONDITIONAL_FLAW_CATEGORIES = ("DATA_SHAPE_ASSUMPTION_UNVERIFIABLE",)
ULTRA_HIGH_RESIDUAL_RISK = "RESIDUAL_RISK"
ULTRA_HIGH_LABEL_TIMINGS = ("future_window", "same_window")

def _validate_runtime_config():
    _rt = TECHNICAL_CONTEXT["runtime"]
    _bounds = {
        "llm_timeout_seconds": (30, 1800),
        "max_retry_attempts": (0, 5),
        "llm_endpoint_demote_after_timeouts": (1, 50),
        "scan_parallelism": (1, 20),
        "metadata_parallelism": (1, 20),
        "global_min_parallelism": (1, 20),
        "global_max_parallelism": (2, 50),
        "default_max_parallelism": (1, 50),
        "cluster_memory_gb": (4, 512),
        "cluster_worker_count": (1, 64),
        "quality_scoring_batch_size": (5, 200),
        "quality_scoring_max_workers": (1, 20),
        "batch_processing_timeout": (60, 3600),
        "batch_processing_max_attempts": (1, 10),
        "table_election_threshold": (1, 500),
        "table_election_transactional_tables_min": (1, 100),
        "aggressive_prune_threshold": (1, 10000),
        "bob_min_threshold": (1, 10000),
        "target_max_use_cases": (1, 10000),
        "target_max_jitter": (1, 1000),
        "bob_scoring_batch_size": (5, 200),
        "bob_theme_max_share": (0.01, 1.0),
        "genie_regen_parallelism": (1, 50),
        "scoring_split_retry_rounds": (0, 4),
        "pass1_temperature": (0.0, 1.0),
        "pass2_temperature": (0.0, 1.0),
        "inspire_score_quality_weight": (0.0, 1.0),
        "inspire_score_priority_weight": (0.0, 1.0),
        "ultra_high_second_look_min_value": (0.0, 10.0),
        "ultra_high_second_look_max": (0, 500),
        "ultra_high_bob_prescreen_min": (0.0, 10.0),
        "ultra_high_judge_batch_size": (1, 25),
        "ultra_high_min_dimension_score": (1, 10),
        "ultra_high_near_miss_band": (0, 3),
        "deep_evidence_budget_seconds": (30, 7200),
        "deep_statement_timeout_seconds": (5, 1800),
        "deep_max_tables": (0, 1000),
        "deep_max_columns_per_table": (1, 1000),
        "deep_max_value_columns_per_table": (0, 100),
        "deep_max_joins": (0, 1000),
        "deep_sample_rows": (1000, 100000000),
        "deep_history_days": (1, 365),
        "deep_history_rows": (0, 100000),
        "deep_min_join_match": (0.0, 1.0),
        "deep_mostly_null_share": (0.5, 1.0),
        "deep_min_signal_rows": (0, 1000000),
        "deep_max_kpi_baselines": (0, 1000),
        "usage_signal_days": (1, 365),
        "usage_election_keep_share": (0.0, 1.0),
        "strategy_alignment_batch_size": (1, 50),
        "ai_call_context_fill_max": (0.1, 0.95),
        "ai_concurrency_floor": (1, 20),
    }
    for key, (lo, hi) in _bounds.items():
        val = _rt.get(key)
        if val is None:
            raise ValueError(f"TECHNICAL_CONTEXT.runtime missing required key: '{key}'")
        if not isinstance(val, (int, float)):
            raise TypeError(f"TECHNICAL_CONTEXT.runtime['{key}'] must be numeric, got {type(val).__name__}")
        if val < lo or val > hi:
            raise ValueError(f"TECHNICAL_CONTEXT.runtime['{key}']={val} out of bounds [{lo}, {hi}]")

    for key in ("ultra_high_second_look_min_value", "ultra_high_bob_prescreen_min"):
        if isinstance(_rt[key], bool):
            raise TypeError(f"TECHNICAL_CONTEXT.runtime['{key}'] must be a number, got bool")
    for key in ("ultra_high_second_look_max", "ultra_high_judge_batch_size",
                "ultra_high_min_dimension_score", "ultra_high_near_miss_band"):
        if isinstance(_rt[key], bool) or not isinstance(_rt[key], int):
            raise TypeError(f"TECHNICAL_CONTEXT.runtime['{key}'] must be an integer, got {type(_rt[key]).__name__}")
    _tiers = _rt.get("ultra_high_allowed_value_tiers")
    if not isinstance(_tiers, list) or not _tiers:
        raise ValueError("TECHNICAL_CONTEXT.runtime.ultra_high_allowed_value_tiers must be a non-empty list")
    _unknown_tiers = [t for t in _tiers if t not in ULTRA_HIGH_VALUE_TIERS]
    if _unknown_tiers or len(set(_tiers)) != len(_tiers):
        raise ValueError(f"TECHNICAL_CONTEXT.runtime.ultra_high_allowed_value_tiers must be distinct values of "
                         f"{list(ULTRA_HIGH_VALUE_TIERS)}, got {_tiers}")
    _weights = _rt.get("ultra_high_score_weights")
    if not isinstance(_weights, dict) or set(_weights) != set(ULTRA_HIGH_DIMENSIONS):
        raise ValueError(f"TECHNICAL_CONTEXT.runtime.ultra_high_score_weights must have exactly the keys "
                         f"{list(ULTRA_HIGH_DIMENSIONS)}")
    for _dim, _weight in _weights.items():
        if isinstance(_weight, bool) or not isinstance(_weight, (int, float)) or not 0.0 <= _weight <= 1.0:
            raise ValueError(f"TECHNICAL_CONTEXT.runtime.ultra_high_score_weights['{_dim}']={_weight} must be a number in [0, 1]")
    if abs(sum(_weights.values()) - 1.0) > 1e-6:
        raise ValueError(f"TECHNICAL_CONTEXT.runtime.ultra_high_score_weights must sum to 1.0, got {sum(_weights.values())}")

    _judges = _rt.get("elite_judge_preferred_endpoints")
    if not isinstance(_judges, list) or not all(isinstance(e, str) and e.strip() for e in _judges):
        raise ValueError("TECHNICAL_CONTEXT.runtime.elite_judge_preferred_endpoints must be a list of endpoint names")

    _chain = _rt.get("model_fallback_chain")
    if not isinstance(_chain, list) or not _chain:
        raise ValueError("TECHNICAL_CONTEXT.runtime.model_fallback_chain must be a non-empty list")
    _valid_types = {"worker", "thinker"}
    _valid_sizes = {"tiny", "small", "large"}
    for _idx, _entry in enumerate(_chain):
        if not isinstance(_entry, dict):
            raise TypeError(f"model_fallback_chain[{_idx}] must be a dict, got {type(_entry).__name__}")
        if _entry.get("type") not in _valid_types:
            raise ValueError(f"model_fallback_chain[{_idx}].type must be one of {sorted(_valid_types)}")
        if _entry.get("size") not in _valid_sizes:
            raise ValueError(f"model_fallback_chain[{_idx}].size must be one of {sorted(_valid_sizes)}")

_validate_runtime_config()


class _LLMEndpointHealth:
    def __init__(self):
        import threading as _threading
        self._lock = _threading.Lock()
        self._unusable = {}
        self._no_temperature = set()
        self._probed = set()
        self._json_format = set()
        self._json_format_decided = set()
        self._probe_outcomes = {}
        self._timeouts = {}
        self._demoted = []

    def record_timeout(self, endpoint, logger=None, source="runtime") -> bool:
        limit = TECHNICAL_CONTEXT["runtime"]["llm_endpoint_demote_after_timeouts"]
        with self._lock:
            if not endpoint:
                return False
            count = self._timeouts.get(endpoint, 0) + 1
            self._timeouts[endpoint] = count
            demote = count >= limit and endpoint not in self._demoted
            if demote:
                self._demoted.append(endpoint)
        if demote and logger is not None:
            logger.warning(f"[LLM_ENDPOINT_DEMOTED FIRED] endpoint={endpoint} timed out {count} times in this run "
                           f"(last: {source}); every cascade tries it after the other endpoints for the rest of the run")
        return demote

    def demoted_last(self, models) -> list:
        with self._lock:
            demoted = list(self._demoted)
        if not demoted:
            return list(models)
        return ([m for m in models if m.get("llm_endpoint_name") not in demoted]
                + [m for m in models if m.get("llm_endpoint_name") in demoted])

    def record_probe_outcome(self, endpoint, outcome):
        with self._lock:
            if endpoint:
                self._probe_outcomes[endpoint] = outcome

    def probe_outcome(self, endpoint):
        with self._lock:
            return self._probe_outcomes.get(endpoint)

    def is_usable(self, endpoint) -> bool:
        with self._lock:
            return endpoint not in self._unusable

    def unusable_reason(self, endpoint):
        with self._lock:
            return self._unusable.get(endpoint)

    def mark_unusable(self, endpoint, reason, logger=None, source="runtime") -> bool:
        reason = " ".join(str(reason or "unspecified capability or permission error").split())[:300]
        with self._lock:
            if not endpoint or endpoint in self._unusable:
                return False
            self._unusable[endpoint] = reason
        if logger is not None:
            logger.warning(f"[LLM_ENDPOINT_UNUSABLE FIRED] endpoint={endpoint} source={source}: {reason}. "
                           f"Every model cascade skips this endpoint for the rest of the run.")
        return True

    def accepts_temperature(self, endpoint) -> bool:
        with self._lock:
            return endpoint not in self._no_temperature

    def mark_temperature_unsupported(self, endpoint, logger=None, source="runtime") -> bool:
        with self._lock:
            if not endpoint or endpoint in self._no_temperature:
                return False
            self._no_temperature.add(endpoint)
        if logger is not None:
            logger.warning(f"[G32 TEMPERATURE_REJECTED_ADAPTED FIRED] endpoint={endpoint} source={source}: the endpoint "
                           f"does not accept a temperature parameter. Its calls are sent without one and run at the "
                           f"endpoint's default sampling, so the per-prompt temperatures configured in "
                           f"TECHNICAL_CONTEXT are not applied on this endpoint.")
        return True

    def claim_probe(self, endpoint) -> bool:
        with self._lock:
            if not endpoint or endpoint in self._probed:
                return False
            self._probed.add(endpoint)
            return True

    def accepts_json_format(self, endpoint) -> bool:
        with self._lock:
            return endpoint in self._json_format

    def mark_json_format(self, endpoint, supported: bool, logger=None, source="startup probe", reason=""):
        with self._lock:
            known = endpoint in self._json_format
            first = endpoint not in self._json_format_decided
            self._json_format_decided.add(endpoint)
            if supported:
                self._json_format.add(endpoint)
            else:
                self._json_format.discard(endpoint)
        if logger is not None and (first or bool(supported) != known):
            logger.info(f"[JSON_FORMAT {'ENABLED' if supported else 'DISABLED'}] endpoint={endpoint} source={source}: "
                        f"the JSON prompts {'use' if supported else 'do not use'} a strict JSON response format there"
                        + (f" ({reason})" if reason and not supported else ""))

    def snapshot(self) -> dict:
        with self._lock:
            return {"unusable": dict(self._unusable),
                    "temperature_unsupported": sorted(self._no_temperature),
                    "probed": sorted(self._probed),
                    "json_format": sorted(self._json_format),
                    "demoted": list(self._demoted)}


LLM_ENDPOINT_HEALTH = _LLMEndpointHealth()


def _model_is_selectable(model) -> bool:
    return bool(model.get("enabled", True)) and LLM_ENDPOINT_HEALTH.is_usable(model.get("llm_endpoint_name"))


def _validate_model_config():
    valid_types = {"worker", "thinker"}
    valid_sizes = {"tiny", "small", "large"}
    models = TECHNICAL_CONTEXT.get("models", [])
    prompt_models = TECHNICAL_CONTEXT.get("prompts_models", [])

    if not isinstance(models, list) or not models:
        raise ValueError("TECHNICAL_CONTEXT.models must be a non-empty list")

    for idx, model in enumerate(models):
        model_type = model.get("type")
        model_size = model.get("size")
        enabled = model.get("enabled")
        if model_type not in valid_types:
            raise ValueError(f"TECHNICAL_CONTEXT.models[{idx}].type must be one of {sorted(valid_types)}")
        if model_size not in valid_sizes:
            raise ValueError(f"TECHNICAL_CONTEXT.models[{idx}].size must be one of {sorted(valid_sizes)}")
        if not isinstance(enabled, bool):
            raise TypeError(f"TECHNICAL_CONTEXT.models[{idx}].enabled must be boolean")

    for idx, pm in enumerate(prompt_models):
        model_type = pm.get("model_type", "worker")
        model_size = pm.get("model_size", DEFAULT_MODEL_SIZE_BY_TYPE.get(model_type, "small"))
        if model_type not in valid_types:
            raise ValueError(f"TECHNICAL_CONTEXT.prompts_models[{idx}].model_type must be one of {sorted(valid_types)}")
        if model_size not in valid_sizes:
            raise ValueError(f"TECHNICAL_CONTEXT.prompts_models[{idx}].model_size must be one of {sorted(valid_sizes)}")
        if not _bucket_cascade(model_type, model_size):
            raise ValueError(
                f"No enabled model reachable for prompts_models[{idx}] '{pm.get('prompt_name')}' "
                f"(designated bucket: {model_type}/{model_size}) via fallback chain"
            )


def get_models_by_type(model_type: str) -> list:
    """
    Get all models of a given type (thinker/worker) sorted by order (ascending).
    The lowest-order model is tried first; on failure, the next one is used.
    """
    return sorted(
        [m for m in TECHNICAL_CONTEXT["models"] if m.get("type") == model_type and _model_is_selectable(m)],
        key=lambda m: m.get("order", 999)
    )

def get_models_by_type_and_size(model_type: str, model_size: str) -> list:
    """
    Get enabled models matching (type, size), sorted by order (ascending).
    """
    return sorted(
        [
            m for m in TECHNICAL_CONTEXT["models"]
            if m.get("type") == model_type and m.get("size") == model_size and _model_is_selectable(m)
        ],
        key=lambda m: m.get("order", 999)
    )

def get_model_cascade_for_prompt(prompt_name: str) -> list:
    """
    Get the ordered list of model configs the LLM client will try for a prompt.
    Used for cascade fallback: try model[0], on failure try model[1], etc.

    Falls back to worker models with default worker size if prompt_name is not in prompts_models
    (e.g., dynamic prompt names like 'Assign_User_Domains_English').
    """
    return get_full_fallback_cascade(prompt_name)

def _get_prompt_bucket(prompt_name: str) -> tuple:
    for pm in TECHNICAL_CONTEXT["prompts_models"]:
        if pm["prompt_name"] == prompt_name:
            mt = pm.get("model_type", "worker")
            ms = pm.get("model_size", DEFAULT_MODEL_SIZE_BY_TYPE.get(mt, "small"))
            return (mt, ms)
    default_type = "worker"
    return (default_type, DEFAULT_MODEL_SIZE_BY_TYPE.get(default_type, "small"))

def _get_prompt_temperature(prompt_name: str) -> float:
    for pm in TECHNICAL_CONTEXT["prompts_models"]:
        if pm["prompt_name"] == prompt_name:
            return pm.get("temperature", 0.3)
    return 0.3


def _bucket_cascade(designated_type: str, designated_size: str) -> list:
    chain = TECHNICAL_CONTEXT["runtime"].get("model_fallback_chain", [])
    buckets = [(entry.get("type"), entry.get("size")) for entry in chain]
    if (designated_type, designated_size) in buckets:
        start = buckets.index((designated_type, designated_size))
        forward = buckets[start:]
        backward = list(reversed(buckets[:start]))
    else:
        forward = [(designated_type, designated_size)]
        backward = list(reversed(buckets))
    cascade = []
    for ct, cs in forward:
        cascade.extend(get_models_by_type_and_size(ct, cs))
    if cascade:
        return cascade
    for ct, cs in backward:
        cascade.extend(get_models_by_type_and_size(ct, cs))
    return cascade


def get_full_fallback_cascade(prompt_name: str) -> list:
    designated_type, designated_size = _get_prompt_bucket(prompt_name)
    cascade = _bucket_cascade(designated_type, designated_size)
    if not cascade:
        all_enabled = sorted(
            [m for m in TECHNICAL_CONTEXT["models"] if _model_is_selectable(m)],
            key=lambda m: m.get("order", 999)
        )
        cascade = all_enabled
    return LLM_ENDPOINT_HEALTH.demoted_last(cascade)


def cascade_from_model(cascade: list, start_model: str) -> list:
    def _is_start(m):
        return start_model in (m.get("llm_endpoint_name"), m.get("name"))

    for idx, m in enumerate(cascade or []):
        if _is_start(m):
            return list(cascade[idx:])
    start = next((m for m in sorted(TECHNICAL_CONTEXT["models"], key=lambda m: m.get("order", 999))
                  if _is_start(m) and _model_is_selectable(m)), None)
    if start is None:
        return []
    start_type = start.get("type", "worker")
    chain = _bucket_cascade(start_type, start.get("size", DEFAULT_MODEL_SIZE_BY_TYPE.get(start_type, "small")))
    for idx, m in enumerate(chain):
        if m.get("llm_endpoint_name") == start.get("llm_endpoint_name"):
            return chain[idx:]
    return [start] + chain

_validate_model_config()


def get_model_endpoint_for_prompt(prompt_name: str) -> str:
    """
    Get the primary LLM endpoint name for a given prompt using model_type cascade.
    Returns the endpoint of the lowest-order model matching the prompt's model_type.
    """
    cascade = get_model_cascade_for_prompt(prompt_name)
    if cascade:
        return cascade[0]["llm_endpoint_name"]
    all_models = sorted(
        [m for m in TECHNICAL_CONTEXT["models"] if _model_is_selectable(m)],
        key=lambda m: m.get("order", 999)
    )
    return all_models[0]["llm_endpoint_name"] if all_models else "databricks-gpt-oss-20b"

def get_model_config_for_prompt(prompt_name: str) -> dict:
    """
    Get the primary model configuration for a given prompt using model_type cascade.
    Returns the config of the lowest-order model matching the prompt's model_type.
    """
    cascade = get_model_cascade_for_prompt(prompt_name)
    if cascade:
        return cascade[0]
    all_models = sorted(
        [m for m in TECHNICAL_CONTEXT["models"] if _model_is_selectable(m)],
        key=lambda m: m.get("order", 999)
    )
    return all_models[0] if all_models else {}

LLM_FAMILY_SPECS = (
    ("claude-opus", "sonnet|haiku", "thinker", "large", 200000, 128000),
    ("claude-sonnet", "opus|haiku", "worker", "large", 200000, 64000),
    ("claude-haiku", "opus|sonnet", "worker", "small", 200000, 32000),
    ("meta-llama|llama-3|llama-4", "gpt|claude|mistral", "thinker", "large", 131072, 32000),
    ("gpt-oss-120b|gpt-oss-large", r"\b20b\b|small", "worker", "small", 131072, 25000),
    ("gpt-oss-20b|gpt-oss-small", r"\b120b\b|large", "worker", "tiny", 131072, 25000),
    ("gpt-oss", "", "worker", "tiny", 131072, 25000),
)


def _llm_endpoint_version(name):
    import re as _re
    match = _re.search(r'(\d+)[-_.](\d+)(?:[-_.](\d+))?', name)
    return tuple(int(x) if x else 0 for x in match.groups()) if match else (0, 0, 0)


def _family_endpoints(endpoints, family_pat, exclude_pat, seen, logger=None) -> list:
    """Databricks-hosted endpoints of one family that can serve ai_query, newest version first, as (name, version)."""
    import re as _re
    family_re = _re.compile(family_pat, _re.IGNORECASE)
    exclude_re = _re.compile(exclude_pat, _re.IGNORECASE) if exclude_pat else None
    matched = []
    for ep in endpoints:
        ep_name = getattr(ep, "name", None) or ""
        if not ep_name or ep_name in seen or not ep_name.lower().startswith("databricks-") or not family_re.search(ep_name):
            continue
        if exclude_re and exclude_re.search(ep_name):
            continue
        _excluded_because = _endpoint_ai_query_exclusion(ep)
        if _excluded_because:
            seen.add(ep_name)
            if logger is not None:
                logger.info(f"[LLM_ENDPOINT_EXCLUDED endpoint={ep_name}] {_excluded_because}; not promoted to the model roster")
            continue
        matched.append((ep_name, _llm_endpoint_version(ep_name)))
    return sorted(matched, key=lambda x: x[1], reverse=True)


def _list_fast_chat_endpoints(logger) -> list:
    """Every Databricks-hosted chat endpoint of a worker family (Claude Sonnet and Haiku, GPT OSS), family by family and
    newest first, whether or not it made the model roster. On any error, returns []."""
    try:
        from databricks.sdk import WorkspaceClient as _WC
        endpoints = list(_WC().serving_endpoints.list())
    except Exception as _e:
        logger.warning(f"Fast chat endpoint listing failed at serving_endpoints.list(): {_e}")
        return []
    seen, names = set(), []
    for family_pat, exclude_pat, mtype, _size, _in_tokens, _out_tokens in LLM_FAMILY_SPECS:
        if mtype != "worker":
            continue
        for ep_name, _ver in _family_endpoints(endpoints, family_pat, exclude_pat, seen):
            seen.add(ep_name)
            names.append(ep_name)
    return names


def _auto_detect_llm_endpoints(logger, max_endpoints: int = 6) -> list:
    """Auto-detect Databricks foundation-model serving endpoints.
    Ranks by architectural family (Claude opus > Claude sonnet > Claude haiku >
    Llama > GPT OSS large > GPT OSS small) and version (latest first). Returns
    a list matching the TECHNICAL_CONTEXT['models'] schema. On any error,
    returns [] so caller falls back to hardcoded list.
    Sentinel [LLM_ENDPOINT_DETECTED family=... version=...] fires per endpoint.
    Industry-agnostic: family patterns only, NO hardcoded specific endpoint IDs.
    """
    try:
        from databricks.sdk import WorkspaceClient as _WC
        _w = _WC()
        _endpoints = list(_w.serving_endpoints.list())
    except Exception as _e:
        logger.warning(f"LLM endpoint auto-detection failed at serving_endpoints.list(): {_e}. Falling back to hardcoded model list.")
        return []
    ranked = []
    seen = set()
    for family_pat, exclude_pat, mtype, msize, in_tokens, out_tokens in LLM_FAMILY_SPECS:
        family_label = family_pat.split("|")[0]
        for ep_name, ver in _family_endpoints(_endpoints, family_pat, exclude_pat, seen, logger):
            if ep_name in seen:
                continue
            seen.add(ep_name)
            ver_str = ".".join(str(v) for v in ver if v > 0) or "unknown"
            ranked.append({
                "name": ep_name.replace("databricks-", ""),
                "order": len(ranked) + 1,
                "type": mtype,
                "size": msize,
                "enabled": True,
                "llm_endpoint_name": ep_name,
                "llm_input_context_tokens_count": in_tokens,
                "llm_output_context_tokens_count": out_tokens,
                "family": family_label,
                "version": ver_str,
            })
            logger.info(f"[LLM_ENDPOINT_DETECTED family={family_label} version={ver_str} endpoint={ep_name} type={mtype} size={msize}]")
            if len(ranked) >= max_endpoints:
                break
        if len(ranked) >= max_endpoints:
            break
    return ranked


_AI_QUERY_TEXT_TASKS = ("llm/v1/chat", "llm/v1/completions")
_LLM_ENDPOINT_PROBE_PROMPT = "Reply with the single word OK."
_LLM_ENDPOINT_PROBE_MAX_TOKENS = 16
_LLM_JSON_PROBE_MAX_TOKENS = 128
_LLM_ENDPOINT_PROBE_TIMEOUT_S = 60
_LLM_JSON_PROBE_TIMEOUT_S = 20
_LLM_JSON_PROBE_PROMPT = 'Return exactly this JSON object and nothing else: {"results": [{"ok": true}]}'
LLM_JSON_FORMAT_PROMPTS = frozenset({"STRATEGY_ALIGNMENT_PROMPT", "COMBINED_VALUE_QUALITY_SCORE_PROMPT",
                                     "BEST_OF_BEST_SCORING_PROMPT", "ULTRA_HIGH_ELITE_JUDGE_PROMPT", "THEME_ASSIGNMENT_PROMPT"})
JSON_ARRAY_RESPONSE_KEY = "results"
JSON_ARRAY_RESPONSE_SCHEMA = {"name": "json_array", "schema": {
    "type": "object", "properties": {JSON_ARRAY_RESPONSE_KEY: {"type": "array", "items": {"type": "object"}}},
    "required": [JSON_ARRAY_RESPONSE_KEY]}}
_RESPONSE_FORMAT_ERROR_MARKERS = ("response_format", "responseformat", "json_schema", "response format", "input_schema",
                                  "structured output")


def is_json_array_text(text) -> bool:
    try:
        return isinstance(json.loads(clean_json_response(text) or "null"), list)
    except (TypeError, ValueError):
        return False


def unwrap_json_array_response(text):
    try:
        parsed = json.loads(clean_json_response(text) or "null")
    except (TypeError, ValueError):
        return text
    if isinstance(parsed, dict) and isinstance(parsed.get(JSON_ARRAY_RESPONSE_KEY), list):
        return json.dumps(parsed[JSON_ARRAY_RESPONSE_KEY], ensure_ascii=False)
    return text


def _endpoint_ai_query_exclusion(endpoint) -> str:
    task = getattr(endpoint, "task", None)
    if task and str(task).strip().lower() not in _AI_QUERY_TEXT_TASKS:
        return f"task {task} is not a chat or completions task that ai_query can serve with a text prompt"
    state = getattr(endpoint, "state", None)
    ready = getattr(state, "ready", None) if state is not None else None
    ready_value = str(getattr(ready, "value", ready) or "").strip().upper()
    if ready_value and ready_value != "READY":
        return f"endpoint state is {ready_value}"
    config = getattr(endpoint, "config", None)
    entities = getattr(config, "served_entities", None) if config is not None else None
    if entities is not None and not list(entities):
        return "endpoint serves no model"
    return ""


def _spark_session_or_none(candidate=None):
    session = candidate if candidate is not None else globals().get("spark")
    try:
        return session if session is not None and callable(getattr(session, "sql", None)) else None
    except Exception:
        return None


def _probe_send(spark_session, endpoint, logger, prompt_text, response_format=None, timeout_seconds=None,
                max_output_tokens=_LLM_ENDPOINT_PROBE_MAX_TOKENS):
    """One logged probe statement: an attempt row, and an exhausted row when it fails."""
    temperature = 0.0 if LLM_ENDPOINT_HEALTH.accepts_temperature(endpoint) else None
    request = ailog_new_request("ENDPOINT_PROBE")
    log_attempt = AILogAttempt(request, endpoint, prompt_text, max_output_tokens, temperature, response_format)
    try:
        return _run_tagged_spark_work(
            spark_session,
            lambda: _ai_query_rows(spark_session, endpoint, prompt_text, max_output_tokens,
                                   temperature=temperature, response_format=response_format, logger=logger,
                                   log_attempt=log_attempt),
            timeout_seconds=timeout_seconds or _LLM_ENDPOINT_PROBE_TIMEOUT_S,
            tag_prefix="inspire_probe", label=f"probe {endpoint}", logger=logger)
    except Exception as exc:
        log_attempt.settle("timedout" if isinstance(exc, SparkWorkTimeout) else ailog_status_for_error(exc),
                           AILOGS_TIMEOUT_NOTE if isinstance(exc, SparkWorkTimeout)
                           else get_clean_error_message(exc, max_chars=AILOGS_ERROR_CHARS))
        AILOG_RECORDER.record_exhausted(request, prompt_text, exc, endpoint_hint=endpoint)
        raise


def _probe_json_format(spark_session, endpoint, logger, timeout_seconds=None) -> bool:
    """A strict JSON array response format is used on an endpoint only after it answered this probe."""
    response_format = json.dumps({"type": "json_schema", "json_schema": JSON_ARRAY_RESPONSE_SCHEMA}, separators=(",", ":"))
    timeout_seconds = timeout_seconds or _LLM_JSON_PROBE_TIMEOUT_S
    try:
        rows = _probe_send(spark_session, endpoint, logger, _LLM_JSON_PROBE_PROMPT, response_format=response_format,
                           timeout_seconds=timeout_seconds, max_output_tokens=_LLM_JSON_PROBE_MAX_TOKENS)
    except SparkWorkTimeout:
        LLM_ENDPOINT_HEALTH.mark_json_format(endpoint, False, logger, reason=f"no answer within {timeout_seconds} s")
        return False
    except Exception as exc:
        message = get_clean_error_message(exc, max_chars=200)
        refused = any(marker in message.lower() for marker in _RESPONSE_FORMAT_ERROR_MARKERS)
        LLM_ENDPOINT_HEALTH.mark_json_format(
            endpoint, False, logger, reason=f"{'the endpoint refused it' if refused else 'the check failed'}: {message}")
        return False
    text = str(getattr(rows[0], "ai_response", "") or "") if rows else ""
    try:
        supported = isinstance(json.loads(clean_json_response(unwrap_json_array_response(text)) or "null"), list)
    except (TypeError, ValueError):
        supported = False
    LLM_ENDPOINT_HEALTH.mark_json_format(endpoint, supported, logger,
                                         reason="" if supported else "its answer was not a JSON array")
    return supported


def _probe_llm_endpoint(spark_session, endpoint, logger, timeout_seconds=None) -> str:
    try:
        _probe_send(spark_session, endpoint, logger, _LLM_ENDPOINT_PROBE_PROMPT, timeout_seconds=timeout_seconds)
        _probe_json_format(spark_session, endpoint, logger, timeout_seconds)
        return "usable"
    except Exception as exc:
        kind = _classify_runtime_error(exc)
        if kind == "temperature_unsupported":
            LLM_ENDPOINT_HEALTH.mark_temperature_unsupported(endpoint, logger, source="startup probe")
            _probe_json_format(spark_session, endpoint, logger, timeout_seconds)
            return "usable"
        if kind == "endpoint_unusable":
            LLM_ENDPOINT_HEALTH.mark_unusable(endpoint, get_clean_error_message(exc), logger, source="startup probe")
            return "unusable"
        logger.info(f"[LLM_ENDPOINT_PROBE] endpoint={endpoint} inconclusive ({kind}: "
                    f"{get_clean_error_message(exc, max_chars=200)}); kept in the roster, the runtime circuit breaker still applies")
        LLM_ENDPOINT_HEALTH.mark_json_format(endpoint, False, logger, reason="its endpoint check did not finish")
        return "inconclusive"


def _probe_llm_endpoints(spark_session, endpoints, logger) -> dict:
    if spark_session is None:
        return {}
    pending = [ep for ep in dict.fromkeys(e for e in endpoints if e) if LLM_ENDPOINT_HEALTH.claim_probe(ep)]
    if not pending:
        return {}
    outcomes = ParallelExecutor.execute_parallel(
        tasks=[(_probe_llm_endpoint, (spark_session, ep, logger)) for ep in pending],
        max_workers=min(len(pending), 8), task_name="LLM endpoint probe", logger=logger,
        total_timeout=_LLM_ENDPOINT_PROBE_TIMEOUT_S + 60, thread_name_prefix="LLMProbe", return_exceptions=True)
    results = {ep: (out if isinstance(out, str) else "inconclusive") for ep, out in zip(pending, outcomes)}
    for ep, out in results.items():
        LLM_ENDPOINT_HEALTH.record_probe_outcome(ep, out)
    logger.info("[LLM_ENDPOINT_PROBE] " + ", ".join(f"{ep}={out}" for ep, out in results.items()))
    return results


def _order_models_by_probe(models, logger) -> list:
    def inconclusive(model):
        return LLM_ENDPOINT_HEALTH.probe_outcome(model.get("llm_endpoint_name")) == "inconclusive"

    if not any(inconclusive(m) for m in models):
        return models
    ranked = sorted(models, key=lambda m: (inconclusive(m), m.get("order", 999)))
    for order, model in enumerate(ranked, start=1):
        model["order"] = order
    behind = [m.get("llm_endpoint_name") for m in ranked if inconclusive(m)]
    logger.info(f"[LLM_ENDPOINT_PROBE_ORDER FIRED] {len(behind)} endpoint(s) whose startup probe did not finish follow "
                f"the probe-passed endpoints in every cascade: {', '.join(behind)}")
    return ranked


def _apply_auto_detected_endpoints(logger, spark_session=None) -> bool:
    """Replace TECHNICAL_CONTEXT['models'] with auto-detected endpoints when a
    minimum safety bar is met. Coverage safety: detected list must be non-empty
    AND must include at least one large-tier model (thinker/large or worker/large)
    so hard prompts survive the universal fallback. On failure, keep hardcoded.
    Returns True on replacement, False otherwise (hardcoded list preserved)."""
    probe_session = _spark_session_or_none(spark_session)
    try:
        detected = _auto_detect_llm_endpoints(logger)
        TECHNICAL_CONTEXT["fast_chat_endpoints"] = _list_fast_chat_endpoints(logger)
        roster_names = [m["llm_endpoint_name"] for m in detected]
        extra = [name for name in TECHNICAL_CONTEXT["fast_chat_endpoints"][:1] if name not in roster_names]
        if detected and probe_session is not None:
            _probe_llm_endpoints(probe_session, roster_names + extra, logger)
            rejected = [m["llm_endpoint_name"] for m in detected if not LLM_ENDPOINT_HEALTH.is_usable(m["llm_endpoint_name"])]
            detected = [m for m in detected if LLM_ENDPOINT_HEALTH.is_usable(m["llm_endpoint_name"])]
            for _order, _model in enumerate(detected, start=1):
                _model["order"] = _order
            if rejected:
                logger.warning(f"[LLM_ENDPOINT_DETECTED] {len(rejected)} detected endpoint(s) cannot serve ai_query and are "
                               f"not promoted: {', '.join(rejected)}")
        if not detected:
            logger.info("[LLM_ENDPOINT_DETECTED] auto-detect returned no usable endpoint; keeping hardcoded model list.")
            return False
        _has_large_tier = any(m.get("size") == "large" for m in detected)
        if not _has_large_tier:
            logger.warning(f"[LLM_ENDPOINT_DETECTED] auto-detected list has no large-tier model ({len(detected)} endpoints, all small/tiny); keeping hardcoded fallback.")
            return False
        _detected_buckets = {(m["type"], m["size"]) for m in detected}
        _prompt_buckets = set()
        for pm in TECHNICAL_CONTEXT.get("prompts_models", []):
            _mt = pm.get("model_type", "worker")
            _ms = pm.get("model_size", DEFAULT_MODEL_SIZE_BY_TYPE.get(_mt, "small"))
            _prompt_buckets.add((_mt, _ms))
        _uncovered = [b for b in _prompt_buckets if b not in _detected_buckets]
        if _uncovered:
            logger.info(f"[LLM_ENDPOINT_DETECTED] auto-detected list has {len(_uncovered)} bucket(s) covered only by fallback: {sorted(_uncovered)}. Runtime will cascade to the nearest populated bucket in model_fallback_chain for those prompts.")
        _previous_models = TECHNICAL_CONTEXT["models"]
        TECHNICAL_CONTEXT["models"] = detected
        try:
            _validate_model_config()
        except (ValueError, TypeError) as _cfg_err:
            TECHNICAL_CONTEXT["models"] = _previous_models
            logger.warning(f"[LLM_ENDPOINT_DETECTED] auto-detected list failed model-config validation ({_cfg_err}); keeping hardcoded fallback.")
            return False
        logger.info(f"[LLM_ENDPOINT_DETECTED] auto-detected list passes safety bar ({len(detected)} endpoints, has large tier); replacing hardcoded model list.")
        return True
    finally:
        if probe_session is not None:
            try:
                _probe_llm_endpoints(probe_session, [m.get("llm_endpoint_name") for m in TECHNICAL_CONTEXT.get("models", [])
                                                     if m.get("enabled", True)], logger)
            except Exception as _probe_err:
                logger.warning(f"[LLM_ENDPOINT_PROBE] probing the active model roster failed non-fatally: {get_clean_error_message(_probe_err)}")
        TECHNICAL_CONTEXT["models"] = _order_models_by_probe(TECHNICAL_CONTEXT.get("models", []), logger)

PROMPT_TRACKING_CONFIG = {}
for _pm in TECHNICAL_CONTEXT.get("prompts_models", []):
    _prompt_name = _pm.get("prompt_name")
    if not _prompt_name:
        continue
    PROMPT_TRACKING_CONFIG[_prompt_name] = {
        "stage_name": _pm.get("stage_name", "Prompt Execution"),
        "step_name": _pm.get("step_name", _prompt_name),
        "progress_increment": float(_pm.get("progress_increment", 0.5)),
    }

def log_print(message: str, level: str = "INFO", flush: bool = True):
    """Print a message with timestamp in logger format for immediate console output.
    For ERROR/CRITICAL levels, also writes to stderr for visibility.
    """
    import time as _time
    import sys as _sys
    timestamp = _time.strftime('%H:%M:%S')
    formatted_msg = f"{timestamp} - {level} - {message}"
    print(formatted_msg, flush=flush)
    if flush:
        _sys.stdout.flush()
    
    level_upper = level.upper()
    if level_upper in ("ERROR", "CRITICAL"):
        print(formatted_msg, file=_sys.stderr, flush=True)


# =============================================================================
# PHASE A / N / H / I SCAFFOLDS (foundation substrate)
# Industry-agnostic, import-free, module-level singletons that later phases
# extend. Kept minimal so they do not affect pipeline behavior today.
# =============================================================================

def _utc_iso8601():
    """Phase N1 / Gap 16: UTC ISO-8601 with millisecond precision and Z suffix."""
    import datetime as _dt
    return _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")


class _NextActionsCollector:
    """Phase H1 scaffold — singleton for operator-actionable findings.
    Full taxonomy + triage + persistence live in Phase H (later).
    Today it just keeps an in-memory list callers can grow from any phase."""
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            inst = super().__new__(cls)
            inst._actions = []
            import threading as _threading
            inst._lock = _threading.Lock()
            cls._instance = inst
        return cls._instance

    def add(self, rule_id, severity="INFO", phase="", step="", evidence="",
            suggested_user_action="", fix_hint="", **provenance):
        allowed = ("BLOCKING", "SAFE_IGNORE", "INFO", "HIGH")
        sev = severity if severity in allowed else "INFO"
        entry = {
            "ts": _utc_iso8601(),
            "rule_id": str(rule_id or "UNKNOWN"),
            "severity": sev,
            "phase": str(phase or ""),
            "step": str(step or ""),
            "evidence": str(evidence or "")[:500],
            "suggested_user_action": str(suggested_user_action or "")[:500],
            "fix_hint": str(fix_hint or "")[:500],
            "provenance": {k: str(v)[:200] for k, v in provenance.items()},
        }
        with self._lock:
            self._actions.append(entry)
        return entry

    def summary(self):
        with self._lock:
            counts = {}
            for a in self._actions:
                counts[a["severity"]] = counts.get(a["severity"], 0) + 1
            return {"total": len(self._actions), "by_severity": counts}

    def snapshot(self):
        """Return a shallow copy of current entries for read-only inspection."""
        with self._lock:
            return list(self._actions)

    def assert_no_blocking(self):
        with self._lock:
            blocking = [a for a in self._actions if a["severity"] == "BLOCKING"]
        if blocking:
            raise RuntimeError(
                f"NEXT_ACTIONS has {len(blocking)} BLOCKING entries: "
                + ", ".join(a["rule_id"] for a in blocking[:10])
            )

    def export_jsonl(self, path):
        with self._lock:
            snapshot = list(self._actions)
        with open(path, "w", encoding="utf-8") as f:
            import json as _json
            for entry in snapshot:
                f.write(_json.dumps(entry, ensure_ascii=False) + "\n")


NEXT_ACTIONS = _NextActionsCollector()


# --- Phase H2: closed taxonomy of NEXT_ACTIONS rule_ids ---
NEXT_ACTIONS_TAXONOMY = {
    "BLOCKING": {
        "METADATA_ONLY_CONTRACT_VIOLATED", "INSPIRE_DATA_ACCESS_PROMPT_LEAKAGE",
        "UC_GENERATION_FAILED", "GENIE_INSTRUCTION_EMPTY_FOR_FLAGGED_UC",
        "UC_REFERENCES_NONEXISTENT_TABLE", "UC_NOT_GROUNDED", "UC_NOT_RELEVANT",
        "UC_DEDUP_RATE_HIGH", "UC_RECALL_GAP",
        "UC_TRANSLATION_DROPPED_IDENTIFIER", "UC_TRANSLATION_DROPPED_NUMBER",
        "UC_TRANSLATION_LENGTH_DRIFT", "UC_TRANSLATION_SEMANTIC_DRIFT",
        "UC_STALE_CATALOG_DRIFT", "SCHEMA_MIGRATION_FAILED",
        "WIDGET_COMPLIANCE_VIOLATED", "PLACEHOLDER_LEAKAGE", "NULL_OUTPUT_SENTINEL",
        "REGRESSED_WARNING", "SUSPICIOUS_SAFE_IGNORE", "VOLUME_ESCALATED",
        "SHADOW_DIFF_DIVERGENCE", "BASELINE_QUALITY_REGRESSION",
        "PREFLIGHT_INVARIANT_FAILURE", "VALIDATION_PARSE_DEFAULT_REJECT",
        "TABLE_FILTER_PARSE_FAILED", "MEMORY_BUDGET_EXCEEDED",
        "VOLUME_LOG_UPLOAD_FAILED", "STEP_TABLE_DUAL_WRITE_DRIFT",
        "BUSINESS_NAME_LOCK_HELD",
        "BUSINESS_NAME_LOCK_FORCE_ACQUIRED",
        "SENSITIVE_CREDENTIAL_OVERRIDE_REFUSED", "LOG_CONTENT_LEAKAGE",
        "PLAN_ANCHOR_DRIFT",
    },
    "SAFE_IGNORE": {
        "LOW_QUALITY_UC", "MISSING_FK_FOR_UC_TABLES", "UC_NAMING_QUIBBLE",
        "BUSINESS_PRIORITY_PARTIAL_MATCH", "PHASE_5B_TIMEOUT_RECOVERED",
        "LANGUAGE_TRANSLATION_FALLBACK", "JOBLAUNCHER_PATH_MIGRATION_NOTICE",
        "DOMAIN_AUTO_DETECTED", "UC_NEAR_DUPLICATE_MERGED",
        "UC_EXACT_DUPLICATE_DROPPED", "UC_RECALL_GAP_ACCEPTED_BY_WIDGET",
        "UC_RECALL_GAP_ACCEPTED_SENSITIVITY", "COLUMN_ID_FUZZY_MATCH_REJECTED",
        "GENIE_CODE_DATA_ACCESS_BY_DESIGN", "R3_DETERMINISTIC_FALLBACK_PASS",
        "R3_DETERMINISTIC_FALLBACK_DROP", "BUSINESS_NAME_PATH_COLLISION_AVOIDED",
        "CATALOG_DRIFT_CHECK_NO_DRIFT",
    },
    "HIGH": {
        "R3_LLM_UNAVAILABLE_FALLBACK_USED", "NO_OPERATIONAL_PATTERNS_DETECTED",
        "DEDUP_LLM_JUDGE_UNAVAILABLE", "MICRO_FIXTURE_NOT_SEEDED",
        "UC_SCHEMA_INCOMPLETE", "UC_CATCHALL_TECHNIQUE", "UC_COLUMN_UNGROUNDED",
    },
    "INFO": {
        "LEXICON_NOT_AVAILABLE",
    },
}


# ============================================================
# Phase F: UseCaseBundle + Visitor + Validator Registry
# ============================================================
# Structured representation of a single use-case for validator/visitor pipelines.
# Lives alongside the dict-based UC representation used across the codebase. Adapters
# (from_dict / to_dict) keep it interop-safe with existing callers.

from dataclasses import dataclass, field, asdict
from typing import Any, Callable, Dict, List, Optional, Tuple


@dataclass
class UseCaseBundle:
    no: str = ""
    name: str = ""
    business_domain: str = ""
    subdomain: str = ""
    type: str = ""
    analytics_technique: str = ""
    priority: str = ""
    primary_table: str = ""
    tables_involved: str = ""
    statement: str = ""
    solution: str = ""
    business_value: str = ""
    quality: str = ""
    result_table: str = ""
    genie_instruction: str = ""
    code_structure_mandate: str = ""
    provenance: Dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "UseCaseBundle":
        g = lambda k, default="": d.get(k, default) if d else default
        return cls(
            no=str(g("No") or ""),
            name=str(g("Name") or ""),
            business_domain=str(g("Business Domain") or ""),
            subdomain=str(g("Subdomain") or ""),
            type=str(g("type") or ""),
            analytics_technique=str(g("Analytics Technique") or ""),
            priority=str(g("Priority") or ""),
            primary_table=str(g("Primary Table") or ""),
            tables_involved=str(g("Tables Involved") or ""),
            statement=str(g("Statement") or ""),
            solution=str(g("Solution") or ""),
            business_value=str(g("Business Value") or ""),
            quality=str(g("Quality") or ""),
            result_table=str(g("result_table") or ""),
            genie_instruction=str(g("genie_instruction") or ""),
            code_structure_mandate=str(g("code_structure_mandate") or ""),
            provenance={k: v for k, v in (d.items() if d else [])
                        if k not in {"No", "Name", "Business Domain", "Subdomain", "type",
                                     "Analytics Technique", "Priority", "Primary Table",
                                     "Tables Involved", "Statement", "Solution",
                                     "Business Value", "Quality", "result_table",
                                     "genie_instruction", "code_structure_mandate"}},
        )

    def to_dict(self) -> Dict[str, Any]:
        out = {
            "No": self.no, "Name": self.name, "Business Domain": self.business_domain,
            "Subdomain": self.subdomain, "type": self.type,
            "Analytics Technique": self.analytics_technique, "Priority": self.priority,
            "Primary Table": self.primary_table, "Tables Involved": self.tables_involved,
            "Statement": self.statement, "Solution": self.solution,
            "Business Value": self.business_value, "Quality": self.quality,
            "result_table": self.result_table, "genie_instruction": self.genie_instruction,
            "code_structure_mandate": self.code_structure_mandate,
        }
        out.update(self.provenance or {})
        return out


class UseCaseBundleVisitor:
    """Visitor-pattern base. Override visit_<field> or visit_bundle to inspect/mutate."""
    def visit_bundle(self, bundle: UseCaseBundle) -> UseCaseBundle:
        return bundle

    def visit_batch(self, bundles: List[UseCaseBundle]) -> List[UseCaseBundle]:
        return [self.visit_bundle(b) for b in bundles]


# Registry: {name: callable(bundle) -> (passed: bool, reason: str)}
# Callers import this and register their own domain-agnostic validators.
VALIDATOR_REGISTRY: Dict[str, Callable[[UseCaseBundle], Tuple[bool, str]]] = {}


def register_validator(name: str):
    def _deco(fn: Callable[[UseCaseBundle], Tuple[bool, str]]):
        VALIDATOR_REGISTRY[name] = fn
        return fn
    return _deco


@register_validator("has_primary_table")
def _v_has_primary_table(b: UseCaseBundle) -> Tuple[bool, str]:
    if not (b.primary_table or "").strip():
        return False, "missing primary table"
    return True, ""


@register_validator("statement_min_length")
def _v_statement_min_length(b: UseCaseBundle) -> Tuple[bool, str]:
    if len((b.statement or "").strip()) < 20:
        return False, "statement too short"
    return True, ""


@register_validator("solution_min_length")
def _v_solution_min_length(b: UseCaseBundle) -> Tuple[bool, str]:
    if len((b.solution or "").strip()) < 20:
        return False, "solution too short"
    return True, ""


@register_validator("technique_present")
def _v_technique_present(b: UseCaseBundle) -> Tuple[bool, str]:
    if not (b.analytics_technique or "").strip():
        return False, "missing analytics technique"
    return True, ""


def run_validators(bundle: UseCaseBundle,
                   validators: Optional[List[str]] = None) -> Dict[str, Tuple[bool, str]]:
    """Run registered validators against a bundle. Returns {name: (passed, reason)}."""
    targets = validators or list(VALIDATOR_REGISTRY.keys())
    out = {}
    for n in targets:
        fn = VALIDATOR_REGISTRY.get(n)
        if not fn:
            continue
        try:
            out[n] = fn(bundle)
        except Exception as _err:
            out[n] = (False, f"validator_exception: {str(_err)[:80]}")
    return out


class _MemoryGuardScaffold:
    """Phase A3 / Phase I scaffold. Later phase extends with RSS checks,
    spill-to-disk, and budget enforcement."""
    def __init__(self, budget_mb=24 * 1024):
        self.budget_mb = int(budget_mb)

    def check(self, step_name=""):
        try:
            import psutil as _psutil, os as _os
            rss_mb = _psutil.Process(_os.getpid()).memory_info().rss / (1024 * 1024)
            return {"step": step_name, "rss_mb": rss_mb, "over_budget": rss_mb > self.budget_mb}
        except Exception:
            return {"step": step_name, "rss_mb": None, "over_budget": False}


_MEMORY_GUARD = _MemoryGuardScaffold()

# =============================================================================
# END PHASE A/N/H/I SCAFFOLDS
# =============================================================================


# =============================================================================
# PHASE C — METADATA-ONLY CONTRACT GUARD
# C1 (stub): inspire_sql wrapper with allowlist. Call sites not yet migrated —
# use as a standalone utility + run C3 banned-phrase scan at load.
# =============================================================================

_INSPIRE_SQL_ALLOWLIST = [
    r"^\s*SELECT\b.*\bFROM\s+[`\w.]+\.information_schema\.",
    r"^\s*(SELECT|MERGE|UPDATE|INSERT|DELETE|CREATE|ALTER|DESCRIBE|DROP)\b",
    r"^\s*CREATE\s+(TABLE|DATABASE|SCHEMA|VOLUME)\s+IF\s+NOT\s+EXISTS\b",
    r"^\s*CREATE\s+OR\s+REPLACE\s+TEMP\s+VIEW\b",
    r"^\s*SHOW\s+(TABLES|SCHEMAS|DATABASES|COLUMNS)\b",
]
_INSPIRE_SQL_BANNED = [
    r"\bDESCRIBE\s+TABLE\s+EXTENDED\b",
    r"\bDESCRIBE\s+EXTENDED\b",
]


def inspire_sql(query, args=None, spark_session=None):
    """Phase C1 wrapper around spark.sql with allowlist + banned-phrase check.
    Drop-in replacement for spark.sql(); raises if the query violates the
    metadata-only contract. This is a utility — existing call sites have NOT
    been migrated yet (39 total). Use for new call sites."""
    import re as _re
    q = str(query or "")
    for pat in _INSPIRE_SQL_BANNED:
        if _re.search(pat, q, _re.IGNORECASE):
            try:
                NEXT_ACTIONS.add("METADATA_ONLY_CONTRACT_VIOLATED", severity="BLOCKING",
                                 phase="C1", evidence=f"Banned pattern matched: {pat}")
            except Exception:
                pass
            raise PermissionError(f"Phase C1: query violates metadata-only contract (banned): {q[:80]}")
    allowed = any(_re.search(pat, q, _re.IGNORECASE) for pat in _INSPIRE_SQL_ALLOWLIST)
    if not allowed:
        try:
            NEXT_ACTIONS.add("METADATA_ONLY_CONTRACT_VIOLATED", severity="BLOCKING",
                             phase="C1", evidence=f"Not on allowlist: {q[:120]}")
        except Exception:
            pass
        raise PermissionError(f"Phase C1: query not on allowlist: {q[:80]}")
    _spark = spark_session
    if _spark is None:
        _spark = globals().get("spark")
    if _spark is None:
        raise RuntimeError("Phase C1: no spark session available")
    if args is None:
        return _spark.sql(q)
    return _spark.sql(q, args)


_C3_BANNED_PROMPT_PHRASES = [
    r"\bsample\s+rows\b",
    r"\bselect\s+\*\s+from\s+\{[^}]*tables_involved",
    r"\bdescribe\s+detail\b",
    r"\bdescribe\s+table\s+extended\b",
    r"\bshow\s+me\s+data\b",
    r"\bquery\s+the\s+table\b",
    r"\bcount\s+the\s+rows\b",
    r"\bread\s+from\s+the\s+table\b",
    r"\bfetch\s+sample\b",
    r"\bget\s+the\s+data\b",
    r"\binspect\s+the\s+rows\b",
]


def audit_prompts_for_data_access_leak(prompt_templates_dict, logger=None):
    """Phase C3: scan every prompt body for banned data-access pressure phrases,
    ignoring fenced code blocks (which belong to generated Genie code per C5 carve-out).
    Emits NEXT_ACTIONS BLOCKING on match; returns list of (prompt_name, phrase, context)."""
    import re as _re
    if not isinstance(prompt_templates_dict, dict):
        return []
    findings = []
    fence_re = _re.compile(r"```.*?```", _re.DOTALL)
    for name, body in prompt_templates_dict.items():
        if not isinstance(body, str):
            continue
        # Strip fenced code blocks (C5 carve-out)
        stripped = fence_re.sub("", body)
        for pat in _C3_BANNED_PROMPT_PHRASES:
            m = _re.search(pat, stripped, _re.IGNORECASE)
            if m:
                context = stripped[max(0, m.start()-40):m.end()+40]
                findings.append({"prompt": name, "pattern": pat, "context": context})
                try:
                    NEXT_ACTIONS.add(
                        "INSPIRE_DATA_ACCESS_PROMPT_LEAKAGE",
                        severity="BLOCKING", phase="C3",
                        evidence=f"{name}: {pat} ~ {context[:100]}",
                        suggested_user_action="Rewrite prompt to work from supplied schema only (no data SELECT pressure).",
                    )
                except Exception:
                    pass
                if logger is not None:
                    logger.warning(f"C3 prompt leak: {name} ~ {pat}")
    return findings

# =============================================================================
# END PHASE C
# =============================================================================


class DuplicateMessageFilter(logging.Filter):
    def __init__(self, window_seconds: float = 1.5):
        super().__init__()
        import threading as _threading
        self.window_seconds = float(window_seconds)
        self._last_seen = {}
        self._lock = _threading.Lock()

    def filter(self, record):
        import time
        if record.levelno >= logging.ERROR:
            return True
        msg = record.getMessage()
        key = f"{record.name}|{record.levelno}|{record.thread}|{msg}"
        now = time.time()
        with self._lock:
            last = self._last_seen.get(key)
            self._last_seen[key] = now
        if last is None:
            return True
        return (now - last) > self.window_seconds

def get_clean_error_message(exception: Exception, max_lines: int = 3, max_chars: int = 600) -> str:
    """
    Extract clean error message from exception without stack traces or embedded SQL.
    Strips JVM stack traces, '== SQL ==' blocks, and Spark internal frames.
    """
    error_str = str(exception)
    if isinstance(exception, (LookupError, AttributeError)) or not error_str.strip():
        error_str = f"{type(exception).__name__}: {error_str}".strip().rstrip(":")

    if '== SQL ==' in error_str:
        error_str = error_str.split('== SQL ==')[0].strip()

    if 'JVM stacktrace:' in error_str:
        error_str = error_str.split('JVM stacktrace:')[0].strip()

    error_str = re.sub(r"\.?[ \t]*\bConfig: [a-z_]+=[^\n]*", "", error_str)

    lines = []
    for line in error_str.split('\n'):
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.startswith('at org.apache') or stripped.startswith('at com.databricks') or stripped.startswith('at scala.'):
            continue
        if stripped.startswith("'") and ('+- ' in stripped or '   ' in stripped):
            continue
        lines.append(stripped)

    result = ' '.join(lines[:max_lines])
    if len(result) > max_chars:
        result = result[:max_chars]
    return result


COMPACT_OUTPUT_PROMPTS = frozenset({
    "REVIEW_USE_CASES_PROMPT",
    "FILTER_BUSINESS_TABLES_PROMPT",
    "BUSINESS_CONTEXT_WORKER_PROMPT",
    "SUBDOMAIN_DETECTOR_PROMPT",
    "DOMAIN_FINDER_PROMPT",
})

MEDIUM_OUTPUT_PROMPTS = frozenset({
    "USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT",
})

# ==============================================================================
# TRULY ADAPTIVE PARALLELISM CALCULATOR
# ==============================================================================
# Parallelism is calculated dynamically based on ACTUAL DATA:
# - Number of items to process (tables, use cases, domains)
# - Estimated prompt/payload size
# - LLM vs non-LLM operations
# - Risk of rate limiting or timeouts
#
# BOUNDS: MIN=2, MAX=10 (enforced globally)
# METADATA QUERIES: FIXED at 5 (no LLM, but too many connections can hang)
# ==============================================================================

METADATA_PARALLELISM = TECHNICAL_CONTEXT["runtime"]["metadata_parallelism"]

def calculate_adaptive_parallelism(
    step_name: str,
    max_parallelism: int,
    num_items: int = 0,
    total_columns: int = 0,
    avg_prompt_chars: int = 0,
    num_domains: int = 0,
    is_llm_operation: bool = True,
    logger=None,
    estimated_mb_per_worker: float = 5.0
) -> tuple:
    """
    Calculate truly adaptive parallelism based on actual data characteristics
    and available memory on the cluster.
    
    Args:
        step_name: Name of the step (for logging)
        max_parallelism: User-configured maximum parallelism
        num_items: Number of items to process (tables, use cases, domains, etc.)
        total_columns: Total columns involved (affects prompt size)
        avg_prompt_chars: Average prompt size in characters
        num_domains: Number of business domains
        is_llm_operation: Whether this step makes LLM calls
        logger: Optional logger
        estimated_mb_per_worker: Estimated MB each worker thread needs (callers
            that hold large data like schema markdown should pass higher values,
            e.g. 20 for quality scoring, 10 for SQL gen)
    
    Returns:
        Tuple of (parallelism: int, reason: str)
    """
    GLOBAL_MIN_PARALLELISM = TECHNICAL_CONTEXT["runtime"]["global_min_parallelism"]
    GLOBAL_MAX_PARALLELISM = TECHNICAL_CONTEXT["runtime"]["global_max_parallelism"]
    CLUSTER_MEMORY_GB = TECHNICAL_CONTEXT["runtime"].get("cluster_memory_gb", 32)
    _SPARK_AND_OS_OVERHEAD_MB = 17500
    available_app_mb = max(1024, (CLUSTER_MEMORY_GB * 1024) - _SPARK_AND_OS_OVERHEAD_MB)
    memory_ceiling = max(GLOBAL_MIN_PARALLELISM, int(available_app_mb / max(1.0, estimated_mb_per_worker)))
    
    effective_max = min(max(1, max_parallelism), GLOBAL_MAX_PARALLELISM)
    effective_min = min(GLOBAL_MIN_PARALLELISM, effective_max)
    
    base = effective_max
    
    # =========================================================================
    # METADATA OPERATIONS (Fixed at 5 - no LLM but DB connections can saturate)
    # =========================================================================
    if step_name in ["schema_discovery", "table_discovery", "column_fetch"]:
        result = METADATA_PARALLELISM
        reason = f"FIXED={METADATA_PARALLELISM} for metadata queries (DB connection limit)"
        if logger:
            logger.info(f"🔧 [{step_name.upper()}] Parallelism = {result} | {reason}")
        return (result, reason)
    
    # =========================================================================
    # FILE I/O OPERATIONS (Can use higher parallelism, but cap based on items)
    # =========================================================================
    if step_name in ["notebook_generation", "artifact_writing"]:
        # Scale with number of items, but cap reasonably
        if num_items <= 5:
            result = min(base, num_items + 2)
            reason = f"few items ({num_items}), using {result} workers"
        elif num_items <= 15:
            result = min(base, 6)
            reason = f"moderate items ({num_items}), capped at 6"
        else:
            result = min(base, 8)
            reason = f"many items ({num_items}), capped at 8 for I/O stability"
        
        result = max(effective_min, min(effective_max, result))
        if logger:
            logger.info(f"🔧 [{step_name.upper()}] Parallelism = {result} | {reason}")
        return (result, reason)
    
    # =========================================================================
    # LLM OPERATIONS - Adaptive based on workload severity
    # =========================================================================
    # Each factor contributes an independent severity score (0.0 = no concern, 1.0 = max concern).
    # We combine the top-2 scores (weighted 70/30) into a single reduction applied ONCE to base.
    # This avoids cascading multiplication that previously drove any input to 0.
    
    factors = []
    severity_scores = []
    
    if num_items > 0:
        if num_items <= 10:
            severity_scores.append(0.0)
            factors.append(f"{num_items} items (small)")
        elif num_items <= 30:
            severity_scores.append(0.15)
            factors.append(f"{num_items} items (medium)")
        elif num_items <= 100:
            severity_scores.append(0.3)
            factors.append(f"{num_items} items (large)")
        else:
            severity_scores.append(0.45)
            factors.append(f"{num_items} items (very large)")
    
    if avg_prompt_chars > 0:
        if avg_prompt_chars > 100000:
            severity_scores.append(0.5)
            factors.append(f"~{avg_prompt_chars//1000}K chars/prompt (huge)")
        elif avg_prompt_chars > 50000:
            severity_scores.append(0.35)
            factors.append(f"~{avg_prompt_chars//1000}K chars/prompt (large)")
        elif avg_prompt_chars > 20000:
            severity_scores.append(0.15)
            factors.append(f"~{avg_prompt_chars//1000}K chars/prompt (medium)")
        else:
            severity_scores.append(0.0)
            factors.append(f"~{avg_prompt_chars//1000}K chars/prompt (small)")
    
    if num_domains > 0:
        if num_domains > 15:
            severity_scores.append(0.45)
            factors.append(f"{num_domains} domains (many)")
        elif num_domains > 8:
            severity_scores.append(0.3)
            factors.append(f"{num_domains} domains (moderate)")
        else:
            severity_scores.append(0.1)
            factors.append(f"{num_domains} domains (few)")
    
    if total_columns > 0:
        if total_columns > 2000:
            severity_scores.append(0.45)
            factors.append(f"{total_columns} cols (massive schema)")
        elif total_columns > 1000:
            severity_scores.append(0.3)
            factors.append(f"{total_columns} cols (large schema)")
        elif total_columns > 500:
            severity_scores.append(0.15)
            factors.append(f"{total_columns} cols (medium schema)")
        else:
            severity_scores.append(0.0)
            factors.append(f"{total_columns} cols (small schema)")
    
    step_severities = {
        "scoring": (0.25, "scoring is rate-limit sensitive"),
        "deduplication": (0.15, "dedup: global one-shot (thinker), fallback per-domain (worker)"),
        "use_case_generation": (0.3, "LLM-intensive, 2-pass for transactional tables"),
        "domain_clustering": (0.3, "domain detection is heavy"),
        "subdomain_detection": (0.15, "subdomain per domain"),
        "translation": (0.15, "translation LLM calls"),
    }
    
    if step_name in step_severities:
        step_sev, adj_reason = step_severities[step_name]
        severity_scores.append(step_sev)
        factors.append(adj_reason)
    
    if severity_scores:
        sorted_scores = sorted(severity_scores, reverse=True)
        if len(sorted_scores) >= 2:
            combined_severity = sorted_scores[0] * 0.7 + sorted_scores[1] * 0.3
        else:
            combined_severity = sorted_scores[0]
        reduction_factor = max(0.25, 1.0 - combined_severity)
        calculated = max(1, int(base * reduction_factor))
    else:
        calculated = base
    
    work_items = num_items if num_items > 0 else (num_domains if num_domains > 0 else calculated)
    calculated = min(calculated, max(1, work_items))
    
    result = max(effective_min, min(effective_max, calculated))
    
    if not factors and is_llm_operation:
        result = max(effective_min, min(4, base))
        factors.append("LLM operation, conservative default")
    
    pre_mem_result = result
    result = min(result, memory_ceiling)
    if result < pre_mem_result:
        factors.append(f"memory-capped={result} ({CLUSTER_MEMORY_GB}GB, ~{estimated_mb_per_worker:.0f}MB/worker, ceiling={memory_ceiling})")
    
    reason = " + ".join(factors) if factors else "default calculation"
    reason = f"calculated={result} based on: {reason}"
    
    if logger:
        logger.info(f"🔧 [{step_name.upper()}] Parallelism = {result} (from max={max_parallelism}, mem_ceil={memory_ceiling}) | {reason}")
    
    return (result, reason)


def log_adaptive_parallelism_decision(step_name: str, parallelism: int, max_parallelism: int, reason: str):
    """
    Log the adaptive parallelism decision with full context.
    """
    log_print(f"🔧 [{step_name.upper()}] Workers: {parallelism} (max={max_parallelism})")
    log_print(f"   └─ Reason: {reason}")


BEST_OF_BEST_DEFAULT_COUNT = 20
BEST_OF_BEST_MAX_COUNT = 100
BEST_OF_BEST_DOMAIN_SHARE = 0.25


def parse_best_of_best_count(value):
    text = str(value if value is not None else "").strip()
    if not text:
        return None
    if not (text.isascii() and text.isdigit() and len(text) <= 3) or int(text) > BEST_OF_BEST_MAX_COUNT:
        raise ValueError(f"must be a whole number from 0 to {BEST_OF_BEST_MAX_COUNT}, or empty for the default "
                         f"{BEST_OF_BEST_DEFAULT_COUNT}; got {text!r}")
    return int(text)


USE_CASES_QUALITY_VALID = ("Good", "High", "Ultra High")

USE_CASES_QUALITY_DEFAULT = "Good"

USE_CASES_QUALITY_ALIASES = {
    "Coverage Mode (All)": "Good",
    "Balanced": "Good",
    "Good Quality": "Good",
    "High Quality": "Good",
    "Strict Quality": "High",
    "Very High Quality": "High",
}


def canonical_use_cases_quality(value) -> str:
    text = str(value if value is not None else "").strip()
    if not text:
        return USE_CASES_QUALITY_DEFAULT
    folded = text.casefold()
    for mode in USE_CASES_QUALITY_VALID:
        if mode.casefold() == folded:
            return mode
    for legacy, mode in USE_CASES_QUALITY_ALIASES.items():
        if legacy.casefold() == folded:
            return mode
    legacy_by_mode = {}
    for legacy, mode in USE_CASES_QUALITY_ALIASES.items():
        legacy_by_mode.setdefault(mode, []).append(repr(legacy))
    legacy_text = "; ".join(f"{', '.join(labels)} (mapped to {mode!r})" for mode, labels in legacy_by_mode.items())
    raise ValueError(
        f"Use Cases Quality {text!r} is not supported. Valid options: "
        f"{', '.join(repr(mode) for mode in USE_CASES_QUALITY_VALID)}. Accepted legacy labels: {legacy_text}."
    )


DISCOVERY_TYPE_VALID = ("Deep", "Standard")

DISCOVERY_TYPE_DEFAULT = "Deep"


def canonical_discovery_type(value) -> str:
    text = str(value if value is not None else "").strip()
    if not text:
        return DISCOVERY_TYPE_DEFAULT
    for mode in DISCOVERY_TYPE_VALID:
        if mode.casefold() == text.casefold():
            return mode
    raise ValueError(
        f"Discovery Type {text!r} is not supported. Valid options: "
        f"{', '.join(repr(mode) for mode in DISCOVERY_TYPE_VALID)}."
    )


STRATEGY_SCHEMA_VERSION = 1
STRATEGY_MAX_DEPTH = 4
STRATEGY_MAX_ENTRIES = 200
STRATEGY_MAX_NAME_CHARS = 300
STRATEGY_MAX_INLINE_BYTES = 900_000
STRATEGY_DEFAULT_LEVEL = "Strategic Goal"
STRATEGY_NOT_ALIGNED = "Not aligned"
STRATEGY_BUILTIN_NO_MATCH = "General Improvement"
STRATEGY_GAP_TEXT = "No data-backed use case found in the scanned data"
STRATEGY_LABEL_KEYS = ("priority", "kpi", "key_result", "sponsor")
STRATEGY_MAX_ALIGNMENTS = 2
STRATEGY_PRIORITY_BOOST = 0.25
STRATEGY_PRIORITY_BOOST_MAX_MATCHES = 2
KPI_DIRECTIONS = ("increase", "decrease", "maintain")

VALUE_FACTOR_WEIGHTS_BLIND = (("Return on Investment", 0.80), ("Time to Value", 0.10), ("Reusability", 0.10))
VALUE_FACTOR_WEIGHTS_ALIGNED = (("Return on Investment", 0.60), ("Strategic Alignment", 0.25),
                                ("Time to Value", 0.075), ("Reusability", 0.075))
FEASIBILITY_FACTOR_KEYS = ("Data Availability", "Data Accessibility", "Architecture Fitness", "Team Skills",
                           "Domain Knowledge", "People Allocation", "Budget Allocation", "Time to Production")
PRIORITY_VALUE_WEIGHT = 1.5
PRIORITY_FEASIBILITY_WEIGHT = 0.5


def _sub_score(uc, key):
    try:
        value = float(uc.get(key))
    except (TypeError, ValueError):
        return None
    return value if value == value and 1.0 <= value <= 5.0 else None


def compute_value_priority(uc: dict, with_alignment: bool = False):
    """(value, feasibility, priority) from the 1-5 sub-scores. Before selection Value never uses
    Strategic Alignment; after the alignment pass it carries 25%. None when a value factor is missing
    or out of range."""
    weights = VALUE_FACTOR_WEIGHTS_ALIGNED if with_alignment else VALUE_FACTOR_WEIGHTS_BLIND
    parts = [(_sub_score(uc, key), weight) for key, weight in weights]
    if any(score is None for score, _weight in parts):
        return None
    value = round(sum(score * weight for score, weight in parts), 2)
    factors = [score for score in (_sub_score(uc, key) for key in FEASIBILITY_FACTOR_KEYS) if score is not None]
    feasibility = round(sum(factors) / len(factors), 2) if factors else _sub_score(uc, "Feasibility")
    if feasibility is None:
        return None
    return value, feasibility, round(value * PRIORITY_VALUE_WEIGHT + feasibility * PRIORITY_FEASIBILITY_WEIGHT, 2)

AILOGS_TABLE = "_ailogs"
AILOGS_AGENT = "inspire"
AILOGS_COLUMNS = (
    ("ailogs_id", "STRING"), ("request_id", "STRING"), ("session_id", "BIGINT"),
    ("attempt", "INT"), ("agent", "STRING"), ("prompt", "STRING"), ("endpoint", "STRING"),
    ("temperature", "DOUBLE"), ("parameters", "STRING"), ("request", "STRING"),
    ("response", "STRING"), ("request_length", "BIGINT"), ("request_tokens", "BIGINT"),
    ("response_length", "BIGINT"), ("response_tokens", "BIGINT"), ("status", "STRING"),
    ("start_time", "TIMESTAMP"), ("end_time", "TIMESTAMP"), ("request_hash", "STRING"),
    ("server_ingested_at", "TIMESTAMP"),
)
AILOGS_STATUSES = ("success", "failed", "timedout", "exhausted", "hedged")
AILOGS_NON_SUCCESS = ("failed", "timedout", "exhausted")
AILOGS_CLUSTER_BY = ("session_id", "agent")
AILOGS_TABLE_COMMENT = "Append-only log of every LLM call attempt with full request and response."
AILOGS_CHARS_PER_TOKEN = 3.2
AILOGS_ERROR_CHARS = 2000
AILOGS_TIMEOUT_NOTE = "timed out waiting for the endpoint"
AILOGS_EXHAUSTED_NOTE = "exhausted: no attempt succeeded"

FEEDBACK_TABLE = "__inspire_feedback"
FEEDBACK_STATUSES = ("accepted", "rejected", "planned", "implemented")
FEEDBACK_COLUMNS = (
    ("session_id", "BIGINT"), ("business_name", "STRING"), ("use_case_id", "STRING"),
    ("use_case_key", "STRING"), ("use_case_title", "STRING"), ("status", "STRING"),
    ("reason", "STRING"), ("updated_by", "STRING"), ("updated_at", "TIMESTAMP"),
)

IMPLEMENT_OPERATION = "Implement Use Cases"
INSPIRE_OPERATIONS = ("Discover Use Cases", "Generate Use Cases", IMPLEMENT_OPERATION)
GENIE_CODE_DEFAULT_COUNT = "0"
INSPIRE_ATTACHING_OPERATIONS = ("Generate Use Cases", IMPLEMENT_OPERATION)


def _semunity_column_comment(table: str, column: str) -> str:
    if column.endswith("_id"):
        return f"Identifier: {column}"
    return f"{table.strip('_')} {column.replace('_', ' ')}"


def ailogs_create_table_sql(ailogs_table: str) -> str:
    columns = ", ".join(f"`{name}` {kind} COMMENT {inspire_sql_literal(_semunity_column_comment(AILOGS_TABLE, name))}"
                        for name, kind in AILOGS_COLUMNS)
    cluster = ", ".join(f"`{name}`" for name in AILOGS_CLUSTER_BY)
    return (f"CREATE TABLE IF NOT EXISTS {ailogs_table} ({columns}) USING DELTA "
            f"CLUSTER BY ({cluster}) COMMENT {inspire_sql_literal(AILOGS_TABLE_COMMENT)}")


TECHNICAL_EXCLUSION_STRATEGY = "Aggressive"


def resolve_technical_exclusion_strategy(value=None) -> str:
    text = str(value if value is not None else "").strip()
    if text and text.casefold() != TECHNICAL_EXCLUSION_STRATEGY.casefold():
        raise ValueError(
            f"Technical exclusion strategy {text!r} is not supported: Inspire always applies the "
            f"{TECHNICAL_EXCLUSION_STRATEGY!r} technical-table filter"
        )
    return TECHNICAL_EXCLUSION_STRATEGY


def _defer_stale_widget(name, default, choices, label, multiselect, deferred):
    try:
        dbutils.widgets.remove(name)
    except Exception:
        return False
    deferred.append((name, default, list(choices), label, multiselect))
    return True


def recreate_deferred_widgets(deferred):
    while deferred:
        name, default, choices, label, multiselect = deferred.pop(0)
        try:
            if multiselect:
                dbutils.widgets.multiselect(name, default, choices, label)
            else:
                dbutils.widgets.dropdown(name, default, choices, label)
            log_print(f"✅ Widget '{label}' recreated with options {choices} (value: {default})")
        except Exception as e:
            log_print(f"⚠️ Widget '{label}' could not be recreated: {e}", level="WARNING")


def create_widgets():
    """
    Creates widgets if they don't exist. Retains existing widget values.
    
    Widget Order:
    0- Operation (Discover Use Cases / Generate Use Cases / Implement Use Cases)
    1- Business Name
    2- UC Metadata
    3- Inspire Database
    4- Discovery Type (Deep / Standard)
    5- Table Election
    6- Use Cases Quality
    7- Business Domains
    8- Business Priorities
    9- Generation Instructions
    10- Generation Options (PDF Catalog / Presentation)
    11- Strategic Context (JSON or a /Workspace or /Volumes .json path)
    12- Generation Path
    13- Documents Languages
    14- Inspire Session ID
    15- Generate Genie Code Instruction For
    """
    
    log_print("Creating widgets (retaining existing values)...")
    
    widget_errors = []
    deferred_widgets = []
    
    try:
        operation_options = list(INSPIRE_OPERATIONS)
        dbutils.widgets.dropdown("15_operation", "Discover Use Cases", operation_options, "00. Operation")
    except Exception as e:
        widget_errors.append(f"Operation: {e}")
    
    # --- 0. Business Name (REQUIRED) ---
    try:
        dbutils.widgets.text("00_business_name", "", "01. Business Name")
    except Exception as e:
        widget_errors.append(f"Business Name: {e}")
    
    # --- 1. UC Metadata (catalogs/schemas/tables OR JSON file path) ---
    try:
        dbutils.widgets.text("01_uc_metadata", "", "02. UC Metadata")
    except Exception as e:
        widget_errors.append(f"UC Metadata: {e}")
    
    # --- 2. Inspire Database (catalog.database format - where all results are stored) ---
    try:
        dbutils.widgets.text("02_inspire_database", "", "03. Inspire Database")
    except Exception as e:
        widget_errors.append(f"Inspire Database: {e}")

    try:
        dbutils.widgets.dropdown("03_discovery_type", DISCOVERY_TYPE_DEFAULT, list(DISCOVERY_TYPE_VALID), "04. Discovery Type")
    except Exception as e:
        widget_errors.append(f"Discovery Type: {e}")
    

    # --- 4. Table Election (controls which tables are used for use case generation) ---
    try:
        table_election_options = [
            "Let Inspire Decides",
            "All Tables",
            "Transactional Only"
        ]
        dbutils.widgets.dropdown("04_table_election", "Let Inspire Decides", table_election_options, "05. Table Election")
    except Exception as e:
        widget_errors.append(f"Table Election: {e}")
    
    # --- 5. Use Cases Quality (controls post-generation selection) ---
    try:
        try:
            _stale_quality = (dbutils.widgets.get("05_use_cases_quality") or "").strip()
        except Exception:
            _stale_quality = ""
        _quality_deferred = False
        if _stale_quality and _stale_quality not in USE_CASES_QUALITY_VALID:
            try:
                _migrated_quality = canonical_use_cases_quality(_stale_quality)
            except ValueError:
                _migrated_quality = None
            if _migrated_quality:
                _quality_deferred = _defer_stale_widget(
                    "05_use_cases_quality", _migrated_quality, USE_CASES_QUALITY_VALID,
                    "06. Use Cases Quality", False, deferred_widgets)
        if not _quality_deferred:
            dbutils.widgets.dropdown("05_use_cases_quality", USE_CASES_QUALITY_DEFAULT, list(USE_CASES_QUALITY_VALID), "06. Use Cases Quality")
    except Exception as e:
        widget_errors.append(f"Use Cases Quality: {e}")
    
    # --- 6. Business Domains (comma-separated list of domains) ---
    try:
        dbutils.widgets.text("06_business_domains", "", "07. Business Domains")
    except Exception as e:
        widget_errors.append(f"Business Domains: {e}")
    
    # --- 7. Business Priorities (multi-select) ---
    try:
        business_priorities_options = [
            "Increase Revenue",
            "Reduce Cost",
            "Optimize Operations",
            "Mitigate Risk",
            "Empower Talent",
            "Enhance Experience",
            "Drive Innovation",
            "Achieve ESG",
            "Protect Revenue",
            "Execute Strategy"
        ]
        dbutils.widgets.multiselect("07_business_priorities", "Increase Revenue", business_priorities_options, "08. Business Priorities")
    except Exception as e:
        widget_errors.append(f"Business Priorities: {e}")
    
    # --- 8. Generation Instructions ---
    try:
        dbutils.widgets.text("08_generation_instructions", "", "09. Generation Instructions")
    except Exception as e:
        widget_errors.append(f"Generation Instructions: {e}")
    
    # --- 9. Generation Options (multiselect with generation choices) ---
    try:
        try:
            _stale_value = dbutils.widgets.get("09_generation_options") or ""
        except Exception:
            _stale_value = ""
        generation_options = [
            "PDF Catalog",
            "Presentation",
        ]
        _options_deferred = False
        if "Genie Code Instructions" in _stale_value:
            _options_deferred = _defer_stale_widget(
                "09_generation_options", "PDF Catalog", generation_options,
                "10. Generation Options", True, deferred_widgets)
        if not _options_deferred:
            dbutils.widgets.multiselect(
                "09_generation_options", 
                "PDF Catalog",
                generation_options, 
                "10. Generation Options"
            )
    except Exception as e:
        widget_errors.append(f"Generation Options: {e}")

    try:
        dbutils.widgets.text("10_strategic_context", "", "11. Strategic Context")
    except Exception as e:
        widget_errors.append(f"Strategic Context: {e}")
    
    # --- 10. Generation Path ---
    try:
        dbutils.widgets.text("11_generation_path", "./../demos/", "12. Generation Path")
    except Exception as e:
        widget_errors.append(f"Generation Path: {e}")
    
    # --- 12. Documents Languages (multiselect) ---
    try:
        lang_choices = [
            "English", "French", "German", "Spanish", "Hindi",
            "Chinese (Mandarin)", "Japanese", "Arabic", "Portuguese", "Russian",
            "Swedish", "Danish", "Norwegian", "Finnish",
            "Italian", "Polish", "Romanian", "Ukrainian", "Dutch", "Korean",
            "Indonesian", "Malay", "Tamil"
        ]
        dbutils.widgets.multiselect("12_documents_languages", "English", lang_choices, "13. Documents Languages")
    except Exception as e:
        widget_errors.append(f"Documents Languages: {e}")
    
    # --- 13. Inspire Session ID (optional - auto-generated if empty) ---
    try:
        dbutils.widgets.text("14_session_id", "", "14. Inspire Session ID")
    except Exception as e:
        widget_errors.append(f"Inspire Session ID: {e}")

    try:
        dbutils.widgets.text(
            "13_generate_genie_code_for", GENIE_CODE_DEFAULT_COUNT,
            "15. Generate Genie Code Instruction For"
        )
    except Exception as e:
        widget_errors.append(f"13_generate_genie_code_for: {e}")

    try:
        dbutils.widgets.text("16_best_of_best_count", "", "16. Best of Best Count")
    except Exception as e:
        widget_errors.append(f"16_best_of_best_count: {e}")

    try:
        dbutils.widgets.text("inspire_implement_use_case", "", "Implement use case id")
    except Exception as e:
        widget_errors.append(f"inspire_implement_use_case: {e}")

    try:
        dbutils.widgets.text("inspire_implement_spec_version", "", "Implement spec version")
    except Exception as e:
        widget_errors.append(f"inspire_implement_spec_version: {e}")
    if widget_errors:
        log_print(f"⚠️ Some widgets had errors during creation:", level="WARNING")
        for err in widget_errors:
            log_print(f"   - {err}", level="WARNING")
        log_print("   Try running: dbutils.widgets.removeAll() and then run this cell again")
    else:
        log_print("✅ Widgets created successfully.")
    if deferred_widgets:
        log_print(f"ℹ️ Widget(s) with retired options removed: {', '.join(w[3] for w in deferred_widgets)}. "
                  f"main() recreates them with the current options and the mapped value.")
    
    log_print("")
    log_print(">>> Fill in the widget values at the TOP of this notebook, then run main().")
    return deferred_widgets

# ---
# Run this cell to create widgets.
# Fill in the widget values at the TOP of the notebook.
# Then, proceed to run the 'main()' cell below.
# ---

_DEFERRED_WIDGETS = create_widgets()

_NOTEBOOK_WIDGET_NAMES = [
    "15_operation",
    "00_business_name",
    "01_uc_metadata",
    "02_inspire_database",
    "03_discovery_type",
    "04_table_election",
    "05_use_cases_quality",
    "06_business_domains",
    "07_business_priorities",
    "08_generation_instructions",
    "09_generation_options",
    "10_strategic_context",
    "11_generation_path",
    "12_documents_languages",
    "13_generate_genie_code_for",
    "14_session_id",
    "16_best_of_best_count",
    "inspire_implement_use_case",
    "inspire_implement_spec_version",
]

# COMMAND ----------

# DBTITLE 1,Imports & Commons
# ==============================================================================
# 0. IMPORTS & CONFIGURATION
# ==============================================================================
import os
import pandas as pd
import logging
import ast
import re
import subprocess
import sys
import json
import csv
import io
import uuid
import base64
import random
import tempfile
import shutil
import datetime
import decimal
import html
import unicodedata
import warnings
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException
from collections import Counter, defaultdict
import time
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
import threading
import gc
import hashlib
import zlib

NUMERIC_CLAIM_RE = re.compile(
    r"[$€£¥]\s?\d[\d,]*(?:\.\d+)?(?:\s?(?:k|m|mm|bn|b|million|billion|thousand)\b)?"
    r"|\b\d+(?:\.\d+)?\s?(?:%|percent\b|pct\b|bps\b|basis points\b)"
    r"|\b\d+(?:\.\d+)?\s?x\b"
    r"|\b\d+(?:\.\d+)?\s?(?:million|billion|thousand|mn|bn)\b", re.IGNORECASE)

# --- Databricks SDK Imports for Notebook Creation ---
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import workspace

# --- Global Configuration ---
AI_MODEL_NAME = "databricks-gpt-oss-20b"

# Token-to-Character Ratios (for context limit calculations)
# English: 1 token ≈ 4 characters
# Non-English: 1 token ≈ 2 characters
TOKEN_TO_CHAR_RATIO_ENGLISH = 4
TOKEN_TO_CHAR_RATIO_NON_ENGLISH = 2

def get_model_token_limit(prompt_name: str = None) -> int:
    """
    Get the input token limit for a model based on TECHNICAL_CONTEXT.
    Uses the primary model (lowest-order) from the prompt's model_type cascade.
    """
    if prompt_name:
        model_config = get_model_config_for_prompt(prompt_name)
        if model_config:
            return model_config.get("llm_input_context_tokens_count", 200000)
    
    first_worker = get_models_by_type("worker")
    return first_worker[0].get("llm_input_context_tokens_count", 200000) if first_worker else 200000

def get_model_output_token_limit(prompt_name: str = None) -> int:
    """
    Get the OUTPUT token limit for a model based on TECHNICAL_CONTEXT.
    Uses the primary model (lowest-order) from the prompt's model_type cascade.
    """
    _safe_default = min(m.get("llm_output_context_tokens_count", 25000) for m in TECHNICAL_CONTEXT["models"])
    if prompt_name:
        model_config = get_model_config_for_prompt(prompt_name)
        if model_config:
            return model_config.get("llm_output_context_tokens_count", _safe_default)
    
    first_worker = get_models_by_type("worker")
    return first_worker[0].get("llm_output_context_tokens_count", _safe_default) if first_worker else _safe_default

def get_max_context_chars(language: str = "English", prompt_name: str = None) -> int:
    """
    Calculate the maximum character limit based on language and model's token limit.
    Uses TECHNICAL_CONTEXT to determine model-specific token limits.
    
    Applies the same conservative limit (tokens * 3.2) used by the actual LLM call
    pre-flight check, ensuring the pre-check here matches the runtime check.
    
    Args:
        language: Target language (default: "English")
        prompt_name: Name of the prompt to lookup model for (optional)
    
    Returns:
        Maximum character limit for the given language and model
    """
    max_tokens = get_model_token_limit(prompt_name)
    
    if language.lower() == "english":
        char_limit = max_tokens * TOKEN_TO_CHAR_RATIO_ENGLISH
    else:
        char_limit = max_tokens * TOKEN_TO_CHAR_RATIO_NON_ENGLISH
    
    conservative_limit = int(max_tokens * 3.2)
    return min(char_limit, conservative_limit, context_budget_chars(max_tokens))


def context_budget_chars(window_tokens: int) -> int:
    """The largest prompt sent to a window: ai_call_context_fill_max of it, at 3.2 chars per token."""
    fill = float(TECHNICAL_CONTEXT.get("runtime", {}).get("ai_call_context_fill_max", 0.45))
    return int(int(window_tokens) * fill * 3.2)


def get_context_limit_chars_for_model(model_config: dict, language: str = "English") -> int:
    """
    Calculate the effective character limit for a specific model config.
    Mirrors the pre-flight logic in _call_ai_query for consistency.
    """
    input_tokens = model_config.get("llm_input_context_tokens_count", 131000)
    char_ratio = TOKEN_TO_CHAR_RATIO_NON_ENGLISH if language.lower() != "english" else TOKEN_TO_CHAR_RATIO_ENGLISH
    max_context_chars = input_tokens * char_ratio
    conservative_limit = int(input_tokens * 3.2)
    return min(max_context_chars, conservative_limit, context_budget_chars(input_tokens))

def get_safe_context_limit(language: str = "English", buffer_percent: float = 0.9, prompt_name: str = None) -> int:
    """
    Calculate a safe context limit with buffer to proactively avoid LLM errors.
    Uses TECHNICAL_CONTEXT to determine model-specific token limits.
    
    Formula: model_token_limit * char_ratio * buffer_percent
    
    Args:
        language: Target language (default: "English")
        buffer_percent: Safety buffer (default: 0.9 = 10% buffer)
        prompt_name: Name of the prompt to lookup model for (optional)
    
    Returns:
        Safe character limit with buffer applied
    """
    max_chars = get_max_context_chars(language, prompt_name)
    safe_limit = int(max_chars * buffer_percent)
    return safe_limit

# Legacy constants for backward compatibility (uses default model's English ratio)
MAX_CONTEXT_TOKENS = get_model_token_limit()
MAX_CONTEXT_CHARS = min(MAX_CONTEXT_TOKENS * TOKEN_TO_CHAR_RATIO_ENGLISH, int(MAX_CONTEXT_TOKENS * 3.2))

# ==============================================================================
# IDENTIFIER NORMALIZATION UTILITIES
# ==============================================================================
# These functions handle SQL identifier parsing and quoting consistently.
# All user input should be normalized (backticks stripped) on input,
# and backticks should be added when constructing SQL queries.
# ==============================================================================

def normalize_identifier(identifier: str) -> str:
    """
    Strip backticks from an identifier.
    
    Args:
        identifier: A SQL identifier that may or may not have backticks
        
    Returns:
        The identifier without backticks
        
    Examples:
        normalize_identifier("`my-schema`") -> "my-schema"
        normalize_identifier("my_table") -> "my_table"
    """
    if identifier is None:
        return ""
    return identifier.strip().strip('`')

def quote_identifier(identifier: str) -> str:
    """
    Add backticks to an identifier for safe SQL usage.
    First normalizes (strips existing backticks) then adds fresh backticks.
    
    Args:
        identifier: A SQL identifier (normalized or not)
        
    Returns:
        The identifier wrapped in backticks
        
    Examples:
        quote_identifier("my-schema") -> "`my-schema`"
        quote_identifier("`my-schema`") -> "`my-schema`" (not double-quoted)
    """
    if identifier is None:
        return "``"
    normalized = normalize_identifier(identifier)
    return "`" + normalized.replace("`", "``") + "`"

def compute_result_table_name(use_case_id: str, use_case_name: str, tables_involved: str, inspire_database: str = "") -> str:
    """
    Compute the Unity Catalog result table name for a use case.
    
    Format: `<catalog>`.`<schema>`.`_inspire_<id>_<name>`, every part backtick-quoted.
    
    Rules:
        - Catalog and schema come from the user-provided inspire_database (catalog.schema format,
          backticks optional); without one the documented placeholder `main`.`_inspire` is used
        - Prefix is always '_inspire_'
        - Use case ID and name are lowercased with every run of non [a-z0-9] characters turned into one underscore
        - Table part capped at 200 chars (Unity Catalog allows 255 per identifier)
    """
    def _snake(value):
        return re.sub(r'[^a-z0-9]+', '_', str(value or '').lower()).strip('_')

    table_name = "_".join(part for part in ("_inspire", _snake(use_case_id), _snake(use_case_name)) if part)[:200]
    catalog, schema = parse_two_level_name(inspire_database) if inspire_database else (None, None)
    if not catalog or not schema:
        catalog, schema = "main", "_inspire"
    return build_fqn(catalog, schema, table_name)

def parse_three_level_name(name: str) -> tuple:
    """
    Parse a three-level name (catalog.schema.table or catalog.schema.column) into parts.
    Handles names with or without backticks at any level.
    
    Args:
        name: A three-level name like "catalog.schema.table" or "`cat`.`sch`.`tbl`"
        
    Returns:
        Tuple of (part1, part2, part3) with backticks stripped from each part,
        or (None, None, None) if parsing fails
        
    Examples:
        parse_three_level_name("cat.schema.table") -> ("cat", "schema", "table")
        parse_three_level_name("`cat`.`my-schema`.`table`") -> ("cat", "my-schema", "table")
        parse_three_level_name("invalid") -> (None, None, None)
    """
    if not name:
        return (None, None, None)
    
    clean_name = name.replace('`', '')
    parts = clean_name.split('.')
    
    if len(parts) == 3:
        return (parts[0].strip(), parts[1].strip(), parts[2].strip())
    return (None, None, None)

def parse_two_level_name(name: str) -> tuple:
    """
    Parse a two-level name (catalog.schema) into parts.
    Handles names with or without backticks.
    
    Args:
        name: A two-level name like "catalog.schema" or "`cat`.`my-schema`"
        
    Returns:
        Tuple of (part1, part2) with backticks stripped from each part,
        or (None, None) if parsing fails
        
    Examples:
        parse_two_level_name("cat.schema") -> ("cat", "schema")
        parse_two_level_name("`cat`.`my-schema`") -> ("cat", "my-schema")
    """
    if not name:
        return (None, None)
    
    clean_name = name.replace('`', '')
    parts = clean_name.split('.', 1)
    
    if len(parts) == 2:
        return (parts[0].strip(), parts[1].strip())
    return (None, None)

def parse_four_level_name(name: str) -> tuple:
    """
    Parse a four-level name (catalog.schema.table.column) into parts.
    Handles names with or without backticks at any level.
    
    Args:
        name: A four-level name like "catalog.schema.table.column"
        
    Returns:
        Tuple of (catalog, schema, table, column) with backticks stripped,
        or (None, None, None, None) if parsing fails
    """
    if not name:
        return (None, None, None, None)
    
    clean_name = name.replace('`', '')
    parts = clean_name.split('.')
    
    if len(parts) == 4:
        return (parts[0].strip(), parts[1].strip(), parts[2].strip(), parts[3].strip())
    return (None, None, None, None)

def build_fqn(catalog: str, schema: str, table: str = None) -> str:
    """
    Build a fully qualified name with proper backtick quoting.
    
    Args:
        catalog: Catalog name (will be normalized and quoted)
        schema: Schema name (will be normalized and quoted)
        table: Optional table name (will be normalized and quoted if provided)
        
    Returns:
        Properly quoted FQN like `catalog`.`schema` or `catalog`.`schema`.`table`
        
    Examples:
        build_fqn("cat", "my-schema") -> "`cat`.`my-schema`"
        build_fqn("cat", "schema", "table") -> "`cat`.`schema`.`table`"
    """
    cat_quoted = quote_identifier(catalog)
    schema_quoted = quote_identifier(schema)
    
    if table:
        table_quoted = quote_identifier(table)
        return f"{cat_quoted}.{schema_quoted}.{table_quoted}"
    return f"{cat_quoted}.{schema_quoted}"

def quote_qualified_name(name: str) -> str:
    parts = [p for p in str(name or "").split(".") if p.strip()]
    return ".".join(quote_identifier(p) for p in parts)

# --- LLM Model Resolution ---
# Model resolution is now handled dynamically via model_type + model_size cascade.
# Each prompt bucket is resolved at runtime to an ordered list of enabled models.
# On failure, the next enabled model in the same bucket is tried automatically.
# See: get_model_cascade_for_prompt(), _call_ai_query()

# DBTITLE 1,Prompts
# --- 1. Main Prompt Templates Dictionary ---
PROMPT_TEMPLATES = {}

USER_DIRECTIVE_SUPREMACY_PREAMBLE = """### USER DIRECTIVES ARE THE SUPREME AUTHORITY
Explicit user directives outrank every other instruction in this prompt. They come from the run inputs (business name, business domains, business priorities, strategic context, quality mode, output languages) and from the Generation Instructions section. If any rule, heuristic, quota, default, scoring anchor or example below conflicts with an explicit user directive, the user directive WINS without exception. A business domain or use case the user explicitly requires is never removed, renamed or replaced. The one thing a user directive never changes is the required response format (columns, keys, order), because downstream parsers depend on it.

"""


def with_user_directive_preamble(prompt) -> str:
    text = "" if prompt is None else str(prompt)
    if not text or text.startswith(USER_DIRECTIVE_SUPREMACY_PREAMBLE):
        return text
    return USER_DIRECTIVE_SUPREMACY_PREAMBLE + text


def smoke_render_all_prompts(templates=None, dummy_values=None, verbose=False) -> dict:
    """Phase A1 helper: best-effort render every PROMPT_TEMPLATES entry with
    dummy values for every `{placeholder}` detected. Returns a per-template
    report {name: {"ok": bool, "error": str, "placeholders": [...]}.

    Use this in calibration mode or test suites to catch stray placeholders
    after prompt edits. No network calls, no LLM invocation.
    """
    import string as _ss
    t = templates if templates is not None else PROMPT_TEMPLATES
    dv = dict(dummy_values or {})
    report = {}
    for name, body in sorted(t.items()):
        if not isinstance(body, str):
            report[name] = {"ok": False, "error": f"non-string body type={type(body).__name__}", "placeholders": []}
            continue
        placeholders = []
        try:
            placeholders = sorted({f.split(".")[0].split("[")[0] for _, f, _, _ in _ss.Formatter().parse(body) if f})
            fill = {k: dv.get(k, f"<{k}>") for k in placeholders}
            # Use format_map so missing keys fall through to __missing__ (default dict)
            class _Defaulting(dict):
                def __missing__(self, key):
                    return f"<missing:{key}>"
            rendered = body.format_map(_Defaulting(fill))
            report[name] = {"ok": True, "error": None, "placeholders": placeholders, "rendered_len": len(rendered)}
        except (KeyError, IndexError, ValueError) as _re:
            report[name] = {"ok": False, "error": f"{type(_re).__name__}: {_re}", "placeholders": placeholders}
        except Exception as _re:
            report[name] = {"ok": False, "error": f"{type(_re).__name__}: {_re}", "placeholders": placeholders}
    if verbose:
        for n, r in report.items():
            status = "OK " if r["ok"] else "FAIL"
            print(f"[A1] {status} {n} (placeholders={len(r['placeholders'])})")
            if not r["ok"]:
                print(f"      -> {r['error']}")
    return report

QUALITY_GATE_RULES = """
**D1 - CAUSAL SIGNAL STRENGTH** (Critical):
Does the data contain fields that DIRECTLY CAUSE or INDICATE the outcome being predicted/analyzed?
If you cannot trace a logical cause-effect chain from available columns to the outcome, the use case is INVALID.
No causal signals = No use case.

**D2 - CAUSE-EFFECT VALIDITY** (Critical):
Does the high level design use variables with a LOGICAL, PROVEN, INDUSTRY-RECOGNIZED cause-effect relationship with the outcome?
Correlation is NOT causation. If the relationship is speculative, the use case is INVALID.

**D3 - DATA GRANULARITY**:
Does the data exist at the RIGHT LEVEL OF DETAIL for the analysis?
If the analysis requires per-customer data but only per-store aggregates exist, the use case is INVALID.

**D4 - CRITICAL DIMENSIONS PRESENT**:
Are ALL critical dimensions (columns) required for the analysis present in the schema?
If even ONE critical column is missing, the use case is INVALID. Do NOT assume derived data can compensate.

**D5 - LOGICAL POSSIBILITY**:
Is the high level design logically possible with the given data types and structure?
If operations cannot be performed on available data, the use case is INVALID.

**D6 - METRIC VALIDITY**:
Can the claimed business metrics ACTUALLY be calculated from available fields?
If the key metric requires fabrication or heavy approximation, the use case is INVALID.

**D7 - HIGH LEVEL DESIGN MATCH**:
Does the high level design match what the data can actually support?
If the design assumes columns, tables, or relationships that do not exist in the schema, the use case is INVALID.

**D8 - SEMANTIC UNIQUENESS** (Critical: ZERO TOLERANCE):
Does this use case address a FUNDAMENTALLY DIFFERENT business question than all others? Apply ALL 5 layers of duplicate detection; a match on ANY layer means DUPLICATE:

**LAYER 1: VERB NORMALIZATION + SUFFIX STRIP**:
Normalize verb (Forecast/Predict/Anticipate/Envision = PREDICT; Detect/Identify/Reveal/Find/Discover/Uncover/Expose = DETECT; Classify/Segment/Categorize/Tier/Rank = CLASSIFY; Simulate/Model/Project = SIMULATE; Optimize/Maximize/Minimize = OPTIMIZE; Assess/Evaluate/Measure/Gauge/Quantify = ASSESS). Strip "with [...]" activation suffix. If remaining core phrase matches another use case → DUPLICATE.

**LAYER 2: NOUN SYNONYM DETECTION**:
Treat these nouns as IDENTICAL: Revenue=Income=Sales=Earnings; Cost=Expense=Spend=Expenditure; Churn=Attrition=Turnover=Defection; Leakage=Loss=Erosion=Slippage; Demand=Consumption=Usage=Volume; Risk=Exposure=Vulnerability=Threat; Delay=Latency=Bottleneck=Slowdown; Fraud=Anomaly=Irregularity; Customer=Client=Account=Subscriber; Employee=Staff=Worker=Personnel; Supplier=Vendor=Provider; Inventory=Stock=Supply.
If after replacing synonyms the core phrases match → DUPLICATE.

**LAYER 3: ENTITY-METRIC-QUESTION OVERLAP**:
Extract the BUSINESS ENTITY (what is being analyzed: customers, orders, suppliers, products, etc.), the BUSINESS METRIC (what is being measured: revenue, churn, cost, risk, etc.) and the BUSINESS QUESTION (the decision the output informs). If two use cases share the SAME entity + SAME metric + SAME question → DUPLICATE regardless of wording. The same entity and metric with a DIFFERENT question (what will it be / is it behaving abnormally / how should resources be allocated to change it) are DISTINCT use cases; keep both.
Example: "Forecast Customer Churn Rate" and "Predict Client Attrition Probability" → same entity (customer), same metric (churn), same question (who will leave?) → DUPLICATE.

**LAYER 4: TECHNIQUE SWAP (NARROW, SAME QUESTION ONLY)**:
Two use cases are duplicates ONLY if they ask the SAME business question and merely swap the technique. Example: "Forecast Customer Churn" and "Predict Customer Churn" = DUPLICATE (same question: will the customer leave?). BUT "Forecast Revenue" and "Detect Revenue Anomalies" are NOT duplicates: the first asks "what will revenue be?" and the second asks "is revenue behaving abnormally?". Different QUESTIONS on the same entity/metric are DISTINCT use cases; keep BOTH. Do NOT collapse technique diversity.

**LAYER 5: TABLE OVERLAP + SIMILAR INTENT**:
If two use cases use the SAME primary tables AND address a SIMILAR business question (even with different wording) → DUPLICATE. Different tables = potentially different use case. Same tables + same intent = definitely DUPLICATE.

**D9 - ANALYTICAL DEPTH** (Critical: RUTHLESS STANDARD):
Apply the "STRIP TEST": mentally remove all advanced analytical techniques (forecasting, classification, anomaly detection, simulation, etc.) from the High Level Design. What remains? If what remains is a trivial operation, the advanced techniques are DECORATION, not substance.

TRIVIAL (D9 ≤ 1.5: AUTOMATIC REJECTION):
- Date arithmetic: simple date comparisons → trivial calendar logic
- Threshold checks: simple value filters → trivial filter
- Basic aggregation: counting or summing by one dimension → any junior analyst can do this
- Single-column statistical measure: basic deviation from mean → basic statistics
- Ratio computation: dividing one column by another → arithmetic, not intelligence
- Sorting/ranking: ordering by a metric → trivial

WEAK (D9 = 2.0-2.5: NOT ACCEPTABLE FOR EXTREME QUALITY):
- Two-column correlation without cross-table context
- Simple moving averages or period-over-period comparisons
- Grouping by one dimension with a threshold overlay

GENUINE ANALYTICAL DEPTH (D9 ≥ 3.5: MINIMUM FOR ACCEPTANCE):
- Multi-dimensional pattern detection across 3+ variables
- Cross-table correlation requiring JOINs on 2+ tables with derived features
- Time-series forecasting with external factor integration
- Multi-step classification with contextual enrichment
- Simulation or what-if analysis with scenario comparison
- Network/graph analysis of entity relationships
- Trained ML models (XGBoost, Random Forest, LightGBM, K-Means, etc.) applied to business problems with feature engineering
- Survival analysis or time-to-event modeling (e.g., customer lifetime, equipment failure)
- Optimization models (resource allocation, scheduling, pricing) with constraint satisfaction
- Recommendation systems using collaborative filtering or content-based approaches
- Ensemble methods combining multiple models or analytical approaches for robust scoring

**D10 - ACTIVATION QUALITY** (Critical: ZERO TOLERANCE FOR TECHNICAL METHOD SUFFIXES):
Apply the "DELIVERABLE TEST": the "with [...]" suffix must describe a business DELIVERABLE: the actionable output the user receives (recommendation, action plan, priority queue, strategy, intervention, escalation brief). It must NEVER describe a technical method, data source, statistical technique, or analytical approach.

BANNED CATEGORIES (D10 = 1.0: AUTOMATIC REJECTION):
- Technical method suffixes: "with Z-Score Deviation", "with Correlation Analysis", "with Rolling STDDEV", "with Regression Slope"
- Data source suffixes: "with Payment History Cross-Reference", "with Invoice-to-Payment Date Gap Analysis", "with RFM Scoring and Cohort Aging"
- Technique suffixes: "with Seasonal Decomposition", "with Anomaly Profiling", "with Monte Carlo Simulation", "with Trend Analysis"
- Buzzword suffixes: "with Intelligence", "with Insights", "with Analytics", "with Context", "with Pattern Recognition"
- Circular suffixes: where the suffix merely restates the verb/name (e.g., "Segment Customers with Customer Segmentation")
- Domain jargon in action part: Any industry-specific technical acronym (e.g., RASK, PNR, AWB, MEL, CDL, ULD, GDS, MTOW for airlines; SKU, BOM, WIP, OEE for manufacturing; LTV, CAC, MRR, ARR for SaaS; NPA, NIM, CAR for banking; RFM, CLTV for marketing) or statistical term (Z-score, variance, coefficient, kurtosis, regression, STDDEV, deviation, correlation, decomposition, cross-reference) appearing ANYWHERE in the name

ACCEPTABLE (D10 ≥ 3.5: MINIMUM FOR EXTREME QUALITY):
- Describes a specific business deliverable: "with Pricing Recalibration Recommendations", "with Pre-Invoice Correction Actions"
- Describes an actionable intervention: "with Prioritized Outreach Queue", "with Audit Escalation Brief", "with Collection Escalation Actions"
- Describes a decision-support artifact: "with Budget Reallocation Strategy", "with Capacity Planning Recommendations", "with Retention Intervention Playbook"
- The activation suffix answers "What does the business user DO with this?" not "How was this analyzed?"

**D11 - DOMAIN & TECHNIQUE BALANCE** (portfolio-level):
Balance is a property of the PORTFOLIO, not a merit of one use case. These are the only balance targets:
- No single business domain above 25% of the use cases
- No single Analytics Technique above 30% of the use cases
- When an area is over target, look for equally strong use cases in under-represented areas. Never drop, weaken or relabel a strong use case to meet a target, and never add a weak use case to fill an under-represented area.
- Balance is never a reason to lower the score of an individual use case.

**D12 - BUSINESS RELEVANCE & OPPORTUNITY** (Critical: put yourself in the business stakeholder's seat):
Answer ALL of these for THIS use case. If you answer "no" to ANY, the use case FAILS this gate:
1. Is this use case RELEVANT to how the business actually operates day-to-day? (Not just analytically possible, actually relevant.)
2. Does it deliver TANGIBLE VALUE to the business: revenue protection, cost reduction, risk mitigation, or opportunity capture?
3. Does SOMEONE INSIDE THE BUSINESS have the exact issue this solves? Can you name the team/role/function that would consume this output and change their decisions?
4. Does this represent either a CURRENT PAIN the business wants solved, OR a FUTURE OPPORTUNITY the business could expand into?

If you cannot ground this use case in a specific business function or named stakeholder, it is an analytical exercise, not a fundable use case: FAIL.

**D13 - SPONSOR TEST** (Critical: put yourself in the Sponsor's/CFO's seat):
Answer these two questions from the SPONSOR's perspective:
1. ARE YOU WILLING TO FUND THIS USE CASE with a real budget line? (yes / no; if no, FAIL)
2. CAN YOU PITCH THIS TO THE CEO and get approval without caveats or hand-waving? (yes / no; if no, FAIL)

The sponsor is the person who signs the check. If you cannot personally stand behind funding this and defending it to the CEO, do NOT generate it.

**D14 - ENGINEERING TEST** (Critical: put yourself in the Production Engineer's seat):
Answer these three questions from the ENGINEERING / PRODUCTION team's perspective:
1. CAN YOU PUT THIS TO PRODUCTION with existing Databricks capabilities (ai_query, Delta, MLflow, Spark) and the available columns, with no fabricated external services? (yes / no; if no, FAIL)
2. CAN YOU SUPPORT AND MAINTAIN THIS long-term once deployed (monitor drift, refresh, retrain) with the existing team skills? (yes / no; if no, FAIL)
3. CAN YOU FIX ISSUES with this use case when they arise (bad predictions, schema drift, edge cases) without external dependencies or bespoke infrastructure? (yes / no; if no, FAIL)

Use cases that cannot be operated by the production team are shelfware. FAIL them here.

**D15 - DECISION CADENCE MATCH** (Critical):
Does the data refresh cadence match the business decision cadence?
- Weekly operational decisions REQUIRE at-least-weekly data freshness.
- Monthly strategic decisions can use monthly refresh.
- If decision cadence is FASTER than what the data can deliver, the UC is unactionable in practice -> FAIL.

**D16 - THE MONDAY TEST** (Critical):
If a stakeholder reads this UC's output on Monday morning, does their Monday look DIFFERENT from last Monday? Does it change a specific action they would take?
- If the output is "interesting to know" but changes nothing in the user's actual workflow -> decorative analytics -> FAIL.
- Must name at least one concrete action the output triggers.

**D17 - EXPLAINABILITY & AUDIT DEFENSIBILITY**:
If a regulator, auditor, CFO, or unhappy customer asked "how did you reach this conclusion?" -> can the output + methodology be explained in 5 minutes without invoking proprietary magic?
- For high-stakes decisions (compliance, pricing, risk, customer-facing), black-box models without local explainability -> FAIL.
- For operational/internal decisions, statistical-grounded techniques are acceptable.

**D18 - 18-MONTH LONGEVITY**:
In 18 months, will this UC still be running, or will it be abandoned?
- If the UC encodes a SHORT-TERM TACTICAL pattern (e.g., "this quarter's specific promotion mix") -> FAIL, it's a one-off not a use case.
- If it provides PERSISTENT operational advantage (always-on anomaly detection, recurring forecast, ongoing optimization) -> PASS.

**D19 - ATTRIBUTABLE IMPACT**:
Can the business team PROVE, 6 months after deployment, that this UC changed a measurable dollar amount in revenue / cost / risk?
- The output must enable a counterfactual or A/B-testable outcome.
- If impact is diffuse ("awareness", "insight", "visibility") without a traceable decision chain -> FAIL.

**COMBINED GATE RESOLUTION**:
- If D12 fails on ANY of the 4 sub-questions -> quality_score CAPPED at 2.0 (Low).
- If D13 fails on EITHER sub-question -> quality_score CAPPED at 2.0 (Low).
- If D14 fails on ANY of the 3 sub-questions -> quality_score CAPPED at 2.0 (Low).
- If D15-D19 FAIL -> quality_score CAPPED at 2.0 (Low).
- These are business-value vetoes; they override any technical quality score.
"""

QUALITY_GATE_HIGH = f"""
### 🚨 DATA QUALITY GATE 🚨

**GOAL: Generate every use case that passes the hard gates below.** One broad generation pass feeds every quality mode; value and quality tiers are assigned downstream on an absolute scale. Do not withhold a use case that passes the hard gates because you expect it to score low, and never emit one that fails them.

**SELF-CHECK PROCEDURE**: Before writing each CSV row, verify all checks below. If a use case fails D8 (duplicate) or D9 (too trivial) → SKIP it. For D10, rewrite the activation rather than skipping an otherwise strong use case. D11 is a portfolio target, never a reason to skip a strong use case.

**🚨 HARD GATES (skip use case if any fails) 🚨**:

**D8 (5-LAYER DUPLICATE DETECTION)**: Apply all 5 layers exactly as defined in D8 below. Match on ANY layer → DO NOT GENERATE (it's a duplicate).

**D9 (ANALYTICAL DEPTH: STRIP TEST)**: Remove all AI function calls mentally. If what remains is ONLY a WHERE clause, single threshold check, basic GROUP BY count, date arithmetic, or single-column ratio → DO NOT GENERATE. The use case must require at least multi-column analysis, cross-table JOINs with derived insights, or genuine statistical/predictive reasoning.

**QUALITY GUIDELINES (apply but do not over-filter)**:
- **D10 (DELIVERABLE TEST)**: The "with [...]" suffix should describe a business DELIVERABLE (recommendation, action plan, queue, strategy, intervention), NOT a technical method. Rewrite weak activations rather than skipping the use case.
- **D11 (BALANCE)**: Follow the D11 targets below. Reach balance by finding equally strong use cases in under-represented areas, never by skipping or weakening valid ones.
{QUALITY_GATE_RULES}
**FINAL CHECK** (for each use case):
1. "Can an engineer implement this using ONLY the columns in the schema, with logical cause-effect reasoning, producing a real calculable metric?" If NO → DO NOT GENERATE IT.
2. "Is this substantially different from every other use case I have generated so far (D8 check)?" If NO → DO NOT GENERATE IT.
3. "Does this require genuine analytical work beyond basic aggregation (D9 check)?" If NO → DO NOT GENERATE IT.
4. "Does the 'with [activation]' part describe a business deliverable?" If not, try to rewrite. If impossible, still include if the core analysis is strong.
5. "Have I chosen an appropriate analytical approach for this problem?" If unsure, default to the technique that best fits the data.
"""

CONTEXT_NOT_PROVIDED_MARKER = "UNKNOWN_FROM_METADATA"


def prompt_context_or_not_provided(value, field_label: str) -> str:
    text = str(value if value is not None else "").strip()
    if text:
        return text
    return f"{CONTEXT_NOT_PROVIDED_MARKER} ({field_label}): not provided for this run; do not assume or invent one"


ASSUMED_CONTEXT_RULE = (
    "**Assumed context:** Text marked `[ASSUMPTION ...]` in the context above was inferred from the business name "
    "only. It is a hypothesis, never evidence: it cannot support a claim, raise a score or justify a figure. "
    f"`{CONTEXT_NOT_PROVIDED_MARKER}` means the value is unknown. Only the schema and explicit user inputs are evidence."
)

NO_EM_DASH_RULE = (
    "Never write an em dash (U+2014) in any text you return; use a comma, colon, period or parentheses instead."
)

_EM_DASH_RUN = re.compile(r"[ \t]*[\u2014\u2015]+[ \t]*")
_EM_DASH_PUNCT_BEFORE = tuple(",.;:!?([{\u3001\u3002\uff0c\uff1a\uff1b\uff08\u060c")
_EM_DASH_PUNCT_AFTER = tuple(",.;:!?)]}\u3001\u3002\uff0c\uff1a\uff1b\uff09\u060c")


def inspire_no_em_dash(text) -> str:
    text = "" if text is None else str(text)
    if "\u2014" not in text and "\u2015" not in text:
        return text
    if re.search("[\u3040-\u30ff]", text):
        comma = "\u3001"
    elif re.search("[\u3400-\u9fff]", text):
        comma = "\uff0c"
    elif re.search("[\u0600-\u06ff]", text):
        comma = "\u060c "
    else:
        comma = ", "

    def replace(match):
        before, after = text[:match.start()], text[match.end():]
        if not before.strip() or not after.strip() or before.endswith("\n") or after.startswith("\n"):
            return ""
        if before.endswith(_EM_DASH_PUNCT_BEFORE) or after.startswith(_EM_DASH_PUNCT_AFTER):
            return "" if comma.strip() == comma else " "
        return comma
    return _EM_DASH_RUN.sub(replace, text)


QUALITY_GATE_SCORING_BLOCK = f"""
## QUALITY SCORING: ABSOLUTE, ANCHORED, ONE USE CASE AT A TIME

**CALIBRATION (applies to every dimension):**
- Score each use case on its own merits against the fixed anchors below. There is no curve, no quota, no expected distribution and no default score. It is valid for every use case in a batch, or for none, to score high.
- A dimension scores at the highest anchor whose evidence is fully present in the use case text and the AVAILABLE SCHEMA. If the evidence for an anchor is missing, use the anchor below it. Never score on presumed quality, and never pull a score toward the middle of the scale.
- Every dimension is judged on the use case alone, except D8, which compares the use case with the others in this batch only to catch duplicates.

**WHAT IS SCORED (the 19 gates are defined at the end of this block):**
- D1–D11 are scored 1.0–5.0 and combined with the weights below.
- D12–D19 are pass/fail business gates. Any FAIL is a hard veto.

**D1–D7 ANCHORS (data grounding):**
- 5.0: Fully met. Every element the gate asks about maps to named columns that exist in the schema, at the right grain, with compatible types.
- 4.0: Fully met, with one minor gap that does not change the result.
- 3.0: Met with a material gap: a critical element must be derived or approximated, or the grain needs an aggregation that weakens the signal.
- 2.0: The core claim rests on an assumption the schema does not show.
- 1.0: Not met: required columns, relationships or logic are absent.
Use half points only when the evidence sits between two anchors.

**D8: SEMANTIC UNIQUENESS** (apply the 5 duplicate-detection layers exactly as defined in D8 below, against every other use case in this batch):
- D8 = 5.0: no overlap on any layer.
- D8 = 3.0: similar topic, but Layers 3 and 4 show a different business question.
- D8 = 1.0: duplicate on any layer. Give 1.0 only to the weaker use case of the pair (less specific Statement, fewer grounded columns; if equal, the later one) and score the stronger one as if the other did not exist.

**D9: ANALYTICAL DEPTH** (apply the STRIP TEST and the categories of D9 below, plus the TECHNIQUE MATCH TEST). Mentally remove all AI function calls from the High Level Design and score what is LEFT:
- D9 = 1.0-1.5 (VETO): a TRIVIAL pattern of D9 (date arithmetic, thresholds, basic aggregation, filtering, sorting, ratios, single-column statistics, basic period-over-period comparison).
- D9 = 2.0 (VETO): a WEAK pattern of D9 (two-column correlation without cross-table context, simple moving average, one-dimensional grouping + threshold).
- D9 = 2.0 (TECHNIQUE MISMATCH: VETO): the Analytics Technique is any of [Anomaly Detection, Clustering, Predictive Modeling, Classification, Forecasting, Regression Analysis, Segmentation, Survival Analysis, Recommendation, Network Analysis, Optimization, Simulation] but the High Level Design describes ONLY SQL operations (z-scores, window functions, CASE WHEN, GROUP BY, NTILE, simple thresholds) without model training, statistical modeling, or a technique-appropriate algorithm. This cap applies regardless of SQL complexity.
- D9 = 3.0 (MODERATE): several statistical operations on 2-3 variables without multi-dimensional reasoning or ML. Acceptable ONLY for techniques where SQL is genuinely appropriate (Pareto Analysis, basic Cohort Analysis, Funnel Analysis).
- D9 = 3.5 (GENUINE DEPTH MINIMUM): multi-dimensional analysis across 3+ variables with derived features, or a single trained model on well-prepared features.
- D9 = 4.0-4.5 (STRONG): cross-table feature engineering with a trained model (XGBoost, Random Forest, LightGBM, Isolation Forest, K-Means, DBSCAN); a multi-step predictive pipeline with model evaluation; clustering/segmentation with profiling; survival analysis with time-to-event modeling; or Monte Carlo/scenario simulation.
- D9 = 5.0 (EXCEPTIONAL): a multi-technique pipeline (feature engineering → model comparison → ensemble scoring → optimization or simulation → ai_query reasoning), graph analysis combined with ML, or time-series modeling with multiple drivers and scenarios.

**D10: ACTIVATION QUALITY** (apply the DELIVERABLE TEST of D10 below to the "with [...]" suffix and its jargon check to the action part before "with"):
- D10 = 1.0 (VETO): any BANNED CATEGORY of D10: a technical method, technique, data source or buzzword suffix, or domain/statistical jargon anywhere in the name.
- D10 = 2.0: circular; the suffix restates the verb or name ("Segment Customers with Customer Segmentation", "Detect Anomalies with Anomaly Detection").
- D10 = 3.0: vague deliverable ("with Recommendations", "with Action Plan", "with Strategy").
- D10 = 4.0-5.0: a specific, role-appropriate deliverable as in the ACCEPTABLE list of D10 ("with Prioritized Outreach Queue", "with Audit Escalation Brief", "with Retention Intervention Playbook").
A C-suite executive must be able to read the FULL name in 3 seconds and understand both the problem AND the deliverable.

**D11 (DOMAIN & TECHNIQUE BALANCE):** Balance is a portfolio property applied during generation and selection, never to an individual score. Score D11 = 5.0 for every use case.

**DIMENSION WEIGHTS (D1–D11):** D1 Causal Signal 15%, D2 Cause-Effect 15%, D3 Granularity 10%, D4 Dimensions 10%, D5 Logical 5%, D6 Metric 5%, D7 Design Match 5%, D8 Semantic Uniqueness 15%, D9 Analytical Depth 10%, D10 Activation Quality 5%, D11 Domain Balance 5%.

**QUALITY SCORE CALCULATION:**
Step 1: weighted average = (D1*0.15) + (D2*0.15) + (D3*0.10) + (D4*0.10) + (D5*0.05) + (D6*0.05) + (D7*0.05) + (D8*0.15) + (D9*0.10) + (D10*0.05) + (D11*0.05)
Step 2 (HARD VETO): quality_score = min(weighted average, 2.0) if ANY of these is true:
- D1 ≤ 2.5 (weak or no causal signal)
- D2 ≤ 2.5 (weak or no cause-effect validity)
- D8 ≤ 2.5 (duplicate on any of the 5 layers)
- D9 ≤ 2.5 (trivial or weak depth; fails the STRIP TEST)
- D10 ≤ 2.0 (technical method, data source or filler as activation, or jargon in the name; fails the DELIVERABLE TEST)
- COLUMN or TABLE HALLUCINATION (the use case relies on a column or table that is not in the AVAILABLE SCHEMA)
- Any of D12–D19 FAILS as defined below (business relevance, sponsor, engineering, decision cadence, Monday test, explainability, longevity, attributable impact)
Step 3 (SOFT VETO): quality_score = min(quality_score, 3.0) if ANY of these is true:
- 2.5 < D9 ≤ 3.0 (moderate depth)
- 2.0 < D10 ≤ 3.0 (vague deliverable)
- D3 ≤ 2.5 (insufficient granularity)
- D7 ≤ 2.5 (design does not match the schema)
A use case cannot compensate for a vetoed flaw with high scores on other dimensions.

**QUALITY LABEL** (from the final quality_score): 4.5–5.0 = "Ultra High"; 4.0–4.49 = "Very High"; 3.5–3.99 = "High"; 2.5–3.49 = "Medium"; 2.0–2.49 = "Low"; 1.5–1.99 = "Very Low"; 1.0–1.49 = "Ultra Low".

**🔥 QUALITY REASONS (quality_summary field): DATA-TO-VALUE CHAIN EXPLANATION 🔥**

The `quality_summary` field is NOT a hallucination check report. It MUST be a detailed, step-by-step explanation of HOW the available data columns will deliver the promised business value. The reader should finish reading this field and think: "Yes, this data is clearly sufficient to produce that outcome."

**MANDATORY STRUCTURE for quality_summary (follow this pattern):**
1. **Identify the signal columns**: Name the specific columns that carry the analytical signal (e.g., "Column `order_date` and `ship_date` provide the temporal gap signal")
2. **Describe the analytical operation**: Explain what aggregation, comparison, or transformation will be performed (e.g., "by computing DATEDIFF(ship_date, order_date) we get fulfillment latency per order")
3. **Derive the metric/KPI**: Show what measurable metric emerges (e.g., "this produces an avg_fulfillment_days KPI per warehouse")
4. **Connect to business value**: Explain how that metric delivers the promised use case outcome (e.g., "comparing this KPI across warehouses reveals bottleneck locations, directly enabling the promised logistics cost reduction")

**EXAMPLE of a GOOD quality_summary:**
"Gap analysis signal detected via columns `order_date` and `ship_date` in orders table; computing DATEDIFF yields per-order fulfillment latency. Aggregating by `warehouse_id` with AVG and PERCENTILE produces warehouse-level fulfillment KPI. Joining with `warehouse_costs.operational_cost` enables cost-per-day-delayed calculation. Combining fulfillment KPI with cost data surfaces the top bottleneck warehouses driving excess logistics spend, directly delivering the promised supply chain cost optimization. Column `product_category` adds dimensional richness for category-level drill-down."

**EXAMPLE of a BAD quality_summary (DO NOT produce this):**
"NO HALLUCINATION: All columns verified. Strong causal signals. Unique business problem."
↑ This tells the reader NOTHING about HOW the data delivers value. REJECTED.

**RULES for quality_summary:**
- MUST reference at least 2-3 specific column names from the schema
- MUST describe at least one concrete analytical operation (aggregation, join, comparison, statistical function)
- MUST name at least one derived metric or KPI
- MUST connect the metric back to the specific business outcome promised by the use case
- Never state an invented percentage or currency figure; if a magnitude matters, write `[ESTIMATE: depends on <column_name>]`
- Length: 150-400 characters. Be specific and dense, not verbose.
- Do NOT start with "NO HALLUCINATION"; that check goes in `hallucination_check` field only
{QUALITY_GATE_RULES}
"""

HONESTY_CHECK_CSV = """

### 🎯 HONESTY SELF-ASSESSMENT (MANDATORY - INTEGRATED IN CSV) 🎯

You MUST include honesty self-assessment as TWO ADDITIONAL COLUMNS at the END of your CSV:
- **"honesty_score"**: Your honest score 0-100 for the ENTIRE output quality
- **"honesty_justification"**: Brief justification (max 250 chars)

Another more powerful LLM will review your output and generate its own honesty score to compare against yours - BE EXTREMELY HONEST. Try EXTREMELY hard to achieve 100% honesty - do not inflate your score.

**IMPORTANT**: Add these 2 columns to EVERY row. Use the SAME score and justification for all rows (it's for the entire output, not per-row).
"""

HONESTY_CHECK_JSON = """

### 🎯 HONESTY SELF-ASSESSMENT (MANDATORY - INTEGRATED IN JSON) 🎯

You MUST wrap your JSON output with honesty self-assessment fields. Your response must be a JSON object with this structure:
```json
{{
  "honesty_score": <your score 0-100>,
  "honesty_justification": "<brief justification, max 250 chars>",
  "data": <your actual output here>
}}
```

Another more powerful LLM will review your output and generate its own honesty score to compare against yours - BE EXTREMELY HONEST. Try EXTREMELY hard to achieve 100% honesty - do not inflate your score.
"""


HONESTY_CHECK_TABLE = """

### 🎯 HONESTY SELF-ASSESSMENT (MANDATORY - AS TABLE FOOTER) 🎯

You MUST include honesty self-assessment as TWO ADDITIONAL COLUMNS at the END of your table:
- **"honesty_score"**: Your honest score 0-100 for the ENTIRE output quality  
- **"honesty_justification"**: Brief justification (max 250 chars)

Another more powerful LLM will review your output and generate its own honesty score to compare against yours - BE EXTREMELY HONEST. Use the SAME score and justification for all rows.
"""

# --- 1. Business Context Worker Prompt ---
PROMPT_TEMPLATES["BUSINESS_CONTEXT_WORKER_PROMPT"] = """
### PERSONA

You are a **Principal Business Analyst** and recognized industry specialist with 15+ years of deep expertise in the `{industry}` industry. You are a master of business strategy, operations, and data-driven decision making.

### CONTEXT

**Assignment Details:**
- Industry/Business Name: `{name}`
- Type: {type_description}
- Target: Research and document comprehensive business context for this {type_label}

### TASK DEFINITION

Research and provide comprehensive business context information across 6 specific dimensions. Generate industry-specific information that will serve as the foundation for building a data model. Your output must be a single, well-structured JSON object.

### WORKFLOW

**Step 1: Research**
Leverage your deep industry knowledge of `{industry}` to understand the {type_label} named `{name}`.

**Step 2: Information Gathering**
For each of the 6 required fields, identify comprehensive and specific details:
1. **Business Context**: A concise, grounded description of the organization assembled ONLY from (a) the supplied `{industry}` and `{name}` inputs, (b) signals inferable from the schema (table and column names, relationships), and (c) widely-known industry characteristics that are NOT firm-specific. Do NOT invent specific figures such as revenue, location counts, customer counts, employee counts, market share, Fortune-500 status, ISO certifications, years in business, or competitor comparisons; if such a figure is not explicitly provided in the inputs, write `UNKNOWN_FROM_METADATA` in its place instead of fabricating a number. Keep the context factual: what the industry typically does, what a business of this name/industry plausibly offers, and what the supplied data could support analytically. If nothing can be said with confidence, write `UNKNOWN_FROM_METADATA`.
2. **Strategic Goals**: The high-level long-term strategic objectives. You MUST select 3-7 goals from this standard list that are MOST relevant to this business:
   - "Reduce Cost" (automation, efficiency, waste reduction)
   - "Boost Productivity" (faster processes, better tools, streamlined workflows)
   - "Increase Revenue" (new revenue streams, upselling, cross-selling, market expansion)
   - "Mitigate Risk" (fraud detection, compliance, security, audit trails)
   - "Protect Revenue" (churn prevention, retention, customer satisfaction)
   - "Align to Regulations" (compliance automation, regulatory reporting, audit support)
   - "Improve Customer Experience" (personalization, faster service, quality improvements)
   - "Enable Data-Driven Decisions" (analytics, insights, forecasting, predictions)
3. **Business Priorities**: Immediate and near-term focus areas for the organization.
4. **Strategic Initiative**: `UNKNOWN_FROM_METADATA` unless the inputs above name an initiative this business is running; never infer one from the business name.
5. **Value Chain**: The primary activities that create value for the customer.
6. **Revenue Model**: `UNKNOWN_FROM_METADATA` unless the inputs above state how this business earns revenue; never infer it from the business name.

**Step 3: JSON Construction**
Format all information as a single JSON object with 6 keys. Values should be descriptive strings (not lists).

### RULES AND CONSTRAINTS

1. **Descriptive Strings**: Clear, concise descriptions in specific, industry-relevant terminology. The only placeholder allowed is `UNKNOWN_FROM_METADATA`, written wherever the inputs do not support a statement.
2. **Strategic Goals Format**: Comma-separated list of 3-7 goals from the standard list above with brief elaboration. Example: "Reduce Cost (automate manual processes), Increase Revenue (expand digital channels), Mitigate Risk (enhance fraud detection)"
3. **Business Context Grounding**: MUST be specific, not generic: describe the core operations, how the business creates value, and what the supplied inputs and data can support analytically. It MUST NOT contain invented scale or financial figures (revenue, location or country counts, headcount, customer counts, market share, rankings, certifications); write `UNKNOWN_FROM_METADATA` wherever such a figure is not supplied in the inputs. REJECTED: "General business operations", "A company in a region" (generic), and any figure that is not in the inputs (invented).
4. **Punctuation**: """ + NO_EM_DASH_RULE + """

### OUTPUT FORMAT

Return ONLY a valid JSON object (no text before or after, no explanations), wrapped exactly as shown, with the 6 fields inside `data`. Start with `{{"honesty_score":`.

```json
{{
  "honesty_score": 85,
  "honesty_justification": "brief justification, max 250 chars",
  "data": {{
    "business_context": "What the business does, how it creates value, and what the supplied data can support; UNKNOWN_FROM_METADATA for any figure not supplied",
    "strategic_goals": "Goal1 (elaboration), Goal2 (elaboration), Goal3 (elaboration)",
    "business_priorities": "string description",
    "strategic_initiative": "UNKNOWN_FROM_METADATA unless the inputs name an initiative",
    "value_chain": "string description",
    "revenue_model": "UNKNOWN_FROM_METADATA unless the inputs state how revenue is earned"
  }}
}}
```
""" + HONESTY_CHECK_JSON


PROMPT_TEMPLATES["GENERAL_INSTRUCTION_INTERPRETER_PROMPT"] = """
### PERSONA

You are a **Principal Data Strategy Interpreter** with expertise in data engineering, analytics, and business intelligence. Your job is to parse free-text user instructions into structured, actionable directives that will be injected into downstream AI prompts.

### TASK

Parse the user's free-text generation instructions into the 8 structured categories below (categories 5-8 are optional). Extract EVERY directive the user specified -- do not ignore or summarize away any instruction. If the user did not provide instructions for a category, leave it empty.

### INPUT

**User Generation Instructions:** {generation_instructions}
**Business Name:** {business_name}

### CATEGORIES TO EXTRACT

**1. Table Joining Instructions** -- User wants specific tables joined together.
Examples: "join table X and Y on column Z", "join catalog.schema.tableA.colA with catalog.schema.tableB.colB", "combine customers with orders on customer_id"
Extract: left table (fully qualified if possible), right table, join column(s), join type (default inner).
CRITICAL: When joins are specified, the joined tables MUST be passed as a single combined unit (not separately) to use case generation.

**2. Goal Setting** -- User wants to focus use case generation on a specific business goal.
Examples: "find all use cases that discover fraud", "focus on customer churn prevention", "optimize supply chain efficiency"
Extract: goal description, a summary sentence, and map to the closest standard strategic goal from this list:
- "Reduce Cost", "Boost Productivity", "Increase Revenue", "Mitigate Risk", "Protect Revenue", "Align to Regulations", "Improve Customer Experience", "Enable Data-Driven Decisions"
CRITICAL: The goal sets the scope and the alignment of the portfolio. Use cases that serve the goal MUST be generated, kept in scope, scored HIGH on strategic alignment by the alignment step after selection, and never penalized for domain or technique balance. The goal NEVER exempts a use case from the quality bar of the selected quality mode: a goal-aligned use case is scored on the same absolute quality anchors as every other use case, and one that fails the bar is still filtered out.

**3. Forced Techniques** -- User wants specific tools, algorithms, or AI techniques used.
Examples: "use Random Forest", "use XGBoost for classification", "apply LSTM for forecasting"
Extract: technique name and the context for when to apply it.
CRITICAL: The Genie Code implementation instructions MUST use the specified technique.

**4. Table Ignore Instructions** -- User wants certain tables excluded.
Examples: "ignore tables named __log*", "ignore tables starting with tmp_", "exclude audit_trail table"
Extract: the pattern and its type (prefix, suffix, exact, or glob pattern).
CRITICAL: These tables MUST be filtered out during table classification.

**5. Construct Exclusions** (OPTIONAL) -- User explicitly states business does NOT have certain constructs that LLMs might otherwise hallucinate.
Examples: "we don't have a loyalty program", "no NPS survey data", "no supplier scorecard", "no tiered pricing", "no commission scheme"
Extract: a list of explicitly-stated non-existent constructs. Only extract what the USER EXPLICITLY STATED -- do NOT infer from silence. If the user did not mention any exclusion, return an empty list.
CRITICAL: Use cases referencing any of these constructs MUST be rejected downstream.

**6. Change-Management Posture** (OPTIONAL) -- User states the business's tolerance for deployment risk and operational change.
Examples: "we're conservative about production changes" / "we prefer experimental pilots" / "we're in crisis-response mode, ship fast" / "balanced cautious but willing"
Extract: one of `conservative`, `balanced`, `experimental`, `crisis-response`, or empty string if not stated.
CRITICAL: Conservative posture MUST trigger rollback mandates in UC Solutions. Experimental posture MUST trigger pilot-scope mandates. Other postures have no mandates.

**7. Data Characteristics** (OPTIONAL) -- User states factual properties of specific tables that constrain what analyses are possible.
Examples: "claims table has no labeled fraud outcomes", "sales_transactions only has 6 months history", "provider NPIs are reliable but taxonomies are stale", "inventory counts are monthly snapshots, not daily"
Extract: a dict mapping `table_name` → plain-English description. Only extract user-explicit statements. Do NOT infer from column names.
CRITICAL: UC technique selection MUST respect these characteristics. If user states no labeled outcomes for a table, do NOT propose supervised ML on that table.

**8. Best of Best Count** (OPTIONAL) -- User states how many use cases should be starred as Best of Best.
Examples: "star the top 10 use cases", "best of best: 15", "mark the 5 best of the best", "no stars"
Extract: a whole number from 0 to 100 ("no stars" is 0), or null if the user did not state a number. Only extract what the USER EXPLICITLY STATED.
CRITICAL: The number only decides how many use cases are starred; it never adds, removes or reorders a use case.

### OUTPUT RULES

1. For EACH downstream prompt listed in per_prompt_instructions, generate a SPECIFIC, TAILORED instruction string. Do NOT copy-paste the same text everywhere -- each prompt has a different purpose and needs different emphasis.
2. If the user provided no instructions for a category, set it to empty array/null.
3. The per_prompt_instructions values are INJECTED VERBATIM into each prompt as non-negotiable rules. Write them as direct commands the LLM must follow.
4. Be thorough -- if the user said "focus on fraud detection", the alignment prompt instruction should say "Score any use case related to fraud detection as HIGH Strategic Alignment (4.8-5.0)" not just "consider fraud".


### DOWNSTREAM PROMPTS REFERENCE

Below is the list of downstream prompts and their purpose. Use this to generate SPECIFIC, TAILORED instructions for each one:

| Prompt Name | Purpose | What Instructions Are Relevant |
|---|---|---|
| `FILTER_BUSINESS_TABLES_PROMPT` | Classifies database tables as BUSINESS or TECHNICAL. Decides which tables to keep or exclude. | Table ignore patterns, table join hints (keep joined tables as BUSINESS) |
| `BASE_USE_CASE_GEN_PROMPT` | Generates analytics/ML use cases from table schemas and business context. This is the core use case generator. | Goal focus (what use cases to prioritize), table joins (treat joined tables as one unit), forced techniques (mention them as preferred approaches) |
| `COMBINED_VALUE_QUALITY_SCORE_PROMPT` | Combines value and quality scores into a final ranking. Decides which use cases survive to the final output. | Goal protection (goal-aligned use cases are never marked down for domain or technique balance; value and quality are scored on the same absolute anchors, so the goal never exempts a use case from the quality bar), construct exclusions |
| `STRATEGY_ALIGNMENT_PROMPT` | After selection, aligns each final use case to the strategy entries and scores Strategic Alignment (1-5). It never adds, keeps or removes a use case. | Goal alignment (score goal-aligned use cases 4.8-5.0 on Strategic Alignment) |
| `BEST_OF_BEST_SCORING_PROMPT` | Scores every surviving use case on the 19 quality gates (0-10) for final selection and ranking. | Goal relevance (score goal-aligned use cases high on D12 and never lower them for portfolio balance), construct exclusions, data characteristics, forced techniques |
| `ULTRA_HIGH_ELITE_JUDGE_PROMPT` | Ultra High mode only: judges each candidate use case against the absolute elite bar (business value, metadata evidence, implementability, actionability, red team) to decide which use cases are elite. | Goal relevance (judge business value and actionability against the goal; the elite bar stays identical for goal-aligned use cases), construct exclusions (REJECT them), data characteristics (judge evidence and implementability against them), forced techniques (never penalize a required technique) |
| `REVIEW_USE_CASES_PROMPT` | Deduplicates use cases by removing overlapping or redundant ones. | Goal coverage (never remove a goal-aligned use case because others cover the same domain or technique; true duplicates still collapse to the stronger one) |
| `DOMAIN_FINDER_PROMPT` | Assigns each use case to a Business Domain (e.g., "Fraud Detection", "Supply Chain"). | Goal alignment (ensure goal-related use cases land in the most relevant domain) |
| `SUBDOMAIN_DETECTOR_PROMPT` | Refines domains into Subdomains (e.g., under "Fraud Detection": "Transaction Fraud", "Identity Fraud"). | Goal alignment (group goal-related use cases into meaningful sub-categories) |
| `USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT` | Generates detailed implementation instructions for each use case (the "Genie Code" -- analytical approach, table joins, MLflow, output schema). | Table joins (specify exact JOIN SQL), forced techniques (MANDATE the specified algorithm), goal framing (frame implementation around the goal) |
| `THEME_ASSIGNMENT_PROMPT` | After Best-of-Best scoring, names the idea themes of the whole scored portfolio in one call and puts every use case in one, so the trim sees one name per idea. | Theme granularity (keep apart or merge themes the user names) |
| `SUMMARY_GEN_PROMPT` | Creates the executive summary of all generated use cases, domains, and strategic alignment. | Goal emphasis (highlight goal-aligned use cases prominently in the executive narrative) |

### OUTPUT FORMAT

Return ONLY a valid JSON object. Do NOT wrap in markdown code fences (no ```json). Do NOT use single quotes. Do NOT add any text before or after the JSON. The very first character of your response MUST be {{ and the very last must be }}. Start with {{.
Return ONLY a valid JSON object. Start with {{.

```json
{{
  "table_joins": [
    {{"left_table": "catalog.schema.table", "right_table": "catalog.schema.table2", "join_columns": ["col1"], "join_type": "inner"}}
  ],
  "goal_setting": {{
    "has_goal": true,
    "goal_description": "Discover fraud in any shape or form across all data",
    "goal_summary_sentence": "Comprehensive fraud detection and prevention across all business processes",
    "strategic_goal_match": "Mitigate Risk"
  }},
  "forced_techniques": [
    {{"technique": "XGBoost", "context": "for all classification use cases"}}
  ],
  "table_ignores": [
    {{"pattern": "__log*", "type": "prefix"}}
  ],
  "construct_exclusions": ["loyalty program", "NPS survey"],
  "change_management_posture": "conservative",
  "data_characteristics": {{
    "samples.schema.table_name": "no labeled outcomes; 6 months history"
  }},
  "best_of_best_count": null,
  "per_prompt_instructions": {{
    "FILTER_BUSINESS_TABLES_PROMPT": "MUST exclude all tables matching these patterns: [patterns]. Mark them as TECHNICAL with score 0 regardless of content.",
    "BASE_USE_CASE_GEN_PROMPT": "MANDATORY GOAL: [goal]. Generate EVERY possible use case related to this goal. Tables [X] and [Y] MUST be treated as pre-joined on [column]. Use [technique] for [context]. USER-DECLARED CONSTRUCT EXCLUSIONS: do NOT frame any Use Case around [exclusions] -- these do not exist in this business. CHANGE-MANAGEMENT POSTURE: [posture] -- if conservative, every Solution MUST name a baseline-fallback or rollback trigger; if experimental, every Solution MUST name a pilot scope (top-N units / single-dimension test). DATA CHARACTERISTICS: [per-table notes] -- Technique selection MUST respect these; do NOT propose supervised ML on tables with no labeled outcomes.",
    "COMBINED_VALUE_QUALITY_SCORE_PROMPT": "Score use cases related to [goal] on the same absolute value and quality anchors as every other use case: the goal never exempts a use case from the quality bar of the selected quality mode, and goal-aligned use cases are never marked down for domain or technique balance. Use cases referencing USER-DECLARED CONSTRUCT EXCLUSIONS [exclusions] MUST be filtered out.",
    "STRATEGY_ALIGNMENT_PROMPT": "Any use case related to [goal] MUST receive a Strategic Alignment score of 4.8-5.0.",
    "BEST_OF_BEST_SCORING_PROMPT": "Use cases related to [goal] MUST score high on D12 and MUST NOT be lowered for portfolio balance. Use cases referencing USER-DECLARED CONSTRUCT EXCLUSIONS [exclusions] MUST score 0-2 on D12. Score D1-D7 against USER-STATED DATA CHARACTERISTICS [per-table notes] (no supervised ML on tables without labeled outcomes). Use cases that use [technique] where the user required it MUST NOT be penalized for that choice.",
    "ULTRA_HIGH_ELITE_JUDGE_PROMPT": "Judge the business_value and actionability of use cases related to [goal] against [goal], and apply the same absolute elite bar to every use case: goal alignment never exempts a use case from that bar, and goal-aligned use cases are never marked down for domain or technique balance. A use case that references USER-DECLARED CONSTRUCT EXCLUSIONS [exclusions] is never elite: it MUST receive the REJECT verdict. Judge evidence_strength and implementability against USER-STATED DATA CHARACTERISTICS [per-table notes] (no supervised technique on tables without labeled outcomes). Use cases that use [technique] where the user required it MUST NOT be penalized for that choice.",
    "REVIEW_USE_CASES_PROMPT": "Never remove a use case that serves [goal] because other use cases cover the same domain or technique; when two goal-aligned use cases are true duplicates, keep the stronger one. DROP any use case that references USER-DECLARED CONSTRUCT EXCLUSIONS: [exclusions]. DROP any use case whose technique contradicts USER-STATED DATA CHARACTERISTICS: [per-table notes].",
    "DOMAIN_FINDER_PROMPT": "Ensure use cases related to [goal] are assigned to the most relevant domain. Create a dedicated domain if needed.",
    "SUBDOMAIN_DETECTOR_PROMPT": "Ensure [goal]-related use cases are grouped into meaningful subdomains that reflect different aspects of [goal].",
    "USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT": "MANDATORY: Join tables [X] and [Y] on [column] as specified. Use [technique] as the primary analytical algorithm. Frame the implementation around [goal]. Do NOT reference USER-DECLARED CONSTRUCT EXCLUSIONS: [exclusions].",
    "THEME_ASSIGNMENT_PROMPT": "Keep [theme A] and [theme B] as separate themes.",
    "SUMMARY_GEN_PROMPT": "Emphasize [goal] as a key strategic priority in the executive summary. Highlight use cases that address this goal prominently."
  }}
}}
```

If the user provided NO generation instructions (empty string), return:
```json
{{
  "table_joins": [],
  "goal_setting": {{"has_goal": false, "goal_description": "", "goal_summary_sentence": "", "strategic_goal_match": ""}},
  "forced_techniques": [],
  "table_ignores": [],
  "construct_exclusions": [],
  "change_management_posture": "",
  "data_characteristics": {{}},
  "best_of_best_count": null,
  "per_prompt_instructions": {{
    "FILTER_BUSINESS_TABLES_PROMPT": "",
    "BASE_USE_CASE_GEN_PROMPT": "",
    "COMBINED_VALUE_QUALITY_SCORE_PROMPT": "",
    "STRATEGY_ALIGNMENT_PROMPT": "",
    "BEST_OF_BEST_SCORING_PROMPT": "",
    "ULTRA_HIGH_ELITE_JUDGE_PROMPT": "",
    "REVIEW_USE_CASES_PROMPT": "",
    "DOMAIN_FINDER_PROMPT": "",
    "SUBDOMAIN_DETECTOR_PROMPT": "",
    "USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT": "",
    "THEME_ASSIGNMENT_PROMPT": "",
    "SUMMARY_GEN_PROMPT": ""
  }}
}}
```
""" + HONESTY_CHECK_JSON

GENERATION_DIRECTIVE_CATEGORIES_BY_PROMPT = {
    "FILTER_BUSINESS_TABLES_PROMPT": ("table_ignores", "table_joins"),
    "BASE_USE_CASE_GEN_PROMPT": ("goal_setting", "table_joins", "forced_techniques", "construct_exclusions", "change_management_posture", "data_characteristics"),
    "COMBINED_VALUE_QUALITY_SCORE_PROMPT": ("goal_setting", "forced_techniques", "construct_exclusions", "data_characteristics"),
    "STRATEGY_ALIGNMENT_PROMPT": ("goal_setting",),
    "BEST_OF_BEST_SCORING_PROMPT": ("goal_setting", "forced_techniques", "construct_exclusions", "change_management_posture", "data_characteristics"),
    "ULTRA_HIGH_ELITE_JUDGE_PROMPT": ("goal_setting", "forced_techniques", "construct_exclusions", "change_management_posture", "data_characteristics"),
    "REVIEW_USE_CASES_PROMPT": ("goal_setting", "construct_exclusions", "data_characteristics"),
    "DOMAIN_FINDER_PROMPT": ("goal_setting",),
    "SUBDOMAIN_DETECTOR_PROMPT": ("goal_setting",),
    "USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT": ("goal_setting", "table_joins", "forced_techniques", "construct_exclusions", "change_management_posture", "data_characteristics"),
    "SUMMARY_GEN_PROMPT": ("goal_setting",),
    "THEME_ASSIGNMENT_PROMPT": (),
}

PROMPT_TEMPLATES["BASE_USE_CASE_GEN_PROMPT"] = """### 0. PERSONA ACTIVATION

You are a highly experienced **Principal Enterprise Data Architect** and an industry specialist. Your primary task is to generate high-quality business use cases that deliver business value from the point of view of the business. These use cases will be implemented by Genie Code, which understands Databricks SQL, AI functions, and statistical techniques.

### BUSINESS CONTEXT
**Industry:** {industry}
**Business Context:** {business_context}
**Strategic Initiative:** {strategic_initiative}
**Value Chain:** {value_chain}
**Revenue Model:** {revenue_model}

""" + ASSUMED_CONTEXT_RULE + """

---

### 🔥🔥🔥 HIGHEST PRIORITY: USER-PROVIDED ADDITIONAL CONTEXT 🔥🔥🔥

{additional_context_section}

---


---

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}

---

### 🚨 ANTI-HALLUCINATION (TABLE + COLUMN LEVEL): REJECTION ON VIOLATION 🚨

**PRODUCTION-READINESS CHECKLIST (EVERY UC MUST PASS BEFORE YOU EMIT IT):**

Each UC you write must pass ALL of the following. If any fails, either fix the UC or don't emit it. No partial-credit shipping:

1. **DECISION VERB (not analytical verb)**: Name must start with a decision action: `Detect / Reduce / Increase / Predict / Prevent / Optimize / Classify / Segment / Forecast / Prioritize / Recover / Accelerate / Extend / Stabilize / Rebalance / Eliminate / Recommend / Personalize / Protect / Identify (only when paired with an action-enabled output)`. REJECT: `Map / Analyze / Understand / Study / Examine / Investigate / Explore / Measure` as primary verbs (these describe curiosity, not decisions; a business owner can't "approve" them).

2. **TECHNIQUE-DATA FIT**: The chosen `analytics_technique` must match the data shape:
   - `Sentiment Analysis / Extraction / Document Processing` (and any language-model reasoning over text) → ONLY if the UC cites a genuine free-text column (review / comment / description / note / transcript / feedback / body / message / review_text). REJECT if the cited "text" column is a short structured field (name, label, 2-3 word desc, category string).
   - `Forecasting / Trend Analysis / Cohort Analysis / Survival Analysis` → ONLY if there's a temporal key usable as time axis.
   - `Clustering / Segmentation / Classification` → need ≥3 engineered features per subject.
   - `Anomaly Detection` → needs repeated observations per entity (an entity key plus an event timestamp or event grain), not one row per entity.

3. **TECHNIQUE-LABEL CONSISTENCY**: `analytics_technique` column MUST match what the Solution / HLD actually does. If HLD runs Ridge regression, technique must be "Regression Analysis", not "Correlation Analysis". If HLD runs STL decomposition + gap check, technique is "Trend Analysis", not "Anomaly Detection".

4. **SOLUTION SPECIFICITY (HLD IMPLEMENTABLE)**: The High-Level-Design must be concrete enough that a data scientist could implement without follow-up. Required elements:
   - Exact table joins with ON-clause columns
   - Named feature-engineering formulas (not "compute features")
   - Specific model choice and key hyperparameter (e.g., "Ridge regression α=1.0", "Isolation Forest contamination=0.05", "XGBoost max_depth=6")
   - Output schema: what columns/rows does Genie produce (score list / tier label / ranked recommendations / alert queue)
   - Explicit `ai_query` call describing the natural-language recommendation output

5. **ACTIONABLE OUTPUT (NOT JUST INSIGHT)**: Every UC must end with a decision-ready artifact a human can act on TOMORROW. Valid endings: "prioritized intervention list / alert queue / score-sorted customer list / tier-label assignment / reallocation recommendation / investigation queue / policy trigger". INVALID endings: "dashboard / report / understanding / visibility / map / summary" alone.

6. **UNIQUE CONTRIBUTION**: Before emitting a UC, mentally scan the prior ones in this batch. If it varies from an earlier UC only in a modifier (`page format` vs `page yield` vs `page conversion`), COLLAPSE into ONE UC with the best technique. Same verb + same object + same primary table on same batch = dup. Different modifier ≠ different decision.

7. **HARD GATES, NOT GUESSED SCORES**: Emit every UC that passes this checklist and the hard gates of the quality gate below (grounding, D8 duplicates, D9 strip test, D10 deliverable). Do not withhold a passing UC because you expect it to score low; value and quality tiers are assigned downstream on an absolute scale. Never emit a UC that fails a hard gate, and never add filler to reach a count.

8. **NO COMPOSITE / STACKED UCs**: One decision per UC. If a UC has 2+ unrelated sub-decisions ("Reduce X AND Forecast Y"), split into two UCs.

---

**PERSONA & WORKING METHOD (apply throughout):**

You are a **20-year Principal Business Analyst** with deep domain expertise. Enumerate EVERY impactful business use case the data structurally supports: operational, strategic, tactical, risk, growth, efficiency, compliance, customer-experience, across every function the schema touches. Do NOT curate a small subset.

**Two-phase mental workflow:**
1. **Enumerate decisions** (business-first, no algorithms). List every operational/strategic choice a business user could make; frame each as a decision verb (prioritize / allocate / approve / reduce / target / escalate / retain / schedule / price / renegotiate / alert / recover). Cover every decision the schema supports; the count follows from the schema and is never a target.
2. **Select the best algorithm per decision**: zero bias toward "easy" or "familiar". A heavier algorithm is acceptable IF the data supports it AND the decision warrants it. Consider the full spectrum:
   - Statistical: Pareto / Trend / Correlation / Cohort / Survival / Ranking
   - Classical ML: Classification / Regression / Clustering / Random Forest / XGBoost / Isolation Forest / Recommendation
   - Heavy ML: multi-stage pipelines, large-scale Monte Carlo, network/graph analysis (deep-learning frameworks are out of scope for the implementer)
   - Language AI over genuine free-text columns: sentiment, extraction, document processing, `ai_query` reasoning
   - Hybrid (statistical pre-filter + ML scorer, ML + LLM enrichment, etc.)

**Portfolio coverage (consider everything, force nothing):**
- Consider every structural decision pattern the schema shows; emit a UC for a pattern only when one passes the hard gates.
- Choose the technique each decision needs. Technique variety is a consequence of good fits, never a target: do NOT pick, keep or relabel a technique to raise a share or a count, and never force language-model techniques onto short structured string columns.
- Portfolio balance follows D11 in the quality gate below (single domain ≤ 25%, single technique ≤ 30%); reach it by finding equally strong UCs in under-represented areas, never by weakening, dropping or relabelling strong ones.
- Beneficiaries span the functions the schema actually touches (not a one-function portfolio when the schema covers several).

**Principle:** Data + decision dictate the algorithm. Impact drives selection. Ease is NEVER a criterion.

---

**ABSOLUTE RULES:**
1. EVERY use case MUST reference at least one table from the schema below. No schema table = REJECTED.
2. EVERY column in your High Level Design MUST exist EXACTLY in the schema. No guessing, no "makes sense" assumptions.
3. Copy table names EXACTLY in `catalog.schema.table` format. Copy column names EXACTLY as listed.
4. **COLUMN-LEVEL GROUNDING (MANDATORY)**: The `Statement` AND `Solution` fields MUST EACH name AT LEAST 2 specific columns drawn from the tables cited in `Tables Involved`, each referenced as backticked `` `column_name` `` using the EXACT column identifier as it appears in the schema. Statements or Solutions written in pure business prose without any column anchors are REJECTED. A reader must be able to see WHICH columns in WHICH tables compute every claim in the Statement and support every step in the Solution. This rule is industry-agnostic and applies to every schema, every domain, every business.
5. **NO INVENTED CAPABILITIES**: Do NOT propose UCs that require concepts, fields, entities, or signals the schema does not prove exist. Before including ANY concept in a UC (e.g., a segment, tier, score, category, status, classification, loyalty, risk, health, churn, forecast, anomaly, sentiment, priority, or any domain construct), verify that either (a) a column with that concept exists by name, OR (b) a DERIVATION from existing columns is EXPLICITLY named in the `Solution` (e.g., "derived from `<col_a>` + `<col_b>` over `<date_col>`"). If a concept is neither present nor derivable from existing columns, SKIP that UC rather than invent the capability. This applies equally to retail, finance, healthcare, manufacturing, logistics, telecom, public-sector, and every other industry; inventions are rejected regardless of domain.
6. **FULL PORTFOLIO (operational decisions included)**: Cover the COMPLETE set of impactful decisions the data structurally supports, INCLUDING operational ones an experienced analyst would consider obvious. Treat each STRUCTURAL signal as a prompt to look for a decision-changing UC (adapt the wording to the supplied business context; the pattern itself is industry-agnostic):  (i) ANY `(date/timestamp column, numeric measure column)` pair -> forecasting, change-point or capacity decisions on that measure.  (ii) ANY high-cardinality dimension referenced by a fact table -> prioritization or allocation decisions across that dimension.  (iii) ANY status-like or enum-like column with a timestamp -> SLA-breach, state-transition or exception-handling decisions.  (iv) ANY grouping id on a fact table (order id, basket id, session id, or equivalent) -> basket, funnel or cohort decisions.  (v) ANY dimension + fact FK pair -> decisions that need both tables.  Emit a UC for a signal only when it passes the hard gates (grounding, D8, D9 strip test, D10 deliverable): a plain trend chart, top-N list or dashboard is reporting, not a use case. Aim for BREADTH across every domain detected AND DEPTH within each. Quality grounded in columns matters more than novelty. This rule applies to ANY industry and ANY schema.
7. **NO INVENTED NUMBERS**: Do NOT quote dollar, percentage, multiplier or time-saving figures in any field; you see metadata only, so every such figure would be invented. Name the KPI and its direction instead, and if a magnitude is unavoidable write `[ESTIMATE: depends on <column_name from supplied schema>]` inline (see E. QUANTIFICATION DISCIPLINE).
13. **OUTPUT DISCIPLINE (NO COMMENTARY)**: Your response MUST be pure CSV, nothing else. No preamble. No afterword. No reasoning, commentary, or chain-of-thought between rows. No lines starting with `#`, `-`, `//`, or narrative phrases like "Let me...", "Wait...", "I need to...", "I will...", "Let's...", "First let me check...", "All use cases pass.", "Now let me write...", or any similar thinking-aloud markers. The first character of your response MUST be the first character of the CSV header; the last character MUST be the last character of the final CSV row. If you need to think, do it silently. Any non-CSV line in the response triggers whole-batch rejection and the UCs are discarded. This rule applies EQUALLY to Pass 1 and Pass 2 and to every industry and every schema.

---

**PRE-EMISSION SELF-REVIEW (apply silently to every use case BEFORE you emit it)**

Before including any UC in your CSV output, answer the following self-checks internally. If a check marked **REQUIRED** fails, rewrite the UC until it passes, or drop it. These checks apply to every industry and every schema; they are structural, not business-specific.

**A. PURPOSE (REQUIRED)**
- *Who funds it?* Name the budgeted role inside the business that would pay for this UC out of their own budget. If the answer is "nobody specific", the UC is decorative; drop it.
- *What decision does it change?* Name one specific operational choice this UC alters versus the status quo. "Produces a report" is NOT a decision; reframe the UC or drop it.
- *Who owns the outcome?* Name the role accountable when the UC produces a bad answer. If the answer is "nobody", the UC is not operational.

**B. FEASIBILITY (REQUIRED)**
- *Compute proof:* Using ONLY the columns you cite in `Tables Involved`, could a skilled data engineer write the SQL today and produce the Solution output? If the UC needs data not in the schema, either drop it OR tag the missing input inline as `[MISSING_DATA: <what>]`.
- *Technique–data fit:* Does the `Analytics Technique` you chose have its structural prerequisite satisfied by your cited columns (Funnel needs ordered stage columns; Cohort needs an onboarding/first-observation timestamp; Survival needs a start event AND an end-or-status column; Basket/association needs a grouping id that ties multiple rows into one transaction; Forecasting needs a date + numeric measure; Top-N needs a numeric column + a dimension to rank over)? If not, pick a different technique that IS supported.
- *Data-fit sanity:* Does the data have sufficient history, cardinality, and per-subject observation depth to actually support the claim the technique makes (a 14-day forecast needs multiple cycles of history; a customer segmentation needs enough repeat interactions per customer)? Check structurally; drop or scope down if the data is too thin.

**C. COVERAGE & PORTFOLIO DISCIPLINE (REQUIRED before the batch ends)**
- *Pattern coverage:* For every structural signal present in the supplied schema: (i) date/timestamp + numeric measure, (ii) fact + dimension FK, (iii) grouping id on a fact, (iv) status/enum-like column + date column, (v) high-cardinality subject id + fact FK; check that you EVALUATED a decision-changing UC exploiting that signal. Emit it only when it passes the hard gates; never add a UC just to cover a signal.
- *FK exploitation:* For every foreign-key edge in the schema, evaluate at least one decision that needs both tables, and emit it when the analysis requires the join and the UC passes the hard gates. A single-table-only batch on a richly-connected schema means this evaluation was skipped; go back and do it.
- *STRUCTURAL JOIN HEURISTIC (industry-agnostic):* The `foreign_key_relationships` block above lists every shared-identifier JOIN available in your schema. COUNT THEM, and aim to exercise at least `ceil(join_count / 3)` distinct joins through multi-table UCs whose analysis REQUIRES the join. A multi-table UC is one whose `Tables Involved` lists 2+ comma-separated FQNs AND whose `Solution` explicitly performs the join (e.g., `JOIN T1 ON T1.col = T2.col`). Single-table UCs that merely name a second table without joining on it DO NOT COUNT. If you fall short, look for more join-requiring decisions; never add a table to a UC only to satisfy the count (see I. Multi-table necessity).
- *Multi-table guidance when no FK lines are listed:* If the schema has >=2 tables but foreign_key_relationships says "None", infer joinable pairs structurally from column-name overlap (`*_id`, `*_code`, `*_key`, or identical identifier columns like `<subject>_id` / `<subject>ID` / `<subject>id`). Then apply the same `ceil(join_count / 3)` guidance against the inferred joins.
- *Quality over quantity:* A batch with fewer UCs that genuinely require their joins is ALWAYS preferred over a larger batch that ignores the joins. Drop single-table UCs that duplicate the decision of an existing multi-table UC in the same batch.
- *REFERENCE-dim enrichment:* If small lookup/dimension tables are available (e.g., geographic dim, category dim, calendar), at least one UC should join through them for enrichment so the portfolio exploits dimensional context, not just raw facts.
- *Type balance:* The portfolio should contain a mix of Opportunity / Risk / Improvement / Problem types. A 100%-Opportunity portfolio reads salesy; a 100%-Risk portfolio reads paranoid; aim for reasonable distribution.
- *Technique redundancy:* Follow D11 (no single technique above 30% unless the schema genuinely and structurally demands it). If one technique is over-represented, look for equally strong UCs that need other techniques; change a UC's technique only when the other technique fits its decision equally well.

**D. STAKEHOLDER PLAUSIBILITY (REQUIRED)**
- *Beneficiary and Sponsor roles* must be realistic for the business and for THIS UC's scope and dollar impact. Do NOT put C-suite roles (CFO, CEO, CTO) on every UC; do NOT put a single role on more than ~30% of UCs; diversify across the organization (Operations, Finance, Marketing, Customer Success, Procurement, Risk, HR, etc.; pick the ones that fit).
- *No invented programs / systems / committees / initiatives.* Every organizational construct referenced in Statement/Solution/Business Value must map to a column in `Tables Involved`. If there is no `<program>_tier` / `<subscription>_plan` / `<promo>_code` or similar column, do not frame the UC around a loyalty or subscription program.
- *Role MUST appear in Business Value text* (Fix-A / S4). The Beneficiary role (e.g. Pricing Manager, Operations Director, Network Analyst, Compliance Officer, Planner, Specialist, Supervisor, Coordinator; whatever fits the business) must be NAMED in the first sentence of `Business Value`. Passive voice ("enables the team", "supports the business") is REJECTED. If the UC cannot identify a specific role that would act on the output, the UC is not fundable and should not be generated.

**E. QUANTIFICATION DISCIPLINE (REQUIRED: one rule for every field)**
- Never write a concrete percentage, currency amount, multiplier or time saving in any UC field. You see metadata only, so any such figure would be invented. When a magnitude matters, write `[ESTIMATE: depends on <column_name from supplied schema>]` in its place. No exceptions.
- *Measurable impact MUST appear in Business Value* (Fix-B / S5). Every `Business Value` MUST name at least one KPI with its direction (reduce / increase / recover / prevent) that is computable from named columns in `Tables Involved`; any magnitude appears only as an `[ESTIMATE: depends on <column>]` tag. Text like "protects revenue" / "improves efficiency" / "reduces risk" without a named KPI and direction is REJECTED as immeasurable.
- *Concrete output artifact MUST appear in Solution* (Fix-C / S2). Every `Solution` MUST name a specific deliverable artifact in its final sentence: queue / ranked list / playbook / action list / alert pipeline / scorecard / briefing / intervention plan / routing rule / escalation workflow. Text like "surfaces patterns" / "enables identification" / "dashboard" without naming the action the output triggers is REJECTED as non-actionable.
- Every cost, fee, rate, comparison, or externality claim must also trace to a column. Do NOT cite industry benchmarks, competitor figures, or common-sense estimates without a column to back them.

**F. VOICE & NAMING (REQUIRED)**
- No marketing copy: drop phrases like "unlock hidden value", "drive actionable insights", "deliver measurable ROI", "comprehensive solution", "next-generation analytics", "transformative intelligence". Write in plain operational terms: name the inputs (columns), name the outputs (what the Solution produces), name the decision (what changes).
- No vague umbrella techniques: "AI Analysis", "General Analytics", "Data Science", "Advanced Analytics", "ML Analysis" are REJECTED. Pick a specific technique from the supported list.
- No generic entity names: no invented named competitors, vendor products, or specific companies unless they appear in a supplied column value.
- """ + NO_EM_DASH_RULE + """

**G. DEDUPLICATION SELF-CHECK (REQUIRED before emitting any UC)**
- Scan your CURRENT-batch and prior-batch UCs (if any were shown to you). If your new UC has identical `Tables Involved` + functionally the same business question + only the technique differs from another UC, merge into a single UC (longer Solution wins, technique named in the Solution); do NOT emit both.

**I. COHERENCE SELF-CHECK (REQUIRED, mirrors §4.11-B)**
- *Technique matches the problem:* Does the Statement's problem actually REQUIRE the chosen `Analytics Technique`, or could a simpler method (a COUNT, a GROUP BY, a ratio) answer the same question? If simpler would do, either drop the UC as too shallow OR reframe the Statement to require the heavier technique.
- *Solution produces the claimed Business Value:* Trace the Solution steps end-to-end. Does the output they produce LITERALLY deliver the outcome the Business Value promises? If the Solution computes X but Business Value claims Y, REWRITE one so they match.
- *No topic-switching:* Does the Business Value stay on the same subject as the Statement? If the Statement is about customer churn and the Business Value talks about supply chain, REWRITE.
- *Multi-table necessity:* If `Tables Involved` lists 2+ tables, is the join STRUCTURALLY REQUIRED by the analysis, or is one table cited for flavor? If flavor-only, drop the extra table OR rewrite the UC to actually use it.

**J. SELF-TEST: "would I build it this way?" (REQUIRED, mirrors §4.11-C)**
- *Same-code reproducibility:* If a skilled analyst were handed ONLY the `Solution` text (no other fields), would they write the same code the UC implies? If the Solution is ambiguous or under-specified, EXPAND it with a concrete sketch (inputs → transforms → outputs).
- *Cheaper/simpler alternative missed:* Is there a cheaper, simpler, or more robust method that answers the same business question? If yes, PICK IT instead of the heavier one.
- *Precision proportional to stakes:* Don't propose Monte Carlo or deep ML for decisions whose dollar impact is trivial; don't propose a single COUNT for decisions worth millions. Match method weight to decision weight.
- *Senior-reviewer test:* If a senior data leader read this UC cold, would they change the `Analytics Technique` or the scope? If yes, change it NOW before emitting.

**K. PORTFOLIO-SMELL SELF-CHECK (REQUIRED before the batch ends, mirrors §4.11-I)**
- *Technique redundancy:* Follow D11 (no single `Analytics Technique` above 30% unless the data genuinely demands it). If one technique is over-represented, look for equally strong UCs that need other techniques; never relabel a UC to rebalance.
- *Wording repetition:* Do NOT repeat marketing copy across UCs ("unlock hidden value", "drive actionable insights", "deliver measurable ROI", "comprehensive solution", "next-generation analytics", "transformative intelligence"). Write differentiated operational prose per UC.
- *Domain starvation:* If a business area the schema clearly supports has only 1-2 UCs, check whether you missed decisions there; add them only when they pass the hard gates.
- *Industry self-contradiction:* If the supplied business context implies industry X (e.g., healthcare provider), do not frame UCs in industry Y's vocabulary (e.g., customer cross-sell). Stay coherent with the inferred industry.

**L. BATCH DISCIPLINE**
- If a self-check fails and you cannot repair the UC within the rules above, DROP it. A smaller portfolio of strong UCs is always better than a larger portfolio padded with weak ones.
- Do NOT write an "I reviewed these and they all pass" note in the output. The self-review is internal; the output is pure CSV.
8. **NO NEAR-DUPLICATE USE CASES**: Two UCs MUST NOT target the same underlying business question on the same set of `Tables Involved` with only a technique change. If the same decision would be supported by both, MERGE them into ONE UC whose `Solution` names the best-fit technique (and optionally a secondary validation technique in the same sentence). A near-duplicate is any pair of UCs that (i) share identical `Tables Involved`, (ii) answer functionally the same business question, and (iii) differ only in which technique is cited or in cosmetic wording. Near-duplicates are REJECTED. Each UC in the final portfolio must answer a materially different business question OR exercise materially different columns. This rule applies to every industry and every schema.
9. **NO INVENTED ORGANIZATIONAL CONSTRUCTS**: Do NOT frame a UC around an organizational program, product feature, system, initiative, tier, policy, process, committee, team, partner relationship, or any other internal construct unless a column in `Tables Involved` parameterizes or identifies that construct. The test is structural: every organizational construct referenced in Statement / Solution / Business Value must map to a column in the supplied schema (whether the column IS the construct, or the column identifies / parameterizes / categorizes it). If no such column exists, the construct is invented; either (a) rephrase the UC so it does not depend on the invented construct (often the real UC is segmentation, scoring, ranking, or anomaly detection grounded in existing columns and should be framed that way), or (b) skip the UC. This rule applies in every industry and domain: retail, finance, healthcare, manufacturing, logistics, telecom, public sector, tech, and any other.
10. **NO INFERRED-EXTERNALITY CLAIMS**: Do NOT cite costs, fees, rates, prices, margins, regulations, market conditions, competitive dynamics, physical effects, health outcomes, legal constraints, widely-known industry benchmarks, or any other real-world fact that depends on knowledge OUTSIDE the schema. The test: every numeric comparison, cost claim, rate claim, externality effect, or differential statement in Statement / Solution / Business Value must either (a) be computable from a named column in `Tables Involved`, or (b) be removed. If a claim requires a fact not in the schema, whether from industry knowledge, common sense, textbook benchmarks, or published statistics, rephrase to stay within what the schema measures, or drop the claim. Inferred-externality framing is REJECTED regardless of which industry or domain the business operates in.
11. **TECHNIQUE–DATA FIT (structural proof required)**: The technique chosen in `Analytics Technique` MUST be structurally supported by the columns of `Tables Involved`. Before selecting a technique, verify the structural prerequisites exist:
   - *Funnel / pathing*: REQUIRES ordered event stages present either as distinct columns (stage_1_ts, stage_2_ts, ...) OR as a stage/state column with multiple ordered values per subject. If the data is only realized outcomes (every row is already a purchase), Funnel is NOT supported.
   - *Survival / time-to-event*: REQUIRES a start event timestamp AND either an end event or an explicit status/event-observed column.
   - *Cohort analysis*: REQUIRES a subject identifier AND a cohort-forming timestamp column (the column that captures when the subject first appeared, was onboarded, was created, or was otherwise first observed; a minimum or "first" aggregation over a timestamp column on the subject also qualifies).
   - *Association / market basket*: REQUIRES a transaction/basket/session grouping column that ties multiple rows into one event.
   - *Forecasting / time-series*: REQUIRES a date/timestamp column AND a numeric measure column.
   - *Top-N / ranking*: REQUIRES a numeric column to rank by AND a dimension column to rank over.
   If the structural prerequisites are missing, choose a different technique that IS supported, or skip the UC. A UC whose technique is structurally unsupported by its cited columns is REJECTED.
12. **NO VAGUE / UMBRELLA TECHNIQUE NAMES**: The `Analytics Technique` field MUST be exactly one value from the allowed list in the `Analytics Technique` column instructions below. Catch-all labels like "AI Analysis", "ML Analysis", "General Analytics", "Data Science", "AI", "Statistical Analysis", "Advanced Analytics" are REJECTED. Specific algorithms (XGBoost, Random Forest, Isolation Forest, K-Means, Kaplan-Meier, Prophet, ALS, Monte Carlo, association rules) are NOT technique labels: name them in the `Solution` and `High Level Design` and choose the matching allowed label (e.g., Isolation Forest → `Anomaly Detection`; K-Means → `Clustering` or `Segmentation`; Kaplan-Meier → `Survival Analysis`; Monte Carlo → `Simulation`; association rules → `Recommendation`). If a UC uses AI primitives on genuine free-text columns (e.g., `ai_query` on comment/description/review fields), use `Sentiment Analysis`, `Extraction` or `Document Processing`, or `Classification` / `Clustering` when the text feeds a classifier or topic clustering; NEVER the generic "AI Analysis". If no allowed technique fits the UC, skip the UC rather than mislabel it.
4. If a use case requires data not in the schema, SKIP it entirely.

**High Level Design column requirement:** List EXACT columns: "Using columns: [col1], [col2] from table. Approach: [describe analytical steps]"

**Hallucination examples (REJECTED):** "Calculate tax" with no tax_amount column; "Analyze age" with no birth_date column.
**Valid examples:** schema has invoice_date, due_date, payment_date → "Analyze payment delays using date differences"

---

### 1. CORE TASK

Your single, primary task is to produce a **single CSV** response.
The CSV MUST have EXACTLY the following 12 data columns, followed by the 2 honesty columns `"honesty_score","honesty_justification"` defined at the end of this prompt (14 columns per row):
`"No","Name","usecase","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Tables Involved","High Level Design"`

🚨 **COLUMN NAME RULES** 🚨:
- Column 6 MUST be called `"Statement"`; do NOT rename it to "Opportunity", "Problem Statement", or anything else.
- `"Analytics Technique"` is a SINGLE column. Values like "Forecasting", "Classification", "Anomaly Detection" are ROW VALUES inside this column; do NOT add them as separate column headers.
- `"usecase"` MUST be present in the header exactly as written.
- Do NOT add any extra columns beyond these 12 data columns and the 2 honesty columns (columns 13-14).

---

{quality_gate_block}

---

### 2. USE CASE GENERATION RULES

You must follow these rules to generate the content for the use cases. All text output must be in **English**.

**🚨🚨🚨 STRATEGIC BUSINESS IMPACT REQUIREMENTS 🚨🚨🚨**

You MUST ONLY generate use cases that meet at least one of these criteria:
1.  **DIRECT FINANCIAL IMPACT**: The decision directly moves revenue (new or protected), cost, or loss.
2.  **BUSINESS MODEL FIT**: The decision moves a line of the Revenue Model, a Value Chain step or the Strategic Initiative stated above.
3.  **PRODUCTIVITY & AUTOMATION**: Increases team productivity by automating manual work.

**❌ STRICTLY PROHIBITED**:
-   **NO MASTER/REFERENCE-CENTERED USE CASES**: EVERY use case MUST derive its analytical question from TRANSACTIONAL events: things that HAPPENED over time. You may JOIN master and reference tables for enrichment, but the CORE BUSINESS QUESTION must be about patterns in EVENTS (transactions, movements, interactions, measurements), NOT about static entity attributes. Ask: "Does this use case analyze WHAT HAPPENED or WHAT EXISTS?" If it only describes entity attributes without analyzing event patterns → REJECT.
-   **NO MARGINAL BENEFIT**: Do not generate trivial use cases.
-   **NO "NICE TO HAVE"**: Focus on "MUST HAVE" decisions with DIRECT IMPACT on revenue, cost, risk or productivity.
-   **NO IT/TECHNICAL MAINTENANCE**: Ignore tables that are purely for IT/system maintenance unless they impact business operations directly.
-   **NO SEMANTIC DUPLICATES**: Do not generate two use cases that address the same core business problem with different wording, different verbs, or different "with [activation]" suffixes.
-   **NO TRIVIAL AI WRAPPERS**: Do not generate use cases where the core analysis is a simple date comparison, threshold check, or basic aggregation, even if you wrap it in AI function calls.
-   **NO TECHNICAL ACTIVATION**: The "with [activation]" part must describe a SPECIFIC business DELIVERABLE (recommendation, action plan, priority queue, strategy, intervention), NEVER a technical method, data source, or analytical technique. Ban: "with Correlation Analysis", "with Z-Score Deviation", "with RFM Scoring", "with Cross-Reference".
-   **TECHNIQUE FIT FIRST (EVERY ALLOWED TECHNIQUE CONSIDERED)**: For each business problem, consider every technique in the allowed `Analytics Technique` list below and choose the one the decision and the data structurally need. Do NOT default to only a handful out of habit, and do NOT pick a technique to raise variety; a technique without its structural prerequisites is a REJECTED fit. Do NOT avoid a valid use case just because its technique has been used before.
-   **DOMAIN DIVERSITY**: Spread use cases across business domains. If one domain has many use cases, also look for opportunities in other domains, but do NOT skip valid use cases just because a domain already has several.

**COLUMN INSTRUCTIONS:**

  - **`No`**: Sequential number (e.g., 1, 2, 3...).
  
  - **`Name`**: A short, clear name that emphasizes BUSINESS VALUE and ACTIONABLE OUTCOMES, not technical implementation.
    *   **🚨🚨🚨 MANDATORY "ACTION + ACTIVATION" PATTERN 🚨🚨🚨**: EVERY use case name MUST follow this two-part pattern:
        **"[Action Part] with [Activation Part]"**
        - **ACTION PART** (before "with"): A business-oriented verb + the business problem or opportunity being surfaced. This conveys URGENCY and tells the user WHAT needs attention. Maximum 6-8 words.
        - **ACTIVATION PART** (after "with"): The DELIVERABLE, what the use case produces as an actionable output for the business user. This is the resolution, recommendation, or decision-support artifact. Maximum 3-5 words.
    *   **Use business-oriented ACTION verbs**: Anticipate, Predict, Detect, Identify, Reveal, Classify, Segment, Forecast, Simulate, Extract.
    *   **🚨 CRITICAL RULES FOR THE ACTIVATION PART (after "with") 🚨**:
        - The "with" suffix MUST describe what the USER GETS: a recommendation, action plan, priority queue, intervention, strategy, or decision tool.
        - The "with" suffix must NEVER describe a technical method, statistical technique, data source, column name, or analytical approach.
        - **BOARD ROOM TEST**: If a C-suite executive cannot read the full name in 3 seconds and immediately understand both the problem AND the deliverable, the name FAILS.
    *   **🚨 CRITICAL RULES FOR THE ACTION PART (before "with") 🚨**:
        - ZERO technical jargon: No industry-specific acronyms (e.g., RASK, PNR, AWB, MEL for airlines; SKU, BOM, OEE for manufacturing; LTV, CAC, MRR for SaaS; NPA, NIM for banking; RFM), no statistical terms (Z-score, correlation, STDDEV, regression, variance, coefficient, decomposition), no implementation language (rolling, partitioned, cross-reference).
        - Only use acronyms that a CEO in ANY industry would understand (ROI, SLA, KPI).
        - Name the BUSINESS PROBLEM, not the DATA OBJECT. "Revenue Underperformance" not "RASK Deviation". "Payment Fraud" not "Transaction Anomaly Patterns". "Inventory Shortage" not "SKU Stockout Variance".
    *   **✅ CORRECT Examples (Action = business problem, Activation = deliverable; CROSS-INDUSTRY)**:
        - "Predict Customer Churn Risk with Prioritized Retention Outreach Queue" (Retail/Telecom, ML: XGBoost churn model)
        - "Detect Revenue Underperformance on High-Demand Products with Pricing Recalibration Recommendations" (Retail/Manufacturing)
        - "Classify Loan Default Probability with Early Intervention Actions" (Banking, ML: Gradient Boosting classifier)
        - "Forecast Demand by Product Category with Inventory Rebalancing Strategy" (Supply Chain, ML: Prophet + external factors)
        - "Segment Patients by Readmission Risk with Care Coordination Plans" (Healthcare, ML: Survival analysis)
        - "Predict Equipment Failure Probability with Pre-Emptive Maintenance Scheduling" (Manufacturing/Energy, ML: Random Forest)
        - "Detect Fraudulent Transactions with Real-Time Hold Recommendations" (Banking/Insurance, ML: Isolation Forest)
        - "Reveal Unauthorized Discount Erosion with Audit Escalation Brief" (Any industry)
        - "Simulate Pricing Scenario Impact with Revenue Optimization Recommendations" (Any industry, Simulation + ML)
        - "Classify Supplier Reliability with Procurement Diversification Strategy" (Manufacturing/Retail, ML: clustering + classification)
    *   **❌ REJECTED Examples (activation describes technical METHOD instead of business DELIVERABLE)**:
        - "Detect Revenue Anomalies with Z-Score Deviation and Correlation Breakdown" → REJECTED (activation = technical method)
        - "Predict Churn with Purchase Recency and Contract Tenure Signals" → REJECTED (activation = data signals, not deliverable)
        - "Forecast Demand with Seasonal Decomposition and Holiday Calendar" → REJECTED (activation = analytical technique)
        - "Classify Risk Tiers with Payment History Cross-Reference" → REJECTED (activation = data source, not deliverable)
        - "Segment Buyers with RFM Scoring and Cohort Aging" → REJECTED (activation = statistical technique)
    *   **❌ REJECTED Examples (generic/hollow activation: BANNED)**:
        - "Detect Fraud with Pattern Recognition" → REJECTED ("Pattern Recognition" is a method, not a deliverable)
        - "Identify Issues with Intelligence" → REJECTED ("Intelligence" is meaningless filler)
        - "Classify Risk with Risk Scoring" → REJECTED (circular: activation restates the action)
        - "Forecast Revenue with Trend Analysis" → REJECTED (activation = technique, not deliverable)
    *   **❌ REJECTED Examples (action contains technical jargon: BANNED)**:
        - "Detect SKU-Level OEE Degradation with Maintenance Recommendations" → REJECTED (action contains "SKU", "OEE": manufacturing jargon)
        - "Identify NPA Portfolio Concentration with Risk Mitigation Queue" → REJECTED (action contains "NPA": banking acronym)
        - "Forecast MRR Churn Cohort Variance with Retention Actions" → REJECTED (action contains "MRR": SaaS jargon)
        - "Predict NPS Score Degradation with Outreach Strategy" → REJECTED (action contains "NPS": domain acronym)
        - "Classify CDR Anomaly Patterns with Investigation Queue" → REJECTED (action contains "CDR": telecom jargon)
    *   **❌ REJECTED Examples (missing "with" entirely)**:
        - "Detect Invoice Fraud Patterns Using Anomaly Detection" → REJECTED (uses "Using" instead of "with")
        - "Identify High-Value Overdue Invoices for Executive Escalation" → REJECTED (uses "for" instead of "with")
        - "Correlation Analysis Between Payment Terms and Overdue Risk" → REJECTED (no "with" at all)
    *   **ACTIVATION QUALITY TEST**: The "with [...]" phrase must answer: "What actionable DELIVERABLE does the business user receive from this use case?" If the suffix describes HOW the analysis works rather than WHAT the user gets to DO, it is TECHNICAL and REJECTED.
    *   **YOUR RESPONSE WILL BE REJECTED if ANY use case name has a technical method as the activation suffix, or if the action part contains domain-specific jargon or statistical terminology.**
    *   **READABILITY TEST**: Keep `Name` easy to comprehend for non-technical business users. Use plain business language, avoid overloaded wording, and cap the full `Name` at 14 words.

  - **`usecase`**: Short, benefit-led phrase (5–10 words) that answers TWO questions: "What are we doing?" AND "Why does it matter to the business?"
    *   **🚨🚨🚨 MANDATORY PATTERN: [Verb] [Business Problem/Area] to [Business Outcome] 🚨🚨🚨**
        Every `usecase` MUST contain **"to"** connecting the ACTION to the BUSINESS WHY.
        - **BEFORE "to"**: What business lever we are pulling (the action). Start with an action verb.
        - **AFTER "to"**: The tangible business outcome: WHY a CEO would care (revenue, cost, risk, growth, margin, retention, compliance, efficiency).
    *   **ACTION VERBS** (start with one): Reduce, Recover, Improve, Stabilize, Minimize, Protect, Cut, Fix, Increase, Accelerate, Prevent, Optimize, Detect, Forecast, Identify
    *   **BUSINESS OUTCOMES** (end with one): Protect Revenue, Grow Revenue, Reduce Cost, Lower Risk, Improve Margins, Increase Retention, Prevent Losses, Accelerate Growth, Ensure Compliance, Improve Efficiency
    *   **🚨 TITLE CASE: MANDATORY 🚨**: EVERY word in the `usecase` MUST be Title Case (capitalize the first letter of every word). Lowercase usecase values are AUTOMATICALLY REJECTED.
        ❌ "detect transaction anomalies to prevent revenue fraud" → REJECTED (lowercase)
        ✅ "Detect Transaction Anomalies to Prevent Revenue Fraud" → CORRECT (Title Case)
    *   **🚨 VERB DIVERSITY: MANDATORY 🚨**: Do NOT reuse the same starting verb across multiple use cases when a more precise synonym exists. The verb list above is a STARTING POINT; you MUST expand it by discovering the verb that BEST fits each specific use case's intent. Different business actions demand different verbs. If more than 30% of use cases in a batch start with the SAME verb, you MUST rewrite using more precise alternatives that better convey the specific nature of each use case.
    *   **🚨 VOCABULARY SPECIFICITY: MANDATORY 🚨**: Always choose the MOST SPECIFIC word that describes the actual business threat, action, or problem; never fall back on a generic catch-all term when a more precise word exists. The name should make a C-suite executive immediately understand the NATURE of the problem, not just that "a problem exists." Ask: "Does this word tell the reader WHAT SPECIFICALLY is happening, or is it a vague umbrella term?" If vague → find the precise word.
    *   **WORD ORDER TEST**: Read the `usecase` aloud. If it does NOT make grammatical sense as a complete English sentence fragment, it FAILS. No garbled noun-first constructions.
    *   ✅ GOOD (notice: the usecase NEVER echoes the Analytics Technique; it focuses on BUSINESS action and BUSINESS outcome):
        - "Reduce Customer Attrition to Protect Revenue" (even if technique = "Predictive Modeling", usecase says NOTHING about prediction/modeling)
        - "Increase Basket Size to Grow Revenue" (even if technique = "Recommendation", usecase says NOTHING about recommendation)
        - "Prevent Payment Fraud to Protect Revenue" (even if technique = "Anomaly Detection", usecase says NOTHING about detection/anomaly)
        - "Improve Demand Planning to Reduce Stockouts" (even if technique = "Forecasting", usecase says NOTHING about forecasting/forecast)
        - "Cut Inventory Waste to Lower Carrying Cost" (even if technique = "Optimization", usecase says NOTHING about optimization)
        - "Recover Missed Revenue from Pricing Gaps"
        - "Improve Supplier Reliability to Prevent Stockouts"
        - "Stabilize Revenue in Volatile Demand Periods"
        - "Protect Margins through Dynamic Discount Controls"
        - "Reduce Default Losses to Improve Portfolio Health"
        - "Accelerate Order Fulfillment to Improve Customer Satisfaction"
    *   ❌ BAD (REJECTED: missing the "to [WHY]"):
        - "Reduce Customer Wait Time" → REJECTED (no business WHY: reduce wait time TO WHAT END?)
        - "Protect Revenue from Payment Fraud" → REJECTED (starts with outcome instead of action; flip it: "Detect Payment Fraud to Protect Revenue")
        - "Recover Lost Revenue from Dormant Customers" → REJECTED (no explicit "to" connecting action to outcome)
    *   ❌ BAD (REJECTED: garbled word order):
        - "Demand to Optimize Stock Levels Forecast" → REJECTED (nonsensical word order; correct: "Forecast Demand to Optimize Stock Levels")
        - "Churn Pattern Identification" → REJECTED (noun phrase, no verb, no outcome)
        - "Inventory Inefficiency Detection" → REJECTED (trailing filler, no benefit)
    *   🚨🚨🚨 **ANALYTICS TECHNIQUE CONTAMINATION: THE #1 FAILURE MODE** 🚨🚨🚨
        You WILL be tempted to take the value you plan to write in the `Analytics Technique` column (e.g., "Forecasting", "Optimization", "Classification", "Anomaly Detection", "Simulation")
        and **append its verb/noun form as the LAST WORD of the `usecase`**. This produces GIBBERISH. Detect and eliminate this.
        **HOW THIS BUG HAPPENS**: You generate the `usecase` and `Analytics Technique` columns near-simultaneously.
        The technique word leaks BACKWARD into the usecase as a trailing noun/verb:
        - Analytics Technique = "Forecasting"  → usecase ends with "...Forecast"     → GIBBERISH
        - Analytics Technique = "Optimization"  → usecase ends with "...Optimization" → GIBBERISH
        - Analytics Technique = "Classification" → usecase ends with "...Classification" → GIBBERISH
        - Analytics Technique = "Anomaly Detection" → usecase ends with "...Detection" → GIBBERISH
        - Analytics Technique = "Simulation"    → usecase ends with "...Simulation"   → GIBBERISH
        - Analytics Technique = "Predictive Modeling" → usecase ends with "...Prediction" or "...Modeling" → GIBBERISH
        - Analytics Technique = "Clustering"    → usecase ends with "...Clustering"   → GIBBERISH
        **CONCRETE EXAMPLES OF THIS BUG (ALL REJECTED):**
        - ❌ "Demand to Reduce Inventory Cost Forecast" (technique "Forecasting" leaked as trailing "Forecast")
          ✅ CORRECT: "Forecast Demand to Reduce Inventory Cost" (verb FIRST, business outcome LAST)
        - ❌ "Staffing to Reduce Labor Cost Optimization" (technique "Optimization" leaked as trailing "Optimization")
          ✅ CORRECT: "Optimize Staffing to Reduce Labor Cost" (verb FIRST, business outcome LAST)
        - ❌ "Customer Segment Profitability Classification" (technique "Classification" leaked as trailing "Classification")
          ✅ CORRECT: "Classify Customer Segments to Improve Profitability" (verb FIRST, business outcome LAST)
        - ❌ "Transaction Pattern Anomaly Detection" (technique "Anomaly Detection" leaked as trailing "Detection")
          ✅ CORRECT: "Detect Transaction Anomalies to Prevent Fraud" (verb FIRST, business outcome LAST)
        - ❌ "Revenue Scenario Impact Simulation" (technique "Simulation" leaked as trailing "Simulation")
          ✅ CORRECT: "Simulate Revenue Scenarios to Reduce Planning Risk" (verb FIRST, business outcome LAST)
        **THE RULE**: The LAST WORD(S) of `usecase` must ALWAYS be a **BUSINESS OUTCOME** (Revenue, Cost, Risk, Growth, Margin, Losses, Retention, Efficiency, Compliance, Satisfaction, Health, Stockouts, Waste).
        The last word must NEVER be a technique-derived word (Forecast, Optimization, Classification, Detection, Simulation, Modeling, Clustering, Analysis, Prediction, Identification, Segmentation, Recommendation).
    *   **SELF-CHECK** (do this for EVERY `usecase` before outputting):
        (1) Does it start with a VERB? If NO → rewrite.
        (2) Does it contain "to" connecting action to outcome? If NO → rewrite.
        (3) Read the LAST WORD. Is it derived from the `Analytics Technique` value you are about to write in column 5? If YES → you have the contamination bug. DELETE that trailing word and restructure: move the technique verb to position 1, put the business outcome at the end.
        (4) Read aloud. Does it make grammatical sense as natural English? If NO → rewrite.
        (5) Can a non-technical executive immediately understand WHAT we do and WHY? If NO → rewrite.
        (6) Is EVERY word Title Case (first letter capitalized)? If NO → capitalize. Lowercase usecase values are REJECTED.
        (7) Look at ALL use cases generated so far. Does the SAME starting verb appear on more than 30% of them? If YES → find more precise verbs that better convey the distinct nature of each use case and rewrite.
        (8) Is any key noun in the usecase a vague umbrella term where a more specific word would tell the reader WHAT EXACTLY is happening? If YES → replace with the most precise word that describes the actual business threat or action.

  - **`type`**: One of "Problem", "Risk", "Opportunity", "Improvement".

  - **`Analytics Technique`**: The PRIMARY analytics technique used. ⚠️ THIS VALUE MUST STAY IN THIS COLUMN ONLY; do NOT let it leak into the `usecase` column in any word form (verb, noun, gerund). MUST be ONE of these values:
    * `Forecasting` - Time-series prediction and demand planning (Prophet, ARIMA, ai_forecast, or trained regression models)
    * `Classification` - Categorizing data into predefined business groups (XGBoost, Random Forest, Logistic Regression, ai_classify, or rule-based)
    * `Anomaly Detection` - Identifying outliers, deviations, unusual patterns (Isolation Forest, DBSCAN, statistical methods, or hybrid)
    * `Predictive Modeling` - Training ML models (XGBoost, LightGBM, Random Forest, Gradient Boosting) to predict business outcomes (churn, default, demand, failure)
    * `Clustering` - Unsupervised grouping using ML algorithms (K-Means, DBSCAN, Hierarchical) to discover natural segments in data
    * `Regression Analysis` - Modeling relationships between variables to quantify drivers and predict continuous outcomes (Linear, Ridge, Lasso, GLM)
    * `Optimization` - Resource allocation, scheduling, pricing, or constraint satisfaction to maximize/minimize business objectives
    * `Cohort Analysis` - Grouping entities by shared characteristics over time
    * `Segmentation` - Clustering customers/products into distinct groups using ML or rule-based approaches
    * `Sentiment Analysis` - Analyzing text for emotional tone
    * `Trend Analysis` - Identifying patterns over time (growth, decline)
    * `Correlation Analysis` - Finding relationships between variables
    * `Pareto Analysis` - 80/20 rule, identifying top contributors
    * `Funnel Analysis` - Conversion tracking through stages
    * `Document Processing` - Extracting structured data from documents
    * `Extraction` - Extracting specific entities from text
    * `Simulation` - What-If analysis, scenario modeling, sensitivity analysis, Monte Carlo simulation
    * `Geospatial Analysis` - Location-based pattern analysis and mapping
    * `Survival Analysis` - Time-to-event modeling (customer lifetime, equipment failure, contract renewal probability)
    * `Recommendation` - Collaborative filtering or content-based recommendation engines for products, actions, or next-best-offers
    * `Network Analysis` - Graph-based analysis of entity relationships, influence propagation, community detection

  - **`Statement`**: 1-2 sentences describing the business challenge or problem being addressed. Focus on IMPACT (Revenue, Cost, Risk). This column MUST be named "Statement" in the CSV header.

  - **`Solution`**: 1-2 sentences high-level business solution. **MUST explicitly highlight "Databricks Genie Code"**.

  - **`Business Value`**: **CRITICAL**. Articulate the tangible business impact (Revenue, Cost, Risk, Efficiency).
    *   **Focus on WHY this matters**: name the beneficiary role first, then the decision it changes and at least one KPI with its direction (reduce / increase / recover / prevent), computable from named columns.
    *   **IMPORTANT CONSTRAINT (same rule as E. QUANTIFICATION DISCIPLINE)**: Never commit to a concrete number (no "10% more revenue", no "reduce cost by 20%"). If a magnitude matters, write `[ESTIMATE: depends on <column_name>]`.
    *   **GOOD**: "The `<role>` re-prioritizes `<entity>` follow-ups every week, reducing the count of breaches recorded in `<status_col>` and the cost carried in `<cost_col>` [ESTIMATE: depends on `<cost_col>`]."
    *   **BAD**: "Optimizes performance..." (too generic, no KPI) and "Cuts cost by 25%" (invented figure).

  - **`Beneficiary`**: The primary person/role (e.g., "Loan Officer").
  - **`Sponsor`**: The main executive (e.g., "CRO").

  - **`Tables Involved`**: Comma-separated **FULLY-QUALIFIED** table names (`catalog.schema.table`). MUST exist in schema.
    *   **CRITICAL**: Use the EXACT THREE-LEVEL FORMAT shown in the schema.
    *   **🔗 MULTI-TABLE PRIORITY**: ALWAYS look for opportunities to JOIN related tables! Use the FK relationships section to identify joinable tables.
    *   **PREFER MULTIPLE TABLES**: Single-table use cases are often weak. Cross-table analysis yields higher business value.

  - **`High Level Design`**: A high-level analytical approach (2-4 sentences) describing the data analysis strategy.
    *   **MANDATORY COLUMN LISTING**: List EXACT column names from the schema: "Using columns: [col1], [col2], [col3]"
    *   **DESCRIBE THE ANALYTICAL APPROACH AND ALGORITHM CHOICE (TECHNIQUE-APPROPRIATE -- NO SQL BIAS)**: Explain what data to extract, how to join tables, what features to engineer, and WHICH ALGORITHM or TECHNIQUE to apply. The algorithm MUST match the Analytics Technique: (1) Anomaly Detection MUST use Isolation Forest, LOF, DBSCAN, or ensemble ML methods -- NOT just z-scores or SQL thresholds; (2) Forecasting MUST use Prophet, ARIMA, ai_forecast, or temporal ML models -- NOT just SQL moving averages; (3) Classification MUST use XGBoost, Random Forest, Logistic Regression, or ai_classify -- NOT just SQL CASE WHEN rules; (4) Predictive Modeling MUST use trained ML models (XGBoost, LightGBM, Random Forest) -- NOT just SQL aggregations; (5) Clustering/Segmentation MUST use K-Means, DBSCAN, GMM -- NOT just SQL NTILE or CASE WHEN tiers; (6) Regression Analysis MUST use Linear/Ridge/Lasso/GLM regression -- NOT just SQL CORR(); (7) Survival Analysis MUST use Kaplan-Meier, Cox PH -- NOT just SQL DATEDIFF; (8) Simulation MUST use Monte Carlo or scenario modeling -- NOT just SQL CASE WHEN with 2-3 scenarios; (9) Recommendation MUST use ALS or similarity algorithms -- NOT just SQL top-N by COUNT; (10) Network Analysis MUST use GraphFrames, PageRank, centrality -- NOT just SQL self-joins. For ALL techniques, specify the model type explicitly (e.g., "Train an Isolation Forest on features [col1, col2, col3] to detect anomalies", "Train an XGBoost classifier to predict [outcome]"). SQL-only approaches are ONLY acceptable for Pareto Analysis, basic Funnel Analysis, and Cohort Analysis construction. Choose the BEST approach for the problem -- do NOT default to SQL when a trained ML model or statistical model would yield better, more defensible results.
    *   **JOIN PRIORITY**: CHECK FK Relationships section. If related tables exist, describe JOINs. Multi-table > single-table for business value.
    *   **EXTERNAL DATA (NEVER SIMULATED)**: Do not plan to simulate, fabricate or assume external data. If an external source would materially strengthen the analysis, name it once as `[MISSING_DATA: <source>]`; the design must still work on the supplied columns alone.
    *   **AI REASONING AS FINAL EXPLANATION LAYER (MANDATORY for ALL use cases)**: Every use case MUST conclude with ai_query as the FINAL step to EXPLAIN pre-computed analytical results, diagnose root causes, and generate actionable recommendations. ai_query does NOT perform the core analysis -- the ML model, statistical method, or AI function performs the analysis first, and ai_query receives those results (scores, predictions, clusters, anomaly flags) to generate business narratives. This separation is NON-NEGOTIABLE: ML/statistical analysis first, ai_query explanation last.
    *   Only reference columns that EXIST in the schema. No assumptions.

**FOCUS AREAS:**
{focus_areas_instruction}

**GENERATION PRINCIPLES:**
  - **BREADTH WITH HARD GATES**: Generate ALL valid use cases the data can support. The hard gates (grounding, D8 duplicates, D9 strip test, D10 deliverable, and the production-readiness checklist) apply now; value and quality tiers are assigned downstream on an absolute scale. Never withhold a use case that passes the hard gates, and never emit one that fails them.
  - **ANTI-HALLUCINATION**: Every use case MUST reference real columns from the schema. If the data doesn't contain the required columns, DO NOT GENERATE IT.
  - **COVERAGE**: Revenue impact, Cost reduction, Risk mitigation, Operational efficiency; cover ALL applicable areas.
  - **NO TRIVIAL USE CASES**: Skip basic reporting, simple aggregations, or use cases any analyst could do with basic filtering and grouping.
  - **NO REDUNDANT EXTRACTION**: Do not use AI to extract data that already exists in structured columns.

**🔥 ANALYTICAL APPROACH FAMILIES (CONSIDER ALL; SHARES ARE AN OUTCOME, NEVER A TARGET) 🔥**:
Consider every family below for each business problem and choose the BEST algorithm for that problem; do NOT default to SQL-only approaches when a trained ML model would be more accurate or a hybrid approach would be superior, and do NOT pick a family to fill a share.

**CANONICAL TECHNIQUE → FAMILY MAPPING** (every allowed `Analytics Technique` belongs to exactly one family):
A. ML & PREDICTIVE: `Predictive Modeling`, `Classification`, `Clustering`, `Regression Analysis`, `Segmentation`, `Anomaly Detection`, `Survival Analysis`, `Recommendation`, `Forecasting`
B. LANGUAGE & DOCUMENT AI (genuine free-text columns only): `Sentiment Analysis`, `Document Processing`, `Extraction`
C. STATISTICAL: `Correlation Analysis`, `Trend Analysis`, `Pareto Analysis`, `Cohort Analysis`, `Funnel Analysis`
D. SIMULATION & ADVANCED: `Simulation`, `Optimization`, `Geospatial Analysis`, `Network Analysis`

A. **MACHINE LEARNING & PREDICTIVE MODELS**. Techniques: `Predictive Modeling`, `Classification`, `Clustering`, `Regression Analysis`, `Segmentation`, `Anomaly Detection`, `Survival Analysis`, `Recommendation`, `Forecasting`:
   - Train XGBoost, LightGBM, or Random Forest models to predict business outcomes (churn, default risk, demand, equipment failure, fraud) → `Predictive Modeling`
   - K-Means or DBSCAN clustering to discover natural customer/product/entity segments → `Clustering` or `Segmentation`
   - Logistic/Linear/Ridge/Lasso regression to quantify feature importance and business drivers → `Regression Analysis`
   - Cox proportional hazards, Kaplan-Meier for time-to-event problems (customer lifetime, contract renewal, asset failure) → `Survival Analysis`
   - Collaborative filtering (ALS) or content-based recommendation engines for products, actions, next-best-offers → `Recommendation`
   - Isolation Forest or ensemble anomaly detection trained on historical patterns → `Anomaly Detection`
   - XGBoost, Random Forest, Logistic Regression for categorizing entities into predefined business groups → `Classification`
   - Time-series forecasting using ai_forecast or Prophet with strategic recommendations → `Forecasting`
   - ALL ML use cases MUST end with ai_query to explain model predictions, generate recommendations, and provide business narratives

B. **LANGUAGE & DOCUMENT AI**. Techniques: `Sentiment Analysis`, `Document Processing`, `Extraction` (only over genuine free-text columns):
   - Analyzing text for emotional tone in feedback, reviews, comments → `Sentiment Analysis`
   - Extracting structured data from unstructured text documents → `Document Processing`
   - Extracting specific entities (names, dates, amounts, codes) from text → `Extraction`

C. **STATISTICAL & ANALYTICAL**. Techniques: `Correlation Analysis`, `Trend Analysis`, `Pareto Analysis`, `Cohort Analysis`, `Funnel Analysis`:
   - Finding relationships between business variables → `Correlation Analysis`
   - Identifying patterns over time (growth, decline, seasonality) → `Trend Analysis`
   - 80/20 rule, identifying top contributors to business outcomes → `Pareto Analysis`
   - Grouping entities by shared characteristics and tracking over time → `Cohort Analysis`
   - Conversion tracking through business stages (lead→customer, order→delivery) → `Funnel Analysis`

D. **SIMULATION & ADVANCED ANALYTICS**. Techniques: `Simulation`, `Optimization`, `Geospatial Analysis`, `Network Analysis`:
   - What-If analysis, scenario modeling, Monte Carlo simulation → `Simulation`
   - Resource allocation, scheduling, pricing, constraint satisfaction → `Optimization`
   - Location-based pattern analysis and spatial mapping → `Geospatial Analysis`
   - Graph-based analysis of entity relationships, influence propagation, community detection → `Network Analysis`

**🚨🚨🚨 ANTI-PATTERN RULES (MANDATORY: violations cause AUTOMATIC REJECTION) 🚨🚨🚨**

**AP1 - NO SEMANTIC DUPLICATES (5-LAYER DETECTION, ZERO TOLERANCE)**: Before generating each use case, apply all 5 layers of D8 exactly as defined in the quality gate above. A match on ANY layer = DUPLICATE = DO NOT GENERATE. Different QUESTIONS on the same entity and metric are DISTINCT use cases: keep BOTH; do NOT collapse technique diversity by marking genuinely different business questions as duplicates.

**AP2 - NO TRIVIAL WRAPPERS**: Do NOT dress up simple operations as advanced analytics. These are AUTOMATICALLY REJECTED:
  - Date expiry/deadline checks (e.g., "Detect Expiry Risk" = just a date comparison)
  - Simple threshold violations (e.g., "Detect Overdue X" = just a basic filter)
  - Basic count/sum aggregations dressed up as AI
  - One-dimensional anomaly detection that is just a basic statistical measure on a single column
  **TEST**: If the core business insight requires no advanced analytical technique, it is TRIVIAL and INVALID.

**AP3 - ACTIVATION MUST BE A BUSINESS DELIVERABLE**: The "with [activation]" part MUST pass the D10 DELIVERABLE TEST in the quality gate above and the `Name` rules: it describes what the business user RECEIVES (e.g., "with Prioritized Intervention Queue", "with Audit Escalation Brief"), never a technical method, data source, generic filler or jargon.

**AP4 - DOMAIN AND TECHNIQUE BALANCE (D11)**:
  - Follow the D11 targets in the quality gate above (single domain ≤ 25%, single technique ≤ 30%)
  - After completing the systematic enumeration, check each over-represented area and actively look for equally strong use cases in under-represented domains and techniques
  - Never skip, weaken or relabel a valid use case, and never add a weak one, to meet a balance target

**AP5 - NO SUBDOMAIN DRIFT**: Each subdomain MUST belong to exactly ONE parent domain. A subdomain cannot appear under multiple parent domains. Pick the MOST natural parent domain and be consistent.

**AP6 - NO IDENTICAL USE CASES**: Two use cases with the EXACT same name are ABSOLUTELY FORBIDDEN. This indicates a generation loop. If you detect this, immediately stop and reassess.

**🚨 BUSINESS REALISM GATE (every use case must pass) 🚨**
Before generating, verify: (1) DIRECT, PROVABLE cause-and-effect between variables, (2) Industry-recognized analysis, (3) A 20-year veteran and a skeptical CFO would both approve. If ANY test fails, DO NOT generate. Do NOT correlate unrelated variables, invent relationships from temporal patterns, or add external data without clear cause-and-effect.

---

### 3. ANALYTICAL CAPABILITIES & SCHEMA

#### CONTEXT: AVAILABLE ANALYTICAL CAPABILITIES

Genie Code has access to the full Databricks platform including:
- **AI Capabilities**: Forecasting, classification, sentiment analysis, text extraction, entity recognition, document parsing, semantic similarity, text generation, translation, summarization, and custom AI model invocation
- **Statistical Techniques**: Central tendency, dispersion analysis, distribution shape, percentile analysis, trend detection, correlation analysis, volatility measurement, outlier detection, ranking, and time series analysis
- **Advanced Analytics**: What-if analysis, scenario simulation, sensitivity analysis, geospatial analysis, market basket analysis, and Monte Carlo simulation
- **Data Processing**: Complex joins, window functions, aggregations, and CTEs

You do NOT need to specify exact function names or SQL syntax. Focus on describing the ANALYTICAL APPROACH and BUSINESS LOGIC. Genie Code will determine the best implementation.

#### INPUT DATA FORMAT
##### 1. Structured Data Schema
Each table is a `### catalog.schema.table` block. Under the heading, these metadata evidence lines appear when the metadata exists (they are metadata only, never row values):
- `Table comment:` the description the table owner wrote. Treat it as primary evidence of what the table records.
- `Data category:` TRANSACTIONAL, MASTER or REFERENCE as classified by the table filter. A trailing "(unverified default ...)" means the filter could not classify the table, so the category is unconfirmed.
- `Primary key:` the declared primary-key column(s).
- `Foreign keys:` the table's outgoing join edges as `column -> catalog.schema.table.column`. An edge ending in `(inferred)` was inferred from column naming, not declared: it is a likely join key, not a guaranteed one.
- `Columns shown: X of Y`: the table is wide and only X of its Y columns are listed (declared keys, commented, numeric-measure and date/timestamp columns are always kept). Use only the listed columns.
After these lines, a `| column | type | column_description |` table lists the columns.
{schema_markdown}

##### 1b. Foreign Key Relationships (CRITICAL FOR MULTI-TABLE USE CASES)

**🔗 USE THESE RELATIONSHIPS TO JOIN TABLES:**
{foreign_key_relationships}

**IMPORTANT INSTRUCTIONS FOR FK RELATIONSHIPS:**
- If relationships are listed above, you MUST evaluate use cases that JOIN these tables and generate the ones whose analysis requires the join
- Format: `source_table.column -> target_table.column` means you can JOIN on these columns
- Multi-table use cases that combine data from related tables deliver HIGHER BUSINESS VALUE
- Do NOT generate single-table use cases when FK relationships exist unless there's a specific single-table insight

---

### 4. CSV ROW PATTERN (adapt to YOUR industry, tables, and columns)

Genie Code will handle implementation. Replace all [PLACEHOLDERS] with actual values. Choose the BEST algorithm for each use case, whether SQL, ML, AI functions, or hybrid:

⚠️ **CRITICAL**: In the examples below, notice that the `usecase` column (column 3) NEVER echoes or derives from the `Analytics Technique` column (column 5). The `usecase` describes the BUSINESS goal; the technique is a separate, independent column.

`"1","Forecast Monthly [METRIC] with Capacity Planning Recommendations","Improve Planning Accuracy to Reduce Cost","Risk","Forecasting","[Statement]","[Solution using Databricks Genie Code]","[Business Value]","[Role]","[Executive]","[catalog.schema.table]","Using columns: [col1], [col2], [col3]. Deduplicate base data, aggregate by period, fit a time-series forecast with Prophet or ai_forecast using calendar features from the date column, then use ai_query to explain the forecast and recommend capacity actions."`

`"2","Predict [ENTITY] [OUTCOME] with Intervention Priority Queue","Reduce [OUTCOME] to Protect Revenue","Risk","Predictive Modeling","[Statement]","[Solution using Databricks Genie Code]","[Business Value]","[Role]","[Executive]","[catalog.schema.table1],[catalog.schema.table2]","Using columns: [col1], [col2], [col3], [col4] from table1 joined with [col5], [col6] from table2. Engineer features (recency, frequency, monetary metrics, behavioral patterns), train an XGBoost model to predict [outcome] probability, rank entities by risk score, then use ai_query to explain each prediction and recommend specific interventions."`

`"3","Classify [ENTITY] by [CATEGORY] with Risk Mitigation Actions","Reduce [ENTITY] Risk to Improve Margins","Improvement","Classification","[Statement]","[Solution using Databricks Genie Code]","[Business Value]","[Role]","[Executive]","[catalog.schema.table]","Using columns: [col1], [col2], [col3]. Engineer features from each entity's history, classify entities into risk tiers using Random Forest (or ai_classify for genuine free-text fields), then use ai_query to recommend mitigation actions per tier."`

---

### 5. FORMATTING RULES
- **HEADER** (EXACT: do NOT rename/reorder; the only additions are the 2 honesty columns at the end): `"No","Name","usecase","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Tables Involved","High Level Design","honesty_score","honesty_justification"`
- **EXACTLY 12 DATA COLUMNS** per row (matching the 12 header columns above), then the 2 honesty columns. Do NOT insert extra columns between them.
- ❌ WRONG: `"No","Name","usecase","type","Analytics Technique","Forecasting","Statement",...` (extra column "Forecasting" inserted, REJECTED)
- ✅ CORRECT: `"No","Name","usecase","type","Analytics Technique","Statement",...` (the value "Forecasting" goes inside the "Analytics Technique" cell of each data row)
- ALL values double-quoted. English. ONLY CSV output. Append the 2 honesty columns as columns 13-14.

### 6. SYSTEMATIC ENUMERATION MATRIX (MANDATORY)

**🚨 EXHAUSTIVE DISCOVERY PROTOCOL: DO NOT SKIP 🚨**

Before generating use cases free-form, you MUST systematically walk through the enumeration matrix below. For each cell in the matrix, ask: "Can this combination produce a VALID, NON-TRIVIAL business use case supported by the actual columns in the schema?" If YES, generate it. If NO (because the required columns don't exist, or the combination makes no business sense), SKIP it and move to the next cell.

**STEP 1: COLUMN-DRIVEN ENUMERATION (EVERY ALLOWED TECHNIQUE)**
For EACH numeric/date/text column in the schema, systematically consider EVERY applicable technique from the allowed `Analytics Technique` list:
- **Forecasting**: Can future values of this column be predicted? (requires temporal data)
- **Anomaly Detection**: Can unusual patterns in this column reveal business problems?
- **Classification**: Can entities be categorized into predefined business groups using this column?
- **Predictive Modeling**: Can trained ML models predict an outcome related to this column?
- **Clustering**: Can entities be grouped into natural, data-driven segments using this column?
- **Regression Analysis**: Can this column quantify drivers or predict continuous outcomes?
- **Optimization**: Can this column's values be optimized for better business outcomes?
- **Segmentation**: Can rule-based or ML grouping on this column create actionable business tiers?
- **Simulation**: Can what-if scenarios be modeled around this column?
- **Trend Analysis**: Does this column show growth, decline, or cyclical patterns over time?
- **Cohort Analysis**: Can entities sharing this column's value be tracked as cohorts over time?
- **Pareto Analysis**: Does this column reveal 80/20 concentration in business outcomes?
- **Survival Analysis**: Can time-to-event (churn, failure, renewal) be modeled with this column?
- **Correlation Analysis**: Are there business-relevant relationships between this and other columns?

**STEP 2: TEXT & UNSTRUCTURED COLUMN ENUMERATION**
For EACH text/string column in the schema, consider:
- **Sentiment Analysis**: Does this column contain text with emotional tone (feedback, reviews, comments)?
- **Document Processing**: Can structured data be extracted from this text column?
- **Extraction**: Can specific entities (names, dates, amounts, codes) be pulled from this text?
- **Classification / Clustering on text**: Can the text be classified into business categories (ai_classify) or clustered into topics that change a decision?

**STEP 3: MULTI-COLUMN & CROSS-TABLE ENUMERATION**
For EACH meaningful pair/group of columns (especially across JOINed tables), consider:
- **Correlation Analysis / Regression Analysis**: Is there a business-relevant relationship between these columns?
- **Predictive Modeling**: Can one column predict the other for business value?
- **Network Analysis**: Do entity relationships (supplier-customer, referral, hierarchy) form a graph worth analyzing?
- **Recommendation**: Can patterns across entities suggest next-best-actions, products, or offers?
- **Funnel Analysis**: Do these columns represent stages in a conversion/process funnel?
- **Geospatial Analysis**: Do location-related columns enable spatial pattern discovery?

**STEP 4: BUSINESS LENS ENUMERATION**
For the complete schema, consider EACH business perspective:
- **Revenue**: How can data increase, protect, or recover revenue?
- **Cost**: How can data reduce operational, inventory, or staffing costs?
- **Risk/Fraud**: How can data detect, prevent, or mitigate risks?
- **Customer**: How can data improve retention, satisfaction, or lifetime value?
- **Operations**: How can data improve efficiency, scheduling, or resource allocation?
- **Product/Inventory**: How can data optimize mix, pricing, or availability?
- **Competitive**: How can data create competitive advantages?

**🚨 BUSINESS VALUE & QUALITY GATE FOR EVERY USE CASE 🚨**
Before generating EACH use case from the matrix, verify ALL of the following:
1. **BUSINESS VALUE**: Does this use case deliver DIRECT, MEASURABLE business impact? (Revenue growth, Cost reduction, Risk mitigation, Productivity improvement). If the value is vague or marginal → SKIP IT.
2. **ANTI-HALLUCINATION**: Do the EXACT columns needed exist in the schema? If NO → SKIP IT.
3. **GENUINE FIT**: Is this technique genuinely the BEST approach for this business problem? NEVER force-fit a technique to a column just to fill the matrix; the business reason must be GENUINE.
4. **NON-TRIVIAL**: Does this require genuine analytical depth? If it's just a simple threshold, basic count, or date check → SKIP IT.
5. **NOT DUPLICATE**: Is this substantially different from every use case you already generated? If duplicate → SKIP IT.
6. **DECISION**: Does a named role change a specific decision because of the output? If NO → SKIP IT.
- After exhausting the matrix, generate any additional use cases you identify through domain expertise.

**STEP 5: TECHNIQUE FIT CHECK**
After completing steps 1-4, review the allowed techniques you did not use. For each, confirm that its structural prerequisites are absent or that no decision needs it. If a GENUINE, decision-changing use case exists for it, add it; never add a use case only to raise the number of distinct techniques. Do NOT skip this step.

### 7. PREVIOUS RUN FEEDBACK (ENSEMBLE PASS)

{previous_use_cases_feedback}

### 8. FINAL INSTRUCTION
Begin generation now. Output ONLY the CSV with the additional honesty columns at the end.
""" + HONESTY_CHECK_CSV





# --- 1c. PDF Summary Prompt (MODIFIED FOR REQUEST #1) ---
PROMPT_TEMPLATES["SUMMARY_GEN_PROMPT"] = """
You are a highly experienced Databricks Principal Strategist and business analyst.
Your task is to generate a **CSV** response containing a strategic summary for a customer.
The customer is: {business_name}
We have identified use cases across these domains: {domain_list}
Your output MUST be in **{output_language}**.

### USE CASE HIGHLIGHTS PER DOMAIN
Below are the top use cases per domain, each listed by its displayed title. You MUST reference specific use cases and business outcomes in your summaries to make them concrete and compelling (not generic):
{domain_use_case_highlights}

### NO COUNTS, TITLES ONLY (HARD RULE)
* Never state how many use cases the portfolio, the catalog or any domain has, neither in digits nor in words (for example never write "thirteen use cases", "its four use cases" or "33 identified use cases"). The document shows the exact counts next to your text, so any number you write can contradict it.
* Do not count the use cases you mention either: never write "Two use cases make this tangible" or "three examples stand out". Name each use case by its title instead. Any sentence that puts a number in front of "use case" is removed from the document.
* Reference a use case ONLY by its displayed title exactly as written in the highlights above (translated into {output_language} when the title is in another language). Never refer to a use case by a number, an ID or a paraphrased name.

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}

### TASK
Generate a **single CSV** with THREE content columns: "Type", "Summary", "TransliteratedBusinessName", followed by the 2 honesty columns defined at the end of this prompt.
Your response MUST start with the header: `"Type","Summary","TransliteratedBusinessName","honesty_score","honesty_justification"`
The CSV must contain:
1.s: One row for the "Executive" summary.
2.s: One row for **EACH** business domain listed in {domain_list}.

### COLUMN DEFINITIONS
1.s: **"Type"**:
    * Use the exact string "Executive" for the first row.
    * For all other rows, use the exact Business Domain name from the list.
2.s: **"Summary"**:
    * **For the "Executive" row**: Write a 2-3 paragraph, professional executive summary. Start with a powerful opening that names {business_name} specifically. Reference 2-3 of the most impactful use cases by their displayed titles across domains as concrete examples of what AI can deliver. Emphasize the value of AI and Databricks Genie Code in unlocking the potential of their data. Be specific to {business_name}'s industry; avoid generic AI platitudes.
    * **For each "Business Domain" row**: Write a 2-3 paragraph professional and engaging summary in prose. You MUST mention at least 2 specific use cases from that domain by their displayed titles and describe their expected business impact. This summary should narrate the domain's strategic importance, its core responsibilities, and the key opportunities AI (specifically Databricks Genie Code) can unlock. The tone should be identical to the Executive Summary.
    * **CRITICAL - "Databricks Genie Code" HANDLING**: When {output_language} uses a non-Latin script (Arabic, Chinese, Japanese, Korean, Hindi, Russian, etc.), you MUST transliterate "Databricks Genie Code" phonetically into the target script. The word order must remain exactly: Databricks → Genie → Code. Examples: Arabic: "داتا بريكس جيني كود", Chinese: "达塔布里克斯 吉尼 科德", Japanese: "データブリックス ジーニー コード", Korean: "데이터브릭스 지니 코드", Hindi: "डेटाब्रिक्स जीनी कोड", Russian: "Датабрикс Джини Код". Do NOT leave "Databricks Genie Code" in English for non-Latin scripts. For Latin-script languages, keep "Databricks Genie Code" as-is.
    * **FORMATTING**: ALL summaries (both Executive and Domain) **MUST** be enclosed in `<p>` HTML tags (e.g., `<p>Paragraph 1.</p><p>Paragraph 2.</p>`).
    * **PUNCTUATION**: """ + NO_EM_DASH_RULE + """
3.s: **"TransliteratedBusinessName"**:
    * **For the "Executive" row**: Provide the transliteration of {business_name} into {output_language}. If {output_language} is English, just repeat {business_name}.
    * **For all "Business Domain" rows**: This field MUST be an empty string `""`.

### CRITICAL CSV FORMATTING RULES
* **CSV Format**: The entire output MUST be a valid CSV. All values MUST be enclosed in **double quotes** (`"`).
* **NO CODE FENCES**: Do NOT include markdown code fences like ```csv or ``` at the beginning or end.
* **ONLY CSV**: Your response must contain ONLY the CSV data, starting with the header line.
* **Proper Escaping**: If a field value contains double quotes, escape them by doubling them ("").
* **No Extra Text**: Do NOT include any explanatory text, comments, or anything other than the CSV data.


### EXAMPLE CSV OUTPUT (for {output_language}=Arabic and {business_name}=Global Enterprises)
"Type","Summary","TransliteratedBusinessName"
"Executive","<p>تقف غلوبال إنتربرايزز في طليعة التميز في قطاعها...</p><p>يحدد هذا الكتالوج مسارًا واضحًا للاستفادة من البيانات المدعومة بـ داتا بريكس جيني كود...</p>","غلوبال إنتربرايزز"
"Customer Management","<p>يعد مجال إدارة العملاء أمرًا محوريًا لنجاح غلوبال إنتربرايزز...</p><p>من خلال الاستفادة من داتا بريكس جيني كود، يمكن لغلوبال إنتربرايزز تحويل هذا المجال...</p>",""
"Finance & Billing","<p>إدارة الصحة المالية للمؤسسة، يعد مجال المالية والفوترة أمرًا بالغ الأهمية...</p><p>يمثل داتا بريكس جيني كود فرصة كبيرة لأتمتة هذه العمليات...</p>",""

### EXAMPLE CSV OUTPUT (for {output_language}=English and {business_name}=Global Enterprises)
"Type","Summary","TransliteratedBusinessName"
"Executive","<p>Global Enterprises stands at the forefront of its industry, and the use cases in this catalog position the organization to revolutionize its operations through the strategic implementation of Databricks Genie Code.</p><p>This catalog outlines a clear path to leveraging data and AI to drive innovation, enhance efficiency, and create significant business value across all key domains.</p>","Global Enterprises"
"Customer Management","<p>The Customer Management domain is central to Global Enterprises's success, as it governs all direct interactions and the entire customer lifecycle. Its primary responsibility is to ensure high rates of acquisition, satisfaction, and retention.</p><p>By leveraging Databricks Genie Code, Global Enterprises can transform this domain, moving from reactive support to proactive engagement. Opportunities include developing sophisticated churn prediction models and deploying generative AI agents to provide instant, personalized customer service, dramatically improving loyalty and reducing operational costs.</p>",""
"Finance & Billing","<p>Managing the financial health of the organization, the Finance & Billing domain is critical for ensuring revenue integrity, compliance, and accurate forecasting. This domain oversees everything from invoicing and payments to financial reporting and risk analysis.</p><p>Databricks Genie Code presents a significant opportunity to automate and intelligentize these processes. For instance, AI can be used to parse document invoices, detect payment anomalies in real-time, and generate highly accurate revenue forecasts, thereby strengthening the organization's financial posture and decision-making capabilities.</p>",""

### FINAL INSTRUCTION
Begin generation now. Produce ONLY the CSV text, starting with the header `"Type","Summary","TransliteratedBusinessName","honesty_score","honesty_justification"`.

🚨 ABSOLUTE RULE - OUTPUT FORMAT 🚨:

❌ DO NOT INCLUDE:
- Any conversational or explanatory text ("Here is...", "I've generated...", "The...")
- Any thoughts or analysis descriptions

✅ OUTPUT REQUIREMENTS:
- Your response must START with: "Type","Summary","TransliteratedBusinessName","honesty_score","honesty_justification"
- Include honesty columns in header and all rows
""" + HONESTY_CHECK_CSV

# --- 1d. Domain Consolidation Prompt (ENHANCED FOR 6-90 USE CASES REQUIREMENT) ---
PROMPT_TEMPLATES["DOMAIN_FINDER_PROMPT"] = """
You are an expert business analyst specializing in BALANCED domain taxonomy design with deep industry knowledge.

**🎯 YOUR TASK**: Analyze the provided use cases and assign each one to appropriate Business Domains (NO subdomains yet).

**HARD CONSTRAINTS (automated rejection on violation)**:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. **DOMAIN COUNT**: 3-25 domains. MAXIMUM 25 is an ABSOLUTE HARD LIMIT (>25 → REJECTED).
   - Target: round(√total_use_cases), capped at 25. Examples: 20 UCs → 4-5 domains; 30 UCs → 5-6; 50 → 7; 100 → 10; 400 → 20.
   - For small datasets (<40 UCs): aim for 4-6 domains. NEVER collapse to only 2-3 mega-domains.
   - For small datasets, domains with 3-5 UCs each are IDEAL. Prefer more domains with fewer UCs over fewer domains with many UCs.
2. **EXACTLY ONE WORD per domain name**: Zero spaces, zero CamelCase concatenation. Only one capital letter (at the start).
   - ❌ REJECTED: "NetworkManagement", "CareTransitions", "CustomerService", "Network Management"
   - ✅ CORRECT: "Network", "Care", "Customers", "Risk", "Maintenance", "Revenue"
   - Rule: Strip adjectives/verbs/descriptors/compound words → keep ONLY the core business noun.
3. **NO OVERLAPPING WORDS**: No two domains may share any word. If overlap detected, MERGE into the shortest (1-word) name.
   - "Customer Service" + "Customer Support" → "Customers"; "Risk Management" + "Risk Analysis" → "Risk"
4. **BALANCED DISTRIBUTION**: No single domain should contain more than 35% of all use cases. Do NOT consolidate into mega-domains.
   - For <40 UCs: 3-10 UCs per domain is ideal. For 40-100 UCs: 5-20 UCs per domain. For >100 UCs: 8-80 UCs per domain. Split any domain >80 UCs.
5. **INDUSTRY-SPECIFIC NAMING**: Infer industry from the actual data. Use business-specific terms. NEVER use overly broad catch-all names.
   - ❌ BANNED generic names: "Operations", "Engagement", "Management", "Services", "Activities", "Processes", "Functions", "Solutions", "Systems", "General"
   - ✅ Use specific business nouns: "Franchise", "Inventory", "Revenue", "Customers", "Fraud", "Workforce", "Pricing", "Logistics", "Compliance", "Marketing"
   - Infer domain names from the actual tables, columns, and use case themes in the schema. Each domain should represent a DISTINCT business function.
6. **NO TWO DOMAINS share the same core business name**.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**BUSINESS CONTEXT**:
Business Name: {business_name}
Industries: {industries}
Business Context: {business_context}

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}


**INPUT DATA**:
CSV columns: "No","Name","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Tables Involved"

**TASK WORKFLOW**:
1. Analyze use case names, statements, solutions, techniques, and tables
2. Identify natural business groupings by: functionality, tables/data domains, beneficiaries, business value
3. Create 1-WORD industry-specific domain names (NOT generic catch-alls like "Operations" or "Engagement")
4. **BALANCE CHECK**: Count UCs per domain. If ANY domain has >35% of total UCs, you MUST split it into more specific domains
5. **MINIMUM DOMAINS CHECK**: Ensure you have at least round(√total_use_cases) domains. For 20-30 UCs that means 4-6 domains minimum
6. Validate all hard constraints above before outputting

**MERGE PATTERNS** (adapt to YOUR actual industry from data):
- Shared-word domains → merge to 1 word: "Sales Operations" + "Sales Analytics" → "Sales"
- BUT different concepts stay separate: "Risk" ≠ "Compliance"; "Revenue" ≠ "Sales"

The output language for domain names should be {output_language}.

**OUTPUT FORMAT**: CSV with header and EXACTLY ONE row per input use case: every `No` value in the input CSV appears exactly once as `use_case_id`, copied verbatim. Never omit, duplicate or invent a use case id. Return ONLY the CSV — no text, no fences, no explanations.

Example:
use_case_id,domain
N1-AI01,Revenue
N1-AI04,Customers
N1-AI07,Inventory
N1-AI10,Maintenance

**INPUT USE CASES CSV**:
{use_cases_csv}

{previous_violations}

🚨 ANTI-PROSE PROTOCOL (read carefully):
- Your FIRST character MUST be 'u' (first letter of "use_case_id" header).
- NO preamble. NO "I'll analyze". NO "Looking at". NO "Let me". NO "**Analysis:**". NO "First, I'll...".
- NO explanatory paragraphs before the CSV. NO bullet list summaries.
- NO text after the CSV either.
- Output ONLY: the CSV header line, then data rows. Nothing else.
- If you catch yourself writing prose, STOP and start over with 'use_case_id,'.

Your response must START with: use_case_id,domain,honesty_score,honesty_justification
Begin your CSV response NOW (first character must be 'u'):
""" + HONESTY_CHECK_CSV

# --- 1d2. Subdomain Detector Prompt (NEW - PER-DOMAIN SUBDOMAIN ASSIGNMENT) ---
PROMPT_TEMPLATES["SUBDOMAIN_DETECTOR_PROMPT"] = """
You are an expert business analyst specializing in subdomain taxonomy design within business domains.

**🎯 YOUR TASK**: Analyze the use cases for a SINGLE domain and assign each to appropriate Subdomains.

**HARD CONSTRAINTS (automated rejection on violation)**:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. **SUBDOMAIN COUNT**: 1-10 per domain. For domains with ≤4 UCs, exactly 1 subdomain is acceptable and preferred.
2. **NAMING**: EXACTLY 2 WORDS per subdomain. 1 word → REJECTED. 3+ words → REJECTED.
   - ✅ "Revenue Pricing", "Quality Control", "Risk Management", "Supply Planning"
   - ❌ "Scheduling" (1 word), "Quality Control Management" (3 words)
3. **🚨 ABSOLUTE MINIMUM 2 USE CASES PER SUBDOMAIN 🚨**: This is the #1 cause of rejections.
   Every subdomain MUST contain AT LEAST 2 use cases. If a subdomain would have only 1 UC, you MUST merge it into the most semantically related subdomain. Do NOT create a subdomain for a single use case under any circumstances.
   **VERIFICATION STEP**: Before outputting, COUNT the use cases per subdomain. If ANY subdomain has <2, MERGE IT.
4. **NO OVERLAPPING WORDS**: No two subdomains within this domain may share any word. If overlap, merge (keep 2 words).
5. **BUSINESS-FOCUSED**: Use business terms, NOT technical terms.
6. **BALANCED DISTRIBUTION**: Distribute use cases evenly across subdomains. Prefer fewer subdomains with more UCs over many subdomains with barely 2.
7. **NO PLACEHOLDER NAMES**: NEVER use generic or sequential names like "Sub Domain1", "Domain 1", "Category 1", "Group A", "Area 1", "N/A", "Other", "General", "Miscellaneous", or "Uncategorized". Every subdomain name must be a MEANINGFUL 2-word business term that describes the actual functional area. If you cannot find a meaningful name, merge those use cases into an existing subdomain.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**PRACTICAL SIZING GUIDANCE** (CRITICAL — follow strictly):
- For 2-4 use cases: create EXACTLY 1 subdomain (all UCs go into one meaningful subdomain)
- For 5-6 use cases: create EXACTLY 2 subdomains (split roughly evenly)
- For 7-12 use cases: create 2-3 subdomains
- For 13-20 use cases: create 3-5 subdomains
- For 21+ use cases: create 4-8 subdomains
FEWER subdomains is ALWAYS better. A subdomain with 4-5 UCs is ideal. A subdomain with only 2 UCs is acceptable only if semantically necessary. NEVER create subdomains where each has exactly 2 UCs if you could have 1-2 broader subdomains with 3-5 UCs each.

**BUSINESS CONTEXT**:
Domain Name: {domain_name}
Business Name: {business_name}
Industries: {industries}
Business Context: {business_context}

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}


**INPUT DATA**:
CSV of use cases belonging to domain "{domain_name}":
"No","Name","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Tables Involved"

**TASK WORKFLOW**:
1. Analyze use case names, statements, solutions, techniques, and tables
2. Identify natural business groupings by: processes, functional areas, activities, stakeholders
3. Create 2-WORD business-focused subdomain names
4. **CRITICAL**: Count use cases per subdomain. Merge any subdomain that has fewer than 2 UCs into the nearest related subdomain
5. Validate all hard constraints before outputting

The output language for subdomain names should be {output_language}.

**OUTPUT FORMAT**: CSV with header and EXACTLY ONE row per input use case: every `No` value in the input CSV appears exactly once as `use_case_id`, copied verbatim. Never omit, duplicate or invent a use case id. Return ONLY the CSV — no text, no fences, no explanations.

Example for "Revenue" domain:
use_case_id,subdomain
N1-AI01,Pricing Strategy
N1-AI04,Yield Management
N1-AI07,Revenue Forecasting

**INPUT USE CASES CSV FOR DOMAIN "{domain_name}"**:
{use_cases_csv}

{previous_violations}

Your response must START with: use_case_id,subdomain,honesty_score,honesty_justification
If you cannot meet requirements, still output CSV (merge subdomains as needed).
🚨 ANTI-PROSE PROTOCOL (read carefully):
- Your FIRST character MUST be 'u' (first letter of "use_case_id" header).
- NO preamble. NO "I'll analyze". NO "Looking at". NO "Let me". NO "**Analysis:**". NO bullet lists.
- Output ONLY: CSV header line, then data rows. Nothing else.
- If you catch yourself writing prose, STOP and start over with 'use_case_id,'.

Begin your CSV response now:
""" + HONESTY_CHECK_CSV
# --- 1f. Translation Prompts (MODIFIED FOR REQUEST #1, #2) ---
PROMPT_TEMPLATES["KEYWORDS_TRANSLATE_PROMPT"] = """You are an expert translator. Your task is to translate the *values* (and only the values) of the following JSON object into {target_language}.
Do NOT translate the keys.

**CRITICAL JSON SYNTAX VALIDATION**:
- Every key-value pair MUST have a colon (:) separator
- Format: "key": "translated value"
- **MOST COMMON ERROR**: Missing colon - `"key" "value"` is WRONG, must be `"key": "value"`
- **FOR ARABIC/CHINESE/JAPANESE/HINDI**: Verify the colon `:` exists between key and value
- Count your colons - you need ONE colon per key-value pair
- Validate your JSON structure before responding

SPECIAL INSTRUCTIONS:
🚨 CRITICAL: You MUST translate ALL text values in the JSON object. Do NOT leave ANY value in English (except for the specific exceptions listed below).

**MANDATORY TRANSLATION REQUIREMENT:**
- EVERY single value in the JSON MUST be translated to {target_language}
- This includes short words like "Type", "Quality", "Subdomain", "Statement", "Solution", "Beneficiary", "Sponsor", etc.
- DO NOT assume any value should stay in English just because it's short or seems technical
- If a value is in English in the input, it MUST be in {target_language} in the output (except for the specific exceptions below)

**ONLY THESE SPECIFIC EXCEPTIONS - DO NOT TRANSLATE:**
1. "N/A" - Keep as "N/A" in all languages

**PLACEHOLDERS:** Some values contain placeholders in curly braces, such as {{count}}, {{total_cases}} or {{business_name}}. Keep every placeholder exactly as written (same name, same braces) and translate only the words around it.

**SPECIAL RULE FOR TITLES CONTAINING "Powered by Databricks Genie Code":**
- "Powered by" MUST be TRANSLATED into the target language (e.g., Arabic: "مدعومة بـ", Spanish: "Impulsados por", French: "Propulsés par", German: "Unterstützt von", Chinese: "由...驱动", Japanese: "を活用した", Korean: "기반의", Hindi: "द्वारा संचालित", Russian: "на базе", Portuguese: "Impulsionados por", Italian: "Alimentati da")
- "Databricks Genie Code" MUST be TRANSLITERATED phonetically for non-Latin scripts (Arabic: "داتا بريكس جيني كود", Chinese: "达塔布里克斯 吉尼 科德", Japanese: "データブリックス ジーニー コード", Korean: "데이터브릭스 지니 코드", Hindi: "डेटाब्रिक्स जीनी कोड", Russian: "Датабрикс Джини Код"). For Latin-script languages, keep "Databricks Genie Code" as-is.
- The word order of "Databricks Genie Code" MUST remain exactly: Databricks, then Genie, then Code (transliterated in same order)

**EVERYTHING ELSE MUST BE TRANSLATED WITHOUT EXCEPTION**

**FOR ALL LANGUAGES - UNIVERSAL REQUIREMENTS:**
- Provide EXACT, DIRECT translations of the English text
- Do NOT use special phrasings or adaptations
- Translate literally and accurately
- Use standard terminology for the target language
- Do NOT leave any English words in the translation (except "N/A" and the transliterated "Databricks Genie Code" described above)
- Ensure complete and proper character encoding for all scripts (Arabic, Chinese, Japanese, Hindi, Cyrillic, etc.)

**MANDATORY TRANSLATIONS FOR ALL LANGUAGES (these MUST be translated, NOT left in English):**

**Common UI Terms (translate to your target language):**
  * "Type" → MUST translate
  * "Subdomain" → MUST translate
  * "Analytics Technique" → MUST translate
  * "Primary Table" → MUST translate
  * "Quality" → MUST translate
  * "Statement" → MUST translate
  * "Solution" → MUST translate
  * "Business Value" → MUST translate
  * "Beneficiary" → MUST translate
  * "Sponsor" → MUST translate
  * "Business Domain" → MUST translate
  * "Tables Involved" → MUST translate

**Value Terms (translate to your target language):**
  * "Risk" → MUST translate
  * "Problem" → MUST translate
  * "Opportunity" → MUST translate
  * "Improvement" → MUST translate
  * "Very High" → MUST translate
  * "High" → MUST translate
  * "Medium" → MUST translate
  * "Low" → MUST translate
  * "Very Low" → MUST translate

**REFERENCE TRANSLATIONS BY LANGUAGE:**

**Arabic:**
  * "Type" = "النوع", "Subdomain" = "المجال الفرعي", "Analytics Technique" = "تقنية التحليل", "Primary Table" = "الجدول الرئيسي", "Quality" = "الجودة"
  * "Statement" = "البيان", "Solution" = "الحل", "Business Value" = "القيمة التجارية"
  * "Beneficiary" = "المستفيد", "Sponsor" = "الراعي", "Business Domain" = "مجال الأعمال"
  * "Risk" = "مخاطرة", "Problem" = "مشكلة", "Opportunity" = "فرصة", "Improvement" = "تحسين"
  * "High" = "عالية", "Very High" = "عالية جداً", "Medium" = "متوسطة", "Low" = "منخفضة", "Very Low" = "منخفضة جداً"

**Spanish:**
  * "Type" = "Tipo", "Subdomain" = "Subdominio", "Analytics Technique" = "Técnica de Análisis", "Primary Table" = "Tabla Principal", "Quality" = "Calidad"
  * "Statement" = "Declaración", "Solution" = "Solución", "Business Value" = "Valor Comercial"
  * "Beneficiary" = "Beneficiario", "Sponsor" = "Patrocinador", "Business Domain" = "Dominio Empresarial"
  * "Risk" = "Riesgo", "Problem" = "Problema", "Opportunity" = "Oportunidad", "Improvement" = "Mejora"
  * "High" = "Alto", "Very High" = "Muy Alto", "Medium" = "Medio", "Low" = "Bajo", "Very Low" = "Muy Bajo"

**French:**
  * "Type" = "Type", "Subdomain" = "Sous-domaine", "Analytics Technique" = "Technique d'Analyse", "Primary Table" = "Table Principale", "Quality" = "Qualité"
  * "Statement" = "Déclaration", "Solution" = "Solution", "Business Value" = "Valeur Commerciale"
  * "Beneficiary" = "Bénéficiaire", "Sponsor" = "Sponsor", "Business Domain" = "Domaine d'Activité"
  * "Risk" = "Risque", "Problem" = "Problème", "Opportunity" = "Opportunité", "Improvement" = "Amélioration"
  * "High" = "Élevé", "Very High" = "Très Élevé", "Medium" = "Moyen", "Low" = "Faible", "Very Low" = "Très Faible"

**German:**
  * "Type" = "Typ", "Subdomain" = "Unterbereich", "Analytics Technique" = "Analysetechnik", "Primary Table" = "Haupttabelle", "Quality" = "Qualität"
  * "Statement" = "Aussage", "Solution" = "Lösung", "Business Value" = "Geschäftswert"
  * "Beneficiary" = "Begünstigter", "Sponsor" = "Sponsor", "Business Domain" = "Geschäftsbereich"
  * "Risk" = "Risiko", "Problem" = "Problem", "Opportunity" = "Chance", "Improvement" = "Verbesserung"
  * "High" = "Hoch", "Very High" = "Sehr Hoch", "Medium" = "Mittel", "Low" = "Niedrig", "Very Low" = "Sehr Niedrig"

**Chinese (Simplified):**
  * "Type" = "类型", "Subdomain" = "子域", "Analytics Technique" = "分析技术", "Primary Table" = "主表", "Quality" = "质量"
  * "Statement" = "声明", "Solution" = "解决方案", "Business Value" = "业务价值"
  * "Beneficiary" = "受益人", "Sponsor" = "赞助者", "Business Domain" = "业务领域"
  * "Risk" = "风险", "Problem" = "问题", "Opportunity" = "机会", "Improvement" = "改进"
  * "High" = "高", "Very High" = "非常高", "Medium" = "中等", "Low" = "低", "Very Low" = "非常低"

**Japanese:**
  * "Type" = "タイプ", "Subdomain" = "サブドメイン", "Analytics Technique" = "分析技術", "Primary Table" = "主要テーブル", "Quality" = "品質"
  * "Statement" = "ステートメント", "Solution" = "ソリューション", "Business Value" = "ビジネス価値"
  * "Beneficiary" = "受益者", "Sponsor" = "スポンサー", "Business Domain" = "ビジネスドメイン"
  * "Risk" = "リスク", "Problem" = "問題", "Opportunity" = "機会", "Improvement" = "改善"
  * "High" = "高", "Very High" = "非常に高い", "Medium" = "中程度", "Low" = "低", "Very Low" = "非常に低い"

**Portuguese:**
  * "Type" = "Tipo", "Subdomain" = "Subdomínio", "Analytics Technique" = "Técnica de Análise", "Primary Table" = "Tabela Principal", "Quality" = "Qualidade"
  * "Statement" = "Declaração", "Solution" = "Solução", "Business Value" = "Valor Comercial"
  * "Beneficiary" = "Beneficiário", "Sponsor" = "Patrocinador", "Business Domain" = "Domínio de Negócio"
  * "Risk" = "Risco", "Problem" = "Problema", "Opportunity" = "Oportunidade", "Improvement" = "Melhoria"
  * "High" = "Alto", "Very High" = "Muito Alto", "Medium" = "Médio", "Low" = "Baixo", "Very Low" = "Muito Baixo"

**Russian:**
  * "Type" = "Тип", "Subdomain" = "Поддомен", "Analytics Technique" = "Аналитическая техника", "Primary Table" = "Основная таблица", "Quality" = "Качество"
  * "Statement" = "Заявление", "Solution" = "Решение", "Business Value" = "Бизнес-ценность"
  * "Beneficiary" = "Бенефициар", "Sponsor" = "Спонсор", "Business Domain" = "Бизнес-домен"
  * "Risk" = "Риск", "Problem" = "Проблема", "Opportunity" = "Возможность", "Improvement" = "Улучшение"
  * "High" = "Высокий", "Very High" = "Очень Высокий", "Medium" = "Средний", "Low" = "Низкий", "Very Low" = "Очень Низкий"

**Hindi:**
  * "Type" = "प्रकार", "Subdomain" = "उपडोमेन", "Analytics Technique" = "विश्लेषण तकनीक", "Primary Table" = "प्राथमिक तालिका", "Quality" = "गुणवत्ता"
  * "Statement" = "कथन", "Solution" = "समाधान", "Business Value" = "व्यावसायिक मूल्य"
  * "Beneficiary" = "लाभार्थी", "Sponsor" = "प्रायोजक", "Business Domain" = "व्यावसायिक डोमेन"
  * "Risk" = "जोखिम", "Problem" = "समस्या", "Opportunity" = "अवसर", "Improvement" = "सुधार"
  * "High" = "उच्च", "Very High" = "बहुत उच्च", "Medium" = "मध्यम", "Low" = "निम्न", "Very Low" = "बहुत निम्न"

**VALIDATION CHECKLIST - Before submitting your translation:**
✓ Every value that was in English is now in {target_language} (except "N/A" and the "Databricks Genie Code" rule)
✓ "Type" is translated (NOT left as "Type")
✓ "Subdomain" is translated (NOT left as "Subdomain")
✓ "Analytics Technique" is translated (NOT left as "Analytics Technique")
✓ "Primary Table" is translated (NOT left as "Primary Table")
✓ "Quality" is translated (NOT left as "Quality")
✓ "Statement" is translated (NOT left as "Statement")
✓ "Solution" is translated (NOT left as "Solution")
✓ "Beneficiary" is translated (NOT left as "Beneficiary")
✓ "Sponsor" is translated (NOT left as "Sponsor")
✓ "Business Value" is translated (NOT left as "Business Value")
✓ "Business Domain" is translated (NOT left as "Business Domain")
✓ "Tables Involved" is translated (NOT left as "Tables Involved")
✓ ALL keys with "Type", "Quality" values are translated (including "type", "quality", "aspect_quality")
✓ All other English words are translated
✓ ONLY "N/A" remains in English. Title phrases have "Powered by" translated and "Databricks Genie Code" transliterated

**CRITICAL - COUNT YOUR TRANSLATIONS:**
The input JSON has approximately 70+ key-value pairs. Your output JSON must have THE SAME number of key-value pairs with 99%+ of values in {target_language} (only "N/A" stays in English; title phrases have "Powered by" translated and "Databricks Genie Code" transliterated for non-Latin scripts).

If you see ANY English words in your output other than "N/A" and the "Databricks Genie Code" rule, you FAILED. Go back and translate them.

Return ONLY a single, valid JSON object with the exact same structure, but with the values translated.

🚨 ABSOLUTE RULE - OUTPUT FORMAT 🚨:

❌ DO NOT INCLUDE:
- Any conversational or explanatory text ("Here is...", "I've...", "The...")
- Any thoughts or analysis descriptions

✅ OUTPUT REQUIREMENTS:
- Your response must be wrapped in honesty JSON: {{"honesty_score": XX, "honesty_justification": "...", "data": <your_translated_json>}}
- Start with: {{"honesty_score":

Input JSON:
{json_payload}

Respond with the honesty-wrapped JSON object.
""" + HONESTY_CHECK_JSON

PROMPT_TEMPLATES["USE_CASE_TRANSLATE_PROMPT"] = """You are an expert translator. Your task is to translate a list of use cases into {target_language}.

**🚨 CRITICAL OUTPUT REQUIREMENT - READ THIS FIRST 🚨**
Your response MUST be PURE CSV DATA ONLY. 
- NO explanations before or after the CSV
- NO conversational text
- NO SQL code fragments
- NO commentary
- NO markdown fences (```)
- ONLY the CSV header and data rows
- Start immediately with the header row: "No","Name",...

**FIELDS TO TRANSLATE** (translate these 18 fields):
"Name", "Headline", "Business Domain", "Subdomain", "type", "Technique Label", "Statement", "Solution", "Business Value", "Beneficiary", "Sponsor", "Business Priority Alignment", "Strategic Goals Alignment", "Measures of Success", "Financial Benefits", "Non-Financial Benefits", "KPI Names", "Proposed Key Results"

**SPECIAL RULES FOR MEASURES AND BENEFITS**:
- "Measures of Success", "Financial Benefits" and "Non-Financial Benefits" are full sentences: translate them fully into {target_language}.
- Keep every backticked identifier (such as `column_name` or `catalog.schema.table`) and every bracketed placeholder (such as [BASELINE: ...], [ESTIMATE: ...], [ASSUMPTION: ...], [TARGET]) exactly as written.
- If a field is "" in the input, output "" for it.

**SPECIAL RULES FOR KPI NAMES AND PROPOSED KEY RESULTS**:
- "KPI Names" lists short KPI names and "Proposed Key Results" lists full sentences, each separated by " | ". Translate every item into {target_language} and keep the " | " separators and the number of items exactly as in the input, in the same order.
- Keep every backticked identifier and every bracketed placeholder exactly as written.
- If a field is "" in the input, output "" for it.

**SPECIAL RULES FOR HEADLINE AND TECHNIQUE LABEL**:
- "Headline" is the short title of the use case. Translate it into a short, natural {target_language} title with the same meaning; keep it short. Output only the title itself, never a label such as "Headline:" in front of it.
- "Technique Label" is an analytics technique name. Translate it into the standard {target_language} term for that technique.
- If either field is "" in the input, output "" for it.
- Write every row in EXACTLY the column order of the header below: "Headline" and "Technique Label" come after "Quality", then "Measures of Success", "Financial Benefits", "Non-Financial Benefits", "KPI Names" and "Proposed Key Results". Never move a value into another column.

Do NOT translate: "No", "Tables Involved", "Analytics Technique", "Quality"

**SPECIAL RULES FOR STRATEGIC GOAL ALIGNMENT FIELD**:
- This field contains strategic goal values like: "Reduce Cost", "Increase Revenue", "Boost Productivity", "Mitigate Risk", "Protect Revenue", "Align to Regulations", "Improve Customer Experience", "Enable Data-Driven Decisions", "General Improvement"
- Translate these goal values to {target_language} using proper business terminology
- If multiple goals are comma-separated (e.g., "Reduce Cost, Increase Revenue"), translate each goal separately and keep them comma-separated
- Examples for MULTIPLE languages:
  * Arabic: "Reduce Cost" → "تقليل التكلفة", "Increase Revenue" → "زيادة الإيرادات"
  * French: "Reduce Cost" → "Réduire les coûts", "Boost Productivity" → "Améliorer la productivité"
  * Spanish: "Reduce Cost" → "Reducir costos", "Mitigate Risk" → "Mitigar riesgos"
  * Chinese: "Reduce Cost" → "降低成本", "Increase Revenue" → "增加收入"
  * German: "Reduce Cost" → "Kosten senken", "Protect Revenue" → "Umsatz schützen"

**NOTE**: The SQL field is NOT included in the input to reduce payload size. You only need to translate the fields listed above.

**SPECIAL RULES FOR SPONSOR FIELD**:
- If the Sponsor field contains a name in the format "Name (Title)" (e.g., "John Smith (Chief Technology Officer)"), you MUST transliterate/translate BOTH the name AND the title for ALL languages
- Person names should be transliterated phonetically into the target language writing system
- **CRITICAL**: This applies to ALL languages, not just non-Latin scripts. Transliterate names appropriately for each target language.
- Examples for MULTIPLE languages:
  * French: "John Smith (Chief Technology Officer)" → "Jean Smith (Directeur de la Technologie)"
  * Spanish: "John Smith (Chief Technology Officer)" → "Juan Smith (Director de Tecnología)"
  * Arabic: "John Smith (Chief Technology Officer)" → "جون سميث (المدير التنفيذي للتكنولوجيا)"
  * Chinese: "John Smith (Chief Technology Officer)" → "约翰·史密斯 (首席技术官)"
  * Japanese: "John Smith (Chief Technology Officer)" → "ジョン・スミス (最高技術責任者)"
  * German: "John Smith (Chief Technology Officer)" → "Johann Schmidt (Technischer Leiter)"
  * Russian: "John Smith (Chief Technology Officer)" → "Джон Смит (Технический директор)"
  * Hindi: "John Smith (Chief Technology Officer)" → "जॉन स्मिथ (मुख्य प्रौद्योगिकी अधिकारी)"
- If the Sponsor field contains ONLY a title (no name), translate it normally
- Always adapt the name to sound natural in the target language and culture

**OUTPUT FORMAT: CSV (NOT JSON)**
Your response MUST be in CSV format with the following structure:
- First line: Header row with the 22 column names below followed by the 2 honesty columns "honesty_score","honesty_justification" (24 columns; SQL field is not included)
- Subsequent lines: One row per use case with 24 fields (22 translation fields + 2 honesty fields)

**CSV HEADER (MUST BE EXACTLY THIS)**:
"No","Name","Business Domain","Subdomain","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Business Priority Alignment","Strategic Goals Alignment","Tables Involved","Quality","Headline","Technique Label","Measures of Success","Financial Benefits","Non-Financial Benefits","KPI Names","Proposed Key Results"

**OPTIONAL REASON COLUMNS (only when the input has them)**:
Some input use cases carry the extra keys "Coverage Pick Reason" and "Below Bar Reason".
- For each of these keys that ANY input use case has, append it as 1 extra column right after "Proposed Key Results" (and before the 2 honesty columns), in that order, for EVERY row. Use "" for a use case that does not have the key.
- Translate their values FULLY into {target_language}. Keep every backticked identifier (such as `column_name` or `catalog.schema.table`) exactly as written.
- If NO input use case has a key, do NOT add its column.

**CRITICAL CSV FORMATTING RULES**:
1. ALL fields MUST be enclosed in double quotes (")
2. Use comma (,) as the field separator
3. For fields containing commas or quotes, keep them inside the double quotes
4. Each row must have exactly 24 fields: the 22 fields of the header above plus the 2 honesty fields (SQL is NOT included); each optional reason column that applies adds 1 field, so a row has 25 fields with one of them and 26 with both
5. Keep untranslated fields (No, Tables Involved, Analytics Technique, Quality) EXACTLY as they appear in the input
6. The Analytics Technique field should be copied EXACTLY as-is in English (do not translate)
7. The Quality field should be copied EXACTLY as-is in English (do not translate "Very High", "High", "Medium", "Low", "Very Low")

**IMPORTANT**: Analytics Technique and Quality values remain in English in the CSV. The UI will handle translation separately.

**TRANSLATION REQUIREMENTS**:
1. Translate ALL 18 designated fields for EVERY use case, including ALL long text fields
2. "Business Domain" - MUST be translated to {target_language}
3. "Subdomain" - MUST be translated to {target_language}
4. "Name" - MUST be translated to {target_language}
5. "Business Priority Alignment" - MUST be translated to {target_language} (translate the priority values like "Reduce Cost" → appropriate translation)
6. "Strategic Goals Alignment" - MUST be translated to {target_language} (this is FREE-TEXT describing strategic goals; translate the entire text naturally)
7. "Statement" - MUST be FULLY translated to {target_language} (this is a long text field; do NOT leave it in English)
8. "Solution" - MUST be FULLY translated to {target_language} (this is a long text field; do NOT leave it in English)
9. "Business Value" - MUST be FULLY translated to {target_language} (this is a long text field; do NOT leave it in English)
10. "Beneficiary" - MUST be translated to {target_language} (translate role titles like "Revenue Management Analyst")

🚨 **CRITICAL FOR NON-LATIN SCRIPTS (Chinese, Arabic, Japanese, Korean, Hindi, Russian)**:
- You MUST translate ALL 18 fields including long text (Statement, Solution, Business Value)
- If ANY field remains in English/Latin script (except No, Tables Involved, Analytics Technique, Quality), your translation is WRONG
- The Statement, Solution, and Business Value fields are LONG paragraphs; you MUST translate them FULLY, not skip them

**SPECIAL INSTRUCTIONS**:
- "Powered by Databricks Genie Code" → "Powered by" MUST be TRANSLATED into {target_language}. "Databricks Genie Code" MUST be TRANSLITERATED phonetically for non-Latin scripts (Arabic: "مدعومة بـ داتا بريكس جيني كود", Chinese: "由 达塔布里克斯 吉尼 科德 驱动", Japanese: "データブリックス ジーニー コード を活用した", Korean: "데이터브릭스 지니 코드 기반의", Hindi: "डेटाब्रिक्स जीनी कोड द्वारा संचालित", Russian: "на базе Датабрикс Джини Код"). For Latin-script languages, translate "Powered by" and keep "Databricks Genie Code" as-is. Word order of "Databricks Genie Code" must be preserved exactly.
- Keep numeric IDs (like "AI-001") unchanged
- """ + NO_EM_DASH_RULE + """
- Keep Analytics Technique and Quality values in English (not translated in CSV)
- For Arabic: Ensure complete and proper Arabic text encoding. Double-check all Arabic characters are properly formed.

**🛑 ABSOLUTELY FORBIDDEN - DO NOT INCLUDE 🛑**:
- ❌ NO text before the CSV header
- ❌ NO explanatory text like "Here is the translation..." or "I've translated..."
- ❌ NO SQL code fragments or queries outside the CSV structure
- ❌ NO examples or demonstrations
- ❌ NO line breaks or empty lines before the header
- ❌ NO commentary about the translation process
- ❌ NO additional rows beyond the exact number provided in the input
- ✅ ONLY: Pure CSV starting with the header, nothing else

**VALIDATION CHECKLIST**:
✓ Response starts with the exact 22-column header followed by the 2 honesty columns (no text before it)
✓ All fields are enclosed in double quotes
✓ Each row has exactly 24 comma-separated fields: 22 translation fields + 2 honesty fields (SQL is NOT included)
✓ No markdown fences (```) around the CSV
✓ Analytics Technique field is copied exactly as received (do not translate it)
✓ Quality field is copied exactly as received (do not translate it)
✓ Business Priority Alignment field is translated to target language
✓ Strategic Goals Alignment field is translated to target language
✓ Statement, Solution, Business Value fields are FULLY translated (not left in English)
✓ All 18 translatable fields are translated for every row

**CRITICAL ROW COUNT REQUIREMENT**:
- You will receive EXACTLY a specific number of use cases in the input
- Your CSV output MUST contain EXACTLY the same number of data rows (plus the header row)
- DO NOT add any extra rows beyond what was provided in the input
- DO NOT omit any rows from the input
- Each row in your output MUST correspond to exactly one row from the input, matched by the "No" field

**INPUT USE CASES** (as JSON for readability):
{json_payload}

🚨 FINAL INSTRUCTION 🚨
Your ENTIRE response must be ONLY the CSV data.
- Start your response with: "No","Name","Business Domain","Subdomain","type","Analytics Technique","Statement",...
- Do NOT write anything before this header
- Do NOT write anything after the last data row
- Do NOT include markdown code fences (```)
- Do NOT include any explanatory text
- NO thoughts or reasoning
- NO "Here is the translation..." or similar statements
- The number of data rows MUST exactly match the number of use cases above

🚨 ABSOLUTE RULE - OUTPUT FORMAT 🚨:

❌ DO NOT INCLUDE:
- Any conversational or explanatory text ("Here is...", "I've...", "The...")
- Any thoughts or analysis descriptions

✅ OUTPUT REQUIREMENTS:
- Your response must START with: "No","Name","Business Domain",...,"honesty_score","honesty_justification"
- Include honesty columns in header and all rows

Begin your CSV response now (header row first, include honesty columns):
""" + HONESTY_CHECK_CSV


# --- 1f. Use Case Review Prompt (RENAMED, ENHANCED) ---
PROMPT_TEMPLATES["REVIEW_USE_CASES_PROMPT"] = """You are a **Deduplication Specialist** for an analytics use case portfolio. Your job is to remove semantic duplicates — use cases that address the SAME business problem with the SAME core analysis on the SAME entity/metric, OR that analyze the SAME underlying data with the SAME analytical intent even if the wording differs. You must PRESERVE genuinely distinct use cases.

**DECISION BIAS: When in doubt, KEEP both use cases.** Remove a use case only when it is clearly the weaker copy of a duplicate under the criteria below; near duplicates are checked again by later stages.
**TARGET: Remove genuine duplicates. There is no expected removal rate: remove exactly the duplicates you find, whether that is none or many.**

**INPUT**: You receive ID, Name, Business Domain, Analytics Technique, Tables (primary table names, without catalog/schema), and a 1-line Statement. Judge duplicates on BUSINESS INTENT AND DATA GROUNDING (tables + technique + statement together).

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}


**REMOVAL CRITERIA** — a use case is removed if ANY of the following is true:

**LAYER A — IDENTICAL NAME:** Two or more use cases share the EXACT same Name (case-insensitive, whitespace-normalized). Keep the earliest ID; remove the rest.

**LAYER B — SEMANTIC DUPLICATE (verb + noun synonym + entity/metric all match):**
  1. Normalize leading verb: Forecast/Predict/Anticipate/Envision → PREDICT; Detect/Identify/Reveal/Find/Discover/Uncover/Expose → DETECT; Classify/Segment/Categorize/Tier/Rank → CLASSIFY; Simulate/Model/Project → SIMULATE; Optimize/Maximize/Minimize → OPTIMIZE; Assess/Evaluate/Measure/Gauge/Quantify → ASSESS. Strip "with [...]" activation suffix.
  2. Apply noun synonym map: Revenue=Income=Sales; Cost=Expense=Spend; Churn=Attrition=Turnover; Demand=Consumption=Volume; Customer=Client=Account; Supplier=Vendor; Inventory=Stock=Supply; Fraud=Anomaly; Leakage=Loss=Erosion.
  3. Extract ENTITY and METRIC. If (normalized verb + entity + metric) all match → DUPLICATE. Example: "Forecast Customer Churn Rate" and "Predict Client Attrition Probability" → entity=Customer, metric=Churn → DUPLICATE.

**LAYER C — TECHNIQUE SWAP ON SAME QUESTION:** Same entity + same metric + same analytical intent (from Statement), but the Analytics Technique column differs. This is a duplicate — the underlying business question is identical; only the technique label varies. Example: two use cases on "predicting customer churn" where one lists Technique=Classification and the other lists Technique=Predictive Modeling → DUPLICATE.

**LAYER D — TABLE + INTENT OVERLAP:** Primary Tables heavily overlap (≥60% common base-table names) AND the Statements describe the same analytical question (same entity + same metric + same direction of analysis) even if names and techniques differ. This is the most common silent-duplicate pattern. EXAMPLE: UC-X "Identify low-utilization assets via clustering" on table `asset_utilization` and UC-Y "Detect underused equipment with anomaly analysis" on `asset_utilization` → same tables, same question → DUPLICATE.

**EXPLICITLY NOT DUPLICATES — KEEP BOTH (even under Layer D):**
- Different analytical DIMENSIONS on the same entity ("Revenue by Product" vs "Revenue by Customer").
- Different METRICS on the same entity ("Customer Churn" vs "Customer Lifetime Value").
- Different analytical QUESTIONS on the same data ("Forecast demand" vs "Detect demand anomalies" — prediction ≠ anomaly detection).
- Different BUSINESS DOMAINS even with superficial textual similarity (treat each domain's portfolio independently unless Layer A matches exactly).

**AMBIGUOUS CASES:**
If you are unsure whether two use cases are duplicates (partial match, close but not identical business intent), KEEP both.

**DUPLICATE RESOLUTION — which ID to KEEP when removing:**
a. Keep the use case whose Statement is more specific and actionable.
b. Prefer the one whose Tables cover a broader analytical join (more tables listed).
c. Tie-break: keep the earliest ID.

**SCOPE — DEDUPLICATION ONLY:** Apart from explicit user directives in the Generation Instructions above, never drop a use case for business value, quality or portfolio size; those are scored later on an absolute scale, and the final count is never a target here. A use case leaves this step only as the weaker copy of a duplicate.

**RULES**: Keep the stronger copy of each duplicate pair using the resolution order above (earliest ID only as the final tie-break). Reviewing ALL {total_count} use cases in one pass.

Here is the markdown table of ALL use cases to analyze:
{use_case_markdown}

**OUTPUT**: CSV listing IDs to KEEP, one kept ID per row. No markdown fences, no prose: do not write the layer analysis, notes, bullet points or explanations of merged pairs anywhere in the response; only the CSV rows are read.

Your response must START with: use_case_id,honesty_score,honesty_justification
""" + HONESTY_CHECK_CSV

# --- 1g-2. COMBINED VALUE + QUALITY SCORING PROMPT (SINGLE PASS) ---
PROMPT_TEMPLATES["COMBINED_VALUE_QUALITY_SCORE_PROMPT"] = """# COMBINED VALUE & QUALITY SCORER

## YOUR ROLE

You are BOTH a **Chief Investment Officer** (scoring business value and ROI) AND a **Chief Data Scientist** (scoring data quality, schema validity, hallucination detection). You will produce BOTH value scores AND quality scores for each use case in a SINGLE pass.

---

## CONTEXT & INPUTS

**Business Context:** {business_context}
**Industry:** {industry}

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}

**Strategic Initiative:** {strategic_initiative}
**Value Chain:** {value_chain}
**Revenue Model:** {revenue_model}

""" + ASSUMED_CONTEXT_RULE + """

---

## AVAILABLE SCHEMA (THE ONLY DATA THAT EXISTS)

{schema_details}

---

## USE CASES TO SCORE

{use_cases_to_validate}

---

## PART A: VALUE SCORING

For each use case, score these factors (1.0 - 5.0):

**VALUE FACTORS:**
1. **Return on Investment (ROI)** [Weight: 80% of Value]: Judge against the Revenue Model from evidence, never from invented multipliers. 4.8-5.0 = the output changes a recurring decision that directly moves a revenue, margin, cost or loss line of the Revenue Model, and the moved quantity is computable from named columns; 4.0-4.7 = directly moves a P&L or risk-exposure KPI computable from named columns, but the decision is occasional or the link to the Revenue Model is one step removed; 3.0-3.9 = improves an efficiency or quality KPI whose money effect is indirect; 1.0-2.9 = soft benefit (visibility, awareness) with no KPI computable from the schema.
2. **Time to Value** [Weight: 10%]: 4.8-5.0=value visible within 1 month, 3.0-4.7=within 1-3 months, 1.0-2.9=more than 3 months.
3. **Reusability** [Weight: 10%]: 4.8-5.0=Platform Asset, 3.0-4.7=Modular, 1.0-2.9=One-Off.

Value is scored on the data and the business context only. Strategic alignment is assessed in a later step, after selection.

**FEASIBILITY FACTORS** (average of all 8):
4. Data Availability, 5. Data Accessibility, 6. Architecture Fitness, 7. Team Skills,
8. Domain Knowledge, 9. People Allocation, 10. Budget Allocation, 11. Time to Production.

**FEASIBILITY SCORING GUIDANCE (critical for unbiased scoring):**
- **Architecture Fitness** (4.8-5.0 = Native): All Databricks-native capabilities are "Native": this includes SQL, PySpark, scikit-learn, XGBoost, LightGBM, Spark MLlib, MLflow, Prophet, AI functions (ai_query, ai_classify, ai_forecast, ai_extract, ai_similarity, ai_gen). Do NOT penalize ML-based approaches; they run natively on Databricks.
- **Team Skills** (4.8-5.0 = Standard): SQL, Python, scikit-learn, XGBoost, standard ML/statistical modeling, and Databricks AI functions are all "Standard" data team skills. Do NOT assume ML requires specialized talent that SQL doesn't.

**FORMULAS:**
- Value = (ROI * 0.80) + (TTV * 0.10) + (Reusability * 0.10)
- Feasibility = Average of factors 4-11
- Priority = (Value * 1.5) + (Feasibility * 0.5)

**VALUE SCORING RULES:**
- NO CURVE / NO DISTRIBUTION: Score based on Absolute Merit.
- ZERO-BASED SCORING: Start every score at 1.0, earn points by evidence.
- IRRELEVANT CORRELATIONS = LOW SCORE (ROI <= 2.0).
- BOARDROOM TEST: Would a senior executive approve budget for this?

---

## PART B: QUALITY SCORING

""" + QUALITY_GATE_SCORING_BLOCK + """

**METADATA-GROUNDED EVALUATION (grounding dimensions D1–D7):**
You already see the complete AVAILABLE SCHEMA above: every table, every column name, every column type. When evaluating grounding:
- IF all columns referenced by the UC appear in AVAILABLE SCHEMA -> GROUNDED.
- IF the analytical technique is compatible with the column TYPES visible in the schema (numeric columns support aggregation/regression; string columns support grouping/text analysis; date columns support time-series) -> GROUNDED.
- DO NOT lower grounding because the user did not explicitly say "yes this column is reliable" or "yes this technique is supported." Columns + types are sufficient evidence of support.
- DO NOT lower grounding for "uncertainty about data availability" if the column is present in the schema; presence in the schema IS availability.
- Grounding fails only for EXPLICIT hallucination (UC references a column or table NOT present in schema) or EXPLICIT user exclusion (construct the user said doesn't exist). Depth, uniqueness, activation and the D12–D19 business gates are judged by their own rules above.

**COLUMN HALLUCINATION DETECTION (MANDATORY):**
- VERIFY every column referenced in the High Level Design exists in the schema.
- AUTOMATIC LOW quality score (<=2.0) if ANY column is hallucinated.
- State "HALLUCINATION DETECTED: Column [X] missing" or "NO HALLUCINATION: All columns verified."

---

## OUTPUT FORMAT

Return a **JSON array** with one object per use case. Each object MUST contain ALL these fields:

```json
[
  {{
    "use_case_id": "N01-AI01",
    "use_case_name": "Predict Customer Churn",

    "roi": 4.8,
    "reusability": 4.0,
    "time_to_value": 3.5,
    "data_availability": 4.0,
    "data_accessibility": 4.0,
    "architecture_fitness": 4.5,
    "team_skills": 4.0,
    "domain_knowledge": 4.0,
    "people_allocation": 4.5,
    "budget_allocation": 4.0,
    "time_to_production": 3.5,
    "value_score": 4.59,
    "feasibility_score": 4.06,
    "priority_score": 8.92,
    "justification": "Directly prevents churn in top-tier accounts, protecting recurring revenue.",

    "quality_score": 4.2,
    "quality_label": "Very High",
    "quality_summary": "Churn signal derived from `last_login_date` and `transaction_date` columns; computing days-since-last-activity per customer yields an engagement_decay metric. Joining with `subscription.plan_type` and `billing.monthly_amount` enables revenue-at-risk quantification per churn-probability tier. Aggregating by `customer_segment` produces segment-level churn-risk KPIs. Combining engagement_decay with revenue-at-risk directly powers the promised churn prediction, prioritizing retention efforts on highest-value accounts.",
    "hallucination_check": "NO HALLUCINATION: All columns verified to exist in schema",
    "strengths": "Rich historical data, direct causal indicators present.",
    "weaknesses": "Minor granularity limitations in time-series data.",
    "d1_causal_signal": 4.5,
    "d2_cause_effect": 4.0,
    "d3_granularity": 4.5,
    "d4_dimensions": 4.0,
    "d5_logical": 4.5,
    "d6_metric": 4.0,
    "d7_design_match": 3.5,
    "d8_semantic_uniqueness": 5.0,
    "d9_analytical_depth": 4.5,
    "d10_activation_quality": 4.0,
    "d11_domain_balance": 5.0
  }}
]
```

---

## CRITICAL RULES

1. **SCORE EVERY SINGLE USE CASE** - output one JSON object per input use case. Missing = CRITICAL FAILURE.
2. **OBJECTIVE QUALITY SCORING**: Score quality based on DATA REALITY, not appeal. If a column is not in the schema, it DOES NOT EXIST.
3. **NO ASSUMPTIONS**: Do not assume data exists that is not in the schema.
4. **VALUE-FIRST PRIORITY**: The Priority formula mathematically forces high-value cases to outrank high-feasibility-only cases.
5. **JUSTIFICATION must be SPECIFIC to the use case** - no generic buzzwords.
6. **JSON FORMATTING**: Double quotes only, no trailing commas, no line breaks inside strings.
7. **HONEST D8-D10 SCORING**: Do NOT inflate D8-D10 scores. If you see semantic duplicates, trivial analyses, technical method suffixes, or domain jargon in names, score them LOW (1.0-2.0). These dimensions exist specifically because previous scoring inflated quality scores on use cases that should have been rejected. D10 specifically checks that the "with" suffix is a business DELIVERABLE (recommendation, action plan, queue, strategy), NOT a technical method, data source, or analytical technique.
8. **DUPLICATE DETECTION IS MANDATORY**: For D8, you MUST compare each use case against ALL others in the batch using the 5 layers of D8, and give D8 = 1.0 only to the weaker copy of a duplicate pair.
9. **PUNCTUATION**: """ + NO_EM_DASH_RULE + """

Begin your JSON response now:
"""


log_print("PROMPT_TEMPLATES dictionary defined successfully with all required prompts.")

# COMMAND ----------

# --- Global Logger ---
# This will be configured by the DatabricksInspire class
logger = logging.getLogger(__name__)

# --- Custom Exceptions ---
class InputTooLongError(RuntimeError):
    """Raised when input exceeds the model's context limit."""
    pass

class CascadeRebatchError(RuntimeError):
    """Raised when a model in the cascade timed out and the prompt is too large for remaining models.
    Signals the caller to re-batch with smaller chunks that fit the target model's context window."""
    def __init__(self, message, target_model_name=None, target_model_endpoint=None,
                 target_context_limit_chars=None, failed_model_name=None):
        super().__init__(message)
        self.target_model_name = target_model_name
        self.target_model_endpoint = target_model_endpoint
        self.target_context_limit_chars = target_context_limit_chars
        self.failed_model_name = failed_model_name

class TruncatedResponseError(RuntimeError):
    """Raised when LLM response is truncated (missing END marker)."""
    pass

class LLMEndpointUnusableError(RuntimeError):
    pass

# ==============================================================================
# CENTRALIZED DATA STRUCTURES (Maximizing Reuse & Reducing LOC)
# ==============================================================================

# ==============================================================================
# 1. REQUIRED HELPER FUNCTIONS
# (Dependencies for AIAgent and DatabricksInspire)
# ==============================================================================

### --- Logging ---

class ConsoleErrorFormatter(logging.Formatter):
    """A custom formatter that logs error messages but not stack traces to the console."""
    def format(self, record):
        original_exc_info = record.exc_info
        original_exc_text = record.exc_text
        if record.levelno >= logging.ERROR:
            record.exc_info = None
            record.exc_text = None
        formatted_message = super().format(record)
        record.exc_info = original_exc_info
        record.exc_text = original_exc_text
        return formatted_message

class FlushingStreamHandler(logging.StreamHandler):
    """StreamHandler that flushes after every emit for Databricks notebook compatibility."""
    def emit(self, record):
        super().emit(record)
        self.flush()

LIBRARY_LOG_NAMESPACES = ("urllib3", "databricks.sdk", "fontTools", "weasyprint", "matplotlib", "PIL", "mlflow", "py4j")


def setup_logging(output_dir):
    """Configures dual logging: detailed logs to a file and high-level logs to the console.

    B17 (v1.0.6): returns the actual output_dir used. On Databricks Serverless UI
    standard-access-mode, /tmp/ file creation is restricted despite makedirs succeeding;
    when FileHandler(mode='w') raises EACCES/PermissionError, we fall back to a fresh
    tempfile.mkdtemp() dir (mkstemp+O_EXCL requests proper perms) and return that path
    so the caller stores it as self.local_log_output_dir — required so downstream code
    (volume log streamer at ~line 17486) tails the correct file."""
    log_file_path = os.path.join(output_dir, "log.txt")
    try:
        os.makedirs(output_dir, exist_ok=True)
    except (PermissionError, OSError):
        pass  # re-raised implicitly below if we can't write anywhere
    # B14: defensive stale-file removal — prior partial init on a warm-reused Serverless container can leave log.txt with restrictive perms, causing EACCES on FileHandler mode='w' when session_id is user-supplied and reused across re-runs.
    if os.path.isfile(log_file_path):
        try:
            os.chmod(log_file_path, 0o644)
        except OSError:
            pass
        try:
            os.unlink(log_file_path)
        except OSError:
            pass
        # B18 (v1.0.7): verify unlink actually happened. On Databricks Serverless the stale file
        # can have ACLs beyond POSIX perms; the unlink above silently returns yet the file
        # survives, and FileHandler(mode='w') then crashes on the O_TRUNC. Fall back to
        # tempfile.mkdtemp() immediately with a distinct marker so the operator sees the
        # stale-file cause (not just the downstream EACCES).
        if os.path.isfile(log_file_path):
            import tempfile as _b18_tmp_verify
            _b18_verify_fallback = _b18_tmp_verify.mkdtemp(prefix="inspire_fallback_")
            print(f"[B18] Stale log file at {log_file_path} could not be removed (Serverless ACL). Falling back to {_b18_verify_fallback}.")
            output_dir = _b18_verify_fallback
            log_file_path = os.path.join(output_dir, "log.txt")
    try:
        os.chmod(output_dir, 0o755)
    except OSError:
        pass
    # B18 (v1.0.7): belt-and-suspenders probe open — try open('w') BEFORE FileHandler runs.
    # Narrow catch (PermissionError only) preserves IsADirectoryError propagation for
    # programmer bugs (v1.0.5 B14 test_C invariant); falls back to tempfile.mkdtemp() on EACCES.
    try:
        _b18_probe = open(log_file_path, 'w')
        _b18_probe.close()
    except PermissionError as _b18_probe_err:
        import tempfile as _b18_tmp_probe
        _b18_probe_fallback = _b18_tmp_probe.mkdtemp(prefix="inspire_fallback_")
        print(f"[B18] Cannot write log at {log_file_path} ({_b18_probe_err}). Falling back to {_b18_probe_fallback}.")
        output_dir = _b18_probe_fallback
        log_file_path = os.path.join(output_dir, "log.txt")
    root_logger = logging.getLogger() # Get root logger
    root_logger.setLevel(logging.DEBUG)

    if root_logger.hasHandlers():
        root_logger.handlers.clear()

    # --- File Handler (Detailed) ---
    # B17: try primary /tmp path; on PermissionError (Databricks Serverless UI
    # standard-access-mode restricts /tmp file creation even when makedirs succeeds),
    # fall back to tempfile.mkdtemp() which uses mkstemp+O_EXCL to request proper
    # permissions. Narrower than catching OSError so non-perm failures (e.g. a directory
    # at the log path — programmer bug — or ENOSPC) still surface loudly.
    file_handler = None
    try:
        file_handler = logging.FileHandler(log_file_path, mode='w')
    except PermissionError as _b17_primary_err:
        try:
            import tempfile as _b17_tmp
            _b17_fallback_dir = _b17_tmp.mkdtemp(prefix="inspire_log_")
            log_file_path = os.path.join(_b17_fallback_dir, "log.txt")
            output_dir = _b17_fallback_dir  # B17: propagate fallback path back to caller
            file_handler = logging.FileHandler(log_file_path, mode='w')
            print(f"[B17-FALLBACK] Primary /tmp log path failed ({_b17_primary_err}); using {log_file_path}")
        except (PermissionError, OSError) as _b17_fallback_err:
            # Ultimate fallback: console-only logging. Downstream file-log consumers (volume
            # streamer, _upload_log_file) will find no source; they already tolerate missing files.
            print(f"[B17-FALLBACK-CONSOLE-ONLY] All file-log paths failed. Primary: {_b17_primary_err}. Fallback: {_b17_fallback_err}. Continuing with console logging only.")
            file_handler = None

    if file_handler is not None:
        file_handler.setLevel(logging.DEBUG)
        file_formatter = logging.Formatter(
            '%(asctime)s - %(name)s - %(levelname)s - %(message)s [in %(pathname)s:%(lineno)d]', 
            datefmt='%Y-%m-%d %H:%M:%S'
        )
        file_handler.setFormatter(file_formatter)
        root_logger.addHandler(file_handler)

    # --- Console Handler (High-Level, Clean) ---
    console_handler = FlushingStreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_formatter = ConsoleErrorFormatter(
        '%(asctime)s - %(levelname)s - %(message)s', 
        datefmt='%H:%M:%S'
    )
    console_handler.setFormatter(console_formatter)
    root_logger.addHandler(console_handler)
    for library in LIBRARY_LOG_NAMESPACES:
        logging.getLogger(library).setLevel(logging.WARNING)
    
    logger.info(f"Logging configured. High-level logs to console, detailed logs to {log_file_path}")
    return output_dir

def _ensure_inspire_database_ready(spark_session, inspire_database, log_print_fn=None, get_error_msg_fn=None):
    """B19 (v1.0.7): widget-validation-phase pre-flight for the inspire_database schema.

    Fails EARLY (before any DatabricksInspire instantiation, before ~60-90s of pipeline
    warm-up) with a clean actionable message when the target catalog/schema is missing.
    Prior behavior: the pipeline ran ~70s of useful work then crashed at
    finalize_atomic_writer with `TABLE_OR_VIEW_NOT_FOUND` on `__inspire_step` when the
    schema didn't exist — the operator saw a downstream table error instead of the real
    root cause (missing schema).

    Args:
        spark_session: live SparkSession.
        inspire_database: 'catalog.schema' string; empty string is a no-op.
        log_print_fn: optional (msg: str) -> None. Falls back to builtin print.
        get_error_msg_fn: optional (exc: Exception) -> str. Falls back to str(exc).

    Returns:
        (ok: bool, validation_error_msg: str or None). ok=True means the schema is ready
        for downstream `CREATE TABLE IF NOT EXISTS`. ok=False means widget validation must
        add the returned message to validation_errors and dbutils.notebook.exit.
    """
    _b19_log = log_print_fn if callable(log_print_fn) else (lambda m: print(m))
    _b19_err = get_error_msg_fn if callable(get_error_msg_fn) else (lambda e: str(e))
    if not inspire_database or not str(inspire_database).strip():
        return (True, None)
    inspire_database = str(inspire_database).strip()
    if inspire_database.count('.') != 1:
        return (True, None)  # format validation lives upstream; don't double-report
    _b19_cat, _b19_sch = inspire_database.split('.', 1)
    _b19_cat, _b19_sch = _b19_cat.strip(), _b19_sch.strip()
    if not _b19_cat or not _b19_sch:
        return (True, None)
    # Step 1: is the schema already there?
    try:
        spark_session.sql(f"DESCRIBE SCHEMA `{_b19_cat}`.`{_b19_sch}`").collect()
        return (True, None)
    except Exception:
        pass  # schema does not exist OR catalog missing — try to create
    # Step 2: try to auto-create the schema (idempotent).
    try:
        spark_session.sql(f"CREATE SCHEMA IF NOT EXISTS `{_b19_cat}`.`{_b19_sch}`")
        _b19_log(f"[B19] Auto-created schema {inspire_database} (did not previously exist).")
        return (True, None)
    except Exception as _b19_create_err:
        _b19_msg = _b19_err(_b19_create_err)
        _b19_msg_lower = _b19_msg.lower()
        # Distinguish catalog-missing (user must create catalog) from other create failures.
        if ("catalog_not_found" in _b19_msg_lower
                or f"catalog '{_b19_cat.lower()}'" in _b19_msg_lower
                or ("catalog" in _b19_msg_lower and "not found" in _b19_msg_lower)
                or ("catalog" in _b19_msg_lower and "does not exist" in _b19_msg_lower)):
            return (False,
                f"❌ 'Inspire Database' ({inspire_database}): catalog '{_b19_cat}' does not exist on "
                f"this workspace. Please create it manually (CREATE CATALOG {_b19_cat}) or set "
                f"02_inspire_database to a catalog you already have access to. Root cause: {_b19_msg[:300]}")
        return (False,
            f"❌ 'Inspire Database' ({inspire_database}): schema does not exist and could not be "
            f"created. Root cause: {_b19_msg[:300]}. Please create the schema manually "
            f"(CREATE SCHEMA {inspire_database}) or set 02_inspire_database to an existing schema.")

def print_ascii_banner():
    """Prints the Databricks Inspire AI ASCII art banner."""
    print(DATABRICKS_INSPIRE_BANNER)
    print(f"[AGENT_VERSION FIRED] {__AGENT_VERSION__}")

def extract_honesty_score(response: str, logger: logging.Logger = None) -> tuple:
    """
    Extracts the honesty score and justification from an LLM response.
    Supports multiple formats: JSON wrapper, CSV columns, SQL comments.
    
    Args:
        response: The raw LLM response text
        logger: Optional logger for debug output
    
    Returns:
        tuple: (score: int or None, justification: str or None, cleaned_response: str)
               - score: The honesty score (0-100) or None if not found
               - justification: The justification text (max 250 chars) or None if not found
               - cleaned_response: The response with honesty data removed for downstream processing
    """
    import re
    import json as json_module
    
    if not response:
        return None, None, response
    
    score = None
    justification = None
    cleaned_response = response
    
    try:
        response_stripped = response.strip()
        
        json_candidate = response_stripped
        if not json_candidate.startswith('{'):
            fence_match = re.search(r'```(?:\w+)?\s*([\s\S]*?)```', json_candidate)
            if fence_match:
                inner = fence_match.group(1).strip()
                if inner.startswith('{'):
                    json_candidate = inner
        
        if json_candidate.startswith('{') and '"honesty_score"' in json_candidate:
            try:
                parsed = json_module.loads(json_candidate)
                if isinstance(parsed, dict) and 'honesty_score' in parsed:
                    score = int(parsed.get('honesty_score', 0))
                    justification = str(parsed.get('honesty_justification', ''))[:250]
                    if 'data' in parsed:
                        cleaned_response = json_module.dumps(parsed['data'], ensure_ascii=False)
                    else:
                        cleaned_parsed = {k: v for k, v in parsed.items() 
                                         if k not in ('honesty_score', 'honesty_justification')}
                        cleaned_response = json_module.dumps(cleaned_parsed, ensure_ascii=False)
            except json_module.JSONDecodeError:
                pass
        
        if score is None and response_stripped.startswith('--'):
            sql_score_pattern = r'^--\s*HONESTY_SCORE:\s*(\d+)'
            sql_just_pattern = r'^--\s*HONESTY_JUSTIFICATION:\s*(.+?)$'
            
            lines = response_stripped.split('\n')
            cleaned_lines = []
            for line in lines:
                score_match = re.match(sql_score_pattern, line.strip())
                if score_match:
                    score = int(score_match.group(1))
                    continue
                just_match = re.match(sql_just_pattern, line.strip())
                if just_match:
                    justification = just_match.group(1).strip()[:250]
                    continue
                cleaned_lines.append(line)
            cleaned_response = '\n'.join(cleaned_lines)
        
        structured = score is not None
        if score is None and ('honesty_score' in response_stripped.lower() or 'honesty_justification' in response_stripped.lower()):
            import csv
            from io import StringIO

            def _csv_cells(line):
                return [c.strip().strip('"\'`').strip().lower() for c in line.split(',')]

            lines = response_stripped.split('\n')
            header_idx = next((i for i, line in enumerate(lines)
                               if len(_csv_cells(line)) > 1
                               and {'honesty_score', 'honesty_justification'} & set(_csv_cells(line))), None)
            if header_idx is not None:
                try:
                    rows = list(csv.reader(StringIO('\n'.join(lines[header_idx:]))))
                    header = [h.strip().strip('"\'`').strip().lower() for h in rows[0]]
                    score_idx = header.index('honesty_score') if 'honesty_score' in header else None
                    just_idx = header.index('honesty_justification') if 'honesty_justification' in header else None
                    if score_idx is not None and len(rows) > 1:
                        try:
                            score = int(rows[1][score_idx])
                        except (ValueError, IndexError):
                            pass
                    if just_idx is not None and len(rows) > 1:
                        try:
                            justification = str(rows[1][just_idx])[:250]
                        except IndexError:
                            pass
                    if not CSVParser.has_open_quote('\n'.join(lines[header_idx:])):
                        output = StringIO()
                        csv.writer(output).writerows(
                            [CSVParser.strip_honesty(row, len(rows[0]), score_idx, just_idx) for row in rows])
                        cleaned_response = '\n'.join(lines[:header_idx] + [output.getvalue().strip()])
                    structured = True
                except Exception:
                    pass
        
        if not structured and '|' in response_stripped and 'honesty_score' in response_stripped.lower():
            lines = response_stripped.split('\n')
            header_line = None
            header_idx = -1
            for idx, line in enumerate(lines):
                if '|' in line and 'honesty_score' in line.lower():
                    header_line = line
                    header_idx = idx
                    break
            
            if header_line:
                structured = True
                cells = [c.strip().strip('"').lower() for c in header_line.split('|')]
                score_idx = None
                just_idx = None
                for i, cell in enumerate(cells):
                    if 'honesty_score' in cell:
                        score_idx = i
                    elif 'honesty_justification' in cell:
                        just_idx = i
                
                if score_idx is not None and header_idx + 2 < len(lines):
                    data_line = lines[header_idx + 2] if lines[header_idx + 1].replace('|', '').replace('-', '').strip() == '' else lines[header_idx + 1]
                    data_cells = [c.strip().strip('"') for c in data_line.split('|')]
                    
                    if score_idx < len(data_cells):
                        try:
                            score = int(data_cells[score_idx])
                        except ValueError:
                            pass
                    if just_idx is not None and just_idx < len(data_cells):
                        justification = data_cells[just_idx][:250]
                    
                    cleaned_lines = []
                    for line in lines:
                        if '|' in line:
                            parts = line.split('|')
                            new_parts = [p for i, p in enumerate(parts) if i != score_idx and i != just_idx]
                            cleaned_lines.append('|'.join(new_parts))
                        else:
                            cleaned_lines.append(line)
                    cleaned_response = '\n'.join(cleaned_lines)
        
        if score is not None:
            if score < 0:
                score = 0
            elif score > 100:
                score = 100
        
        _honesty_free_text_markers = [
            'honesty self-assessment',
            'honesty_score',
            'honesty score',
            'honesty_justification',
            'honesty justification',
        ]
        _score_trailer_line = re.compile(r'^\W*score\W*\d')
        _lines = cleaned_response.split('\n') if not structured else []
        _first_marker_idx = None
        for _idx in range(len(_lines) - 1, max(0, len(_lines) - 30) - 1, -1):
            _line_stripped = _lines[_idx].strip()
            if not _line_stripped:
                continue
            _is_csv_data = _line_stripped.startswith('"') and '","' in _line_stripped
            if _is_csv_data:
                break
            _line_lower = _line_stripped.lower()
            if _score_trailer_line.match(_line_lower) or any(_marker in _line_lower for _marker in _honesty_free_text_markers):
                _first_marker_idx = _idx
        if _first_marker_idx is not None and '\n'.join(_lines[:_first_marker_idx]).strip():
            cleaned_response = '\n'.join(_lines[:_first_marker_idx]).rstrip()

    except Exception as e:
        if logger:
            logger.debug(f"Failed to extract honesty score: {e}")
        cleaned_response = response
    
    return score, justification, cleaned_response

### --- AIAgent Dependencies ---
# (Assumed to be available for the AIAgent class)

def execute_sql(spark: SparkSession, query: str, logger: logging.Logger, args: dict = None):
    """
    Executes a Spark SQL query and returns the collected rows.
    Supports parameterized queries via args dict to avoid SQL injection / escaping issues.
    """
    try:
        logger.debug(f"Executing Spark SQL: {query[:200]}...")
        if args:
            result = spark.sql(query, args=args).collect()
        else:
            result = spark.sql(query).collect()
        return result
    except Exception as e:
        logger.debug(f"Spark SQL query failed: {get_clean_error_message(e)}")
        raise

def _ai_query_rows(spark_session, endpoint, prompt_text, max_output_tokens, temperature=None, response_format=None, logger=None,
                   log_attempt=None):
    if temperature is None:
        _model_params_sql = f"named_struct('max_tokens', {max_output_tokens})"
    else:
        _model_params_sql = f"named_struct('max_tokens', {max_output_tokens}, 'temperature', CAST({temperature} AS DOUBLE))"
    ai_query_args = {"prompt_text": prompt_text}
    if response_format:
        ai_query_args["resp_fmt"] = response_format
        ai_query_sql = f"SELECT ai_query('{endpoint}', :prompt_text, responseFormat => :resp_fmt, modelParameters => {_model_params_sql}) AS ai_response"
    else:
        ai_query_sql = f"SELECT ai_query('{endpoint}', :prompt_text, modelParameters => {_model_params_sql}) AS ai_response"
    try:
        rows = execute_sql(spark_session, ai_query_sql, logger or logging.getLogger(__name__), args=ai_query_args)
    except Exception as error:
        if log_attempt is not None:
            log_attempt.settle(ailog_status_for_error(error), get_clean_error_message(error, max_chars=AILOGS_ERROR_CHARS))
        raise
    if log_attempt is not None:
        first = rows[0] if rows else None
        log_attempt.settle("success", str(getattr(first, "ai_response", "") or "") if first is not None else "")
    return rows


_AILOG_CALL = threading.local()


def _ailog_now() -> str:
    return datetime.datetime.now(datetime.timezone.utc).isoformat()


def _ailog_tokens(text) -> int:
    return int(round(len(text or "") / AILOGS_CHARS_PER_TOKEN))


def ailog_status_for_error(error) -> str:
    return "timedout" if _classify_runtime_error(error) == "timeout" else "failed"


def ailog_new_request(prompt_name: str) -> dict:
    return {"request_id": uuid.uuid4().hex, "attempts": 0, "succeeded": False, "prompt": str(prompt_name or "")}


class AILogAttempt:
    """One statement sent to an endpoint. It settles exactly once: the transport settles success or
    failed, the caller settles timedout, whichever happens first writes the row."""

    def __init__(self, request: dict, endpoint: str, request_text: str, max_tokens=None, temperature=None,
                 response_format=None):
        request["attempts"] += 1
        sent = {"max_tokens": max_tokens, "response_format": response_format, "temperature": temperature}
        text = str(request_text or "")
        self.request = request
        self.row = {"ailogs_id": uuid.uuid4().hex, "request_id": request["request_id"], "session_id": None,
                    "attempt": request["attempts"], "agent": AILOGS_AGENT, "prompt": request["prompt"],
                    "endpoint": str(endpoint or ""), "temperature": None if temperature is None else float(temperature),
                    "parameters": json.dumps({k: v for k, v in sent.items() if v is not None}, sort_keys=True, default=str),
                    "request": text, "request_length": len(text), "request_tokens": _ailog_tokens(text),
                    "request_hash": hashlib.sha256(text.encode("utf-8")).hexdigest(), "start_time": _ailog_now()}
        self._lock = threading.Lock()
        self._settled = False

    def settle(self, status: str, response_text) -> bool:
        with self._lock:
            if self._settled:
                return False
            self._settled = True
        text = str(response_text or "")
        self.row.update({"response": text, "response_length": len(text), "response_tokens": _ailog_tokens(text),
                         "status": status, "end_time": _ailog_now()})
        if status == "success":
            self.request["succeeded"] = True
        AILOG_RECORDER.record(self.row)
        return True


class AILogRecorder:
    """Process-wide sink for _ailogs rows. Recording never raises and never blocks an LLM call: rows
    wait in a bounded buffer until a run attaches its writer (the endpoint probes run before the
    session exists), then go to the writer's spool or the JSONL fallback."""

    MAX_PENDING = 5000

    def __init__(self):
        self._lock = threading.Lock()
        self._pending = []
        self._sink = None
        self.session_id = None
        self.dropped = 0
        self.keys = Counter()
        self.requests = {}
        self.success_by_prompt = Counter()
        self.probe_endpoints = {}
        self.probe_requests = set()

    def attach(self, session_id, sink):
        with self._lock:
            self.session_id = session_id
            self._sink = sink
            pending, self._pending = self._pending, []
        for row in pending:
            self._deliver(row)

    def detach(self):
        with self._lock:
            self._sink = None

    def record(self, row: dict):
        try:
            with self._lock:
                self.keys[(row["request_id"], row["attempt"])] += 1
                self.requests.setdefault(row["request_id"], []).append(row["status"])
                if row["status"] == "success":
                    self.success_by_prompt[row["prompt"]] += 1
                if row["prompt"] == "ENDPOINT_PROBE":
                    self.probe_requests.add(row["request_id"])
                    if row["status"] in ("success", "exhausted"):
                        self.probe_endpoints.setdefault(row.get("probe_endpoint") or row["endpoint"], row["status"])
                if self._sink is None:
                    if len(self._pending) < self.MAX_PENDING:
                        self._pending.append(row)
                    else:
                        self.dropped += 1
                    return
            self._deliver(row)
        except Exception:
            self.dropped += 1

    def record_exhausted(self, request: dict, request_text: str, error=None, endpoint_hint: str = ""):
        text = str(request_text or "")
        note = AILOGS_EXHAUSTED_NOTE + (f" ({get_clean_error_message(error, max_chars=300)})" if error is not None else "")
        now = _ailog_now()
        self.record({"ailogs_id": uuid.uuid4().hex, "request_id": request["request_id"], "session_id": None,
                     "attempt": request["attempts"] + 1, "agent": AILOGS_AGENT, "prompt": request["prompt"],
                     "endpoint": "", "temperature": None, "parameters": "{}", "request": text,
                     "request_length": len(text), "request_tokens": _ailog_tokens(text), "response": note,
                     "response_length": len(note), "response_tokens": _ailog_tokens(note), "status": "exhausted",
                     "start_time": now, "end_time": now, "request_hash": hashlib.sha256(text.encode("utf-8")).hexdigest(),
                     "probe_endpoint": endpoint_hint})

    def _deliver(self, row: dict):
        try:
            delivered = {name: row.get(name) for name, _kind in AILOGS_COLUMNS if name != "server_ingested_at"}
            if delivered["session_id"] is None:
                delivered["session_id"] = self.session_id
            self._sink(delivered)
        except Exception:
            self.dropped += 1

    def self_check(self, expected_success_by_prompt: dict = None, bound_endpoints=()) -> dict:
        with self._lock:
            keys, requests = dict(self.keys), {k: list(v) for k, v in self.requests.items()}
            success, probes = dict(self.success_by_prompt), dict(self.probe_endpoints)
            probe_ids = set(self.probe_requests)
        rows = sum(keys.values())
        non_success = sum(1 for statuses in requests.values() for s in statuses if s in AILOGS_NON_SUCCESS)
        call_statuses = [s for rid, statuses in requests.items() if rid not in probe_ids for s in statuses]
        call_non_success = sum(1 for s in call_statuses if s in AILOGS_NON_SUCCESS)
        checks = {
            "unique_request_attempt": all(count == 1 for count in keys.values()),
            "statuses_closed_set": all(s in AILOGS_STATUSES for statuses in requests.values() for s in statuses),
            "attempt_at_least_one": all(attempt >= 1 for _rid, attempt in keys),
            "every_request_terminates": all(("success" in statuses) or statuses[-1] == "exhausted"
                                            for statuses in requests.values()),
            "attempts_cover_calls": all(success.get(prompt, 0) >= int(count or 0)
                                        for prompt, count in (expected_success_by_prompt or {}).items()),
            "every_endpoint_probed": all(endpoint in probes for endpoint in bound_endpoints or ()),
            "non_success_rate": (call_non_success / len(call_statuses)) <= 0.25 if call_statuses else True,
        }
        return {"rows": rows, "requests": len(requests), "non_success": non_success,
                "probe_non_success": non_success - call_non_success, "dropped": self.dropped,
                "checks": checks, "passed": sum(1 for ok in checks.values() if ok), "total": len(checks)}


AILOG_RECORDER = AILogRecorder()

def load_and_format_prompt(prompt_key: str, prompt_vars: dict, logger: logging.Logger) -> str:
    try:
        # Check if the prompt_key (variable name) exists in the global scope
        if prompt_key not in globals():
            raise NameError(f"Global prompt variable '{prompt_key}' not found. Please make sure the cell defining it has been run.")
            
        # Get the template string from the global variable
        template = globals()[prompt_key]
        
        if not template or not isinstance(template, str):
             raise ValueError(f"Global prompt variable '{prompt_key}' is empty or not a string.")

        # Format the template using the provided dictionary
        return template.format(**prompt_vars)
    except KeyError as e:
        logger.error(f"Missing key in prompt vars for '{prompt_key}': {get_clean_error_message(e)}")
        # Re-raise with more context
        raise KeyError(f"Missing formatting key {e} for prompt '{prompt_key}'")
    except Exception as e:
        logger.error(f"Failed to load or format prompt for '{prompt_key}': {get_clean_error_message(e)}")
        raise

def clean_csv_response(raw_string: str, strip_preamble: bool = True) -> str:
    """
    Removes markdown code fences AND strips prose preamble from a CSV response.
    """
    if not raw_string: return ""
    cleaned = raw_string.strip()
    cleaned = re.sub(r'^```(?:csv|json)?\s*\n?', '', cleaned, flags=re.IGNORECASE | re.MULTILINE)
    cleaned = re.sub(r'\n?```\s*$', '', cleaned, flags=re.IGNORECASE | re.MULTILINE)
    cleaned = re.sub(r'```(?:csv|json)?\s*$', '', cleaned, flags=re.IGNORECASE | re.MULTILINE)
    cleaned = cleaned.strip()
    if not strip_preamble:
        return cleaned
    # Strip prose preamble (NEW). Find first line that looks like CSV header/row
    # and drop everything before it. Heuristics: short first token, has a comma,
    # not starting with a prose-signal word, total line length <400.
    lines = cleaned.split("\n")
    prose_signal = re.compile(r"^(?:Looking|I\s|Let me|First|Total|Analysis|Here|Note|The following|"
                              r"Based on|Given|Since|Actually|Now|Well|After|Before|In this|"
                              r"Using|Applying|To summar|Summary:|Notes?:|\*\*)",
                              re.IGNORECASE)
    csv_start_idx = None
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        if prose_signal.match(stripped):
            continue
        if "," not in stripped:
            continue
        if len(stripped) > 400 and not CSVParser.header_like(stripped):
            continue
        first_tok = stripped.split(",", 1)[0].strip().strip("\"\'")
        if len(first_tok) > 60:
            continue
        if " " in first_tok and len(first_tok.split()) > 3:
            continue
        csv_start_idx = i
        break
    if csv_start_idx is not None and csv_start_idx > 0:
        cleaned = "\n".join(lines[csv_start_idx:])
    return cleaned.strip()


def sanitize_llm_special_chars(text: str) -> str:
    if not text:
        return text
    replacements = {
        '\u2011': '-',   # non-breaking hyphen
        '\u2010': '-',   # hyphen
        '\u2012': '-',   # figure dash
        '\u2013': '-',   # en dash
        '\u2014': '--',  # em dash
        '\u2015': '--',  # horizontal bar
        '\u2018': "'",   # left single quote
        '\u2019': "'",   # right single quote
        '\u201A': "'",   # single low-9 quote
        '\u201B': "'",   # single high-reversed-9 quote
        '\u201C': '"',   # left double quote
        '\u201D': '"',   # right double quote
        '\u201E': '"',   # double low-9 quote
        '\u201F': '"',   # double high-reversed-9 quote
        '\u2026': '...', # ellipsis
        '\u00A0': ' ',   # non-breaking space
        '\u202F': ' ',   # narrow no-break space
        '\u2009': ' ',   # thin space
        '\u200B': '',    # zero-width space
        '\u200C': '',    # zero-width non-joiner
        '\u200D': '',    # zero-width joiner
        '\uFEFF': '',    # byte order mark
        '\u00AB': '"',   # left-pointing double angle quote
        '\u00BB': '"',   # right-pointing double angle quote
        '\u2039': "'",   # single left-pointing angle quote
        '\u203A': "'",   # single right-pointing angle quote
        '\u02BC': "'",   # modifier letter apostrophe
        '\u2032': "'",   # prime
        '\u2033': '"',   # double prime
        '\u00B7': '.',   # middle dot
        '\u2022': '-',   # bullet
        '\u2023': '-',   # triangular bullet
        '\u25E6': '-',   # white bullet
        '\u2043': '-',   # hyphen bullet
        '\u00D7': 'x',   # multiplication sign
        '\u2212': '-',   # minus sign
        '\u00B1': '+/-', # plus-minus sign
        '\u2264': '<=',  # less-than or equal
        '\u2265': '>=',  # greater-than or equal
        '\u2260': '!=',  # not equal
        '\u00AE': '(R)', # registered sign
        '\u2122': '(TM)',# trademark
        '\u00A9': '(C)', # copyright
        '\u20AC': 'EUR', # euro sign
        '\u00A3': 'GBP', # pound sign
        '\u00A5': 'JPY', # yen sign
        '\u20B9': 'INR', # rupee sign
        '\u00BD': '1/2', # vulgar fraction one half
        '\u00BC': '1/4', # vulgar fraction one quarter
        '\u00BE': '3/4', # vulgar fraction three quarters
        '\u00B2': '^2',  # superscript two
        '\u00B3': '^3',  # superscript three
        '\u00B9': '^1',  # superscript one
    }
    for char, replacement in replacements.items():
        text = text.replace(char, replacement)
    return text


def _iter_balanced_json_objects(text):
    objects = []
    depth = 0
    start = None
    in_string = False
    escaped = False
    for i, ch in enumerate(text or ""):
        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    objects.append(text[start:i + 1])
                    start = None
    return objects


def clean_json_response(raw_string: str) -> str:
    """
    Removes markdown code fences and other noise from a raw LLM response.
    Also extracts JSON object/array if extra text is present before or after.
    Renamed from clean_llm_response to match AIAgent dependency.
    """
    if not raw_string: return ""
    
    cleaned = raw_string.strip()
    
    # Remove markdown code fences - handle multiple patterns
    # Pattern 1: ```json\n{...}\n``` or ```\n{...}\n```
    cleaned = re.sub(r'^```(?:json|csv)?\s*', '', cleaned, flags=re.IGNORECASE | re.MULTILINE)
    cleaned = re.sub(r'\s*```\s*$', '', cleaned, flags=re.IGNORECASE | re.MULTILINE)
    
    # Pattern 2: Handle cases where ``` appears in the middle (trailing after JSON)
    cleaned = re.sub(r'```(?:json|csv)?\s*$', '', cleaned, flags=re.IGNORECASE | re.MULTILINE)
    
    cleaned = cleaned.strip()
    
    # Try to extract JSON object or array from the response
    # Look for the first { or [ and the last matching } or ]
    start_obj = cleaned.find('{')
    start_arr = cleaned.find('[')
    
    # Determine which comes first
    if start_obj == -1 and start_arr == -1:
        return cleaned  # No JSON found, return as-is
    elif start_obj == -1:
        start = start_arr
        end_char = ']'
    elif start_arr == -1:
        start = start_obj
        end_char = '}'
    else:
        start = min(start_obj, start_arr)
        end_char = '}' if start == start_obj else ']'
    
    # Find the last occurrence of the closing character
    # Use a more robust method to find the matching closing brace/bracket
    end = -1
    depth = 0
    open_char = '{' if end_char == '}' else '['
    in_string = False
    escaped = False
    
    # Find the matching closing character by tracking depth
    for i in range(start, len(cleaned)):
        ch = cleaned[i]
        if in_string:
            if escaped:
                escaped = False
            elif ch == '\\':
                escaped = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == open_char:
            depth += 1
        elif ch == end_char:
            depth -= 1
            if depth == 0:
                end = i
                break
    
    # Fallback to rfind if depth tracking doesn't work
    if end == -1:
        end = cleaned.rfind(end_char)
    
    if start != -1 and end != -1 and end > start:
        # Extract the JSON portion
        json_portion = cleaned[start:end+1]
        return json_portion
    
    return cleaned

def retry_with_logging(func, max_attempts=1, logger=None, fallback=None, context=""):
    """
    Generic retry wrapper for functions that may fail transiently.
    
    Args:
        func: Callable to execute (should take no arguments; use lambda if needed)
        max_attempts: Maximum number of retry attempts (default: 3)
        logger: Logger instance for logging retries (optional)
        fallback: Fallback value or callable to return on failure (optional)
        context: Context string for logging (e.g., "Domain consolidation for English")
    
    Returns:
        Result of func() on success, fallback on failure (if provided), otherwise raises
    
    Raises:
        Last exception if all attempts fail and no fallback provided
    """
    last_exception = None
    for attempt in range(1, max_attempts + 1):
        try:
            if attempt > 1 and logger:
                logger.info(f"Retry attempt {attempt}/{max_attempts}{f' for {context}' if context else ''}...")
            return func()
        except Exception as e:
            last_exception = e
            if logger:
                error_msg = get_clean_error_message(e)
                if attempt == max_attempts:
                    logger.error(f"All {max_attempts} attempts failed{f' for {context}' if context else ''}: {error_msg}")
                else:
                    logger.warning(f"Attempt {attempt}/{max_attempts} failed{f' for {context}' if context else ''}: {error_msg}")
            if attempt == max_attempts:
                if fallback is not None:
                    if callable(fallback):
                        return fallback()
                    return fallback
                raise last_exception

# ==============================================================================
# MEMORY MANAGEMENT UTILITIES
# ==============================================================================

def _force_gc(logger=None, reason: str = ""):
    """Force garbage collection and log memory stats."""
    gc.collect()
    try:
        import psutil
        proc = psutil.Process(os.getpid())
        mem_mb = proc.memory_info().rss / (1024 * 1024)
        if logger:
            logger.info(f"🧹 GC forced{f' ({reason})' if reason else ''}: RSS={mem_mb:.0f} MB")
    except ImportError:
        if logger:
            logger.debug(f"🧹 GC forced{f' ({reason})' if reason else ''}")


# ==============================================================================
# CENTRALIZED UTILITY CLASSES (Code Reuse & LOC Reduction)
# ==============================================================================

class RetryHandler:
    """
    Centralized retry handler with exponential backoff and flexible error handling.
    Replaces all scattered retry logic throughout the codebase.
    """
    
    @staticmethod
    def execute_with_retry(
        func,
        max_attempts=1,
        logger=None,
        context="",
        fallback=None,
        exponential_backoff=True,
        base_delay=1.0,
        max_delay=60.0,
        retryable_errors=None,
        non_retryable_errors=None
    ):
        """
        Execute a function with retry logic and exponential backoff.
        
        Args:
            func: Callable to execute
            max_attempts: Maximum retry attempts (default: 1)
            logger: Logger instance for tracking
            context: Context string for logging
            fallback: Fallback value on failure
            exponential_backoff: Use exponential backoff (default: True)
            base_delay: Base delay in seconds (default: 1.0)
            max_delay: Maximum delay between retries (default: 60.0)
            retryable_errors: List of error types/keywords that should be retried
            non_retryable_errors: List of error types/keywords that should NOT be retried
            
        Returns:
            Result of func() on success, fallback on failure
        """
        import time
        last_exception = None
        
        for attempt in range(1, max_attempts + 1):
            try:
                if attempt > 1 and logger:
                    logger.info(f"🔄 Retry attempt {attempt}/{max_attempts}{f' for {context}' if context else ''}...")
                return func()
            except Exception as e:
                last_exception = e
                error_str = str(e).lower()
                
                if non_retryable_errors:
                    is_non_retryable = any(
                        (isinstance(err, type) and isinstance(e, err)) or 
                        (isinstance(err, str) and err.lower() in error_str)
                        for err in non_retryable_errors
                    )
                    if is_non_retryable:
                        if logger:
                            logger.error(f"❌ Non-retryable error{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
                        if fallback is not None:
                            return fallback() if callable(fallback) else fallback
                        raise
                
                is_retryable = True
                if retryable_errors:
                    is_retryable = any(
                        (isinstance(err, type) and isinstance(e, err)) or 
                        (isinstance(err, str) and err.lower() in error_str)
                        for err in retryable_errors
                    )
                
                if not is_retryable or attempt == max_attempts:
                    if logger:
                        logger.error(f"❌ Failed after {attempt} attempt{'s' if attempt > 1 else ''}{f' for {context}' if context else ''}: "
                                     f"{get_clean_error_message(e)}")
                    if fallback is not None:
                        return fallback() if callable(fallback) else fallback
                    raise
                
                if exponential_backoff:
                    wait_time = min(base_delay * (2 ** (attempt - 1)), max_delay)
                    jitter = random.uniform(0, wait_time * 0.1)
                    wait_time += jitter
                else:
                    wait_time = base_delay
                
                if logger:
                    logger.warning(f"⚠️  Attempt {attempt} failed{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
                    logger.info(f"   Waiting {wait_time:.1f}s before retry...")
                
                time.sleep(wait_time)
        
        if fallback is not None:
            return fallback() if callable(fallback) else fallback
        raise last_exception


_THREAD_CANCEL_REGISTRY = {}
_THREAD_CANCEL_LOCK = threading.Lock()

def _current_thread_cancel_events():
    with _THREAD_CANCEL_LOCK:
        return tuple(_THREAD_CANCEL_REGISTRY.get(threading.current_thread().ident, ()))

def _register_thread_cancel(event, inherited=()):
    chain = tuple(e for e in ((event,) + tuple(inherited or ())) if e is not None)
    with _THREAD_CANCEL_LOCK:
        _THREAD_CANCEL_REGISTRY[threading.current_thread().ident] = chain

def _unregister_thread_cancel():
    with _THREAD_CANCEL_LOCK:
        _THREAD_CANCEL_REGISTRY.pop(threading.current_thread().ident, None)

def _is_thread_cancelled():
    with _THREAD_CANCEL_LOCK:
        chain = _THREAD_CANCEL_REGISTRY.get(threading.current_thread().ident, ())
    return any(ev.is_set() for ev in chain)

def _cancellable_sleep(seconds, poll_interval=1.0):
    import time as _time_mod
    deadline = _time_mod.time() + max(0.0, float(seconds))
    while True:
        if _is_thread_cancelled():
            return True
        remaining = deadline - _time_mod.time()
        if remaining <= 0:
            return False
        _time_mod.sleep(min(poll_interval, remaining))


class SparkWorkTimeout(TimeoutError):
    def __init__(self, label, elapsed, timeout_seconds, tag):
        super().__init__(f"[{label}] Spark work timed out after {elapsed:.1f}s (budget {timeout_seconds}s, tag {tag})")
        self.elapsed = elapsed
        self.timeout_seconds = timeout_seconds
        self.tag = tag


class SparkWorkCancelled(RuntimeError):
    def __init__(self, label, tag):
        super().__init__(f"[{label}] Spark work abandoned because its parent task was cancelled (tag {tag})")
        self.tag = tag


def _interrupt_spark_tag(spark_session, tag, label, reason, logger=None) -> bool:
    try:
        spark_session.interruptTag(tag)
    except AttributeError:
        if logger is not None:
            logger.debug(f"[{label}] spark.interruptTag not available (non-Spark-Connect runtime); {reason} work keeps running until it ends")
        return False
    except Exception as _interrupt_err:
        if logger is not None:
            logger.debug(f"[{label}] interruptTag failed: {str(_interrupt_err)[:100]}")
        return False
    if logger is not None:
        logger.info(f"🛑 [{label}] Sent interruptTag for {reason} Spark work (tag: {tag})")
    return True


def _run_tagged_spark_work(spark_session, work, timeout_seconds=None, tag_prefix="inspire", label="spark",
                           on_wait=None, logger=None, poll_seconds=1.0):
    import time as _time_mod
    tag = f"{re.sub(r'[^A-Za-z0-9_]', '_', str(tag_prefix or 'inspire'))}_{uuid.uuid4().hex}"
    outcome = {}
    done = threading.Event()

    def _target():
        tagged = False
        try:
            try:
                spark_session.addTag(tag)
                tagged = True
            except Exception:
                pass
            outcome["value"] = work()
        except Exception as exc:
            outcome["error"] = exc
        finally:
            if tagged:
                try:
                    spark_session.removeTag(tag)
                except Exception:
                    pass
            done.set()

    worker = threading.Thread(target=_target, name=f"SparkWork_{tag[:40]}", daemon=True)
    started = _time_mod.time()
    worker.start()
    while not done.wait(poll_seconds):
        elapsed = _time_mod.time() - started
        if _is_thread_cancelled():
            _interrupt_spark_tag(spark_session, tag, label, "cancelled", logger)
            raise SparkWorkCancelled(label, tag)
        if timeout_seconds and elapsed >= timeout_seconds:
            _interrupt_spark_tag(spark_session, tag, label, "timed-out", logger)
            raise SparkWorkTimeout(label, elapsed, timeout_seconds, tag)
        if on_wait is not None:
            on_wait(elapsed)
    if "error" in outcome:
        raise outcome["error"]
    return outcome.get("value")


_TRANSIENT_ERROR_KINDS = frozenset({"timeout", "throttled", "server"})
_THROTTLE_ERROR_MARKERS = ("throttl", "rate limit", "rate_limit", "too many requests", "quota", "request_limit_exceeded",
                           "resource exhausted", "resource_exhausted", "capacity", "overload")
_TIMEOUT_ERROR_MARKERS = ("timeout", "timed out", "deadline", "time limit")
_SERVER_ERROR_MARKERS = ("internal server", "service unavailable", "bad gateway", "gateway timeout",
                         "temporarily unavailable", "temporarily_unavailable", "statuscode.unavailable",
                         "connection reset", "connection refused", "broken pipe")
_ENDPOINT_PERMISSION_MARKERS = ("permission_denied", "permission denied", "does not have permission",
                                "not authorized to", "is not authorized")
_ENDPOINT_MISSING_MARKERS = ("resource_does_not_exist", "does not exist", "endpoint_not_found", "not found")


_CONTEXT_OVERFLOW_MARKERS = ('input is too long', 'too long for requested model', 'input length', 'exceeds context limit',
                             'context window', 'token limit exceeded', 'maximum context length', 'context_length_exceeded',
                             'prompt is too long')
_NOT_CONTEXT_OVERFLOW_MARKERS = ('does not support the parameter', 'invalid parameter') + _RESPONSE_FORMAT_ERROR_MARKERS


def is_context_overflow_error(error_msg) -> bool:
    text = str(error_msg or "").lower()
    if any(marker in text for marker in _CONTEXT_OVERFLOW_MARKERS):
        return True
    if any(marker in text for marker in _NOT_CONTEXT_OVERFLOW_MARKERS):
        return False
    return ('400' in text or 'bad_request' in text or 'bad request' in text) and (
        'input' in text or 'length' in text or 'model' in text)


def _classify_runtime_error(error) -> str:
    if isinstance(error, SparkWorkCancelled):
        return "cancelled"
    if isinstance(error, SparkWorkTimeout):
        return "timeout"
    text = str(error or "").lower()
    if "operation_canceled" in text or "operation has been canceled" in text or "cascade aborted" in text:
        return "cancelled"
    if inspire_temperature_rejected(text):
        return "temperature_unsupported"
    throttled = any(m in text for m in _THROTTLE_ERROR_MARKERS) or re.search(r"\b429\b", text) is not None
    if "not supported for batch inference" in text:
        return "endpoint_unusable"
    if "endpoint" in text and not throttled and any(m in text for m in _ENDPOINT_PERMISSION_MARKERS + _ENDPOINT_MISSING_MARKERS):
        return "endpoint_unusable"
    if throttled:
        return "throttled"
    if any(m in text for m in _TIMEOUT_ERROR_MARKERS):
        return "timeout"
    if any(m in text for m in _SERVER_ERROR_MARKERS) or re.search(r"\b50[0234]\b", text) is not None:
        return "server"
    return "other"


WAIT_HEARTBEAT_LOG_SECONDS = 60


def _log_wait_heartbeat(log, label, done, total, in_progress, elapsed, timeout):
    budget = f"of {timeout:.0f}s" if timeout else "with no time limit"
    (log or logger).info(f"[WAIT-HEARTBEAT] {label}: {done}/{total} done, {in_progress} in progress, "
                         f"{elapsed:.0f}s elapsed {budget}")


def safe_as_completed(futures, timeout=None, label="Parallel tasks", log=None):
    """Drop-in replacement for concurrent.futures.as_completed() that keeps
    the main thread active via a polling loop. On Databricks, the notebook
    output buffer only flushes when the main thread executes Python code.

    Uses manual future.done() polling + time.sleep() instead of
    concurrent.futures.wait(). The latter uses C-level Condition.wait()
    which can block far beyond its timeout when the GIL is contended by
    worker threads executing C-extensions (e.g. PySpark JNI calls via
    Py4J). time.sleep() properly releases the GIL and always returns on
    schedule, guaranteeing the main thread stays responsive.
    """
    import time as _time_mod
    import sys as _sys_mod

    if isinstance(futures, dict):
        future_set = set(futures.keys())
    else:
        future_set = set(futures)

    total = len(future_set)
    remaining = set(future_set)
    start_time = _time_mod.time()
    heartbeat_interval = 10
    sleep_interval = 2
    completed_count = 0
    last_heartbeat = start_time
    last_logged = start_time

    while remaining:
        newly_done = {f for f in remaining if f.done()}
        remaining -= newly_done

        if newly_done:
            for future in newly_done:
                completed_count += 1
                yield future
            _sys_mod.stdout.flush()
            _sys_mod.stderr.flush()
            continue

        elapsed = _time_mod.time() - start_time
        if timeout is not None and elapsed >= timeout:
            raise concurrent.futures.TimeoutError(
                f"safe_as_completed timed out after {timeout:.0f}s with {len(remaining)} futures remaining"
            )
        else:
            now = _time_mod.time()
            if now - last_heartbeat >= heartbeat_interval:
                timeout_str = f"{timeout:.0f}" if timeout is not None else "∞"
                timestamp = _time_mod.strftime('%H:%M:%S', _time_mod.localtime())
                _sys_mod.stdout.write(f"{timestamp} - INFO - ⏳ [{completed_count}/{total}] {len(remaining)} task(s) in progress ({elapsed:.0f}s / {timeout_str}s)\n")
                last_heartbeat = now
            if now - last_logged >= WAIT_HEARTBEAT_LOG_SECONDS:
                _log_wait_heartbeat(log, label, completed_count, total, len(remaining), now - start_time, timeout)
                last_logged = now
            _sys_mod.stdout.flush()
            _sys_mod.stderr.flush()
            remaining_timeout = (timeout - elapsed) if timeout is not None else sleep_interval
            _time_mod.sleep(min(sleep_interval, max(0.1, remaining_timeout)))


def _safe_future_result(future, timeout=None, poll_interval=2):
    """Get a future's result with guaranteed timeout enforcement.

    Replaces direct future.result(timeout=X) calls which use C-level
    Condition.wait() that can block beyond the timeout when the GIL is
    contended (e.g. PySpark JNI calls). This polls future.done() +
    time.sleep() instead, both of which behave correctly under GIL pressure.
    """
    import time as _time_mod
    import sys as _sys_mod

    if future.done():
        return future.result(timeout=0)

    start = _time_mod.time()
    while not future.done():
        elapsed = _time_mod.time() - start
        if timeout is not None and elapsed >= timeout:
            raise concurrent.futures.TimeoutError(
                f"_safe_future_result timed out after {timeout:.0f}s"
            )
        remaining = (timeout - elapsed) if timeout is not None else poll_interval
        _time_mod.sleep(min(poll_interval, max(0.1, remaining)))
        _sys_mod.stdout.flush()
        _sys_mod.stderr.flush()

    return future.result(timeout=0)


class _SafeExecutorContext:
    """
    Wraps a ThreadPoolExecutor with safe shutdown on timeout.
    Call mark_timed_out() when a timeout occurs so __exit__ uses
    shutdown(wait=False, cancel_futures=True) instead of blocking forever.
    """
    __slots__ = ('_executor', '_timed_out', '_logger', '_name', '_cancel_event', '_futures', '_task_events')

    def __init__(self, max_workers, thread_name_prefix="Worker", logger=None, name=""):
        self._executor = ThreadPoolExecutor(max_workers=max_workers, thread_name_prefix=thread_name_prefix)
        self._timed_out = False
        self._logger = logger
        self._name = name or thread_name_prefix
        self._cancel_event = threading.Event()
        self._futures = []
        self._task_events = {}

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.shutdown(abandon=exc_type is not None)
        return False

    def shutdown(self, abandon: bool = False):
        _pending = [f for f in self._futures if not f.done()]
        _abandon = self._timed_out or abandon or bool(_pending)
        if _abandon:
            self._cancel_event.set()
            if self._logger:
                self._logger.warning(
                    f"🛑 [{self._name}] Forcing executor shutdown (wait=False) and cancelling "
                    f"{len(_pending)} in-flight task(s)"
                )
        self._executor.shutdown(wait=False, cancel_futures=_abandon)

    def completed(self, futures, timeout=None):
        return safe_as_completed(futures, timeout=timeout, label=self._name, log=self._logger)

    def submit(self, fn, *args, **kwargs):
        _inherited = (self._cancel_event,) + tuple(_current_thread_cancel_events())
        _task_event = threading.Event()

        def _run_with_cancel_scope():
            _register_thread_cancel(_task_event, _inherited)
            try:
                return fn(*args, **kwargs)
            finally:
                _unregister_thread_cancel()

        future = self._executor.submit(_run_with_cancel_scope)
        self._futures.append(future)
        self._task_events[future] = _task_event
        return future

    def cancel(self, future) -> bool:
        _task_event = self._task_events.get(future)
        if _task_event is None:
            return False
        _task_event.set()
        future.cancel()
        return True

    def mark_timed_out(self):
        self._timed_out = True
        self._cancel_event.set()


class ParallelExecutor:
    """
    Centralized parallel execution manager using ThreadPoolExecutor.
    Replaces all scattered ThreadPoolExecutor usage throughout the codebase.
    """
    
    @staticmethod
    def execute_parallel(
        tasks,
        max_workers,
        task_name="Task",
        logger=None,
        timeout_per_task=None,
        total_timeout=None,
        thread_name_prefix="Worker",
        return_exceptions=False
    ):
        """
        Execute multiple tasks in parallel using ThreadPoolExecutor.
        
        Uses a polling loop instead of as_completed() to keep the main thread
        active. This is critical on Databricks where the notebook output buffer
        only flushes when the main thread executes Python code. Without polling,
        worker thread output (logs, heartbeats) accumulates invisibly while the
        main thread is blocked in as_completed()'s C-level Condition.wait().
        
        Args:
            tasks: List of callables or (callable, args) tuples
            max_workers: Maximum number of parallel workers
            task_name: Name for logging purposes
            logger: Logger instance
            timeout_per_task: Timeout per individual task in seconds
            total_timeout: Total timeout for all tasks in seconds
            thread_name_prefix: Prefix for thread names
            return_exceptions: If True, return exceptions instead of raising them
            
        Returns:
            List of results (or exceptions if return_exceptions=True)
        """
        import time as _time_mod
        import sys as _sys_mod
        import traceback as _tb_mod
        
        results = []
        exceptions = []
        poll_interval = 15
        
        cancel_events = {}
        task_start_times = {}
        inherited_cancel = _current_thread_cancel_events()

        def _safe_wrapper(idx, fn, fn_args, _cancel_evt=None):
            task_start_times[idx] = _time_mod.time()
            if _cancel_evt is not None:
                _register_thread_cancel(_cancel_evt, inherited_cancel)
            try:
                return fn(*fn_args) if fn_args else fn()
            except Exception as exc:
                _sys_mod.stderr.write(f"[ParallelExecutor] {task_name} #{idx} FAILED: {exc}\n{_tb_mod.format_exc()}")
                _sys_mod.stderr.flush()
                raise
            finally:
                if _cancel_evt is not None:
                    _unregister_thread_cancel()
        
        timed_out = False
        executor = ThreadPoolExecutor(max_workers=max_workers, thread_name_prefix=thread_name_prefix)
        try:
            future_to_task = {}
            for i, task in enumerate(tasks):
                evt = threading.Event()
                cancel_events[i] = evt
                if isinstance(task, tuple):
                    func, args = task
                    future = executor.submit(_safe_wrapper, i, func, args, evt)
                else:
                    future = executor.submit(_safe_wrapper, i, task, None, evt)
                future_to_task[future] = i
            
            remaining = set(future_to_task.keys())
            completed_indices = set()
            start_time = _time_mod.time()
            last_heartbeat = start_time
            last_logged = start_time
            heartbeat_interval = poll_interval
            sleep_interval = 0.5
            
            while remaining:
                done = {f for f in remaining if f.done()}
                remaining -= done
                
                for future in done:
                    task_idx = future_to_task[future]
                    completed_indices.add(task_idx)
                    try:
                        result = future.result()
                        results.append((task_idx, result))
                    except concurrent.futures.TimeoutError:
                        error_msg = f"{task_name} #{task_idx} timed out"
                        if logger:
                            logger.warning(f"⏱️  {error_msg}")
                        if return_exceptions:
                            results.append((task_idx, TimeoutError(error_msg)))
                        else:
                            exceptions.append((task_idx, TimeoutError(error_msg)))
                    except Exception as e:
                        if logger:
                            logger.warning(f"❌ {task_name} #{task_idx} failed: {get_clean_error_message(e)}")
                        if return_exceptions:
                            results.append((task_idx, e))
                        else:
                            exceptions.append((task_idx, e))
                if not remaining:
                    break
                
                elapsed = _time_mod.time() - start_time
                parent_cancelled = any(evt.is_set() for evt in inherited_cancel)
                
                if parent_cancelled or (total_timeout and elapsed >= total_timeout):
                    timed_out_count = len(remaining)
                    if parent_cancelled:
                        reason = "was cancelled because its parent task was cancelled"
                        msg = f"🛑 {task_name}: parent task cancelled. Cancelling {timed_out_count} task(s) that did not complete."
                    else:
                        reason = f"exceeded total timeout of {total_timeout}s"
                        msg = f"⏱️  Total timeout ({total_timeout}s) exceeded for {task_name}. {timed_out_count} task(s) did not complete."
                    if logger:
                        logger.error(msg)
                    else:
                        log_print(msg, level="ERROR")
                    timed_out = True
                    for evt in cancel_events.values():
                        evt.set()
                    if return_exceptions:
                        for future in remaining:
                            task_idx = future_to_task[future]
                            results.append((task_idx, TimeoutError(f"{task_name} #{task_idx} {reason}")))
                            future.cancel()
                    else:
                        raise concurrent.futures.TimeoutError(msg)
                    break
                
                if timeout_per_task:
                    now = _time_mod.time()
                    expired = [
                        f for f in remaining
                        if not f.done()
                        and future_to_task[f] in task_start_times
                        and now - task_start_times[future_to_task[f]] >= timeout_per_task
                    ]
                    for future in expired:
                        task_idx = future_to_task[future]
                        cancel_events[task_idx].set()
                        remaining.discard(future)
                        completed_indices.add(task_idx)
                        error_msg = f"{task_name} #{task_idx} exceeded per-task timeout of {timeout_per_task}s"
                        if logger:
                            logger.warning(f"⏱️  {error_msg}; cancelling its in-flight work")
                        if return_exceptions:
                            results.append((task_idx, TimeoutError(error_msg)))
                        else:
                            exceptions.append((task_idx, TimeoutError(error_msg)))
                
                if remaining:
                    now = _time_mod.time()
                    if now - last_heartbeat >= heartbeat_interval:
                        elapsed = _time_mod.time() - start_time
                        timeout_str = f"{total_timeout:.0f}" if total_timeout else "∞"
                        log_print(f"⏳ [{task_name}] {len(completed_indices)}/{len(future_to_task)} done, {len(remaining)} in progress ({elapsed:.0f}s / {timeout_str}s)")
                        last_heartbeat = now
                    if now - last_logged >= WAIT_HEARTBEAT_LOG_SECONDS:
                        _log_wait_heartbeat(logger, task_name, len(completed_indices), len(future_to_task), len(remaining),
                                            now - start_time, total_timeout)
                        last_logged = now
                    _sys_mod.stdout.flush()
                    _sys_mod.stderr.flush()
                    if not done:
                        remaining_timeout = (total_timeout - elapsed) if total_timeout else sleep_interval
                        _time_mod.sleep(min(sleep_interval, max(0.1, remaining_timeout)))
        finally:
            if timed_out:
                if logger:
                    logger.warning(f"🛑 [{task_name}] Forcing executor shutdown (wait=False) after timeout to prevent hang")
                else:
                    log_print(f"🛑 [{task_name}] Forcing executor shutdown (wait=False) after timeout to prevent hang", level="WARNING")
            executor.shutdown(wait=False, cancel_futures=timed_out)
        
        results.sort(key=lambda x: x[0])
        
        if not return_exceptions and exceptions:
            if logger:
                logger.error(f"❌ {len(exceptions)} {task_name}(s) failed")
            raise exceptions[0][1]
        
        return [r[1] for r in results]


csv.field_size_limit(min(sys.maxsize, 2147483647))


class CSVParser:
    """
    Centralized CSV parsing utility with consistent error handling.
    Replaces all scattered csv.DictReader usage throughout the codebase.
    """
    
    @staticmethod
    def parse_csv_string(
        csv_data,
        logger=None,
        context="",
        quoting=csv.QUOTE_ALL,
        delimiter=',',
        skipinitialspace=True,
        expected_fields=None
    ):
        """
        Parse CSV string into list of dictionaries.
        
        Args:
            csv_data: CSV string data
            logger: Logger instance
            context: Context string for logging
            quoting: CSV quoting mode (default: QUOTE_ALL)
            delimiter: Field delimiter (default: ',')
            skipinitialspace: Skip initial spaces (default: True)
            expected_fields: Optional list of expected field names for validation
            
        Returns:
            List of dictionaries (one per row)
        """
        if not csv_data or not csv_data.strip():
            if logger:
                logger.warning(f"⚠️  Empty CSV data{f' for {context}' if context else ''}")
            return []
        csv_data = csv_data.replace('\x00', '')
        
        try:
            reader = csv.DictReader(
                io.StringIO(csv_data),
                quoting=quoting,
                delimiter=delimiter,
                skipinitialspace=skipinitialspace
            )
            rows = list(reader)
            
            if expected_fields and rows:
                actual_fields_lower = {k.lower().strip(): k for k in rows[0].keys() if k is not None}
                missing_fields = {f for f in expected_fields if f.lower().strip() not in actual_fields_lower}
                if missing_fields and logger:
                    logger.warning(f"⚠️  Missing expected CSV fields{f' for {context}' if context else ''}: {missing_fields}")
            
            if logger:
                logger.debug(f"✅ Parsed {len(rows)} CSV rows{f' for {context}' if context else ''}")
            
            return rows
        except Exception as e:
            if logger:
                logger.error(f"❌ CSV parsing failed{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
            return []
    
    @staticmethod
    def parse_csv_list(
        csv_data,
        logger=None,
        context="",
        quoting=csv.QUOTE_ALL,
        delimiter=',',
        quotechar='"',
        skipinitialspace=True
    ):
        """
        Parse CSV string into list of lists (for non-dictionary CSV).
        
        Args:
            csv_data: CSV string data
            logger: Logger instance
            context: Context string for logging
            quoting: CSV quoting mode (default: QUOTE_ALL)
            delimiter: Field delimiter (default: ',')
            quotechar: Quote character (default: '"')
            skipinitialspace: Skip initial spaces (default: True)
            
        Returns:
            List of lists (one per row)
        """
        if not csv_data or not csv_data.strip():
            if logger:
                logger.warning(f"⚠️  Empty CSV data{f' for {context}' if context else ''}")
            return []
        csv_data = csv_data.replace('\x00', '')
        
        try:
            reader = csv.reader(
                io.StringIO(csv_data),
                delimiter=delimiter,
                quotechar=quotechar,
                quoting=quoting,
                skipinitialspace=skipinitialspace
            )
            rows = list(reader)
            
            if logger:
                logger.debug(f"✅ Parsed {len(rows)} CSV rows{f' for {context}' if context else ''}")
            
            return rows
        except Exception as e:
            if logger:
                logger.error(f"❌ CSV parsing failed{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
            return []

    _HONESTY_SCORE_VALUE = re.compile(r"\s*\d{1,3}(?:\.\d+)?\s*(?:%|/\s*100)?\s*")
    _HONESTY_KINDS = {"honestyscore": "score", "honestyjustification": "justification"}

    @classmethod
    def is_honesty_score(cls, value) -> bool:
        return value is not None and bool(cls._HONESTY_SCORE_VALUE.fullmatch(str(value)))

    @classmethod
    def honesty_kind(cls, column):
        return cls._HONESTY_KINDS.get(re.sub(r"[^a-z]", "", str(column or "").lower()))

    @staticmethod
    def header_like(line: str) -> bool:
        """A wide CSV header: 3 or more fields and none longer than a column name. Long prose with commas is not."""
        try:
            fields = next(csv.reader([line]))
        except (csv.Error, StopIteration):
            return False
        return len(fields) >= 3 and all(0 < len(field.strip()) <= 60 for field in fields)

    @staticmethod
    def anchor_text(value) -> str:
        return str(value if value is not None else "").replace("\ufeff", "").strip().strip("`\"'").strip().casefold()

    @classmethod
    def one_of(cls, values):
        accepted = frozenset(cls.anchor_text(value) for value in values) - {""}
        return lambda value: cls.anchor_text(value) in accepted

    @staticmethod
    def has_open_quote(text) -> bool:
        return str(text or "").count('"') % 2 == 1

    @staticmethod
    def row_dict(header, values) -> dict:
        values = list(values or [])
        row = {str(name): (values[i] if i < len(values) else None) for i, name in enumerate(header)}
        if len(values) > len(header):
            row[None] = values[len(header):]
        return row

    @classmethod
    def strip_honesty(cls, row, width: int, score_idx, just_idx) -> list:
        row = list(row)
        honesty = sorted(i for i in (score_idx, just_idx) if i is not None)
        if not honesty:
            return row
        if honesty != list(range(width - len(honesty), width)):
            return [value for i, value in enumerate(row) if i not in honesty]
        start = honesty[0]
        if len(row) == width:
            return row[:start]
        if score_idx == start and len(row) > width and cls.is_honesty_score(row[start]):
            return row[:start]
        if score_idx == start and len(honesty) == 2 and len(row) == width - 1 and cls.is_honesty_score(row[-1]):
            return row[:start]
        return row

    @classmethod
    def misaligned_row(cls, row: dict, required=(), anchors=None, echoes=None, numeric=()) -> tuple:
        anchors, echoes = anchors or {}, echoes or {}
        columns = [k for k in row if k is not None]
        filled = [(k, row[k]) for k in columns if row[k] is not None]
        score_column = next((k for k in columns if cls.honesty_kind(k) == "score"), None)
        extra = [v for v in (row.get(None) or []) if str(v or "").strip()]
        if extra:
            last = columns[-1] if columns else None
            appendix = ((last in anchors and anchors[last](row.get(last)))
                        or (cls.honesty_kind(last) == "justification" and score_column is not None
                            and cls.is_honesty_score(row.get(score_column)))
                        or (score_column is None and cls.is_honesty_score(extra[0])))
            if not appendix:
                return "extra_values", f"has {len(extra)} extra value(s)"
        short = [k for k in columns if row[k] is None and k in required]
        if short:
            return "short_row", f"is missing {short[:3]}"
        score = row.get(score_column) if score_column is not None else None
        if score is not None and str(score).strip() and not cls.is_honesty_score(score):
            return "honesty_misplaced", f"has a non-numeric {score_column}"
        for column, value in [(k, v) for k, v in filled if not cls.honesty_kind(k)][-2:]:
            if column not in numeric and column not in anchors and cls.is_honesty_score(value):
                return "honesty_misplaced", f"holds a number in '{column}'"
        for anchor, holds in anchors.items():
            if anchor not in row or holds(row[anchor]):
                continue
            moved = [k for k, v in filled if k != anchor and k not in echoes.get(anchor, ()) and holds(v)]
            if moved:
                return "anchor_shifted", f"has the {anchor} value in '{moved[0]}'"
        return "", ""

    @classmethod
    def aligned_rows(cls, rows, logger=None, context="", header=None, required=(), anchors=None, echoes=None,
                     numeric=(), text=None) -> tuple:
        from collections import Counter
        rows = [row for row in (rows or []) if header is None or any(str(cell or "").strip() for cell in row)]
        records = [cls.row_dict(header, row) if header is not None else row for row in rows]
        cut_index = len(records) - 1 if text is not None and cls.has_open_quote(text) else None
        kept, reasons = [], Counter()
        for index, (row, record) in enumerate(zip(rows, records)):
            reason, detail = cls.misaligned_row(record, required, anchors, echoes, numeric)
            if not reason and index == cut_index and cls.cut_in_data(record):
                reason, detail = "cut_final_row", "ends inside an open quoted field"
            if not reason:
                kept.append(row)
                continue
            reasons[reason] += 1
            if logger is not None:
                first = next((str(v) for k, v in record.items() if k is not None and v is not None), "")
                (logger.warning if sum(reasons.values()) <= 3 else logger.debug)(
                    f"[CSV-ROW misaligned] {context}: row {first[:40]!r} {detail}; it is treated as missing")
        cls.log_misaligned(logger, context, reasons, len(rows))
        return kept, reasons

    @classmethod
    def cut_in_data(cls, record: dict) -> bool:
        filled = [k for k, v in record.items() if k is not None and v is not None]
        return bool(filled) and not cls.honesty_kind(filled[-1])

    @classmethod
    def truncation_reason(cls, text, records, record_ids, required, expected_ids) -> str:
        if cls.has_open_quote(text):
            return "the answer ends inside an open quoted field"
        ids = [str(value or "").strip() for value in record_ids]
        if records and ids[-1] and any(expected.startswith(ids[-1]) for expected in expected_ids):
            last = records[-1]
            if any(last.get(column, "") is None for column in required):
                return f"the final row {ids[-1]!r} stops after {sum(1 for k, v in last.items() if k is not None and v is not None)} field(s)"
        absent = [expected for expected in expected_ids if expected not in set(ids)]
        if absent and expected_ids[-len(absent):] == absent:
            return f"the closing row(s) {absent[:3]} never started"
        return ""

    @staticmethod
    def log_misaligned(logger, context, reasons, total) -> None:
        if logger is not None and reasons:
            logger.warning(f"[CSV-ROW misaligned] {context}: rejected {sum(reasons.values())} of {total} row(s) "
                           f"({', '.join(f'{reason}={count}' for reason, count in sorted(reasons.items()))}); "
                           f"they are treated as missing")


class JSONParser:
    """
    Centralized JSON parsing utility with consistent error handling.
    Replaces scattered json.loads/dumps usage throughout the codebase.
    """
    
    @staticmethod
    def safe_loads(json_string, logger=None, context="", fallback=None):
        """
        Safely parse JSON string with error handling.
        
        Args:
            json_string: JSON string to parse
            logger: Logger instance
            context: Context string for logging
            fallback: Fallback value on parsing failure
            
        Returns:
            Parsed JSON object or fallback value
        """
        if not json_string:
            return fallback
        
        try:
            return json.loads(json_string)
        except json.JSONDecodeError as e:
            if logger:
                logger.warning(f"⚠️  JSON parsing failed{f' for {context}' if context else ''}: {str(e)[:100]}")
            return fallback
        except Exception as e:
            if logger:
                logger.error(f"❌ Unexpected error parsing JSON{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
            return fallback
    
    @staticmethod
    def safe_dumps(obj, logger=None, context="", fallback="{}", indent=None, separators=None):
        """
        Safely serialize object to JSON string with error handling.
        
        Args:
            obj: Object to serialize
            logger: Logger instance
            context: Context string for logging
            fallback: Fallback string on serialization failure
            indent: Indentation level (default: None)
            separators: Custom separators (default: None)
            
        Returns:
            JSON string or fallback value
        """
        try:
            if separators:
                return json.dumps(obj, indent=indent, separators=separators)
            return json.dumps(obj, indent=indent)
        except TypeError as e:
            if logger:
                logger.warning(f"⚠️  JSON serialization failed{f' for {context}' if context else ''}: {str(e)[:100]}")
            return fallback
        except Exception as e:
            if logger:
                logger.error(f"❌ Unexpected error serializing JSON{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
            return fallback


class TimeoutHandler:
    """
    Centralized timeout handling utility.
    Provides consistent timeout behavior across the codebase.
    """
    
    @staticmethod
    def execute_with_timeout(func, timeout_seconds, logger=None, context="", fallback=None):
        """
        Execute a function with a timeout.
        
        Args:
            func: Callable to execute
            timeout_seconds: Timeout in seconds
            logger: Logger instance
            context: Context string for logging
            fallback: Fallback value on timeout
            
        Returns:
            Result of func() or fallback on timeout
        """
        from concurrent.futures import TimeoutError as FuturesTimeoutError
        
        executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="Timeout")
        timed_out = False
        cancel_event = threading.Event()
        inherited_cancel = _current_thread_cancel_events()

        def _run_with_cancel_scope():
            _register_thread_cancel(cancel_event, inherited_cancel)
            try:
                return func()
            finally:
                _unregister_thread_cancel()

        try:
            future = executor.submit(_run_with_cancel_scope)
            try:
                result = _safe_future_result(future, timeout=timeout_seconds)
                return result
            except (FuturesTimeoutError, concurrent.futures.TimeoutError):
                timed_out = True
                cancel_event.set()
                future.cancel()
                if logger:
                    logger.warning(f"⏱️  Timeout ({timeout_seconds}s) exceeded{f' for {context}' if context else ''}")
                if fallback is not None:
                    return fallback() if callable(fallback) else fallback
                raise TimeoutError(f"Operation timed out after {timeout_seconds}s{f' for {context}' if context else ''}")
            except Exception as e:
                if logger:
                    logger.error(f"❌ Error during execution{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
                raise
        finally:
            executor.shutdown(wait=False, cancel_futures=timed_out)



# DBTITLE 1,Table Size Analyzer & Dynamic Batch Optimizer

_METADATA_IN_LIST_CHUNK = 200
_NUMERIC_TYPE_RE = re.compile(r"^\s*(tinyint|smallint|integer|int|bigint|long|short|byte|float|double|real|decimal|dec|numeric)\b", re.IGNORECASE)
_TEMPORAL_TYPE_RE = re.compile(r"^\s*(timestamp_ntz|timestamp_ltz|timestamp|datetime|date|time)\b", re.IGNORECASE)
_STRING_TYPE_RE = re.compile(r"^\s*(string|varchar|char|text)\b", re.IGNORECASE)
_KEY_NAME_SUFFIXES = ("_id", "_key", "_sk", "_pk", "_fk", "_nk", "_code", "_no", "_num", "_number", "_ref", "_uuid", "_guid")
_KEY_NAME_PREFIXES = ("id_", "fk_", "sk_", "pk_")
_GENERIC_KEY_NAMES = frozenset({"id", "key", "pk", "sk", "nk", "fk", "code", "no", "num", "number", "ref",
                                "uuid", "guid", "rowid", "row_id", "record_id"})
_CAMEL_KEY_NAME_RE = re.compile(r"[a-z0-9](Id|ID|Key|KEY|Sk|SK|Code|CODE|No|NO|Num|NUM|Number|NUMBER|Ref|REF|Uuid|UUID|Guid|GUID)$")
_CONCAT_KEY_SUFFIXES = ("id", "key", "sk", "code", "num", "number")


def _metadata_type_family(data_type):
    text = str(data_type or "")
    if _NUMERIC_TYPE_RE.match(text):
        return "numeric"
    if _TEMPORAL_TYPE_RE.match(text):
        return "temporal"
    if _STRING_TYPE_RE.match(text):
        return "string"
    if re.match(r"^\s*bool", text, re.IGNORECASE):
        return "boolean"
    return "other"


def _is_key_like_column(name):
    raw = str(name or "").strip()
    low = raw.lower()
    if not low:
        return False
    if low in _GENERIC_KEY_NAMES or low.endswith(_KEY_NAME_SUFFIXES) or low.startswith(_KEY_NAME_PREFIXES):
        return True
    if _CAMEL_KEY_NAME_RE.search(raw):
        return True
    return any(low.endswith(suffix) and len(low) - len(suffix) >= 4 and low[:-len(suffix)].isalpha()
               for suffix in _CONCAT_KEY_SUFFIXES)


def _name_tokens(name):
    spaced = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", str(name or ""))
    return [t for t in re.split(r"[^a-z0-9]+", spaced.lower()) if t]


def _singular_token(token):
    t = str(token or "").lower()
    if len(t) > 4 and t.endswith("ies"):
        return t[:-3] + "y"
    if len(t) > 4 and t.endswith(("sses", "shes", "ches", "xes")):
        return t[:-2]
    if len(t) > 3 and t.endswith("s") and not t.endswith(("ss", "us", "is")):
        return t[:-1]
    return t


def _key_entity_tokens(column_name):
    tokens = _name_tokens(column_name)
    while tokens and tokens[-1] in _GENERIC_KEY_NAMES:
        tokens = tokens[:-1]
    if len(tokens) == 1:
        for suffix in _CONCAT_KEY_SUFFIXES:
            stem = tokens[0][:-len(suffix)]
            if tokens[0].endswith(suffix) and len(stem) >= 4 and stem.isalpha():
                tokens = [stem]
                break
    return [_singular_token(t) for t in tokens]


_ROW_SEQUENCE_TOKENS = frozenset({"line", "seq", "sequence", "position", "ordinal"})
_DIMENSION_TABLE_AFFIXES = ("dim_", "_dim", "dimension_", "_dimension")
_TABLE_ROLE_TOKENS = frozenset({"dim", "dimension", "fact", "fct", "tbl"})


def _is_own_row_key(table_name, column_name, primary_key=(), table_columns=()):
    column = str(column_name or "").strip().lower()
    declared = [str(p).strip().lower() for p in (primary_key or ()) if str(p).strip()]
    if declared:
        return declared == [column]
    if any(set(_name_tokens(other)) & _ROW_SEQUENCE_TOKENS for other in (table_columns or ())
           if str(other).strip().lower() != column):
        return False
    if column in _GENERIC_KEY_NAMES:
        return True
    entity = _key_entity_tokens(column_name)
    table_tokens = [_singular_token(t) for t in _name_tokens(str(table_name or "").rsplit(".", 1)[-1])
                    if t not in _TABLE_ROLE_TOKENS]
    return bool(entity and table_tokens and entity[-1] == table_tokens[-1])


def _sql_in_list_params(values, prefix):
    names = [f"{prefix}{idx}" for idx in range(len(values))]
    return ", ".join(f":{name}" for name in names), dict(zip(names, values))


def _clip_metadata_text(text, limit=None):
    cleaned = " ".join(str(text or "").split()).replace("|", "/")
    if limit is not None and len(cleaned) > limit:
        return cleaned[:max(0, limit - 1)].rstrip() + "…"
    return cleaned


def _parse_describe_rows(rows):
    columns = []
    for row in rows or []:
        name = getattr(row, "col_name", None)
        if not name or str(name).startswith("#"):
            break
        columns.append((str(name), getattr(row, "data_type", None), getattr(row, "comment", None)))
    return columns


_METADATA_QUERY_TIMEOUT_S = 180
_METADATA_QUERY_ATTEMPTS = 3
_METADATA_RETRY_BACKOFF_S = (5, 15)
_METADATA_BATCH_TIMEOUT_S = 600


def _with_metadata_retries(spark_session, work, logger=None, context="metadata query"):
    attempts = max(1, int(_METADATA_QUERY_ATTEMPTS))
    for attempt in range(1, attempts + 1):
        if _is_thread_cancelled():
            raise SparkWorkCancelled(context, "not started")
        try:
            return _run_tagged_spark_work(spark_session, work, timeout_seconds=_METADATA_QUERY_TIMEOUT_S,
                                          tag_prefix="inspire_meta", label=context, logger=logger)
        except SparkWorkCancelled:
            raise
        except Exception as exc:
            kind = _classify_runtime_error(exc)
            if kind not in _TRANSIENT_ERROR_KINDS:
                raise
            if attempt >= attempts:
                raise RuntimeError(f"{kind} after {attempts} attempt(s): {get_clean_error_message(exc, max_chars=300)}") from exc
            delay = _METADATA_RETRY_BACKOFF_S[min(attempt - 1, len(_METADATA_RETRY_BACKOFF_S) - 1)]
            if logger is not None:
                logger.warning(f"[METADATA_QUERY_RETRY FIRED] {context}: {kind} on attempt {attempt}/{attempts} "
                               f"({get_clean_error_message(exc, max_chars=200)}); retrying in {delay}s")
            if _cancellable_sleep(delay):
                raise SparkWorkCancelled(context, "backoff")


def _metadata_rows(spark_session, query, args=None, logger=None, context="metadata query"):
    if args:
        return _with_metadata_retries(spark_session, lambda: spark_session.sql(query, args=args).collect(), logger, context)
    return _with_metadata_retries(spark_session, lambda: spark_session.sql(query).collect(), logger, context)


class StandardModeDataReadError(RuntimeError):
    pass


class DeepEvidenceBudgetSpent(RuntimeError):
    pass


DEEP_PII_NAME_RE = re.compile(
    r"(e[-_]?mail|phone|mobile|msisdn|fax|first[_-]?name|last[_-]?name|full[_-]?name|middle[_-]?name|surname|"
    r"given[_-]?name|maiden|address|street|postcode|post[_-]?code|zip[_-]?code|zipcode|\bssn\b|social[_-]?security|"
    r"national[_-]?id|tax[_-]?id|passport|licen[cs]e[_-]?(no|num|number)|iban|swift|account[_-]?(no|num|number)|"
    r"card[_-]?(no|num|number)|credit[_-]?card|\bcvv\b|birth|\bdob\b|ip[_-]?addr|(^|_)ip($|_)|latitude|longitude|geo[_-]?loc)",
    re.IGNORECASE)
DEEP_PII_TAG_RE = re.compile(r"pii|personal|sensitive|confidential|gdpr|hipaa|\bphi\b|\bpci\b|restricted", re.IGNORECASE)
DEEP_TOP_VALUES = 20
DEEP_LOW_CARDINALITY_MAX = 50
DEEP_VALUE_MAX_CHARS = 80
DEEP_PROFILED_FAMILIES = ("numeric", "temporal", "string", "boolean")
DEEP_KPI_AGGREGATES = ("SUM", "AVG", "COUNT", "COUNT_DISTINCT", "MIN", "MAX")
DEEP_KPI_VALUE_AGGREGATES = ("SUM", "AVG", "MIN", "MAX")
METRIC_VIEW_TABLE_TYPES = ("METRIC_VIEW", "METRIC VIEW")
_PERMISSION_MARKERS = ("permission", "unauthorized", "access denied", "insufficient privilege", "not authorized",
                       "permission_denied")


def deep_column_is_pii(column_name, column_tags=None) -> bool:
    if DEEP_PII_NAME_RE.search(str(column_name or "")):
        return True
    for tag_name, tag_value in (column_tags or {}).items():
        if DEEP_PII_TAG_RE.search(f"{tag_name} {tag_value or ''}"):
            return True
    return False


def _evidence_error_status(error) -> str:
    if isinstance(error, SparkWorkTimeout):
        return "timeout"
    if isinstance(error, SparkWorkCancelled):
        return "cancelled"
    text = str(error or "").lower()
    if any(marker in text for marker in _PERMISSION_MARKERS):
        return "denied"
    return "failed"


class DeepEvidence:
    """The only code that reads customer data. It refuses to exist or run unless the session's
    Discovery Type is Deep; every statement is logged and counted for data_access."""

    def __init__(self, spark_session, discovery_type, logger, runtime=None):
        if canonical_discovery_type(discovery_type) != "Deep":
            raise StandardModeDataReadError("Discovery Type is Standard: Inspire never reads customer data")
        rt = dict(TECHNICAL_CONTEXT["runtime"]) if runtime is None else dict(runtime)
        self.spark = spark_session
        self.discovery_type = "Deep"
        self.logger = logger
        self.budget_seconds = float(rt["deep_evidence_budget_seconds"])
        self.statement_timeout_seconds = float(rt["deep_statement_timeout_seconds"])
        self.max_tables = int(rt["deep_max_tables"])
        self.max_columns = int(rt["deep_max_columns_per_table"])
        self.max_value_columns = int(rt["deep_max_value_columns_per_table"])
        self.max_joins = int(rt["deep_max_joins"])
        self.sample_rows = int(rt["deep_sample_rows"])
        self.history_days = int(rt["deep_history_days"])
        self.history_rows = int(rt["deep_history_rows"])
        self.max_kpi_baselines = int(rt["deep_max_kpi_baselines"])
        self.spent_seconds = 0.0
        self.statements = []
        self.skipped_tables = {}
        self.pii_excluded = set()
        self._lock = threading.Lock()

    def _require_deep(self):
        if self.discovery_type != "Deep":
            raise StandardModeDataReadError("Discovery Type is Standard: Inspire never reads customer data")

    def budget_left(self) -> float:
        return self.budget_seconds - self.spent_seconds

    def _read(self, kind, table, sql, args=None):
        self._require_deep()
        left = self.budget_left()
        if left <= 0:
            raise DeepEvidenceBudgetSpent(f"Deep evidence budget of {self.budget_seconds:.0f}s is spent")
        started = time.time()
        status = "ok"
        rows = None
        try:
            if args:
                work = lambda: self.spark.sql(sql, args=args).collect()
            else:
                work = lambda: self.spark.sql(sql).collect()
            rows = _run_tagged_spark_work(self.spark, work,
                                          timeout_seconds=max(5.0, min(self.statement_timeout_seconds, left)),
                                          tag_prefix="deep_evidence", label=f"DeepEvidence {kind}", logger=self.logger)
            return rows
        except Exception as error:
            status = _evidence_error_status(error)
            raise
        finally:
            seconds = round(time.time() - started, 2)
            tables = [str(t) for t in table] if isinstance(table, (list, tuple)) else [str(table)]
            with self._lock:
                self.spent_seconds += time.time() - started
                self.statements.append({"kind": kind, "tables": tables, "status": status, "seconds": seconds,
                                        "rows": len(rows) if rows is not None else 0})
            self.logger.info(f"[DEEP-EVIDENCE] {kind} on {' & '.join(tables)}: {status} in {seconds}s | {sql[:400]}")

    def _source(self, table_sql, row_count):
        if row_count is not None and row_count > self.sample_rows:
            return f"(SELECT * FROM {table_sql} LIMIT {self.sample_rows})", True
        return table_sql, False

    def profile_table(self, table_sql, table_label, columns, column_tags=None):
        """columns: [(name, data_type)]. Returns aggregates only; values only for non-PII columns
        with at most DEEP_LOW_CARDINALITY_MAX distinct values."""
        column_tags = {str(name).lower(): tags for name, tags in (column_tags or {}).items()}
        profiled = [(name, dtype) for name, dtype in columns
                    if _metadata_type_family(dtype) in DEEP_PROFILED_FAMILIES][:self.max_columns]
        try:
            count_rows = self._read("row_count", table_label, f"SELECT COUNT(*) AS n FROM {table_sql}")
        except (DeepEvidenceBudgetSpent, SparkWorkCancelled):
            raise
        except Exception as error:
            self.skipped_tables[table_label] = _evidence_error_status(error)
            self.logger.warning(f"[DEEP-EVIDENCE SKIPPED] {table_label}: {self.skipped_tables[table_label]} "
                                f"({get_clean_error_message(error, max_chars=200)}); this table stays Standard")
            return None
        row_count = int(count_rows[0]["n"] or 0) if count_rows else 0
        profile = {"rows": row_count, "sampled": False, "columns": {}}
        if row_count == 0 or not profiled:
            return profile
        source, sampled = self._source(table_sql, row_count)
        profile["sampled"] = sampled
        parts = ["COUNT(*) AS `__n`"]
        for index, (name, dtype) in enumerate(profiled):
            column = quote_identifier(name)
            parts.append(f"COUNT({column}) AS `nn{index}`")
            parts.append(f"approx_count_distinct({column}) AS `dc{index}`")
        try:
            stats = self._read("column_stats", table_label, f"SELECT {', '.join(parts)} FROM {source}")
        except (DeepEvidenceBudgetSpent, SparkWorkCancelled):
            raise
        except Exception as error:
            self.logger.warning(f"[DEEP-EVIDENCE] column stats unavailable for {table_label}: "
                                f"{get_clean_error_message(error, max_chars=200)}")
            return profile
        row = stats[0] if stats else None
        sample_n = int(row["__n"] or 0) if row is not None else 0
        temporal = []
        for index, (name, dtype) in enumerate(profiled):
            non_null = int(row[f"nn{index}"] or 0) if row is not None else 0
            distinct = int(row[f"dc{index}"] or 0) if row is not None else 0
            profile["columns"][name] = {"null_share": round(1.0 - (non_null / sample_n), 4) if sample_n else None,
                                        "distinct": distinct, "type_family": _metadata_type_family(dtype)}
            if _metadata_type_family(dtype) == "temporal":
                if deep_column_is_pii(name, column_tags.get(str(name).lower())):
                    self.pii_excluded.add(f"{table_label}.{name}")
                else:
                    temporal.append((index, name))
        if temporal:
            ranges = ", ".join(f"MIN({quote_identifier(name)}) AS `lo{index}`, MAX({quote_identifier(name)}) AS `hi{index}`"
                               for index, name in temporal)
            try:
                bounds = self._read("date_range", table_label, f"SELECT {ranges} FROM {table_sql}")
                if bounds:
                    for index, name in temporal:
                        lo, hi = bounds[0][f"lo{index}"], bounds[0][f"hi{index}"]
                        profile["columns"][name]["min"] = str(lo)[:32] if lo is not None else None
                        profile["columns"][name]["max"] = str(hi)[:32] if hi is not None else None
            except (DeepEvidenceBudgetSpent, SparkWorkCancelled):
                raise
            except Exception as error:
                self.logger.warning(f"[DEEP-EVIDENCE] date range unavailable for {table_label}: "
                                    f"{get_clean_error_message(error, max_chars=200)}")
        value_columns = []
        for name, dtype in profiled:
            info = profile["columns"].get(name) or {}
            if _metadata_type_family(dtype) not in ("string", "boolean"):
                continue
            if not 0 < (info.get("distinct") or 0) <= DEEP_LOW_CARDINALITY_MAX:
                continue
            if deep_column_is_pii(name, column_tags.get(str(name).lower())):
                self.pii_excluded.add(f"{table_label}.{name}")
                continue
            value_columns.append(name)
        for name in value_columns[:self.max_value_columns]:
            column = quote_identifier(name)
            try:
                values = self._read("top_values", table_label,
                                    f"SELECT {column} AS v, COUNT(*) AS c FROM {source} GROUP BY {column} "
                                    f"ORDER BY c DESC LIMIT {DEEP_TOP_VALUES}")
            except (DeepEvidenceBudgetSpent, SparkWorkCancelled):
                raise
            except Exception as error:
                self.logger.warning(f"[DEEP-EVIDENCE] top values unavailable for {table_label}.{name}: "
                                    f"{get_clean_error_message(error, max_chars=200)}")
                continue
            total = sum(int(r["c"] or 0) for r in values) or 1
            profile["columns"][name]["top_values"] = [
                {"value": None if r["v"] is None else str(r["v"])[:DEEP_VALUE_MAX_CHARS],
                 "share": round(int(r["c"] or 0) / total, 4)} for r in values]
        return profile

    def join_match_rate(self, child_sql, child_label, child_col, parent_sql, parent_label, parent_col):
        child_column, parent_column = quote_identifier(child_col), quote_identifier(parent_col)
        sql = (f"SELECT COUNT(*) AS n, COUNT(p.k) AS m FROM "
               f"(SELECT {child_column} AS k FROM {child_sql} WHERE {child_column} IS NOT NULL LIMIT {self.sample_rows}) c "
               f"LEFT JOIN (SELECT DISTINCT {parent_column} AS k FROM {parent_sql}) p ON c.k = p.k")
        rows = self._read("join_match", (child_label, parent_label), sql)
        if not rows:
            return None
        n, m = int(rows[0]["n"] or 0), int(rows[0]["m"] or 0)
        return round(m / n, 4) if n else None

    def query_history_statements(self, catalogs=None):
        names = sorted({str(c).replace("`", "").strip().lower() for c in catalogs or [] if str(c).strip()})
        args = {f"c{index}": f"%{name}.%" for index, name in enumerate(names)}
        scope = " OR ".join(f"lower(statement_text) LIKE :c{index}" for index in range(len(names)))
        sql = ("SELECT substring(statement_text, 1, 16000) AS statement_text FROM system.query.history "
               f"WHERE start_time >= current_timestamp() - INTERVAL {int(self.history_days)} DAYS "
               "AND execution_status = 'FINISHED' AND statement_type = 'SELECT' "
               + (f"AND ({scope}) " if scope else "")
               + f"ORDER BY start_time DESC LIMIT {int(self.history_rows)}")
        rows = self._read("query_history", "system.query.history", sql, args=args or None)
        return [str(r["statement_text"] or "") for r in rows or []]

    @staticmethod
    def canonical_aggregate(aggregate) -> str:
        aggregate = str(aggregate or "").strip().upper().replace(" ", "_")
        if aggregate not in DEEP_KPI_AGGREGATES:
            raise ValueError(f"Unsupported KPI aggregate {aggregate!r}; valid: {', '.join(DEEP_KPI_AGGREGATES)}")
        return aggregate

    @staticmethod
    def kpi_sql(spec: dict) -> str:
        """The only builder of KPI SQL: a read-only aggregate from a validated spec. Standard mode
        emits this text without running it; the LLM never writes SQL that Inspire executes."""
        if spec.get("metric_view"):
            view_sql = quote_qualified_name(spec["metric_view"])
            return f"SELECT CAST(MEASURE({quote_identifier(spec['measure'])}) AS DOUBLE) AS v FROM {view_sql}"
        table_sql = quote_qualified_name(spec["table"])
        denominator = spec.get("denominator")
        conditions = []
        if denominator:
            numerator_sql = DeepEvidence._kpi_aggregate(spec.get("aggregation"), spec.get("column"),
                                                        spec.get("filter_column"), spec.get("filter_value"))
            denominator_sql = DeepEvidence._kpi_aggregate(denominator.get("aggregation"), denominator.get("column"),
                                                          denominator.get("filter_column"), denominator.get("filter_value"))
            scale = " * 100" if str(spec.get("unit") or "").strip().lower() in ("%", "percent") else ""
            expression = f"CAST({numerator_sql} AS DOUBLE) / NULLIF(CAST({denominator_sql} AS DOUBLE), 0){scale}"
        else:
            expression = DeepEvidence._kpi_aggregate(spec.get("aggregation"), spec.get("column"), None, None)
            if DeepEvidence.canonical_aggregate(spec.get("aggregation")) in DEEP_KPI_VALUE_AGGREGATES:
                expression = f"CAST({expression} AS DOUBLE)"
            if spec.get("filter_column") and spec.get("filter_value") is not None:
                conditions.append(f"{quote_identifier(spec['filter_column'])} = {inspire_sql_literal(spec['filter_value'])}")
        if spec.get("time_column") and spec.get("window_days"):
            conditions.append(f"{quote_identifier(spec['time_column'])} >= current_date() - INTERVAL "
                              f"{int(spec['window_days'])} DAYS")
        where = f" WHERE {' AND '.join(conditions)}" if conditions else ""
        return f"SELECT {expression} AS v FROM {table_sql}{where}"

    @staticmethod
    def _kpi_aggregate(aggregation, column, filter_column, filter_value) -> str:
        aggregate = DeepEvidence.canonical_aggregate(aggregation)
        if not column:
            if aggregate != "COUNT":
                raise ValueError(f"KPI aggregate {aggregate} needs a column")
            expression = "COUNT(*)"
        elif aggregate == "COUNT_DISTINCT":
            expression = f"COUNT(DISTINCT {quote_identifier(column)})"
        elif aggregate in ("MIN", "MAX") and deep_column_is_pii(column):
            raise ValueError(f"KPI {aggregate} of the personal column {column!r} is never measured")
        else:
            expression = f"{aggregate}({quote_identifier(column)})"
        if filter_column and filter_value is not None:
            expression += f" FILTER (WHERE {quote_identifier(filter_column)} = {inspire_sql_literal(filter_value)})"
        return expression

    def measure_kpi(self, spec: dict):
        sql = DeepEvidence.kpi_sql(spec)
        rows = self._read("kpi_baseline", spec.get("metric_view") or spec.get("table"), sql)
        value = rows[0]["v"] if rows else None
        try:
            number = None if value is None else float(value)
        except (TypeError, ValueError, OverflowError):
            return None
        return number if number is not None and number == number and abs(number) != float("inf") else None

    def summary(self) -> dict:
        tables = sorted({t for s in self.statements if s["status"] == "ok" for t in s["tables"]
                         if t != "system.query.history"})
        return {"discovery_type": "Deep", "tables_read": tables, "tables_read_count": len(tables),
                "statements": len(self.statements),
                "statements_by_status": dict(Counter(s["status"] for s in self.statements)),
                "tables_skipped": dict(self.skipped_tables),
                "pii_columns_excluded": len(self.pii_excluded),
                "seconds": round(sum(s["seconds"] for s in self.statements), 1),
                "rows_sent_to_ai": 0}


_HISTORY_TABLE_REF_RE = re.compile(r"\b(?:FROM|JOIN)\s+([`\w.]+)(?:\s+(?:AS\s+)?([A-Za-z_]\w*))?", re.IGNORECASE)
_HISTORY_EQ_RE = re.compile(r"([`\w]+)\s*\.\s*([`\w]+)\s*=\s*([`\w]+)\s*\.\s*([`\w]+)")
_HISTORY_RESERVED = {"on", "where", "join", "left", "right", "inner", "outer", "full", "cross", "group", "order",
                     "limit", "union", "select", "using", "natural", "lateral", "having", "qualify", "window"}


def mine_history_join_edges(statements, known_tables) -> list:
    """Join edges between known tables from query text. The text never leaves this function:
    only (table.column, table.column, count) edges are returned."""
    known = {str(t).replace("`", "").lower(): str(t).replace("`", "") for t in known_tables or []}
    by_short = defaultdict(set)
    for low, original in known.items():
        by_short[low.split(".")[-1]].add(original)
    counts = Counter()
    for text in statements or []:
        aliases = {}
        for ref, alias in _HISTORY_TABLE_REF_RE.findall(text or ""):
            name = ref.replace("`", "").lower()
            table = known.get(name)
            if table is None and len(by_short.get(name.split(".")[-1], ())) == 1:
                table = next(iter(by_short[name.split(".")[-1]]))
            if table is None:
                continue
            aliases[name.split(".")[-1]] = table
            if alias and alias.lower() not in _HISTORY_RESERVED:
                aliases[alias.lower()] = table
        if len(set(aliases.values())) < 2:
            continue
        for left_alias, left_col, right_alias, right_col in _HISTORY_EQ_RE.findall(text or ""):
            left_table = aliases.get(left_alias.replace("`", "").lower())
            right_table = aliases.get(right_alias.replace("`", "").lower())
            if not left_table or not right_table or left_table == right_table:
                continue
            edge = tuple(sorted([(left_table, left_col.replace("`", "")), (right_table, right_col.replace("`", ""))]))
            counts[edge] += 1
    return [{"left": f"{a[0]}.{a[1]}", "right": f"{b[0]}.{b[1]}", "count": n}
            for (a, b), n in counts.most_common()]


def _parse_metric_view_definition(text) -> dict:
    payload = None
    try:
        import yaml as _yaml
        payload = _yaml.safe_load(text)
    except Exception:
        payload = None
    if not isinstance(payload, dict):
        measures = [{"name": m.group(1).strip().strip("'\""), "expr": m.group(2).strip().strip("'\"")}
                    for m in re.finditer(r"-\s*name:\s*(.+)\n\s+expr:\s*(.+)", str(text or ""))]
        source = re.search(r"^\s*source:\s*(.+)$", str(text or ""), re.MULTILINE)
        return {"source": source.group(1).strip().strip("'\"") if source else "", "measures": measures}
    return {"source": str(payload.get("source") or ""),
            "measures": [{"name": str(m.get("name") or ""), "expr": str(m.get("expr") or "")}
                         for m in payload.get("measures") or [] if isinstance(m, dict) and m.get("name")]}


def collect_metric_views(spark_session, schema_pairs, logger) -> list:
    """Unity Catalog metric views in the scanned schemas, read from information_schema and DESCRIBE only."""
    views = []
    for catalog, schema in sorted({(str(c), str(s)) for c, s in schema_pairs or []}):
        try:
            rows = _metadata_rows(
                spark_session,
                f"SELECT table_catalog, table_schema, table_name, comment FROM {quote_identifier(catalog)}.`information_schema`.`tables` "
                f"WHERE table_schema = :schema AND table_type IN ('METRIC_VIEW', 'METRIC VIEW')",
                args={"schema": schema.lower()}, logger=logger, context=f"metric views {catalog}.{schema}")
        except Exception as error:
            logger.info(f"Metric views not listed for {catalog}.{schema}: {get_clean_error_message(error, max_chars=200)}")
            continue
        for row in rows:
            fqn = build_fqn(row.table_catalog or catalog, row.table_schema or schema, row.table_name)
            definition = {"source": "", "measures": []}
            try:
                described = _metadata_rows(spark_session, f"DESCRIBE TABLE EXTENDED {fqn} AS JSON", logger=logger,
                                           context=f"metric view {fqn}")
                raw = described[0][0] if described else ""
                meta = json.loads(raw) if isinstance(raw, str) and raw.strip().startswith("{") else {}
                definition = _parse_metric_view_definition(meta.get("view_text") or meta.get("view_definition") or "")
            except Exception as error:
                logger.info(f"Metric view {fqn} definition unreadable: {get_clean_error_message(error, max_chars=200)}")
            views.append({"fqn": f"{row.table_catalog or catalog}.{row.table_schema or schema}.{row.table_name}",
                          "comment": row.comment or "", "source": definition["source"],
                          "measures": definition["measures"]})
    return views


def collect_usage_signals(spark_session, table_fqns, logger, days=90) -> dict:
    """Read counts, distinct readers and co-use from system.access.table_lineage. Metadata only:
    no user identity and no query text is kept."""
    names = sorted({str(t).replace("`", "") for t in table_fqns or [] if str(t).strip()})
    result = {"available": False, "reason": "", "reads": {}, "readers": {}, "downstream": {}, "co_used": []}
    if not names:
        result["reason"] = "no tables"
        return result
    in_sql, params = _sql_in_list_params(names, "t")
    window = f"event_time >= current_timestamp() - INTERVAL {int(days)} DAYS"
    try:
        rows = _metadata_rows(
            spark_session,
            f"SELECT source_table_full_name AS t, COUNT(*) AS n, COUNT(DISTINCT created_by) AS u, "
            f"COLLECT_SET(entity_type) AS kinds FROM system.access.table_lineage "
            f"WHERE {window} AND source_table_full_name IN ({in_sql}) GROUP BY source_table_full_name",
            args=params, logger=logger, context="usage signals")
        for row in rows:
            result["reads"][row.t] = int(row.n or 0)
            result["readers"][row.t] = int(row.u or 0)
            result["downstream"][row.t] = sorted(str(k) for k in (row.kinds or []) if k)
        pairs = _metadata_rows(
            spark_session,
            f"SELECT a.source_table_full_name AS t1, b.source_table_full_name AS t2, COUNT(DISTINCT a.entity_run_id) AS runs "
            f"FROM system.access.table_lineage a JOIN system.access.table_lineage b "
            f"ON a.entity_run_id = b.entity_run_id AND a.source_table_full_name < b.source_table_full_name "
            f"WHERE a.{window} AND b.{window} AND a.entity_run_id IS NOT NULL "
            f"AND a.source_table_full_name IN ({in_sql}) AND b.source_table_full_name IN ({in_sql}) "
            f"GROUP BY 1, 2 ORDER BY runs DESC LIMIT 200",
            args=params, logger=logger, context="co-usage signals")
        result["co_used"] = [{"left": r.t1, "right": r.t2, "runs": int(r.runs or 0)} for r in pairs]
        result["available"] = True
    except Exception as error:
        result["reason"] = get_clean_error_message(error, max_chars=200)
        logger.info(f"Usage signals unavailable (system.access.table_lineage): {result['reason']}")
    return result


class TableSizeInfo:
    """Metadata about a table's size and structure."""
    UNKNOWN_SIZE_WEIGHT = 1250

    def __init__(self, catalog, schema, table, num_columns=0, estimated_row_count=0, size_category="unknown"):
        self.catalog = catalog
        self.schema = schema
        self.table = table
        self.num_columns = num_columns
        self.estimated_row_count = estimated_row_count
        self.size_category = size_category  # "small", "medium", "wide", "very_wide", "unknown"
        self.memory_weight = self._calculate_memory_weight()
    
    def _calculate_memory_weight(self):
        """Calculate memory weight for batching decisions."""
        if self.size_category == "unknown" and self.num_columns <= 0:
            return self.UNKNOWN_SIZE_WEIGHT
        if self.num_columns > 1000:
            return self.num_columns * 10  # Very heavy
        elif self.num_columns > 250:
            return self.num_columns * 5   # Heavy
        elif self.num_columns > 100:
            return self.num_columns * 2   # Medium
        else:
            return self.num_columns       # Light
    
    def __repr__(self):
        return f"{self.catalog}.{self.schema}.{self.table} ({self.num_columns} cols, {self.size_category})"


class TableSizeAnalyzer:
    """
    Two-pass analyzer: First pass collects table sizes, second pass uses this info for intelligent batching.
    """
    def __init__(self, spark, logger):
        self.spark = spark
        self.logger = logger
        self.size_cache = {}  # {(catalog, schema, table): TableSizeInfo}
    
    def analyze_table_sizes_batch(self, table_tuples, use_info_schema_map, max_parallelism=20):
        """
        Analyze sizes for a batch of tables efficiently using parallel queries.
        
        Args:
            table_tuples: List of (catalog, schema, table) tuples
            use_info_schema_map: Dict mapping catalog -> bool for information_schema support
            max_parallelism: Maximum number of parallel queries (default: 20)
            
        Returns:
            List of TableSizeInfo objects
        """
        results = []
        
        # Group by catalog.schema for efficient querying
        schema_groups = defaultdict(list)
        for cat, schema, table in table_tuples:
            schema_groups[(cat, schema)].append(table)
        
        # Process schemas in parallel for speed
        def analyze_schema_group(schema_key_and_tables):
            (catalog, schema), tables = schema_key_and_tables
            use_info_schema = use_info_schema_map.get(catalog, False)
            schema_results = []
            
            self.logger.debug(f"   Analyzing {len(tables)} tables in {catalog}.{schema}...")
            
            column_counts = None
            if use_info_schema:
                try:
                    in_sql, params = _sql_in_list_params([str(t).lower() for t in tables], "t")
                    params["schema"] = str(schema).lower()
                    query = f"""
                        SELECT table_name, COUNT(*) as num_columns
                        FROM {quote_identifier(catalog)}.`information_schema`.`columns`
                        WHERE table_schema = :schema
                        AND table_name IN ({in_sql})
                        GROUP BY table_name
                    """
                    column_counts = {str(row.table_name).lower(): row.num_columns
                                     for row in _metadata_rows(self.spark, query, args=params, logger=self.logger,
                                                               context=f"column counts {catalog}.{schema}")}
                except SparkWorkCancelled:
                    raise
                except Exception as e:
                    self.logger.warning(f"Column-count query failed for {len(tables)} table(s) in {catalog}.{schema}; "
                                        f"sizing each table with DESCRIBE instead: {get_clean_error_message(e)}")
            unknown_tables = []
            for table in tables:
                num_cols = (column_counts or {}).get(str(table).lower(), 0)
                if num_cols == 0:
                    num_cols = self._get_column_count_fallback(catalog, schema, table)
                    if num_cols and column_counts is not None:
                        self.logger.info(f"info_schema returned 0 for {catalog}.{schema}.{table}; DESCRIBE fallback found {num_cols} columns")
                if num_cols:
                    size_info = TableSizeInfo(catalog, schema, table, num_columns=num_cols,
                                              size_category=self._categorize_table(num_cols))
                else:
                    unknown_tables.append(table)
                    size_info = TableSizeInfo(catalog, schema, table, num_columns=0, size_category="unknown")
                schema_results.append(size_info)
                self.size_cache[(catalog, schema, table)] = size_info
            if unknown_tables:
                shown = ", ".join(f"{catalog}.{schema}.{t}" for t in unknown_tables[:10])
                more = f" (+{len(unknown_tables) - 10} more)" if len(unknown_tables) > 10 else ""
                self.logger.warning(f"Column count unknown for {len(unknown_tables)} table(s) "
                                    f"(information_schema and DESCRIBE returned nothing): {shown}{more}; kept with size_category=unknown")
            return schema_results
        
        # Execute schema analysis - parallel or sequential based on max_parallelism
        if max_parallelism == 1:
            # Sequential execution when nested in another thread pool
            for item in schema_groups.items():
                try:
                    schema_results = analyze_schema_group(item)
                    results.extend(schema_results)
                except Exception as e:
                    schema_key = item[0]
                    self.logger.error(f"Failed to analyze schema {schema_key}: {get_clean_error_message(e)}")
        else:
            # Parallel execution for top-level calls
            with _SafeExecutorContext(max_workers=max_parallelism, thread_name_prefix="SchemaAnalyzer", logger=self.logger, name="SchemaAnalyzer") as ctx:
                futures = {ctx.submit(analyze_schema_group, item): item[0] 
                          for item in schema_groups.items()}
                
                total_timeout = len(futures) * 300
                try:
                    for future in ctx.completed(futures, timeout=total_timeout):
                        schema_key = futures[future]
                        try:
                            schema_results = future.result(timeout=300)
                            results.extend(schema_results)
                        except concurrent.futures.TimeoutError:
                            self.logger.error(f"Schema analysis timed out for {schema_key} after 5 minutes")
                        except Exception as e:
                            self.logger.error(f"Failed to analyze schema {schema_key}: {get_clean_error_message(e)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error(f"Overall schema analysis timeout ({total_timeout}s). Proceeding with partial results.")
        
        return results
    
    def _get_column_count_fallback(self, catalog, schema, table):
        """Fallback method to get column count using DESCRIBE."""
        try:
            rows = _metadata_rows(self.spark, f"DESCRIBE TABLE {build_fqn(catalog, schema, table)}", logger=self.logger,
                                  context=f"DESCRIBE {catalog}.{schema}.{table}")
            return len(_parse_describe_rows(rows))
        except SparkWorkCancelled:
            raise
        except Exception as e:
            self.logger.debug(f"DESCRIBE failed while sizing {catalog}.{schema}.{table}: {get_clean_error_message(e, max_chars=2000)}")
            return 0
    
    def _categorize_table(self, num_columns):
        """Categorize table based on column count."""
        if num_columns > 1000:
            return "very_wide"
        elif num_columns > 250:
            return "wide"
        elif num_columns > 100:
            return "medium"
        else:
            return "small"
    
    def get_cached_size(self, catalog, schema, table):
        """Get cached size info for a table."""
        return self.size_cache.get((catalog, schema, table))


class DynamicBatchOptimizer:
    """
    Intelligently groups tables into batches based on their size and memory requirements.
    
    Strategy:
    - Very wide tables (>1000 cols): 1-2 per batch
    - Wide tables (250-1000 cols): 5-10 per batch
    - Medium tables (100-250 cols): 20-50 per batch
    - Small tables (<100 cols): 100-500 per batch
    """
    
    # Memory weights for batching (in arbitrary units)
    MAX_BATCH_WEIGHT = 10000  # Adjust based on cluster memory
    MIN_BATCH_SIZE = 1
    MAX_BATCH_SIZE = 500
    
    def __init__(self, logger, max_batch_weight=None):
        self.logger = logger
        self.max_batch_weight = max_batch_weight or self.MAX_BATCH_WEIGHT
    
    def create_optimized_batches(self, table_size_infos):
        """
        Create optimized batches from table size information.
        
        Args:
            table_size_infos: List of TableSizeInfo objects
            
        Returns:
            List of lists, where each inner list is a batch of (catalog, schema, table) tuples
        """
        if not table_size_infos:
            return []
        
        # Sort tables by size (largest first for better packing)
        sorted_tables = sorted(table_size_infos, key=lambda t: t.memory_weight, reverse=True)
        
        batches = []
        current_batch = []
        current_weight = 0
        
        for table_info in sorted_tables:
            table_tuple = (table_info.catalog, table_info.schema, table_info.table)
            table_weight = table_info.memory_weight
            
            # Check if adding this table would exceed batch weight
            if current_batch and (current_weight + table_weight > self.max_batch_weight or 
                                 len(current_batch) >= self.MAX_BATCH_SIZE):
                # Start new batch
                batches.append(current_batch)
                self.logger.debug(f"Created batch with {len(current_batch)} tables, weight={current_weight}")
                current_batch = []
                current_weight = 0
            
            # Add table to current batch
            current_batch.append(table_tuple)
            current_weight += table_weight
        
        # Add final batch
        if current_batch:
            batches.append(current_batch)
            self.logger.debug(f"Created final batch with {len(current_batch)} tables, weight={current_weight}")
        
        # Log batch statistics
        self._log_batch_stats(batches, table_size_infos)
        
        return batches
    
    def _log_batch_stats(self, batches, table_size_infos):
        """Log statistics about the created batches."""
        total_tables = len(table_size_infos)
        num_batches = len(batches)
        
        size_categories = defaultdict(int)
        for table_info in table_size_infos:
            size_categories[table_info.size_category] += 1
        
        avg_batch_size = total_tables / num_batches if num_batches > 0 else 0
        
        self.logger.info(f"📊 Dynamic Batch Optimization Complete:")
        self.logger.info(f"   • Total tables: {total_tables}")
        self.logger.info(f"   • Created batches: {num_batches}")
        self.logger.info(f"   • Average batch size: {avg_batch_size:.1f} tables")
        self.logger.info(f"   • Table size distribution:")
        self.logger.info(f"      - Small (<100 cols): {size_categories['small']}")
        self.logger.info(f"      - Medium (100-250 cols): {size_categories['medium']}")
        self.logger.info(f"      - Wide (250-1000 cols): {size_categories['wide']}")
        self.logger.info(f"      - Very Wide (>1000 cols): {size_categories['very_wide']}")
        if size_categories['unknown']:
            self.logger.info(f"      - Unknown size: {size_categories['unknown']}")


class ColumnSampler:
    """
    Samples columns from very wide tables to reduce memory footprint.
    
    For tables with >250 columns, selects representative columns:
    - Evidence columns are never dropped: declared PK/FK, commented, numeric measure and date/timestamp columns
    - Key-like names that are not declared keys, capped at KEY_COLUMN_CAP
    - Columns with business-meaningful names
    - Evenly spaced sample of the remaining columns
    """
    
    WIDE_TABLE_THRESHOLD = 250
    TARGET_SAMPLE_SIZE = 200
    KEY_COLUMN_CAP = 50
    BUSINESS_KEYWORDS = (
        'name', 'date', 'time', 'amount', 'total', 'count', 'quantity',
        'price', 'cost', 'revenue', 'customer', 'order', 'product', 'status',
        'type', 'category', 'description', 'address', 'email', 'phone'
    )
    
    def __init__(self, logger):
        self.logger = logger
    
    def should_sample(self, num_columns):
        """Determine if column sampling is needed."""
        return num_columns > self.WIDE_TABLE_THRESHOLD

    @staticmethod
    def evidence_rank(column_name, data_type, comment, protected_columns=()):
        """0=declared PK/FK, 1=commented, 2=date/timestamp, 3=numeric measure, 4=key-like name, 5=other."""
        name = str(column_name or "")
        if name.lower() in protected_columns:
            return 0
        if comment is not None and str(comment).strip():
            return 1
        family = _metadata_type_family(data_type)
        if family == "temporal":
            return 2
        key_like = _is_key_like_column(name)
        if family == "numeric" and not key_like:
            return 3
        return 4 if key_like else 5
    
    def sample_columns(self, column_details, table_info, protected_columns=None):
        """
        Sample columns from a wide table.
        
        Args:
            column_details: List of (catalog, schema, table, col_name, data_type, comment) tuples
            table_info: TableSizeInfo object
            protected_columns: Column names (any case) of declared PK/FK columns that must be kept
            
        Returns:
            Sampled list of column details (original order) + flag indicating whether columns were dropped
        """
        if not self.should_sample(len(column_details)):
            return column_details, False  # No sampling needed
        
        protected = {str(c).lower() for c in (protected_columns or ())}
        must_keep, key_columns, business_columns, other_columns = [], [], [], []
        for idx, col_detail in enumerate(column_details):
            comment = col_detail[5] if len(col_detail) > 5 else None
            rank = self.evidence_rank(col_detail[3], col_detail[4], comment, protected)
            if rank <= 3:
                must_keep.append(idx)
            elif rank == 4:
                key_columns.append(idx)
            elif any(keyword in str(col_detail[3]).lower() for keyword in self.BUSINESS_KEYWORDS):
                business_columns.append(idx)
            else:
                other_columns.append(idx)
        
        selected = set(must_keep)
        remaining_slots = max(0, self.TARGET_SAMPLE_SIZE - len(selected))
        kept_keys = key_columns[:min(self.KEY_COLUMN_CAP, remaining_slots)]
        selected.update(kept_keys)
        remaining_slots -= len(kept_keys)
        kept_business = business_columns[:remaining_slots]
        selected.update(kept_business)
        remaining_slots -= len(kept_business)
        kept_other = []
        if remaining_slots > 0 and other_columns:
            step = max(1, len(other_columns) // remaining_slots)
            kept_other = other_columns[::step][:remaining_slots]
            selected.update(kept_other)
        
        sampled = [column_details[idx] for idx in sorted(selected)]
        was_sampled = len(sampled) < len(column_details)
        self.logger.info(
            f"🎯 Sampled wide table {table_info}: {len(column_details)} -> {len(sampled)} cols "
            f"({len(must_keep)} evidence cols kept [PK/FK, commented, measure, date/timestamp], "
            f"{len(kept_keys)}/{len(key_columns)} key-like, {len(kept_business)}/{len(business_columns)} business, "
            f"{len(kept_other)}/{len(other_columns)} other)"
        )
        return sampled, was_sampled


# COMMAND ----------

# DBTITLE 1,DataLoader

class DataLoader:
    # === MODIFIED: Added tables parameter + memory optimization features ===
    def __init__(self, catalogs: str, schemas: str, tables: str, logger: logging.Logger, 
                 enable_two_pass=True, enable_column_sampling=True, streaming_batch_size=1000,
                 max_parallelism=10, schema_timeout_seconds=900):
        self.spark = SparkSession.builder.getOrCreate()
        self.max_parallelism = max_parallelism  # For parallel schema discovery and column loading
        self.schema_timeout_seconds = schema_timeout_seconds  # Timeout per schema query (15 minutes)
        self.logger = logger
        self.foreign_key_graph = defaultdict(list)
        self._metadata_lock = threading.RLock()
        self._table_columns = {}
        self._columns_absent = set()
        self._canonical_names = {}
        self.table_metadata = {}
        self.primary_keys = {}
        self._constraints_loaded = set()
        self._constraint_failures = defaultdict(int)
        self._describe_failures = defaultdict(int)
        self._table_tags = {}
        self._column_tags = {}
        self._tags_loaded = set()
        self._inferred_fk = {}
        self._join_inferred_schemas = set()
        self._join_inference_lock = threading.Lock()
        self.sampled_tables = {}
        self._evidence_cache = {}
        self._metadata_errors = {}
        self.metadata_unavailable = {}
        
        # === NEW: Memory optimization features ===
        self.enable_two_pass = enable_two_pass  # Enable intelligent batching based on table sizes
        self.enable_column_sampling = enable_column_sampling  # Sample columns from very wide tables
        self.streaming_batch_size = streaming_batch_size  # Number of tables to process in each streaming chunk
        
        # === NEW: Initialize optimization components ===
        self.size_analyzer = TableSizeAnalyzer(self.spark, self.logger) if enable_two_pass else None
        self.batch_optimizer = DynamicBatchOptimizer(self.logger) if enable_two_pass else None
        self.column_sampler = ColumnSampler(self.logger) if enable_column_sampling else None

        # === MODIFIED: Process schemas, catalogs, and individual tables ===
        # Use utility functions to normalize identifiers (strip backticks from user input)
        self.schemas_to_process = [s.strip() for s in schemas.split(',') if s.strip()]
        self.catalogs_to_process = [normalize_identifier(c) for c in catalogs.split(',') if c.strip()]
        self.tables_to_process = [t.strip() for t in tables.split(',') if t.strip()]
        
        # Get all unique catalogs mentioned to check capabilities
        self.catalog_capabilities = {}
        unique_catalogs = set(self.catalogs_to_process)
        for s in self.schemas_to_process:
            cat, _ = parse_two_level_name(s)
            if cat:
                unique_catalogs.add(cat)
        for t in self.tables_to_process:
            cat, _, _ = parse_three_level_name(t)
            if cat:
                unique_catalogs.add(cat)
        
        self.logger.info(f"Initializing capabilities for {len(unique_catalogs)} catalogs in parallel: {unique_catalogs}")
        if len(unique_catalogs) > 1:
            with _SafeExecutorContext(max_workers=min(len(unique_catalogs), METADATA_PARALLELISM), thread_name_prefix="cat_cap", logger=self.logger, name="CatalogCapabilities") as ctx:
                cap_futures = {ctx.submit(self._check_catalog_capability, cat): cat for cat in unique_catalogs}
                try:
                    for future in ctx.completed(cap_futures, timeout=120):
                        cat = cap_futures[future]
                        try:
                            self.catalog_capabilities[cat] = future.result(timeout=60)
                        except Exception as exc:
                            self.logger.warning(f"Catalog capability check failed for '{cat}': {get_clean_error_message(exc)}")
                            self.catalog_capabilities[cat] = False
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error("Catalog capability checks timed out globally (120s)")
                    for future in cap_futures:
                        if not future.done():
                            cat = cap_futures[future]
                            self.catalog_capabilities[cat] = False
        else:
            for catalog in unique_catalogs:
                self.catalog_capabilities[catalog] = self._check_catalog_capability(catalog)
        self.logger.info(f"Capabilities found: {self.catalog_capabilities}")

        # === MODIFIED: Build the database queue and individual tables list ===
        self.database_queue = []
        db_set = set()
        
        # Track individual tables separately
        self.individual_tables = []  # List of (catalog, schema, table) tuples
        
        # Track explicitly provided schemas (to know which schemas should expand ALL tables)
        self.explicit_schemas_set = set()  # Set of (catalog, schema) tuples

        # 1. From explicit schemas (use parse_two_level_name for consistent normalization)
        for s in self.schemas_to_process:
            cat, db = parse_two_level_name(s)
            if cat and db:
                db_set.add((cat, db))
                self.explicit_schemas_set.add((cat, db))  # Mark as explicitly provided
            else:
                self.logger.warning(f"Skipping malformed schema name: {s}")
        
        # 2. From individual tables (use parse_three_level_name for consistent normalization)
        for t in self.tables_to_process:
            cat, db, table = parse_three_level_name(t)
            if cat and db and table:
                self.individual_tables.append((cat, db, table))
                # Also add the schema to db_set so we can process it
                db_set.add((cat, db))
            else:
                self.logger.warning(f"Skipping malformed table name: {t} (expected format: catalog.schema.table)")
        
        # 3. From catalogs (schemas from catalogs should also expand ALL tables)
        eligible_catalogs = list(dict.fromkeys(self.catalogs_to_process))
        for cat in eligible_catalogs:
            if not self.catalog_capabilities.get(cat, False):
                self.logger.info(f"Catalog `{cat}` has no information_schema; discovering its schemas with SHOW SCHEMAS.")
        
        if len(eligible_catalogs) > 1:
            _schema_lock = threading.Lock()
            def _fetch_catalog_schemas(cat):
                self.logger.info(f"Fetching schemas for catalog: {cat}")
                return cat, self._fetch_schemas_for_catalog(cat)
            
            with _SafeExecutorContext(max_workers=min(len(eligible_catalogs), METADATA_PARALLELISM), thread_name_prefix="cat_sch", logger=self.logger, name="SchemaDiscovery") as ctx:
                sch_futures = {ctx.submit(_fetch_catalog_schemas, cat): cat for cat in eligible_catalogs}
                try:
                    for future in ctx.completed(sch_futures, timeout=300):
                        try:
                            cat, schemas_in_cat = future.result(timeout=120)
                            with _schema_lock:
                                for db in schemas_in_cat:
                                    db_set.add((cat, db))
                                    self.explicit_schemas_set.add((cat, db))
                        except Exception as exc:
                            self.logger.error(f"Schema discovery failed for catalog: {get_clean_error_message(exc)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error("Schema discovery timed out globally (300s)")
        else:
            for cat in eligible_catalogs:
                self.logger.info(f"Fetching schemas for catalog: {cat}")
                schemas_in_cat = self._fetch_schemas_for_catalog(cat)
                for db in schemas_in_cat:
                    db_set.add((cat, db))
                    self.explicit_schemas_set.add((cat, db))
        
        self.database_queue = sorted(list(db_set)) # Sort for deterministic order
        self.logger.info(f"Found {len(self.database_queue)} unique databases to process.")
        
        # Log explicit schemas (will expand ALL tables)
        if self.explicit_schemas_set:
            explicit_names = [f"{cat}.{db}" for cat, db in sorted(self.explicit_schemas_set)]
            self.logger.info(f"📋 Explicit schemas (will expand ALL tables): {', '.join(explicit_names)}")
        
        # Log individual tables (specific tables only)
        if self.individual_tables:
            db_stats = {}
            for cat, db, tbl in self.individual_tables:
                key = f"{cat}.{db}"
                db_stats[key] = db_stats.get(key, 0) + 1
            self.logger.info(f"Found {len(self.individual_tables)} individual tables to process across {len(db_stats)} databases:")
            for db_key, count in sorted(db_stats.items()):
                self.logger.info(f"  database {db_key}: {count} tables loaded")
        
        # === NEW: State variables for table-level batching ===
        self.all_table_tuples = []  # List of (catalog, schema, table) tuples
        self.optimized_batches = []  # List of optimized batches (two-pass mode)
        self.current_batch_idx = 0   # Current batch index (two-pass mode)
        self.current_table_idx = 0   # Current position in all_table_tuples (single-pass mode)
        self._tables_initialized = False
        self._size_analysis_complete = False

    def _check_catalog_capability(self, catalog_name: str) -> bool:
        try:
            _metadata_rows(self.spark, f"SELECT 1 FROM {quote_identifier(catalog_name)}.`information_schema`.`schemata` LIMIT 1",
                           logger=self.logger, context=f"information_schema check {catalog_name}")
            return True
        except SparkWorkCancelled:
            raise
        except Exception as e:
            self.logger.info(f"Catalog `{catalog_name}` does not support information_schema ({get_clean_error_message(e, max_chars=200)}). "
                             f"Will attempt fallback 'SHOW' commands.")
            return False # Rely on fallbacks

    def _fetch_schemas_for_catalog(self, catalog_name: str):
        use_info_schema = self.catalog_capabilities.get(catalog_name, False)

        try:
            if not use_info_schema:
                raise Exception("Catalog does not support information_schema, using fallback.")
            
            query = f"""
                SELECT `schema_name` FROM {quote_identifier(catalog_name)}.`information_schema`.`schemata` 
                WHERE `schema_name` != 'information_schema' 
                ORDER BY `schema_name`
            """
            return [row.schema_name for row in _metadata_rows(self.spark, query, logger=self.logger,
                                                              context=f"schema listing {catalog_name}")]
        except SparkWorkCancelled:
            raise
        except Exception as e_info:
            try:
                query = f"SHOW SCHEMAS IN {quote_identifier(catalog_name)}"
                all_schemas = _with_metadata_retries(self.spark, lambda: self._show_names(query, ("databaseName", "namespace")),
                                                     self.logger, f"SHOW SCHEMAS {catalog_name}")
                return [s for s in all_schemas if s != 'information_schema']
            except Exception as e_show_schemas:
                self.logger.error(f"Error listing schemas for catalog `{catalog_name}`. All fallbacks failed: {get_clean_error_message(e_show_schemas)}")
                self._record_metadata_unavailable((catalog_name, "*", "*"), f"schema listing failed: {get_clean_error_message(e_show_schemas, max_chars=200)}")
                return []

    def _show_names(self, query, name_columns):
        df = self.spark.sql(query)
        col_name = next((c for c in name_columns if c in df.columns), name_columns[-1])
        return [row[col_name] for row in df.orderBy(col_name).collect()]

    def _show_table_names(self, query, limit=None, offset=0):
        df = self.spark.sql(query)
        if 'isTemporary' in df.columns:
            df = df.filter(df.isTemporary == False)
        if limit:
            df = df.orderBy("tableName").limit(limit).offset(offset)
        else:
            df = df.orderBy("tableName")
        return [row.tableName for row in df.collect()]

    def _record_metadata_unavailable(self, names, reason=None):
        key = self._lkey(*names)
        with self._metadata_lock:
            reason = " ".join(str(reason or self._metadata_errors.get(key)
                                  or "not listed in information_schema and DESCRIBE returned no columns").split())[:300]
            first = key not in self.metadata_unavailable
            fqn = ".".join(str(part) for part in self._canonical_names.get(key, names))
            self.metadata_unavailable[key] = {"table": fqn, "reason": reason}
        if first:
            self.logger.warning(f"[METADATA_TABLE_UNAVAILABLE FIRED] table={fqn}: {reason}; it is excluded from analysis")
        return first

    def metadata_unavailable_report(self) -> dict:
        with self._metadata_lock:
            tables = sorted(self.metadata_unavailable.values(), key=lambda entry: entry["table"])
        return {"count": len(tables), "tables": [dict(entry) for entry in tables]}

    def _report_unavailable_batch_tables(self, missing, requested_count, label):
        if not missing:
            return
        with self._metadata_lock:
            entries = [self.metadata_unavailable.get(self._lkey(*t)) or {"table": ".".join(map(str, t)), "reason": "unknown"}
                       for t in missing]
        listing = "; ".join(f"{e['table']} ({e['reason']})" for e in entries[:20])
        more = f"; +{len(entries) - 20} more" if len(entries) > 20 else ""
        self.logger.error(f"[METADATA_TABLES_UNAVAILABLE FIRED] {label}: {len(missing)} of {requested_count} table(s) could not be "
                          f"loaded and are NOT analysed: {listing}{more}")
        try:
            NEXT_ACTIONS.add("METADATA_TABLE_UNAVAILABLE", severity="HIGH", phase="Metadata", step=label,
                             evidence=f"{len(missing)} table(s) without metadata: {listing}{more}",
                             suggested_user_action="Check that the metastore is reachable and the principal can read these "
                                                   "tables' metadata, then re-run.")
        except Exception:
            pass

    # === MODIFIED: Added streaming/pagination support for memory efficiency ===
    def _fetch_tables_for_schema(self, catalog_name: str, schema_name: str, limit=None, offset=0):
        """
        Fetch tables for a schema with optional pagination.
        
        Args:
            catalog_name: Catalog name
            schema_name: Schema name
            limit: Optional limit for pagination (for very large schemas)
            offset: Offset for pagination
            
        Returns:
            List of fully qualified table names
        """
        cat_normalized = normalize_identifier(catalog_name)
        schema_normalized = normalize_identifier(schema_name)
        fq_schema = build_fqn(cat_normalized, schema_normalized)
        tables = []
        use_info_schema = self.catalog_capabilities.get(cat_normalized, False)

        try:
            if not use_info_schema:
                raise Exception("Catalog does not support information_schema, using fallback.")
            
            # Use pagination for large schemas
            limit_clause = f"LIMIT {int(limit)} OFFSET {int(offset)}" if limit else ""
            cat_quoted = quote_identifier(cat_normalized)
            query = f"""
                SELECT `table_name` FROM {cat_quoted}.`information_schema`.`tables`
                WHERE `table_schema` = :schema
                ORDER BY `table_name`
                {limit_clause}
            """
            rows = _metadata_rows(self.spark, query, args={"schema": schema_normalized.lower()}, logger=self.logger,
                                  context=f"table listing {cat_normalized}.{schema_normalized}")
            tables = [f"{fq_schema}.`{row.table_name}`" for row in rows]
                
        except SparkWorkCancelled:
            raise
        except Exception:
            try:
                query = f"SHOW TABLES IN {fq_schema}"
                tables = [f"{fq_schema}.`{name}`" for name in _with_metadata_retries(
                    self.spark, lambda: self._show_table_names(query, limit, offset), self.logger,
                    f"SHOW TABLES {cat_normalized}.{schema_normalized}")]
            except Exception as e:
                self.logger.warning(f"Error listing tables for {fq_schema}: {get_clean_error_message(e)}")
                self._record_metadata_unavailable((cat_normalized, schema_normalized, "*"),
                                                  f"table listing failed: {get_clean_error_message(e, max_chars=200)}")
                tables = []
        
        return tables
    
    def _fetch_tables_for_schema_streaming(self, catalog_name: str, schema_name: str, chunk_size=1000):
        """
        Generator that yields tables in chunks for memory-efficient streaming.
        
        Args:
            catalog_name: Catalog name  
            schema_name: Schema name
            chunk_size: Number of tables to yield per chunk
            
        Yields:
            Lists of fully qualified table names
        """
        offset = 0
        while True:
            tables = self._fetch_tables_for_schema(catalog_name, schema_name, limit=chunk_size, offset=offset)
            if not tables:
                break
            yield tables
            if len(tables) < chunk_size:
                break  # Last chunk
            offset += chunk_size

    @staticmethod
    def _lkey(catalog, schema, table):
        return (normalize_identifier(str(catalog or "")).lower(),
                normalize_identifier(str(schema or "")).lower(),
                normalize_identifier(str(table or "")).lower())

    @staticmethod
    def _fqn_list(fqns, limit=10):
        fqns = list(fqns)
        more = f" (+{len(fqns) - limit} more)" if len(fqns) > limit else ""
        return ", ".join(fqns[:limit]) + more

    def _catalog_capable(self, catalog):
        cat = normalize_identifier(str(catalog or ""))
        if cat in self.catalog_capabilities:
            return bool(self.catalog_capabilities[cat])
        for known, capable in self.catalog_capabilities.items():
            if known.lower() == cat.lower():
                return bool(capable)
        capable = self._check_catalog_capability(cat)
        self.catalog_capabilities[cat] = capable
        return capable

    def _load_schema_metadata(self, catalog, schema, tables, include_tags=False):
        cat = normalize_identifier(str(catalog or ""))
        sch = normalize_identifier(str(schema or ""))
        names = sorted({normalize_identifier(str(t)).lower() for t in tables if normalize_identifier(str(t or ""))})
        if not (cat and sch and names) or not self._catalog_capable(cat):
            return set()
        cat_quoted = quote_identifier(cat)
        loaded = set()
        for start in range(0, len(names), _METADATA_IN_LIST_CHUNK):
            chunk = names[start:start + _METADATA_IN_LIST_CHUNK]
            in_sql, params = _sql_in_list_params(chunk, "t")
            params["schema"] = sch.lower()
            chunk_fqns = [f"{cat}.{sch}.{t}" for t in chunk]
            try:
                rows = _metadata_rows(
                    self.spark,
                    f"SELECT table_catalog, table_schema, table_name, column_name, data_type, comment, is_nullable, ordinal_position "
                    f"FROM {cat_quoted}.`information_schema`.`columns` "
                    f"WHERE table_schema = :schema AND table_name IN ({in_sql}) "
                    f"ORDER BY table_name, ordinal_position",
                    args=params, logger=self.logger, context=f"column metadata {cat}.{sch}")
                grouped = {}
                for row in rows:
                    canonical = (row.table_catalog or cat, row.table_schema or sch, row.table_name)
                    nullable = row.is_nullable
                    grouped.setdefault(canonical, []).append({
                        "name": row.column_name,
                        "type": row.data_type,
                        "comment": row.comment,
                        "nullable": None if nullable is None else str(nullable).strip().upper() == "YES",
                        "ordinal": row.ordinal_position,
                    })
                with self._metadata_lock:
                    for canonical, cols in grouped.items():
                        key = self._lkey(*canonical)
                        self._table_columns[key] = cols
                        self._canonical_names[key] = canonical
                        loaded.add(key)
                    for name in chunk:
                        key = self._lkey(cat, sch, name)
                        self._metadata_errors.pop(key, None)
                        if key not in loaded:
                            self._columns_absent.add(key)
            except SparkWorkCancelled:
                raise
            except Exception as e:
                self.logger.warning(f"Column metadata query failed for {len(chunk)} table(s) in {cat}.{sch} "
                                    f"({self._fqn_list(chunk_fqns)}): {get_clean_error_message(e)}")
                with self._metadata_lock:
                    for name in chunk:
                        self._metadata_errors[self._lkey(cat, sch, name)] = \
                            f"information_schema column query failed: {get_clean_error_message(e, max_chars=200)}"
            try:
                rows = _metadata_rows(
                    self.spark,
                    f"SELECT table_catalog, table_schema, table_name, table_type, comment, data_source_format, last_altered "
                    f"FROM {cat_quoted}.`information_schema`.`tables` "
                    f"WHERE table_schema = :schema AND table_name IN ({in_sql})",
                    args=params, logger=self.logger, context=f"table metadata {cat}.{sch}")
                with self._metadata_lock:
                    for row in rows:
                        key = self._lkey(row.table_catalog or cat, row.table_schema or sch, row.table_name)
                        altered = row.last_altered
                        self.table_metadata[key] = {
                            "comment": row.comment,
                            "table_type": row.table_type,
                            "data_source_format": row.data_source_format,
                            "last_altered": None if altered is None else (altered.isoformat() if hasattr(altered, "isoformat") else str(altered)),
                        }
            except SparkWorkCancelled:
                raise
            except Exception as e:
                self.logger.warning(f"Table metadata (comment/type) query failed for {len(chunk)} table(s) in {cat}.{sch} "
                                    f"({self._fqn_list(chunk_fqns)}): {get_clean_error_message(e)}")
            self._load_constraints(cat, sch, chunk)
            if include_tags:
                self._load_tags(cat, sch, chunk)
        return loaded

    def _load_constraints(self, catalog, schema, tables):
        cat = normalize_identifier(str(catalog or ""))
        sch = normalize_identifier(str(schema or ""))
        names = sorted({normalize_identifier(str(t)).lower() for t in tables if normalize_identifier(str(t or ""))})
        if not (cat and sch and names):
            return False
        keys = [self._lkey(cat, sch, t) for t in names]
        if not self._catalog_capable(cat):
            return False
        cat_quoted = quote_identifier(cat)
        in_sql, params = _sql_in_list_params(names, "t")
        params["schema"] = sch.lower()
        try:
            pk_rows = _metadata_rows(
                self.spark,
                f"SELECT tc.table_catalog AS table_catalog, tc.table_schema AS table_schema, tc.table_name AS table_name, "
                f"kcu.column_name AS column_name, kcu.ordinal_position AS ordinal_position "
                f"FROM {cat_quoted}.`information_schema`.`table_constraints` tc "
                f"JOIN {cat_quoted}.`information_schema`.`key_column_usage` kcu "
                f"ON tc.constraint_catalog = kcu.constraint_catalog AND tc.constraint_schema = kcu.constraint_schema "
                f"AND tc.constraint_name = kcu.constraint_name "
                f"WHERE tc.table_schema = :schema AND tc.table_name IN ({in_sql}) AND tc.constraint_type = 'PRIMARY KEY' "
                f"ORDER BY tc.table_name, kcu.ordinal_position",
                args=params, logger=self.logger, context=f"primary keys {cat}.{sch}")
            fk_rows = None
            last_error = None
            for ref_position in ("COALESCE(kcu.position_in_unique_constraint, kcu.ordinal_position)", "kcu.ordinal_position"):
                try:
                    fk_rows = _metadata_rows(
                        self.spark,
                        f"SELECT kcu.table_catalog AS table_catalog, kcu.table_schema AS table_schema, kcu.table_name AS table_name, "
                        f"kcu.column_name AS column_name, kcu.ordinal_position AS ordinal_position, tc.constraint_name AS constraint_name, "
                        f"rc.unique_constraint_catalog AS ref_constraint_catalog, rc.unique_constraint_schema AS ref_constraint_schema, "
                        f"rc.unique_constraint_name AS ref_constraint_name, {ref_position} AS ref_position, "
                        f"kcu_ref.table_catalog AS referenced_table_catalog, kcu_ref.table_schema AS referenced_table_schema, "
                        f"kcu_ref.table_name AS referenced_table_name, kcu_ref.column_name AS referenced_column_name "
                        f"FROM {cat_quoted}.`information_schema`.`table_constraints` tc "
                        f"JOIN {cat_quoted}.`information_schema`.`referential_constraints` rc "
                        f"ON tc.constraint_catalog = rc.constraint_catalog AND tc.constraint_schema = rc.constraint_schema "
                        f"AND tc.constraint_name = rc.constraint_name "
                        f"JOIN {cat_quoted}.`information_schema`.`key_column_usage` kcu "
                        f"ON tc.constraint_catalog = kcu.constraint_catalog AND tc.constraint_schema = kcu.constraint_schema "
                        f"AND tc.constraint_name = kcu.constraint_name "
                        f"LEFT JOIN {cat_quoted}.`information_schema`.`key_column_usage` kcu_ref "
                        f"ON rc.unique_constraint_catalog = kcu_ref.constraint_catalog AND rc.unique_constraint_schema = kcu_ref.constraint_schema "
                        f"AND rc.unique_constraint_name = kcu_ref.constraint_name AND kcu_ref.ordinal_position = {ref_position} "
                        f"WHERE tc.table_schema = :schema AND tc.table_name IN ({in_sql}) AND tc.constraint_type = 'FOREIGN KEY' "
                        f"ORDER BY kcu.table_name, tc.constraint_name, kcu.ordinal_position",
                        args=params, logger=self.logger, context=f"foreign keys {cat}.{sch}")
                    break
                except SparkWorkCancelled:
                    raise
                except Exception as e:
                    last_error = e
            if fk_rows is None:
                raise last_error
        except SparkWorkCancelled:
            raise
        except Exception as e:
            with self._metadata_lock:
                for key in keys:
                    self._constraint_failures[key] += 1
            self.logger.warning(f"PK/FK constraint query failed for {len(names)} table(s) in {cat}.{sch} "
                                f"({self._fqn_list(f'{cat}.{sch}.{t}' for t in names)}); not cached, will retry on next use: "
                                f"{get_clean_error_message(e)}")
            return False
        primary_keys = defaultdict(list)
        for row in pk_rows:
            primary_keys[self._lkey(row.table_catalog or cat, row.table_schema or sch, row.table_name)].append(row.column_name)
        relations = defaultdict(list)
        unresolved = []
        for row in fk_rows:
            source = (row.table_catalog or cat, row.table_schema or sch, row.table_name)
            if row.referenced_table_name:
                relations[source].append((source[0], source[1], source[2], row.column_name,
                                          row.referenced_table_catalog, row.referenced_table_schema,
                                          row.referenced_table_name, row.referenced_column_name))
            else:
                unresolved.append((source, row))
        if unresolved:
            for source, rel in self._resolve_cross_catalog_references(unresolved):
                relations[source].append(rel)
        with self._metadata_lock:
            for key in keys:
                self._constraints_loaded.add(key)
                self._constraint_failures.pop(key, None)
                if primary_keys.get(key):
                    self.primary_keys[key] = primary_keys[key]
                else:
                    self.primary_keys.pop(key, None)
            for source, rels in relations.items():
                for old_rel in self.foreign_key_graph.get(source, []):
                    self._inferred_fk.pop(old_rel, None)
                self.foreign_key_graph[source] = rels
        return True

    def _resolve_cross_catalog_references(self, unresolved):
        by_catalog = defaultdict(list)
        for source, row in unresolved:
            by_catalog[normalize_identifier(str(row.ref_constraint_catalog or ""))].append((source, row))
        resolved = []
        for ref_catalog, items in by_catalog.items():
            constraint_names = sorted({str(row.ref_constraint_name) for _, row in items if row.ref_constraint_name})
            lookup = {}
            if ref_catalog and constraint_names and self._catalog_capable(ref_catalog):
                in_sql, params = _sql_in_list_params(constraint_names, "c")
                try:
                    for ref in _metadata_rows(
                            self.spark,
                            f"SELECT constraint_schema, constraint_name, table_catalog, table_schema, table_name, column_name, ordinal_position "
                            f"FROM {quote_identifier(ref_catalog)}.`information_schema`.`key_column_usage` "
                            f"WHERE constraint_name IN ({in_sql})",
                            args=params, logger=self.logger, context=f"cross-catalog key lookup {ref_catalog}"):
                        lookup[(str(ref.constraint_schema), str(ref.constraint_name), int(ref.ordinal_position))] = ref
                except SparkWorkCancelled:
                    raise
                except Exception as e:
                    self.logger.warning(f"Cross-catalog FK target lookup failed in catalog {ref_catalog}: {get_clean_error_message(e)}")
            for source, row in items:
                ref = lookup.get((str(row.ref_constraint_schema), str(row.ref_constraint_name), int(row.ref_position or row.ordinal_position or 0)))
                if ref is None:
                    self.logger.warning(f"Declared FK {source[0]}.{source[1]}.{source[2]}.{row.column_name} "
                                        f"(constraint {row.constraint_name}) references an unresolvable key "
                                        f"{row.ref_constraint_catalog}.{row.ref_constraint_schema}.{row.ref_constraint_name}; skipped")
                    continue
                resolved.append((source, (source[0], source[1], source[2], row.column_name,
                                          ref.table_catalog, ref.table_schema, ref.table_name, ref.column_name)))
        return resolved

    def _load_tags(self, catalog, schema, tables):
        cat = normalize_identifier(str(catalog or ""))
        sch = normalize_identifier(str(schema or ""))
        names = sorted({normalize_identifier(str(t)).lower() for t in tables if normalize_identifier(str(t or ""))})
        if not (cat and sch and names) or not self._catalog_capable(cat):
            return
        cat_quoted = quote_identifier(cat)
        in_sql, params = _sql_in_list_params(names, "t")
        params["schema"] = sch.lower()
        fqns = [f"{cat}.{sch}.{t}" for t in names]
        table_tags = defaultdict(dict)
        column_tags = defaultdict(lambda: defaultdict(dict))
        try:
            for row in _metadata_rows(
                    self.spark,
                    f"SELECT table_name, tag_name, tag_value FROM {cat_quoted}.`information_schema`.`table_tags` "
                    f"WHERE schema_name = :schema AND table_name IN ({in_sql})",
                    args=params, logger=self.logger, context=f"table tags {cat}.{sch}"):
                table_tags[self._lkey(cat, sch, row.table_name)][row.tag_name] = row.tag_value
            for row in _metadata_rows(
                    self.spark,
                    f"SELECT table_name, column_name, tag_name, tag_value FROM {cat_quoted}.`information_schema`.`column_tags` "
                    f"WHERE schema_name = :schema AND table_name IN ({in_sql})",
                    args=params, logger=self.logger, context=f"column tags {cat}.{sch}"):
                column_tags[self._lkey(cat, sch, row.table_name)][str(row.column_name).lower()][row.tag_name] = row.tag_value
        except SparkWorkCancelled:
            raise
        except Exception as e:
            self.logger.warning(f"Tag metadata unavailable for {len(names)} table(s) in {cat}.{sch} "
                                f"({self._fqn_list(fqns)}); evidence keeps empty tags: {get_clean_error_message(e)}")
        with self._metadata_lock:
            for name in names:
                key = self._lkey(cat, sch, name)
                self._table_tags[key] = dict(table_tags.get(key, {}))
                self._column_tags[key] = {c: dict(v) for c, v in column_tags.get(key, {}).items()}
                self._tags_loaded.add(key)

    def _describe_table_columns(self, catalog, schema, table):
        key = self._lkey(catalog, schema, table)
        if self._describe_failures.get(key, 0) >= 2:
            return None
        try:
            rows = _metadata_rows(self.spark, f"DESCRIBE TABLE {build_fqn(catalog, schema, table)}", logger=self.logger,
                                  context=f"DESCRIBE {catalog}.{schema}.{table}")
        except SparkWorkCancelled:
            raise
        except Exception as e:
            with self._metadata_lock:
                self._describe_failures[key] += 1
            lowered = str(e).lower()
            kind = "permission denied" if any(k in lowered for k in ("permission", "unauthorized", "access denied", "insufficient privilege")) else "failed"
            with self._metadata_lock:
                self._metadata_errors[key] = f"DESCRIBE {kind}: {get_clean_error_message(e, max_chars=200)}"
            self.logger.warning(f"Column metadata unavailable for {catalog}.{schema}.{table}: DESCRIBE {kind} "
                                f"(attempt {self._describe_failures[key]}/2): {get_clean_error_message(e)}")
            return None
        parsed = _parse_describe_rows(rows)
        if not parsed:
            self.logger.warning(f"Column metadata unavailable for {catalog}.{schema}.{table}: DESCRIBE returned no columns")
            return None
        columns = [{"name": name, "type": dtype, "comment": comment, "nullable": None, "ordinal": idx + 1}
                   for idx, (name, dtype, comment) in enumerate(parsed)]
        with self._metadata_lock:
            self._table_columns[key] = columns
            self._canonical_names.setdefault(key, (normalize_identifier(catalog), normalize_identifier(schema), normalize_identifier(table)))
        return columns

    def _preload_metadata(self, table_tuples, max_workers=None):
        groups = {}
        for (catalog, schema, table) in table_tuples:
            key = self._lkey(catalog, schema, table)
            if key in self._table_columns or key in self._columns_absent or not self._catalog_capable(catalog):
                continue
            groups.setdefault((normalize_identifier(catalog), normalize_identifier(schema)), []).append(table)
        if not groups:
            return
        workers = max(1, min(len(groups), max_workers or METADATA_PARALLELISM))
        if workers == 1:
            for (catalog, schema), tables in groups.items():
                self._load_schema_metadata(catalog, schema, tables)
            return
        with _SafeExecutorContext(max_workers=workers, thread_name_prefix="meta_preload", logger=self.logger, name="MetadataPreload") as ctx:
            futures = {ctx.submit(self._load_schema_metadata, catalog, schema, tables): (catalog, schema)
                       for (catalog, schema), tables in groups.items()}
            try:
                for future in ctx.completed(futures, timeout=_METADATA_BATCH_TIMEOUT_S):
                    try:
                        future.result(timeout=300)
                    except Exception as exc:
                        catalog, schema = futures[future]
                        self.logger.warning(f"Metadata preload failed for {catalog}.{schema}: {get_clean_error_message(exc)}")
            except concurrent.futures.TimeoutError:
                ctx.mark_timed_out()
                self.logger.warning(f"Metadata preload timed out ({_METADATA_BATCH_TIMEOUT_S}s); remaining tables load individually")

    def _fetch_batch_details(self, batch_table_tuples, max_workers, executor_name, label):
        all_column_details = []
        self._preload_metadata(batch_table_tuples, max_workers=max_workers)

        def fetch_details_with_sampling(args):
            return self._get_table_details(*args, apply_sampling=self.enable_column_sampling)

        timed_out = False
        with _SafeExecutorContext(max_workers=max_workers, thread_name_prefix=executor_name, logger=self.logger, name=executor_name) as ctx:
            detail_futures = [ctx.submit(fetch_details_with_sampling, args) for args in batch_table_tuples]
            try:
                for future in ctx.completed(detail_futures, timeout=_METADATA_BATCH_TIMEOUT_S):
                    try:
                        column_list = future.result(timeout=120)
                        if column_list:
                            all_column_details.extend(column_list)
                    except Exception as exc:
                        self.logger.warning(f"Table detail fetch failed: {get_clean_error_message(exc)}")
            except concurrent.futures.TimeoutError:
                timed_out = True
                ctx.mark_timed_out()
                self.logger.error(f"{label}: table details timed out globally ({_METADATA_BATCH_TIMEOUT_S}s)")
        loaded = {self._lkey(row[0], row[1], row[2]) for row in all_column_details}
        missing = [t for t in batch_table_tuples if self._lkey(*t) not in loaded]
        with self._metadata_lock:
            already_recorded = set(self.metadata_unavailable)
        for t in missing:
            if self._lkey(*t) not in already_recorded:
                self._record_metadata_unavailable(
                    t, f"metadata load did not finish within the {_METADATA_BATCH_TIMEOUT_S}s batch budget" if timed_out else None)
        self.logger.info(f"   ✓ {label} loaded: {len(all_column_details)} columns from "
                         f"{len(batch_table_tuples) - len(missing)}/{len(batch_table_tuples)} tables")
        self._report_unavailable_batch_tables(missing, len(batch_table_tuples), label)
        return all_column_details

    def _get_table_details(self, catalog: str, schema: str, table: str, apply_sampling=True):
        """
        Get table column details with optional column sampling for very wide tables.
        
        Args:
            catalog: Catalog name
            schema: Schema name
            table: Table name
            apply_sampling: Whether to apply column sampling for wide tables
            
        Returns:
            List of (catalog, schema, table, column_name, data_type, comment) tuples
        """
        cat_normalized = normalize_identifier(catalog)
        schema_normalized = normalize_identifier(schema)
        table_normalized = normalize_identifier(table)
        key = self._lkey(cat_normalized, schema_normalized, table_normalized)
        columns = self._table_columns.get(key)
        if columns is None and key not in self._columns_absent and self._catalog_capable(cat_normalized):
            self._load_schema_metadata(cat_normalized, schema_normalized, [table_normalized])
            columns = self._table_columns.get(key)
        if columns is None:
            columns = self._describe_table_columns(cat_normalized, schema_normalized, table_normalized)
        if not columns:
            self._record_metadata_unavailable((cat_normalized, schema_normalized, table_normalized))
            return []
        c_name, s_name, t_name = self._canonical_names.get(key, (cat_normalized, schema_normalized, table_normalized))
        details = [(c_name, s_name, t_name, col["name"], col["type"], col["comment"]) for col in columns]
        if apply_sampling and self.column_sampler and len(details) > ColumnSampler.WIDE_TABLE_THRESHOLD:
            protected = {str(p).lower() for p in self.primary_keys.get(key, [])}
            protected.update(str(rel[3]).lower() for rel in self.foreign_key_graph.get((c_name, s_name, t_name), [])
                             if rel not in self._inferred_fk)
            table_info = self.size_analyzer.get_cached_size(c_name, s_name, t_name) if self.size_analyzer else None
            if not table_info:
                table_info = TableSizeInfo(c_name, s_name, t_name, num_columns=len(details))
            sampled, was_sampled = self.column_sampler.sample_columns(details, table_info, protected_columns=protected)
            if was_sampled:
                with self._metadata_lock:
                    self.sampled_tables[key] = (len(sampled), len(details))
            return sampled
        return details

    def _ensure_constraints(self, table_tuples):
        groups = {}
        for (catalog, schema, table) in table_tuples:
            key = self._lkey(catalog, schema, table)
            if key in self._constraints_loaded or self._constraint_failures.get(key, 0) >= 2:
                continue
            if not self._catalog_capable(catalog):
                continue
            groups.setdefault((normalize_identifier(catalog), normalize_identifier(schema)), []).append(table)
        for (catalog, schema), tables in groups.items():
            self._load_constraints(catalog, schema, tables)

    def get_foreign_key_relations(self, table_tuples: set):
        self._ensure_constraints(table_tuples)
        relations = []
        for tbl in table_tuples:
            relations.extend(self.foreign_key_graph.get(tbl, []))
        return relations

    def get_table_comment(self, catalog, schema, table):
        return (self.table_metadata.get(self._lkey(catalog, schema, table)) or {}).get("comment") or ""

    def get_primary_key(self, catalog, schema, table):
        return list(self.primary_keys.get(self._lkey(catalog, schema, table), []))

    def get_sampling_info(self, catalog, schema, table):
        return self.sampled_tables.get(self._lkey(catalog, schema, table))

    def get_table_relations(self, catalog, schema, table):
        """Outgoing FKs of a table as (relation_8_tuple, source, confidence); source is declared or inferred."""
        rels = self.foreign_key_graph.get((catalog, schema, table))
        if rels is None:
            rels = self.foreign_key_graph.get(self._lkey(catalog, schema, table))
        out = []
        for rel in rels or []:
            if rel in self._inferred_fk:
                out.append((rel, "inferred", self._inferred_fk[rel]))
            else:
                out.append((rel, "declared", 1.0))
        return out

    def get_metadata_evidence(self, table_fqns) -> dict:
        """Metadata-only evidence per table, keyed by lowercase "cat.sch.tbl": comment, table_type,
        data_source_format, last_altered, tags, full column list (never sampled), primary_key, foreign_keys.
        Batched per schema from information_schema; DESCRIBE TABLE only for tables information_schema does not list.
        Cached per run; tables that cannot be found are absent keys and logged at WARNING."""
        requested = []
        pending = {}
        for fqn in table_fqns or []:
            cat, sch, tbl = parse_three_level_name(str(fqn or ""))
            if not (cat and sch and tbl):
                self.logger.warning(f"Metadata evidence skipped for malformed table name {fqn!r} (expected catalog.schema.table)")
                continue
            key = self._lkey(cat, sch, tbl)
            if key not in requested:
                requested.append(key)
            if ".".join(key) not in self._evidence_cache:
                pending.setdefault((cat, sch), set()).add(tbl)
        for (cat, sch), tables in pending.items():
            if self._catalog_capable(cat):
                missing = [t for t in tables if self._lkey(cat, sch, t) not in self._table_columns
                           or self._lkey(cat, sch, t) not in self.table_metadata]
                if missing:
                    self._load_schema_metadata(cat, sch, missing)
                self._ensure_constraints([(cat, sch, t) for t in tables])
                untagged = [t for t in tables if self._lkey(cat, sch, t) not in self._tags_loaded]
                if untagged:
                    self._load_tags(cat, sch, untagged)
            for t in tables:
                if self._lkey(cat, sch, t) not in self._table_columns:
                    self._describe_table_columns(cat, sch, t)
            for t in tables:
                key = self._lkey(cat, sch, t)
                columns = self._table_columns.get(key)
                if not columns:
                    self.logger.warning(f"Metadata evidence unavailable for {'.'.join(key)}: table not found in "
                                        f"information_schema/DESCRIBE or not visible to this principal")
                    continue
                meta = self.table_metadata.get(key) or {}
                col_tags = self._column_tags.get(key, {})
                self._evidence_cache[".".join(key)] = {
                    "comment": meta.get("comment"),
                    "table_type": meta.get("table_type"),
                    "data_source_format": meta.get("data_source_format"),
                    "last_altered": meta.get("last_altered"),
                    "tags": dict(self._table_tags.get(key, {})),
                    "columns": {str(col["name"]).lower(): {"name": col["name"], "type": col["type"], "comment": col["comment"],
                                                           "nullable": col["nullable"],
                                                           "tags": dict(col_tags.get(str(col["name"]).lower(), {}))}
                                for col in columns},
                    "primary_key": list(self.primary_keys.get(key, [])),
                }
        evidence = {}
        for key in requested:
            fqn = ".".join(key)
            cached = self._evidence_cache.get(fqn)
            if cached is None:
                continue
            entry = dict(cached)
            entry["columns"] = {k: dict(v) for k, v in cached["columns"].items()}
            entry["foreign_keys"] = [
                {"column": rel[3],
                 "ref_table": f"{rel[4] or key[0]}.{rel[5] or key[1]}.{rel[6]}".lower(),
                 "ref_column": rel[7],
                 "source": source,
                 "confidence": confidence}
                for rel, source, confidence in self.get_table_relations(*self._canonical_names.get(key, key))
            ]
            evidence[fqn] = entry
        return evidence

    @staticmethod
    def _table_name_forms(table_name, strip_affixes=True):
        name = str(table_name or "").lower()
        bases = {name}
        for affix in _DIMENSION_TABLE_AFFIXES if strip_affixes else ():
            core = name[len(affix):] if affix.endswith("_") and name.startswith(affix) else (
                name[:-len(affix)] if affix.startswith("_") and name.endswith(affix) else "")
            if len(core) >= 2:
                bases.add(core)
        forms = set(bases)
        for base in bases:
            if base.endswith("ies") and len(base) > 4:
                forms.add(base[:-3] + "y")
            elif base.endswith(("sses", "shes", "ches", "xes", "zes")):
                forms.add(base[:-2])
            elif base.endswith("s") and not base.endswith(("ss", "us", "is")) and len(base) > 3:
                forms.add(base[:-1])
        return forms

    @staticmethod
    def _strip_short_prefix(name):
        head, sep, rest = str(name or "").partition("_")
        return rest if sep and rest and len(head) <= 4 else name

    _FK_SUFFIX_RE = re.compile(r"^(?P<stem>.+?)_(?P<tok>id|fk|ref|key|sk|nk)$")
    _FK_PREFIX_RE = re.compile(r"^(?P<tok>fk|id|sk)_(?P<stem>.+)$")
    _FK_CONCAT_RE = re.compile(r"^(?P<stem>[a-z][a-z0-9]{2,}?)(?P<tok>id|key|code|sk)$")
    _FK_ROLE_WORD_ENDINGS = ("ed", "ing")
    _FK_PK_TOKENS = {"id": ("id",), "fk": ("id", "key", "sk"), "ref": ("id", "key", "sk"), "key": ("key",),
                     "sk": ("sk",), "nk": ("nk",), "code": ("code",)}

    def _fk_name_candidates(self, col_lower):
        out = []
        for regex, allow_strip in ((self._FK_SUFFIX_RE, True), (self._FK_PREFIX_RE, False), (self._FK_CONCAT_RE, False)):
            match = regex.match(col_lower)
            if match:
                out.append((match.group("stem"), match.group("tok"), allow_strip))
        return out

    @staticmethod
    def _unique_fk_target(source_key, candidates):
        others = [c for c in candidates if c != source_key]
        for in_scope in (lambda c: c[:2] == source_key[:2], lambda c: c[0] == source_key[0], lambda c: True):
            hits = [c for c in others if in_scope(c)]
            if len(hits) == 1:
                return hits[0]
            if len(hits) > 1:
                return None
        return None

    def _pick_fk_target_column(self, target, stem, token, fk_col_lower, family, tables_columns, stripped_core):
        target_cols = tables_columns.get(target, {})

        def compatible(name_lower):
            return name_lower in target_cols and _metadata_type_family(target_cols[name_lower][1]) == family

        declared = [str(p).lower() for p in self.primary_keys.get(self._lkey(*target), [])]
        if len(declared) == 1 and compatible(declared[0]):
            pk = declared[0]
            if stripped_core is None:
                return target_cols[pk][0], 0.95
            if pk == stripped_core or self._strip_short_prefix(pk) == stripped_core:
                return target_cols[pk][0], 0.85
        if len(declared) > 1:
            return None
        if stripped_core is not None:
            for name_lower in target_cols:
                if (name_lower == stripped_core or self._strip_short_prefix(name_lower) == stripped_core) and compatible(name_lower):
                    return target_cols[name_lower][0], 0.8
            return None
        pk_tokens = self._FK_PK_TOKENS.get(token, (token,))
        bases = [stem] + sorted(self._table_name_forms(target[2]) - {stem})
        token_names = [f"{base}{sep}{ptok}" for ptok in pk_tokens for base in bases for sep in ("_", "")]
        ranked = []
        if fk_col_lower in target_cols:
            ranked.append((fk_col_lower, 0.9))
        ranked.extend((name, 0.85) for name in token_names)
        token_name_set = set(token_names)
        ranked.extend((name_lower, 0.8) for name_lower in target_cols
                      if self._strip_short_prefix(name_lower) != name_lower and self._strip_short_prefix(name_lower) in token_name_set)
        ranked.extend((generic, 0.7) for generic in ("id", "pk") + tuple(pk_tokens))
        for name_lower, confidence in ranked:
            if compatible(name_lower):
                return target_cols[name_lower][0], confidence
        return None

    def _fk_own_key_index(self, tables_columns) -> dict:
        index = defaultdict(list)
        for key, columns in tables_columns.items():
            table = str(key[2]).lower()
            words = [w for w in re.split(r"[^a-z0-9]+", table) if w]
            forms = self._table_name_forms(key[2])
            for col_lower in columns:
                stripped = self._strip_short_prefix(col_lower)
                match = self._FK_SUFFIX_RE.match(stripped) if stripped != col_lower else None
                if match is None:
                    continue
                core = match.group("stem")
                if core in forms or core in words or (len(core) >= 3 and any(w.startswith(core) for w in words + [table])):
                    index[core].append((key, col_lower, col_lower.partition("_")[0], match.group("tok")))
        return index

    def _fk_own_key_target(self, source_key, stems, token, family, own_keys, tables_columns):
        allowed = self._FK_PK_TOKENS.get(token, (token,))
        for stem in stems:
            tiers = (
                own_keys.get(stem, []),
                [entry for core, entries in own_keys.items() if stem.endswith(core) and 1 <= len(stem) - len(core) <= 2
                 for entry in entries if entry[2].startswith(stem[:len(stem) - len(core)])],
                [entry for core, entries in own_keys.items() if len(stem) >= 3 and core != stem and core.startswith(stem)
                 for entry in entries],
            )
            for entries in tiers:
                if not entries:
                    continue
                if any(target == source_key for target, _pk, _head, _tok in entries):
                    return None
                found = {}
                for target, pk, _head, pk_tok in entries:
                    if pk_tok in allowed and _metadata_type_family(tables_columns[target][pk][1]) == family:
                        found.setdefault(target, pk)
                target = self._unique_fk_target(source_key, list(found))
                if target is not None:
                    return target, tables_columns[target][found[target]][0], 0.75
                if found:
                    return None
        return None

    def _infer_foreign_keys(self, all_column_details):
        tables_columns = {}
        for row in all_column_details or []:
            catalog, schema, table, column, dtype = row[0], row[1], row[2], row[3], row[4]
            if not column:
                continue
            tables_columns.setdefault((catalog, schema, table), {}).setdefault(str(column).lower(), (str(column), dtype))
        stem_index, affix_index = defaultdict(list), defaultdict(list)
        for key in tables_columns:
            exact_forms = self._table_name_forms(key[2], strip_affixes=False)
            for form in exact_forms:
                stem_index[form].append(key)
            for form in self._table_name_forms(key[2]) - exact_forms:
                affix_index[form].append(key)

        def _stem_candidates(stem):
            return stem_index.get(stem) or affix_index.get(stem, [])

        own_keys = self._fk_own_key_index(tables_columns)
        inferred = defaultdict(list)
        for key, columns in tables_columns.items():
            for col_lower, (col_original, dtype) in columns.items():
                family = _metadata_type_family(dtype)
                if family not in ("numeric", "string"):
                    continue
                for stem, token, allow_strip in self._fk_name_candidates(col_lower):
                    candidates = _stem_candidates(stem)
                    if key in candidates:
                        break
                    target = self._unique_fk_target(key, candidates)
                    stripped_core = None
                    stems = [stem]
                    if target is None and allow_strip and "_" in stem:
                        parts = stem.split("_")
                        for cut in range(1, min(3, len(parts))):
                            if all(len(p) <= 4 or p.endswith(self._FK_ROLE_WORD_ENDINGS) for p in parts[:cut]):
                                core_stem = "_".join(parts[cut:])
                                stems.append(core_stem)
                                core_candidates = _stem_candidates(core_stem)
                                if key in core_candidates:
                                    continue
                                target = self._unique_fk_target(key, core_candidates)
                                if target is not None:
                                    stripped_core = f"{core_stem}_{token}"
                                    break
                    if target is None:
                        own = self._fk_own_key_target(key, stems, token, family, own_keys, tables_columns)
                        if own is None:
                            continue
                        target, picked = own[0], own[1:]
                    else:
                        picked = self._pick_fk_target_column(target, stem, token, col_lower, family, tables_columns, stripped_core)
                    if picked is None:
                        continue
                    ref_col, confidence = picked
                    rel = (key[0], key[1], key[2], col_original, target[0], target[1], target[2], ref_col)
                    inferred[key].append((rel, confidence))
                    self.logger.debug(f"Inferred FK ({confidence:.2f}): {key[0]}.{key[1]}.{key[2]}.{col_original} -> "
                                      f"{target[0]}.{target[1]}.{target[2]}.{ref_col}")
                    break
        return inferred

    def infer_foreign_keys_by_naming(self, all_column_details: list) -> dict:
        """
        Infer PK-FK relationships from column naming conventions when explicit FK constraints don't exist.
        
        Args:
            all_column_details: List of (catalog, schema, table, column, type, comment) tuples
            
        Returns:
            dict: Mapping of table tuples to list of inferred 8-tuple relationships
        """
        return {key: [rel for rel, _ in scored] for key, scored in self._infer_foreign_keys(all_column_details).items()}
    
    def merge_inferred_foreign_keys(self, all_column_details: list):
        """
        Merge inferred FK relationships into the foreign_key_graph.
        Only adds inferred relationships for tables that don't already have declared FKs.
        
        Args:
            all_column_details: List of all column details to analyze
        """
        inferred = self._infer_foreign_keys(all_column_details)
        added_count = 0
        skipped_declared = 0
        with self._metadata_lock:
            self._join_inferred_schemas.update(self._lkey(row[0], row[1], row[2])[:2] for row in all_column_details or [])
            for table_key, scored in inferred.items():
                existing = self.foreign_key_graph.get(table_key, [])
                if any(rel not in self._inferred_fk for rel in existing):
                    skipped_declared += len(scored)
                    continue
                for rel, confidence in scored:
                    if rel in existing:
                        continue
                    self.foreign_key_graph[table_key].append(rel)
                    existing = self.foreign_key_graph[table_key]
                    self._inferred_fk[rel] = confidence
                    added_count += 1
        
        if added_count > 0:
            self.logger.info(f"✅ Inferred {added_count} additional FK relationships from column naming patterns")
        if skipped_declared:
            self.logger.info(f"🔗 Skipped {skipped_declared} naming-inferred FK(s) on tables that already declare FK constraints")
        
        return added_count

    def _scope_tables_in_schema(self, catalog, schema):
        cat, sch, _ = self._lkey(catalog, schema, "")
        if cat in {self._lkey(c, "", "")[0] for c in self.catalogs_to_process}:
            return None
        if (cat, sch) in {self._lkey(*parse_two_level_name(s), "")[:2] for s in self.schemas_to_process}:
            return None
        named = (self._lkey(*parse_three_level_name(t)) for t in self.tables_to_process)
        return {key[2] for key in named if key[:2] == (cat, sch) and key[2]}

    def _schema_table_names(self, catalog, schema):
        try:
            rows = _metadata_rows(self.spark,
                                  f"SELECT table_name FROM {quote_identifier(catalog)}.`information_schema`.`tables` "
                                  f"WHERE table_schema = :schema",
                                  args={"schema": schema}, logger=self.logger, context=f"table list {catalog}.{schema}")
        except SparkWorkCancelled:
            raise
        except Exception as e:
            self.logger.warning(f"[SCOPE-JOINS] cannot list the tables of {catalog}.{schema} ({get_clean_error_message(e)}); "
                                f"joins are inferred among the cited tables only")
            return set()
        return {str(row.table_name).lower() for row in rows if row.table_name}

    def infer_scope_joins(self, table_fqns) -> int:
        requested = {}
        for fqn in table_fqns or []:
            key = self._lkey(*parse_three_level_name(str(fqn or "")))
            if all(key):
                requested.setdefault(key[:2], set()).add(key[2])
        added = 0
        for (cat, sch), names in sorted(requested.items()):
            with self._join_inference_lock:
                if (cat, sch) in self._join_inferred_schemas or not self._catalog_capable(cat):
                    continue
                scope = self._scope_tables_in_schema(cat, sch)
                tables = sorted((self._schema_table_names(cat, sch) if scope is None else scope) | names)
                missing = [t for t in tables if (cat, sch, t) not in self._table_columns]
                if missing:
                    self._load_schema_metadata(cat, sch, missing)
                details = [(*self._canonical_names.get((cat, sch, t), (cat, sch, t)), col["name"], col["type"], col["comment"])
                           for t in tables for col in self._table_columns.get((cat, sch, t)) or []]
                count = self.merge_inferred_foreign_keys(details) if details else 0
                self._join_inferred_schemas.add((cat, sch))
            added += count
            if count:
                self.logger.info(f"[SCOPE-JOINS FIRED] {cat}.{sch}: {count} join(s) inferred from column names over {len(tables)} "
                                 f"table(s) of this run's scope, as Discover infers them over its scan")
            else:
                self.logger.info(f"[SCOPE-JOINS] {cat}.{sch}: no join inferred from column names over {len(tables)} table(s)")
        return added

    # === REMOVED: _fill_from_direct_tables, _fill_from_schemas, _fill_from_catalogs ===

    # === MODIFIED: Support both single-pass and two-pass optimized modes ===
    def getNextTables(self, batch_size=None):
        """
        Returns a batch of tables with their column details.
        Maintains state across all catalogs and databases, continuing seamlessly
        across database boundaries.
        
        Supports two modes:
        1. Single-pass mode (default): Uses batch_size parameter for simple batching
        2. Two-pass mode (enable_two_pass=True): Uses pre-optimized batches, ignores batch_size
        
        Args:
            batch_size (int): Maximum number of tables to return in this batch (single-pass mode only)
            
        Returns:
            list of tuples: Each tuple is (catalog, schema, table, column_name, data_type, comment)
            None: if all tables have been processed
        """
        
        # Initialize table list on first call
        if not self._tables_initialized:
            self.logger.info("Initializing all tables from all databases...")
            self._initialize_all_tables()
            self._tables_initialized = True
        
        # === TWO-PASS OPTIMIZED MODE ===
        if self.enable_two_pass and self._size_analysis_complete:
            # Check if we've exhausted all batches
            if self.current_batch_idx >= len(self.optimized_batches):
                return None  # Signal no more batches
            
            # Get the next optimized batch
            batch_table_tuples = self.optimized_batches[self.current_batch_idx]
            batch_num = self.current_batch_idx + 1
            
            self.logger.info(f"📦 Fetching optimized batch {batch_num}/{len(self.optimized_batches)}: "
                           f"{len(batch_table_tuples)} tables")
            
            # Fetch column details for this optimized batch in parallel
            # ADAPTIVE PARALLELISM: Fixed for metadata (DB connection limits)
            column_parallelism, reason = calculate_adaptive_parallelism(
                "column_fetch", self.max_parallelism, 
                num_items=len(batch_table_tuples),
                is_llm_operation=False, logger=self.logger
            )
            
            all_column_details = self._fetch_batch_details(batch_table_tuples, column_parallelism, "TableDetailsBatch",
                                                           f"Batch {batch_num}")
            
            # Update position for next call
            self.current_batch_idx += 1
            
            # Return the column details for this optimized batch
            return all_column_details
        
        # === SINGLE-PASS MODE (backward compatible) ===
        else:
            if batch_size is None:
                self.logger.warning("batch_size not provided in single-pass mode, using default of 25")
                batch_size = 25
                
            # Check if we've exhausted all tables
            if self.current_table_idx >= len(self.all_table_tuples):
                return None  # Signal no more tables
                
            # Get the next batch of tables
            end_idx = min(self.current_table_idx + batch_size, len(self.all_table_tuples))
            batch_table_tuples = self.all_table_tuples[self.current_table_idx:end_idx]
            
            self.logger.info(f"Fetching batch of {len(batch_table_tuples)} tables (indices {self.current_table_idx} to {end_idx-1} of {len(self.all_table_tuples)} total)")
            
            # Fetch column details for this batch of tables in parallel
            # ADAPTIVE PARALLELISM: Fixed for metadata (DB connection limits)
            column_parallelism, reason = calculate_adaptive_parallelism(
                "column_fetch", self.max_parallelism,
                num_items=len(batch_table_tuples),
                is_llm_operation=False, logger=self.logger
            )
            
            all_column_details = self._fetch_batch_details(batch_table_tuples, column_parallelism, "TableDetailsSinglePass",
                                                           f"Tables {self.current_table_idx}-{end_idx - 1}")
            
            # Update position for next call
            self.current_table_idx = end_idx
            
            # Return the column details for this batch
            return all_column_details
    
    def reset(self):
        """Reset the data loader to start from the beginning."""
        self.current_table_idx = 0
        self.current_batch_idx = 0
        self.logger.info("DataLoader reset to beginning")
    
    def _initialize_all_tables(self):
        """
        Fetches all table names from all databases in the queue and stores them
        as (catalog, schema, table) tuples for later batch processing.
        
        Supports both streaming (memory-efficient) and two-pass (size-optimized) modes.
        """
        self.logger.info(f"Discovering all tables from {len(self.database_queue)} databases...")
        
        # Check if we have specific individual tables to filter by
        individual_tables_set = set(self.individual_tables) if self.individual_tables else None
        
        # Get set of explicitly provided schemas (these should expand ALL tables, not filter)
        explicit_schemas_set = getattr(self, 'explicit_schemas_set', set())
        
        # Log discovery strategy
        if explicit_schemas_set and individual_tables_set:
            explicit_schema_names = [f"{cat}.{db}" for cat, db in explicit_schemas_set]
            self.logger.info(f"📋 Discovery strategy: {len(explicit_schemas_set)} explicit schemas will expand ALL tables: {', '.join(explicit_schema_names)}")
            self.logger.info(f"📋 {len(individual_tables_set)} individual tables will be included from other schemas")
        
        # === TWO-PASS MODE: First analyze table sizes, then create optimized batches ===
        if self.enable_two_pass:
            self.logger.info("🔄 TWO-PASS MODE ENABLED: Analyzing table sizes for intelligent batching...")
            self.logger.info(f"⚡ Using parallel schema discovery with {self.max_parallelism} workers for speed")
            
            # PASS 1: Discover all tables and analyze their sizes IN PARALLEL
            all_table_size_infos = []
            all_raw_table_tuples = []
            
            # Function to process a single schema in parallel
            def discover_and_analyze_schema(schema_tuple):
                catalog_name, schema_name = schema_tuple
                cat_normalized = normalize_identifier(catalog_name)
                schema_normalized = normalize_identifier(schema_name)
                db_fqn = build_fqn(cat_normalized, schema_normalized)
                self.logger.info(f"🔍 Starting discovery for {db_fqn}...")
                
                schema_table_tuples = []
                schema_key = (cat_normalized, schema_normalized)
                is_explicit_schema = schema_key in explicit_schemas_set
                
                # Check if schema has a huge number of tables (>10K) - use streaming
                try:
                    self.logger.debug(f"   Counting tables in {db_fqn}...")
                    cat_quoted = quote_identifier(cat_normalized)
                    count_query = f"""
                        SELECT COUNT(*) as cnt FROM {cat_quoted}.`information_schema`.`tables`
                        WHERE `table_schema` = :schema
                    """
                    table_count = _metadata_rows(self.spark, count_query, args={"schema": schema_normalized.lower()},
                                                 logger=self.logger, context=f"table count {db_fqn}")[0].cnt
                    self.logger.debug(f"   {db_fqn} has {table_count} tables")
                    use_streaming = table_count > 10000
                except SparkWorkCancelled:
                    raise
                except Exception as e:
                    # Extract clean error message without stack trace for console
                    error_msg = get_clean_error_message(e)
                    self.logger.warning(f"   Could not count tables in {db_fqn}: {error_msg}. Using non-streaming mode.")
                    self.logger.debug(f"   Full error details: {get_clean_error_message(e, max_chars=2000)}")  # Full trace to log file only
                    use_streaming = False  # Fallback to non-streaming
                
                if use_streaming:
                    self.logger.info(f"   Large schema detected in {db_fqn} ({table_count:,} tables). Using streaming mode...")
                    # Process in chunks
                    for table_chunk in self._fetch_tables_for_schema_streaming(cat_normalized, schema_normalized, chunk_size=self.streaming_batch_size):
                        for fq_table_name in table_chunk:
                            cat, db, tbl = parse_three_level_name(fq_table_name)
                            if not (cat and db and tbl):
                                self.logger.warning(f"Skipping malformed table name: {fq_table_name}")
                                continue
                            table_tuple = (cat, db, tbl)
                            
                            # Only filter by individual_tables if this schema was NOT explicitly provided
                            # Explicit schemas should expand ALL tables
                            if not is_explicit_schema and individual_tables_set and table_tuple not in individual_tables_set:
                                continue
                            
                            schema_table_tuples.append(table_tuple)
                else:
                    # Non-streaming mode for smaller schemas
                    table_names_fq = self._fetch_tables_for_schema(cat_normalized, schema_normalized)
                    
                    if not table_names_fq:
                        self.logger.debug(f"No tables found in {db_fqn}. Skipping.")
                        return []  # Return empty list for this schema
                    
                    for fq_table_name in table_names_fq:
                        cat, db, tbl = parse_three_level_name(fq_table_name)
                        if not (cat and db and tbl):
                            self.logger.warning(f"Skipping malformed table name: {fq_table_name}")
                            continue
                        table_tuple = (cat, db, tbl)
                        
                        # Only filter by individual_tables if this schema was NOT explicitly provided
                        # Explicit schemas should expand ALL tables
                        if not is_explicit_schema and individual_tables_set and table_tuple not in individual_tables_set:
                            continue
                        
                        schema_table_tuples.append(table_tuple)
                
                # Analyze sizes for this schema's tables
                schema_size_infos = []
                if schema_table_tuples:
                    self.logger.debug(f"   Analyzing column counts for {len(schema_table_tuples)} tables in {db_fqn}...")
                    
                    # Process in chunks to avoid memory issues
                    # IMPORTANT: Use max_parallelism=1 to avoid nested parallelism deadlock
                    # (outer loop already runs schemas in parallel with self.max_parallelism workers)
                    chunk_size = 100
                    for i in range(0, len(schema_table_tuples), chunk_size):
                        chunk = schema_table_tuples[i:i+chunk_size]
                        size_infos = self.size_analyzer.analyze_table_sizes_batch(chunk, self.catalog_capabilities, max_parallelism=1)
                        schema_size_infos.extend(size_infos)
                    
                    self.logger.info(f"   ✓ {db_fqn}: Found {len(schema_table_tuples)} tables")
                
                return (schema_table_tuples, schema_size_infos)
            
            # ADAPTIVE PARALLELISM: Fixed for metadata (DB connection limits)
            discovery_parallelism, reason = calculate_adaptive_parallelism(
                "schema_discovery", self.max_parallelism,
                num_items=len(self.database_queue),
                is_llm_operation=False, logger=self.logger
            )
            log_adaptive_parallelism_decision("schema_discovery", discovery_parallelism, self.max_parallelism, reason)
            
            with _SafeExecutorContext(max_workers=discovery_parallelism, thread_name_prefix="SchemaDiscovery", logger=self.logger, name="SchemaDiscovery") as ctx:
                futures = {ctx.submit(discover_and_analyze_schema, schema_tuple): schema_tuple 
                          for schema_tuple in self.database_queue}
                
                completed = 0
                total = len(self.database_queue)
                self.logger.info(f"      Submitted {total} schemas for parallel discovery...")
                
                timeout_per_schema = self.schema_timeout_seconds
                
                try:
                    for future in ctx.completed(futures, timeout=timeout_per_schema * total):
                        schema_tuple = futures[future]
                        completed += 1
                        try:
                            _raw_tuples, schema_size_infos = future.result(timeout=timeout_per_schema)
                            all_raw_table_tuples.extend(_raw_tuples)
                            all_table_size_infos.extend(schema_size_infos)
                            self.logger.info(f"      Progress: {completed}/{total} schemas analyzed, {len(all_table_size_infos)} tables discovered so far")
                        except concurrent.futures.TimeoutError:
                            timeout_minutes = timeout_per_schema // 60
                            self.logger.error(f"⏱️  Timeout analyzing schema {schema_tuple} (>{timeout_minutes} min). Skipping this schema.")
                            self._record_metadata_unavailable((schema_tuple[0], schema_tuple[1], "*"),
                                                              f"schema discovery timed out after {timeout_per_schema}s")
                        except Exception as e:
                            self.logger.error(f"Failed to analyze schema {schema_tuple}: {get_clean_error_message(e)}")
                            self._record_metadata_unavailable((schema_tuple[0], schema_tuple[1], "*"),
                                                              f"schema discovery failed: {get_clean_error_message(e, max_chars=200)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error(f"⏱️  Overall schema discovery timeout ({timeout_per_schema * total}s). Proceeding with {completed}/{total} schemas.")
                    for _pending_future, _pending_schema in futures.items():
                        if not _pending_future.done():
                            self._record_metadata_unavailable((_pending_schema[0], _pending_schema[1], "*"),
                                                              f"schema discovery did not finish within {timeout_per_schema * total}s")
                
                if completed < total:
                    self.logger.warning(f"⚠️  Only {completed}/{total} schemas completed. {total - completed} schemas timed out or failed.")
            
            self.logger.info(f"📊 Pass 1 complete: Analyzed {len(all_table_size_infos)} tables")
            
            # Deduplicate tables (in case same table was included via both schema and individual table)
            seen_tables = set()
            unique_table_size_infos = []
            for info in all_table_size_infos:
                table_key = (info.catalog, info.schema, info.table)
                if table_key not in seen_tables:
                    seen_tables.add(table_key)
                    unique_table_size_infos.append(info)
            
            if len(unique_table_size_infos) < len(all_table_size_infos):
                duplicates_removed = len(all_table_size_infos) - len(unique_table_size_infos)
                self.logger.info(f"🔄 Deduplicated: Removed {duplicates_removed} duplicate tables, {len(unique_table_size_infos)} unique tables remaining")
                all_table_size_infos = unique_table_size_infos
            
            _sized_keys = {(info.catalog, info.schema, info.table) for info in all_table_size_infos}
            _missing_seen = set()
            _missing_ordered = []
            for _raw in all_raw_table_tuples:
                if _raw not in _sized_keys and _raw not in _missing_seen:
                    _missing_seen.add(_raw)
                    _missing_ordered.append(_raw)
            if _missing_ordered:
                _rescued_infos = [TableSizeInfo(_c, _s, _t, num_columns=0, size_category="unknown") for (_c, _s, _t) in _missing_ordered]
                all_table_size_infos.extend(_rescued_infos)
                _rescued_schemas = sorted({(_c, _s) for (_c, _s, _t) in _missing_ordered})
                self.logger.warning(
                    f"[G19 ZERO_BATCH_FALLBACK FIRED] rescued={len(_missing_ordered)} table(s) with size_category=unknown "
                    f"(column count not measured) - size analysis did not return them (timeout or failed schema analysis). "
                    f"Schemas affected: {_rescued_schemas[:10]}"
                )
            
            # PASS 2: Create optimized batches based on size analysis
            self.logger.info("🎯 Pass 2: Creating optimized batches based on table sizes...")
            self.optimized_batches = self.batch_optimizer.create_optimized_batches(all_table_size_infos)
            self._size_analysis_complete = True
            
            # Store all table tuples for reference
            self.all_table_tuples = [
                (info.catalog, info.schema, info.table) for info in all_table_size_infos
            ]
            
            self.logger.info(f"✅ Two-pass initialization complete: {len(self.all_table_tuples)} tables in {len(self.optimized_batches)} optimized batches")
        
        # === SINGLE-PASS MODE: Standard behavior (for backward compatibility) ===
        else:
            self.logger.info("📋 SINGLE-PASS MODE: Standard table discovery with parallel queries...")
            self.logger.info(f"⚡ Using {self.max_parallelism} parallel workers for schema queries")
            
            # Function to fetch tables for a single schema
            def fetch_schema_tables(schema_tuple):
                catalog_name, schema_name = schema_tuple
                cat_normalized = normalize_identifier(catalog_name)
                schema_normalized = normalize_identifier(schema_name)
                db_fqn = build_fqn(cat_normalized, schema_normalized)
                self.logger.info(f"🔍 Starting table fetch for {db_fqn}...")
                
                schema_key = (cat_normalized, schema_normalized)
                is_explicit_schema = schema_key in explicit_schemas_set
                
                # Fetch all table names for this database
                table_names_fq = self._fetch_tables_for_schema(cat_normalized, schema_normalized)
                
                schema_tuples = []
                if not table_names_fq:
                    self.logger.debug(f"No tables found in {db_fqn}. Skipping.")
                    return schema_tuples
                    
                # Parse into (cat, schema, table) tuples
                for fq_table_name in table_names_fq:
                    cat, db, tbl = parse_three_level_name(fq_table_name)
                    if not (cat and db and tbl):
                        self.logger.warning(f"Skipping malformed table name: {fq_table_name}")
                        continue
                    table_tuple = (cat, db, tbl)
                    
                    # Explicit schemas expand ALL tables; otherwise filter by individual_tables
                    if is_explicit_schema:
                        # Schema was explicitly provided - include ALL tables
                        schema_tuples.append(table_tuple)
                    elif individual_tables_set:
                        # Schema came from individual tables - only include those specific tables
                        if table_tuple in individual_tables_set:
                            schema_tuples.append(table_tuple)
                    else:
                        # No individual tables specified - include all
                        schema_tuples.append(table_tuple)
                
                self.logger.info(f"   ✓ {db_fqn}: Found {len(schema_tuples)} tables")
                return schema_tuples
            
            # ADAPTIVE PARALLELISM: Fixed for metadata (DB connection limits)
            table_discovery_parallelism, reason = calculate_adaptive_parallelism(
                "table_discovery", self.max_parallelism,
                num_items=len(self.database_queue),
                is_llm_operation=False, logger=self.logger
            )
            log_adaptive_parallelism_decision("table_discovery", table_discovery_parallelism, self.max_parallelism, reason)
            
            with _SafeExecutorContext(max_workers=table_discovery_parallelism, thread_name_prefix="TableDiscovery", logger=self.logger, name="TableDiscovery") as ctx:
                futures = {ctx.submit(fetch_schema_tables, schema_tuple): schema_tuple 
                          for schema_tuple in self.database_queue}
                
                completed = 0
                total = len(self.database_queue)
                self.logger.info(f"      Submitted {total} schemas for parallel discovery...")
                
                timeout_per_schema = self.schema_timeout_seconds
                
                try:
                    for future in ctx.completed(futures, timeout=timeout_per_schema * total):
                        schema_tuple = futures[future]
                        completed += 1
                        try:
                            schema_tuples = future.result(timeout=timeout_per_schema)
                            self.all_table_tuples.extend(schema_tuples)
                            self.logger.info(f"      Progress: {completed}/{total} schemas processed, {len(self.all_table_tuples)} tables discovered so far")
                        except concurrent.futures.TimeoutError:
                            timeout_minutes = timeout_per_schema // 60
                            self.logger.error(f"⏱️  Timeout fetching tables for schema {schema_tuple} (>{timeout_minutes} min). Skipping this schema.")
                            self._record_metadata_unavailable((schema_tuple[0], schema_tuple[1], "*"),
                                                              f"table discovery timed out after {timeout_per_schema}s")
                        except Exception as e:
                            self.logger.error(f"Failed to fetch tables for schema {schema_tuple}: {get_clean_error_message(e)}")
                            self._record_metadata_unavailable((schema_tuple[0], schema_tuple[1], "*"),
                                                              f"table discovery failed: {get_clean_error_message(e, max_chars=200)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error(f"⏱️  Overall table discovery timeout ({timeout_per_schema * total}s). Proceeding with {completed}/{total} schemas.")
                    for _pending_future, _pending_schema in futures.items():
                        if not _pending_future.done():
                            self._record_metadata_unavailable((_pending_schema[0], _pending_schema[1], "*"),
                                                              f"table discovery did not finish within {timeout_per_schema * total}s")
                
                if completed < total:
                    self.logger.warning(f"⚠️  Only {completed}/{total} schemas completed. {total - completed} schemas timed out or failed.")
            
            # Deduplicate tables (in case same table was included via both schema and individual table)
            original_count = len(self.all_table_tuples)
            self.all_table_tuples = list(dict.fromkeys(self.all_table_tuples))  # Preserves order, removes duplicates
            if len(self.all_table_tuples) < original_count:
                duplicates_removed = original_count - len(self.all_table_tuples)
                self.logger.info(f"🔄 Deduplicated: Removed {duplicates_removed} duplicate tables, {len(self.all_table_tuples)} unique tables remaining")
            
            self.logger.info(f"Table discovery complete. Found {len(self.all_table_tuples)} total tables across all databases.")

# COMMAND ----------

# DBTITLE 1,Chart Type Reference Guide
# Chart Type Reference Guide:
# The generator maps SQL columns to chart fields in two ways:
# 1. By Prefix: Looks for columns starting with a specific prefix (e.g., `category_`, `value_`).
# 2. By Position: If no prefixes are found, it uses the column order from the SELECT statement.
#
# Supported formats (optional columns in brackets `[]`):
# --------------------------------------------------------------------------------
# Bar Chart:      category_col, value_col, [group_col]
# Pie Chart:      category_col, value_col
# Line Chart:     x_axis_col, y_axis_col, [group_col]
# Area Chart:     x_axis_col, y_axis_col, [group_col]
# Scatter Plot:   x_axis_col, y_axis_col, [group_col]
# Heatmap:        x_axis_col, y_axis_col, color_col
# Combo Chart:    x_axis_col, bar_value_col, line_value_col
#
# Counter:        value_col
# Histogram:      value_col
# Funnel Chart:   stage_col, value_col
#
# Box Plot:       category_col, min_col, q1_col, median_col, q3_col, max_col
# Sankey Chart:   source_col, destination_col, value_col
# Pivot Table:    row_col, column_col, cell_value_col
# Choropleth Map: location_col, value_col
# Symbol Map:     lat_col, lon_col, [size_col], [group_col]
#
# Table:          col_1, col_2, ... (all columns are displayed)
# --------------------------------------------------------------------------------

# (Chart Type Reference Guide... unchanged)

class WidgetFailedToCreate(Exception):
    """Custom exception raised when widget creation fails due to unmet requirements."""
    pass

TECHNIQUE_EXECUTION_FAMILY = {
    "Classification": "A",
    "Predictive Modeling": "A",
    "Survival Analysis": "A",
    "Anomaly Detection": "B",
    "Clustering": "C",
    "Segmentation": "C",
    "Recommendation": "D",
    "Forecasting": "E",
    "Trend Analysis": "F",
    "Correlation Analysis": "F",
    "Pareto Analysis": "F",
    "Cohort Analysis": "F",
    "Funnel Analysis": "F",
    "Regression Analysis": "F",
    "Geospatial Analysis": "F",
    "Network Analysis": "F",
    "AI Analysis": "F",
    "Sentiment Analysis": "G",
    "Document Processing": "G",
    "Extraction": "G",
    "Optimization": "H",
    "Simulation": "H",
}

GENIE_FAMILY_SPECS = {
    "A": {"label": "Supervised scoring", "value_type": "probability",
          "pattern": ("every run trains on closed outcome windows whose horizon ends before the most recent closed window, compares the "
                      "candidates with the baseline on that most recent closed window, "
                      "registers the accepted model in Unity Catalog as `{model_name}` with the alias champion, and scores the entities whose outcome is still open")},
    "B": {"label": "Anomaly scoring", "value_type": "anomaly score",
          "pattern": ("every run fits the detector on a reference window, checks it on held-out reference data, registers the accepted detector "
                      "in Unity Catalog as `{model_name}` with the alias champion, and scores the current window")},
    "C": {"label": "Segmentation", "value_type": "segment measure",
          "pattern": ("every run fits an assignable segmentation model, checks segment quality and stability, registers the accepted model in "
                      "Unity Catalog as `{model_name}` with the alias champion, and assigns every entity to a segment")},
    "D": {"label": "Recommendation", "value_type": "recommendation score",
          "pattern": ("every run fits the recommender on all past interactions (eligibility rules such as margin, stock or items already "
                      "bought apply only when ranking, never before fitting), checks it against the popularity baseline on a time-based holdout, "
                      "registers the accepted model in Unity Catalog as `{model_name}` with the alias champion, and writes the top recommendations per entity")},
    "E": {"label": "Forecasting", "value_type": "forecast",
          "pattern": ("every run refits the forecaster on the full history up to the as-of date (the period grid runs from the first period "
                      "with data to the as-of date, never past the data), backtests it against the seasonal naive forecast, and writes the "
                      "forecast with its prediction interval; nothing is registered")},
    "F": {"label": "Analytical refresh", "value_type": "effect size",
          "pattern": "every run recomputes each finding with its effect size against its baseline and significance corrected for multiple tests; nothing is registered"},
    "G": {"label": "AI text processing", "value_type": "score",
          "pattern": ("every run processes only the rows newer than the latest as_of_date already in the business output table, with Databricks "
                      "AI functions as the core method; nothing is registered")},
    "H": {"label": "Optimization and simulation", "value_type": "planned value",
          "pattern": ("every run solves or simulates the plan for the next planning cycle with a fixed seed and compares it with the current "
                      "plan; simulations draw from aggregates computed once before the draw loop, never from a scan of the source rows "
                      "inside it; nothing is registered")},
}

FAMILY_F_SHORTFALL_GUIDANCE = ("If data is insufficient: {units} below the minimum rows are left out of the findings and counted{kept}; "
                               "fewer reportable {units} than `min_reportable_groups`, or fewer periods than `min_periods`, stop the run at "
                               "the readiness gate.")
TECHNIQUE_IMPLEMENTATION_GUIDANCE = {
    "Anomaly Detection": (
        "Method: Isolation Forest, or Local Outlier Factor with novelty=True, from scikit-learn on numeric signals, fitted on a reference window that always precedes the scored window.\n"
        "Baseline: a robust z-score rule (median and MAD) on the same signals.\n"
        "Accept: the alerts fit the review budget, the top alerts of two fits with different seeds agree at the stability threshold, and precision on any known cases is not below the baseline.\n"
        "If data is insufficient: too few reference rows or mostly missing key signals stop the run at the readiness gate; never score with a detector trained on a thin window."
    ),
    "Forecasting": (
        "Method: refit on the full history every run (statsmodels exponential smoothing or SARIMAX per series, or one gradient-boosted model with calendar and lag features across series); never register per-series models. ai_forecast() runs only through a SQL warehouse, not in this notebook.\n"
        "Baseline: the seasonal naive forecast (the value one season earlier).\n"
        "Accept: a rolling-origin backtest with at least three folds gives MASE below 1 against the seasonal naive forecast; otherwise the notebook publishes the seasonal naive forecast, flagged.\n"
        "If data is insufficient: fewer than two seasonal cycles or many missing periods stop the run at the readiness gate; stale data flags every row and never stops it."
    ),
    "Classification": (
        "Method: a scikit-learn HistGradientBoostingClassifier or a regularised logistic regression on as-of features, with class weights for imbalance; frame the outcome as an event per the outcome definition.\n"
        "Baseline: the class rate plus a balanced logistic regression on the strongest signals.\n"
        "Accept: out-of-time AUC and PR-AUC meet the thresholds and AUC beats the baseline by the minimum gain, measured on the most recent closed outcome window (time-aware split, never a random split across time).\n"
        "If data is insufficient: too few positives, an extreme class rate or fewer than two closed outcome windows stop the run at the readiness gate."
    ),
    "Predictive Modeling": (
        "Method: a scikit-learn HistGradientBoostingClassifier on as-of features predicting the outcome event in the outcome definition (a continuous target becomes the event of crossing its business threshold within the window), with class weights for imbalance.\n"
        "Baseline: the class rate plus a balanced logistic regression on the strongest signals.\n"
        "Accept: out-of-time AUC and PR-AUC meet the thresholds and AUC beats the baseline by the minimum gain on the most recent closed outcome window (time-aware split).\n"
        "If data is insufficient: too few positives, an extreme class rate or fewer than two closed outcome windows stop the run at the readiness gate."
    ),
    "Clustering": (
        "Method: K-Means or a Gaussian mixture inside a scikit-learn Pipeline with imputation and scaling, choosing the number of segments by silhouette. Use only models that can assign new entities, so segment IDs stay stable between runs.\n"
        "Baseline: quantile tiers on the most variable feature.\n"
        "Accept: silhouette, the smallest segment share and bootstrap stability (adjusted Rand index) meet the thresholds.\n"
        "If data is insufficient: too few entities for the number of segments or fewer than three usable numeric features stop the run at the readiness gate."
    ),
    "Segmentation": (
        "Method: K-Means or a Gaussian mixture inside a scikit-learn Pipeline with imputation and scaling, on behaviour measures such as recency, frequency and value per entity; choose the number of segments by silhouette. Use only models that can assign new entities, so segment IDs stay stable between runs.\n"
        "Baseline: quantile tiers on the most variable feature.\n"
        "Accept: silhouette, the smallest segment share and bootstrap stability (adjusted Rand index) meet the thresholds.\n"
        "If data is insufficient: too few entities for the number of segments or fewer than three usable numeric features stop the run at the readiness gate."
    ),
    "Regression Analysis": (
        "Method: driver analysis with a regularised linear or generalised linear model (statsmodels) of the KPI on candidate drivers; report each driver's effect size with a confidence interval and p-value, and check multicollinearity (VIF) before interpreting coefficients.\n"
        "Baseline: the KPI's prior-period or overall level (a model with no drivers).\n"
        "Accept: every reported driver carries an effect size and a Benjamini-Hochberg corrected significance; non-significant drivers are reported as Monitor only.\n"
        + FAMILY_F_SHORTFALL_GUIDANCE.format(units="groups", kept="")
    ),
    "Cohort Analysis": (
        "Method: cohorts by first-event period, leaving out the cohort of the data's first period (its members may predate the data); retention or conversion per cohort and period; compare each cohort with the prior cohorts that started in the same season (the same quarter or month of the year) at the same period offset, on the offsets that are closed for every cohort compared, using a two-proportion test.\n"
        "Baseline: the prior same-season cohorts' rate for the same period offset; with no earlier same-season cohort, all prior cohorts, with the row flagged 'no same-season baseline'.\n"
        "Accept: every reported difference carries an effect size and a Benjamini-Hochberg corrected significance.\n"
        + FAMILY_F_SHORTFALL_GUIDANCE.format(units="cohorts", kept="")
    ),
    "Trend Analysis": (
        "Method: per series, statsmodels OLS of the observed values on time plus season (month or week of year) indicators over the full history, with HAC standard errors when there are at least 30 points; the time coefficient is the trend. Never test an STL component or the observed values minus one, never apply a smoother before the test, and never write a test statistic by hand; flag level shifts where the trend changes.\n"
        "Baseline: the prior period's level.\n"
        "Accept: every reported trend or shift carries an effect size and a Benjamini-Hochberg corrected significance.\n"
        + FAMILY_F_SHORTFALL_GUIDANCE.format(units="series", kept="")
    ),
    "Correlation Analysis": (
        "Method: Spearman or Pearson correlation with p-values, plus partial correlation controlling for the obvious shared drivers; state that correlation is not causation.\n"
        "Baseline: no relationship (zero correlation).\n"
        "Accept: every reported relationship carries an effect size and a Benjamini-Hochberg corrected significance.\n"
        + FAMILY_F_SHORTFALL_GUIDANCE.format(units="pairs", kept="")
    ),
    "Pareto Analysis": (
        "Method: cumulative share of the KPI by contributor with the Gini coefficient, and a bootstrap confidence interval for the top contributors' share.\n"
        "Baseline: an even split across contributors and the prior period's concentration.\n"
        "Accept: every tested finding is a contributor's own, such as its change in share against the prior period, and carries an effect size and a Benjamini-Hochberg corrected significance; that finding is the contributor's value and decision test, and a group's concentration, its Gini or its change is reported in the summary, never as the value or the test of its contributors' rows.\n"
        + FAMILY_F_SHORTFALL_GUIDANCE.format(units="groups", kept=", and every contributor of a reported group keeps its place in that group's share")
    ),
    "Funnel Analysis": (
        "Method: stage-to-stage conversion from stage or status columns with confidence intervals, counting only the entrants whose conversion window is closed; compare segments with a two-proportion test.\n"
        "Baseline: the overall conversion rate and the prior period's rate.\n"
        "Accept: every reported bottleneck carries an effect size and a Benjamini-Hochberg corrected significance.\n"
        + FAMILY_F_SHORTFALL_GUIDANCE.format(units="segments", kept=", and every stage of a reported segment keeps its conversion")
    ),
    "Sentiment Analysis": (
        "Method: Databricks ai_analyze_sentiment, or ai_query with a JSON response format and failOnError => false, as the core method on new text since the last watermark.\n"
        "Baseline: a keyword rule checked on a 50-row spot sample.\n"
        "Accept: the AI function error rate is at or below the threshold.\n"
        "If data is insufficient: too little text coverage or very short texts stop the run at the readiness gate."
    ),
    "Document Processing": (
        "Method: Databricks ai_parse_document, ai_classify, ai_extract, ai_summarize or ai_query with a JSON response format and failOnError => false as the core method; validate the returned labels or fields against each other.\n"
        "Baseline: a code or pattern rule checked on a 50-row spot sample.\n"
        "Accept: the AI function error rate is at or below the threshold.\n"
        "If data is insufficient: too little text coverage or very short texts stop the run at the readiness gate."
    ),
    "Extraction": (
        "Method: Databricks ai_extract or ai_query with a JSON response format and failOnError => false as the core method, then normalise and deduplicate the extracted entities.\n"
        "Baseline: a pattern rule checked on a 50-row spot sample.\n"
        "Accept: the AI function error rate is at or below the threshold.\n"
        "If data is insufficient: too little text coverage or very short texts stop the run at the readiness gate."
    ),
    "AI Analysis": (
        "Method: the simplest analysis that answers the decision: an aggregate comparison with significance, a scikit-learn model, or a Databricks AI function when meaning in text is the signal.\n"
        "Baseline: the prior period or the overall rate.\n"
        "Accept: every reported finding carries an effect size and a Benjamini-Hochberg corrected significance.\n"
        + FAMILY_F_SHORTFALL_GUIDANCE.format(units="groups", kept="")
    ),
    "Simulation": (
        "Method: Monte Carlo with numpy default_rng and a fixed seed, at least the minimum number of draws, sampling inputs from distributions estimated on history; report P10, P50 and P90 per entity.\n"
        "Baseline: the current plan or status quo evaluated under the same draws.\n"
        "Accept: the seed is fixed and the draw count meets the threshold, so the same seed gives identical results.\n"
        "If data is insufficient: inputs whose distributions cannot be estimated from the data stop the run at the readiness gate."
    ),
    "Geospatial Analysis": (
        "Method: aggregate to cells with the Databricks h3_* SQL functions or to the existing region columns, compute rates per cell and compare cells with a rate test; use haversine distances where proximity matters.\n"
        "Baseline: the overall rate and the prior period's rate per cell.\n"
        "Accept: every reported hotspot carries an effect size and a Benjamini-Hochberg corrected significance.\n"
        + FAMILY_F_SHORTFALL_GUIDANCE.format(units="cells", kept="")
    ),
    "Survival Analysis": (
        "Method: time-to-event with censoring: a risk score from a scikit-learn classifier of the event within the window, or a Cox model if the user approves installing a pinned survival library; describe groups with Kaplan-Meier curves.\n"
        "Baseline: a balanced logistic regression on the strongest signals.\n"
        "Accept: the concordance index on the most recent closed window meets the threshold and beats the baseline by the minimum gain (time-aware split).\n"
        "If data is insufficient: too few events or fewer than two closed windows stop the run at the readiness gate."
    ),
    "Recommendation": (
        "Method: item-item cosine similarity on the interaction matrix, or truncated SVD, with scikit-learn and scipy sparse matrices; exclude items the entity already has, which in a holdout case are the items the case gives the model, never the held-out item it must find. Keep the work sparse: keep only the top neighbors of each item, score users in batches keeping each user's top items with numpy (np.argpartition on each batch's score rows, converted to a dense array with .toarray() first, because np.argpartition does not take a sparse matrix), never a Python loop that builds or sorts a pandas frame per user, measure precision@K on a seeded sample of at most 5,000 holdout users, and never build a dense items-by-items or users-by-items matrix. Never use Spark ML ALS: serverless compute rejects it (UC_COMMAND_NOT_SUPPORTED, because Unity Catalog does not support the higher-order functions it runs).\n"
        "Baseline: the most popular items.\n"
        "Accept: out-of-time precision@K is at least the minimum lift over popularity.\n"
        "If data is insufficient: too few interactions, users or items stop the run at the readiness gate."
    ),
    "Network Analysis": (
        "Method: build the graph from the relationship columns as one scipy.sparse adjacency matrix, then compute degree, connected components (scipy.sparse.csgraph) and a power-iteration PageRank on that matrix with a dense numpy rank vector, spreading the rank of nodes without out-links evenly; JAR-based graph libraries cannot load on serverless notebooks.\n"
        "Baseline: degree alone.\n"
        "Accept: every reported influence or community finding carries an effect size and a Benjamini-Hochberg corrected significance.\n"
        + FAMILY_F_SHORTFALL_GUIDANCE.format(units="groups", kept=", while every node is still scored and a node's own influence test needs only its per-unit minimum")
    ),
    "Optimization": (
        "Method: scipy.optimize.milp for discrete decisions or linprog for continuous ones, with the objective and constraints taken from cost and capacity columns; check feasibility first.\n"
        "Baseline: the current plan evaluated with the same objective.\n"
        "Accept: the problem is feasible and the plan is better than the current plan.\n"
        "If data is insufficient: costs or capacities that cannot be derived from the data stop the run at the readiness gate."
    ),
}

GENIE_READINESS_THRESHOLDS = {
    "A": {"min_rows": 1, "min_join_coverage": 0.80, "min_positives": 200, "min_class_rate": 0.005, "max_class_rate": 0.95,
          "min_outcome_windows": 2, "min_auc": 0.60, "min_pr_auc_lift": 1.5, "min_auc_gain": 0.02, "min_c_index": 0.60},
    "B": {"min_rows": 1, "min_join_coverage": 0.80, "min_reference_rows": 1000, "max_key_signal_null_rate": 0.20,
          "max_alerts_per_run": 200, "alert_quantile": 0.99, "min_topk_stability": 0.6},
    "C": {"min_rows": 1, "min_join_coverage": 0.80, "min_entities_per_k": 10, "min_numeric_features": 3, "min_silhouette": 0.15,
          "min_segment_share": 0.03, "min_bootstrap_ari": 0.6},
    "D": {"min_rows": 1, "min_join_coverage": 0.80, "min_interactions": 1000, "min_users": 100, "min_items": 100,
          "min_precision_lift": 1.1},
    "E": {"min_rows": 1, "min_join_coverage": 0.80, "min_seasonal_cycles": 2, "max_staleness_cadences": 2,
          "max_missing_period_rate": 0.20, "min_backtest_folds": 3, "max_mase": 1.0},
    "F": {"min_rows": 1, "min_join_coverage": 0.80, "min_group_rows": 30, "min_reportable_groups": 2, "min_periods": 2,
          "fdr_alpha": 0.05},
    "G": {"min_rows": 1, "min_join_coverage": 0.80, "min_text_non_null_rate": 0.50, "min_median_text_length": 20,
          "max_ai_error_rate": 0.02, "spot_check_rows": 50},
    "H": {"min_rows": 1, "min_join_coverage": 0.80, "min_simulation_draws": 1000},
}

GENIE_WIDGET_LABELS = {
    "result_table": "Business output table", "model_name": "Unity Catalog model name",
    "ai_endpoint": "AI endpoint for explanations", "as_of_date": "As-of date (latest = last period complete in every source table)",
    "horizon": "Forecast horizon (periods)",
    "season_length": "Season length (periods)", "act_now_share": "Share of rows to act on now",
    "review_share": "Share of rows to review next", "currency": "Currency of the money at stake (ISO code)",
    "act_now_amount": "Money at stake to act now", "review_amount": "Money at stake to review",
    "min_rows": "Minimum usable rows", "min_join_coverage": "Minimum join coverage",
    "min_positives": "Minimum positive outcomes", "min_class_rate": "Minimum outcome rate",
    "max_class_rate": "Maximum outcome rate", "min_outcome_windows": "Minimum closed outcome windows",
    "min_auc": "Minimum AUC to accept a model", "min_pr_auc_lift": "Minimum PR-AUC lift over the base rate",
    "min_auc_gain": "Minimum AUC gain over the baseline", "min_c_index": "Minimum concordance index",
    "min_reference_rows": "Minimum reference rows", "max_key_signal_null_rate": "Maximum null rate of key signals",
    "max_alerts_per_run": "Maximum alerts per run", "alert_quantile": "Alert score quantile",
    "min_topk_stability": "Minimum top-k stability", "min_entities_per_k": "Minimum entities per segment",
    "min_numeric_features": "Minimum numeric features", "min_silhouette": "Minimum silhouette score",
    "min_segment_share": "Minimum segment share", "min_bootstrap_ari": "Minimum bootstrap ARI",
    "min_interactions": "Minimum interactions", "min_users": "Minimum users", "min_items": "Minimum items",
    "min_precision_lift": "Minimum precision lift over popularity", "min_seasonal_cycles": "Minimum seasonal cycles",
    "max_staleness_cadences": "Maximum staleness in cadences", "max_missing_period_rate": "Maximum missing period rate",
    "min_backtest_folds": "Minimum backtest folds", "max_mase": "Maximum MASE to accept a forecast",
    "min_group_rows": "Minimum rows per reported group",
    "min_reportable_groups": "Minimum groups large enough to report", "min_periods": "Minimum periods", "fdr_alpha": "False discovery rate",
    "min_text_non_null_rate": "Minimum text non-null rate", "min_median_text_length": "Minimum median text length",
    "max_ai_error_rate": "Maximum AI error rate", "spot_check_rows": "Rows to spot-check",
    "min_simulation_draws": "Minimum simulation draws",
}
GENIE_MAX_USE_CASE_WIDGETS = 6
GENIE_WIDGET_CHOOSE = "you choose"
GENIE_WIDGET_FALLBACKS = {"horizon": "1", "season_length": "1"}
GENIE_WIDGET_ALLOWED = {
    "horizon": (1, None, True), "season_length": (1, None, True), "act_now_share": (0, 1, False), "review_share": (0, 1, False),
    "act_now_amount": (0, None, False), "review_amount": (0, None, False), "min_rows": (1, None, True), "min_join_coverage": (0, 1, False),
    "min_positives": (1, None, True), "min_class_rate": (0, 1, False), "max_class_rate": (0, 1, False),
    "min_outcome_windows": (1, None, True), "min_auc": (0.5, 1, False), "min_pr_auc_lift": (0, None, False),
    "min_auc_gain": (0, 1, False), "min_c_index": (0.5, 1, False), "min_reference_rows": (1, None, True),
    "max_key_signal_null_rate": (0, 1, False), "max_alerts_per_run": (1, None, True), "alert_quantile": (0, 1, False),
    "min_topk_stability": (0, 1, False), "min_entities_per_k": (1, None, True), "min_numeric_features": (1, None, True),
    "min_silhouette": (-1, 1, False), "min_segment_share": (0, 1, False), "min_bootstrap_ari": (-1, 1, False),
    "min_interactions": (1, None, True), "min_users": (1, None, True), "min_items": (1, None, True),
    "min_precision_lift": (0, None, False), "min_seasonal_cycles": (1, None, True), "max_staleness_cadences": (0, None, True),
    "max_missing_period_rate": (0, 1, False), "min_backtest_folds": (1, None, True), "max_mase": (0, None, False),
    "min_group_rows": (1, None, True), "min_reportable_groups": (1, None, True), "min_periods": (1, None, True),
    "fdr_alpha": (0, 1, False), "min_text_non_null_rate": (0, 1, False), "min_median_text_length": (0, None, True),
    "max_ai_error_rate": (0, 1, False), "spot_check_rows": (0, None, True), "min_simulation_draws": (1, None, True),
}


def _genie_number_text(value) -> str:
    return inspire_number_text(value)


def genie_allowed_text(rule) -> str:
    low, high, whole = rule or (None, None, False)
    if low is not None and high is not None:
        bounds = f"{_genie_number_text(low)} to {_genie_number_text(high)}"
    elif low is not None:
        bounds = f"at least {_genie_number_text(low)}"
    elif high is not None:
        bounds = f"at most {_genie_number_text(high)}"
    else:
        bounds = ""
    return ("whole number" + (f", {bounds}" if bounds else "")) if whole else bounds


_GENIE_ALLOWED_RE = re.compile(r"(?:(whole number)(?:, |$))?(?:(-?[0-9]+(?:\.[0-9]+)?) to (-?[0-9]+(?:\.[0-9]+)?)"
                               r"|at least (-?[0-9]+(?:\.[0-9]+)?)|at most (-?[0-9]+(?:\.[0-9]+)?))?")


def genie_allowed_rule(text):
    text = str(text if text is not None else "").strip()
    match = _GENIE_ALLOWED_RE.fullmatch("" if text.lower() == "any" else text)
    if not match:
        return None
    whole, low, high, at_least, at_most = match.groups()
    low = low if low is not None else at_least
    high = high if high is not None else at_most
    return (float(low) if low is not None else None, float(high) if high is not None else None, bool(whole))


_GENIE_WIDGET_NUMBER_RE = re.compile(r"-?[0-9]+(?:\.[0-9]+)?")


def genie_widget_value_problem(name: str, value, allowed) -> str:
    rule = genie_allowed_rule(allowed)
    if not rule or rule == (None, None, False):
        return ""
    low, high, whole = rule
    text = str(value if value is not None else "").strip()
    number = float(text) if _GENIE_WIDGET_NUMBER_RE.fullmatch(text) and not (whole and "." in text) else None
    if number is None or (low is not None and number < low) or (high is not None and number > high):
        return f"{name} is '{text}'; allowed: {genie_allowed_text(rule)}"
    return ""


def genie_widget_set_problems(values: dict) -> list:
    def number(name):
        text = str(values.get(name) if values.get(name) is not None else "").strip()
        return float(text) if _GENIE_WIDGET_NUMBER_RE.fullmatch(text) else None

    problems = []
    act, review = number("act_now_share"), number("review_share")
    if act is not None and review is not None and act + review > 1 + 1e-9:
        problems.append(f"act_now_share + review_share is {_genie_number_text(act + review)}; allowed: at most 1")
    order = inspire_amount_order_problem(number("act_now_amount"), number("review_amount"))
    if order:
        problems.append(order)
    for low_name in sorted(values):
        high_name = "max_" + low_name[4:]
        if low_name.startswith("min_") and high_name in values:
            low, high = number(low_name), number(high_name)
            if low is not None and high is not None and low > high:
                problems.append(f"{low_name} is {_genie_number_text(low)}, above {high_name} {_genie_number_text(high)}")
    return problems

GENIE_NOTEBOOK_ENV_VERSION = "4"
GENIE_SERVERLESS_RULE = ("Databricks serverless only: no `.cache()`, `.persist()`, `.unpersist()`, `.checkpoint()`, `sparkContext`, "
                         "RDDs or `spark.databricks.*` configs; always use fully qualified table names.")
GENIE_DECIMAL_RULE = ("Spark Connect returns DECIMAL columns and aggregates as Python `Decimal`, which formatting and "
                      "`spark.createDataFrame(...)` reject: cast them to DOUBLE in Spark before `toPandas()`, `first()` or `collect()`, and "
                      "cast pandas columns that hold `Decimal` values to float before `spark.createDataFrame(...)`. Keep keys, ids, ranks "
                      "and counts as integers, never as floats, so an id reads `10`, not `10.0`, and hold one that can be missing, "
                      "such as a count from a left join or from a NULL median, as pandas' nullable integer, "
                      "`pd.to_numeric(x, errors='coerce').astype('Int64')`, never with `astype(int)`, which raises IntCastingNaNError "
                      "on a missing value; a comparison of such a column is <NA> where a value is missing, which np.where, `if` and "
                      "a `.loc` mask reject ('boolean value of NA is ambiguous'), so fill it with the answer a missing value gets, "
                      "`(n < m).fillna(False)`, before it picks rows; call int() or round() only in a function that returns first on a "
                      "missing value, as "
                      "fmt_count does, never behind np.where, which computes both of its branches on every row, so "
                      "`np.where(s.notna(), s.map(lambda x: round(x)), '')` still raises 'cannot convert float NaN to integer'. "
                      "Convert numpy values to Python types with bool(), int() or float() before json.dumps or dbutils.notebook.exit, "
                      "which reject numpy booleans and numbers, and pass every json.dumps "
                      "`default=lambda v: v.tolist() if hasattr(v, 'tolist') else str(v)`, since a comparison of numpy numbers, such "
                      "as an AUC against its threshold, and an `and` of such comparisons are numpy booleans too. DATE values arrive as Python `datetime.date` objects, never Timestamps, "
                      "both in pandas columns and from `first()` or `collect()`, and a date parameter such as as_of_date is one too: "
                      "convert each side with pd.to_datetime, or pd.Timestamp for a single date, before comparing it with a "
                      "Timestamp, a datetime64 column or another date, adding a Timedelta or calling a Timestamp method such as "
                      "`.date()`. Spark never "
                      "converts between BOOLEAN and numbers: cast a 0/1 flag to BOOLEAN before OR, AND, `|` or `&`, and a BOOLEAN to INT "
                      "before SUM or arithmetic. Serverless runs Spark SQL in ANSI mode, where dividing by zero, an invalid cast or an "
                      "overflow raises an error instead of returning NULL: write every ratio of columns as `F.try_divide(a, b)` "
                      "(`try_divide(a, b)` in SQL) or `F.when(b != 0, a / b)`, and cast text with `try_cast`; in pandas, dividing by zero "
                      "or the log of 0 gives inf, so replace inf with NaN after the last step that derives a feature and before a fit, "
                      "a score or the write, in the numeric columns only "
                      "(`num = x_pdf.select_dtypes('number').columns`, then `x_pdf[num] = x_pdf[num].replace([np.inf, -np.inf], np.nan)`): "
                      "a replace over the whole frame also compares each array value of an ARRAY column and fails. A ratio of two "
                      "Python numbers, such as a share computed for one row from the values `itertuples()` or `to_dict('records')` "
                      "yields, raises ZeroDivisionError on a zero denominator instead: compute it only when the "
                      "denominator is above 0, `a / b if b > 0 else None`, and leave a reason that states it out of a row whose "
                      "denominator is 0.")
GENIE_EMPTY_INPUT_RULE = ("Before every fit, score, forecast, simulation or ai_query step, count the rows it receives and never call it on an "
                          "empty input: when it has none, print how many rows each filter and each feature removed and stop by the STOP RULE. "
                          "When a step or a model wrapper splits its input by a key or a model type, skip each empty slice instead of "
                          "calling the model on it. Build each per-group or per-unit result frame from an explicit column list, "
                          "`pd.DataFrame(rows, columns=[...])`, so a step left with no group or unit to report still returns its columns "
                          "and the join back gives NULL instead of a KeyError.")
GENIE_TRAINING_CLASSES_RULE = ("A classifier's training rows must hold at least `min_positives` examples of each class: when they do not, "
                               "print the class counts and stop by the STOP RULE, because a model fit on one class writes the same value on "
                               "every row.")
GENIE_MISSING_VALUE_TEXTS = ("the whole word 'None', 'nan' or 'NaT' or the text '<NA>', which Python and pandas write for a "
                             "missing value, each word of the check matched case-sensitively, never with `re.IGNORECASE`, `(?i)`, "
                             "`case=False` or a lowered text, so neither a name such as 'Nan' or 'Nat' nor a category such as "
                             "'Unknown' matches")
GENIE_NUMBER_FORMAT_RULE = ("Every number written into top_reasons, recommended_action, ai_facts, the summary facts or an ai_query "
                            "prompt is formatted first by one "
                            "Python function, `fmt_num(x)`: 'unknown' for a missing value, which `x is None or pd.isna(x)` finds for "
                            "None, NaN and pandas' NA and NaT, and for inf, which `isinstance(x, numbers.Real) and math.isinf(x)` finds, "
                            "so a text, such as a carrier's name, is never missing: the test never calls float() on its input, which "
                            "raises on every text, so a test that returns missing on that error marks every label and name missing; "
                            "and '0' for 0; otherwise round to 3 significant "
                            "figures first, `x = round(x, 2 - math.floor(math.log10(abs(x))))`, so 109.96 becomes 110 and 999.6 becomes "
                            "1000, then divide by 1e9, 1e6 or 1e3 for a B, M or K suffix when abs(x) is at least that amount, then write "
                            "a whole result as `str(int(x))` and any other with `f'{x:.{d}f}'`, "
                            "d = max(0, 2 - math.floor(math.log10(abs(x)))), stripping trailing zeros and the point only from text that "
                            "has a point, because `'120'.rstrip('0')` is '12'. P- and q-values below 0.001 are written '< 0.001'. "
                            "`run_use_case()` asserts, before any step builds text, that fmt_num gives '110' for 109.96, '120' for 120.4, "
                            "'1K' for 999.6, '1.06K' for 1060, '105K' for 104999, '250M' for 250000000, '0.5' for 0.5, '-120' for "
                            "-120.4 and 'unknown' for None, float('nan') and pd.NA, which a nullable column holds and on which "
                            "`float(x)` raises a TypeError. A ratio carries its unit in the code and in every text template: a multiple ends in 'x' (7.48x the "
                            "lane median), a share of at most 1 is multiplied by 100 and ends in '%' (23.5% of sales), and a "
                            "part-to-whole ratio above 1 is written as a multiple (a ship cost 23.5x the sales price), never as a "
                            "percent such as '2.35K%'. A count's noun agrees with it: '1 prior order', '2 prior orders', '1 day since "
                            "the last order', so a count and its noun, a number of days, weeks or months among them, are written by "
                            "`fmt_count(n, noun, plural=None)`: fmt_num(n), a space and the noun when n is "
                            "exactly 1, otherwise its plural (the noun plus 's' unless `plural` is given), never by a template that "
                            "puts a count before a fixed noun, such as f'{n} tickets', which writes '1 tickets', and a reason the "
                            "instruction words with a count before a plural noun, such as 'N days since the last order', is built as "
                            "fmt_count(n, 'day') + ' since the last order'; a missing count gives "
                            "'unknown' and the plural, like fmt_num, never an error. A count is never negative, so fmt_count raises a "
                            "ValueError on one: a difference that can fall on either side of zero, such as the days left before a "
                            "window opens, takes its words from its sign and passes fmt_count its absolute value ('the window opens "
                            "in 13 days', 'the window opened 13 days ago'). An amount that can fall on either side of its "
                            "reference, such as a cost's excess over its peer median, takes its words from its sign the same way and "
                            "writes fmt_num of its absolute value ('excess freight of 9.6K over the lane peer median', 'freight 216 "
                            "below the lane peer median'), never a negative number after words of one side, such as 'excess freight "
                            "of -216 over the lane peer median'. A rank is written "
                            "by `fmt_rank(r)`: the whole number with thousands separators and its ordinal suffix ('st', 'nd' or 'rd' "
                            "after a last digit of 1, 2 or 3, except in 11, 12 and 13, and 'th' otherwise), never through fmt_num, "
                            "which writes '1.01Kth', with the total it ranks among written as `f'{total:,}'`, and a missing rank gives "
                            "'unknown', like fmt_num, never an error. `run_use_case()` also "
                            "asserts that fmt_count gives '1 prior order', '2 prior orders', '0 days' and '1.06K tickets' for (1, "
                            "'prior order'), (2, 'prior order'), (0, 'day') and (1060, 'ticket'), 'unknown days' for (None, 'day') and "
                            "for (pd.NA, 'day'), that it raises a ValueError for (-1, 'day'), and that fmt_rank gives '1st', "
                            "'2nd', '3rd', '11th', '12th', '13th', '21st', '112th' and '21,302nd' for 1, 2, 3, 11, 12, 13, 21, 112 "
                            "and 21302 and 'unknown' for None, float('nan') and pd.NA. Never write a raw double or "
                            "scientific notation, so never Python's 'g' format, which turns 1060 into 1.06e+03, and never turn a number "
                            "into text inside SQL, where CAST(... AS STRING), concat and format_string write the raw double, such as "
                            "0.6793932502246776. A fact whose number is missing, which fmt_num writes as 'unknown', is left out of "
                            "top_reasons, recommended_action, ai_facts and the summary facts, so no text says 'unknown' in place of a "
                            "number: "
                            "every row's top_reasons is built by `keep_reasons(facts)`, which takes (text, numbers) pairs in order "
                            "and returns the texts whose numbers are all present (none of them missing by fmt_num's test, or inf), "
                            "without a repeat and at most 3, and `run_use_case()` asserts that it returns ['a', 'd', 'e'] for "
                            "[('a', (1,)), ('b', (None,)), ('c', (float('nan'),)), ('a', (1,)), ('g', (pd.NA,)), ('d', (2, 3)), "
                            "('e', (0,)), ('f', (4,))]. Each "
                            "text is built before keep_reasons reads its numbers, so every function a fact's text calls, such as "
                            "fmt_num, fmt_count and fmt_rank, returns a text for a missing number and never raises on one. A "
                            "reason with the same text on every row of a run that writes more than one row, such as a rate that is "
                            "0% for every item, says nothing about the row: drop it from every row before the write and print it "
                            "once in the log. "
                            "Before the write, check that no top_reasons, recommended_action or ai_facts text holds the whole word "
                            "'unknown' or 'inf', or " + GENIE_MISSING_VALUE_TEXTS + ", and stop the run on the first row that does. Check "
                            "entity_id, entity_label and segment_or_item for " + GENIE_MISSING_VALUE_TEXTS + ", never for a value "
                            "the data itself names, such as an 'Unknown' category, and for an empty part, such as two separators "
                            "with nothing between them (' |  | ') or a separator at either end, and check that no top_reasons or "
                            "recommended_action text holds a minus "
                            "sign before a digit that follows no letter, digit or point (the '-216' of 'excess freight of -216', never "
                            "the '-' of a lane such as 'TN-KY' or of a date), stopping the run on the first row that does. Before the "
                            "write, also check that no "
                            "top_reasons entry holds a second number after ', ' or '; ', apart from the reference after ', against' and "
                            "a count with its minimum ('16 tickets, the test needs 17'), and stop the run on the first row that does; "
                            "this check never reads recommended_action, whose next step names the record and its amount together.")
GENIE_NUMBER_TEXT_SOURCE = '''from math import floor, isfinite, log10


def fmt_num(x, p_value=False):
    try:
        x = float(x)
    except (TypeError, ValueError):
        return "unknown"
    if not isfinite(x):
        return "unknown"
    if p_value and x < 0.001:
        return "< 0.001"
    if x == 0:
        return "0"
    x = round(x, 2 - floor(log10(abs(x))))
    suffix = ""
    for scale, unit in ((1e9, "B"), (1e6, "M"), (1e3, "K")):
        if abs(x) >= scale:
            x, suffix = x / scale, unit
            break
    if float(x).is_integer():
        return str(int(x)) + suffix
    text = f"{x:.{max(0, 2 - floor(log10(abs(x))))}f}"
    if "." in text:
        text = text.rstrip("0").rstrip(".")
    return text + suffix
'''
GENIE_NUMBER_TEXT_NAME = "fmt_num"
GENIE_NUMBER_FUNCTIONS = ("fmt_num", "fmt_count", "fmt_rank")
GENIE_PERIOD_COMPLETE_RULE = ("Decide which periods are complete from each source table's own dates before any filter, so a filter never "
                              "makes a partial period look complete. Drop the rows missing the time value or any column of the row key, the "
                              "entity key and each column a row is scored per, such as the store of a store-item row, from the frame "
                              "every later step uses, not only from the readiness count. Compute period starts and ends with pandas periods "
                              "(`pd.Period(day, 'Q').start_time`, `.end_time`, `pd.offsets`) or the date table's own period columns, never "
                              "by adding months to a date by hand, which fails at the year end. A time column that holds something other "
                              "than a date, such as a key into a date table, a yyyymmdd number or epoch seconds, becomes a calendar date "
                              "before any period, window or as-of date is computed: join the date table on the key and read its date "
                              "column, parse the yyyymmdd number, convert the epoch seconds. `as_of_date` is always a calendar date, never "
                              "a key or a number, which a cast to a date turns into NULL or a wrong date.")
GENIE_OUTCOME_WINDOW_FAMILIES = ("A", "F")
GENIE_HOLDOUT_FAMILIES = ("A", "D", "E")
GENIE_UNREACHED_QUANTILE_RULE = ("A survival curve fit on censored times, such as a Kaplan-Meier curve, that stays above a quantile's "
                                 "level to the end of its follow-up has not reached that quantile, which is then longer than the "
                                 "follow-up, never missing or NULL, whatever a build step says: a reason states it as more than the "
                                 "follow-up ('more than 365 days'), and a date built from it, such as the day a lapse window opens, "
                                 "uses the follow-up's end, the earliest the quantile can be, so a group whose event is rare still "
                                 "gets its date, and a reason that states that date, or the days to or since it, says it is the earliest "
                                 "('the lapse window opens in 192 days at the earliest', 'the earliest the lapse window opened is 36 days ago').")
GENIE_CLOSED_WINDOW_RULE = ("A window whose outcome is observed over a horizon (an outcome, churn or lapse window, a treatment's response "
                            "window, a cohort's period offset, a funnel's conversion window) is closed only when its last day plus that "
                            "horizon is before as_of_date. A window labeled by its first day, such as a month from "
                            "`F.trunc(date, 'month')`, ends the day before the next window starts: test that last day, "
                            "`F.last_day(date)` for a month or `pd.Timestamp(m) + pd.offsets.MonthEnd(0)` in pandas, never the label, "
                            "which counts the window closed up to a month too early, so its outcomes reach into the holdout's days. "
                            "Choose the training, holdout and baseline windows by that test on the window, "
                            "never by which units already have an outcome: in a window that is not closed, the units with a known outcome "
                            "are the ones whose event came early, so a holdout or a rate built from them holds one class or leans to it. "
                            "A unit whose horizon is still open is censored: score it or leave it out, never label it as a negative, and "
                            "leave a cohort offset or a funnel stage the data has not reached out of every comparison instead of counting "
                            "it as zero. A closed window with no event is a negative, 0: over a frame that holds no rows, `F.sum`, "
                            "`F.max`, `F.min` and `F.avg` return NULL while `F.count` returns 0, so an outcome is "
                            "`F.coalesce(F.sum(x).over(w), F.lit(0)) > 0` or `F.count(x).over(w) > 0`, and a NULL outcome marks only "
                            "a censored unit. The holdout is the most recent closed window, and training takes the anchors of every "
                            "earlier window whose last day plus the horizon is before the holdout's first day, so no training label "
                            "reads an outcome from the holdout's days: at most one anchor per entity and date, never only each "
                            "entity's latest anchor, which puts every example in one window and leaves the others empty. Readiness "
                            "counts the positives of the fit and the holdout on those same anchors. " + GENIE_UNREACHED_QUANTILE_RULE)
GENIE_FRAME_NAMES_RULE = ("Name pandas DataFrames `*_pdf` and Spark DataFrames `*_sdf`, and move between them only with `toPandas()` and "
                          "`spark.createDataFrame(...)`, so no step mixes the two. Clear `attrs` on every frame `toPandas()` returns "
                          "(`x_pdf.attrs = {}`): Spark Connect stores the query's metrics there, and pandas copies them into every row "
                          "`iterrows()` yields, which made one 28,000-row loop run 39 minutes. Build row text with vectorized column "
                          "operations, or loop with `itertuples()`, never `iterrows()`. `itertuples()` renames a column whose name starts "
                          "with an underscore or is not a valid identifier to its position (`_1`, `_2`), so give such columns plain names "
                          "first, and its rows are named tuples: read a field as `row.name`, never `row['name']`, which raises TypeError. "
                          "Read and assign pandas columns as `x_pdf['name']`, never `x_pdf.name`: a column named like a DataFrame "
                          "method, such as rank, count, tail or size, returns the method instead. `toPandas()` turns each value of an "
                          "ARRAY or MAP column into a numpy array or a dict, and NULL into None: read one as "
                          "`np.asarray(x if x is not None else [], dtype=float)` and count it with `len()`, never test it as a condition "
                          "such as `if x` or `x or []`, which fails on an array of two or more elements. Serverless environment "
                          + GENIE_NOTEBOOK_ENV_VERSION + " runs pandas 2.2.3, where `len()`, `list()`, `sorted()` or `.groups` of a "
                          "groupby on one key with `dropna=False` raises 'Categorical categories cannot be null' when a key is missing, "
                          "and `get_group` cannot select the missing group: to keep the rows whose key is missing as a group, label them "
                          "in a new column, `x_pdf['segment_group'] = x_pdf['segment'].astype(object).where(x_pdf['segment'].notna(), "
                          "'missing')`, group on that column and build labels from the original one. A fact about the whole run, such as "
                          "a test flagging every series, is one bool: test it with `if` before assigning rows, never use it as the row "
                          "key of `.loc` or join it to a row mask with `and` or `or`, which return the bool itself: pandas raises "
                          "KeyError 'cannot use a single bool to index into setitem' on a Python bool, and on a numpy bool writes row 0 "
                          "or 1 and appends a row labeled False or True. A step that adds rows, such as "
                          "`pd.concat`, `explode` or a landmark or window expansion, repeats index labels: call `reset_index(drop=True)` "
                          "right after it, and line up labels, predictions and features by key columns or position, never by index labels. "
                          "Each step reads only the columns the step before it returns: a CTE, a select or a feature step keeps every "
                          "column a later step reads, the keys and measures of a filter, a join, a grouping, a flag, a reason or the "
                          "output, and a filter on a derived column runs after the step that adds it.")
GENIE_COLUMN_NAMING_RULE = ("Name every column a step adds after what it holds, never after a pandas or Spark DataFrame method or "
                            "property such as `tail`, `head`, `count`, `size`, `mean`, `sum`, `values` or `index`: read as an "
                            "attribute (`x_pdf.tail`), such a name returns the method instead of the column, so write `is_tail`, "
                            "not `tail`. The business output columns keep their names.")
GENIE_SQL_NESTING_RULE = ("Spark rejects a window function inside an aggregate, an aggregate inside another aggregate, "
                          "COUNT(DISTINCT ...) over a window and a window function in a filter, a join condition or a HAVING clause "
                          "(WINDOW_FUNCTION_NOT_ALLOWED_IN_CLAUSE): compute the window or the inner aggregate in its own step, as a "
                          "column added with `withColumn`, then aggregate or filter on that column. On serverless, Spark Connect "
                          "analyzes a DataFrame only at its first action, so a column that does not resolve, such as a column of "
                          "another frame, fails at a later count, collect or write, whose error names the line that wrote it after "
                          "'was called from', and a later `drop()` or `withColumn` of that column never removes the expression from "
                          "the plan: delete the code a fix supersedes. In PySpark, wrap every comparison in parentheses before "
                          "combining it with `&` or `|`, which bind "
                          "tighter than `==`, `>` and `<`: `(F.col('a') == 1) & (F.col('b') > 0)`. Combine a list of conditions with "
                          "`functools.reduce(operator.and_, conditions)` or `operator.or_`, never with Python's `and`, `or`, `all()` "
                          "or `any()` or NumPy's `logical_and`, which call `bool()` on a Column and raise "
                          "CANNOT_CONVERT_COLUMN_INTO_BOOL. The condition of `F.when` is a Column, `F.when(F.col('closed'), value)`, "
                          "never a column name such as `F.when('closed', value)`, which raises NOT_COLUMN; a condition written as SQL "
                          "text goes through `F.expr('days > 0')`. Count the rows that meet a condition with "
                          "`F.count_if(condition)`, which gives 0 when none do, never `F.sum(condition.cast('int'))` or "
                          "`F.sum(F.when(condition, 1))`, which give NULL when the condition is NULL on every row of a group, and wrap "
                          "any other sum whose answer over no value is 0 in `F.coalesce(..., F.lit(0))`. Name every field of a struct "
                          "that goes into an array, a `when` or a union with another struct, `F.struct(F.lit('after as_of_date')"
                          ".alias('check'), condition.alias('failed'))`: Spark names an unnamed field after its expression, so structs "
                          "built from different expressions are different types, and `F.array` of them stops with "
                          "DATATYPE_MISMATCH.DATA_DIFF_TYPES.")
GENIE_SPARSE_MATRIX_TECHNIQUES = ("Recommendation", "Network Analysis", "Optimization")
GENIE_SPARSE_MATRIX_RULE = ("A matrix with a row or column for each entity, item or decision variable, such as an interaction, adjacency "
                            "or constraint matrix, is one scipy.sparse matrix built from coordinate arrays, "
                            "`scipy.sparse.csr_matrix((data, (rows, cols)), shape=(n_rows, n_cols))`, and stays sparse: scale its rows "
                            "or columns by multiplying it with `scipy.sparse.diags(...)`, such as a PageRank transition matrix "
                            "`scipy.sparse.diags(inverse_out_degree) @ adjacency` with 0 for a node without out-links, never by "
                            "assigning to its rows, columns or elements, where a row assigned from a dense array stores each of its "
                            "elements, zeros included, and call `.toarray()` only on a batch of rows, never on the whole matrix or "
                            "on a product of two such matrices; "
                            "scipy.optimize.milp and linprog take a sparse constraint matrix as it is.")
GENIE_JOIN_RULE = ("Join on a list of key names, such as `a_sdf.join(b_sdf, ['key'], 'left')`, only when every key has that name on both "
                   "sides; when a key has another name on one side, or a step renamed it, join on an explicit condition, "
                   "`a_sdf.join(b_sdf, a_sdf['key'] == b_sdf['other_key'])`, and drop the other side's key. Before a join, rename or "
                   "drop every column other than the keys that both sides hold, so each name exists once after it, and rename the "
                   "columns of a frame joined to itself or to a frame built from it, such as its aggregate or a cross join of its "
                   "distinct keys, first: `anchors_sdf['customer_sk'] == orders_sdf['customer_sk']`, where `anchors_sdf` was built "
                   "from `orders_sdf`, stops with AMBIGUOUS_COLUMN_REFERENCE. Stack frames with `unionByName` over one "
                   "column list selected from each, `a_sdf.select(cols).unionByName(b_sdf.select(cols))`: `unionByName` stops "
                   "on frames whose columns differ, such as fit rows and scoring rows, which have no outcome, and `union` "
                   "pairs columns by position. In SQL, qualify every column read after a "
                   "join with its table alias. A join or `map` that adds columns to a frame reads them from a side with one row "
                   "per key (aggregate that side to its key first, or key both sides by a row id made before they split), and "
                   "the step asserts that the frame kept its row count. A key that can be NULL, such as a class or a segment "
                   "whose rows stay flagged as missing, joins on a null-safe condition, `a_sdf['key'].eqNullSafe(b_sdf['key'])` "
                   "(<=> in SQL), and drops the other side's key, because a grouping keeps NULL as one group while an equality "
                   "condition and a join on key names never match it, so the rows with a NULL key would lose every column the "
                   "join adds. A step that needs to know which rows found a match, such as a join's coverage or a missing-input "
                   "flag, reads a flag the other side adds before the join, `b_sdf.withColumn('b_matched', F.lit(True))`, which "
                   "is NULL on the rows without a match, never the other side's key, which is gone once it is dropped and is "
                   "NULL on a row that a null-safe join matched on a NULL key.")
GENIE_DATE_FRAME_RULE = ("A window over earlier or later dates, such as features from the orders strictly before an anchor's date or "
                         "an outcome in the days after it, is a range frame on the date as a day number: "
                         "`Window.partitionBy(key).orderBy(F.datediff(F.col(date), F.lit('1970-01-01').cast('date')))` with "
                         "`.rangeBetween(Window.unboundedPreceding, -1)` for the earlier dates and `.rangeBetween(1, horizon_days)` "
                         "for the days after, or `ORDER BY datediff(date, DATE'1970-01-01') RANGE BETWEEN UNBOUNDED PRECEDING AND 1 "
                         "PRECEDING` in SQL. A rows frame also counts the other rows of the same date, so a second order on a day "
                         "reads the first one as its history and the leakage check fails, unless the frame runs over one row per key "
                         "and date. The leakage check tests the dates each step read, never a second computation of its values, "
                         "which differs on a missing history, a repeated date or a sample and stops a run that has no leakage: the "
                         "feature step keeps on each row the latest date its window or rows read (`F.max(date)` over the same window, "
                         "NULL when it read none) and an outcome step the earliest, and the check stops on any row whose latest "
                         "feature date is on or after its anchor date or whose earliest outcome date is on or before it, and on any "
                         "training or reference window whose last day, plus the outcome horizon when its rows carry an outcome, is "
                         "not before the first day of the holdout or scored window. The holdout and the scored rows are never "
                         "checked against each other: a scored row has no outcome, so its features may read the days of the "
                         "holdout's outcome horizon, which are the present it is scored on, and the scoring population is never cut "
                         "to pass a leakage check. A window the closed-window choice left out of "
                         "training is not a training window, so the check never tests it. A statistic the fit computes over its own "
                         "window, such as a peer median, a MAD or an alert threshold, is tested on the holdout and scored rows that "
                         "read it, never on the fit rows it was computed from, and a window's first and last day are the first and "
                         "last dates its rows hold, never a bound the window excludes. Every date, "
                         "as_of_date and window bound is a DATE or "
                         "TIMESTAMP value: when a table holds its time as a key to a date table, such as an integer date key, read the "
                         "date from that table's date column through its foreign key (the date table is listed under TABLES), never "
                         "by parsing or converting the key.")
GENIE_WEIGHT_CHECK_RULE = ("Print the distribution of every weight computed from the data that shapes `value` or eligibility, such as a "
                           "margin, a price or a capacity, before it is used. A weight that is zero, negative or the same for most rows "
                           "never multiplies, clips or filters `value` or the candidates: rank and keep the rows by the method's own score "
                           "instead, say so in the summary, and the decision test, not an eligibility filter, fails the rows whose "
                           "expected impact is a loss, so the notebook still writes its ranked rows, every row Monitor when every row is a "
                           "loss. A share, a cumulative share or a concentration (a Pareto curve, a Gini, a top-k share) needs values of "
                           "one sign. When its measure can be negative, such as a profit, print how many groups have a positive total and "
                           "how many a negative one, then analyze a group with a negative total on its losses (its negative values as "
                           "positive amounts, so the rows that lose the most lead) and a group with a positive total on its positive "
                           "values, count the rows of the other sign as zero in that share, and say which in that share's own reason "
                           "('56.9% of its class's losses', '12.3% of its class's profit'), never as a fact joined to it, and a row of "
                           "the other sign, which counts as zero, states its own amount against its group's total ('a profit of "
                           "1.2K, against its class's loss of 45.6K'), never a 0% share: no group "
                           "is left out, and no test is left with no group, because of the sign of its measure, and the step that "
                           "writes these share reasons asserts that none of them is a 0% share on each share's number before fmt_num "
                           "formats it, which is 0 only for a share of 0, never on the reason's text, where a test for '0%' also "
                           "matches '10%' and '100%'. Before the write, check that `value` "
                           "varies across the rows: a value that is the same for every row means a step collapsed it, so print which "
                           "step.")
GENIE_COMPUTE_ONCE_RULE = ("Serverless has no cache, so Spark recomputes a DataFrame's whole lineage on every action: run each expensive "
                           "step (a model fit per group, an ai_query call, a large join) in one action, collect its small result to pandas, "
                           "and take later counts and checks from that pandas frame. Compute a value several steps use, such as the escaped "
                           "AI endpoint, once in the Parameters cell, never inside a branch. Work on thousands of entities with vectorized "
                           "numpy, pandas or Spark operations, never a Python loop that filters, builds or sorts a frame per entity: group "
                           "the rows once (groupby, or a dict of arrays keyed by entity) and look each entity up; evaluate "
                           "a ranking on a seeded sample of at most 5,000 entities when there are more, and say so in the summary; the "
                           "sample is for the evaluation only, and the scoring and the business output keep every entity. Sample per "
                           "group with `x_pdf.sample(frac=1, random_state=seed).groupby(key).head(n)`, which keeps the group key "
                           "column and every row of a group smaller than n, never with `groupby(key).sample(n=...)`, which raises "
                           "when a group holds fewer than n rows, and never inside `groupby(...).apply(...)`. Aggregate "
                           "each fact table to the entity and period before joining it, never joining raw fact rows to a date window per "
                           "entity, and fit, score or test more than 50 independent series or units with groupBy(...).applyInPandas(...), "
                           "which runs them in parallel where Spark holds the rows, never in a loop on the driver over event rows moved "
                           "there with `toPandas()`, which made one test of 107,132 units run 16 minutes.")
GENIE_SIGNIFICANCE_RULE = ("Run every significance test on observed values, never on a smoothed or fitted series such as an STL trend, a "
                           "moving average or a model's fitted values, and never on observed values minus an STL seasonal component, whose "
                           "trend absorbs a short series' noise: each makes almost every series look significant. Handle seasonality inside "
                           "the test, with season indicators in an OLS over the full history or by comparing each period with the same period "
                           "a cycle earlier, and use HAC standard errors only with at least 30 points. Use a library test from statsmodels or "
                           "scipy, never a hand-written test statistic. In seasonal data, compare a period or cohort with the same season of earlier cycles, never "
                           "with an average over all seasons. Leave out the latest periods of an event that follows another by a lag (a "
                           "return after its sale, a claim after its policy), which are still incomplete. Correct for multiple tests, print "
                           "how many series the test flags in each "
                           "direction, and when it flags every tested series, one fact about the whole run that an `if` tests, set each "
                           "row's data_quality_flag to 'test flags every series'.")
GENIE_PER_UNIT_SAMPLE_RULE = ("When a test, rate or mean is computed once per unit (an item, a sender, a store) and no readiness "
                              "threshold, such as `min_group_rows` for the groups a comparison reports, sets that unit's minimum, its "
                              "per-unit minimum sample comes from the test, never a round number: the smallest sample at which either of "
                              "its two extreme outcomes (such as every trial a success, or none) reaches, in the same library test, the "
                              "significance level (`fdr_alpha`, else 0.05) divided by the number of units with at least that "
                              "sample, which is the level at which one unit passes the multiple-test correction on its own, and "
                              "at least 5: at the uncorrected level most units tested are ones no outcome could make significant, "
                              "as when 49,803 items with a median of 10 tickets were tested from 6 tickets and every q-value "
                              "was 1, since one item needed 21 tickets to pass a correction over 49,803 tests on its own. "
                              "Readiness prints "
                              "how many units reach it and their median sample size. Compute it once, with one function, from the "
                              "sample the test computes on, before readiness checks it, and pass that value to readiness, the test, "
                              "the shrinkage and each reason, so readiness never prints another minimum than the one the test uses. "
                              "That minimum is a unit's only gate: a unit is never "
                              "also left out for the size of a group it belongs to, such as a cohort or a segment, and readiness counts "
                              "each group and unit with the same rows, filters and minimum as the step that tests it, so a readiness pass "
                              "always leaves that step something to test. Whether the minimum comes from the test or from a "
                              "readiness threshold such as `min_group_rows`, the count compared with it, in readiness, in the test "
                              "and in a reason, is the sample the test computes on, after the test drops what it drops (such as "
                              "the ties of a sign test, or the recipients whose outcome window has not closed), never another count "
                              "of the unit or group (such as its tickets or its senders), and a test that compares two samples, "
                              "such as a group against its prior period, checks each of them, so no unit or group that passes "
                              "meets its test with fewer observations, or none. Units below it are not tested and stay out of the "
                              "multiple-test correction, and their rows stay, flagged 'too few <observations>' and Monitor, with top "
                              "reasons that give the unit's own count and the minimum ('12 tickets, the test needs 21'), never a test "
                              "result such as 'not significant', which only a tested unit has, and only a unit below the minimum gives "
                              "that reason, never a row the test skipped for a missing input, whose data_quality_flag names "
                              "its reason instead; a test left "
                              "with no unit to test is not an empty input and never stops the run, so the rows are still written with its "
                              "columns NULL and the summary says so. Before a unit's rate or mean shapes `value` or the rank, shrink it "
                              "toward its baseline by its sample size, (sum + m * baseline) / (n + m) with m the per-unit minimum, so a "
                              "single observation barely moves a unit from the baseline.")
GENIE_RANK_ORDER_RULE = ("Build `rank` with one function, `rank_rows(passes, data_quality_flag, impact, key)`, which takes four "
                         "pandas Series on one index and returns the ranks 1 to n as a Series on that index, so `frame['rank'] = "
                         "rank_rows(...)` lines up on any index, such as one a filter or a concat left: the rows whose `passes` "
                         "(the decision test) is True first, then the rows whose `data_quality_flag` is 'ok', then the flagged rows, "
                         "each group by `impact` (the expected impact or value) descending, compared at 6 significant figures, "
                         "`impact.map(lambda x: float(f'{x:.6g}'), na_action='ignore')`, so a difference of float rounding never "
                         "orders rows, with a missing impact last, then by `key` "
                         "(the row key: entity_id, followed by segment_or_item when the method writes several rows for an entity, "
                         "so tied rows of one entity keep one order on every run) ascending. Sort on `data_quality_flag == 'ok'` as a boolean, descending like `passes`. "
                         "Never sort on the flag text, whose alphabetical order puts a flag such as 'missing covariate' before 'ok' "
                         "and one such as 'too few rows' after it. "
                         "`run_use_case()` asserts, before any step ranks rows, that with `i = [5, 3, 9, 0, 7, 1]`, "
                         "`rank_rows(pd.Series([False, True, False, False, False, False], index=i), pd.Series(['ok', 'ok', "
                         "'missing covariate', 'too few rows', 'ok', 'ok'], index=i), pd.Series([5, 1, 9, 8, 7, None], index=i, "
                         "dtype=float), pd.Series(['a', 'b', 'c', 'd', 'e', 'f'], index=i)).to_dict()` is "
                         "{5: 3, 3: 1, 9: 5, 0: 6, 7: 2, 1: 4}, and that `rank_rows(pd.Series([False, False, False]), "
                         "pd.Series(['ok', 'ok', 'ok']), pd.Series([0.010462203274983, 0.010462203274965, 0.0105]), "
                         "pd.Series(['c', 'b', 'a'])).tolist()` is [3, 2, 1]. Before the write, check the rank and stop the run on the first row that breaks it: "
                         "ranks run from 1 to the row count without a gap or a repeat, every row that passes the decision test ranks "
                         "before every row that does not, and every 'ok' row before every flagged row.")
GENIE_BASELINE_VALUE_RULE = ("The baseline's rows, written when no candidate passes acceptance, carry the baseline model's own "
                             "score for each entity of the scoring population as `value` and the baseline's name as `method`, also "
                             "when the candidate scores another quantity, such as an uplift the baseline does not estimate; a "
                             "baseline that does estimate the candidate's quantity writes that estimate as `value`, such as a "
                             "baseline given the treatment as a feature, whose prediction with the treatment set to 1 minus its "
                             "prediction with the treatment set to 0 is its uplift, so the rank, the reasons and the decision test "
                             "read the quantity the use case decides on; a "
                             "constant such as 0 or the class rate is only the reference the baseline is compared with and is never "
                             "written as `value`, and the summary says which score `value` holds.")
GENIE_BASELINE_MONITOR_RULE = ("When the baseline itself misses the acceptance thresholds, every row is Monitor whatever its decision "
                               "test gives, such as an uplift above 0 from a baseline whose AUC is at chance, and the summary says so.")
GENIE_MEASURED_TIER_RULE = ("This use case acts on measured amounts, so its tiers come from value_at_stake and the `act_now_amount` "
                            "and `review_amount` widgets alone: a candidate or a baseline that misses acceptance, or a test that is "
                            "not significant, never moves a row to Monitor. It is evidence for the row's confidence_level, which is "
                            "never 'High' on such a row, and the summary says so.")
GENIE_VALUE_AT_STAKE_RULE = ("`value_at_stake` is the money each record puts at stake, in the currency the `currency` widget names: one "
                             "formula in code over the record's own amounts, its assumptions named constants that the notebook "
                             "prints with their values before the write and the summary states, such as 'margin lost = (last "
                             "year's margin rate - this period's margin rate) x this period's sales'. {basis} It is never negative: "
                             "a record with nothing at stake has 0, and a record the formula cannot price, such as one without a "
                             "sales amount, has NULL and that reason in its data_quality_flag. A widget default the Parameters "
                             "section leaves for you to choose is set once while you build, as a fixed value in the widget, never "
                             "computed by a run: `currency` is the currency of the source amounts{amounts}.")
GENIE_VALUE_AT_STAKE_BASIS = {
    "measured": ("This use case acts on measured amounts, so the formula prices what the data already shows is lost, overpaid or "
                 "at risk now, never a forecast.",
                 ", and `act_now_amount` and `review_amount` come from the money at stake in the first full test run, such as the "
                 "amounts at stake on the records at its 95th and 80th percentiles, each rounded to 2 significant digits, and the "
                 "summary states them with that reason"),
    "predicted": ("This use case acts on a prediction, so the formula is the predicted probability, share or change times the "
                  "amount it applies to, such as the chance a customer leaves times their sales over the last 12 months.", ""),
}
GENIE_ACCEPTANCE_SECTION_TITLE = "Acceptance and Stop Rule"
GENIE_READINESS_STOP_RULE = ("The notebook measures data readiness before any modeling. When a readiness threshold is not met, the "
                             "data cannot support this use case today, and that is a valid, finished outcome: the notebook prints "
                             "which check failed with its measured and required values, writes nothing, and stops.")
GENIE_NO_FORCED_PASS_RULE = ("Genie Code never lowers a threshold, fabricates data, drops a check or switches to other tables to force "
                             "a pass; a threshold changes only when the user asks for it.")
GENIE_STOP_RULE_POINTER = f"STOP RULE: see {GENIE_ACCEPTANCE_SECTION_TITLE}."
GENIE_INSTRUCTION_REPAIR_ROUNDS = 3
GENIE_EVALUATION_PARITY_RULE = ("Score every candidate and the baseline with one evaluation function that takes the model's scorer "
                                "as an argument, on the same holdout cases with the same inputs and the same exclusions, so a gap "
                                "between their metrics comes from the models alone. Each case's held-out answer, such as the item "
                                "a basket is missing, an outcome label or a later value, is what the function checks a score "
                                "against, so nothing a scorer is given or has removed from its list contains it: a ranking leaves "
                                "out only the items the case gives the model, such as the basket's other items, never the held-out "
                                "item. Before the real evaluation, `run_use_case()` asserts that an oracle scorer, one that "
                                "returns each case's held-out answer, gets the best score the metric allows on that function, a "
                                "hit on every case, a perfect AUC or a zero MASE, so an evaluation that can never find the answer "
                                "stops the run instead of rejecting every candidate. Print each model's hits or errors beside its "
                                "rate, and the number of cases.")
GENIE_MODEL_FEATURE_RULE = ("Every model, the baseline among them, reads at least one feature that every row it scores has, "
                            "such as the days since the entity's last event, and gets one when a build step lists none: an imputer fills "
                            "a missing feature with one value, so rows that lack every feature a model reads, such as single-order "
                            "customers scored only on the gaps between their orders, all get one score, which is no result of their own. "
                            "That feature takes many values across the rows, such as a number of days or an amount, and is added when "
                            "every feature a build step lists is a category or a small count, such as one-hot segments and a log of the "
                            "prior order count: such a model gives each combination of their values one score, so hundreds of rows share "
                            "each score and its rank. The model reads that feature as its number, scaled or logged, also when a build step "
                            "bins it to name each row's driver, which its coefficient times the row's scaled value gives: quantile bins of "
                            "it are categories too, so a model that reads only bins, categories and small counts gives every row of one "
                            "combination of bins one score.")
GENIE_FEATURE_COLUMNS_RULE = ("Fit, evaluate and score every model on one list of feature column names, `feature_cols`, that holds no "
                              "name twice, selecting `x_pdf[feature_cols]` each time, so every frame a model reads has the fit's "
                              "columns in the fit's order: scikit-learn stops a prediction whose columns differ from the fit's in "
                              "order or name, and a repeated name, such as a current and a prior copy of one attribute under one "
                              "name, makes `x_pdf[feature_cols]` return both copies and stops the fit or the prediction. Score "
                              "another arm, such as the untreated arm of an uplift model or of its baseline, on a copy of the "
                              "scoring frame with that arm's columns set before the selection, "
                              "`scoring_pdf.assign(**{treatment_col: 0})[feature_cols]`, since the scoring frame has no treatment "
                              "column of its own. Cast a column to pandas' category dtype, as `categorical_features='from_dtype'` "
                              "needs, only in the frame a model reads, `frame[feature_cols].astype({c: 'category' for c in "
                              "categorical_cols})`, never in the frame later steps fill, label or write, where a text that is not one "
                              "of its categories, such as a label for a missing value, raises 'Cannot setitem on a Categorical with a "
                              "new category'. " + GENIE_MODEL_FEATURE_RULE)
GENIE_MLFLOW_RULE = (f"Serverless environment {GENIE_NOTEBOOK_ENV_VERSION} runs MLflow 2: call `mlflow.set_registry_uri(\"databricks-uc\")`, "
                     "log the model with keyword arguments only, because the first positional parameter of "
                     "`mlflow.pyfunc.log_model` is `artifact_path` and a model passed by position collides with it: "
                     "`mlflow.sklearn.log_model(sk_model=model, artifact_path=\"model\", signature=..., input_example=..., "
                     "registered_model_name=model_name)` or, for a pyfunc wrapper, `mlflow.pyfunc.log_model(python_model=wrapper, "
                     "artifact_path=\"model\", signature=..., input_example=..., registered_model_name=model_name)` (never `name=`), "
                     "take the new version from the returned model info's "
                     "`registered_model_version` (Unity Catalog rejects a `run_id` filter in `search_model_versions`), then set the "
                     "alias with `mlflow.MlflowClient().set_registered_model_alias(model_name, \"champion\", version)`. A pyfunc "
                     "wrapper subclasses `mlflow.pyfunc.PythonModel`, `class Wrapper(mlflow.pyfunc.PythonModel)` with `import mlflow` "
                     "among the notebook's imports, and defines `predict(self, context, model_input)`: MLflow 2 rejects an instance of "
                     "a plain class with a predict method ('`python_model` must be a PythonModel instance'). Write that "
                     "version as `model_version` on every row the accepted model scores, and assert before the write that it is not "
                     "empty, so a registration that never ran stops the test run instead of writing the accepted model's rows with "
                     "'' as their version.")
IMPLEMENT_RESULT_HISTORY_SOURCE = r'''
import hashlib as _inspire_hashlib
import json as _inspire_json
import re as _inspire_re
import time as _inspire_time
import uuid as _inspire_uuid

INSPIRE_HISTORY_COLUMNS = (("record_key", "STRING"), ("valid_from", "TIMESTAMP"), ("valid_to", "TIMESTAMP"),
                           ("is_current", "BOOLEAN"), ("change_source", "STRING"), ("changed_by", "STRING"))
INSPIRE_HISTORY_NAMES = tuple(name for name, _kind in INSPIRE_HISTORY_COLUMNS)
INSPIRE_ACTION_COLUMNS = (("action_owner", "STRING"), ("action_status", "STRING"), ("action_due", "DATE"), ("action_note", "STRING"),
                          ("dismiss_reason", "STRING"), ("action_outcome", "STRING"))
INSPIRE_ACTION_NAMES = tuple(name for name, _kind in INSPIRE_ACTION_COLUMNS)
INSPIRE_CARRIED_ACTIONS = ("action_owner", "action_status", "action_due", "action_note", "dismiss_reason")
INSPIRE_REPLACEABLE_SOURCES = ("run", "action")
INSPIRE_FLAGGED_TIERS = ("Act now", "Review")
INSPIRE_CURRENT_ROWS = "is_current AND change_source <> 'delete'"
INSPIRE_REPLACE_EDITS_PROPERTY = "inspire.replace_edits"
INSPIRE_RUN_KEY_COLUMNS = ("use_case_id", "run_id", "as_of_date", "entity_id")
INSPIRE_AI_FACTS = "ai_facts"
INSPIRE_CONFIDENCE = "confidence_level"
INSPIRE_CONFIDENCE_LEVELS = ("High", "Medium", "Low")
INSPIRE_VALUE_AT_STAKE = "value_at_stake"
INSPIRE_DECISION_BASES = ("measured", "predicted")
INSPIRE_TIER_AMOUNTS = ("act_now_amount", "review_amount")
INSPIRE_CURRENCY_RE = r"[A-Z]{3}"
INSPIRE_OUTCOME_COLUMNS = (("act_now_records", "BIGINT"), ("review_records", "BIGINT"), ("records", "BIGINT"),
                           (INSPIRE_VALUE_AT_STAKE, "DOUBLE"), ("currency", "STRING"), ("decision_basis", "STRING"),
                           ("act_now_amount", "DOUBLE"), ("review_amount", "DOUBLE"), ("nothing_to_act", "STRING"))
INSPIRE_AI_COLUMNS = (("ai_explanation", "STRING"), ("ai_confidence_reason", "STRING"), ("ai_facts_hash", "STRING"))
INSPIRE_AI_NAMES = tuple(name for name, _kind in INSPIRE_AI_COLUMNS)
INSPIRE_AI_CATEGORY_NAME = r"^ai_cat_[a-z0-9_]{1,40}$"
INSPIRE_AI_UNEXPLAINED = "Not explained yet"
INSPIRE_AI_AUTO_ROWS = 4000
INSPIRE_AI_PREVIEW_ROWS = 40
INSPIRE_AI_CHUNK_ROWS = 2000
INSPIRE_AI_ROWS_PER_SECOND = 4.5
INSPIRE_AI_PROMPT_VERSION = "2"
INSPIRE_AI_SUMMARY_SUFFIX = "_summary"
INSPIRE_AI_HEDGE = (r"(?i)\b(?:i(?:['’]m| am| only| have| haven['’]t| have not| do not| don['’]t| cannot| can['’]t| can not| would| "
                    r"could| need| lack| see| infer|['’]ve|['’]d)\b|as an ai\b|language model|(?:you|the user) (?:gave|provided|"
                    r"supplied|shared|sent|listed)\b|(?:facts|details|figures?|information|data|fields) (?:you )?(?:gave|given|"
                    r"provided|supplied|shared|listed)\b|(?:the|these|those|given|provided|supplied) (?:facts|prompt)\b|based (?:only|"
                    r"solely|purely|just) on\b|(?:not enough|insufficient) (?:information|context|detail|details|data)\b|without "
                    r"(?:more|additional|further) (?:information|context|data|details?)\b|cannot (?:be )?(?:determined?|told|"
                    r"assessed|confirmed)\b|it is (?:unclear|not clear)\b|unclear (?:from|whether)\b|(?:not|isn['’]t|is not) "
                    r"(?:specified|provided|given|included|stated) (?:in|here|above)\b)")


def inspire_table_parts(name):
    parts = [part.strip() for part in str(name or "").replace("`", "").split(".")]
    if len(parts) != 3 or not all(parts):
        raise ValueError(f"expected a three-level table name catalog.schema.table, got {name!r}")
    return parts


def inspire_table_name(name, suffix=""):
    catalog, schema, table = inspire_table_parts(name)
    return ".".join(f"`{part}`" for part in (catalog, schema, table + suffix))


def inspire_sql_literal(text):
    return "'" + str("" if text is None else text).replace("\\", "\\\\").replace("'", "\\'") + "'"


def inspire_column(name, alias=""):
    return (f"{alias}." if alias else "") + "`" + str(name).replace("`", "``") + "`"


def inspire_rows(frame):
    return [row.asDict() if hasattr(row, "asDict") else dict(row) for row in frame.collect()]


def inspire_table_schema(spark, table):
    try:
        rows = inspire_rows(spark.sql(f"DESCRIBE TABLE {inspire_table_name(table)}"))
    except Exception as error:
        if "TABLE_OR_VIEW_NOT_FOUND" in str(error) or "cannot be found" in str(error):
            return None
        raise
    schema = []
    for row in rows:
        name = str(row.get("col_name") or "").strip()
        if not name or name.startswith("#"):
            break
        schema.append((name, " ".join(str(row.get("data_type") or "").split())))
    return schema or None


def inspire_history_missing(columns, wanted=INSPIRE_HISTORY_COLUMNS):
    have = {str(column).lower() for column in columns or []}
    return [(name, kind) for name, kind in wanted if name not in have]


def inspire_sql_list(values):
    return ", ".join(inspire_sql_literal(value) for value in values)


def inspire_replaceable_sql(alias, replace_edits):
    return "TRUE" if replace_edits else f"{alias}.change_source IN ({inspire_sql_list(INSPIRE_REPLACEABLE_SOURCES)})"


def inspire_unique_sql(columns, column):
    if column not in {str(c).lower() for c in columns}:
        return "FALSE"
    return f"COUNT(*) = COUNT(DISTINCT NULLIF(TRIM(CAST({inspire_column(column)} AS STRING)), ''))"


def inspire_record_key_rule(columns, id_unique, label_unique):
    have = {str(c).lower() for c in columns}
    if id_unique:
        return ("entity_id",)
    if label_unique and "entity_label" in have:
        return ("entity_label",)
    return ("entity_id", "segment_or_item") if "segment_or_item" in have else ("entity_id",)


def inspire_record_key_sql(key_names, alias=""):
    parts = [f"CAST({inspire_column(name, alias)} AS STRING)" for name in key_names]
    return parts[0] if len(parts) == 1 else f"concat_ws(' | ', {', '.join(parts)})"


def inspire_number_text(value):
    return ("%.10f" % (float(value) + 0.0)).rstrip("0").rstrip(".")


def inspire_money_text(amount, currency):
    return f"{float(amount):,.2f}".rstrip("0").rstrip(".") + f" {currency}"


def inspire_amount(text):
    text = str("" if text is None else text).strip()
    return float(text) if _inspire_re.fullmatch(r"[0-9]+(?:\.[0-9]+)?", text) else None


def inspire_amount_order_problem(act_now, review):
    if act_now is None or review is None or review <= act_now:
        return ""
    return (f"review_amount is {inspire_number_text(review)}, above act_now_amount {inspire_number_text(act_now)}; a record "
            "needs at least as much at stake to act on now as to review")


def inspire_value_run_config(names):
    currency = inspire_widget("currency", None)
    if INSPIRE_VALUE_AT_STAKE not in names and currency is None:
        return None
    run = globals().get("INSPIRE_RUN") or {}
    basis = str(run.get("decision_basis") or "").strip().lower()
    return {"basis": basis, "currency": str(currency or "").strip(),
            "amounts": {name: inspire_widget(name, "") for name in INSPIRE_TIER_AMOUNTS} if basis == "measured" else {}}


def inspire_value_amounts(value):
    return tuple(inspire_amount((value or {}).get("amounts", {}).get(name)) for name in INSPIRE_TIER_AMOUNTS)


def inspire_value_problems(value):
    problems = []
    if value["basis"] not in INSPIRE_DECISION_BASES:
        problems.append(f"the run's decision basis is '{value['basis']}', and it must be one of {', '.join(INSPIRE_DECISION_BASES)}")
    if not _inspire_re.fullmatch(INSPIRE_CURRENCY_RE, value["currency"]):
        problems.append(f"the currency widget is '{value['currency']}'; set its default to the 3-letter ISO code of the money "
                        f"{INSPIRE_VALUE_AT_STAKE} holds, such as USD")
    for name, text in value["amounts"].items():
        if inspire_amount(text) is None:
            problems.append(f"the {name} widget is '{text}'; set its default to an amount of at least 0 in the currency of "
                            f"{INSPIRE_VALUE_AT_STAKE}, since this use case acts on measured amounts")
    order = inspire_amount_order_problem(*inspire_value_amounts(value))
    return problems + ([order] if order else [])


def inspire_run_profile_sql(stage, columns, value=None):
    blank = "COUNT_IF({0} IS NULL OR TRIM(CAST({0} AS STRING)) = '')"
    named = {str(c).lower(): inspire_column(c) for c in columns}
    checks = ""
    if value and INSPIRE_VALUE_AT_STAKE in named:
        stake, tier = named[INSPIRE_VALUE_AT_STAKE], inspire_tier_column(columns)
        act_now, review = inspire_value_amounts(value)
        checks += f", COUNT_IF({stake} < 0) AS negative_stake"
        if value["basis"] == "measured" and tier and act_now is not None and review is not None:
            ok = f"{named['data_quality_flag']} = 'ok'" if "data_quality_flag" in named else "TRUE"
            checks += (f", COUNT_IF(NOT ({inspire_column(tier)} <=> CASE WHEN {ok} AND {stake} >= {act_now!r} THEN 'Act now' "
                       f"WHEN {ok} AND {stake} >= {review!r} THEN 'Review' ELSE 'Monitor' END)) AS off_tier")
    if INSPIRE_CONFIDENCE in named:
        level = named[INSPIRE_CONFIDENCE]
        checks += (f", COUNT_IF({level} IS NULL OR {level} NOT IN ({inspire_sql_list(INSPIRE_CONFIDENCE_LEVELS)})) "
                   "AS bad_levels")
    if INSPIRE_AI_FACTS in named:
        tier = inspire_tier_column(columns)
        flagged = f"{inspire_column(tier)} IN ({inspire_sql_list(INSPIRE_FLAGGED_TIERS)})" if tier else "FALSE"
        checks += f", COUNT_IF({flagged} AND ({named[INSPIRE_AI_FACTS]} IS NULL OR TRIM({named[INSPIRE_AI_FACTS]}) = '')) AS bare_flagged"
    return (f"SELECT COUNT(*) AS row_count, COUNT(DISTINCT use_case_id) AS use_cases, MIN(use_case_id) AS use_case_id, "
            f"COUNT(DISTINCT run_id) AS runs, MIN(run_id) AS run_id, COUNT_IF(as_of_date IS NULL) AS undated, "
            f"{blank.format('use_case_id')} AS no_use_case, {blank.format('run_id')} AS no_run, "
            f"{inspire_unique_sql(columns, 'entity_id')} AS id_unique, {inspire_unique_sql(columns, 'entity_label')} AS label_unique"
            f"{checks} FROM {inspire_table_name(stage)}")


def inspire_run_problems(profile, value=None):
    problems = []
    if int(profile.get("negative_stake") or 0):
        problems.append(f"{profile['negative_stake']} row(s) have a {INSPIRE_VALUE_AT_STAKE} below 0; it holds the money the record "
                        "puts at stake, 0 when nothing is at stake")
    if int(profile.get("off_tier") or 0):
        act_now, review = inspire_value_amounts(value)
        problems.append(f"{profile['off_tier']} row(s) have a priority_tier that does not follow their {INSPIRE_VALUE_AT_STAKE}: this "
                        f"use case acts on measured amounts, so a row whose data_quality_flag is 'ok' is 'Act now' at "
                        f"{inspire_money_text(act_now, value['currency'])} or more, 'Review' at "
                        f"{inspire_money_text(review, value['currency'])} or more, and every other row is 'Monitor'")
    if int(profile.get("no_use_case") or 0) or int(profile.get("use_cases") or 0) != 1:
        problems.append("every row must carry this use case's id in use_case_id, and only that id")
    if int(profile.get("no_run") or 0) or int(profile.get("runs") or 0) != 1:
        problems.append("every row must carry this run's one run_id")
    if int(profile.get("undated") or 0):
        problems.append(f"{profile['undated']} row(s) have no as_of_date; write the resolved as-of date on every row")
    if int(profile.get("bad_levels") or 0):
        problems.append(f"{profile['bad_levels']} row(s) have a {INSPIRE_CONFIDENCE} other than "
                        f"{', '.join(repr(level) for level in INSPIRE_CONFIDENCE_LEVELS)}; set it on every row with the evidence rule")
    if int(profile.get("bare_flagged") or 0):
        problems.append(f"{profile['bare_flagged']} 'Act now' or 'Review' row(s) have no {INSPIRE_AI_FACTS}, the facts Inspire "
                        "explains each flagged record from; build them on every row")
    return problems


def inspire_key_repeats_sql(stage, key_names):
    return (f"SELECT k AS record_key, COUNT(*) AS row_count FROM (SELECT {inspire_record_key_sql(key_names)} AS k "
            f"FROM {inspire_table_name(stage)}) GROUP BY k HAVING COUNT(*) > 1 OR k IS NULL OR TRIM(k) = '' LIMIT 3")


def inspire_history_create_sql(table, stage, columns, key_names, ai=None):
    picked = ", ".join(inspire_column(c, "s") for c in columns)
    actions = ", ".join(f"CAST(NULL AS {kind}) AS {inspire_column(name)}"
                        for name, kind in INSPIRE_ACTION_COLUMNS + inspire_ai_columns(ai))
    return (f"CREATE TABLE {inspire_table_name(table)} AS SELECT {picked}, {actions}, {inspire_record_key_sql(key_names, 's')} AS record_key, "
            f"current_timestamp() AS valid_from, CAST(NULL AS TIMESTAMP) AS valid_to, TRUE AS is_current, 'run' AS change_source, "
            f"concat('run ', s.run_id) AS changed_by FROM {inspire_table_name(stage)} AS s")


def inspire_history_backup_sql(table):
    return f"CREATE OR REPLACE TABLE {inspire_table_name(table, '_backup')} DEEP CLONE {inspire_table_name(table)}"


def inspire_history_add_sql(table, missing):
    return f"ALTER TABLE {inspire_table_name(table)} ADD COLUMNS ({', '.join(f'{name} {kind}' for name, kind in missing)})"


def inspire_history_pending_sql(table):
    return f"SELECT COUNT_IF(is_current IS NULL) AS pending FROM {inspire_table_name(table)}"


def inspire_history_backfill_sql(table, columns):
    target = inspire_table_name(table)
    key = (f"CASE WHEN k.id_unique THEN {inspire_record_key_sql(('entity_id',), 'r')} "
           f"WHEN k.label_unique THEN {inspire_record_key_sql(inspire_record_key_rule(columns, False, True), 'r')} "
           f"ELSE {inspire_record_key_sql(inspire_record_key_rule(columns, False, False), 'r')} END")
    computed = {"record_key": key, "valid_from": "COALESCE(CAST(r.as_of_date AS TIMESTAMP), current_timestamp())",
                "valid_to": ("CASE WHEN r.as_of_date IS NULL THEN current_timestamp() WHEN d.next_date IS NOT NULL THEN "
                             "CAST(d.next_date AS TIMESTAMP) WHEN v.use_case_id IS NOT NULL THEN current_timestamp() END"),
                "is_current": "COALESCE(r.as_of_date IS NOT NULL AND d.next_date IS NULL AND v.use_case_id IS NULL, FALSE)",
                "change_source": "'run'", "changed_by": "COALESCE(concat('run ', r.run_id), 'run')"}
    picked = ", ".join(f"CASE WHEN r.is_current IS NULL THEN {computed[str(c).lower()]} ELSE {inspire_column(c, 'r')} END"
                       if str(c).lower() in computed else inspire_column(c, "r") for c in columns)
    return (f"INSERT OVERWRITE {target} SELECT {picked} FROM {target} AS r LEFT JOIN (SELECT use_case_id, as_of_date, "
            f"LEAD(as_of_date) OVER (PARTITION BY use_case_id ORDER BY as_of_date) AS next_date FROM (SELECT DISTINCT use_case_id, "
            f"as_of_date FROM {target} WHERE is_current IS NULL AND as_of_date IS NOT NULL)) AS d ON r.is_current IS NULL AND "
            f"d.use_case_id <=> r.use_case_id AND d.as_of_date = r.as_of_date LEFT JOIN (SELECT use_case_id, run_id, "
            f"{inspire_unique_sql(columns, 'entity_id')} AS id_unique, {inspire_unique_sql(columns, 'entity_label')} AS label_unique "
            f"FROM {target} WHERE is_current IS NULL GROUP BY use_case_id, run_id) AS k ON r.is_current IS NULL AND "
            f"k.use_case_id <=> r.use_case_id AND k.run_id <=> r.run_id LEFT JOIN (SELECT DISTINCT use_case_id FROM {target} "
            f"WHERE is_current) AS v ON r.is_current IS NULL AND v.use_case_id <=> r.use_case_id")


def inspire_type_change_sql(table, table_schema, run_schema, replace_edits):
    wanted = {str(name).lower(): kind for name, kind in run_schema}
    picked, changed = [], []
    for name, kind in table_schema:
        target = wanted.get(str(name).lower())
        column = inspire_column(name, "r")
        have, want = kind.replace(" ", "").lower(), (target or "").replace(" ", "").lower()
        if target is None or have == want:
            picked.append(column)
            continue
        if have.startswith("struct<") and ("<result:" in have or ",result:" in have) and want == "string":
            value = f"CAST({column}.result AS STRING)"
        elif have == "string" and want == "array<string>":
            value = f"IF({column} IS NULL, NULL, array({column}))"
        else:
            value = f"TRY_CAST({column} AS {target})"
        picked.append(f"{value} AS {inspire_column(name)}")
        changed.append(f"{name} ({kind} to {target})")
    if not changed:
        return "", []
    flag = inspire_sql_literal("true" if replace_edits else "false")
    return (f"CREATE OR REPLACE TABLE {inspire_table_name(table)} TBLPROPERTIES ({inspire_sql_literal(INSPIRE_REPLACE_EDITS_PROPERTY)} = "
            f"{flag}) AS SELECT {', '.join(picked)} FROM {inspire_table_name(table)} AS r", changed)


def inspire_add_columns_sql(table, table_schema, run_schema):
    have = {str(name).lower() for name, _kind in table_schema}
    new = [(name, kind) for name, kind in run_schema if str(name).lower() not in have]
    if not new:
        return "", []
    return (f"ALTER TABLE {inspire_table_name(table)} ADD COLUMNS ({', '.join(f'{inspire_column(n)} {k}' for n, k in new)})",
            [name for name, _kind in new])


def inspire_kept_edits_sql(table, use_case_id):
    return (f"SELECT COUNT(*) AS kept FROM {inspire_table_name(table)} WHERE use_case_id = {inspire_sql_literal(use_case_id)} "
            f"AND is_current AND change_source NOT IN ({inspire_sql_list(INSPIRE_REPLACEABLE_SOURCES)})")


def inspire_tier_column(columns):
    return next((c for c in columns if str(c).lower() == "priority_tier"), None)


def inspire_history_merge_sql(table, stage, columns, key_names, use_case_id, replace_edits, ai=None):
    target, source = inspire_table_name(table), inspire_table_name(stage)
    key = inspire_record_key_sql(key_names, "s")
    picked = ", ".join(inspire_column(c, "s") for c in columns)
    reasoning = [name for name, _kind in inspire_ai_columns(ai)]
    same_facts = f"p.ai_facts_hash = {inspire_ai_hash_sql(ai['fingerprint'], 's')}" if ai else ""
    carried = ", ".join([inspire_column(c, "p") for c in INSPIRE_ACTION_NAMES]
                        + [f"CASE WHEN {same_facts} THEN {inspire_column(c, 'p')} END AS {inspire_column(c)}" for c in reasoning])
    close = "SET valid_to = current_timestamp(), is_current = FALSE"
    prior = ", ".join(inspire_column(c, "r") for c in ("record_key", "is_current", "change_source") + INSPIRE_ACTION_NAMES
                      + tuple(reasoning))
    latest = (f"SELECT {prior} FROM {target} AS r WHERE r.use_case_id = {inspire_sql_literal(use_case_id)} QUALIFY ROW_NUMBER() "
              f"OVER (PARTITION BY r.record_key ORDER BY r.is_current DESC, r.valid_from DESC) = 1")
    tier = inspire_tier_column(columns)
    status, shown = inspire_column("action_status", "u"), inspire_column(tier, "u") if tier else "NULL"
    values = [f"CASE WHEN {status} = 'Dismissed' THEN 'Monitor' ELSE {shown} END" if c == tier else inspire_column(c, "u")
              for c in columns]
    outcome = (f"CASE WHEN {status} = 'Done' AND {shown} IN ({inspire_sql_list(INSPIRE_FLAGGED_TIERS)}) THEN 'Still flagged' "
               f"WHEN {status} = 'Done' THEN 'Resolved' ELSE {inspire_column('action_outcome', 'u')} END")

    def replaceable(alias):
        return inspire_replaceable_sql(alias, replace_edits)

    return (f"MERGE INTO {target} AS t USING (SELECT {key} AS _inspire_key, {key} AS _inspire_record_key, {picked}, {carried} "
            f"FROM {source} AS s LEFT JOIN ({latest}) AS p ON p.record_key = {key} "
            f"UNION ALL SELECT NULL AS _inspire_key, {key} AS _inspire_record_key, {picked}, {carried} FROM {source} AS s JOIN "
            f"({latest}) AS p ON p.record_key = {key} AND p.is_current AND {replaceable('p')}) AS u "
            f"ON t.use_case_id = u.use_case_id AND t.record_key = u._inspire_key AND t.is_current "
            f"WHEN MATCHED AND {replaceable('t')} THEN UPDATE {close} "
            f"WHEN NOT MATCHED THEN INSERT ({', '.join(inspire_column(c) for c in columns + list(INSPIRE_ACTION_NAMES) + reasoning)}, "
            f"record_key, valid_from, valid_to, is_current, change_source, changed_by) VALUES ({', '.join(values)}, "
            f"{', '.join(inspire_column(c, 'u') for c in INSPIRE_CARRIED_ACTIONS)}, {outcome}, "
            + "".join(f"{inspire_column(c, 'u')}, " for c in reasoning) +
            f"u._inspire_record_key, current_timestamp(), NULL, TRUE, 'run', concat('run ', u.run_id)) "
            f"WHEN NOT MATCHED BY SOURCE AND t.use_case_id = {inspire_sql_literal(use_case_id)} AND t.is_current AND "
            f"{replaceable('t')} THEN UPDATE {close}")


def inspire_action_counts_sql(table, stage, columns, key_names):
    target, source = inspire_table_name(table), inspire_table_name(stage)
    tier = inspire_tier_column(columns)
    kept_out = f"t.action_status = 'Dismissed' AND {inspire_column(tier, 's')} = 'Act now'" if tier else "FALSE"
    return (f"SELECT COUNT_IF(t.action_status = 'Dismissed') AS dismissed, COUNT_IF({kept_out}) AS kept_out, "
            f"COUNT_IF(t.action_status = 'Done' AND t.action_outcome = 'Resolved') AS resolved, "
            f"COUNT_IF(t.action_status = 'Done' AND t.action_outcome = 'Still flagged') AS still_flagged FROM {target} AS t JOIN "
            f"{source} AS s ON t.use_case_id = s.use_case_id AND t.run_id = s.run_id AND t.record_key = "
            f"{inspire_record_key_sql(key_names, 's')} WHERE t.is_current AND t.change_source = 'run'")


def inspire_replace_edits(spark, table):
    for row in inspire_rows(spark.sql(f"SHOW TBLPROPERTIES {inspire_table_name(table)}")):
        if str(row.get("key")) == INSPIRE_REPLACE_EDITS_PROPERTY:
            return str(row.get("value")).strip().lower() == "true"
    return False


def inspire_ensure_history(spark, table, schema=None, backup=True, out=print):
    schema = inspire_table_schema(spark, table) if schema is None else list(schema)
    if schema is None:
        return None
    columns = [name for name, _kind in schema]
    missing = inspire_history_missing(columns)
    pending = 0 if missing else int(inspire_rows(spark.sql(inspire_history_pending_sql(table)))[0].get("pending") or 0)
    if not missing and not pending:
        return {"added": [], "backfilled": False}
    absent = [name for name in INSPIRE_RUN_KEY_COLUMNS if name not in {str(c).lower() for c in columns}]
    if absent:
        raise ValueError(f"{table} has no {', '.join(absent)} column, so Inspire cannot tell its runs apart to keep their change history")
    if backup:
        spark.sql(inspire_history_backup_sql(table))
    if missing:
        spark.sql(inspire_history_add_sql(table, missing))
        columns = columns + [name for name, _kind in missing]
    spark.sql(inspire_history_backfill_sql(table, columns))
    out(f"Change history: {table} now keeps every version of each record"
        + (f" (added {', '.join(name for name, _kind in missing)})" if missing else "")
        + "; each row already there became a version of its run, the newest run's rows current"
        + (f"; the table as it was is in {table}_backup" if backup else ""))
    return {"added": [name for name, _kind in missing], "backfilled": True}


def inspire_ensure_actions(spark, table, schema=None, out=print):
    schema = inspire_table_schema(spark, table) if schema is None else list(schema)
    if schema is None:
        return None
    missing = inspire_history_missing([name for name, _kind in schema], INSPIRE_ACTION_COLUMNS)
    if missing:
        spark.sql(inspire_history_add_sql(table, missing))
        out(f"Actions: added {', '.join(name for name, _kind in missing)} to {table}, where people record in the App what they do "
            "about each record; the rows already there hold NULL in them")
    return [name for name, _kind in missing]


def inspire_widget(name, default=""):
    dbu = globals().get("dbutils")
    if dbu is None:
        return default
    try:
        return str(dbu.widgets.get(name)).strip()
    except Exception:
        return default


def inspire_is_preview(run_profile):
    return str(run_profile).strip().lower() not in ("", "full")


def inspire_ai_owned(name):
    text = str(name).lower()
    return text in INSPIRE_AI_NAMES or _inspire_re.match(INSPIRE_AI_CATEGORY_NAME, text) is not None


def inspire_ai_columns(ai):
    return INSPIRE_AI_COLUMNS + tuple((name, "STRING") for name in ai["categories"]) if ai else ()


def inspire_ai_category_problems(categories):
    if not isinstance(categories, dict) or not 1 <= len(categories) <= 3:
        return ["AI_CATEGORIES must be a top-level dict of 1 to 3 categories, each keyed ai_cat_<lowercase letters, digits or _> "
                "and holding a meaning and 2 to 8 distinct values, such as {'ai_cat_driver': {'meaning': 'what moved the record', "
                "'values': ['Price', 'Volume', 'Mix']}}"]
    problems = []
    for name, spec in categories.items():
        if not isinstance(name, str) or _inspire_re.match(INSPIRE_AI_CATEGORY_NAME, name) is None:
            problems.append(f"the AI category {name!r} is not named ai_cat_<lowercase letters, digits or _>")
            continue
        spec = spec if isinstance(spec, dict) else {}
        values = spec.get("values")
        texts = [v.strip() for v in values] if isinstance(values, (list, tuple)) and all(isinstance(v, str) for v in values) else []
        if not str(spec.get("meaning") or "").strip():
            problems.append(f"{name} has no meaning")
        if not 2 <= len(texts) <= 8 or not all(texts) or len({t.lower() for t in texts}) != len(texts):
            problems.append(f"{name} needs 2 to 8 distinct, non-empty text values")
        if any(t.lower() == INSPIRE_AI_UNEXPLAINED.lower() for t in texts):
            problems.append(f"{name} uses {INSPIRE_AI_UNEXPLAINED!r} as a value, which the App keeps for records without reasoning")
        if any(len(t) > 60 for t in texts):
            problems.append(f"{name} has a value longer than 60 characters")
    return problems


def inspire_ai_prompt(categories, context):
    lines = [str(context or "").strip(), "",
             "If anything below conflicts with an explicit directive from the user above, the user's directive wins.", "",
             "You explain one record of this use case's results to the person who acts on it. The facts at the end are what is "
             "known about the record. Reply with one JSON object and nothing else, with these keys:",
             '- "explanation": 2 to 4 plain sentences: what stands out about this record, why it matters to the business, and what '
             "to check or do next. State the facts as facts. Never write about yourself, about the facts or this message, or about "
             'what is not known, and never use the word "I".',
             '- "confidence_reason": one plain sentence, in the same style, naming the facts behind the record\'s confidence '
             "level, which the facts end with; never change the level or give another one."]
    for name, spec in categories.items():
        lines.append(f'- "{name}": exactly one of {", ".join(_inspire_json.dumps(v) for v in spec["values"])}: {spec["meaning"]}.')
    return "\n".join(lines + ["", "Facts:", ""]).lstrip()


def inspire_ai_fingerprint(endpoint, prompt):
    return _inspire_hashlib.sha256(f"{INSPIRE_AI_PROMPT_VERSION}\n{endpoint}\n{prompt}".encode("utf-8")).hexdigest()[:16]


def inspire_ai_facts_sql(alias):
    return (f"concat(coalesce({inspire_column(INSPIRE_AI_FACTS, alias)}, ''), {inspire_sql_literal(chr(10) + 'Confidence level: ')}, "
            f"coalesce({inspire_column(INSPIRE_CONFIDENCE, alias)}, ''))")


def inspire_ai_hash_sql(fingerprint, alias):
    return f"sha2(concat({inspire_sql_literal(fingerprint)}, {inspire_ai_facts_sql(alias)}), 256)"


def inspire_ai_config(categories, endpoint, context):
    problems = inspire_ai_category_problems(categories)
    if problems:
        raise ValueError("Inspire did not write the business output: " + "; ".join(problems))
    clean = {name: {"meaning": str(spec["meaning"]).strip(), "values": [v.strip() for v in spec["values"]]}
             for name, spec in categories.items()}
    endpoint = str(endpoint or "").strip()
    prompt = inspire_ai_prompt(clean, context)
    return {"endpoint": endpoint, "categories": clean, "prompt": prompt, "fingerprint": inspire_ai_fingerprint(endpoint, prompt),
            "version": INSPIRE_AI_PROMPT_VERSION}


def inspire_ai_run_config(categories=None, endpoint=None, context=None):
    run = globals().get("INSPIRE_RUN") or {}
    return inspire_ai_config(globals().get("AI_CATEGORIES") if categories is None else categories,
                             inspire_widget("ai_endpoint") if endpoint is None else endpoint,
                             run.get("ai_context", "") if context is None else context)


def inspire_ai_duration(rows):
    seconds = max(0, int(rows)) / INSPIRE_AI_ROWS_PER_SECOND
    if seconds < 90:
        return f"about {max(1, round(seconds))} seconds"
    if seconds < 5400:
        return f"about {round(seconds / 60)} minutes"
    return f"about {round(seconds / 3600)} hours"


def inspire_ai_flagged_sql(tier, alias):
    return f"{inspire_column(tier, alias)} IN ({inspire_sql_list(INSPIRE_FLAGGED_TIERS)})"


def inspire_ai_need_sql(table, use_case_id, ai, tier):
    return (f"SELECT t.* FROM {inspire_table_name(table)} AS t WHERE t.use_case_id = {inspire_sql_literal(use_case_id)} AND "
            f"t.is_current AND t.change_source <> 'delete' AND {inspire_ai_flagged_sql(tier, 't')} AND "
            f"nullif(trim(t.{INSPIRE_AI_FACTS}), '') IS NOT NULL AND "
            f"(t.ai_facts_hash IS NULL OR t.ai_facts_hash <> {inspire_ai_hash_sql(ai['fingerprint'], 't')})")


def inspire_ai_work_table(table, suffix, part):
    return inspire_table_name(table, f"{suffix}_{part}")


def inspire_ai_work_sql(table, suffix, need, tier, rank, rows):
    order = (f"CASE {inspire_column(tier, 'n')} " + " ".join(f"WHEN {inspire_sql_literal(t)} THEN {i}" for i, t in
                                                            enumerate(INSPIRE_FLAGGED_TIERS)) + " END, "
             + (f"{inspire_column(rank, 'n')} ASC NULLS LAST, " if rank else "") + "n.record_key")
    picked = (f"n.use_case_id, n.record_key, n.{INSPIRE_AI_FACTS}, n.{INSPIRE_CONFIDENCE}, {inspire_column(tier, 'n')}"
              + (f", {inspire_column(rank, 'n')}" if rank else ""))
    return (f"CREATE TABLE {inspire_ai_work_table(table, suffix, 'work')} AS SELECT n.use_case_id, n.record_key, "
            f"n.{INSPIRE_AI_FACTS}, n.{INSPIRE_CONFIDENCE}, CAST(floor((ROW_NUMBER() OVER (ORDER BY {order}) - 1) / "
            f"{INSPIRE_AI_CHUNK_ROWS}) AS INT) AS ai_chunk FROM (SELECT {picked} FROM ({need}) AS n ORDER BY {order} "
            f"LIMIT {int(rows)}) AS n")


def inspire_ai_reply_sql(batch, ai, retry=False):
    asked = f"concat({inspire_sql_literal(ai['prompt'])}, {inspire_ai_facts_sql('w')}"
    if retry:
        asked += (f", {inspire_sql_literal(chr(10) * 2 + 'Your last reply was rejected (')}, w.ai_problem, "
                  f"{inspire_sql_literal('). Reply again with one JSON object that follows every rule above.')}")
    return (f"SELECT w.use_case_id, w.record_key, w.{INSPIRE_AI_FACTS}, w.{INSPIRE_CONFIDENCE}, w.ai_chunk, "
            f"ai_query({inspire_sql_literal(ai['endpoint'])}, {asked}), failOnError => false) AS ai_reply FROM ({batch}) AS w")


def inspire_ai_checks(ai):
    hedge = inspire_sql_literal(INSPIRE_AI_HEDGE)
    checks = [("endpoint error", "c.ai_error IS NOT NULL", "c.ai_error"),
              ("unreadable reply", "coalesce(c.ai_explanation, '') = ''", "left(c.ai_raw, 300)"),
              ("hedged explanation", f"c.ai_explanation RLIKE {hedge}", f"regexp_extract(c.ai_explanation, {hedge}, 0)"),
              ("no confidence reason", "coalesce(c.ai_confidence_reason, '') = ''", "NULL"),
              ("hedged confidence reason", f"c.ai_confidence_reason RLIKE {hedge}",
               f"regexp_extract(c.ai_confidence_reason, {hedge}, 0)")]
    return checks + [(f"{name} outside its values", f"{inspire_column(name, 'c')} IS NULL", inspire_column(name + "_given", "c"))
                     for name in ai["categories"]]


def inspire_ai_checked_sql(table, suffix, ai):
    keys = ("explanation", "confidence_reason") + tuple(ai["categories"])
    shape = inspire_sql_literal(", ".join(f"{key} STRING" for key in keys))
    parsed = (f"SELECT a.use_case_id, a.record_key, a.{INSPIRE_AI_FACTS}, a.{INSPIRE_CONFIDENCE}, a.ai_chunk, "
              f"a.ai_reply.errorMessage AS ai_error, a.ai_reply.result AS ai_raw, "
              f"from_json(regexp_extract(a.ai_reply.result, '(?s)[{{].*[}}]', 0), {shape}) AS ai_json "
              f"FROM {inspire_ai_work_table(table, suffix, 'reply')} AS a")
    categories = []
    for name, spec in ai["categories"].items():
        given = inspire_column(name, "b.ai_json")
        picks = " ".join(f"WHEN {inspire_sql_literal(v.lower())} THEN {inspire_sql_literal(v)}" for v in spec["values"])
        categories.append(f"{given} AS {inspire_column(name + '_given')}, CASE lower(trim({given})) {picks} END AS {inspire_column(name)}")
    fields = (f"SELECT b.use_case_id, b.record_key, b.{INSPIRE_AI_FACTS}, b.{INSPIRE_CONFIDENCE}, b.ai_chunk, b.ai_error, b.ai_raw, "
              f"trim(b.ai_json.explanation) AS ai_explanation, trim(b.ai_json.confidence_reason) AS ai_confidence_reason, "
              f"{', '.join(categories)} FROM ({parsed}) AS b")
    checks = inspire_ai_checks(ai)
    kind = " ".join(f"WHEN {test} THEN {inspire_sql_literal(label)}" for label, test, _detail in checks)
    problem = " ".join(f"WHEN {test} THEN concat({inspire_sql_literal(label)}, coalesce(concat(': ', {detail}), ''))"
                       for label, test, detail in checks)
    return f"SELECT c.*, CASE {kind} END AS ai_problem_kind, CASE {problem} END AS ai_problem FROM ({fields}) AS c"


def inspire_ai_update_sql(table, suffix, ai):
    names = [name for name, _kind in inspire_ai_columns(ai) if name != "ai_facts_hash"]
    sets = ", ".join(f"{inspire_column(n)} = {inspire_column(n, 'c')}" for n in names)
    return (f"MERGE INTO {inspire_table_name(table)} AS t USING (SELECT * FROM {inspire_ai_work_table(table, suffix, 'checked')} "
            f"WHERE ai_problem_kind IS NULL) AS c "
            f"ON t.use_case_id = c.use_case_id AND t.record_key = c.record_key AND t.is_current AND "
            f"t.{INSPIRE_AI_FACTS} <=> c.{INSPIRE_AI_FACTS} AND t.{INSPIRE_CONFIDENCE} <=> c.{INSPIRE_CONFIDENCE} "
            f"WHEN MATCHED THEN UPDATE SET {sets}, ai_facts_hash = {inspire_ai_hash_sql(ai['fingerprint'], 'c')}")


def inspire_ai_status_sql(table, use_case_id, ai, tier):
    same = f"t.ai_facts_hash = {inspire_ai_hash_sql(ai['fingerprint'], 't')}"
    return (f"SELECT COUNT(*) AS records, COUNT_IF({same}) AS explained, COUNT(DISTINCT t.{INSPIRE_CONFIDENCE}) AS levels, "
            f"MIN(t.{INSPIRE_CONFIDENCE}) AS level FROM {inspire_table_name(table)} AS t WHERE t.use_case_id = "
            f"{inspire_sql_literal(use_case_id)} AND t.is_current AND t.change_source <> 'delete' AND "
            f"{inspire_ai_flagged_sql(tier, 't')} AND nullif(trim(t.{INSPIRE_AI_FACTS}), '') IS NOT NULL")


def inspire_ensure_ai(spark, table, schema, ai, out=print):
    missing = inspire_history_missing([name for name, _kind in schema or []], inspire_ai_columns(ai))
    if missing:
        spark.sql(inspire_history_add_sql(table, missing))
        out(f"AI reasoning: added {', '.join(name for name, _kind in missing)} to {table}, where Inspire keeps the AI reasoning of "
            "each record; the rows already there hold NULL in them until they are explained")
    return [name for name, _kind in missing]


def inspire_ai_explain(spark, table, use_case_id, ai, cap, out=print):
    columns = [name for name, _kind in inspire_table_schema(spark, table) or []]
    tier = inspire_tier_column(columns)
    rank = next((c for c in columns if str(c).lower() == "rank"), None)
    report = {"needed": 0, "explained": 0, "without": 0, "kinds": {}, "later": 0}
    if not tier:
        out(f"AI reasoning: {table} has no priority_tier column, so no record is flagged 'Act now' or 'Review' for the AI to explain")
        return report
    need = inspire_ai_need_sql(table, use_case_id, ai, tier)
    count = report["needed"] = int(inspire_rows(spark.sql(f"SELECT COUNT(*) AS n FROM ({need}) AS q"))[0].get("n") or 0)
    if not count:
        out(f"AI reasoning: every current 'Act now' and 'Review' record with facts already holds the reasoning of {ai['endpoint']} "
            "for those facts")
        return report
    rows = min(count, max(1, int(cap)))
    report["later"] = count - rows
    suffix, started = "_inspire_ai_" + _inspire_uuid.uuid4().hex[:12], _inspire_time.time()
    chunks = (rows + INSPIRE_AI_CHUNK_ROWS - 1) // INSPIRE_AI_CHUNK_ROWS
    out(f"AI reasoning: explaining {rows} 'Act now' and 'Review' record(s) with {ai['endpoint']}, best ranked first, in {chunks} "
        f"batch(es), {inspire_ai_duration(rows)}"
        + (f"; the next runs explain the other {count - rows}, at most {rows} per run like this one" if rows < count else ""))
    try:
        spark.sql(inspire_ai_work_sql(table, suffix, need, tier, rank, rows))
        for chunk in range(chunks):
            batch = f"SELECT * FROM {inspire_ai_work_table(table, suffix, 'work')} WHERE ai_chunk = {chunk}"
            for attempt in ("first", "retry"):
                spark.sql(f"CREATE OR REPLACE TABLE {inspire_ai_work_table(table, suffix, 'reply')} AS "
                          f"{inspire_ai_reply_sql(batch, ai, retry=attempt == 'retry')}")
                spark.sql(f"CREATE OR REPLACE TABLE {inspire_ai_work_table(table, suffix, 'checked')} AS "
                          f"{inspire_ai_checked_sql(table, suffix, ai)}")
                inspire_sql_retrying(spark, inspire_ai_update_sql(table, suffix, ai), table, "Inspire saved AI reasoning to it",
                                     "saving the reasoning", out)
                tally = {str(r.get("kind") or ""): int(r.get("n") or 0) for r in inspire_rows(spark.sql(
                    f"SELECT coalesce(ai_problem_kind, '') AS kind, COUNT(*) AS n "
                    f"FROM {inspire_ai_work_table(table, suffix, 'checked')} GROUP BY 1"))}
                report["explained"] += tally.pop("", 0)
                if not tally:
                    break
                if attempt == "retry":
                    for kind, n in tally.items():
                        report["kinds"][kind] = report["kinds"].get(kind, 0) + n
                batch = f"SELECT * FROM {inspire_ai_work_table(table, suffix, 'checked')} WHERE ai_problem_kind IS NOT NULL"
    finally:
        for part in ("work", "reply", "checked"):
            spark.sql(f"DROP TABLE IF EXISTS {inspire_ai_work_table(table, suffix, part)}")
    report["without"] = sum(report["kinds"].values())
    status = inspire_rows(spark.sql(inspire_ai_status_sql(table, use_case_id, ai, tier)))[0]
    explained, records = int(status.get("explained") or 0), int(status.get("records") or 0)
    out(f"AI reasoning: {report['explained']} record(s) explained by {ai['endpoint']} in "
        f"{round(_inspire_time.time() - started)} seconds"
        + (f"; {report['without']} left without reasoning after one retry ("
           + ", ".join(f"{n} {kind}" for kind, n in sorted(report["kinds"].items())) + ")" if report["without"] else "")
        + f"; {explained} of the {records} current 'Act now' and 'Review' record(s) with facts now hold AI reasoning")
    if records >= 10 and int(status.get("levels") or 0) == 1:
        out(f"AI reasoning: warning: all {records} 'Act now' and 'Review' records have the {INSPIRE_CONFIDENCE} "
            f"{status.get('level')!r}, so the rule that sets it reads too little evidence to tell them apart, such as the distance "
            "from the threshold or baseline, the history length or the data completeness")
    return report


def inspire_ai_summary_table(table):
    return ".".join(inspire_table_parts(table)) + INSPIRE_AI_SUMMARY_SUFFIX


def inspire_ai_summary_merge_sql(table, use_case_id, run_id, values):
    names = list(values)
    return (f"MERGE INTO {inspire_table_name(inspire_ai_summary_table(table))} AS s USING (SELECT "
            f"{inspire_sql_literal(use_case_id)} AS use_case_id, {inspire_sql_literal(run_id)} AS run_id) AS n ON s.use_case_id = n.use_case_id AND s.run_id = n.run_id "
            f"WHEN MATCHED THEN UPDATE SET {', '.join(f'{inspire_column(k)} = {values[k]}' for k in names)}, "
            f"updated_at = current_timestamp() WHEN NOT MATCHED THEN INSERT (use_case_id, run_id, "
            f"{', '.join(inspire_column(k) for k in names)}, updated_at) VALUES (n.use_case_id, n.run_id, "
            f"{', '.join(values[k] for k in names)}, current_timestamp())")


def inspire_ai_summary_create_sql(table):
    outcome = ", ".join(f"{name} {kind}" for name, kind in INSPIRE_OUTCOME_COLUMNS)
    return (f"CREATE TABLE IF NOT EXISTS {inspire_table_name(inspire_ai_summary_table(table))} (use_case_id STRING, run_id STRING, "
            f"as_of_date DATE, summary STRING, ai_endpoint STRING, ai_config STRING, {outcome}, updated_at TIMESTAMP)")


def inspire_ensure_summary(spark, table):
    spark.sql(inspire_ai_summary_create_sql(table))
    have = [name for name, _kind in inspire_table_schema(spark, inspire_ai_summary_table(table)) or []]
    missing = inspire_history_missing(have, INSPIRE_OUTCOME_COLUMNS)
    if missing:
        spark.sql(inspire_history_add_sql(inspire_ai_summary_table(table), missing))


def inspire_outcome_sql(table, use_case_id, columns, value):
    named = {str(c).lower(): inspire_column(c, "t") for c in columns}
    tier = named["priority_tier"]
    picks = [f"COUNT_IF({tier} = 'Act now') AS act_now_records", f"COUNT_IF({tier} = 'Review') AS review_records",
             "COUNT(*) AS records"]
    if "data_quality_flag" in named:
        flag = named["data_quality_flag"]
        picks += [f"COUNT_IF(coalesce({flag}, '') <> 'ok') AS flagged_records", f"mode(nullif({flag}, 'ok')) AS top_flag"]
    if value and INSPIRE_VALUE_AT_STAKE in named:
        stake = named[INSPIRE_VALUE_AT_STAKE]
        picks += [f"SUM(CASE WHEN {tier} IN ({inspire_sql_list(INSPIRE_FLAGGED_TIERS)}) THEN {stake} END) AS value_at_stake",
                  f"MAX({stake}) AS top_stake"]
    return (f"SELECT {', '.join(picks)} FROM {inspire_table_name(table)} AS t WHERE t.use_case_id = "
            f"{inspire_sql_literal(use_case_id)} AND t.is_current AND t.change_source <> 'delete'")


def inspire_nothing_to_act(found, value, given):
    if int(found.get("act_now_records") or 0) + int(found.get("review_records") or 0):
        return None
    if str(given or "").strip():
        return str(given).strip()
    records, flagged = int(found.get("records") or 0), int(found.get("flagged_records") or 0)
    if not records:
        return "The last run wrote no current records."
    if flagged >= records:
        top = found.get("top_flag")
        return "Every record has a data quality issue" + (f", most often: {top}." if top else ".")
    review = inspire_value_amounts(value)[1]
    if value and value["basis"] == "measured" and review is not None:
        top = found.get("top_stake")
        return (f"No record has {inspire_money_text(review, value['currency'])} or more at stake, the amount this use case "
                "reviews from" + (f"; the most at stake on one record is {inspire_money_text(top, value['currency'])}."
                                  if top is not None else "."))
    return "No record passed this use case's decision test in the last run."


def inspire_sql_number(number):
    return "NULL" if number is None else repr(float(number))


def inspire_save_outcome(spark, table, use_case_id, run_id, columns, value, given, out=print):
    if "priority_tier" not in {str(c).lower() for c in columns}:
        return None
    try:
        found = (inspire_rows(spark.sql(inspire_outcome_sql(table, use_case_id, columns, value))) or [{}])[0]
        reason = inspire_nothing_to_act(found, value, given)
        values = {name: str(int(found.get(name) or 0)) for name in ("act_now_records", "review_records", "records")}
        values["nothing_to_act"] = "NULL" if reason is None else inspire_sql_literal(reason)
        if value:
            act_now, review = inspire_value_amounts(value)
            values.update({INSPIRE_VALUE_AT_STAKE: inspire_sql_number(found.get(INSPIRE_VALUE_AT_STAKE)),
                           "currency": inspire_sql_literal(value["currency"]), "decision_basis": inspire_sql_literal(value["basis"]),
                           "act_now_amount": inspire_sql_number(act_now), "review_amount": inspire_sql_number(review)})
        inspire_ensure_summary(spark, table)
        spark.sql(inspire_ai_summary_merge_sql(table, use_case_id, run_id, values))
    except Exception as error:
        out(f"Business output: run {run_id} wrote its rows, but Inspire could not record how many of them need action in "
            f"{inspire_ai_summary_table(table)}: {str(error)[:300]}")
        return None
    counts = f"{values['act_now_records']} Act now and {values['review_records']} Review of {values['records']} current record(s)"
    if reason is None:
        ignored = " (the nothing_to_act reason the run gave was left out, since records need action)" if str(given or "").strip() else ""
        out(f"Business output: {counts}{ignored}")
    else:
        source = "the run's own reason" if str(given or "").strip() else "Inspire's reason, since the run gave no nothing_to_act"
        out(f"Business output: {counts}, so the App shows nothing to act on yet, with {source}: {reason}")
    return reason


def inspire_ai_save_config(spark, table, use_case_id, run_id, ai):
    config = dict(ai, hedge=INSPIRE_AI_HEDGE, rows_per_second=INSPIRE_AI_ROWS_PER_SECOND, unexplained=INSPIRE_AI_UNEXPLAINED)
    inspire_ensure_summary(spark, table)
    spark.sql(inspire_ai_summary_merge_sql(table, use_case_id, run_id, {
        "ai_endpoint": inspire_sql_literal(ai["endpoint"]),
        "ai_config": inspire_sql_literal(_inspire_json.dumps(config, ensure_ascii=False, sort_keys=True))}))


def inspire_ai_after_merge(spark, table, use_case_id, run_id, ai, cap, out=print):
    if not ai:
        return None
    try:
        inspire_ai_save_config(spark, table, use_case_id, run_id, ai)
    except Exception as error:
        out(f"AI reasoning: Inspire could not record the AI settings of run {run_id} in {inspire_ai_summary_table(table)}: "
            f"{str(error)[:300]}")
    if not ai["endpoint"]:
        out("AI reasoning: the ai_endpoint widget is empty, so no record was explained")
        return None
    try:
        return inspire_ai_explain(spark, table, use_case_id, ai, cap, out=out)
    except Exception as error:
        out(f"AI reasoning: run {run_id} wrote its rows, but Inspire could not explain them with {ai['endpoint']}: "
            f"{str(error)[:300]}")
        return None


def inspire_ai_summary_prompt(context):
    return "\n".join([str(context or "").strip(), "",
                      "If anything below conflicts with an explicit directive from the user above, the user's directive wins.", "",
                      "Write the summary of this run of the use case for the people who act on its results: 3 to 5 plain "
                      "sentences on how ready the run is, how many records need action now and why, what changed against "
                      "earlier runs, and what to do first. State the facts as facts. Reply with the summary text only, with no "
                      "heading, list or JSON. Never write about yourself, about the facts or this message, or about what is not "
                      'known, and never use the word "I".', "", "Facts:", ""]).lstrip()


def inspire_write_summary(facts, result_table, run_id, as_of_date, spark_session=None, endpoint=None, context=None, out=print):
    session = spark_session if spark_session is not None else globals()["spark"]
    table = ".".join(inspire_table_parts(result_table))
    text = facts if isinstance(facts, str) else _inspire_json.dumps(facts, default=str, ensure_ascii=False, sort_keys=True)
    if not str(text or "").strip():
        raise ValueError("inspire_write_summary needs the facts of the run's summary")
    endpoint = str(inspire_widget("ai_endpoint") if endpoint is None else endpoint).strip()
    if not endpoint:
        out("AI summary: the ai_endpoint widget is empty, so the run has no AI summary")
        return None
    run = globals().get("INSPIRE_RUN") or {}
    prompt = inspire_ai_summary_prompt(run.get("ai_context", "") if context is None else context) + str(text)
    summary, problem = None, ""
    for _attempt in (1, 2):
        asked = prompt + (f"\n\nYour last reply was rejected ({problem}). Write it again following every rule above." if problem else "")
        reply = (inspire_rows(session.sql(f"SELECT ai_query({inspire_sql_literal(endpoint)}, {inspire_sql_literal(asked)}, "
                                          "failOnError => false) AS r")) or [{}])[0].get("r") or {}
        result, error = str(reply["result"] or "").strip() if reply else "", reply["errorMessage"] if reply else "no reply"
        hedged = _inspire_re.search(INSPIRE_AI_HEDGE, result)
        problem = (f"endpoint error: {error}" if error else "empty reply" if not result else
                   f"it wrote about yourself or the facts ('{hedged.group(0)}')" if hedged else "")
        if not problem:
            summary = result
            break
    if summary is None:
        out(f"AI summary: run {run_id} has no AI summary, since {endpoint} gave none Inspire could use after one retry: {problem[:300]}")
        return None
    inspire_ensure_summary(session, table)
    session.sql(inspire_ai_summary_merge_sql(table, str(run.get("use_case_id") or ""), str(run_id), {
        "summary": inspire_sql_literal(summary), "as_of_date": f"try_cast({inspire_sql_literal(as_of_date)} AS DATE)",
        "ai_endpoint": inspire_sql_literal(endpoint)}))
    out(f"AI summary: wrote the summary of run {run_id} to {inspire_ai_summary_table(table)}")
    return summary


def inspire_sql_retrying(spark, sql, table, doing, again, out=print):
    for attempt in (1, 2, 3):
        try:
            return inspire_rows(spark.sql(sql))
        except Exception as error:
            if attempt == 3 or "concurrent" not in str(error).lower():
                raise
            out(f"Business output: {table} changed while {doing}, such as an edit in the App, so {again} runs again "
                f"(attempt {attempt + 1} of 3)")
            _inspire_time.sleep(5 * attempt)


def inspire_merge_run(spark, table, stage, out=print, ai=None, ai_cap=INSPIRE_AI_AUTO_ROWS, value=None, nothing_to_act=None):
    owned = set(INSPIRE_HISTORY_NAMES) | set(INSPIRE_ACTION_NAMES)
    staged = inspire_table_schema(spark, stage) or []
    reasoning = {str(name).lower() for name, _kind in staged if inspire_ai_owned(name)} if ai else set()
    schema = [(name, kind) for name, kind in staged if str(name).lower() not in owned | reasoning]
    ignored = [name for name, _kind in staged if str(name).lower() in INSPIRE_ACTION_NAMES]
    if reasoning:
        out(f"AI reasoning: left out {', '.join(sorted(reasoning))} from the run's rows: Inspire writes them from each row's "
            f"{INSPIRE_AI_FACTS}")
    columns = [name for name, _kind in schema]
    wanted = INSPIRE_RUN_KEY_COLUMNS + ((INSPIRE_CONFIDENCE,) if ai else ()) + ((INSPIRE_VALUE_AT_STAKE,) if value else ())
    absent = [name for name in wanted if name not in {str(c).lower() for c in columns}]
    if absent:
        raise ValueError(f"Inspire did not write the business output: the rows have no {', '.join(absent)} column, and every row needs it")
    profile = inspire_rows(spark.sql(inspire_run_profile_sql(stage, columns, value)))[0]
    rows = int(profile.get("row_count") or 0)
    if not rows:
        out(f"Business output: the run produced no rows, so {table} was left as it was")
        return 0
    problems = (inspire_value_problems(value) if value else []) + inspire_run_problems(profile, value)
    id_unique, label_unique = bool(profile.get("id_unique")), bool(profile.get("label_unique"))
    key_names = inspire_record_key_rule(columns, id_unique, label_unique)
    if not problems and not ((key_names == ("entity_id",) and id_unique) or (key_names == ("entity_label",) and label_unique)):
        repeats = inspire_rows(spark.sql(inspire_key_repeats_sql(stage, key_names)))
        if repeats:
            shown = ", ".join(f"{r.get('record_key')!r} ({r.get('row_count')} rows)" for r in repeats)
            problems.append(f"rows share the record key {' with '.join(key_names)} or leave it empty ({shown}); every row needs its own "
                            "entity_id, or its own entity_id with segment_or_item")
    if problems:
        raise ValueError("Inspire did not write the business output: " + "; ".join(problems))
    use_case_id, run_id, keyed = str(profile["use_case_id"]), str(profile["run_id"]), " with ".join(key_names)
    if ignored:
        out(f"Business output: left out {', '.join(ignored)} from the run's rows: people set them in the App, and each run carries "
            "them forward from the record's current version")
    existing = inspire_table_schema(spark, table)
    if existing is None:
        spark.sql(inspire_history_create_sql(table, stage, columns, key_names, ai))
        out(f"Business output: created {table} with the {rows} row(s) of run {run_id} as the current version of each record, "
            f"keyed by {keyed}")
        inspire_save_outcome(spark, table, use_case_id, run_id, columns, value, nothing_to_act, out)
        inspire_ai_after_merge(spark, table, use_case_id, run_id, ai, ai_cap, out)
        return rows
    spark.sql(inspire_history_backup_sql(table))
    migrated = inspire_ensure_history(spark, table, existing, backup=False, out=out)
    if migrated["backfilled"]:
        existing = inspire_table_schema(spark, table)
    replace_edits = inspire_replace_edits(spark, table)
    retype, changed = inspire_type_change_sql(table, existing, schema, replace_edits)
    if retype:
        spark.sql(retype)
        out(f"Business output: converted {', '.join(changed)} in {table} to the types this run writes")
    added_sql, added = inspire_add_columns_sql(table, existing, schema)
    if added_sql:
        spark.sql(added_sql)
        out(f"Business output: added {', '.join(added)} to {table}; the rows already there hold NULL in them")
    inspire_ensure_actions(spark, table, existing, out=out)
    if ai:
        inspire_ensure_ai(spark, table, existing, ai, out=out)
    unwritten = [name for name, _kind in existing if str(name).lower() not in {str(c).lower() for c in columns}
                 and str(name).lower() not in owned and not inspire_ai_owned(name)]
    if unwritten:
        out(f"Business output: kept {', '.join(unwritten)} in {table}, which this run does not write, so its rows hold NULL there")
    kept = 0 if replace_edits else int(inspire_rows(spark.sql(inspire_kept_edits_sql(table, use_case_id)))[0].get("kept") or 0)
    merge = inspire_history_merge_sql(table, stage, columns, key_names, use_case_id, replace_edits, ai)
    merged = inspire_sql_retrying(spark, merge, table, f"run {run_id} merged into it", "the merge", out)
    counts = merged[0] if merged else {}
    out(f"Business output: run {run_id} wrote {counts.get('num_inserted_rows', rows)} current version(s) to {table} and closed "
        f"{counts.get('num_updated_rows', 0)} earlier one(s), keyed by {keyed}; "
        + ("runs replace the records people changed in the App, as the use case's settings ask" if replace_edits else
           f"kept {kept} record(s) people changed in the App, since runs keep those changes")
        + f"; {inspire_action_summary(spark, table, stage, columns, key_names)}"
        + f"; the table as it was before this run is in {table}_backup")
    inspire_save_outcome(spark, table, use_case_id, run_id, columns, value, nothing_to_act, out)
    inspire_ai_after_merge(spark, table, use_case_id, run_id, ai, ai_cap, out)
    return rows


def inspire_action_summary(spark, table, stage, columns, key_names):
    try:
        found = (inspire_rows(spark.sql(inspire_action_counts_sql(table, stage, columns, key_names))) or [{}])[0]
    except Exception as error:
        return f"the actions people took in the App were carried forward, but Inspire could not count them: {str(error)[:300]}"
    dismissed, kept_out, resolved, flagged = (int(found.get(name) or 0) for name in ("dismissed", "kept_out", "resolved", "still_flagged"))
    return (f"kept {dismissed} dismissed record(s) in Monitor, {kept_out} of them out of Act now, and wrote {resolved + flagged} "
            f"outcome(s) for records people marked Done ({resolved} Resolved, {flagged} Still flagged)")


def inspire_write_result(frame, result_table, spark_session=None, out=print, nothing_to_act=None):
    session = spark_session if spark_session is not None else globals()["spark"]
    if not hasattr(frame, "write"):
        frame = session.createDataFrame(frame)
    for field in frame.schema.fields:
        if field.dataType.typeName() in ("void", "null"):
            frame = frame.withColumn(field.name, frame[field.name].cast("string"))
    catalog, schema, name = inspire_table_parts(result_table)
    table, stage = f"{catalog}.{schema}.{name}", f"{catalog}.{schema}.{name}_inspire_stage"
    names = {f.name.lower() for f in frame.schema.fields}
    ai = inspire_ai_run_config() if INSPIRE_AI_FACTS in names else None
    value = inspire_value_run_config(names)
    preview = inspire_is_preview(inspire_widget("run_profile", "full"))
    frame.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(inspire_table_name(stage))
    try:
        return inspire_merge_run(session, table, stage, out=out, ai=ai,
                                 ai_cap=INSPIRE_AI_PREVIEW_ROWS if preview else INSPIRE_AI_AUTO_ROWS, value=value,
                                 nothing_to_act=nothing_to_act)
    finally:
        session.sql(f"DROP TABLE IF EXISTS {inspire_table_name(stage)}")
'''
exec(compile(IMPLEMENT_RESULT_HISTORY_SOURCE, "<inspire_result_history>", "exec"), globals())
GENIE_EXPLANATION_FACTS_RULE = (f"Build `{INSPIRE_AI_FACTS}` on every row in pandas as one plain-text column of short 'name: value' "
                                "lines, from the facts the code computed and formatted: its entity_label, its value with the unit, its "
                                "value_at_stake with the currency, its top_reasons, the main driver or cause when the method computes one, and the evidence behind its "
                                f"{INSPIRE_CONFIDENCE}, such as how far it sits past the decision threshold or its baseline, how much "
                                "history it has and how complete its data is. The facts are written without markdown and never choose a "
                                f"driver, a cause, a rank or a number from raw values. Set `{INSPIRE_CONFIDENCE}` on every row to "
                                f"{', '.join(repr(level) for level in INSPIRE_CONFIDENCE_LEVELS)} with one rule in code over that "
                                "evidence, its cut-offs named constants, and print how many 'Act now' and 'Review' rows each level "
                                "holds; when they all get one level, the rule reads too little evidence to tell them apart, so fix it "
                                "before the write. Define `AI_CATEGORIES` at top level, after the Parameters cell: a dict of 1 to 3 ways "
                                "to sort this use case's flagged records for the people who act on them, each keyed "
                                "`ai_cat_<lowercase name>` and holding a `meaning` and 2 to 8 short `values`, such as "
                                "{'ai_cat_driver': {'meaning': 'what moved the record most', 'values': ['Price', 'Volume', 'Mix']}}; "
                                "the App shows each as a filter. inspire_write_result explains the 'Act now' and 'Review' rows with "
                                f"ai_query on the endpoint of the `ai_endpoint` widget, from {INSPIRE_AI_FACTS} and "
                                f"{INSPIRE_CONFIDENCE} alone, best ranked first, at most {INSPIRE_AI_AUTO_ROWS} per run and "
                                f"{INSPIRE_AI_PREVIEW_ROWS} per preview run, and writes {', '.join(INSPIRE_AI_NAMES)} and the "
                                "AI_CATEGORIES columns, so never call ai_query to explain rows and never write those columns. When a step "
                                "of the method itself calls ai_query, pass the named argument `failOnError => false`, never "
                                "`named_struct('failOnError', false)`, which ai_query does not read as that option, so it returns plain "
                                "text and fails on the first error. With failOnError => false, ai_query returns a struct: store its "
                                "`result` text in SQL (NULL when `errorMessage` is set), never the struct or its string form. Call it "
                                "without responseFormat, which some endpoints reject. Build the summary step's facts the same way, from "
                                "facts the code computed "
                                "and formatted with fmt_num, fmt_count and fmt_rank, never a raw number, dict or list: each number in a "
                                "fact covers the rows that fact names, so a group's count of passing rows goes with the amount over those "
                                "rows, and a total over all of the group's rows says so; a group whose key is missing is named for what "
                                "it is, such as 'no carrier', never by a missing value's text; a fact whose value is missing, such as a "
                                "correlation over a column that does not vary, is left out of the facts or names why it is missing ('no "
                                "correlation, since every sender has one recipient'), never fmt_num's 'unknown'; the step raises an "
                                "error when the facts hold the whole word 'unknown' or 'inf', or " + GENIE_MISSING_VALUE_TEXTS
                                + ", and otherwise passes them to `inspire_write_summary(facts, result_table, run_id, as_of_date)`, "
                                "which writes the run's AI summary for the App.")
GENIE_RESULT_HISTORY_RULE = ("Write the business output only with `inspire_write_result(frame, result_table)`, which the Run controls "
                             "cell defines: pass the Spark DataFrame of the run's rows, cast to the business output types, and the "
                             "value of the `result_table` widget, and return the row count it returns as `rows_written`. When no "
                             "row of the run is 'Act now' or 'Review', also pass `nothing_to_act`: one plain sentence for the people "
                             "who act on the results that says why, with the measured numbers behind it, and what would change it, "
                             "such as one that names the best candidate's out-of-time AUC, the AUC this use case needs and the "
                             "closed outcome windows still missing; the App shows it in place of an empty list, and without it the "
                             "App can only give a general reason. It keeps "
                             "the table's change history: each row becomes the current version of its record, keyed by entity_id "
                             "(entity_label when entity_id repeats, entity_id with segment_or_item when both repeat), the versions it "
                             "replaces are closed with their valid_to, records people changed in the App stay as they are unless the "
                             "use case lets runs replace them, and a DEEP CLONE backup of the table comes first. So never create, "
                             "write, update or delete the business output table yourself, and never take its newest as_of_date for "
                             f"the latest run: the current rows are `WHERE {INSPIRE_CURRENT_ROWS}`. The columns "
                             f"{', '.join(INSPIRE_ACTION_NAMES)} hold what people do about each record in the App, and "
                             "inspire_write_result carries them to each new version, so the run's rows never set them; a record people "
                             f"dismissed stays Monitor, so never put it back in Act now. The columns {', '.join(INSPIRE_AI_NAMES)} "
                             "and the AI_CATEGORIES columns hold Inspire's AI reasoning, which inspire_write_result writes and carries "
                             f"to a new version whose {INSPIRE_AI_FACTS} and {INSPIRE_CONFIDENCE} did not change, so the run's rows "
                             "never set them either.")
GENIE_CELL_BOUNDARY_RULE = ("When you write the notebook as a source file, start every cell, yours and Inspire's, after its own "
                            "`# COMMAND ----------` line, and keep that line directly above the title line of the Run controls cell "
                            "(`Run controls (Inspire, do not edit)`), so your code and the Run controls never share a cell; after each "
                            "import, check that the notebook's last cell is the Run controls cell, starting with that title and ending "
                            "with `main()`.")
GENIE_COMPLETE_BUILD_RULE = ("Write every step the instruction lists in full before the first run, the business output write and the "
                             "summary step among them, also when the run will stop by the STOP RULE, so a run with other data "
                             "or widget values runs every step: never write a stub, a step whose body only raises, passes or returns a "
                             "placeholder, or a branch of a step that prints its work as done in place of doing it, such as a "
                             "registration step whose accepted branch prints a message and returns an empty version: every branch does "
                             "the work the instruction gives it, and a step prints only what it did. When an edit does not apply, read "
                             "the notebook again and apply the same complete build, never "
                             "a shorter one. A failed run is a defect to fix, never the result to report: fix the code and run again "
                             "until a run returns its outcome, written or stopped by the STOP RULE.")
GENIE_RUN_CONTROLS_RULE = ("Inspire's Run controls cell is the last cell of the notebook and ends with `main()`: never edit, move or remove "
                           "it, and add every cell between the instruction cell and the Run controls cell. " + GENIE_CELL_BOUNDARY_RULE
                           + " " + GENIE_COMPLETE_BUILD_RULE + " Put each build step in a "
                           "function and define `run_use_case()`, with no arguments, which runs the steps in order and passes each step, "
                           "as arguments, every value it reads that an earlier step computed, since a name local to `run_use_case()` or "
                           "to another step does not exist inside a step, and prints each step's name and elapsed seconds as the step "
                           "returns, so the printed log shows which step is slow; only imports, function "
                           "definitions and the Parameters cell's widgets and typed reads run at top level, besides the %pip cell, the "
                           "first cell below the instruction cell, so "
                           "running the notebook runs the use case once, through `main()`. A value computed from the data, such as "
                           "`as_of_date` = `latest` or a default the Parameters section says how to compute, is resolved by the first step "
                           "`run_use_case()` runs, never in the Parameters cell: every top-level statement runs once in the session and "
                           "again in the job run. `main()` runs `run_use_case()` in this session "
                           "when the Execution widget is Here, and as the notebook's tagged Databricks job when it is As job, where the "
                           "job run calls `run_use_case()`. While you build and test, set its Execution widget to Here and run the whole "
                           "notebook, and give a test run you submit as a job `{\"execution\": \"Here\"}` in its notebook task's "
                           "base_parameters, the widget's name, never its label `Execution`, which the run ignores and so launches the "
                           "tagged job; set it back to As job when you finish, so every run of the notebook is a tagged Databricks job. "
                           "`run_use_case()` returns the run's outcome as a dict: {'status': 'written', 'rows_written': <n>, "
                           "'as_of_date': <date>, 'run_id': <id>} after the write, or {'status': 'stopped', 'stop_reason': <the "
                           "message the STOP RULE printed>} when a check stops the run. `main()` prints it and exits with it, followed "
                           "by everything `run_use_case()` printed, because a cell that calls `dbutils.notebook.exit` loses its "
                           "printed output in a job run, and the tagged job run returns that text to the notebook that launched it; "
                           "so never call `dbutils.notebook.exit` in your own cells, and return the outcome instead. Check each test "
                           "run by that outcome, never by counting the output table's rows, which also holds earlier runs. Run one test "
                           "at a time: submit each test run in one command that first lists the active runs with their tasks "
                           "(`databricks jobs list-runs --active-only --expand-tasks`, since without `--expand-tasks` a run lists no "
                           "tasks, so no path matches and nothing is canceled) and cancels every run whose task runs this notebook's "
                           "path, with or without `/Workspace` before it, which covers every earlier test run and the notebook's "
                           "tagged job, which the Run controls name as they launch it ('Running this notebook as the tagged job "
                           "dbx_inspire_ai_...') and which keeps running and writing after the run that launched it ends or is "
                           "canceled, and never submit from two commands at once, since a second submit, such as one repeated after "
                           "a long output, starts a run that writes the output table while the first still does; submit each test "
                           "run with retries off (the task's `max_retries` 0 and "
                           "`disable_auto_optimization` true, since serverless otherwise retries a failed task on its own), because a "
                           "retry that starts after a newer run writes its older rows over that run's current rows. After the last "
                           "run, check that the current rows of this use case that runs wrote (`change_source = 'run' AND "
                           f"{INSPIRE_CURRENT_ROWS}`) carry the run_id its outcome returned. Each run closes the versions the run "
                           "before it wrote, so the rows of an earlier test run stay in the table only as closed change history, "
                           "which no output reads: never delete them. " + GENIE_RESULT_HISTORY_RULE)
GENIE_RUN_CONTROLS_TEMPLATE = '''# Run controls (Inspire, do not edit; keep this the last cell). main() runs run_use_case(), which Genie Code builds in
# the cells above: Execution "As job" runs this notebook as a tagged Databricks job and waits for it; "Here" runs it in this session, untagged.
@@JOB_LAUNCHER@@
@@RESULT_HISTORY@@

import contextlib
import io
import sys

INSPIRE_RUN = @@RUN@@


def inspire_run_controls(run, dbu=None, launcher_cls=None, out=print):
    dbu = dbu if dbu is not None else dbutils
    launcher_type = launcher_cls if launcher_cls is not None else JobLauncher
    dbu.widgets.dropdown("execution", "As job", ["As job", "Here"], "Execution")
    dbu.widgets.text(run["marker"], "0", "Inspire job run (set by the job)")
    if str(dbu.widgets.get(run["marker"])).strip() == "1":
        out("Running as the tagged job of this notebook: the use case runs here.")
        return {"execution": "child"}
    if dbu.widgets.get("execution") != "As job":
        out("Execution = Here: the notebook runs in this session, and this run is not tagged.")
        return {"execution": "here"}
    notebook_path = launcher_type.get_current_notebook_path()
    if not notebook_path:
        out("Could not determine this notebook's path, so no job was created. Running here instead; this run is not tagged.")
        return {"execution": "here", "fallback": "no notebook path"}
    tags = dict(run["job_tags"])
    tags["dbx_inspire_ai_notebook"] = launcher_type._sanitize_tag(notebook_path.rsplit("/", 1)[-1])
    parameters = {}
    for name in list(run["widget_names"]) + ([] if "run_profile" in run["widget_names"] else ["run_profile"]):
        try:
            parameters[name] = str(dbu.widgets.get(name)).strip()
        except Exception:
            pass
    parameters[run["marker"]] = "1"
    scheduled = dict(parameters, **run["scheduled_overrides"])
    if "run_profile" in scheduled:
        scheduled["run_profile"] = "full"
    job_name = launcher_type.persistent_job_name(run["job_slug"], notebook_path)
    launcher = launcher_type(notebook_path, scheduled, tags)
    out(f"Running this notebook as the tagged job {job_name}.")
    preview = inspire_is_preview(parameters.get("run_profile", "full"))
    result = launcher.launch(job_name=job_name, run_name=job_name, run_parameters=parameters, max_concurrent_runs=1,
                             queue=True, schedule_cron=run["schedule_cron"],
                             **({"only_tasks": [launcher_type.TASK_KEY]} if preview else {}))
    if not result.get("success"):
        out(f"The job could not be created or started ({result.get('error')}). Running here instead; this run is not tagged.")
        return {"execution": "here", "fallback": "launch failed", "error": result.get("error")}
    outcome = launcher.wait_for_run(result["run_id"], timeout_s=int(run["wait_minutes"]) * 60, out=out)
    link = result.get("job_url") or ""
    if outcome["status"] == "SUCCESS":
        reported = outcome.get("output") or "the job run returned no outcome"
        headline, newline, printed = str(reported).partition("\\n")
        out(f"The tagged job run finished: {headline}. Results table: {run['result_table']}. Run: {link}")
        dbu.notebook.exit(f"Job run finished: {headline}. Results table: {run['result_table']}. Run: {link}{newline}{printed}")
        return {"execution": "job", "status": "SUCCESS", "outcome": headline, "result": result}
    if outcome["status"] == "RUNNING":
        out(f"The tagged job run is still running after {run['wait_minutes']} minute(s); follow it here: {link}")
        dbu.notebook.exit(f"Job run still running after {run['wait_minutes']} minute(s); follow it here: {link}")
        return {"execution": "job", "status": "RUNNING", "result": result}
    raise RuntimeError(f"The tagged job run did not succeed ({outcome['status']}): {outcome.get('error') or 'no error message'}. Run: {link}")


def inspire_outcome_text(outcome):
    if outcome is None:
        return "run_use_case() returned no outcome, so this run's printed summary is its only record"
    if isinstance(outcome, dict) and outcome.get("status") == "written":
        return (f"wrote {outcome.get('rows_written')} row(s) for as_of_date {outcome.get('as_of_date')} "
                f"with run_id {outcome.get('run_id')}")
    if isinstance(outcome, dict) and outcome.get("status") == "stopped":
        return f"stopped without writing: {outcome.get('stop_reason') or 'no reason given'}"
    return str(outcome)


class InspireTee:
    def __init__(self, stream, copy):
        self.stream, self.copy = stream, copy

    def write(self, text):
        self.copy.write(text)
        return self.stream.write(text)

    def flush(self):
        self.stream.flush()

    def __getattr__(self, name):
        return getattr(self.stream, name)


def inspire_exit_text(headline, printed, limit=100000):
    printed = (printed or "").strip()
    if not printed:
        return headline
    if len(printed) > limit:
        head = limit // 5
        printed = (f"{printed[:head]}\\n... ({len(printed) - limit} characters of the printed log left out) ...\\n"
                   f"{printed[-(limit - head):]}")
    return f"{headline}\\n\\nPrinted log of this run:\\n{printed}"


def main(run=INSPIRE_RUN, dbu=None, launcher_cls=None, out=print):
    use_case = globals().get("run_use_case")
    if not callable(use_case):
        out("Genie Code has not built this notebook yet: copy the instruction cell into Genie Code first. Nothing ran.")
        return {"execution": "not built"}
    dbu = dbu if dbu is not None else dbutils
    gate = inspire_run_controls(run, dbu=dbu, launcher_cls=launcher_cls, out=out)
    if gate["execution"] in ("child", "here"):
        printed = io.StringIO()
        with contextlib.redirect_stdout(InspireTee(sys.stdout, printed)):
            outcome = use_case()
        gate["outcome"] = inspire_outcome_text(outcome)
        out(f"Use case outcome: {gate['outcome']}.")
        dbu.notebook.exit(inspire_exit_text(gate["outcome"], printed.getvalue()))
    return gate


main()
'''
GENIE_INSTRUCTION_MAX_CHARS = 20000
GENIE_ENV_PACKAGE_VERSIONS = {"scikit-learn": "1.6.1", "statsmodels": "0.14.4", "scipy": "1.15.1"}
GENIE_FAMILY_PACKAGES = {
    "A": ("scikit-learn",), "B": ("scikit-learn",), "C": ("scikit-learn",), "D": ("scikit-learn", "scipy"),
    "E": ("statsmodels",), "F": ("statsmodels", "scipy"), "G": (), "H": ("scipy",),
}
GENIE_PINNED_API_RULE = ("The %pip line pins {pins}: call each library as that version defines it, never with an argument, a "
                         "function or a default that a later release added, such as `KBinsDiscretizer(quantile_method=...)`, which "
                         "scikit-learn added in 1.7, so an earlier release stops it with \"__init__() got an unexpected keyword "
                         "argument 'quantile_method'\".")

INSPIRE_MODEL_FAMILIES = ("A", "B", "C", "D")
GENIE_FAMILY_DECISION_BASIS = {"A": "predicted", "B": "measured", "C": "measured", "D": "predicted", "E": "predicted",
                               "F": "measured", "G": "measured", "H": "predicted"}
INSPIRE_PRIORITY_TIERS = ("Act now", "Review", "Monitor")
INSPIRE_TIER_SHARES = {"act_now": 0.05, "review": 0.20}
INSPIRE_CHILD_MARKER = "dbx_inspire_ai_child_run"
INSPIRE_CORE_COLUMNS = (
    ("as_of_date", "DATE"), ("entity_id", "STRING"), ("entity_label", "STRING"), ("value", "DOUBLE"),
    ("value_type", "STRING"), (INSPIRE_VALUE_AT_STAKE, "DOUBLE"), ("segment_or_item", "STRING"), ("rank", "INT"), ("priority_tier", "STRING"),
    ("top_reasons", "ARRAY<STRING>"), ("recommended_action", "STRING"), (INSPIRE_AI_FACTS, "STRING"),
    (INSPIRE_CONFIDENCE, "STRING"),
    ("data_quality_flag", "STRING"), ("method", "STRING"), ("model_version", "STRING"), ("run_id", "STRING"),
    ("use_case_id", "STRING"),
)
INSPIRE_FLAG_ONLY_READINESS = {"staleness_cadences": "data ends <newest date>"}
INSPIRE_READINESS_RULES = {
    "common": (
        ("rows", "row_count", ">=", "min_rows",
         "No usable rows were found for this use case: {observed} row(s), at least {required} needed."),
        ("join coverage", "join_coverage", ">=", "min_join_coverage",
         "Only {observed:.0%} of rows match along the verified join path; at least {required:.0%} must match."),
    ),
    "A": (
        ("outcome", "outcome_derivable", "is_true", None,
         "The outcome this use case predicts cannot be derived from the available data."),
        ("positives", "positives", ">=", "min_positives",
         "History holds {observed} past example(s) of the outcome; at least {required} are needed to learn from."),
        ("class rate (low)", "class_rate", ">=", "min_class_rate",
         "The outcome occurs in {observed:.2%} of cases; below {required:.2%} it cannot be separated reliably."),
        ("class rate (high)", "class_rate", "<=", "max_class_rate",
         "The outcome occurs in {observed:.1%} of cases; above {required:.0%} there is nothing left to predict."),
        ("outcome windows", "outcome_windows", ">=", "min_outcome_windows",
         "History covers {observed} closed outcome window(s); at least {required} are needed to test on unseen time."),
    ),
    "B": (
        ("reference rows", "reference_rows", ">=", "min_reference_rows",
         "The reference period has {observed} row(s); at least {required} are needed to learn what normal looks like."),
        ("key signal gaps", "key_signal_null_rate", "<=", "max_key_signal_null_rate",
         "{observed:.0%} of the key signal values are missing; at most {required:.0%} is allowed."),
    ),
    "C": (
        ("entities per segment", "entities", ">=", "min_entities_per_k*k",
         "There are {observed} entities, too few for the requested segments; at least {required} are needed."),
        ("numeric features", "numeric_features", ">=", "min_numeric_features",
         "Only {observed} usable numeric feature(s) exist; at least {required} are needed to form segments."),
    ),
    "D": (
        ("interactions", "interactions", ">=", "min_interactions",
         "There are {observed} past interaction(s); at least {required} are needed to learn preferences."),
        ("users", "users", ">=", "min_users",
         "Only {observed} distinct user(s) interacted; at least {required} are needed."),
        ("items", "items", ">=", "min_items",
         "Only {observed} distinct item(s) were interacted with; at least {required} are needed."),
    ),
    "E": (
        ("seasonal cycles", "seasonal_cycles", ">=", "min_seasonal_cycles",
         "History covers {observed:.1f} seasonal cycle(s); at least {required} are needed to forecast."),
        ("data freshness", "staleness_cadences", "<=", "max_staleness_cadences",
         "The newest data is {observed:.1f} decision cycle(s) older than today; at most {required} is allowed to act on. The "
         "forecast is still written from the newest data with every row flagged; to act on older data, raise the "
         "max_staleness_cadences widget."),
        ("missing periods", "missing_period_rate", "<=", "max_missing_period_rate",
         "{observed:.0%} of the periods have no data; at most {required:.0%} is allowed."),
    ),
    "F": (
        ("reportable groups", "reportable_groups", ">=", "min_reportable_groups",
         "Only {observed} group(s) hold at least the minimum rows per group; at least {required} are needed for a comparison."),
        ("periods", "periods", ">=", "min_periods",
         "History covers {observed} period(s); at least {required} are needed to compare against."),
    ),
    "G": (
        ("text coverage", "text_non_null_rate", ">=", "min_text_non_null_rate",
         "Only {observed:.0%} of the records carry text; at least {required:.0%} must."),
        ("text length", "median_text_length", ">=", "min_median_text_length",
         "The median text is {observed} character(s) long; at least {required} are needed to read meaning."),
    ),
    "H": (
        ("planning inputs", "inputs_derivable", "is_true", None,
         "The costs or capacities this plan needs cannot be derived from the available data."),
    ),
}
INSPIRE_ASSUMPTION_RULES = {
    "KEY_HAS_REPEATED_ROWS": ("values on more than one row", 1,
                              "No {column} value in {table} appears on more than one row, so this use case cannot run as designed: {statement}"),
    "KEY_HAS_SINGLE_ROWS": ("values on exactly one row", 1,
                            "Every {column} value in {table} appears on more than one row, so this use case cannot run as designed: {statement}"),
    "COLUMN_HAS_TWO_VALUES": ("distinct values", 2,
                              "{column} in {table} holds {observed} distinct value(s) and at least 2 are needed, so this use case cannot run as designed: {statement}"),
    "COLUMN_HAS_VALUES": ("non-empty values", 1,
                          "{column} in {table} holds no values, so this use case cannot run as designed: {statement}"),
}


def inspire_temperature_rejected(message):
    text = str(message or "").lower()
    return "does not support the parameter" in text and "temperature" in text


def inspire_output_names(result_table):
    catalog, schema, table = inspire_table_parts(result_table)
    return {"result_table": inspire_table_name(result_table), "model_name": f"{catalog}.{schema}.{table}_model"}

TRAIN_INFER_TECHNIQUES = frozenset(t for t, family in TECHNIQUE_EXECUTION_FAMILY.items() if family in INSPIRE_MODEL_FAMILIES)
TEMPORAL_TECHNIQUES = frozenset({"Forecasting", "Trend Analysis", "Cohort Analysis", "Survival Analysis"})

GENIE_AGENTS_FILE_TEMPLATE = """# Genie Code instructions for Inspire use-case notebooks

The notebooks in this folder were generated by Databricks Inspire AI. Each notebook holds the use-case card, then one Genie Code instruction in its second cell, then the Run controls cell as its last cell. The user copies the instruction into Genie Code, and Genie Code builds and runs the rest of the notebook from it.

- The instruction is complete: follow it for the tables, joins, outcome rule, method, thresholds and the business output table.
- Add your cells between the instruction cell and the Run controls cell, which stays the last cell. Never edit or delete Inspire's three cells. """ + GENIE_CELL_BOUNDARY_RULE + """
- Put each step in a function and define run_use_case(), which runs the steps in order, passes each step as arguments the values earlier steps computed (a name local to run_use_case() or to another step does not exist inside a step), prints each step's elapsed seconds, and returns the run's outcome (written, with its row count, as_of_date and run_id, or stopped, with the reason); only imports, function definitions and the Parameters widgets run at top level, and a value computed from the data, such as as_of_date = latest, is resolved by the first step of run_use_case(). The Run controls cell ends with main(), which runs run_use_case() here, or as the notebook's tagged Databricks job when Execution is As job, and prints that outcome. While you build and test, set Execution to Here and check each run by its outcome; set it back to As job when you finish.
- Run the notebook on serverless compute and fix your own code until it runs from top to bottom. """ + GENIE_COMPLETE_BUILD_RULE + """
- When a readiness check fails, that is a valid outcome: print the measured and required values and stop, unless the instruction says the check only flags rows. When no candidate passes the acceptance rule, do what the instruction's acceptance rule says. Never lower a threshold, fabricate data, drop a check or switch tables to force a pass.
""" + "- " + GENIE_SERVERLESS_RULE + """
- Write the business output only with inspire_write_result(frame, result_table), which the Run controls cell defines and which keeps the table's change history, and, after acceptance, write only the Unity Catalog model the instruction names. Never write to source tables and never create or delete jobs yourself; the Run controls cell manages the notebook's own tagged job.
"""

def _technique_registry_key(technique):
    label = " ".join(str(technique if technique is not None else "").split()).lower()
    for key in TECHNIQUE_EXECUTION_FAMILY:
        if key.lower() == label:
            return key
    return None


def _execution_family(technique, logger=None, context=""):
    key = _technique_registry_key(technique)
    if key is not None:
        return TECHNIQUE_EXECUTION_FAMILY[key]
    label = " ".join(str(technique if technique is not None else "").split()) or "(empty)"
    message = (f"[TECHNIQUE_UNKNOWN] {context or 'Use case'}: Analytics Technique '{label}' is not in the technique registry; "
               "it runs as execution family F (analytical refresh)")
    if logger is not None:
        logger.warning(message)
    else:
        log_print(message, level="WARNING")
    return "F"


def _gate_family_sets(families, free_text_detections):
    unknown = sorted({label for _key, label, _aliases, _methods in families} - set(TECHNIQUE_EXECUTION_FAMILY))
    if unknown:
        raise ValueError(f"technique families label techniques that are not in TECHNIQUE_EXECUTION_FAMILY: {unknown}")
    stray = sorted(set(free_text_detections) - {key for key, _label, _aliases, _methods in families})
    if stray:
        raise ValueError(f"free-text detections that are not technique families: {stray}")

    def keys_where(predicate):
        return frozenset(key for key, label, _aliases, _methods in families if predicate(label))

    return {
        "supervised": keys_where(lambda label: TECHNIQUE_EXECUTION_FAMILY[label] == "A"),
        "model": keys_where(lambda label: label in TRAIN_INFER_TECHNIQUES),
        "text": keys_where(lambda label: TECHNIQUE_EXECUTION_FAMILY[label] == "G") | frozenset(free_text_detections),
        "temporal": keys_where(lambda label: label in TEMPORAL_TECHNIQUES),
    }


def _validate_technique_registry():
    families = set(GENIE_FAMILY_SPECS)
    if set(TECHNIQUE_EXECUTION_FAMILY) != set(TECHNIQUE_IMPLEMENTATION_GUIDANCE):
        raise ValueError("TECHNIQUE_EXECUTION_FAMILY and TECHNIQUE_IMPLEMENTATION_GUIDANCE must list the same techniques; "
                         f"only in the family registry: {sorted(set(TECHNIQUE_EXECUTION_FAMILY) - set(TECHNIQUE_IMPLEMENTATION_GUIDANCE))}, "
                         f"only in the guidance: {sorted(set(TECHNIQUE_IMPLEMENTATION_GUIDANCE) - set(TECHNIQUE_EXECUTION_FAMILY))}")
    unknown = sorted(t for t, family in TECHNIQUE_EXECUTION_FAMILY.items() if family not in families)
    if unknown:
        raise ValueError(f"techniques mapped to an undefined execution family: {unknown}")
    stray = sorted(TEMPORAL_TECHNIQUES - set(TECHNIQUE_EXECUTION_FAMILY))
    if stray:
        raise ValueError(f"TEMPORAL_TECHNIQUES names techniques that are not in TECHNIQUE_EXECUTION_FAMILY: {stray}")
    for name, table in (("GENIE_READINESS_THRESHOLDS", GENIE_READINESS_THRESHOLDS), ("GENIE_FAMILY_PACKAGES", GENIE_FAMILY_PACKAGES)):
        if set(table) != families:
            raise ValueError(f"{name} must define exactly the execution families {sorted(families)}; got {sorted(table)}")
    unpinned = sorted({p for packages in GENIE_FAMILY_PACKAGES.values() for p in packages} - set(GENIE_ENV_PACKAGE_VERSIONS))
    if unpinned:
        raise ValueError(f"packages without a pinned version for environment {GENIE_NOTEBOOK_ENV_VERSION}: {unpinned}")
    if len(GENIE_AGENTS_FILE_TEMPLATE) > 20000:
        raise ValueError("GENIE_AGENTS_FILE_TEMPLATE exceeds the 20,000-character limit Genie Code reads from AGENTS.md")


_validate_technique_registry()


PROMPT_TEMPLATES["ULTRA_HIGH_ELITE_JUDGE_PROMPT"] = """# ULTRA HIGH ELITE JUDGE: absolute selection of the best of the best

You are the final investment committee for this business: a principal business analyst, a principal data architect and a skeptical executive sponsor in one person. For each use case below, decide whether it is ELITE: it changes a decision that matters, the metadata proves the data needed to build it exists, and it can be built and run on Databricks.

## Calibration (absolute, one use case at a time)

- Judge each use case on its own merits against the fixed anchors below. There is no quota, no curve, no ranking within the batch and no default score. Every use case in the batch may be ELITE, or none may be.
- Earlier pipeline stages do not make a use case elite. The Best-of-Best gate scores shown with each use case are an earlier opinion; re-judge every use case from the evidence.
- A dimension scores at the highest anchor whose evidence is present. Missing evidence means a lower score, never a middle score.

## Evidence rules (metadata only)

- The only evidence is the AVAILABLE SCHEMA, the metadata evidence listed with each use case (table and column names, types, comments, keys, tags) and the business context. No data values are available; never assume more about the data than its names, types and comments state.
- Cite columns as fully qualified `catalog.schema.table.column` names, copied exactly from the AVAILABLE SCHEMA, and only from the use case's own Tables Involved. A column that is not listed does not exist.
- Never state money amounts, percentages or multipliers. Value is argued from the decision, the KPI and the columns that compute the KPI.
- Some use cases list the **verification failures** of an earlier assessment and the **real columns** of their tables. Re-cite every role from those real columns and never repeat a failed citation. When no real column can fill a role the technique needs, the metadata does not support that role: name the gap in `blockers` and score `evidence_strength` and `implementability` from it.

## What to establish for each use case

1. `decision`: the specific recurring business decision this use case changes.
2. `decision_owner`: the role that makes that decision. `cadence`: how often the decision is made (for example daily, weekly, monthly, quarterly or per event).
3. `kpi`: the business KPI the decision moves. `kpi_direction`: "increase" or "decrease".
4. `value_driver`: one sentence that links the decision to the KPI through backticked columns, with no numbers.
5. `value_tier`: "Transformational" (moves a core revenue, cost or risk line of the Revenue Model), "Material" (moves a KPI a sponsor funds on its own), "Incremental" (an efficiency or quality gain whose money effect is indirect) or "Marginal" (visibility or reporting).
6. `signal_columns`: the input columns that carry the signal. `time_column`: the DATE or TIMESTAMP column the technique orders events by (null when the technique is not temporal); when the use case's tables hold no DATE or TIMESTAMP column but a date key that the evidence lists as joining a dated table, cite that date key. `entity_key_column`: the key column of the entity the decision acts on. `join_path`: the key-to-key joins between the cited tables, each written as `catalog.schema.table.column = catalog.schema.table.column` (empty for one table). A text technique (sentiment analysis, extraction, document processing, topic modelling or text classification) lists in `signal_columns` the free-text column its method takes as input.
7. `outcome_definition`: required for supervised techniques (classification, prediction, regression) and for any KPI defined as the rate of a condition. It is an object with `column` (an existing outcome column, or null), `derived_from` (the existing columns the outcome is computed from), `rule` (how the outcome is computed from them) and `label_timing`: "future_window" when the outcome is observed in a period after the period the signal columns describe, "same_window" otherwise. Null otherwise.
8. The KPI is computed from a numeric or boolean column, from the time between two DATE or TIMESTAMP columns, or from the `outcome_definition`.
9. `build_path`: the Databricks implementation path in one sentence (for example a SQL feature table, a model tracked in MLflow, an AI function, a scheduled job that feeds a dashboard or an alert). `blockers`: concrete missing prerequisites (empty when there are none).
10. `runtime_assumptions`: every data shape the use case depends on that the metadata cannot prove, as objects with `kind`, `table`, `column` (both fully qualified, from the use case's own tables), `statement` (the assumption in plain words) and `central` (true when the use case produces nothing if the assumption is false). `kind` is one of: "KEY_HAS_REPEATED_ROWS" (values of `column` occur on more than one row of `table`), "KEY_HAS_SINGLE_ROWS" (some values of `column` occur on exactly one row of `table`), "COLUMN_HAS_TWO_VALUES" (`column` holds at least two distinct values) or "COLUMN_HAS_VALUES" (`column` holds values). The notebook checks every assumption before it builds anything. Empty only when the metadata proves every data shape the use case needs.
11. `fatal_flaws`: every reason that defeats the use case, as objects with `category` and `explanation` (one sentence). Empty when nothing defeats it. `category` is one of:
   - "OUTCOME_DERIVED_FROM_INPUTS": the outcome is a rule or threshold over the use case's own signal columns in the same period (label leakage or a tautology), so a model would only relearn the rule: the use case is a ranking or a rule, not a prediction.
   - "NO_DECISION_CHANGED": no recurring business decision changes; the output is informational.
   - "DUPLICATE_OF_ANOTHER_USE_CASE": another use case in this batch changes the same decision for the same entity (for example the same analysis on another channel's table); flag the weaker copy.
   - "DATA_SHAPE_ASSUMPTION_UNVERIFIABLE": the use case produces nothing unless a data shape holds, and the table's own keys or names indicate that it does not (for example several rows per group are needed but the group key is the table's own row key). A central assumption the metadata can neither confirm nor contradict is not fatal: list it in `runtime_assumptions` with `central` true.
   - "INFEASIBLE_WITH_AVAILABLE_COLUMNS": a role the technique needs (signal, outcome, entity key or time) has no existing column.
   - "VALUE_OVERCLAIMED": the honest value is Incremental or Marginal while the use case presents itself as more.
12. `red_team`: the strongest reason this use case fails or becomes shelfware. `red_team_category`: the category of that reason: one of the fatal-flaw categories when it defeats the use case (and then it is also listed in `fatal_flaws`), or "RESIDUAL_RISK" when it does not. `red_team_verdict` is derived by Inspire from `fatal_flaws` ("fatal" when it is not empty); do not return it. An assessment whose `red_team` names a fatal-flaw category that is missing from `fatal_flaws`, or whose verdict is ELITE or NEAR_MISS while `fatal_flaws` is not empty, contradicts itself and is discarded.

## Absolute anchors (integer scores 1-10)

Return the four scores in `scores`, an object with the keys `business_value`, `evidence_strength`, `implementability` and `actionability`.

**business_value**
- 9-10: changes a recurring, high-stakes decision tied to a stated Strategic Initiative or Revenue Model line; the KPI is a core revenue, cost or risk line.
- 7-8: changes a recurring decision on a KPI a sponsor funds; the link to the stated business context is one step removed.
- 5-6: a plausible owner and KPI, but the decision is occasional or generic.
- 3-4: mostly visibility; the decision or the owner is vague.
- 1-2: no identifiable decision or KPI.

**evidence_strength**
- 9-10: every role (signal, time, entity key, outcome) maps to existing columns with compatible types; the joins follow declared keys; the cited tables and columns are documented by comments or by unambiguous names.
- 7-8: every role maps to existing columns; a join is name-inferred or the documentation is thin.
- 5-6: one role rests on a proxy or on a derivation with a stated assumption.
- 3-4: a critical role rests on an assumption the metadata does not support.
- 1-2: a required table, column, key or outcome is missing.

**implementability**
- 9-10: standard Databricks capabilities only; the build path and the join path are concrete; no blockers.
- 7-8: one non-trivial component, such as an outcome derivation or an entity-matching step.
- 5-6: several non-trivial components, or a dependency the metadata does not prove.
- 3-4: needs data or systems the metadata does not show.
- 1-2: not buildable as described.

**actionability**
- 9-10: the output triggers a named action by the decision owner at the decision cadence, and the impact is attributable on the KPI through a before/after or controlled comparison.
- 7-8: a clear action; the cadence or the attribution has one weak point.
- 5-6: the action is implied, not named.
- 3-4: informational output only.
- 1-2: no action follows.

## Verdict

`verdict` is exactly one of:
- "ELITE": every dimension scores at least {min_dimension_score}, `value_tier` is {allowed_value_tiers}, and `fatal_flaws` is empty.
- "NEAR_MISS": `value_tier` and `fatal_flaws` meet the ELITE conditions and no dimension is more than {near_miss_band} point(s) below {min_dimension_score}.
- "REJECT": everything else.
- `verdict_reason`: one sentence naming the deciding evidence or the weakest dimension.

## Output

Return ONLY a JSON array with one object per use case, in input order. Every key is required; use null or [] only where the rules above allow it. Each object carries `No` (the use case id) and these keys: `decision`, `decision_owner`, `cadence`, `kpi`, `kpi_direction`, `value_driver`, `value_tier`, `signal_columns`, `time_column`, `entity_key_column`, `join_path`, `outcome_definition`, `build_path`, `blockers`, `runtime_assumptions`, `fatal_flaws`, `red_team`, `red_team_category`, `scores`, `verdict` and `verdict_reason`. Never return `red_team_verdict`.
""" + NO_EM_DASH_RULE + """

```json
[
  {{
    "No": "N01-U03",
    "decision": "Which <entities> receive the <intervention> each week",
    "decision_owner": "<role>",
    "cadence": "weekly",
    "kpi": "<business KPI>",
    "kpi_direction": "decrease",
    "value_driver": "Ranking `<catalog>.<schema>.<events>.<measure>` per `<catalog>.<schema>.<entities>.<entity_id>` focuses the weekly <intervention> on the <entities> that move the <business KPI>",
    "value_tier": "Material",
    "signal_columns": ["<catalog>.<schema>.<events>.<measure>", "<catalog>.<schema>.<events>.<event_ts>"],
    "outcome_definition": {{"column": null, "derived_from": ["<catalog>.<schema>.<events>.<status>"], "rule": "<status> reaches the <condition> value within the window after <event_ts>", "label_timing": "future_window"}},
    "time_column": "<catalog>.<schema>.<events>.<event_ts>",
    "entity_key_column": "<catalog>.<schema>.<entities>.<entity_id>",
    "join_path": ["<catalog>.<schema>.<events>.<entity_id> = <catalog>.<schema>.<entities>.<entity_id>"],
    "build_path": "A weekly SQL feature table, a classifier tracked in MLflow, and scores written to a table that feeds the owner's work queue",
    "blockers": [],
    "runtime_assumptions": [{{"kind": "COLUMN_HAS_TWO_VALUES", "table": "<catalog>.<schema>.<events>", "column": "<catalog>.<schema>.<events>.<status>", "statement": "Past <events> reached the <condition> value as well as other values, so the outcome has examples of both kinds", "central": true}}],
    "fatal_flaws": [],
    "red_team": "If the <condition> is recorded late, the outcome lags the decision and the model learns stale behaviour",
    "red_team_category": "RESIDUAL_RISK",
    "scores": {{"business_value": 9, "evidence_strength": 8, "implementability": 8, "actionability": 9}},
    "verdict": "ELITE",
    "verdict_reason": "A weekly owner decision on a funded KPI, grounded in declared keys and a dated outcome"
  }}
]
```

## Inputs

**Business Context**: {business_context}
**Industry**: {industry}
**Strategic Initiative**: {strategic_initiative}
**Value Chain**: {value_chain}
**Revenue Model**: {revenue_model}

""" + ASSUMED_CONTEXT_RULE + """

{generation_instructions_section}

## AVAILABLE SCHEMA (THE ONLY DATA THAT EXISTS)

{schema_details}

## Use Cases to Judge

{use_cases_to_judge}
"""

PROMPT_TEMPLATES["BEST_OF_BEST_SCORING_PROMPT"] = """# BEST-OF-BEST SCORING: absolute 19-gate scoring

You are a Principal Business Analyst. Score every use case below on the 19 quality gates (D1-D19) on an ABSOLUTE 0-10 scale. Downstream code may later trim the portfolio to about {target_max} use cases using these scores; that must never change how you score any single use case.

## Calibration (absolute, one use case at a time)

- Score each use case on its own merits against the fixed anchors below. There is no curve, no ranking within the batch, no forced spread and no default score. It is valid for every use case in the batch, or for none, to score high.
- A gate scores at the highest anchor whose evidence is present in the use case text, the AVAILABLE SCHEMA and the business context below. Missing evidence means a lower score; it never means a middle score.
- Earlier pipeline stages do not guarantee that a use case passes any gate. Judge each one as if seeing it for the first time.
- Verify D1-D7 against the AVAILABLE SCHEMA: every table and column the use case relies on must exist there, with a compatible type and grain. A use case that relies on a table or column missing from the schema scores 0-2 on D4 and D7.
- D8 compares the use case with the others in this batch only to detect duplicates: score 0-2 only for the weaker copy of a duplicate pair (less specific, fewer grounded columns; if equal, the later one).
- D11 (domain and technique balance) is a portfolio property, never a merit of one use case: score D11 = 10 for every use case.

## The 19 gates (D1-D19)

- D1 Causal Signal: how directly do the data columns CAUSE or indicate the outcome?
- D2 Cause-Effect Validity: how industry-recognized is the cause-effect relationship?
- D3 Data Granularity: how well does the data level match the analysis level?
- D4 Critical Dimensions Present: are ALL required columns actually in the schema?
- D5 Logical Possibility: is the operation possible given data types?
- D6 Metric Validity: is the claimed metric truly calculable from the fields?
- D7 Design-Schema Match: does the technical design cite columns that exist?
- D8 Semantic Uniqueness: after verb/noun/entity/metric/question normalization, is this a distinct business question?
- D9 Analytical Depth (Strip Test): remove ML mentions, is real substance left?
- D10 Activation Quality (Deliverable Test): "with [...]" names a business deliverable, not a technique?
- D11 Domain and Technique Balance: portfolio property (always 10, see Calibration).
- D12 Business Relevance & Opportunity: (business stakeholder) relevant to day-to-day ops? tangible value? named consumer team? current pain or future opportunity?
- D13 Sponsor Test: (CFO) fundable with a real budget line? pitchable to the CEO without caveats?
- D14 Engineering Test: (production engineer) deployable on existing Databricks capabilities? supportable long-term? fixable without bespoke infra?
- D15 Decision Cadence Match: does data refresh speed match decision speed?
- D16 The Monday Test: would a stakeholder's Monday look DIFFERENT? names a concrete action?
- D17 Explainability & Audit Defensibility: can the output be defended in 5 minutes without proprietary magic?
- D18 18-Month Longevity: still running in 18 months or abandoned?
- D19 Attributable Impact: can a measurable change in revenue, cost or risk be proven 6 months post-deployment?

## Absolute anchors

**Technical tier (D1-D10):**
- 9-10: Every claim maps to named columns that exist in the AVAILABLE SCHEMA, at the right grain, with compatible types; joins follow declared or clearly named keys; the technique's structural prerequisites are all present; the design needs genuine analytical work.
- 7-8: All critical columns exist; one non-critical element must be derived or approximated.
- 5-6: The core signal exists, but a critical element must be derived with a stated assumption, or the grain needs re-aggregation.
- 3-4: The core claim rests on an unverified assumption, a weak proxy, or a join the schema does not show.
- 0-2: A required table, column or relationship is missing or invented, or the logic is impossible on the stated types.

**Business tier (D12-D14):**
- 9-10: A named role owns a recurring decision; the KPI it moves, with its direction, is computable from named columns and tied to a stated Strategic Initiative, Value Chain step or the Revenue Model; it runs on standard Databricks capabilities.
- 7-8: Role, decision and KPI are clear; the link to the stated business context is one step removed, or the build needs one non-trivial component.
- 5-6: Plausible owner and KPI, but the decision or the funding case is generic.
- 3-4: The owner, decision or KPI is vague; the value is mostly visibility.
- 0-2: No identifiable owner or decision, or not buildable without services the schema and the platform do not provide.

**Principal-analyst tier (D15-D19):**
- 9-10: The timestamp columns support the decision cadence; the output triggers a named action; the method can be explained in 5 minutes; the need is persistent; the impact is attributable through a before/after or A/B comparison on a named KPI.
- 7-8: All of the above hold, with one weak point.
- 5-6: Two weak points.
- 3-4: Cadence, action or attribution is doubtful.
- 0-2: One-off, decorative or unattributable.

Never cite invented money or percentage figures as evidence. Evidence is columns, tables, joins and the stated business context.

## Output

Return a JSON array, one object per UC, in the same order as the input. Zero prose outside the JSON.

```json
[
  {{
    "No": "N01-U03",
    "bob_scores": {{"D1": 8, "D2": 7, "D3": 9, "D4": 9, "D5": 10, "D6": 8, "D7": 9, "D8": 7, "D9": 6, "D10": 8, "D11": 10, "D12": 9, "D13": 7, "D14": 8, "D15": 7, "D16": 9, "D17": 6, "D18": 8, "D19": 7}},
    "bob_justification_short": "`<amount_col>` by `<entity_id>` over `<event_ts>` gives a direct signal and a daily investigation queue; weakest gate D17 (scores need per-case explanations)."
  }},
  ...
]
```

**Rules**
- Every UC gets all 19 D-scores as integers 0-10. None omitted.
- `bob_justification_short` is one sentence, ≤200 chars, that cites at least one backticked column from the AVAILABLE SCHEMA as evidence and names the weakest gate.
- """ + NO_EM_DASH_RULE + """
- Return ONLY the JSON array.

## Inputs

**Business Context**: {business_context}
**Industry**: {industry}
**Strategic Initiative**: {strategic_initiative}
**Value Chain**: {value_chain}
**Revenue Model**: {revenue_model}

""" + ASSUMED_CONTEXT_RULE + """

{generation_instructions_section}

**Target portfolio size after trim (selection only, never a scoring input)**: {target_max}

## AVAILABLE SCHEMA (THE ONLY DATA THAT EXISTS)

{schema_details}

## Use Cases to Score

{use_cases_to_validate}
"""

PROMPT_TEMPLATES["USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT"] = """You write the instruction a user pastes into Genie Code, the Databricks coding agent, in a Databricks notebook. Genie Code reads ONLY this instruction. It adds every cell the notebook needs, runs it on serverless compute and fixes its own code until the notebook runs from top to bottom. So the instruction must stand on its own: name every table, column, rule, threshold and output, and never refer to cells, functions, variables or files that Genie Code would not have.

**RULES:**
- Cite only the tables and columns listed under TABLES: every table as `catalog.schema.table`, every column as `table.column`. An invented table or column name = FAILURE.
- Use the join path, entity key, time column and outcome definition from the DECISION BLOCK. When a field says "not assessed", choose it from TABLES and say which columns you chose and why.
- No invented numbers: you see metadata only, so never an invented dollar or percentage figure, accuracy gain, revenue, cost, volume or count that is not in the inputs. When a magnitude matters, write `[ESTIMATE: depends on <table.column>]`. Readiness thresholds, acceptance thresholds and statistical intervals (for example a 95% prediction interval) are allowed.
- Never fabricate unobserved inputs: no random numbers standing in for missing data. Seeded sampling and seeded simulation are allowed.
- No deep learning (no TensorFlow, PyTorch, or Keras).{model_feature_rule}
- The role of `ai_query` depends on the execution pattern: for AI text processing, Databricks AI functions are the core method; for every other pattern, the notebook never calls `ai_query`: `inspire_write_result` explains the flagged rows and `inspire_write_summary` writes the summary. Always name the AI endpoint given under RUN PATTERN AND BUSINESS OUTPUT.
- """ + GENIE_SERVERLESS_RULE + """ Read and join with Spark; move to pandas only after aggregating or sampling.
- The notebook writes ONLY the business output table and, for the execution patterns that register a model, the Unity Catalog model named below. It never writes to the source tables and never creates or deletes jobs (Inspire's Run controls cell runs its tagged job).
- Write prose with numbered build steps; each step after the Parameters cell is a function that `run_use_case()` calls in order (Platform Rules). A short SQL or Python snippet (at most 12 lines) is allowed only where it removes ambiguity, for example the exact outcome rule.{unreached_quantile_rule}
- Use only plain ASCII characters and straight quotes. """ + NO_EM_DASH_RULE + """

**STOP RULE:** """ + GENIE_READINESS_STOP_RULE + """ {acceptance_stop_rule} """ + GENIE_NO_FORCED_PASS_RULE + """ Inspire appends both rules verbatim: never restate them; where a section asks for them, write `""" + GENIE_STOP_RULE_POINTER + """`.

**BUSINESS CONTEXT:**
- Company: {business_name}
- Context: {enriched_business_context}
- Strategic Goals: {enriched_strategic_goals}
- Business Priorities: {enriched_business_priorities}
- Strategic Initiative: {enriched_strategic_initiative}
- Value Chain: {enriched_value_chain}
- Revenue Model: {enriched_revenue_model}

""" + ASSUMED_CONTEXT_RULE + """

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}

**USE CASE:**
- ID: {use_case_id} | Name: {use_case_name}
- Domain: {business_domain} > {subdomain} | Type: {use_case_type}
- Technique: {analytics_technique} | Execution pattern: {execution_family}
- Statement: {statement}
- Solution: {solution}
- Business Value: {business_value}
- High Level Design: {high_level_design}
- Beneficiary: {beneficiary} | Sponsor: {sponsor}
- Tables: {tables_involved} | Primary: {primary_table}
- Business output table: {result_table} | Unity Catalog model: {model_name}

**DECISION BLOCK** (from the Ultra High elite assessment, else "not assessed"):
- Decision changed: {decision}
- Decision owner: {decision_owner}
- Cadence: {cadence}
- KPI: {kpi} (direction: {kpi_direction})
- Value tier: {value_tier}
- Outcome definition: {outcome_definition}
- Time column: {time_column}
- Entity key column: {entity_key_column}
- Join path: {join_path}
- Signal columns: {signal_columns}
- Build path: {build_path}
- Blockers: {blockers}

**TABLES (directly involved: real columns, types and descriptions from Unity Catalog metadata; the ONLY source columns you may cite):**
{directly_involved_schema}

**FOREIGN KEY RELATIONSHIPS:**
{foreign_key_relationships}

**OTHER TABLES IN SCOPE** (use one only when the plan cannot work without it, and say why):
{available_tables_listing}

**TECHNIQUE GUIDANCE ({analytics_technique}):**
{technique_implementation_guidance}

**RUN PATTERN AND BUSINESS OUTPUT:**
{code_structure_mandate}

**READINESS AND ACCEPTANCE THRESHOLDS:**
{readiness_thresholds}

**CORE WIDGETS:**
{widget_table}

---

Write the instruction using EXACTLY these 8 sections, each starting with `## N. Section Name`. Output ONLY the instruction text: no preamble, no closing remarks, no fences around the whole output. Keep it under 1,800 words: Genie Code needs the facts and the rules, not explanations of why.

## 1. Goal
Two to four sentences: what this notebook decides, for whom and how often, the KPI and the direction that counts as better, and that the result lands in the business output table. Reuse the DECISION BLOCK wording; do not invent roles, team sizes or targets.

## 2. Data
Each table with its role (event facts, entity, reference), the join path with the join columns, the entity key, the time column, the as-of rule (every feature for an as-of date uses only data observed strictly before that date), the population the decision acts on, and how nulls in each signal, rating or outcome column are handled: left out of the calculation and counted, or carried with a `data_quality_flag` reason; never filled with an invented value. For supervised use cases state the outcome rule exactly: the event, its window, and how open cases are excluded.

## 3. Parameters
The widgets the notebook creates first, as one table with the columns `widget | type | default | allowed | used for`: every core widget above copied exactly (a `you choose` default becomes your value inside its allowed range, with a one-line reason), then the use-case widgets your steps need (at most 6, such as a lookback window or a minimum entity size), each with a non-empty default from the data, the business context or the method, and an allowed range. Use `text` for every widget and `dropdown` only for a fixed set of choices. Every threshold, window and limit in the later sections is read from its widget. A default that depends on the data is written as how to compute it. `as_of_date` = `latest` means the last period that is complete in every table the features and the outcome read: take each event-facts table's newest date on its own time column and use the earliest of them; entity and reference tables do not count. The first step of `run_use_case()`, never the Parameters cell, computes these defaults, resolves `latest` and prints each table's newest date. Every window, holdout, period grid and scoring population ends at the as-of date.

## 4. Readiness Check
The first thing the notebook computes after the Parameters cell. Every count is taken on the joined source rows inside the windows, before any feature engineering: `row_count` counts the rows of the primary table whose entity key and time column are present; a missing feature never removes a row from a readiness count. When a check fails, the notebook also prints how many rows each filter removed. After feature engineering, the notebook counts the rows each fit, holdout and scoring step receives; an empty one fails, printing how many rows each feature's nulls removed. One line per readiness threshold above: what is counted, over which rows, along which join, the required value, and the message the notebook prints when it fails, with the measured value filled in. Join coverage is the share of rows on the primary table that find a match along the join path; use 1.0 when the use case reads one table. Then the STOP RULE.

## 5. Build Steps
Numbered steps in notebook order, from installing the packages to the final summary step. Step 1 creates every widget of section 3 in one cell titled Parameters (`dbutils.widgets.text` or `dropdown`, with the used-for text as the label) and reads each into a typed variable; no later cell hardcodes a value a widget holds. The notebook contains every step, fully written, including the business output write and the summary step, even when this run stops by the STOP RULE: a stop skips that work but never removes or stubs it, so a run with fresher data or other widget values runs every step. Each step says what it computes, from which tables and columns, and what it returns. Cover the leakage guard, the time-aware split or backtest, the baseline, the candidates, the acceptance check, the model registration when the run pattern registers one, and the rows the run writes.

## 6. Method and Acceptance
The baseline, one or two candidate methods (the simplest that can beat the baseline), and how the method handles the risks of this data (class imbalance, seasonality, missing periods, sparse entities, text quality). The acceptance rule is exactly the one under READINESS AND ACCEPTANCE THRESHOLDS: never add a criterion to it. Compare the candidates with the baseline and report both, but the comparison decides acceptance only where that rule says so. For AI functions, name the function and the response format.

## 7. Business Output
How each output column is filled: what `value` means and its unit, the `value_at_stake` formula, the entity label, the segment, how the top reasons are derived (model contributions or rules), the recommended action template per priority tier, which conditions set `data_quality_flag` to something other than ok (never a result such as not significant), the facts in `ai_facts`, the evidence rule behind `confidence_level` and the `AI_CATEGORIES`. Priority tiers and recommended actions are rule-based; never ask a model to assign them.

## 8. Definition of Done
A checklist Genie Code verifies before it stops: the notebook ran from top to bottom on serverless without errors, with every step fully written and none stubbed; every widget of section 3 exists with its default and every later cell reads its value from the widgets; the readiness check printed its measured values against the required ones; the model is registered only when a candidate passed acceptance and the run pattern registers one; the business output table has every column listed above and rows with this run's run_id; the summary step prints the readiness values, the metrics against the baseline, the rows written per priority tier, the ten top-ranked rows, any `nothing_to_act` reason, and passes its facts to `inspire_write_summary`. End with the STOP RULE.
"""

# --- 1i. Business vs Technical Table Filtering Prompt (NEW - CRITICAL FOR QUALITY) ---
PROMPT_TEMPLATES["THEME_ASSIGNMENT_PROMPT"] = """# IDEA THEMES FOR THE WHOLE PORTFOLIO

Name the idea themes of this portfolio and put every use case below in exactly one of them. A theme names the analytical idea its use cases embody, not the business domain: for example "Anomaly Detection", "Churn Prediction", "Demand Forecasting", "Pricing Optimization", "Customer Segmentation", "Root Cause Attribution", "Risk Scoring", "Inventory Optimization". The themes keep the final portfolio spread across ideas, so one idea must carry one name across the whole list.

{generation_instructions_section}

## USE CASES
{use_cases_listing}
{existing_themes_section}
## RULES
1. A theme is 2 to 4 words in Title Case, without punctuation, and names one analytical idea: the kind of method and the kind of question it answers.
2. Use cases built with the same kind of method to answer the same kind of question share one theme, even when their domains, tables or objects differ: "Sensor Anomaly Detection" and "Payment Anomaly Detection" are both "Anomaly Detection"; "Machine Failure Prediction" and "Equipment Failure Prediction" share one theme.
3. Keep apart ideas that answer different business questions, even with the same method: "Demand Forecasting" and "Cash Flow Forecasting", "Pricing Optimization" and "Inventory Optimization", "Customer Segmentation" and "Product Segmentation" stay separate.
4. A theme uses exactly one kind of method, one of: {method_families}. Never put two kinds of method in one theme.
5. Most themes hold 2 to 6 use cases. A theme with one use case is right only when no other use case shares its idea. Never make a catch-all theme such as "Analytics", "Insights" or "Other".
6. Every use case appears exactly once in your answer, with its id written exactly as given.

## OUTPUT
Return ONLY a JSON array with one object per use case, in the same order: [{{"No": "<id as given>", "idea_theme": "<theme>"}}]. No other text.
"""

PROMPT_TEMPLATES["GOAL_ALIGNMENT_FILTER_PROMPT"] = """You are a **Goal Alignment Classifier**. Your ONLY job is to determine whether each use case aligns with the user's stated goal.

### USER GOAL
{goal_description}

### USE CASES TO CLASSIFY
{use_cases_table}

### RULES
1. For EACH use case, output ONLY the use case ID and YES or NO.
2. YES = the use case DIRECTLY serves the user's stated goal.
3. NO = the use case does NOT serve the user's stated goal, even if it is a valid business use case.
4. **EXPLICIT-OVERLAP TEST — mark YES only when at least one of these holds for the goal AS WORDED:**
   - **Same outcome**: the use case's stated outcome or KPI is the goal's outcome (same quantity, same direction), or a direct, named component of it (for example a loss, leakage, cost or revenue line that the goal's outcome adds up from).
   - **Same decision**: the decision the use case changes is a decision the goal names or directly requires.
   - **Loss prevention** (fraud, abuse, leakage, refund or pricing integrity, compliance or policy exposure, data errors): YES only when the loss or exposure it prevents is a direct component of the goal's outcome — for example the goal is about cost, margin, revenue, loss, risk or compliance. Never mark YES because of the topic alone.
5. Be STRICT OTHERWISE. If the use case is tangentially related but not directly about the goal, mark it NO. Do not reject a use case merely because it mentions "fraud" or "compliance" either — apply the same overlap test to it.
6. Do NOT add explanations, comments, or any other text.

### OUTPUT FORMAT
Return a CSV with exactly 2 columns: ID,Aligned
No header row. One row per use case. Example:
AI-P1_1-U1,YES
AI-P1_1-U2,NO
AI-P1_1-U3,YES

Start your response with the first ID immediately. No preamble.
"""

PROMPT_TEMPLATES["STRATEGY_ALIGNMENT_PROMPT"] = """# STRATEGY ALIGNMENT AND VALUE DEFINITION

You are a principal business analyst. The use cases below were already found and selected from the data alone. Your job is to align each one to the strategy, define the KPIs that measure it, its key results, measures of success and benefits. You never add, remove, rename or rank use cases: every use case below gets exactly one answer.

## BUSINESS
**Business:** {business_name}
**Business Context:** {business_context}

{generation_instructions_section}

## STRATEGY ({strategy_source})
Levels, top to bottom: {strategy_levels}
{strategy_entries}

## BUSINESS PRIORITIES
{business_priorities}

## SPONSOR ROSTER
{sponsor_roster}

## METRIC VIEWS (governed measures; prefer them for KPIs)
{metric_views}

## USE CASES
Each use case lists its tables with their columns and types, and data evidence where it was measured.
{use_cases_json}

## RULES
1. **Alignment**: pick at most 2 strategy entries per use case, by their id (S1, S1.2, ...), preferring the deepest entry that fits. Each alignment carries `evidence`: one sentence that names at least one of the use case's own columns in backticks, or one of its KPIs, and says how the use case moves that entry. Align only to an entry the use case's data can measure: when the entry lists KPIs, the use case must compute one of them (list it under `kpis` with its `kpi_id`); a use case that only helps the entry indirectly (efficiency, cost, staffing or a side benefit) is not aligned to it, and an entry whose subject no column measures stays unaligned so it is reported as a gap. An empty `alignments` list ("Not aligned") is valid and expected when no entry clearly fits; never stretch a link.
2. **Priorities**: `priority_alignment` lists only business priorities from the list above, worded exactly as given; an empty list when none fits.
3. **Strategic alignment score** (1.0-5.0): 4.8-5.0 = the use case's decision directly moves a KPI or key result of an aligned entry; 3.5-4.7 = a strong, one-step link; 1.0-3.4 = weak or no link. With no alignment, score 1.0.
4. **KPIs**: 1 to 3 per use case, each with `name`, `direction` (increase, decrease or maintain) and `unit`. When a strategy KPI (K1, K2, ...) measures the use case, set `kpi_id` to it and keep its name, but only when your computation measures that same quantity in its unit: never relabel another quantity (a cost, a count or revenue) as an emission, energy, weight, distance or satisfaction KPI; leave `kpi_id` empty instead. Define how the KPI is computed in one of two ways, never as SQL:
   - preferred: `metric_view` and `measure`, taken from the metric views listed above;
   - for a rate, share or ratio (for example a repeat purchase rate): the numerator as below plus `denominator` with its own `aggregation`, `column` and optional `filter_column` and `filter_value` on the same table; the KPI is numerator / denominator, times 100 when the unit is %. The numerator must measure part of what the denominator measures: filter it to the rows that count (for example a quantity on hand of 0 for out-of-stock rows) or use another column; a numerator equal to its denominator is always 100% and is refused. A % unit needs a share: COUNT over COUNT, COUNT_DISTINCT over COUNT or COUNT_DISTINCT, or SUM over SUM; a SUM over a count is a value per unit, so give it that unit. Name a KPI only for what it computes: a total is never called a rate, and a value only a model produces later (flagged, predicted, at risk) is never a baseline;
   - otherwise: `aggregation` (SUM, AVG, COUNT, COUNT_DISTINCT, MIN or MAX), `table` (one of the use case's tables), `column` (a column of that table; null only for COUNT), optionally `filter_column` and `filter_value` (for a text or true/false column, a value the data evidence lists for it; for a number column, a number), and optionally `time_column` (a DATE or TIMESTAMP column of that table) with `window_days`.
5. **Key results**: `key_result_ids` lists strategy key results (R1, R2, ...) that sit on an entry you aligned to. `proposed_key_results` adds 1 or 2 key results of the form "<verb> <KPI> from [BASELINE] to [TARGET] within [WINDOW]".
6. **Measures of success**: 1 to 3 observable checks that show the use case works in operation.
7. **Benefits**: `financial_benefits` (1 to 3) names the money line moved (revenue, margin, cost, loss, working capital) and the driver columns in backticks; `non_financial_benefits` (1 to 3) names customer, risk, compliance, employee or speed outcomes. Both lists are required. Where a measured magnitude belongs, write `[BASELINE:K<n>]`, where n is the position of the KPI in your `kpis` list (1-based); label any uplift as `[ASSUMPTION: uplift to be agreed with the sponsor]`.
8. **Sponsor**: when a sponsor roster is listed, `sponsor_id` is the roster id (SP1, SP2, ...) of the role best placed to own the decision; otherwise null.
9. **Numbers**: never write money amounts, percentages, multipliers or counts that are not written in the strategy, priorities or business context above. Placeholders carry every other number.
10. **PUNCTUATION**: """ + NO_EM_DASH_RULE + """

## OUTPUT
Return ONLY a JSON array, one object per use case, in input order:
```json
[
  {{
    "use_case_id": "N01-U01",
    "alignments": [{{"id": "S1.1", "evidence": "Weekly repeat orders per `customer_id` in `orders` move repeat purchase"}}],
    "priority_alignment": ["<priority as listed>"],
    "strategic_alignment": 4.6,
    "kpis": [{{"name": "<KPI name>", "kpi_id": "K1", "direction": "increase", "unit": "%", "metric_view": null, "measure": null,
              "aggregation": "COUNT_DISTINCT", "table": "<catalog>.<schema>.<table>", "column": "<column>", "filter_column": null,
              "filter_value": null, "time_column": "<date column>", "window_days": 90}}],
    "key_result_ids": ["R1"],
    "proposed_key_results": ["Raise <KPI> from [BASELINE] to [TARGET] within [WINDOW]"],
    "measures_of_success": ["<observable check>"],
    "financial_benefits": ["<money line> driven by `<column>` from [BASELINE:K1], [ASSUMPTION: uplift to be agreed with the sponsor]"],
    "non_financial_benefits": ["<outcome>"],
    "sponsor_id": null
  }}
]
```
"""

PROMPT_TEMPLATES["FILTER_BUSINESS_TABLES_PROMPT"] = """You are a **Senior Data Architect** classifying database tables as BUSINESS or TECHNICAL.

**BUSINESS CONTEXT**:
- **Business**: {business_name} | **Industry**: {industry}
- **Description**: {business_context}
- **Exclusion Strategy**: {exclusion_strategy}

{additional_context_section}

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}

**TABLE-EXCLUSION RULES** (apply them to every exclusion directive in the user instructions above; those directives override every other rule in this prompt, and tables matching an explicit ignore pattern were already removed before this prompt):
1. **Catalog / schema exclusions**: if the user ignores or excludes a catalog, database or schema, EVERY table under it is TECHNICAL.
2. **Table patterns**: a table whose name matches an ignore pattern (prefix, suffix, exact name or glob) is TECHNICAL.
3. **Entity exclusions**: when the user excludes a business entity, tables that hold the same entity under a synonym or an equivalent name are also TECHNICAL. Use business-domain knowledge to recognise equivalent names.
4. **Exclusivity**: a directive of the form "only X" or "nothing except X" makes tables unrelated to X TECHNICAL.
5. **Goals are not exclusions**: a goal, focus area, priority, technique or join instruction does NOT make other tables TECHNICAL. Classify those tables on their own comment and columns.
6. **Plausible matches**: when a table plausibly falls under an explicit exclusion, classify it as TECHNICAL.
Ignoring an explicit user exclusion rejects the entire output.


**CONTEXT-AWARE RULE**: Terms technical for most businesses may be BUSINESS for companies where they are core product (e.g., `clusters` is BUSINESS for Databricks but TECHNICAL for a retailer. `cluster_logs` is always TECHNICAL).

**UNIVERSAL TECHNICAL PATTERNS** (always TECHNICAL regardless of context):
`*_logs`, `*_audit_trail`, `*_changelog`, `*_snapshot`, `*_backup`, `information_schema.*`, `sys.*`, `*_metrics`, `*_health`, `*_monitoring`, `etl_*`, `pipeline_*` (orchestration), `*_error`, `*_exception`, `*_debug`, `*_config` (system), `*_test`, `*_staging`, `*_temp`, ML/platform infra metadata.
**Exception**: If business sells these as products (e.g., observability company), they're BUSINESS.


**🧱 MEDALLION + RAG/AI-PREP PATTERNS — CURATED BUSINESS TABLES, NOT TECHNICAL**

Tables named with medallion-architecture suffixes describe DATA QUALITY TIERS, not ETL artifacts. The content is still business data:

- `*_bronze_*` / `*_bronze` — raw landing; **TECHNICAL** (lake raw-tier, still transformed downstream)
- `*_silver_*` / `*_silver` — cleaned, joined; **BUSINESS** — ready-to-analyze business entities
- `*_gold_*` / `*_gold` — curated, aggregated, BI-ready; **BUSINESS** — the HIGHEST-VALUE tier, almost always master or transactional business data
- `*_chunked` / `*_embeddings` / `*_embeds` / `*_vectors` / `*_rag` — RAG-prepared variants of business data for LLM consumption. **If the base entity is BUSINESS, the chunked/embedding variant is ALSO BUSINESS.** These are first-class inputs for AI-powered analytics use cases (semantic search, topic modeling, sentiment analysis, LLM summarization) — NOT ETL artifacts.

**Example**: `media_gold_reviews_chunked` = gold-layer (curated) + reviews (customer-feedback, obviously business) + chunked (RAG-ready). This is BUSINESS with Data Category MASTER. **Never classify as TECHNICAL solely because of 'chunked', 'gold', 'silver', 'embedded', 'vectorized' suffixes** — reason about the BASE ENTITY.

**STRATEGY: {exclusion_strategy}**
{strategy_rules}

---

**DATA CATEGORY CLASSIFICATION (SEMANTIC — not score-based)**:

Use the business context above to understand what this company does, then classify EACH table by applying the 3-QUESTION FRAMEWORK below. Do NOT rely on table names alone — reason about what data the table holds in the context of THIS specific industry and business. Each table is listed with its owner-written comment (when one exists) and its column names and types: treat the comment and the columns as the PRIMARY evidence and the table name as secondary evidence.

**🚨 3-QUESTION FRAMEWORK (apply to EVERY table) 🚨**

**Q1: "Does a NEW row get inserted every time a business EVENT occurs?"**
  YES → likely **TRANSACTIONAL**. The table records discrete business events — things that HAPPENED at a specific moment. Each row represents one occurrence (one sale, one payment, one inspection, one shipment, one measurement). The table grows continuously over time. Rows are rarely updated after insertion.

**Q2: "Does this table describe WHAT something IS — its identity, attributes, or configuration — rather than WHAT HAPPENED?"**
  YES → likely **MASTER**. The table represents business ENTITIES — things that EXIST and have a lifecycle (active/inactive). Think: people, organizations, physical assets, products, locations, contracts, plans. The table is updated occasionally when entity attributes change, but new rows are NOT inserted for each business event.

**Q3: "Is this a finite, controlled lookup set that classifies or categorizes data in other tables?"**
  YES → likely **REFERENCE**. The table is a small, stable set of codes, types, categories, or rules. It changes rarely (quarterly/annually). Other tables reference it via foreign keys to categorize their data.

**TRANSACTIONAL** — "VERBS" of the business (events that HAPPENED):
- Each row = one discrete business event with a PRIMARY BUSINESS TIMESTAMP — a timestamp that records WHEN THE EVENT OCCURRED (not when the row was created/modified).
- Insert-heavy, rarely updated after creation. High volume, grows continuously over time.
- Typically references MASTER entities via FK (e.g., a transaction row points to a customer row).

**MASTER** — "NOUNS" of the business (entities that EXIST):
- Core business entity with lifecycle. Has a natural business identifier (ID, code, number).
- Timestamps on master tables are METADATA ABOUT the entity (created_at, last_updated, effective_date, hire_date, registration_date) — these do NOT make it transactional. These timestamps describe WHEN THE ENTITY WAS CREATED/CHANGED, not when business events happened.
- Updated occasionally when attributes change. Does NOT grow with each business transaction.

**REFERENCE** — "ADJECTIVES" of the business (lookup/configuration codes):
- Finite controlled set (typically <1000 rows). Classifies, categorizes, or configures other data.
- Changes rarely (quarterly, annually, or never). Other tables point to it for classification.

**🚨 CRITICAL ANTI-MISCLASSIFICATION RULES (apply to ALL industries) 🚨**

The #1 error is classifying MASTER tables as TRANSACTIONAL. Apply these rules to catch it:

1. **THE ENTITY TEST**: If the table represents a WHO (person, organization, partner), a WHAT (asset, product, service, resource), or a WHERE (location, facility, infrastructure) → it is MASTER or REFERENCE, NOT transactional. These tables describe things that EXIST, regardless of whether business events happen.

2. **THE EVENT TEST**: A table is ONLY transactional if each row captures a DISCRETE BUSINESS EVENT — something that happened at a specific point in time that would NOT exist if the event hadn't occurred. Ask: "If this business event hadn't happened, would this row not exist?" If yes → TRANSACTIONAL. If the row exists because the entity exists (regardless of events) → MASTER.

3. **THE TIMESTAMP TEST**: EVERY table has timestamps. The question is: does the timestamp represent the REASON the row exists (an event occurring), or is it just metadata about the row? If the table's primary purpose is to record "something happened at time X" → TRANSACTIONAL. If the timestamp is just "this entity was created/updated at time X" → MASTER/REFERENCE.

4. **THE GROWTH TEST**: Does this table grow proportionally with business activity? If transaction volume doubles, does this table's row count roughly double? If yes → TRANSACTIONAL. If the table stays roughly the same size regardless of business volume → MASTER/REFERENCE.

5. **THE FK DIRECTION TEST**: Does this table POINT TO other tables (has FK columns referencing master/reference entities), or do other tables POINT TO it (other tables have FK columns referencing this table's ID)? Tables that are POINTED TO by transactions are typically MASTER. Tables that POINT TO master entities are typically TRANSACTIONAL.

6. **PLANS/SCHEDULES/CONFIGURATIONS are MASTER, not TRANSACTIONAL**: A planned schedule, a pricing rule, a capacity allocation, a route definition, or a configuration table describes HOW things SHOULD work — not events that happened. Even if schedules change over time, each row represents a PLAN, not an event.

7. **CATALOGS/DIRECTORIES/REGISTRIES are MASTER or REFERENCE**: A product catalog, a partner directory, an asset registry, a service listing — these describe what CAN exist, not what DID happen.

**WHEN IN DOUBT → MASTER**: It is ALWAYS safer to classify as MASTER than as TRANSACTIONAL. MASTER tables are still available as JOIN context for use case generation. Misclassifying as TRANSACTIONAL wastes generation resources on static data and produces low-quality use cases centered on entity attributes rather than business events.

**TECHNICAL tables**: ONLY for IT infrastructure with ZERO business value: IT system logs, DB monitoring, server health, ETL orchestration, CI/CD, schema migrations, system configs.
**Edge cases default to BUSINESS**: Business user activity logs → TRANSACTIONAL. Business audit tables → TRANSACTIONAL. Business config/pricing → REFERENCE/MASTER. Mixed (>10% business columns) → BUSINESS.

**BUSINESS SCORE** (criticality, NOT category): 90-100=high-frequency transactional; 80-89=core master; 60-79=operational; 40-59=supporting; 20-39=static reference; 1-19=marginal. TECHNICAL=0.

---

**TABLES TO CLASSIFY** (one table per line: fully qualified name, then `comment:` when the owner wrote one, then `columns (N):` as `name type` pairs; a trailing `+K more` means K further columns exist):
{tables_markdown}

**OUTPUT**: CSV with columns: "Table Name","Classification","Data Category","Business Score","Reason"
- "Table Name" MUST be the fully qualified name exactly as listed (catalog.schema.table, without backticks). Never shorten, rename or invent a table name.
- Classification: "BUSINESS" or "TECHNICAL". Data Category: "MASTER", "TRANSACTIONAL", "REFERENCE", or "TECHNICAL".
- ALL fields double-quoted. Classify EVERY listed table exactly once. When in doubt, prefer BUSINESS.

Your response must START with: "Table Name","Classification","Data Category","Business Score","Reason","honesty_score","honesty_justification"
""" + HONESTY_CHECK_CSV

# COMMAND ----------

# DBTITLE 1,AIAgent
# ==============================================================================
# 2.4. AI AGENT MANAGER (Centralised AI Query Authority)
# ==============================================================================

class AdaptiveConcurrencyLimit:
    """A bounded semaphore whose limit follows AIMD: a throttle halves it, at most once per burst (a
    throttle seen by a call that started before the last halving is ignored); every INCREASE_EVERY
    successes add one slot; the limit stays between floor and ceiling."""

    INCREASE_EVERY = 10

    def __init__(self, ceiling: int, floor: int):
        self.ceiling = max(1, int(ceiling))
        self.floor = max(1, min(int(floor), self.ceiling))
        self.limit = self.ceiling
        self.in_flight = 0
        self.successes = 0
        self.epoch = 0
        self.throttles = 0
        self._cond = threading.Condition()

    def acquire(self, blocking: bool = True, timeout=None) -> bool:
        with self._cond:
            if not blocking:
                if self.in_flight < self.limit:
                    self.in_flight += 1
                    return True
                return False
            deadline = None if timeout is None else time.time() + timeout
            while self.in_flight >= self.limit:
                remaining = None if deadline is None else deadline - time.time()
                if remaining is not None and remaining <= 0:
                    return False
                self._cond.wait(remaining)
            self.in_flight += 1
            return True

    def release(self):
        with self._cond:
            if self.in_flight <= 0:
                raise ValueError("AdaptiveConcurrencyLimit released more times than acquired")
            self.in_flight -= 1
            self._cond.notify_all()

    def on_success(self):
        with self._cond:
            if self.limit >= self.ceiling:
                self.successes = 0
                return
            self.successes += 1
            if self.successes >= self.INCREASE_EVERY:
                self.limit += 1
                self.successes = 0
                self._cond.notify_all()

    def on_throttle(self, epoch) -> bool:
        with self._cond:
            self.throttles += 1
            self.successes = 0
            if epoch is not None and epoch < self.epoch:
                return False
            self.limit = max(self.floor, self.limit // 2)
            self.epoch += 1
            return True


class AIAgentManager:
    def __init__(self, max_concurrent_ai_calls: int, fallback_chain: list, logger=None):
        import threading
        self._max_concurrent = max(1, int(max_concurrent_ai_calls))
        self._semaphore = AdaptiveConcurrencyLimit(
            self._max_concurrent, TECHNICAL_CONTEXT.get("runtime", {}).get("ai_concurrency_floor", 4))
        self._fallback_chain = list(fallback_chain)
        self._logger = logger
        self._lock = threading.Lock()
        self._model_stats = {}

    def _ensure_model_stats(self, model_name: str):
        if model_name not in self._model_stats:
            self._model_stats[model_name] = {"calls": 0, "successes": 0, "timeouts": 0, "errors": 0}

    def record_call(self, model_name: str):
        with self._lock:
            self._ensure_model_stats(model_name)
            self._model_stats[model_name]["calls"] += 1

    def record_success(self, model_name: str):
        with self._lock:
            self._ensure_model_stats(model_name)
            self._model_stats[model_name]["successes"] += 1
        self._semaphore.on_success()

    def concurrency_epoch(self) -> int:
        return self._semaphore.epoch

    def on_throttle(self, epoch, model_name: str = ""):
        before = self._semaphore.limit
        if self._semaphore.on_throttle(epoch) and self._logger is not None:
            self._logger.warning(f"[AIMD FIRED] throttled on {model_name or 'an endpoint'}: in-flight LLM limit "
                                 f"{before} -> {self._semaphore.limit} (floor {self._semaphore.floor}); one slot "
                                 f"comes back every {AdaptiveConcurrencyLimit.INCREASE_EVERY} successes")

    def record_timeout(self, model_name: str):
        with self._lock:
            self._ensure_model_stats(model_name)
            self._model_stats[model_name]["timeouts"] += 1

    def record_error(self, model_name: str):
        with self._lock:
            self._ensure_model_stats(model_name)
            self._model_stats[model_name]["errors"] += 1

    def get_stats_summary(self) -> dict:
        with self._lock:
            return {k: dict(v) for k, v in self._model_stats.items()}

    def build_full_cascade(self, prompt_name: str) -> list:
        return get_full_fallback_cascade(prompt_name)

    def acquire(self):
        while not self._semaphore.acquire(timeout=1.0):
            if _is_thread_cancelled():
                raise Exception("LLM concurrency slot wait aborted: parent task was cancelled/timed out")

    def release(self):
        self._semaphore.release()

    @property
    def max_concurrent(self) -> int:
        return self._max_concurrent


# ==============================================================================
# 2.5. AI AGENT CLASS (MODIFIED)
# ==============================================================================

class AIAgent:
    _total_ai_calls = 0
    _total_input_chars = 0
    _total_output_chars = 0
    _step_stats = defaultdict(lambda: {"calls": 0, "input_chars": 0, "output_chars": 0})
    _usage_stats_lock = threading.Lock()

    # === MODIFIED: Added prompt_templates parameter ===
    def __init__(self, spark, logger, worker_llm_config, judge_llm_config, prompt_templates: dict,
                 default_timeout_seconds: int = None, max_retry_attempts: int = None, status_emitter=None):
        if default_timeout_seconds is None:
            default_timeout_seconds = TECHNICAL_CONTEXT["runtime"]["llm_timeout_seconds"]
        if max_retry_attempts is None:
            max_retry_attempts = TECHNICAL_CONTEXT["runtime"]["max_retry_attempts"]
        self.spark = spark
        self.logger = logger
        self.worker_llm = worker_llm_config
        self.judge_llm = judge_llm_config
        self.prompt_templates = prompt_templates  # <-- NEW: Store the dictionary
        self._language_local = threading.local()
        self._language_local.language = "English"
        self.default_timeout_seconds = default_timeout_seconds
        self.max_retry_attempts = max(0, max_retry_attempts)
        self.status_emitter = status_emitter
        self._status_step_ids = {}
        self._warning_seq = 0
        self._status_lock = threading.Lock()
        self._status_emit_failed = False
        
        self._llm_cache_dir = None
        self._llm_cache_lock = threading.Lock()

        _runtime = TECHNICAL_CONTEXT.get("runtime", {})
        _max_ai_calls = _runtime.get("global_max_parallelism", 20)
        _fallback_chain = _runtime.get("model_fallback_chain", [])
        self._manager = AIAgentManager(
            max_concurrent_ai_calls=_max_ai_calls,
            fallback_chain=_fallback_chain,
            logger=self.logger,
        )
        self.logger.info(
            f"AIAgentManager initialized: max_concurrent_ai_calls={self._manager.max_concurrent}, "
            f"fallback_chain_length={len(_fallback_chain)}"
        )
        
        if not self.prompt_templates:
            self.logger.warning("AIAgent initialized with an empty prompt_templates dictionary.")

    def _utc_event_at(self):
        from datetime import datetime, timezone
        return datetime.now(timezone.utc).isoformat()

    def _derive_entity_id(self, prompt_name: str, step_name: str, result_json) -> str:
        if isinstance(result_json, dict):
            for key in ("entity_id", "domain_name", "use_case_id", "batch_id"):
                value = result_json.get(key)
                if value:
                    return f"{prompt_name}:{value}"
        return f"{prompt_name}:{step_name}"

    def _enrich_status_payload(self, prompt_name: str, step_name: str, result_json):
        payload = result_json if isinstance(result_json, dict) else {}
        if "event_at" not in payload:
            payload["event_at"] = self._utc_event_at()
        if "entity_id" not in payload:
            payload["entity_id"] = self._derive_entity_id(prompt_name, step_name, payload)
        payload.setdefault("step_name", step_name)
        payload.setdefault("prompt_name", prompt_name)
        return payload

    def _emit_runtime_warning(self, prompt_name: str, step_name: str, message: str, payload: dict = None):
        import time
        with self._status_lock:
            self._warning_seq += 1
            seq = self._warning_seq
        warning_step = f"{step_name}__warning_{int(time.time() * 1000)}_{seq}"
        self._emit_status(
            prompt_name=prompt_name,
            step_name=warning_step,
            status="ended_warning",
            message=message,
            result_json=payload if payload is not None else {}
        )

    def _emit_status(self, prompt_name: str, step_name: str, status: str, message: str = "", result_json=None):
        if not self.status_emitter:
            return
        try:
            payload = self._enrich_status_payload(prompt_name, step_name, result_json if result_json is not None else {})
            key = f"{prompt_name}|{step_name}"
            if status == "started":
                step_id = self.status_emitter(
                    prompt_name=prompt_name,
                    step_name=step_name,
                    status=status,
                    message=message,
                    result_json=payload,
                    step_id=None
                )
                if step_id is not None:
                    with self._status_lock:
                        self._status_step_ids[key] = step_id
                return

            with self._status_lock:
                existing_step_id = self._status_step_ids.get(key)
            self.status_emitter(
                prompt_name=prompt_name,
                step_name=step_name,
                status=status,
                message=message,
                result_json=payload,
                step_id=existing_step_id
            )
            if status.startswith("ended_"):
                with self._status_lock:
                    self._status_step_ids.pop(key, None)
        except Exception as _emit_err:
            with self._status_lock:
                _first_failure = not self._status_emit_failed
                self._status_emit_failed = True
            if _first_failure:
                self.logger.warning(
                    f"Status emission failed for {prompt_name}/{step_name} ({get_clean_error_message(_emit_err)}); "
                    f"later status emission failures are logged at DEBUG."
                )
            else:
                self.logger.debug(f"Status emission failed for {prompt_name}/{step_name}: {get_clean_error_message(_emit_err)}")

    def _count_csv_rows_hint(self, csv_text: str) -> int:
        if not csv_text:
            return 0
        lines = [ln for ln in str(csv_text).splitlines() if ln.strip()]
        if not lines:
            return 0
        return max(0, len(lines) - 1)

    def _extract_csv_rows(self, raw_response: str, max_rows: int = 200) -> tuple:
        import csv
        from io import StringIO
        cleaned = clean_csv_response(raw_response) if raw_response else ""
        if not cleaned or "," not in cleaned:
            return [], 0, False
        try:
            reader = csv.DictReader(StringIO(cleaned))
            records = list(reader)
            kept, _ = CSVParser.aligned_rows(
                records, required=[f for f in (reader.fieldnames or []) if not CSVParser.honesty_kind(f)], text=cleaned)
            rows = [{str(k): str(v) for k, v in row.items()} for row in kept[:max_rows]]
            return rows, len(kept), len(kept) > len(rows)
        except Exception:
            return [], 0, False

    def _extract_json_object(self, raw_response: str):
        import json
        candidates = []
        if raw_response:
            candidates.append(raw_response)
            try:
                candidates.append(clean_json_response(raw_response))
            except Exception:
                pass
        for candidate in candidates:
            if not candidate:
                continue
            try:
                return json.loads(candidate)
            except Exception:
                continue
        return None

    def _build_started_status(self, prompt_name: str, step_name: str, prompt_vars: dict) -> tuple:
        pv = prompt_vars if isinstance(prompt_vars, dict) else {}
        summary = {"step_name": step_name, "prompt_name": prompt_name}
        if "schema_markdown" in pv:
            summary["schema_chars"] = len(str(pv.get("schema_markdown", "")))
        if "foreign_key_relationships" in pv:
            summary["fk_chars"] = len(str(pv.get("foreign_key_relationships", "")))
        if "use_cases_csv" in pv:
            summary["use_cases_input_count"] = self._count_csv_rows_hint(pv.get("use_cases_csv", ""))
        if "batch_csv" in pv:
            summary["batch_rows"] = self._count_csv_rows_hint(pv.get("batch_csv", ""))
        if "domain_name" in pv:
            summary["domain_name"] = str(pv.get("domain_name", ""))
        if "target_language" in pv:
            summary["target_language"] = str(pv.get("target_language", ""))
        if "strategic_goals" in pv:
            summary["strategic_goals_chars"] = len(str(pv.get("strategic_goals", "")))
        message = f"{prompt_name} started"
        details = []
        if "schema_chars" in summary:
            details.append(f"schema_chars={summary['schema_chars']}")
        if "use_cases_input_count" in summary:
            details.append(f"use_cases={summary['use_cases_input_count']}")
        if "domain_name" in summary and summary["domain_name"]:
            details.append(f"domain={summary['domain_name']}")
        if details:
            message = f"{prompt_name} started ({', '.join(details)})"
        return message, summary

    def _build_success_status(self, prompt_name: str, step_name: str, raw_response: str, response_schema) -> tuple:
        payload = {
            "step_name": step_name,
            "prompt_name": prompt_name,
            "response_type": "json" if response_schema else "raw",
            "response_chars": len(raw_response or "")
        }
        rows, total_rows, truncated = self._extract_csv_rows(raw_response)
        parsed_json = self._extract_json_object(raw_response)

        if prompt_name == "BASE_USE_CASE_GEN_PROMPT":
            use_cases = []
            for r in rows:
                use_cases.append({
                    "id": r.get("No", ""),
                    "name": r.get("Name", ""),
                    "usecase": r.get("usecase", ""),
                    "business_domain": r.get("Business Domain", ""),
                    "subdomain": r.get("Subdomain", ""),
                    "type": r.get("type", ""),
                    "priority": r.get("Priority", "")
                })
            payload["use_cases"] = use_cases
            payload["use_cases_count"] = total_rows
            payload["is_truncated"] = truncated
            return f"{prompt_name} completed with {total_rows} use cases", payload

        if prompt_name in ("DOMAIN_FINDER_PROMPT", "SUBDOMAIN_DETECTOR_PROMPT", "DOMAINS_MERGER_PROMPT", "SUBDOMAINS_MERGER_PROMPT", "DOMAIN_SPLITTER_PROMPT"):
            if isinstance(parsed_json, dict):
                domains = parsed_json.get("domains", parsed_json.get("domain_clusters", []))
                payload["domains"] = domains if isinstance(domains, list) else []
                payload["domains_count"] = len(payload["domains"])
            elif rows:
                payload["domains"] = rows
                payload["domains_count"] = total_rows
                payload["is_truncated"] = truncated
            return f"{prompt_name} completed with {payload.get('domains_count', 0)} domains", payload

        if prompt_name in ("COMBINED_VALUE_QUALITY_SCORE_PROMPT", "REVIEW_USE_CASES_PROMPT"):
            priorities = []
            scored_use_cases = []
            for r in rows:
                item = {
                    "id": r.get("No", ""),
                    "name": r.get("Name", ""),
                    "usecase": r.get("usecase", ""),
                    "priority": r.get("Priority", r.get("priority", "")),
                    "value": r.get("Value", r.get("value", "")),
                    "feasibility": r.get("Feasibility", r.get("feasibility", "")),
                    "quality": r.get("Quality", r.get("quality", ""))
                }
                scored_use_cases.append(item)
                if item["priority"]:
                    priorities.append({"id": item["id"], "priority": item["priority"]})
            payload["scored_use_cases"] = scored_use_cases
            payload["priorities"] = priorities
            payload["scored_count"] = total_rows
            payload["is_truncated"] = truncated
            return f"{prompt_name} completed with {total_rows} scored use cases", payload

        if rows:
            payload["rows"] = rows
            payload["rows_count"] = total_rows
            payload["is_truncated"] = truncated
            return f"{prompt_name} completed with {total_rows} rows", payload
        if parsed_json is not None:
            payload["json"] = parsed_json
            return f"{prompt_name} completed", payload
        return f"{prompt_name} completed", payload
    
    @property
    def current_language(self) -> str:
        return getattr(self._language_local, 'language', 'English')

    @current_language.setter
    def current_language(self, value: str):
        self._language_local.language = value

    def set_language(self, language: str):
        self._language_local.language = language

    def enable_llm_cache(self, session_id):
        self._llm_cache_dir = b17_writable_dir(
            os.path.join(tempfile.gettempdir(), f"inspire_{session_id}", "llm_cache"), "inspire_llm_cache_", self.logger)
        self.logger.info(f"LLM disk cache enabled: {self._llm_cache_dir}")

    def cleanup_llm_cache(self):
        import shutil
        if self._llm_cache_dir:
            try:
                shutil.rmtree(self._llm_cache_dir, ignore_errors=True)
                self.logger.info(f"LLM disk cache cleaned up: {self._llm_cache_dir}")
            except Exception:
                pass
            self._llm_cache_dir = None

    def _llm_cache_key(self, prompt: str, model: str, params: str = "") -> str:
        import hashlib
        h = hashlib.sha256((prompt + "||" + model + "||" + params).encode('utf-8', errors='replace')).hexdigest()[:32]
        return h

    def _llm_request_params(self, model_config: dict, prompt_name: str, response_schema=None, temperature_override=None) -> tuple:
        _min_output_default = min(m.get("llm_output_context_tokens_count", 25000) for m in TECHNICAL_CONTEXT["models"])
        output_token_limit = model_config.get("llm_output_context_tokens_count", _min_output_default)
        if prompt_name in COMPACT_OUTPUT_PROMPTS:
            max_output_tokens = min(int(output_token_limit * 0.25), output_token_limit)
        elif prompt_name in MEDIUM_OUTPUT_PROMPTS:
            max_output_tokens = min(20480, output_token_limit)
        else:
            max_output_tokens = min(int(output_token_limit * 0.9), output_token_limit)
        max_output_tokens = min(max_output_tokens, output_token_limit)
        effective_temperature = temperature_override if temperature_override is not None else _get_prompt_temperature(prompt_name)
        if not LLM_ENDPOINT_HEALTH.accepts_temperature(model_config.get("llm_endpoint_name")):
            effective_temperature = None
        response_format_str = ""
        if ((response_schema or prompt_name in LLM_JSON_FORMAT_PROMPTS)
                and LLM_ENDPOINT_HEALTH.accepts_json_format(model_config.get("llm_endpoint_name"))):
            response_format_str = json.dumps({"type": "json_schema", "json_schema": response_schema or JSON_ARRAY_RESPONSE_SCHEMA},
                                             separators=(',', ':'))
        cache_params = f"max_tokens={max_output_tokens}|temperature={effective_temperature}|response_format={response_format_str}"
        return max_output_tokens, effective_temperature, response_format_str, cache_params

    def _llm_cache_get(self, prompt: str, model: str, params: str = ""):
        if not self._llm_cache_dir:
            return None
        import os
        key = self._llm_cache_key(prompt, model, params)
        path = os.path.join(self._llm_cache_dir, f"{key}.txt")
        if os.path.exists(path):
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    cached = f.read()
                if cached:
                    self.logger.info(f"   LLM cache HIT: key={key[:12]}... ({len(cached):,} chars)")
                    return cached
            except Exception:
                pass
        return None

    def _llm_cache_put(self, prompt: str, model: str, response: str, params: str = ""):
        if not self._llm_cache_dir or not response:
            return
        import os
        key = self._llm_cache_key(prompt, model, params)
        path = os.path.join(self._llm_cache_dir, f"{key}.txt")
        try:
            with self._llm_cache_lock:
                with open(path, 'w', encoding='utf-8') as f:
                    f.write(response)
        except Exception:
            pass

    def _llm_cache_invalidate(self, prompt: str, model: str, params: str = ""):
        if not self._llm_cache_dir:
            return
        import os
        key = self._llm_cache_key(prompt, model, params)
        path = os.path.join(self._llm_cache_dir, f"{key}.txt")
        try:
            if os.path.exists(path):
                with self._llm_cache_lock:
                    os.remove(path)
                self.logger.info(f"   LLM cache INVALIDATED: key={key[:12]}... (retry requested a fresh response)")
        except Exception:
            pass

    _UNESCAPED_BRACE_RE = re.compile(r'(?<!\{)\{([^{}]+)\}(?!\})')

    def _sanitize_template_braces(self, template: str, prompt_key: str) -> str:
        """
        Pre-processes a prompt template to auto-escape {content} patterns where
        the content is NOT a valid Python .format() variable name.

        Prevents KeyError from unescaped JSON/text braces (e.g. {"key": "val"})
        that were accidentally left in the template string. Logs a warning for
        each pattern it auto-escapes so the developer can fix the source.
        """
        auto_fixed = []

        def _replacer(match):
            content = match.group(1)
            field_part = content.split('!')[0].split(':')[0].strip()
            base_name = field_part.split('.')[0].split('[')[0]
            if base_name and base_name.isidentifier():
                return match.group(0)
            auto_fixed.append(content[:100])
            return '{{' + content + '}}'

        sanitized = self._UNESCAPED_BRACE_RE.sub(_replacer, template)

        if auto_fixed:
            for fix_desc in auto_fixed:
                self.logger.warning(
                    f"[{prompt_key}] ⚠️ TEMPLATE BUG: Auto-escaped unescaped braces "
                    f"containing non-variable content: '{fix_desc}'"
                )

        return sanitized

    def _load_and_format_prompt(self, prompt_key: str, prompt_vars: dict) -> str:
        """
        Loads a prompt template from the internal dictionary and formats it.

        Includes a safety layer that auto-escapes any {content} pattern in the
        template where content is not a valid .format() variable name (e.g.
        unescaped JSON like {"key": "val"}). This prevents cryptic KeyError
        failures that silently break entire generation pipelines.
        """
        if prompt_key not in self.prompt_templates:
            self.logger.error(f"Prompt key '{prompt_key}' not found in prompt dictionary.")
            raise KeyError(f"Prompt key '{prompt_key}' not found in AIAgent's prompt_templates.")

        prompt_template = self._sanitize_template_braces(
            self.prompt_templates[prompt_key], prompt_key
        )

        try:
            return with_user_directive_preamble(prompt_template.format(**prompt_vars))
        except KeyError as e:
            self.logger.error(f"Failed to format prompt '{prompt_key}'. Missing variable: {get_clean_error_message(e)}")
            raise e
        except Exception as e:
            self.logger.error(f"An unexpected error occurred during prompt formatting for '{prompt_key}': {get_clean_error_message(e)}")
            raise

    @staticmethod
    def adaptive_timeout_seconds(base_timeout_seconds, prompt_chars) -> tuple:
        _adaptive_baseline_chars = 40000
        _adaptive_max_multiplier = 6
        _size_multiplier = max(1.0, min(float(_adaptive_max_multiplier), prompt_chars / float(max(1, _adaptive_baseline_chars))))
        return int(base_timeout_seconds * _size_multiplier), _size_multiplier

    def _call_ai_query(self, prompt: str, prompt_name: str, response_schema=None, model_override=None, timeout_seconds=None, max_retries=None, display_name=None, start_from_model=None, skip_cache=False, status_step_name=None, temperature_override=None) -> str:
        prompt = with_user_directive_preamble(prompt)
        request = ailog_new_request(prompt_name)
        outer_request = getattr(_AILOG_CALL, "request", None)
        _AILOG_CALL.request = request
        self._manager.acquire()
        try:
            return self._call_ai_query_impl(
                prompt, prompt_name, response_schema, model_override,
                timeout_seconds, max_retries, display_name, start_from_model,
                skip_cache, status_step_name, temperature_override
            )
        except BaseException as error:
            if request["attempts"] and not request["succeeded"]:
                AILOG_RECORDER.record_exhausted(request, prompt, error)
            raise
        finally:
            _AILOG_CALL.request = outer_request
            self._manager.release()

    def _call_ai_query_impl(self, prompt: str, prompt_name: str, response_schema=None, model_override=None, timeout_seconds=None, max_retries=None, display_name=None, start_from_model=None, skip_cache=False, status_step_name=None, temperature_override=None) -> str:
        """
        Calls Databricks AI query function with the given prompt.
        
        Uses cross-type model fallback chain: resolves the prompt's designated model bucket
        and builds a cascade through all remaining buckets in the fallback chain.
        Tries models in chain order; on failure, cascades to the next enabled model.
        Each model gets up to max_retries attempts before cascading.
        Concurrency is gated by AIAgentManager's BoundedSemaphore (acquired in _call_ai_query).
        
        Args:
            prompt: The prompt text
            prompt_name: Name of the prompt template (for logging and model_type resolution)
            response_schema: Optional JSON schema for structured responses
            model_override: Optional endpoint name override (skips cascade, uses only this model)
            timeout_seconds: Timeout in seconds for LLM call
            max_retries: Maximum number of retry attempts per model for throttling/timeouts
            display_name: Optional display name for heartbeat logs (defaults to prompt_name)
            start_from_model: Optional model name/endpoint to start cascade from (skips earlier models)
        
        Returns:
            Raw response string from the LLM
        """
        heartbeat_name = display_name if display_name else prompt_name
        status_step = status_step_name if status_step_name else heartbeat_name
        
        attempts_per_model = (max_retries if max_retries is not None else self.max_retry_attempts) + 1
        timeout_val = timeout_seconds if timeout_seconds is not None else self.default_timeout_seconds
        
        if not timeout_val or timeout_val <= 0:
            self.logger.warning(f"Timeout value '{timeout_val}' is invalid. Enforcing safety fallback of 300s.")
            timeout_val = 300

        _base_timeout_val = timeout_val
        _prompt_size_chars = len(prompt or "")
        _adaptive_timeout, _size_multiplier = AIAgent.adaptive_timeout_seconds(_base_timeout_val, _prompt_size_chars)
        if _size_multiplier > 1.0:
            self.logger.info(f"[LLM_CASCADE_ADAPTIVE_TIMEOUT FIRED] prompt_size={_prompt_size_chars:,} chars base={_base_timeout_val}s multiplier={_size_multiplier:.2f}x adaptive={_adaptive_timeout}s prompt={prompt_name}")
        timeout_val = _adaptive_timeout

        # --- Build model cascade ---
        if model_override:
            _override_unusable = LLM_ENDPOINT_HEALTH.unusable_reason(model_override)
            if _override_unusable is not None:
                raise LLMEndpointUnusableError(
                    f"[{prompt_name}] model_override endpoint '{model_override}' is unusable for this run "
                    f"({_override_unusable}); not calling it")
            override_config = next(
                (m for m in TECHNICAL_CONTEXT["models"] if m["llm_endpoint_name"] == model_override),
                None
            )
            if not override_config:
                min_output_limit = min(m.get("llm_output_context_tokens_count", 25000) for m in TECHNICAL_CONTEXT["models"])
                override_config = {
                    "name": model_override,
                    "llm_endpoint_name": model_override,
                    "llm_input_context_tokens_count": 131000,
                    "llm_output_context_tokens_count": min_output_limit,
                    "order": 999,
                    "type": "worker"
                }
            model_cascade = [override_config]
        elif start_from_model:
            full_cascade = self._manager.build_full_cascade(prompt_name)
            model_cascade = cascade_from_model(full_cascade, start_from_model)
            if not model_cascade:
                self.logger.warning(f"start_from_model '{start_from_model}' not found in cascade, using full cascade")
                model_cascade = full_cascade
        else:
            model_cascade = self._manager.build_full_cascade(prompt_name)
        if not model_cascade:
            _unusable = LLM_ENDPOINT_HEALTH.snapshot()["unusable"]
            if _unusable:
                raise LLMEndpointUnusableError(
                    f"No usable model left for prompt bucket of '{prompt_name}': "
                    + "; ".join(f"{ep} ({why})" for ep, why in sorted(_unusable.items())))
            raise ValueError(f"No enabled models available for prompt bucket of '{prompt_name}'")

        cascade_names = [m.get("name", m["llm_endpoint_name"]) for m in model_cascade]
        self.logger.info(f"   [{prompt_name}] Model cascade ({len(model_cascade)}): {' -> '.join(cascade_names)}")

        if self._llm_cache_dir and not skip_cache:
            for _mc in model_cascade:
                _mc_params = self._llm_request_params(_mc, prompt_name, response_schema, temperature_override)[3]
                cached = self._llm_cache_get(prompt, _mc["llm_endpoint_name"], _mc_params)
                if cached is not None:
                    return cached
        elif skip_cache and self._llm_cache_dir:
            self.logger.debug(f"   [{prompt_name}] Cache SKIPPED (retry wave — forcing fresh LLM call)")
            for _mc in model_cascade:
                _mc_params = self._llm_request_params(_mc, prompt_name, response_schema, temperature_override)[3]
                self._llm_cache_invalidate(prompt, _mc["llm_endpoint_name"], _mc_params)

        last_error = None
        prompt_len = len(prompt)
        any_model_attempted = False
        _ailog_request = getattr(_AILOG_CALL, "request", None)
        if _ailog_request is None or _ailog_request.get("prompt") != prompt_name:
            _ailog_request = ailog_new_request(prompt_name)

        # --- Outer loop: iterate through models in cascade order ---
        for model_idx, model_config in enumerate(model_cascade):
            if _is_thread_cancelled():
                raise Exception(f"[{prompt_name}] Cascade aborted: parent task was cancelled/timed out")

            is_last_model = (model_idx == len(model_cascade) - 1)
            model = model_config["llm_endpoint_name"]
            model_name = model_config.get("name", model)
            _unusable_reason = LLM_ENDPOINT_HEALTH.unusable_reason(model)
            if _unusable_reason is not None:
                self.logger.debug(f"   [{prompt_name}] Skipping '{model_name}': unusable for this run ({_unusable_reason})")
                if last_error is None:
                    last_error = LLMEndpointUnusableError(f"endpoint '{model}' is unusable for this run ({_unusable_reason})")
                continue
            input_token_limit_val = model_config.get("llm_input_context_tokens_count", 131000)
            _min_output_default = min(m.get("llm_output_context_tokens_count", 25000) for m in TECHNICAL_CONTEXT["models"])
            output_token_limit = model_config.get("llm_output_context_tokens_count", _min_output_default)

            if model_idx > 0:
                self.logger.info(
                    f"🔀 [{prompt_name}] Cascading to model '{model_name}' "
                    f"(order: {model_config.get('order', '?')}, type: {model_config.get('type', '?')}, "
                    f"input: {input_token_limit_val:,} tokens, output: {output_token_limit:,} tokens)"
                )

            # PRE-FLIGHT: Check if prompt fits this model's context window
            effective_limit = get_context_limit_chars_for_model(model_config, self.current_language)
            _planned_output = self._llm_request_params(model_config, prompt_name, response_schema, temperature_override)[0]
            if int(prompt_len / 3.2) + _planned_output > input_token_limit_val:
                effective_limit = min(effective_limit, max(0, int((input_token_limit_val - _planned_output) * 3.2)))

            if prompt_len > effective_limit:
                estimated_tokens = int(prompt_len / 3.2)
                self.logger.info(f"[CONTEXT-BUDGET FIRED] [{prompt_name}] {prompt_len:,} chars exceed the budget of "
                                 f"{effective_limit:,} chars for '{model_name}' (at most "
                                 f"{TECHNICAL_CONTEXT['runtime']['ai_call_context_fill_max']:.0%} of its "
                                 f"{input_token_limit_val:,}-token window, with room for {_planned_output:,} output tokens)")

                if any_model_attempted:
                    remaining_models = [m for m in model_cascade[model_idx:]
                                        if LLM_ENDPOINT_HEALTH.is_usable(m["llm_endpoint_name"])] or model_cascade[model_idx:]
                    best_remaining = max(remaining_models, key=lambda m: m.get("llm_input_context_tokens_count", 0))
                    best_limit = get_context_limit_chars_for_model(best_remaining, self.current_language)
                    failed_name = cascade_names[model_idx - 1] if model_idx > 0 else "unknown"
                    self.logger.warning(
                        f"🔄 [{prompt_name}] Prompt ({prompt_len:,} chars) exceeds remaining models' limits after "
                        f"'{failed_name}' failed. Best remaining: '{best_remaining.get('name')}' ({best_limit:,} chars). "
                        f"Raising CascadeRebatchError so caller can split into smaller batches."
                    )
                    raise CascadeRebatchError(
                        f"Prompt ({prompt_len:,} chars) exceeds '{best_remaining.get('name')}' limit ({best_limit:,} chars) "
                        f"after '{failed_name}' failed. Caller should rebatch to fit {best_limit:,} chars.",
                        target_model_name=best_remaining.get("name"),
                        target_model_endpoint=best_remaining["llm_endpoint_name"],
                        target_context_limit_chars=best_limit,
                        failed_model_name=failed_name
                    )

                if not is_last_model:
                    self.logger.warning(
                        f"⚠️  [{prompt_name}] Prompt ({prompt_len:,} chars, ~{estimated_tokens:,} tokens) exceeds "
                        f"'{model_name}' limit ({effective_limit:,} chars / {input_token_limit_val:,} tokens). "
                        f"Skipping to next model in cascade."
                    )
                    continue
                self.logger.error(
                    f"Prompt length ({prompt_len:,} chars, ~{estimated_tokens:,} est. tokens) exceeds limit "
                    f"({effective_limit:,} chars / {input_token_limit_val:,} tokens) "
                    f"for language '{self.current_language}' and prompt: {prompt_name}. All cascade models exhausted."
                )
                raise InputTooLongError(
                    f"Input length: {prompt_len:,} characters (~{estimated_tokens:,} tokens) exceeds context limit of {effective_limit:,} characters "
                    f"({input_token_limit_val:,} tokens) "
                    f"for language '{self.current_language}'. "
                    f"Prompt: {prompt_name}, Model: {model_name}. All {len(model_cascade)} cascade models exhausted. Please batch your input."
                )

            any_model_attempted = True
            self._manager.record_call(model_name)

            # --- Inner loop: retry attempts on this specific model ---
            cascade_to_next = False

            attempt = 1
            _temperature_retry_used = False
            _format_retry_used = False
            response_format_str = ""
            while attempt <= attempts_per_model:
                if _is_thread_cancelled():
                    raise Exception(f"[{prompt_name}] Cascade aborted: parent task was cancelled/timed out")
                try:
                    if attempt > 1:
                        self.logger.info(f"🔄 [{prompt_name}] Retry attempt {attempt}/{attempts_per_model} on model '{model_name}'...")
                        self._emit_runtime_warning(
                            prompt_name=prompt_name,
                            step_name=status_step,
                            message=f"{prompt_name} retry attempt {attempt}/{attempts_per_model} on {model_name}",
                            payload={"model": model_name, "attempt": attempt, "attempts_per_model": attempts_per_model}
                        )

                    max_output_tokens, effective_temperature, response_format_str, _cache_params = self._llm_request_params(
                        model_config, prompt_name, response_schema, temperature_override
                    )

                    self.logger.info(f"   [{prompt_name}] Setting max_tokens={max_output_tokens:,} (model: {model_name}, limit: {output_token_limit:,})")
                    if effective_temperature is None:
                        self.logger.debug(f"   [{prompt_name}] Temperature not sent: endpoint {model} does not accept it")
                    else:
                        self.logger.debug(f"   [{prompt_name}] Using temperature={effective_temperature} (override={temperature_override is not None})")

                    _attempt_timeout = timeout_val
                    _heartbeat = {"minute": 0}

                    def _heartbeat_log(elapsed, _state=_heartbeat):
                        current_minute = int(elapsed) // 60
                        if current_minute > _state["minute"]:
                            _state["minute"] = current_minute
                            waiting = f"[{heartbeat_name}] Still waiting... {elapsed:.0f}s elapsed (timeout: {_attempt_timeout}s, model: {model_name})"
                            log_print(waiting)
                            self.logger.debug(waiting)

                    _ailog = AILogAttempt(_ailog_request, model, prompt, max_output_tokens, effective_temperature,
                                          response_format_str or None)
                    _aimd_epoch_of = getattr(self._manager, "concurrency_epoch", None)
                    _aimd_epoch = _aimd_epoch_of() if _aimd_epoch_of else None
                    try:
                        _rows = _run_tagged_spark_work(
                            self.spark,
                            lambda _m=model, _tokens=max_output_tokens, _temp=effective_temperature, _fmt=(response_format_str or None), _log=_ailog:
                                _ai_query_rows(self.spark, _m, prompt, _tokens, _temp, _fmt, self.logger, log_attempt=_log),
                            timeout_seconds=_attempt_timeout, tag_prefix=f"llm_{prompt_name}", label=prompt_name,
                            on_wait=_heartbeat_log, logger=self.logger)
                    except SparkWorkCancelled:
                        _ailog.settle("failed", "cancelled by the parent task")
                        self.logger.warning(f"🛑 [{prompt_name}] Parent task cancelled - aborting LLM poll (model: {model_name})")
                        raise Exception(f"[{prompt_name}] Cascade aborted: parent task was cancelled/timed out")
                    except SparkWorkTimeout as _timed_out:
                        _ailog.settle("timedout", AILOGS_TIMEOUT_NOTE)
                        self.logger.warning(f"⏱️  [{prompt_name}] LLM call timed out after {_timed_out.elapsed:.1f}s (model: {model_name}, attempt {attempt}/{attempts_per_model}, budget: {_attempt_timeout}s)")
                        self._emit_runtime_warning(
                            prompt_name=prompt_name,
                            step_name=status_step,
                            message=f"{prompt_name} timeout on {model_name} after {int(_timed_out.elapsed)}s",
                            payload={
                                "model": model_name,
                                "attempt": attempt,
                                "attempts_per_model": attempts_per_model,
                                "elapsed_seconds": int(_timed_out.elapsed),
                                "event_type": "timeout"
                            }
                        )
                        raise Exception(f"LLM call timed out after {_attempt_timeout} seconds (model: {model_name})")

                    raw_response = _rows[0].ai_response if _rows and _rows[0] else ""
                    _rows = None

                    honesty_score, honesty_justification, cleaned_response = extract_honesty_score(raw_response, self.logger)
                    del raw_response
                    if response_format_str and not response_schema:
                        cleaned_response = unwrap_json_array_response(cleaned_response)
                        if not is_json_array_text(cleaned_response) and not _format_retry_used:
                            _format_retry_used = True
                            LLM_ENDPOINT_HEALTH.mark_json_format(model, False, self.logger, source=f"{prompt_name} answer")
                            self.logger.warning(f"[{prompt_name}] '{model_name}' answered the strict JSON format without the "
                                                f"array; re-sending without the format, which does not use up a retry attempt")
                            last_error = None
                            continue

                    if honesty_score is not None:
                        if honesty_justification:
                            self.logger.info(f"🔮✨ HONESTY CHECK [{prompt_name}] Score: {honesty_score}% | {honesty_justification} ✨🔮")
                        else:
                            self.logger.info(f"🔮✨ HONESTY CHECK [{prompt_name}] Score: {honesty_score}% ✨🔮")

                    input_len = len(prompt)
                    output_len = len(cleaned_response)

                    with AIAgent._usage_stats_lock:
                        AIAgent._total_ai_calls += 1
                        AIAgent._total_input_chars += input_len
                        AIAgent._total_output_chars += output_len

                        AIAgent._step_stats[prompt_name]["calls"] += 1
                        AIAgent._step_stats[prompt_name]["input_chars"] += input_len
                        AIAgent._step_stats[prompt_name]["output_chars"] += output_len

                    self._llm_cache_put(prompt, model, cleaned_response, _cache_params)
                    self._manager.record_success(model_name)
                    return cleaned_response

                except InputTooLongError:
                    raise
                except CascadeRebatchError:
                    raise
                except TruncatedResponseError:
                    raise
                except Exception as e:
                    if _is_thread_cancelled():
                        raise
                    last_error = e
                    error_msg = str(e).lower()
                    _error_kind = _classify_runtime_error(e)

                    _is_temperature_rejected = _error_kind == "temperature_unsupported"
                    if _is_temperature_rejected and not _temperature_retry_used:
                        _temperature_retry_used = True
                        LLM_ENDPOINT_HEALTH.mark_temperature_unsupported(model, self.logger, source=f"{prompt_name} call")
                        self.logger.debug(f"   [{prompt_name}] Re-sending to '{model_name}' without temperature; this does not use up a retry attempt")
                        last_error = None
                        continue

                    if (response_format_str and not _format_retry_used
                            and any(marker in error_msg for marker in _RESPONSE_FORMAT_ERROR_MARKERS)):
                        _format_retry_used = True
                        LLM_ENDPOINT_HEALTH.mark_json_format(model, False, self.logger, source=f"{prompt_name} call")
                        last_error = None
                        continue

                    if _error_kind == "endpoint_unusable":
                        _first_report = LLM_ENDPOINT_HEALTH.mark_unusable(model, get_clean_error_message(e), self.logger,
                                                                          source=f"{prompt_name} call")
                        self._manager.record_error(model_name)
                        if not is_last_model:
                            (self.logger.info if _first_report else self.logger.debug)(
                                f"🔀 [{prompt_name}] '{model_name}' cannot serve ai_query; cascading to the next model without retrying it")
                            cascade_to_next = True
                            break
                        self.logger.error(f"❌ [{prompt_name}] AI Query failed on '{model_name}' (last in cascade): {get_clean_error_message(e)}")
                        raise

                    is_sql_construction_error = (
                        'parse_syntax_error' in error_msg and
                        'select ai_query' in error_msg
                    )
                    is_max_tokens_error = 'max_new_tokens' in error_msg and 'cannot be greater' in error_msg
                    if is_sql_construction_error or is_max_tokens_error:
                        self.logger.error(
                            f"❌ [{prompt_name}] Non-retryable AI query error: {get_clean_error_message(e)}"
                        )
                        raise

                    if is_context_overflow_error(error_msg):
                        self.logger.error(
                            f"❌ [{prompt_name}] Input too long on model '{model_name}': {get_clean_error_message(e)}"
                        )
                        if not is_last_model:
                            next_cfg = model_cascade[model_idx + 1]
                            next_char_limit = int(next_cfg.get("llm_input_context_tokens_count", 131000) * 3.2)
                            if prompt_len <= next_char_limit:
                                self.logger.info(
                                    f"🔀 [{prompt_name}] Prompt fits next model '{next_cfg.get('name')}' "
                                    f"({prompt_len:,} chars <= {next_char_limit:,} limit). Cascading."
                                )
                                cascade_to_next = True
                                break
                        raise InputTooLongError(
                            f"Input length: {len(prompt)} characters exceeds model's context limit. "
                            f"Prompt: {prompt_name}, Model: {model_name}. All cascade models exhausted."
                        ) from e

                    is_throttling = _error_kind == "throttled"
                    is_timeout = _error_kind == "timeout"
                    is_server_error = _error_kind == "server"
                    is_retryable = is_throttling or is_timeout or is_server_error
                    if is_throttling and getattr(self._manager, "on_throttle", None):
                        self._manager.on_throttle(_aimd_epoch, model_name)
                    if is_timeout:
                        self._manager.record_timeout(model_name)
                        LLM_ENDPOINT_HEALTH.record_timeout(model, self.logger, source=prompt_name)
                    else:
                        self._manager.record_error(model_name)

                    error_type = "Throttling" if is_throttling else ("Timeout" if is_timeout else ("Server error" if is_server_error else "Error"))

                    clean_err_msg = get_clean_error_message(e)

                    _skip_retry_and_cascade = bool(is_timeout and not is_last_model)
                    if _skip_retry_and_cascade:
                        self.logger.info(f"[G31 CASCADE_SKIP_RETRY_ON_TIMEOUT FIRED] prompt={prompt_name} model={model_name} attempt={attempt}/{attempts_per_model} — cascading to next model instead of retrying same slow model")

                    if is_retryable and attempt < attempts_per_model and not _skip_retry_and_cascade:
                        wait_time = min(2 ** attempt * 5, 120)
                        self.logger.warning(
                            f"⚠️  [{prompt_name}] {error_type} on model '{model_name}' "
                            f"(attempt {attempt}/{attempts_per_model}): {clean_err_msg}"
                        )
                        self.logger.info(f"   Waiting {wait_time}s before retry on same model '{model_name}'...")
                        self._emit_runtime_warning(
                            prompt_name=prompt_name,
                            step_name=status_step,
                            message=f"{prompt_name} transient {error_type.lower()} on {model_name}, retrying in {wait_time}s",
                            payload={
                                "model": model_name,
                                "attempt": attempt,
                                "attempts_per_model": attempts_per_model,
                                "wait_seconds": wait_time,
                                "event_type": "retry_scheduled",
                                "error_type": error_type
                            }
                        )
                        if _cancellable_sleep(wait_time):
                            raise Exception(f"[{prompt_name}] Cascade aborted: parent task was cancelled/timed out")
                        attempt += 1
                        continue

                    if not is_last_model:
                        self.logger.warning(
                            f"⚠️  [{prompt_name}] {error_type} on model '{model_name}' "
                            f"(attempt {attempt}/{attempts_per_model}, retries exhausted). "
                            f"Cascading to next model. Error: {clean_err_msg}"
                        )
                        next_model_name = model_cascade[model_idx + 1].get("name", model_cascade[model_idx + 1]["llm_endpoint_name"])
                        self._emit_runtime_warning(
                            prompt_name=prompt_name,
                            step_name=status_step,
                            message=f"{prompt_name} cascading from {model_name} to {next_model_name}",
                            payload={
                                "from_model": model_name,
                                "to_model": next_model_name,
                                "attempt": attempt,
                                "event_type": "cascade"
                            }
                        )
                        cascade_to_next = True
                        break

                    if is_retryable:
                        self.logger.error(
                            f"❌ [{prompt_name}] All {len(model_cascade)} cascade models and retries exhausted "
                            f"for retryable error: {clean_err_msg}"
                        )
                    self.logger.error(f"❌ [{prompt_name}] AI Query failed on '{model_name}' (last in cascade): {get_clean_error_message(e)}")
                    raise

            if cascade_to_next:
                continue

        if last_error:
            _err_msg_lower = str(last_error).lower()
            _is_timeout_exhaustion = any(_k in _err_msg_lower for _k in ('timeout', 'timed out', 'deadline', 'time limit'))
            if _is_timeout_exhaustion and len(model_cascade) > 0:
                _smallest = min(model_cascade, key=lambda _m: _m.get("llm_input_context_tokens_count", 131000))
                _smallest_limit_chars = get_context_limit_chars_for_model(_smallest, self.current_language)
                _target_split_limit = max(4000, _smallest_limit_chars // 2)
                self.logger.error(f"[LLM_CASCADE_ADAPTIVE_TIMEOUT FIRED] cascade_exhausted models={len(model_cascade)} prompt={prompt_name} prompt_size={prompt_len:,} chars target_split_limit={_target_split_limit:,} chars - forcing batch split")
                raise CascadeRebatchError(f"Cascade exhausted on timeout for '{prompt_name}' ({prompt_len:,} chars). Rebatch under {_target_split_limit:,} chars.", target_model_name=_smallest.get("name"), target_model_endpoint=_smallest["llm_endpoint_name"], target_context_limit_chars=_target_split_limit, failed_model_name=cascade_names[-1] if cascade_names else "unknown")
            raise last_error
        raise Exception(f"All {len(model_cascade)} cascade models exhausted for prompt: {prompt_name}")

    def run_worker(self, step_name, worker_prompt_path, prompt_vars, response_schema, model_override=None, timeout_override=None, max_retries_override=None, start_from_model=None, skip_cache=False, temperature_override=None):
        try:
            start_message, start_payload = self._build_started_status(worker_prompt_path, step_name, prompt_vars)
            self._emit_status(
                prompt_name=worker_prompt_path,
                step_name=step_name,
                status="started",
                message=start_message,
                result_json=start_payload
            )
            worker_prompt = self._load_and_format_prompt(worker_prompt_path, prompt_vars)
            
            if not worker_prompt:
                raise ValueError(f"Failed to load prompt: {worker_prompt_path}")
            
            raw_response = self._call_ai_query(
                worker_prompt,
                worker_prompt_path,
                response_schema,
                model_override,
                timeout_seconds=timeout_override,
                max_retries=max_retries_override,
                display_name=step_name,
                start_from_model=start_from_model,
                skip_cache=skip_cache,
                status_step_name=step_name,
                temperature_override=temperature_override
            )
            
            if response_schema:
                parsed = clean_json_response(raw_response)
                end_message, end_payload = self._build_success_status(worker_prompt_path, step_name, raw_response, response_schema)
                self._emit_status(
                    prompt_name=worker_prompt_path,
                    step_name=step_name,
                    status="ended_success",
                    message=end_message,
                    result_json=end_payload
                )
                return parsed
            else:
                if not raw_response:
                    self.logger.warning(f"AI Worker for {step_name} (Raw) returned an empty response.")
                    self._emit_status(
                        prompt_name=worker_prompt_path,
                        step_name=step_name,
                        status="ended_warning",
                        message=f"{worker_prompt_path} returned empty response",
                        result_json={"step_name": step_name}
                    )
                    return ""
                end_message, end_payload = self._build_success_status(worker_prompt_path, step_name, raw_response, response_schema)
                self._emit_status(
                    prompt_name=worker_prompt_path,
                    step_name=step_name,
                    status="ended_success",
                    message=end_message,
                    result_json=end_payload
                )
                return raw_response
        except CascadeRebatchError:
            self._emit_status(
                prompt_name=worker_prompt_path,
                step_name=step_name,
                status="ended_warning",
                message=f"{worker_prompt_path} requires rebatching",
                result_json={"step_name": step_name}
            )
            raise
        except Exception as e:
            cancelled = _is_thread_cancelled()
            self._emit_status(
                prompt_name=worker_prompt_path,
                step_name=step_name,
                status="ended_warning" if cancelled else "ended_error",
                message=f"{worker_prompt_path} {'cancelled by its parent task' if cancelled else 'failed'}",
                result_json={"step_name": step_name, "error": get_clean_error_message(e, max_chars=300)}
            )
            if cancelled:
                self.logger.warning(f"AI Worker for {step_name} stopped because its parent task cancelled it: "
                                    f"{get_clean_error_message(e)}")
            else:
                self.logger.error(f"AI Worker process failed for {step_name}: {get_clean_error_message(e)}")
            raise

    # --- START OF MODIFICATIONS ---

    def _deep_parse_json_values(self, data, task):
        """
        (Helper) Parses known stringified keys ('attributes', 'domains') within a data object.
        Operates on the dictionary, not the JSON string.
        """
        if not isinstance(data, dict):
            return data # Not a dict, can't fix

        keys_to_check = []
        if task == 'attributes':
            keys_to_check = ['attributes']
        elif task == 'domains':
            keys_to_check = ['domains']
        
        for key in keys_to_check:
            value = data.get(key)
            if isinstance(value, str):
                try:
                    data[key] = json.loads(value) # Replace string with parsed object
                except (json.JSONDecodeError, TypeError):
                    self.logger.warning(f"Failed to deep-parse stringified key '{key}' in task '{task}'.")
                    data[key] = [] # Set to empty list on failure
        return data

    # === MODIFIED: Uses internal _load_and_format_prompt ===
    def run_worker_judge(self, step_name, worker_prompt_path, judge_prompt_path, base_prompt_vars, worker_response_schema, config, randomization_params={}, task_info_lambda=None, validation_lambda=None):
        
        task_type, log_context = task_info_lambda(base_prompt_vars) if task_info_lambda else ("unknown task", "")
        # Worker/judge process starting - high-level logging only
        
        # --- Added Retry Loop ---
        for attempt in range(config["MAX_RETRIES"]):
            try:
                # --- Worker Generation ---
                worker_outputs = []
                for i in range(2):
                    randomized_vars = base_prompt_vars.copy()
                    for key, val in randomization_params.items():
                        base_min = int(config["PROMPT_VARIABLES"][f"min_{key}"])
                        base_max = int(config["PROMPT_VARIABLES"][f"max_{key}"])
                        new_min = base_min + random.randint(0, val)
                        randomized_vars[f"min_{key}"] = new_min
                        randomized_vars[f"max_{key}"] = max(new_min + 1, base_max + random.randint(0, val))
                    
                    # === MODIFIED ===
                    worker_prompt = self._load_and_format_prompt(worker_prompt_path, randomized_vars)
                    
                    # --- Worker call ---
                    worker_step_name = f"{step_name}_worker_{i+1}"
                    raw_response = self._call_ai_query(
                        worker_prompt, worker_prompt_path, worker_response_schema,
                        display_name=worker_step_name, status_step_name=worker_step_name,
                        skip_cache=(attempt > 0 or i > 0)
                    )
                    worker_outputs.append(clean_json_response(raw_response))

                # --- Helper Functions (Modified to use _deep_parse) ---
                def summarize_output(json_string, task):
                    try:
                        data = json.loads(json_string)
                        data = self._deep_parse_json_values(data, task) # Fix stringified values
                        if task == 'domains': return f"domains: {', '.join([d.get('domain', 'N/A') for d in data.get('domains', [])])}"
                        if task == 'attributes': return f"{len(data.get('attributes', []))} attributes"
                        if 'dashboard' in task: return f"{sum(len(p.get('queries', [])) for p in data)} dashboard queries"
                        return "summary not available"
                    except (json.JSONDecodeError, TypeError): return "invalid JSON"

                def is_response_empty(json_string, task):
                    try:
                        data = json.loads(json_string)
                        if not data: return True
                        data = self._deep_parse_json_values(data, task) # Fix stringified values
                        if task == 'domains': return not data.get('domains')
                        if task == 'attributes': return not data.get('attributes')
                        if 'dashboard' in task: return not isinstance(data, list) or not any(p.get('queries') for p in data)
                        return True
                    except (json.JSONDecodeError, TypeError):
                        return True

                def get_best_worker_output(outputs, task):
                    best_output, max_count = "", -1
                    for out_str in outputs:
                        if is_response_empty(out_str, task): continue
                        try:
                            data, count = json.loads(out_str), 0
                            data = self._deep_parse_json_values(data, task) # Fix stringified values
                            if task == 'domains': count = len(data.get('domains', []))
                            elif task == 'attributes': count = len(data.get('attributes', []))
                            elif 'dashboard' in task: count = sum(len(p.get('queries', [])) for p in data)
                            if count > max_count:
                                max_count, best_output = count, out_str
                        except (json.JSONDecodeError, TypeError): continue
                    return best_output if best_output else max(outputs, key=len, default="")
                
                # --- End Helper Functions ---

                for i, output in enumerate(worker_outputs): self.logger.debug(f"LLM Worker {i+1} suggested {log_context}: {summarize_output(output, task_type)}")
                
                best_worker_fallback = get_best_worker_output(worker_outputs, task_type)
                
                # --- START: NEW JUDGE-SKIP LOGIC ---
                skip_judge = False
                final_cleaned_json = ""
                
                if task_type == 'domains' and judge_prompt_path:
                    try:
                        data1 = json.loads(worker_outputs[0])
                        data1 = self._deep_parse_json_values(data1, task_type)
                        domains1 = set(d.get('domain') for d in data1.get('domains', []))
                        
                        data2 = json.loads(worker_outputs[1])
                        data2 = self._deep_parse_json_values(data2, task_type)
                        domains2 = set(d.get('domain') for d in data2.get('domains', []))
                        
                        if domains1 and domains1 == domains2:
                            self.logger.debug("Worker domain outputs match. Skipping judge and using worker 1 output.")
                            final_cleaned_json = worker_outputs[0]
                            skip_judge = True
                    except Exception as e:
                        self.logger.warning(f"Could not compare worker outputs for judge skip: {get_clean_error_message(e)}")
                # --- END: NEW JUDGE-SKIP LOGIC ---

                if not judge_prompt_path or skip_judge:
                    if not judge_prompt_path:
                        self.logger.debug("No judge prompt provided. Skipping judge and using best worker output.")
                    
                    if not final_cleaned_json:
                        final_cleaned_json = best_worker_fallback
                else:
                    judge_prompt_vars = {**base_prompt_vars, **{f'llm{i+1}_output': out for i, out in enumerate(worker_outputs)}}
                    
                    # === MODIFIED ===
                    judge_prompt = self._load_and_format_prompt(judge_prompt_path, judge_prompt_vars)
                    
                    if len(judge_prompt) > 120000:
                        self.logger.debug(f"Judge prompt too long ({len(judge_prompt)} chars). Using best worker response.")
                        final_cleaned_json = best_worker_fallback
                    else:
                        judge_step_name = f"{step_name}_judge"
                        final_raw_response = self._call_ai_query(
                            judge_prompt, judge_prompt_path, worker_response_schema,
                            display_name=judge_step_name, status_step_name=judge_step_name,
                            skip_cache=(attempt > 0)
                        )
                        final_cleaned_json = clean_json_response(final_raw_response)
                        if is_response_empty(final_cleaned_json, task_type):
                            self.logger.warning(f"Judge returned a malformed or empty result for {task_type}. Rejecting and using best worker output.")
                            final_cleaned_json = best_worker_fallback
                
                try:
                    final_data = json.loads(final_cleaned_json)
                    final_data = self._deep_parse_json_values(final_data, task_type)
                    final_fixed_json_string = json.dumps(final_data)
                except (json.JSONDecodeError, TypeError):
                    self.logger.error(f"Failed to parse or fix final JSON output for {task_type}. Using raw cleaned JSON for validation.")
                    final_fixed_json_string = final_cleaned_json

                if validation_lambda:
                    validation_lambda(final_fixed_json_string, task_type)
                
                self.logger.debug(f"Judge adjudicated the final output {log_context}, resulting in: {summarize_output(final_fixed_json_string, task_type)}")
                return final_fixed_json_string

            except Exception as e:
                self.logger.warning(f"Attempt {attempt + 1}/{config['MAX_RETRIES']} for {step_name} {log_context} failed: {get_clean_error_message(e)}")
                if attempt == config["MAX_RETRIES"] - 1:
                    self.logger.error(f"FAILED to generate valid output for {step_name} {log_context} after all retries.")
                    raise e
        
        raise Exception(f"AI Worker/Judge for {step_name} failed after all retries.")

    @staticmethod
    def get_summary_report():
        """
        Generates a summary report of AI usage, grouped by prompt type.
        More generic and aggregated across all instances.
        """
        report = []
        report.append("\n" + "="*70)
        report.append("--- 📊 AI Usage Summary ---")
        report.append("="*70)
        report.append(f"Total AI Calls:     {AIAgent._total_ai_calls}")
        
        # Calculate estimated tokens (chars / 4)
        input_tokens = AIAgent._total_input_chars / 4
        output_tokens = AIAgent._total_output_chars / 4
        
        report.append(f"Total Input Tokens:  ~{input_tokens:,.2f}  ({AIAgent._total_input_chars:,} chars)")
        report.append(f"Total Output Tokens: ~{output_tokens:,.2f}  ({AIAgent._total_output_chars:,} chars)")
        report.append("\n--- Prompt Type Details ---")
        
        if not AIAgent._step_stats:
            report.append("No AI calls were tracked.")
        else:
            # Group by prompt type (extract the general prompt name from step names)
            prompt_aggregates = {}
            for step_name, stats in AIAgent._step_stats.items():
                # Extract prompt type from step name (e.g., "Generate_UseCases_Batch_5" -> "Generate_UseCases")
                # Common patterns: "PromptName_Language_Batch_X", "PromptName_Batch_X", etc
                parts = step_name.split('_')
                
                # Identify the core prompt name
                prompt_type = step_name
                if 'Batch' in step_name:
                    # Extract everything before "_Batch"
                    batch_idx = step_name.find('_Batch')
                    if batch_idx > 0:
                        prompt_type = step_name[:batch_idx]
                elif any(lang in step_name for lang in ['English', 'Arabic', 'Chinese', 'French', 'Spanish']):
                    # Remove language suffix
                    for lang in ['English', 'Arabic', 'Chinese', 'French', 'Spanish', 'German', 'Portuguese', 'Italian', 'Japanese', 'Korean']:
                        if f"_{lang}" in step_name:
                            prompt_type = step_name.replace(f"_{lang}", "")
                            break
                
                # Aggregate stats by prompt type
                if prompt_type not in prompt_aggregates:
                    prompt_aggregates[prompt_type] = {'calls': 0, 'input_chars': 0, 'output_chars': 0}
                
                prompt_aggregates[prompt_type]['calls'] += stats['calls']
                prompt_aggregates[prompt_type]['input_chars'] += stats['input_chars']
                prompt_aggregates[prompt_type]['output_chars'] += stats['output_chars']
            
            # Display aggregated results
            prompt_col_width = 40
            header = f"{'Prompt Type':<{prompt_col_width}} | {'Calls':<7} | {'Input Tokens':<18} | {'Output Tokens':<18}"
            report.append(header)
            report.append("-" * len(header))
            
            # Sort by number of calls (descending)
            for prompt_type, stats in sorted(prompt_aggregates.items(), key=lambda x: x[1]['calls'], reverse=True):
                input_tok = stats['input_chars'] / 4
                output_tok = stats['output_chars'] / 4
                report.append(f"{prompt_type:<{prompt_col_width}} | {stats['calls']:<7} | ~{input_tok:<16,.0f} | ~{output_tok:<16,.0f}")
        
        report.append("="*70)

        final_report_str = "\n".join(report)
        log_print(final_report_str)
        return final_report_str

    def get_manager_stats_report(self) -> str:
        if not hasattr(self, '_manager'):
            return ""
        mgr_stats = self._manager.get_stats_summary()
        if not mgr_stats:
            return ""
        lines = []
        lines.append("\n" + "="*70)
        lines.append("--- AIAgentManager Model Stats ---")
        lines.append(f"Max concurrent AI calls: {self._manager.max_concurrent}")
        mgr_header = f"{'Model':<30} | {'Calls':<7} | {'Success':<9} | {'Timeouts':<9} | {'Errors':<7}"
        lines.append(mgr_header)
        lines.append("-" * len(mgr_header))
        for mname, mstats in sorted(mgr_stats.items(), key=lambda x: x[1].get("calls", 0), reverse=True):
            lines.append(
                f"{mname:<30} | {mstats.get('calls',0):<7} | "
                f"{mstats.get('successes',0):<9} | {mstats.get('timeouts',0):<9} | "
                f"{mstats.get('errors',0):<7}"
            )
        health = LLM_ENDPOINT_HEALTH.snapshot()
        for endpoint, reason in sorted(health["unusable"].items()):
            lines.append(f"Unusable endpoint skipped by every cascade: {endpoint} ({reason})")
        if health["temperature_unsupported"]:
            lines.append("Endpoints without temperature support (configured temperatures not applied): "
                         + ", ".join(health["temperature_unsupported"]))
        lines.append("="*70)
        report_str = "\n".join(lines)
        log_print(report_str)
        return report_str

# COMMAND ----------

# DBTITLE 1,Translations
# ==============================================================================
# 1.5. TRANSLATION SERVICE (MODIFIED)
# ==============================================================================

def inspire_plain_text(value, joiner: str = ", ") -> str:
    if value is None:
        return ""
    if isinstance(value, dict):
        parts = [(key, inspire_plain_text(item)) for key, item in value.items()]
        return "; ".join(f"{key}: {text}" for key, text in parts if text)
    if isinstance(value, (list, tuple, set)):
        return joiner.join(text for text in (inspire_plain_text(item) for item in value) if text)
    return str(value).strip()


class TranslationService:
    """
    Handles all language translation tasks by calling an AI agent in parallel.
    Relies on an AIAgent that has been initialized with the PROMPT_TEMPLATES dictionary.
    """
    
    # === MODIFIED: Added pdf_disclaimer_title and updated titles ===
    ENGLISH_TRANSLATIONS = {
        "main_title": "Use Case Generator Powered by Databricks Genie Code",
        "intro": "This notebook contains AI-generated use cases based on your schemas. Below is a summary of the generated scenarios by business domain.",
        "domain": "Business Domain",
        "total": "Total Use Cases",
        "summaries": "Use Case Summaries",
        "sum_id": "ID",
        "sum_name": "Use Case",
        "sum_value": "Business Value",
        "sum_outcome": "Expected Outcome",
        "warning_header": "WARNING",
        "warning_body": "Do not run this notebook. It is intended for demonstration and cataloging purposes only.",
        "disclaimer": "This content is AI-generated and should be reviewed by a qualified engineer before being used in production. Databricks is not liable for any issues arising from the use of this content.",
        "detailed_scenarios": "Use Cases Details",
        "aspect": "Aspect",
        "description": "Description",
        "aspect_domain": "Business Domain",
        "type": "Type",
        "analytics_technique": "Analytics Technique",
        "primary_table": "Primary Table",
        "quality": "Quality",
        "priority": "Priority",
        # Value translations for Type field
        "value_type_problem": "Problem",
        "value_type_risk": "Risk",
        "value_type_opportunity": "Opportunity",
        "value_type_improvement": "Improvement",
        # Value translations for Priority field
        "value_priority_ultra_high": "Ultra High",
        "value_priority_very_high": "Very High",
        "value_priority_high": "High",
        "value_priority_medium": "Medium",
        "value_priority_low": "Low",
        "value_priority_very_low": "Very Low",
        "value_priority_ultra_low": "Ultra Low",
        "statement": "Statement",
        "solution": "Solution",
        "aspect_beneficiary": "Beneficiary",
        "beneficiary": "Beneficiary",
        "aspect_sponsor": "Sponsor",
        "sponsor": "Sponsor",
        "business_priority_alignment": "Business Priority Alignment",
        "strategic_goals_alignment": "Strategic Goals Alignment",
        "subdomain": "Subdomain",
        "usecase": "Use Case",
        "description_label": "Description",
        "aspect_value": "Business Value",
        "business_value": "Business Value",
        "aspect_tables": "Tables Involved",
        "quality_reasons": "Quality Reasons",
        "aspect_ai_function": "AI Function",
        "aspect_analytics_technique": "Analytics Technique",
        "aspect_primary_table": "Primary Table",
        "aspect_quality": "Quality",
        "aspect_priority": "Priority",
        # Scoring field labels (for Excel)
        "strategic_alignment": "Strategic Alignment",
        "return_on_investment": "Return on Investment",
        "reusability": "Reusability",
        "time_to_value": "Time to Value",
        "data_availability": "Data Availability",
        "data_accessibility": "Data Accessibility",
        "architecture_fitness": "Architecture Fitness",
        "team_skills": "Team Skills",
        "domain_knowledge": "Domain Knowledge",
        "people_allocation": "People Allocation",
        "budget_allocation": "Budget Allocation",
        "time_to_production": "Time to Production",
        "value_score": "Value",
        "feasibility_score": "Feasibility",
        "priority_score": "Priority",
        "inspire_score": "Inspire Score",
        "quality_score_header": "Quality Score",
        "pdf_title": "Strategic AI Use Cases Powered by Databricks Genie Code", # MODIFIED
        "pdf_for": "For",
        "pdf_exec_summary": "Executive Summary",
        "pdf_toc_title": "Use Case Domains",
        "pdf_detailed_view": "Detailed Use Case Catalog",
        "pdf_disclaimer_title": "Disclaimer", # NEW
        "pdf_fallback_summary_p1": "This document outlines {total_cases} high-value analytical use cases identified for {business_name}. These scenarios, powered by Databricks Genie Code, are designed to drive significant business outcomes by leveraging your existing data assets.",
        "pdf_fallback_summary_p2": "The following pages provide a detailed breakdown of these opportunities, categorized by business domain, to help prioritize your AI initiatives.",
        "pptx_main_title": "Strategic AI Use Cases Powered by Databricks Genie Code", # MODIFIED
        "pptx_for": "For",
        "pptx_disclaimer_title": "Disclaimer",
        "pptx_domain_suffix": "Use Cases",
        "elite_verdict": "Verdict",
        "elite_decision": "Decision",
        "elite_decision_owner": "Decision Owner",
        "elite_kpi": "KPI",
        "elite_value_driver": "Value Driver",
        "elite_evidence_columns": "Evidence Columns",
        "elite_join_path": "Join Path",
        "elite_build_path": "Build Path",
        "elite_red_team_note": "Red-Team Note",
        "elite_coverage_pick": "Domain coverage pick, below the bar",
        "elite_fallback_pick": "Below the {mode} bar",
        "elite_fallback_near_miss": "Near miss, below the {mode} bar",
        "doc_fallback_pick": "Below bar",
        "doc_fallback_notice": "No use case met the {mode} bar. These are the closest candidates, each labelled below the bar.",
        "elite_reason": "Reason",
        "untranslated_notice": "Shown in English (translation unavailable)",
        "more_use_cases": "...and {count} more use cases.",
        "executive_summary_not_available": "Executive summary not available.",
        "domain_summary_not_available": "Domain summary not available.",
        "summary_not_available": "Summary not available.",
        "doc_catalog_kicker": "AI use case catalog",
        "doc_prepared_for": "Prepared for",
        "doc_generated_on": "Generated on",
        "doc_quality_mode": "Quality mode",
        "doc_use_cases": "Use cases",
        "doc_domains": "Business domains",
        "doc_elite_use_cases": "Elite use cases",
        "doc_high_or_better": "High quality or better",
        "doc_best_score": "Best Inspire Score",
        "doc_not_scored": "Not scored",
        "doc_coverage_picks": "Coverage picks",
        "doc_coverage_pick": "Coverage pick",
        "value_correlation_analysis": "Correlation Analysis",
        "value_pareto_analysis": "Pareto Analysis",
        "value_funnel_analysis": "Funnel Analysis",
        "value_document_processing": "Document Processing",
        "value_extraction": "Extraction",
        "value_ai_analysis": "AI Analysis",
        "value_simulation": "Simulation",
        "value_geospatial_analysis": "Geospatial Analysis",
        "value_predictive_modeling": "Predictive Modeling",
        "value_clustering": "Clustering",
        "value_regression_analysis": "Regression Analysis",
        "value_survival_analysis": "Survival Analysis",
        "value_network_analysis": "Network Analysis",
        "value_mode_good": "Good",
        "value_mode_high": "High",
        "value_mode_ultra_high": "Ultra High",
        "doc_exec_kicker": "The opportunity",
        "doc_guide_kicker": "Reading guide",
        "doc_how_to_read": "How to read this catalog",
        "doc_guide_lead": "Every use case comes from the organization's own metadata and is scored on one absolute scale. Use the scores to compare, the tiers to filter and the domain pages to navigate.",
        "doc_inspire_score_title": "Inspire Score, from 0 to 10",
        "doc_inspire_score_body": "Combines the quality of a use case with its business priority. Higher is better, and the catalog is ordered by it.",
        "doc_quality_tiers_title": "Quality tiers",
        "doc_quality_tiers_body": "Each use case receives a tier from {lowest} to {highest} that reflects how specific, valuable and buildable it is with the available data.",
        "doc_modes_title": "Quality modes",
        "doc_mode_good_body": "Keeps use cases rated {medium} or better.",
        "doc_mode_high_body": "Keeps only use cases rated {high} or better.",
        "doc_mode_ultra_body": "Keeps only use cases that clear an absolute bar on value, evidence, implementability and actionability. There is no quota.",
        "doc_this_catalog": "This catalog",
        "doc_elite_score_title": "Elite score",
        "doc_elite_score_body": "Used in {ultra} mode. It weighs business value, evidence strength, implementability and actionability, and an elite use case scores at least {min_score} out of 10 on each.",
        "doc_coverage_title": "Domain coverage picks",
        "doc_coverage_body": "When a requested business domain has no use case above the bar, its strongest candidate is kept and labelled as a coverage pick below the bar.",
        "doc_portfolio_kicker": "Portfolio",
        "doc_portfolio_glance": "Portfolio at a glance",
        "doc_per_domain": "Use cases per domain",
        "doc_quality_distribution": "Quality distribution",
        "doc_top_use_cases": "Top use cases",
        "doc_rank": "Rank",
        "doc_domain_label": "Domain",
        "doc_domain_overview": "About this domain",
        "doc_what_it_covers": "What it covers",
        "doc_top_opportunities": "Top opportunities",
        "doc_domain_fallback": "This domain brings together use cases across {subdomains}, {count} in total. Its strongest opportunities are listed below and detailed on the following pages.",
        "doc_open_notebook": "Open the Genie notebook",
        "doc_elite_score": "Elite Score",
        "doc_best_of_best": "Best of Best",
        "doc_appendix": "Appendix",
        "doc_methodology": "Methodology",
        "strategy_path": "Strategy Path",
        "kpis": "KPIs",
        "expected_key_results": "Expected Key Results",
        "measures_of_success": "Measures of Success",
        "financial_benefits": "Financial Benefits",
        "non_financial_benefits": "Non-Financial Benefits",
        "foundations": "Data Foundations",
        "doc_foundations_title": "Shared data foundations",
        "doc_foundations_lead": "Tables and joins that two or more use cases need. Building them first unlocks the most use cases. The order is a sequence, not a schedule.",
        "doc_foundation_tables": "Tables",
        "doc_foundation_unlocks": "Use cases unlocked",
        "doc_data_access_standard": "Standard discovery: Inspire read no customer data. It used metadata, metric view definitions and usage counts only.",
        "doc_data_access_deep": "Deep discovery: Inspire read aggregates from the tables in scope, {tables} in total (row counts, date coverage, null rates, distinct counts and short value lists of non-personal columns). No data rows were sent to the AI.",
        "doc_method_1_deep": "Inspire reads metadata, and in Deep discovery also aggregates computed over the tables. It never sends data rows to the AI.",
        "doc_guide_lead_deep": "Every use case comes from the organization's own metadata and data profiles and is scored on one absolute scale. Use the scores to compare, the tiers to filter and the domain pages to navigate.",
        "value_not_aligned": "Not aligned",
        "kpi_baseline": "baseline {value}, last {days} days, measured {date}",
        "kpi_baseline_undated": "baseline {value}, measured {date}",
        "doc_method_1": "Inspire reads metadata only: table and column names, types, comments, keys and tags. It never reads row data.",
        "doc_method_2": "It derives the business context, then generates candidate use cases for every business domain the data supports.",
        "doc_method_3": "Duplicates are removed, every candidate is scored on value and feasibility, and quality gates drop weak or unsupported ideas.",
        "doc_method_4": "In {ultra} mode, an elite judge reviews each remaining candidate and deterministic checks confirm that every cited table and column exists.",
        "doc_method_5": "The final list is ranked, grouped by business domain and, where selected, paired with a Databricks Genie Code notebook.",
        "doc_glossary": "Glossary",
        "doc_gl_domain": "A business area that groups related use cases.",
        "doc_gl_subdomain": "A narrower theme inside a business domain.",
        "doc_gl_technique": "The analytical method, for example forecasting or classification.",
        "doc_gl_inspire": "The score from 0 to 10 that combines quality and business priority.",
        "doc_gl_quality": "The tier from {lowest} to {highest} given to each use case.",
        "doc_gl_elite": "The {ultra} score across value, evidence, implementability and actionability.",
        "doc_gl_coverage": "A use case kept below the bar so that a requested domain is represented.",
        "doc_gl_notebook": "A Databricks notebook with Genie Code instructions to build the use case.",
        "doc_notebook_term": "Genie notebook",
        "doc_next_steps": "Next steps",
        "doc_closing_title": "From use cases to outcomes",
        "doc_closing_body": "Choose the use cases that matter most, open their Genie notebooks and build the first one with the team. The steps are a suggested order; the team sets the pace.",
        "doc_step_select": "Select",
        "doc_step_select_body": "Pick the use cases with the strongest score and a clear sponsor.",
        "doc_step_validate": "Validate the data",
        "doc_step_validate_body": "Confirm the tables involved and their joins with the data owners.",
        "doc_step_build": "Build with Genie Code",
        "doc_step_build_body": "Open the Genie notebook and generate the first working version.",
        "doc_step_schedule": "Schedule",
        "doc_step_schedule_body": "Run the notebook as a Databricks job that refreshes the result table.",
        "doc_step_measure": "Measure and scale",
        "doc_step_measure_body": "Track the business value it delivers, then extend to the next use case.",
        "doc_agenda": "Agenda",
        "doc_roadmap": "Roadmap and next steps",
        "doc_use_case_index": "Use case index",
        "doc_continued": "continued",
        "doc_not_provided": "Not provided",
        "doc_bob_score": "Best-of-Best Score",
        "elite_cadence": "Decision Cadence",
        "value_cadence_hourly": "Hourly",
        "value_cadence_daily": "Daily",
        "value_cadence_weekly": "Weekly",
        "value_cadence_biweekly": "Every two weeks",
        "value_cadence_monthly": "Monthly",
        "value_cadence_quarterly": "Quarterly",
        "value_cadence_yearly": "Yearly",
        "value_cadence_real_time": "Real time",
        "value_cadence_per_event": "Per event",
        "doc_generated_by": "Generated by {brand} {version} · {date}",
        "doc_disclaimer_ai": "The use cases, scores and narratives in this document are generated by AI. Validate them with the data owners and the business sponsors before any decision is made.",
        "doc_disclaimer_figures": "Scores, values and outcomes are estimates for prioritization only. No figure in this document is guaranteed.",
        "doc_contents": "Contents",
        "doc_index_lead": "All use cases in final rank order, {count} in total.",
        "doc_deck_scope": "This presentation details the top-ranked use cases, {count} of {total}. The PDF catalog details all of them.",
        "doc_deck_scope_all": "This presentation details every use case, {total} in total, one card each.",
        # Strategic Goal/Priority Alignment value translations
        "value_general_improvement": "General Improvement",
        "value_reduce_cost": "Reduce Cost",
        "value_increase_revenue": "Increase Revenue",
        "value_boost_productivity": "Boost Productivity",
        "value_mitigate_risk": "Mitigate Risk",
        "value_protect_revenue": "Protect Revenue",
        "value_align_to_regulations": "Align to Regulations",
        "value_improve_customer_experience": "Improve Customer Experience",
        "value_enable_data_driven_decisions": "Enable Data-Driven Decisions",
        "value_optimize_operations": "Optimize Operations",
        "value_empower_talent": "Empower Talent",
        "value_enhance_experience": "Enhance Experience",
        "value_drive_innovation": "Drive Innovation",
        "value_achieve_esg": "Achieve ESG",
        "value_execute_strategy": "Execute Strategy",
        # Analytics Technique value translations
        "value_forecasting": "Forecasting",
        "value_classification": "Classification",
        "value_anomaly_detection": "Anomaly Detection",
        "value_cohort_analysis": "Cohort Analysis",
        "value_segmentation": "Segmentation",
        "value_sentiment_analysis": "Sentiment Analysis",
        "value_trend_analysis": "Trend Analysis",
        "value_prescriptive_analytics": "Prescriptive Analytics",
        "value_root_cause_analysis": "Root Cause Analysis",
        "value_optimization": "Optimization",
        "value_recommendation": "Recommendation",
        "value_time_series_analysis": "Time Series Analysis",
        "value_predictive_analytics": "Predictive Analytics",
        "value_descriptive_analytics": "Descriptive Analytics"
    }

    def __init__(self, ai_agent, logger=None):
        """
        Initializes the TranslationService with an AIAgent instance.
        """
        self.ai_agent = ai_agent
        self.logger = logger or logging.getLogger(self.__class__.__name__)
        self.translation_cache = {}
        self._translation_cache_lock = threading.Lock()

    # Complete fallback translations for ALL keys - ensures translations are 100% reliable
    TRANSLATION_FALLBACKS = {
        "Arabic": {
            "main_title": "مولد حالات الاستخدام مدعوم بـ داتا بريكس جيني كود",
            "intro": "يحتوي هذا الدفتر على حالات استخدام تم إنشاؤها بواسطة الذكاء الاصطناعي استناداً إلى مخططاتك. فيما يلي ملخص للسيناريوهات المُنشأة حسب مجال الأعمال.",
            "domain": "مجال الأعمال",
            "total": "إجمالي حالات الاستخدام",
            "summaries": "ملخصات حالات الاستخدام",
            "sum_id": "المعرف",
            "sum_name": "الاسم",
            "sum_value": "القيمة التجارية",
            "sum_outcome": "النتيجة المتوقعة",
            "warning_header": "تحذير",
            "warning_body": "لا تقم بتشغيل هذا الدفتر. وهو مخصص للعرض التوضيحي والفهرسة فقط.",
            "disclaimer": "هذا المحتوى تم إنشاؤه بواسطة الذكاء الاصطناعي ويجب مراجعته من قبل مهندس مؤهل قبل استخدامه في الإنتاج. داتابريكس غير مسؤولة عن أي مشاكل ناتجة عن استخدام هذا المحتوى.",
            "detailed_scenarios": "تفاصيل حالات الاستخدام",
            "aspect": "الجانب",
            "description": "الوصف",
            "aspect_domain": "مجال الأعمال",
            "type": "النوع",
            "analytics_technique": "تقنية التحليل",
            "primary_table": "الجدول الرئيسي",
            "quality": "الجودة",
            "priority": "الأولوية",
            "usecase": "حالة الاستخدام",
            "description_label": "الوصف",
            "value_type_problem": "مشكلة",
            "value_type_risk": "مخاطرة",
            "value_type_opportunity": "فرصة",
            "value_type_improvement": "تحسين",
            "value_priority_ultra_high": "عالية للغاية",
            "value_priority_very_high": "عالية جداً",
            "value_priority_high": "عالية",
            "value_priority_medium": "متوسطة",
            "value_priority_low": "منخفضة",
            "value_priority_very_low": "منخفضة جداً",
            "value_priority_ultra_low": "منخفضة للغاية",
            "statement": "البيان",
            "solution": "الحل",
            "aspect_beneficiary": "المستفيد",
            "beneficiary": "المستفيد",
            "aspect_sponsor": "الراعي",
            "sponsor": "الراعي",
            "business_priority_alignment": "توافق أولوية الأعمال",
            "strategic_goals_alignment": "التوافق مع الأهداف الاستراتيجية",
            "subdomain": "النطاق الفرعي",
            "aspect_value": "القيمة التجارية",
            "business_value": "القيمة التجارية",
            "aspect_tables": "الجداول المستخدمة",
            "quality_reasons": "أسباب الجودة",
            "aspect_ai_function": "وظيفة الذكاء الاصطناعي",
            "aspect_analytics_technique": "تقنية التحليل",
            "aspect_primary_table": "الجدول الرئيسي",
            "aspect_quality": "الجودة",
            "aspect_priority": "الأولوية",
            "strategic_alignment": "التوافق الاستراتيجي",
            "return_on_investment": "العائد على الاستثمار",
            "reusability": "قابلية إعادة الاستخدام",
            "time_to_value": "الوقت للقيمة",
            "data_availability": "توفر البيانات",
            "data_accessibility": "إمكانية الوصول للبيانات",
            "architecture_fitness": "ملاءمة البنية",
            "team_skills": "مهارات الفريق",
            "domain_knowledge": "المعرفة بالمجال",
            "people_allocation": "تخصيص الموارد البشرية",
            "budget_allocation": "تخصيص الميزانية",
            "time_to_production": "الوقت للإنتاج",
            "value_score": "درجة القيمة",
            "feasibility_score": "درجة الجدوى",
            "priority_score": "درجة الأولوية",
            "inspire_score": "درجة إنسباير",
            "quality_score_header": "درجة الجودة",
            "pdf_title": "حالات استخدام الذكاء الاصطناعي الاستراتيجية مدعومة بـ داتا بريكس جيني كود",
            "pdf_for": "لـ",
            "pdf_exec_summary": "الملخص التنفيذي",
            "pdf_toc_title": "مجالات حالات الاستخدام",
            "pdf_detailed_view": "كتالوج حالات الاستخدام التفصيلية",
            "pdf_disclaimer_title": "إخلاء المسؤولية",
            "pdf_fallback_summary_p1": "يوضح هذا المستند {total_cases} حالة استخدام تحليلية عالية القيمة تم تحديدها لـ {business_name}. هذه السيناريوهات، مدعومة بـ داتا بريكس جيني كود، مصممة لتحقيق نتائج أعمال مهمة من خلال الاستفادة من أصول البيانات الحالية.",
            "pdf_fallback_summary_p2": "توفر الصفحات التالية تفصيلاً مفصلاً لهذه الفرص، مصنفة حسب مجال الأعمال، للمساعدة في تحديد أولويات مبادرات الذكاء الاصطناعي الخاصة بك.",
            "pptx_main_title": "حالات استخدام الذكاء الاصطناعي الاستراتيجية مدعومة بـ داتا بريكس جيني كود",
            "pptx_for": "لـ",
            "pptx_disclaimer_title": "إخلاء المسؤولية",
            "pptx_domain_suffix": "حالات الاستخدام",
            "elite_verdict": "الحكم",
            "elite_decision": "القرار",
            "elite_decision_owner": "صاحب القرار",
            "elite_kpi": "مؤشر الأداء الرئيسي",
            "elite_value_driver": "محرك القيمة",
            "elite_evidence_columns": "أعمدة الأدلة",
            "elite_join_path": "مسار الربط",
            "elite_build_path": "مسار التنفيذ",
            "elite_red_team_note": "ملاحظة الفريق الأحمر",
            "elite_coverage_pick": "اختيار لتغطية المجال، دون المستوى المطلوب",
            "elite_fallback_pick": "دون مستوى {mode}",
            "elite_fallback_near_miss": "قريب من المعيار، دون مستوى {mode}",
            "doc_fallback_pick": "دون المعيار",
            "doc_fallback_notice": "لم تستوفِ أي حالة استخدام مستوى {mode}. هذه أقرب الحالات، وكلٌّ منها موسوم بأنه دون المستوى.",
            "elite_reason": "السبب",
            "untranslated_notice": "معروض باللغة الإنجليزية (الترجمة غير متاحة)",
            "more_use_cases": "...و{count} حالات استخدام أخرى.",
            "executive_summary_not_available": "الملخص التنفيذي غير متوفر.",
            "domain_summary_not_available": "ملخص المجال غير متوفر.",
            "summary_not_available": "الملخص غير متوفر.",
            "doc_catalog_kicker": "كتالوج حالات استخدام الذكاء الاصطناعي",
            "doc_prepared_for": "أُعدّ لـ",
            "doc_generated_on": "تاريخ الإنشاء",
            "doc_quality_mode": "وضع الجودة",
            "doc_use_cases": "حالات الاستخدام",
            "doc_domains": "مجالات الأعمال",
            "doc_elite_use_cases": "حالات الاستخدام النخبوية",
            "doc_high_or_better": "جودة عالية أو أعلى",
            "doc_best_score": "أفضل درجة إنسباير",
            "doc_not_scored": "غير مُقيَّم",
            "doc_coverage_picks": "اختيارات التغطية",
            "doc_coverage_pick": "اختيار تغطية",
            "value_correlation_analysis": "تحليل الارتباط",
            "value_pareto_analysis": "تحليل باريتو",
            "value_funnel_analysis": "تحليل مسار التحويل",
            "value_document_processing": "معالجة المستندات",
            "value_extraction": "الاستخراج",
            "value_ai_analysis": "تحليل بالذكاء الاصطناعي",
            "value_simulation": "المحاكاة",
            "value_geospatial_analysis": "التحليل الجغرافي المكاني",
            "value_predictive_modeling": "النمذجة التنبؤية",
            "value_clustering": "التجميع",
            "value_regression_analysis": "تحليل الانحدار",
            "value_survival_analysis": "تحليل البقاء",
            "value_network_analysis": "تحليل الشبكات",
            "value_mode_good": "جيد",
            "value_mode_high": "عالٍ",
            "value_mode_ultra_high": "عالٍ للغاية",
            "doc_exec_kicker": "الفرصة",
            "doc_guide_kicker": "دليل القراءة",
            "doc_how_to_read": "كيفية قراءة هذا الكتالوج",
            "doc_guide_lead": "تنبع كل حالة استخدام من البيانات الوصفية الخاصة بالمؤسسة، وتُقيَّم على مقياس مطلق واحد. استخدم الدرجات للمقارنة، والمستويات للتصفية، وصفحات المجالات للتنقل.",
            "doc_inspire_score_title": "درجة إنسباير، من 0 إلى 10",
            "doc_inspire_score_body": "تجمع بين جودة حالة الاستخدام وأولويتها في الأعمال. الدرجة الأعلى أفضل، ويُرتَّب الكتالوج وفقًا لها.",
            "doc_quality_tiers_title": "مستويات الجودة",
            "doc_quality_tiers_body": "تحصل كل حالة استخدام على مستوى يتراوح من {lowest} إلى {highest}، يعكس مدى تحديدها وقيمتها وقابلية بنائها بالبيانات المتاحة.",
            "doc_modes_title": "أوضاع الجودة",
            "doc_mode_good_body": "يحتفظ بحالات الاستخدام المصنفة {medium} أو أعلى.",
            "doc_mode_high_body": "يحتفظ فقط بحالات الاستخدام المصنفة {high} أو أعلى.",
            "doc_mode_ultra_body": "يحتفظ فقط بحالات الاستخدام التي تتجاوز معيارًا مطلقًا في القيمة والأدلة وقابلية التنفيذ وقابلية التطبيق العملي. لا توجد حصة محددة.",
            "doc_this_catalog": "هذا الكتالوج",
            "doc_elite_score_title": "درجة النخبة",
            "doc_elite_score_body": "تُستخدم في وضع {ultra}. تَزِن القيمة التجارية وقوة الأدلة وقابلية التنفيذ وقابلية التطبيق العملي، وتحقق حالة الاستخدام النخبوية {min_score} من 10 على الأقل في كل منها.",
            "doc_coverage_title": "اختيارات تغطية المجالات",
            "doc_coverage_body": "عندما لا يحتوي مجال أعمال مطلوب على أي حالة استخدام فوق المعيار، يُحتفظ بأقوى مرشح فيه ويُوسم بأنه اختيار تغطية دون المعيار.",
            "doc_portfolio_kicker": "المحفظة",
            "doc_portfolio_glance": "المحفظة في لمحة",
            "doc_per_domain": "حالات الاستخدام لكل مجال",
            "doc_quality_distribution": "توزيع الجودة",
            "doc_top_use_cases": "أبرز حالات الاستخدام",
            "doc_rank": "الترتيب",
            "doc_domain_label": "المجال",
            "doc_domain_overview": "نبذة عن هذا المجال",
            "doc_what_it_covers": "ما يغطيه",
            "doc_top_opportunities": "أبرز الفرص",
            "doc_domain_fallback": "يجمع هذا المجال حالات استخدام عبر {subdomains}، وعددها {count}. تظهر أقوى فرصه أدناه، مع تفاصيلها في الصفحات التالية.",
            "doc_open_notebook": "افتح دفتر جيني",
            "doc_elite_score": "درجة النخبة",
            "doc_best_of_best": "الأفضل على الإطلاق",
            "doc_appendix": "الملحق",
            "doc_methodology": "المنهجية",
            "strategy_path": "مسار الاستراتيجية",
            "kpis": "مؤشرات الأداء الرئيسية",
            "expected_key_results": "النتائج الرئيسية المتوقعة",
            "measures_of_success": "مقاييس النجاح",
            "financial_benefits": "الفوائد المالية",
            "non_financial_benefits": "الفوائد غير المالية",
            "foundations": "أسس البيانات",
            "doc_foundations_title": "أسس البيانات المشتركة",
            "doc_foundations_lead": "الجداول والروابط التي تحتاجها حالتا استخدام أو أكثر. بناؤها أولاً يتيح أكبر عدد من حالات الاستخدام. الترتيب تسلسل وليس جدولاً زمنياً.",
            "doc_foundation_tables": "الجداول",
            "doc_foundation_unlocks": "حالات الاستخدام المتاحة",
            "doc_data_access_standard": "الاكتشاف القياسي: لم يقرأ Inspire أي بيانات للعميل. استخدم البيانات الوصفية وتعريفات طرق عرض المقاييس وأعداد الاستخدام فقط.",
            "doc_data_access_deep": "الاكتشاف العميق: قرأ Inspire تجميعات من الجداول المشمولة، وعددها {tables} (أعداد الصفوف، التغطية الزمنية، نسب القيم الفارغة، أعداد القيم المميزة وقوائم قصيرة لقيم الأعمدة غير الشخصية). لم تُرسل أي صفوف بيانات إلى الذكاء الاصطناعي.",
            "doc_method_1_deep": "يقرأ Inspire البيانات الوصفية، وفي الاكتشاف العميق يقرأ أيضاً تجميعات محسوبة على الجداول. لا يرسل أبداً صفوف البيانات إلى الذكاء الاصطناعي.",
            "doc_guide_lead_deep": "كل حالة استخدام مستمدة من البيانات الوصفية للمؤسسة وملفات تعريف بياناتها، ويتم تقييمها على مقياس مطلق واحد. استخدم الدرجات للمقارنة والفئات للتصفية وصفحات المجالات للتنقل.",
            "value_not_aligned": "غير متوافق",
            "kpi_baseline": "خط الأساس {value}، آخر {days} يومًا، تم القياس في {date}",
            "kpi_baseline_undated": "خط الأساس {value}، تم القياس في {date}",
            "doc_method_1": "يقرأ إنسباير البيانات الوصفية فقط: أسماء الجداول والأعمدة وأنواعها والتعليقات والمفاتيح والوسوم. ولا يقرأ بيانات الصفوف أبدًا.",
            "doc_method_2": "يستخلص سياق الأعمال، ثم يُولّد حالات استخدام مرشحة لكل مجال أعمال تدعمه البيانات.",
            "doc_method_3": "تُزال الحالات المكررة، ويُقيَّم كل مرشح من حيث القيمة والجدوى، وتستبعد بوابات الجودة الأفكار الضعيفة أو غير المدعومة.",
            "doc_method_4": "في وضع {ultra}، يراجع مُحكِّم النخبة كل مرشح متبقٍ، وتتحقق فحوص حتمية من وجود كل جدول وعمود مذكور.",
            "doc_method_5": "تُرتَّب القائمة النهائية وتُجمَّع حسب مجال الأعمال، وتُقرن عند الاختيار بدفتر داتا بريكس جيني كود.",
            "doc_glossary": "مسرد المصطلحات",
            "doc_gl_domain": "مجال أعمال يجمع حالات الاستخدام المترابطة.",
            "doc_gl_subdomain": "موضوع أضيق داخل مجال الأعمال.",
            "doc_gl_technique": "الأسلوب التحليلي، مثل التنبؤ أو التصنيف.",
            "doc_gl_inspire": "الدرجة من 0 إلى 10 التي تجمع بين الجودة وأولوية الأعمال.",
            "doc_gl_quality": "المستوى من {lowest} إلى {highest} الممنوح لكل حالة استخدام.",
            "doc_gl_elite": "درجة وضع {ultra} عبر القيمة والأدلة وقابلية التنفيذ وقابلية التطبيق العملي.",
            "doc_gl_coverage": "حالة استخدام دون المعيار يُحتفظ بها لتمثيل مجال مطلوب.",
            "doc_gl_notebook": "دفتر داتا بريكس يتضمن تعليمات جيني كود لبناء حالة الاستخدام.",
            "doc_notebook_term": "دفتر جيني",
            "doc_next_steps": "الخطوات التالية",
            "doc_closing_title": "من حالات الاستخدام إلى النتائج",
            "doc_closing_body": "اختر حالات الاستخدام الأهم، وافتح دفاتر جيني الخاصة بها، وابنِ أولها مع الفريق. الخطوات ترتيب مقترح، والفريق هو من يحدد الوتيرة.",
            "doc_step_select": "الاختيار",
            "doc_step_select_body": "اختر حالات الاستخدام ذات الدرجة الأقوى والراعي الواضح.",
            "doc_step_validate": "التحقق من البيانات",
            "doc_step_validate_body": "أكّد الجداول المعنية والروابط بينها مع مالكي البيانات.",
            "doc_step_build": "البناء باستخدام جيني كود",
            "doc_step_build_body": "افتح دفتر جيني وأنشئ أول نسخة عاملة.",
            "doc_step_schedule": "الجدولة",
            "doc_step_schedule_body": "شغّل الدفتر كمهمة داتا بريكس تُحدّث جدول النتائج.",
            "doc_step_measure": "القياس والتوسع",
            "doc_step_measure_body": "تابع القيمة التجارية التي تحققها، ثم انتقل إلى حالة الاستخدام التالية.",
            "doc_agenda": "جدول الأعمال",
            "doc_roadmap": "خارطة الطريق والخطوات التالية",
            "doc_use_case_index": "فهرس حالات الاستخدام",
            "doc_continued": "تتمة",
            "doc_not_provided": "غير متوفر",
            "doc_bob_score": "درجة الأفضل بين الأفضل",
            "elite_cadence": "وتيرة القرار",
            "value_cadence_hourly": "كل ساعة",
            "value_cadence_daily": "يومي",
            "value_cadence_weekly": "أسبوعي",
            "value_cadence_biweekly": "كل أسبوعين",
            "value_cadence_monthly": "شهري",
            "value_cadence_quarterly": "ربع سنوي",
            "value_cadence_yearly": "سنوي",
            "value_cadence_real_time": "فوري",
            "value_cadence_per_event": "عند كل حدث",
            "doc_generated_by": "تم الإنشاء بواسطة {brand} {version} · {date}",
            "doc_disclaimer_ai": "حالات الاستخدام والدرجات والنصوص الواردة في هذا المستند مولدة بالذكاء الاصطناعي. تحقق منها مع مالكي البيانات والرعاة من الأعمال قبل اتخاذ أي قرار.",
            "doc_disclaimer_figures": "الدرجات والقيم والنتائج تقديرات لغرض تحديد الأولويات فقط. لا يوجد أي رقم مضمون في هذا المستند.",
            "doc_contents": "المحتويات",
            "doc_index_lead": "جميع حالات الاستخدام البالغ عددها {count} مرتبة حسب الترتيب النهائي.",
            "doc_deck_scope": "يعرض هذا العرض تفاصيل حالات الاستخدام الأعلى ترتيبًا، وعددها {count} من أصل {total}. ويعرض كتالوج PDF تفاصيلها جميعًا.",
            "doc_deck_scope_all": "يعرض هذا العرض تفاصيل جميع حالات الاستخدام البالغ عددها {total}، لكل حالة بطاقة.",
            "value_general_improvement": "تحسين عام",
            "value_reduce_cost": "تقليل التكلفة",
            "value_increase_revenue": "زيادة الإيرادات",
            "value_boost_productivity": "تعزيز الإنتاجية",
            "value_mitigate_risk": "تخفيف المخاطر",
            "value_protect_revenue": "حماية الإيرادات",
            "value_align_to_regulations": "الامتثال للوائح",
            "value_improve_customer_experience": "تحسين تجربة العملاء",
            "value_enable_data_driven_decisions": "تمكين القرارات المبنية على البيانات",
            "value_optimize_operations": "تحسين العمليات",
            "value_empower_talent": "تمكين المواهب",
            "value_enhance_experience": "تعزيز التجربة",
            "value_drive_innovation": "دفع الابتكار",
            "value_achieve_esg": "تحقيق ESG",
            "value_execute_strategy": "تنفيذ الاستراتيجية",
            "value_forecasting": "التنبؤ",
            "value_classification": "التصنيف",
            "value_anomaly_detection": "كشف الشذوذ",
            "value_cohort_analysis": "تحليل الأتراب",
            "value_segmentation": "التجزئة",
            "value_sentiment_analysis": "تحليل المشاعر",
            "value_trend_analysis": "تحليل الاتجاهات",
            "value_prescriptive_analytics": "التحليلات التوجيهية",
            "value_root_cause_analysis": "تحليل السبب الجذري",
            "value_optimization": "التحسين",
            "value_recommendation": "التوصية",
            "value_time_series_analysis": "تحليل السلاسل الزمنية",
            "value_predictive_analytics": "التحليلات التنبؤية",
            "value_descriptive_analytics": "التحليلات الوصفية",
        },
        "Spanish": {
            "main_title": "Generador de Casos de Uso Impulsado por Databricks Genie Code",
            "intro": "Este cuaderno contiene casos de uso generados por IA basados en sus esquemas. A continuación se muestra un resumen de los escenarios generados por dominio de negocio.",
            "domain": "Dominio de Negocio",
            "total": "Total de Casos de Uso",
            "summaries": "Resúmenes de Casos de Uso",
            "sum_id": "ID",
            "sum_name": "Nombre",
            "sum_value": "Valor Comercial",
            "sum_outcome": "Resultado Esperado",
            "warning_header": "ADVERTENCIA",
            "warning_body": "No ejecute este cuaderno. Está destinado solo para demostración y catalogación.",
            "disclaimer": "Este contenido es generado por IA y debe ser revisado por un ingeniero calificado antes de su uso en producción. Databricks no es responsable de ningún problema derivado del uso de este contenido.",
            "detailed_scenarios": "Detalles de Casos de Uso",
            "aspect": "Aspecto",
            "description": "Descripción",
            "aspect_domain": "Dominio de Negocio",
            "type": "Tipo",
            "analytics_technique": "Técnica de Análisis",
            "primary_table": "Tabla Principal",
            "quality": "Calidad",
            "priority": "Prioridad",
            "usecase": "Caso de Uso",
            "description_label": "Descripción",
            "value_type_problem": "Problema",
            "value_type_risk": "Riesgo",
            "value_type_opportunity": "Oportunidad",
            "value_type_improvement": "Mejora",
            "value_priority_ultra_high": "Extremadamente Alta",
            "value_priority_very_high": "Muy Alta",
            "value_priority_high": "Alta",
            "value_priority_medium": "Media",
            "value_priority_low": "Baja",
            "value_priority_very_low": "Muy Baja",
            "value_priority_ultra_low": "Extremadamente Baja",
            "statement": "Declaración",
            "solution": "Solución",
            "aspect_beneficiary": "Beneficiario",
            "beneficiary": "Beneficiario",
            "aspect_sponsor": "Patrocinador",
            "sponsor": "Patrocinador",
            "business_priority_alignment": "Alineación de Prioridad Empresarial",
            "strategic_goals_alignment": "Alineación con Objetivos Estratégicos",
            "subdomain": "Subdominio",
            "aspect_value": "Valor Comercial",
            "business_value": "Valor Comercial",
            "aspect_tables": "Tablas Involucradas",
            "quality_reasons": "Razones de Calidad",
            "aspect_ai_function": "Función de IA",
            "aspect_analytics_technique": "Técnica de Análisis",
            "aspect_primary_table": "Tabla Principal",
            "aspect_quality": "Calidad",
            "aspect_priority": "Prioridad",
            "strategic_alignment": "Alineación Estratégica",
            "return_on_investment": "Retorno de Inversión",
            "reusability": "Reusabilidad",
            "time_to_value": "Tiempo al Valor",
            "data_availability": "Disponibilidad de Datos",
            "data_accessibility": "Accesibilidad de Datos",
            "architecture_fitness": "Aptitud de Arquitectura",
            "team_skills": "Habilidades del Equipo",
            "domain_knowledge": "Conocimiento del Dominio",
            "people_allocation": "Asignación de Personal",
            "budget_allocation": "Asignación de Presupuesto",
            "time_to_production": "Tiempo a Producción",
            "value_score": "Puntuación de Valor",
            "feasibility_score": "Puntuación de Viabilidad",
            "priority_score": "Puntuación de Prioridad",
            "inspire_score": "Puntuación Inspire",
            "quality_score_header": "Puntuación de Calidad",
            "pdf_title": "Casos de Uso de IA Estratégica Impulsados por Databricks Genie Code",
            "pdf_for": "Para",
            "pdf_exec_summary": "Resumen Ejecutivo",
            "pdf_toc_title": "Dominios de Casos de Uso",
            "pdf_detailed_view": "Catálogo Detallado de Casos de Uso",
            "pdf_disclaimer_title": "Descargo de Responsabilidad",
            "pdf_fallback_summary_p1": "Este documento describe {total_cases} casos de uso analíticos de alto valor identificados para {business_name}.",
            "pdf_fallback_summary_p2": "Las siguientes páginas proporcionan un desglose detallado de estas oportunidades, categorizadas por dominio de negocio.",
            "pptx_main_title": "Casos de Uso de IA Estratégica Impulsados por Databricks Genie Code",
            "pptx_for": "Para",
            "pptx_disclaimer_title": "Descargo de Responsabilidad",
            "pptx_domain_suffix": "Casos de Uso",
            "elite_verdict": "Veredicto",
            "elite_decision": "Decisión",
            "elite_decision_owner": "Responsable de la decisión",
            "elite_kpi": "KPI",
            "elite_value_driver": "Impulsor de valor",
            "elite_evidence_columns": "Columnas de evidencia",
            "elite_join_path": "Ruta de unión",
            "elite_build_path": "Ruta de implementación",
            "elite_red_team_note": "Nota del equipo rojo",
            "elite_coverage_pick": "Selección por cobertura de dominio, por debajo del umbral",
            "elite_fallback_pick": "Por debajo del umbral {mode}",
            "elite_fallback_near_miss": "Casi seleccionado, por debajo del umbral {mode}",
            "doc_fallback_pick": "Bajo el umbral",
            "doc_fallback_notice": "Ningún caso de uso alcanzó el umbral {mode}. Estos son los candidatos más cercanos, cada uno etiquetado por debajo del umbral.",
            "elite_reason": "Motivo",
            "untranslated_notice": "Se muestra en inglés (traducción no disponible)",
            "more_use_cases": "...y {count} casos de uso más.",
            "executive_summary_not_available": "Resumen ejecutivo no disponible.",
            "domain_summary_not_available": "Resumen del dominio no disponible.",
            "summary_not_available": "Resumen no disponible.",
            "doc_catalog_kicker": "Catálogo de casos de uso de IA",
            "doc_prepared_for": "Preparado para",
            "doc_generated_on": "Generado el",
            "doc_quality_mode": "Modo de calidad",
            "doc_use_cases": "Casos de uso",
            "doc_domains": "Dominios de negocio",
            "doc_elite_use_cases": "Casos de uso de élite",
            "doc_high_or_better": "Calidad alta o superior",
            "doc_best_score": "Mejor puntuación Inspire",
            "doc_not_scored": "Sin puntuar",
            "doc_coverage_picks": "Selecciones de cobertura",
            "doc_coverage_pick": "Selección de cobertura",
            "value_correlation_analysis": "Análisis de Correlación",
            "value_pareto_analysis": "Análisis de Pareto",
            "value_funnel_analysis": "Análisis de Embudo",
            "value_document_processing": "Procesamiento de Documentos",
            "value_extraction": "Extracción",
            "value_ai_analysis": "Análisis con IA",
            "value_simulation": "Simulación",
            "value_geospatial_analysis": "Análisis Geoespacial",
            "value_predictive_modeling": "Modelado Predictivo",
            "value_clustering": "Agrupamiento",
            "value_regression_analysis": "Análisis de Regresión",
            "value_survival_analysis": "Análisis de Supervivencia",
            "value_network_analysis": "Análisis de Redes",
            "value_mode_good": "Bueno",
            "value_mode_high": "Alto",
            "value_mode_ultra_high": "Ultra alto",
            "doc_exec_kicker": "La oportunidad",
            "doc_guide_kicker": "Guía de lectura",
            "doc_how_to_read": "Cómo leer este catálogo",
            "doc_guide_lead": "Cada caso de uso proviene de los metadatos de la propia organización y se puntúa en una única escala absoluta. Use las puntuaciones para comparar, los niveles para filtrar y las páginas de dominio para navegar.",
            "doc_inspire_score_title": "Puntuación Inspire, de 0 a 10",
            "doc_inspire_score_body": "Combina la calidad de un caso de uso con su prioridad de negocio. Cuanto más alta, mejor, y el catálogo se ordena por ella.",
            "doc_quality_tiers_title": "Niveles de calidad",
            "doc_quality_tiers_body": "Cada caso de uso recibe un nivel de {lowest} a {highest} que refleja lo específico, valioso y construible que es con los datos disponibles.",
            "doc_modes_title": "Modos de calidad",
            "doc_mode_good_body": "Conserva los casos de uso calificados como {medium} o superior.",
            "doc_mode_high_body": "Conserva solo los casos de uso calificados como {high} o superior.",
            "doc_mode_ultra_body": "Conserva solo los casos de uso que superan un umbral absoluto de valor, evidencia, viabilidad de implementación y accionabilidad. No hay cuota.",
            "doc_this_catalog": "Este catálogo",
            "doc_elite_score_title": "Puntuación de élite",
            "doc_elite_score_body": "Se usa en el modo {ultra}. Pondera el valor de negocio, la solidez de la evidencia, la viabilidad de implementación y la accionabilidad, y un caso de uso de élite obtiene al menos {min_score} de 10 en cada uno.",
            "doc_coverage_title": "Selecciones por cobertura de dominio",
            "doc_coverage_body": "Cuando un dominio de negocio solicitado no tiene ningún caso de uso por encima del umbral, se conserva su candidato más sólido y se etiqueta como selección de cobertura por debajo del umbral.",
            "doc_portfolio_kicker": "Cartera",
            "doc_portfolio_glance": "La cartera de un vistazo",
            "doc_per_domain": "Casos de uso por dominio",
            "doc_quality_distribution": "Distribución de calidad",
            "doc_top_use_cases": "Principales casos de uso",
            "doc_rank": "Posición",
            "doc_domain_label": "Dominio",
            "doc_domain_overview": "Sobre este dominio",
            "doc_what_it_covers": "Qué abarca",
            "doc_top_opportunities": "Principales oportunidades",
            "doc_domain_fallback": "Este dominio reúne casos de uso en {subdomains}, {count} en total. Sus oportunidades más sólidas se enumeran a continuación y se detallan en las páginas siguientes.",
            "doc_open_notebook": "Abrir el notebook de Genie",
            "doc_elite_score": "Puntuación de élite",
            "doc_best_of_best": "Lo mejor de lo mejor",
            "doc_appendix": "Apéndice",
            "doc_methodology": "Metodología",
            "strategy_path": "Ruta de la estrategia",
            "kpis": "KPI",
            "expected_key_results": "Resultados clave esperados",
            "measures_of_success": "Medidas de éxito",
            "financial_benefits": "Beneficios financieros",
            "non_financial_benefits": "Beneficios no financieros",
            "foundations": "Fundamentos de datos",
            "doc_foundations_title": "Fundamentos de datos compartidos",
            "doc_foundations_lead": "Tablas y uniones que necesitan dos o más casos de uso. Construirlas primero habilita la mayor cantidad de casos de uso. El orden es una secuencia, no un calendario.",
            "doc_foundation_tables": "Tablas",
            "doc_foundation_unlocks": "Casos de uso habilitados",
            "doc_data_access_standard": "Descubrimiento estándar: Inspire no leyó datos del cliente. Usó solo metadatos, definiciones de vistas de métricas y recuentos de uso.",
            "doc_data_access_deep": "Descubrimiento profundo: Inspire leyó agregados de las tablas del alcance, {tables} en total (recuentos de filas, cobertura de fechas, tasas de nulos, recuentos de valores distintos y listas cortas de valores de columnas no personales). No se envió ninguna fila de datos a la IA.",
            "doc_method_1_deep": "Inspire lee metadatos y, en el descubrimiento profundo, también agregados calculados sobre las tablas. Nunca envía filas de datos a la IA.",
            "doc_guide_lead_deep": "Cada caso de uso procede de los metadatos y los perfiles de datos de la organización y se puntúa en una única escala absoluta. Use las puntuaciones para comparar, los niveles para filtrar y las páginas de dominio para navegar.",
            "value_not_aligned": "No alineado",
            "kpi_baseline": "línea base {value}, últimos {days} días, medida el {date}",
            "kpi_baseline_undated": "línea base {value}, medida el {date}",
            "doc_method_1": "Inspire solo lee metadatos: nombres de tablas y columnas, tipos, comentarios, claves y etiquetas. Nunca lee datos de filas.",
            "doc_method_2": "Deriva el contexto de negocio y luego genera casos de uso candidatos para cada dominio de negocio que los datos respaldan.",
            "doc_method_3": "Se eliminan los duplicados, cada candidato se puntúa por valor y viabilidad, y los filtros de calidad descartan las ideas débiles o sin respaldo.",
            "doc_method_4": "En el modo {ultra}, un juez de élite revisa cada candidato restante y comprobaciones deterministas confirman que existe cada tabla y columna citada.",
            "doc_method_5": "La lista final se ordena, se agrupa por dominio de negocio y, cuando se selecciona, se acompaña de un notebook de Databricks Genie Code.",
            "doc_glossary": "Glosario",
            "doc_gl_domain": "Un área de negocio que agrupa casos de uso relacionados.",
            "doc_gl_subdomain": "Un tema más específico dentro de un dominio de negocio.",
            "doc_gl_technique": "El método analítico, por ejemplo pronóstico o clasificación.",
            "doc_gl_inspire": "La puntuación de 0 a 10 que combina calidad y prioridad de negocio.",
            "doc_gl_quality": "El nivel de {lowest} a {highest} asignado a cada caso de uso.",
            "doc_gl_elite": "La puntuación del modo {ultra} sobre valor, evidencia, viabilidad de implementación y accionabilidad.",
            "doc_gl_coverage": "Un caso de uso conservado por debajo del umbral para que un dominio solicitado esté representado.",
            "doc_gl_notebook": "Un notebook de Databricks con instrucciones de Genie Code para construir el caso de uso.",
            "doc_notebook_term": "Notebook de Genie",
            "doc_next_steps": "Próximos pasos",
            "doc_closing_title": "De los casos de uso a los resultados",
            "doc_closing_body": "Elija los casos de uso más importantes, abra sus notebooks de Genie y construya el primero con el equipo. Los pasos son un orden sugerido; el equipo marca el ritmo.",
            "doc_step_select": "Seleccionar",
            "doc_step_select_body": "Elija los casos de uso con la mejor puntuación y un patrocinador claro.",
            "doc_step_validate": "Validar los datos",
            "doc_step_validate_body": "Confirme las tablas involucradas y sus uniones con los responsables de los datos.",
            "doc_step_build": "Construir con Genie Code",
            "doc_step_build_body": "Abra el notebook de Genie y genere la primera versión funcional.",
            "doc_step_schedule": "Programar",
            "doc_step_schedule_body": "Ejecute el notebook como un job de Databricks que actualiza la tabla de resultados.",
            "doc_step_measure": "Medir y escalar",
            "doc_step_measure_body": "Haga seguimiento del valor de negocio que aporta y luego extienda el trabajo al siguiente caso de uso.",
            "doc_agenda": "Agenda",
            "doc_roadmap": "Hoja de ruta y próximos pasos",
            "doc_use_case_index": "Índice de casos de uso",
            "doc_continued": "continuación",
            "doc_not_provided": "No disponible",
            "doc_bob_score": "Puntuación Best-of-Best",
            "elite_cadence": "Frecuencia de la decisión",
            "value_cadence_hourly": "Cada hora",
            "value_cadence_daily": "Diaria",
            "value_cadence_weekly": "Semanal",
            "value_cadence_biweekly": "Cada dos semanas",
            "value_cadence_monthly": "Mensual",
            "value_cadence_quarterly": "Trimestral",
            "value_cadence_yearly": "Anual",
            "value_cadence_real_time": "En tiempo real",
            "value_cadence_per_event": "Por evento",
            "doc_generated_by": "Generado por {brand} {version} · {date}",
            "doc_disclaimer_ai": "Los casos de uso, las puntuaciones y los textos de este documento están generados por IA. Valídelos con los responsables de los datos y los patrocinadores de negocio antes de tomar cualquier decisión.",
            "doc_disclaimer_figures": "Las puntuaciones, los valores y los resultados son estimaciones solo para priorizar. Ninguna cifra de este documento está garantizada.",
            "doc_contents": "Contenido",
            "doc_index_lead": "Todos los casos de uso en orden de clasificación final, {count} en total.",
            "doc_deck_scope": "Esta presentación detalla los casos de uso mejor clasificados, {count} de {total}. El catálogo PDF los detalla todos.",
            "doc_deck_scope_all": "Esta presentación detalla todos los casos de uso, {total} en total, con una tarjeta para cada uno.",
            "value_general_improvement": "Mejora General",
            "value_reduce_cost": "Reducir Costos",
            "value_increase_revenue": "Aumentar Ingresos",
            "value_boost_productivity": "Impulsar Productividad",
            "value_mitigate_risk": "Mitigar Riesgos",
            "value_protect_revenue": "Proteger Ingresos",
            "value_align_to_regulations": "Cumplir Regulaciones",
            "value_improve_customer_experience": "Mejorar Experiencia del Cliente",
            "value_enable_data_driven_decisions": "Habilitar Decisiones Basadas en Datos",
            "value_optimize_operations": "Optimizar Operaciones",
            "value_empower_talent": "Empoderar Talento",
            "value_enhance_experience": "Mejorar Experiencia",
            "value_drive_innovation": "Impulsar Innovación",
            "value_achieve_esg": "Lograr ESG",
            "value_execute_strategy": "Ejecutar Estrategia",
            "value_forecasting": "Pronóstico",
            "value_classification": "Clasificación",
            "value_anomaly_detection": "Detección de Anomalías",
            "value_cohort_analysis": "Análisis de Cohortes",
            "value_segmentation": "Segmentación",
            "value_sentiment_analysis": "Análisis de Sentimiento",
            "value_trend_analysis": "Análisis de Tendencias",
            "value_prescriptive_analytics": "Analítica Prescriptiva",
            "value_root_cause_analysis": "Análisis de Causa Raíz",
            "value_optimization": "Optimización",
            "value_recommendation": "Recomendación",
            "value_time_series_analysis": "Análisis de Series Temporales",
            "value_predictive_analytics": "Analítica Predictiva",
            "value_descriptive_analytics": "Analítica Descriptiva",
        },
        "French": {
            "main_title": "Générateur de Cas d'Utilisation Propulsé par Databricks Genie Code",
            "intro": "Ce carnet contient des cas d'utilisation générés par l'IA basés sur vos schémas. Voici un résumé des scénarios générés par domaine d'activité.",
            "domain": "Domaine d'Activité",
            "total": "Total des Cas d'Utilisation",
            "summaries": "Résumés des Cas d'Utilisation",
            "sum_id": "ID",
            "sum_name": "Nom",
            "sum_value": "Valeur Commerciale",
            "sum_outcome": "Résultat Attendu",
            "warning_header": "AVERTISSEMENT",
            "warning_body": "N'exécutez pas ce carnet. Il est destiné uniquement à la démonstration et au catalogage.",
            "disclaimer": "Ce contenu est généré par l'IA et doit être examiné par un ingénieur qualifié avant toute mise en production. Databricks n'est pas responsable des problèmes découlant de l'utilisation de ce contenu.",
            "detailed_scenarios": "Détails des Cas d'Utilisation",
            "aspect": "Aspect",
            "description": "Description",
            "aspect_domain": "Domaine d'Activité",
            "type": "Type",
            "analytics_technique": "Technique d'Analyse",
            "primary_table": "Table Principale",
            "quality": "Qualité",
            "priority": "Priorité",
            "usecase": "Cas d'Utilisation",
            "description_label": "Description",
            "value_type_problem": "Problème",
            "value_type_risk": "Risque",
            "value_type_opportunity": "Opportunité",
            "value_type_improvement": "Amélioration",
            "value_priority_ultra_high": "Extrêmement Haute",
            "value_priority_very_high": "Très Haute",
            "value_priority_high": "Haute",
            "value_priority_medium": "Moyenne",
            "value_priority_low": "Basse",
            "value_priority_very_low": "Très Basse",
            "value_priority_ultra_low": "Extrêmement Basse",
            "statement": "Énoncé",
            "solution": "Solution",
            "aspect_beneficiary": "Bénéficiaire",
            "beneficiary": "Bénéficiaire",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Alignement de Priorité d'Entreprise",
            "strategic_goals_alignment": "Alignement aux Objectifs Stratégiques",
            "subdomain": "Sous-domaine",
            "aspect_value": "Valeur Commerciale",
            "business_value": "Valeur Commerciale",
            "aspect_tables": "Tables Impliquées",
            "quality_reasons": "Raisons de Qualité",
            "aspect_ai_function": "Fonction IA",
            "aspect_analytics_technique": "Technique d'Analyse",
            "aspect_primary_table": "Table Principale",
            "aspect_quality": "Qualité",
            "aspect_priority": "Priorité",
            "strategic_alignment": "Alignement Stratégique",
            "return_on_investment": "Retour sur Investissement",
            "reusability": "Réutilisabilité",
            "time_to_value": "Délai de Valorisation",
            "data_availability": "Disponibilité des Données",
            "data_accessibility": "Accessibilité des Données",
            "architecture_fitness": "Adéquation Architecturale",
            "team_skills": "Compétences de l'Équipe",
            "domain_knowledge": "Connaissance du Domaine",
            "people_allocation": "Allocation du Personnel",
            "budget_allocation": "Allocation Budgétaire",
            "time_to_production": "Délai de Production",
            "value_score": "Score de Valeur",
            "feasibility_score": "Score de Faisabilité",
            "priority_score": "Score de Priorité",
            "inspire_score": "Score Inspire",
            "quality_score_header": "Score de Qualité",
            "pdf_title": "Cas d'Utilisation IA Stratégiques Propulsés par Databricks Genie Code",
            "pdf_for": "Pour",
            "pdf_exec_summary": "Résumé Exécutif",
            "pdf_toc_title": "Domaines des Cas d'Utilisation",
            "pdf_detailed_view": "Catalogue Détaillé des Cas d'Utilisation",
            "pdf_disclaimer_title": "Avertissement",
            "pdf_fallback_summary_p1": "Ce document présente {total_cases} cas d'utilisation analytiques à forte valeur identifiés pour {business_name}.",
            "pdf_fallback_summary_p2": "Les pages suivantes fournissent une répartition détaillée de ces opportunités, classées par domaine d'activité.",
            "pptx_main_title": "Cas d'Utilisation IA Stratégiques Propulsés par Databricks Genie Code",
            "pptx_for": "Pour",
            "pptx_disclaimer_title": "Avertissement",
            "pptx_domain_suffix": "Cas d'Utilisation",
            "elite_verdict": "Verdict",
            "elite_decision": "Décision",
            "elite_decision_owner": "Responsable de la décision",
            "elite_kpi": "KPI",
            "elite_value_driver": "Levier de valeur",
            "elite_evidence_columns": "Colonnes de preuve",
            "elite_join_path": "Chemin de jointure",
            "elite_build_path": "Chemin de mise en œuvre",
            "elite_red_team_note": "Note de l'équipe rouge",
            "elite_coverage_pick": "Sélection pour couverture du domaine, sous le seuil",
            "elite_fallback_pick": "Sous le seuil {mode}",
            "elite_fallback_near_miss": "Presque retenu, sous le seuil {mode}",
            "doc_fallback_pick": "Sous le seuil",
            "doc_fallback_notice": "Aucun cas d'usage n'a atteint le seuil {mode}. Voici les candidats les plus proches, chacun signalé sous le seuil.",
            "elite_reason": "Raison",
            "untranslated_notice": "Affiché en anglais (traduction indisponible)",
            "more_use_cases": "...et {count} autres cas d'usage.",
            "executive_summary_not_available": "Résumé exécutif non disponible.",
            "domain_summary_not_available": "Résumé du domaine non disponible.",
            "summary_not_available": "Résumé non disponible.",
            "doc_catalog_kicker": "Catalogue de cas d'usage IA",
            "doc_prepared_for": "Préparé pour",
            "doc_generated_on": "Généré le",
            "doc_quality_mode": "Mode de qualité",
            "doc_use_cases": "Cas d'usage",
            "doc_domains": "Domaines d'activité",
            "doc_elite_use_cases": "Cas d'usage d'élite",
            "doc_high_or_better": "Qualité haute ou supérieure",
            "doc_best_score": "Meilleur score Inspire",
            "doc_not_scored": "Non évalué",
            "doc_coverage_picks": "Sélections de couverture",
            "doc_coverage_pick": "Sélection de couverture",
            "value_correlation_analysis": "Analyse de Corrélation",
            "value_pareto_analysis": "Analyse de Pareto",
            "value_funnel_analysis": "Analyse d'Entonnoir",
            "value_document_processing": "Traitement de Documents",
            "value_extraction": "Extraction",
            "value_ai_analysis": "Analyse par IA",
            "value_simulation": "Simulation",
            "value_geospatial_analysis": "Analyse Géospatiale",
            "value_predictive_modeling": "Modélisation Prédictive",
            "value_clustering": "Regroupement",
            "value_regression_analysis": "Analyse de Régression",
            "value_survival_analysis": "Analyse de Survie",
            "value_network_analysis": "Analyse de Réseaux",
            "value_mode_good": "Bon",
            "value_mode_high": "Élevé",
            "value_mode_ultra_high": "Ultra élevé",
            "doc_exec_kicker": "L'opportunité",
            "doc_guide_kicker": "Guide de lecture",
            "doc_how_to_read": "Comment lire ce catalogue",
            "doc_guide_lead": "Chaque cas d'usage provient des métadonnées propres à l'organisation et reçoit un score sur une seule échelle absolue. Utilisez les scores pour comparer, les niveaux pour filtrer et les pages de domaine pour naviguer.",
            "doc_inspire_score_title": "Score Inspire, de 0 à 10",
            "doc_inspire_score_body": "Combine la qualité d'un cas d'usage et sa priorité métier. Plus il est élevé, mieux c'est, et le catalogue est trié selon ce score.",
            "doc_quality_tiers_title": "Niveaux de qualité",
            "doc_quality_tiers_body": "Chaque cas d'usage reçoit un niveau allant de {lowest} à {highest}, qui reflète sa précision, sa valeur et sa faisabilité avec les données disponibles.",
            "doc_modes_title": "Modes de qualité",
            "doc_mode_good_body": "Conserve les cas d'usage notés {medium} ou mieux.",
            "doc_mode_high_body": "Conserve uniquement les cas d'usage notés {high} ou mieux.",
            "doc_mode_ultra_body": "Conserve uniquement les cas d'usage qui franchissent un seuil absolu en valeur, preuves, faisabilité et actionnabilité. Il n'y a pas de quota.",
            "doc_this_catalog": "Ce catalogue",
            "doc_elite_score_title": "Score d'élite",
            "doc_elite_score_body": "Utilisé en mode {ultra}. Il pondère la valeur métier, la solidité des preuves, la faisabilité et l'actionnabilité, et un cas d'usage d'élite obtient au moins {min_score} sur 10 pour chacun.",
            "doc_coverage_title": "Sélections pour couverture de domaine",
            "doc_coverage_body": "Lorsqu'un domaine d'activité demandé n'a aucun cas d'usage au-dessus du seuil, son meilleur candidat est conservé et signalé comme sélection de couverture sous le seuil.",
            "doc_portfolio_kicker": "Portefeuille",
            "doc_portfolio_glance": "Le portefeuille en un coup d'œil",
            "doc_per_domain": "Cas d'usage par domaine",
            "doc_quality_distribution": "Répartition de la qualité",
            "doc_top_use_cases": "Principaux cas d'usage",
            "doc_rank": "Rang",
            "doc_domain_label": "Domaine",
            "doc_domain_overview": "À propos de ce domaine",
            "doc_what_it_covers": "Ce qu'il couvre",
            "doc_top_opportunities": "Principales opportunités",
            "doc_domain_fallback": "Ce domaine réunit des cas d'usage répartis sur {subdomains}, {count} au total. Ses meilleures opportunités sont listées ci-dessous et détaillées dans les pages suivantes.",
            "doc_open_notebook": "Ouvrir le notebook Genie",
            "doc_elite_score": "Score d'élite",
            "doc_best_of_best": "Le meilleur du meilleur",
            "doc_appendix": "Annexe",
            "doc_methodology": "Méthodologie",
            "strategy_path": "Chemin stratégique",
            "kpis": "Indicateurs clés (KPI)",
            "expected_key_results": "Résultats clés attendus",
            "measures_of_success": "Mesures de réussite",
            "financial_benefits": "Bénéfices financiers",
            "non_financial_benefits": "Bénéfices non financiers",
            "foundations": "Fondations de données",
            "doc_foundations_title": "Fondations de données partagées",
            "doc_foundations_lead": "Tables et jointures nécessaires à au moins deux cas d'usage. Les construire en premier débloque le plus de cas d'usage. L'ordre est une séquence, pas un calendrier.",
            "doc_foundation_tables": "Tables",
            "doc_foundation_unlocks": "Cas d'usage débloqués",
            "doc_data_access_standard": "Découverte standard : Inspire n'a lu aucune donnée client. Il a utilisé uniquement les métadonnées, les définitions des vues de métriques et les comptes d'utilisation.",
            "doc_data_access_deep": "Découverte approfondie : Inspire a lu des agrégats sur les tables du périmètre, {tables} au total (nombre de lignes, couverture temporelle, taux de valeurs nulles, nombres de valeurs distinctes et courtes listes de valeurs de colonnes non personnelles). Aucune ligne de données n'a été envoyée à l'IA.",
            "doc_method_1_deep": "Inspire lit les métadonnées et, en découverte approfondie, des agrégats calculés sur les tables. Il n'envoie jamais de lignes de données à l'IA.",
            "doc_guide_lead_deep": "Chaque cas d'usage provient des métadonnées et des profils de données de l'organisation et est noté sur une seule échelle absolue. Utilisez les scores pour comparer, les niveaux pour filtrer et les pages de domaine pour naviguer.",
            "value_not_aligned": "Non aligné",
            "kpi_baseline": "référence {value}, {days} derniers jours, mesurée le {date}",
            "kpi_baseline_undated": "référence {value}, mesurée le {date}",
            "doc_method_1": "Inspire lit uniquement les métadonnées : noms des tables et des colonnes, types, commentaires, clés et tags. Il ne lit jamais les données des lignes.",
            "doc_method_2": "Il déduit le contexte métier, puis génère des cas d'usage candidats pour chaque domaine d'activité que les données permettent de couvrir.",
            "doc_method_3": "Les doublons sont supprimés, chaque candidat est évalué sur sa valeur et sa faisabilité, et des contrôles de qualité écartent les idées faibles ou non étayées.",
            "doc_method_4": "En mode {ultra}, un juge d'élite examine chaque candidat restant et des contrôles déterministes confirment l'existence de chaque table et colonne citée.",
            "doc_method_5": "La liste finale est classée, regroupée par domaine d'activité et, si l'option est choisie, associée à un notebook Databricks Genie Code.",
            "doc_glossary": "Glossaire",
            "doc_gl_domain": "Un secteur d'activité qui regroupe des cas d'usage liés.",
            "doc_gl_subdomain": "Un thème plus précis au sein d'un domaine d'activité.",
            "doc_gl_technique": "La méthode analytique, par exemple la prévision ou la classification.",
            "doc_gl_inspire": "Le score de 0 à 10 qui combine qualité et priorité métier.",
            "doc_gl_quality": "Le niveau, de {lowest} à {highest}, attribué à chaque cas d'usage.",
            "doc_gl_elite": "Le score du mode {ultra} sur la valeur, les preuves, la faisabilité et l'actionnabilité.",
            "doc_gl_coverage": "Un cas d'usage conservé sous le seuil afin qu'un domaine demandé soit représenté.",
            "doc_gl_notebook": "Un notebook Databricks avec des instructions Genie Code pour construire le cas d'usage.",
            "doc_notebook_term": "Notebook Genie",
            "doc_next_steps": "Prochaines étapes",
            "doc_closing_title": "Des cas d'usage aux résultats",
            "doc_closing_body": "Choisissez les cas d'usage les plus importants, ouvrez leurs notebooks Genie et construisez le premier avec l'équipe. Les étapes suivent un ordre suggéré ; l'équipe fixe le rythme.",
            "doc_step_select": "Sélectionner",
            "doc_step_select_body": "Retenez les cas d'usage au meilleur score et au sponsor clairement identifié.",
            "doc_step_validate": "Valider les données",
            "doc_step_validate_body": "Confirmez les tables concernées et leurs jointures avec les responsables des données.",
            "doc_step_build": "Construire avec Genie Code",
            "doc_step_build_body": "Ouvrez le notebook Genie et générez une première version fonctionnelle.",
            "doc_step_schedule": "Planifier",
            "doc_step_schedule_body": "Exécutez le notebook comme un job Databricks qui actualise la table de résultats.",
            "doc_step_measure": "Mesurer et étendre",
            "doc_step_measure_body": "Suivez la valeur métier obtenue, puis passez au cas d'usage suivant.",
            "doc_agenda": "Ordre du jour",
            "doc_roadmap": "Feuille de route et prochaines étapes",
            "doc_use_case_index": "Index des cas d'usage",
            "doc_continued": "suite",
            "doc_not_provided": "Non renseigné",
            "doc_bob_score": "Score Best-of-Best",
            "elite_cadence": "Fréquence de décision",
            "value_cadence_hourly": "Toutes les heures",
            "value_cadence_daily": "Quotidienne",
            "value_cadence_weekly": "Hebdomadaire",
            "value_cadence_biweekly": "Toutes les deux semaines",
            "value_cadence_monthly": "Mensuelle",
            "value_cadence_quarterly": "Trimestrielle",
            "value_cadence_yearly": "Annuelle",
            "value_cadence_real_time": "En temps réel",
            "value_cadence_per_event": "À chaque événement",
            "doc_generated_by": "Généré par {brand} {version} · {date}",
            "doc_disclaimer_ai": "Les cas d'usage, les scores et les textes de ce document sont générés par l'IA. Validez-les avec les propriétaires des données et les sponsors métier avant toute décision.",
            "doc_disclaimer_figures": "Les scores, les valeurs et les résultats sont des estimations destinées uniquement à la priorisation. Aucun chiffre de ce document n'est garanti.",
            "doc_contents": "Sommaire",
            "doc_index_lead": "Tous les cas d'usage dans l'ordre du classement final, {count} au total.",
            "doc_deck_scope": "Cette présentation détaille les cas d'usage les mieux classés, {count} sur {total}. Le catalogue PDF les détaille tous.",
            "doc_deck_scope_all": "Cette présentation détaille tous les cas d'usage, {total} au total, avec une fiche pour chacun.",
            "value_general_improvement": "Amélioration Générale",
            "value_reduce_cost": "Réduire les Coûts",
            "value_increase_revenue": "Augmenter les Revenus",
            "value_boost_productivity": "Améliorer la Productivité",
            "value_mitigate_risk": "Atténuer les Risques",
            "value_protect_revenue": "Protéger les Revenus",
            "value_align_to_regulations": "Se Conformer aux Réglementations",
            "value_improve_customer_experience": "Améliorer l'Expérience Client",
            "value_enable_data_driven_decisions": "Permettre les Décisions Basées sur les Données",
            "value_optimize_operations": "Optimiser les Opérations",
            "value_empower_talent": "Autonomiser les Talents",
            "value_enhance_experience": "Améliorer l'Expérience",
            "value_drive_innovation": "Stimuler l'Innovation",
            "value_achieve_esg": "Atteindre ESG",
            "value_execute_strategy": "Exécuter la Stratégie",
            "value_forecasting": "Prévision",
            "value_classification": "Classification",
            "value_anomaly_detection": "Détection d'Anomalies",
            "value_cohort_analysis": "Analyse de Cohortes",
            "value_segmentation": "Segmentation",
            "value_sentiment_analysis": "Analyse de Sentiments",
            "value_trend_analysis": "Analyse des Tendances",
            "value_prescriptive_analytics": "Analytique Prescriptive",
            "value_root_cause_analysis": "Analyse des Causes Profondes",
            "value_optimization": "Optimisation",
            "value_recommendation": "Recommandation",
            "value_time_series_analysis": "Analyse de Séries Temporelles",
            "value_predictive_analytics": "Analytique Prédictive",
            "value_descriptive_analytics": "Analytique Descriptive",
        },
        "German": {
            "main_title": "Anwendungsfall-Generator Unterstützt von Databricks Genie Code",
            "intro": "Dieses Notebook enthält KI-generierte Anwendungsfälle basierend auf Ihren Schemas. Nachfolgend finden Sie eine Zusammenfassung der generierten Szenarien nach Geschäftsbereich.",
            "domain": "Geschäftsbereich",
            "total": "Gesamtzahl der Anwendungsfälle",
            "summaries": "Zusammenfassungen der Anwendungsfälle",
            "sum_id": "ID",
            "sum_name": "Name",
            "sum_value": "Geschäftswert",
            "sum_outcome": "Erwartetes Ergebnis",
            "warning_header": "WARNUNG",
            "warning_body": "Führen Sie dieses Notebook nicht aus. Es dient nur zur Demonstration und Katalogisierung.",
            "disclaimer": "Dieser Inhalt wurde von KI generiert und sollte von einem qualifizierten Ingenieur überprüft werden, bevor er in der Produktion eingesetzt wird. Databricks übernimmt keine Haftung für Probleme, die aus der Verwendung dieses Inhalts entstehen.",
            "detailed_scenarios": "Anwendungsfall-Details",
            "aspect": "Aspekt",
            "description": "Beschreibung",
            "aspect_domain": "Geschäftsbereich",
            "type": "Typ",
            "analytics_technique": "Analysetechnik",
            "primary_table": "Haupttabelle",
            "quality": "Qualität",
            "priority": "Priorität",
            "usecase": "Anwendungsfall",
            "description_label": "Beschreibung",
            "value_type_problem": "Problem",
            "value_type_risk": "Risiko",
            "value_type_opportunity": "Chance",
            "value_type_improvement": "Verbesserung",
            "value_priority_ultra_high": "Extrem Hoch",
            "value_priority_very_high": "Sehr Hoch",
            "value_priority_high": "Hoch",
            "value_priority_medium": "Mittel",
            "value_priority_low": "Niedrig",
            "value_priority_very_low": "Sehr Niedrig",
            "value_priority_ultra_low": "Extrem Niedrig",
            "statement": "Aussage",
            "solution": "Lösung",
            "aspect_beneficiary": "Begünstigter",
            "beneficiary": "Begünstigter",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Geschäftspriorität Ausrichtung",
            "strategic_goals_alignment": "Strategische Zielausrichtung",
            "subdomain": "Subdomäne",
            "aspect_value": "Geschäftswert",
            "business_value": "Geschäftswert",
            "aspect_tables": "Beteiligte Tabellen",
            "quality_reasons": "Qualitätsgründe",
            "aspect_ai_function": "KI-Funktion",
            "aspect_analytics_technique": "Analysetechnik",
            "aspect_primary_table": "Haupttabelle",
            "aspect_quality": "Qualität",
            "aspect_priority": "Priorität",
            "strategic_alignment": "Strategische Ausrichtung",
            "return_on_investment": "Kapitalrendite",
            "reusability": "Wiederverwendbarkeit",
            "time_to_value": "Zeit bis zum Wert",
            "data_availability": "Datenverfügbarkeit",
            "data_accessibility": "Datenzugänglichkeit",
            "architecture_fitness": "Architektureignung",
            "team_skills": "Teamfähigkeiten",
            "domain_knowledge": "Fachwissen",
            "people_allocation": "Personalzuweisung",
            "budget_allocation": "Budgetzuweisung",
            "time_to_production": "Zeit bis zur Produktion",
            "value_score": "Wertpunktzahl",
            "feasibility_score": "Machbarkeitspunktzahl",
            "priority_score": "Prioritätspunktzahl",
            "inspire_score": "Inspire-Punktzahl",
            "quality_score_header": "Qualitätspunktzahl",
            "pdf_title": "Strategische KI-Anwendungsfälle Unterstützt von Databricks Genie Code",
            "pdf_for": "Für",
            "pdf_exec_summary": "Zusammenfassung",
            "pdf_toc_title": "Anwendungsfall-Bereiche",
            "pdf_detailed_view": "Detaillierter Anwendungsfallkatalog",
            "pdf_disclaimer_title": "Haftungsausschluss",
            "pdf_fallback_summary_p1": "Dieses Dokument beschreibt {total_cases} hochwertige analytische Anwendungsfälle, die für {business_name} identifiziert wurden.",
            "pdf_fallback_summary_p2": "Die folgenden Seiten bieten eine detaillierte Aufschlüsselung dieser Möglichkeiten, kategorisiert nach Geschäftsbereich.",
            "pptx_main_title": "Strategische KI-Anwendungsfälle Unterstützt von Databricks Genie Code",
            "pptx_for": "Für",
            "pptx_disclaimer_title": "Haftungsausschluss",
            "pptx_domain_suffix": "Anwendungsfälle",
            "elite_verdict": "Urteil",
            "elite_decision": "Entscheidung",
            "elite_decision_owner": "Entscheidungsverantwortlicher",
            "elite_kpi": "KPI",
            "elite_value_driver": "Werttreiber",
            "elite_evidence_columns": "Nachweisspalten",
            "elite_join_path": "Verknüpfungspfad",
            "elite_build_path": "Umsetzungspfad",
            "elite_red_team_note": "Red-Team-Hinweis",
            "elite_coverage_pick": "Auswahl zur Domänenabdeckung, unterhalb der Schwelle",
            "elite_fallback_pick": "Unterhalb der {mode}-Schwelle",
            "elite_fallback_near_miss": "Knapp verfehlt, unterhalb der {mode}-Schwelle",
            "doc_fallback_pick": "Unter der Schwelle",
            "doc_fallback_notice": "Kein Anwendungsfall erreichte die {mode}-Schwelle. Dies sind die nächstbesten Kandidaten, jeweils unterhalb der Schwelle gekennzeichnet.",
            "elite_reason": "Grund",
            "untranslated_notice": "Auf Englisch angezeigt (Übersetzung nicht verfügbar)",
            "more_use_cases": "...und {count} weitere Anwendungsfälle.",
            "executive_summary_not_available": "Zusammenfassung nicht verfügbar.",
            "domain_summary_not_available": "Bereichszusammenfassung nicht verfügbar.",
            "summary_not_available": "Zusammenfassung nicht verfügbar.",
            "doc_catalog_kicker": "KI-Anwendungsfallkatalog",
            "doc_prepared_for": "Erstellt für",
            "doc_generated_on": "Erstellt am",
            "doc_quality_mode": "Qualitätsmodus",
            "doc_use_cases": "Anwendungsfälle",
            "doc_domains": "Geschäftsbereiche",
            "doc_elite_use_cases": "Elite-Anwendungsfälle",
            "doc_high_or_better": "Hohe Qualität oder besser",
            "doc_best_score": "Beste Inspire-Punktzahl",
            "doc_not_scored": "Nicht bewertet",
            "doc_coverage_picks": "Abdeckungsauswahlen",
            "doc_coverage_pick": "Abdeckungsauswahl",
            "value_correlation_analysis": "Korrelationsanalyse",
            "value_pareto_analysis": "Pareto-Analyse",
            "value_funnel_analysis": "Trichteranalyse",
            "value_document_processing": "Dokumentenverarbeitung",
            "value_extraction": "Extraktion",
            "value_ai_analysis": "KI-Analyse",
            "value_simulation": "Simulation",
            "value_geospatial_analysis": "Geodatenanalyse",
            "value_predictive_modeling": "Prädiktive Modellierung",
            "value_clustering": "Clusteranalyse",
            "value_regression_analysis": "Regressionsanalyse",
            "value_survival_analysis": "Überlebensanalyse",
            "value_network_analysis": "Netzwerkanalyse",
            "value_mode_good": "Gut",
            "value_mode_high": "Hoch",
            "value_mode_ultra_high": "Ultra hoch",
            "doc_exec_kicker": "Die Chance",
            "doc_guide_kicker": "Lesehilfe",
            "doc_how_to_read": "So lesen Sie diesen Katalog",
            "doc_guide_lead": "Jeder Anwendungsfall stammt aus den eigenen Metadaten der Organisation und wird auf einer einzigen absoluten Skala bewertet. Nutzen Sie die Punktzahlen zum Vergleichen, die Stufen zum Filtern und die Bereichsseiten zur Navigation.",
            "doc_inspire_score_title": "Inspire-Punktzahl, von 0 bis 10",
            "doc_inspire_score_body": "Verbindet die Qualität eines Anwendungsfalls mit seiner geschäftlichen Priorität. Höher ist besser, und der Katalog ist danach sortiert.",
            "doc_quality_tiers_title": "Qualitätsstufen",
            "doc_quality_tiers_body": "Jeder Anwendungsfall erhält eine Stufe von {lowest} bis {highest}, die zeigt, wie konkret, wertvoll und mit den verfügbaren Daten umsetzbar er ist.",
            "doc_modes_title": "Qualitätsmodi",
            "doc_mode_good_body": "Behält Anwendungsfälle mit der Bewertung {medium} oder besser.",
            "doc_mode_high_body": "Behält nur Anwendungsfälle mit der Bewertung {high} oder besser.",
            "doc_mode_ultra_body": "Behält nur Anwendungsfälle, die eine absolute Schwelle bei Wert, Evidenz, Umsetzbarkeit und Handlungsrelevanz überschreiten. Es gibt keine Quote.",
            "doc_this_catalog": "Dieser Katalog",
            "doc_elite_score_title": "Elite-Punktzahl",
            "doc_elite_score_body": "Wird im Modus {ultra} verwendet. Sie gewichtet Geschäftswert, Evidenzstärke, Umsetzbarkeit und Handlungsrelevanz, und ein Elite-Anwendungsfall erreicht in jeder Dimension mindestens {min_score} von 10.",
            "doc_coverage_title": "Auswahlen zur Bereichsabdeckung",
            "doc_coverage_body": "Hat ein angeforderter Geschäftsbereich keinen Anwendungsfall über der Schwelle, wird sein stärkster Kandidat behalten und als Abdeckungsauswahl unterhalb der Schwelle gekennzeichnet.",
            "doc_portfolio_kicker": "Portfolio",
            "doc_portfolio_glance": "Das Portfolio auf einen Blick",
            "doc_per_domain": "Anwendungsfälle pro Bereich",
            "doc_quality_distribution": "Qualitätsverteilung",
            "doc_top_use_cases": "Top-Anwendungsfälle",
            "doc_rank": "Rang",
            "doc_domain_label": "Bereich",
            "doc_domain_overview": "Über diesen Bereich",
            "doc_what_it_covers": "Was er abdeckt",
            "doc_top_opportunities": "Wichtigste Chancen",
            "doc_domain_fallback": "Dieser Bereich vereint Anwendungsfälle aus {subdomains}, insgesamt {count}. Die stärksten Chancen sind unten aufgeführt und auf den folgenden Seiten beschrieben.",
            "doc_open_notebook": "Genie-Notebook öffnen",
            "doc_elite_score": "Elite-Punktzahl",
            "doc_best_of_best": "Das Beste vom Besten",
            "doc_appendix": "Anhang",
            "doc_methodology": "Methodik",
            "strategy_path": "Strategiepfad",
            "kpis": "KPIs",
            "expected_key_results": "Erwartete Schlüsselergebnisse",
            "measures_of_success": "Erfolgskennzahlen",
            "financial_benefits": "Finanzieller Nutzen",
            "non_financial_benefits": "Nicht-finanzieller Nutzen",
            "foundations": "Datengrundlagen",
            "doc_foundations_title": "Gemeinsame Datengrundlagen",
            "doc_foundations_lead": "Tabellen und Verknüpfungen, die zwei oder mehr Anwendungsfälle benötigen. Wer sie zuerst aufbaut, ermöglicht die meisten Anwendungsfälle. Die Reihenfolge ist eine Abfolge, kein Zeitplan.",
            "doc_foundation_tables": "Tabellen",
            "doc_foundation_unlocks": "Ermöglichte Anwendungsfälle",
            "doc_data_access_standard": "Standard-Erkennung: Inspire hat keine Kundendaten gelesen. Es nutzte nur Metadaten, Definitionen von Metrik-Views und Nutzungszahlen.",
            "doc_data_access_deep": "Tiefe Erkennung: Inspire hat Aggregate aus den Tabellen im Umfang gelesen, insgesamt {tables} (Zeilenzahlen, Datumsabdeckung, Null-Anteile, Anzahl eindeutiger Werte und kurze Wertelisten nicht personenbezogener Spalten). Es wurden keine Datenzeilen an die KI gesendet.",
            "doc_method_1_deep": "Inspire liest Metadaten und bei der tiefen Erkennung auch über die Tabellen berechnete Aggregate. Es sendet nie Datenzeilen an die KI.",
            "doc_guide_lead_deep": "Jeder Anwendungsfall stammt aus den Metadaten und Datenprofilen der Organisation und wird auf einer einzigen absoluten Skala bewertet. Nutzen Sie die Werte zum Vergleichen, die Stufen zum Filtern und die Domänenseiten zur Navigation.",
            "value_not_aligned": "Nicht ausgerichtet",
            "kpi_baseline": "Ausgangswert {value}, letzte {days} Tage, gemessen am {date}",
            "kpi_baseline_undated": "Ausgangswert {value}, gemessen am {date}",
            "doc_method_1": "Inspire liest ausschließlich Metadaten: Tabellen- und Spaltennamen, Typen, Kommentare, Schlüssel und Tags. Zeilendaten liest es nie.",
            "doc_method_2": "Es leitet den Geschäftskontext ab und erzeugt dann Kandidaten für jeden Geschäftsbereich, den die Daten tragen.",
            "doc_method_3": "Duplikate werden entfernt, jeder Kandidat wird nach Wert und Machbarkeit bewertet, und Qualitätsprüfungen verwerfen schwache oder unbelegte Ideen.",
            "doc_method_4": "Im Modus {ultra} prüft ein Elite-Gutachter jeden verbleibenden Kandidaten, und deterministische Prüfungen bestätigen, dass jede genannte Tabelle und Spalte existiert.",
            "doc_method_5": "Die endgültige Liste wird gerankt, nach Geschäftsbereich gruppiert und bei Auswahl mit einem Databricks Genie Code Notebook verknüpft.",
            "doc_glossary": "Glossar",
            "doc_gl_domain": "Ein Geschäftsfeld, das zusammengehörige Anwendungsfälle bündelt.",
            "doc_gl_subdomain": "Ein engeres Thema innerhalb eines Geschäftsbereichs.",
            "doc_gl_technique": "Die analytische Methode, zum Beispiel Prognose oder Klassifikation.",
            "doc_gl_inspire": "Die Punktzahl von 0 bis 10, die Qualität und geschäftliche Priorität verbindet.",
            "doc_gl_quality": "Die Stufe von {lowest} bis {highest}, die jeder Anwendungsfall erhält.",
            "doc_gl_elite": "Die Punktzahl im Modus {ultra} über Wert, Evidenz, Umsetzbarkeit und Handlungsrelevanz.",
            "doc_gl_coverage": "Ein Anwendungsfall unterhalb der Schwelle, der behalten wird, damit ein angeforderter Bereich vertreten ist.",
            "doc_gl_notebook": "Ein Databricks-Notebook mit Genie-Code-Anweisungen zum Aufbau des Anwendungsfalls.",
            "doc_notebook_term": "Genie-Notebook",
            "doc_next_steps": "Nächste Schritte",
            "doc_closing_title": "Von Anwendungsfällen zu Ergebnissen",
            "doc_closing_body": "Wählen Sie die wichtigsten Anwendungsfälle, öffnen Sie ihre Genie-Notebooks und bauen Sie den ersten gemeinsam mit dem Team. Die Schritte sind eine empfohlene Reihenfolge; das Team bestimmt das Tempo.",
            "doc_step_select": "Auswählen",
            "doc_step_select_body": "Wählen Sie die Anwendungsfälle mit der höchsten Punktzahl und einem klaren Sponsor.",
            "doc_step_validate": "Daten prüfen",
            "doc_step_validate_body": "Bestätigen Sie die beteiligten Tabellen und ihre Verknüpfungen mit den Datenverantwortlichen.",
            "doc_step_build": "Mit Genie Code bauen",
            "doc_step_build_body": "Öffnen Sie das Genie-Notebook und erzeugen Sie die erste lauffähige Version.",
            "doc_step_schedule": "Planen",
            "doc_step_schedule_body": "Führen Sie das Notebook als Databricks-Job aus, der die Ergebnistabelle aktualisiert.",
            "doc_step_measure": "Messen und skalieren",
            "doc_step_measure_body": "Verfolgen Sie den erzielten Geschäftswert und erweitern Sie dann auf den nächsten Anwendungsfall.",
            "doc_agenda": "Agenda",
            "doc_roadmap": "Roadmap und nächste Schritte",
            "doc_use_case_index": "Verzeichnis der Anwendungsfälle",
            "doc_continued": "Fortsetzung",
            "doc_not_provided": "Nicht angegeben",
            "doc_bob_score": "Best-of-Best-Score",
            "elite_cadence": "Entscheidungsrhythmus",
            "value_cadence_hourly": "Stündlich",
            "value_cadence_daily": "Täglich",
            "value_cadence_weekly": "Wöchentlich",
            "value_cadence_biweekly": "Alle zwei Wochen",
            "value_cadence_monthly": "Monatlich",
            "value_cadence_quarterly": "Vierteljährlich",
            "value_cadence_yearly": "Jährlich",
            "value_cadence_real_time": "In Echtzeit",
            "value_cadence_per_event": "Pro Ereignis",
            "doc_generated_by": "Erstellt mit {brand} {version} · {date}",
            "doc_disclaimer_ai": "Die Anwendungsfälle, Bewertungen und Texte in diesem Dokument wurden von KI erzeugt. Prüfen Sie sie mit den Dateneigentümern und den fachlichen Sponsoren, bevor eine Entscheidung getroffen wird.",
            "doc_disclaimer_figures": "Bewertungen, Werte und Ergebnisse sind Schätzungen, die nur der Priorisierung dienen. Keine Zahl in diesem Dokument ist garantiert.",
            "doc_contents": "Inhalt",
            "doc_index_lead": "Alle Anwendungsfälle in der endgültigen Rangfolge, insgesamt {count}.",
            "doc_deck_scope": "Diese Präsentation beschreibt die bestplatzierten Anwendungsfälle, {count} von {total}. Der PDF-Katalog beschreibt alle.",
            "doc_deck_scope_all": "Diese Präsentation beschreibt alle Anwendungsfälle, insgesamt {total}, jeweils mit einer eigenen Karte.",
            "value_general_improvement": "Allgemeine Verbesserung",
            "value_reduce_cost": "Kosten Reduzieren",
            "value_increase_revenue": "Umsatz Steigern",
            "value_boost_productivity": "Produktivität Steigern",
            "value_mitigate_risk": "Risiken Mindern",
            "value_protect_revenue": "Umsatz Schützen",
            "value_align_to_regulations": "Vorschriften Einhalten",
            "value_improve_customer_experience": "Kundenerlebnis Verbessern",
            "value_enable_data_driven_decisions": "Datengestützte Entscheidungen Ermöglichen",
            "value_optimize_operations": "Betrieb Optimieren",
            "value_empower_talent": "Talente Fördern",
            "value_enhance_experience": "Erlebnis Verbessern",
            "value_drive_innovation": "Innovation Vorantreiben",
            "value_achieve_esg": "ESG Erreichen",
            "value_execute_strategy": "Strategie Umsetzen",
            "value_forecasting": "Prognose",
            "value_classification": "Klassifizierung",
            "value_anomaly_detection": "Anomalieerkennung",
            "value_cohort_analysis": "Kohortenanalyse",
            "value_segmentation": "Segmentierung",
            "value_sentiment_analysis": "Stimmungsanalyse",
            "value_trend_analysis": "Trendanalyse",
            "value_prescriptive_analytics": "Präskriptive Analytik",
            "value_root_cause_analysis": "Ursachenanalyse",
            "value_optimization": "Optimierung",
            "value_recommendation": "Empfehlung",
            "value_time_series_analysis": "Zeitreihenanalyse",
            "value_predictive_analytics": "Prädiktive Analytik",
            "value_descriptive_analytics": "Deskriptive Analytik",
        },
        "Portuguese": {
            "main_title": "Gerador de Casos de Uso Impulsionado por Databricks Genie Code",
            "intro": "Este notebook contém casos de uso gerados por IA baseados em seus esquemas. Abaixo está um resumo dos cenários gerados por domínio de negócio.",
            "domain": "Domínio de Negócio",
            "total": "Total de Casos de Uso",
            "summaries": "Resumos de Casos de Uso",
            "sum_id": "ID",
            "sum_name": "Nome",
            "sum_value": "Valor de Negócio",
            "sum_outcome": "Resultado Esperado",
            "warning_header": "AVISO",
            "warning_body": "Não execute este notebook. É destinado apenas para demonstração e catalogação.",
            "disclaimer": "Este conteúdo foi gerado por IA e deve ser revisado por um engenheiro qualificado antes de ser utilizado em produção. A Databricks não é responsável por quaisquer problemas decorrentes do uso deste conteúdo.",
            "detailed_scenarios": "Detalhes dos Casos de Uso",
            "aspect": "Aspecto",
            "description": "Descrição",
            "aspect_domain": "Domínio de Negócio",
            "type": "Tipo",
            "analytics_technique": "Técnica de Análise",
            "primary_table": "Tabela Principal",
            "quality": "Qualidade",
            "priority": "Prioridade",
            "usecase": "Caso de Uso",
            "description_label": "Descrição",
            "value_type_problem": "Problema",
            "value_type_risk": "Risco",
            "value_type_opportunity": "Oportunidade",
            "value_type_improvement": "Melhoria",
            "value_priority_ultra_high": "Extremamente Alta",
            "value_priority_very_high": "Muito Alta",
            "value_priority_high": "Alta",
            "value_priority_medium": "Média",
            "value_priority_low": "Baixa",
            "value_priority_very_low": "Muito Baixa",
            "value_priority_ultra_low": "Extremamente Baixa",
            "statement": "Declaração",
            "solution": "Solução",
            "aspect_beneficiary": "Beneficiário",
            "beneficiary": "Beneficiário",
            "aspect_sponsor": "Patrocinador",
            "sponsor": "Patrocinador",
            "business_priority_alignment": "Alinhamento de Prioridade de Negócio",
            "strategic_goals_alignment": "Alinhamento com Objetivos Estratégicos",
            "subdomain": "Subdomínio",
            "aspect_value": "Valor de Negócio",
            "business_value": "Valor de Negócio",
            "aspect_tables": "Tabelas Envolvidas",
            "quality_reasons": "Razões de Qualidade",
            "aspect_ai_function": "Função de IA",
            "aspect_analytics_technique": "Técnica de Análise",
            "aspect_primary_table": "Tabela Principal",
            "aspect_quality": "Qualidade",
            "aspect_priority": "Prioridade",
            "strategic_alignment": "Alinhamento Estratégico",
            "return_on_investment": "Retorno sobre Investimento",
            "reusability": "Reusabilidade",
            "time_to_value": "Tempo para Valor",
            "data_availability": "Disponibilidade de Dados",
            "data_accessibility": "Acessibilidade de Dados",
            "architecture_fitness": "Adequação da Arquitetura",
            "team_skills": "Habilidades da Equipe",
            "domain_knowledge": "Conhecimento do Domínio",
            "people_allocation": "Alocação de Pessoas",
            "budget_allocation": "Alocação de Orçamento",
            "time_to_production": "Tempo para Produção",
            "value_score": "Pontuação de Valor",
            "feasibility_score": "Pontuação de Viabilidade",
            "priority_score": "Pontuação de Prioridade",
            "inspire_score": "Pontuação Inspire",
            "quality_score_header": "Pontuação de Qualidade",
            "pdf_title": "Casos de Uso de IA Estratégica Impulsionados por Databricks Genie Code",
            "pdf_for": "Para",
            "pdf_exec_summary": "Resumo Executivo",
            "pdf_toc_title": "Domínios de Casos de Uso",
            "pdf_detailed_view": "Catálogo Detalhado de Casos de Uso",
            "pdf_disclaimer_title": "Aviso Legal",
            "pdf_fallback_summary_p1": "Este documento descreve {total_cases} casos de uso analíticos de alto valor identificados para {business_name}.",
            "pdf_fallback_summary_p2": "As páginas seguintes fornecem uma análise detalhada dessas oportunidades, categorizadas por domínio de negócio.",
            "pptx_main_title": "Casos de Uso de IA Estratégica Impulsionados por Databricks Genie Code",
            "pptx_for": "Para",
            "pptx_disclaimer_title": "Aviso Legal",
            "pptx_domain_suffix": "Casos de Uso",
            "elite_verdict": "Veredito",
            "elite_decision": "Decisão",
            "elite_decision_owner": "Responsável pela decisão",
            "elite_kpi": "KPI",
            "elite_value_driver": "Motor de valor",
            "elite_evidence_columns": "Colunas de evidência",
            "elite_join_path": "Caminho de junção",
            "elite_build_path": "Caminho de implementação",
            "elite_red_team_note": "Nota da equipe vermelha",
            "elite_coverage_pick": "Seleção por cobertura de domínio, abaixo do limite",
            "elite_fallback_pick": "Abaixo do limite {mode}",
            "elite_fallback_near_miss": "Quase selecionado, abaixo do limite {mode}",
            "doc_fallback_pick": "Abaixo do limite",
            "doc_fallback_notice": "Nenhum caso de uso atingiu o limite {mode}. Estes são os candidatos mais próximos, cada um identificado abaixo do limite.",
            "elite_reason": "Motivo",
            "untranslated_notice": "Exibido em inglês (tradução indisponível)",
            "more_use_cases": "...e mais {count} casos de uso.",
            "executive_summary_not_available": "Resumo executivo não disponível.",
            "domain_summary_not_available": "Resumo do domínio não disponível.",
            "summary_not_available": "Resumo não disponível.",
            "doc_catalog_kicker": "Catálogo de casos de uso de IA",
            "doc_prepared_for": "Preparado para",
            "doc_generated_on": "Gerado em",
            "doc_quality_mode": "Modo de qualidade",
            "doc_use_cases": "Casos de uso",
            "doc_domains": "Domínios de negócio",
            "doc_elite_use_cases": "Casos de uso de elite",
            "doc_high_or_better": "Qualidade alta ou superior",
            "doc_best_score": "Melhor pontuação Inspire",
            "doc_not_scored": "Sem pontuação",
            "doc_coverage_picks": "Seleções de cobertura",
            "doc_coverage_pick": "Seleção de cobertura",
            "value_correlation_analysis": "Análise de Correlação",
            "value_pareto_analysis": "Análise de Pareto",
            "value_funnel_analysis": "Análise de Funil",
            "value_document_processing": "Processamento de Documentos",
            "value_extraction": "Extração",
            "value_ai_analysis": "Análise com IA",
            "value_simulation": "Simulação",
            "value_geospatial_analysis": "Análise Geoespacial",
            "value_predictive_modeling": "Modelagem Preditiva",
            "value_clustering": "Agrupamento",
            "value_regression_analysis": "Análise de Regressão",
            "value_survival_analysis": "Análise de Sobrevivência",
            "value_network_analysis": "Análise de Redes",
            "value_mode_good": "Bom",
            "value_mode_high": "Alto",
            "value_mode_ultra_high": "Ultra alto",
            "doc_exec_kicker": "A oportunidade",
            "doc_guide_kicker": "Guia de leitura",
            "doc_how_to_read": "Como ler este catálogo",
            "doc_guide_lead": "Cada caso de uso vem dos próprios metadados da organização e é pontuado em uma única escala absoluta. Use as pontuações para comparar, os níveis para filtrar e as páginas de domínio para navegar.",
            "doc_inspire_score_title": "Pontuação Inspire, de 0 a 10",
            "doc_inspire_score_body": "Combina a qualidade de um caso de uso com sua prioridade de negócio. Quanto mais alta, melhor, e o catálogo é ordenado por ela.",
            "doc_quality_tiers_title": "Níveis de qualidade",
            "doc_quality_tiers_body": "Cada caso de uso recebe um nível de {lowest} a {highest}, que reflete o quão específico, valioso e construível ele é com os dados disponíveis.",
            "doc_modes_title": "Modos de qualidade",
            "doc_mode_good_body": "Mantém os casos de uso classificados como {medium} ou superior.",
            "doc_mode_high_body": "Mantém apenas os casos de uso classificados como {high} ou superior.",
            "doc_mode_ultra_body": "Mantém apenas os casos de uso que superam um patamar absoluto de valor, evidência, viabilidade de implementação e acionabilidade. Não há cota.",
            "doc_this_catalog": "Este catálogo",
            "doc_elite_score_title": "Pontuação de elite",
            "doc_elite_score_body": "Usada no modo {ultra}. Pondera o valor de negócio, a solidez da evidência, a viabilidade de implementação e a acionabilidade, e um caso de uso de elite obtém pelo menos {min_score} em 10 em cada uma.",
            "doc_coverage_title": "Seleções por cobertura de domínio",
            "doc_coverage_body": "Quando um domínio de negócio solicitado não tem nenhum caso de uso acima do patamar, seu candidato mais forte é mantido e identificado como seleção de cobertura abaixo do patamar.",
            "doc_portfolio_kicker": "Portfólio",
            "doc_portfolio_glance": "O portfólio em resumo",
            "doc_per_domain": "Casos de uso por domínio",
            "doc_quality_distribution": "Distribuição da qualidade",
            "doc_top_use_cases": "Principais casos de uso",
            "doc_rank": "Posição",
            "doc_domain_label": "Domínio",
            "doc_domain_overview": "Sobre este domínio",
            "doc_what_it_covers": "O que abrange",
            "doc_top_opportunities": "Principais oportunidades",
            "doc_domain_fallback": "Este domínio reúne casos de uso em {subdomains}, {count} no total. As oportunidades mais fortes estão listadas abaixo e detalhadas nas páginas seguintes.",
            "doc_open_notebook": "Abrir o notebook do Genie",
            "doc_elite_score": "Pontuação de elite",
            "doc_best_of_best": "O melhor dos melhores",
            "doc_appendix": "Anexo",
            "doc_methodology": "Metodologia",
            "strategy_path": "Caminho estratégico",
            "kpis": "KPIs",
            "expected_key_results": "Resultados-chave esperados",
            "measures_of_success": "Medidas de sucesso",
            "financial_benefits": "Benefícios financeiros",
            "non_financial_benefits": "Benefícios não financeiros",
            "foundations": "Fundações de dados",
            "doc_foundations_title": "Fundações de dados partilhadas",
            "doc_foundations_lead": "Tabelas e junções de que dois ou mais casos de uso precisam. Construí-las primeiro desbloqueia o maior número de casos de uso. A ordem é uma sequência, não um calendário.",
            "doc_foundation_tables": "Tabelas",
            "doc_foundation_unlocks": "Casos de uso desbloqueados",
            "doc_data_access_standard": "Descoberta padrão: o Inspire não leu dados do cliente. Usou apenas metadados, definições de vistas de métricas e contagens de utilização.",
            "doc_data_access_deep": "Descoberta profunda: o Inspire leu agregados das tabelas no âmbito, {tables} no total (contagens de linhas, cobertura de datas, taxas de nulos, contagens de valores distintos e listas curtas de valores de colunas não pessoais). Nenhuma linha de dados foi enviada para a IA.",
            "doc_method_1_deep": "O Inspire lê metadados e, na descoberta profunda, também agregados calculados sobre as tabelas. Nunca envia linhas de dados para a IA.",
            "doc_guide_lead_deep": "Cada caso de uso provém dos metadados e dos perfis de dados da organização e é pontuado numa única escala absoluta. Use as pontuações para comparar, os níveis para filtrar e as páginas de domínio para navegar.",
            "value_not_aligned": "Não alinhado",
            "kpi_baseline": "linha de base {value}, últimos {days} dias, medida em {date}",
            "kpi_baseline_undated": "linha de base {value}, medida em {date}",
            "doc_method_1": "O Inspire lê apenas metadados: nomes de tabelas e colunas, tipos, comentários, chaves e etiquetas. Nunca lê dados das linhas.",
            "doc_method_2": "Ele deriva o contexto de negócio e depois gera casos de uso candidatos para cada domínio de negócio que os dados suportam.",
            "doc_method_3": "Os duplicados são removidos, cada candidato é pontuado por valor e viabilidade, e filtros de qualidade descartam ideias fracas ou sem suporte.",
            "doc_method_4": "No modo {ultra}, um juiz de elite analisa cada candidato restante e verificações determinísticas confirmam que cada tabela e coluna citada existe.",
            "doc_method_5": "A lista final é ordenada, agrupada por domínio de negócio e, quando selecionado, acompanhada de um notebook Databricks Genie Code.",
            "doc_glossary": "Glossário",
            "doc_gl_domain": "Uma área de negócio que agrupa casos de uso relacionados.",
            "doc_gl_subdomain": "Um tema mais restrito dentro de um domínio de negócio.",
            "doc_gl_technique": "O método analítico, por exemplo previsão ou classificação.",
            "doc_gl_inspire": "A pontuação de 0 a 10 que combina qualidade e prioridade de negócio.",
            "doc_gl_quality": "O nível de {lowest} a {highest} atribuído a cada caso de uso.",
            "doc_gl_elite": "A pontuação do modo {ultra} em valor, evidência, viabilidade de implementação e acionabilidade.",
            "doc_gl_coverage": "Um caso de uso mantido abaixo do patamar para que um domínio solicitado fique representado.",
            "doc_gl_notebook": "Um notebook Databricks com instruções do Genie Code para construir o caso de uso.",
            "doc_notebook_term": "Notebook do Genie",
            "doc_next_steps": "Próximos passos",
            "doc_closing_title": "Dos casos de uso aos resultados",
            "doc_closing_body": "Escolha os casos de uso mais importantes, abra seus notebooks do Genie e construa o primeiro com a equipe. Os passos seguem uma ordem sugerida; a equipe define o ritmo.",
            "doc_step_select": "Selecionar",
            "doc_step_select_body": "Escolha os casos de uso com a melhor pontuação e um patrocinador claro.",
            "doc_step_validate": "Validar os dados",
            "doc_step_validate_body": "Confirme as tabelas envolvidas e as suas junções com os responsáveis pelos dados.",
            "doc_step_build": "Construir com Genie Code",
            "doc_step_build_body": "Abra o notebook do Genie e gere a primeira versão funcional.",
            "doc_step_schedule": "Agendar",
            "doc_step_schedule_body": "Execute o notebook como um job do Databricks que atualiza a tabela de resultados.",
            "doc_step_measure": "Medir e escalar",
            "doc_step_measure_body": "Acompanhe o valor de negócio gerado e depois avance para o próximo caso de uso.",
            "doc_agenda": "Agenda",
            "doc_roadmap": "Roteiro e próximos passos",
            "doc_use_case_index": "Índice de casos de uso",
            "doc_continued": "continuação",
            "doc_not_provided": "Não informado",
            "doc_bob_score": "Pontuação Best-of-Best",
            "elite_cadence": "Cadência da decisão",
            "value_cadence_hourly": "De hora em hora",
            "value_cadence_daily": "Diária",
            "value_cadence_weekly": "Semanal",
            "value_cadence_biweekly": "A cada duas semanas",
            "value_cadence_monthly": "Mensal",
            "value_cadence_quarterly": "Trimestral",
            "value_cadence_yearly": "Anual",
            "value_cadence_real_time": "Em tempo real",
            "value_cadence_per_event": "Por evento",
            "doc_generated_by": "Gerado por {brand} {version} · {date}",
            "doc_disclaimer_ai": "Os casos de uso, as pontuações e os textos deste documento são gerados por IA. Valide-os com os responsáveis pelos dados e os patrocinadores de negócio antes de qualquer decisão.",
            "doc_disclaimer_figures": "As pontuações, os valores e os resultados são estimativas apenas para priorização. Nenhum número deste documento é garantido.",
            "doc_contents": "Conteúdo",
            "doc_index_lead": "Todos os casos de uso pela ordem final de classificação, {count} no total.",
            "doc_deck_scope": "Esta apresentação detalha os casos de uso mais bem classificados, {count} de {total}. O catálogo PDF detalha todos.",
            "doc_deck_scope_all": "Esta apresentação detalha todos os casos de uso, {total} no total, um cartão para cada um.",
            "value_general_improvement": "Melhoria Geral",
            "value_reduce_cost": "Reduzir Custos",
            "value_increase_revenue": "Aumentar Receita",
            "value_boost_productivity": "Aumentar Produtividade",
            "value_mitigate_risk": "Mitigar Riscos",
            "value_protect_revenue": "Proteger Receita",
            "value_align_to_regulations": "Cumprir Regulamentos",
            "value_improve_customer_experience": "Melhorar Experiência do Cliente",
            "value_enable_data_driven_decisions": "Habilitar Decisões Baseadas em Dados",
            "value_optimize_operations": "Otimizar Operações",
            "value_empower_talent": "Capacitar Talentos",
            "value_enhance_experience": "Melhorar Experiência",
            "value_drive_innovation": "Impulsionar Inovação",
            "value_achieve_esg": "Alcançar ESG",
            "value_execute_strategy": "Executar Estratégia",
            "value_forecasting": "Previsão",
            "value_classification": "Classificação",
            "value_anomaly_detection": "Detecção de Anomalias",
            "value_cohort_analysis": "Análise de Coorte",
            "value_segmentation": "Segmentação",
            "value_sentiment_analysis": "Análise de Sentimento",
            "value_trend_analysis": "Análise de Tendências",
            "value_prescriptive_analytics": "Análise Prescritiva",
            "value_root_cause_analysis": "Análise de Causa Raiz",
            "value_optimization": "Otimização",
            "value_recommendation": "Recomendação",
            "value_time_series_analysis": "Análise de Séries Temporais",
            "value_predictive_analytics": "Análise Preditiva",
            "value_descriptive_analytics": "Análise Descritiva",
        },
        "Italian": {
            "main_title": "Generatore di Casi d'Uso Alimentato da Databricks Genie Code",
            "intro": "Questo notebook contiene casi d'uso generati dall'IA basati sui tuoi schemi. Di seguito è riportato un riepilogo degli scenari generati per dominio aziendale.",
            "domain": "Dominio Aziendale",
            "total": "Totale Casi d'Uso",
            "summaries": "Riepiloghi dei Casi d'Uso",
            "sum_id": "ID",
            "sum_name": "Nome",
            "sum_value": "Valore Aziendale",
            "sum_outcome": "Risultato Atteso",
            "warning_header": "AVVERTIMENTO",
            "warning_body": "Non eseguire questo notebook. È destinato solo a scopi dimostrativi e di catalogazione.",
            "disclaimer": "Questo contenuto è generato dall'IA e deve essere verificato da un ingegnere qualificato prima dell'uso in produzione. Databricks non è responsabile per eventuali problemi derivanti dall'uso di questo contenuto.",
            "detailed_scenarios": "Dettagli dei Casi d'Uso",
            "aspect": "Aspetto",
            "description": "Descrizione",
            "aspect_domain": "Dominio Aziendale",
            "type": "Tipo",
            "analytics_technique": "Tecnica di Analisi",
            "primary_table": "Tabella Principale",
            "quality": "Qualità",
            "priority": "Priorità",
            "usecase": "Caso d'Uso",
            "description_label": "Descrizione",
            "value_type_problem": "Problema",
            "value_type_risk": "Rischio",
            "value_type_opportunity": "Opportunità",
            "value_type_improvement": "Miglioramento",
            "value_priority_ultra_high": "Estremamente Alta",
            "value_priority_very_high": "Molto Alta",
            "value_priority_high": "Alta",
            "value_priority_medium": "Media",
            "value_priority_low": "Bassa",
            "value_priority_very_low": "Molto Bassa",
            "value_priority_ultra_low": "Estremamente Bassa",
            "statement": "Dichiarazione",
            "solution": "Soluzione",
            "aspect_beneficiary": "Beneficiario",
            "beneficiary": "Beneficiario",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Allineamento Priorità Aziendale",
            "strategic_goals_alignment": "Allineamento agli Obiettivi Strategici",
            "subdomain": "Sottodominio",
            "aspect_value": "Valore Aziendale",
            "business_value": "Valore Aziendale",
            "aspect_tables": "Tabelle Coinvolte",
            "quality_reasons": "Ragioni di Qualità",
            "aspect_ai_function": "Funzione IA",
            "aspect_analytics_technique": "Tecnica di Analisi",
            "aspect_primary_table": "Tabella Principale",
            "aspect_quality": "Qualità",
            "aspect_priority": "Priorità",
            "strategic_alignment": "Allineamento Strategico",
            "return_on_investment": "Ritorno sull'Investimento",
            "reusability": "Riutilizzabilità",
            "time_to_value": "Tempo al Valore",
            "data_availability": "Disponibilità dei Dati",
            "data_accessibility": "Accessibilità dei Dati",
            "architecture_fitness": "Idoneità dell'Architettura",
            "team_skills": "Competenze del Team",
            "domain_knowledge": "Conoscenza del Dominio",
            "people_allocation": "Allocazione del Personale",
            "budget_allocation": "Allocazione del Budget",
            "time_to_production": "Tempo alla Produzione",
            "value_score": "Punteggio di Valore",
            "feasibility_score": "Punteggio di Fattibilità",
            "priority_score": "Punteggio di Priorità",
            "inspire_score": "Punteggio Inspire",
            "quality_score_header": "Punteggio di Qualità",
            "pdf_title": "Casi d'Uso IA Strategici Alimentati da Databricks Genie Code",
            "pdf_for": "Per",
            "pdf_exec_summary": "Riepilogo Esecutivo",
            "pdf_toc_title": "Domini dei Casi d'Uso",
            "pdf_detailed_view": "Catalogo Dettagliato dei Casi d'Uso",
            "pdf_disclaimer_title": "Disclaimer",
            "pdf_fallback_summary_p1": "Questo documento descrive {total_cases} casi d'uso analitici ad alto valore identificati per {business_name}.",
            "pdf_fallback_summary_p2": "Le pagine seguenti forniscono un'analisi dettagliata di queste opportunità, categorizzate per dominio aziendale.",
            "pptx_main_title": "Casi d'Uso IA Strategici Alimentati da Databricks Genie Code",
            "pptx_for": "Per",
            "pptx_disclaimer_title": "Disclaimer",
            "pptx_domain_suffix": "Casi d'Uso",
            "elite_verdict": "Verdetto",
            "elite_decision": "Decisione",
            "elite_decision_owner": "Responsabile della decisione",
            "elite_kpi": "KPI",
            "elite_value_driver": "Leva di valore",
            "elite_evidence_columns": "Colonne di evidenza",
            "elite_join_path": "Percorso di join",
            "elite_build_path": "Percorso di implementazione",
            "elite_red_team_note": "Nota del red team",
            "elite_coverage_pick": "Selezione per copertura del dominio, sotto la soglia",
            "elite_fallback_pick": "Sotto la soglia {mode}",
            "elite_fallback_near_miss": "Quasi selezionato, sotto la soglia {mode}",
            "doc_fallback_pick": "Sotto la soglia",
            "doc_fallback_notice": "Nessun caso d'uso ha raggiunto la soglia {mode}. Questi sono i candidati più vicini, ciascuno etichettato sotto la soglia.",
            "elite_reason": "Motivo",
            "untranslated_notice": "Mostrato in inglese (traduzione non disponibile)",
            "more_use_cases": "...e altri {count} casi d'uso.",
            "executive_summary_not_available": "Riepilogo esecutivo non disponibile.",
            "domain_summary_not_available": "Riepilogo del dominio non disponibile.",
            "summary_not_available": "Riepilogo non disponibile.",
            "doc_catalog_kicker": "Catalogo dei casi d'uso di IA",
            "doc_prepared_for": "Preparato per",
            "doc_generated_on": "Generato il",
            "doc_quality_mode": "Modalità di qualità",
            "doc_use_cases": "Casi d'uso",
            "doc_domains": "Domini aziendali",
            "doc_elite_use_cases": "Casi d'uso d'élite",
            "doc_high_or_better": "Qualità alta o superiore",
            "doc_best_score": "Miglior punteggio Inspire",
            "doc_not_scored": "Non valutato",
            "doc_coverage_picks": "Selezioni di copertura",
            "doc_coverage_pick": "Selezione di copertura",
            "value_correlation_analysis": "Analisi di Correlazione",
            "value_pareto_analysis": "Analisi di Pareto",
            "value_funnel_analysis": "Analisi del Funnel",
            "value_document_processing": "Elaborazione di Documenti",
            "value_extraction": "Estrazione",
            "value_ai_analysis": "Analisi con IA",
            "value_simulation": "Simulazione",
            "value_geospatial_analysis": "Analisi Geospaziale",
            "value_predictive_modeling": "Modellazione Predittiva",
            "value_clustering": "Raggruppamento",
            "value_regression_analysis": "Analisi di Regressione",
            "value_survival_analysis": "Analisi di Sopravvivenza",
            "value_network_analysis": "Analisi di Rete",
            "value_mode_good": "Buona",
            "value_mode_high": "Alta",
            "value_mode_ultra_high": "Ultra alta",
            "doc_exec_kicker": "L'opportunità",
            "doc_guide_kicker": "Guida alla lettura",
            "doc_how_to_read": "Come leggere questo catalogo",
            "doc_guide_lead": "Ogni caso d'uso nasce dai metadati dell'organizzazione e riceve un punteggio su un'unica scala assoluta. Usa i punteggi per confrontare, i livelli per filtrare e le pagine di dominio per navigare.",
            "doc_inspire_score_title": "Punteggio Inspire, da 0 a 10",
            "doc_inspire_score_body": "Combina la qualità di un caso d'uso con la sua priorità di business. Più è alto, meglio è, e il catalogo è ordinato in base a esso.",
            "doc_quality_tiers_title": "Livelli di qualità",
            "doc_quality_tiers_body": "Ogni caso d'uso riceve un livello da {lowest} a {highest}, che riflette quanto è specifico, utile e realizzabile con i dati disponibili.",
            "doc_modes_title": "Modalità di qualità",
            "doc_mode_good_body": "Mantiene i casi d'uso valutati {medium} o superiore.",
            "doc_mode_high_body": "Mantiene solo i casi d'uso valutati {high} o superiore.",
            "doc_mode_ultra_body": "Mantiene solo i casi d'uso che superano una soglia assoluta di valore, evidenza, attuabilità e operatività. Non ci sono quote.",
            "doc_this_catalog": "Questo catalogo",
            "doc_elite_score_title": "Punteggio d'élite",
            "doc_elite_score_body": "Usato in modalità {ultra}. Pesa valore di business, solidità dell'evidenza, attuabilità e operatività, e un caso d'uso d'élite ottiene almeno {min_score} su 10 in ciascuno.",
            "doc_coverage_title": "Selezioni per copertura del dominio",
            "doc_coverage_body": "Quando un dominio aziendale richiesto non ha alcun caso d'uso sopra la soglia, il suo candidato più forte viene mantenuto ed etichettato come selezione di copertura sotto la soglia.",
            "doc_portfolio_kicker": "Portafoglio",
            "doc_portfolio_glance": "Il portafoglio in sintesi",
            "doc_per_domain": "Casi d'uso per dominio",
            "doc_quality_distribution": "Distribuzione della qualità",
            "doc_top_use_cases": "Casi d'uso principali",
            "doc_rank": "Posizione",
            "doc_domain_label": "Dominio",
            "doc_domain_overview": "Informazioni su questo dominio",
            "doc_what_it_covers": "Cosa copre",
            "doc_top_opportunities": "Opportunità principali",
            "doc_domain_fallback": "Questo dominio riunisce casi d'uso in {subdomains}, {count} in totale. Le opportunità più forti sono elencate qui sotto e descritte nelle pagine seguenti.",
            "doc_open_notebook": "Apri il notebook di Genie",
            "doc_elite_score": "Punteggio d'élite",
            "doc_best_of_best": "Il meglio del meglio",
            "doc_appendix": "Appendice",
            "doc_methodology": "Metodologia",
            "strategy_path": "Percorso strategico",
            "kpis": "KPI",
            "expected_key_results": "Risultati chiave attesi",
            "measures_of_success": "Misure di successo",
            "financial_benefits": "Benefici finanziari",
            "non_financial_benefits": "Benefici non finanziari",
            "foundations": "Fondamenta dei dati",
            "doc_foundations_title": "Fondamenta dei dati condivise",
            "doc_foundations_lead": "Tabelle e join necessari a due o più casi d'uso. Costruirli per primi sblocca il maggior numero di casi d'uso. L'ordine è una sequenza, non un calendario.",
            "doc_foundation_tables": "Tabelle",
            "doc_foundation_unlocks": "Casi d'uso sbloccati",
            "doc_data_access_standard": "Scoperta standard: Inspire non ha letto dati del cliente. Ha usato solo metadati, definizioni delle viste metriche e conteggi di utilizzo.",
            "doc_data_access_deep": "Scoperta approfondita: Inspire ha letto aggregati dalle tabelle nell'ambito, {tables} in totale (conteggi di righe, copertura temporale, percentuali di valori nulli, conteggi di valori distinti e brevi elenchi di valori di colonne non personali). Nessuna riga di dati è stata inviata all'IA.",
            "doc_method_1_deep": "Inspire legge i metadati e, nella scoperta approfondita, anche aggregati calcolati sulle tabelle. Non invia mai righe di dati all'IA.",
            "doc_guide_lead_deep": "Ogni caso d'uso deriva dai metadati e dai profili dei dati dell'organizzazione ed è valutato su un'unica scala assoluta. Usa i punteggi per confrontare, i livelli per filtrare e le pagine dei domini per navigare.",
            "value_not_aligned": "Non allineato",
            "kpi_baseline": "valore di riferimento {value}, ultimi {days} giorni, misurato il {date}",
            "kpi_baseline_undated": "valore di riferimento {value}, misurato il {date}",
            "doc_method_1": "Inspire legge solo metadati: nomi di tabelle e colonne, tipi, commenti, chiavi e tag. Non legge mai i dati delle righe.",
            "doc_method_2": "Ricava il contesto di business, poi genera casi d'uso candidati per ogni dominio aziendale supportato dai dati.",
            "doc_method_3": "I duplicati vengono rimossi, ogni candidato riceve un punteggio per valore e fattibilità, e i controlli di qualità scartano le idee deboli o non supportate.",
            "doc_method_4": "In modalità {ultra}, un giudice d'élite esamina ogni candidato rimanente e controlli deterministici confermano che ogni tabella e colonna citata esiste.",
            "doc_method_5": "L'elenco finale viene ordinato, raggruppato per dominio aziendale e, se selezionato, abbinato a un notebook Databricks Genie Code.",
            "doc_glossary": "Glossario",
            "doc_gl_domain": "Un'area di business che raggruppa casi d'uso correlati.",
            "doc_gl_subdomain": "Un tema più specifico all'interno di un dominio aziendale.",
            "doc_gl_technique": "Il metodo analitico, ad esempio previsione o classificazione.",
            "doc_gl_inspire": "Il punteggio da 0 a 10 che combina qualità e priorità di business.",
            "doc_gl_quality": "Il livello da {lowest} a {highest} assegnato a ogni caso d'uso.",
            "doc_gl_elite": "Il punteggio della modalità {ultra} su valore, evidenza, attuabilità e operatività.",
            "doc_gl_coverage": "Un caso d'uso mantenuto sotto la soglia perché un dominio richiesto sia rappresentato.",
            "doc_gl_notebook": "Un notebook Databricks con istruzioni Genie Code per realizzare il caso d'uso.",
            "doc_notebook_term": "Notebook di Genie",
            "doc_next_steps": "Prossimi passi",
            "doc_closing_title": "Dai casi d'uso ai risultati",
            "doc_closing_body": "Scegli i casi d'uso più importanti, apri i loro notebook di Genie e realizza il primo con il team. I passi sono un ordine suggerito; il ritmo lo decide il team.",
            "doc_step_select": "Selezionare",
            "doc_step_select_body": "Scegli i casi d'uso con il punteggio migliore e uno sponsor chiaro.",
            "doc_step_validate": "Validare i dati",
            "doc_step_validate_body": "Conferma le tabelle coinvolte e i relativi join con i responsabili dei dati.",
            "doc_step_build": "Realizzare con Genie Code",
            "doc_step_build_body": "Apri il notebook di Genie e genera la prima versione funzionante.",
            "doc_step_schedule": "Pianificare",
            "doc_step_schedule_body": "Esegui il notebook come job Databricks che aggiorna la tabella dei risultati.",
            "doc_step_measure": "Misurare e scalare",
            "doc_step_measure_body": "Monitora il valore di business generato, poi estendi al caso d'uso successivo.",
            "doc_agenda": "Agenda",
            "doc_roadmap": "Roadmap e prossimi passi",
            "doc_use_case_index": "Indice dei casi d'uso",
            "doc_continued": "continua",
            "doc_not_provided": "Non indicato",
            "doc_bob_score": "Punteggio Best-of-Best",
            "elite_cadence": "Cadenza della decisione",
            "value_cadence_hourly": "Ogni ora",
            "value_cadence_daily": "Giornaliera",
            "value_cadence_weekly": "Settimanale",
            "value_cadence_biweekly": "Ogni due settimane",
            "value_cadence_monthly": "Mensile",
            "value_cadence_quarterly": "Trimestrale",
            "value_cadence_yearly": "Annuale",
            "value_cadence_real_time": "In tempo reale",
            "value_cadence_per_event": "A ogni evento",
            "doc_generated_by": "Generato da {brand} {version} · {date}",
            "doc_disclaimer_ai": "I casi d'uso, i punteggi e i testi di questo documento sono generati dall'IA. Convalidali con i responsabili dei dati e gli sponsor di business prima di prendere qualsiasi decisione.",
            "doc_disclaimer_figures": "Punteggi, valori e risultati sono stime utili solo per definire le priorità. Nessun dato di questo documento è garantito.",
            "doc_contents": "Indice",
            "doc_index_lead": "Tutti i casi d'uso nell'ordine della classifica finale, {count} in totale.",
            "doc_deck_scope": "Questa presentazione descrive i casi d'uso meglio classificati, {count} su {total}. Il catalogo PDF li descrive tutti.",
            "doc_deck_scope_all": "Questa presentazione descrive tutti i casi d'uso, {total} in totale, una scheda per ciascuno.",
            "value_general_improvement": "Miglioramento Generale",
            "value_reduce_cost": "Ridurre i Costi",
            "value_increase_revenue": "Aumentare i Ricavi",
            "value_boost_productivity": "Aumentare la Produttività",
            "value_mitigate_risk": "Mitigare i Rischi",
            "value_protect_revenue": "Proteggere i Ricavi",
            "value_align_to_regulations": "Conformarsi alle Normative",
            "value_improve_customer_experience": "Migliorare l'Esperienza del Cliente",
            "value_enable_data_driven_decisions": "Abilitare Decisioni Basate sui Dati",
            "value_optimize_operations": "Ottimizzare le Operazioni",
            "value_empower_talent": "Valorizzare i Talenti",
            "value_enhance_experience": "Migliorare l'Esperienza",
            "value_drive_innovation": "Promuovere l'Innovazione",
            "value_achieve_esg": "Raggiungere ESG",
            "value_execute_strategy": "Eseguire la Strategia",
            "value_forecasting": "Previsione",
            "value_classification": "Classificazione",
            "value_anomaly_detection": "Rilevamento Anomalie",
            "value_cohort_analysis": "Analisi di Coorte",
            "value_segmentation": "Segmentazione",
            "value_sentiment_analysis": "Analisi del Sentimento",
            "value_trend_analysis": "Analisi delle Tendenze",
            "value_prescriptive_analytics": "Analisi Prescrittiva",
            "value_root_cause_analysis": "Analisi delle Cause Profonde",
            "value_optimization": "Ottimizzazione",
            "value_recommendation": "Raccomandazione",
            "value_time_series_analysis": "Analisi delle Serie Temporali",
            "value_predictive_analytics": "Analisi Predittiva",
            "value_descriptive_analytics": "Analisi Descrittiva",
        },
        "Chinese (Mandarin)": {
            "main_title": "由 达塔布里克斯 吉尼 科德 驱动的用例生成器",
            "intro": "本笔记本包含基于您的架构由AI生成的用例。以下是按业务领域生成的场景摘要。",
            "domain": "业务领域",
            "total": "用例总数",
            "summaries": "用例摘要",
            "sum_id": "ID",
            "sum_name": "名称",
            "sum_value": "商业价值",
            "sum_outcome": "预期结果",
            "warning_header": "警告",
            "warning_body": "请勿运行此笔记本。它仅用于演示和编目目的。",
            "disclaimer": "此内容由AI生成，应由合格工程师审核后方可用于生产环境。Databricks不对因使用此内容而产生的任何问题承担责任。",
            "detailed_scenarios": "用例详情",
            "aspect": "方面",
            "description": "描述",
            "aspect_domain": "业务领域",
            "type": "类型",
            "analytics_technique": "分析技术",
            "primary_table": "主表",
            "quality": "质量",
            "priority": "优先级",
            "usecase": "用例",
            "description_label": "描述",
            "value_type_problem": "问题",
            "value_type_risk": "风险",
            "value_type_opportunity": "机会",
            "value_type_improvement": "改进",
            "value_priority_ultra_high": "极高",
            "value_priority_very_high": "非常高",
            "value_priority_high": "高",
            "value_priority_medium": "中等",
            "value_priority_low": "低",
            "value_priority_very_low": "非常低",
            "value_priority_ultra_low": "极低",
            "statement": "陈述",
            "solution": "解决方案",
            "aspect_beneficiary": "受益者",
            "beneficiary": "受益者",
            "aspect_sponsor": "发起人",
            "sponsor": "发起人",
            "business_priority_alignment": "业务优先级对齐",
            "strategic_goals_alignment": "战略目标对齐",
            "subdomain": "子领域",
            "aspect_value": "商业价值",
            "business_value": "商业价值",
            "aspect_tables": "涉及的表",
            "quality_reasons": "质量原因",
            "aspect_ai_function": "AI功能",
            "aspect_analytics_technique": "分析技术",
            "aspect_primary_table": "主表",
            "aspect_quality": "质量",
            "aspect_priority": "优先级",
            "strategic_alignment": "战略对齐",
            "return_on_investment": "投资回报率",
            "reusability": "可重用性",
            "time_to_value": "价值实现时间",
            "data_availability": "数据可用性",
            "data_accessibility": "数据可访问性",
            "architecture_fitness": "架构适配性",
            "team_skills": "团队技能",
            "domain_knowledge": "领域知识",
            "people_allocation": "人员分配",
            "budget_allocation": "预算分配",
            "time_to_production": "投产时间",
            "value_score": "价值分数",
            "feasibility_score": "可行性分数",
            "priority_score": "优先级分数",
            "inspire_score": "Inspire 评分",
            "quality_score_header": "质量评分",
            "pdf_title": "战略AI用例 由 达塔布里克斯 吉尼 科德 驱动",
            "pdf_for": "为",
            "pdf_exec_summary": "执行摘要",
            "pdf_toc_title": "用例领域",
            "pdf_detailed_view": "详细用例目录",
            "pdf_disclaimer_title": "免责声明",
            "pdf_fallback_summary_p1": "本文档概述了为{business_name}识别的{total_cases}个高价值分析用例。",
            "pdf_fallback_summary_p2": "以下页面按业务领域分类提供这些机会的详细分析。",
            "pptx_main_title": "战略AI用例 由 达塔布里克斯 吉尼 科德 驱动",
            "pptx_for": "为",
            "pptx_disclaimer_title": "免责声明",
            "pptx_domain_suffix": "用例",
            "elite_verdict": "结论",
            "elite_decision": "决策",
            "elite_decision_owner": "决策负责人",
            "elite_kpi": "关键绩效指标",
            "elite_value_driver": "价值驱动因素",
            "elite_evidence_columns": "证据列",
            "elite_join_path": "关联路径",
            "elite_build_path": "实施路径",
            "elite_red_team_note": "红队意见",
            "elite_coverage_pick": "领域覆盖选择，低于标准",
            "elite_fallback_pick": "低于{mode}标准",
            "elite_fallback_near_miss": "接近入选，低于{mode}标准",
            "doc_fallback_pick": "低于标准",
            "doc_fallback_notice": "没有用例达到{mode}标准。以下是最接近的候选用例，均标注为低于标准。",
            "elite_reason": "原因",
            "untranslated_notice": "以英文显示（无法提供翻译）",
            "more_use_cases": "……以及另外 {count} 个用例。",
            "executive_summary_not_available": "执行摘要不可用。",
            "domain_summary_not_available": "领域摘要不可用。",
            "summary_not_available": "摘要不可用。",
            "doc_catalog_kicker": "AI 用例目录",
            "doc_prepared_for": "编制对象",
            "doc_generated_on": "生成日期",
            "doc_quality_mode": "质量模式",
            "doc_use_cases": "用例",
            "doc_domains": "业务领域",
            "doc_elite_use_cases": "精英用例",
            "doc_high_or_better": "高质量及以上",
            "doc_best_score": "最佳 Inspire 评分",
            "doc_not_scored": "未评分",
            "doc_coverage_picks": "覆盖选择",
            "doc_coverage_pick": "覆盖选择",
            "value_correlation_analysis": "相关性分析",
            "value_pareto_analysis": "帕累托分析",
            "value_funnel_analysis": "漏斗分析",
            "value_document_processing": "文档处理",
            "value_extraction": "信息抽取",
            "value_ai_analysis": "AI 分析",
            "value_simulation": "模拟",
            "value_geospatial_analysis": "地理空间分析",
            "value_predictive_modeling": "预测建模",
            "value_clustering": "聚类分析",
            "value_regression_analysis": "回归分析",
            "value_survival_analysis": "生存分析",
            "value_network_analysis": "网络分析",
            "value_mode_good": "良好",
            "value_mode_high": "高",
            "value_mode_ultra_high": "超高",
            "doc_exec_kicker": "机遇",
            "doc_guide_kicker": "阅读指南",
            "doc_how_to_read": "如何阅读本目录",
            "doc_guide_lead": "每个用例都源自组织自身的元数据，并在同一绝对标尺上评分。可用评分进行比较，用等级进行筛选，用领域页面进行浏览。",
            "doc_inspire_score_title": "Inspire 评分（0 至 10）",
            "doc_inspire_score_body": "综合用例的质量与业务优先级。分数越高越好，本目录按此排序。",
            "doc_quality_tiers_title": "质量等级",
            "doc_quality_tiers_body": "每个用例都会获得从{lowest}到{highest}的等级，反映其在现有数据条件下的具体程度、价值和可构建性。",
            "doc_modes_title": "质量模式",
            "doc_mode_good_body": "保留评级为{medium}及以上的用例。",
            "doc_mode_high_body": "仅保留评级为{high}及以上的用例。",
            "doc_mode_ultra_body": "仅保留在价值、证据、可实施性和可操作性方面均超过绝对标准的用例。不设配额。",
            "doc_this_catalog": "本目录",
            "doc_elite_score_title": "精英评分",
            "doc_elite_score_body": "用于{ultra}模式。它综合业务价值、证据强度、可实施性和可操作性，精英用例在每一项上至少达到 {min_score} 分（满分 10 分）。",
            "doc_coverage_title": "领域覆盖选择",
            "doc_coverage_body": "当某个指定的业务领域没有达到标准的用例时，会保留其中最强的候选用例，并标注为低于标准的覆盖选择。",
            "doc_portfolio_kicker": "组合",
            "doc_portfolio_glance": "用例组合概览",
            "doc_per_domain": "各领域用例数",
            "doc_quality_distribution": "质量分布",
            "doc_top_use_cases": "重点用例",
            "doc_rank": "排名",
            "doc_domain_label": "领域",
            "doc_domain_overview": "关于本领域",
            "doc_what_it_covers": "涵盖内容",
            "doc_top_opportunities": "重点机会",
            "doc_domain_fallback": "本领域汇集了{subdomains}方面的 {count} 个用例。其最佳机会列于下方，详情见后续页面。",
            "doc_open_notebook": "打开 吉尼 笔记本",
            "doc_elite_score": "精英评分",
            "doc_best_of_best": "优中之优",
            "doc_appendix": "附录",
            "doc_methodology": "方法论",
            "strategy_path": "战略路径",
            "kpis": "关键绩效指标",
            "expected_key_results": "预期关键结果",
            "measures_of_success": "成功衡量标准",
            "financial_benefits": "财务收益",
            "non_financial_benefits": "非财务收益",
            "foundations": "数据基础",
            "doc_foundations_title": "共享数据基础",
            "doc_foundations_lead": "两个或更多用例需要的表和关联。优先构建它们可以解锁最多的用例。该顺序只是先后次序，不是时间表。",
            "doc_foundation_tables": "表",
            "doc_foundation_unlocks": "解锁的用例",
            "doc_data_access_standard": "标准发现：Inspire 未读取任何客户数据，只使用了元数据、指标视图定义和使用次数。",
            "doc_data_access_deep": "深度发现：Inspire 读取了 {tables} 张表的聚合信息（行数、日期覆盖范围、空值比例、不同值数量以及非个人字段的少量取值）。没有任何数据行被发送给 AI。",
            "doc_method_1_deep": "Inspire 读取元数据，在深度发现中还会读取基于表计算的聚合信息。它从不将数据行发送给 AI。",
            "doc_guide_lead_deep": "每个用例都来自组织自身的元数据和数据概况，并在同一绝对标准上评分。用分数进行比较，用等级进行筛选，用领域页面进行浏览。",
            "value_not_aligned": "未对齐",
            "kpi_baseline": "基线 {value}，最近 {days} 天，测量于 {date}",
            "kpi_baseline_undated": "基线 {value}，测量于 {date}",
            "doc_method_1": "Inspire 只读取元数据：表名和列名、类型、注释、键和标签，从不读取行数据。",
            "doc_method_2": "它先推导业务背景，再为数据所支持的每个业务领域生成候选用例。",
            "doc_method_3": "去除重复项后，每个候选用例都会按价值和可行性评分，质量关卡会剔除薄弱或缺乏依据的想法。",
            "doc_method_4": "在{ultra}模式下，精英评审会复核每个剩余候选用例，确定性检查会确认所引用的每个表和列都真实存在。",
            "doc_method_5": "最终列表会进行排序、按业务领域分组，并在选择时配套 达塔布里克斯 吉尼 科德 笔记本。",
            "doc_glossary": "术语表",
            "doc_gl_domain": "将相关用例归为一组的业务范畴。",
            "doc_gl_subdomain": "业务领域内更细的主题。",
            "doc_gl_technique": "分析方法，例如预测或分类。",
            "doc_gl_inspire": "综合质量与业务优先级的 0 至 10 分评分。",
            "doc_gl_quality": "为每个用例给出的从{lowest}到{highest}的等级。",
            "doc_gl_elite": "{ultra}模式下基于价值、证据、可实施性和可操作性的评分。",
            "doc_gl_coverage": "为使指定领域有所体现而保留的低于标准的用例。",
            "doc_gl_notebook": "包含 吉尼 科德 指令、用于构建该用例的 达塔布里克斯 笔记本。",
            "doc_notebook_term": "吉尼 笔记本",
            "doc_next_steps": "后续步骤",
            "doc_closing_title": "从用例到成果",
            "doc_closing_body": "选择最重要的用例，打开对应的 吉尼 笔记本，并与团队一起构建第一个用例。以下步骤仅为建议顺序，节奏由团队决定。",
            "doc_step_select": "选择",
            "doc_step_select_body": "挑选评分最高且有明确负责人的用例。",
            "doc_step_validate": "验证数据",
            "doc_step_validate_body": "与数据负责人确认所涉及的表及其关联关系。",
            "doc_step_build": "使用 吉尼 科德 构建",
            "doc_step_build_body": "打开 吉尼 笔记本，生成第一个可运行版本。",
            "doc_step_schedule": "调度",
            "doc_step_schedule_body": "将笔记本作为 达塔布里克斯 作业运行，定期刷新结果表。",
            "doc_step_measure": "衡量与扩展",
            "doc_step_measure_body": "跟踪其带来的业务价值，然后扩展到下一个用例。",
            "doc_agenda": "议程",
            "doc_roadmap": "路线图与后续步骤",
            "doc_use_case_index": "用例索引",
            "doc_continued": "续",
            "doc_not_provided": "未提供",
            "doc_bob_score": "Best-of-Best 评分",
            "elite_cadence": "决策频率",
            "value_cadence_hourly": "每小时",
            "value_cadence_daily": "每天",
            "value_cadence_weekly": "每周",
            "value_cadence_biweekly": "每两周",
            "value_cadence_monthly": "每月",
            "value_cadence_quarterly": "每季度",
            "value_cadence_yearly": "每年",
            "value_cadence_real_time": "实时",
            "value_cadence_per_event": "按事件",
            "doc_generated_by": "由 {brand} {version} 生成 · {date}",
            "doc_disclaimer_ai": "本文档中的用例、评分和描述均由 AI 生成。在做出任何决策之前，请与数据负责人和业务发起人一起核实。",
            "doc_disclaimer_figures": "评分、数值和结果仅为用于确定优先级的估计。本文档中的任何数字均不作保证。",
            "doc_contents": "目录",
            "doc_index_lead": "全部 {count} 个用例，按最终排名排列。",
            "doc_deck_scope": "本演示文稿详细介绍排名前 {count} 的用例。PDF 目录详细介绍全部 {total} 个用例。",
            "doc_deck_scope_all": "本演示文稿详细介绍全部 {total} 个用例，每个用例一张卡片。",
            "value_general_improvement": "一般改进",
            "value_reduce_cost": "降低成本",
            "value_increase_revenue": "增加收入",
            "value_boost_productivity": "提高生产力",
            "value_mitigate_risk": "降低风险",
            "value_protect_revenue": "保护收入",
            "value_align_to_regulations": "符合法规",
            "value_improve_customer_experience": "改善客户体验",
            "value_enable_data_driven_decisions": "实现数据驱动决策",
            "value_optimize_operations": "优化运营",
            "value_empower_talent": "赋能人才",
            "value_enhance_experience": "提升体验",
            "value_drive_innovation": "推动创新",
            "value_achieve_esg": "实现ESG",
            "value_execute_strategy": "执行战略",
            "value_forecasting": "预测",
            "value_classification": "分类",
            "value_anomaly_detection": "异常检测",
            "value_cohort_analysis": "队列分析",
            "value_segmentation": "细分",
            "value_sentiment_analysis": "情感分析",
            "value_trend_analysis": "趋势分析",
            "value_prescriptive_analytics": "规范性分析",
            "value_root_cause_analysis": "根因分析",
            "value_optimization": "优化",
            "value_recommendation": "推荐",
            "value_time_series_analysis": "时间序列分析",
            "value_predictive_analytics": "预测分析",
            "value_descriptive_analytics": "描述性分析",
        },
        "Japanese": {
            "main_title": "データブリックス ジーニー コード を活用したユースケースジェネレーター",
            "intro": "このノートブックには、スキーマに基づいてAIが生成したユースケースが含まれています。以下は、ビジネスドメイン別に生成されたシナリオの概要です。",
            "domain": "ビジネスドメイン",
            "total": "ユースケース総数",
            "summaries": "ユースケース概要",
            "sum_id": "ID",
            "sum_name": "名前",
            "sum_value": "ビジネス価値",
            "sum_outcome": "期待される成果",
            "warning_header": "警告",
            "warning_body": "このノートブックを実行しないでください。デモンストレーションとカタログ作成のみを目的としています。",
            "disclaimer": "このコンテンツはAIによって生成されたものであり、本番環境で使用する前に資格のあるエンジニアによるレビューが必要です。Databricksはこのコンテンツの使用に起因するいかなる問題についても責任を負いません。",
            "detailed_scenarios": "ユースケースの詳細",
            "aspect": "側面",
            "description": "説明",
            "aspect_domain": "ビジネスドメイン",
            "type": "タイプ",
            "analytics_technique": "分析技術",
            "primary_table": "主要テーブル",
            "quality": "品質",
            "priority": "優先度",
            "usecase": "ユースケース",
            "description_label": "説明",
            "value_type_problem": "問題",
            "value_type_risk": "リスク",
            "value_type_opportunity": "機会",
            "value_type_improvement": "改善",
            "value_priority_ultra_high": "極めて高い",
            "value_priority_very_high": "非常に高い",
            "value_priority_high": "高い",
            "value_priority_medium": "中程度",
            "value_priority_low": "低い",
            "value_priority_very_low": "非常に低い",
            "value_priority_ultra_low": "極めて低い",
            "statement": "ステートメント",
            "solution": "ソリューション",
            "aspect_beneficiary": "受益者",
            "beneficiary": "受益者",
            "aspect_sponsor": "スポンサー",
            "sponsor": "スポンサー",
            "business_priority_alignment": "ビジネス優先度整合性",
            "strategic_goals_alignment": "戦略目標との整合性",
            "subdomain": "サブドメイン",
            "aspect_value": "ビジネス価値",
            "business_value": "ビジネス価値",
            "aspect_tables": "関連テーブル",
            "quality_reasons": "品質理由",
            "aspect_ai_function": "AI機能",
            "aspect_analytics_technique": "分析技術",
            "aspect_primary_table": "主要テーブル",
            "aspect_quality": "品質",
            "aspect_priority": "優先度",
            "strategic_alignment": "戦略的整合性",
            "return_on_investment": "投資収益率",
            "reusability": "再利用性",
            "time_to_value": "価値実現までの時間",
            "data_availability": "データ可用性",
            "data_accessibility": "データアクセシビリティ",
            "architecture_fitness": "アーキテクチャ適合性",
            "team_skills": "チームスキル",
            "domain_knowledge": "ドメイン知識",
            "people_allocation": "人員配置",
            "budget_allocation": "予算配分",
            "time_to_production": "本番化までの時間",
            "value_score": "価値スコア",
            "feasibility_score": "実現可能性スコア",
            "priority_score": "優先度スコア",
            "inspire_score": "Inspireスコア",
            "quality_score_header": "品質スコア",
            "pdf_title": "データブリックス ジーニー コード を活用した戦略的AIユースケース",
            "pdf_for": "対象",
            "pdf_exec_summary": "エグゼクティブサマリー",
            "pdf_toc_title": "ユースケースドメイン",
            "pdf_detailed_view": "詳細ユースケースカタログ",
            "pdf_disclaimer_title": "免責事項",
            "pdf_fallback_summary_p1": "本ドキュメントは{business_name}向けに特定された{total_cases}件の高価値分析ユースケースを概説しています。",
            "pdf_fallback_summary_p2": "以下のページでは、ビジネスドメイン別に分類されたこれらの機会の詳細な分析を提供します。",
            "pptx_main_title": "データブリックス ジーニー コード を活用した戦略的AIユースケース",
            "pptx_for": "対象",
            "pptx_disclaimer_title": "免責事項",
            "pptx_domain_suffix": "ユースケース",
            "elite_verdict": "判定",
            "elite_decision": "意思決定",
            "elite_decision_owner": "意思決定者",
            "elite_kpi": "KPI",
            "elite_value_driver": "価値ドライバー",
            "elite_evidence_columns": "根拠となる列",
            "elite_join_path": "結合パス",
            "elite_build_path": "実装パス",
            "elite_red_team_note": "レッドチームの所見",
            "elite_coverage_pick": "ドメインカバレッジ枠（基準未達）",
            "elite_fallback_pick": "{mode}基準未達",
            "elite_fallback_near_miss": "僅差で未達、{mode}基準未達",
            "doc_fallback_pick": "基準未達",
            "doc_fallback_notice": "{mode}基準に達したユースケースはありません。以下は最も近い候補で、いずれも基準未達として明示しています。",
            "elite_reason": "理由",
            "untranslated_notice": "英語で表示（翻訳を利用できません）",
            "more_use_cases": "...ほか {count} 件のユースケース。",
            "executive_summary_not_available": "エグゼクティブサマリーは利用できません。",
            "domain_summary_not_available": "ドメインサマリーは利用できません。",
            "summary_not_available": "サマリーは利用できません。",
            "doc_catalog_kicker": "AIユースケースカタログ",
            "doc_prepared_for": "提出先",
            "doc_generated_on": "作成日",
            "doc_quality_mode": "品質モード",
            "doc_use_cases": "ユースケース",
            "doc_domains": "ビジネスドメイン",
            "doc_elite_use_cases": "エリートユースケース",
            "doc_high_or_better": "高品質以上",
            "doc_best_score": "最高Inspireスコア",
            "doc_not_scored": "未評価",
            "doc_coverage_picks": "カバレッジ枠",
            "doc_coverage_pick": "カバレッジ枠",
            "value_correlation_analysis": "相関分析",
            "value_pareto_analysis": "パレート分析",
            "value_funnel_analysis": "ファネル分析",
            "value_document_processing": "ドキュメント処理",
            "value_extraction": "情報抽出",
            "value_ai_analysis": "AI分析",
            "value_simulation": "シミュレーション",
            "value_geospatial_analysis": "地理空間分析",
            "value_predictive_modeling": "予測モデリング",
            "value_clustering": "クラスタリング",
            "value_regression_analysis": "回帰分析",
            "value_survival_analysis": "生存分析",
            "value_network_analysis": "ネットワーク分析",
            "value_mode_good": "良好",
            "value_mode_high": "高",
            "value_mode_ultra_high": "超高",
            "doc_exec_kicker": "機会",
            "doc_guide_kicker": "読み方ガイド",
            "doc_how_to_read": "このカタログの読み方",
            "doc_guide_lead": "すべてのユースケースは組織自身のメタデータから生まれ、ひとつの絶対的な尺度で評価されています。比較にはスコアを、絞り込みには等級を、閲覧にはドメインページをご利用ください。",
            "doc_inspire_score_title": "Inspireスコア（0〜10）",
            "doc_inspire_score_body": "ユースケースの品質とビジネス上の優先度を組み合わせたスコアです。高いほど良く、カタログはこの順に並んでいます。",
            "doc_quality_tiers_title": "品質等級",
            "doc_quality_tiers_body": "各ユースケースには、利用可能なデータでどれだけ具体的で価値があり構築可能かを示す、「{lowest}」から「{highest}」までの等級が付与されます。",
            "doc_modes_title": "品質モード",
            "doc_mode_good_body": "「{medium}」以上と評価されたユースケースを残します。",
            "doc_mode_high_body": "「{high}」以上と評価されたユースケースのみを残します。",
            "doc_mode_ultra_body": "価値、根拠、実装可能性、実行可能性のすべてで絶対的な基準を満たすユースケースのみを残します。件数の割り当てはありません。",
            "doc_this_catalog": "このカタログ",
            "doc_elite_score_title": "エリートスコア",
            "doc_elite_score_body": "{ultra}モードで使用されます。ビジネス価値、根拠の強さ、実装可能性、実行可能性を評価し、エリートユースケースは各項目で10点中{min_score}点以上を獲得しています。",
            "doc_coverage_title": "ドメインカバレッジ枠",
            "doc_coverage_body": "指定されたビジネスドメインに基準を超えるユースケースがない場合、最も有力な候補を残し、基準未達のカバレッジ枠として明示します。",
            "doc_portfolio_kicker": "ポートフォリオ",
            "doc_portfolio_glance": "ポートフォリオの概要",
            "doc_per_domain": "ドメイン別ユースケース数",
            "doc_quality_distribution": "品質の分布",
            "doc_top_use_cases": "主要ユースケース",
            "doc_rank": "順位",
            "doc_domain_label": "ドメイン",
            "doc_domain_overview": "このドメインについて",
            "doc_what_it_covers": "対象範囲",
            "doc_top_opportunities": "主な機会",
            "doc_domain_fallback": "このドメインには、{subdomains}にわたる{count}件のユースケースがあります。有力な機会を以下に示し、詳細は次ページ以降で説明します。",
            "doc_open_notebook": "ジーニー ノートブックを開く",
            "doc_elite_score": "エリートスコア",
            "doc_best_of_best": "ベスト・オブ・ベスト",
            "doc_appendix": "付録",
            "doc_methodology": "手法",
            "strategy_path": "戦略パス",
            "kpis": "KPI",
            "expected_key_results": "期待される主要成果",
            "measures_of_success": "成功指標",
            "financial_benefits": "財務的メリット",
            "non_financial_benefits": "非財務的メリット",
            "foundations": "データ基盤",
            "doc_foundations_title": "共有データ基盤",
            "doc_foundations_lead": "2 つ以上のユースケースが必要とするテーブルと結合です。これらを先に構築すると、最も多くのユースケースが利用可能になります。順序は手順であり、スケジュールではありません。",
            "doc_foundation_tables": "テーブル",
            "doc_foundation_unlocks": "利用可能になるユースケース",
            "doc_data_access_standard": "標準検出: Inspire は顧客データを一切読み取っていません。メタデータ、メトリクスビューの定義、利用回数のみを使用しました。",
            "doc_data_access_deep": "詳細検出: Inspire は {tables} 個のテーブルから集計値を読み取りました (行数、日付の範囲、NULL の割合、一意の値の数、個人情報ではない列の短い値リスト)。データ行は AI に送信されていません。",
            "doc_method_1_deep": "Inspire はメタデータを読み取り、詳細検出ではテーブル上で計算した集計値も読み取ります。データ行を AI に送信することはありません。",
            "doc_guide_lead_deep": "各ユースケースは組織のメタデータとデータプロファイルに基づき、単一の絶対尺度で評価されます。スコアで比較し、ティアで絞り込み、ドメインページで移動してください。",
            "value_not_aligned": "整合なし",
            "kpi_baseline": "ベースライン {value}、直近 {days} 日間、{date} 測定",
            "kpi_baseline_undated": "ベースライン {value}、{date} 測定",
            "doc_method_1": "Inspireが読み取るのはメタデータのみです。テーブル名と列名、型、コメント、キー、タグを読み取り、行データは一切読み取りません。",
            "doc_method_2": "ビジネスの文脈を導き出したうえで、データが支えるビジネスドメインごとに候補となるユースケースを生成します。",
            "doc_method_3": "重複を取り除き、各候補を価値と実現可能性で評価し、品質ゲートで根拠の弱いアイデアを除外します。",
            "doc_method_4": "{ultra}モードでは、エリート審査が残りの各候補を確認し、決定論的なチェックで引用されたすべてのテーブルと列の存在を確かめます。",
            "doc_method_5": "最終リストは順位付けされ、ビジネスドメインごとにまとめられ、選択した場合はデータブリックス ジーニー コードのノートブックが付きます。",
            "doc_glossary": "用語集",
            "doc_gl_domain": "関連するユースケースをまとめるビジネス領域。",
            "doc_gl_subdomain": "ビジネスドメイン内のより細かなテーマ。",
            "doc_gl_technique": "予測や分類などの分析手法。",
            "doc_gl_inspire": "品質とビジネス上の優先度を組み合わせた0〜10のスコア。",
            "doc_gl_quality": "各ユースケースに付与される、「{lowest}」から「{highest}」までの等級。",
            "doc_gl_elite": "価値、根拠、実装可能性、実行可能性に基づく{ultra}モードのスコア。",
            "doc_gl_coverage": "指定されたドメインを反映するために残された、基準未達のユースケース。",
            "doc_gl_notebook": "ユースケースを構築するためのジーニー コードの指示を含むデータブリックスのノートブック。",
            "doc_notebook_term": "ジーニー ノートブック",
            "doc_next_steps": "次のステップ",
            "doc_closing_title": "ユースケースから成果へ",
            "doc_closing_body": "最も重要なユースケースを選び、そのジーニー ノートブックを開いて、チームと一緒に最初のひとつを構築しましょう。手順は推奨の順序であり、進めるペースはチームが決めます。",
            "doc_step_select": "選定",
            "doc_step_select_body": "スコアが最も高く、スポンサーが明確なユースケースを選びます。",
            "doc_step_validate": "データの検証",
            "doc_step_validate_body": "関係するテーブルとその結合をデータオーナーと確認します。",
            "doc_step_build": "ジーニー コードで構築",
            "doc_step_build_body": "ジーニー ノートブックを開き、最初に動くバージョンを生成します。",
            "doc_step_schedule": "スケジュール",
            "doc_step_schedule_body": "ノートブックをデータブリックスのジョブとして実行し、結果テーブルを更新します。",
            "doc_step_measure": "測定と拡大",
            "doc_step_measure_body": "得られるビジネス価値を追跡し、次のユースケースへ展開します。",
            "doc_agenda": "アジェンダ",
            "doc_roadmap": "ロードマップと次のステップ",
            "doc_use_case_index": "ユースケース索引",
            "doc_continued": "続き",
            "doc_not_provided": "未記入",
            "doc_bob_score": "Best-of-Best スコア",
            "elite_cadence": "意思決定の頻度",
            "value_cadence_hourly": "毎時",
            "value_cadence_daily": "毎日",
            "value_cadence_weekly": "毎週",
            "value_cadence_biweekly": "隔週",
            "value_cadence_monthly": "毎月",
            "value_cadence_quarterly": "四半期ごと",
            "value_cadence_yearly": "毎年",
            "value_cadence_real_time": "リアルタイム",
            "value_cadence_per_event": "イベントごと",
            "doc_generated_by": "{brand} {version} により生成 · {date}",
            "doc_disclaimer_ai": "この文書のユースケース、スコア、説明文は AI によって生成されています。意思決定の前に、データオーナーと事業スポンサーとともに検証してください。",
            "doc_disclaimer_figures": "スコア、数値、成果は優先順位付けのための見積もりにすぎません。この文書の数値はいずれも保証されません。",
            "doc_contents": "目次",
            "doc_index_lead": "全 {count} 件のユースケースを最終順位の順に掲載しています。",
            "doc_deck_scope": "このプレゼンテーションでは上位 {count} 件のユースケースを詳しく説明します。PDF カタログでは全 {total} 件を詳しく説明します。",
            "doc_deck_scope_all": "このプレゼンテーションでは全 {total} 件のユースケースを、1 件ずつカードで詳しく説明します。",
            "value_general_improvement": "一般的な改善",
            "value_reduce_cost": "コスト削減",
            "value_increase_revenue": "収益増加",
            "value_boost_productivity": "生産性向上",
            "value_mitigate_risk": "リスク軽減",
            "value_protect_revenue": "収益保護",
            "value_align_to_regulations": "規制遵守",
            "value_improve_customer_experience": "顧客体験の改善",
            "value_enable_data_driven_decisions": "データ駆動型意思決定の実現",
            "value_optimize_operations": "業務最適化",
            "value_empower_talent": "人材育成",
            "value_enhance_experience": "体験向上",
            "value_drive_innovation": "イノベーション推進",
            "value_achieve_esg": "ESG達成",
            "value_execute_strategy": "戦略実行",
            "value_forecasting": "予測",
            "value_classification": "分類",
            "value_anomaly_detection": "異常検知",
            "value_cohort_analysis": "コホート分析",
            "value_segmentation": "セグメンテーション",
            "value_sentiment_analysis": "感情分析",
            "value_trend_analysis": "トレンド分析",
            "value_prescriptive_analytics": "処方的分析",
            "value_root_cause_analysis": "根本原因分析",
            "value_optimization": "最適化",
            "value_recommendation": "レコメンデーション",
            "value_time_series_analysis": "時系列分析",
            "value_predictive_analytics": "予測分析",
            "value_descriptive_analytics": "記述的分析",
        },
        "Korean": {
            "main_title": "데이터브릭스 지니 코드 기반의 유스케이스 생성기",
            "intro": "이 노트북에는 스키마를 기반으로 AI가 생성한 유스케이스가 포함되어 있습니다. 아래는 비즈니스 도메인별 생성된 시나리오 요약입니다.",
            "domain": "비즈니스 도메인",
            "total": "총 유스케이스",
            "summaries": "유스케이스 요약",
            "sum_id": "ID",
            "sum_name": "이름",
            "sum_value": "비즈니스 가치",
            "sum_outcome": "예상 결과",
            "warning_header": "경고",
            "warning_body": "이 노트북을 실행하지 마십시오. 데모 및 카탈로그 작성 목적으로만 사용됩니다.",
            "disclaimer": "이 콘텐츠는 AI가 생성한 것이며 프로덕션에 사용하기 전에 자격을 갖춘 엔지니어의 검토가 필요합니다. Databricks는 이 콘텐츠 사용으로 인해 발생하는 문제에 대해 책임을 지지 않습니다.",
            "detailed_scenarios": "유스케이스 세부정보",
            "aspect": "측면",
            "description": "설명",
            "aspect_domain": "비즈니스 도메인",
            "type": "유형",
            "analytics_technique": "분석 기법",
            "primary_table": "주요 테이블",
            "quality": "품질",
            "priority": "우선순위",
            "usecase": "유스케이스",
            "description_label": "설명",
            "value_type_problem": "문제",
            "value_type_risk": "위험",
            "value_type_opportunity": "기회",
            "value_type_improvement": "개선",
            "value_priority_ultra_high": "초고",
            "value_priority_very_high": "매우 높음",
            "value_priority_high": "높음",
            "value_priority_medium": "보통",
            "value_priority_low": "낮음",
            "value_priority_very_low": "매우 낮음",
            "value_priority_ultra_low": "초저",
            "statement": "설명",
            "solution": "솔루션",
            "aspect_beneficiary": "수혜자",
            "beneficiary": "수혜자",
            "aspect_sponsor": "후원자",
            "sponsor": "후원자",
            "business_priority_alignment": "비즈니스 우선순위 정렬",
            "strategic_goals_alignment": "전략 목표 정렬",
            "subdomain": "하위 도메인",
            "aspect_value": "비즈니스 가치",
            "business_value": "비즈니스 가치",
            "aspect_tables": "관련 테이블",
            "quality_reasons": "품질 사유",
            "aspect_ai_function": "AI 기능",
            "aspect_analytics_technique": "분석 기법",
            "aspect_primary_table": "주요 테이블",
            "aspect_quality": "품질",
            "aspect_priority": "우선순위",
            "strategic_alignment": "전략적 정렬",
            "return_on_investment": "투자 수익률",
            "reusability": "재사용성",
            "time_to_value": "가치 실현 시간",
            "data_availability": "데이터 가용성",
            "data_accessibility": "데이터 접근성",
            "architecture_fitness": "아키텍처 적합성",
            "team_skills": "팀 역량",
            "domain_knowledge": "도메인 지식",
            "people_allocation": "인력 배치",
            "budget_allocation": "예산 배분",
            "time_to_production": "프로덕션까지 시간",
            "value_score": "가치 점수",
            "feasibility_score": "실현 가능성 점수",
            "priority_score": "우선순위 점수",
            "inspire_score": "Inspire 점수",
            "quality_score_header": "품질 점수",
            "pdf_title": "데이터브릭스 지니 코드 기반의 전략적 AI 유스케이스",
            "pdf_for": "대상",
            "pdf_exec_summary": "경영진 요약",
            "pdf_toc_title": "유스케이스 도메인",
            "pdf_detailed_view": "상세 유스케이스 카탈로그",
            "pdf_disclaimer_title": "면책조항",
            "pdf_fallback_summary_p1": "이 문서는 {business_name}을 위해 식별된 {total_cases}개의 고가치 분석 유스케이스를 설명합니다.",
            "pdf_fallback_summary_p2": "다음 페이지에서는 비즈니스 도메인별로 분류된 이러한 기회의 상세 분석을 제공합니다.",
            "pptx_main_title": "데이터브릭스 지니 코드 기반의 전략적 AI 유스케이스",
            "pptx_for": "대상",
            "pptx_disclaimer_title": "면책조항",
            "pptx_domain_suffix": "유스케이스",
            "elite_verdict": "판정",
            "elite_decision": "의사결정",
            "elite_decision_owner": "의사결정 책임자",
            "elite_kpi": "핵심 성과 지표",
            "elite_value_driver": "가치 동인",
            "elite_evidence_columns": "근거 열",
            "elite_join_path": "조인 경로",
            "elite_build_path": "구현 경로",
            "elite_red_team_note": "레드팀 의견",
            "elite_coverage_pick": "도메인 커버리지 선택, 기준 미달",
            "elite_fallback_pick": "{mode} 기준 미달",
            "elite_fallback_near_miss": "근소한 차이, {mode} 기준 미달",
            "doc_fallback_pick": "기준 미달",
            "doc_fallback_notice": "{mode} 기준을 충족한 유스케이스가 없습니다. 다음은 가장 근접한 후보이며, 각각 기준 미달로 표시됩니다.",
            "elite_reason": "사유",
            "untranslated_notice": "영어로 표시됨(번역 불가)",
            "more_use_cases": "...외 {count}개의 사용 사례.",
            "executive_summary_not_available": "경영진 요약을 사용할 수 없습니다.",
            "domain_summary_not_available": "도메인 요약을 사용할 수 없습니다.",
            "summary_not_available": "요약을 사용할 수 없습니다.",
            "doc_catalog_kicker": "AI 유스케이스 카탈로그",
            "doc_prepared_for": "제출 대상",
            "doc_generated_on": "생성일",
            "doc_quality_mode": "품질 모드",
            "doc_use_cases": "유스케이스",
            "doc_domains": "비즈니스 도메인",
            "doc_elite_use_cases": "엘리트 유스케이스",
            "doc_high_or_better": "높음 이상 품질",
            "doc_best_score": "최고 Inspire 점수",
            "doc_not_scored": "평가되지 않음",
            "doc_coverage_picks": "커버리지 선택",
            "doc_coverage_pick": "커버리지 선택",
            "value_correlation_analysis": "상관 분석",
            "value_pareto_analysis": "파레토 분석",
            "value_funnel_analysis": "퍼널 분석",
            "value_document_processing": "문서 처리",
            "value_extraction": "정보 추출",
            "value_ai_analysis": "AI 분석",
            "value_simulation": "시뮬레이션",
            "value_geospatial_analysis": "지리공간 분석",
            "value_predictive_modeling": "예측 모델링",
            "value_clustering": "클러스터링",
            "value_regression_analysis": "회귀 분석",
            "value_survival_analysis": "생존 분석",
            "value_network_analysis": "네트워크 분석",
            "value_mode_good": "양호",
            "value_mode_high": "높음",
            "value_mode_ultra_high": "최상",
            "doc_exec_kicker": "기회",
            "doc_guide_kicker": "읽기 가이드",
            "doc_how_to_read": "이 카탈로그를 읽는 방법",
            "doc_guide_lead": "모든 유스케이스는 조직 자체의 메타데이터에서 도출되며 하나의 절대 척도로 평가됩니다. 비교에는 점수를, 필터링에는 등급을, 탐색에는 도메인 페이지를 활용하세요.",
            "doc_inspire_score_title": "Inspire 점수(0~10)",
            "doc_inspire_score_body": "유스케이스의 품질과 비즈니스 우선순위를 결합한 점수입니다. 높을수록 좋으며 카탈로그는 이 점수 순으로 정렬됩니다.",
            "doc_quality_tiers_title": "품질 등급",
            "doc_quality_tiers_body": "각 유스케이스는 사용 가능한 데이터로 얼마나 구체적이고 가치 있으며 구축 가능한지를 반영하여 {lowest}부터 {highest}까지의 등급을 받습니다.",
            "doc_modes_title": "품질 모드",
            "doc_mode_good_body": "{medium} 이상으로 평가된 유스케이스를 유지합니다.",
            "doc_mode_high_body": "{high} 이상으로 평가된 유스케이스만 유지합니다.",
            "doc_mode_ultra_body": "가치, 근거, 구현 가능성, 실행 가능성 모두에서 절대 기준을 통과한 유스케이스만 유지합니다. 할당량은 없습니다.",
            "doc_this_catalog": "이 카탈로그",
            "doc_elite_score_title": "엘리트 점수",
            "doc_elite_score_body": "{ultra} 모드에서 사용됩니다. 비즈니스 가치, 근거의 강도, 구현 가능성, 실행 가능성을 평가하며, 엘리트 유스케이스는 각 항목에서 10점 만점에 {min_score}점 이상을 받습니다.",
            "doc_coverage_title": "도메인 커버리지 선택",
            "doc_coverage_body": "요청된 비즈니스 도메인에 기준을 넘는 유스케이스가 없으면, 가장 강력한 후보를 유지하고 기준 미달 커버리지 선택으로 표시합니다.",
            "doc_portfolio_kicker": "포트폴리오",
            "doc_portfolio_glance": "포트폴리오 한눈에 보기",
            "doc_per_domain": "도메인별 유스케이스",
            "doc_quality_distribution": "품질 분포",
            "doc_top_use_cases": "주요 유스케이스",
            "doc_rank": "순위",
            "doc_domain_label": "도메인",
            "doc_domain_overview": "이 도메인 소개",
            "doc_what_it_covers": "다루는 범위",
            "doc_top_opportunities": "주요 기회",
            "doc_domain_fallback": "이 도메인에는 {subdomains} 전반의 유스케이스 {count}개가 포함됩니다. 가장 강력한 기회는 아래에 정리되어 있으며 다음 페이지에서 자세히 설명합니다.",
            "doc_open_notebook": "지니 노트북 열기",
            "doc_elite_score": "엘리트 점수",
            "doc_best_of_best": "최고 중의 최고",
            "doc_appendix": "부록",
            "doc_methodology": "방법론",
            "strategy_path": "전략 경로",
            "kpis": "KPI",
            "expected_key_results": "기대 핵심 결과",
            "measures_of_success": "성공 지표",
            "financial_benefits": "재무적 효과",
            "non_financial_benefits": "비재무적 효과",
            "foundations": "데이터 기반",
            "doc_foundations_title": "공유 데이터 기반",
            "doc_foundations_lead": "두 개 이상의 유스케이스에 필요한 테이블과 조인입니다. 이를 먼저 구축하면 가장 많은 유스케이스를 실행할 수 있습니다. 순서는 일정이 아니라 진행 순서입니다.",
            "doc_foundation_tables": "테이블",
            "doc_foundation_unlocks": "가능해지는 유스케이스",
            "doc_data_access_standard": "표준 탐색: Inspire는 고객 데이터를 읽지 않았습니다. 메타데이터, 메트릭 뷰 정의, 사용 횟수만 사용했습니다.",
            "doc_data_access_deep": "심층 탐색: Inspire는 {tables}개 테이블에서 집계값을 읽었습니다(행 수, 날짜 범위, NULL 비율, 고유 값 수, 개인 정보가 아닌 열의 짧은 값 목록). AI로 전송된 데이터 행은 없습니다.",
            "doc_method_1_deep": "Inspire는 메타데이터를 읽고, 심층 탐색에서는 테이블에서 계산한 집계값도 읽습니다. 데이터 행은 AI로 전송하지 않습니다.",
            "doc_guide_lead_deep": "모든 유스케이스는 조직의 메타데이터와 데이터 프로필에서 도출되며 하나의 절대 척도로 평가됩니다. 점수로 비교하고, 등급으로 필터링하고, 도메인 페이지로 탐색하세요.",
            "value_not_aligned": "정렬되지 않음",
            "kpi_baseline": "기준값 {value}, 최근 {days}일, {date} 측정",
            "kpi_baseline_undated": "기준값 {value}, {date} 측정",
            "doc_method_1": "Inspire는 메타데이터만 읽습니다. 테이블 및 열 이름, 유형, 주석, 키, 태그를 읽으며 행 데이터는 절대 읽지 않습니다.",
            "doc_method_2": "비즈니스 맥락을 도출한 다음, 데이터가 뒷받침하는 모든 비즈니스 도메인에 대해 후보 유스케이스를 생성합니다.",
            "doc_method_3": "중복을 제거하고, 각 후보를 가치와 실현 가능성으로 평가하며, 품질 게이트가 약하거나 근거 없는 아이디어를 걸러냅니다.",
            "doc_method_4": "{ultra} 모드에서는 엘리트 심사가 남은 각 후보를 검토하고, 결정론적 검사로 인용된 모든 테이블과 열이 존재하는지 확인합니다.",
            "doc_method_5": "최종 목록은 순위가 매겨지고 비즈니스 도메인별로 묶이며, 선택한 경우 데이터브릭스 지니 코드 노트북과 함께 제공됩니다.",
            "doc_glossary": "용어집",
            "doc_gl_domain": "관련 유스케이스를 묶는 비즈니스 영역.",
            "doc_gl_subdomain": "비즈니스 도메인 안의 더 세부적인 주제.",
            "doc_gl_technique": "예측이나 분류와 같은 분석 방법.",
            "doc_gl_inspire": "품질과 비즈니스 우선순위를 결합한 0~10점 점수.",
            "doc_gl_quality": "각 유스케이스에 부여되는 {lowest}부터 {highest}까지의 등급.",
            "doc_gl_elite": "가치, 근거, 구현 가능성, 실행 가능성에 대한 {ultra} 모드 점수.",
            "doc_gl_coverage": "요청된 도메인이 대표되도록 기준 미달 상태로 유지된 유스케이스.",
            "doc_gl_notebook": "유스케이스 구축을 위한 지니 코드 지침이 담긴 데이터브릭스 노트북.",
            "doc_notebook_term": "지니 노트북",
            "doc_next_steps": "다음 단계",
            "doc_closing_title": "유스케이스에서 성과로",
            "doc_closing_body": "가장 중요한 유스케이스를 고르고, 해당 지니 노트북을 열어 팀과 함께 첫 번째 유스케이스를 구축하세요. 단계는 권장 순서이며, 진행 속도는 팀이 정합니다.",
            "doc_step_select": "선정",
            "doc_step_select_body": "점수가 가장 높고 스폰서가 분명한 유스케이스를 고릅니다.",
            "doc_step_validate": "데이터 검증",
            "doc_step_validate_body": "데이터 담당자와 함께 관련 테이블과 조인을 확인합니다.",
            "doc_step_build": "지니 코드로 구축",
            "doc_step_build_body": "지니 노트북을 열고 첫 번째 작동 버전을 생성합니다.",
            "doc_step_schedule": "예약 실행",
            "doc_step_schedule_body": "노트북을 결과 테이블을 갱신하는 데이터브릭스 작업으로 실행합니다.",
            "doc_step_measure": "측정 및 확장",
            "doc_step_measure_body": "창출되는 비즈니스 가치를 추적한 다음 다음 유스케이스로 확장합니다.",
            "doc_agenda": "안건",
            "doc_roadmap": "로드맵 및 다음 단계",
            "doc_use_case_index": "유스케이스 색인",
            "doc_continued": "계속",
            "doc_not_provided": "제공되지 않음",
            "doc_bob_score": "Best-of-Best 점수",
            "elite_cadence": "의사 결정 주기",
            "value_cadence_hourly": "매시간",
            "value_cadence_daily": "매일",
            "value_cadence_weekly": "매주",
            "value_cadence_biweekly": "격주",
            "value_cadence_monthly": "매월",
            "value_cadence_quarterly": "분기별",
            "value_cadence_yearly": "매년",
            "value_cadence_real_time": "실시간",
            "value_cadence_per_event": "이벤트별",
            "doc_generated_by": "{brand} {version}에서 생성 · {date}",
            "doc_disclaimer_ai": "이 문서의 사용 사례, 점수 및 설명은 AI가 생성했습니다. 결정을 내리기 전에 데이터 소유자 및 비즈니스 스폰서와 함께 검증하세요.",
            "doc_disclaimer_figures": "점수, 값 및 결과는 우선순위를 정하기 위한 추정치일 뿐입니다. 이 문서의 어떤 수치도 보장되지 않습니다.",
            "doc_contents": "목차",
            "doc_index_lead": "전체 {count}개 사용 사례를 최종 순위 순으로 나열합니다.",
            "doc_deck_scope": "이 프레젠테이션은 상위 {count}개 사용 사례를 자세히 설명합니다. PDF 카탈로그는 전체 {total}개를 자세히 설명합니다.",
            "doc_deck_scope_all": "이 프레젠테이션은 전체 {total}개 사용 사례를 사례별 카드로 자세히 설명합니다.",
            "value_general_improvement": "일반 개선",
            "value_reduce_cost": "비용 절감",
            "value_increase_revenue": "수익 증대",
            "value_boost_productivity": "생산성 향상",
            "value_mitigate_risk": "위험 완화",
            "value_protect_revenue": "수익 보호",
            "value_align_to_regulations": "규정 준수",
            "value_improve_customer_experience": "고객 경험 개선",
            "value_enable_data_driven_decisions": "데이터 기반 의사결정 지원",
            "value_optimize_operations": "운영 최적화",
            "value_empower_talent": "인재 역량 강화",
            "value_enhance_experience": "경험 향상",
            "value_drive_innovation": "혁신 추진",
            "value_achieve_esg": "ESG 달성",
            "value_execute_strategy": "전략 실행",
            "value_forecasting": "예측",
            "value_classification": "분류",
            "value_anomaly_detection": "이상 탐지",
            "value_cohort_analysis": "코호트 분석",
            "value_segmentation": "세분화",
            "value_sentiment_analysis": "감정 분석",
            "value_trend_analysis": "추세 분석",
            "value_prescriptive_analytics": "처방적 분석",
            "value_root_cause_analysis": "근본 원인 분석",
            "value_optimization": "최적화",
            "value_recommendation": "추천",
            "value_time_series_analysis": "시계열 분석",
            "value_predictive_analytics": "예측 분석",
            "value_descriptive_analytics": "기술적 분석",
        },
        "Hindi": {
            "main_title": "डेटाब्रिक्स जीनी कोड द्वारा संचालित उपयोग केस जनरेटर",
            "intro": "इस नोटबुक में आपके स्कीमा के आधार पर AI-जनित उपयोग केस शामिल हैं। नीचे व्यापार डोमेन द्वारा जनित परिदृश्यों का सारांश है।",
            "domain": "व्यापार डोमेन",
            "total": "कुल उपयोग केस",
            "summaries": "उपयोग केस सारांश",
            "sum_id": "ID",
            "sum_name": "नाम",
            "sum_value": "व्यापारिक मूल्य",
            "sum_outcome": "अपेक्षित परिणाम",
            "warning_header": "चेतावनी",
            "warning_body": "इस नोटबुक को न चलाएं। यह केवल प्रदर्शन और कैटलॉगिंग उद्देश्यों के लिए है।",
            "disclaimer": "यह सामग्री AI-जनित है और उत्पादन में उपयोग करने से पहले एक योग्य इंजीनियर द्वारा समीक्षा की जानी चाहिए। Databricks इस सामग्री के उपयोग से उत्पन्न किसी भी समस्या के लिए उत्तरदायी नहीं है।",
            "detailed_scenarios": "उपयोग केस विवरण",
            "aspect": "पहलू",
            "description": "विवरण",
            "aspect_domain": "व्यापार डोमेन",
            "type": "प्रकार",
            "analytics_technique": "विश्लेषण तकनीक",
            "primary_table": "प्राथमिक तालिका",
            "quality": "गुणवत्ता",
            "priority": "प्राथमिकता",
            "usecase": "उपयोग केस",
            "description_label": "विवरण",
            "value_type_problem": "समस्या",
            "value_type_risk": "जोखिम",
            "value_type_opportunity": "अवसर",
            "value_type_improvement": "सुधार",
            "value_priority_ultra_high": "अत्यधिक उच्च",
            "value_priority_very_high": "बहुत उच्च",
            "value_priority_high": "उच्च",
            "value_priority_medium": "मध्यम",
            "value_priority_low": "कम",
            "value_priority_very_low": "बहुत कम",
            "value_priority_ultra_low": "अत्यधिक कम",
            "statement": "कथन",
            "solution": "समाधान",
            "aspect_beneficiary": "लाभार्थी",
            "beneficiary": "लाभार्थी",
            "aspect_sponsor": "प्रायोजक",
            "sponsor": "प्रायोजक",
            "business_priority_alignment": "व्यापार प्राथमिकता संरेखण",
            "strategic_goals_alignment": "रणनीतिक लक्ष्य संरेखण",
            "subdomain": "उप-डोमेन",
            "aspect_value": "व्यापारिक मूल्य",
            "business_value": "व्यापारिक मूल्य",
            "aspect_tables": "शामिल तालिकाएं",
            "quality_reasons": "गुणवत्ता कारण",
            "aspect_ai_function": "AI फ़ंक्शन",
            "aspect_analytics_technique": "विश्लेषण तकनीक",
            "aspect_primary_table": "प्राथमिक तालिका",
            "aspect_quality": "गुणवत्ता",
            "aspect_priority": "प्राथमिकता",
            "strategic_alignment": "रणनीतिक संरेखण",
            "return_on_investment": "निवेश पर प्रतिफल",
            "reusability": "पुन: प्रयोज्यता",
            "time_to_value": "मूल्य तक समय",
            "data_availability": "डेटा उपलब्धता",
            "data_accessibility": "डेटा पहुंच",
            "architecture_fitness": "आर्किटेक्चर उपयुक्तता",
            "team_skills": "टीम कौशल",
            "domain_knowledge": "डोमेन ज्ञान",
            "people_allocation": "लोग आवंटन",
            "budget_allocation": "बजट आवंटन",
            "time_to_production": "उत्पादन तक समय",
            "value_score": "मूल्य स्कोर",
            "feasibility_score": "व्यवहार्यता स्कोर",
            "priority_score": "प्राथमिकता स्कोर",
            "inspire_score": "Inspire स्कोर",
            "quality_score_header": "गुणवत्ता स्कोर",
            "pdf_title": "डेटाब्रिक्स जीनी कोड द्वारा संचालित रणनीतिक AI उपयोग केस",
            "pdf_for": "के लिए",
            "pdf_exec_summary": "कार्यकारी सारांश",
            "pdf_toc_title": "उपयोग केस डोमेन",
            "pdf_detailed_view": "विस्तृत उपयोग केस कैटलॉग",
            "pdf_disclaimer_title": "अस्वीकरण",
            "pdf_fallback_summary_p1": "यह दस्तावेज़ {business_name} के लिए पहचाने गए {total_cases} उच्च-मूल्य विश्लेषणात्मक उपयोग केस का वर्णन करता है।",
            "pdf_fallback_summary_p2": "निम्नलिखित पृष्ठ व्यापार डोमेन द्वारा वर्गीकृत इन अवसरों का विस्तृत विश्लेषण प्रदान करते हैं।",
            "pptx_main_title": "डेटाब्रिक्स जीनी कोड द्वारा संचालित रणनीतिक AI उपयोग केस",
            "pptx_for": "के लिए",
            "pptx_disclaimer_title": "अस्वीकरण",
            "pptx_domain_suffix": "उपयोग केस",
            "elite_verdict": "निर्णय परिणाम",
            "elite_decision": "निर्णय",
            "elite_decision_owner": "निर्णय स्वामी",
            "elite_kpi": "मुख्य प्रदर्शन संकेतक",
            "elite_value_driver": "मूल्य चालक",
            "elite_evidence_columns": "साक्ष्य कॉलम",
            "elite_join_path": "जॉइन पथ",
            "elite_build_path": "कार्यान्वयन पथ",
            "elite_red_team_note": "रेड टीम टिप्पणी",
            "elite_coverage_pick": "डोमेन कवरेज चयन, मानक से नीचे",
            "elite_fallback_pick": "{mode} मानक से नीचे",
            "elite_fallback_near_miss": "लगभग चयनित, {mode} मानक से नीचे",
            "doc_fallback_pick": "मानक से नीचे",
            "doc_fallback_notice": "कोई भी उपयोग केस {mode} मानक तक नहीं पहुंचा। ये सबसे निकटतम उम्मीदवार हैं, प्रत्येक को मानक से नीचे चिह्नित किया गया है।",
            "elite_reason": "कारण",
            "untranslated_notice": "अंग्रेज़ी में दिखाया गया (अनुवाद उपलब्ध नहीं)",
            "more_use_cases": "...और {count} अन्य उपयोग मामले।",
            "executive_summary_not_available": "कार्यकारी सारांश उपलब्ध नहीं है।",
            "domain_summary_not_available": "डोमेन सारांश उपलब्ध नहीं है।",
            "summary_not_available": "सारांश उपलब्ध नहीं है।",
            "doc_catalog_kicker": "AI उपयोग केस कैटलॉग",
            "doc_prepared_for": "इनके लिए तैयार",
            "doc_generated_on": "बनाने की तिथि",
            "doc_quality_mode": "गुणवत्ता मोड",
            "doc_use_cases": "उपयोग केस",
            "doc_domains": "व्यापार डोमेन",
            "doc_elite_use_cases": "एलीट उपयोग केस",
            "doc_high_or_better": "उच्च गुणवत्ता या उससे बेहतर",
            "doc_best_score": "सर्वश्रेष्ठ Inspire स्कोर",
            "doc_not_scored": "स्कोर नहीं किया गया",
            "doc_coverage_picks": "कवरेज चयन",
            "doc_coverage_pick": "कवरेज चयन",
            "value_correlation_analysis": "सहसंबंध विश्लेषण",
            "value_pareto_analysis": "पैरेटो विश्लेषण",
            "value_funnel_analysis": "फ़नल विश्लेषण",
            "value_document_processing": "दस्तावेज़ प्रसंस्करण",
            "value_extraction": "सूचना निष्कर्षण",
            "value_ai_analysis": "AI विश्लेषण",
            "value_simulation": "सिमुलेशन",
            "value_geospatial_analysis": "भू-स्थानिक विश्लेषण",
            "value_predictive_modeling": "भविष्य कथन मॉडलिंग",
            "value_clustering": "क्लस्टरिंग",
            "value_regression_analysis": "प्रतिगमन विश्लेषण",
            "value_survival_analysis": "अस्तित्व विश्लेषण",
            "value_network_analysis": "नेटवर्क विश्लेषण",
            "value_mode_good": "अच्छा",
            "value_mode_high": "उच्च",
            "value_mode_ultra_high": "अति उच्च",
            "doc_exec_kicker": "अवसर",
            "doc_guide_kicker": "पढ़ने की मार्गदर्शिका",
            "doc_how_to_read": "इस कैटलॉग को कैसे पढ़ें",
            "doc_guide_lead": "हर उपयोग केस संगठन के अपने मेटाडेटा से आता है और एक ही निरपेक्ष पैमाने पर स्कोर किया जाता है। तुलना के लिए स्कोर, छाँटने के लिए स्तर और नेविगेशन के लिए डोमेन पेजों का उपयोग करें।",
            "doc_inspire_score_title": "Inspire स्कोर, 0 से 10 तक",
            "doc_inspire_score_body": "यह उपयोग केस की गुणवत्ता को उसकी व्यापारिक प्राथमिकता के साथ जोड़ता है। जितना अधिक, उतना बेहतर, और कैटलॉग इसी क्रम में है।",
            "doc_quality_tiers_title": "गुणवत्ता स्तर",
            "doc_quality_tiers_body": "हर उपयोग केस को {lowest} से {highest} तक का स्तर मिलता है, जो दर्शाता है कि उपलब्ध डेटा के साथ वह कितना विशिष्ट, मूल्यवान और निर्माण योग्य है।",
            "doc_modes_title": "गुणवत्ता मोड",
            "doc_mode_good_body": "{medium} या उससे बेहतर रेटिंग वाले उपयोग केस रखता है।",
            "doc_mode_high_body": "केवल {high} या उससे बेहतर रेटिंग वाले उपयोग केस रखता है।",
            "doc_mode_ultra_body": "केवल उन उपयोग केसों को रखता है जो मूल्य, साक्ष्य, कार्यान्वयन क्षमता और कार्रवाई योग्यता पर एक निरपेक्ष मानक पार करते हैं। कोई कोटा नहीं है।",
            "doc_this_catalog": "यह कैटलॉग",
            "doc_elite_score_title": "एलीट स्कोर",
            "doc_elite_score_body": "{ultra} मोड में उपयोग होता है। यह व्यापारिक मूल्य, साक्ष्य की मज़बूती, कार्यान्वयन क्षमता और कार्रवाई योग्यता को तौलता है, और एलीट उपयोग केस हर एक पर 10 में से कम से कम {min_score} अंक पाता है।",
            "doc_coverage_title": "डोमेन कवरेज चयन",
            "doc_coverage_body": "जब किसी अनुरोधित व्यापार डोमेन में मानक से ऊपर कोई उपयोग केस नहीं होता, तो उसका सबसे मज़बूत उम्मीदवार रखा जाता है और उसे मानक से नीचे का कवरेज चयन चिह्नित किया जाता है।",
            "doc_portfolio_kicker": "पोर्टफोलियो",
            "doc_portfolio_glance": "एक नज़र में पोर्टफोलियो",
            "doc_per_domain": "प्रति डोमेन उपयोग केस",
            "doc_quality_distribution": "गुणवत्ता वितरण",
            "doc_top_use_cases": "शीर्ष उपयोग केस",
            "doc_rank": "रैंक",
            "doc_domain_label": "डोमेन",
            "doc_domain_overview": "इस डोमेन के बारे में",
            "doc_what_it_covers": "इसमें क्या शामिल है",
            "doc_top_opportunities": "प्रमुख अवसर",
            "doc_domain_fallback": "यह डोमेन {subdomains} में फैले उपयोग केसों को एक साथ लाता है, जिनकी कुल संख्या {count} है। इसके सबसे मज़बूत अवसर नीचे दिए गए हैं और अगले पेजों में विस्तार से बताए गए हैं।",
            "doc_open_notebook": "जीनी नोटबुक खोलें",
            "doc_elite_score": "एलीट स्कोर",
            "doc_best_of_best": "सर्वश्रेष्ठ में सर्वश्रेष्ठ",
            "doc_appendix": "परिशिष्ट",
            "doc_methodology": "कार्यप्रणाली",
            "strategy_path": "रणनीति पथ",
            "kpis": "मुख्य प्रदर्शन संकेतक",
            "expected_key_results": "अपेक्षित प्रमुख परिणाम",
            "measures_of_success": "सफलता के मापदंड",
            "financial_benefits": "वित्तीय लाभ",
            "non_financial_benefits": "गैर-वित्तीय लाभ",
            "foundations": "डेटा आधार",
            "doc_foundations_title": "साझा डेटा आधार",
            "doc_foundations_lead": "वे टेबल और जॉइन जिनकी दो या अधिक उपयोग मामलों को आवश्यकता है। इन्हें पहले बनाने से सबसे अधिक उपयोग मामले संभव होते हैं। यह क्रम एक अनुक्रम है, समय-सारणी नहीं।",
            "doc_foundation_tables": "टेबल",
            "doc_foundation_unlocks": "संभव होने वाले उपयोग मामले",
            "doc_data_access_standard": "मानक खोज: Inspire ने ग्राहक का कोई डेटा नहीं पढ़ा। उसने केवल मेटाडेटा, मेट्रिक व्यू परिभाषाओं और उपयोग गणनाओं का उपयोग किया।",
            "doc_data_access_deep": "गहन खोज: Inspire ने दायरे की टेबलों से समेकित आँकड़े पढ़े, जिनकी कुल संख्या {tables} है (पंक्तियों की संख्या, तिथि कवरेज, रिक्त मानों का अनुपात, विशिष्ट मानों की संख्या और गैर-व्यक्तिगत कॉलमों के मानों की छोटी सूचियाँ)। कोई भी डेटा पंक्ति AI को नहीं भेजी गई।",
            "doc_method_1_deep": "Inspire मेटाडेटा पढ़ता है और गहन खोज में टेबलों पर गणना किए गए समेकित आँकड़े भी पढ़ता है। यह कभी भी डेटा पंक्तियाँ AI को नहीं भेजता।",
            "doc_guide_lead_deep": "प्रत्येक उपयोग मामला संगठन के अपने मेटाडेटा और डेटा प्रोफाइल से आता है और एक ही निरपेक्ष पैमाने पर अंकित किया जाता है। तुलना के लिए अंकों, फ़िल्टर करने के लिए स्तरों और नेविगेट करने के लिए डोमेन पृष्ठों का उपयोग करें।",
            "value_not_aligned": "संरेखित नहीं",
            "kpi_baseline": "आधार रेखा {value}, पिछले {days} दिन, {date} को मापा गया",
            "kpi_baseline_undated": "आधार रेखा {value}, {date} को मापा गया",
            "doc_method_1": "Inspire केवल मेटाडेटा पढ़ता है: टेबल और कॉलम के नाम, प्रकार, टिप्पणियाँ, कुंजियाँ और टैग। यह कभी भी पंक्ति डेटा नहीं पढ़ता।",
            "doc_method_2": "यह व्यापारिक संदर्भ निकालता है, फिर डेटा द्वारा समर्थित हर व्यापार डोमेन के लिए संभावित उपयोग केस बनाता है।",
            "doc_method_3": "डुप्लिकेट हटाए जाते हैं, हर उम्मीदवार को मूल्य और व्यवहार्यता पर स्कोर किया जाता है, और गुणवत्ता जाँच कमज़ोर या बिना आधार वाले विचारों को हटा देती है।",
            "doc_method_4": "{ultra} मोड में, एक एलीट निर्णायक हर शेष उम्मीदवार की समीक्षा करता है और निर्धारक जाँचें पुष्टि करती हैं कि हर उल्लिखित टेबल और कॉलम मौजूद है।",
            "doc_method_5": "अंतिम सूची को रैंक किया जाता है, व्यापार डोमेन के अनुसार समूहित किया जाता है और चुने जाने पर डेटाब्रिक्स जीनी कोड नोटबुक के साथ जोड़ा जाता है।",
            "doc_glossary": "शब्दावली",
            "doc_gl_domain": "एक व्यापार क्षेत्र जो संबंधित उपयोग केसों को समूहित करता है।",
            "doc_gl_subdomain": "व्यापार डोमेन के भीतर एक संकीर्ण विषय।",
            "doc_gl_technique": "विश्लेषण की विधि, जैसे पूर्वानुमान या वर्गीकरण।",
            "doc_gl_inspire": "0 से 10 तक का स्कोर जो गुणवत्ता और व्यापारिक प्राथमिकता को जोड़ता है।",
            "doc_gl_quality": "हर उपयोग केस को दिया गया {lowest} से {highest} तक का स्तर।",
            "doc_gl_elite": "मूल्य, साक्ष्य, कार्यान्वयन क्षमता और कार्रवाई योग्यता पर आधारित {ultra} मोड का स्कोर।",
            "doc_gl_coverage": "मानक से नीचे रखा गया उपयोग केस, ताकि अनुरोधित डोमेन का प्रतिनिधित्व हो।",
            "doc_gl_notebook": "उपयोग केस बनाने के लिए जीनी कोड निर्देशों वाली डेटाब्रिक्स नोटबुक।",
            "doc_notebook_term": "जीनी नोटबुक",
            "doc_next_steps": "अगले कदम",
            "doc_closing_title": "उपयोग केस से परिणाम तक",
            "doc_closing_body": "सबसे महत्वपूर्ण उपयोग केस चुनें, उनकी जीनी नोटबुक खोलें और टीम के साथ पहला उपयोग केस बनाएँ। ये कदम एक सुझाया गया क्रम हैं; गति टीम तय करती है।",
            "doc_step_select": "चयन करें",
            "doc_step_select_body": "सबसे मज़बूत स्कोर और स्पष्ट प्रायोजक वाले उपयोग केस चुनें।",
            "doc_step_validate": "डेटा सत्यापित करें",
            "doc_step_validate_body": "डेटा स्वामियों के साथ शामिल तालिकाओं और उनके जॉइन की पुष्टि करें।",
            "doc_step_build": "जीनी कोड से बनाएँ",
            "doc_step_build_body": "जीनी नोटबुक खोलें और पहला चालू संस्करण बनाएँ।",
            "doc_step_schedule": "शेड्यूल करें",
            "doc_step_schedule_body": "नोटबुक को डेटाब्रिक्स जॉब के रूप में चलाएँ जो परिणाम टेबल को ताज़ा करता है।",
            "doc_step_measure": "मापें और विस्तार करें",
            "doc_step_measure_body": "इससे मिलने वाले व्यावसायिक मूल्य को ट्रैक करें, फिर अगले उपयोग केस तक विस्तार करें।",
            "doc_agenda": "कार्यसूची",
            "doc_roadmap": "रोडमैप और अगले कदम",
            "doc_use_case_index": "उपयोग केस सूची",
            "doc_continued": "जारी",
            "doc_not_provided": "उपलब्ध नहीं",
            "doc_bob_score": "Best-of-Best स्कोर",
            "elite_cadence": "निर्णय की आवृत्ति",
            "value_cadence_hourly": "हर घंटे",
            "value_cadence_daily": "दैनिक",
            "value_cadence_weekly": "साप्ताहिक",
            "value_cadence_biweekly": "हर दो सप्ताह",
            "value_cadence_monthly": "मासिक",
            "value_cadence_quarterly": "त्रैमासिक",
            "value_cadence_yearly": "वार्षिक",
            "value_cadence_real_time": "रियल टाइम",
            "value_cadence_per_event": "हर घटना पर",
            "doc_generated_by": "{brand} {version} द्वारा बनाया गया · {date}",
            "doc_disclaimer_ai": "इस दस्तावेज़ के उपयोग के मामले, स्कोर और विवरण AI द्वारा बनाए गए हैं। कोई भी निर्णय लेने से पहले डेटा स्वामियों और व्यावसायिक प्रायोजकों के साथ इनकी पुष्टि करें।",
            "doc_disclaimer_figures": "स्कोर, मान और परिणाम केवल प्राथमिकता तय करने के लिए अनुमान हैं। इस दस्तावेज़ का कोई भी आँकड़ा गारंटीकृत नहीं है।",
            "doc_contents": "विषय सूची",
            "doc_index_lead": "सभी उपयोग के मामले अंतिम रैंक के क्रम में, कुल {count}।",
            "doc_deck_scope": "यह प्रस्तुति शीर्ष रैंक वाले उपयोग के मामलों का विवरण देती है, {total} में से {count}। PDF कैटलॉग सभी का विवरण देता है।",
            "doc_deck_scope_all": "यह प्रस्तुति सभी उपयोग के मामलों का विवरण देती है, कुल {total}, हर मामले के लिए एक कार्ड।",
            "value_general_improvement": "सामान्य सुधार",
            "value_reduce_cost": "लागत कम करें",
            "value_increase_revenue": "राजस्व बढ़ाएं",
            "value_boost_productivity": "उत्पादकता बढ़ाएं",
            "value_mitigate_risk": "जोखिम कम करें",
            "value_protect_revenue": "राजस्व की रक्षा करें",
            "value_align_to_regulations": "नियमों का पालन करें",
            "value_improve_customer_experience": "ग्राहक अनुभव सुधारें",
            "value_enable_data_driven_decisions": "डेटा-आधारित निर्णय सक्षम करें",
            "value_optimize_operations": "संचालन अनुकूलित करें",
            "value_empower_talent": "प्रतिभा को सशक्त बनाएं",
            "value_enhance_experience": "अनुभव बढ़ाएं",
            "value_drive_innovation": "नवाचार को बढ़ावा दें",
            "value_achieve_esg": "ESG प्राप्त करें",
            "value_execute_strategy": "रणनीति निष्पादित करें",
            "value_forecasting": "पूर्वानुमान",
            "value_classification": "वर्गीकरण",
            "value_anomaly_detection": "विसंगति पहचान",
            "value_cohort_analysis": "समूह विश्लेषण",
            "value_segmentation": "विभाजन",
            "value_sentiment_analysis": "भावना विश्लेषण",
            "value_trend_analysis": "रुझान विश्लेषण",
            "value_prescriptive_analytics": "निर्देशात्मक विश्लेषण",
            "value_root_cause_analysis": "मूल कारण विश्लेषण",
            "value_optimization": "अनुकूलन",
            "value_recommendation": "अनुशंसा",
            "value_time_series_analysis": "समय श्रृंखला विश्लेषण",
            "value_predictive_analytics": "भविष्यवाणी विश्लेषण",
            "value_descriptive_analytics": "वर्णनात्मक विश्लेषण",
        },
        "Russian": {
            "main_title": "Генератор бизнес-кейсов на базе Датабрикс Джини Код",
            "intro": "Эта записная книжка содержит бизнес-кейсы, созданные ИИ на основе ваших схем. Ниже приведено резюме созданных сценариев по бизнес-доменам.",
            "domain": "Бизнес-домен",
            "total": "Всего бизнес-кейсов",
            "summaries": "Резюме бизнес-кейсов",
            "sum_id": "ID",
            "sum_name": "Название",
            "sum_value": "Бизнес-ценность",
            "sum_outcome": "Ожидаемый результат",
            "warning_header": "ПРЕДУПРЕЖДЕНИЕ",
            "warning_body": "Не запускайте эту записную книжку. Она предназначена только для демонстрации и каталогизации.",
            "disclaimer": "Этот контент создан ИИ и должен быть проверен квалифицированным инженером перед использованием в рабочей среде. Databricks не несёт ответственности за любые проблемы, возникающие в результате использования этого контента.",
            "detailed_scenarios": "Детали бизнес-кейсов",
            "aspect": "Аспект",
            "description": "Описание",
            "aspect_domain": "Бизнес-домен",
            "type": "Тип",
            "analytics_technique": "Аналитическая техника",
            "primary_table": "Основная таблица",
            "quality": "Качество",
            "priority": "Приоритет",
            "usecase": "Бизнес-кейс",
            "description_label": "Описание",
            "value_type_problem": "Проблема",
            "value_type_risk": "Риск",
            "value_type_opportunity": "Возможность",
            "value_type_improvement": "Улучшение",
            "value_priority_ultra_high": "Крайне высокий",
            "value_priority_very_high": "Очень высокий",
            "value_priority_high": "Высокий",
            "value_priority_medium": "Средний",
            "value_priority_low": "Низкий",
            "value_priority_very_low": "Очень низкий",
            "value_priority_ultra_low": "Крайне низкий",
            "statement": "Описание",
            "solution": "Решение",
            "aspect_beneficiary": "Выгодоприобретатель",
            "beneficiary": "Выгодоприобретатель",
            "aspect_sponsor": "Спонсор",
            "sponsor": "Спонсор",
            "business_priority_alignment": "Соответствие бизнес-приоритетам",
            "strategic_goals_alignment": "Соответствие стратегическим целям",
            "subdomain": "Поддомен",
            "aspect_value": "Бизнес-ценность",
            "business_value": "Бизнес-ценность",
            "aspect_tables": "Связанные таблицы",
            "quality_reasons": "Причины качества",
            "aspect_ai_function": "Функция ИИ",
            "aspect_analytics_technique": "Аналитическая техника",
            "aspect_primary_table": "Основная таблица",
            "aspect_quality": "Качество",
            "aspect_priority": "Приоритет",
            "strategic_alignment": "Стратегическое соответствие",
            "return_on_investment": "Рентабельность инвестиций",
            "reusability": "Возможность повторного использования",
            "time_to_value": "Время до получения ценности",
            "data_availability": "Доступность данных",
            "data_accessibility": "Доступ к данным",
            "architecture_fitness": "Соответствие архитектуре",
            "team_skills": "Навыки команды",
            "domain_knowledge": "Знание предметной области",
            "people_allocation": "Распределение персонала",
            "budget_allocation": "Распределение бюджета",
            "time_to_production": "Время до внедрения",
            "value_score": "Оценка ценности",
            "feasibility_score": "Оценка реализуемости",
            "priority_score": "Оценка приоритета",
            "inspire_score": "Оценка Inspire",
            "quality_score_header": "Оценка качества",
            "pdf_title": "Стратегические ИИ бизнес-кейсы на базе Датабрикс Джини Код",
            "pdf_for": "Для",
            "pdf_exec_summary": "Резюме для руководства",
            "pdf_toc_title": "Домены бизнес-кейсов",
            "pdf_detailed_view": "Подробный каталог бизнес-кейсов",
            "pdf_disclaimer_title": "Отказ от ответственности",
            "pdf_fallback_summary_p1": "Этот документ описывает {total_cases} высокоценных аналитических бизнес-кейсов, выявленных для {business_name}.",
            "pdf_fallback_summary_p2": "На следующих страницах представлен подробный анализ этих возможностей, классифицированных по бизнес-доменам.",
            "pptx_main_title": "Стратегические ИИ бизнес-кейсы на базе Датабрикс Джини Код",
            "pptx_for": "Для",
            "pptx_disclaimer_title": "Отказ от ответственности",
            "pptx_domain_suffix": "Бизнес-кейсы",
            "elite_verdict": "Вердикт",
            "elite_decision": "Решение",
            "elite_decision_owner": "Владелец решения",
            "elite_kpi": "Ключевой показатель (KPI)",
            "elite_value_driver": "Драйвер ценности",
            "elite_evidence_columns": "Столбцы-доказательства",
            "elite_join_path": "Путь соединения",
            "elite_build_path": "Путь реализации",
            "elite_red_team_note": "Замечание красной команды",
            "elite_coverage_pick": "Выбор для покрытия домена, ниже порога",
            "elite_fallback_pick": "Ниже порога {mode}",
            "elite_fallback_near_miss": "Почти прошёл, ниже порога {mode}",
            "doc_fallback_pick": "Ниже порога",
            "doc_fallback_notice": "Ни один сценарий не достиг порога {mode}. Это ближайшие кандидаты, каждый помечен как ниже порога.",
            "elite_reason": "Причина",
            "untranslated_notice": "Показано на английском (перевод недоступен)",
            "more_use_cases": "...и ещё {count} сценариев использования.",
            "executive_summary_not_available": "Резюме для руководства недоступно.",
            "domain_summary_not_available": "Резюме домена недоступно.",
            "summary_not_available": "Резюме недоступно.",
            "doc_catalog_kicker": "Каталог ИИ бизнес-кейсов",
            "doc_prepared_for": "Подготовлено для",
            "doc_generated_on": "Дата создания",
            "doc_quality_mode": "Режим качества",
            "doc_use_cases": "Бизнес-кейсы",
            "doc_domains": "Бизнес-домены",
            "doc_elite_use_cases": "Элитные бизнес-кейсы",
            "doc_high_or_better": "Высокое качество и выше",
            "doc_best_score": "Лучшая оценка Inspire",
            "doc_not_scored": "Без оценки",
            "doc_coverage_picks": "Выборы для покрытия",
            "doc_coverage_pick": "Выбор для покрытия",
            "value_correlation_analysis": "Корреляционный Анализ",
            "value_pareto_analysis": "Анализ Парето",
            "value_funnel_analysis": "Анализ Воронки",
            "value_document_processing": "Обработка Документов",
            "value_extraction": "Извлечение Данных",
            "value_ai_analysis": "ИИ-Анализ",
            "value_simulation": "Моделирование",
            "value_geospatial_analysis": "Геопространственный Анализ",
            "value_predictive_modeling": "Предиктивное Моделирование",
            "value_clustering": "Кластеризация",
            "value_regression_analysis": "Регрессионный Анализ",
            "value_survival_analysis": "Анализ Выживаемости",
            "value_network_analysis": "Сетевой Анализ",
            "value_mode_good": "Хороший",
            "value_mode_high": "Высокий",
            "value_mode_ultra_high": "Сверхвысокий",
            "doc_exec_kicker": "Возможность",
            "doc_guide_kicker": "Как читать",
            "doc_how_to_read": "Как читать этот каталог",
            "doc_guide_lead": "Каждый бизнес-кейс основан на собственных метаданных организации и оценивается по единой абсолютной шкале. Используйте оценки для сравнения, уровни для фильтрации и страницы доменов для навигации.",
            "doc_inspire_score_title": "Оценка Inspire, от 0 до 10",
            "doc_inspire_score_body": "Объединяет качество бизнес-кейса и его бизнес-приоритет. Чем выше, тем лучше; каталог упорядочен по этой оценке.",
            "doc_quality_tiers_title": "Уровни качества",
            "doc_quality_tiers_body": "Каждый бизнес-кейс получает уровень от «{lowest}» до «{highest}», который отражает, насколько он конкретен, ценен и реализуем на доступных данных.",
            "doc_modes_title": "Режимы качества",
            "doc_mode_good_body": "Сохраняет бизнес-кейсы с оценкой «{medium}» и выше.",
            "doc_mode_high_body": "Сохраняет только бизнес-кейсы с оценкой «{high}» и выше.",
            "doc_mode_ultra_body": "Сохраняет только бизнес-кейсы, которые превышают абсолютную планку по ценности, доказательности, реализуемости и применимости. Квот нет.",
            "doc_this_catalog": "Этот каталог",
            "doc_elite_score_title": "Элитная оценка",
            "doc_elite_score_body": "Используется в режиме «{ultra}». Учитывает бизнес-ценность, силу доказательств, реализуемость и применимость; элитный бизнес-кейс получает не менее {min_score} из 10 по каждому критерию.",
            "doc_coverage_title": "Выборы для покрытия доменов",
            "doc_coverage_body": "Если в запрошенном бизнес-домене нет бизнес-кейса выше планки, сохраняется его самый сильный кандидат с пометкой «выбор для покрытия, ниже планки».",
            "doc_portfolio_kicker": "Портфель",
            "doc_portfolio_glance": "Портфель в цифрах",
            "doc_per_domain": "Бизнес-кейсы по доменам",
            "doc_quality_distribution": "Распределение по качеству",
            "doc_top_use_cases": "Лучшие бизнес-кейсы",
            "doc_rank": "Место",
            "doc_domain_label": "Домен",
            "doc_domain_overview": "Об этом домене",
            "doc_what_it_covers": "Что охватывает",
            "doc_top_opportunities": "Главные возможности",
            "doc_domain_fallback": "Этот домен объединяет бизнес-кейсы в областях: {subdomains} (всего: {count}). Самые сильные возможности перечислены ниже и подробно описаны на следующих страницах.",
            "doc_open_notebook": "Открыть блокнот Джини",
            "doc_elite_score": "Элитная оценка",
            "doc_best_of_best": "Лучшее из лучших",
            "doc_appendix": "Приложение",
            "doc_methodology": "Методология",
            "strategy_path": "Стратегический путь",
            "kpis": "KPI",
            "expected_key_results": "Ожидаемые ключевые результаты",
            "measures_of_success": "Показатели успеха",
            "financial_benefits": "Финансовые выгоды",
            "non_financial_benefits": "Нефинансовые выгоды",
            "foundations": "Основы данных",
            "doc_foundations_title": "Общие основы данных",
            "doc_foundations_lead": "Таблицы и соединения, которые нужны двум и более бизнес-кейсам. Если построить их первыми, станет доступно больше всего бизнес-кейсов. Порядок является последовательностью, а не графиком.",
            "doc_foundation_tables": "Таблицы",
            "doc_foundation_unlocks": "Доступные бизнес-кейсы",
            "doc_data_access_standard": "Стандартное обнаружение: Inspire не читал данные клиента. Использовались только метаданные, определения представлений метрик и счётчики использования.",
            "doc_data_access_deep": "Глубокое обнаружение: Inspire прочитал агрегаты из таблиц в области анализа (всего: {tables}): число строк, охват дат, доля пустых значений, число уникальных значений и короткие списки значений неперсональных столбцов. Ни одна строка данных не была отправлена ИИ.",
            "doc_method_1_deep": "Inspire читает метаданные, а при глубоком обнаружении также агрегаты, вычисленные по таблицам. Он никогда не отправляет строки данных ИИ.",
            "doc_guide_lead_deep": "Каждый бизнес-кейс основан на метаданных и профилях данных организации и оценивается по единой абсолютной шкале. Используйте оценки для сравнения, уровни для фильтрации, а страницы доменов для навигации.",
            "value_not_aligned": "Не согласовано",
            "kpi_baseline": "базовое значение {value}, последние {days} дн., измерено {date}",
            "kpi_baseline_undated": "базовое значение {value}, измерено {date}",
            "doc_method_1": "Inspire читает только метаданные: имена таблиц и столбцов, типы, комментарии, ключи и теги. Данные строк он никогда не читает.",
            "doc_method_2": "Он определяет бизнес-контекст, а затем формирует кандидатов для каждого бизнес-домена, который поддерживают данные.",
            "doc_method_3": "Дубликаты удаляются, каждый кандидат оценивается по ценности и выполнимости, а фильтры качества отсеивают слабые или необоснованные идеи.",
            "doc_method_4": "В режиме «{ultra}» элитный эксперт проверяет каждого оставшегося кандидата, а детерминированные проверки подтверждают существование каждой упомянутой таблицы и столбца.",
            "doc_method_5": "Итоговый список ранжируется, группируется по бизнес-доменам и при выборе дополняется блокнотом Датабрикс Джини Код.",
            "doc_glossary": "Глоссарий",
            "doc_gl_domain": "Бизнес-область, объединяющая связанные бизнес-кейсы.",
            "doc_gl_subdomain": "Более узкая тема внутри бизнес-домена.",
            "doc_gl_technique": "Аналитический метод, например прогнозирование или классификация.",
            "doc_gl_inspire": "Оценка от 0 до 10, объединяющая качество и бизнес-приоритет.",
            "doc_gl_quality": "Уровень от «{lowest}» до «{highest}», присваиваемый каждому бизнес-кейсу.",
            "doc_gl_elite": "Оценка режима «{ultra}» по ценности, доказательности, реализуемости и применимости.",
            "doc_gl_coverage": "Бизнес-кейс ниже планки, сохранённый, чтобы запрошенный домен был представлен.",
            "doc_gl_notebook": "Блокнот Датабрикс с инструкциями Джини Код для реализации бизнес-кейса.",
            "doc_notebook_term": "Блокнот Джини",
            "doc_next_steps": "Следующие шаги",
            "doc_closing_title": "От бизнес-кейсов к результатам",
            "doc_closing_body": "Выберите самые важные бизнес-кейсы, откройте их блокноты Джини и реализуйте первый вместе с командой. Шаги показывают рекомендуемый порядок; темп задаёт команда.",
            "doc_step_select": "Выбор",
            "doc_step_select_body": "Выберите бизнес-кейсы с наилучшей оценкой и понятным спонсором.",
            "doc_step_validate": "Проверка данных",
            "doc_step_validate_body": "Согласуйте задействованные таблицы и их соединения с владельцами данных.",
            "doc_step_build": "Реализация с Джини Код",
            "doc_step_build_body": "Откройте блокнот Джини и создайте первую рабочую версию.",
            "doc_step_schedule": "Расписание",
            "doc_step_schedule_body": "Запускайте блокнот как задание Датабрикс, обновляющее таблицу результатов.",
            "doc_step_measure": "Измерение и масштабирование",
            "doc_step_measure_body": "Отслеживайте полученную бизнес-ценность, затем переходите к следующему бизнес-кейсу.",
            "doc_agenda": "Повестка",
            "doc_roadmap": "Дорожная карта и следующие шаги",
            "doc_use_case_index": "Указатель бизнес-кейсов",
            "doc_continued": "продолжение",
            "doc_not_provided": "Не указано",
            "doc_bob_score": "Оценка Best-of-Best",
            "elite_cadence": "Периодичность решения",
            "value_cadence_hourly": "Ежечасно",
            "value_cadence_daily": "Ежедневно",
            "value_cadence_weekly": "Еженедельно",
            "value_cadence_biweekly": "Раз в две недели",
            "value_cadence_monthly": "Ежемесячно",
            "value_cadence_quarterly": "Ежеквартально",
            "value_cadence_yearly": "Ежегодно",
            "value_cadence_real_time": "В реальном времени",
            "value_cadence_per_event": "По событию",
            "doc_generated_by": "Создано с помощью {brand} {version} · {date}",
            "doc_disclaimer_ai": "Сценарии использования, оценки и описания в этом документе созданы ИИ. Проверьте их вместе с владельцами данных и бизнес-спонсорами, прежде чем принимать решения.",
            "doc_disclaimer_figures": "Оценки, значения и результаты являются приблизительными и служат только для расстановки приоритетов. Ни одна цифра в этом документе не гарантирована.",
            "doc_contents": "Содержание",
            "doc_index_lead": "Все сценарии использования в порядке итогового рейтинга (всего: {count}).",
            "doc_deck_scope": "В этой презентации подробно описаны сценарии использования с наивысшим рейтингом ({count} из {total}). В PDF-каталоге описаны все.",
            "doc_deck_scope_all": "В этой презентации подробно описаны все сценарии использования (всего: {total}), по одной карточке на каждый.",
            "value_general_improvement": "Общее Улучшение",
            "value_reduce_cost": "Сократить Затраты",
            "value_increase_revenue": "Увеличить Доход",
            "value_boost_productivity": "Повысить Производительность",
            "value_mitigate_risk": "Снизить Риски",
            "value_protect_revenue": "Защитить Доход",
            "value_align_to_regulations": "Соответствовать Нормативам",
            "value_improve_customer_experience": "Улучшить Клиентский Опыт",
            "value_enable_data_driven_decisions": "Обеспечить Решения на Основе Данных",
            "value_optimize_operations": "Оптимизировать Операции",
            "value_empower_talent": "Развивать Таланты",
            "value_enhance_experience": "Улучшить Опыт",
            "value_drive_innovation": "Стимулировать Инновации",
            "value_achieve_esg": "Достичь ESG",
            "value_execute_strategy": "Реализовать Стратегию",
            "value_forecasting": "Прогнозирование",
            "value_classification": "Классификация",
            "value_anomaly_detection": "Обнаружение Аномалий",
            "value_cohort_analysis": "Когортный Анализ",
            "value_segmentation": "Сегментация",
            "value_sentiment_analysis": "Анализ Настроений",
            "value_trend_analysis": "Анализ Трендов",
            "value_prescriptive_analytics": "Предписывающая Аналитика",
            "value_root_cause_analysis": "Анализ Первопричин",
            "value_optimization": "Оптимизация",
            "value_recommendation": "Рекомендация",
            "value_time_series_analysis": "Анализ Временных Рядов",
            "value_predictive_analytics": "Прогнозная Аналитика",
            "value_descriptive_analytics": "Описательная Аналитика",
        },
        "Swedish": {
            "main_title": "Användningsfallsgenerator Driven av Databricks Genie Code",
            "intro": "Denna notebook innehåller AI-genererade användningsfall baserade på dina scheman. Nedan finns en sammanfattning av genererade scenarier per affärsdomän.",
            "domain": "Affärsdomän",
            "total": "Totalt antal användningsfall",
            "summaries": "Sammanfattningar av användningsfall",
            "sum_id": "ID",
            "sum_name": "Namn",
            "sum_value": "Affärsvärde",
            "sum_outcome": "Förväntat resultat",
            "warning_header": "VARNING",
            "warning_body": "Kör inte denna notebook. Den är avsedd enbart för demonstration och katalogisering.",
            "disclaimer": "Detta innehåll är AI-genererat och bör granskas av en kvalificerad ingenjör innan det används i produktion. Databricks ansvarar inte för eventuella problem som uppstår vid användning av detta innehåll.",
            "detailed_scenarios": "Detaljer om användningsfall",
            "aspect": "Aspekt",
            "description": "Beskrivning",
            "aspect_domain": "Affärsdomän",
            "type": "Typ",
            "analytics_technique": "Analysteknik",
            "primary_table": "Primärtabell",
            "quality": "Kvalitet",
            "priority": "Prioritet",
            "usecase": "Användningsfall",
            "description_label": "Beskrivning",
            "value_type_problem": "Problem",
            "value_type_risk": "Risk",
            "value_type_opportunity": "Möjlighet",
            "value_type_improvement": "Förbättring",
            "value_priority_ultra_high": "Extremt hög",
            "value_priority_very_high": "Mycket hög",
            "value_priority_high": "Hög",
            "value_priority_medium": "Medel",
            "value_priority_low": "Låg",
            "value_priority_very_low": "Mycket låg",
            "value_priority_ultra_low": "Extremt låg",
            "statement": "Uttalande",
            "solution": "Lösning",
            "aspect_beneficiary": "Förmånstagare",
            "beneficiary": "Förmånstagare",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Affärsprioritet anpassning",
            "strategic_goals_alignment": "Strategisk målanpassning",
            "subdomain": "Underdomän",
            "aspect_value": "Affärsvärde",
            "business_value": "Affärsvärde",
            "aspect_tables": "Involverade tabeller",
            "quality_reasons": "Kvalitetsskäl",
            "aspect_ai_function": "AI-funktion",
            "aspect_analytics_technique": "Analysteknik",
            "aspect_primary_table": "Primärtabell",
            "aspect_quality": "Kvalitet",
            "aspect_priority": "Prioritet",
            "strategic_alignment": "Strategisk anpassning",
            "return_on_investment": "Avkastning på investering",
            "reusability": "Återanvändbarhet",
            "time_to_value": "Tid till värde",
            "data_availability": "Datatillgänglighet",
            "data_accessibility": "Dataåtkomlighet",
            "architecture_fitness": "Arkitekturlämplighet",
            "team_skills": "Teamkompetens",
            "domain_knowledge": "Domänkunskap",
            "people_allocation": "Personalallokering",
            "budget_allocation": "Budgetallokering",
            "time_to_production": "Tid till produktion",
            "value_score": "Värdepoäng",
            "feasibility_score": "Genomförbarhetspoäng",
            "priority_score": "Prioritetspoäng",
            "inspire_score": "Inspire-poäng",
            "quality_score_header": "Kvalitetspoäng",
            "pdf_title": "Strategiska AI-användningsfall drivna av Databricks Genie Code",
            "pdf_for": "För",
            "pdf_exec_summary": "Sammanfattning",
            "pdf_toc_title": "Användningsfallsdomäner",
            "pdf_detailed_view": "Detaljerad användningsfallskatalog",
            "pdf_disclaimer_title": "Ansvarsfriskrivning",
            "pdf_fallback_summary_p1": "Detta dokument beskriver {total_cases} högvärdiga analytiska användningsfall identifierade för {business_name}.",
            "pdf_fallback_summary_p2": "Följande sidor ger en detaljerad uppdelning av dessa möjligheter, kategoriserade efter affärsdomän.",
            "pptx_main_title": "Strategiska AI-användningsfall drivna av Databricks Genie Code",
            "pptx_for": "För",
            "pptx_disclaimer_title": "Ansvarsfriskrivning",
            "pptx_domain_suffix": "Användningsfall",
            "elite_verdict": "Bedömning",
            "elite_decision": "Beslut",
            "elite_decision_owner": "Beslutsägare",
            "elite_kpi": "KPI",
            "elite_value_driver": "Värdedrivare",
            "elite_evidence_columns": "Evidenskolumner",
            "elite_join_path": "Kopplingsväg",
            "elite_build_path": "Genomförandeväg",
            "elite_red_team_note": "Red team-anteckning",
            "elite_coverage_pick": "Urval för domäntäckning, under ribban",
            "elite_fallback_pick": "Under {mode}-ribban",
            "elite_fallback_near_miss": "Nära, under {mode}-ribban",
            "doc_fallback_pick": "Under ribban",
            "doc_fallback_notice": "Inget användningsfall nådde {mode}-ribban. Detta är de närmaste kandidaterna, var och en märkt under ribban.",
            "elite_reason": "Anledning",
            "untranslated_notice": "Visas på engelska (översättning saknas)",
            "more_use_cases": "...och {count} fler användningsfall.",
            "executive_summary_not_available": "Sammanfattning ej tillgänglig.",
            "domain_summary_not_available": "Domänsammanfattning ej tillgänglig.",
            "summary_not_available": "Sammanfattning ej tillgänglig.",
            "doc_catalog_kicker": "Katalog över AI-användningsfall",
            "doc_prepared_for": "Framtagen för",
            "doc_generated_on": "Skapad",
            "doc_quality_mode": "Kvalitetsläge",
            "doc_use_cases": "Användningsfall",
            "doc_domains": "Affärsdomäner",
            "doc_elite_use_cases": "Elitanvändningsfall",
            "doc_high_or_better": "Hög kvalitet eller bättre",
            "doc_best_score": "Bästa Inspire-poäng",
            "doc_not_scored": "Ej poängsatt",
            "doc_coverage_picks": "Täckningsval",
            "doc_coverage_pick": "Täckningsval",
            "value_correlation_analysis": "Korrelationsanalys",
            "value_pareto_analysis": "Paretoanalys",
            "value_funnel_analysis": "Trattanalys",
            "value_document_processing": "Dokumentbehandling",
            "value_extraction": "Extraktion",
            "value_ai_analysis": "AI-analys",
            "value_simulation": "Simulering",
            "value_geospatial_analysis": "Geospatial analys",
            "value_predictive_modeling": "Prediktiv modellering",
            "value_clustering": "Klustring",
            "value_regression_analysis": "Regressionsanalys",
            "value_survival_analysis": "Överlevnadsanalys",
            "value_network_analysis": "Nätverksanalys",
            "value_mode_good": "Bra",
            "value_mode_high": "Hög",
            "value_mode_ultra_high": "Ultrahög",
            "doc_exec_kicker": "Möjligheten",
            "doc_guide_kicker": "Läsguide",
            "doc_how_to_read": "Så läser du katalogen",
            "doc_guide_lead": "Varje användningsfall bygger på organisationens egna metadata och poängsätts på en och samma absoluta skala. Använd poängen för att jämföra, nivåerna för att filtrera och domänsidorna för att navigera.",
            "doc_inspire_score_title": "Inspire-poäng, från 0 till 10",
            "doc_inspire_score_body": "Kombinerar ett användningsfalls kvalitet med dess affärsprioritet. Högre är bättre, och katalogen är sorterad efter den.",
            "doc_quality_tiers_title": "Kvalitetsnivåer",
            "doc_quality_tiers_body": "Varje användningsfall får en nivå från {lowest} till {highest} som speglar hur specifikt, värdefullt och byggbart det är med tillgängliga data.",
            "doc_modes_title": "Kvalitetslägen",
            "doc_mode_good_body": "Behåller användningsfall med betyget {medium} eller bättre.",
            "doc_mode_high_body": "Behåller bara användningsfall med betyget {high} eller bättre.",
            "doc_mode_ultra_body": "Behåller bara användningsfall som klarar en absolut ribba för värde, evidens, genomförbarhet och handlingsbarhet. Det finns ingen kvot.",
            "doc_this_catalog": "Den här katalogen",
            "doc_elite_score_title": "Elitpoäng",
            "doc_elite_score_body": "Används i läget {ultra}. Den väger affärsvärde, evidensstyrka, genomförbarhet och handlingsbarhet, och ett elitanvändningsfall får minst {min_score} av 10 på varje.",
            "doc_coverage_title": "Val för domäntäckning",
            "doc_coverage_body": "När en efterfrågad affärsdomän saknar användningsfall över ribban behålls dess starkaste kandidat och märks som täckningsval under ribban.",
            "doc_portfolio_kicker": "Portfölj",
            "doc_portfolio_glance": "Portföljen i korthet",
            "doc_per_domain": "Användningsfall per domän",
            "doc_quality_distribution": "Kvalitetsfördelning",
            "doc_top_use_cases": "Främsta användningsfall",
            "doc_rank": "Placering",
            "doc_domain_label": "Domän",
            "doc_domain_overview": "Om domänen",
            "doc_what_it_covers": "Vad den omfattar",
            "doc_top_opportunities": "Främsta möjligheter",
            "doc_domain_fallback": "Den här domänen samlar {count} användningsfall inom {subdomains}. De starkaste möjligheterna listas nedan och beskrivs på de följande sidorna.",
            "doc_open_notebook": "Öppna Genie-anteckningsboken",
            "doc_elite_score": "Elitpoäng",
            "doc_best_of_best": "Det bästa av det bästa",
            "doc_appendix": "Bilaga",
            "doc_methodology": "Metod",
            "strategy_path": "Strategisk sökväg",
            "kpis": "KPI:er",
            "expected_key_results": "Förväntade nyckelresultat",
            "measures_of_success": "Framgångsmått",
            "financial_benefits": "Ekonomiska fördelar",
            "non_financial_benefits": "Icke-ekonomiska fördelar",
            "foundations": "Datagrunder",
            "doc_foundations_title": "Delade datagrunder",
            "doc_foundations_lead": "Tabeller och kopplingar som två eller fler användningsfall behöver. Att bygga dem först möjliggör flest användningsfall. Ordningen är en sekvens, inte ett schema.",
            "doc_foundation_tables": "Tabeller",
            "doc_foundation_unlocks": "Möjliggjorda användningsfall",
            "doc_data_access_standard": "Standardupptäckt: Inspire läste inga kunddata. Endast metadata, definitioner av måttvyer och användningsräkningar användes.",
            "doc_data_access_deep": "Djup upptäckt: Inspire läste aggregat från tabellerna i omfånget, totalt {tables} (radantal, datumtäckning, andel tomma värden, antal unika värden och korta värdelistor för icke-personliga kolumner). Inga datarader skickades till AI:n.",
            "doc_method_1_deep": "Inspire läser metadata och vid djup upptäckt även aggregat som beräknas över tabellerna. Den skickar aldrig datarader till AI:n.",
            "doc_guide_lead_deep": "Varje användningsfall kommer från organisationens egna metadata och dataprofiler och poängsätts på en enda absolut skala. Använd poängen för att jämföra, nivåerna för att filtrera och domänsidorna för att navigera.",
            "value_not_aligned": "Ej anpassad",
            "kpi_baseline": "utgångsvärde {value}, senaste {days} dagarna, mätt {date}",
            "kpi_baseline_undated": "utgångsvärde {value}, mätt {date}",
            "doc_method_1": "Inspire läser bara metadata: tabell- och kolumnnamn, typer, kommentarer, nycklar och taggar. Den läser aldrig raddata.",
            "doc_method_2": "Den härleder affärskontexten och genererar sedan kandidater för varje affärsdomän som data stöder.",
            "doc_method_3": "Dubbletter tas bort, varje kandidat poängsätts på värde och genomförbarhet, och kvalitetsgrindar sållar bort svaga eller ostödda idéer.",
            "doc_method_4": "I läget {ultra} granskar en elitbedömare varje återstående kandidat, och deterministiska kontroller bekräftar att varje citerad tabell och kolumn finns.",
            "doc_method_5": "Den slutliga listan rangordnas, grupperas per affärsdomän och kompletteras vid val med en Databricks Genie Code-anteckningsbok.",
            "doc_glossary": "Ordlista",
            "doc_gl_domain": "Ett affärsområde som samlar relaterade användningsfall.",
            "doc_gl_subdomain": "Ett smalare tema inom en affärsdomän.",
            "doc_gl_technique": "Den analytiska metoden, till exempel prognos eller klassificering.",
            "doc_gl_inspire": "Poängen från 0 till 10 som kombinerar kvalitet och affärsprioritet.",
            "doc_gl_quality": "Nivån från {lowest} till {highest} som varje användningsfall får.",
            "doc_gl_elite": "Poängen i läget {ultra} för värde, evidens, genomförbarhet och handlingsbarhet.",
            "doc_gl_coverage": "Ett användningsfall under ribban som behålls så att en efterfrågad domän finns representerad.",
            "doc_gl_notebook": "En Databricks-anteckningsbok med Genie Code-instruktioner för att bygga användningsfallet.",
            "doc_notebook_term": "Genie-anteckningsbok",
            "doc_next_steps": "Nästa steg",
            "doc_closing_title": "Från användningsfall till resultat",
            "doc_closing_body": "Välj de viktigaste användningsfallen, öppna deras Genie-anteckningsböcker och bygg det första tillsammans med teamet. Stegen är en föreslagen ordning; teamet bestämmer takten.",
            "doc_step_select": "Välj",
            "doc_step_select_body": "Välj användningsfallen med högst poäng och en tydlig sponsor.",
            "doc_step_validate": "Validera data",
            "doc_step_validate_body": "Bekräfta de berörda tabellerna och deras kopplingar med dataägarna.",
            "doc_step_build": "Bygg med Genie Code",
            "doc_step_build_body": "Öppna Genie-anteckningsboken och skapa den första fungerande versionen.",
            "doc_step_schedule": "Schemalägg",
            "doc_step_schedule_body": "Kör anteckningsboken som ett Databricks-jobb som uppdaterar resultattabellen.",
            "doc_step_measure": "Mät och skala",
            "doc_step_measure_body": "Följ det affärsvärde som skapas och utöka sedan till nästa användningsfall.",
            "doc_agenda": "Agenda",
            "doc_roadmap": "Färdplan och nästa steg",
            "doc_use_case_index": "Register över användningsfall",
            "doc_continued": "fortsättning",
            "doc_not_provided": "Inte angivet",
            "doc_bob_score": "Best-of-Best-poäng",
            "elite_cadence": "Beslutsfrekvens",
            "value_cadence_hourly": "Varje timme",
            "value_cadence_daily": "Dagligen",
            "value_cadence_weekly": "Varje vecka",
            "value_cadence_biweekly": "Varannan vecka",
            "value_cadence_monthly": "Varje månad",
            "value_cadence_quarterly": "Varje kvartal",
            "value_cadence_yearly": "Varje år",
            "value_cadence_real_time": "I realtid",
            "value_cadence_per_event": "Per händelse",
            "doc_generated_by": "Skapad med {brand} {version} · {date}",
            "doc_disclaimer_ai": "Användningsfallen, poängen och texterna i detta dokument är AI-genererade. Validera dem med dataägarna och verksamhetens sponsorer innan något beslut fattas.",
            "doc_disclaimer_figures": "Poäng, värden och utfall är uppskattningar som endast används för prioritering. Ingen siffra i detta dokument är garanterad.",
            "doc_contents": "Innehåll",
            "doc_index_lead": "Alla användningsfall i slutlig rangordning, totalt {count}.",
            "doc_deck_scope": "Den här presentationen beskriver de högst rankade användningsfallen, {count} av {total}. PDF-katalogen beskriver alla.",
            "doc_deck_scope_all": "Den här presentationen beskriver alla användningsfall, totalt {total}, ett kort för varje.",
            "value_general_improvement": "Allmän förbättring",
            "value_reduce_cost": "Minska kostnader",
            "value_increase_revenue": "Öka intäkter",
            "value_boost_productivity": "Öka produktivitet",
            "value_mitigate_risk": "Minska risker",
            "value_protect_revenue": "Skydda intäkter",
            "value_align_to_regulations": "Efterleva regelverk",
            "value_improve_customer_experience": "Förbättra kundupplevelse",
            "value_enable_data_driven_decisions": "Möjliggöra datadrivna beslut",
            "value_optimize_operations": "Optimera verksamheten",
            "value_empower_talent": "Stärka talanger",
            "value_enhance_experience": "Förbättra upplevelsen",
            "value_drive_innovation": "Driva innovation",
            "value_achieve_esg": "Uppnå ESG",
            "value_execute_strategy": "Genomföra strategi",
            "value_forecasting": "Prognos",
            "value_classification": "Klassificering",
            "value_anomaly_detection": "Anomalidetektering",
            "value_cohort_analysis": "Kohortanalys",
            "value_segmentation": "Segmentering",
            "value_sentiment_analysis": "Sentimentanalys",
            "value_trend_analysis": "Trendanalys",
            "value_prescriptive_analytics": "Preskriptiv analys",
            "value_root_cause_analysis": "Rotorsaksanalys",
            "value_optimization": "Optimering",
            "value_recommendation": "Rekommendation",
            "value_time_series_analysis": "Tidsserieanalys",
            "value_predictive_analytics": "Prediktiv analys",
            "value_descriptive_analytics": "Deskriptiv analys",
        },
        "Danish": {
            "main_title": "Anvendelsescase-generator drevet af Databricks Genie Code",
            "intro": "Denne notebook indeholder AI-genererede anvendelsescases baseret på dine skemaer. Nedenfor er en oversigt over genererede scenarier pr. forretningsdomæne.",
            "domain": "Forretningsdomæne",
            "total": "Samlet antal anvendelsescases",
            "summaries": "Oversigter over anvendelsescases",
            "sum_id": "ID",
            "sum_name": "Navn",
            "sum_value": "Forretningsværdi",
            "sum_outcome": "Forventet resultat",
            "warning_header": "ADVARSEL",
            "warning_body": "Kør ikke denne notebook. Den er udelukkende beregnet til demonstration og katalogisering.",
            "disclaimer": "Dette indhold er AI-genereret og bør gennemgås af en kvalificeret ingeniør, før det bruges i produktion. Databricks er ikke ansvarlig for problemer, der opstår ved brug af dette indhold.",
            "detailed_scenarios": "Detaljer om anvendelsescases",
            "aspect": "Aspekt",
            "description": "Beskrivelse",
            "aspect_domain": "Forretningsdomæne",
            "type": "Type",
            "analytics_technique": "Analyseteknik",
            "primary_table": "Primærtabel",
            "quality": "Kvalitet",
            "priority": "Prioritet",
            "usecase": "Anvendelsescase",
            "description_label": "Beskrivelse",
            "value_type_problem": "Problem",
            "value_type_risk": "Risiko",
            "value_type_opportunity": "Mulighed",
            "value_type_improvement": "Forbedring",
            "value_priority_ultra_high": "Ekstremt høj",
            "value_priority_very_high": "Meget høj",
            "value_priority_high": "Høj",
            "value_priority_medium": "Middel",
            "value_priority_low": "Lav",
            "value_priority_very_low": "Meget lav",
            "value_priority_ultra_low": "Ekstremt lav",
            "statement": "Erklæring",
            "solution": "Løsning",
            "aspect_beneficiary": "Modtager",
            "beneficiary": "Modtager",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Forretningsprioritets tilpasning",
            "strategic_goals_alignment": "Strategisk måltilpasning",
            "subdomain": "Underdomæne",
            "aspect_value": "Forretningsværdi",
            "business_value": "Forretningsværdi",
            "aspect_tables": "Involverede tabeller",
            "quality_reasons": "Kvalitetsårsager",
            "aspect_ai_function": "AI-funktion",
            "aspect_analytics_technique": "Analyseteknik",
            "aspect_primary_table": "Primærtabel",
            "aspect_quality": "Kvalitet",
            "aspect_priority": "Prioritet",
            "strategic_alignment": "Strategisk tilpasning",
            "return_on_investment": "Investeringsafkast",
            "reusability": "Genanvendelighed",
            "time_to_value": "Tid til værdi",
            "data_availability": "Datatilgængelighed",
            "data_accessibility": "Dataadgang",
            "architecture_fitness": "Arkitekturegnethed",
            "team_skills": "Teamkompetencer",
            "domain_knowledge": "Domæneviden",
            "people_allocation": "Personaleallokering",
            "budget_allocation": "Budgetallokering",
            "time_to_production": "Tid til produktion",
            "value_score": "Værdiscore",
            "feasibility_score": "Gennemførlighedsscore",
            "priority_score": "Prioritetsscore",
            "inspire_score": "Inspire-score",
            "quality_score_header": "Kvalitetsscore",
            "pdf_title": "Strategiske AI-anvendelsescases drevet af Databricks Genie Code",
            "pdf_for": "For",
            "pdf_exec_summary": "Resumé",
            "pdf_toc_title": "Anvendelsescasedomæner",
            "pdf_detailed_view": "Detaljeret anvendelsescasekatalog",
            "pdf_disclaimer_title": "Ansvarsfraskrivelse",
            "pdf_fallback_summary_p1": "Dette dokument beskriver {total_cases} højværdi analytiske anvendelsescases identificeret for {business_name}.",
            "pdf_fallback_summary_p2": "De følgende sider giver en detaljeret opdeling af disse muligheder, kategoriseret efter forretningsdomæne.",
            "pptx_main_title": "Strategiske AI-anvendelsescases drevet af Databricks Genie Code",
            "pptx_for": "For",
            "pptx_disclaimer_title": "Ansvarsfraskrivelse",
            "pptx_domain_suffix": "Anvendelsescases",
            "elite_verdict": "Vurdering",
            "elite_decision": "Beslutning",
            "elite_decision_owner": "Beslutningsejer",
            "elite_kpi": "KPI",
            "elite_value_driver": "Værdidriver",
            "elite_evidence_columns": "Evidenskolonner",
            "elite_join_path": "Join-sti",
            "elite_build_path": "Implementeringssti",
            "elite_red_team_note": "Red team-note",
            "elite_coverage_pick": "Valg for domænedækning, under niveauet",
            "elite_fallback_pick": "Under {mode}-niveauet",
            "elite_fallback_near_miss": "Tæt på, under {mode}-niveauet",
            "doc_fallback_pick": "Under niveauet",
            "doc_fallback_notice": "Ingen anvendelsescase nåede {mode}-niveauet. Dette er de nærmeste kandidater, hver markeret under niveauet.",
            "elite_reason": "Årsag",
            "untranslated_notice": "Vist på engelsk (oversættelse ikke tilgængelig)",
            "more_use_cases": "...og {count} flere use cases.",
            "executive_summary_not_available": "Resumé ikke tilgængeligt.",
            "domain_summary_not_available": "Domæneoversigt ikke tilgængelig.",
            "summary_not_available": "Oversigt ikke tilgængelig.",
            "doc_catalog_kicker": "Katalog over AI-anvendelsescases",
            "doc_prepared_for": "Udarbejdet til",
            "doc_generated_on": "Oprettet",
            "doc_quality_mode": "Kvalitetstilstand",
            "doc_use_cases": "Anvendelsescases",
            "doc_domains": "Forretningsdomæner",
            "doc_elite_use_cases": "Elite-anvendelsescases",
            "doc_high_or_better": "Høj kvalitet eller bedre",
            "doc_best_score": "Bedste Inspire-score",
            "doc_not_scored": "Ikke scoret",
            "doc_coverage_picks": "Dækningsvalg",
            "doc_coverage_pick": "Dækningsvalg",
            "value_correlation_analysis": "Korrelationsanalyse",
            "value_pareto_analysis": "Paretoanalyse",
            "value_funnel_analysis": "Tragtanalyse",
            "value_document_processing": "Dokumentbehandling",
            "value_extraction": "Udtrækning",
            "value_ai_analysis": "AI-analyse",
            "value_simulation": "Simulering",
            "value_geospatial_analysis": "Geospatial analyse",
            "value_predictive_modeling": "Prædiktiv modellering",
            "value_clustering": "Klyngeanalyse",
            "value_regression_analysis": "Regressionsanalyse",
            "value_survival_analysis": "Overlevelsesanalyse",
            "value_network_analysis": "Netværksanalyse",
            "value_mode_good": "God",
            "value_mode_high": "Høj",
            "value_mode_ultra_high": "Ultrahøj",
            "doc_exec_kicker": "Muligheden",
            "doc_guide_kicker": "Læsevejledning",
            "doc_how_to_read": "Sådan læser du kataloget",
            "doc_guide_lead": "Hver anvendelsescase stammer fra organisationens egne metadata og scores på én og samme absolutte skala. Brug scorerne til at sammenligne, niveauerne til at filtrere og domænesiderne til at navigere.",
            "doc_inspire_score_title": "Inspire-score, fra 0 til 10",
            "doc_inspire_score_body": "Kombinerer en anvendelsescases kvalitet med dens forretningsprioritet. Højere er bedre, og kataloget er sorteret efter den.",
            "doc_quality_tiers_title": "Kvalitetsniveauer",
            "doc_quality_tiers_body": "Hver anvendelsescase får et niveau fra {lowest} til {highest}, der afspejler, hvor specifik, værdifuld og byggebar den er med de tilgængelige data.",
            "doc_modes_title": "Kvalitetstilstande",
            "doc_mode_good_body": "Beholder anvendelsescases vurderet som {medium} eller bedre.",
            "doc_mode_high_body": "Beholder kun anvendelsescases vurderet som {high} eller bedre.",
            "doc_mode_ultra_body": "Beholder kun anvendelsescases, der klarer en absolut bar for værdi, evidens, gennemførlighed og handlingsrettethed. Der er ingen kvote.",
            "doc_this_catalog": "Dette katalog",
            "doc_elite_score_title": "Elitescore",
            "doc_elite_score_body": "Bruges i tilstanden {ultra}. Den vægter forretningsværdi, evidensstyrke, gennemførlighed og handlingsrettethed, og en elite-anvendelsescase scorer mindst {min_score} ud af 10 på hver.",
            "doc_coverage_title": "Valg for domænedækning",
            "doc_coverage_body": "Når et efterspurgt forretningsdomæne ikke har nogen anvendelsescase over baren, beholdes dens stærkeste kandidat og markeres som dækningsvalg under baren.",
            "doc_portfolio_kicker": "Portefølje",
            "doc_portfolio_glance": "Porteføljen i overblik",
            "doc_per_domain": "Anvendelsescases pr. domæne",
            "doc_quality_distribution": "Kvalitetsfordeling",
            "doc_top_use_cases": "Vigtigste anvendelsescases",
            "doc_rank": "Placering",
            "doc_domain_label": "Domæne",
            "doc_domain_overview": "Om dette domæne",
            "doc_what_it_covers": "Hvad det dækker",
            "doc_top_opportunities": "Største muligheder",
            "doc_domain_fallback": "Dette domæne samler anvendelsescases inden for {subdomains}, i alt {count}. De stærkeste muligheder står nedenfor og uddybes på de følgende sider.",
            "doc_open_notebook": "Åbn Genie-notesbogen",
            "doc_elite_score": "Elitescore",
            "doc_best_of_best": "Det bedste af det bedste",
            "doc_appendix": "Bilag",
            "doc_methodology": "Metode",
            "strategy_path": "Strategisti",
            "kpis": "KPI'er",
            "expected_key_results": "Forventede nøgleresultater",
            "measures_of_success": "Succeskriterier",
            "financial_benefits": "Økonomiske fordele",
            "non_financial_benefits": "Ikke-økonomiske fordele",
            "foundations": "Datafundamenter",
            "doc_foundations_title": "Fælles datafundamenter",
            "doc_foundations_lead": "Tabeller og joins, som to eller flere anvendelsescases har brug for. Når de bygges først, åbnes flest anvendelsescases. Rækkefølgen er en sekvens, ikke en tidsplan.",
            "doc_foundation_tables": "Tabeller",
            "doc_foundation_unlocks": "Mulige anvendelsescases",
            "doc_data_access_standard": "Standardopdagelse: Inspire læste ingen kundedata. Der blev kun brugt metadata, definitioner af metrikvisninger og brugsoptællinger.",
            "doc_data_access_deep": "Dyb opdagelse: Inspire læste aggregater fra tabellerne i omfanget, i alt {tables} (antal rækker, datodækning, andel tomme værdier, antal unikke værdier og korte værdilister for ikke-personlige kolonner). Ingen datarækker blev sendt til AI'en.",
            "doc_method_1_deep": "Inspire læser metadata og ved dyb opdagelse også aggregater beregnet over tabellerne. Den sender aldrig datarækker til AI'en.",
            "doc_guide_lead_deep": "Hver anvendelsescase stammer fra organisationens egne metadata og dataprofiler og scores på én absolut skala. Brug scorerne til at sammenligne, niveauerne til at filtrere og domænesiderne til at navigere.",
            "value_not_aligned": "Ikke tilpasset",
            "kpi_baseline": "udgangsværdi {value}, seneste {days} dage, målt {date}",
            "kpi_baseline_undated": "udgangsværdi {value}, målt {date}",
            "doc_method_1": "Inspire læser kun metadata: tabel- og kolonnenavne, typer, kommentarer, nøgler og tags. Den læser aldrig rækkedata.",
            "doc_method_2": "Den udleder forretningskonteksten og genererer derefter kandidater til hvert forretningsdomæne, som dataene understøtter.",
            "doc_method_3": "Dubletter fjernes, hver kandidat scores på værdi og gennemførlighed, og kvalitetsporte frasorterer svage eller udokumenterede idéer.",
            "doc_method_4": "I tilstanden {ultra} gennemgår en elitebedømmer hver resterende kandidat, og deterministiske kontroller bekræfter, at hver nævnt tabel og kolonne findes.",
            "doc_method_5": "Den endelige liste rangordnes, grupperes efter forretningsdomæne og suppleres ved valg med en Databricks Genie Code-notesbog.",
            "doc_glossary": "Ordliste",
            "doc_gl_domain": "Et forretningsområde, der samler relaterede anvendelsescases.",
            "doc_gl_subdomain": "Et smallere tema inden for et forretningsdomæne.",
            "doc_gl_technique": "Den analytiske metode, for eksempel prognose eller klassifikation.",
            "doc_gl_inspire": "Scoren fra 0 til 10, der kombinerer kvalitet og forretningsprioritet.",
            "doc_gl_quality": "Niveauet fra {lowest} til {highest}, som hver anvendelsescase får.",
            "doc_gl_elite": "Scoren i tilstanden {ultra} for værdi, evidens, gennemførlighed og handlingsrettethed.",
            "doc_gl_coverage": "En anvendelsescase under baren, der beholdes, så et efterspurgt domæne er repræsenteret.",
            "doc_gl_notebook": "En Databricks-notesbog med Genie Code-instruktioner til at bygge anvendelsescasen.",
            "doc_notebook_term": "Genie-notesbog",
            "doc_next_steps": "Næste skridt",
            "doc_closing_title": "Fra anvendelsescases til resultater",
            "doc_closing_body": "Vælg de vigtigste anvendelsescases, åbn deres Genie-notesbøger, og byg den første sammen med teamet. Trinene er en foreslået rækkefølge; teamet bestemmer tempoet.",
            "doc_step_select": "Vælg",
            "doc_step_select_body": "Vælg de anvendelsescases, der har den højeste score og en klar sponsor.",
            "doc_step_validate": "Valider data",
            "doc_step_validate_body": "Bekræft de involverede tabeller og deres joins med dataejerne.",
            "doc_step_build": "Byg med Genie Code",
            "doc_step_build_body": "Åbn Genie-notesbogen, og generér den første fungerende version.",
            "doc_step_schedule": "Planlæg",
            "doc_step_schedule_body": "Kør notesbogen som et Databricks-job, der opdaterer resultattabellen.",
            "doc_step_measure": "Mål og skalér",
            "doc_step_measure_body": "Følg den forretningsværdi, der skabes, og udvid derefter til næste anvendelsescase.",
            "doc_agenda": "Dagsorden",
            "doc_roadmap": "Køreplan og næste skridt",
            "doc_use_case_index": "Oversigt over anvendelsescases",
            "doc_continued": "fortsat",
            "doc_not_provided": "Ikke angivet",
            "doc_bob_score": "Best-of-Best-score",
            "elite_cadence": "Beslutningsfrekvens",
            "value_cadence_hourly": "Hver time",
            "value_cadence_daily": "Dagligt",
            "value_cadence_weekly": "Ugentligt",
            "value_cadence_biweekly": "Hver anden uge",
            "value_cadence_monthly": "Månedligt",
            "value_cadence_quarterly": "Kvartalsvis",
            "value_cadence_yearly": "Årligt",
            "value_cadence_real_time": "I realtid",
            "value_cadence_per_event": "Pr. hændelse",
            "doc_generated_by": "Oprettet med {brand} {version} · {date}",
            "doc_disclaimer_ai": "Use cases, scorer og tekster i dette dokument er genereret af AI. Valider dem med dataejerne og forretningens sponsorer, før der træffes en beslutning.",
            "doc_disclaimer_figures": "Scorer, værdier og resultater er skøn, der kun bruges til prioritering. Intet tal i dette dokument er garanteret.",
            "doc_contents": "Indhold",
            "doc_index_lead": "Alle use cases i endelig rækkefølge, i alt {count}.",
            "doc_deck_scope": "Denne præsentation beskriver de højest rangerede use cases, {count} af {total}. PDF-kataloget beskriver dem alle.",
            "doc_deck_scope_all": "Denne præsentation beskriver alle use cases, i alt {total}, med ét kort til hver.",
            "value_general_improvement": "Generel forbedring",
            "value_reduce_cost": "Reducer omkostninger",
            "value_increase_revenue": "Øg omsætning",
            "value_boost_productivity": "Øg produktivitet",
            "value_mitigate_risk": "Mindsk risici",
            "value_protect_revenue": "Beskyt omsætning",
            "value_align_to_regulations": "Overhold regler",
            "value_improve_customer_experience": "Forbedr kundeoplevelse",
            "value_enable_data_driven_decisions": "Muliggør datadrevne beslutninger",
            "value_optimize_operations": "Optimer drift",
            "value_empower_talent": "Styrk talenter",
            "value_enhance_experience": "Forbedr oplevelsen",
            "value_drive_innovation": "Driv innovation",
            "value_achieve_esg": "Opnå ESG",
            "value_execute_strategy": "Udfør strategi",
            "value_forecasting": "Prognose",
            "value_classification": "Klassificering",
            "value_anomaly_detection": "Anomalidetektion",
            "value_cohort_analysis": "Kohorteanalyse",
            "value_segmentation": "Segmentering",
            "value_sentiment_analysis": "Sentimentanalyse",
            "value_trend_analysis": "Trendanalyse",
            "value_prescriptive_analytics": "Præskriptiv analyse",
            "value_root_cause_analysis": "Rodårsagsanalyse",
            "value_optimization": "Optimering",
            "value_recommendation": "Anbefaling",
            "value_time_series_analysis": "Tidsserieanalyse",
            "value_predictive_analytics": "Prædiktiv analyse",
            "value_descriptive_analytics": "Deskriptiv analyse",
        },
        "Norwegian": {
            "main_title": "Brukscasegenerator drevet av Databricks Genie Code",
            "intro": "Denne notatboken inneholder AI-genererte brukscaser basert på skjemaene dine. Nedenfor er en oppsummering av genererte scenarier etter forretningsdomene.",
            "domain": "Forretningsdomene",
            "total": "Totalt antall brukscaser",
            "summaries": "Oppsummeringer av brukscaser",
            "sum_id": "ID",
            "sum_name": "Navn",
            "sum_value": "Forretningsverdi",
            "sum_outcome": "Forventet resultat",
            "warning_header": "ADVARSEL",
            "warning_body": "Ikke kjør denne notatboken. Den er kun ment for demonstrasjon og katalogisering.",
            "disclaimer": "Dette innholdet er AI-generert og bør gjennomgås av en kvalifisert ingeniør før det brukes i produksjon. Databricks er ikke ansvarlig for problemer som oppstår ved bruk av dette innholdet.",
            "detailed_scenarios": "Detaljer om brukscaser",
            "aspect": "Aspekt",
            "description": "Beskrivelse",
            "aspect_domain": "Forretningsdomene",
            "type": "Type",
            "analytics_technique": "Analyseteknikk",
            "primary_table": "Primærtabell",
            "quality": "Kvalitet",
            "priority": "Prioritet",
            "usecase": "Brukscase",
            "description_label": "Beskrivelse",
            "value_type_problem": "Problem",
            "value_type_risk": "Risiko",
            "value_type_opportunity": "Mulighet",
            "value_type_improvement": "Forbedring",
            "value_priority_ultra_high": "Ekstremt høy",
            "value_priority_very_high": "Veldig høy",
            "value_priority_high": "Høy",
            "value_priority_medium": "Middels",
            "value_priority_low": "Lav",
            "value_priority_very_low": "Veldig lav",
            "value_priority_ultra_low": "Ekstremt lav",
            "statement": "Utsagn",
            "solution": "Løsning",
            "aspect_beneficiary": "Mottaker",
            "beneficiary": "Mottaker",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Forretningsprioritetstilpasning",
            "strategic_goals_alignment": "Strategisk måltilpasning",
            "subdomain": "Underdomene",
            "aspect_value": "Forretningsverdi",
            "business_value": "Forretningsverdi",
            "aspect_tables": "Involverte tabeller",
            "quality_reasons": "Kvalitetsårsaker",
            "aspect_ai_function": "AI-funksjon",
            "aspect_analytics_technique": "Analyseteknikk",
            "aspect_primary_table": "Primærtabell",
            "aspect_quality": "Kvalitet",
            "aspect_priority": "Prioritet",
            "strategic_alignment": "Strategisk tilpasning",
            "return_on_investment": "Avkastning på investering",
            "reusability": "Gjenbrukbarhet",
            "time_to_value": "Tid til verdi",
            "data_availability": "Datatilgjengelighet",
            "data_accessibility": "Datatilgang",
            "architecture_fitness": "Arkitekturegnethet",
            "team_skills": "Teamkompetanse",
            "domain_knowledge": "Domenekunnskap",
            "people_allocation": "Personalallokering",
            "budget_allocation": "Budsjettallokering",
            "time_to_production": "Tid til produksjon",
            "value_score": "Verdiscore",
            "feasibility_score": "Gjennomførbarhetsscore",
            "priority_score": "Prioritetsscore",
            "inspire_score": "Inspire-score",
            "quality_score_header": "Kvalitetsscore",
            "pdf_title": "Strategiske AI-brukscaser drevet av Databricks Genie Code",
            "pdf_for": "For",
            "pdf_exec_summary": "Sammendrag",
            "pdf_toc_title": "Brukscasedomener",
            "pdf_detailed_view": "Detaljert brukscasekatalog",
            "pdf_disclaimer_title": "Ansvarsfraskrivelse",
            "pdf_fallback_summary_p1": "Dette dokumentet beskriver {total_cases} høyverdige analytiske brukscaser identifisert for {business_name}.",
            "pdf_fallback_summary_p2": "De følgende sidene gir en detaljert oversikt over disse mulighetene, kategorisert etter forretningsdomene.",
            "pptx_main_title": "Strategiske AI-brukscaser drevet av Databricks Genie Code",
            "pptx_for": "For",
            "pptx_disclaimer_title": "Ansvarsfraskrivelse",
            "pptx_domain_suffix": "Brukscaser",
            "elite_verdict": "Vurdering",
            "elite_decision": "Beslutning",
            "elite_decision_owner": "Beslutningseier",
            "elite_kpi": "KPI",
            "elite_value_driver": "Verdidriver",
            "elite_evidence_columns": "Bevis-kolonner",
            "elite_join_path": "Koblingssti",
            "elite_build_path": "Implementeringssti",
            "elite_red_team_note": "Red team-merknad",
            "elite_coverage_pick": "Valg for domenedekning, under nivået",
            "elite_fallback_pick": "Under {mode}-nivået",
            "elite_fallback_near_miss": "Nære på, under {mode}-nivået",
            "doc_fallback_pick": "Under nivået",
            "doc_fallback_notice": "Ingen brukstilfelle nådde {mode}-nivået. Dette er de nærmeste kandidatene, hver merket under nivået.",
            "elite_reason": "Årsak",
            "untranslated_notice": "Vist på engelsk (oversettelse ikke tilgjengelig)",
            "more_use_cases": "...og {count} flere brukstilfeller.",
            "executive_summary_not_available": "Sammendrag ikke tilgjengelig.",
            "domain_summary_not_available": "Domeneoversikt ikke tilgjengelig.",
            "summary_not_available": "Oversikt ikke tilgjengelig.",
            "doc_catalog_kicker": "Katalog over AI-brukscaser",
            "doc_prepared_for": "Utarbeidet for",
            "doc_generated_on": "Opprettet",
            "doc_quality_mode": "Kvalitetsmodus",
            "doc_use_cases": "Brukscaser",
            "doc_domains": "Forretningsdomener",
            "doc_elite_use_cases": "Elitebrukscaser",
            "doc_high_or_better": "Høy kvalitet eller bedre",
            "doc_best_score": "Beste Inspire-score",
            "doc_not_scored": "Ikke vurdert",
            "doc_coverage_picks": "Dekningsvalg",
            "doc_coverage_pick": "Dekningsvalg",
            "value_correlation_analysis": "Korrelasjonsanalyse",
            "value_pareto_analysis": "Paretoanalyse",
            "value_funnel_analysis": "Traktanalyse",
            "value_document_processing": "Dokumentbehandling",
            "value_extraction": "Uttrekk",
            "value_ai_analysis": "AI-analyse",
            "value_simulation": "Simulering",
            "value_geospatial_analysis": "Geospatial analyse",
            "value_predictive_modeling": "Prediktiv modellering",
            "value_clustering": "Klyngeanalyse",
            "value_regression_analysis": "Regresjonsanalyse",
            "value_survival_analysis": "Overlevelsesanalyse",
            "value_network_analysis": "Nettverksanalyse",
            "value_mode_good": "God",
            "value_mode_high": "Høy",
            "value_mode_ultra_high": "Ultrahøy",
            "doc_exec_kicker": "Muligheten",
            "doc_guide_kicker": "Leseveiledning",
            "doc_how_to_read": "Slik leser du katalogen",
            "doc_guide_lead": "Hver brukscase kommer fra organisasjonens egne metadata og får score på én og samme absolutte skala. Bruk scorene til å sammenligne, nivåene til å filtrere og domenesidene til å navigere.",
            "doc_inspire_score_title": "Inspire-score, fra 0 til 10",
            "doc_inspire_score_body": "Kombinerer kvaliteten til en brukscase med forretningsprioriteten. Høyere er bedre, og katalogen er sortert etter den.",
            "doc_quality_tiers_title": "Kvalitetsnivåer",
            "doc_quality_tiers_body": "Hver brukscase får et nivå fra {lowest} til {highest} som viser hvor spesifikk, verdifull og byggbar den er med tilgjengelige data.",
            "doc_modes_title": "Kvalitetsmoduser",
            "doc_mode_good_body": "Beholder brukscaser vurdert som {medium} eller bedre.",
            "doc_mode_high_body": "Beholder bare brukscaser vurdert som {high} eller bedre.",
            "doc_mode_ultra_body": "Beholder bare brukscaser som klarer en absolutt terskel for verdi, evidens, gjennomførbarhet og handlingsrettethet. Det finnes ingen kvote.",
            "doc_this_catalog": "Denne katalogen",
            "doc_elite_score_title": "Elitescore",
            "doc_elite_score_body": "Brukes i modusen {ultra}. Den vekter forretningsverdi, evidensstyrke, gjennomførbarhet og handlingsrettethet, og en elitebrukscase får minst {min_score} av 10 på hver.",
            "doc_coverage_title": "Valg for domenedekning",
            "doc_coverage_body": "Når et etterspurt forretningsdomene ikke har noen brukscase over terskelen, beholdes den sterkeste kandidaten og merkes som dekningsvalg under terskelen.",
            "doc_portfolio_kicker": "Portefølje",
            "doc_portfolio_glance": "Porteføljen i korte trekk",
            "doc_per_domain": "Brukscaser per domene",
            "doc_quality_distribution": "Kvalitetsfordeling",
            "doc_top_use_cases": "Viktigste brukscaser",
            "doc_rank": "Plassering",
            "doc_domain_label": "Domene",
            "doc_domain_overview": "Om dette domenet",
            "doc_what_it_covers": "Hva det dekker",
            "doc_top_opportunities": "Største muligheter",
            "doc_domain_fallback": "Dette domenet samler brukscaser innen {subdomains}, totalt {count}. De sterkeste mulighetene står nedenfor og beskrives på de neste sidene.",
            "doc_open_notebook": "Åpne Genie-notatboken",
            "doc_elite_score": "Elitescore",
            "doc_best_of_best": "Det beste av det beste",
            "doc_appendix": "Vedlegg",
            "doc_methodology": "Metode",
            "strategy_path": "Strategisti",
            "kpis": "KPI-er",
            "expected_key_results": "Forventede nøkkelresultater",
            "measures_of_success": "Suksesskriterier",
            "financial_benefits": "Økonomiske fordeler",
            "non_financial_benefits": "Ikke-økonomiske fordeler",
            "foundations": "Datafundamenter",
            "doc_foundations_title": "Felles datafundamenter",
            "doc_foundations_lead": "Tabeller og koblinger som to eller flere brukscaser trenger. Når de bygges først, åpnes flest brukscaser. Rekkefølgen er en sekvens, ikke en tidsplan.",
            "doc_foundation_tables": "Tabeller",
            "doc_foundation_unlocks": "Mulige brukscaser",
            "doc_data_access_standard": "Standard oppdagelse: Inspire leste ingen kundedata. Bare metadata, definisjoner av metrikkvisninger og brukstellinger ble brukt.",
            "doc_data_access_deep": "Dyp oppdagelse: Inspire leste aggregater fra tabellene i omfanget, totalt {tables} (antall rader, datodekning, andel tomme verdier, antall unike verdier og korte verdilister for ikke-personlige kolonner). Ingen datarader ble sendt til KI-en.",
            "doc_method_1_deep": "Inspire leser metadata og ved dyp oppdagelse også aggregater beregnet over tabellene. Den sender aldri datarader til KI-en.",
            "doc_guide_lead_deep": "Hver brukscase kommer fra organisasjonens egne metadata og dataprofiler og scores på én absolutt skala. Bruk poengene til å sammenligne, nivåene til å filtrere og domenesidene til å navigere.",
            "value_not_aligned": "Ikke tilpasset",
            "kpi_baseline": "utgangsverdi {value}, siste {days} dager, målt {date}",
            "kpi_baseline_undated": "utgangsverdi {value}, målt {date}",
            "doc_method_1": "Inspire leser bare metadata: tabell- og kolonnenavn, typer, kommentarer, nøkler og tagger. Den leser aldri raddata.",
            "doc_method_2": "Den utleder forretningskonteksten og genererer deretter kandidater for hvert forretningsdomene som dataene støtter.",
            "doc_method_3": "Duplikater fjernes, hver kandidat får score på verdi og gjennomførbarhet, og kvalitetsporter luker ut svake eller udokumenterte ideer.",
            "doc_method_4": "I modusen {ultra} gjennomgår en elitedommer hver gjenværende kandidat, og deterministiske kontroller bekrefter at hver nevnte tabell og kolonne finnes.",
            "doc_method_5": "Den endelige listen rangeres, grupperes etter forretningsdomene og kobles ved valg til en Databricks Genie Code-notatbok.",
            "doc_glossary": "Ordliste",
            "doc_gl_domain": "Et forretningsområde som samler relaterte brukscaser.",
            "doc_gl_subdomain": "Et smalere tema innenfor et forretningsdomene.",
            "doc_gl_technique": "Den analytiske metoden, for eksempel prognose eller klassifisering.",
            "doc_gl_inspire": "Scoren fra 0 til 10 som kombinerer kvalitet og forretningsprioritet.",
            "doc_gl_quality": "Nivået fra {lowest} til {highest} som hver brukscase får.",
            "doc_gl_elite": "Scoren i modusen {ultra} for verdi, evidens, gjennomførbarhet og handlingsrettethet.",
            "doc_gl_coverage": "En brukscase under terskelen som beholdes slik at et etterspurt domene er representert.",
            "doc_gl_notebook": "En Databricks-notatbok med Genie Code-instruksjoner for å bygge brukscasen.",
            "doc_notebook_term": "Genie-notatbok",
            "doc_next_steps": "Neste steg",
            "doc_closing_title": "Fra brukscaser til resultater",
            "doc_closing_body": "Velg de viktigste brukscasene, åpne Genie-notatbøkene deres og bygg den første sammen med teamet. Stegene er en foreslått rekkefølge; teamet bestemmer tempoet.",
            "doc_step_select": "Velg",
            "doc_step_select_body": "Velg brukscasene med høyest score og en tydelig sponsor.",
            "doc_step_validate": "Valider dataene",
            "doc_step_validate_body": "Bekreft tabellene som inngår og koblingene mellom dem med dataeierne.",
            "doc_step_build": "Bygg med Genie Code",
            "doc_step_build_body": "Åpne Genie-notatboken og generer den første fungerende versjonen.",
            "doc_step_schedule": "Planlegg",
            "doc_step_schedule_body": "Kjør notatboken som en Databricks-jobb som oppdaterer resultattabellen.",
            "doc_step_measure": "Mål og skaler",
            "doc_step_measure_body": "Følg forretningsverdien som skapes, og utvid deretter til neste brukscase.",
            "doc_agenda": "Agenda",
            "doc_roadmap": "Veikart og neste steg",
            "doc_use_case_index": "Oversikt over brukscaser",
            "doc_continued": "fortsatt",
            "doc_not_provided": "Ikke oppgitt",
            "doc_bob_score": "Best-of-Best-poeng",
            "elite_cadence": "Beslutningsfrekvens",
            "value_cadence_hourly": "Hver time",
            "value_cadence_daily": "Daglig",
            "value_cadence_weekly": "Ukentlig",
            "value_cadence_biweekly": "Annenhver uke",
            "value_cadence_monthly": "Månedlig",
            "value_cadence_quarterly": "Kvartalsvis",
            "value_cadence_yearly": "Årlig",
            "value_cadence_real_time": "I sanntid",
            "value_cadence_per_event": "Per hendelse",
            "doc_generated_by": "Laget med {brand} {version} · {date}",
            "doc_disclaimer_ai": "Brukstilfellene, poengene og tekstene i dette dokumentet er generert av KI. Valider dem med dataeierne og forretningssponsorene før en beslutning tas.",
            "doc_disclaimer_figures": "Poeng, verdier og resultater er anslag som kun brukes til prioritering. Ingen tall i dette dokumentet er garantert.",
            "doc_contents": "Innhold",
            "doc_index_lead": "Alle brukstilfeller i endelig rangering, totalt {count}.",
            "doc_deck_scope": "Denne presentasjonen beskriver de høyest rangerte brukstilfellene, {count} av {total}. PDF-katalogen beskriver alle.",
            "doc_deck_scope_all": "Denne presentasjonen beskriver alle brukstilfellene, totalt {total}, ett kort for hvert.",
            "value_general_improvement": "Generell forbedring",
            "value_reduce_cost": "Reduser kostnader",
            "value_increase_revenue": "Øk inntekter",
            "value_boost_productivity": "Øk produktivitet",
            "value_mitigate_risk": "Reduser risiko",
            "value_protect_revenue": "Beskytt inntekter",
            "value_align_to_regulations": "Overhold regler",
            "value_improve_customer_experience": "Forbedre kundeopplevelse",
            "value_enable_data_driven_decisions": "Muliggjøre datadrevne beslutninger",
            "value_optimize_operations": "Optimere drift",
            "value_empower_talent": "Styrke talenter",
            "value_enhance_experience": "Forbedre opplevelsen",
            "value_drive_innovation": "Drive innovasjon",
            "value_achieve_esg": "Oppnå ESG",
            "value_execute_strategy": "Gjennomføre strategi",
            "value_forecasting": "Prognose",
            "value_classification": "Klassifisering",
            "value_anomaly_detection": "Anomalideteksjon",
            "value_cohort_analysis": "Kohortanalyse",
            "value_segmentation": "Segmentering",
            "value_sentiment_analysis": "Sentimentanalyse",
            "value_trend_analysis": "Trendanalyse",
            "value_prescriptive_analytics": "Preskriptiv analyse",
            "value_root_cause_analysis": "Rotårsaksanalyse",
            "value_optimization": "Optimalisering",
            "value_recommendation": "Anbefaling",
            "value_time_series_analysis": "Tidsserieanalyse",
            "value_predictive_analytics": "Prediktiv analyse",
            "value_descriptive_analytics": "Deskriptiv analyse",
        },
        "Finnish": {
            "main_title": "Käyttötapausgeneraattori Databricks Genie Coden avulla",
            "intro": "Tämä muistikirja sisältää tekoälyn luomia käyttötapauksia skeemojesi perusteella. Alla on yhteenveto luoduista skenaarioista liiketoiminta-alueittain.",
            "domain": "Liiketoiminta-alue",
            "total": "Käyttötapauksia yhteensä",
            "summaries": "Käyttötapausten yhteenvedot",
            "sum_id": "ID",
            "sum_name": "Nimi",
            "sum_value": "Liiketoiminta-arvo",
            "sum_outcome": "Odotettu tulos",
            "warning_header": "VAROITUS",
            "warning_body": "Älä suorita tätä muistikirjaa. Se on tarkoitettu vain esittely- ja luettelointitarkoituksiin.",
            "disclaimer": "Tämä sisältö on tekoälyn luomaa ja pätevän insinöörin tulee tarkistaa se ennen tuotantokäyttöä. Databricks ei ole vastuussa mistään tämän sisällön käytöstä aiheutuvista ongelmista.",
            "detailed_scenarios": "Käyttötapausten yksityiskohdat",
            "aspect": "Näkökulma",
            "description": "Kuvaus",
            "aspect_domain": "Liiketoiminta-alue",
            "type": "Tyyppi",
            "analytics_technique": "Analytiikkatekniikka",
            "primary_table": "Ensisijainen taulu",
            "quality": "Laatu",
            "priority": "Prioriteetti",
            "usecase": "Käyttötapaus",
            "description_label": "Kuvaus",
            "value_type_problem": "Ongelma",
            "value_type_risk": "Riski",
            "value_type_opportunity": "Mahdollisuus",
            "value_type_improvement": "Parannus",
            "value_priority_ultra_high": "Erittäin korkea",
            "value_priority_very_high": "Hyvin korkea",
            "value_priority_high": "Korkea",
            "value_priority_medium": "Keskitaso",
            "value_priority_low": "Matala",
            "value_priority_very_low": "Hyvin matala",
            "value_priority_ultra_low": "Erittäin matala",
            "statement": "Lausunto",
            "solution": "Ratkaisu",
            "aspect_beneficiary": "Edunsaaja",
            "beneficiary": "Edunsaaja",
            "aspect_sponsor": "Sponsori",
            "sponsor": "Sponsori",
            "business_priority_alignment": "Liiketoimintaprioriteetin kohdistaminen",
            "strategic_goals_alignment": "Strategisten tavoitteiden kohdistaminen",
            "subdomain": "Alidomain",
            "aspect_value": "Liiketoiminta-arvo",
            "business_value": "Liiketoiminta-arvo",
            "aspect_tables": "Liittyvät taulut",
            "quality_reasons": "Laadun syyt",
            "aspect_ai_function": "Tekoälytoiminto",
            "aspect_analytics_technique": "Analytiikkatekniikka",
            "aspect_primary_table": "Ensisijainen taulu",
            "aspect_quality": "Laatu",
            "aspect_priority": "Prioriteetti",
            "strategic_alignment": "Strateginen kohdistaminen",
            "return_on_investment": "Sijoitetun pääoman tuotto",
            "reusability": "Uudelleenkäytettävyys",
            "time_to_value": "Aika arvon saavuttamiseen",
            "data_availability": "Datan saatavuus",
            "data_accessibility": "Datan saavutettavuus",
            "architecture_fitness": "Arkkitehtuurin soveltuvuus",
            "team_skills": "Tiimin osaaminen",
            "domain_knowledge": "Toimialaosaaminen",
            "people_allocation": "Henkilöstöallokointi",
            "budget_allocation": "Budjettiallokointi",
            "time_to_production": "Aika tuotantoon",
            "value_score": "Arvopisteet",
            "feasibility_score": "Toteutettavuuspisteet",
            "priority_score": "Prioriteettipisteet",
            "inspire_score": "Inspire-pisteet",
            "quality_score_header": "Laatupisteet",
            "pdf_title": "Strategiset tekoälykäyttötapaukset Databricks Genie Coden avulla",
            "pdf_for": "Kohde",
            "pdf_exec_summary": "Yhteenveto",
            "pdf_toc_title": "Käyttötapausdomainit",
            "pdf_detailed_view": "Yksityiskohtainen käyttötapausluettelo",
            "pdf_disclaimer_title": "Vastuuvapauslauseke",
            "pdf_fallback_summary_p1": "Tämä dokumentti kuvaa {total_cases} korkea-arvoista analyyttistä käyttötapausta, jotka on tunnistettu {business_name} toimintaan.",
            "pdf_fallback_summary_p2": "Seuraavat sivut tarjoavat yksityiskohtaisen erittelyn näistä mahdollisuuksista liiketoiminta-alueittain.",
            "pptx_main_title": "Strategiset tekoälykäyttötapaukset Databricks Genie Coden avulla",
            "pptx_for": "Kohde",
            "pptx_disclaimer_title": "Vastuuvapauslauseke",
            "pptx_domain_suffix": "Käyttötapaukset",
            "elite_verdict": "Arvio",
            "elite_decision": "Päätös",
            "elite_decision_owner": "Päätöksen omistaja",
            "elite_kpi": "KPI",
            "elite_value_driver": "Arvoajuri",
            "elite_evidence_columns": "Näyttösarakkeet",
            "elite_join_path": "Liitospolku",
            "elite_build_path": "Toteutuspolku",
            "elite_red_team_note": "Red team -huomio",
            "elite_coverage_pick": "Toimialueen kattavuusvalinta, alle rajan",
            "elite_fallback_pick": "Alle {mode}-rajan",
            "elite_fallback_near_miss": "Lähellä, alle {mode}-rajan",
            "doc_fallback_pick": "Alle rajan",
            "doc_fallback_notice": "Yksikään käyttötapaus ei yltänyt {mode}-rajalle. Nämä ovat lähimmät ehdokkaat, kukin merkitty rajan alle.",
            "elite_reason": "Syy",
            "untranslated_notice": "Näytetään englanniksi (käännös ei saatavilla)",
            "more_use_cases": "...ja {count} muuta käyttötapausta.",
            "executive_summary_not_available": "Yhteenveto ei saatavilla.",
            "domain_summary_not_available": "Alueen yhteenveto ei saatavilla.",
            "summary_not_available": "Yhteenveto ei saatavilla.",
            "doc_catalog_kicker": "Tekoälyn käyttötapausluettelo",
            "doc_prepared_for": "Laadittu organisaatiolle",
            "doc_generated_on": "Luotu",
            "doc_quality_mode": "Laatutila",
            "doc_use_cases": "Käyttötapaukset",
            "doc_domains": "Liiketoiminta-alueet",
            "doc_elite_use_cases": "Eliittikäyttötapaukset",
            "doc_high_or_better": "Korkea laatu tai parempi",
            "doc_best_score": "Paras Inspire-pistemäärä",
            "doc_not_scored": "Ei pisteytetty",
            "doc_coverage_picks": "Kattavuusvalinnat",
            "doc_coverage_pick": "Kattavuusvalinta",
            "value_correlation_analysis": "Korrelaatioanalyysi",
            "value_pareto_analysis": "Pareto-analyysi",
            "value_funnel_analysis": "Suppiloanalyysi",
            "value_document_processing": "Asiakirjojen käsittely",
            "value_extraction": "Tiedon poiminta",
            "value_ai_analysis": "Tekoälyanalyysi",
            "value_simulation": "Simulointi",
            "value_geospatial_analysis": "Paikkatietoanalyysi",
            "value_predictive_modeling": "Ennustava mallinnus",
            "value_clustering": "Klusterointi",
            "value_regression_analysis": "Regressioanalyysi",
            "value_survival_analysis": "Elinaika-analyysi",
            "value_network_analysis": "Verkostoanalyysi",
            "value_mode_good": "Hyvä",
            "value_mode_high": "Korkea",
            "value_mode_ultra_high": "Ultrakorkea",
            "doc_exec_kicker": "Mahdollisuus",
            "doc_guide_kicker": "Lukuopas",
            "doc_how_to_read": "Näin luet tätä luetteloa",
            "doc_guide_lead": "Jokainen käyttötapaus perustuu organisaation omiin metatietoihin, ja se pisteytetään yhdellä ja samalla absoluuttisella asteikolla. Käytä pisteitä vertailuun, tasoja suodatukseen ja alueiden sivuja selaamiseen.",
            "doc_inspire_score_title": "Inspire-pistemäärä, 0:sta 10:een",
            "doc_inspire_score_body": "Yhdistää käyttötapauksen laadun ja sen liiketoimintaprioriteetin. Suurempi on parempi, ja luettelo on järjestetty sen mukaan.",
            "doc_quality_tiers_title": "Laatutasot",
            "doc_quality_tiers_body": "Jokainen käyttötapaus saa tason väliltä {lowest} ja {highest}, ja taso kertoo, kuinka täsmällinen, arvokas ja toteutettavissa se on käytettävissä olevilla tiedoilla.",
            "doc_modes_title": "Laatutilat",
            "doc_mode_good_body": "Säilyttää käyttötapaukset, joiden arvio on {medium} tai parempi.",
            "doc_mode_high_body": "Säilyttää vain käyttötapaukset, joiden arvio on {high} tai parempi.",
            "doc_mode_ultra_body": "Säilyttää vain käyttötapaukset, jotka ylittävät absoluuttisen rajan arvossa, näytössä, toteutettavuudessa ja toimeenpantavuudessa. Kiintiötä ei ole.",
            "doc_this_catalog": "Tämä luettelo",
            "doc_elite_score_title": "Eliittipisteet",
            "doc_elite_score_body": "Käytetään tilassa {ultra}. Se painottaa liiketoiminta-arvoa, näytön vahvuutta, toteutettavuutta ja toimeenpantavuutta, ja eliittikäyttötapaus saa jokaisesta vähintään {min_score}/10.",
            "doc_coverage_title": "Toimialueen kattavuusvalinnat",
            "doc_coverage_body": "Kun pyydetyllä liiketoiminta-alueella ei ole yhtään rajan ylittävää käyttötapausta, sen vahvin ehdokas säilytetään ja merkitään rajan alittavaksi kattavuusvalinnaksi.",
            "doc_portfolio_kicker": "Portfolio",
            "doc_portfolio_glance": "Portfolio yhdellä silmäyksellä",
            "doc_per_domain": "Käyttötapaukset alueittain",
            "doc_quality_distribution": "Laadun jakauma",
            "doc_top_use_cases": "Tärkeimmät käyttötapaukset",
            "doc_rank": "Sijoitus",
            "doc_domain_label": "Alue",
            "doc_domain_overview": "Tietoa alueesta",
            "doc_what_it_covers": "Mitä se kattaa",
            "doc_top_opportunities": "Tärkeimmät mahdollisuudet",
            "doc_domain_fallback": "Tämä alue kokoaa käyttötapauksia seuraavista aiheista: {subdomains}, yhteensä {count}. Vahvimmat mahdollisuudet on lueteltu alla ja kuvattu tarkemmin seuraavilla sivuilla.",
            "doc_open_notebook": "Avaa Genie-muistikirja",
            "doc_elite_score": "Eliittipisteet",
            "doc_best_of_best": "Parhaista parhaat",
            "doc_appendix": "Liite",
            "doc_methodology": "Menetelmä",
            "strategy_path": "Strategiapolku",
            "kpis": "KPI-mittarit",
            "expected_key_results": "Odotetut avaintulokset",
            "measures_of_success": "Onnistumisen mittarit",
            "financial_benefits": "Taloudelliset hyödyt",
            "non_financial_benefits": "Muut kuin taloudelliset hyödyt",
            "foundations": "Datan perustat",
            "doc_foundations_title": "Jaetut datan perustat",
            "doc_foundations_lead": "Taulut ja liitokset, joita kaksi tai useampi käyttötapaus tarvitsee. Kun ne rakennetaan ensin, käyttöön tulee eniten käyttötapauksia. Järjestys on vaihejärjestys, ei aikataulu.",
            "doc_foundation_tables": "Taulut",
            "doc_foundation_unlocks": "Mahdollistuvat käyttötapaukset",
            "doc_data_access_standard": "Vakiotunnistus: Inspire ei lukenut asiakkaan dataa. Se käytti vain metatietoja, mittarinäkymien määritelmiä ja käyttömääriä.",
            "doc_data_access_deep": "Syvä tunnistus: Inspire luki koosteita rajauksen tauluista, yhteensä {tables} (rivimäärät, päivämäärien kattavuus, tyhjien arvojen osuudet, erillisten arvojen määrät ja lyhyet arvoluettelot muista kuin henkilötietosarakkeista). Yhtään datariviä ei lähetetty tekoälylle.",
            "doc_method_1_deep": "Inspire lukee metatiedot ja syvässä tunnistuksessa myös tauluista lasketut koosteet. Se ei koskaan lähetä datarivejä tekoälylle.",
            "doc_guide_lead_deep": "Jokainen käyttötapaus perustuu organisaation omiin metatietoihin ja dataprofiileihin, ja se pisteytetään yhdellä absoluuttisella asteikolla. Vertaa pisteillä, suodata tasoilla ja liiku toimialuesivujen avulla.",
            "value_not_aligned": "Ei kohdistettu",
            "kpi_baseline": "lähtötaso {value}, viimeiset {days} päivää, mitattu {date}",
            "kpi_baseline_undated": "lähtötaso {value}, mitattu {date}",
            "doc_method_1": "Inspire lukee vain metatietoja: taulujen ja sarakkeiden nimiä, tyyppejä, kommentteja, avaimia ja tunnisteita. Se ei koskaan lue rivitietoja.",
            "doc_method_2": "Se johtaa liiketoimintakontekstin ja luo sitten ehdokkaat jokaiselle liiketoiminta-alueelle, jota tiedot tukevat.",
            "doc_method_3": "Kaksoiskappaleet poistetaan, jokainen ehdokas pisteytetään arvon ja toteutettavuuden mukaan, ja laatuportit karsivat heikot tai perusteettomat ideat.",
            "doc_method_4": "Tilassa {ultra} eliittiarvioija tarkastaa jokaisen jäljelle jääneen ehdokkaan, ja deterministiset tarkistukset varmistavat, että jokainen mainittu taulu ja sarake on olemassa.",
            "doc_method_5": "Lopullinen luettelo järjestetään, ryhmitellään liiketoiminta-alueittain ja liitetään valittaessa Databricks Genie Code -muistikirjaan.",
            "doc_glossary": "Sanasto",
            "doc_gl_domain": "Liiketoiminnan alue, joka kokoaa toisiinsa liittyvät käyttötapaukset.",
            "doc_gl_subdomain": "Kapeampi teema liiketoiminta-alueen sisällä.",
            "doc_gl_technique": "Analyysimenetelmä, esimerkiksi ennustaminen tai luokittelu.",
            "doc_gl_inspire": "Pistemäärä 0:sta 10:een, joka yhdistää laadun ja liiketoimintaprioriteetin.",
            "doc_gl_quality": "Taso väliltä {lowest} ja {highest}, jonka kukin käyttötapaus saa.",
            "doc_gl_elite": "Tilan {ultra} pistemäärä arvosta, näytöstä, toteutettavuudesta ja toimeenpantavuudesta.",
            "doc_gl_coverage": "Rajan alittava käyttötapaus, joka säilytetään, jotta pyydetty alue on edustettuna.",
            "doc_gl_notebook": "Databricks-muistikirja, jossa on Genie Code -ohjeet käyttötapauksen rakentamiseen.",
            "doc_notebook_term": "Genie-muistikirja",
            "doc_next_steps": "Seuraavat vaiheet",
            "doc_closing_title": "Käyttötapauksista tuloksiin",
            "doc_closing_body": "Valitse tärkeimmät käyttötapaukset, avaa niiden Genie-muistikirjat ja rakenna ensimmäinen yhdessä tiimin kanssa. Vaiheet ovat ehdotettu järjestys; tiimi päättää tahdin.",
            "doc_step_select": "Valitse",
            "doc_step_select_body": "Valitse käyttötapaukset, joilla on paras pistemäärä ja selkeä sponsori.",
            "doc_step_validate": "Varmista tiedot",
            "doc_step_validate_body": "Vahvista mukana olevat taulut ja niiden liitokset tietojen omistajien kanssa.",
            "doc_step_build": "Rakenna Genie Codella",
            "doc_step_build_body": "Avaa Genie-muistikirja ja luo ensimmäinen toimiva versio.",
            "doc_step_schedule": "Ajoita",
            "doc_step_schedule_body": "Suorita muistikirja Databricks-työnä, joka päivittää tulostaulun.",
            "doc_step_measure": "Mittaa ja laajenna",
            "doc_step_measure_body": "Seuraa saavutettua liiketoiminta-arvoa ja laajenna sitten seuraavaan käyttötapaukseen.",
            "doc_agenda": "Esityslista",
            "doc_roadmap": "Tiekartta ja seuraavat vaiheet",
            "doc_use_case_index": "Käyttötapausten hakemisto",
            "doc_continued": "jatkuu",
            "doc_not_provided": "Ei ilmoitettu",
            "doc_bob_score": "Best-of-Best-pisteet",
            "elite_cadence": "Päätöksenteon tiheys",
            "value_cadence_hourly": "Tunneittain",
            "value_cadence_daily": "Päivittäin",
            "value_cadence_weekly": "Viikoittain",
            "value_cadence_biweekly": "Joka toinen viikko",
            "value_cadence_monthly": "Kuukausittain",
            "value_cadence_quarterly": "Neljännesvuosittain",
            "value_cadence_yearly": "Vuosittain",
            "value_cadence_real_time": "Reaaliaikaisesti",
            "value_cadence_per_event": "Tapahtumakohtaisesti",
            "doc_generated_by": "Luotu: {brand} {version} · {date}",
            "doc_disclaimer_ai": "Tämän asiakirjan käyttötapaukset, pisteet ja kuvaukset ovat tekoälyn tuottamia. Vahvista ne tietojen omistajien ja liiketoiminnan sponsorien kanssa ennen päätöksiä.",
            "doc_disclaimer_figures": "Pisteet, arvot ja tulokset ovat arvioita, jotka on tarkoitettu vain priorisointiin. Mikään tämän asiakirjan luku ei ole taattu.",
            "doc_contents": "Sisällys",
            "doc_index_lead": "Kaikki käyttötapaukset lopullisessa järjestyksessä, yhteensä {count}.",
            "doc_deck_scope": "Tämä esitys kuvaa parhaiten sijoittuneet käyttötapaukset, {count}/{total}. PDF-luettelo kuvaa ne kaikki.",
            "doc_deck_scope_all": "Tämä esitys kuvaa kaikki käyttötapaukset, yhteensä {total}, kustakin oma kortti.",
            "value_general_improvement": "Yleinen parannus",
            "value_reduce_cost": "Vähennä kustannuksia",
            "value_increase_revenue": "Kasvata liikevaihtoa",
            "value_boost_productivity": "Lisää tuottavuutta",
            "value_mitigate_risk": "Vähennä riskejä",
            "value_protect_revenue": "Suojaa liikevaihtoa",
            "value_align_to_regulations": "Noudata säädöksiä",
            "value_improve_customer_experience": "Paranna asiakaskokemusta",
            "value_enable_data_driven_decisions": "Mahdollista datapohjaiset päätökset",
            "value_optimize_operations": "Optimoi toimintaa",
            "value_empower_talent": "Vahvista osaajia",
            "value_enhance_experience": "Paranna kokemusta",
            "value_drive_innovation": "Edistä innovaatiota",
            "value_achieve_esg": "Saavuta ESG",
            "value_execute_strategy": "Toteuta strategia",
            "value_forecasting": "Ennustaminen",
            "value_classification": "Luokittelu",
            "value_anomaly_detection": "Poikkeamien tunnistus",
            "value_cohort_analysis": "Kohortti-analyysi",
            "value_segmentation": "Segmentointi",
            "value_sentiment_analysis": "Sentimenttianalyysi",
            "value_trend_analysis": "Trendianalyysi",
            "value_prescriptive_analytics": "Preskriptiivinen analytiikka",
            "value_root_cause_analysis": "Juurisyyanalyysi",
            "value_optimization": "Optimointi",
            "value_recommendation": "Suositus",
            "value_time_series_analysis": "Aikasarja-analyysi",
            "value_predictive_analytics": "Prediktiivinen analytiikka",
            "value_descriptive_analytics": "Deskriptiivinen analytiikka",
        },
        "Polish": {
            "main_title": "Generator przypadków użycia oparty na Databricks Genie Code",
            "intro": "Ten notatnik zawiera przypadki użycia wygenerowane przez AI na podstawie Twoich schematów. Poniżej znajduje się podsumowanie wygenerowanych scenariuszy według domeny biznesowej.",
            "domain": "Domena biznesowa",
            "total": "Łączna liczba przypadków użycia",
            "summaries": "Podsumowania przypadków użycia",
            "sum_id": "ID",
            "sum_name": "Nazwa",
            "sum_value": "Wartość biznesowa",
            "sum_outcome": "Oczekiwany wynik",
            "warning_header": "OSTRZEŻENIE",
            "warning_body": "Nie uruchamiaj tego notatnika. Jest przeznaczony wyłącznie do celów demonstracyjnych i katalogowania.",
            "disclaimer": "Ta treść została wygenerowana przez AI i powinna zostać zweryfikowana przez wykwalifikowanego inżyniera przed wdrożeniem na produkcję. Databricks nie ponosi odpowiedzialności za jakiekolwiek problemy wynikające z korzystania z tej treści.",
            "detailed_scenarios": "Szczegóły przypadków użycia",
            "aspect": "Aspekt",
            "description": "Opis",
            "aspect_domain": "Domena biznesowa",
            "type": "Typ",
            "analytics_technique": "Technika analityczna",
            "primary_table": "Tabela główna",
            "quality": "Jakość",
            "priority": "Priorytet",
            "usecase": "Przypadek użycia",
            "description_label": "Opis",
            "value_type_problem": "Problem",
            "value_type_risk": "Ryzyko",
            "value_type_opportunity": "Szansa",
            "value_type_improvement": "Ulepszenie",
            "value_priority_ultra_high": "Ekstremalnie wysoki",
            "value_priority_very_high": "Bardzo wysoki",
            "value_priority_high": "Wysoki",
            "value_priority_medium": "Średni",
            "value_priority_low": "Niski",
            "value_priority_very_low": "Bardzo niski",
            "value_priority_ultra_low": "Ekstremalnie niski",
            "statement": "Stwierdzenie",
            "solution": "Rozwiązanie",
            "aspect_beneficiary": "Beneficjent",
            "beneficiary": "Beneficjent",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Zgodność z priorytetami biznesowymi",
            "strategic_goals_alignment": "Zgodność z celami strategicznymi",
            "subdomain": "Poddomena",
            "aspect_value": "Wartość biznesowa",
            "business_value": "Wartość biznesowa",
            "aspect_tables": "Powiązane tabele",
            "quality_reasons": "Przyczyny jakości",
            "aspect_ai_function": "Funkcja AI",
            "aspect_analytics_technique": "Technika analityczna",
            "aspect_primary_table": "Tabela główna",
            "aspect_quality": "Jakość",
            "aspect_priority": "Priorytet",
            "strategic_alignment": "Zgodność strategiczna",
            "return_on_investment": "Zwrot z inwestycji",
            "reusability": "Możliwość ponownego użycia",
            "time_to_value": "Czas do wartości",
            "data_availability": "Dostępność danych",
            "data_accessibility": "Dostęp do danych",
            "architecture_fitness": "Dopasowanie architektoniczne",
            "team_skills": "Umiejętności zespołu",
            "domain_knowledge": "Wiedza domenowa",
            "people_allocation": "Alokacja personelu",
            "budget_allocation": "Alokacja budżetu",
            "time_to_production": "Czas do produkcji",
            "value_score": "Wynik wartości",
            "feasibility_score": "Wynik wykonalności",
            "priority_score": "Wynik priorytetu",
            "inspire_score": "Wynik Inspire",
            "quality_score_header": "Wynik jakości",
            "pdf_title": "Strategiczne przypadki użycia AI oparte na Databricks Genie Code",
            "pdf_for": "Dla",
            "pdf_exec_summary": "Podsumowanie wykonawcze",
            "pdf_toc_title": "Domeny przypadków użycia",
            "pdf_detailed_view": "Szczegółowy katalog przypadków użycia",
            "pdf_disclaimer_title": "Zastrzeżenie",
            "pdf_fallback_summary_p1": "Ten dokument opisuje {total_cases} wartościowych analitycznych przypadków użycia zidentyfikowanych dla {business_name}.",
            "pdf_fallback_summary_p2": "Kolejne strony zawierają szczegółowy podział tych możliwości, skategoryzowanych według domeny biznesowej.",
            "pptx_main_title": "Strategiczne przypadki użycia AI oparte na Databricks Genie Code",
            "pptx_for": "Dla",
            "pptx_disclaimer_title": "Zastrzeżenie",
            "pptx_domain_suffix": "Przypadki użycia",
            "elite_verdict": "Werdykt",
            "elite_decision": "Decyzja",
            "elite_decision_owner": "Właściciel decyzji",
            "elite_kpi": "KPI",
            "elite_value_driver": "Czynnik wartości",
            "elite_evidence_columns": "Kolumny dowodowe",
            "elite_join_path": "Ścieżka złączenia",
            "elite_build_path": "Ścieżka wdrożenia",
            "elite_red_team_note": "Uwaga red teamu",
            "elite_coverage_pick": "Wybór dla pokrycia domeny, poniżej progu",
            "elite_fallback_pick": "Poniżej progu {mode}",
            "elite_fallback_near_miss": "Blisko progu, poniżej progu {mode}",
            "doc_fallback_pick": "Poniżej progu",
            "doc_fallback_notice": "Żaden przypadek użycia nie osiągnął progu {mode}. To najbliżsi kandydaci, każdy oznaczony poniżej progu.",
            "elite_reason": "Powód",
            "untranslated_notice": "Wyświetlono po angielsku (tłumaczenie niedostępne)",
            "more_use_cases": "...i jeszcze {count} przypadków użycia.",
            "executive_summary_not_available": "Podsumowanie wykonawcze niedostępne.",
            "domain_summary_not_available": "Podsumowanie domeny niedostępne.",
            "summary_not_available": "Podsumowanie niedostępne.",
            "doc_catalog_kicker": "Katalog przypadków użycia AI",
            "doc_prepared_for": "Przygotowano dla",
            "doc_generated_on": "Data utworzenia",
            "doc_quality_mode": "Tryb jakości",
            "doc_use_cases": "Przypadki użycia",
            "doc_domains": "Domeny biznesowe",
            "doc_elite_use_cases": "Elitarne przypadki użycia",
            "doc_high_or_better": "Wysoka jakość lub lepsza",
            "doc_best_score": "Najlepszy wynik Inspire",
            "doc_not_scored": "Bez oceny",
            "doc_coverage_picks": "Wybory dla pokrycia",
            "doc_coverage_pick": "Wybór dla pokrycia",
            "value_correlation_analysis": "Analiza korelacji",
            "value_pareto_analysis": "Analiza Pareto",
            "value_funnel_analysis": "Analiza lejka",
            "value_document_processing": "Przetwarzanie dokumentów",
            "value_extraction": "Ekstrakcja",
            "value_ai_analysis": "Analiza AI",
            "value_simulation": "Symulacja",
            "value_geospatial_analysis": "Analiza geoprzestrzenna",
            "value_predictive_modeling": "Modelowanie predykcyjne",
            "value_clustering": "Klasteryzacja",
            "value_regression_analysis": "Analiza regresji",
            "value_survival_analysis": "Analiza przeżycia",
            "value_network_analysis": "Analiza sieci",
            "value_mode_good": "Dobry",
            "value_mode_high": "Wysoki",
            "value_mode_ultra_high": "Ultrawysoki",
            "doc_exec_kicker": "Szansa",
            "doc_guide_kicker": "Przewodnik",
            "doc_how_to_read": "Jak czytać ten katalog",
            "doc_guide_lead": "Każdy przypadek użycia pochodzi z własnych metadanych organizacji i jest oceniany na jednej, bezwzględnej skali. Używaj wyników do porównań, poziomów do filtrowania, a stron domen do nawigacji.",
            "doc_inspire_score_title": "Wynik Inspire, od 0 do 10",
            "doc_inspire_score_body": "Łączy jakość przypadku użycia z jego priorytetem biznesowym. Im wyżej, tym lepiej, a katalog jest według niego uporządkowany.",
            "doc_quality_tiers_title": "Poziomy jakości",
            "doc_quality_tiers_body": "Każdy przypadek użycia otrzymuje poziom od {lowest} do {highest}, który pokazuje, na ile jest konkretny, wartościowy i możliwy do zbudowania na dostępnych danych.",
            "doc_modes_title": "Tryby jakości",
            "doc_mode_good_body": "Zachowuje przypadki użycia ocenione jako {medium} lub lepiej.",
            "doc_mode_high_body": "Zachowuje tylko przypadki użycia ocenione jako {high} lub lepiej.",
            "doc_mode_ultra_body": "Zachowuje tylko przypadki użycia, które przekraczają bezwzględny próg wartości, dowodów, wykonalności i użyteczności w działaniu. Nie ma limitu.",
            "doc_this_catalog": "Ten katalog",
            "doc_elite_score_title": "Wynik elitarny",
            "doc_elite_score_body": "Używany w trybie {ultra}. Waży wartość biznesową, siłę dowodów, wykonalność i użyteczność w działaniu, a elitarny przypadek użycia uzyskuje w każdym z tych wymiarów co najmniej {min_score} na 10.",
            "doc_coverage_title": "Wybory dla pokrycia domen",
            "doc_coverage_body": "Gdy żądana domena biznesowa nie ma żadnego przypadku użycia powyżej progu, jej najsilniejszy kandydat zostaje zachowany i oznaczony jako wybór dla pokrycia poniżej progu.",
            "doc_portfolio_kicker": "Portfel",
            "doc_portfolio_glance": "Portfel w skrócie",
            "doc_per_domain": "Przypadki użycia według domen",
            "doc_quality_distribution": "Rozkład jakości",
            "doc_top_use_cases": "Najlepsze przypadki użycia",
            "doc_rank": "Pozycja",
            "doc_domain_label": "Domena",
            "doc_domain_overview": "O tej domenie",
            "doc_what_it_covers": "Co obejmuje",
            "doc_top_opportunities": "Najważniejsze szanse",
            "doc_domain_fallback": "Ta domena obejmuje przypadki użycia w obszarach: {subdomains} (łącznie: {count}). Najsilniejsze szanse wymieniono poniżej i opisano na kolejnych stronach.",
            "doc_open_notebook": "Otwórz notatnik Genie",
            "doc_elite_score": "Wynik elitarny",
            "doc_best_of_best": "Najlepsze z najlepszych",
            "doc_appendix": "Załącznik",
            "doc_methodology": "Metodyka",
            "strategy_path": "Ścieżka strategii",
            "kpis": "KPI",
            "expected_key_results": "Oczekiwane kluczowe rezultaty",
            "measures_of_success": "Miary sukcesu",
            "financial_benefits": "Korzyści finansowe",
            "non_financial_benefits": "Korzyści niefinansowe",
            "foundations": "Podstawy danych",
            "doc_foundations_title": "Wspólne podstawy danych",
            "doc_foundations_lead": "Tabele i złączenia potrzebne co najmniej dwóm przypadkom użycia. Zbudowanie ich w pierwszej kolejności umożliwia najwięcej przypadków użycia. Kolejność to sekwencja, a nie harmonogram.",
            "doc_foundation_tables": "Tabele",
            "doc_foundation_unlocks": "Umożliwione przypadki użycia",
            "doc_data_access_standard": "Odkrywanie standardowe: Inspire nie odczytał żadnych danych klienta. Użyto wyłącznie metadanych, definicji widoków metryk i liczników użycia.",
            "doc_data_access_deep": "Odkrywanie dogłębne: Inspire odczytał agregaty z tabel w zakresie (łącznie: {tables}): liczby wierszy, zakres dat, odsetki wartości pustych, liczby unikalnych wartości i krótkie listy wartości kolumn niezawierających danych osobowych. Do AI nie wysłano żadnego wiersza danych.",
            "doc_method_1_deep": "Inspire odczytuje metadane, a przy odkrywaniu dogłębnym także agregaty obliczone na tabelach. Nigdy nie wysyła wierszy danych do AI.",
            "doc_guide_lead_deep": "Każdy przypadek użycia pochodzi z metadanych i profili danych organizacji i jest oceniany na jednej bezwzględnej skali. Używaj ocen do porównań, poziomów do filtrowania, a stron domen do nawigacji.",
            "value_not_aligned": "Niedopasowany",
            "kpi_baseline": "wartość bazowa {value}, ostatnie {days} dni, zmierzono {date}",
            "kpi_baseline_undated": "wartość bazowa {value}, zmierzono {date}",
            "doc_method_1": "Inspire czyta wyłącznie metadane: nazwy tabel i kolumn, typy, komentarze, klucze i tagi. Nigdy nie czyta danych z wierszy.",
            "doc_method_2": "Ustala kontekst biznesowy, a następnie generuje kandydatów dla każdej domeny biznesowej, którą wspierają dane.",
            "doc_method_3": "Duplikaty są usuwane, każdy kandydat jest oceniany pod kątem wartości i wykonalności, a bramki jakości odrzucają słabe lub nieuzasadnione pomysły.",
            "doc_method_4": "W trybie {ultra} elitarny sędzia przegląda każdego pozostałego kandydata, a deterministyczne kontrole potwierdzają, że każda wskazana tabela i kolumna istnieje.",
            "doc_method_5": "Ostateczna lista jest uszeregowana, pogrupowana według domen biznesowych i, jeśli wybrano tę opcję, uzupełniona notatnikiem Databricks Genie Code.",
            "doc_glossary": "Słowniczek",
            "doc_gl_domain": "Obszar biznesowy grupujący powiązane przypadki użycia.",
            "doc_gl_subdomain": "Węższy temat w ramach domeny biznesowej.",
            "doc_gl_technique": "Metoda analityczna, na przykład prognozowanie lub klasyfikacja.",
            "doc_gl_inspire": "Wynik od 0 do 10 łączący jakość i priorytet biznesowy.",
            "doc_gl_quality": "Poziom od {lowest} do {highest} nadawany każdemu przypadkowi użycia.",
            "doc_gl_elite": "Wynik trybu {ultra} dla wartości, dowodów, wykonalności i użyteczności w działaniu.",
            "doc_gl_coverage": "Przypadek użycia poniżej progu zachowany po to, by żądana domena była reprezentowana.",
            "doc_gl_notebook": "Notatnik Databricks z instrukcjami Genie Code do zbudowania przypadku użycia.",
            "doc_notebook_term": "Notatnik Genie",
            "doc_next_steps": "Kolejne kroki",
            "doc_closing_title": "Od przypadków użycia do wyników",
            "doc_closing_body": "Wybierz najważniejsze przypadki użycia, otwórz ich notatniki Genie i zbuduj pierwszy wspólnie z zespołem. Kroki to sugerowana kolejność; tempo ustala zespół.",
            "doc_step_select": "Wybierz",
            "doc_step_select_body": "Wybierz przypadki użycia z najwyższym wynikiem i jasnym sponsorem.",
            "doc_step_validate": "Zweryfikuj dane",
            "doc_step_validate_body": "Potwierdź używane tabele i ich złączenia z właścicielami danych.",
            "doc_step_build": "Zbuduj z Genie Code",
            "doc_step_build_body": "Otwórz notatnik Genie i wygeneruj pierwszą działającą wersję.",
            "doc_step_schedule": "Zaplanuj",
            "doc_step_schedule_body": "Uruchom notatnik jako zadanie Databricks, które odświeża tabelę wyników.",
            "doc_step_measure": "Mierz i skaluj",
            "doc_step_measure_body": "Śledź uzyskiwaną wartość biznesową, a następnie rozszerz działania na kolejny przypadek użycia.",
            "doc_agenda": "Agenda",
            "doc_roadmap": "Plan działania i kolejne kroki",
            "doc_use_case_index": "Indeks przypadków użycia",
            "doc_continued": "ciąg dalszy",
            "doc_not_provided": "Nie podano",
            "doc_bob_score": "Wynik Best-of-Best",
            "elite_cadence": "Częstotliwość decyzji",
            "value_cadence_hourly": "Co godzinę",
            "value_cadence_daily": "Codziennie",
            "value_cadence_weekly": "Co tydzień",
            "value_cadence_biweekly": "Co dwa tygodnie",
            "value_cadence_monthly": "Co miesiąc",
            "value_cadence_quarterly": "Co kwartał",
            "value_cadence_yearly": "Co rok",
            "value_cadence_real_time": "W czasie rzeczywistym",
            "value_cadence_per_event": "Przy każdym zdarzeniu",
            "doc_generated_by": "Wygenerowano przez {brand} {version} · {date}",
            "doc_disclaimer_ai": "Przypadki użycia, wyniki i opisy w tym dokumencie zostały wygenerowane przez AI. Zweryfikuj je z właścicielami danych i sponsorami biznesowymi przed podjęciem decyzji.",
            "doc_disclaimer_figures": "Wyniki, wartości i efekty są szacunkami służącymi wyłącznie do ustalania priorytetów. Żadna liczba w tym dokumencie nie jest gwarantowana.",
            "doc_contents": "Spis treści",
            "doc_index_lead": "Wszystkie przypadki użycia ({count}) w kolejności rankingu końcowego.",
            "doc_deck_scope": "Ta prezentacja opisuje najwyżej ocenione przypadki użycia ({count} z {total}). Katalog PDF opisuje wszystkie.",
            "doc_deck_scope_all": "Ta prezentacja opisuje wszystkie przypadki użycia ({total}), każdy na osobnej karcie.",
            "value_general_improvement": "Ogólne ulepszenie",
            "value_reduce_cost": "Redukcja kosztów",
            "value_increase_revenue": "Zwiększenie przychodów",
            "value_boost_productivity": "Zwiększenie produktywności",
            "value_mitigate_risk": "Ograniczenie ryzyka",
            "value_protect_revenue": "Ochrona przychodów",
            "value_align_to_regulations": "Zgodność z przepisami",
            "value_improve_customer_experience": "Poprawa doświadczenia klienta",
            "value_enable_data_driven_decisions": "Wsparcie decyzji opartych na danych",
            "value_optimize_operations": "Optymalizacja operacji",
            "value_empower_talent": "Wzmocnienie talentów",
            "value_enhance_experience": "Poprawa doświadczenia",
            "value_drive_innovation": "Napędzanie innowacji",
            "value_achieve_esg": "Osiągnięcie ESG",
            "value_execute_strategy": "Realizacja strategii",
            "value_forecasting": "Prognozowanie",
            "value_classification": "Klasyfikacja",
            "value_anomaly_detection": "Wykrywanie anomalii",
            "value_cohort_analysis": "Analiza kohortowa",
            "value_segmentation": "Segmentacja",
            "value_sentiment_analysis": "Analiza sentymentu",
            "value_trend_analysis": "Analiza trendów",
            "value_prescriptive_analytics": "Analityka preskrypcyjna",
            "value_root_cause_analysis": "Analiza przyczyn źródłowych",
            "value_optimization": "Optymalizacja",
            "value_recommendation": "Rekomendacja",
            "value_time_series_analysis": "Analiza szeregów czasowych",
            "value_predictive_analytics": "Analityka predykcyjna",
            "value_descriptive_analytics": "Analityka opisowa",
        },
        "Romanian": {
            "main_title": "Generator de cazuri de utilizare alimentat de Databricks Genie Code",
            "intro": "Acest notebook conține cazuri de utilizare generate de AI pe baza schemelor dvs. Mai jos este un rezumat al scenariilor generate pe domeniu de afaceri.",
            "domain": "Domeniu de afaceri",
            "total": "Total cazuri de utilizare",
            "summaries": "Rezumate ale cazurilor de utilizare",
            "sum_id": "ID",
            "sum_name": "Nume",
            "sum_value": "Valoare de afaceri",
            "sum_outcome": "Rezultat așteptat",
            "warning_header": "AVERTISMENT",
            "warning_body": "Nu rulați acest notebook. Este destinat exclusiv pentru demonstrație și catalogare.",
            "disclaimer": "Acest conținut este generat de AI și trebuie verificat de un inginer calificat înainte de utilizarea în producție. Databricks nu este responsabil pentru nicio problemă rezultată din utilizarea acestui conținut.",
            "detailed_scenarios": "Detalii cazuri de utilizare",
            "aspect": "Aspect",
            "description": "Descriere",
            "aspect_domain": "Domeniu de afaceri",
            "type": "Tip",
            "analytics_technique": "Tehnică analitică",
            "primary_table": "Tabel principal",
            "quality": "Calitate",
            "priority": "Prioritate",
            "usecase": "Caz de utilizare",
            "description_label": "Descriere",
            "value_type_problem": "Problemă",
            "value_type_risk": "Risc",
            "value_type_opportunity": "Oportunitate",
            "value_type_improvement": "Îmbunătățire",
            "value_priority_ultra_high": "Extrem de ridicat",
            "value_priority_very_high": "Foarte ridicat",
            "value_priority_high": "Ridicat",
            "value_priority_medium": "Mediu",
            "value_priority_low": "Scăzut",
            "value_priority_very_low": "Foarte scăzut",
            "value_priority_ultra_low": "Extrem de scăzut",
            "statement": "Declarație",
            "solution": "Soluție",
            "aspect_beneficiary": "Beneficiar",
            "beneficiary": "Beneficiar",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Aliniere la prioritățile de afaceri",
            "strategic_goals_alignment": "Aliniere la obiectivele strategice",
            "subdomain": "Subdomeniu",
            "aspect_value": "Valoare de afaceri",
            "business_value": "Valoare de afaceri",
            "aspect_tables": "Tabele implicate",
            "quality_reasons": "Motive de calitate",
            "aspect_ai_function": "Funcție AI",
            "aspect_analytics_technique": "Tehnică analitică",
            "aspect_primary_table": "Tabel principal",
            "aspect_quality": "Calitate",
            "aspect_priority": "Prioritate",
            "strategic_alignment": "Aliniere strategică",
            "return_on_investment": "Randamentul investiției",
            "reusability": "Reutilizabilitate",
            "time_to_value": "Timp până la valoare",
            "data_availability": "Disponibilitatea datelor",
            "data_accessibility": "Accesibilitatea datelor",
            "architecture_fitness": "Compatibilitate arhitecturală",
            "team_skills": "Competențe echipă",
            "domain_knowledge": "Cunoștințe de domeniu",
            "people_allocation": "Alocare personal",
            "budget_allocation": "Alocare buget",
            "time_to_production": "Timp până la producție",
            "value_score": "Scor valoare",
            "feasibility_score": "Scor fezabilitate",
            "priority_score": "Scor prioritate",
            "inspire_score": "Scor Inspire",
            "quality_score_header": "Scor calitate",
            "pdf_title": "Cazuri de utilizare AI strategice alimentate de Databricks Genie Code",
            "pdf_for": "Pentru",
            "pdf_exec_summary": "Rezumat executiv",
            "pdf_toc_title": "Domenii cazuri de utilizare",
            "pdf_detailed_view": "Catalog detaliat de cazuri de utilizare",
            "pdf_disclaimer_title": "Declinare a responsabilității",
            "pdf_fallback_summary_p1": "Acest document descrie {total_cases} cazuri de utilizare analitice de mare valoare identificate pentru {business_name}.",
            "pdf_fallback_summary_p2": "Paginile următoare oferă o defalcare detaliată a acestor oportunități, clasificate pe domenii de afaceri.",
            "pptx_main_title": "Cazuri de utilizare AI strategice alimentate de Databricks Genie Code",
            "pptx_for": "Pentru",
            "pptx_disclaimer_title": "Declinare a responsabilității",
            "pptx_domain_suffix": "Cazuri de utilizare",
            "elite_verdict": "Verdict",
            "elite_decision": "Decizie",
            "elite_decision_owner": "Responsabilul deciziei",
            "elite_kpi": "KPI",
            "elite_value_driver": "Factor de valoare",
            "elite_evidence_columns": "Coloane de dovezi",
            "elite_join_path": "Cale de joncțiune",
            "elite_build_path": "Cale de implementare",
            "elite_red_team_note": "Notă red team",
            "elite_coverage_pick": "Selecție pentru acoperirea domeniului, sub prag",
            "elite_fallback_pick": "Sub pragul {mode}",
            "elite_fallback_near_miss": "Aproape selectat, sub pragul {mode}",
            "doc_fallback_pick": "Sub prag",
            "doc_fallback_notice": "Niciun caz de utilizare nu a atins pragul {mode}. Aceștia sunt cei mai apropiați candidați, fiecare marcat sub prag.",
            "elite_reason": "Motiv",
            "untranslated_notice": "Afișat în engleză (traducere indisponibilă)",
            "more_use_cases": "...și încă {count} cazuri de utilizare.",
            "executive_summary_not_available": "Rezumat executiv indisponibil.",
            "domain_summary_not_available": "Rezumat domeniu indisponibil.",
            "summary_not_available": "Rezumat indisponibil.",
            "doc_catalog_kicker": "Catalog de cazuri de utilizare AI",
            "doc_prepared_for": "Pregătit pentru",
            "doc_generated_on": "Generat la",
            "doc_quality_mode": "Mod de calitate",
            "doc_use_cases": "Cazuri de utilizare",
            "doc_domains": "Domenii de afaceri",
            "doc_elite_use_cases": "Cazuri de utilizare de elită",
            "doc_high_or_better": "Calitate ridicată sau mai bună",
            "doc_best_score": "Cel mai bun scor Inspire",
            "doc_not_scored": "Fără scor",
            "doc_coverage_picks": "Selecții de acoperire",
            "doc_coverage_pick": "Selecție de acoperire",
            "value_correlation_analysis": "Analiză de corelație",
            "value_pareto_analysis": "Analiză Pareto",
            "value_funnel_analysis": "Analiza pâlniei",
            "value_document_processing": "Procesarea documentelor",
            "value_extraction": "Extragere",
            "value_ai_analysis": "Analiză AI",
            "value_simulation": "Simulare",
            "value_geospatial_analysis": "Analiză geospațială",
            "value_predictive_modeling": "Modelare predictivă",
            "value_clustering": "Grupare",
            "value_regression_analysis": "Analiză de regresie",
            "value_survival_analysis": "Analiză de supraviețuire",
            "value_network_analysis": "Analiză de rețea",
            "value_mode_good": "Bun",
            "value_mode_high": "Ridicat",
            "value_mode_ultra_high": "Ultra ridicat",
            "doc_exec_kicker": "Oportunitatea",
            "doc_guide_kicker": "Ghid de lectură",
            "doc_how_to_read": "Cum se citește acest catalog",
            "doc_guide_lead": "Fiecare caz de utilizare provine din propriile metadate ale organizației și primește un scor pe o singură scală absolută. Folosiți scorurile pentru comparații, nivelurile pentru filtrare și paginile de domeniu pentru navigare.",
            "doc_inspire_score_title": "Scor Inspire, de la 0 la 10",
            "doc_inspire_score_body": "Combină calitatea unui caz de utilizare cu prioritatea sa de afaceri. Cu cât este mai mare, cu atât mai bine, iar catalogul este ordonat după el.",
            "doc_quality_tiers_title": "Niveluri de calitate",
            "doc_quality_tiers_body": "Fiecare caz de utilizare primește un nivel de la {lowest} la {highest}, care arată cât de specific, valoros și realizabil este cu datele disponibile.",
            "doc_modes_title": "Moduri de calitate",
            "doc_mode_good_body": "Păstrează cazurile de utilizare evaluate {medium} sau mai bine.",
            "doc_mode_high_body": "Păstrează doar cazurile de utilizare evaluate {high} sau mai bine.",
            "doc_mode_ultra_body": "Păstrează doar cazurile de utilizare care depășesc un prag absolut de valoare, dovezi, implementabilitate și aplicabilitate. Nu există cote.",
            "doc_this_catalog": "Acest catalog",
            "doc_elite_score_title": "Scor de elită",
            "doc_elite_score_body": "Folosit în modul {ultra}. Cântărește valoarea de afaceri, puterea dovezilor, implementabilitatea și aplicabilitatea, iar un caz de utilizare de elită obține cel puțin {min_score} din 10 la fiecare.",
            "doc_coverage_title": "Selecții pentru acoperirea domeniilor",
            "doc_coverage_body": "Când un domeniu de afaceri solicitat nu are niciun caz de utilizare peste prag, cel mai puternic candidat al său este păstrat și marcat ca selecție de acoperire sub prag.",
            "doc_portfolio_kicker": "Portofoliu",
            "doc_portfolio_glance": "Portofoliul pe scurt",
            "doc_per_domain": "Cazuri de utilizare pe domeniu",
            "doc_quality_distribution": "Distribuția calității",
            "doc_top_use_cases": "Principalele cazuri de utilizare",
            "doc_rank": "Poziție",
            "doc_domain_label": "Domeniu",
            "doc_domain_overview": "Despre acest domeniu",
            "doc_what_it_covers": "Ce acoperă",
            "doc_top_opportunities": "Principalele oportunități",
            "doc_domain_fallback": "Acest domeniu reunește cazuri de utilizare din {subdomains}, în total {count}. Cele mai puternice oportunități sunt enumerate mai jos și detaliate în paginile următoare.",
            "doc_open_notebook": "Deschide notebook-ul Genie",
            "doc_elite_score": "Scor de elită",
            "doc_best_of_best": "Cel mai bun dintre cei mai buni",
            "doc_appendix": "Anexă",
            "doc_methodology": "Metodologie",
            "strategy_path": "Calea strategică",
            "kpis": "KPI-uri",
            "expected_key_results": "Rezultate cheie așteptate",
            "measures_of_success": "Măsuri ale succesului",
            "financial_benefits": "Beneficii financiare",
            "non_financial_benefits": "Beneficii nefinanciare",
            "foundations": "Fundamente de date",
            "doc_foundations_title": "Fundamente de date comune",
            "doc_foundations_lead": "Tabele și asocieri de care au nevoie două sau mai multe cazuri de utilizare. Construirea lor mai întâi deblochează cele mai multe cazuri de utilizare. Ordinea este o succesiune, nu un calendar.",
            "doc_foundation_tables": "Tabele",
            "doc_foundation_unlocks": "Cazuri de utilizare deblocate",
            "doc_data_access_standard": "Descoperire standard: Inspire nu a citit date ale clientului. A folosit doar metadate, definiții ale vizualizărilor de metrici și numărători de utilizare.",
            "doc_data_access_deep": "Descoperire aprofundată: Inspire a citit agregate din tabelele incluse, în total {tables} (numere de rânduri, acoperirea datelor, ponderi ale valorilor nule, numere de valori distincte și liste scurte de valori ale coloanelor fără date personale). Niciun rând de date nu a fost trimis către AI.",
            "doc_method_1_deep": "Inspire citește metadatele și, în descoperirea aprofundată, și agregate calculate pe tabele. Nu trimite niciodată rânduri de date către AI.",
            "doc_guide_lead_deep": "Fiecare caz de utilizare provine din metadatele și profilurile de date ale organizației și este punctat pe o singură scală absolută. Folosiți scorurile pentru comparare, nivelurile pentru filtrare și paginile de domeniu pentru navigare.",
            "value_not_aligned": "Nealiniat",
            "kpi_baseline": "valoare de referință {value}, ultimele {days} zile, măsurată pe {date}",
            "kpi_baseline_undated": "valoare de referință {value}, măsurată pe {date}",
            "doc_method_1": "Inspire citește doar metadate: nume de tabele și coloane, tipuri, comentarii, chei și etichete. Nu citește niciodată datele din rânduri.",
            "doc_method_2": "Deduce contextul de afaceri, apoi generează cazuri de utilizare candidate pentru fiecare domeniu de afaceri susținut de date.",
            "doc_method_3": "Duplicatele sunt eliminate, fiecare candidat primește un scor pentru valoare și fezabilitate, iar porțile de calitate elimină ideile slabe sau nesusținute.",
            "doc_method_4": "În modul {ultra}, un evaluator de elită analizează fiecare candidat rămas, iar verificări deterministe confirmă că fiecare tabel și coloană citată există.",
            "doc_method_5": "Lista finală este ordonată, grupată pe domenii de afaceri și, când este selectat, însoțită de un notebook Databricks Genie Code.",
            "doc_glossary": "Glosar",
            "doc_gl_domain": "O zonă de afaceri care grupează cazuri de utilizare înrudite.",
            "doc_gl_subdomain": "O temă mai restrânsă în cadrul unui domeniu de afaceri.",
            "doc_gl_technique": "Metoda analitică, de exemplu prognoza sau clasificarea.",
            "doc_gl_inspire": "Scorul de la 0 la 10 care combină calitatea și prioritatea de afaceri.",
            "doc_gl_quality": "Nivelul de la {lowest} la {highest} atribuit fiecărui caz de utilizare.",
            "doc_gl_elite": "Scorul modului {ultra} pentru valoare, dovezi, implementabilitate și aplicabilitate.",
            "doc_gl_coverage": "Un caz de utilizare sub prag, păstrat pentru ca un domeniu solicitat să fie reprezentat.",
            "doc_gl_notebook": "Un notebook Databricks cu instrucțiuni Genie Code pentru construirea cazului de utilizare.",
            "doc_notebook_term": "Notebook Genie",
            "doc_next_steps": "Pașii următori",
            "doc_closing_title": "De la cazuri de utilizare la rezultate",
            "doc_closing_body": "Alegeți cazurile de utilizare cele mai importante, deschideți notebook-urile lor Genie și construiți primul împreună cu echipa. Pașii sunt o ordine sugerată; ritmul îl stabilește echipa.",
            "doc_step_select": "Selectați",
            "doc_step_select_body": "Alegeți cazurile de utilizare cu cel mai bun scor și un sponsor clar.",
            "doc_step_validate": "Validați datele",
            "doc_step_validate_body": "Confirmați tabelele implicate și legăturile dintre ele cu proprietarii datelor.",
            "doc_step_build": "Construiți cu Genie Code",
            "doc_step_build_body": "Deschideți notebook-ul Genie și generați prima versiune funcțională.",
            "doc_step_schedule": "Programați",
            "doc_step_schedule_body": "Rulați notebook-ul ca job Databricks care actualizează tabelul de rezultate.",
            "doc_step_measure": "Măsurați și extindeți",
            "doc_step_measure_body": "Urmăriți valoarea de business obținută, apoi extindeți la următorul caz de utilizare.",
            "doc_agenda": "Agendă",
            "doc_roadmap": "Foaie de parcurs și pașii următori",
            "doc_use_case_index": "Index al cazurilor de utilizare",
            "doc_continued": "continuare",
            "doc_not_provided": "Nespecificat",
            "doc_bob_score": "Scor Best-of-Best",
            "elite_cadence": "Frecvența deciziei",
            "value_cadence_hourly": "Din oră în oră",
            "value_cadence_daily": "Zilnic",
            "value_cadence_weekly": "Săptămânal",
            "value_cadence_biweekly": "La două săptămâni",
            "value_cadence_monthly": "Lunar",
            "value_cadence_quarterly": "Trimestrial",
            "value_cadence_yearly": "Anual",
            "value_cadence_real_time": "În timp real",
            "value_cadence_per_event": "La fiecare eveniment",
            "doc_generated_by": "Generat de {brand} {version} · {date}",
            "doc_disclaimer_ai": "Cazurile de utilizare, scorurile și textele din acest document sunt generate de AI. Validați-le cu proprietarii datelor și cu sponsorii de business înainte de orice decizie.",
            "doc_disclaimer_figures": "Scorurile, valorile și rezultatele sunt estimări folosite doar pentru prioritizare. Nicio cifră din acest document nu este garantată.",
            "doc_contents": "Cuprins",
            "doc_index_lead": "Toate cazurile de utilizare în ordinea clasamentului final, în total {count}.",
            "doc_deck_scope": "Această prezentare detaliază cazurile de utilizare cel mai bine clasate, {count} din {total}. Catalogul PDF le detaliază pe toate.",
            "doc_deck_scope_all": "Această prezentare detaliază toate cazurile de utilizare, în total {total}, câte un card pentru fiecare.",
            "value_general_improvement": "Îmbunătățire generală",
            "value_reduce_cost": "Reducerea costurilor",
            "value_increase_revenue": "Creșterea veniturilor",
            "value_boost_productivity": "Creșterea productivității",
            "value_mitigate_risk": "Reducerea riscurilor",
            "value_protect_revenue": "Protejarea veniturilor",
            "value_align_to_regulations": "Conformitate cu reglementările",
            "value_improve_customer_experience": "Îmbunătățirea experienței clientului",
            "value_enable_data_driven_decisions": "Facilitarea deciziilor bazate pe date",
            "value_optimize_operations": "Optimizarea operațiunilor",
            "value_empower_talent": "Consolidarea talentelor",
            "value_enhance_experience": "Îmbunătățirea experienței",
            "value_drive_innovation": "Stimularea inovației",
            "value_achieve_esg": "Atingerea ESG",
            "value_execute_strategy": "Executarea strategiei",
            "value_forecasting": "Prognoză",
            "value_classification": "Clasificare",
            "value_anomaly_detection": "Detectarea anomaliilor",
            "value_cohort_analysis": "Analiză de cohortă",
            "value_segmentation": "Segmentare",
            "value_sentiment_analysis": "Analiza sentimentelor",
            "value_trend_analysis": "Analiza tendințelor",
            "value_prescriptive_analytics": "Analiză prescriptivă",
            "value_root_cause_analysis": "Analiza cauzelor profunde",
            "value_optimization": "Optimizare",
            "value_recommendation": "Recomandare",
            "value_time_series_analysis": "Analiza seriilor temporale",
            "value_predictive_analytics": "Analiză predictivă",
            "value_descriptive_analytics": "Analiză descriptivă",
        },
        "Ukrainian": {
            "main_title": "Генератор бізнес-кейсів на базі Датабрікс Джіні Код",
            "intro": "Цей ноутбук містить бізнес-кейси, створені ШІ на основі ваших схем. Нижче наведено резюме створених сценаріїв за бізнес-доменами.",
            "domain": "Бізнес-домен",
            "total": "Загальна кількість бізнес-кейсів",
            "summaries": "Резюме бізнес-кейсів",
            "sum_id": "ID",
            "sum_name": "Назва",
            "sum_value": "Бізнес-цінність",
            "sum_outcome": "Очікуваний результат",
            "warning_header": "ПОПЕРЕДЖЕННЯ",
            "warning_body": "Не запускайте цей ноутбук. Він призначений лише для демонстрації та каталогізації.",
            "disclaimer": "Цей контент створений ШІ і повинен бути перевірений кваліфікованим інженером перед використанням у виробничому середовищі. Databricks не несе відповідальності за будь-які проблеми, що виникають внаслідок використання цього контенту.",
            "detailed_scenarios": "Деталі бізнес-кейсів",
            "aspect": "Аспект",
            "description": "Опис",
            "aspect_domain": "Бізнес-домен",
            "type": "Тип",
            "analytics_technique": "Аналітична техніка",
            "primary_table": "Основна таблиця",
            "quality": "Якість",
            "priority": "Пріоритет",
            "usecase": "Бізнес-кейс",
            "description_label": "Опис",
            "value_type_problem": "Проблема",
            "value_type_risk": "Ризик",
            "value_type_opportunity": "Можливість",
            "value_type_improvement": "Покращення",
            "value_priority_ultra_high": "Надзвичайно високий",
            "value_priority_very_high": "Дуже високий",
            "value_priority_high": "Високий",
            "value_priority_medium": "Середній",
            "value_priority_low": "Низький",
            "value_priority_very_low": "Дуже низький",
            "value_priority_ultra_low": "Надзвичайно низький",
            "statement": "Опис ситуації",
            "solution": "Рішення",
            "aspect_beneficiary": "Вигодонабувач",
            "beneficiary": "Вигодонабувач",
            "aspect_sponsor": "Спонсор",
            "sponsor": "Спонсор",
            "business_priority_alignment": "Відповідність бізнес-пріоритетам",
            "strategic_goals_alignment": "Відповідність стратегічним цілям",
            "subdomain": "Піддомен",
            "aspect_value": "Бізнес-цінність",
            "business_value": "Бізнес-цінність",
            "aspect_tables": "Пов'язані таблиці",
            "quality_reasons": "Причини якості",
            "aspect_ai_function": "Функція ШІ",
            "aspect_analytics_technique": "Аналітична техніка",
            "aspect_primary_table": "Основна таблиця",
            "aspect_quality": "Якість",
            "aspect_priority": "Пріоритет",
            "strategic_alignment": "Стратегічна відповідність",
            "return_on_investment": "Рентабельність інвестицій",
            "reusability": "Можливість повторного використання",
            "time_to_value": "Час до отримання цінності",
            "data_availability": "Доступність даних",
            "data_accessibility": "Доступ до даних",
            "architecture_fitness": "Відповідність архітектурі",
            "team_skills": "Навички команди",
            "domain_knowledge": "Знання предметної області",
            "people_allocation": "Розподіл персоналу",
            "budget_allocation": "Розподіл бюджету",
            "time_to_production": "Час до впровадження",
            "value_score": "Оцінка цінності",
            "feasibility_score": "Оцінка реалізованості",
            "priority_score": "Оцінка пріоритету",
            "inspire_score": "Оцінка Inspire",
            "quality_score_header": "Оцінка якості",
            "pdf_title": "Стратегічні ШІ бізнес-кейси на базі Датабрікс Джіні Код",
            "pdf_for": "Для",
            "pdf_exec_summary": "Резюме для керівництва",
            "pdf_toc_title": "Домени бізнес-кейсів",
            "pdf_detailed_view": "Детальний каталог бізнес-кейсів",
            "pdf_disclaimer_title": "Відмова від відповідальності",
            "pdf_fallback_summary_p1": "Цей документ описує {total_cases} високоцінних аналітичних бізнес-кейсів, виявлених для {business_name}.",
            "pdf_fallback_summary_p2": "На наступних сторінках представлений детальний аналіз цих можливостей, класифікованих за бізнес-доменами.",
            "pptx_main_title": "Стратегічні ШІ бізнес-кейси на базі Датабрікс Джіні Код",
            "pptx_for": "Для",
            "pptx_disclaimer_title": "Відмова від відповідальності",
            "pptx_domain_suffix": "Бізнес-кейси",
            "elite_verdict": "Вердикт",
            "elite_decision": "Рішення",
            "elite_decision_owner": "Власник рішення",
            "elite_kpi": "Ключовий показник (KPI)",
            "elite_value_driver": "Драйвер цінності",
            "elite_evidence_columns": "Стовпці-докази",
            "elite_join_path": "Шлях з'єднання",
            "elite_build_path": "Шлях реалізації",
            "elite_red_team_note": "Зауваження червоної команди",
            "elite_coverage_pick": "Вибір для покриття домену, нижче порогу",
            "elite_fallback_pick": "Нижче порогу {mode}",
            "elite_fallback_near_miss": "Майже пройшов, нижче порогу {mode}",
            "doc_fallback_pick": "Нижче порогу",
            "doc_fallback_notice": "Жоден сценарій не досяг порогу {mode}. Це найближчі кандидати, кожен позначений як нижче порогу.",
            "elite_reason": "Причина",
            "untranslated_notice": "Показано англійською (переклад недоступний)",
            "more_use_cases": "...і ще {count} сценаріїв використання.",
            "executive_summary_not_available": "Резюме для керівництва недоступне.",
            "domain_summary_not_available": "Резюме домену недоступне.",
            "summary_not_available": "Резюме недоступне.",
            "doc_catalog_kicker": "Каталог ШІ бізнес-кейсів",
            "doc_prepared_for": "Підготовлено для",
            "doc_generated_on": "Дата створення",
            "doc_quality_mode": "Режим якості",
            "doc_use_cases": "Бізнес-кейси",
            "doc_domains": "Бізнес-домени",
            "doc_elite_use_cases": "Елітні бізнес-кейси",
            "doc_high_or_better": "Висока якість і вище",
            "doc_best_score": "Найкраща оцінка Inspire",
            "doc_not_scored": "Без оцінки",
            "doc_coverage_picks": "Вибори для покриття",
            "doc_coverage_pick": "Вибір для покриття",
            "value_correlation_analysis": "Кореляційний Аналіз",
            "value_pareto_analysis": "Аналіз Парето",
            "value_funnel_analysis": "Аналіз Воронки",
            "value_document_processing": "Обробка Документів",
            "value_extraction": "Вилучення Даних",
            "value_ai_analysis": "ШІ-Аналіз",
            "value_simulation": "Моделювання",
            "value_geospatial_analysis": "Геопросторовий Аналіз",
            "value_predictive_modeling": "Предиктивне Моделювання",
            "value_clustering": "Кластеризація",
            "value_regression_analysis": "Регресійний Аналіз",
            "value_survival_analysis": "Аналіз Виживаності",
            "value_network_analysis": "Мережевий Аналіз",
            "value_mode_good": "Добрий",
            "value_mode_high": "Високий",
            "value_mode_ultra_high": "Надвисокий",
            "doc_exec_kicker": "Можливість",
            "doc_guide_kicker": "Як читати",
            "doc_how_to_read": "Як читати цей каталог",
            "doc_guide_lead": "Кожен бізнес-кейс походить із власних метаданих організації та оцінюється за єдиною абсолютною шкалою. Використовуйте оцінки для порівняння, рівні для фільтрації, а сторінки доменів для навігації.",
            "doc_inspire_score_title": "Оцінка Inspire, від 0 до 10",
            "doc_inspire_score_body": "Поєднує якість бізнес-кейсу з його бізнес-пріоритетом. Що вище, то краще; каталог упорядковано за цією оцінкою.",
            "doc_quality_tiers_title": "Рівні якості",
            "doc_quality_tiers_body": "Кожен бізнес-кейс отримує рівень від «{lowest}» до «{highest}», який показує, наскільки він конкретний, цінний і здійсненний на доступних даних.",
            "doc_modes_title": "Режими якості",
            "doc_mode_good_body": "Зберігає бізнес-кейси з оцінкою «{medium}» і вище.",
            "doc_mode_high_body": "Зберігає лише бізнес-кейси з оцінкою «{high}» і вище.",
            "doc_mode_ultra_body": "Зберігає лише бізнес-кейси, що перевищують абсолютну планку за цінністю, доказовістю, реалізовністю та застосовністю. Квот немає.",
            "doc_this_catalog": "Цей каталог",
            "doc_elite_score_title": "Елітна оцінка",
            "doc_elite_score_body": "Використовується в режимі «{ultra}». Враховує бізнес-цінність, силу доказів, реалізовність і застосовність; елітний бізнес-кейс отримує не менше {min_score} з 10 за кожним критерієм.",
            "doc_coverage_title": "Вибори для покриття доменів",
            "doc_coverage_body": "Якщо в запитаному бізнес-домені немає бізнес-кейсу вище планки, зберігається його найсильніший кандидат із позначкою «вибір для покриття, нижче планки».",
            "doc_portfolio_kicker": "Портфель",
            "doc_portfolio_glance": "Портфель у цифрах",
            "doc_per_domain": "Бізнес-кейси за доменами",
            "doc_quality_distribution": "Розподіл за якістю",
            "doc_top_use_cases": "Найкращі бізнес-кейси",
            "doc_rank": "Місце",
            "doc_domain_label": "Домен",
            "doc_domain_overview": "Про цей домен",
            "doc_what_it_covers": "Що охоплює",
            "doc_top_opportunities": "Головні можливості",
            "doc_domain_fallback": "Цей домен об'єднує бізнес-кейси у сферах: {subdomains} (усього: {count}). Найсильніші можливості наведено нижче та докладно описано на наступних сторінках.",
            "doc_open_notebook": "Відкрити блокнот Джіні",
            "doc_elite_score": "Елітна оцінка",
            "doc_best_of_best": "Найкраще з найкращого",
            "doc_appendix": "Додаток",
            "doc_methodology": "Методологія",
            "strategy_path": "Стратегічний шлях",
            "kpis": "KPI",
            "expected_key_results": "Очікувані ключові результати",
            "measures_of_success": "Показники успіху",
            "financial_benefits": "Фінансові вигоди",
            "non_financial_benefits": "Нефінансові вигоди",
            "foundations": "Основи даних",
            "doc_foundations_title": "Спільні основи даних",
            "doc_foundations_lead": "Таблиці та з'єднання, потрібні двом і більше бізнес-кейсам. Якщо побудувати їх першими, стане доступно найбільше бізнес-кейсів. Порядок є послідовністю, а не графіком.",
            "doc_foundation_tables": "Таблиці",
            "doc_foundation_unlocks": "Доступні бізнес-кейси",
            "doc_data_access_standard": "Стандартне виявлення: Inspire не читав даних клієнта. Використано лише метадані, визначення подань метрик і лічильники використання.",
            "doc_data_access_deep": "Глибоке виявлення: Inspire прочитав агрегати з таблиць в області аналізу (усього: {tables}): кількість рядків, охоплення дат, частку порожніх значень, кількість унікальних значень і короткі списки значень неперсональних стовпців. Жоден рядок даних не надіслано ШІ.",
            "doc_method_1_deep": "Inspire читає метадані, а під час глибокого виявлення також агрегати, обчислені за таблицями. Він ніколи не надсилає рядки даних ШІ.",
            "doc_guide_lead_deep": "Кожен бізнес-кейс походить із метаданих і профілів даних організації та оцінюється за єдиною абсолютною шкалою. Використовуйте оцінки для порівняння, рівні для фільтрації, а сторінки доменів для навігації.",
            "value_not_aligned": "Не узгоджено",
            "kpi_baseline": "базове значення {value}, останні {days} дн., виміряно {date}",
            "kpi_baseline_undated": "базове значення {value}, виміряно {date}",
            "doc_method_1": "Inspire читає лише метадані: назви таблиць і стовпців, типи, коментарі, ключі та теги. Дані рядків він ніколи не читає.",
            "doc_method_2": "Він визначає бізнес-контекст, а потім формує кандидатів для кожного бізнес-домену, який підтримують дані.",
            "doc_method_3": "Дублікати видаляються, кожного кандидата оцінюють за цінністю та здійсненністю, а фільтри якості відсіюють слабкі або необґрунтовані ідеї.",
            "doc_method_4": "У режимі «{ultra}» елітний експерт перевіряє кожного кандидата, що залишився, а детерміновані перевірки підтверджують існування кожної згаданої таблиці та стовпця.",
            "doc_method_5": "Підсумковий список ранжується, групується за бізнес-доменами і за вибору доповнюється блокнотом Датабрікс Джіні Код.",
            "doc_glossary": "Глосарій",
            "doc_gl_domain": "Бізнес-сфера, що об'єднує пов'язані бізнес-кейси.",
            "doc_gl_subdomain": "Вужча тема всередині бізнес-домену.",
            "doc_gl_technique": "Аналітичний метод, наприклад прогнозування або класифікація.",
            "doc_gl_inspire": "Оцінка від 0 до 10, що поєднує якість і бізнес-пріоритет.",
            "doc_gl_quality": "Рівень від «{lowest}» до «{highest}», що присвоюється кожному бізнес-кейсу.",
            "doc_gl_elite": "Оцінка режиму «{ultra}» за цінністю, доказовістю, реалізовністю та застосовністю.",
            "doc_gl_coverage": "Бізнес-кейс нижче планки, збережений, щоб запитаний домен був представлений.",
            "doc_gl_notebook": "Блокнот Датабрікс з інструкціями Джіні Код для реалізації бізнес-кейсу.",
            "doc_notebook_term": "Блокнот Джіні",
            "doc_next_steps": "Наступні кроки",
            "doc_closing_title": "Від бізнес-кейсів до результатів",
            "doc_closing_body": "Оберіть найважливіші бізнес-кейси, відкрийте їхні блокноти Джіні та реалізуйте перший разом із командою. Кроки показують рекомендований порядок; темп визначає команда.",
            "doc_step_select": "Вибір",
            "doc_step_select_body": "Оберіть бізнес-кейси з найкращою оцінкою та зрозумілим спонсором.",
            "doc_step_validate": "Перевірка даних",
            "doc_step_validate_body": "Погодьте задіяні таблиці та їхні з'єднання з власниками даних.",
            "doc_step_build": "Реалізація з Джіні Код",
            "doc_step_build_body": "Відкрийте блокнот Джіні та створіть першу робочу версію.",
            "doc_step_schedule": "Розклад",
            "doc_step_schedule_body": "Запускайте блокнот як завдання Датабрікс, що оновлює таблицю результатів.",
            "doc_step_measure": "Вимірювання та масштабування",
            "doc_step_measure_body": "Відстежуйте отриману бізнес-цінність, потім переходьте до наступного бізнес-кейсу.",
            "doc_agenda": "Порядок денний",
            "doc_roadmap": "Дорожня карта та наступні кроки",
            "doc_use_case_index": "Покажчик бізнес-кейсів",
            "doc_continued": "продовження",
            "doc_not_provided": "Не вказано",
            "doc_bob_score": "Оцінка Best-of-Best",
            "elite_cadence": "Періодичність рішення",
            "value_cadence_hourly": "Щогодини",
            "value_cadence_daily": "Щодня",
            "value_cadence_weekly": "Щотижня",
            "value_cadence_biweekly": "Раз на два тижні",
            "value_cadence_monthly": "Щомісяця",
            "value_cadence_quarterly": "Щокварталу",
            "value_cadence_yearly": "Щороку",
            "value_cadence_real_time": "У реальному часі",
            "value_cadence_per_event": "За подією",
            "doc_generated_by": "Створено за допомогою {brand} {version} · {date}",
            "doc_disclaimer_ai": "Сценарії використання, оцінки та описи в цьому документі створені ШІ. Перевірте їх разом із власниками даних і бізнес-спонсорами, перш ніж ухвалювати рішення.",
            "doc_disclaimer_figures": "Оцінки, значення та результати є орієнтовними й слугують лише для визначення пріоритетів. Жодна цифра в цьому документі не гарантована.",
            "doc_contents": "Зміст",
            "doc_index_lead": "Усі сценарії використання в порядку підсумкового рейтингу (усього: {count}).",
            "doc_deck_scope": "Ця презентація докладно описує сценарії використання з найвищим рейтингом ({count} з {total}). PDF-каталог описує всі.",
            "doc_deck_scope_all": "Ця презентація докладно описує всі сценарії використання (усього: {total}), по одній картці на кожен.",
            "value_general_improvement": "Загальне покращення",
            "value_reduce_cost": "Скорочення витрат",
            "value_increase_revenue": "Збільшення доходу",
            "value_boost_productivity": "Підвищення продуктивності",
            "value_mitigate_risk": "Зниження ризиків",
            "value_protect_revenue": "Захист доходу",
            "value_align_to_regulations": "Дотримання нормативів",
            "value_improve_customer_experience": "Покращення клієнтського досвіду",
            "value_enable_data_driven_decisions": "Забезпечення рішень на основі даних",
            "value_optimize_operations": "Оптимізація операцій",
            "value_empower_talent": "Розвиток талантів",
            "value_enhance_experience": "Покращення досвіду",
            "value_drive_innovation": "Стимулювання інновацій",
            "value_achieve_esg": "Досягнення ESG",
            "value_execute_strategy": "Реалізація стратегії",
            "value_forecasting": "Прогнозування",
            "value_classification": "Класифікація",
            "value_anomaly_detection": "Виявлення аномалій",
            "value_cohort_analysis": "Когортний аналіз",
            "value_segmentation": "Сегментація",
            "value_sentiment_analysis": "Аналіз настроїв",
            "value_trend_analysis": "Аналіз трендів",
            "value_prescriptive_analytics": "Приписна аналітика",
            "value_root_cause_analysis": "Аналіз першопричин",
            "value_optimization": "Оптимізація",
            "value_recommendation": "Рекомендація",
            "value_time_series_analysis": "Аналіз часових рядів",
            "value_predictive_analytics": "Прогнозна аналітика",
            "value_descriptive_analytics": "Описова аналітика",
        },
        "Dutch": {
            "main_title": "Gebruikscasegenerator aangedreven door Databricks Genie Code",
            "intro": "Dit notebook bevat door AI gegenereerde gebruikscases op basis van uw schema's. Hieronder vindt u een samenvatting van de gegenereerde scenario's per bedrijfsdomein.",
            "domain": "Bedrijfsdomein",
            "total": "Totaal aantal gebruikscases",
            "summaries": "Samenvattingen van gebruikscases",
            "sum_id": "ID",
            "sum_name": "Naam",
            "sum_value": "Bedrijfswaarde",
            "sum_outcome": "Verwacht resultaat",
            "warning_header": "WAARSCHUWING",
            "warning_body": "Voer dit notebook niet uit. Het is uitsluitend bedoeld voor demonstratie en catalogisering.",
            "disclaimer": "Deze inhoud is door AI gegenereerd en dient door een gekwalificeerde engineer te worden beoordeeld voordat deze in productie wordt gebruikt. Databricks is niet aansprakelijk voor problemen die voortvloeien uit het gebruik van deze inhoud.",
            "detailed_scenarios": "Details van gebruikscases",
            "aspect": "Aspect",
            "description": "Beschrijving",
            "aspect_domain": "Bedrijfsdomein",
            "type": "Type",
            "analytics_technique": "Analysetechniek",
            "primary_table": "Primaire tabel",
            "quality": "Kwaliteit",
            "priority": "Prioriteit",
            "usecase": "Gebruikscase",
            "description_label": "Beschrijving",
            "value_type_problem": "Probleem",
            "value_type_risk": "Risico",
            "value_type_opportunity": "Kans",
            "value_type_improvement": "Verbetering",
            "value_priority_ultra_high": "Extreem hoog",
            "value_priority_very_high": "Zeer hoog",
            "value_priority_high": "Hoog",
            "value_priority_medium": "Gemiddeld",
            "value_priority_low": "Laag",
            "value_priority_very_low": "Zeer laag",
            "value_priority_ultra_low": "Extreem laag",
            "statement": "Verklaring",
            "solution": "Oplossing",
            "aspect_beneficiary": "Begunstigde",
            "beneficiary": "Begunstigde",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Afstemming bedrijfsprioriteit",
            "strategic_goals_alignment": "Afstemming strategische doelen",
            "subdomain": "Subdomein",
            "aspect_value": "Bedrijfswaarde",
            "business_value": "Bedrijfswaarde",
            "aspect_tables": "Betrokken tabellen",
            "quality_reasons": "Kwaliteitsredenen",
            "aspect_ai_function": "AI-functie",
            "aspect_analytics_technique": "Analysetechniek",
            "aspect_primary_table": "Primaire tabel",
            "aspect_quality": "Kwaliteit",
            "aspect_priority": "Prioriteit",
            "strategic_alignment": "Strategische afstemming",
            "return_on_investment": "Rendement op investering",
            "reusability": "Herbruikbaarheid",
            "time_to_value": "Tijd tot waarde",
            "data_availability": "Databeschikbaarheid",
            "data_accessibility": "Datatoegang",
            "architecture_fitness": "Architectuurgeschiktheid",
            "team_skills": "Teamvaardigheden",
            "domain_knowledge": "Domeinkennis",
            "people_allocation": "Personeelstoewijzing",
            "budget_allocation": "Budgettoewijzing",
            "time_to_production": "Tijd tot productie",
            "value_score": "Waardescore",
            "feasibility_score": "Haalbaarheidsscore",
            "priority_score": "Prioriteitsscore",
            "inspire_score": "Inspire-score",
            "quality_score_header": "Kwaliteitsscore",
            "pdf_title": "Strategische AI-gebruikscases aangedreven door Databricks Genie Code",
            "pdf_for": "Voor",
            "pdf_exec_summary": "Samenvatting",
            "pdf_toc_title": "Gebruikscasedomeinen",
            "pdf_detailed_view": "Gedetailleerde gebruikscasecatalogus",
            "pdf_disclaimer_title": "Disclaimer",
            "pdf_fallback_summary_p1": "Dit document beschrijft {total_cases} hoogwaardige analytische gebruikscases geïdentificeerd voor {business_name}.",
            "pdf_fallback_summary_p2": "De volgende pagina's bieden een gedetailleerde uitsplitsing van deze kansen, gecategoriseerd per bedrijfsdomein.",
            "pptx_main_title": "Strategische AI-gebruikscases aangedreven door Databricks Genie Code",
            "pptx_for": "Voor",
            "pptx_disclaimer_title": "Disclaimer",
            "pptx_domain_suffix": "Gebruikscases",
            "elite_verdict": "Oordeel",
            "elite_decision": "Beslissing",
            "elite_decision_owner": "Beslissingseigenaar",
            "elite_kpi": "KPI",
            "elite_value_driver": "Waardedrijver",
            "elite_evidence_columns": "Bewijskolommen",
            "elite_join_path": "Koppelpad",
            "elite_build_path": "Implementatiepad",
            "elite_red_team_note": "Red-team-opmerking",
            "elite_coverage_pick": "Keuze voor domeindekking, onder de lat",
            "elite_fallback_pick": "Onder de {mode}-lat",
            "elite_fallback_near_miss": "Net niet, onder de {mode}-lat",
            "doc_fallback_pick": "Onder de lat",
            "doc_fallback_notice": "Geen enkele use case haalde de {mode}-lat. Dit zijn de dichtstbijzijnde kandidaten, elk gemarkeerd onder de lat.",
            "elite_reason": "Reden",
            "untranslated_notice": "Weergegeven in het Engels (vertaling niet beschikbaar)",
            "more_use_cases": "...en nog {count} use cases.",
            "executive_summary_not_available": "Samenvatting niet beschikbaar.",
            "domain_summary_not_available": "Domeinsamenvatting niet beschikbaar.",
            "summary_not_available": "Samenvatting niet beschikbaar.",
            "doc_catalog_kicker": "Catalogus van AI-gebruikscases",
            "doc_prepared_for": "Opgesteld voor",
            "doc_generated_on": "Aangemaakt op",
            "doc_quality_mode": "Kwaliteitsmodus",
            "doc_use_cases": "Gebruikscases",
            "doc_domains": "Bedrijfsdomeinen",
            "doc_elite_use_cases": "Elite-gebruikscases",
            "doc_high_or_better": "Hoge kwaliteit of beter",
            "doc_best_score": "Beste Inspire-score",
            "doc_not_scored": "Niet gescoord",
            "doc_coverage_picks": "Dekkingskeuzes",
            "doc_coverage_pick": "Dekkingskeuze",
            "value_correlation_analysis": "Correlatieanalyse",
            "value_pareto_analysis": "Pareto-analyse",
            "value_funnel_analysis": "Trechteranalyse",
            "value_document_processing": "Documentverwerking",
            "value_extraction": "Extractie",
            "value_ai_analysis": "AI-analyse",
            "value_simulation": "Simulatie",
            "value_geospatial_analysis": "Geospatiale analyse",
            "value_predictive_modeling": "Voorspellende modellering",
            "value_clustering": "Clusteranalyse",
            "value_regression_analysis": "Regressieanalyse",
            "value_survival_analysis": "Overlevingsanalyse",
            "value_network_analysis": "Netwerkanalyse",
            "value_mode_good": "Goed",
            "value_mode_high": "Hoog",
            "value_mode_ultra_high": "Ultrahoog",
            "doc_exec_kicker": "De kans",
            "doc_guide_kicker": "Leeswijzer",
            "doc_how_to_read": "Zo leest u deze catalogus",
            "doc_guide_lead": "Elke gebruikscase komt uit de eigen metadata van de organisatie en krijgt een score op één absolute schaal. Gebruik de scores om te vergelijken, de niveaus om te filteren en de domeinpagina's om te navigeren.",
            "doc_inspire_score_title": "Inspire-score, van 0 tot 10",
            "doc_inspire_score_body": "Combineert de kwaliteit van een gebruikscase met de bedrijfsprioriteit. Hoger is beter, en de catalogus is erop gesorteerd.",
            "doc_quality_tiers_title": "Kwaliteitsniveaus",
            "doc_quality_tiers_body": "Elke gebruikscase krijgt een niveau van {lowest} tot {highest} dat laat zien hoe specifiek, waardevol en bouwbaar die is met de beschikbare data.",
            "doc_modes_title": "Kwaliteitsmodi",
            "doc_mode_good_body": "Behoudt gebruikscases met de beoordeling {medium} of beter.",
            "doc_mode_high_body": "Behoudt alleen gebruikscases met de beoordeling {high} of beter.",
            "doc_mode_ultra_body": "Behoudt alleen gebruikscases die een absolute lat halen voor waarde, bewijs, uitvoerbaarheid en bruikbaarheid in de praktijk. Er is geen quotum.",
            "doc_this_catalog": "Deze catalogus",
            "doc_elite_score_title": "Elitescore",
            "doc_elite_score_body": "Wordt gebruikt in de modus {ultra}. De score weegt bedrijfswaarde, bewijskracht, uitvoerbaarheid en bruikbaarheid in de praktijk, en een elite-gebruikscase scoort op elk onderdeel minstens {min_score} van 10.",
            "doc_coverage_title": "Keuzes voor domeindekking",
            "doc_coverage_body": "Als een gevraagd bedrijfsdomein geen gebruikscase boven de lat heeft, blijft de sterkste kandidaat behouden met het label dekkingskeuze onder de lat.",
            "doc_portfolio_kicker": "Portfolio",
            "doc_portfolio_glance": "Het portfolio in één oogopslag",
            "doc_per_domain": "Gebruikscases per domein",
            "doc_quality_distribution": "Kwaliteitsverdeling",
            "doc_top_use_cases": "Belangrijkste gebruikscases",
            "doc_rank": "Positie",
            "doc_domain_label": "Domein",
            "doc_domain_overview": "Over dit domein",
            "doc_what_it_covers": "Wat het omvat",
            "doc_top_opportunities": "Grootste kansen",
            "doc_domain_fallback": "Dit domein bundelt gebruikscases binnen {subdomains}, in totaal {count}. De sterkste kansen staan hieronder en worden op de volgende pagina's toegelicht.",
            "doc_open_notebook": "Open het Genie-notebook",
            "doc_elite_score": "Elitescore",
            "doc_best_of_best": "Het beste van het beste",
            "doc_appendix": "Bijlage",
            "doc_methodology": "Methodologie",
            "strategy_path": "Strategiepad",
            "kpis": "KPI's",
            "expected_key_results": "Verwachte kernresultaten",
            "measures_of_success": "Succesmaatstaven",
            "financial_benefits": "Financiële voordelen",
            "non_financial_benefits": "Niet-financiële voordelen",
            "foundations": "Datafundamenten",
            "doc_foundations_title": "Gedeelde datafundamenten",
            "doc_foundations_lead": "Tabellen en joins die twee of meer gebruikscases nodig hebben. Door ze eerst te bouwen worden de meeste gebruikscases mogelijk. De volgorde is een reeks, geen planning.",
            "doc_foundation_tables": "Tabellen",
            "doc_foundation_unlocks": "Mogelijk gemaakte gebruikscases",
            "doc_data_access_standard": "Standaardontdekking: Inspire heeft geen klantgegevens gelezen. Alleen metadata, definities van metric views en gebruikstellingen zijn gebruikt.",
            "doc_data_access_deep": "Diepe ontdekking: Inspire heeft aggregaten gelezen uit de tabellen in de scope, in totaal {tables} (aantallen rijen, datumdekking, aandelen lege waarden, aantallen unieke waarden en korte waardelijsten van niet-persoonlijke kolommen). Er zijn geen gegevensrijen naar de AI gestuurd.",
            "doc_method_1_deep": "Inspire leest metadata en bij diepe ontdekking ook aggregaten die over de tabellen worden berekend. Het stuurt nooit gegevensrijen naar de AI.",
            "doc_guide_lead_deep": "Elke gebruikscase komt uit de eigen metadata en dataprofielen van de organisatie en wordt beoordeeld op één absolute schaal. Gebruik de scores om te vergelijken, de niveaus om te filteren en de domeinpagina's om te navigeren.",
            "value_not_aligned": "Niet afgestemd",
            "kpi_baseline": "nulmeting {value}, laatste {days} dagen, gemeten op {date}",
            "kpi_baseline_undated": "nulmeting {value}, gemeten op {date}",
            "doc_method_1": "Inspire leest alleen metadata: tabel- en kolomnamen, typen, opmerkingen, sleutels en tags. Rijgegevens leest het nooit.",
            "doc_method_2": "Het leidt de bedrijfscontext af en genereert vervolgens kandidaten voor elk bedrijfsdomein dat de data ondersteunt.",
            "doc_method_3": "Dubbele kandidaten worden verwijderd, elke kandidaat krijgt een score voor waarde en haalbaarheid, en kwaliteitspoorten filteren zwakke of ongefundeerde ideeën weg.",
            "doc_method_4": "In de modus {ultra} beoordeelt een elitejury elke overgebleven kandidaat, en deterministische controles bevestigen dat elke genoemde tabel en kolom bestaat.",
            "doc_method_5": "De definitieve lijst wordt gerangschikt, gegroepeerd per bedrijfsdomein en, indien gekozen, aangevuld met een Databricks Genie Code-notebook.",
            "doc_glossary": "Woordenlijst",
            "doc_gl_domain": "Een bedrijfsgebied dat verwante gebruikscases groepeert.",
            "doc_gl_subdomain": "Een smaller thema binnen een bedrijfsdomein.",
            "doc_gl_technique": "De analytische methode, bijvoorbeeld voorspelling of classificatie.",
            "doc_gl_inspire": "De score van 0 tot 10 die kwaliteit en bedrijfsprioriteit combineert.",
            "doc_gl_quality": "Het niveau van {lowest} tot {highest} dat elke gebruikscase krijgt.",
            "doc_gl_elite": "De score in de modus {ultra} voor waarde, bewijs, uitvoerbaarheid en bruikbaarheid in de praktijk.",
            "doc_gl_coverage": "Een gebruikscase onder de lat die behouden blijft zodat een gevraagd domein vertegenwoordigd is.",
            "doc_gl_notebook": "Een Databricks-notebook met Genie Code-instructies om de gebruikscase te bouwen.",
            "doc_notebook_term": "Genie-notebook",
            "doc_next_steps": "Volgende stappen",
            "doc_closing_title": "Van gebruikscases naar resultaten",
            "doc_closing_body": "Kies de belangrijkste gebruikscases, open hun Genie-notebooks en bouw de eerste samen met het team. De stappen zijn een voorgestelde volgorde; het team bepaalt het tempo.",
            "doc_step_select": "Selecteren",
            "doc_step_select_body": "Kies de gebruikscases met de hoogste score en een duidelijke sponsor.",
            "doc_step_validate": "Data valideren",
            "doc_step_validate_body": "Bevestig de betrokken tabellen en hun koppelingen met de data-eigenaren.",
            "doc_step_build": "Bouwen met Genie Code",
            "doc_step_build_body": "Open het Genie-notebook en genereer de eerste werkende versie.",
            "doc_step_schedule": "Plannen",
            "doc_step_schedule_body": "Voer het notebook uit als een Databricks-job die de resultatentabel bijwerkt.",
            "doc_step_measure": "Meten en opschalen",
            "doc_step_measure_body": "Volg de gerealiseerde bedrijfswaarde en breid daarna uit naar de volgende gebruikscase.",
            "doc_agenda": "Agenda",
            "doc_roadmap": "Roadmap en volgende stappen",
            "doc_use_case_index": "Index van gebruikscases",
            "doc_continued": "vervolg",
            "doc_not_provided": "Niet opgegeven",
            "doc_bob_score": "Best-of-Best-score",
            "elite_cadence": "Beslisfrequentie",
            "value_cadence_hourly": "Elk uur",
            "value_cadence_daily": "Dagelijks",
            "value_cadence_weekly": "Wekelijks",
            "value_cadence_biweekly": "Om de twee weken",
            "value_cadence_monthly": "Maandelijks",
            "value_cadence_quarterly": "Per kwartaal",
            "value_cadence_yearly": "Jaarlijks",
            "value_cadence_real_time": "Realtime",
            "value_cadence_per_event": "Per gebeurtenis",
            "doc_generated_by": "Gemaakt met {brand} {version} · {date}",
            "doc_disclaimer_ai": "De use cases, scores en teksten in dit document zijn door AI gegenereerd. Valideer ze met de data-eigenaren en de business-sponsors voordat er een beslissing wordt genomen.",
            "doc_disclaimer_figures": "Scores, waarden en resultaten zijn schattingen die alleen dienen om te prioriteren. Geen enkel cijfer in dit document is gegarandeerd.",
            "doc_contents": "Inhoud",
            "doc_index_lead": "Alle use cases in de definitieve rangorde, in totaal {count}.",
            "doc_deck_scope": "Deze presentatie beschrijft de hoogst gerangschikte use cases, {count} van {total}. De PDF-catalogus beschrijft ze allemaal.",
            "doc_deck_scope_all": "Deze presentatie beschrijft alle use cases, in totaal {total}, met een kaart per use case.",
            "value_general_improvement": "Algemene verbetering",
            "value_reduce_cost": "Kosten verlagen",
            "value_increase_revenue": "Omzet verhogen",
            "value_boost_productivity": "Productiviteit verhogen",
            "value_mitigate_risk": "Risico's beperken",
            "value_protect_revenue": "Omzet beschermen",
            "value_align_to_regulations": "Voldoen aan regelgeving",
            "value_improve_customer_experience": "Klantervaring verbeteren",
            "value_enable_data_driven_decisions": "Datagedreven besluitvorming mogelijk maken",
            "value_optimize_operations": "Operaties optimaliseren",
            "value_empower_talent": "Talent versterken",
            "value_enhance_experience": "Ervaring verbeteren",
            "value_drive_innovation": "Innovatie stimuleren",
            "value_achieve_esg": "ESG bereiken",
            "value_execute_strategy": "Strategie uitvoeren",
            "value_forecasting": "Voorspelling",
            "value_classification": "Classificatie",
            "value_anomaly_detection": "Anomaliedetectie",
            "value_cohort_analysis": "Cohortanalyse",
            "value_segmentation": "Segmentatie",
            "value_sentiment_analysis": "Sentimentanalyse",
            "value_trend_analysis": "Trendanalyse",
            "value_prescriptive_analytics": "Prescriptieve analyse",
            "value_root_cause_analysis": "Oorzaakanalyse",
            "value_optimization": "Optimalisatie",
            "value_recommendation": "Aanbeveling",
            "value_time_series_analysis": "Tijdreeksanalyse",
            "value_predictive_analytics": "Voorspellende analyse",
            "value_descriptive_analytics": "Beschrijvende analyse",
        },
        "Indonesian": {
            "main_title": "Generator Kasus Penggunaan Didukung oleh Databricks Genie Code",
            "intro": "Notebook ini berisi kasus penggunaan yang dihasilkan oleh AI berdasarkan skema Anda. Di bawah ini adalah ringkasan skenario yang dihasilkan berdasarkan domain bisnis.",
            "domain": "Domain Bisnis",
            "total": "Total Kasus Penggunaan",
            "summaries": "Ringkasan Kasus Penggunaan",
            "sum_id": "ID",
            "sum_name": "Nama",
            "sum_value": "Nilai Bisnis",
            "sum_outcome": "Hasil yang Diharapkan",
            "warning_header": "PERINGATAN",
            "warning_body": "Jangan jalankan notebook ini. Ini dimaksudkan hanya untuk demonstrasi dan katalogisasi.",
            "disclaimer": "Konten ini dihasilkan oleh AI dan harus ditinjau oleh insinyur yang berkualifikasi sebelum digunakan dalam produksi. Databricks tidak bertanggung jawab atas masalah apa pun yang timbul dari penggunaan konten ini.",
            "detailed_scenarios": "Detail Kasus Penggunaan",
            "aspect": "Aspek",
            "description": "Deskripsi",
            "aspect_domain": "Domain Bisnis",
            "type": "Tipe",
            "analytics_technique": "Teknik Analitik",
            "primary_table": "Tabel Utama",
            "quality": "Kualitas",
            "priority": "Prioritas",
            "usecase": "Kasus Penggunaan",
            "description_label": "Deskripsi",
            "value_type_problem": "Masalah",
            "value_type_risk": "Risiko",
            "value_type_opportunity": "Peluang",
            "value_type_improvement": "Peningkatan",
            "value_priority_ultra_high": "Sangat Tinggi Sekali",
            "value_priority_very_high": "Sangat Tinggi",
            "value_priority_high": "Tinggi",
            "value_priority_medium": "Sedang",
            "value_priority_low": "Rendah",
            "value_priority_very_low": "Sangat Rendah",
            "value_priority_ultra_low": "Sangat Rendah Sekali",
            "statement": "Pernyataan",
            "solution": "Solusi",
            "aspect_beneficiary": "Penerima Manfaat",
            "beneficiary": "Penerima Manfaat",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Keselarasan Prioritas Bisnis",
            "strategic_goals_alignment": "Keselarasan Tujuan Strategis",
            "subdomain": "Subdomain",
            "aspect_value": "Nilai Bisnis",
            "business_value": "Nilai Bisnis",
            "aspect_tables": "Tabel yang Terlibat",
            "quality_reasons": "Alasan Kualitas",
            "aspect_ai_function": "Fungsi AI",
            "aspect_analytics_technique": "Teknik Analitik",
            "aspect_primary_table": "Tabel Utama",
            "aspect_quality": "Kualitas",
            "aspect_priority": "Prioritas",
            "strategic_alignment": "Keselarasan Strategis",
            "return_on_investment": "Pengembalian Investasi",
            "reusability": "Dapat Digunakan Kembali",
            "time_to_value": "Waktu ke Nilai",
            "data_availability": "Ketersediaan Data",
            "data_accessibility": "Aksesibilitas Data",
            "architecture_fitness": "Kesesuaian Arsitektur",
            "team_skills": "Keterampilan Tim",
            "domain_knowledge": "Pengetahuan Domain",
            "people_allocation": "Alokasi Personel",
            "budget_allocation": "Alokasi Anggaran",
            "time_to_production": "Waktu ke Produksi",
            "value_score": "Skor Nilai",
            "feasibility_score": "Skor Kelayakan",
            "priority_score": "Skor Prioritas",
            "inspire_score": "Skor Inspire",
            "quality_score_header": "Skor Kualitas",
            "pdf_title": "Kasus Penggunaan AI Strategis Didukung oleh Databricks Genie Code",
            "pdf_for": "Untuk",
            "pdf_exec_summary": "Ringkasan Eksekutif",
            "pdf_toc_title": "Domain Kasus Penggunaan",
            "pdf_detailed_view": "Katalog Detail Kasus Penggunaan",
            "pdf_disclaimer_title": "Penafian",
            "pdf_fallback_summary_p1": "Dokumen ini menjelaskan {total_cases} kasus penggunaan analitik bernilai tinggi yang diidentifikasi untuk {business_name}.",
            "pdf_fallback_summary_p2": "Halaman-halaman berikut memberikan uraian terperinci tentang peluang-peluang ini, dikategorikan berdasarkan domain bisnis.",
            "pptx_main_title": "Kasus Penggunaan AI Strategis Didukung oleh Databricks Genie Code",
            "pptx_for": "Untuk",
            "pptx_disclaimer_title": "Penafian",
            "pptx_domain_suffix": "Kasus Penggunaan",
            "elite_verdict": "Putusan",
            "elite_decision": "Keputusan",
            "elite_decision_owner": "Pemilik keputusan",
            "elite_kpi": "KPI",
            "elite_value_driver": "Pendorong nilai",
            "elite_evidence_columns": "Kolom bukti",
            "elite_join_path": "Jalur penggabungan",
            "elite_build_path": "Jalur implementasi",
            "elite_red_team_note": "Catatan tim merah",
            "elite_coverage_pick": "Pilihan cakupan domain, di bawah standar",
            "elite_fallback_pick": "Di bawah standar {mode}",
            "elite_fallback_near_miss": "Nyaris lolos, di bawah standar {mode}",
            "doc_fallback_pick": "Di bawah standar",
            "doc_fallback_notice": "Tidak ada kasus penggunaan yang memenuhi standar {mode}. Ini adalah kandidat terdekat, masing-masing ditandai di bawah standar.",
            "elite_reason": "Alasan",
            "untranslated_notice": "Ditampilkan dalam bahasa Inggris (terjemahan tidak tersedia)",
            "more_use_cases": "...dan {count} kasus penggunaan lainnya.",
            "executive_summary_not_available": "Ringkasan eksekutif tidak tersedia.",
            "domain_summary_not_available": "Ringkasan domain tidak tersedia.",
            "summary_not_available": "Ringkasan tidak tersedia.",
            "doc_catalog_kicker": "Katalog kasus penggunaan AI",
            "doc_prepared_for": "Disiapkan untuk",
            "doc_generated_on": "Dibuat pada",
            "doc_quality_mode": "Mode kualitas",
            "doc_use_cases": "Kasus penggunaan",
            "doc_domains": "Domain bisnis",
            "doc_elite_use_cases": "Kasus penggunaan elite",
            "doc_high_or_better": "Kualitas tinggi atau lebih baik",
            "doc_best_score": "Skor Inspire terbaik",
            "doc_not_scored": "Belum dinilai",
            "doc_coverage_picks": "Pilihan cakupan",
            "doc_coverage_pick": "Pilihan cakupan",
            "value_correlation_analysis": "Analisis Korelasi",
            "value_pareto_analysis": "Analisis Pareto",
            "value_funnel_analysis": "Analisis Corong",
            "value_document_processing": "Pemrosesan Dokumen",
            "value_extraction": "Ekstraksi",
            "value_ai_analysis": "Analisis AI",
            "value_simulation": "Simulasi",
            "value_geospatial_analysis": "Analisis Geospasial",
            "value_predictive_modeling": "Pemodelan Prediktif",
            "value_clustering": "Klasterisasi",
            "value_regression_analysis": "Analisis Regresi",
            "value_survival_analysis": "Analisis Kelangsungan Hidup",
            "value_network_analysis": "Analisis Jaringan",
            "value_mode_good": "Baik",
            "value_mode_high": "Tinggi",
            "value_mode_ultra_high": "Ultra tinggi",
            "doc_exec_kicker": "Peluang",
            "doc_guide_kicker": "Panduan membaca",
            "doc_how_to_read": "Cara membaca katalog ini",
            "doc_guide_lead": "Setiap kasus penggunaan berasal dari metadata milik organisasi sendiri dan dinilai pada satu skala absolut. Gunakan skor untuk membandingkan, tingkat untuk menyaring, dan halaman domain untuk bernavigasi.",
            "doc_inspire_score_title": "Skor Inspire, dari 0 hingga 10",
            "doc_inspire_score_body": "Menggabungkan kualitas kasus penggunaan dengan prioritas bisnisnya. Makin tinggi makin baik, dan katalog diurutkan berdasarkan skor ini.",
            "doc_quality_tiers_title": "Tingkat kualitas",
            "doc_quality_tiers_body": "Setiap kasus penggunaan mendapat tingkat dari {lowest} hingga {highest} yang mencerminkan seberapa spesifik, bernilai, dan dapat dibangun dengan data yang tersedia.",
            "doc_modes_title": "Mode kualitas",
            "doc_mode_good_body": "Mempertahankan kasus penggunaan dengan peringkat {medium} atau lebih baik.",
            "doc_mode_high_body": "Hanya mempertahankan kasus penggunaan dengan peringkat {high} atau lebih baik.",
            "doc_mode_ultra_body": "Hanya mempertahankan kasus penggunaan yang melewati standar absolut untuk nilai, bukti, keterlaksanaan, dan kemudahan ditindaklanjuti. Tidak ada kuota.",
            "doc_this_catalog": "Katalog ini",
            "doc_elite_score_title": "Skor elite",
            "doc_elite_score_body": "Digunakan dalam mode {ultra}. Skor ini menimbang nilai bisnis, kekuatan bukti, keterlaksanaan, dan kemudahan ditindaklanjuti, dan kasus penggunaan elite meraih minimal {min_score} dari 10 pada setiap aspek.",
            "doc_coverage_title": "Pilihan cakupan domain",
            "doc_coverage_body": "Jika domain bisnis yang diminta tidak memiliki kasus penggunaan di atas standar, kandidat terkuatnya dipertahankan dan diberi label pilihan cakupan di bawah standar.",
            "doc_portfolio_kicker": "Portofolio",
            "doc_portfolio_glance": "Sekilas portofolio",
            "doc_per_domain": "Kasus penggunaan per domain",
            "doc_quality_distribution": "Distribusi kualitas",
            "doc_top_use_cases": "Kasus penggunaan teratas",
            "doc_rank": "Peringkat",
            "doc_domain_label": "Domain",
            "doc_domain_overview": "Tentang domain ini",
            "doc_what_it_covers": "Cakupannya",
            "doc_top_opportunities": "Peluang utama",
            "doc_domain_fallback": "Domain ini menghimpun {count} kasus penggunaan di bidang {subdomains}. Peluang terkuatnya tercantum di bawah dan dijelaskan pada halaman berikutnya.",
            "doc_open_notebook": "Buka notebook Genie",
            "doc_elite_score": "Skor elite",
            "doc_best_of_best": "Terbaik dari yang terbaik",
            "doc_appendix": "Lampiran",
            "doc_methodology": "Metodologi",
            "strategy_path": "Jalur strategi",
            "kpis": "KPI",
            "expected_key_results": "Hasil utama yang diharapkan",
            "measures_of_success": "Ukuran keberhasilan",
            "financial_benefits": "Manfaat finansial",
            "non_financial_benefits": "Manfaat non-finansial",
            "foundations": "Fondasi data",
            "doc_foundations_title": "Fondasi data bersama",
            "doc_foundations_lead": "Tabel dan join yang dibutuhkan oleh dua atau lebih kasus penggunaan. Membangunnya lebih dulu membuka kasus penggunaan terbanyak. Urutan ini adalah rangkaian, bukan jadwal.",
            "doc_foundation_tables": "Tabel",
            "doc_foundation_unlocks": "Kasus penggunaan yang terbuka",
            "doc_data_access_standard": "Penemuan standar: Inspire tidak membaca data pelanggan. Hanya metadata, definisi tampilan metrik, dan jumlah penggunaan yang dipakai.",
            "doc_data_access_deep": "Penemuan mendalam: Inspire membaca agregat dari {tables} tabel (jumlah baris, cakupan tanggal, proporsi nilai kosong, jumlah nilai unik, dan daftar singkat nilai kolom non-pribadi). Tidak ada baris data yang dikirim ke AI.",
            "doc_method_1_deep": "Inspire membaca metadata dan, pada penemuan mendalam, juga agregat yang dihitung atas tabel. Inspire tidak pernah mengirim baris data ke AI.",
            "doc_guide_lead_deep": "Setiap kasus penggunaan berasal dari metadata dan profil data milik organisasi dan dinilai pada satu skala mutlak. Gunakan skor untuk membandingkan, tingkatan untuk memfilter, dan halaman domain untuk bernavigasi.",
            "value_not_aligned": "Tidak selaras",
            "kpi_baseline": "nilai dasar {value}, {days} hari terakhir, diukur {date}",
            "kpi_baseline_undated": "nilai dasar {value}, diukur {date}",
            "doc_method_1": "Inspire hanya membaca metadata: nama tabel dan kolom, tipe, komentar, kunci, dan tag. Inspire tidak pernah membaca data baris.",
            "doc_method_2": "Inspire menyimpulkan konteks bisnis, lalu membuat kandidat kasus penggunaan untuk setiap domain bisnis yang didukung data.",
            "doc_method_3": "Duplikat dihapus, setiap kandidat dinilai berdasarkan nilai dan kelayakan, dan gerbang kualitas menyaring ide yang lemah atau tidak didukung.",
            "doc_method_4": "Dalam mode {ultra}, juri elite meninjau setiap kandidat yang tersisa, dan pemeriksaan deterministik memastikan setiap tabel dan kolom yang disebut benar-benar ada.",
            "doc_method_5": "Daftar akhir diurutkan, dikelompokkan per domain bisnis, dan bila dipilih dilengkapi notebook Databricks Genie Code.",
            "doc_glossary": "Glosarium",
            "doc_gl_domain": "Area bisnis yang mengelompokkan kasus penggunaan terkait.",
            "doc_gl_subdomain": "Tema yang lebih sempit di dalam domain bisnis.",
            "doc_gl_technique": "Metode analitik, misalnya peramalan atau klasifikasi.",
            "doc_gl_inspire": "Skor 0 hingga 10 yang menggabungkan kualitas dan prioritas bisnis.",
            "doc_gl_quality": "Tingkat dari {lowest} hingga {highest} yang diberikan pada setiap kasus penggunaan.",
            "doc_gl_elite": "Skor mode {ultra} untuk nilai, bukti, keterlaksanaan, dan kemudahan ditindaklanjuti.",
            "doc_gl_coverage": "Kasus penggunaan di bawah standar yang dipertahankan agar domain yang diminta tetap terwakili.",
            "doc_gl_notebook": "Notebook Databricks berisi instruksi Genie Code untuk membangun kasus penggunaan.",
            "doc_notebook_term": "Notebook Genie",
            "doc_next_steps": "Langkah berikutnya",
            "doc_closing_title": "Dari kasus penggunaan menuju hasil",
            "doc_closing_body": "Pilih kasus penggunaan yang paling penting, buka notebook Genie-nya, lalu bangun yang pertama bersama tim. Langkah-langkah ini adalah urutan yang disarankan; tim yang menentukan kecepatannya.",
            "doc_step_select": "Pilih",
            "doc_step_select_body": "Pilih kasus penggunaan dengan skor terkuat dan sponsor yang jelas.",
            "doc_step_validate": "Validasi data",
            "doc_step_validate_body": "Konfirmasikan tabel yang terlibat dan penggabungannya dengan pemilik data.",
            "doc_step_build": "Bangun dengan Genie Code",
            "doc_step_build_body": "Buka notebook Genie dan hasilkan versi pertama yang berfungsi.",
            "doc_step_schedule": "Jadwalkan",
            "doc_step_schedule_body": "Jalankan notebook sebagai job Databricks yang memperbarui tabel hasil.",
            "doc_step_measure": "Ukur dan perluas",
            "doc_step_measure_body": "Pantau nilai bisnis yang dihasilkan, lalu perluas ke kasus penggunaan berikutnya.",
            "doc_agenda": "Agenda",
            "doc_roadmap": "Peta jalan dan langkah berikutnya",
            "doc_use_case_index": "Indeks kasus penggunaan",
            "doc_continued": "lanjutan",
            "doc_not_provided": "Tidak tersedia",
            "doc_bob_score": "Skor Best-of-Best",
            "elite_cadence": "Frekuensi keputusan",
            "value_cadence_hourly": "Setiap jam",
            "value_cadence_daily": "Harian",
            "value_cadence_weekly": "Mingguan",
            "value_cadence_biweekly": "Setiap dua minggu",
            "value_cadence_monthly": "Bulanan",
            "value_cadence_quarterly": "Triwulanan",
            "value_cadence_yearly": "Tahunan",
            "value_cadence_real_time": "Waktu nyata",
            "value_cadence_per_event": "Per peristiwa",
            "doc_generated_by": "Dibuat dengan {brand} {version} · {date}",
            "doc_disclaimer_ai": "Kasus penggunaan, skor, dan narasi dalam dokumen ini dihasilkan oleh AI. Validasikan bersama pemilik data dan sponsor bisnis sebelum keputusan apa pun diambil.",
            "doc_disclaimer_figures": "Skor, nilai, dan hasil adalah perkiraan untuk penentuan prioritas saja. Tidak ada angka dalam dokumen ini yang dijamin.",
            "doc_contents": "Daftar isi",
            "doc_index_lead": "Semua {count} kasus penggunaan dalam urutan peringkat akhir.",
            "doc_deck_scope": "Presentasi ini merinci {count} kasus penggunaan teratas. Katalog PDF merinci seluruh {total}.",
            "doc_deck_scope_all": "Presentasi ini merinci seluruh {total} kasus penggunaan, satu kartu untuk masing-masing.",
            "value_general_improvement": "Peningkatan Umum",
            "value_reduce_cost": "Mengurangi Biaya",
            "value_increase_revenue": "Meningkatkan Pendapatan",
            "value_boost_productivity": "Meningkatkan Produktivitas",
            "value_mitigate_risk": "Mengurangi Risiko",
            "value_protect_revenue": "Melindungi Pendapatan",
            "value_align_to_regulations": "Mematuhi Peraturan",
            "value_improve_customer_experience": "Meningkatkan Pengalaman Pelanggan",
            "value_enable_data_driven_decisions": "Memungkinkan Keputusan Berbasis Data",
            "value_optimize_operations": "Mengoptimalkan Operasi",
            "value_empower_talent": "Memberdayakan Talenta",
            "value_enhance_experience": "Meningkatkan Pengalaman",
            "value_drive_innovation": "Mendorong Inovasi",
            "value_achieve_esg": "Mencapai ESG",
            "value_execute_strategy": "Melaksanakan Strategi",
            "value_forecasting": "Peramalan",
            "value_classification": "Klasifikasi",
            "value_anomaly_detection": "Deteksi Anomali",
            "value_cohort_analysis": "Analisis Kohort",
            "value_segmentation": "Segmentasi",
            "value_sentiment_analysis": "Analisis Sentimen",
            "value_trend_analysis": "Analisis Tren",
            "value_prescriptive_analytics": "Analitik Preskriptif",
            "value_root_cause_analysis": "Analisis Akar Masalah",
            "value_optimization": "Optimasi",
            "value_recommendation": "Rekomendasi",
            "value_time_series_analysis": "Analisis Deret Waktu",
            "value_predictive_analytics": "Analitik Prediktif",
            "value_descriptive_analytics": "Analitik Deskriptif",
        },
        "Malay": {
            "main_title": "Penjana Kes Penggunaan Dikuasakan oleh Databricks Genie Code",
            "intro": "Buku nota ini mengandungi kes penggunaan yang dijana oleh AI berdasarkan skema anda. Di bawah ialah ringkasan senario yang dijana mengikut domain perniagaan.",
            "domain": "Domain Perniagaan",
            "total": "Jumlah Kes Penggunaan",
            "summaries": "Ringkasan Kes Penggunaan",
            "sum_id": "ID",
            "sum_name": "Nama",
            "sum_value": "Nilai Perniagaan",
            "sum_outcome": "Hasil Dijangka",
            "warning_header": "AMARAN",
            "warning_body": "Jangan jalankan buku nota ini. Ia bertujuan untuk demonstrasi dan pengkatalogan sahaja.",
            "disclaimer": "Kandungan ini dijana oleh AI dan perlu disemak oleh jurutera yang berkelayakan sebelum digunakan dalam pengeluaran. Databricks tidak bertanggungjawab atas sebarang masalah yang timbul daripada penggunaan kandungan ini.",
            "detailed_scenarios": "Butiran Kes Penggunaan",
            "aspect": "Aspek",
            "description": "Penerangan",
            "aspect_domain": "Domain Perniagaan",
            "type": "Jenis",
            "analytics_technique": "Teknik Analitik",
            "primary_table": "Jadual Utama",
            "quality": "Kualiti",
            "priority": "Keutamaan",
            "usecase": "Kes Penggunaan",
            "description_label": "Penerangan",
            "value_type_problem": "Masalah",
            "value_type_risk": "Risiko",
            "value_type_opportunity": "Peluang",
            "value_type_improvement": "Penambahbaikan",
            "value_priority_ultra_high": "Amat Tinggi Sekali",
            "value_priority_very_high": "Amat Tinggi",
            "value_priority_high": "Tinggi",
            "value_priority_medium": "Sederhana",
            "value_priority_low": "Rendah",
            "value_priority_very_low": "Amat Rendah",
            "value_priority_ultra_low": "Amat Rendah Sekali",
            "statement": "Pernyataan",
            "solution": "Penyelesaian",
            "aspect_beneficiary": "Penerima Manfaat",
            "beneficiary": "Penerima Manfaat",
            "aspect_sponsor": "Penaja",
            "sponsor": "Penaja",
            "business_priority_alignment": "Penjajaran Keutamaan Perniagaan",
            "strategic_goals_alignment": "Penjajaran Matlamat Strategik",
            "subdomain": "Subdomain",
            "aspect_value": "Nilai Perniagaan",
            "business_value": "Nilai Perniagaan",
            "aspect_tables": "Jadual Terlibat",
            "quality_reasons": "Sebab Kualiti",
            "aspect_ai_function": "Fungsi AI",
            "aspect_analytics_technique": "Teknik Analitik",
            "aspect_primary_table": "Jadual Utama",
            "aspect_quality": "Kualiti",
            "aspect_priority": "Keutamaan",
            "strategic_alignment": "Penjajaran Strategik",
            "return_on_investment": "Pulangan Pelaburan",
            "reusability": "Kebolehgunaan Semula",
            "time_to_value": "Masa ke Nilai",
            "data_availability": "Ketersediaan Data",
            "data_accessibility": "Kebolehcapaian Data",
            "architecture_fitness": "Kesesuaian Seni Bina",
            "team_skills": "Kemahiran Pasukan",
            "domain_knowledge": "Pengetahuan Domain",
            "people_allocation": "Peruntukan Kakitangan",
            "budget_allocation": "Peruntukan Bajet",
            "time_to_production": "Masa ke Pengeluaran",
            "value_score": "Skor Nilai",
            "feasibility_score": "Skor Kebolehlaksanaan",
            "priority_score": "Skor Keutamaan",
            "inspire_score": "Skor Inspire",
            "quality_score_header": "Skor Kualiti",
            "pdf_title": "Kes Penggunaan AI Strategik Dikuasakan oleh Databricks Genie Code",
            "pdf_for": "Untuk",
            "pdf_exec_summary": "Ringkasan Eksekutif",
            "pdf_toc_title": "Domain Kes Penggunaan",
            "pdf_detailed_view": "Katalog Terperinci Kes Penggunaan",
            "pdf_disclaimer_title": "Penafian",
            "pdf_fallback_summary_p1": "Dokumen ini menerangkan {total_cases} kes penggunaan analitik bernilai tinggi yang dikenal pasti untuk {business_name}.",
            "pdf_fallback_summary_p2": "Halaman-halaman berikut menyediakan perincian terperinci peluang-peluang ini, dikategorikan mengikut domain perniagaan.",
            "pptx_main_title": "Kes Penggunaan AI Strategik Dikuasakan oleh Databricks Genie Code",
            "pptx_for": "Untuk",
            "pptx_disclaimer_title": "Penafian",
            "pptx_domain_suffix": "Kes Penggunaan",
            "elite_verdict": "Keputusan penilaian",
            "elite_decision": "Keputusan",
            "elite_decision_owner": "Pemilik keputusan",
            "elite_kpi": "KPI",
            "elite_value_driver": "Pemacu nilai",
            "elite_evidence_columns": "Lajur bukti",
            "elite_join_path": "Laluan cantuman",
            "elite_build_path": "Laluan pelaksanaan",
            "elite_red_team_note": "Nota pasukan merah",
            "elite_coverage_pick": "Pilihan liputan domain, di bawah standard",
            "elite_fallback_pick": "Di bawah standard {mode}",
            "elite_fallback_near_miss": "Hampir lolos, di bawah standard {mode}",
            "doc_fallback_pick": "Di bawah standard",
            "doc_fallback_notice": "Tiada kes penggunaan yang mencapai standard {mode}. Ini ialah calon terdekat, setiap satu ditandai di bawah standard.",
            "elite_reason": "Sebab",
            "untranslated_notice": "Dipaparkan dalam bahasa Inggeris (terjemahan tidak tersedia)",
            "more_use_cases": "...dan {count} lagi kes penggunaan.",
            "executive_summary_not_available": "Ringkasan eksekutif tidak tersedia.",
            "domain_summary_not_available": "Ringkasan domain tidak tersedia.",
            "summary_not_available": "Ringkasan tidak tersedia.",
            "doc_catalog_kicker": "Katalog kes penggunaan AI",
            "doc_prepared_for": "Disediakan untuk",
            "doc_generated_on": "Dijana pada",
            "doc_quality_mode": "Mod kualiti",
            "doc_use_cases": "Kes penggunaan",
            "doc_domains": "Domain perniagaan",
            "doc_elite_use_cases": "Kes penggunaan elit",
            "doc_high_or_better": "Kualiti tinggi atau lebih baik",
            "doc_best_score": "Skor Inspire terbaik",
            "doc_not_scored": "Belum dinilai",
            "doc_coverage_picks": "Pilihan liputan",
            "doc_coverage_pick": "Pilihan liputan",
            "value_correlation_analysis": "Analisis Korelasi",
            "value_pareto_analysis": "Analisis Pareto",
            "value_funnel_analysis": "Analisis Corong",
            "value_document_processing": "Pemprosesan Dokumen",
            "value_extraction": "Pengekstrakan",
            "value_ai_analysis": "Analisis AI",
            "value_simulation": "Simulasi",
            "value_geospatial_analysis": "Analisis Geospatial",
            "value_predictive_modeling": "Pemodelan Ramalan",
            "value_clustering": "Pengelompokan",
            "value_regression_analysis": "Analisis Regresi",
            "value_survival_analysis": "Analisis Kelangsungan Hidup",
            "value_network_analysis": "Analisis Rangkaian",
            "value_mode_good": "Baik",
            "value_mode_high": "Tinggi",
            "value_mode_ultra_high": "Ultra tinggi",
            "doc_exec_kicker": "Peluang",
            "doc_guide_kicker": "Panduan membaca",
            "doc_how_to_read": "Cara membaca katalog ini",
            "doc_guide_lead": "Setiap kes penggunaan berasal daripada metadata organisasi itu sendiri dan dinilai pada satu skala mutlak. Gunakan skor untuk membandingkan, tahap untuk menapis dan halaman domain untuk menavigasi.",
            "doc_inspire_score_title": "Skor Inspire, dari 0 hingga 10",
            "doc_inspire_score_body": "Menggabungkan kualiti kes penggunaan dengan keutamaan perniagaannya. Lebih tinggi lebih baik, dan katalog disusun mengikut skor ini.",
            "doc_quality_tiers_title": "Tahap kualiti",
            "doc_quality_tiers_body": "Setiap kes penggunaan menerima tahap daripada {lowest} hingga {highest} yang mencerminkan sejauh mana ia khusus, bernilai dan boleh dibina dengan data yang tersedia.",
            "doc_modes_title": "Mod kualiti",
            "doc_mode_good_body": "Mengekalkan kes penggunaan yang dinilai {medium} atau lebih baik.",
            "doc_mode_high_body": "Hanya mengekalkan kes penggunaan yang dinilai {high} atau lebih baik.",
            "doc_mode_ultra_body": "Hanya mengekalkan kes penggunaan yang melepasi tanda aras mutlak bagi nilai, bukti, kebolehlaksanaan dan kebolehtindakan. Tiada kuota.",
            "doc_this_catalog": "Katalog ini",
            "doc_elite_score_title": "Skor elit",
            "doc_elite_score_body": "Digunakan dalam mod {ultra}. Ia menimbang nilai perniagaan, kekuatan bukti, kebolehlaksanaan dan kebolehtindakan, dan kes penggunaan elit mencapai sekurang-kurangnya {min_score} daripada 10 bagi setiap satu.",
            "doc_coverage_title": "Pilihan liputan domain",
            "doc_coverage_body": "Apabila domain perniagaan yang diminta tiada kes penggunaan melepasi tanda aras, calon terkuatnya dikekalkan dan dilabel sebagai pilihan liputan di bawah tanda aras.",
            "doc_portfolio_kicker": "Portfolio",
            "doc_portfolio_glance": "Portfolio secara ringkas",
            "doc_per_domain": "Kes penggunaan mengikut domain",
            "doc_quality_distribution": "Taburan kualiti",
            "doc_top_use_cases": "Kes penggunaan utama",
            "doc_rank": "Kedudukan",
            "doc_domain_label": "Domain",
            "doc_domain_overview": "Tentang domain ini",
            "doc_what_it_covers": "Liputannya",
            "doc_top_opportunities": "Peluang utama",
            "doc_domain_fallback": "Domain ini menghimpunkan {count} kes penggunaan merentas {subdomains}. Peluang terkuatnya disenaraikan di bawah dan diperincikan pada halaman seterusnya.",
            "doc_open_notebook": "Buka buku nota Genie",
            "doc_elite_score": "Skor elit",
            "doc_best_of_best": "Terbaik antara yang terbaik",
            "doc_appendix": "Lampiran",
            "doc_methodology": "Metodologi",
            "strategy_path": "Laluan strategi",
            "kpis": "KPI",
            "expected_key_results": "Keputusan utama yang dijangka",
            "measures_of_success": "Ukuran kejayaan",
            "financial_benefits": "Manfaat kewangan",
            "non_financial_benefits": "Manfaat bukan kewangan",
            "foundations": "Asas data",
            "doc_foundations_title": "Asas data dikongsi",
            "doc_foundations_lead": "Jadual dan cantuman yang diperlukan oleh dua atau lebih kes penggunaan. Membinanya dahulu membuka kes penggunaan yang paling banyak. Susunan ini ialah urutan, bukan jadual waktu.",
            "doc_foundation_tables": "Jadual",
            "doc_foundation_unlocks": "Kes penggunaan yang dibuka",
            "doc_data_access_standard": "Penemuan standard: Inspire tidak membaca sebarang data pelanggan. Hanya metadata, definisi paparan metrik dan kiraan penggunaan digunakan.",
            "doc_data_access_deep": "Penemuan mendalam: Inspire membaca agregat daripada {tables} jadual (bilangan baris, liputan tarikh, kadar nilai kosong, bilangan nilai unik dan senarai pendek nilai lajur bukan peribadi). Tiada baris data dihantar kepada AI.",
            "doc_method_1_deep": "Inspire membaca metadata dan, dalam penemuan mendalam, juga agregat yang dikira ke atas jadual. Ia tidak pernah menghantar baris data kepada AI.",
            "doc_guide_lead_deep": "Setiap kes penggunaan datang daripada metadata dan profil data organisasi sendiri dan diberi skor pada satu skala mutlak. Gunakan skor untuk membandingkan, peringkat untuk menapis dan halaman domain untuk menavigasi.",
            "value_not_aligned": "Tidak sejajar",
            "kpi_baseline": "nilai asas {value}, {days} hari terakhir, diukur {date}",
            "kpi_baseline_undated": "nilai asas {value}, diukur {date}",
            "doc_method_1": "Inspire hanya membaca metadata: nama jadual dan lajur, jenis, komen, kunci dan tag. Ia tidak pernah membaca data baris.",
            "doc_method_2": "Ia menyimpulkan konteks perniagaan, kemudian menjana calon kes penggunaan bagi setiap domain perniagaan yang disokong oleh data.",
            "doc_method_3": "Pendua dibuang, setiap calon dinilai berdasarkan nilai dan kebolehlaksanaan, dan get kualiti menapis idea yang lemah atau tidak disokong.",
            "doc_method_4": "Dalam mod {ultra}, hakim elit menyemak setiap calon yang tinggal, dan semakan deterministik mengesahkan bahawa setiap jadual dan lajur yang disebut wujud.",
            "doc_method_5": "Senarai akhir disusun mengikut kedudukan, dikumpulkan mengikut domain perniagaan dan, jika dipilih, disertakan dengan buku nota Databricks Genie Code.",
            "doc_glossary": "Glosari",
            "doc_gl_domain": "Bidang perniagaan yang mengumpulkan kes penggunaan berkaitan.",
            "doc_gl_subdomain": "Tema yang lebih khusus dalam domain perniagaan.",
            "doc_gl_technique": "Kaedah analitik, contohnya ramalan atau pengelasan.",
            "doc_gl_inspire": "Skor 0 hingga 10 yang menggabungkan kualiti dan keutamaan perniagaan.",
            "doc_gl_quality": "Tahap daripada {lowest} hingga {highest} yang diberikan kepada setiap kes penggunaan.",
            "doc_gl_elite": "Skor mod {ultra} bagi nilai, bukti, kebolehlaksanaan dan kebolehtindakan.",
            "doc_gl_coverage": "Kes penggunaan di bawah tanda aras yang dikekalkan supaya domain yang diminta diwakili.",
            "doc_gl_notebook": "Buku nota Databricks dengan arahan Genie Code untuk membina kes penggunaan.",
            "doc_notebook_term": "Buku nota Genie",
            "doc_next_steps": "Langkah seterusnya",
            "doc_closing_title": "Daripada kes penggunaan kepada hasil",
            "doc_closing_body": "Pilih kes penggunaan yang paling penting, buka buku nota Genie masing-masing dan bina yang pertama bersama pasukan. Langkah-langkah ini ialah susunan cadangan; pasukan yang menentukan rentaknya.",
            "doc_step_select": "Pilih",
            "doc_step_select_body": "Pilih kes penggunaan dengan skor terkuat dan penaja yang jelas.",
            "doc_step_validate": "Sahkan data",
            "doc_step_validate_body": "Sahkan jadual yang terlibat dan cantumannya bersama pemilik data.",
            "doc_step_build": "Bina dengan Genie Code",
            "doc_step_build_body": "Buka buku nota Genie dan jana versi pertama yang berfungsi.",
            "doc_step_schedule": "Jadualkan",
            "doc_step_schedule_body": "Jalankan buku nota sebagai kerja Databricks yang mengemas kini jadual hasil.",
            "doc_step_measure": "Ukur dan kembangkan",
            "doc_step_measure_body": "Jejaki nilai perniagaan yang dihasilkan, kemudian kembangkan kepada kes penggunaan seterusnya.",
            "doc_agenda": "Agenda",
            "doc_roadmap": "Peta jalan dan langkah seterusnya",
            "doc_use_case_index": "Indeks kes penggunaan",
            "doc_continued": "sambungan",
            "doc_not_provided": "Tidak diberikan",
            "doc_bob_score": "Skor Best-of-Best",
            "elite_cadence": "Kekerapan keputusan",
            "value_cadence_hourly": "Setiap jam",
            "value_cadence_daily": "Harian",
            "value_cadence_weekly": "Mingguan",
            "value_cadence_biweekly": "Setiap dua minggu",
            "value_cadence_monthly": "Bulanan",
            "value_cadence_quarterly": "Suku tahunan",
            "value_cadence_yearly": "Tahunan",
            "value_cadence_real_time": "Masa nyata",
            "value_cadence_per_event": "Bagi setiap peristiwa",
            "doc_generated_by": "Dijana oleh {brand} {version} · {date}",
            "doc_disclaimer_ai": "Kes penggunaan, skor dan naratif dalam dokumen ini dijana oleh AI. Sahkan bersama pemilik data dan penaja perniagaan sebelum sebarang keputusan dibuat.",
            "doc_disclaimer_figures": "Skor, nilai dan hasil ialah anggaran untuk penetapan keutamaan sahaja. Tiada angka dalam dokumen ini yang dijamin.",
            "doc_contents": "Kandungan",
            "doc_index_lead": "Kesemua {count} kes penggunaan mengikut kedudukan akhir.",
            "doc_deck_scope": "Pembentangan ini memperincikan {count} kes penggunaan teratas. Katalog PDF memperincikan kesemua {total}.",
            "doc_deck_scope_all": "Pembentangan ini memperincikan kesemua {total} kes penggunaan, satu kad bagi setiap kes.",
            "value_general_improvement": "Penambahbaikan Umum",
            "value_reduce_cost": "Mengurangkan Kos",
            "value_increase_revenue": "Meningkatkan Hasil",
            "value_boost_productivity": "Meningkatkan Produktiviti",
            "value_mitigate_risk": "Mengurangkan Risiko",
            "value_protect_revenue": "Melindungi Hasil",
            "value_align_to_regulations": "Mematuhi Peraturan",
            "value_improve_customer_experience": "Meningkatkan Pengalaman Pelanggan",
            "value_enable_data_driven_decisions": "Membolehkan Keputusan Berasaskan Data",
            "value_optimize_operations": "Mengoptimumkan Operasi",
            "value_empower_talent": "Memperkasakan Bakat",
            "value_enhance_experience": "Meningkatkan Pengalaman",
            "value_drive_innovation": "Memacu Inovasi",
            "value_achieve_esg": "Mencapai ESG",
            "value_execute_strategy": "Melaksanakan Strategi",
            "value_forecasting": "Ramalan",
            "value_classification": "Pengelasan",
            "value_anomaly_detection": "Pengesanan Anomali",
            "value_cohort_analysis": "Analisis Kohort",
            "value_segmentation": "Segmentasi",
            "value_sentiment_analysis": "Analisis Sentimen",
            "value_trend_analysis": "Analisis Trend",
            "value_prescriptive_analytics": "Analitik Preskriptif",
            "value_root_cause_analysis": "Analisis Punca Masalah",
            "value_optimization": "Pengoptimuman",
            "value_recommendation": "Cadangan",
            "value_time_series_analysis": "Analisis Siri Masa",
            "value_predictive_analytics": "Analitik Ramalan",
            "value_descriptive_analytics": "Analitik Deskriptif",
        },
        "Tamil": {
            "main_title": "Databricks Genie Code மூலம் இயக்கப்படும் பயன்பாட்டு வழக்கு உருவாக்கி",
            "intro": "இந்த நோட்புக் உங்கள் திட்டமைப்புகளின் அடிப்படையில் AI உருவாக்கிய பயன்பாட்டு வழக்குகளைக் கொண்டுள்ளது. கீழே வணிக துறை வாரியாக உருவாக்கப்பட்ட காட்சிகளின் சுருக்கம் உள்ளது.",
            "domain": "வணிக துறை",
            "total": "மொத்த பயன்பாட்டு வழக்குகள்",
            "summaries": "பயன்பாட்டு வழக்கு சுருக்கங்கள்",
            "sum_id": "ID",
            "sum_name": "பெயர்",
            "sum_value": "வணிக மதிப்பு",
            "sum_outcome": "எதிர்பார்க்கப்படும் முடிவு",
            "warning_header": "எச்சரிக்கை",
            "warning_body": "இந்த நோட்புக்கை இயக்க வேண்டாம். இது ஆர்ப்பாட்டம் மற்றும் பட்டியலிடல் நோக்கங்களுக்கு மட்டுமே.",
            "disclaimer": "இந்த உள்ளடக்கம் AI ஆல் உருவாக்கப்பட்டது மற்றும் உற்பத்தியில் பயன்படுத்துவதற்கு முன் தகுதியான பொறியாளரால் மதிப்பாய்வு செய்யப்பட வேண்டும். இந்த உள்ளடக்கத்தின் பயன்பாட்டினால் எழும் எந்தப் பிரச்சனைகளுக்கும் Databricks பொறுப்பல்ல.",
            "detailed_scenarios": "பயன்பாட்டு வழக்கு விவரங்கள்",
            "aspect": "அம்சம்",
            "description": "விளக்கம்",
            "aspect_domain": "வணிக துறை",
            "type": "வகை",
            "analytics_technique": "பகுப்பாய்வு நுட்பம்",
            "primary_table": "முதன்மை அட்டவணை",
            "quality": "தரம்",
            "priority": "முன்னுரிமை",
            "usecase": "பயன்பாட்டு வழக்கு",
            "description_label": "விளக்கம்",
            "value_type_problem": "சிக்கல்",
            "value_type_risk": "அபாயம்",
            "value_type_opportunity": "வாய்ப்பு",
            "value_type_improvement": "மேம்பாடு",
            "value_priority_ultra_high": "மிக மிக உயர்ந்த",
            "value_priority_very_high": "மிக உயர்ந்த",
            "value_priority_high": "உயர்ந்த",
            "value_priority_medium": "நடுத்தர",
            "value_priority_low": "குறைந்த",
            "value_priority_very_low": "மிக குறைந்த",
            "value_priority_ultra_low": "மிக மிக குறைந்த",
            "statement": "அறிக்கை",
            "solution": "தீர்வு",
            "aspect_beneficiary": "பயனாளி",
            "beneficiary": "பயனாளி",
            "aspect_sponsor": "ஆதரவாளர்",
            "sponsor": "ஆதரவாளர்",
            "business_priority_alignment": "வணிக முன்னுரிமை சீரமைப்பு",
            "strategic_goals_alignment": "மூலோபாய இலக்கு சீரமைப்பு",
            "subdomain": "துணை துறை",
            "aspect_value": "வணிக மதிப்பு",
            "business_value": "வணிக மதிப்பு",
            "aspect_tables": "தொடர்புடைய அட்டவணைகள்",
            "quality_reasons": "தர காரணங்கள்",
            "aspect_ai_function": "AI செயல்பாடு",
            "aspect_analytics_technique": "பகுப்பாய்வு நுட்பம்",
            "aspect_primary_table": "முதன்மை அட்டவணை",
            "aspect_quality": "தரம்",
            "aspect_priority": "முன்னுரிமை",
            "strategic_alignment": "மூலோபாய சீரமைப்பு",
            "return_on_investment": "முதலீட்டு வருவாய்",
            "reusability": "மறுபயன்பாட்டுத் திறன்",
            "time_to_value": "மதிப்புக்கான நேரம்",
            "data_availability": "தரவு கிடைக்கும் தன்மை",
            "data_accessibility": "தரவு அணுகல்",
            "architecture_fitness": "கட்டமைப்பு பொருத்தம்",
            "team_skills": "குழு திறன்கள்",
            "domain_knowledge": "துறை அறிவு",
            "people_allocation": "பணியாளர் ஒதுக்கீடு",
            "budget_allocation": "பட்ஜெட் ஒதுக்கீடு",
            "time_to_production": "உற்பத்திக்கான நேரம்",
            "value_score": "மதிப்பு மதிப்பெண்",
            "feasibility_score": "சாத்தியக்கூறு மதிப்பெண்",
            "priority_score": "முன்னுரிமை மதிப்பெண்",
            "inspire_score": "Inspire மதிப்பெண்",
            "quality_score_header": "தர மதிப்பெண்",
            "pdf_title": "Databricks Genie Code மூலம் இயக்கப்படும் மூலோபாய AI பயன்பாட்டு வழக்குகள்",
            "pdf_for": "இதற்காக",
            "pdf_exec_summary": "நிர்வாக சுருக்கம்",
            "pdf_toc_title": "பயன்பாட்டு வழக்கு துறைகள்",
            "pdf_detailed_view": "விரிவான பயன்பாட்டு வழக்கு பட்டியல்",
            "pdf_disclaimer_title": "மறுப்பு",
            "pdf_fallback_summary_p1": "இந்த ஆவணம் {business_name} க்காக அடையாளம் காணப்பட்ட {total_cases} உயர் மதிப்புள்ள பகுப்பாய்வு பயன்பாட்டு வழக்குகளை விவரிக்கிறது.",
            "pdf_fallback_summary_p2": "அடுத்த பக்கங்கள் வணிக துறை வாரியாக வகைப்படுத்தப்பட்ட இந்த வாய்ப்புகளின் விரிவான பகுப்பாய்வை வழங்குகின்றன.",
            "pptx_main_title": "Databricks Genie Code மூலம் இயக்கப்படும் மூலோபாய AI பயன்பாட்டு வழக்குகள்",
            "pptx_for": "இதற்காக",
            "pptx_disclaimer_title": "மறுப்பு",
            "pptx_domain_suffix": "பயன்பாட்டு வழக்குகள்",
            "elite_verdict": "தீர்ப்பு",
            "elite_decision": "முடிவு",
            "elite_decision_owner": "முடிவு உரிமையாளர்",
            "elite_kpi": "முக்கிய செயல்திறன் குறிகாட்டி",
            "elite_value_driver": "மதிப்பு இயக்கி",
            "elite_evidence_columns": "சான்று நெடுவரிசைகள்",
            "elite_join_path": "இணைப்பு பாதை",
            "elite_build_path": "செயல்படுத்தும் பாதை",
            "elite_red_team_note": "சிவப்பு அணி குறிப்பு",
            "elite_coverage_pick": "துறை உள்ளடக்கத் தேர்வு, தரத்திற்குக் கீழே",
            "elite_fallback_pick": "{mode} தரத்திற்குக் கீழே",
            "elite_fallback_near_miss": "நெருக்கமான தோல்வி, {mode} தரத்திற்குக் கீழே",
            "doc_fallback_pick": "தரத்திற்குக் கீழே",
            "doc_fallback_notice": "எந்த பயன்பாட்டு நிகழ்வும் {mode} தரத்தை எட்டவில்லை. இவை மிக நெருக்கமான தேர்வுகள், ஒவ்வொன்றும் தரத்திற்குக் கீழே எனக் குறிக்கப்பட்டுள்ளது.",
            "elite_reason": "காரணம்",
            "untranslated_notice": "ஆங்கிலத்தில் காட்டப்படுகிறது (மொழிபெயர்ப்பு கிடைக்கவில்லை)",
            "more_use_cases": "...மேலும் {count} பயன்பாட்டு வழக்குகள்.",
            "executive_summary_not_available": "நிர்வாக சுருக்கம் கிடைக்கவில்லை.",
            "domain_summary_not_available": "துறை சுருக்கம் கிடைக்கவில்லை.",
            "summary_not_available": "சுருக்கம் கிடைக்கவில்லை.",
            "doc_catalog_kicker": "AI பயன்பாட்டு வழக்குப் பட்டியல்",
            "doc_prepared_for": "இதற்காகத் தயாரிக்கப்பட்டது",
            "doc_generated_on": "உருவாக்கப்பட்ட தேதி",
            "doc_quality_mode": "தர முறை",
            "doc_use_cases": "பயன்பாட்டு வழக்குகள்",
            "doc_domains": "வணிகத் துறைகள்",
            "doc_elite_use_cases": "சிறப்புநிலை பயன்பாட்டு வழக்குகள்",
            "doc_high_or_better": "உயர் தரம் அல்லது அதற்கு மேல்",
            "doc_best_score": "சிறந்த Inspire மதிப்பெண்",
            "doc_not_scored": "மதிப்பிடப்படவில்லை",
            "doc_coverage_picks": "உள்ளடக்கத் தேர்வுகள்",
            "doc_coverage_pick": "உள்ளடக்கத் தேர்வு",
            "value_correlation_analysis": "தொடர்பு பகுப்பாய்வு",
            "value_pareto_analysis": "பரேட்டோ பகுப்பாய்வு",
            "value_funnel_analysis": "புனல் பகுப்பாய்வு",
            "value_document_processing": "ஆவண செயலாக்கம்",
            "value_extraction": "தகவல் பிரித்தெடுப்பு",
            "value_ai_analysis": "AI பகுப்பாய்வு",
            "value_simulation": "உருவகப்படுத்துதல்",
            "value_geospatial_analysis": "புவியியல் பகுப்பாய்வு",
            "value_predictive_modeling": "முன்கணிப்பு மாதிரியாக்கம்",
            "value_clustering": "கொத்தாக்கம்",
            "value_regression_analysis": "பின்னடைவு பகுப்பாய்வு",
            "value_survival_analysis": "உயிர்வாழ்வு பகுப்பாய்வு",
            "value_network_analysis": "வலையமைப்பு பகுப்பாய்வு",
            "value_mode_good": "நல்லது",
            "value_mode_high": "உயர்",
            "value_mode_ultra_high": "மிக உயர்",
            "doc_exec_kicker": "வாய்ப்பு",
            "doc_guide_kicker": "வாசிப்பு வழிகாட்டி",
            "doc_how_to_read": "இந்தப் பட்டியலை எப்படிப் படிப்பது",
            "doc_guide_lead": "ஒவ்வொரு பயன்பாட்டு வழக்கும் நிறுவனத்தின் சொந்த மெட்டாடேட்டாவிலிருந்து வருகிறது, ஒரே முழுமையான அளவுகோலில் மதிப்பிடப்படுகிறது. ஒப்பிட மதிப்பெண்களையும், வடிகட்ட நிலைகளையும், வழிசெல்ல துறைப் பக்கங்களையும் பயன்படுத்துங்கள்.",
            "doc_inspire_score_title": "Inspire மதிப்பெண், 0 முதல் 10 வரை",
            "doc_inspire_score_body": "பயன்பாட்டு வழக்கின் தரத்தை அதன் வணிக முன்னுரிமையுடன் இணைக்கிறது. அதிகமாக இருந்தால் சிறந்தது; பட்டியல் இதன்படி வரிசைப்படுத்தப்பட்டுள்ளது.",
            "doc_quality_tiers_title": "தர நிலைகள்",
            "doc_quality_tiers_body": "ஒவ்வொரு பயன்பாட்டு வழக்கும் {lowest} முதல் {highest} வரையிலான நிலையைப் பெறுகிறது; கிடைக்கும் தரவைக் கொண்டு அது எவ்வளவு குறிப்பானது, மதிப்புமிக்கது, உருவாக்கக்கூடியது என்பதை அது காட்டுகிறது.",
            "doc_modes_title": "தர முறைகள்",
            "doc_mode_good_body": "{medium} அல்லது அதற்கு மேல் மதிப்பிடப்பட்ட பயன்பாட்டு வழக்குகளை வைத்திருக்கிறது.",
            "doc_mode_high_body": "{high} அல்லது அதற்கு மேல் மதிப்பிடப்பட்ட பயன்பாட்டு வழக்குகளை மட்டும் வைத்திருக்கிறது.",
            "doc_mode_ultra_body": "மதிப்பு, சான்று, செயல்படுத்தும் தன்மை, நடைமுறைப் பயன்பாடு ஆகியவற்றில் முழுமையான தரத்தைக் கடக்கும் பயன்பாட்டு வழக்குகளை மட்டும் வைத்திருக்கிறது. ஒதுக்கீடு இல்லை.",
            "doc_this_catalog": "இந்தப் பட்டியல்",
            "doc_elite_score_title": "சிறப்புநிலை மதிப்பெண்",
            "doc_elite_score_body": "{ultra} முறையில் பயன்படுத்தப்படுகிறது. இது வணிக மதிப்பு, சான்றின் வலிமை, செயல்படுத்தும் தன்மை, நடைமுறைப் பயன்பாடு ஆகியவற்றை எடைபோடுகிறது; சிறப்புநிலை பயன்பாட்டு வழக்கு ஒவ்வொன்றிலும் 10க்கு குறைந்தது {min_score} பெறுகிறது.",
            "doc_coverage_title": "துறை உள்ளடக்கத் தேர்வுகள்",
            "doc_coverage_body": "கோரப்பட்ட வணிகத் துறையில் தரத்திற்கு மேல் எந்தப் பயன்பாட்டு வழக்கும் இல்லாதபோது, அதன் வலிமையான வேட்பாளர் வைக்கப்பட்டு, தரத்திற்குக் கீழான உள்ளடக்கத் தேர்வு எனக் குறிக்கப்படுகிறது.",
            "doc_portfolio_kicker": "தொகுப்பு",
            "doc_portfolio_glance": "தொகுப்பு ஒரு பார்வையில்",
            "doc_per_domain": "துறை வாரியான பயன்பாட்டு வழக்குகள்",
            "doc_quality_distribution": "தரப் பரவல்",
            "doc_top_use_cases": "முன்னணி பயன்பாட்டு வழக்குகள்",
            "doc_rank": "தரவரிசை",
            "doc_domain_label": "துறை",
            "doc_domain_overview": "இந்தத் துறை பற்றி",
            "doc_what_it_covers": "இது உள்ளடக்குவது",
            "doc_top_opportunities": "முக்கிய வாய்ப்புகள்",
            "doc_domain_fallback": "இந்தத் துறை {subdomains} ஆகியவற்றில் உள்ள பயன்பாட்டு வழக்குகளை ஒன்றிணைக்கிறது, மொத்தம் {count}. அதன் வலிமையான வாய்ப்புகள் கீழே பட்டியலிடப்பட்டு, அடுத்த பக்கங்களில் விரிவாக விளக்கப்பட்டுள்ளன.",
            "doc_open_notebook": "Genie நோட்புக்கைத் திறக்கவும்",
            "doc_elite_score": "சிறப்புநிலை மதிப்பெண்",
            "doc_best_of_best": "சிறந்தவற்றில் சிறந்தது",
            "doc_appendix": "பின்னிணைப்பு",
            "doc_methodology": "வழிமுறை",
            "strategy_path": "உத்தி பாதை",
            "kpis": "முக்கிய செயல்திறன் குறிகாட்டிகள்",
            "expected_key_results": "எதிர்பார்க்கப்படும் முக்கிய முடிவுகள்",
            "measures_of_success": "வெற்றியின் அளவீடுகள்",
            "financial_benefits": "நிதி நன்மைகள்",
            "non_financial_benefits": "நிதி சாரா நன்மைகள்",
            "foundations": "தரவு அடித்தளங்கள்",
            "doc_foundations_title": "பகிரப்பட்ட தரவு அடித்தளங்கள்",
            "doc_foundations_lead": "இரண்டு அல்லது அதற்கு மேற்பட்ட பயன்பாட்டு வழக்குகளுக்குத் தேவையான அட்டவணைகளும் இணைப்புகளும். இவற்றை முதலில் உருவாக்கினால் அதிகமான பயன்பாட்டு வழக்குகள் சாத்தியமாகின்றன. இந்த வரிசை ஒரு படிநிலை மட்டுமே, கால அட்டவணை அல்ல.",
            "doc_foundation_tables": "அட்டவணைகள்",
            "doc_foundation_unlocks": "சாத்தியமாகும் பயன்பாட்டு வழக்குகள்",
            "doc_data_access_standard": "நிலையான கண்டறிதல்: Inspire வாடிக்கையாளர் தரவு எதையும் படிக்கவில்லை. மெட்டாடேட்டா, அளவீட்டு காட்சி வரையறைகள் மற்றும் பயன்பாட்டு எண்ணிக்கைகள் மட்டுமே பயன்படுத்தப்பட்டன.",
            "doc_data_access_deep": "ஆழமான கண்டறிதல்: Inspire வரம்பில் உள்ள அட்டவணைகளிலிருந்து தொகுப்பு மதிப்புகளைப் படித்தது, மொத்தம் {tables} (வரிசைகளின் எண்ணிக்கை, தேதி வரம்பு, வெற்று மதிப்புகளின் விகிதம், தனித்த மதிப்புகளின் எண்ணிக்கை மற்றும் தனிப்பட்டதல்லாத நெடுவரிசைகளின் சுருக்கமான மதிப்புப் பட்டியல்கள்). எந்த தரவு வரிசையும் AI-க்கு அனுப்பப்படவில்லை.",
            "doc_method_1_deep": "Inspire மெட்டாடேட்டாவைப் படிக்கிறது; ஆழமான கண்டறிதலில் அட்டவணைகளின் மீது கணக்கிடப்பட்ட தொகுப்பு மதிப்புகளையும் படிக்கிறது. இது ஒருபோதும் தரவு வரிசைகளை AI-க்கு அனுப்புவதில்லை.",
            "doc_guide_lead_deep": "ஒவ்வொரு பயன்பாட்டு வழக்கும் நிறுவனத்தின் சொந்த மெட்டாடேட்டா மற்றும் தரவு விவரக்குறிப்புகளிலிருந்து பெறப்பட்டு ஒரே முழுமையான அளவுகோலில் மதிப்பிடப்படுகிறது. ஒப்பிட மதிப்பெண்களையும், வடிகட்ட நிலைகளையும், வழிசெலுத்த டொமைன் பக்கங்களையும் பயன்படுத்துங்கள்.",
            "value_not_aligned": "சீரமைக்கப்படவில்லை",
            "kpi_baseline": "அடிப்படை {value}, கடந்த {days} நாட்கள், {date} அன்று அளவிடப்பட்டது",
            "kpi_baseline_undated": "அடிப்படை {value}, {date} அன்று அளவிடப்பட்டது",
            "doc_method_1": "Inspire மெட்டாடேட்டாவை மட்டுமே படிக்கிறது: அட்டவணை மற்றும் நெடுவரிசைப் பெயர்கள், வகைகள், கருத்துகள், விசைகள், குறிச்சொற்கள். அது ஒருபோதும் வரிசைத் தரவைப் படிப்பதில்லை.",
            "doc_method_2": "அது வணிகச் சூழலைக் கண்டறிந்து, தரவு ஆதரிக்கும் ஒவ்வொரு வணிகத் துறைக்கும் வேட்பாளர் பயன்பாட்டு வழக்குகளை உருவாக்குகிறது.",
            "doc_method_3": "நகல்கள் நீக்கப்படுகின்றன, ஒவ்வொரு வேட்பாளரும் மதிப்பு மற்றும் சாத்தியக்கூறுக்காக மதிப்பிடப்படுகிறது, பலவீனமான அல்லது ஆதாரமற்ற யோசனைகளைத் தர வாயில்கள் நீக்குகின்றன.",
            "doc_method_4": "{ultra} முறையில், ஒரு சிறப்புநிலை நடுவர் மீதமுள்ள ஒவ்வொரு வேட்பாளரையும் மதிப்பாய்வு செய்கிறார்; குறிப்பிடப்பட்ட ஒவ்வொரு அட்டவணையும் நெடுவரிசையும் உள்ளன என்பதை உறுதியான சோதனைகள் உறுதிப்படுத்துகின்றன.",
            "doc_method_5": "இறுதிப் பட்டியல் தரவரிசைப்படுத்தப்பட்டு, வணிகத் துறை வாரியாகத் தொகுக்கப்பட்டு, தேர்ந்தெடுக்கப்பட்டால் Databricks Genie Code நோட்புக்குடன் இணைக்கப்படுகிறது.",
            "doc_glossary": "சொற்களஞ்சியம்",
            "doc_gl_domain": "தொடர்புடைய பயன்பாட்டு வழக்குகளைக் குழுவாக்கும் வணிகப் பகுதி.",
            "doc_gl_subdomain": "ஒரு வணிகத் துறைக்குள் உள்ள குறுகிய கருப்பொருள்.",
            "doc_gl_technique": "பகுப்பாய்வு முறை, எடுத்துக்காட்டாக முன்கணிப்பு அல்லது வகைப்படுத்தல்.",
            "doc_gl_inspire": "தரத்தையும் வணிக முன்னுரிமையையும் இணைக்கும் 0 முதல் 10 வரையிலான மதிப்பெண்.",
            "doc_gl_quality": "ஒவ்வொரு பயன்பாட்டு வழக்குக்கும் வழங்கப்படும் {lowest} முதல் {highest} வரையிலான நிலை.",
            "doc_gl_elite": "மதிப்பு, சான்று, செயல்படுத்தும் தன்மை, நடைமுறைப் பயன்பாடு ஆகியவற்றுக்கான {ultra} முறை மதிப்பெண்.",
            "doc_gl_coverage": "கோரப்பட்ட துறை இடம்பெறும் வகையில், தரத்திற்குக் கீழே வைக்கப்பட்ட பயன்பாட்டு வழக்கு.",
            "doc_gl_notebook": "பயன்பாட்டு வழக்கை உருவாக்க Genie Code வழிமுறைகளைக் கொண்ட Databricks நோட்புக்.",
            "doc_notebook_term": "Genie நோட்புக்",
            "doc_next_steps": "அடுத்த படிகள்",
            "doc_closing_title": "பயன்பாட்டு வழக்குகளிலிருந்து விளைவுகளுக்கு",
            "doc_closing_body": "மிக முக்கியமான பயன்பாட்டு வழக்குகளைத் தேர்ந்தெடுத்து, அவற்றின் Genie நோட்புக்குகளைத் திறந்து, குழுவுடன் சேர்ந்து முதலாவதை உருவாக்குங்கள். இந்தப் படிகள் பரிந்துரைக்கப்பட்ட வரிசை மட்டுமே; வேகத்தைக் குழுவே தீர்மானிக்கிறது.",
            "doc_step_select": "தேர்வு",
            "doc_step_select_body": "வலுவான மதிப்பெண்ணும் தெளிவான ஆதரவாளரும் உள்ள பயன்பாட்டு வழக்குகளைத் தேர்ந்தெடுக்கவும்.",
            "doc_step_validate": "தரவைச் சரிபார்க்கவும்",
            "doc_step_validate_body": "தொடர்புடைய அட்டவணைகளையும் அவற்றின் இணைப்புகளையும் தரவு உரிமையாளர்களுடன் உறுதிப்படுத்தவும்.",
            "doc_step_build": "Genie Code மூலம் உருவாக்கவும்",
            "doc_step_build_body": "Genie நோட்புக்கைத் திறந்து முதல் செயல்படும் பதிப்பை உருவாக்கவும்.",
            "doc_step_schedule": "திட்டமிடுங்கள்",
            "doc_step_schedule_body": "முடிவு அட்டவணையைப் புதுப்பிக்கும் Databricks வேலையாக நோட்புக்கை இயக்கவும்.",
            "doc_step_measure": "அளவிட்டு விரிவாக்குங்கள்",
            "doc_step_measure_body": "அது தரும் வணிக மதிப்பைக் கண்காணித்து, பின்னர் அடுத்த பயன்பாட்டு வழக்குக்கு விரிவாக்குங்கள்.",
            "doc_agenda": "நிகழ்ச்சி நிரல்",
            "doc_roadmap": "செயல்திட்டம் மற்றும் அடுத்த படிகள்",
            "doc_use_case_index": "பயன்பாட்டு வழக்குச் சுட்டி",
            "doc_continued": "தொடர்ச்சி",
            "doc_not_provided": "வழங்கப்படவில்லை",
            "doc_bob_score": "Best-of-Best மதிப்பெண்",
            "elite_cadence": "முடிவு எடுக்கும் இடைவெளி",
            "value_cadence_hourly": "ஒவ்வொரு மணி நேரமும்",
            "value_cadence_daily": "தினசரி",
            "value_cadence_weekly": "வாராந்திரம்",
            "value_cadence_biweekly": "இரண்டு வாரங்களுக்கு ஒருமுறை",
            "value_cadence_monthly": "மாதாந்திரம்",
            "value_cadence_quarterly": "காலாண்டுக்கு ஒருமுறை",
            "value_cadence_yearly": "ஆண்டுதோறும்",
            "value_cadence_real_time": "நிகழ்நேரம்",
            "value_cadence_per_event": "ஒவ்வொரு நிகழ்வுக்கும்",
            "doc_generated_by": "{brand} {version} மூலம் உருவாக்கப்பட்டது · {date}",
            "doc_disclaimer_ai": "இந்த ஆவணத்தில் உள்ள பயன்பாட்டு நிகழ்வுகள், மதிப்பெண்கள் மற்றும் விளக்கங்கள் AI மூலம் உருவாக்கப்பட்டவை. எந்த முடிவையும் எடுப்பதற்கு முன் தரவு உரிமையாளர்கள் மற்றும் வணிக ஆதரவாளர்களுடன் சரிபார்க்கவும்.",
            "doc_disclaimer_figures": "மதிப்பெண்கள், மதிப்புகள் மற்றும் விளைவுகள் முன்னுரிமை அமைப்பதற்கான மதிப்பீடுகள் மட்டுமே. இந்த ஆவணத்தில் உள்ள எந்த எண்ணும் உத்தரவாதம் அளிக்கப்படவில்லை.",
            "doc_contents": "உள்ளடக்கம்",
            "doc_index_lead": "அனைத்து பயன்பாட்டு நிகழ்வுகளும் இறுதி தரவரிசை வரிசையில், மொத்தம் {count}.",
            "doc_deck_scope": "இந்த விளக்கக்காட்சி உயர் தரவரிசையில் உள்ள பயன்பாட்டு நிகழ்வுகளை விவரிக்கிறது, {total} இல் {count}. PDF பட்டியல் அனைத்தையும் விவரிக்கிறது.",
            "doc_deck_scope_all": "இந்த விளக்கக்காட்சி அனைத்து பயன்பாட்டு நிகழ்வுகளையும் ஒவ்வொன்றுக்கும் ஒரு அட்டையுடன் விவரிக்கிறது, மொத்தம் {total}.",
            "value_general_improvement": "பொது மேம்பாடு",
            "value_reduce_cost": "செலவைக் குறைத்தல்",
            "value_increase_revenue": "வருவாயை அதிகரித்தல்",
            "value_boost_productivity": "உற்பத்தித்திறனை அதிகரித்தல்",
            "value_mitigate_risk": "அபாயத்தைக் குறைத்தல்",
            "value_protect_revenue": "வருவாயைப் பாதுகாத்தல்",
            "value_align_to_regulations": "விதிமுறைகளுக்கு இணங்குதல்",
            "value_improve_customer_experience": "வாடிக்கையாளர் அனுபவத்தை மேம்படுத்துதல்",
            "value_enable_data_driven_decisions": "தரவு அடிப்படையிலான முடிவுகளை செயல்படுத்துதல்",
            "value_optimize_operations": "செயல்பாடுகளை மேம்படுத்துதல்",
            "value_empower_talent": "திறமையை வலுப்படுத்துதல்",
            "value_enhance_experience": "அனுபவத்தை மேம்படுத்துதல்",
            "value_drive_innovation": "புதுமையை உந்துதல்",
            "value_achieve_esg": "ESG ஐ அடைதல்",
            "value_execute_strategy": "மூலோபாயத்தை செயல்படுத்துதல்",
            "value_forecasting": "முன்கணிப்பு",
            "value_classification": "வகைப்பாடு",
            "value_anomaly_detection": "அசாதாரண கண்டறிதல்",
            "value_cohort_analysis": "குழு பகுப்பாய்வு",
            "value_segmentation": "பிரிவுபடுத்தல்",
            "value_sentiment_analysis": "உணர்வு பகுப்பாய்வு",
            "value_trend_analysis": "போக்கு பகுப்பாய்வு",
            "value_prescriptive_analytics": "பரிந்துரை பகுப்பாய்வு",
            "value_root_cause_analysis": "மூல காரண பகுப்பாய்வு",
            "value_optimization": "உகப்பாக்கம்",
            "value_recommendation": "பரிந்துரை",
            "value_time_series_analysis": "நேரத் தொடர் பகுப்பாய்வு",
            "value_predictive_analytics": "முன்கணிப்பு பகுப்பாய்வு",
            "value_descriptive_analytics": "விளக்க பகுப்பாய்வு",
        },
    }

    def _apply_translation_fallbacks(self, translations: dict, target_language: str) -> dict:
        """
        Applies known fallback translations for commonly untranslated terms.
        This fixes cases where the LLM fails to translate certain terms.
        ALWAYS applies fallbacks for keys that are missing OR still have English values.
        """
        if target_language not in self.TRANSLATION_FALLBACKS:
            return translations
        
        fallbacks = self.TRANSLATION_FALLBACKS[target_language]
        english_values = list(self.ENGLISH_TRANSLATIONS.values())
        english_values_lower = [v.lower() for v in english_values]
        
        applied_count = 0
        for key, fallback_value in fallbacks.items():
            should_apply = False
            current_value = translations.get(key, None)
            
            # Apply fallback if: key is missing, value is empty, or value is still in English
            if key not in translations:
                should_apply = True
                self.logger.debug(f"Translation MISSING for '{key}', applying fallback: '{fallback_value}'")
            elif not current_value or (isinstance(current_value, str) and not current_value.strip()):
                should_apply = True
                self.logger.debug(f"Translation EMPTY for '{key}', applying fallback: '{fallback_value}'")
            elif isinstance(current_value, str) and current_value.lower() in english_values_lower:
                should_apply = True
                self.logger.debug(f"Translation still ENGLISH for '{key}': '{current_value}' → '{fallback_value}'")
            
            if should_apply:
                translations[key] = fallback_value
                applied_count += 1
        
        if applied_count > 0:
            self.logger.info(f"Applied {applied_count} fallback translations for {target_language}")
        
        return translations

    def _validate_translations(self, translations: dict, target_language: str) -> bool:
        """
        Validates that critical fields are actually translated and not left in English.
        Returns True if translations are valid, False otherwise.
        Handles cognates/loanwords that are legitimately the same across languages
        (e.g., 'Sponsor' in German/French/Italian, 'Problem' in German/Dutch/Swedish).
        """
        critical_keys = [
            "type", "subdomain", "analytics_technique", "primary_table", "priority", 
            "aspect_analytics_technique", "aspect_primary_table", "aspect_priority",
            "statement", "solution", "business_value", "beneficiary", "sponsor",
            "strategic_goals_alignment",
            "pdf_detailed_view", "pptx_domain_suffix", "domain",
            "value_type_opportunity", "value_type_problem", "value_type_risk", "value_type_improvement",
            "value_priority_very_high", "value_priority_high", "value_priority_medium", "value_priority_low", "value_priority_very_low",
        ]
        
        fallbacks = self.TRANSLATION_FALLBACKS.get(target_language, {})
        
        failed_count = 0
        for key in critical_keys:
            if key not in translations:
                continue
            value = translations[key]
            if not isinstance(value, str):
                continue
            english_value = self.ENGLISH_TRANSLATIONS.get(key, "")
            if not english_value:
                continue
            if value.lower() != english_value.lower():
                continue
            if key in fallbacks and fallbacks[key].lower() == english_value.lower():
                continue
            failed_count += 1
            self.logger.warning(f"Translation validation: Key '{key}' still has English value '{value}' for {target_language}")
        
        if failed_count > 3:
            self.logger.warning(f"Translation validation FAILED for {target_language}: {failed_count} keys still in English")
            return False
        
        self.logger.info(f"Translation validation PASSED for {target_language} ({failed_count} cognate/loanword keys accepted)")
        return True

    def get_translations(self, target_language: str) -> dict:
        """
        Gets translations for UI elements for a given language.
        Uses AI agent and caches the result.
        Supports retry logic (max 2 attempts).
        """
        if target_language == "English":
            self.logger.info("Using default English UI translations.")
            return self.ENGLISH_TRANSLATIONS
        
        with self._translation_cache_lock:
            if target_language in self.translation_cache:
                cached = self.translation_cache[target_language]
                missing_keys = set(self.ENGLISH_TRANSLATIONS.keys()) - set(cached.keys())
                if not missing_keys:
                    if self._validate_translations(cached, target_language):
                        self.logger.info(f"Using cached UI translations for {target_language}.")
                        return cached
                    else:
                        self.logger.warning(f"Cached translations for {target_language} contain English values. Forcing re-translation...")
                        del self.translation_cache[target_language]
                else:
                    self.logger.info(f"Cache for {target_language} is outdated (missing {len(missing_keys)} keys). Refreshing...")
                    del self.translation_cache[target_language]

        self.logger.debug(f"Calling LLM to get UI translations for {target_language}...")
        
        # Retry loop: up to 2 attempts
        for attempt in range(1, 3):
            try:
                if attempt > 1:
                    self.logger.info(f"Retrying UI translation for {target_language} (Attempt {attempt}/2)...")
                
                prompt_vars = {
                    "json_payload": json.dumps(self.ENGLISH_TRANSLATIONS, indent=2, ensure_ascii=False),
                    "target_language": target_language
                }
                
                self.logger.info(f"⏳ Waiting for LLM response (translating UI to {target_language})...")
                response_raw = self.ai_agent.run_worker(
                    step_name=f"Translate_UI_{target_language}_Attempt{attempt}",
                    worker_prompt_path="KEYWORDS_TRANSLATE_PROMPT",
                    prompt_vars=prompt_vars,
                    response_schema=None,
                    skip_cache=(attempt > 1)
                )
                self.logger.info(f"✅ Received LLM response, parsing translations...")
                
                response_clean = clean_json_response(response_raw)
                translated_dict = json.loads(response_clean)
                
                final_translations = self.ENGLISH_TRANSLATIONS.copy()
                final_translations.update(translated_dict) 
                
                # Apply known fallback fixes for commonly untranslated terms
                final_translations = self._apply_translation_fallbacks(final_translations, target_language)
                
                # Validate translations before caching
                if not self._validate_translations(final_translations, target_language):
                    if attempt < 2:
                        self.logger.warning(f"Translation validation failed on attempt {attempt}. Will retry...")
                        continue  # Retry
                    else:
                        # Apply fallbacks one more time before giving up
                        final_translations = self._apply_translation_fallbacks(final_translations, target_language)
                        if self._validate_translations(final_translations, target_language):
                            self.logger.info(f"Fallback translations fixed the issue. Using fallbacks.")
                            with self._translation_cache_lock:
                                self.translation_cache[target_language] = final_translations
                            return final_translations
                        self.logger.error(f"Translation validation failed on final attempt for {target_language}.")
                        return self._fallback_ui_translations(target_language)
                
                self.logger.info(f"Successfully fetched and cached UI translations for {target_language} on attempt {attempt}.")
                with self._translation_cache_lock:
                    self.translation_cache[target_language] = final_translations
                return final_translations

            except Exception as e:
                self.logger.warning(f"Failed to get UI translations for {target_language} (Attempt {attempt}/2): {get_clean_error_message(e)}")
                if attempt == 2:
                    self.logger.error(f"All retry attempts exhausted for UI translations of {target_language}.")
                    return self._fallback_ui_translations(target_language)

    def _fallback_ui_translations(self, target_language: str) -> dict:
        curated = self._apply_translation_fallbacks(dict(self.ENGLISH_TRANSLATIONS), target_language)
        if target_language in self.TRANSLATION_FALLBACKS:
            self.logger.warning(f"[TRANSLATION GAP] UI labels for {target_language} use the curated fallback translations "
                                f"because the LLM translation failed")
        else:
            self.logger.warning(f"[TRANSLATION GAP] UI labels for {target_language} are shown in English: the LLM translation "
                                f"failed and no curated fallback exists for this language")
        return curated

    _TRANSLATION_FIELDS = (
        "Name", "Business Domain", "Subdomain", "type", "Statement", "Solution",
        "Business Value", "Beneficiary", "Sponsor", "Business Priority Alignment",
        "Strategic Goals Alignment", "Measures of Success", "Financial Benefits", "Non-Financial Benefits",
    )
    _TRANSLATION_LIST_FIELDS = ("Measures of Success", "Financial Benefits", "Non-Financial Benefits")
    _TRANSLATION_COPY_FIELDS = ("Analytics Technique", "Tables Involved", "Quality")
    _COVERAGE_TRANSLATION_FIELD = ("Coverage Pick Reason", "coverage_pick_reason")
    _FALLBACK_TRANSLATION_FIELD = ("Below Bar Reason", "fallback_reason")
    _BELOW_BAR_TRANSLATION_FIELDS = (("Coverage Pick Reason", "coverage_pick_reason"), ("Below Bar Reason", "fallback_reason"))
    _DERIVED_TRANSLATION_FIELDS = (("Headline", "usecase"), ("Technique Label", "Analytics Technique Label"),
                                   ("KPI Names", "_kpi_names_translated"),
                                   ("Proposed Key Results", "_key_results_translated"))
    _DERIVED_LIST_SEPARATOR = " | "
    _TRANSLATION_COLUMN_ORDER = (
        "No", "Name", "Business Domain", "Subdomain", "type", "Analytics Technique", "Statement", "Solution",
        "Business Value", "Beneficiary", "Sponsor", "Business Priority Alignment", "Strategic Goals Alignment",
        "Tables Involved", "Quality", "Headline", "Technique Label", "Measures of Success", "Financial Benefits",
        "Non-Financial Benefits", "KPI Names", "Proposed Key Results",
    )
    _LABEL_PREFIX = re.compile(r"^\s*[^:\uff1a\s][^:\uff1a]{0,14}[:\uff1a]\s*(\S.*)$", re.S)
    _NON_LATIN_LANGUAGES = {"Arabic", "Chinese (Mandarin)", "Chinese (Simplified)", "Japanese", "Korean",
                            "Hindi", "Russian", "Ukrainian", "Tamil"}

    @staticmethod
    def _has_text(value) -> bool:
        return bool(inspire_plain_text(value))

    @classmethod
    def _derived_sources(cls, uc: dict) -> dict:
        headline = ""
        if cls._has_text(uc.get("usecase")) or cls._has_text(uc.get("Name")):
            headline = DatabricksInspire._normalize_usecase(inspire_plain_text(uc.get("usecase")), inspire_plain_text(uc.get("Name")))
        technique = inspire_plain_text(uc.get("Analytics Technique"))
        if not technique or DatabricksInspire._canonical_technique(technique):
            technique = ""
        kpi_names = [inspire_plain_text(spec.get("name")) for spec in cls._generated_kpis(uc)]
        proposed = [inspire_plain_text(kr.get("statement")) for kr in cls._proposed_key_results(uc)]
        return {"Headline": "" if headline == "N/A" else headline, "Technique Label": technique,
                "KPI Names": cls._DERIVED_LIST_SEPARATOR.join(kpi_names) if any(kpi_names) else "",
                "Proposed Key Results": cls._DERIVED_LIST_SEPARATOR.join(proposed) if any(proposed) else ""}

    @staticmethod
    def _generated_kpis(uc: dict) -> list:
        """KPIs Inspire wrote (translated); KPIs from the customer's strategy keep the customer's wording."""
        return [spec for spec in uc.get("KPIs") or [] if isinstance(spec, dict) and not spec.get("kpi_id")]

    @staticmethod
    def _proposed_key_results(uc: dict) -> list:
        return [kr for kr in uc.get("Expected Key Results") or [] if isinstance(kr, dict) and kr.get("source") == "proposed"]

    @classmethod
    def _translated_items(cls, uc: dict, key: str, count: int):
        """The translated items of a derived list column, or None when the count does not match."""
        text = str(uc.get(key) or "").strip()
        if not text or not count:
            return None
        items = [item.strip() for item in text.split(cls._DERIVED_LIST_SEPARATOR.strip())]
        return items if len(items) == count and all(items) else None

    @classmethod
    def _translatable_fields_present(cls, uc: dict) -> list:
        fields = [f for f in cls._TRANSLATION_FIELDS if cls._has_text(uc.get(f))]
        fields += [column for column, value in cls._derived_sources(uc).items() if value]
        for column, key in cls._BELOW_BAR_TRANSLATION_FIELDS:
            if cls._has_text(uc.get(key)):
                fields.append(column)
        return fields

    @classmethod
    def untranslated_copy(cls, uc: dict) -> dict:
        import copy
        merged = copy.deepcopy(uc)
        merged["_translation_status"] = "untranslated"
        merged["_untranslated_fields"] = cls._translatable_fields_present(uc)
        return merged

    @classmethod
    def _translation_payload(cls, uc: dict) -> dict:
        values = {"No": str(uc.get("No") or "").strip()}
        for field in cls._TRANSLATION_FIELDS + cls._TRANSLATION_COPY_FIELDS:
            values[field] = inspire_plain_text(uc.get(field), joiner="; " if field in cls._TRANSLATION_LIST_FIELDS else ", ")
        values.update(cls._derived_sources(uc))
        payload = {column: values.get(column, "") for column in cls._TRANSLATION_COLUMN_ORDER}
        for column, key in cls._BELOW_BAR_TRANSLATION_FIELDS:
            if cls._has_text(uc.get(key)):
                payload[column] = inspire_plain_text(uc.get(key))
        return payload

    @classmethod
    def _looks_untranslated(cls, source: str, value: str, target_language: str) -> bool:
        if target_language == "English":
            return False
        source, value = str(source or "").strip(), str(value or "").strip()
        if not value:
            return True
        if value == source and len(source) >= 20 and " " in source:
            return True
        if target_language in cls._NON_LATIN_LANGUAGES:
            prose = re.sub(r"`[^`]*`", "", value)
            if len(prose) > 10 and sum(1 for c in prose if ord(c) < 128) / len(prose) > 0.85:
                return True
        return False

    def _merge_translation(self, original: dict, row: dict, target_language: str) -> dict:
        import copy
        if row is None:
            return self.untranslated_copy(original)
        merged = copy.deepcopy(original)
        untranslated = []
        for field in self._TRANSLATION_FIELDS:
            if not self._has_text(original.get(field)):
                continue
            value = str(row.get(field) or "").strip()
            if value:
                merged[field] = value
            if (self._looks_untranslated(inspire_plain_text(original.get(field)), value, target_language)
                    and not DatabricksInspire._label_rendered(field, merged.get(field))):
                untranslated.append(field)
        derived = self._derived_sources(original)
        for column, key in self._DERIVED_TRANSLATION_FIELDS:
            if not derived[column]:
                continue
            value = str(row.get(column) or "").strip()
            labelled = self._LABEL_PREFIX.match(value)
            if labelled and ":" not in derived[column]:
                value = labelled.group(1).strip()
            if value:
                merged[key] = value
            if self._looks_untranslated(derived[column], value, target_language):
                untranslated.append(column)
        for coverage_column, coverage_key in self._BELOW_BAR_TRANSLATION_FIELDS:
            if not self._has_text(original.get(coverage_key)):
                continue
            value = str(row.get(coverage_column) or "").strip()
            if value:
                merged[coverage_key] = value
            if self._looks_untranslated(inspire_plain_text(original.get(coverage_key)), value, target_language):
                untranslated.append(coverage_column)
        merged["_translation_status"] = "partial" if untranslated else "translated"
        merged["_untranslated_fields"] = untranslated
        return merged

    def _merge_translated_batch(self, originals: list, rows: list, target_language: str) -> list:
        from collections import deque
        by_id = defaultdict(deque)
        for row in rows or []:
            by_id[str(row.get("No") or "").strip()].append(row)
        merged = []
        for original in originals:
            queue = by_id.get(str(original.get("No") or "").strip())
            merged.append(self._merge_translation(original, queue.popleft() if queue else None, target_language))
        return merged

    def translate_use_case_list(self, english_use_cases: list, target_language: str, max_parallelism: int = 20, enable_parallelization: bool = True) -> list:
        """
        Returns one entry per input use case, in input order. Each entry is a deep copy of the
        original use case with only its text fields (and the elite text fields) replaced by the
        translation, so every score, flag and identifier survives; `_translation_status`
        ("translated" | "partial" | "untranslated") and `_untranslated_fields` record any text
        that is still in English so non-English artifacts can flag it.
        
        Args:
            english_use_cases: List of use case dictionaries to translate
            target_language: Target language name
            max_parallelism: Maximum parallel workers (only used if enable_parallelization=True)
            enable_parallelization: If False, processes batches sequentially to avoid nested ThreadPoolExecutors
        """
        if target_language == "English":
            return english_use_cases
        
        if not english_use_cases:
             self.logger.warning("No use cases provided to translate.")
             return []

        self.logger.info(f"Starting translation for {target_language}... (parallelization: {'enabled' if enable_parallelization else 'disabled'})")
        
        # FIXED BATCH SIZE: 3 use cases per batch for ALL languages
        # This prevents LLM output truncation issues with long SQL queries (8-10K chars each)
        # Optimal balance between translation speed and reliability
        batch_size = 3
        
        self.logger.info(f"Translation batch sizing: {batch_size} use cases per batch (FIXED for all languages to prevent truncation)")
        
        batches = [english_use_cases[i:i + batch_size] for i in range(0, len(english_use_cases), batch_size)]
        batch_results = [None] * len(batches)
        
        if enable_parallelization:
            # Parallel processing (used when NOT called from within another ThreadPoolExecutor)
            with _SafeExecutorContext(max_workers=max_parallelism, thread_name_prefix="Translator", logger=self.logger, name="Translation") as ctx:
                futures = {ctx.submit(self._translate_batch, batch, target_language, i): i for i, batch in enumerate(batches)}
                
                total_timeout = (len(batches) * 600) // max_parallelism + 300
                self.logger.info(f"Translation timeout set to {total_timeout}s for {len(batches)} batches")
                
                try:
                    for future in ctx.completed(list(futures), timeout=total_timeout):
                        try:
                            batch_results[futures[future]] = future.result(timeout=600)
                        except concurrent.futures.TimeoutError:
                            self.logger.error(f"Translation batch timed out after 10 minutes")
                        except Exception as e:
                            self.logger.error(f"A translation future failed unexpectedly: {get_clean_error_message(e)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error(f"Overall translation timeout reached ({total_timeout}s); unfinished batches stay in English and are flagged.")
        else:
            # Sequential processing (used when called from within another ThreadPoolExecutor to avoid nesting)
            self.logger.info(f"Processing {len(batches)} translation batches sequentially to avoid nested ThreadPoolExecutors...")
            for i, batch in enumerate(batches):
                try:
                    batch_results[i] = self._translate_batch(batch, target_language, i)
                except Exception as e:
                    self.logger.error(f"Translation batch {i} failed: {get_clean_error_message(e)}")
        
        translated_use_cases = []
        for batch, result in zip(batches, batch_results):
            if result is None or len(result) != len(batch):
                translated_use_cases.extend(self.untranslated_copy(uc) for uc in batch)
            else:
                translated_use_cases.extend(result)
        untranslated = sum(1 for uc in translated_use_cases if uc.get("_translation_status") == "untranslated")
        partial = sum(1 for uc in translated_use_cases if uc.get("_translation_status") == "partial")
        if untranslated or partial:
            self.logger.warning(f"[TRANSLATION GAP] {target_language}: {untranslated} use case(s) untranslated and {partial} "
                                f"partially translated out of {len(translated_use_cases)}; the gaps are flagged in the {target_language} documents")
        self.logger.info(f"Translation completed for {target_language}.")
        return translated_use_cases

    def _translate_batch(self, use_case_batch: list, target_language: str, batch_num, attempt: int = 1, split_attempt: int = 0, allow_rerequest: bool = True) -> list:
        """
        Private method to translate a single batch of use cases. Returns merged use case copies
        (see translate_use_case_list); on any final failure the copies are flagged untranslated.
        An answer missing rows at a row boundary keeps the rows that arrived and re-requests only the
        missing rows once; a genuinely cut-off answer, or a re-request that is still incomplete, splits
        the batch (up to 3 split attempts); a response that cannot be parsed at all is retried once with
        the cache bypassed.

        Args:
            use_case_batch: List of use case dictionaries to translate
            target_language: Target language for translation
            batch_num: Batch number for logging
            attempt: Current attempt number (1 or 2)
            split_attempt: Number of times batch has been split (0-3)
            allow_rerequest: False while serving a re-request, so the missing rows are asked for only once
        """
        self.logger.debug(f"Translating batch #{batch_num} (Attempt {attempt}/2, Split {split_attempt}/3) ({len(use_case_batch)} use cases) to {target_language}...")
        try:
            batch_for_translation = [self._translation_payload(uc) for uc in use_case_batch]
            prompt_vars = {
                "json_payload": json.dumps(batch_for_translation, indent=2, ensure_ascii=False),
                "target_language": target_language
            }
            self.logger.info(f"⏳ [Batch {batch_num}] Waiting for LLM response (translating {len(use_case_batch)} use cases to {target_language})...")
            response_raw = self.ai_agent.run_worker(
                step_name=f"Translate_UseCases_{target_language}_Batch_{batch_num}_Attempt{attempt}_Split{split_attempt}",
                worker_prompt_path="USE_CASE_TRANSLATE_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None,
                skip_cache=(attempt > 1 or split_attempt > 0 or not allow_rerequest)
            )
            self.logger.info(f"✅ [Batch {batch_num}] Received LLM response, parsing translated use cases...")
            translated_by_id, missing_ids, detail = self._parse_translation_rows(response_raw, batch_for_translation, batch_num, target_language)
        except InputTooLongError as e:
            if len(use_case_batch) <= 1:
                self.logger.error(f"Batch #{batch_num} has single use case that's too large for translation. It stays in English and is flagged.")
                return [self.untranslated_copy(uc) for uc in use_case_batch]
            self.logger.warning(f"Batch #{batch_num} exceeds context limit ({str(e)[:100]}). Splitting into smaller sub-batches...")
            return self._split_translate_batch(use_case_batch, target_language, batch_num, split_attempt)
        except Exception as e:
            self.logger.warning(f"Failed to translate use case batch #{batch_num} (Attempt {attempt}/2, Split {split_attempt}/3) for {target_language}: {get_clean_error_message(e)}")
            if allow_rerequest and attempt < 2:
                self.logger.info(f"Retrying batch #{batch_num} for {target_language} (Attempt 2/2)...")
                return self._translate_batch(use_case_batch, target_language, batch_num, attempt=2, split_attempt=split_attempt)
            self.logger.error(f"All retry attempts exhausted for batch #{batch_num}. It stays in English and is flagged.")
            return [self.untranslated_copy(uc) for uc in use_case_batch]

        if not missing_ids:
            self.logger.debug(f"Successfully translated batch #{batch_num} on attempt {attempt}, split {split_attempt}.")
            return self._merge_translated_batch(use_case_batch, list(translated_by_id.values()), target_language)

        if allow_rerequest and translated_by_id and len(missing_ids) < len(use_case_batch):
            missing_set = set(missing_ids)
            missing_batch = [uc for uc in use_case_batch if str(uc.get("No") or "").strip() in missing_set]
            self.logger.warning(f"Batch #{batch_num}: {len(missing_ids)} of {len(use_case_batch)} row(s) missing"
                                f"{f' ({detail})' if detail else ''}; re-requesting only the missing row(s) once...")
            recovered = self._translate_batch(missing_batch, target_language, f"{batch_num}r", attempt=1,
                                              split_attempt=split_attempt, allow_rerequest=False)
            recovered_by_id = {str(uc.get("No") or "").strip(): uc for uc in recovered}
            merged = []
            for uc in use_case_batch:
                uc_id = str(uc.get("No") or "").strip()
                if uc_id in translated_by_id:
                    merged.append(self._merge_translation(uc, translated_by_id[uc_id], target_language))
                else:
                    merged.append(recovered_by_id.get(uc_id) or self.untranslated_copy(uc))
            return merged

        if (detail or not allow_rerequest) and len(use_case_batch) > 1 and split_attempt < 3:
            self.logger.warning(f"Batch #{batch_num} {'appears cut off' if detail else 'came back incomplete on the re-request'}; "
                                f"reducing batch size (split attempt {split_attempt + 1}/3)...")
            return self._split_translate_batch(use_case_batch, target_language, batch_num, split_attempt)
        if allow_rerequest and attempt < 2:
            self.logger.info(f"Retrying batch #{batch_num} for {target_language} (Attempt 2/2)...")
            return self._translate_batch(use_case_batch, target_language, batch_num, attempt=2, split_attempt=split_attempt)
        self.logger.error(f"All retry attempts exhausted for batch #{batch_num}. It stays in English and is flagged.")
        return [self.untranslated_copy(uc) for uc in use_case_batch]

    def _split_translate_batch(self, use_case_batch: list, target_language: str, batch_num, split_attempt: int) -> list:
        pieces = 3 if len(use_case_batch) >= 3 else 2
        size = len(use_case_batch) // pieces
        bounds = [i * size for i in range(pieces)] + [len(use_case_batch)]
        combined = []
        for index in range(pieces):
            chunk = use_case_batch[bounds[index]:bounds[index + 1]]
            if chunk:
                combined += self._translate_batch(chunk, target_language, f"{batch_num}{'abc'[index]}",
                                                  attempt=1, split_attempt=split_attempt + 1)
        self.logger.info(f"Batch #{batch_num} translated after reducing batch size (split {split_attempt + 1}); "
                         f"{sum(1 for uc in combined if uc.get('_translation_status') == 'untranslated')} use case(s) stay in English")
        return combined

    _ALIGNMENT_ANCHORS = ("Analytics Technique", "Tables Involved", "Quality")

    @classmethod
    def _misaligned_translation_row(cls, raw: dict, wanted: set, original: dict = None) -> tuple:
        anchors = {}
        for anchor in cls._ALIGNMENT_ANCHORS:
            expected = str((original or {}).get(anchor) or "").strip()
            if len(expected) >= 4:
                anchors[anchor] = CSVParser.one_of([expected])
        return CSVParser.misaligned_row(raw, required=wanted, anchors=anchors,
                                        echoes={"Analytics Technique": ("Technique Label",)})

    def _parse_translation_rows(self, csv_response: str, original_batch: list, batch_num, target_language: str) -> tuple:
        if not csv_response or len(csv_response.strip()) < 100:
            raise ValueError(f"Response is empty or too short (len={len(csv_response) if csv_response else 0})")
        lines = clean_csv_response(csv_response).split("\n")
        header_idx = next((i for i, line in enumerate(lines)
                           if re.match(r'^\s*"?No"?\s*,\s*"?Name"?\s*,', line)), None)
        if header_idx is None:
            first_line = lines[0][:200] if lines else "Empty"
            self.logger.error(f"Batch #{batch_num}: CSV header not found. First line: {first_line}")
            raise ValueError(f"Could not find CSV header. First line was: {first_line}")
        body = "\n".join(lines[header_idx:])
        parsed_rows = CSVParser.parse_csv_string(
            body,
            logger=self.logger,
            context=f"Batch #{batch_num}",
            quoting=csv.QUOTE_ALL
        )
        header = [str(k) for k in (parsed_rows[0].keys() if parsed_rows else []) if k is not None]
        expected_ids = [str(uc.get("No") or "").strip() for uc in original_batch]
        expected_set = set(expected_ids)
        originals = {str(uc.get("No") or "").strip(): uc for uc in original_batch}
        wanted = set(self._TRANSLATION_FIELDS) | {column for column, _ in self._BELOW_BAR_TRANSLATION_FIELDS} | {column for column, _ in self._DERIVED_TRANSLATION_FIELDS} | {"No"}

        def _clean(value):
            if value is None:
                return ""
            if isinstance(value, list):
                return ", ".join(str(item) for item in value).strip()
            return str(value).strip()

        from collections import Counter
        cut_index = len(parsed_rows) - 1 if CSVParser.has_open_quote(body) else None
        translated_by_id, row_ids, reasons = {}, [], Counter()
        for index, raw in enumerate(parsed_rows):
            row = {str(k): _clean(v) for k, v in raw.items() if k is not None}
            row_id = row.get("No", "")
            if row_id.lower() == "no" and row.get("Name", "") in expected_set and len(header) > 1:
                row = {header[i]: row.get(header[i + 1], "") for i in range(len(header) - 1)}
                row_id = row.get("No", "")
                row_ids.append(row_id)
                self.logger.warning(f"Batch #{batch_num}: realigned a column-shifted row for {row_id}")
            else:
                row_ids.append(row_id)
                reason, problem = self._misaligned_translation_row(raw, wanted, originals.get(row_id))
                if not reason and index == cut_index and CSVParser.cut_in_data(raw):
                    reason, problem = "cut_final_row", "ends inside an open quoted field"
                if reason and row_id in expected_set:
                    reasons[reason] += 1
                    self.logger.warning(f"[TRANSLATE-ROW malformed] Batch #{batch_num}: row {row_id} {problem}; "
                                        f"its fields would shift into the wrong columns, so it is treated as missing")
                    continue
            if not row_id or row_id not in expected_set:
                if row_id:
                    self.logger.warning(f"Batch #{batch_num}: Skipping unexpected row with ID '{row_id[:50]}' (not in batch)")
                continue
            translated_by_id.setdefault(row_id, {k: v for k, v in row.items() if k in wanted})
        CSVParser.log_misaligned(self.logger, f"Translation batch #{batch_num}", reasons, len(parsed_rows))
        missing_ids = [uc_id for uc_id in expected_ids if uc_id not in translated_by_id]
        detail = CSVParser.truncation_reason(body, parsed_rows, row_ids, wanted, expected_ids) if missing_ids else ""
        return translated_by_id, missing_ids, detail

    def _parse_translation_csv(self, csv_response: str, original_batch: list, batch_num: int, target_language: str) -> list:
        """
        Parse the CSV translation response into one row dict per expected use case, matched by
        "No" (compared as text). Only the translation columns are read; a row whose ID column holds
        the literal header "No" is realigned one column to the left. Raises ValueError when rows are
        missing (with "TRUNCATED" in the message when the response looks cut off).
        """
        try:
            translated_by_id, missing_ids, detail = self._parse_translation_rows(csv_response, original_batch, batch_num, target_language)
            expected_ids = [str(uc.get("No") or "").strip() for uc in original_batch]
            if missing_ids:
                self.logger.error(f"Batch #{batch_num}: Missing translations for IDs: {missing_ids}")
                if detail:
                    self.logger.error(f"Batch #{batch_num}: Response appears TRUNCATED: {detail}. "
                                      f"Last 100 chars: ...{(csv_response or '')[-100:].strip()}")
                    raise ValueError(f"Translation response was TRUNCATED ({detail}) - missing {len(missing_ids)} rows. Reduce batch size or simplify content.")
                raise ValueError(f"Translation missing {len(missing_ids)} expected rows: {missing_ids}")
            translated_rows = [translated_by_id[uc_id] for uc_id in expected_ids]
            self.logger.debug(f"Parsed {len(translated_rows)} translated rows from CSV for batch #{batch_num}")
            return translated_rows

        except Exception as e:
            error_msg = str(e)
            self.logger.error(f"Failed to parse translation CSV for batch #{batch_num}: {get_clean_error_message(e)}")
            snippet = csv_response[:500] if csv_response else "Empty response"
            self.logger.error(f"CSV response snippet: {snippet}")
            if "TRUNCATED" in error_msg:
                raise
            return []


# COMMAND ----------

# DBTITLE 1,Inspire
# ==============================================================================
# 2. MAIN APPLICATION CLASS (MODIFIED FOR STRICT ORDERING)
# ==============================================================================

# NOTE: Global dependency checks have been removed per user request.
# Dependencies will be checked and installed on-demand.

# ==============================================================================
# MEMORY-EFFICIENT STORAGE MANAGER
# ==============================================================================
def b17_fallback_dir(prefix: str, failed_target: str, error, logger=None) -> str:
    fallback_dir = tempfile.mkdtemp(prefix=prefix)
    message = f"[B17-FALLBACK] {failed_target} is not writable ({error}); using {fallback_dir}"
    if logger is not None:
        logger.warning(message)
    else:
        print(message)
    return fallback_dir


def b17_writable_dir(path: str, fallback_prefix: str, logger=None) -> str:
    try:
        os.makedirs(path, exist_ok=True)
        probe_fd, probe_path = tempfile.mkstemp(prefix=".inspire_probe_", dir=path)
        os.close(probe_fd)
        os.unlink(probe_path)
        return path
    except OSError as primary_err:
        return b17_fallback_dir(fallback_prefix, path, primary_err, logger)


def b17_writable_file(path: str, fallback_prefix: str, logger=None) -> str:
    try:
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "a", encoding="utf-8"):
            pass
        return path
    except OSError as primary_err:
        fallback_path = os.path.join(b17_fallback_dir(fallback_prefix, path, primary_err, logger), os.path.basename(path))
        with open(fallback_path, "a", encoding="utf-8"):
            pass
        return fallback_path


def copy_file_sequentially(src_path: str, dst_path: str) -> None:
    with open(src_path, "rb") as src, open(dst_path, "wb") as dst:
        shutil.copyfileobj(src, dst, 1024 * 1024)


class IntermediateStorageManager:
    """
    Manages file-based intermediate storage for large datasets to prevent memory explosion.
    Stores batch results on disk and provides memory-efficient iteration.
    """
    
    def __init__(self, base_path="/tmp", logger=None):
        """
        Initialize the storage manager.
        
        Args:
            base_path: Base path for temporary storage (default: /tmp)
            logger: Logger instance
        """
        self.logger = logger or logging.getLogger(self.__class__.__name__)
        self.base_path = base_path
        self.temp_dir = None
        self.batch_files = []
        self.initialized = False
        
    def initialize(self):
        """Create temporary directory for intermediate storage."""
        if not self.initialized:
            try:
                self.temp_dir = tempfile.mkdtemp(prefix="inspire_", dir=self.base_path)
            except OSError as primary_err:
                self.temp_dir = b17_fallback_dir("inspire_", self.base_path, primary_err, self.logger)
            self.logger.info(f"📁 Initialized intermediate storage at: {self.temp_dir}")
            self.initialized = True
            
    def save_batch(self, batch_num, data):
        """
        Save batch data to disk.
        
        Args:
            batch_num: Batch identifier (can be int or str)
            data: List of use case dictionaries
        """
        if not self.initialized:
            self.initialize()
            
        # Handle both int and string batch_num
        if isinstance(batch_num, int):
            batch_file = os.path.join(self.temp_dir, f"batch_{batch_num:04d}.json")
        else:
            batch_file = os.path.join(self.temp_dir, f"batch_{batch_num}.json")
        with open(batch_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        
        self.batch_files.append(batch_file)
        file_size = os.path.getsize(batch_file) / (1024 * 1024)  # Size in MB
        self.logger.debug(f"💾 Saved batch {batch_num} to disk ({len(data)} use cases, {file_size:.2f} MB)")
        
    def load_batch(self, batch_file):
        """Load a single batch from disk."""
        with open(batch_file, 'r', encoding='utf-8') as f:
            return json.load(f)
            
    def iter_batches(self):
        """
        Iterator over all batches.
        Memory-efficient: loads one batch at a time.
        """
        for batch_file in self.batch_files:
            yield self.load_batch(batch_file)
            
    def iter_all_use_cases(self):
        """
        Iterator over all use cases across all batches.
        Memory-efficient: loads one batch at a time and yields individual use cases.
        """
        for batch in self.iter_batches():
            for use_case in batch:
                yield use_case
                
    def load_all_use_cases(self):
        """
        Load all use cases into memory.
        Use this only when necessary (e.g., for deduplication).
        """
        all_use_cases = []
        for batch in self.iter_batches():
            all_use_cases.extend(batch)
        return all_use_cases
        
    def get_total_count(self):
        """Get total count of use cases without loading all into memory."""
        count = 0
        for batch in self.iter_batches():
            count += len(batch)
        return count
        
    def cleanup(self):
        """Remove temporary directory and all files."""
        if self.temp_dir and os.path.exists(self.temp_dir):
            try:
                shutil.rmtree(self.temp_dir)
                self.logger.info(f"🧹 Cleaned up intermediate storage: {self.temp_dir}")
            except Exception as e:
                self.logger.warning(f"Failed to cleanup temp directory {self.temp_dir}: {get_clean_error_message(e)}")
            finally:
                self.temp_dir = None
                self.batch_files = []
                self.initialized = False
                
    def get_stats(self):
        """Get storage statistics."""
        if not self.temp_dir or not os.path.exists(self.temp_dir):
            return {"initialized": False}
            
        total_size = 0
        for batch_file in self.batch_files:
            if os.path.exists(batch_file):
                total_size += os.path.getsize(batch_file)
                
        return {
            "initialized": True,
            "temp_dir": self.temp_dir,
            "num_batches": len(self.batch_files),
            "total_size_mb": total_size / (1024 * 1024),
            "use_case_count": self.get_total_count()
        }
    
    def save_schema_details(self, data: list, name: str = "all_columns_for_sql"):
        """Save schema column details to disk to free memory."""
        if not self.initialized:
            self.initialize()
        schema_file = os.path.join(self.temp_dir, f"{name}.json")
        serializable = [list(t) for t in data]
        with open(schema_file, 'w', encoding='utf-8') as f:
            json.dump(serializable, f, ensure_ascii=False)
        size_mb = os.path.getsize(schema_file) / (1024 * 1024)
        self.logger.debug(f"💾 Saved {name} to disk ({len(data)} rows, {size_mb:.2f} MB)")
    
    def load_schema_details(self, name: str = "all_columns_for_sql") -> list:
        """Load schema column details from disk."""
        if not self.temp_dir:
            return []
        schema_file = os.path.join(self.temp_dir, f"{name}.json")
        if not os.path.exists(schema_file):
            return []
        with open(schema_file, 'r', encoding='utf-8') as f:
            raw = json.load(f)
        return [tuple(t) for t in raw]

    def save_scored_use_cases(self, data, tag: str = "combined_scored"):
        if not self.initialized:
            self.initialize()
        path = os.path.join(self.temp_dir, f"{tag}.json")
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=1)
        self.logger.debug(f"Saved {len(data)} scored use cases to disk ({tag})")

    def load_scored_use_cases(self, tag: str = "combined_scored"):
        if not self.temp_dir:
            return None
        path = os.path.join(self.temp_dir, f"{tag}.json")
        if not os.path.exists(path):
            return None
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)

    def save_schema_markdown(self, markdown: str, tag: str = "schema_markdown"):
        if not self.initialized:
            self.initialize()
        path = os.path.join(self.temp_dir, f"{tag}.txt")
        with open(path, 'w', encoding='utf-8') as f:
            f.write(markdown)
        self.logger.debug(f"Saved schema markdown to disk ({len(markdown):,} chars)")

    def load_schema_markdown(self, tag: str = "schema_markdown"):
        if not self.temp_dir:
            return None
        path = os.path.join(self.temp_dir, f"{tag}.txt")
        if not os.path.exists(path):
            return None
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()

    def save_column_tracking(self, fqtn: str, column_names: list):
        """
        Save column tracking for a table to disk.
        
        Args:
            fqtn: Fully qualified table name (catalog.schema.table)
            column_names: List of column names that were kept
        """
        if not self.initialized:
            self.initialize()
        
        # Create column_tracking subdirectory if it doesn't exist
        tracking_dir = os.path.join(self.temp_dir, "column_tracking")
        os.makedirs(tracking_dir, exist_ok=True)
        
        # Use a safe filename (replace dots and special chars)
        safe_filename = fqtn.replace('.', '_').replace('`', '') + ".json"
        tracking_file = os.path.join(tracking_dir, safe_filename)
        
        with open(tracking_file, 'w', encoding='utf-8') as f:
            json.dump({
                "fqtn": fqtn,
                "columns": column_names,
                "timestamp": datetime.datetime.now().isoformat()
            }, f, ensure_ascii=False, indent=2)
        
        self.logger.debug(f"💾 Saved column tracking for {fqtn}: {len(column_names)} columns")
    
    def load_column_tracking(self, fqtn: str) -> list:
        """
        Load column tracking for a table from disk.
        
        Args:
            fqtn: Fully qualified table name (catalog.schema.table)
            
        Returns:
            List of column names that were kept, or None if no tracking exists
        """
        if not self.temp_dir or not os.path.exists(self.temp_dir):
            return None
        
        tracking_dir = os.path.join(self.temp_dir, "column_tracking")
        if not os.path.exists(tracking_dir):
            return None
        
        # Use a safe filename (replace dots and special chars)
        safe_filename = fqtn.replace('.', '_').replace('`', '') + ".json"
        tracking_file = os.path.join(tracking_dir, safe_filename)
        
        if not os.path.exists(tracking_file):
            return None
        
        try:
            with open(tracking_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                return data.get("columns", None)
        except Exception as e:
            self.logger.warning(f"Failed to load column tracking for {fqtn}: {get_clean_error_message(e)}")
            return None
    
    def has_column_tracking(self, fqtn: str) -> bool:
        """
        Check if column tracking exists for a table.
        
        Args:
            fqtn: Fully qualified table name (catalog.schema.table)
            
        Returns:
            True if tracking exists, False otherwise
        """
        if not self.temp_dir or not os.path.exists(self.temp_dir):
            return False
        
        tracking_dir = os.path.join(self.temp_dir, "column_tracking")
        if not os.path.exists(tracking_dir):
            return False
        
        safe_filename = fqtn.replace('.', '_').replace('`', '') + ".json"
        tracking_file = os.path.join(tracking_dir, safe_filename)
        return os.path.exists(tracking_file)
    
    def save_pass1_ids(self, use_case_ids: list):
        """Save PASS 1 use case IDs to disk for memory-efficient comparison."""
        if not self.initialized:
            self.initialize()
        ids_file = os.path.join(self.temp_dir, "pass1_ids.json")
        with open(ids_file, 'w', encoding='utf-8') as f:
            json.dump(use_case_ids, f)
        self.logger.debug(f"💾 Saved {len(use_case_ids)} PASS 1 IDs to disk")
    
    def load_pass1_ids(self) -> set:
        """Load PASS 1 use case IDs from disk as a set for O(1) lookup."""
        if not self.temp_dir or not os.path.exists(self.temp_dir):
            return set()
        ids_file = os.path.join(self.temp_dir, "pass1_ids.json")
        if not os.path.exists(ids_file):
            return set()
        try:
            with open(ids_file, 'r', encoding='utf-8') as f:
                return set(json.load(f))
        except Exception as e:
            self.logger.warning(f"Failed to load PASS 1 IDs: {get_clean_error_message(e)}")
            return set()
    
    def save_feedback_file(self, feedback_lines: list):
        """Save feedback string to disk to avoid keeping large string in memory."""
        if not self.initialized:
            self.initialize()
        feedback_file = os.path.join(self.temp_dir, "pass1_feedback.txt")
        with open(feedback_file, 'w', encoding='utf-8') as f:
            f.write("\n".join(feedback_lines))
        self.logger.debug(f"💾 Saved feedback to disk ({len(feedback_lines)} lines)")
    
    def load_feedback_file(self) -> str:
        """Load feedback string from disk."""
        if not self.temp_dir or not os.path.exists(self.temp_dir):
            return ""
        feedback_file = os.path.join(self.temp_dir, "pass1_feedback.txt")
        if not os.path.exists(feedback_file):
            return ""
        try:
            with open(feedback_file, 'r', encoding='utf-8') as f:
                return f.read()
        except Exception as e:
            self.logger.warning(f"Failed to load feedback file: {get_clean_error_message(e)}")
            return ""
    
    def iter_pass1_use_cases_for_feedback(self, limit: int = 200):
        """
        Memory-efficient iterator for building feedback.
        Yields (idx, name, tables) tuples from PASS 1 batches without loading all into memory.
        """
        idx = 0
        for batch in self.iter_batches():
            for uc in batch:
                if idx >= limit:
                    return
                idx += 1
                name = str(uc.get('Name', '')).replace('|', '-')[:80]
                tables = str(uc.get('Tables Involved', '')).replace('|', '-')[:60]
                yield (idx, name, tables)
    
    def save_domain_results(self, domain_name: str, use_cases: list):
        """Save completed domain use cases to disk to free memory during domain-by-domain processing."""
        if not self.initialized:
            self.initialize()
        domain_dir = os.path.join(self.temp_dir, "domain_results")
        os.makedirs(domain_dir, exist_ok=True)
        safe_name = re.sub(r'[^\w\-]', '_', domain_name)
        domain_file = os.path.join(domain_dir, f"{safe_name}.json")
        with open(domain_file, 'w', encoding='utf-8') as f:
            json.dump(use_cases, f, ensure_ascii=False)
        size_mb = os.path.getsize(domain_file) / (1024 * 1024)
        self.logger.debug(f"💾 Saved domain '{domain_name}' results to disk ({len(use_cases)} use cases, {size_mb:.2f} MB)")

    def load_domain_results(self, domain_name: str) -> list:
        """Load domain use cases from disk."""
        if not self.temp_dir:
            return []
        domain_dir = os.path.join(self.temp_dir, "domain_results")
        safe_name = re.sub(r'[^\w\-]', '_', domain_name)
        domain_file = os.path.join(domain_dir, f"{safe_name}.json")
        if not os.path.exists(domain_file):
            return []
        with open(domain_file, 'r', encoding='utf-8') as f:
            return json.load(f)

    def load_all_domain_results(self) -> list:
        """Load all domain results from disk in one pass."""
        if not self.temp_dir:
            return []
        domain_dir = os.path.join(self.temp_dir, "domain_results")
        if not os.path.exists(domain_dir):
            return []
        all_results = []
        for filename in sorted(os.listdir(domain_dir)):
            if filename.endswith('.json'):
                filepath = os.path.join(domain_dir, filename)
                with open(filepath, 'r', encoding='utf-8') as f:
                    all_results.extend(json.load(f))
        return all_results

    def save_phase_data(self, phase_name: str, data):
        """Save arbitrary phase data to disk for cross-phase handoff without holding in memory."""
        if not self.initialized:
            self.initialize()
        phase_file = os.path.join(self.temp_dir, f"phase_{phase_name}.json")
        with open(phase_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False)

    def load_phase_data(self, phase_name: str):
        """Load phase data from disk."""
        if not self.temp_dir:
            return None
        phase_file = os.path.join(self.temp_dir, f"phase_{phase_name}.json")
        if not os.path.exists(phase_file):
            return None
        with open(phase_file, 'r', encoding='utf-8') as f:
            return json.load(f)

    def save_id_maps(self, column_id_map: dict, id_column_map: dict, table_id_map: dict, id_table_map: dict):
        """Save column/table ID maps to disk to reduce memory footprint."""
        if not self.initialized:
            self.initialize()
        maps_file = os.path.join(self.temp_dir, "id_maps.json")
        data = {
            "column_id_map": column_id_map,
            "id_column_map": id_column_map,
            "table_id_map": table_id_map,
            "id_table_map": id_table_map
        }
        with open(maps_file, 'w', encoding='utf-8') as f:
            json.dump(data, f)
        self.logger.debug(f"💾 Saved ID maps to disk (columns: {len(column_id_map)}, tables: {len(table_id_map)})")
    
    def load_id_maps(self) -> tuple:
        """Load ID maps from disk. Returns (column_id_map, id_column_map, table_id_map, id_table_map)."""
        if not self.temp_dir or not os.path.exists(self.temp_dir):
            return {}, {}, {}, {}
        maps_file = os.path.join(self.temp_dir, "id_maps.json")
        if not os.path.exists(maps_file):
            return {}, {}, {}, {}
        try:
            with open(maps_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return (data.get("column_id_map", {}), data.get("id_column_map", {}),
                    data.get("table_id_map", {}), data.get("id_table_map", {}))
        except Exception as e:
            self.logger.warning(f"Failed to load ID maps: {get_clean_error_message(e)}")
            return {}, {}, {}, {}

class SessionMutationWriter:
    def __init__(self, spark, session_table, session_id, logger, max_retries=5, base_backoff_seconds=0.2):
        import threading, queue
        self.spark = spark
        self.session_table = str(session_table or "")
        self.session_id = int(session_id)
        self.logger = logger
        self.max_retries = int(max_retries)
        self.base_backoff_seconds = float(base_backoff_seconds)
        self._queue = queue.Queue()
        self._stop_event = threading.Event()
        self._worker = threading.Thread(target=self._run, daemon=True, name=f"SessionMutationWriter_{self.session_id}")
        self._worker.start()

    def submit(self, sql, tag=None):
        self._queue.put((str(sql or ""), str(tag or "unnamed"), None))

    def submit_and_wait(self, sql, tag=None, timeout=60.0) -> bool:
        import threading
        tag = str(tag or "unnamed")
        if not self._worker.is_alive():
            self.logger.error(f"[G19 SINGLE_WRITER STOPPED] session={self.session_id} tag={tag}: the writer thread is not running; the write was not applied")
            return False
        done = threading.Event()
        outcome = {"ok": False}
        def _cb(ok):
            outcome["ok"] = bool(ok)
            done.set()
        self._queue.put((str(sql or ""), tag, _cb))
        if not done.wait(timeout=timeout):
            self.logger.error(f"[G19 SINGLE_WRITER TIMEOUT] session={self.session_id} tag={tag} timeout_s={timeout}: the write did not complete in time")
            return False
        return outcome["ok"]

    def _is_concurrent_error(self, exc):
        low = str(exc).lower()
        if "concurrentappend" in low or "delta_concurrent_append" in low:
            return True
        if "concurrent" in low and "append" in low:
            return True
        return False

    def _run(self):
        import time
        while not self._stop_event.is_set() or not self._queue.empty():
            try:
                item = self._queue.get(timeout=0.5)
            except Exception:
                continue
            sql, tag, cb = item
            attempt = 0
            ok = False
            while True:
                attempt += 1
                try:
                    self.spark.sql(sql)
                    ok = True
                    try:
                        self.logger.info(f"[G19 SINGLE_WRITER FIRED] session={self.session_id} tag={tag} retries={attempt-1}")
                    except Exception:
                        pass
                    break
                except Exception as e:
                    if attempt >= self.max_retries or not self._is_concurrent_error(e):
                        try:
                            self.logger.warning(f"[G19 SINGLE_WRITER FAILED] session={self.session_id} tag={tag} attempts={attempt} err={str(e)[:200]}")
                        except Exception:
                            pass
                        break
                    backoff = self.base_backoff_seconds * (2 ** (attempt - 1))
                    try:
                        self.logger.info(f"[G19 SINGLE_WRITER RETRY] session={self.session_id} tag={tag} attempt={attempt} backoff_s={backoff:.2f}")
                    except Exception:
                        pass
                    time.sleep(backoff)
            if callable(cb):
                try:
                    cb(ok)
                except Exception:
                    pass
            try:
                self._queue.task_done()
            except Exception:
                pass

    def stop(self, drain_timeout=15.0):
        self._stop_event.set()
        self._worker.join(timeout=drain_timeout)
        if self._worker.is_alive():
            try:
                self.logger.warning(f"[G19 SINGLE_WRITER MAX_RETRIES] session={self.session_id} drain_timeout_s={drain_timeout} queue_remaining={self._queue.qsize()}")
            except Exception:
                pass


JOB_RUN_ENDED_STATES = ("TERMINATED", "SKIPPED", "INTERNAL_ERROR")


def current_job_run_id() -> str:
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    except Exception:
        return ""
    for name in ("idInJob", "parentRunId"):
        try:
            value = getattr(context, name)()
            if hasattr(value, "isDefined"):
                value = value.get() if value.isDefined() else ""
        except Exception:
            continue
        value = str(value or "").strip()
        if value.isdigit():
            return value
    return ""


def job_run_ended(run_id: str):
    try:
        run = WorkspaceClient().jobs.get_run(run_id=int(run_id))
    except Exception:
        return None
    state, status = getattr(run, "state", None), getattr(run, "status", None)
    life = getattr(state, "life_cycle_state", None) or getattr(status, "state", None)
    life = str(getattr(life, "value", life) or "").upper()
    if life not in JOB_RUN_ENDED_STATES:
        return None
    result = getattr(state, "result_state", None)
    end_time = getattr(run, "end_time", None)
    return {"state": str(getattr(result, "value", result) or life), "end_time": int(end_time) if end_time else None}


IMPLEMENT_TABLE = "__inspire_implement"
IMPLEMENT_STAGE = "Implement Use Cases"
IMPLEMENT_STATUSES = ("queued", "filling", "previewing", "building_outputs", "ready", "ready_with_warnings", "failed")
IMPLEMENT_DONE_STATUSES = ("ready", "ready_with_warnings")
IMPLEMENT_OUTPUTS = ("genie_space", "dashboard", "app", "alert")
IMPLEMENT_OUTPUT_LABELS = {"dashboard": "Dashboard", "genie_space": "Genie Agent", "app": "App", "alert": "Alert"}
IMPLEMENT_APP_HEALTH_OK = ("passed", "running")
IMPLEMENT_STOPPED_STATES = ("deleted", "released")
IMPLEMENT_NOTEBOOK_EDITED = "edited"
IMPLEMENT_RESULT_HISTORY = 1
IMPLEMENT_REFRESH_PARAMETER = "refresh_requested_by"
IMPLEMENT_REFRESH_DEFAULT = "schedule"
IMPLEMENT_DISMISS_REASONS = 10
IMPLEMENT_HISTORY_UPGRADE_CHOICE = ("- The notebook was built before Inspire kept the change history of the business output table: "
                                    "change its write step so it writes the run's rows only with inspire_write_result(frame, "
                                    "result_table), as the Platform Rules say, and keep every other cell that still holds.\n")
_IMPLEMENT_CALENDAR_DATE_RE = re.compile(r"\d{4}-\d{2}-\d{2}(?:[ T][0-9:.]+Z?)?")
_IMPLEMENT_NUMBER_NS = {}
exec(GENIE_NUMBER_TEXT_SOURCE, _IMPLEMENT_NUMBER_NS)
implement_number_text = _IMPLEMENT_NUMBER_NS[GENIE_NUMBER_TEXT_NAME]
IMPLEMENT_OUTPUT_ID_COLUMNS = {"dashboard": "dashboard_id", "genie_space": "genie_space_id", "app": "app_name", "alert": "alert_id"}
IMPLEMENT_CHANGE_CLASSES = ("initial", "data", "config", "schedule", "outputs", "deselect", "sharing", "retry", "rollback")
IMPLEMENT_RUN_PROFILES = ("preview", "full")
IMPLEMENT_FILTER_OPS = ("=", "!=", ">", ">=", "<", "<=", "in", "not in", "between")
IMPLEMENT_COLUMN_ROLES = ("entity_id", "event_date", "measure", "label", "segment")
IMPLEMENT_OUTPUT_OPTIONS = {"dashboard": ("dashboard_name", "dashboard_filters", "dashboard_vibes"),
                            "genie_space": ("genie_space_name", "genie_questions", "genie_space_vibes"),
                            "app": ("app_purpose", "app_access", "app_look", "app_options", "app_backend", "app_vibes"),
                            "alert": ("alert_name", "alert_tiers", "alert_operator", "alert_threshold", "alert_threshold_basis",
                                      "alert_condition_text", "alert_recipients", "alert_destinations", "alert_vibes")}
IMPLEMENT_RESTYLE_OPTIONS = {"app": ("app_look", "app_backend")}
IMPLEMENT_VIBES_MAX_CHARS = 2000
IMPLEMENT_VIBES_NOT_DONE = "vibes_not_done"
IMPLEMENT_NOTEBOOK = "notebook"
IMPLEMENT_NOTEBOOK_LABEL = "notebook"
IMPLEMENT_RETRY_TARGETS = (IMPLEMENT_NOTEBOOK,) + IMPLEMENT_OUTPUTS
IMPLEMENT_MERGE_OUTPUTS = ("dashboard", "genie_space", "app")
IMPLEMENT_MERGE_LABELS = {"dashboard": "dashboard", "genie_space": "Genie Agent", "app": "App"}
IMPLEMENT_ASSET_ID_RE = re.compile(r"^[A-Za-z0-9_-]{1,64}$")
IMPLEMENT_ACTION_TIERS = INSPIRE_PRIORITY_TIERS[:2]
IMPLEMENT_ALERT_TIER_SETS = {"act_now": IMPLEMENT_ACTION_TIERS[:1], "act_now_review": IMPLEMENT_ACTION_TIERS}
IMPLEMENT_JOIN_TAGS = tuple(f"--rt=FROM_RELATIONSHIP_TYPE_{kind}--" for kind in ("MANY_TO_ONE", "ONE_TO_MANY", "ONE_TO_ONE", "MANY_TO_MANY"))
IMPLEMENT_ALERT_OPERATORS = {">=": "GREATER_THAN_OR_EQUAL", ">": "GREATER_THAN", "=": "EQUAL", "!=": "NOT_EQUAL",
                             "<": "LESS_THAN", "<=": "LESS_THAN_OR_EQUAL"}
IMPLEMENT_ALERT_TASK_KEY = "inspire_alert"
IMPLEMENT_ALERT_FIELDS = ("display_name", "query_text", "warehouse_id", "custom_summary", "custom_description", "evaluation", "schedule")
IMPLEMENT_ALERT_MAX_SUBSCRIBERS = 100
IMPLEMENT_ALERT_THRESHOLD_BASES = ("fixed", "baseline")
IMPLEMENT_ALERT_BASELINE_DIRECTIONS = {">=": "+", ">": "+", "<=": "-", "<": "-"}
IMPLEMENT_ALERT_BASELINE_SIGMAS = 2
IMPLEMENT_ALERT_BASELINE_MIN_RUNS = 5
IMPLEMENT_ALERT_BASELINE_RUNS = 30
IMPLEMENT_ALERT_IDLE_CRON = "0 0 6 * * ?"
IMPLEMENT_GENIE_SNIPPET_KINDS = ("filters", "expressions", "measures")
IMPLEMENT_GENIE_MIN_BENCHMARKS = 3
IMPLEMENT_GENIE_MAX_SYNONYMS = 10
IMPLEMENT_GENIE_SYNONYM_CHARS = 60
IMPLEMENT_GENIE_MAX_ENTITY_COLUMNS = 120
IMPLEMENT_GENIE_FREE_TEXT_TOKENS = ("explanation",)
IMPLEMENT_MAX_GENIE_ROUNDS = 3
IMPLEMENT_LOCK_TTL_SECONDS = 4 * 3600
IMPLEMENT_UPDATE_ATTEMPTS = 5
IMPLEMENT_USE_CASE_PARAM = "inspire_implement_use_case"
IMPLEMENT_SPEC_VERSION_PARAM = "inspire_implement_spec_version"
IMPLEMENT_RUN_ID_PARAM = "inspire_implement_run_id"
IMPLEMENT_DECISION_DATASET = "decision_list"
IMPLEMENT_DECISION_TOP_K = 20
IMPLEMENT_APP_NAME_RE = re.compile(r"^[a-z][a-z0-9-]{1,28}[a-z0-9]$")
IMPLEMENT_WIDGET_NAME_RE = re.compile(r"^[A-Za-z][A-Za-z0-9_]{0,40}$")
IMPLEMENT_TABLE_NAME_RE = re.compile(r"^[A-Za-z0-9_`-]+\.[A-Za-z0-9_`-]+\.[A-Za-z0-9_`-]+$")
IMPLEMENT_PRINCIPAL_TYPES = ("user", "group", "service_principal")
IMPLEMENT_COLUMNS = (
    ("session_id", "BIGINT"), ("use_case_id", "STRING"), ("spec_version", "INT"), ("based_on_version", "INT"),
    ("spec_json", "VARIANT"), ("status", "STRING"), ("status_message", "STRING"), ("change_classes", "STRING"),
    ("notebook_path", "STRING"), ("job_id", "BIGINT"), ("last_preview_run_id", "BIGINT"), ("result_rows", "BIGINT"),
    ("dashboard_id", "STRING"), ("dashboard_url", "STRING"), ("genie_space_id", "STRING"), ("genie_space_url", "STRING"),
    ("app_name", "STRING"), ("app_url", "STRING"), ("genie_widgets", "VARIANT"), ("data_profiles", "VARIANT"),
    ("suggested_schedule", "VARIANT"), ("asset_status", "VARIANT"), ("created_by", "STRING"),
    ("created_at", "TIMESTAMP"), ("updated_at", "TIMESTAMP"), ("alert_id", "STRING"), ("alert_url", "STRING"),
    ("implement_run_id", "BIGINT"), ("asset_snapshots", "STRING"),
)
IMPLEMENT_TABLE_COMMENT = ("One row per Start Use Case PoC/MVP with Genie wizard submission: the Inspire App writes the spec and the id of the run "
                           "that builds it, the Implement Use Cases operation writes status, the built notebook, the job and the "
                           "dashboard, Genie Agent and App ids.")
GENIE_CODE_CLI_VERSION = "0.2.1"
GENIE_DATABRICKS_CLI_VERSION = "1.22.1"
GENIE_CODE_MODEL = "gpt-5.6-sol"
GENIE_CODE_TIMEOUT_S = 45 * 60
GENIE_CODE_EXIT_GRACE_S = 30
GENIE_CLI_AUTH_NOTE = ("\n\nThe databricks CLI here is already signed in through the DATABRICKS_HOST and DATABRICKS_TOKEN environment "
                       "variables (its DEFAULT profile reads them): use it as it is, and never write a token, a .databrickscfg or any "
                       "other credentials file. jq is not installed here: read the CLI's JSON output with python instead.")
GENIE_ARTIFACT_SCRATCH_FILES = ("genie_last.txt", "genie_output_schema.json")
GENIE_CODE_RELEASE_URL = "https://github.com/databricks/genie-code-cli/releases/download/v{version}/genie-package-{target}.tar.gz"
GENIE_DATABRICKS_CLI_URL = "https://github.com/databricks/cli/releases/download/v{version}/databricks_cli_{version}_linux_{arch}.zip"
GENIE_CODE_TARGETS = {"x86_64": "x86_64-unknown-linux-musl", "aarch64": "aarch64-unknown-linux-musl",
                      "arm64": "aarch64-unknown-linux-musl", "amd64": "x86_64-unknown-linux-musl"}
GENIE_CLI_ARCHES = {"x86_64": "amd64", "amd64": "amd64", "aarch64": "arm64", "arm64": "arm64"}
GENIE_DOWNLOAD_ATTEMPTS = 4
GENIE_DOWNLOAD_TIMEOUT_S = 120
GENIE_FILL_RESULT_SCHEMA = {
    "type": "object", "additionalProperties": False,
    "required": ["status", "summary", "stop_reason", "widgets", "profiles", "schedule"],
    "properties": {
        "status": {"type": "string", "enum": ["built", "stopped", "failed"]},
        "summary": {"type": "string"},
        "stop_reason": {"type": "string"},
        "widgets": {"type": "array", "items": {
            "type": "object", "additionalProperties": False, "required": ["name", "type", "default", "choices", "allowed", "label"],
            "properties": {"name": {"type": "string"}, "type": {"type": "string", "enum": ["text", "dropdown"]},
                           "default": {"type": "string"}, "choices": {"type": "array", "items": {"type": "string"}},
                           "allowed": {"type": "string"}, "label": {"type": "string"}}}},
        "profiles": {"type": "object", "additionalProperties": False, "required": ["training", "full_load"],
                     "properties": {name: {"type": "object", "additionalProperties": False,
                                           "required": ["description", "rows_scope", "window"],
                                           "properties": {"description": {"type": "string"}, "rows_scope": {"type": "string"},
                                                          "window": {"type": "string"}}}
                                    for name in ("training", "full_load")}},
        "schedule": {"type": "object", "additionalProperties": False, "required": ["cron", "timezone", "reason"],
                     "properties": {"cron": {"type": "string"}, "timezone": {"type": "string"}, "reason": {"type": "string"}}},
    },
}
GENIE_ARTIFACT_RESULT_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["status", "files", "summary"],
    "properties": {"status": {"type": "string", "enum": ["written", "failed"]},
                   "files": {"type": "array", "items": {"type": "string"}}, "summary": {"type": "string"}},
}
_IMPLEMENT_WRITE_SQL_RE = re.compile(r"\b(INSERT\s+INTO|UPDATE\s+[`\w.]+\s+SET|DELETE\s+FROM|MERGE\s+INTO|DROP\s+(TABLE|VIEW|SCHEMA|CATALOG)|"
                                     r"CREATE\s+(OR\s+REPLACE\s+)?(TABLE|VIEW|SCHEMA|CATALOG)|ALTER\s+TABLE|TRUNCATE\s+TABLE|"
                                     r"GRANT\s+\w+|REVOKE\s+\w+)\b", re.I)
_IMPLEMENT_NAME_PART = r"`?([A-Za-z_][\w-]*)`?\.`?([A-Za-z_][\w-]*)`?\.`?([A-Za-z_][\w-]*)`?"
_IMPLEMENT_TABLE_REF_RE = re.compile(r"(?:\b(?:FROM|JOIN|TABLE|INTO|UPDATE)\s+" + _IMPLEMENT_NAME_PART + r"(?![\w`.])(?!\s+import\b)"
                                     r"|[\"']" + _IMPLEMENT_NAME_PART + r"[\"'])", re.I)


def implement_table_create_sql(implement_table: str) -> str:
    columns = ", ".join(f"`{name}` {kind} COMMENT {inspire_sql_literal(_semunity_column_comment(IMPLEMENT_TABLE, name))}"
                        for name, kind in IMPLEMENT_COLUMNS)
    return (f"CREATE TABLE IF NOT EXISTS {implement_table} ({columns}) USING DELTA "
            f"CLUSTER BY (`session_id`, `use_case_id`) COMMENT {inspire_sql_literal(IMPLEMENT_TABLE_COMMENT)}")


def implement_removed_table_problems(spec: dict, involved, primary: str = "") -> list:
    removed = [_implement_plain_table(t) for t in ((spec or {}).get("data") or {}).get("removed_tables") or []]
    if not removed:
        return []
    gone = {t.lower() for t in removed}
    known = {_implement_plain_table(t).lower() for t in involved or ()}
    primary = _implement_plain_table(primary)
    problems = []
    if primary and primary.lower() in gone:
        problems.append(f"{primary} is the primary table of this use case; swap it instead of removing it")
    strangers = [t for t in removed if t.lower() not in known]
    if strangers:
        problems.append(f"{', '.join(strangers)} {'is' if len(strangers) == 1 else 'are'} not among the tables of this use case")
    if known and known <= gone:
        problems.append("every table of this use case was removed; keep at least one")
    return problems


IMPLEMENT_PERSON_NAME_TOKENS = ("name", "email", "phone", "address")


def implement_column_role_problems(roles: dict, evidence: dict, dated: dict, trains_model: bool = True,
                                   free_text_tokens=()) -> list:
    evidence = {str(k).lower(): v for k, v in (evidence or {}).items() if isinstance(v, dict)}
    dated = {(str(t).lower(), str(c).lower()) for t, c in (dated or {})}
    text_tokens = set(free_text_tokens) | set(IMPLEMENT_PERSON_NAME_TOKENS)
    problems = []
    for role, ref in sorted((roles or {}).items()):
        parts = str(ref or "").strip().strip("`").split(".")
        if len(parts) != 4 or (role == "label" and not trains_model):
            continue
        table, column = ".".join(parts[:3]).lower(), parts[3].lower()
        if table not in evidence:
            continue
        columns = {str(k).lower(): v for k, v in (evidence[table].get("columns") or {}).items() if isinstance(v, dict)}
        if column not in columns:
            problems.append(f"{ref} is not a column of {'.'.join(parts[:3])}")
            continue
        kind = str(columns[column].get("type") or "unknown type").upper()
        family = _metadata_type_family(kind)
        date_key = (table, column) in dated
        key_like = _is_key_like_column(column)
        free_text = family == "string" and bool(text_tokens & set(_name_tokens(column)))
        shown = f"{ref} ({kind}{', a date key' if date_key else ''})"
        label = role.replace("_", " ")
        if role == "entity_id" and (date_key or family in ("temporal", "boolean") or (family == "numeric" and not key_like)):
            problems.append(f"{shown} cannot be the entity id: a date, a date key or a measure is not the entity the decision acts on")
        elif role == "event_date" and not (family == "temporal" or date_key):
            problems.append(f"{shown} cannot be the event date: it is neither a DATE or TIMESTAMP column nor a date key that joins "
                            "a date table")
        elif role == "measure" and (family != "numeric" or key_like or date_key):
            problems.append(f"{shown} cannot be the measure: a measure is a numeric column that is not a key")
        elif role == "label" and (family == "temporal" or date_key or key_like or free_text):
            problems.append(f"{shown} cannot be the {label}: an outcome is a flag, a number or a category, never a date, a key, "
                            "a name or free text")
        elif role == "segment" and (family in ("temporal", "other") or date_key or free_text):
            problems.append(f"{shown} cannot be the {label}: a segment is a category column, never a date or free text")
    return problems


def implement_stop_message(outcome: dict, spec: dict, trains_model: bool = True) -> str:
    outcome = outcome if isinstance(outcome, dict) else {}
    if outcome.get("denied"):
        return f"the preview run was denied a privilege Genie cannot fix in the notebook: {outcome['denied']}"
    reason = str(outcome.get("stop_reason") or outcome.get("error") or outcome or "no reason given")
    by_genie = outcome.get("status") == "stopped" and outcome.get("stopped_by") == "genie"
    text = f"{'Genie stopped the build' if by_genie else 'the preview run did not write usable rows'}: {reason}"
    if outcome.get("not_a_stop"):
        text += (f". {'Inspire sent it back to Genie, and it still stopped' if outcome.get('sent_back') else 'Build the notebook again so Genie fixes it'}"
                 f": {str(outcome['not_a_stop'])[:600]}")
    roles = [(role, ref) for role, ref in sorted((((spec or {}).get("data") or {}).get("column_roles") or {}).items())
             if ref and (role != "label" or trains_model)]
    if outcome.get("status") == "stopped" and roles:
        chosen = ", ".join(f"{role.replace('_', ' ')} = {ref}" for role, ref in roles)
        text += (f". The run used the column roles set in the wizard ({chosen}) in place of the use case's own; if they do not "
                 "fit, set them back with Reset roles to Inspire's choice in the Data step")
    return text


_IMPLEMENT_TABLE_GONE_RE = re.compile(r"\b(TABLE_OR_VIEW_NOT_FOUND|SCHEMA_NOT_FOUND|CATALOG_NOT_FOUND|NO_SUCH_CATALOG_EXCEPTION)\b"
                                      r"|cannot be found|was not found", re.IGNORECASE)
_IMPLEMENT_TABLE_DENIED_RE = re.compile(r"\b(INSUFFICIENT_PERMISSIONS|PERMISSION_DENIED)\b|does not have \w+( \w+)? (privilege )?on",
                                        re.IGNORECASE)


def implement_source_tables(spec: dict, involved) -> list:
    data = (spec or {}).get("data") or {}
    tables = data.get("tables") if isinstance(data.get("tables"), dict) else {}
    swaps = {_implement_plain_table(original).lower(): _implement_plain_table(chosen) for original, chosen in tables.items()}
    removed = data.get("removed_tables") if isinstance(data.get("removed_tables"), list) else []
    gone = {_implement_plain_table(t).lower() for t in removed}
    out = []
    for table in involved or ():
        key = _implement_plain_table(table).lower()
        if key in gone:
            continue
        chosen = swaps.get(key) or _implement_plain_table(table)
        if chosen.lower() not in {t.lower() for t in out}:
            out.append(chosen)
    return out


def implement_table_access_reason(table: str, message: str) -> str:
    text = str(message or "")
    catalog = _implement_plain_table(table).split(".")[0].lower()
    if not catalog or catalog not in re.sub(r"[`'\"]", "", text).lower():
        return ""
    if _IMPLEMENT_TABLE_GONE_RE.search(text):
        return "not found"
    if _IMPLEMENT_TABLE_DENIED_RE.search(text):
        return "no read access"
    return ""


_IMPLEMENT_DENIAL_RE = re.compile(
    r"(?:(?:Principal|User)\s+(?:'(?P<who>[^']+)'\s+)?)?does not have\s+(?:permission\s+)?(?P<what>[A-Z][A-Z_ ]*?)\s+(?:privilege\s+)?on\s+"
    r"(?P<kind>\w+)\s+(?:'(?P<quoted>[^']+)'|(?P<ticked>`[^`]+`(?:\.`[^`]+`)*)|(?P<plain>[\w.]+))", re.IGNORECASE)


def implement_owned_objects(spec: dict, involved=()) -> set:
    config = (spec or {}).get("config") or {}
    result_table = str(config.get("result_table") or "")
    model = str((config.get("widgets") or {}).get("model_name") or "").strip()
    if not model and result_table:
        try:
            model = inspire_output_names(result_table)["model_name"]
        except ValueError:
            model = ""
    owned = set()
    for name in [result_table, model] + implement_source_tables(spec, involved):
        parts = [part.lower() for part in _implement_plain_table(name).split(".") if part]
        owned.update(".".join(parts[:size]) for size in range(1, len(parts) + 1))
    return owned


def implement_privilege_denial(error, owned) -> str:
    text = str(error or "")
    found = _IMPLEMENT_DENIAL_RE.search(text) if _IMPLEMENT_TABLE_DENIED_RE.search(text) else None
    if not found:
        return ""
    target = _implement_plain_table(found.group("quoted") or found.group("ticked") or found.group("plain").rstrip("."))
    if target.lower() not in owned:
        return ""
    who = found.group("who") or "the run identity"
    what = " ".join(found.group("what").upper().split())
    kind = found.group("kind").lower()
    return (f"{who} does not have {what} on {kind} {target}, and no change to the notebook can grant it: grant {what} on "
            f"{kind} {target} to {who}, then build again")


def implement_missing_tables_message(missing: list) -> str:
    return ("the preview cannot run because these source tables are missing or not readable: "
            + ", ".join(f"{m['table']} ({m['reason']})" for m in missing)
            + ". Open Edit, Data step, to swap each for a table with the same role or remove it")


def implement_settled_message(row: dict) -> str:
    version, use_case, status = row.get("spec_version"), row.get("use_case_id"), row.get("status")
    if status == "failed":
        return (f"The use case PoC/MVP already ran spec version {version} of {use_case} and it failed: "
                f"{row.get('status_message') or 'no error was recorded'}. Submit the wizard again to build a new version.")
    return (f"Spec version {version} of {use_case} is {status}: another run is building it, or the run that was building it "
            "stopped before it finished. Submit the wizard again to build a new version.")


def implement_lock_key(business_name: str, use_case_id: str) -> str:
    return f"implement::{str(business_name or '').strip()}::{str(use_case_id or '').strip()}"


def implement_notebook_edited(modified_at, recorded, rewrite_ended_at=None) -> bool:
    if modified_at is None or not isinstance(recorded, dict) or recorded.get("modified_at") is None:
        return False
    if int(modified_at) == int(recorded["modified_at"]):
        return False
    return not (recorded.get("rewriting") and rewrite_ended_at is not None and int(modified_at) <= int(rewrite_ended_at))


def implement_notebook_edited_message(path: str) -> str:
    return (f"the notebook {path} was edited after Inspire last built it, so this version did not rebuild it: rebuild it with "
            "Edit the PoC/MVP on the Build page to overwrite those edits, or keep them and change only the outputs or the schedule")


IMPLEMENT_SPEC_SPACE = " \t\n\v\f\r\u00a0\u1680\u2000\u2001\u2002\u2003\u2004\u2005\u2006\u2007\u2008\u2009\u200a\u2028\u2029\u202f\u205f\u3000\ufeff"
IMPLEMENT_SAFE_WHOLE = 2 ** 53 - 1


def _implement_plain_table(name) -> str:
    return ".".join(part.strip(IMPLEMENT_SPEC_SPACE).strip("`") for part in str(name or "").split("."))


def _implement_js_number(value) -> str:
    try:
        number = float(value)
    except OverflowError:
        return "null"
    if number != number or number in (float("inf"), float("-inf")):
        return "null"
    if number == 0:
        return "0"
    _, digits, exponent = decimal.Decimal(repr(abs(number))).normalize().as_tuple()
    text, point = "".join(map(str, digits)), len(digits) + exponent
    if len(text) <= point <= 21:
        shown = text + "0" * (point - len(text))
    elif 0 < point <= 21:
        shown = f"{text[:point]}.{text[point:]}"
    elif -6 < point <= 0:
        shown = "0." + "0" * -point + text
    else:
        shown = text[0] + (f".{text[1:]}" if len(text) > 1 else "") + f"e{'+' if point > 0 else '-'}{abs(point - 1)}"
    return ("-" if number < 0 else "") + shown


def _implement_json_text(value) -> str:
    if value is None or isinstance(value, bool):
        return json.dumps(value)
    if isinstance(value, (int, float)):
        return _implement_js_number(value)
    if isinstance(value, (list, tuple)):
        return "[" + ",".join(_implement_json_text(v) for v in value) + "]"
    if isinstance(value, dict):
        items = sorted(((str(k), v) for k, v in value.items()), key=lambda kv: kv[0].encode("utf-16-be", "surrogatepass"))
        return "{" + ",".join(f"{json.dumps(k, ensure_ascii=False)}:{_implement_json_text(v)}" for k, v in items) + "}"
    return json.dumps(str(value), ensure_ascii=False)


def _implement_shown(value) -> str:
    return f"'{value}'" if isinstance(value, str) else _implement_json_text(value)


def _implement_text_value(value):
    if value is None:
        return ""
    return value.strip(IMPLEMENT_SPEC_SPACE) if isinstance(value, str) else None


def _implement_text(value, field: str, errors: list) -> str:
    text = _implement_text_value(value)
    if text is None:
        errors.append(f"{field} must be text, not {_implement_shown(value)}")
        return ""
    return text


def _implement_table_text(value):
    text = _implement_text_value(value)
    return None if text is None else _implement_plain_table(text)


def _implement_flag(value, field: str, errors: list, default: bool = False) -> bool:
    if value is None:
        return default
    if isinstance(value, bool):
        return value
    errors.append(f"{field} must be true or false, not {_implement_shown(value)}")
    return default


def _implement_value_text(value):
    if isinstance(value, str):
        return value
    whole = implement_whole_number(value) if isinstance(value, (int, float)) else None
    return None if whole is None else str(whole)


def _implement_list(value, field: str, errors: list) -> list:
    if value is None or value == "":
        return []
    if isinstance(value, list):
        return value
    errors.append(f"{field} must be a list")
    return []


def _implement_mapping(value, field: str, errors: list) -> dict:
    if value is None or value == "":
        return {}
    if isinstance(value, dict):
        return value
    errors.append(f"{field} must be an object")
    return {}


_IMPLEMENT_DECIMAL_RE = re.compile(rf"[{IMPLEMENT_SPEC_SPACE}]*[-+]?([0-9]+(\.[0-9]*)?|\.[0-9]+)([eE][-+]?[0-9]+)?[{IMPLEMENT_SPEC_SPACE}]*")
_IMPLEMENT_WHOLE_RE = re.compile(rf"[{IMPLEMENT_SPEC_SPACE}]*[-+]?[0-9]+[{IMPLEMENT_SPEC_SPACE}]*")


def implement_whole_number(value):
    if isinstance(value, bool):
        return None
    if isinstance(value, int):
        number = value
    elif isinstance(value, float):
        number = int(value) if value.is_integer() else None
    else:
        text = value if isinstance(value, str) else ""
        number = int(text.strip(IMPLEMENT_SPEC_SPACE)) if _IMPLEMENT_WHOLE_RE.fullmatch(text) else None
    return number if number is not None and abs(number) <= IMPLEMENT_SAFE_WHOLE else None


def implement_decimal(value):
    if isinstance(value, bool):
        return None
    try:
        if isinstance(value, (int, float)):
            number = float(value)
        elif isinstance(value, str) and _IMPLEMENT_DECIMAL_RE.fullmatch(value):
            number = float(value.strip(IMPLEMENT_SPEC_SPACE))
        else:
            return None
    except OverflowError:
        return None
    return number if number == number and number not in (float("inf"), float("-inf")) else None


_IMPLEMENT_CRON_FIELDS = (("second", 0, 59, ()), ("minute", 0, 59, ()), ("hour", 0, 23, ()), ("day of month", 1, 31, ()),
                          ("month", 1, 12, ("JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC")),
                          ("day of week", 1, 7, ("SUN", "MON", "TUE", "WED", "THU", "FRI", "SAT")), ("year", 1970, 2099, ()))


def _implement_cron_value_problem(part: str, label: str, low: int, high: int, names: tuple) -> str:
    if part in names or (re.fullmatch(r"[0-9]+", part) and low <= int(part) <= high):
        return ""
    return f"the {label} value '{part}' is not {low} to {high}" + (f" or {names[0]} to {names[-1]}" if names else "")


def implement_cron_problem(cron) -> str:
    fields = [part for part in re.split(f"[{IMPLEMENT_SPEC_SPACE}]+", str(cron or "").upper()) if part]
    if len(fields) not in (6, 7):
        return (f"it has {len(fields)} fields; a Quartz cron has 6 or 7: second, minute, hour, day of month, month, day of week "
                "and an optional year")
    for text, (label, low, high, names) in zip(fields, _IMPLEMENT_CRON_FIELDS):
        if text == "?" and label in ("day of month", "day of week"):
            continue
        last = re.fullmatch(r"L(?:-([0-9]+))?|LW|([0-9]+)W", text) if label == "day of month" else None
        if last:
            if last.group(1) is not None and not 1 <= int(last.group(1)) <= 30:
                return f"the day of month offset '{last.group(1)}' is not 1 to 30"
            if last.group(2) is not None:
                problem = _implement_cron_value_problem(last.group(2), label, low, high, names)
                if problem:
                    return problem
            continue
        nth = re.fullmatch(r"L|([0-9]+|[A-Z]+)(?:L|#([0-9]+))", text) if label == "day of week" else None
        if nth:
            problem = _implement_cron_value_problem(nth.group(1), label, low, high, names) if nth.group(1) else ""
            if problem:
                return problem
            if nth.group(2) is not None and not 1 <= int(nth.group(2)) <= 5:
                return f"the day of week occurrence '{nth.group(2)}' is not 1 to 5"
            continue
        for item in text.split(","):
            base, slash, step = item.partition("/")
            top = 129 if label == "year" else high
            if slash and not (re.fullmatch(r"[0-9]+", step) and 1 <= int(step) <= top):
                return f"the {label} step '{step}' is not 1 to {top}"
            if base == "*":
                continue
            start, dash, end = base.partition("-")
            for part in (start, end) if dash else (start,):
                problem = _implement_cron_value_problem(part, label, low, high, names)
                if problem:
                    return problem
    if (fields[3] == "?") == (fields[5] == "?"):
        return "exactly one of day of month and day of week must be '?'"
    return ""


_IMPLEMENT_EMAIL_RE = re.compile(rf"[^@{IMPLEMENT_SPEC_SPACE}]+@[^@{IMPLEMENT_SPEC_SPACE}]+\.[^@{IMPLEMENT_SPEC_SPACE}]+")
_IMPLEMENT_ROLE_REF_RE = re.compile(rf"[^.{IMPLEMENT_SPEC_SPACE}`]+(\.[^.{IMPLEMENT_SPEC_SPACE}`]+){{3}}")
_IMPLEMENT_DATE_RE = re.compile(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")


def _implement_texts(values, field: str, errors: list) -> list:
    texts = (_implement_text(item, f"each item of {field}", errors) for item in _implement_list(values, field, errors))
    return [text for text in texts if text]


def validate_implement_spec(spec) -> tuple:
    errors = []
    if isinstance(spec, str):
        try:
            spec = json.loads(spec)
        except ValueError as e:
            return {}, [f"the spec is not valid JSON: {e}"]
    if not isinstance(spec, dict):
        return {}, ["the spec must be a JSON object"]
    out = {"use_case_id": _implement_text(spec.get("use_case_id"), "use_case_id", errors),
           "use_case_name": _implement_text(spec.get("use_case_name"), "use_case_name", errors),
           "confirm_notebook_overwrite": _implement_flag(spec.get("confirm_notebook_overwrite"), "confirm_notebook_overwrite", errors)}
    if _implement_text_value(spec.get("use_case_id")) == "":
        errors.append("use_case_id is required")
    based_on = spec.get("based_on_version")
    if based_on not in (None, ""):
        if implement_whole_number(based_on) is None:
            errors.append(f"based_on_version {_implement_shown(based_on)} must be an integer")
        else:
            out["based_on_version"] = implement_whole_number(based_on)
    framing = _implement_mapping(spec.get("use_case"), "use_case", errors)
    measures = _implement_texts(framing.get("success_measures"), "use_case.success_measures", errors)
    if len(measures) > 3:
        errors.append("at most 3 success measures are allowed")
    out["use_case"] = {"goal": _implement_text(framing.get("goal"), "use_case.goal", errors)[:500],
                       "decision_owner": _implement_text(framing.get("decision_owner"), "use_case.decision_owner", errors)[:200],
                       "success_measures": measures[:3]}
    data = _implement_mapping(spec.get("data"), "data", errors)
    tables = {}
    for original, chosen in _implement_mapping(data.get("tables"), "data.tables", errors).items():
        o, c = _implement_table_text(original), _implement_table_text(chosen)
        if o is None or c is None or not IMPLEMENT_TABLE_NAME_RE.fullmatch(o) or not IMPLEMENT_TABLE_NAME_RE.fullmatch(c):
            errors.append(f"table swap {_implement_shown(original)} -> {_implement_shown(chosen)} must name two catalog.schema.table tables")
            continue
        if o.lower() != c.lower():
            tables[o] = c
    filters = []
    for item in _implement_list(data.get("filters"), "data.filters", errors):
        if not isinstance(item, dict):
            errors.append("each data filter must be an object")
            continue
        table, column, op = _implement_table_text(item.get("table")), _implement_text_value(item.get("column")), _implement_text_value(item.get("op"))
        op = None if op is None else op.lower()
        raw_values = item.get("values")
        raw_values = raw_values if isinstance(raw_values, list) else ([] if raw_values is None or raw_values == "" else [raw_values])
        values = [_implement_value_text(v) for v in raw_values]
        bad = [raw for raw, value in zip(raw_values, values) if value is None]
        if (table is None or column is None or not IMPLEMENT_TABLE_NAME_RE.fullmatch(table)
                or not IMPLEMENT_WIDGET_NAME_RE.fullmatch(column.replace("-", "_"))):
            errors.append(f"filter on {_implement_shown(item.get('table'))}.{_implement_shown(item.get('column'))} must name a "
                          "catalog.schema.table table and a column")
        elif op not in IMPLEMENT_FILTER_OPS:
            errors.append(f"filter operator {_implement_shown(item.get('op'))} is not one of {', '.join(IMPLEMENT_FILTER_OPS)}")
        elif bad:
            errors.append(f"filter {table}.{column} value {_implement_shown(bad[0])} must be text or a whole number")
        elif not values or (op == "between" and len(values) != 2) or (op in ("=", "!=", ">", ">=", "<", "<=") and len(values) != 1):
            errors.append(f"filter {table}.{column} {op} needs " + ("two values" if op == "between" else "one value" if op not in ("in", "not in") else "at least one value"))
        else:
            filters.append({"table": table, "column": column, "op": op, "values": values})
    profile = _implement_text_value(data.get("run_profile"))
    profile = "" if profile is None else (profile or "preview").lower()
    if profile not in IMPLEMENT_RUN_PROFILES:
        errors.append(f"run_profile must be one of {', '.join(IMPLEMENT_RUN_PROFILES)}")
    roles = {}
    for role, raw in _implement_mapping(data.get("column_roles"), "data.column_roles", errors).items():
        ref = _implement_text_value(raw)
        ref = None if ref is None else ref.strip("`")
        if role not in IMPLEMENT_COLUMN_ROLES:
            errors.append(f"column role {_implement_shown(role)} is not one of {', '.join(IMPLEMENT_COLUMN_ROLES)}")
        elif ref is None or (ref and not _IMPLEMENT_ROLE_REF_RE.fullmatch(ref)):
            errors.append(f"column role {role} must name catalog.schema.table.column, not {_implement_shown(raw if ref is None else ref)}")
        elif ref:
            roles[role] = ref
    removed_tables = []
    for raw in _implement_list(data.get("removed_tables"), "data.removed_tables", errors):
        table = _implement_table_text(raw)
        if table is None or not IMPLEMENT_TABLE_NAME_RE.fullmatch(table):
            errors.append(f"removed table {_implement_shown(raw)} must be catalog.schema.table")
        elif any(table.lower() == o.lower() for o in tables):
            errors.append(f"{table} cannot be swapped and removed at once")
        elif table.lower() not in [r.lower() for r in removed_tables]:
            removed_tables.append(table)
    gone = {r.lower() for r in removed_tables}
    for role, ref in sorted(roles.items()):
        if ".".join(ref.split(".")[:3]).lower() in gone:
            errors.append(f"the {role.replace('_', ' ')} column {ref} is in a table you removed")
    for item in filters:
        if item["table"].lower() in gone:
            errors.append(f"the filter on {item['table']}.{item['column']} reads a table you removed")
    window = _implement_mapping(data.get("window"), "data.window", errors)
    dates = {}
    for label in ("start", "end"):
        raw = window.get(label)
        value = _implement_text_value(raw)
        if value is None or (value and not _IMPLEMENT_DATE_RE.fullmatch(value)):
            errors.append(f"the data window {label} {_implement_shown(raw if value is None else value)} must be a YYYY-MM-DD date")
        dates[label] = value or ""
    start, end = dates["start"], dates["end"]
    if start and end and _IMPLEMENT_DATE_RE.fullmatch(start) and _IMPLEMENT_DATE_RE.fullmatch(end) and start > end:
        errors.append(f"the data window starts ({start}) after it ends ({end})")
    out["data"] = {"tables": tables, "removed_tables": removed_tables, "filters": filters, "run_profile": profile, "column_roles": roles,
                   "window": {"start": start, "end": end},
                   "notes": _implement_vibes_text(data.get("notes"), "data.notes", IMPLEMENT_NOTEBOOK_LABEL, errors)}
    config = _implement_mapping(spec.get("config"), "config", errors)
    widgets = {}
    for name, value in _implement_mapping(config.get("widgets"), "config.widgets", errors).items():
        text = "" if value is None else _implement_value_text(value)
        if not IMPLEMENT_WIDGET_NAME_RE.fullmatch(str(name)):
            errors.append(f"widget name {_implement_shown(name)} is not a valid widget name")
        elif text is None:
            errors.append(f"widget {name} must be text or a whole number, not {_implement_shown(value)}")
        else:
            widgets[str(name)] = text
    raw_table = config.get("result_table")
    result_table = _implement_table_text(raw_table)
    if result_table is None or (result_table and not IMPLEMENT_TABLE_NAME_RE.fullmatch(result_table)):
        errors.append(f"result_table {_implement_shown(raw_table)} must be catalog.schema.table")
    profiles_in = _implement_mapping(config.get("profiles"), "config.profiles", errors)
    profiles = {}
    for name, fields in (("training", ("window_days", "sample_entities")), ("full_load", ("window_days",)),
                         ("preview", ("entity_cap",)), ("retrain", ("every_runs",))):
        section = _implement_mapping(profiles_in.get(name), f"config.profiles.{name}", errors)
        clean = {}
        for field in fields:
            value = section.get(field)
            if value is None or value == "":
                continue
            number = implement_whole_number(value)
            if number is None:
                errors.append(f"{name} {field} {_implement_shown(value)} must be a whole number")
            elif number < 1:
                errors.append(f"{name} {field} must be at least 1")
            else:
                clean[field] = number
        if name == "training":
            stratify = _implement_text(section.get("stratify_by"), "config.profiles.training.stratify_by", errors)
            if stratify:
                clean["stratify_by"] = stratify
        drift = section.get("drift_threshold")
        if name == "retrain" and drift is not None and drift != "":
            number = implement_decimal(drift)
            if number is None or not 0 < number <= 1:
                errors.append(f"retrain drift_threshold {_implement_shown(drift)} must be a number above 0 and at most 1")
            else:
                clean["drift_threshold"] = number
        profiles[name] = clean
    out["config"] = {"widgets": widgets, "result_table": result_table or "",
                     "ai_endpoint": _implement_text(config.get("ai_endpoint"), "config.ai_endpoint", errors),
                     "warehouse_id": _implement_text(config.get("warehouse_id"), "config.warehouse_id", errors), "profiles": profiles}
    schedule = _implement_mapping(spec.get("schedule"), "schedule", errors)
    cron = _implement_text(schedule.get("cron"), "schedule.cron", errors)
    enabled = _implement_flag(schedule.get("enabled"), "schedule.enabled", errors)
    cron_problem = implement_cron_problem(cron) if cron else ""
    if cron_problem:
        errors.append(f"schedule cron {_implement_shown(cron)} is not a valid Quartz cron: {cron_problem}")
    if enabled and not cron and _implement_text_value(schedule.get("cron")) is not None:
        errors.append("an enabled schedule needs a cron expression")
    emails = []
    for raw in _implement_list(schedule.get("failure_emails"), "schedule.failure_emails", errors):
        email = _implement_text_value(raw)
        if email is None or not _IMPLEMENT_EMAIL_RE.fullmatch(email):
            errors.append(f"failure email {_implement_shown(raw if email is None else email)} is not an email address")
        elif email.lower() not in [e.lower() for e in emails]:
            emails.append(email)
    out["schedule"] = {"enabled": enabled, "cron": cron, "timezone": _implement_text(schedule.get("timezone"), "schedule.timezone", errors) or "UTC",
                       "failure_emails": emails}
    outputs = _implement_mapping(spec.get("outputs"), "outputs", errors)
    chosen = {name: _implement_flag(outputs.get(name), f"outputs.{name}", errors) for name in IMPLEMENT_OUTPUTS}
    if "outputs" not in spec:
        chosen["dashboard"] = True
    if chosen["app"]:
        chosen["genie_space"] = True
    merge, merge_errors = _implement_merge_targets(outputs, chosen)
    errors += merge_errors
    app_name = merge["app"] or (_implement_text_value(outputs.get("app_name")) or "").lower()
    if chosen["app"] and not IMPLEMENT_APP_NAME_RE.fullmatch(app_name):
        errors.append(f"app_name {_implement_shown(outputs.get('app_name'))} must be 3-30 characters: lowercase letters, digits and dashes, "
                      "starting with a letter")
    if any(chosen.values()) and not out["config"]["warehouse_id"] and _implement_text_value(config.get("warehouse_id")) is not None:
        errors.append("a SQL warehouse is required to build a dashboard, a Genie Agent, an App or an alert")
    alert, alert_errors = _implement_alert_options(outputs, chosen["alert"])
    errors += alert_errors
    app_choices, app_errors = implement_app_choices(outputs) if chosen["app"] else ({"app_access": "", "app_look": {}, "app_options": {},
                                                                                    "app_backend": ""}, [])
    errors += app_errors
    removed = {}
    for name, action in _implement_mapping(spec.get("removed_outputs"), "removed_outputs", errors).items():
        action = _implement_text_value(action)
        if name in IMPLEMENT_OUTPUTS and action is not None and action.lower() in ("keep", "delete"):
            removed[name] = action.lower()
    questions = [q[:300] for q in _implement_texts(outputs.get("genie_questions"), "outputs.genie_questions", errors)]
    if len(questions) > 5:
        errors.append("at most 5 Genie Agent sample questions are allowed")
    filters_out = _implement_texts(outputs.get("dashboard_filters"), "outputs.dashboard_filters", errors)
    bad_filters = [w for w in filters_out if not IMPLEMENT_WIDGET_NAME_RE.fullmatch(w)]
    if bad_filters:
        errors.append(f"dashboard filters must be widget names: {', '.join(bad_filters)}")
    out["outputs"] = dict(chosen, app_name=app_name if chosen["app"] else "",
                          dashboard_name=_implement_text(outputs.get("dashboard_name"), "outputs.dashboard_name", errors)[:200]
                          if chosen["dashboard"] else "",
                          dashboard_filters=filters_out if chosen["dashboard"] else [],
                          genie_space_name=_implement_text(outputs.get("genie_space_name"), "outputs.genie_space_name", errors)[:200]
                          if chosen["genie_space"] else "",
                          genie_questions=questions[:5] if chosen["genie_space"] else [],
                          app_purpose=_implement_text(outputs.get("app_purpose"), "outputs.app_purpose", errors)[:1500] if chosen["app"] else "",
                          **alert, merge=merge, **app_choices, **_implement_vibes(outputs, chosen, errors))
    out["removed_outputs"] = removed
    retry = []
    for raw in _implement_list(spec.get("retry_outputs"), "retry_outputs", errors):
        name = _implement_text_value(raw)
        if name not in IMPLEMENT_RETRY_TARGETS:
            errors.append(f"retry output {_implement_shown(raw if name is None else name)} is not one of {', '.join(IMPLEMENT_RETRY_TARGETS)}")
        elif name != IMPLEMENT_NOTEBOOK and not chosen[name]:
            errors.append(f"the {IMPLEMENT_OUTPUT_LABELS[name]} cannot be retried: it is not selected")
        elif name not in retry:
            retry.append(name)
    out["retry_outputs"] = retry
    rollback = {}
    for name, value in _implement_mapping(spec.get("rollback_outputs"), "rollback_outputs", errors).items():
        version = implement_whole_number(value)
        if name not in IMPLEMENT_OUTPUTS:
            errors.append(f"rollback output {_implement_shown(name)} is not one of {', '.join(IMPLEMENT_OUTPUTS)}")
        elif not chosen[name]:
            errors.append(f"the {IMPLEMENT_OUTPUT_LABELS[name]} cannot be rolled back: it is not selected")
        elif version is None or version < 1:
            errors.append(f"rollback_outputs.{name} {_implement_shown(value)} must be a version number, 1 or more")
        elif name in retry:
            errors.append(f"the {IMPLEMENT_OUTPUT_LABELS[name]} cannot be retried and rolled back in the same version")
        else:
            rollback[name] = version
    out["rollback_outputs"] = {name: rollback[name] for name in IMPLEMENT_OUTPUTS if name in rollback}
    sharing = _implement_mapping(spec.get("sharing"), "sharing", errors)
    principals, seen = [], set()
    for item in _implement_list(sharing.get("principals"), "sharing.principals", errors):
        entry = item if isinstance(item, dict) else {}
        kind, name = _implement_text_value(entry.get("type")), _implement_text_value(entry.get("name"))
        kind = None if kind is None else kind.lower()
        if kind not in IMPLEMENT_PRINCIPAL_TYPES or not name or "`" in name:
            errors.append(f"principal {_implement_shown(item)} must have a type ({', '.join(IMPLEMENT_PRINCIPAL_TYPES)}) and a name without backticks")
        elif (kind, name.lower()) not in seen:
            seen.add((kind, name.lower()))
            principals.append({"type": kind, "name": name})
    out["sharing"] = {"principals": principals}
    return out, errors


def _implement_vibes_text(raw, field: str, label: str, errors: list) -> str:
    text = _implement_text(raw, field, errors)
    if len(text) > IMPLEMENT_VIBES_MAX_CHARS:
        errors.append(f"the {label} vibes are {len(text)} characters; at most {IMPLEMENT_VIBES_MAX_CHARS} are allowed")
    return text


def _implement_vibes(outputs: dict, chosen: dict, errors: list) -> dict:
    return {f"{name}_vibes": _implement_vibes_text(outputs.get(f"{name}_vibes") if chosen[name] else None, f"outputs.{name}_vibes",
                                                   IMPLEMENT_OUTPUT_LABELS[name], errors) for name in IMPLEMENT_OUTPUTS}


def _implement_choice(outputs: dict, key: str, default: str, choices, separator: str, errors: list, lower: bool = False) -> str:
    raw = outputs.get(key)
    text = _implement_text_value(raw)
    if text is not None:
        text = (text.lower() if lower else text) or default
    if text is None or text not in choices:
        errors.append(f"{key} {_implement_shown(raw)} must be one of {separator.join(choices)}")
        return default
    return text


def _implement_alert_options(outputs: dict, on: bool) -> tuple:
    errors = []
    outputs = outputs if on else {}
    tiers = _implement_choice(outputs, "alert_tiers", "act_now", IMPLEMENT_ALERT_TIER_SETS, ", ", errors, lower=True)
    operator = _implement_choice(outputs, "alert_operator", ">=", IMPLEMENT_ALERT_OPERATORS, " ", errors)
    raw = outputs.get("alert_threshold")
    threshold = 1.0 if raw is None or raw == "" else implement_decimal(raw)
    if threshold is None or threshold < 0:
        errors.append(f"alert_threshold {_implement_shown(raw)} must be a number of rows, 0 or more")
        threshold = 1.0
    basis = _implement_choice(outputs, "alert_threshold_basis", "fixed", IMPLEMENT_ALERT_THRESHOLD_BASES, ", ", errors, lower=True)
    if basis == "baseline" and operator not in IMPLEMENT_ALERT_BASELINE_DIRECTIONS:
        errors.append(f"a baseline threshold fires on a rise or a fall against earlier runs, so alert_operator must be one of "
                      f"{' '.join(IMPLEMENT_ALERT_BASELINE_DIRECTIONS)}, not {_implement_shown(operator)}")
    recipients = []
    for raw_email in _implement_list(outputs.get("alert_recipients"), "outputs.alert_recipients", errors):
        email = _implement_text_value(raw_email)
        if email is None or not _IMPLEMENT_EMAIL_RE.fullmatch(email):
            errors.append(f"alert recipient {_implement_shown(raw_email if email is None else email)} is not an email address")
        elif email.lower() not in [e.lower() for e in recipients]:
            recipients.append(email)
    destinations, seen = [], set()
    for raw_item in _implement_list(outputs.get("alert_destinations"), "outputs.alert_destinations", errors):
        item = raw_item if isinstance(raw_item, dict) else {"id": raw_item}
        dest_id = _implement_text_value(item.get("id"))
        if dest_id is None or not re.fullmatch(r"[A-Za-z0-9-]{8,64}", dest_id):
            errors.append(f"alert destination {_implement_shown(raw_item)} needs the id of a workspace notification destination")
        elif dest_id not in seen:
            seen.add(dest_id)
            destinations.append({"id": dest_id, "name": _implement_text(item.get("name"), "the name of each alert destination", errors)[:200]})
    if on and not recipients and not destinations:
        errors.append("an alert needs at least one recipient email or notification destination")
    if len(recipients) + len(destinations) > IMPLEMENT_ALERT_MAX_SUBSCRIBERS:
        errors.append(f"an alert can notify at most {IMPLEMENT_ALERT_MAX_SUBSCRIBERS} recipients and destinations")
    return {"alert_name": _implement_text(outputs.get("alert_name"), "outputs.alert_name", errors)[:200] if on else "",
            "alert_tiers": tiers, "alert_operator": operator, "alert_threshold": threshold, "alert_threshold_basis": basis,
            "alert_condition_text": _implement_text(outputs.get("alert_condition_text"), "outputs.alert_condition_text", errors)[:500]
            if on else "",
            "alert_recipients": recipients, "alert_destinations": destinations}, errors


def _implement_merge_targets(outputs: dict, chosen: dict) -> tuple:
    errors = []
    asked = _implement_mapping(outputs.get("merge"), "outputs.merge", errors)
    errors += [f"outputs.merge.{name} is not one of {', '.join(IMPLEMENT_MERGE_OUTPUTS)}" for name in asked
               if name not in IMPLEMENT_MERGE_OUTPUTS]
    merge = {}
    for name in IMPLEMENT_MERGE_OUTPUTS:
        if asked.get(name) is not None and not isinstance(asked.get(name), str):
            errors.append(f'outputs.merge.{name} must be text: the {IMPLEMENT_MERGE_LABELS[name]} to add to, or "" for a new one')
            merge[name] = ""
            continue
        target = _implement_text_value(asked.get(name)) if chosen[name] else ""
        target = target.lower() if name == "app" else target
        if target and not (IMPLEMENT_APP_NAME_RE if name == "app" else IMPLEMENT_ASSET_ID_RE).fullmatch(target):
            errors.append(f"outputs.merge.{name} {_implement_shown(asked.get(name))} must name an existing Inspire {IMPLEMENT_MERGE_LABELS[name]}")
            target = ""
        merge[name] = target
    return merge, errors



def implement_merge_target(outputs: dict, name: str) -> str:
    return str(((outputs or {}).get("merge") or {}).get(name) or "").strip()


def _implement_section(spec: dict, key: str):
    value = (spec or {}).get(key)
    return json.dumps(value if value is not None else {}, sort_keys=True, default=str)


def _implement_option_changes(before: dict, after: dict) -> list:
    return [name for name in IMPLEMENT_OUTPUTS if before.get(name) and after.get(name)
            and any(before.get(o) != after.get(o) for o in IMPLEMENT_OUTPUT_OPTIONS[name])]


def classify_implement_change(previous, current) -> list:
    if not previous:
        return ["initial"]
    classes = set()
    if _implement_section(previous, "data") != _implement_section(current, "data"):
        classes.add("data")
    if (_implement_section(previous, "config") != _implement_section(current, "config")
            or _implement_section(previous, "use_case") != _implement_section(current, "use_case")):
        classes.add("config")
    if _implement_section(previous, "schedule") != _implement_section(current, "schedule"):
        classes.add("schedule")
    before, after = previous.get("outputs") or {}, current.get("outputs") or {}
    if any(after.get(n) and not before.get(n) for n in IMPLEMENT_OUTPUTS) or (
            after.get("app") and before.get("app") and after.get("app_name") != before.get("app_name")) or _implement_option_changes(
            before, after) or any(after.get(n) and before.get(n) and implement_merge_target(before, n) != implement_merge_target(after, n)
                                  for n in IMPLEMENT_MERGE_OUTPUTS):
        classes.add("outputs")
    if any(before.get(n) and not after.get(n) for n in IMPLEMENT_OUTPUTS):
        classes.add("deselect")
    if _implement_section(previous, "sharing") != _implement_section(current, "sharing"):
        classes.add("sharing")
    if current.get("retry_outputs"):
        classes.add("retry")
    if current.get("rollback_outputs"):
        classes.add("rollback")
    return [c for c in IMPLEMENT_CHANGE_CLASSES if c in classes]


def implement_plan(previous, current, previous_assets=None, has_built_notebook=False) -> dict:
    classes = classify_implement_change(previous, current)
    assets = previous_assets or {}
    outputs = current.get("outputs") or {}
    before = (previous or {}).get("outputs") or {}
    refill = (bool({"initial", "data", "config"} & set(classes)) or not has_built_notebook
              or IMPLEMENT_NOTEBOOK in (current.get("retry_outputs") or []))
    build, refresh = [], []
    option_changes = _implement_option_changes(before, outputs) if previous else []
    retry = list(current.get("retry_outputs") or [])
    rollback = {name: int(version) for name, version in (current.get("rollback_outputs") or {}).items()}
    for name in IMPLEMENT_OUTPUTS:
        if not outputs.get(name) or name in rollback:
            continue
        renamed = (name == "app" and before.get("app") and before.get("app_name") != outputs.get("app_name")) or (
            before.get(name) and implement_merge_target(before, name) != implement_merge_target(outputs, name))
        if not assets.get(name) or not before.get(name) or renamed:
            build.append(name)
        elif refill or name in option_changes or name in retry or (name == "alert" and "schedule" in classes):
            refresh.append(name)
    restyle = [n for n in refresh if n in IMPLEMENT_RESTYLE_OPTIONS and n in option_changes and not refill and n not in retry
               and all(before.get(o) == outputs.get(o) for o in IMPLEMENT_OUTPUT_OPTIONS[n] if o not in IMPLEMENT_RESTYLE_OPTIONS[n])]
    deselected = [n for n in IMPLEMENT_OUTPUTS if before.get(n) and not outputs.get(n)]
    delete = [n for n in deselected if (current.get("removed_outputs") or {}).get(n) == "delete" and assets.get(n)]
    keep = [n for n in deselected if n not in delete]
    moved = bool(previous) and (_implement_plain_table(((previous or {}).get("config") or {}).get("result_table")).lower()
                                != _implement_plain_table((current.get("config") or {}).get("result_table")).lower())
    return {"classes": classes, "refill": refill, "preview": refill or bool(build) and not assets.get("preview_run_id"),
            "build": build, "refresh": refresh, "restyle": restyle, "rollback": rollback, "delete": delete, "keep_unmanaged": keep,
            "schedule": bool({"initial", "schedule"} & set(classes)) or refill,
            "sharing": bool({"initial", "sharing", "outputs"} & set(classes)) or bool(build) or bool(rollback) or moved}


def implement_reusable_notebook(previous, notebook_record) -> bool:
    written_by = (notebook_record or {}).get("spec_version")
    if not previous or written_by is None:
        return False
    try:
        return int(written_by) <= int(previous["spec_version"])
    except (KeyError, TypeError, ValueError):
        return False


def implement_notebook_keeps_history(notebook_record) -> bool:
    return (notebook_record or {}).get("result_history") == IMPLEMENT_RESULT_HISTORY


def implement_replace_edits(spec: dict) -> bool:
    return bool((((spec or {}).get("outputs") or {}).get("app_options") or {}).get("replace_edits"))


def implement_replace_edits_sql(result_table: str, replace_edits: bool) -> str:
    return (f"ALTER TABLE {inspire_table_name(result_table)} SET TBLPROPERTIES ({inspire_sql_literal(INSPIRE_REPLACE_EDITS_PROPERTY)} = "
            f"{inspire_sql_literal('true' if replace_edits else 'false')})")


def implement_result_table_move(previous_spec, spec: dict, default_table: str):
    if not previous_spec:
        return None
    old = ".".join(inspire_table_parts((previous_spec.get("config") or {}).get("result_table") or default_table))
    new = ".".join(inspire_table_parts((spec.get("config") or {}).get("result_table") or default_table))
    return (old, new) if old.lower() != new.lower() else None


def implement_result_copy_sql(previous_result_table: str, result_table: str) -> str:
    return f"CREATE TABLE {inspire_table_name(result_table)} DEEP CLONE {inspire_table_name(previous_result_table)}"


def implement_dismissals_sql(result_table: str) -> str:
    return (f"SELECT reason, records, SUM(records) OVER () AS dismissed FROM (SELECT NULLIF(TRIM(dismiss_reason), '') AS reason, "
            f"COUNT(*) AS records FROM {inspire_table_name(result_table)} WHERE use_case_id = :uc AND {INSPIRE_CURRENT_ROWS} AND "
            f"action_status = 'Dismissed' GROUP BY 1) ORDER BY reason IS NULL, records DESC, reason LIMIT {IMPLEMENT_DISMISS_REASONS + 1}")


def implement_dismissal_lines(dismissed) -> list:
    total = int((dismissed or {}).get("dismissed") or 0)
    if total <= 0:
        return []
    reasons = "; ".join(f"\"{reason}\" ({count})" for reason, count in (dismissed.get("reasons") or [])[:IMPLEMENT_DISMISS_REASONS])
    return [f"- The customer's feedback from the App: people dismissed {total} of the current records"
            + (f", most often because: {reasons}" if reasons else ", without giving a reason")
            + ". Treat it as the customer's word on what the decision list gets wrong, and change the method where the data can tell "
              "such records apart. A dismissed record never goes back to Act now: inspire_write_result keeps it in Monitor on every "
              "run, so never work around that."]


def implement_choices_markdown(spec: dict, previous=None, trains_model: bool = True, dismissed=None) -> str:
    lines = ["## Implementation Choices",
             "The user made these choices in the Start Use Case PoC/MVP with Genie wizard. They win over anything above that conflicts with them."]
    choices = _implement_choice_lines(spec, trains_model)
    lines += choices
    lines += implement_dismissal_lines(dismissed)
    changed = classify_implement_change(previous, spec) if previous else []
    if previous and {"data", "config"} & set(changed):
        lines.append("- The notebook was already built for an earlier version of these choices. Update the cells that the changed "
                     "choices affect; keep every cell that still holds.")
        for line in _implement_choice_lines(previous, trains_model):
            if line not in choices:
                lines.append(f"- Withdrawn since the earlier version: {line[2:]} Take out what the cells do for it; where the "
                             "instruction above has its own choice, follow the instruction.")
    if previous and IMPLEMENT_NOTEBOOK in (spec.get("retry_outputs") or []):
        lines.append("- The user asked for the notebook to be built again. Check every cell against the instruction and the platform "
                     "rules above as they read now: rewrite each cell that does not follow them, and keep each cell that does.")
    return "\n".join(lines) + "\n"


def _implement_choice_lines(spec: dict, trains_model: bool = True) -> list:
    data, config = spec.get("data") or {}, spec.get("config") or {}
    lines = []
    framing = spec.get("use_case") or {}
    if framing.get("goal"):
        lines.append(f"- Goal: {framing['goal']}")
    if framing.get("decision_owner"):
        lines.append(f"- Decision owner: {framing['decision_owner']}")
    for measure in framing.get("success_measures") or []:
        lines.append(f"- Success measure: {measure}")
    for original, chosen in sorted((data.get("tables") or {}).items()):
        lines.append(f"- Read `{chosen}` wherever this instruction reads `{original}`; it has the same role. Check its columns with "
                     f"DESCRIBE TABLE first and map each column the instruction uses to its real name in `{chosen}`.")
    for table in data.get("removed_tables") or []:
        lines.append(f"- Do not read `{table}`: the user removed it from this use case. Build every step from the other tables and "
                     "leave out the features that only it provided.")
    for item in data.get("filters") or []:
        values = ", ".join(repr(v) for v in item["values"])
        lines.append(f"- Keep only the rows of `{item['table']}` where `{item['column']}` {item['op']} {values}, applied when the "
                     "table is first read, before any join.")
    for role, ref in sorted((data.get("column_roles") or {}).items()):
        if role != "label" or trains_model:
            lines.append(f"- Use `{ref}` as the {role.replace('_', ' ')} column.")
    window = data.get("window") or {}
    if window.get("start") or window.get("end"):
        lines.append(f"- Read history from {window.get('start') or 'the earliest date'} to {window.get('end') or 'the latest date'} only.")
    profiles = config.get("profiles") or {}
    training, full_load = profiles.get("training") or {}, profiles.get("full_load") or {}
    if training and trains_model:
        parts = [f"the last {training['window_days']} days" if training.get("window_days") else "",
                 f"a sample of {training['sample_entities']} entities" if training.get("sample_entities") else "",
                 f"stratified by `{training['stratify_by']}`" if training.get("stratify_by") else ""]
        lines.append("- Training profile: " + ", ".join(p for p in parts if p) + ".")
    if full_load.get("window_days"):
        lines.append(f"- Full-load profile: score with the last {full_load['window_days']} days of data, as_of_date = latest.")
    retrain = profiles.get("retrain") or {}
    if retrain and trains_model:
        rules = [f"every {retrain['every_runs']} full runs" if retrain.get("every_runs") else "",
                 f"when drift exceeds {retrain['drift_threshold']}" if retrain.get("drift_threshold") else ""]
        lines.append("- Retrain the model " + " or ".join(r for r in rules if r) + "; otherwise reuse the last trained model.")
    if config.get("result_table"):
        own = f"use_case_id = '{spec['use_case_id']}'" if spec.get("use_case_id") else "use_case_id is this use case's id"
        lines.append(f"- Write the business output to `{config['result_table']}` (the default of the `result_table` widget), only with "
                     f"inspire_write_result(frame, result_table): each run's rows, where {own}, become the current version of their "
                     "records, earlier versions stay as change history, and rows of other use cases stay.")
    if config.get("ai_endpoint"):
        lines.append(f"- Use the serving endpoint `{config['ai_endpoint']}` (the default of the `ai_endpoint` widget).")
    for name, value in sorted((config.get("widgets") or {}).items()):
        lines.append(f"- Default of widget `{name}`: `{value}`.")
    cap = (profiles.get("preview") or {}).get("entity_cap") or 1000
    if trains_model:
        profile_rule = ("The training profile is the history the model learns from; the full-load profile is every entity scored by "
                        f"a production run. With `preview`, train on the training profile as usual and score at most {cap} entities, "
                        "so a run finishes in minutes; with `full`, score the whole full-load profile.")
    else:
        profile_rule = ("The full-load profile is every entity a production run covers. With `preview`, cover at most "
                        f"{cap} entities, so a run finishes in minutes; with `full`, cover the whole full-load profile.")
    lines.append("- Add a dropdown widget `run_profile` with the choices `preview` and `full` and the default `full`, in the Parameters "
                 f"cell. {profile_rule} Both write the same columns.")
    if data.get("notes"):
        lines.append(f"- The customer's asks for this notebook, word for word: \"\"\"{data['notes']}\"\"\" They are explicit user "
                     "directives and win over anything above that conflicts with them; Inspire's checks on the notebook still run.")
    return lines


def implement_job_parameters(widget_names, spec: dict, profile: str) -> dict:
    config = spec.get("config") or {}
    params = {}
    names = list(widget_names or [])
    for name, value in (config.get("widgets") or {}).items():
        if name in names:
            params[name] = str(value)
    if config.get("result_table") and "result_table" in names:
        params["result_table"] = config["result_table"]
    if config.get("ai_endpoint") and "ai_endpoint" in names:
        params["ai_endpoint"] = config["ai_endpoint"]
    params["run_profile"] = profile if profile in IMPLEMENT_RUN_PROFILES else "preview"
    params[IMPLEMENT_REFRESH_PARAMETER] = IMPLEMENT_REFRESH_DEFAULT
    params[INSPIRE_CHILD_MARKER] = "1"
    return params


_IMPLEMENT_OUTCOME_RE = re.compile(r"wrote (\d+) row\(s\) for as_of_date (\S+) with run_id ([0-9a-fA-F-]{8,})")


def parse_use_case_outcome(text) -> dict:
    text = str(text or "")
    match = _IMPLEMENT_OUTCOME_RE.search(text)
    if match:
        return {"status": "written", "rows_written": int(match.group(1)), "as_of_date": match.group(2), "run_id": match.group(3)}
    stopped = re.search(r"stopped without writing: (.*)", text)
    if stopped:
        return {"status": "stopped", "stop_reason": stopped.group(1).strip()}
    return {"status": "unknown", "text": text[:500]}


def implement_widget_problems(widgets, values) -> list:
    known = [w for w in widgets or [] if isinstance(w, dict)]
    allowed = {str(w.get("name")): w.get("allowed") for w in known}
    problems = [problem for name, value in sorted((values or {}).items())
                for problem in [genie_widget_value_problem(name, value, allowed.get(name, ""))] if problem]
    merged = {str(w.get("name")): w.get("default") for w in known}
    merged.update(values or {})
    return problems + genie_widget_set_problems(merged)


def normalize_genie_fill_widgets(result, core_widgets) -> list:
    core = {w["name"]: w.get("allowed") or "" for w in core_widgets or []}
    problems = []
    for widget in (result.get("widgets") or []) if isinstance(result, dict) else []:
        if not isinstance(widget, dict):
            continue
        name = str(widget.get("name") or "")
        if name in core:
            widget["allowed"] = core[name]
        elif str(widget.get("type") or "") == "dropdown":
            widget["allowed"] = ""
        else:
            rule = genie_allowed_rule(widget.get("allowed"))
            if rule is None:
                problems.append(f"widget {name} has allowed {widget.get('allowed')!r}, which is not one of Inspire's range forms "
                                "('0 to 1', 'at least 1', 'at most 5', 'whole number, 1 to 10', or empty for any value)")
            widget["allowed"] = genie_allowed_text(rule)
    return problems


def validate_genie_fill_result(result, required_widgets) -> list:
    errors = []
    if not isinstance(result, dict) or not result:
        return ["Genie returned no JSON result"]
    names = []
    for widget in result.get("widgets") or []:
        name, kind = str(widget.get("name") or ""), str(widget.get("type") or "")
        if not IMPLEMENT_WIDGET_NAME_RE.match(name):
            errors.append(f"widget name {name!r} is invalid")
        if kind == "dropdown" and str(widget.get("default")) not in [str(c) for c in widget.get("choices") or []]:
            errors.append(f"dropdown widget {name} has default {widget.get('default')!r} outside its choices")
        problem = genie_widget_value_problem(name, widget.get("default"), widget.get("allowed"))
        if problem:
            errors.append(f"the default of widget {problem}")
        names.append(name)
    missing = [n for n in list(required_widgets or []) + ["run_profile"] if n not in names]
    if missing:
        errors.append(f"widgets missing from the result: {', '.join(missing)}")
    cron = str((result.get("schedule") or {}).get("cron") or "")
    cron_problem = implement_cron_problem(cron) if cron else ""
    if cron_problem:
        errors.append(f"suggested schedule '{cron}' is not a valid Quartz cron: {cron_problem}")
    return errors


def genie_fill_stopped(result) -> bool:
    if not isinstance(result, dict):
        return False
    status = str(result.get("status") or "")
    return status == "stopped" or (status == "built" and bool(str(result.get("stop_reason") or "").strip()))


def implement_table_refs(text) -> list:
    refs = []
    for match in _IMPLEMENT_TABLE_REF_RE.finditer(str(text or "")):
        groups = [g for g in match.groups() if g is not None]
        name = ".".join(groups[:3])
        if name.lower() not in {r.lower() for r in refs}:
            refs.append(name)
    return refs


def implement_join_lines(lines) -> list:
    items = [str(line) for line in lines or []]
    return [line if index == len(items) - 1 or line[-1:] in ("\n", " ", "\t", "\r") else line + "\n"
            for index, line in enumerate(items)]


def _implement_normalize_sql_lines(obj):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key in ("queryLines", "sql") and isinstance(value, list):
                obj[key] = implement_join_lines(value)
            else:
                _implement_normalize_sql_lines(value)
    elif isinstance(obj, list):
        for value in obj:
            _implement_normalize_sql_lines(value)
    return obj


def _implement_sql_texts(obj) -> list:
    texts = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key in ("queryLines", "sql") and isinstance(value, list):
                texts.append("".join(str(v) for v in value))
            elif key in ("query", "sql") and isinstance(value, str):
                texts.append(value)
            else:
                texts.extend(_implement_sql_texts(value))
    elif isinstance(obj, list):
        for value in obj:
            texts.extend(_implement_sql_texts(value))
    return texts


def _implement_unknown_tables(texts, allowed) -> list:
    allowed_lower = {_implement_plain_table(t).lower() for t in allowed or []}
    unknown = []
    for text in texts:
        for ref in implement_table_refs(text):
            if ref.lower() not in allowed_lower and ref not in unknown and not ref.lower().startswith(("system.", "information_schema.")):
                unknown.append(ref)
    return unknown


_IMPLEMENT_FLAG_COUNT_RE = re.compile(r"\b(?:count_if|count|sum|avg)\s*\([^)]*\bdata_quality_flag\b"
                                      r"|\bfilter\s*\(\s*where\b[^)]*\bdata_quality_flag\b", re.I)
_IMPLEMENT_OK_LITERAL_RE = re.compile(r"""['"]ok['"]""", re.I)


def _implement_flag_count_problems(texts, where: str) -> list:
    if any(_IMPLEMENT_FLAG_COUNT_RE.search(str(t)) and not _IMPLEMENT_OK_LITERAL_RE.search(str(t)) for t in texts or ()):
        return [f"{where} counts data_quality_flag without comparing it to 'ok': the notebook writes 'ok' on every row that "
                f"passed its checks, so only a value other than 'ok' marks a flagged row"]
    return []


_IMPLEMENT_G_SPEC = r"(?:.?[<>=^])?[+\- ]?z?#?0?\d*[,_]?(?:\.\d+)?[gG]"
_IMPLEMENT_G_SPEC_RE = re.compile(_IMPLEMENT_G_SPEC)
_IMPLEMENT_BRACE_G_RE = re.compile(r"\{[^{}]*:" + _IMPLEMENT_G_SPEC + r"\}")
_IMPLEMENT_PERCENT_G_RE = re.compile(r"%[-+ #0]*\d*(?:\.\d+)?[gG]")
_IMPLEMENT_ZERO_STRIP = "strips zeros from a number's text with rstrip, which turns 120 into '12'"
_IMPLEMENT_G_FORMAT = "writes a number with Python's 'g' format, which turns 1060 into 1.06e+03"


def _implement_text_constant(node) -> str:
    return node.value if isinstance(node, ast.Constant) and isinstance(node.value, str) else None


def _implement_number_text_spots(tree, skip) -> list:
    spots = []
    for node in ast.walk(tree):
        if id(node) in skip:
            continue
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr == "rstrip" and node.args \
                and "0" in (_implement_text_constant(node.args[0]) or ""):
            spots.append((node.lineno, _IMPLEMENT_ZERO_STRIP))
        elif isinstance(node, ast.FormattedValue) and isinstance(node.format_spec, ast.JoinedStr) \
                and all(_implement_text_constant(v) is not None for v in node.format_spec.values) \
                and _IMPLEMENT_G_SPEC_RE.fullmatch("".join(v.value for v in node.format_spec.values)):
            spots.append((node.lineno, _IMPLEMENT_G_FORMAT))
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "format" and len(node.args) > 1 \
                and _IMPLEMENT_G_SPEC_RE.fullmatch(_implement_text_constant(node.args[1]) or "-"):
            spots.append((node.lineno, _IMPLEMENT_G_FORMAT))
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr == "format" \
                and _IMPLEMENT_BRACE_G_RE.search(_implement_text_constant(node.func.value) or ""):
            spots.append((node.lineno, _IMPLEMENT_G_FORMAT))
        elif isinstance(node, ast.BinOp) and isinstance(node.op, ast.Mod) \
                and _IMPLEMENT_PERCENT_G_RE.search(_implement_text_constant(node.left) or ""):
            spots.append((node.lineno, _IMPLEMENT_G_FORMAT))
    return sorted(set(spots))


def implement_number_text_problems(cells, required: bool = True, vetted: bool = False) -> list:
    vetted_tree = ast.parse(GENIE_NUMBER_TEXT_SOURCE)
    vetted_def = ast.dump(next(n for n in vetted_tree.body if isinstance(n, ast.FunctionDef)))
    needs = sorted(a.name for n in vetted_tree.body if isinstance(n, ast.ImportFrom) for a in n.names)
    formatters = (GENIE_NUMBER_TEXT_NAME,) if vetted else GENIE_NUMBER_FUNCTIONS
    defined, changed, imported, spots, calls = [], [], set(), [], 0
    for label, source in cells:
        tree = _implement_cell_tree(source)
        if tree is None:
            continue
        skip = set()
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef) and node.name in formatters:
                skip.update(id(n) for n in ast.walk(node))
                if node.name == GENIE_NUMBER_TEXT_NAME:
                    defined.append(label)
                    if vetted and ast.dump(node) != vetted_def:
                        changed.append(label)
            elif isinstance(node, ast.ImportFrom) and node.module == "math":
                imported.update(a.name for a in node.names if a.asname is None)
        calls += sum(1 for n in ast.walk(tree) if id(n) not in skip and isinstance(n, ast.Call)
                     and isinstance(n.func, ast.Name) and n.func.id in formatters)
        spots += [f"{label} line {line} {what}" for line, what in _implement_number_text_spots(tree, skip)]
    problems = []
    if len(defined) > 1:
        problems.append(f"{GENIE_NUMBER_TEXT_NAME} is defined in {', '.join(defined)}; define it once")
    if changed:
        problems.append(f"{', '.join(changed)} changes {GENIE_NUMBER_TEXT_NAME}; copy it exactly as Inspire gives it")
    if vetted and defined and not set(needs) <= imported:
        problems.append(f"the import `from math import {', '.join(needs)}` that {GENIE_NUMBER_TEXT_NAME} needs is missing")
    if required and not defined:
        problems.append(f"the notebook does not define {GENIE_NUMBER_TEXT_NAME}; write it as the number rule of the instruction's "
                        "Platform Rules says")
    elif required and not calls:
        problems.append(f"no cell calls {' or '.join(formatters)}; write every number in top_reasons, recommended_action, "
                        "ai_facts and the summary facts with them")
    if spots:
        problems.append("; ".join(spots) + f"; write numbers as text only with {' or '.join(formatters)}")
    return problems


_IMPLEMENT_DENIAL_MARK_RE = re.compile(r"PERMISSION_DENIED|INSUFFICIENT_PERMISSIONS|PermissionDenied|PermissionError|Forbidden"
                                       r"|does not have|permission|privilege|CREATE[ _]MODEL", re.IGNORECASE)


def _implement_cell_tree(source):
    lines = str(source or "").splitlines()
    try:
        return ast.parse("\n".join("" if line.lstrip().startswith(("%", "!")) else line for line in lines))
    except SyntaxError:
        return None


def _implement_mentions_denial(node) -> bool:
    for part in ast.walk(node):
        if isinstance(part, ast.Constant) and isinstance(part.value, str):
            text = part.value
        elif isinstance(part, ast.Name):
            text = part.id
        elif isinstance(part, ast.Attribute):
            text = part.attr
        else:
            continue
        if _IMPLEMENT_DENIAL_MARK_RE.search(text):
            return True
    return False


def _implement_raises(body) -> bool:
    return any(isinstance(node, ast.Raise) for statement in body for node in ast.walk(statement))


def implement_denial_catch_problems(cells) -> list:
    problems = []
    for label, source in cells:
        tree = _implement_cell_tree(source)
        if tree is None:
            continue
        for handler in (node for node in ast.walk(tree) if isinstance(node, ast.ExceptHandler)):
            carries_on = handler.type is not None and _implement_mentions_denial(handler.type) and not _implement_raises(handler.body)
            for node in ast.walk(handler):
                if isinstance(node, ast.If) and _implement_mentions_denial(node.test) and not _implement_raises(node.body):
                    carries_on = True
                elif isinstance(node, ast.IfExp) and _implement_mentions_denial(node.test):
                    carries_on = True
            if carries_on:
                problems.append(f"{label} line {handler.lineno} catches a permission error and carries on; a denied privilege "
                                "must fail the run, so remove that handling and let the error stop the notebook")
    return problems


def implement_cell_title(cell) -> str:
    metadata = cell.get("metadata") if isinstance(cell, dict) else None
    return str(((metadata or {}).get("application/vnd.databricks.v1+cell") or {}).get("title") or "")


def implement_cell_text(cell) -> str:
    source = cell.get("source") if isinstance(cell, dict) else None
    return "".join(source) if isinstance(source, list) else str(source or "")


def _implement_same_code(expected: str, actual: str) -> bool:
    try:
        return ast.dump(ast.parse(expected)) == ast.dump(ast.parse(actual))
    except SyntaxError:
        return expected.strip() == actual.strip()


def implement_restore_owned_cells(cells, instruction: dict, controls: dict) -> tuple:
    restored, changes = list(cells or []), []
    for owned, keep_last in ((instruction, False), (controls, True)):
        title = implement_cell_title(owned)
        found = [i for i, cell in enumerate(restored) if implement_cell_title(cell) == title]
        if not found:
            change = "deleted"
        elif len(found) > 1:
            change = "copied"
        elif not _implement_same_code(implement_cell_text(owned), implement_cell_text(restored[found[0]])):
            change = "edited"
        elif keep_last and found[0] != len(restored) - 1:
            change = "moved"
        else:
            continue
        changes.append(f"{change} the '{title}' cell")
        position = len(restored) if keep_last else (found[0] if found else 1)
        for index in reversed(found):
            del restored[index]
        restored.insert(min(position, len(restored)), owned)
    return restored, changes


def check_dashboard_definition(serialized, allowed_tables, decision: str = IMPLEMENT_DECISION_DATASET) -> tuple:
    errors = []
    try:
        dashboard = json.loads(serialized) if isinstance(serialized, str) else dict(serialized or {})
    except (ValueError, TypeError) as e:
        return {}, [f"dashboard.lvdash.json is not valid JSON: {e}"]
    _implement_normalize_sql_lines(dashboard.get("datasets") or [])
    datasets = dashboard.get("datasets") or []
    names = {str(d.get("name")) for d in datasets if isinstance(d, dict)}
    if not datasets:
        errors.append("the dashboard has no datasets")
    if decision not in names:
        errors.append(f"the dashboard has no dataset named {decision}, the decision list of its first page")
    pages = dashboard.get("pages") or []
    if not pages:
        errors.append("the dashboard has no pages")
    elif decision in names and not any(
            ((q.get("query") or {}).get("datasetName")) == decision
            for item in ([p for p in pages if p.get("pageType") != "PAGE_TYPE_GLOBAL_FILTERS"] or pages)[0].get("layout") or []
            for q in ((item.get("widget") or {}).get("queries") or [])):
        errors.append(f"the first page shows no widget over the {decision} dataset; it must be the decision list")
    errors += implement_dashboard_problems(dashboard)
    sql = ["".join(d.get("queryLines") or []) if isinstance(d.get("queryLines"), list) else str(d.get("query") or "") for d in datasets]
    for text in sql:
        if _IMPLEMENT_WRITE_SQL_RE.search(text):
            errors.append("a dashboard dataset writes data; datasets may only read")
    errors += _implement_flag_count_problems(sql, "a dashboard dataset")
    unknown = _implement_unknown_tables(sql, allowed_tables)
    if unknown:
        errors.append(f"the dashboard reads tables outside this use case: {', '.join(unknown)}")
    return dashboard, errors


def implement_decision_list_problems(decision_rows, truth_rows, top_k: int = IMPLEMENT_DECISION_TOP_K,
                                     dataset: str = IMPLEMENT_DECISION_DATASET) -> list:
    decision = [{str(k).lower(): v for k, v in dict(r).items()} for r in decision_rows or []]
    truth = [{str(k).lower(): v for k, v in dict(r).items()} for r in truth_rows or []]
    if not truth:
        return []
    if not decision:
        return [f"the {dataset} dataset returned no rows, but the business output table has {len(truth)} current row(s) "
                f"(WHERE {INSPIRE_CURRENT_ROWS}); show every one of them ordered by rank, whatever its priority_tier"]
    columns = set(decision[0])
    if "priority_tier" not in columns or not ({"entity_id", "entity_label"} & columns):
        return [f"the {dataset} dataset must return priority_tier and entity_label (or entity_id), "
                f"so each row can be checked against the business output table"]
    ids = [t.get("entity_id") for t in truth]
    key = ("entity_id" if "entity_id" in columns and all(v not in (None, "") for v in ids) and len(set(map(str, ids))) == len(ids)
           else "entity_label")
    stored = {(str(t.get(key)), str(t.get("priority_tier"))) for t in truth}
    stored_keys = {name for name, _tier in stored}
    shown = {(str(d.get(key)), str(d.get("priority_tier"))) for d in decision}
    shown_keys = {name for name, _tier in shown}
    problems = []
    relabeled = [f"{name} shows {tier!r} but the table holds {' or '.join(sorted(repr(t) for n, t in stored if n == name))}"
                 for name, tier in sorted(shown) if name in stored_keys and (name, tier) not in stored]
    if relabeled:
        problems.append(f"the {dataset} dataset changes priority tiers ({'; '.join(relabeled[:5])}); show each "
                        "row's priority_tier exactly as the business output table stores it")
    missing = list(dict.fromkeys(str(t.get(key)) for t in truth[:max(1, int(top_k))] if str(t.get(key)) not in shown_keys))
    if missing:
        problems.append(f"the {dataset} dataset leaves out top-ranked current rows "
                        f"({', '.join(missing[:5])}); list every row WHERE {INSPIRE_CURRENT_ROWS} ordered by rank, never only some "
                        "tiers and never the newest as_of_date instead")
    return problems


def implement_latest_run_filter(result_table: str) -> str:
    return INSPIRE_CURRENT_ROWS


def _implement_positive_tier_counts(tier_counts: dict) -> dict:
    counts = {}
    for tier, rows in (tier_counts or {}).items():
        rows = int(rows or 0)
        if rows > 0:
            counts[str(tier)] = rows
    return counts


def implement_reached_no_action_tier(tier_counts: dict) -> bool:
    counts = _implement_positive_tier_counts(tier_counts)
    return bool(counts) and not any(counts.get(tier) for tier in IMPLEMENT_ACTION_TIERS)


def implement_tier_mix_warning(tier_counts: dict, run_url: str = "", profile: dict = None, methods: dict = None,
                               reason: str = "") -> str:
    if not implement_reached_no_action_tier(tier_counts):
        return ""
    counts = _implement_positive_tier_counts(tier_counts)
    mix = ", ".join(f"{rows} {tier}" for tier, rows in sorted(counts.items(), key=lambda item: (-item[1], item[0])))
    reason = " ".join(str(reason or "").split()).rstrip(".")
    facts = [f"the run's reason: {reason}"] if reason else []
    named = (methods or {}).get("methods") or []
    if named:
        fact = (f"every row comes from {named[0][0]}" if len(named) == 1
                else "its rows come from " + ", ".join(f"{name} ({rows})" for name, rows in named))
        facts.append(fact + ("" if methods.get("versioned_rows") else ", with no registered model"))
    if profile:
        if profile.get("low") is not None and profile.get("high") is not None:
            kind = f" ({profile['value_type']})" if profile.get("value_type") else ""
            facts.append(f"its value{kind} runs from {implement_number_text(profile['low'])} to "
                         f"{implement_number_text(profile['high'])}")
        flags = profile.get("flags") or []
        facts.append("its rows are flagged: " + ", ".join(f"{flag} ({rows})" for flag, rows in flags) if flags
                     else "no row is flagged")
    return (f"Decision list: no row of the latest run reached {' or '.join(IMPLEMENT_ACTION_TIERS)} ({mix}), so it has nobody "
            "to act on yet" + "".join(f"; {fact}" for fact in facts) + (f"; run: {run_url}" if run_url else ""))


def implement_written_key_problems(outcome: dict, uc_id: str, counts: dict, result_table: str) -> list:
    problems = []
    as_of = str(outcome.get("as_of_date") or "")
    if not _IMPLEMENT_CALENDAR_DATE_RE.fullmatch(as_of):
        problems.append(f"run_use_case() returned as_of_date {as_of or '(empty)'}, which is not a calendar date (YYYY-MM-DD): "
                        "resolve the as-of date to a date, and when the source tables hold dates as keys into a date table, "
                        "join that table to read the date")
    rows, run_id = int(counts.get("row_count") or 0), outcome.get("run_id")
    if not rows:
        return problems + [f"the business output table {result_table} holds no row with run_id {run_id}, although run_use_case() "
                           f"said it wrote {outcome.get('rows_written')} row(s): write the run_id the outcome returns on every row"]
    undated = rows - int(counts.get("dated_rows") or 0)
    if undated:
        problems.append(f"{undated} of {rows} row(s) of run {run_id} have no as_of_date, so the change history cannot say which "
                        "run date they belong to: write the resolved as-of date on every row")
    foreign = rows - int(counts.get("own_rows") or 0)
    if foreign:
        problems.append(f"{foreign} of {rows} row(s) of run {run_id} carry a use_case_id other than {uc_id}")
    loose = rows - int(counts["current_rows"] or 0)
    if loose:
        problems.append(f"{loose} of {rows} row(s) of run {run_id} are not the current version of their record in {result_table}, "
                        "so no output would show them: write the business output only with inspire_write_result(frame, "
                        "result_table) from the Run controls cell, never with your own write")
    return problems


def _implement_text_arrays(obj):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key in ("question", "sql", "content", "instruction", "comment", "usage_guidance") and isinstance(value, str):
                obj[key] = [value]
            else:
                _implement_text_arrays(value)
    elif isinstance(obj, list):
        for value in obj:
            _implement_text_arrays(value)
    return obj


def _implement_join_spec_problems(join: dict) -> list:
    lines = [str(line).strip() for line in join.get("sql") or [] if str(line).strip()]
    if len(lines) == 2 and not lines[0].startswith("--") and lines[1] in IMPLEMENT_JOIN_TAGS:
        return []
    names = " to ".join(str((join.get(side) or {}).get("alias") or (join.get(side) or {}).get("identifier") or "?")
                        for side in ("left", "right"))
    return [f"the join spec {names} has sql {json.dumps(join.get('sql'))}; it must be exactly two items, the join condition and then "
            f"the relationship tag that matches the key's cardinality, one of {', '.join(IMPLEMENT_JOIN_TAGS)}"]


def normalize_genie_space(definition, allowed_tables) -> tuple:
    errors = []
    try:
        data = json.loads(definition) if isinstance(definition, str) else dict(definition or {})
    except (ValueError, TypeError) as e:
        return "", "", {}, [f"genie_space.json is not valid JSON: {e}"]
    space = data.get("serialized_space") if isinstance(data.get("serialized_space"), dict) else data
    if isinstance(data.get("serialized_space"), str):
        try:
            space = json.loads(data["serialized_space"])
        except ValueError as e:
            return "", "", {}, [f"serialized_space is not valid JSON: {e}"]
    title, description = str(data.get("title") or "").strip(), str(data.get("description") or "").strip()
    if not title:
        errors.append("the Genie Agent has no title")
    space = _implement_normalize_sql_lines(_implement_text_arrays(json.loads(json.dumps(space))))
    space["version"] = 2

    question_ids, instruction_ids = set(), set()

    def ids(items, used):
        for item in items:
            if not re.fullmatch(r"[0-9a-f]{32}", str(item.get("id") or "")) or item["id"] in used:
                item["id"] = uuid.uuid4().hex
            used.add(item["id"])
        return sorted(items, key=lambda i: i["id"])

    config = space.setdefault("config", {})
    config["sample_questions"] = ids([q if isinstance(q.get("question"), list) else dict(q, question=[str(q.get("question") or "")])
                                      for q in config.get("sample_questions") or [] if isinstance(q, dict)], question_ids)
    if isinstance(space.get("benchmarks"), dict) and "questions" in space["benchmarks"]:
        space["benchmarks"]["questions"] = ids([q for q in space["benchmarks"].get("questions") or [] if isinstance(q, dict)],
                                               question_ids)
    instructions = space.setdefault("instructions", {})
    texts = [i for i in instructions.get("text_instructions") or [] if isinstance(i, dict)]
    if len(texts) > 1:
        merged = "\n\n".join("".join(str(part) for part in (item.get("content") if isinstance(item.get("content"), list)
                                                            else [item.get("content") or ""])).strip() for item in texts)
        texts = [{"id": texts[0].get("id"), "content": merged.splitlines(keepends=True)}]
    if "text_instructions" in instructions:
        instructions["text_instructions"] = ids(texts, instruction_ids)
    for key in ("example_question_sqls", "join_specs"):
        if key in instructions:
            instructions[key] = ids([i for i in instructions.get(key) or [] if isinstance(i, dict)], instruction_ids)
    for join in instructions.get("join_specs") or []:
        errors += _implement_join_spec_problems(join)
    for key, items in list((instructions.get("sql_snippets") or {}).items()):
        instructions["sql_snippets"][key] = ids([i for i in items or [] if isinstance(i, dict)], instruction_ids)
    sources = space.setdefault("data_sources", {})
    tables = [t for t in sources.get("tables") or [] if isinstance(t, dict)]
    if not tables:
        errors.append("the Genie Agent has no tables")
    allowed_lower = {_implement_plain_table(t).lower() for t in allowed_tables or []}
    for kind, key in (("table", "tables"), ("metric view", "metric_views")):
        if key not in sources:
            continue
        items = [s for s in sources.get(key) or [] if isinstance(s, dict)]
        for item in items:
            name = _implement_plain_table(item.get("identifier"))
            if name.lower() not in allowed_lower:
                errors.append(f"the Genie Agent uses {kind} {name}, which is outside this use case")
            if isinstance(item.get("column_configs"), list):
                item["column_configs"] = sorted(item["column_configs"],
                                                key=lambda c: str(c.get("column_name") if isinstance(c, dict) else ""))
        sources[key] = sorted(items, key=lambda s: str(s.get("identifier") or ""))
    if not config["sample_questions"]:
        errors.append("the Genie Agent has no sample questions, so it cannot be smoke-tested")
    sql = _implement_sql_texts(instructions)
    if any(_IMPLEMENT_WRITE_SQL_RE.search(text) for text in sql):
        errors.append("an example SQL of the Genie Agent writes data")
    errors += _implement_flag_count_problems(sql, "an example SQL of the Genie Agent")
    unknown = _implement_unknown_tables(sql, allowed_tables)
    if unknown:
        errors.append(f"the Genie Agent's SQL reads tables outside this use case: {', '.join(unknown)}")
    return title, description, space, errors


def implement_genie_question_problems(definition, sample_questions, user_questions) -> tuple:
    def key(text):
        return " ".join(str(text or "").split()).casefold()

    asked = [" ".join(str(q).split()) for q in user_questions or [] if str(q).strip()]
    present = {key(q) for q in sample_questions or []}
    errors = [f"the space leaves out the user's question {q!r}: include it word for word as a sample question, with an example SQL "
              "that answers it" for q in asked if key(q) not in present]
    try:
        data = json.loads(definition) if isinstance(definition, str) else dict(definition or {})
    except (ValueError, TypeError):
        data = {}
    raw = data.get("question_notes") if isinstance(data, dict) else None
    if raw is None or raw == []:
        return [], errors
    if not isinstance(raw, list):
        return [], errors + ['question_notes must be a list of {"question": ..., "note": ...} objects']
    own = {key(q): q for q in asked}
    notes, noted = [], set()
    for item in raw:
        question = " ".join(str(item.get("question") or "").split()) if isinstance(item, dict) else ""
        note = " ".join(str(item.get("note") or "").split()) if isinstance(item, dict) else ""
        if not question:
            errors.append('each question_notes entry needs a "question" and a "note"')
        elif key(question) not in own:
            errors.append(f"question_notes covers {question!r}, which is not one of the user's questions: rewrite that question so it "
                          "asks about what the business output table holds, and leave it out of question_notes")
        elif not note:
            errors.append(f"the note for the user's question {question!r} is empty: say in one sentence what its answer shows instead")
        elif key(question) not in noted:
            noted.add(key(question))
            notes.append({"question": own[key(question)], "note": note[:500]})
    return notes, errors


def implement_question_note_warning(note: dict) -> str:
    return (f"Genie Agent: your question {note['question']!r} asks about something the results do not hold, so the space "
            f"answers it with what they hold: {note['note']}")


_IMPLEMENT_SQL_QUOTED_RE = re.compile(r"'(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\"|--[^\n]*|/\*.*?\*/", re.S)
_IMPLEMENT_NAME_TOKEN = r"(?:`[^`]+`|[A-Za-z_]\w*)"
_IMPLEMENT_DOTTED_RE = re.compile(r"(?<![\w`.])" + _IMPLEMENT_NAME_TOKEN + r"(?:\s*\.\s*" + _IMPLEMENT_NAME_TOKEN + r")+(?![\w`(])")
_IMPLEMENT_ALIAS_RE = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def _implement_genie_fold(text) -> str:
    return " ".join(str(text or "").split()).casefold()


def _implement_genie_words(value) -> str:
    return " ".join("".join(str(v) for v in value).split() if isinstance(value, list) else str(value or "").split())


def implement_sql_column_refs(sql) -> list:
    plain = _IMPLEMENT_SQL_QUOTED_RE.sub(" ", str(sql or ""))
    return [tuple(part.strip("`") for part in re.findall(_IMPLEMENT_NAME_TOKEN, match.group(0)))
            for match in _IMPLEMENT_DOTTED_RE.finditer(plain)]


def implement_genie_aliases(identifiers) -> dict:
    names = {}
    for identifier in identifiers or []:
        if identifier:
            names.setdefault(_implement_plain_table(identifier).lower(), _implement_plain_table(identifier))
    aliases = {}
    for key in sorted(names):
        parts = names[key].split(".")
        taken = {a.lower() for a in aliases.values()}
        aliases[key] = parts[-1] if parts[-1].lower() not in taken else f"{parts[-2]}_{parts[-1]}"
    return aliases


def implement_genie_column_flags(name: str, data_type, free_text_tokens=()) -> tuple:
    family = _metadata_type_family(data_type)
    if DEEP_PII_NAME_RE.search(str(name or "")) or family not in ("string", "numeric", "temporal", "boolean"):
        return False, False
    words = {_singular_token(t) for t in _name_tokens(name)}
    free = (bool(words & {_singular_token(t) for t in tuple(free_text_tokens) + IMPLEMENT_GENIE_FREE_TEXT_TOKENS})
            or str(name or "").lower() in INSPIRE_AI_NAMES + (INSPIRE_AI_FACTS,))
    return not free, family == "string" and not free and not _is_key_like_column(name)


def implement_genie_synonyms(values, column: str, others=()) -> list:
    blocked = {_implement_genie_fold(n) for name in (column, *others) for n in (name, str(name).replace("_", " "))}
    out, seen = [], set()
    for value in values if isinstance(values, list) else [values] if isinstance(values, str) else []:
        text = " ".join(str(value or "").split())
        key = text.casefold()
        if text and len(text) <= IMPLEMENT_GENIE_SYNONYM_CHARS and key not in blocked and key not in seen:
            seen.add(key)
            out.append(text)
    return out[:IMPLEMENT_GENIE_MAX_SYNONYMS]


def _implement_genie_column_configs(table: dict, known: dict, everything: bool, free_text_tokens, budget: list) -> list:
    identifier, problems, configs = _implement_plain_table(table.get("identifier")), [], {}
    for config in table.get("column_configs") or []:
        name = str((config or {}).get("column_name") or "").strip().strip("`") if isinstance(config, dict) else ""
        if name.lower() not in known:
            problems.append(f"the Genie Agent configures column {name or '(no name)'} of {identifier}, which {identifier} does not have; "
                            f"configure only its real columns: {', '.join(n for n, _t, _c in list(known.values())[:40])}")
        else:
            configs.setdefault(name.lower(), config)
    for key, (name, _kind, _comment) in known.items():
        if key not in configs and (everything or DEEP_PII_NAME_RE.search(name)):
            configs[key] = {}
    out, undescribed = [], []
    for key, config in configs.items():
        name, kind, comment = known[key]
        example_values, entity = implement_genie_column_flags(name, kind, free_text_tokens)
        entity = entity and budget[0] > 0
        budget[0] -= int(entity)
        entry = {"column_name": name}
        description = _implement_genie_words(config.get("description")) or " ".join(str(comment or "").split())
        if description:
            entry["description"] = [description]
        else:
            undescribed.append(name)
        synonyms = implement_genie_synonyms(config.get("synonyms"), name, [n for k, (n, _t, _c) in known.items() if k != key])
        if synonyms:
            entry["synonyms"] = synonyms
        entry["enable_format_assistance"], entry["enable_entity_matching"] = example_values, entity
        out.append(entry)
    if everything and undescribed:
        problems.append(f"describe every column of the business output table {identifier} in business terms, in its column_configs: "
                        f"{', '.join(sorted(undescribed))} {'has' if len(undescribed) == 1 else 'have'} no description")
    table["column_configs"] = sorted(out, key=lambda c: c["column_name"])
    return problems


def _implement_genie_join_problems(instructions: dict, tables: dict, real: dict, aliases: dict, inputs: list) -> list:
    problems, joined = [], set()
    for join in instructions.get("join_specs") or []:
        sides, before = {}, len(problems)
        for side in ("left", "right"):
            ref = join.get(side) if isinstance(join.get(side), dict) else {}
            identifier = _implement_plain_table(ref.get("identifier"))
            if identifier.lower() not in tables:
                problems.append(f"a join spec joins {identifier or '(no table)'}, which is not a table of the Genie Agent")
                continue
            alias = str(ref.get("alias") or "").strip().strip("`") or aliases[identifier.lower()]
            join[side] = {"identifier": tables[identifier.lower()], "alias": alias}
            sides[alias.lower()] = identifier.lower()
        lines = [str(line) for line in join.get("sql") or []]
        if len(sides) != 2:
            if len(problems) == before:
                problems.append("each join spec joins two tables of the Genie Agent with two different aliases")
            continue
        joined.update(sides.values())
        if len(lines) != 2:
            continue
        names = " to ".join(join[side]["alias"] for side in ("left", "right"))
        refs = implement_sql_column_refs(lines[0])
        if not refs:
            problems.append(f"the join spec {names} names no column: write its condition as `alias`.`column` = `alias`.`column`")
        for parts in refs:
            owner = sides.get(parts[0].lower())
            if owner is None or len(parts) != 2:
                problems.append(f"the join spec {names} reads {'.'.join(parts)}: qualify each column with the alias of its left or right "
                                f"table ({', '.join(join[side]['alias'] for side in ('left', 'right'))})")
            elif parts[1].lower() not in real[owner]:
                problems.append(f"the join spec {names} reads {parts[1]}, which {tables[owner]} does not have")
    lonely = [tables[t] for t in inputs if t not in joined]
    if lonely and len(tables) > 1:
        problems.append(f"{', '.join(lonely)} {'joins' if len(lonely) == 1 else 'join'} no other table of the Genie Agent: add a join "
                        "spec for each, on the keys the notebook joins them by")
    return problems


def _implement_genie_snippet_problems(instructions: dict, real: dict, alias_tables: dict) -> list:
    snippets = instructions.get("sql_snippets")
    if snippets is None:
        return []
    if not isinstance(snippets, dict):
        return ["instructions.sql_snippets must be an object with filters, expressions and measures"]
    problems = []
    for kind in list(snippets):
        if kind not in IMPLEMENT_GENIE_SNIPPET_KINDS:
            problems.append(f"sql_snippets.{kind} is not one of {', '.join(IMPLEMENT_GENIE_SNIPPET_KINDS)}")
            continue
        named = set()
        for item in snippets[kind] or []:
            sql = "".join(item.get("sql") or []).strip()
            alias = str(item.get("alias") or "").strip()
            label = f"the {kind[:-1]} {_implement_genie_words(item.get('display_name')) or alias or sql[:60]!r}"
            if kind == "filters":
                item.pop("alias", None)
            elif not _IMPLEMENT_ALIAS_RE.match(alias) or alias.lower() in named:
                problems.append(f"{label} needs its own alias, a name of letters, digits and underscores")
            named.add(alias.lower())
            if not _implement_genie_words(item.get("display_name")):
                if kind == "filters":
                    problems.append(f"{label} needs a display_name")
                else:
                    item["display_name"] = alias.replace("_", " ")
            plain = _IMPLEMENT_SQL_QUOTED_RE.sub(" ", sql)
            if not sql or ";" in plain or re.search(r"\b(SELECT|FROM)\b", plain, re.I) or _IMPLEMENT_WRITE_SQL_RE.search(plain):
                problems.append(f"{label} must be one SQL expression over `table`.`column` names, never a query or a statement")
                continue
            readers, refs = set(), implement_sql_column_refs(sql)
            for parts in refs:
                owner = alias_tables.get(parts[0].lower())
                if owner is None or len(parts) < 2:
                    problems.append(f"{label} reads {'.'.join(parts)}: name each column as `table`.`column` with one of "
                                    f"{', '.join(sorted(alias_tables))}")
                elif parts[1].lower() not in real[owner]:
                    problems.append(f"{label} reads {parts[1]}, which {owner} does not have")
                else:
                    readers.add(owner)
            if not refs:
                problems.append(f"{label} names no column: name each column as `table`.`column`")
            elif len(readers) > 1:
                problems.append(f"{label} reads more than one table; a SQL expression reads one table, so write logic across tables "
                                "as an example SQL")
            item["synonyms"] = implement_genie_synonyms(item.get("synonyms"), alias or "")
            if not item["synonyms"]:
                item.pop("synonyms")
    return problems


def _implement_genie_benchmark_problems(space: dict, asked: set, tables: list) -> list:
    benchmarks = space.get("benchmarks") if isinstance(space.get("benchmarks"), dict) else {}
    problems, kept, seen = [], 0, set()
    for item in benchmarks.get("questions") or []:
        question = _implement_genie_words(item.get("question"))
        answers = [a for a in item.get("answer") or [] if isinstance(a, dict) and str(a.get("format") or "").upper() == "SQL"]
        sql = "".join(str(line) for line in (answers[0].get("content") or [])).strip() if len(answers) == 1 else ""
        if not question or len(answers) != 1 or not sql:
            problems.append(f"the benchmark {question or '(no question)'!r} needs exactly one answer of format SQL with its gold query")
        elif _implement_genie_fold(question) in asked:
            problems.append(f"the benchmark {question!r} repeats a sample question or an example SQL question; a benchmark asks what "
                            "the examples do not answer word for word")
        elif _implement_genie_fold(question) in seen:
            problems.append(f"the benchmark {question!r} is asked twice")
        elif _IMPLEMENT_WRITE_SQL_RE.search(sql):
            problems.append(f"the benchmark SQL for {question!r} writes data")
        elif _implement_unknown_tables([sql], tables):
            problems.append(f"the benchmark SQL for {question!r} reads tables outside the Genie Agent: "
                            f"{', '.join(_implement_unknown_tables([sql], tables))}")
        else:
            seen.add(_implement_genie_fold(question))
            item["answer"] = [{"format": "SQL", "content": implement_join_lines(answers[0].get("content") or [])}]
            kept += 1
    if kept < IMPLEMENT_GENIE_MIN_BENCHMARKS:
        problems.append(f"the Genie Agent has {kept} benchmark question(s) with a gold SQL answer; write at least "
                        f"{IMPLEMENT_GENIE_MIN_BENCHMARKS}, each a question a decision owner asks that the examples do not answer word "
                        "for word, with one answer of format SQL")
    return problems


def implement_genie_space_grounding(space: dict, columns: dict, result_table: str, allowed=(), free_text_tokens=(),
                                    latest_filter: str = "") -> list:
    problems = []
    sources = space.setdefault("data_sources", {})
    instructions = space.setdefault("instructions", {})
    result, reasons = _implement_plain_table(result_table).lower(), implement_reasons_view(result_table).lower()
    tables = {_implement_plain_table(t.get("identifier")).lower(): _implement_plain_table(t.get("identifier"))
              for t in sources.get("tables") or [] if isinstance(t, dict)}
    if result not in tables:
        problems.append(f"the Genie Agent must hold the business output table {_implement_plain_table(result_table)}, the first of its tables")
    real = {}
    for key, identifier in tables.items():
        found = (columns or {}).get(key) or []
        if not found:
            problems.append(f"the columns of {identifier} could not be read, so the Genie Agent cannot be checked against them")
        real[key] = {str(name).lower(): (str(name), kind, comment) for name, kind, comment in found}
    aliases = implement_genie_aliases(list(allowed or ()) + list(tables.values()))
    budget = [IMPLEMENT_GENIE_MAX_ENTITY_COLUMNS]
    for table in sorted((t for t in sources.get("tables") or [] if isinstance(t, dict)),
                        key=lambda t: _implement_plain_table(t.get("identifier")).lower() != result):
        key = _implement_plain_table(table.get("identifier")).lower()
        if isinstance(table.get("description"), str):
            table["description"] = [table["description"]]
        if real.get(key):
            problems += _implement_genie_column_configs(table, real[key], key == result, free_text_tokens, budget)
    problems += _implement_genie_join_problems(instructions, tables, real, {k: aliases[k] for k in tables},
                                               [t for t in tables if t not in (result, reasons)])
    problems += _implement_genie_snippet_problems(instructions, real, {aliases[k].lower(): k for k in tables})
    samples = ["".join(q.get("question") or []) for q in (space.get("config") or {}).get("sample_questions") or []]
    examples = {_implement_genie_fold("".join(e.get("question") or [])) for e in instructions.get("example_question_sqls") or []}
    unanswered = [q for q in samples if _implement_genie_fold(q) not in examples]
    if unanswered:
        problems.append(f"the sample question(s) {', '.join(repr(q) for q in unanswered[:5])} have no example SQL: give each sample "
                        "question an example question SQL with the same question that answers it")
    problems += _implement_genie_benchmark_problems(space, examples | {_implement_genie_fold(q) for q in samples}, list(tables.values()))
    if latest_filter and result in tables:
        fact = (f"The latest run of `{tables[result]}` is its rows where {latest_filter}; answer about the current results from those "
                "rows only, and read earlier runs only when a question asks about the past.")
        texts = instructions.get("text_instructions") or []
        if not texts:
            instructions["text_instructions"] = [{"id": uuid.uuid4().hex, "content": [fact]}]
        elif fact not in "".join(texts[0].get("content") or []):
            texts[0]["content"] = ("".join(texts[0].get("content") or []).rstrip() + "\n\n" + fact).splitlines(keepends=True)
    return problems


def _implement_genie_key_columns(condition: str, sides: dict):
    keys = {alias: [] for alias in sides}
    for part in re.split(r"\s+AND\s+", _IMPLEMENT_SQL_QUOTED_RE.sub(" ", condition).strip(), flags=re.I):
        match = re.fullmatch(r"\(?\s*(" + _IMPLEMENT_NAME_TOKEN + r"\s*\.\s*" + _IMPLEMENT_NAME_TOKEN + r")\s*=\s*("
                             + _IMPLEMENT_NAME_TOKEN + r"\s*\.\s*" + _IMPLEMENT_NAME_TOKEN + r")\s*\)?", part.strip())
        refs = [implement_sql_column_refs(group)[0] for group in match.groups()] if match else []
        if len(refs) != 2 or {r[0].lower() for r in refs} != set(sides):
            return None
        for alias, column in refs:
            keys[alias.lower()].append(column)
    return keys


def implement_result_repeats_sql(result_table: str, columns, suffix: str = "") -> str:
    keys = ", ".join(f"`{c}`" for c in columns)
    present = " AND ".join(f"`{c}` IS NOT NULL" for c in columns)
    return f"SELECT 1 AS repeated FROM {inspire_table_name(result_table, suffix)} WHERE {present} GROUP BY {keys} HAVING COUNT(*) > 1"


def implement_genie_space_checks(space: dict, aliases: dict, result_table: str) -> list:
    checks = []
    instructions = space.get("instructions") or {}
    for example in instructions.get("example_question_sqls") or []:
        question = "".join(example.get("question") or [])
        checks.append({"label": f"the example SQL for {question!r}", "sql": "".join(example.get("sql") or []), "raw_json": True})
    for item in (space.get("benchmarks") or {}).get("questions") or []:
        question = "".join(item.get("question") or [])
        sql = "".join("".join(a.get("content") or []) for a in item.get("answer") or [] if isinstance(a, dict))
        checks.append({"label": f"the benchmark SQL for {question!r}", "sql": sql, "limit": 0})
    by_alias = {alias.lower(): table for table, alias in aliases.items()}
    for kind, items in (instructions.get("sql_snippets") or {}).items():
        for item in items or []:
            sql = "".join(item.get("sql") or []).strip()
            owner = next((by_alias[p[0].lower()] for p in implement_sql_column_refs(sql) if p[0].lower() in by_alias), None)
            if not owner:
                continue
            source = f"{inspire_table_name(owner)} AS `{aliases[owner]}`"
            label = f"the {kind[:-1]} {_implement_genie_words(item.get('display_name')) or item.get('alias')!r}"
            checks.append({"label": label, "limit": 0, "sql": (f"SELECT 1 AS matched FROM {source} WHERE {sql} LIMIT 0" if kind == "filters"
                                                               else f"SELECT {sql} AS inspire_check FROM {source} LIMIT 0")})
    owned = {_implement_plain_table(result_table).lower(): "", implement_reasons_view(result_table).lower(): "_reasons"}
    for join in instructions.get("join_specs") or []:
        lines = [str(line) for line in join.get("sql") or []]
        left, right = join.get("left") or {}, join.get("right") or {}
        if len(lines) != 2 or not left.get("alias") or not right.get("alias"):
            continue
        names = f"{left['alias']} to {right['alias']}"
        pair = (f"{inspire_table_name(left['identifier'])} AS `{left['alias']}` JOIN {inspire_table_name(right['identifier'])} "
                f"AS `{right['alias']}`")
        checks.append({"label": f"the join spec {names}", "limit": 0, "sql": f"SELECT 1 AS joined FROM {pair} ON {lines[0].strip()} LIMIT 0"})
        keys = _implement_genie_key_columns(lines[0], {left["alias"].lower(): "left", right["alias"].lower(): "right"})
        kind = lines[1].strip()[len("--rt=FROM_RELATIONSHIP_TYPE_"):-2] if lines[1].strip() in IMPLEMENT_JOIN_TAGS else ""
        unique = {"MANY_TO_ONE": (right,), "ONE_TO_MANY": (left,), "ONE_TO_ONE": (left, right)}.get(kind, ())
        for ref in unique if keys else ():
            suffix = owned.get(_implement_plain_table(ref["identifier"]).lower())
            if suffix is None:
                continue
            columns = keys[ref["alias"].lower()]
            checks.append({"label": f"the join spec {names}", "expect": "none", "sql": implement_result_repeats_sql(result_table, columns, suffix),
                           "problem": (f"the join spec {names} is tagged {kind}, so each {', '.join(columns)} of {ref['identifier']} "
                                       "must be on one row, but some repeat across its rows and runs: choose the tag that matches the "
                                       "key's cardinality")})
    return checks


IMPLEMENT_APP_SHELL_VERSION = 1
IMPLEMENT_APP_SHELL_MODULES = ("shell_sql", "shell_db", "shell_pg", "shell_core")
IMPLEMENT_APP_PAGE_FILE = "page.json"
IMPLEMENT_APP_CHECK_DIR = ".inspire_check"
IMPLEMENT_APP_CHECK_SCRIPT = "check_page.py"
IMPLEMENT_APP_ACCESS = ("user", "sp")
IMPLEMENT_APP_LOOK = {"style": "a", "layout": "sidebar", "palette": "default", "theme": "light"}
IMPLEMENT_APP_LOOK_CHOICES = {"style": ("a", "b", "c"), "layout": ("sidebar", "tabs", "home"),
                              "palette": ("default", "muted", "vivid", "ocean", "sunset", "earth", "jewel", "nordic", "berry", "pop",
                                          "neon", "solid", "mono"),
                              "theme": ("light", "dark")}
IMPLEMENT_APP_OWNER_RE = re.compile(r"^Inspire use case (\S+) \(session (\d+)\)")
IMPLEMENT_GENIE_MAX_TABLES = 50
IMPLEMENT_APP_OPTIONS = {"modify": False, "replace_edits": False, "decisions": False}
IMPLEMENT_APP_USER_OFF = ("decisions",)
IMPLEMENT_APP_RETIRED_OPTIONS = ("download",)
IMPLEMENT_APP_CLUSTER_COLUMNS = 4
IMPLEMENT_APP_CLUSTER_TYPES = ("date", "timestamp", "timestamp_ntz", "string", "varchar", "char", "int", "bigint", "smallint",
                               "tinyint", "float", "double", "decimal")
IMPLEMENT_APP_JOB_PERMISSION = "CAN_MANAGE_RUN"
IMPLEMENT_APP_CHECK_CHOICES = {"app_access": "sp"}
IMPLEMENT_APP_READ_SCOPES = ("sql:restricted-query", "genie")
IMPLEMENT_APP_EDIT_SCOPES = ("sql", "genie")
IMPLEMENT_APP_CHECK_HEADERS = {"x-forwarded-access-token": "inspire-check", "x-forwarded-email": "inspire-check"}
IMPLEMENT_APP_BACKENDS = ("lakebase", "delta")
IMPLEMENT_LAKEBASE_PROJECT = "inspire-apps"
IMPLEMENT_LAKEBASE_BRANCH = f"projects/{IMPLEMENT_LAKEBASE_PROJECT}/branches/production"
IMPLEMENT_LAKEBASE_ENDPOINT = f"{IMPLEMENT_LAKEBASE_BRANCH}/endpoints/primary"
IMPLEMENT_LAKEBASE_DATABASE = "databricks_postgres"
IMPLEMENT_LAKEBASE_DATABASE_RESOURCE = f"{IMPLEMENT_LAKEBASE_BRANCH}/databases/databricks-postgres"
IMPLEMENT_LAKEBASE_CATALOG = "inspire_lakebase"
IMPLEMENT_LAKEBASE_PROJECT_SPEC = {"display_name": "Inspire Apps", "default_endpoint_settings": {
    "autoscaling_limit_min_cu": 0.5, "autoscaling_limit_max_cu": 2, "suspend_timeout_duration": "300s"}}
IMPLEMENT_LAKEBASE_TASK_KEY = "inspire_lakebase_sync"
IMPLEMENT_LAKEBASE_RESOURCE = "postgres"
IMPLEMENT_LAKEBASE_TYPES = {"bigint": "LONG", "integer": "INT", "smallint": "SHORT", "tinyint": "BYTE", "varchar": "STRING",
                            "char": "STRING"}
IMPLEMENT_LAKEBASE_UNSUPPORTED = ("GEOGRAPHY", "GEOMETRY", "OBJECT", "VOID")
IMPLEMENT_LAKEBASE_ONLINE = ("SYNCED_TABLE_ONLINE", "SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE")
IMPLEMENT_LAKEBASE_TIMEOUT_S = 1800
IMPLEMENT_LAKEBASE_POLL_S = 10
IMPLEMENT_ALERT_VIBES_FILE = "vibes_not_done.json"
IMPLEMENT_ARTIFACT_FILES = {"genie_space": "genie_space.json", "dashboard": IMPLEMENT_APP_PAGE_FILE, "app": IMPLEMENT_APP_PAGE_FILE,
                            "alert": "alert_condition.sql"}


def implement_output_vibes(outputs, name: str) -> str:
    return str((outputs or {}).get(f"{name}_vibes") or "").strip()


def implement_vibes_block(outputs, name: str, where: str) -> str:
    vibes = implement_output_vibes(outputs, name)
    if not vibes:
        return ""
    return (f"{USER_DIRECTIVE_SUPREMACY_PREAMBLE}The customer's asks for this {IMPLEMENT_OUTPUT_LABELS[name]}, word for word:\n"
            f"\"\"\"\n{vibes}\n\"\"\"\nThese asks are explicit user directives: where they conflict with any guideline, rule, default "
            f"or example in this prompt, the customer's asks win, without exception. Inspire's checks still run on what you write and "
            f"decide what ships: every query must run and pass them, nothing may write data or read a table this prompt does not "
            f"name, Inspire reads the input tables through their metadata only, and the business output table rules above still hold.\n"
            f"If an ask cannot be done within those checks, do every other ask and list each one you could not do in {where}, as "
            f'[{{"ask": "<the ask, word for word>", "reason": "<one sentence on why>"}}]; leave it out when you did every ask.\n')


def implement_vibes_not_done(data, outputs, name: str) -> tuple:
    value = data.pop(IMPLEMENT_VIBES_NOT_DONE, None) if isinstance(data, dict) else None
    if not implement_output_vibes(outputs, name) or value is None or value == []:
        return [], []
    if not isinstance(value, list):
        return [], [f'{IMPLEMENT_VIBES_NOT_DONE} must be a list of {{"ask": ..., "reason": ...}} objects']
    warnings, errors = [], []
    for item in value:
        item = item if isinstance(item, dict) else {}
        ask, reason = (" ".join(str(item.get(key) or "").split()) for key in ("ask", "reason"))
        warning = f"{IMPLEMENT_OUTPUT_LABELS[name]} vibe not done: {ask[:300]}: {reason[:500]}"
        if not ask or not reason:
            errors.append(f'each {IMPLEMENT_VIBES_NOT_DONE} entry needs an "ask" and a "reason"')
        elif warning not in warnings:
            warnings.append(warning)
    return warnings, sorted(set(errors))


def implement_artifact_files(name: str, artifact) -> dict:
    text = (str((artifact or {}).get("condition_sql") or "") if name == "alert"
            else json.dumps(artifact, indent=1, ensure_ascii=False))
    return {IMPLEMENT_ARTIFACT_FILES[name]: text}


def implement_app_shell_payload(text: str) -> dict:
    try:
        data = json.loads(zlib.decompress(base64.b64decode(text)).decode("utf-8")) if text else {}
    except (ValueError, zlib.error) as error:
        raise RuntimeError(f"the App shell embedded in this notebook does not decode: {error}")
    files = data.get("files") if isinstance(data, dict) else None
    if (not isinstance(files, dict) or data.get("version") != IMPLEMENT_APP_SHELL_VERSION or not isinstance(data.get("example"), dict)
            or any(f"{name}.py" not in files for name in IMPLEMENT_APP_SHELL_MODULES) or "static/index.html" not in files):
        raise RuntimeError("this notebook carries no usable App shell; embed it with `python3 agent/app_shell/embed.py write`")
    return data


def implement_write_files(root: str, files: dict):
    for relative, text in files.items():
        target = os.path.join(root, relative)
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with open(target, "w", encoding="utf-8") as handle:
            handle.write(text)


def implement_app_check_files(shell_sql_source: str, event_date: bool) -> dict:
    script = ("import json, os, sys\n"
              "here = os.path.dirname(os.path.abspath(__file__))\n"
              "sys.path.insert(0, here)\n"
              "import shell_sql\n"
              "try:\n"
              f"    with open(os.path.join(os.path.dirname(here), {IMPLEMENT_APP_PAGE_FILE!r}), encoding='utf-8') as handle:\n"
              "        page = json.load(handle)\n"
              "except (OSError, ValueError) as error:\n"
              f"    print({IMPLEMENT_APP_PAGE_FILE!r} + ' cannot be read: ' + str(error))\n"
              "    sys.exit(1)\n"
              f"problems = shell_sql.spec_problems(page) or shell_sql.quality_problems(page, {bool(event_date)})\n"
              "print('\\n'.join(problems) or 'no problems')\n"
              "sys.exit(1 if problems else 0)\n")
    return {f"{IMPLEMENT_APP_CHECK_DIR}/shell_sql.py": shell_sql_source, f"{IMPLEMENT_APP_CHECK_DIR}/{IMPLEMENT_APP_CHECK_SCRIPT}": script}


def implement_app_shell_modules(folder: str, files: dict) -> dict:
    import importlib
    implement_write_files(folder, {f"{name}.py": files[f"{name}.py"] for name in IMPLEMENT_APP_SHELL_MODULES})
    for name in IMPLEMENT_APP_SHELL_MODULES:
        sys.modules.pop(name, None)
    if folder in sys.path:
        sys.path.remove(folder)
    sys.path.insert(0, folder)
    importlib.invalidate_caches()
    bytecode, sys.dont_write_bytecode = sys.dont_write_bytecode, True
    try:
        return {name: importlib.import_module(name) for name in IMPLEMENT_APP_SHELL_MODULES}
    finally:
        sys.dont_write_bytecode = bytecode


def implement_app_page_id(uc_id: str) -> str:
    ident = re.sub(r"[^a-z0-9]+", "_", str(uc_id or "").lower()).strip("_")[:60]
    return ident if re.match(r"^[a-z_]", ident) else f"p_{ident}"


def implement_app_page(page, uc_id: str, use_case: dict, result_table: str = "") -> dict:
    if not isinstance(page, dict):
        raise RuntimeError(f"{IMPLEMENT_APP_PAGE_FILE} must hold one JSON object")
    out = dict(page, id=implement_app_page_id(uc_id), use_case_id=uc_id)
    out.pop("result_table", None)
    if result_table:
        out["result_table"] = result_table
    for key, column in (("domain", "Business Domain"), ("subdomain", "Subdomain"), ("sponsor", "Sponsor")):
        value = " ".join(str((use_case or {}).get(column) or "").split())
        if value:
            out[key] = value[:80]
    return out


def implement_app_page_problems(page: dict, shell_sql, allowed, event_date: bool) -> list:
    problems = shell_sql.spec_problems(page)
    if not any(isinstance(d, dict) and d.get("name") == shell_sql.RECORDS_DATASET for d in page.get("datasets") or []):
        problems.append(f"the page has no dataset named {shell_sql.RECORDS_DATASET}: it is the decision list, every row of the "
                        f"latest run, with a records section that names its columns")
    if problems:
        return problems
    problems = shell_sql.quality_problems(page, event_date)
    texts = [str(d.get("sql") or "") for d in page["datasets"]]
    unknown = _implement_unknown_tables(texts, allowed)
    if unknown:
        problems.append(f"the datasets read tables outside this use case: {', '.join(unknown)}")
    expressions = ([str(k.get("expression") or "") for k in page.get("kpis") or []]
                   + [str(y.get("expression") or "") for c in page.get("charts") or [] for y in c.get("y") or []])
    return problems + _implement_flag_count_problems(texts + expressions, "the page")


def _implement_plain_words(text) -> str:
    return " ".join(str(text or "").split()).lower()


def implement_fewer_charts_problems(page: dict, vibes: str, label: str, minimum: int) -> list:
    quote = page.get("fewer_charts")
    if quote is None or (isinstance(quote, str) and not quote.strip()):
        return []
    if not isinstance(quote, str):
        return [f"fewer_charts must be text: the words of the customer's asks for this {label} that ask for fewer than {minimum} "
                "charts, word for word"]
    if not _implement_plain_words(vibes):
        return [f"fewer_charts quotes {quote.strip()[:200]!r}, but the customer gave no asks for this {label}: write at least "
                f"{minimum} charts and leave fewer_charts out"]
    if _implement_plain_words(quote) not in _implement_plain_words(vibes):
        return [f"fewer_charts quotes {quote.strip()[:200]!r}, which is not in the customer's asks for this {label} word for word: "
                f"quote the words that ask for fewer charts, or write at least {minimum} charts and leave fewer_charts out"]
    return []


def implement_app_read_tables(page: dict, allowed) -> list:
    by_name = {_implement_plain_table(t).lower(): t for t in allowed or []}
    out = []
    for dataset in page.get("datasets") or []:
        for ref in implement_table_refs(str(dataset.get("sql") or "")):
            table = by_name.get(ref.lower())
            if table and table not in out:
                out.append(table)
    return out


def implement_cluster_columns(page: dict, table_schema) -> list:
    kinds = {str(name).lower(): (str(name), " ".join(str(kind or "").split()).lower()) for name, kind in table_schema or []}
    wanted = ["as_of_date"] + [str(column) for f in (page or {}).get("filters") or [] if isinstance(f, dict)
                               for column in ((f.get("columns") or {}).values() if isinstance(f.get("columns"), dict) else ())]
    out = []
    for column in wanted:
        name, kind = kinds.get(column.lower(), ("", ""))
        if name and kind.split("(")[0] in IMPLEMENT_APP_CLUSTER_TYPES and name not in out:
            out.append(name)
    return out[:IMPLEMENT_APP_CLUSTER_COLUMNS]


def implement_cluster_detail_sql(result_table: str) -> str:
    return f"DESCRIBE DETAIL {inspire_table_name(result_table)}"


def implement_cluster_sql(result_table: str, columns) -> str:
    return f"ALTER TABLE {inspire_table_name(result_table)} CLUSTER BY ({', '.join(inspire_column(c) for c in columns)})"


def implement_app_choices(outputs) -> tuple:
    outputs = outputs if isinstance(outputs, dict) else {}
    errors = []
    raw_access = outputs.get("app_access")
    text = _implement_text_value(raw_access)
    access = (text or "").lower()
    if access not in IMPLEMENT_APP_ACCESS:
        errors.append(f"choose who the App reads its data as: 'user' (each person sees only the rows and columns their Unity Catalog "
                      f"grants allow) or 'sp' (everyone sees the same data through the App's service principal)"
                      + (f", not {_implement_shown(raw_access)}" if text != "" else ""))
        access = ""
    look = dict(IMPLEMENT_APP_LOOK)
    for key, value in _implement_mapping(outputs.get("app_look"), "outputs.app_look", errors).items():
        choice = _implement_text_value(value)
        choice = None if choice is None else choice.lower()
        if key not in IMPLEMENT_APP_LOOK_CHOICES:
            errors.append(f"the App look has no setting {_implement_shown(key)}; it sets only {', '.join(IMPLEMENT_APP_LOOK_CHOICES)}")
        elif choice is None or (choice and choice not in IMPLEMENT_APP_LOOK_CHOICES[key]):
            errors.append(f"the App {key} {_implement_shown(value)} is not one Inspire offers; pick one of "
                          f"{', '.join(IMPLEMENT_APP_LOOK_CHOICES[key])}")
        elif choice:
            look[key] = choice
    options = dict(IMPLEMENT_APP_OPTIONS)
    for key, value in _implement_mapping(outputs.get("app_options"), "outputs.app_options", errors).items():
        if key not in IMPLEMENT_APP_OPTIONS:
            errors.append(f"the App has no option {_implement_shown(key)}; its options are {', '.join(IMPLEMENT_APP_OPTIONS)}")
        elif value is not None and not isinstance(value, bool):
            errors.append(f"the App option {key} must be true or false, not {_implement_shown(value)}")
        else:
            options[key] = value is True
    if access == "user":
        options.update(dict.fromkeys(IMPLEMENT_APP_USER_OFF, False))
    if not options["modify"]:
        options["replace_edits"] = False
    raw_backend = outputs.get("app_backend")
    backend = _implement_text_value(raw_backend)
    backend = IMPLEMENT_APP_BACKENDS[-1] if backend == "" else None if backend is None else backend.lower()
    if backend not in IMPLEMENT_APP_BACKENDS:
        errors.append(f"choose where the App reads its data: 'lakebase' (its pages load instantly) or 'delta' (Delta tables, its pages "
                      f"take a few seconds to load), not {_implement_shown(raw_backend)}")
        backend = IMPLEMENT_APP_BACKENDS[-1]
    return {"app_access": access, "app_look": look, "app_options": options, "app_backend": backend}, errors


def implement_current_app_options(options: dict) -> dict:
    return {k: v for k, v in options.items() if k not in IMPLEMENT_APP_RETIRED_OPTIONS}


def implement_stored_spec(spec):
    outputs = spec.get("outputs") if isinstance(spec, dict) else None
    if not isinstance(outputs, dict) or not isinstance(outputs.get("app_options"), dict):
        return spec
    return dict(spec, outputs=dict(outputs, app_options=implement_current_app_options(outputs["app_options"])))


def implement_app_held_settings(settings):
    if not isinstance(settings, dict) or not isinstance(settings.get("options"), dict):
        return settings
    return dict(settings, options=implement_current_app_options(settings["options"]))


def implement_app_admin_warnings(settings: dict) -> list:
    if settings.get("auth") != "user" or str(settings.get("admin") or "").strip():
        return []
    return ["App: no one can use Refresh now in this App, because each viewer signs in as themselves and Inspire could not read "
            "the identity it runs as to name the person who built it; rebuild the App to name them"]


def implement_app_user_scopes(outputs) -> list:
    outputs = outputs if isinstance(outputs, dict) else {}
    if outputs.get("app_access") != "user":
        return []
    return list(IMPLEMENT_APP_EDIT_SCOPES if (outputs.get("app_options") or {}).get("modify") else IMPLEMENT_APP_READ_SCOPES)


def implement_app_jobs(jobs) -> dict:
    out = {}
    for page_id, job_id in (jobs.items() if isinstance(jobs, dict) else ()):
        text = "" if isinstance(job_id, bool) or job_id is None else str(job_id).strip()
        if re.fullmatch(r"[0-9]+", text):
            out[str(page_id)] = text
    return out


def implement_app_kept_jobs(held, page_ids, page_id: str = "", job_id=None) -> dict:
    ids = [str(p) for p in page_ids]
    jobs = {p: j for p, j in implement_app_jobs((held or {}).get("jobs")).items() if p in ids}
    jobs.update({p: j for p, j in implement_app_jobs({page_id: job_id}).items() if p in ids})
    return jobs


def implement_app_job_resource(page_id: str, job_id: str) -> dict:
    return {"name": f"job-{str(page_id).replace('_', '-')}", "job": {"id": str(job_id), "permission": IMPLEMENT_APP_JOB_PERMISSION}}


def _implement_holds_job(resource, wanted: dict) -> bool:
    job = resource.get("job") if isinstance(resource, dict) else None
    return (isinstance(job, dict) and resource.get("name") == wanted["name"] and str(job.get("id")) == wanted["job"]["id"]
            and str(job.get("permission") or "").upper() == IMPLEMENT_APP_JOB_PERMISSION)


def implement_app_missing_jobs(resources, jobs) -> list:
    held = list(resources or [])
    return [job_id for page_id, job_id in implement_app_jobs(jobs).items()
            if not any(_implement_holds_job(r, implement_app_job_resource(page_id, job_id)) for r in held)]


def implement_app_resources(resources, jobs, postgres: bool = False) -> list:
    out = list(resources or [])
    for page_id, job_id in implement_app_jobs(jobs).items():
        wanted = implement_app_job_resource(page_id, job_id)
        names = [r.get("name") if isinstance(r, dict) else None for r in out]
        if wanted["name"] not in names:
            out.append(wanted)
        elif not _implement_holds_job(out[names.index(wanted["name"])], wanted):
            out[names.index(wanted["name"])] = wanted
    if postgres and not implement_app_holds_postgres(out):
        out = [r for r in out if not (isinstance(r, dict) and r.get("name") == IMPLEMENT_LAKEBASE_RESOURCE)]
        out.append(implement_app_postgres_resource())
    return out


def implement_app_postgres_resource() -> dict:
    return {"name": IMPLEMENT_LAKEBASE_RESOURCE, "postgres": {"branch": IMPLEMENT_LAKEBASE_BRANCH,
                                                              "database": IMPLEMENT_LAKEBASE_DATABASE_RESOURCE,
                                                              "permission": "CAN_CONNECT_AND_CREATE"}}


IMPLEMENT_APP_YAML = "app.yaml"
IMPLEMENT_APP_WAREHOUSE_RESOURCE = "sql-warehouse"
IMPLEMENT_APP_POSTGRES_ENV = ("PGAPPNAME", "PGDATABASE", "PGHOST", "PGPORT", "PGSSLMODE", "PGUSER")
IMPLEMENT_LAKEBASE_APP_ENV = {"name": "LAKEBASE_ENDPOINT", "valueFrom": IMPLEMENT_LAKEBASE_RESOURCE}
IMPLEMENT_APP_ENV_ENTRY = re.compile(r"^[ ]*-[ ]*name:[ ]*([A-Za-z_][A-Za-z0-9_]*)[ ]*\n[ ]*(value|valueFrom):[ ]*(\S[^\n]*?)[ ]*$", re.M)


def implement_app_yaml_env(text) -> list:
    return [{"name": name, kind: value.strip("'\"")} for name, kind, value in IMPLEMENT_APP_ENV_ENTRY.findall(str(text or ""))]


def implement_app_yaml(text, lakebase: bool) -> str:
    text = str(text or "")
    if not lakebase or IMPLEMENT_LAKEBASE_APP_ENV in implement_app_yaml_env(text):
        return text
    if not re.search(r"^env:[ ]*$", text, re.M):
        text = text.rstrip("\n") + "\nenv:\n"
    return (text.rstrip("\n") + f"\n  - name: {IMPLEMENT_LAKEBASE_APP_ENV['name']}\n"
            f"    valueFrom: {IMPLEMENT_LAKEBASE_APP_ENV['valueFrom']}\n")


def implement_app_runtime_env(text, resolved: dict, injected: dict) -> dict:
    env = dict(injected or {})
    for entry in implement_app_yaml_env(text):
        if "value" in entry:
            env[entry["name"]] = entry["value"]
        elif entry["valueFrom"] in (resolved or {}):
            env[entry["name"]] = resolved[entry["valueFrom"]]
    return env


def implement_app_holds_postgres(resources) -> bool:
    wanted = implement_app_postgres_resource()["postgres"]
    return any(isinstance(r, dict) and isinstance(r.get("postgres"), dict) and r.get("name") == IMPLEMENT_LAKEBASE_RESOURCE
               and all(str(r["postgres"].get(k) or "") == v for k, v in wanted.items() if k != "permission")
               for r in resources or [])


class ImplementLakebaseUnavailable(Exception):
    pass


def implement_lakebase_unavailable(error, project: str = "") -> str:
    code = str(getattr(error, "error_code", "") or "")
    if implement_is_permission_denied(error) and project:
        return (f"the identity Inspire runs as has no Can Use or Can Manage on the Lakebase project {project}; run the Inspire "
                "installer again, or ask a workspace admin to give it Can Manage on that project")
    if implement_is_permission_denied(error):
        return "the identity Inspire runs as has no permission to use Lakebase in this workspace"
    if implement_is_not_found(error) or code in ("FEATURE_DISABLED", "NOT_IMPLEMENTED"):
        return "Lakebase is not available in this workspace"
    return f"Lakebase did not answer: {get_clean_error_message(error)[:300]}"


def implement_lakebase_schema_denied(synced: str, builder: str = "") -> str:
    catalog, schema = str(synced).split(".")[:2]
    ask = (f"ask its owner to run GRANT ALL PRIVILEGES ON CATALOG `{catalog}` TO `{builder}`" if builder else
           "ask its owner to give the identity Inspire runs as ALL PRIVILEGES on that catalog")
    return (f"the identity Inspire runs as cannot add tables to the Lakebase schema {catalog}.{schema}; run the Inspire installer "
            f"again as an owner of the catalog {catalog}, or {ask}")


def implement_lakebase_column(name: str, text: str) -> dict:
    text = " ".join(str(text or "").split())
    head = re.split(r"[<(\s]", text.lower(), maxsplit=1)[0]
    return {"name": str(name), "type": IMPLEMENT_LAKEBASE_TYPES.get(head, head.upper()), "type_text": text.upper()}


def implement_lakebase_schema(app_name: str) -> str:
    return re.sub(r"[^a-z0-9_]", "_", str(app_name or "").lower())[:63]


def implement_lakebase_copies(pages, row: str) -> list:
    out = []
    for page in pages:
        result_table = ".".join(inspire_table_parts(page["result_table"]))
        out.append({"page": page["id"], "dataset": "", "key": page["result_table"], "result_table": result_table, "suffix": "__lb",
                    "view": f"{result_table}__lb", "body": f"SELECT * FROM {inspire_table_name(result_table)}"})
        for dataset in page.get("datasets") or []:
            suffix = f"__lb_{dataset['name']}"
            out.append({"page": page["id"], "dataset": dataset["name"], "key": f"{result_table}{suffix}", "result_table": result_table,
                        "suffix": suffix, "view": f"{result_table}{suffix}", "body": str(dataset["sql"])})
    for copy in out:
        copy["sql"] = implement_lakebase_view_sql(copy["result_table"], copy["suffix"], copy["body"], row)
    return out


def implement_lakebase_view_sql(result_table: str, suffix: str, body: str, row: str) -> str:
    return (f"CREATE OR REPLACE VIEW {inspire_table_name(result_table + suffix)} AS SELECT d.*, monotonically_increasing_id() AS "
            f"`{row}` FROM ({str(body).strip().rstrip(';')}) AS d")


def implement_lakebase_pg_table(copy: dict, columns: list) -> str:
    digest = hashlib.sha256(json.dumps([[c["name"], c["type_text"]] for c in columns]).encode()).hexdigest()[:8]
    base = f"{copy['page']}__d_{copy['dataset']}" if copy["dataset"] else f"{copy['page']}__t"
    return f"{re.sub(r'[^a-z0-9_]', '_', base.lower())[:54]}_{digest}"


def implement_lakebase_type_problems(copy: dict, columns: list) -> list:
    bad = [c["name"] for c in columns if c["type"] in IMPLEMENT_LAKEBASE_UNSUPPORTED]
    what = f"dataset {copy['dataset']} of page {copy['page']}" if copy["dataset"] else copy["key"]
    return [f"{what} has columns Lakebase cannot hold: {', '.join(bad)}"] if bad else []


def implement_lakebase_backend(pipeline_id: str, copies: list) -> dict:
    tables, datasets = {}, {}
    for copy in copies:
        tables[copy["key"]] = {"pg": copy["pg"], "columns": copy["columns"]}
        if copy["dataset"]:
            datasets.setdefault(copy["page"], {})[copy["dataset"]] = copy["key"]
    return {"kind": IMPLEMENT_APP_BACKENDS[0], "pipeline_id": pipeline_id, "tables": tables, "datasets": datasets}


def implement_lakebase_pg_names(backend) -> set:
    if not isinstance(backend, dict) or backend.get("kind") != IMPLEMENT_APP_BACKENDS[0]:
        return set()
    return {str(entry.get("pg") or "") for entry in (backend.get("tables") or {}).values() if isinstance(entry, dict)} - {""}


def implement_app_wanted_backend(outputs: dict, host: dict) -> tuple:
    held = (host.get("settings") or {}) if host.get("merged") else {}
    backend = held.get("backend") if host.get("merged") else {"kind": (outputs or {}).get("app_backend")}
    kind = (backend or {}).get("kind") if (backend or {}).get("kind") in IMPLEMENT_APP_BACKENDS else IMPLEMENT_APP_BACKENDS[-1]
    access = held.get("auth") if host.get("merged") else (outputs or {}).get("app_access")
    if kind == IMPLEMENT_APP_BACKENDS[0] and access != "sp":
        return IMPLEMENT_APP_BACKENDS[-1], ("App: it reads Delta tables, not Lakebase, because each viewer signs in as themselves and "
                                            "Lakebase does not carry Unity Catalog row filters, column masks or grants to each viewer")
    return kind, ""


def implement_lakebase_fallback(reason: str) -> str:
    return (f"App: it reads Delta tables, so its pages take a few seconds to load, because Inspire could not set up Lakebase: "
            f"{' '.join(str(reason).split())[:600]}. Rebuild the App, or pick Lakebase again in the wizard, to switch it to Lakebase")


def implement_app_settings(version: int, title: str, page_ids, genie: dict, genie_note: str, built_at: str, outputs=None, jobs=None,
                           admin: str = "") -> dict:
    choices, problems = implement_app_choices(outputs)
    if problems:
        raise ImplementBlocked("the App cannot be built with the look and access chosen in the wizard: " + "; ".join(problems))
    ids = list(page_ids)
    return {"version": version, "title": " ".join(str(title or "").split())[:80] or "Inspire App", "subtitle": "",
            **choices["app_look"], "auth": choices["app_access"], "options": choices["app_options"], "genie": genie or None,
            "genie_note": "" if genie else genie_note, "pages": ids, "jobs": implement_app_kept_jobs({"jobs": jobs}, ids),
            "admin": " ".join(str(admin or "").split()), "built_at": built_at}


def implement_app_source(files: dict, settings_file: str, pages_dir: str, settings: dict, pages) -> dict:
    out = dict(files)
    out[settings_file] = json.dumps(settings, indent=2, ensure_ascii=False) + "\n"
    for page in pages:
        out[f"{pages_dir}/{page['id']}.json"] = json.dumps(page, indent=2, ensure_ascii=False) + "\n"
    return out


def _implement_page_key(text) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(text or "").lower())


def implement_page_request_problems(page: dict, requested) -> list:
    covered = set()
    for f in page.get("filters") or []:
        if isinstance(f, dict):
            covered |= {_implement_page_key(f.get("id")), _implement_page_key(f.get("label"))}
            covered |= {_implement_page_key(column) for column in (f.get("columns") or {}).values()}
    return [f"the user asked the dashboard to filter on {name}, which no filter of the page covers: add a filter on it"
            for name in requested or [] if _implement_page_key(name) not in covered]


IMPLEMENT_DASHBOARD_GRID = 12
IMPLEMENT_DASHBOARD_HEIGHTS = {"text": 2, "link": 1, "filter": 2, "counter": 3, "chart": 6, "table": 7, "records": 9}
IMPLEMENT_DASHBOARD_FILTERS_PER_ROW = 4
IMPLEMENT_DASHBOARD_PIE_PARTS = 6
IMPLEMENT_DASHBOARD_REASONS_SEPARATOR = " · "
IMPLEMENT_DASHBOARD_GENIE = {"isEnabled": True, "enablementMode": "ENABLED"}
IMPLEMENT_DASHBOARD_RUN_HEALTH = "run_health"
IMPLEMENT_DASHBOARD_RECORD_LABELS = {"rank": "Rank", "label": "Name", "tier": "Priority tier", "reasons": "Reasons",
                                     "action": "Recommended action"}


def implement_dashboard_format(item: dict, measure: bool = True):
    kind, code = (item or {}).get("format", "number"), str((item or {}).get("currency") or "")
    places = {"type": "max", "places": 2}
    if kind == "currency" and re.fullmatch(r"[A-Z]{3}", code):
        return {"type": "number-currency", "currencyCode": code, "abbreviation": "compact", "decimalPlaces": places}
    if kind == "currency":
        kind = "number"
    if kind == "percent":
        return {"type": "number-percent" if measure else "number", "decimalPlaces": {"type": "max", "places": 1}}
    if kind == "integer":
        return {"type": "number", "decimalPlaces": {"type": "max", "places": 0}}
    if kind == "number":
        return {"type": "number", "abbreviation": "compact", "decimalPlaces": places}
    return None


def implement_dashboard_chart_type(chart: dict, rows) -> str:
    kind, x = chart.get("type"), chart.get("x") or {}
    if kind in ("scatter", "heatmap"):
        return kind
    if x.get("kind") == "time":
        return "area" if kind in ("area", "stacked_area", "stacked_bar") else "line"
    if kind == "donut" and len(rows or []) <= IMPLEMENT_DASHBOARD_PIE_PARTS:
        return "pie"
    return "bar"


def implement_dashboard_theme(palettes, palette, hue=None):
    chosen = (palettes or {}).get(palette) if palette and palette != IMPLEMENT_APP_LOOK["palette"] else None
    if not isinstance(chosen, dict):
        return None
    light, dark = chosen.get("light") or {}, chosen.get("dark") or {}

    def pair(key):
        return {"light": light["base"][key], "dark": dark["base"][key]}

    hues = list((light.get("hues") or {}).items())
    hues.sort(key=lambda item: item[0] != hue)
    return {"canvasBackgroundColor": pair("bg"), "widgetBackgroundColor": pair("card"), "widgetBorderColor": pair("line"),
            "fontColor": pair("ink"), "selectionColor": pair("brand"),
            "visualizationColors": [colors["line"] for _name, colors in hues], "widgetHeaderAlignment": "LEFT"}


def _implement_dashboard_label(column: str) -> str:
    return str(column or "").replace("_", " ").strip().capitalize()


def _implement_dashboard_row(widgets: list, y: int, height: int) -> list:
    base, extra = divmod(IMPLEMENT_DASHBOARD_GRID, len(widgets))
    out, x = [], 0
    for index, widget in enumerate(widgets):
        width = base + (1 if index < extra else 0)
        out.append({"widget": widget, "position": {"x": x, "y": y, "width": width, "height": height}})
        x += width
    return out


def _implement_dashboard_text(name: str, lines) -> dict:
    return {"name": name, "multilineTextboxSpec": {"lines": implement_join_lines(lines)}}


def _implement_dashboard_filter(name: str, f: dict) -> dict:
    queries, fields = [], []
    for index, (dataset, column) in enumerate(f["columns"].items()):
        query = f"{name}_q{index}"
        queries.append({"name": query, "query": {"datasetName": dataset, "fields": [{"name": column, "expression": f"`{column}`"}],
                                                 "disaggregated": False}})
        fields.append({"fieldName": column, "displayName": f["label"], "queryName": query})
    kind = "filter-date-range-picker" if f["type"] == "date_range" else "filter-multi-select"
    return {"name": name, "queries": queries, "spec": {"version": 2, "widgetType": kind, "encodings": {"fields": fields},
                                                       "frame": {"showTitle": True, "title": f["label"]}}}


def _implement_dashboard_encoded(field: dict, label: str, item: dict = None, scale: str = None) -> dict:
    out = {"fieldName": field["name"], "displayName": label}
    if scale:
        out["scale"] = {"type": scale}
    number = implement_dashboard_format(item) if item is not None else None
    if number:
        out["format"] = number
    return out


def _implement_dashboard_table(name: str, dataset: str, columns: list, title: str, hint: str, order=None) -> dict:
    seen, fields, encoded = set(), [], []
    for column in columns:
        if column["column"] in seen:
            continue
        seen.add(column["column"])
        field = {"name": column["column"], "expression": f"`{column['column']}`"}
        fields.append(field)
        percent = column.get("format") == "percent"
        entry = {"fieldName": field["name"], "displayName": column["label"] + (" (%)" if percent else "")}
        number = implement_dashboard_format(column, measure=False)
        if number:
            entry["format"] = number
        encoded.append(entry)
    query = {"datasetName": dataset, "fields": fields, "disaggregated": True}
    if order:
        query["orders"] = [{"direction": order[1], "expression": f"`{order[0]}`"}]
    frame = {"showTitle": True, "title": title}
    if hint:
        frame.update(showDescription=True, description=hint)
    return {"name": name, "queries": [{"name": "main_query", "query": query}],
            "spec": {"version": 2, "widgetType": "table", "encodings": {"columns": encoded}, "frame": frame}}


def implement_dashboard_dataset_sql(page: dict, name: str, shell_sql) -> str:
    sql = shell_sql.clean_sql(next(d["sql"] for d in page["datasets"] if d["name"] == name))
    dates = [f["columns"][name] for f in page.get("filters") or [] if f.get("type") == "date_range" and name in (f.get("columns") or {})]
    reasons = (page.get("records") or {}).get("reasons") if name == shell_sql.RECORDS_DATASET else None
    changed = list(dict.fromkeys(dates + ([reasons] if reasons else [])))
    if not changed:
        return sql
    casts = [f"CAST({shell_sql.quote(c)} AS DATE) AS {shell_sql.quote(c)}" for c in dict.fromkeys(dates) if c != reasons]
    texts = ([f"concat_ws('{IMPLEMENT_DASHBOARD_REASONS_SEPARATOR}', {shell_sql.quote(reasons)}) AS {shell_sql.quote(reasons)}"]
             if reasons else [])
    return (f"SELECT * EXCEPT ({', '.join(shell_sql.quote(c) for c in changed)}), {', '.join(casts + texts)}\n"
            f"FROM ({sql}) AS d")


def implement_dashboard_run_health_sql(result_table: str) -> str:
    return (f"SELECT run_id, as_of_date, COUNT(*) AS row_count, MAX(model_version) AS model_version,\n"
            f"100.0 * COUNT_IF(data_quality_flag <> 'ok') / COUNT(*) AS flagged_share\nFROM {inspire_table_name(result_table)}\n"
            f"WHERE change_source = 'run'\nGROUP BY run_id, as_of_date\nORDER BY as_of_date DESC")


def _implement_dashboard_chart(name: str, chart: dict, rows, measure, dated=()) -> dict:
    kind = implement_dashboard_chart_type(chart, rows)
    dataset, x, ys = chart["dataset"], chart.get("x") or {}, chart["y"]
    measures = [measure(dataset, f"{name}_m{index}", y["expression"], y) for index, y in enumerate(ys)]
    if len(ys) == 1:
        y_encoding = _implement_dashboard_encoded(measures[0], ys[0]["label"], ys[0], "quantitative")
    else:
        y_encoding = {"scale": {"type": "quantitative"},
                      "fields": [{"fieldName": m["name"], "displayName": y["label"]} for m, y in zip(measures, ys)]}
    if kind == "scatter":
        group = chart["group"]
        x_measure = measure(dataset, f"{name}_x", x["expression"], x)
        fields = [{"name": group, "expression": f"`{group}`"}, x_measure] + measures
        encodings = {"x": _implement_dashboard_encoded(x_measure, x.get("label") or "", x, "quantitative"), "y": y_encoding,
                     "color": _implement_dashboard_encoded(fields[0], _implement_dashboard_label(group), scale="categorical")}
    else:
        column = {"name": x["column"], "expression": f"`{x['column']}`"}
        fields = [column] + measures
        temporal = kind in ("line", "area") and ((dataset, x["column"]) in dated
                                                 or any(isinstance(r.get("x"), datetime.date) for r in rows or []))
        x_encoding = _implement_dashboard_encoded(column, x.get("label") or _implement_dashboard_label(x["column"]),
                                                  scale="temporal" if temporal else "categorical")
        series = (chart.get("series") or {}).get("column")
        if series:
            fields.append({"name": series, "expression": f"`{series}`"})
        if kind == "pie":
            encodings = {"angle": dict(y_encoding), "color": x_encoding, "label": {"show": True}}
        elif kind == "heatmap":
            encodings = {"x": x_encoding, "y": _implement_dashboard_encoded(fields[-1], _implement_dashboard_label(series),
                                                                            scale="categorical"),
                         "color": dict(y_encoding)}
        else:
            ranked = kind == "bar" and len(ys) == 1 and chart.get("sort", "value_desc") == "value_desc" and not x.get("ordered")
            if ranked:
                x_encoding["scale"]["sort"] = {"by": "y-reversed"}
            encodings = {"x": x_encoding, "y": y_encoding}
            if series:
                encodings["color"] = _implement_dashboard_encoded(fields[-1], _implement_dashboard_label(series), scale="categorical")
    frame = {"showTitle": True, "title": chart["title"], "showDescription": True, "description": chart["hint"]}
    return {"name": name, "queries": [{"name": "main_query", "query": {"datasetName": dataset, "fields": fields, "disaggregated": False}}],
            "spec": {"version": 3, "widgetType": kind, "encodings": encodings, "frame": frame}}


def _implement_dashboard_records(page: dict, shell_sql) -> dict:
    records = page["records"]
    columns = [{"column": records["rank"], "label": IMPLEMENT_DASHBOARD_RECORD_LABELS["rank"], "format": "integer"},
               {"column": records["label"], "label": records.get("label_title") or IMPLEMENT_DASHBOARD_RECORD_LABELS["label"],
                "format": "text"}]
    listed = list(records.get("columns") or []) + list(records.get("detail") or [])
    if records.get("sublabel") and records["sublabel"] not in {c.get("column") for c in listed}:
        columns.append({"column": records["sublabel"], "label": _implement_dashboard_label(records["sublabel"]), "format": "text"})
    for key in ("tier", "reasons", "action"):
        if records.get(key):
            columns.append({"column": records[key], "label": IMPLEMENT_DASHBOARD_RECORD_LABELS[key], "format": "text"})
    return _implement_dashboard_table("records", shell_sql.RECORDS_DATASET, columns + listed, records["title"], records.get("hint"),
                                      (records["rank"], "ASC"))


def implement_dashboard_from_page(page: dict, shell_sql, result_table: str, shapes: dict = None, genie: dict = None,
                                  theme: dict = None) -> dict:
    heights = IMPLEMENT_DASHBOARD_HEIGHTS
    filters = list(page.get("filters") or [])
    dated = {(d, c) for f in filters if f["type"] == "date_range" for d, c in f["columns"].items()}
    used = list(dict.fromkeys([k["dataset"] for k in page.get("kpis") or []] + [c["dataset"] for c in page.get("charts") or []]
                              + [t["dataset"] for t in page.get("tables") or []] + [shell_sql.RECORDS_DATASET]
                              + [d for f in filters for d in f["columns"]]))
    measures = {name: [] for name in used}

    def measure(dataset, name, expression, item):
        text = f"({expression}) / 100" if (item or {}).get("format") == "percent" else str(expression)
        measures[dataset].append({"displayName": name, "expression": text})
        return {"name": f"measure({name})", "expression": f"MEASURE(`{name}`)"}

    layout = [{"widget": _implement_dashboard_text("header", [f"## {page['title']}", "", page["question"]]),
               "position": {"x": 0, "y": 0, "width": IMPLEMENT_DASHBOARD_GRID, "height": heights["text"]}}]
    y = heights["text"]
    if genie:
        title = re.sub(r"[\[\]()]", "", str(genie.get("title") or "")).strip() or "Genie Agent"
        layout.append({"widget": _implement_dashboard_text("genie_link", [
            f"Ask the Genie Agent [{title}]({genie['url']}) about these results in plain language."]),
                       "position": {"x": 0, "y": y, "width": IMPLEMENT_DASHBOARD_GRID, "height": heights["link"]}})
        y += heights["link"]
    widgets = [_implement_dashboard_filter(f"filter_{index}", f) for index, f in enumerate(filters)]
    for start in range(0, len(widgets), IMPLEMENT_DASHBOARD_FILTERS_PER_ROW):
        layout += _implement_dashboard_row(widgets[start:start + IMPLEMENT_DASHBOARD_FILTERS_PER_ROW], y, heights["filter"])
        y += heights["filter"]
    counters = []
    for index, kpi in enumerate(page.get("kpis") or []):
        name = f"kpi_{index}"
        field = measure(kpi["dataset"], f"{name}_m0", kpi["expression"], kpi)
        frame = {"showTitle": True, "title": kpi["label"]}
        if kpi.get("caption"):
            frame.update(showDescription=True, description=kpi["caption"])
        counters.append({"name": name, "queries": [{"name": "main_query", "query": {"datasetName": kpi["dataset"], "fields": [field],
                                                                                    "disaggregated": False}}],
                         "spec": {"version": 2, "widgetType": "counter",
                                  "encodings": {"value": _implement_dashboard_encoded(field, kpi["label"], kpi)}, "frame": frame}})
    if counters:
        layout += _implement_dashboard_row(counters, y, heights["counter"])
        y += heights["counter"]
    rows, pending = [], []
    for index, chart in enumerate(page.get("charts") or []):
        widget = _implement_dashboard_chart(f"chart_{index}", chart, (shapes or {}).get(chart["id"]), measure, dated)
        if chart.get("span", 1) == 2:
            rows += [pending] if pending else []
            rows.append([widget])
            pending = []
        else:
            pending.append(widget)
            if len(pending) == 2:
                rows.append(pending)
                pending = []
    rows += [pending] if pending else []
    for row in rows:
        layout += _implement_dashboard_row(row, y, heights["chart"])
        y += heights["chart"]
    for index, table in enumerate(page.get("tables") or []):
        sort = table.get("sort") or {}
        order = (sort["column"], "DESC" if sort.get("desc", True) else "ASC") if sort.get("column") else None
        layout += _implement_dashboard_row([_implement_dashboard_table(f"table_{index}", table["dataset"], table["columns"],
                                                                       table["title"], table.get("hint"), order)], y, heights["table"])
        y += heights["table"]
    layout += _implement_dashboard_row([_implement_dashboard_records(page, shell_sql)], y, heights["records"])
    run_dates = [f for f in filters if f["type"] == "date_range" and "as_of_date" in f["columns"].values()]
    health = IMPLEMENT_DASHBOARD_RUN_HEALTH
    while health in measures:
        health += "_"
    run_filter = {"type": "date_range", "label": run_dates[0]["label"] if run_dates else "Run date", "columns": {health: "as_of_date"}}
    health_layout = [{"widget": _implement_dashboard_text("run_health_header", [
        "## Run health", "", "One row per run of the use case: its run date, the rows it wrote, the model version and the share of "
                             "rows that data quality checks flagged."]),
                      "position": {"x": 0, "y": 0, "width": IMPLEMENT_DASHBOARD_GRID, "height": heights["text"]}}]
    health_layout += _implement_dashboard_row([_implement_dashboard_filter("run_health_filter", run_filter)], heights["text"],
                                              heights["filter"])
    health_layout += _implement_dashboard_row([_implement_dashboard_table(
        "run_health_table", health, [{"column": "run_id", "label": "Run", "format": "text"},
                                     {"column": "as_of_date", "label": "Run date", "format": "date"},
                                     {"column": "row_count", "label": "Rows written", "format": "integer"},
                                     {"column": "model_version", "label": "Model version", "format": "text"},
                                     {"column": "flagged_share", "label": "Flagged rows", "format": "percent"}],
        "Runs, newest first", "", ("as_of_date", "DESC"))], heights["text"] + heights["filter"], heights["table"])
    datasets = []
    for name in used:
        entry = {"name": name, "displayName": "Decision list" if name == shell_sql.RECORDS_DATASET else _implement_dashboard_label(name),
                 "queryLines": implement_join_lines(implement_dashboard_dataset_sql(page, name, shell_sql).splitlines())}
        if measures[name]:
            entry["columns"] = measures[name]
        datasets.append(entry)
    datasets.append({"name": health, "displayName": "Run health",
                     "queryLines": implement_join_lines(implement_dashboard_run_health_sql(result_table).splitlines())})
    doc = {"datasets": datasets,
           "pages": [{"name": "overview", "displayName": page["title"], "pageType": "PAGE_TYPE_CANVAS", "layoutVersion": "GRID_V1",
                      "layout": layout},
                     {"name": "run_health", "displayName": "Run health", "pageType": "PAGE_TYPE_CANVAS", "layoutVersion": "GRID_V1",
                      "layout": health_layout}]}
    ui = {}
    if theme:
        ui["theme"] = theme
    if genie:
        ui["genieSpace"] = dict(IMPLEMENT_DASHBOARD_GENIE, overrideId=str(genie["space_id"]))
    if ui:
        doc["uiSettings"] = ui
    return doc


def implement_dashboard_shown_columns(dashboard: dict) -> dict:
    shown = {}
    for page in dashboard.get("pages") or []:
        for item in page.get("layout") or []:
            widget = item.get("widget") or {}
            if (widget.get("spec") or {}).get("widgetType") != "table":
                continue
            for query in widget.get("queries") or []:
                body = query.get("query") or {}
                shown.setdefault(body.get("datasetName"), set()).update(f["name"] for f in body.get("fields") or [])
    return shown
def implement_app_owner(app) -> tuple:
    match = IMPLEMENT_APP_OWNER_RE.match(str((app or {}).get("description") or ""))
    return (match.group(1), match.group(2)) if match else None


def implement_app_host(name: str) -> dict:
    return {"name": name, "app": None, "owner": None, "merged": False, "source": "", "settings": None, "pages": [], "notes": []}


def implement_app_merge(host: dict, settings: dict, page: dict, session_id, look: dict) -> tuple:
    job_id = (settings.get("jobs") or {}).get(page["id"])
    if host["merged"] and host["owner"][1] != str(session_id):
        page = dict(page, id=implement_app_page_id(f"{page['use_case_id']}_s{session_id}"))
    pages = [page if p.get("id") == page["id"] else p for p in host["pages"]]
    if page["id"] not in [p.get("id") for p in pages]:
        pages.append(page)
    ids = [p["id"] for p in pages]
    jobs = implement_app_kept_jobs(host["settings"], ids, page["id"], job_id)
    if not host["merged"]:
        return dict(settings, **look, pages=ids, jobs=jobs), pages, page
    held = host["settings"]
    merged = dict(held, version=settings["version"], pages=ids, built_at=settings["built_at"], jobs=jobs,
                  admin=" ".join(str(held.get("admin") or "").split()) or settings.get("admin") or "")
    if not merged.get("genie"):
        merged.update(genie=settings["genie"], genie_note=settings["genie_note"])
    return merged, pages, page


def implement_app_page_reads(pages, skip: str = "") -> list:
    out = []
    for page in pages:
        if page.get("id") == skip:
            continue
        for dataset in page.get("datasets") or []:
            out += [t for t in implement_table_refs(str(dataset.get("sql") or "")) if t not in out]
    return out


def implement_app_page_name(page: dict) -> str:
    title = " ".join(str(page.get("title") or "").split())
    return f"'{title}' of use case {page.get('use_case_id')}" if title else f"{page.get('id')} of use case {page.get('use_case_id')}"


def implement_app_check_failure(app_name: str, problems, pages, page_id: str, label: str = "App") -> Exception:
    broken = [p for p in pages if p.get("id") != page_id and any(f"page {p.get('id')}:" in text for text in problems)]
    if not broken:
        return RuntimeError(f"the {label} was run at its default filters and " + "; ".join(problems[:20]))
    named = [text for text in problems if any(f"page {p.get('id')}:" in text for p in broken)]
    return ImplementBlocked(f"the App {app_name} was not updated because its page{'s' if len(broken) > 1 else ''} "
                            f"{' and '.join(implement_app_page_name(p) for p in broken)} no longer work"
                            f"{'' if len(broken) > 1 else 's'} at the default filters ({'; '.join(named[:10])}). Inspire rebuilds only "
                            f"this use case's page: rebuild {'those use cases' if len(broken) > 1 else 'that use case'} or remove "
                            "the page, then submit again")


def _implement_widget_datasets(page: dict):
    for item in page.get("layout") or []:
        for query in (item.get("widget") or {}).get("queries") or []:
            if isinstance(query.get("query"), dict):
                yield query["query"]


def implement_dashboard_problems(dashboard: dict) -> list:
    problems = []
    names = [str(d.get("name") or "") for d in dashboard.get("datasets") or []]
    pages = dashboard.get("pages") or []
    widgets = [str((item.get("widget") or {}).get("name") or "") for p in pages for item in p.get("layout") or []]
    for kind, values in (("dataset", names), ("page", [str(p.get("name") or "") for p in pages]), ("widget", widgets)):
        repeated = sorted({v for v in values if values.count(v) > 1})
        if repeated:
            problems.append(f"two {kind}s share the name {', '.join(repeated)}")
    missing = sorted({str(q.get("datasetName")) for p in pages for q in _implement_widget_datasets(p)
                      if q.get("datasetName") and q.get("datasetName") not in names})
    if missing:
        problems.append(f"widgets read datasets the dashboard does not define: {', '.join(missing)}")
    if sum(p.get("pageType") == "PAGE_TYPE_GLOBAL_FILTERS" for p in pages) > 1:
        problems.append("the dashboard has more than one global filters page")
    return problems


def implement_merge_dashboard(target: dict, ours: dict, prefix: str, label: str) -> tuple:
    merged = json.loads(json.dumps(target or {}))
    ours = json.loads(json.dumps(ours or {}))
    pages = [p for p in merged.get("pages") or [] if not str(p.get("name") or "").startswith(prefix)]
    for page in pages:
        page["layout"] = [i for i in page.get("layout") or [] if not str((i.get("widget") or {}).get("name") or "").startswith(prefix)]
    datasets = [d for d in merged.get("datasets") or [] if not str(d.get("name") or "").startswith(prefix)]
    renamed = {str(d.get("name")): f"{prefix}{d.get('name')}" for d in ours.get("datasets") or []}
    datasets += [dict(d, name=renamed[str(d.get("name"))]) for d in ours.get("datasets") or []]
    shared_filters = next((p for p in pages if p.get("pageType") == "PAGE_TYPE_GLOBAL_FILTERS"), None)
    for page in ours.get("pages") or []:
        for item in page.get("layout") or []:
            if isinstance(item.get("widget"), dict):
                item["widget"]["name"] = f"{prefix}{item['widget'].get('name')}"
        for query in _implement_widget_datasets(page):
            query["datasetName"] = renamed.get(str(query.get("datasetName")), query.get("datasetName"))
        if page.get("pageType") == "PAGE_TYPE_GLOBAL_FILTERS" and shared_filters is not None:
            bottom = max([int((i.get("position") or {}).get("y") or 0) + int((i.get("position") or {}).get("height") or 0)
                          for i in shared_filters.get("layout") or []] or [0])
            for item in page.get("layout") or []:
                if isinstance(item.get("position"), dict):
                    item["position"]["y"] = bottom + int(item["position"].get("y") or 0)
            shared_filters["layout"] = (shared_filters.get("layout") or []) + (page.get("layout") or [])
            continue
        title = " ".join(str(page.get("displayName") or page.get("name") or "").split())
        pages.append(dict(page, name=f"{prefix}{page.get('name')}",
                          displayName=(title if title.startswith(label) else f"{label} {title}".strip())[:100]))
    merged.update(datasets=datasets, pages=pages)
    return merged, implement_dashboard_problems(merged)


def _implement_genie_text(value) -> str:
    return "".join(str(part) for part in (value if isinstance(value, list) else [value or ""]))


def _implement_genie_key(kind: str, item: dict) -> str:
    if kind in ("sample_questions", "benchmarks", "example_question_sqls"):
        return " ".join(_implement_genie_text(item.get("question")).split()).casefold()
    if kind == "join_specs":
        return "|".join(str((item.get(side) or {}).get("identifier") or "").lower() for side in ("left", "right"))
    return json.dumps({k: v for k, v in item.items() if k != "id"}, sort_keys=True, default=str)


def _implement_genie_kinds(space: dict) -> list:
    kinds = ["sample_questions"] + (["benchmarks"] if isinstance(space.get("benchmarks"), dict) else [])
    instructions = space.get("instructions") if isinstance(space.get("instructions"), dict) else {}
    kinds += [k for k in sorted(instructions) if k not in ("text_instructions", "sql_snippets") and isinstance(instructions[k], list)]
    snippets = instructions.get("sql_snippets")
    return kinds + ([f"sql_snippets.{k}" for k in sorted(snippets) if isinstance(snippets[k], list)] if isinstance(snippets, dict) else [])


def _implement_genie_id_space(kind: str) -> str:
    return "questions" if kind in ("sample_questions", "benchmarks") else "instructions"


def implement_merge_genie_space(target: dict, ours: dict, previous: dict, label: str) -> tuple:
    merged = json.loads(json.dumps(target or {}))
    ours = json.loads(json.dumps(ours or {}))
    previous = previous or {}
    merged.setdefault("version", 2)
    sources, mine = merged.setdefault("data_sources", {}), ours.get("data_sources") or {}
    added = []
    for key in ("tables", "metric_views"):
        have = {str(t.get("identifier") or "").lower() for t in sources.get(key) or []}
        extra = [t for t in mine.get(key) or [] if isinstance(t, dict) and str(t.get("identifier") or "").lower() not in have]
        added += [str(t.get("identifier")) for t in extra]
        if extra or key in sources:
            sources[key] = sorted((sources.get(key) or []) + extra, key=lambda t: str(t.get("identifier") or ""))
    total = sum(len(sources.get(key) or []) for key in ("tables", "metric_views"))
    if total > IMPLEMENT_GENIE_MAX_TABLES:
        listed = ", ".join(added[:5]) + (f" and {len(added) - 5} more" if len(added) > 5 else "")
        return merged, {}, [f"it would hold {total} tables, above the {IMPLEMENT_GENIE_MAX_TABLES} a Genie Agent can hold "
                            f"({total - len(added)} already, and this use case adds {len(added)}: {listed})"]
    kinds = list(dict.fromkeys(_implement_genie_kinds(merged) + _implement_genie_kinds(ours)))
    kept = {kind: [i for i in _implement_genie_items(merged, kind) if _implement_genie_key(kind, i) not in set(previous.get(kind) or [])]
            for kind in kinds}
    used = {"questions": set(), "instructions": set()}
    for kind, items in kept.items():
        used[_implement_genie_id_space(kind)] |= {str(i.get("id")) for i in items}
    used["instructions"] |= {str(i.get("id")) for i in (merged.get("instructions") or {}).get("text_instructions") or []
                             if isinstance(i, dict)}
    record = {"tables": added}
    for kind in kinds:
        items, ids = list(kept[kind]), used[_implement_genie_id_space(kind)]
        have = {_implement_genie_key(kind, i) for i in items}
        record[kind] = []
        for item in _implement_genie_items(ours, kind):
            fingerprint = _implement_genie_key(kind, item)
            if fingerprint in have:
                continue
            ident, salt = str(item.get("id") or ""), 0
            while not re.fullmatch(r"[0-9a-f]{32}", ident) or ident in ids:
                ident, salt = uuid.uuid5(uuid.NAMESPACE_URL, f"{kind}:{fingerprint}:{salt}").hex, salt + 1
            ids.add(ident)
            have.add(fingerprint)
            record[kind].append(fingerprint)
            items.append(dict(item, id=ident))
        if items or _implement_genie_items(merged, kind):
            _implement_genie_put(merged, kind, sorted(items, key=lambda i: str(i.get("id"))))
    instructions = merged.setdefault("instructions", {})
    text = _implement_genie_text(((ours.get("instructions") or {}).get("text_instructions") or [{}])[0].get("content")).strip()
    block = f"{label}\n{text}" if text else ""
    current = (instructions.get("text_instructions") or [None])[0]
    content = _implement_genie_text((current or {}).get("content")).strip()
    old = str(previous.get("text") or "")
    if old and old in content:
        content = content.replace(old, block).strip()
    elif block and block not in content:
        content = f"{content}\n\n{block}".strip()
    if content:
        entry = dict(current or {}, content=content.splitlines(keepends=True))
        if not re.fullmatch(r"[0-9a-f]{32}", str(entry.get("id") or "")):
            entry["id"] = uuid.uuid4().hex
        instructions["text_instructions"] = [entry]
    record["text"] = block
    return merged, record, []


def _implement_genie_path(kind: str) -> tuple:
    if kind == "sample_questions":
        return ("config", "sample_questions")
    if kind == "benchmarks":
        return ("benchmarks", "questions")
    if kind.startswith("sql_snippets."):
        return ("instructions", "sql_snippets", kind.split(".", 1)[1])
    return ("instructions", kind)


def _implement_genie_items(space: dict, kind: str) -> list:
    node = space
    for part in _implement_genie_path(kind):
        node = node.get(part) if isinstance(node, dict) else None
    return [i for i in node if isinstance(i, dict)] if isinstance(node, list) else []


def _implement_genie_put(space: dict, kind: str, items: list) -> None:
    *parents, key = _implement_genie_path(kind)
    node = space
    for part in parents:
        node = node.setdefault(part, {})
    node[key] = items


_IMPLEMENT_MUTATE_SQL_RE = re.compile(r"\b(UPDATE\s+[`\w.{}]+\s+SET|DELETE\s+FROM|MERGE\s+INTO|INSERT\s+OVERWRITE)\b", re.I)


class ImplementBlocked(RuntimeError):
    pass


def implement_is_permission_denied(error) -> bool:
    return type(error).__name__ == "PermissionDenied" or str(getattr(error, "error_code", "") or "") == "PERMISSION_DENIED"


def implement_is_not_found(error) -> bool:
    if type(error).__name__ in ("NotFound", "ResourceDoesNotExist"):
        return True
    if str(getattr(error, "error_code", "") or "") in ("RESOURCE_DOES_NOT_EXIST", "NOT_FOUND"):
        return True
    text = str(error).lower()
    return any(needle in text for needle in ("does not exist", "not found", "resource_does_not_exist", "not_found"))


def implement_retarget_instruction(instruction: str, written_for: str, result_table: str) -> str:
    text = str(instruction or "")
    old, new = inspire_table_parts(written_for), inspire_table_parts(result_table)
    if [part.lower() for part in old] == [part.lower() for part in new]:
        return text
    suffix = "(?P<suffix>_model|_actions)?"
    quoted = re.compile(re.escape("`{}`.`{}`.`{}".format(*old)) + suffix + "`", re.IGNORECASE)
    plain = re.compile(r"(?<![\w.])" + re.escape(".".join(old)) + suffix + r"(?!\w)", re.IGNORECASE)
    text = quoted.sub(lambda m: "`{}`.`{}`.`{}".format(*new) + (m.group("suffix") or "") + "`", text)
    return plain.sub(lambda m: ".".join(new) + (m.group("suffix") or ""), text)


IMPLEMENT_REASONS_HISTORY_COLUMNS = tuple(name for name in INSPIRE_HISTORY_NAMES if name != "changed_by")
IMPLEMENT_REASONS_VIEW_COLUMNS = ("as_of_date, run_id, use_case_id, entity_id, entity_label, rank, priority_tier, "
                                  + ", ".join(IMPLEMENT_REASONS_HISTORY_COLUMNS) + ", reason_position (1 for a row's first reason) and reason")
IMPLEMENT_NO_RAW_JSON_RULE = ("people never see a list, an object or JSON text. Show top_reasons as readable text, joined with "
                              "array_join(top_reasons, ' · ') or one reason per line, give each field of an object its own column, and "
                              "never return an ARRAY, MAP or STRUCT column or a JSON string from a query that people read; Inspire runs "
                              "every query and rejects one that does.")


def implement_reasons_view(result_table: str) -> str:
    return ".".join(inspire_table_parts(result_table)) + "_reasons"


IMPLEMENT_REASONS_VIEW_SELECT = ("as_of_date, run_id, use_case_id, entity_id, entity_label, rank, priority_tier, "
                                 + ", ".join(IMPLEMENT_REASONS_HISTORY_COLUMNS) + ", pos + 1 AS reason_position, reason")


def implement_reasons_view_sql(result_table: str) -> str:
    comment = inspire_sql_literal(f"One row per reason in top_reasons of {result_table}, so filters and charts read each reason on its own")
    return (f"CREATE VIEW IF NOT EXISTS {inspire_table_name(result_table, '_reasons')} COMMENT {comment} AS SELECT "
            f"{IMPLEMENT_REASONS_VIEW_SELECT} FROM {inspire_table_name(result_table)} LATERAL VIEW POSEXPLODE(top_reasons) r AS pos, reason")


def implement_reasons_view_alter_sql(result_table: str) -> str:
    return (f"ALTER VIEW {inspire_table_name(result_table, '_reasons')} AS SELECT {IMPLEMENT_REASONS_VIEW_SELECT} "
            f"FROM {inspire_table_name(result_table)} LATERAL VIEW POSEXPLODE(top_reasons) r AS pos, reason")


def _implement_raw_json_kind(value):
    if hasattr(value, "asDict") or isinstance(value, dict):
        return "an object"
    if isinstance(value, (list, tuple)):
        return "a list"
    if isinstance(value, str):
        text = value.strip()
        if text[:1] in ("[", "{") and text[-1:] in ("]", "}"):
            try:
                parsed = json.loads(text)
            except ValueError:
                return None
            if isinstance(parsed, (list, dict)):
                return "JSON text"
    return None


def implement_raw_json_problems(label: str, rows) -> list:
    found = {}
    for row in rows or []:
        for column, value in dict(row).items():
            if column not in found:
                kind = _implement_raw_json_kind(value)
                if kind:
                    found[column] = kind
    return [f"{label} returns {column} as {kind}, which people would read as raw JSON; return readable text instead "
            f"(array_join({column}, ' · ') for a list, one column per field for an object), or read the one-row-per-reason view"
            for column, kind in found.items()]


def check_alert_condition_sql(sql, allowed_tables) -> list:
    plain = re.sub(r"--[^\n]*|/\*.*?\*/", " ", str(sql or ""), flags=re.S).strip().rstrip(";").strip()
    if not plain:
        return ["alert_condition.sql is empty"]
    errors = []
    if ";" in plain:
        errors.append("alert_condition.sql must hold one SELECT statement")
    if not re.match(r"(?is)^\(?\s*(SELECT|WITH)\b", plain):
        errors.append("alert_condition.sql must be a SELECT (or WITH ... SELECT) query")
    if _IMPLEMENT_WRITE_SQL_RE.search(plain) or _IMPLEMENT_MUTATE_SQL_RE.search(plain):
        errors.append("alert_condition.sql writes data; it may only read")
    if not re.search(r"\btriggered\b", plain, re.I):
        errors.append("alert_condition.sql must return one row with a BOOLEAN column named triggered")
    errors += _implement_flag_count_problems([plain], "alert_condition.sql")
    unknown = _implement_unknown_tables([plain], allowed_tables)
    if unknown:
        errors.append(f"alert_condition.sql reads tables outside this use case: {', '.join(unknown)}")
    return errors


def implement_alert_threshold(outputs: dict) -> float:
    raw = outputs.get("alert_threshold")
    number = 1.0 if raw in (None, "") else implement_decimal(raw)
    if number is None:
        raise ValueError(f"alert_threshold {raw!r} is not a number")
    return number


def _implement_alert_rule_text(outputs: dict) -> str:
    return f"{outputs.get('alert_operator') or '>='} {_genie_number_text(implement_alert_threshold(outputs))}"


def implement_alert_trigger_text(outputs: dict) -> str:
    tiers = " or ".join(IMPLEMENT_ALERT_TIER_SETS[outputs.get("alert_tiers") or "act_now"])
    text = f"{tiers} rows in the latest run {_implement_alert_rule_text(outputs)}"
    if outputs.get("alert_threshold_basis") == "baseline":
        rise = IMPLEMENT_ALERT_BASELINE_DIRECTIONS[outputs.get("alert_operator") or ">="] == "+"
        text = (f"{tiers} rows in the latest run {'rise above' if rise else 'fall below'} the mean {'+' if rise else '-'} "
                f"{IMPLEMENT_ALERT_BASELINE_SIGMAS} standard deviations of up to {IMPLEMENT_ALERT_BASELINE_RUNS} earlier runs "
                f"({_implement_alert_rule_text(outputs)} until {IMPLEMENT_ALERT_BASELINE_MIN_RUNS} earlier runs exist)")
    if outputs.get("alert_condition_text"):
        text += f", or {outputs['alert_condition_text']}"
    return text


def implement_alert_query(result_table: str, outputs: dict, condition_sql: str = "") -> dict:
    tiers = ", ".join(inspire_sql_literal(t) for t in IMPLEMENT_ALERT_TIER_SETS[outputs.get("alert_tiers") or "act_now"])
    operator = outputs.get("alert_operator") or ">="
    threshold = implement_alert_threshold(outputs)
    base = (f"WITH latest AS (SELECT * FROM {inspire_table_name(result_table)} WHERE {implement_latest_run_filter(result_table)}),\n"
            f"tier_rows AS (SELECT COUNT_IF(priority_tier IN ({tiers})) AS tier_rows, COUNT(*) AS run_rows, "
            f"MAX(as_of_date) AS as_of_date, MAX(run_id) AS run_id FROM latest)")
    condition = re.sub(r"--[^\n]*", " ", str(condition_sql or "")).strip().rstrip(";").strip()
    baseline = outputs.get("alert_threshold_basis") == "baseline"
    if not condition and not baseline:
        return {"query": base + "\nSELECT tier_rows, run_rows, as_of_date, run_id FROM tier_rows", "column": "tier_rows",
                "operator": IMPLEMENT_ALERT_OPERATORS[operator], "threshold": threshold}
    rule, columns, joins = f"t.tier_rows {operator} {threshold!r}", "t.tier_rows, t.run_rows", ""
    if baseline:
        sign = IMPLEMENT_ALERT_BASELINE_DIRECTIONS[operator]
        band = f"b.baseline_mean {sign} {IMPLEMENT_ALERT_BASELINE_SIGMAS} * b.baseline_sd"
        enough = f"b.history_runs >= {IMPLEMENT_ALERT_BASELINE_MIN_RUNS}"
        base += (f",\nhistory AS (SELECT as_of_date, tier_rows FROM (SELECT as_of_date, COUNT_IF(priority_tier IN ({tiers})) AS "
                 f"tier_rows, ROW_NUMBER() OVER (PARTITION BY as_of_date ORDER BY MAX(valid_from) DESC) AS pick FROM "
                 f"{inspire_table_name(result_table)} WHERE change_source = 'run' AND as_of_date IS NOT NULL AND as_of_date NOT IN "
                 f"(SELECT as_of_date FROM latest WHERE change_source = 'run' AND as_of_date IS NOT NULL) GROUP BY as_of_date, run_id) "
                 f"AS r WHERE pick = 1 ORDER BY as_of_date DESC LIMIT {IMPLEMENT_ALERT_BASELINE_RUNS}),\n"
                 f"baseline AS (SELECT COUNT(*) AS history_runs, AVG(tier_rows) AS baseline_mean, "
                 f"COALESCE(STDDEV_SAMP(tier_rows), 0) AS baseline_sd FROM history)")
        rule = f"CASE WHEN {enough} THEN t.tier_rows {'>' if sign == '+' else '<'} {band} ELSE {rule} END"
        columns += (f", b.history_runs, ROUND(b.baseline_mean, 2) AS baseline_mean, ROUND(b.baseline_sd, 2) AS baseline_sd, "
                    f"CASE WHEN {enough} THEN ROUND({band}, 2) ELSE {threshold!r} END AS alert_threshold, "
                    f"CASE WHEN {enough} THEN 'baseline' ELSE 'fixed' END AS threshold_basis")
        joins += " CROSS JOIN baseline b"
    if condition:
        base += f",\nuser_condition AS (\n{condition}\n)"
        rule += " OR COALESCE(c.triggered, FALSE)"
        columns += ", COALESCE(c.triggered, FALSE) AS condition_met"
        joins += " LEFT JOIN (SELECT triggered FROM user_condition LIMIT 1) c ON TRUE"
    return {"query": base + f"\nSELECT CASE WHEN {rule} THEN 1 ELSE 0 END AS alert_value, {columns}, t.as_of_date, t.run_id "
                            f"FROM tier_rows t{joins}",
            "column": "alert_value", "operator": "GREATER_THAN_OR_EQUAL", "threshold": 1.0}


def implement_alert_baseline_now(outputs: dict, rows) -> dict:
    if outputs.get("alert_threshold_basis") != "baseline" or not rows:
        return {}
    row = {str(k).lower(): v for k, v in dict(rows[0]).items()}
    return {"threshold_basis": str(row.get("threshold_basis") or "fixed"), "history_runs": int(row.get("history_runs") or 0),
            "baseline_mean": row.get("baseline_mean"), "baseline_sd": row.get("baseline_sd"), "alert_threshold": row.get("alert_threshold")}


def implement_job_cadence(schedule: dict, current=None) -> dict:
    schedule = schedule or {}
    cron = str(schedule.get("cron") or getattr(current, "quartz_cron_expression", None) or "").strip()
    return {"cron": cron, "timezone": str(schedule.get("timezone") or "UTC"), "enabled": bool(cron and schedule.get("enabled"))}


def implement_alert_schedule(cadence: dict) -> dict:
    return {"quartz_cron_schedule": cadence.get("cron") or IMPLEMENT_ALERT_IDLE_CRON, "timezone_id": cadence.get("timezone") or "UTC",
            "pause_status": "PAUSED"}


def implement_alert_cadence_text(cadence: dict) -> str:
    cron, zone = cadence.get("cron"), cadence.get("timezone") or "UTC"
    if cron and cadence.get("enabled"):
        return f"after each scheduled run of the job ({cron}, {zone}), once its notebook task has written the results"
    if cron:
        return f"after each run of the job; its schedule ({cron}, {zone}) is paused, so only when someone runs the job"
    return "after each run of the job; the job has no schedule, so only when someone runs it"


def implement_alert_threshold_reason(outputs: dict) -> str:
    tiers = " and ".join(IMPLEMENT_ALERT_TIER_SETS[outputs.get("alert_tiers") or "act_now"])
    fixed = f"{tiers} rows {_implement_alert_rule_text(outputs)}"
    if outputs.get("alert_threshold_basis") != "baseline":
        return f"{fixed}, the number set in the wizard."
    sign = "plus" if IMPLEMENT_ALERT_BASELINE_DIRECTIONS[outputs.get("alert_operator") or ">="] == "+" else "minus"
    return (f"the mean {sign} {IMPLEMENT_ALERT_BASELINE_SIGMAS} standard deviations of the {tiers} rows of up to "
            f"{IMPLEMENT_ALERT_BASELINE_RUNS} earlier runs, worked out again at every check. The mean and standard deviation use "
            f"every earlier run, so the line moves smoothly as runs are added; a percentile of so few runs is one of the top runs "
            f"and jumps when a run enters or leaves. With fewer than {IMPLEMENT_ALERT_BASELINE_MIN_RUNS} earlier runs the alert "
            f"uses the number set in the wizard ({fixed}), because a mean of so few runs swings with a single run.")


def implement_alert_texts(uc_id: str, use_case: dict, contract: dict, outputs: dict, dashboard_url: str = "", genie_url: str = "",
                          cadence: dict = None) -> tuple:
    trigger = implement_alert_trigger_text(outputs)
    summary = f"{uc_id} {use_case.get('Name') or ''}".strip()[:150] + f": {trigger}"[:200]
    tiers = " and ".join(IMPLEMENT_ALERT_TIER_SETS[outputs.get("alert_tiers") or "act_now"])
    flagged = ("open the Run health page of the dashboard first" if dashboard_url else
               "look at the rows whose data_quality_flag is not 'ok' first")
    steps = [f"Open the decision list{f' ({dashboard_url})' if dashboard_url else ''} and work the {tiers} rows by rank, top first.",
             "Read each row's top reasons and recommended action before you act on it.",
             f"If far more rows fired than usual, {flagged}: flagged rows point to a data problem, not a business one."]
    if genie_url:
        steps.append(f"Ask the Genie Agent ({genie_url}) why a row is on the list or how it compares with earlier runs.")
    lines = ([f"What it means: it fires when {trigger}.",
              f"Why it matters: {str(use_case.get('Statement') or use_case.get('Solution') or '').strip()[:400]}",
              f"Who acts: {contract.get('decision_owner') or 'the decision owner'}",
              f"Decision: {contract.get('decision') or use_case.get('Name') or ''}", "What to do:"]
             + [f"{number}. {step}" for number, step in enumerate(steps, 1)]
             + ["When it clears: you get one more message when the alert returns to OK.",
                f"Threshold: {implement_alert_threshold_reason(outputs)}",
                f"Checked: {implement_alert_cadence_text(cadence or {})}", f"KPI: {contract.get('kpi') or ''}"])
    return summary[:250], "\n".join(line for line in lines if not line.endswith(": "))[:4000]


def implement_alert_subscribers(outputs: dict) -> list:
    return ([{"user_name": email} for email in outputs.get("alert_recipients") or []]
            + [{"destination_id": d["id"]} for d in outputs.get("alert_destinations") or []])[:IMPLEMENT_ALERT_MAX_SUBSCRIBERS]


IMPLEMENT_ALL_USERS = "account users"
IMPLEMENT_READ_NEEDS = (("USE CATALOG", "catalog", ("catalog",)), ("USE SCHEMA", "schema", ("catalog", "schema")),
                        ("SELECT", "table", ("catalog", "schema", "table")))
IMPLEMENT_TRAVERSE_PRIVILEGES = tuple(privilege for privilege, level, _ in IMPLEMENT_READ_NEEDS if level != "table")


def _implement_table_parts(table: str):
    parts = [p.strip().strip("`") for p in str(table or "").split(".")]
    return parts if len(parts) == 3 and all(parts) else None


def implement_access_args(table: str):
    parts = _implement_table_parts(table)
    return {"c": parts[0], "s": parts[1], "t": parts[2]} if parts else None


def implement_access_sql() -> str:
    c = "catalog_name = lower(:c)"
    s = "schema_name = lower(:s)"
    t = "table_catalog = lower(:c) AND table_schema = lower(:s) AND table_name = lower(:t)"
    return ("SELECT level, privilege, principal, "
            "(lower(principal) = lower(current_user()) OR is_account_group_member(principal)) AS mine, current_user() AS me FROM ("
            f"SELECT 'catalog' AS level, 'OWNER' AS privilege, catalog_owner AS principal FROM system.information_schema.catalogs WHERE {c} "
            f"UNION ALL SELECT 'schema', 'OWNER', schema_owner FROM system.information_schema.schemata WHERE {c} AND {s} "
            f"UNION ALL SELECT 'table', 'OWNER', table_owner FROM system.information_schema.tables WHERE {t} "
            f"UNION ALL SELECT 'catalog', privilege_type, grantee FROM system.information_schema.catalog_privileges WHERE {c} "
            f"UNION ALL SELECT 'schema', privilege_type, grantee FROM system.information_schema.schema_privileges WHERE {c} AND {s} "
            f"UNION ALL SELECT 'table', privilege_type, grantee FROM system.information_schema.table_privileges WHERE {t}"
            ") WHERE principal IS NOT NULL")


def implement_grant_sql(need: dict, principal: str) -> str:
    return f"GRANT {need['privilege']} ON {need['level'].upper()} {need['object']} TO `{principal}`"


def _implement_privilege(value) -> str:
    return " ".join(str(value or "").strip().upper().replace("_", " ").split())


def implement_access_plan(table: str, rows, principal: str = "") -> dict:
    parts = _implement_table_parts(table)
    who = str(principal or "").strip()
    if not parts or "`" in who:
        return {"grants": [], "gaps": [], "me": ""}
    objects = {"catalog": quote_identifier(parts[0]), "schema": f"{quote_identifier(parts[0])}.{quote_identifier(parts[1])}",
               "table": ".".join(quote_identifier(p) for p in parts)}
    need = lambda privilege, level: {"privilege": privilege, "level": level, "object": objects[level]}
    if rows is None:
        return {"grants": [need(p, level) for p, level, _ in IMPLEMENT_READ_NEEDS] if who else [], "gaps": [], "me": ""}
    facts = [{"level": str(r.get("level") or "").lower(), "privilege": _implement_privilege(r.get("privilege")),
              "principal": str(r.get("principal") or "").strip(), "mine": str(r.get("mine")).lower() == "true"} for r in rows]
    me = next((str(r.get("me")).strip() for r in rows if r.get("me")), "")
    holders = {IMPLEMENT_ALL_USERS} | ({who.lower()} if who else set())
    table_exists = any(f["level"] == "table" and f["privilege"] == "OWNER" for f in facts)
    plan = {"grants": [], "gaps": [], "me": me}
    for privilege, level, levels in IMPLEMENT_READ_NEEDS:
        owns = bool(who) and any(f["level"] == level and f["privilege"] == "OWNER" and f["principal"].lower() == who.lower()
                                 for f in facts)
        granted = any(f["level"] in levels and f["privilege"] in (privilege, "ALL PRIVILEGES") and f["principal"].lower() in holders
                      for f in facts)
        if owns or granted:
            continue
        manages = any(f["mine"] and f["level"] in levels and f["privilege"] in ("OWNER", "MANAGE") for f in facts)
        if manages or (level == "table" and not table_exists):
            if who:
                plan["grants"].append(need(privilege, level))
            continue
        owner = next((f["principal"] for f in facts if f["level"] == level and f["privilege"] == "OWNER"), "")
        plan["gaps"].append(dict(need(privilege, level), owner=owner))
    return plan


def _implement_gap_list(gaps) -> str:
    items = [f"{g['privilege']} on {g['level']} {g['object']}" + (f" (owner {g['owner']})" if g["owner"] else "") for g in gaps]
    return items[0] if len(items) == 1 else f"{', '.join(items[:-1])} and {items[-1]}"


def implement_access_gap_message(who: str, gaps, table: str = "the result table") -> str:
    statements = "; ".join(implement_grant_sql(g, who) for g in gaps)
    return (f"{who} may not be able to read {table}: Inspire cannot grant {_implement_gap_list(gaps)}, because it "
            f"neither owns nor holds MANAGE on {'it' if len(gaps) == 1 else 'them'}. If {who} does not have "
            f"{'it' if len(gaps) == 1 else 'them'} yet, ask the owner to run: {statements}")


def implement_app_access_message(table: str, gaps, me: str = "") -> str:
    inspire = f"`{me}`" if me else "the identity Inspire runs as"
    statements = "; ".join(f"GRANT MANAGE ON {g['level'].upper()} {g['object']} TO {inspire}" for g in gaps)
    one = len(gaps) == 1
    shared = ""
    if gaps and all(g["privilege"] in IMPLEMENT_TRAVERSE_PRIVILEGES for g in gaps):
        shared = (f", or {'; '.join(implement_grant_sql(g, IMPLEMENT_ALL_USERS) for g in gaps)}, which "
                  f"{'opens' if one else 'open'} no data by {'itself' if one else 'themselves'}")
    return (f"a new App could not read {table}: its service principal needs {_implement_gap_list(gaps)}, which not all "
            f"account users have, and Inspire can grant {'it' if one else 'them'} only with MANAGE or ownership. "
            f"Ask the owner to run: {statements}{shared}; or choose a result table in a catalog and schema Inspire manages")


def implement_permission_acl(principals, level: str) -> list:
    acl = []
    for principal in principals or []:
        key = {"user": "user_name", "group": "group_name", "service_principal": "service_principal_name"}.get(principal.get("type"))
        if key and principal.get("name"):
            acl.append({key: principal["name"], "permission_level": level})
    return acl


IMPLEMENT_PERMISSION_RANK = ("CAN_VIEW", "CAN_READ", "CAN_USE", "CAN_RUN", "CAN_MONITOR", "CAN_MANAGE_RUN", "CAN_EDIT",
                             "CAN_MANAGE", "IS_OWNER")


def implement_principals_below(current, principals, level: str) -> list:
    rank = {name: index for index, name in enumerate(IMPLEMENT_PERMISSION_RANK)}
    held = {}
    entries = current.get("access_control_list") if isinstance(current, dict) else None
    for entry in entries if isinstance(entries, list) else []:
        entry = entry if isinstance(entry, dict) else {}
        who = str(entry.get("user_name") or entry.get("group_name") or entry.get("service_principal_name") or "").strip().lower()
        for permission in entry.get("all_permissions") or []:
            got = rank.get(str((permission or {}).get("permission_level") or "").strip().upper(), -1)
            held[who] = max(held.get(who, -1), got)
    need = rank[level]
    if held.get("users", -1) >= need:
        return []
    return [p for p in principals or [] if held.get(str((p or {}).get("name") or "").strip().lower(), -1) < need]


def genie_message_text(text) -> str:
    raw = str(text or "").strip()
    try:
        parsed = json.loads(raw) if raw.startswith("{") else None
    except ValueError:
        parsed = None
    if isinstance(parsed, dict) and ("summary" in parsed or "stop_reason" in parsed):
        reason = " ".join(str(parsed.get("stop_reason") or "").split())
        raw = f"Stopped: {reason}" if genie_fill_stopped(parsed) and reason else str(parsed.get("summary") or "")
    return " ".join(raw.split())[:1000]


def summarize_genie_event(event) -> str:
    if not isinstance(event, dict):
        return ""
    kind = str(event.get("type") or "")
    item = event.get("item") if isinstance(event.get("item"), dict) else {}
    item_type = str(item.get("type") or "")
    if kind == "item.completed" and item_type == "command_execution":
        command = " ".join(str(item.get("command") or "").split())[:300]
        return f"Genie ran: {command} (exit {item.get('exit_code')})"
    if kind == "item.completed" and item_type == "agent_message":
        return "Genie: " + genie_message_text(item.get("text"))
    if kind == "item.completed" and item_type == "file_change":
        changes = ", ".join(str(c.get("path")) for c in item.get("changes") or [] if isinstance(c, dict))[:300]
        return f"Genie edited: {changes}"
    if kind in ("error", "turn.failed"):
        nested = event.get("error") if isinstance(event.get("error"), dict) else {}
        message = event.get("message") or nested.get("message") or event
        return f"Genie error: {str(message)[:400]}"
    if kind == "turn.completed":
        usage = event.get("usage") or {}
        return f"Genie turn completed (input tokens {usage.get('input_tokens')}, output tokens {usage.get('output_tokens')})"
    return ""


def implement_secret_files(root: str, secret: str, skip=()) -> list:
    if not secret or not root or not os.path.isdir(root):
        return []
    needle, found = secret.encode("utf-8"), []
    for folder, dirs, names in os.walk(root):
        dirs[:] = [d for d in dirs if os.path.relpath(os.path.join(folder, d), root) not in skip]
        for name in names:
            path = os.path.join(folder, name)
            try:
                if os.path.islink(path) or os.path.getsize(path) > 1_000_000:
                    continue
                with open(path, "rb") as f:
                    if needle in f.read():
                        found.append(os.path.relpath(path, root))
            except OSError:
                continue
    return sorted(found)


def implement_carry_files(previous_dir: str, work_dir: str) -> list:
    if not previous_dir or not os.path.isdir(previous_dir):
        return []
    import shutil
    carried = []
    for folder, dirs, names in os.walk(previous_dir):
        dirs[:] = [d for d in dirs if not d.startswith(".") and d != "__pycache__"]
        for name in names:
            relative = os.path.relpath(os.path.join(folder, name), previous_dir)
            if name.startswith(".") or relative in GENIE_ARTIFACT_SCRATCH_FILES:
                continue
            target = os.path.join(work_dir, relative)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            shutil.copy2(os.path.join(folder, name), target)
            carried.append(relative)
    return sorted(carried)


def genie_download_once(url: str, path: str) -> int:
    import urllib.request
    with urllib.request.urlopen(url, timeout=GENIE_DOWNLOAD_TIMEOUT_S) as response, open(path, "wb") as f:
        expected = int(response.headers.get("Content-Length") or -1)
        shutil.copyfileobj(response, f, 1 << 20)
    size = os.path.getsize(path)
    if expected >= 0 and size != expected:
        raise RuntimeError(f"the download of {url} stopped at {size} of {expected} bytes")
    return size


class GenieCodeRunner:

    def __init__(self, host: str, token: str, logger, on_line=None, root: str = None, model: str = GENIE_CODE_MODEL,
                 timeout_s: int = GENIE_CODE_TIMEOUT_S, fetch=None, popen=None):
        self.host = str(host or "").rstrip("/")
        self.token = str(token or "")
        self.logger = logger
        self.on_line = on_line or (lambda line: None)
        self.model = model
        self.timeout_s = int(timeout_s)
        self.root = root or self._writable_root()
        self._fetch = fetch
        self._popen = popen
        self.genie_path = ""
        self.cli_dir = ""

    @staticmethod
    def _writable_root() -> str:
        candidates = [os.path.join(os.path.expanduser("~"), ".inspire_genie"), "/local_disk0/inspire_genie",
                      os.path.join(tempfile.gettempdir(), "inspire_genie")]
        for path in candidates:
            try:
                os.makedirs(path, exist_ok=True)
                probe = os.path.join(path, ".probe")
                with open(probe, "w", encoding="utf-8") as f:
                    f.write("ok")
                os.remove(probe)
                return path
            except OSError:
                continue
        raise RuntimeError(f"no writable folder for Genie Code among {candidates}")

    def config_toml(self) -> str:
        auth = "printf '{\\\"access_token\\\":\\\"%s\\\",\\\"token_type\\\":\\\"Bearer\\\"}' \\\"$DATABRICKS_TOKEN\\\""
        return (f'model_provider = "databricks"\nmodel = "{self.model}"\n\n[model_providers.databricks]\nname = "Unity Gateway"\n'
                f'base_url = "{self.host}/ai-gateway/codex/v1"\nwire_api = "responses"\n\n[model_providers.databricks.auth]\n'
                f'command = "sh"\nargs = ["-c", "{auth}"]\ntimeout_ms = 60000\nrefresh_interval_ms = 900000\n')

    def env(self) -> dict:
        env = dict(os.environ)
        for key in [k for k in env if k.startswith("DATABRICKS_") and k not in ("DATABRICKS_HOST", "DATABRICKS_TOKEN")]:
            env.pop(key)
        env.update({"GENIE_HOME": os.path.join(self.root, "home"), "DATABRICKS_HOST": self.host, "DATABRICKS_TOKEN": self.token,
                    "PATH": f"{self.cli_dir}:" + os.environ.get("PATH", ""), "HOME": self.root, "SHELL": "/bin/bash"})
        return env

    def _download(self, url: str, path: str):
        if self._fetch:
            return self._fetch(url, path)
        try:
            return RetryHandler.execute_with_retry(lambda: genie_download_once(url, path), max_attempts=GENIE_DOWNLOAD_ATTEMPTS,
                                                   logger=self.logger, context=f"downloading {url}", base_delay=2.0,
                                                   non_retryable_errors=["HTTP Error 404"])
        except Exception as error:
            raise RuntimeError(f"could not download {url}: {get_clean_error_message(error)}") from error

    def install(self) -> dict:
        import platform
        import tarfile
        import zipfile
        arch = platform.machine().lower()
        if arch not in GENIE_CODE_TARGETS:
            raise RuntimeError(f"Genie Code has no build for the {arch} architecture of this compute")
        bin_dir = os.path.join(self.root, "bin")
        genie = os.path.join(bin_dir, "genie")
        if not os.path.exists(genie):
            os.makedirs(os.path.join(self.root, "pkg"), exist_ok=True)
            package = os.path.join(self.root, "genie.tgz")
            self._download(GENIE_CODE_RELEASE_URL.format(version=GENIE_CODE_CLI_VERSION, target=GENIE_CODE_TARGETS[arch]), package)
            with tarfile.open(package) as tf:
                tf.extractall(os.path.join(self.root, "pkg"))
            found = next((os.path.join(d, "genie") for d, _s, files in os.walk(os.path.join(self.root, "pkg"))
                          if "genie" in files and d.endswith("bin")), "")
            if not found:
                raise RuntimeError("the Genie Code package holds no bin/genie")
            os.makedirs(bin_dir, exist_ok=True)
            os.symlink(found, genie)
        cli_dir = os.path.join(self.root, "cli")
        if not os.path.exists(os.path.join(cli_dir, "databricks")):
            archive = os.path.join(self.root, "cli.zip")
            self._download(GENIE_DATABRICKS_CLI_URL.format(version=GENIE_DATABRICKS_CLI_VERSION, arch=GENIE_CLI_ARCHES[arch]), archive)
            with zipfile.ZipFile(archive) as z:
                z.extractall(cli_dir)
            os.chmod(os.path.join(cli_dir, "databricks"), 0o755)
        home = os.path.join(self.root, "home")
        os.makedirs(home, exist_ok=True)
        with open(os.path.join(home, "config.toml"), "w", encoding="utf-8") as f:
            f.write(self.config_toml())
        self._write_cli_profile()
        self.genie_path, self.cli_dir = genie, cli_dir
        return {"genie": genie, "cli": os.path.join(cli_dir, "databricks"), "home": home}

    def _write_cli_profile(self):
        with open(os.path.join(self.root, ".databrickscfg"), "w", encoding="utf-8") as f:
            f.write(f"[DEFAULT]\nhost = {self.host}\n")

    def redact(self, text) -> str:
        text = str(text or "")
        return text.replace(self.token, "<redacted>") if self.token else text

    def scrub(self, work_dir: str) -> list:
        if not self.token:
            return []
        profile = os.path.realpath(os.path.join(self.root, ".databrickscfg"))
        needle, artifact_hits, seen = self.token.encode("utf-8"), [], set()
        for base, skip in ((work_dir, ()), (self.root, ("bin", "pkg", "cli"))):
            for relative in implement_secret_files(base, self.token, skip):
                path = os.path.realpath(os.path.join(base, relative))
                if path in seen:
                    continue
                seen.add(path)
                try:
                    if path == profile:
                        self._write_cli_profile()
                        action = "rewrote it without the token"
                    elif os.path.basename(path).startswith(".") or path.endswith(".pyc"):
                        os.remove(path)
                        action = "deleted it"
                    else:
                        with open(path, "rb") as f:
                            data = f.read()
                        with open(path, "wb") as f:
                            f.write(data.replace(needle, b"<redacted>"))
                        action = "redacted the token"
                        if base == work_dir and relative not in GENIE_ARTIFACT_SCRATCH_FILES:
                            artifact_hits.append(relative)
                except OSError as error:
                    action = f"could not clean it ({error})"
                self.logger.warning(f"[IMPLEMENT-SCRUB] {path} held the access token: {action}")
        return artifact_hits

    @staticmethod
    def _kill_group(proc):
        import signal
        try:
            group = int(proc.pid)
            if group in (os.getpid(), os.getpgrp()):
                raise ProcessLookupError("Genie shares this process group")
            os.killpg(group, signal.SIGKILL)
        except Exception:
            try:
                proc.kill()
            except Exception:
                pass

    def run(self, prompt: str, work_dir: str, output_schema: dict = None, label: str = "") -> dict:
        import subprocess
        import time
        if not self.genie_path:
            self.install()
        os.makedirs(work_dir, exist_ok=True)
        last_path = os.path.join(work_dir, "genie_last.txt")
        if os.path.exists(last_path):
            os.remove(last_path)
        command = [self.genie_path, "exec", "--skip-git-repo-check", "--ephemeral", "-s", "danger-full-access", "--json",
                   "-C", work_dir, "-o", last_path]
        if output_schema:
            schema_path = os.path.join(work_dir, "genie_output_schema.json")
            with open(schema_path, "w", encoding="utf-8") as f:
                json.dump(output_schema, f)
            command += ["--output-schema", schema_path]
        command.append(prompt + GENIE_CLI_AUTH_NOTE)
        started = time.time()
        popen = self._popen or subprocess.Popen
        proc = popen(command, env=self.env(), stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                     text=True, bufsize=1, start_new_session=True)
        events, errors, expired = 0, [], []
        import collections
        import threading
        stderr_tail = collections.deque(maxlen=200)
        done = threading.Event()

        def _drain_stderr():
            try:
                for text in proc.stderr or []:
                    stderr_tail.append(text)
            except (ValueError, OSError):
                pass

        def _expire():
            expired.append(True)
            self._kill_group(proc)

        def _reap():
            proc.wait()
            if not done.wait(GENIE_CODE_EXIT_GRACE_S):
                self.logger.warning(f"[IMPLEMENT-GENIE] {label or 'genie'}: Genie exited but a child process kept its output open "
                                    f"for {GENIE_CODE_EXIT_GRACE_S}s; stopping its process group")
                self._kill_group(proc)

        drain = threading.Thread(target=_drain_stderr, daemon=True)
        drain.start()
        threading.Thread(target=_reap, daemon=True).start()
        timer = threading.Timer(self.timeout_s, _expire)
        timer.start()
        try:
            for raw in proc.stdout:
                raw = raw.strip()
                if not raw:
                    continue
                events += 1
                try:
                    event = json.loads(raw)
                except ValueError:
                    continue
                line = self.redact(summarize_genie_event(event))
                if line:
                    self.on_line(f"[{label}] {line}" if label else line)
                    if line.startswith("Genie error"):
                        errors.append(line)
            code = proc.wait()
        finally:
            done.set()
            timer.cancel()
        drain.join(timeout=5)
        timed_out = bool(expired)
        stderr = self.redact("".join(stderr_tail))[-2000:]
        secret_files = self.scrub(work_dir)
        last = open(last_path, encoding="utf-8").read() if os.path.exists(last_path) else ""
        final = None
        if output_schema and last.strip():
            try:
                final = json.loads(last[last.index("{"):last.rindex("}") + 1])
            except ValueError:
                final = None
        return {"exit": code, "seconds": round(time.time() - started, 1), "events": events, "last": self.redact(last)[-4000:],
                "final": final, "errors": errors, "stderr": stderr, "timed_out": timed_out, "secret_files": secret_files}


class AtomicWriter:
    def __init__(self, spark, logger, inspire_database: str, session_id: int, widget_values: dict = None):
        import os
        import threading
        import tempfile
        self.spark = spark
        self.logger = logger
        self.inspire_database = (inspire_database or "").strip()
        self.session_id = int(session_id)
        self.widget_values = widget_values or {}
        self.enabled = bool(self.inspire_database)
        database_sql = quote_qualified_name(self.inspire_database)
        self.session_table = f"{database_sql}.__inspire_session" if self.enabled else ""
        self.steps_table = f"{database_sql}.__inspire_step" if self.enabled else ""
        self.lock_table = f"{database_sql}.__inspire_lock" if self.enabled else ""
        self._lock_holder = f"|run={current_job_run_id()}|holder={uuid.uuid4().hex}"
        self._lock = threading.Lock()
        self._flush_lock = threading.Lock()
        self._flush_wanted = False
        self._flush_running = False
        self._last_flush_attempt_epoch = 0.0
        self._flush_interval_seconds = 2.0
        self._step_counter = 0
        self.console_step_logging_enabled = str(self.widget_values.get("atomic_writer_console_logs", "false")).strip().lower() in {"1", "true", "yes", "y"}
        self.ailogs_table = f"{database_sql}.{AILOGS_TABLE}" if self.enabled else ""
        self.implement_table = f"{database_sql}.{IMPLEMENT_TABLE}" if self.enabled else ""
        self._spool_path = ""
        self._spool_offset = 0
        self._ailogs_spool_path = ""
        self._ailogs_offset = 0
        self._ailogs_ready = False
        self._steps_event_id = False
        self._chunk_failures = {}
        self._denied_spools = {}
        self.quarantined_rows = {}
        self.attached_to_existing_session = False
        self._session_writer = None
        if self.enabled:
            self._spool_path = b17_writable_file(
                os.path.join(tempfile.gettempdir(), "inspire_atomic_writer", f"session_{self.session_id}.jsonl"),
                "inspire_atomic_writer_", self.logger)
            self._spool_offset = os.path.getsize(self._spool_path)
            self._ailogs_spool_path = b17_writable_file(
                os.path.join(tempfile.gettempdir(), "inspire_atomic_writer", f"session_{self.session_id}_ailogs.jsonl"),
                "inspire_atomic_writer_ailogs_", self.logger)
            self._ailogs_offset = os.path.getsize(self._ailogs_spool_path)
            self._session_writer = SessionMutationWriter(self.spark, self.session_table, self.session_id, self.logger)

    def _console_log(self, message: str, level: str = "INFO", force: bool = False):
        if not force and level not in {"WARNING", "ERROR"} and not self.console_step_logging_enabled:
            return
        try:
            log_print(f"[AtomicWriter][session={self.session_id}] {message}", level=level)
        except Exception:
            pass

    def _utc_now_str(self):
        from datetime import datetime, timezone
        return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

    def _safe_json_str(self, payload):
        import json
        try:
            return json.dumps(payload if payload is not None else {}, ensure_ascii=True, default=str)
        except Exception as e:
            logger = getattr(self, "logger", None)
            if logger is not None:
                logger.warning(f"[STEP-PAYLOAD] a payload could not be serialized ({get_clean_error_message(e)}); it is written as {{}}")
            return "{}"

    def _next_step_id(self):
        import time
        with self._lock:
            self._step_counter += 1
            return int((time.time() * 1000) + self._step_counter)

    @staticmethod
    def _normalize_progress_increment(progress_increment: float) -> float:
        inc = float(progress_increment or 0.0)
        if inc <= 0.0:
            return 1.0
        return float(max(1, int(round(inc))))

    def _ensure_tables(self):
        if not self.enabled:
            return
        session_sql = f"""
        CREATE TABLE IF NOT EXISTS {self.session_table} (
            session_id BIGINT,
            business_name STRING,
            inspire_database_name STRING,
            operation_mode STRING,
            table_election_mode STRING,
            use_cases_quality STRING,
            strategic_goals STRING,
            business_priorities STRING,
            business_domains STRING,
            catalogs STRING,
            schemas_str STRING,
            tables_str STRING,
            generate_choices STRING,
            generation_path STRING,
            output_language STRING,
            sql_generation_per_domain STRING,
            technical_exclusion_strategy STRING,
            json_file_path STRING,
            discovery_type STRING,
            strategic_context STRING,
            processing_status STRING,
            completed_percent DOUBLE,
            create_at TIMESTAMP,
            last_updated TIMESTAMP,
            completed_on TIMESTAMP,
            inspire_json VARIANT,
            results_json VARIANT,
            enriched_business_context STRING,
            enriched_strategic_goals STRING,
            enriched_business_priorities STRING,
            enriched_strategic_initiative STRING,
            enriched_value_chain STRING,
            enriched_revenue_model STRING,
            generation_instructions_section STRING
        ) USING DELTA
        """
        steps_sql = f"""
        CREATE TABLE IF NOT EXISTS {self.steps_table} (
            session_id BIGINT,
            step_id BIGINT,
            last_updated TIMESTAMP,
            stage_name STRING,
            step_name STRING,
            sub_step_name STRING,
            progress_increment DOUBLE,
            message STRING,
            status STRING,
            result_json VARIANT
        ) USING DELTA
        """
        lock_sql = f"""
        CREATE TABLE IF NOT EXISTS {self.lock_table} (
            business_name STRING,
            session_id BIGINT,
            acquired_at TIMESTAMP,
            expires_at TIMESTAMP,
            holder_note STRING
        ) USING DELTA
        """
        self.spark.sql(session_sql)
        self.spark.sql(steps_sql)
        try:
            self.spark.sql(lock_sql)
        except Exception as _le:
            self.logger.debug(f"Lock table create failed (non-critical): {_le}")
        self._migrate_session_table_schema()
        self._steps_event_id = self._ensure_steps_event_id()
        try:
            self.spark.sql(ailogs_create_table_sql(self.ailogs_table))
            self._ailogs_ready = True
        except Exception as _ae:
            self.logger.warning(f"AI call log table {self.ailogs_table} unavailable ({get_clean_error_message(_ae)}); "
                                f"AI call rows stay in the local spool {self._ailogs_spool_path}")
        try:
            self._ensure_implement_table()
        except Exception as _ie:
            self.logger.warning(f"Implement table {self.implement_table} unavailable ({get_clean_error_message(_ie)}); "
                                f"Start Use Case PoC/MVP with Genie cannot run against this database")

    def _ensure_implement_table(self):
        if not self.enabled:
            return
        self.spark.sql(implement_table_create_sql(self.implement_table))
        existing = self.described_column_names(self.spark.sql(f"DESCRIBE TABLE {self.implement_table}").collect())
        for name, kind in IMPLEMENT_COLUMNS:
            if name.lower() not in existing:
                self.spark.sql(f"ALTER TABLE {self.implement_table} ADD COLUMN `{name}` {kind}")

    def _ensure_steps_event_id(self) -> bool:
        try:
            columns = self.described_column_names(self.spark.sql(f"DESCRIBE TABLE {self.steps_table}").collect())
        except Exception as e:
            self.logger.warning(f"Step table DESCRIBE failed ({get_clean_error_message(e)}); step re-sends are not de-duplicated")
            return False
        if "event_id" in columns:
            return True
        try:
            self.spark.sql(f"ALTER TABLE {self.steps_table} ADD COLUMN event_id STRING")
            return True
        except Exception as e:
            if "already exists" in str(e).lower() or "duplicate" in str(e).lower():
                return True
            self.logger.warning(f"Step table has no event_id column ({get_clean_error_message(e)}); steps are written as "
                                f"before and a re-send after a failure may repeat a step row")
            return False

    def emit_ailog(self, row: dict):
        """Spool one _ailogs row. Never raises: a log write never fails an LLM call."""
        if not self.enabled or not self._ailogs_spool_path:
            return
        import json
        try:
            line = json.dumps(row, ensure_ascii=True, default=str)
            with self._lock:
                with open(self._ailogs_spool_path, "a", encoding="utf-8") as f:
                    f.write(line + "\n")
        except Exception as e:
            self.logger.debug(f"AI call log spool write failed: {get_clean_error_message(e)}")
            return
        self._request_flush()

    def _ended_lock_holders(self, notes) -> list:
        ended = []
        for note in notes:
            if note.endswith(self._lock_holder):
                continue
            match = re.search(r"\|run=(\d+)\|holder=[0-9a-f]+$", note)
            run = job_run_ended(match.group(1)) if match else None
            if not run:
                return []
            ended.append((note, match.group(1), run["state"]))
        return ended

    def acquire_lock(self, business_name: str, ttl_seconds: int = 7200, holder_note: str = "", max_attempts: int = 5) -> bool:
        """True only when this session verifiably holds the per-business lock, False when another
        live session holds it. Raises RuntimeError when the lock table cannot be read or written,
        so an infrastructure failure is never reported as an acquired lock."""
        if not self.enabled or not (business_name or "").strip():
            return True
        import time
        business_sql = inspire_sql_literal(str(business_name).strip())
        note_sql = inspire_sql_literal(str(holder_note or "")[:150] + self._lock_holder)
        holder_sql = inspire_sql_literal("%" + self._lock_holder)
        last_error = None
        reclaimed = False
        for attempt in range(1, max(1, int(max_attempts)) + 1):
            try:
                self.spark.sql(
                    f"MERGE INTO {self.lock_table} AS target "
                    f"USING (SELECT {business_sql} AS business_name, {int(self.session_id)} AS session_id, "
                    f"current_timestamp() AS acquired_at, current_timestamp() + INTERVAL {int(ttl_seconds)} SECONDS AS expires_at, "
                    f"{note_sql} AS holder_note) AS source "
                    f"ON target.business_name = source.business_name "
                    f"WHEN MATCHED AND (target.holder_note LIKE {holder_sql} OR target.expires_at < current_timestamp()) THEN UPDATE SET "
                    f"target.session_id = source.session_id, target.acquired_at = source.acquired_at, "
                    f"target.expires_at = source.expires_at, target.holder_note = source.holder_note "
                    f"WHEN NOT MATCHED THEN INSERT (business_name, session_id, acquired_at, expires_at, holder_note) "
                    f"VALUES (source.business_name, source.session_id, source.acquired_at, source.expires_at, source.holder_note)"
                )
                holders = self.spark.sql(
                    f"SELECT DISTINCT session_id, holder_note FROM {self.lock_table} WHERE business_name = {business_sql}"
                ).collect()
                notes = [str(row[1] or "") for row in holders]
                if notes and all(note.endswith(self._lock_holder) for note in notes):
                    return True
                dead = [] if reclaimed else self._ended_lock_holders(notes)
                if notes and not dead:
                    return False
                for note, run_id, ended in dead:
                    self.spark.sql(f"DELETE FROM {self.lock_table} WHERE business_name = {business_sql} "
                                   f"AND holder_note = {inspire_sql_literal(note)}")
                    self.logger.warning(f"[LOCK RECLAIMED] the lock for '{business_name}' was held by run {run_id}, which ended "
                                        f"({ended}) without releasing it; this run takes it over")
                if dead:
                    reclaimed = True
                    continue
                last_error = RuntimeError(f"lock row for business '{business_name}' is missing after the MERGE")
            except Exception as err:
                last_error = err
                low = str(err).lower()
                if "concurrent" not in low:
                    break
            if attempt < max_attempts:
                time.sleep(0.2 * (2 ** (attempt - 1)))
        message = (f"Lock for business '{business_name}' could not be acquired or verified in {self.lock_table}: "
                   f"{get_clean_error_message(last_error) if last_error else 'unknown error'}")
        self.logger.warning(message)
        raise RuntimeError(message)

    def release_lock(self, business_name: str) -> None:
        if not self.enabled or not (business_name or "").strip():
            return
        try:
            self.spark.sql(
                f"DELETE FROM {self.lock_table} "
                f"WHERE business_name = {inspire_sql_literal(str(business_name).strip())} "
                f"AND holder_note LIKE {inspire_sql_literal('%' + self._lock_holder)}"
            )
        except Exception as _err:
            self.logger.warning(f"[G15 RELEASE_FAILED] the Generate lock for business '{business_name}' was not released and stays until it "
                                f"expires: {get_clean_error_message(_err)}")
    
    def _execute_session_write(self, sql, tag, wait=False, timeout=60.0) -> bool:
        if not self.enabled:
            return True
        if self._session_writer is None:
            self.spark.sql(sql)
            return True
        if wait:
            return self._session_writer.submit_and_wait(sql, tag=tag, timeout=timeout)
        self._session_writer.submit(sql, tag=tag)
        return True

    def _migrate_session_table_schema(self):
        """Idempotently add the per-session columns introduced after the first release: the 7
        enriched-context columns and, from v1.3.0, discovery_type and strategic_context.
        Logs at WARNING level on failure (was DEBUG — silently hid the bug where
        columns never got added and enriched context was lost on every run)."""
        if not self.enabled:
            return
        existing = set()
        self._session_missing_columns = set()
        try:
            _rows = self.spark.sql(f"DESCRIBE TABLE {self.session_table}").collect()
            existing = self.described_column_names(_rows)
        except Exception as e:
            self.logger.warning(f"Session schema DESCRIBE failed: {get_clean_error_message(e)} — migration will skip. "
                                f"Enriched context UPDATE will fail downstream.")
            return
        add_if_missing = [
            ("enriched_business_context", "STRING"),
            ("enriched_strategic_goals", "STRING"),
            ("enriched_business_priorities", "STRING"),
            ("enriched_strategic_initiative", "STRING"),
            ("enriched_value_chain", "STRING"),
            ("enriched_revenue_model", "STRING"),
            ("generation_instructions_section", "STRING"),
            ("discovery_type", "STRING"),
            ("strategic_context", "STRING"),
        ]
        added = []
        already = []
        failed = []
        for col_name, col_type in add_if_missing:
            if col_name.lower() in existing:
                already.append(col_name)
                continue
            try:
                self.spark.sql(f"ALTER TABLE {self.session_table} ADD COLUMN {col_name} {col_type}")
                added.append(col_name)
            except Exception as e:
                msg = str(e).lower()
                if "already exists" in msg or "duplicate" in msg:
                    already.append(col_name)
                    continue
                failed.append((col_name, get_clean_error_message(e)[:200]))
        self._session_missing_columns = {col_name.lower() for col_name, _error in failed}
        if added:
            self.logger.info(f"Session schema migrate: added {len(added)} columns: {added}")
        if failed:
            self.logger.warning(
                f"Session schema migrate: {len(failed)}/{len(add_if_missing)} ALTER TABLE ADD COLUMN failed. "
                f"Enriched context UPDATEs will fail for these columns. Details: {failed}"
            )

    @staticmethod
    def described_column_names(rows) -> set:
        names = set()
        for row in rows or []:
            try:
                name = row["col_name"]
            except Exception:
                name = getattr(row, "col_name", None)
            if name and not str(name).startswith("#"):
                names.add(str(name).strip().lower())
        return names

    def _widget_to_columns_map(self) -> dict:
        """Map widget_values dict keys to session table column names."""
        wv = self.widget_values or {}
        return {
            "business_name": str(wv.get("business", "") or ""),
            "inspire_database_name": str(wv.get("inspire_database", "") or ""),
            "operation_mode": str(wv.get("operation", wv.get("operation_mode", "")) or ""),
            "table_election_mode": str(wv.get("table_election_mode", "") or ""),
            "use_cases_quality": canonical_use_cases_quality(wv.get("use_cases_quality")),
            "strategic_goals": str(wv.get("generation_instructions", "") or ""),
            "business_priorities": str(wv.get("business_priorities", "") or ""),
            "business_domains": str(wv.get("business_domains", "") or ""),
            "catalogs": str(wv.get("catalogs", "") or ""),
            "schemas_str": str(wv.get("schemas", "") or ""),
            "tables_str": str(wv.get("tables", "") or ""),
            "generate_choices": str(wv.get("generate", "") or ""),
            "generation_path": str(wv.get("generation_path", "") or ""),
            "output_language": str(wv.get("output_language", "") or ""),
            "sql_generation_per_domain": str(wv.get("sql_generation_per_domain", "") or ""),
            "technical_exclusion_strategy": str(wv.get("technical_exclusion_strategy", "") or ""),
            "json_file_path": str(wv.get("json_file_path", "") or ""),
            "discovery_type": canonical_discovery_type(wv.get("discovery_type")),
            "strategic_context": str(wv.get("strategic_context_json", "") or ""),
        }

    def _session_row_exists(self) -> bool:
        try:
            return bool(self.spark.sql(
                f"SELECT session_id FROM {self.session_table} WHERE session_id = {self.session_id} LIMIT 1").collect())
        except Exception as error:
            self.logger.warning(f"AtomicWriter: could not read session {self.session_id} from {self.session_table} "
                                f"({get_clean_error_message(error)}); the Generate run keeps any existing row as it is")
            return True

    def initialize_session(self):
        if not self.enabled:
            self._console_log("inspire_database not provided; running in console-only mode")
            return
        self._ensure_tables()
        ts = self._utc_now_str()
        inspire_json = inspire_sql_literal(self._safe_json_str({}))
        col_map = self._widget_to_columns_map()
        if col_map["operation_mode"] in INSPIRE_ATTACHING_OPERATIONS and self._session_row_exists():
            self.attached_to_existing_session = True
            self._execute_session_write(
                f"UPDATE {self.session_table} SET last_updated = TIMESTAMP('{ts}') WHERE session_id = {self.session_id}",
                tag="attach", wait=True)
            self.logger.info(f"[SESSION-ATTACH FIRED] {col_map['operation_mode']} on existing session {self.session_id}: its row keeps "
                             "its results, settings, creation time and completion; this run only updates last_updated")
            return
        missing_columns = getattr(self, "_session_missing_columns", None) or set()
        missing = [k for k in col_map if k.lower() in missing_columns]
        if missing:
            self.logger.warning(f"Session table {self.session_table} lacks columns {missing}; "
                                f"this session row is written without them")
            col_map = {k: v for k, v in col_map.items() if k.lower() not in missing_columns}
        widget_col_select = ", ".join(
            f"{inspire_sql_literal(v)} AS {k}" for k, v in col_map.items()
        )
        widget_col_update = ", ".join(
            f"target.{k} = source.{k}" for k in col_map.keys()
        )
        widget_col_insert_names = ", ".join(col_map.keys())
        widget_col_insert_vals = ", ".join(f"source.{k}" for k in col_map.keys())

        merge_sql = f"""
        MERGE INTO {self.session_table} AS target
        USING (
            SELECT
                {self.session_id} AS session_id,
                {widget_col_select},
                'done' AS processing_status,
                1.0 AS completed_percent,
                TIMESTAMP('{ts}') AS create_at,
                TIMESTAMP('{ts}') AS last_updated,
                CAST(NULL AS TIMESTAMP) AS completed_on,
                parse_json({inspire_json}) AS inspire_json,
                parse_json({inspire_json}) AS results_json
        ) AS source
        ON target.session_id = source.session_id
        WHEN MATCHED THEN UPDATE SET
            {widget_col_update},
            target.processing_status = source.processing_status,
            target.completed_percent = source.completed_percent,
            target.create_at = source.create_at,
            target.last_updated = source.last_updated,
            target.completed_on = source.completed_on,
            target.inspire_json = source.inspire_json,
            target.results_json = source.results_json
        WHEN NOT MATCHED THEN INSERT (
            session_id, {widget_col_insert_names},
            processing_status, completed_percent, create_at, last_updated, completed_on,
            inspire_json, results_json
        ) VALUES (
            source.session_id, {widget_col_insert_vals},
            source.processing_status, source.completed_percent, source.create_at,
            source.last_updated, source.completed_on,
            source.inspire_json, source.results_json
        )
        """
        if not self._execute_session_write(merge_sql, tag="initialize", wait=True):
            raise RuntimeError(
                f"Inspire session {self.session_id} could not be recorded in {self.session_table}; "
                f"the Inspire App would not see this run. Check write access to {self.inspire_database}.")
        self._console_log("session initialized and persisted", force=True)
    
    def update_session_prompt_context(self, context_fields: dict):
        """Persist the 9 per-session Genie-prompt context fields onto the session row so
        Generate mode can rebuild prompts without re-running business-context extraction.
        Called once after business context is extracted. Idempotent (UPDATE statement)."""
        if not self.enabled or not context_fields:
            return
        allowed_cols = {
            "enriched_business_context",
            "enriched_strategic_goals",
            "enriched_business_priorities",
            "enriched_strategic_initiative",
            "enriched_value_chain",
            "enriched_revenue_model",
            "generation_instructions_section",
        }
        set_parts = []
        for col, val in context_fields.items():
            if col not in allowed_cols:
                continue
            set_parts.append(f"{col} = {inspire_sql_literal(str(val or ''))}")
        if not set_parts:
            return
        try:
            set_clause = ", ".join(set_parts)
            self._execute_session_write(
                f"UPDATE {self.session_table} SET {set_clause} WHERE session_id = {self.session_id}",
                tag="prompt_context",
            )
            self._console_log(f"session prompt context updated ({len(set_parts)} fields)", force=False)
        except Exception as e:
            self._console_log(f"session prompt context update failed (non-critical): {e}", level="WARNING")

    def emit_step(self, stage_name: str, step_name: str, sub_step_name: str = "", progress_increment: float = 0.0,
                  message: str = "", status: str = "started", result_json=None, step_id: int = None):
        effective_step_id = int(step_id) if step_id is not None else self._next_step_id()
        from datetime import datetime, timezone
        payload = result_json if isinstance(result_json, dict) else {}
        payload.setdefault("event_at", datetime.now(timezone.utc).isoformat())
        payload.setdefault("entity_id", f"{self.session_id}:{stage_name}:{step_name}:{sub_step_name}")
        normalized_increment = self._normalize_progress_increment(progress_increment)
        if self.console_step_logging_enabled or status == "ended_error":
            self._console_log(
                f"{status} | stage='{stage_name}' step='{step_name}' sub_step='{sub_step_name}' progress={normalized_increment} | {message}",
                level="ERROR" if status == "ended_error" else "INFO"
            )
        if not self.enabled:
            return effective_step_id
        import json
        event = {
            "session_id": self.session_id,
            "step_id": effective_step_id,
            "stage_name": str(stage_name or ""),
            "step_name": str(step_name or ""),
            "sub_step_name": str(sub_step_name or ""),
            "progress_increment": normalized_increment,
            "message": str(message or ""),
            "status": str(status or "started"),
            "result_json": payload,
            "event_id": uuid.uuid4().hex,
        }
        with self._lock:
            with open(self._spool_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(event, ensure_ascii=True) + "\n")
        self._request_flush()
        return event["step_id"]

    def _request_flush(self):
        import threading
        with self._lock:
            self._flush_wanted = True
            if self._flush_running:
                return
            self._flush_running = True
        threading.Thread(target=self._flush_until_drained, daemon=True, name=f"AtomicWriterFlush_{self.session_id}").start()

    def _flush_until_drained(self):
        import time
        while True:
            wait = self._flush_interval_seconds - (time.time() - self._last_flush_attempt_epoch)
            if wait > 0:
                time.sleep(wait)
            with self._lock:
                if not self._flush_wanted:
                    self._flush_running = False
                    return
                self._flush_wanted = False
                self._last_flush_attempt_epoch = time.time()
            try:
                self.flush_pending(force=False, finalize=False)
            except Exception as e:
                self.logger.warning(f"AtomicWriter: background flush failed, the next event retries it: {get_clean_error_message(e)}")
            except BaseException:
                with self._lock:
                    self._flush_running = False
                raise

    def _get_session_processing_status(self):
        if not self.enabled:
            return None
        try:
            rows = self.spark.sql(
                f"SELECT processing_status FROM {self.session_table} WHERE session_id = {self.session_id} LIMIT 1"
            ).collect()
            if not rows:
                return None
            return str(rows[0]["processing_status"]).lower() if rows[0]["processing_status"] is not None else None
        except Exception as e:
            self.logger.warning(f"AtomicWriter: failed reading processing_status: {get_clean_error_message(e)}")
            return None

    _CHUNK_MAX_BYTES = 8 * 1024 * 1024
    _DRAIN_MAX_BYTES = 4 * _CHUNK_MAX_BYTES
    _QUARANTINE_AFTER = 5

    def _read_spool_entries(self, path: str, offset: int, max_bytes: int) -> list:
        import os
        import json
        entries, read = [], 0
        with self._lock:
            if not path or not os.path.exists(path):
                return entries
            with open(path, "r", encoding="utf-8") as f:
                f.seek(offset)
                while read < max_bytes:
                    line = f.readline()
                    if not line:
                        break
                    end = f.tell()
                    text = line.strip()
                    event = None
                    if text:
                        try:
                            event = json.loads(text)
                        except Exception:
                            event = None
                    entries.append((end, event, len(text)))
                    read += len(text)
        return entries

    def _drain_spool(self, kind: str, path: str, offset_attr: str, insert_chunk, id_field: str,
                     max_rows: int = 300) -> tuple:
        """Insert spooled rows chunk by chunk. Rows are read under the lock and inserted outside it;
        a chunk is capped by rows and bytes; the offset advances per landed chunk; a failed chunk is
        re-sent through an insert-only MERGE on its row id, and quarantined after repeated failures."""
        import json
        if kind in self._denied_spools:
            return 0, 0.0
        entries = self._read_spool_entries(path, getattr(self, offset_attr), self._DRAIN_MAX_BYTES)
        chunks, current, size = [], [], 0
        for entry in entries:
            if current and (len(current) >= max_rows or size + entry[2] > self._CHUNK_MAX_BYTES):
                chunks.append(current)
                current, size = [], 0
            current.append(entry)
            size += entry[2]
        if current:
            chunks.append(current)
        inserted, extra = 0, 0.0
        for chunk in chunks:
            events = [event for _end, event, _size in chunk if isinstance(event, dict)]
            if events:
                key = (kind, str(events[0].get(id_field)))
                failures = self._chunk_failures.get(key, 0)
                try:
                    extra += insert_chunk(events, failures > 0) or 0.0
                except Exception as error:
                    if _evidence_error_status(error) == "denied":
                        self._deny_spool(kind, path, error)
                        break
                    failures += 1
                    if failures < self._QUARANTINE_AFTER:
                        self._chunk_failures[key] = failures
                        self.logger.warning(f"AtomicWriter: {kind} chunk of {len(events)} row(s) failed "
                                            f"({failures}/{self._QUARANTINE_AFTER}): {get_clean_error_message(error)}; "
                                            f"it is re-sent without duplicates on the next flush")
                        break
                    self._chunk_failures.pop(key, None)
                    self.quarantined_rows[kind] = self.quarantined_rows.get(kind, 0) + len(events)
                    self.logger.error(f"AtomicWriter: {kind} chunk of {len(events)} row(s) quarantined after {failures} "
                                      f"failures ({get_clean_error_message(error)}); first row: "
                                      f"{json.dumps(events[0], default=str)[:300]}")
                else:
                    self._chunk_failures.pop(key, None)
                    inserted += len(events)
            with self._lock:
                setattr(self, offset_attr, max(getattr(self, offset_attr), chunk[-1][0]))
        return inserted, extra

    def _deny_spool(self, kind: str, path: str, error):
        table = self.ailogs_table if kind == "ailogs" else self.steps_table
        self._denied_spools[kind] = path
        self._chunk_failures = {key: count for key, count in self._chunk_failures.items() if key[0] != kind}
        self.logger.warning(f"[SPOOL-DENIED FIRED] AtomicWriter: this run's identity may not write {table} "
                            f"({get_clean_error_message(error)}); its {kind} rows stay in {path} and are not sent again "
                            "in this run")

    def denied_spool_path(self, kind: str) -> str:
        return self._denied_spools.get(kind, "")

    def _consume_pending_events_stream(self, ts: str, chunk_size: int = 300):
        inserted_count, progress_increment_sum = self._drain_spool(
            "steps", self._spool_path, "_spool_offset",
            lambda rows, resend: self._insert_steps_chunk(rows, ts, resend=resend), "event_id", max_rows=chunk_size)
        return inserted_count, self._spool_offset, progress_increment_sum

    def _insert_steps_chunk(self, rows: list, ts: str, resend: bool = False):
        if not rows:
            return 0.0
        import time
        from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType
        with_event_id = bool(self._steps_event_id)
        schema = StructType([
            StructField("session_id", LongType(), False),
            StructField("step_id", LongType(), False),
            StructField("stage_name", StringType(), True),
            StructField("step_name", StringType(), True),
            StructField("sub_step_name", StringType(), True),
            StructField("progress_increment", DoubleType(), True),
            StructField("message", StringType(), True),
            StructField("status", StringType(), True),
            StructField("result_json_str", StringType(), True),
        ] + ([StructField("event_id", StringType(), True)] if with_event_id else []))
        payload_rows = []
        success_increment_sum = 0.0
        for row in rows:
            status = str(row.get("status", "") or "")
            inc = float(row.get("progress_increment", 0.0) or 0.0)
            if status == "ended_success":
                success_increment_sum += inc
            payload_rows.append((
                int(row.get("session_id", self.session_id)),
                int(row.get("step_id")),
                str(row.get("stage_name", "")),
                str(row.get("step_name", "")),
                str(row.get("sub_step_name", "")),
                inc,
                str(row.get("message", "")),
                status,
                self._safe_json_str(row.get("result_json", {}))
            ) + ((str(row.get("event_id") or uuid.uuid4().hex),) if with_event_id else ()))
        df = self.spark.createDataFrame(payload_rows, schema=schema)
        view_name = f"_inspire_steps_queue_{self.session_id}_{int(time.time() * 1000)}"
        df.createOrReplaceTempView(view_name)
        event_column = ", event_id" if with_event_id else ""
        columns = f"session_id, step_id, last_updated, stage_name, step_name, sub_step_name, progress_increment, message, status, result_json{event_column}"
        selected = (f"session_id, step_id, TIMESTAMP('{ts}') AS last_updated, stage_name, step_name, sub_step_name, "
                    f"progress_increment, message, status, parse_json(result_json_str) AS result_json{event_column}")
        if resend and with_event_id:
            values = ", ".join(f"s.{c.strip()}" for c in columns.split(","))
            insert_sql = (f"MERGE INTO {self.steps_table} AS t USING (SELECT {selected} FROM {view_name}) AS s "
                          f"ON t.session_id = s.session_id AND t.event_id = s.event_id "
                          f"WHEN NOT MATCHED THEN INSERT ({columns}) VALUES ({values})")
        else:
            insert_sql = f"INSERT INTO {self.steps_table} ({columns}) SELECT {selected} FROM {view_name}"
        self.spark.sql(insert_sql)
        try:
            self.spark.catalog.dropTempView(view_name)
        except Exception:
            pass
        return success_increment_sum

    _AILOGS_SPARK_TYPES = {"STRING": "StringType", "BIGINT": "LongType", "INT": "IntegerType", "DOUBLE": "DoubleType",
                           "TIMESTAMP": "StringType"}

    def _insert_ailogs_chunk(self, rows: list, resend: bool = False) -> float:
        import time
        from pyspark.sql import types as spark_types
        columns = [(name, kind) for name, kind in AILOGS_COLUMNS if name != "server_ingested_at"]
        schema = spark_types.StructType([spark_types.StructField(name, getattr(spark_types, self._AILOGS_SPARK_TYPES[kind])(), True)
                                         for name, kind in columns])

        def _value(value, kind):
            if value is None:
                return None
            if kind in ("BIGINT", "INT"):
                return int(value)
            if kind == "DOUBLE":
                return float(value)
            return str(value)
        df = self.spark.createDataFrame([tuple(_value(row.get(name), kind) for name, kind in columns) for row in rows],
                                        schema=schema)
        view_name = f"_inspire_ailogs_queue_{self.session_id}_{int(time.time() * 1000)}"
        df.createOrReplaceTempView(view_name)
        names = ", ".join(f"`{name}`" for name, _kind in AILOGS_COLUMNS)
        selected = ", ".join(f"CAST(`{name}` AS TIMESTAMP) AS `{name}`" if kind == "TIMESTAMP" else f"`{name}`"
                             for name, kind in columns) + ", current_timestamp() AS `server_ingested_at`"
        if resend:
            values = ", ".join(f"s.`{name}`" for name, _kind in AILOGS_COLUMNS)
            insert_sql = (f"MERGE INTO {self.ailogs_table} AS t USING (SELECT {selected} FROM {view_name}) AS s "
                          f"ON t.`session_id` = s.`session_id` AND t.`ailogs_id` = s.`ailogs_id` "
                          f"WHEN NOT MATCHED THEN INSERT ({names}) VALUES ({values})")
        else:
            insert_sql = f"INSERT INTO {self.ailogs_table} ({names}) SELECT {selected} FROM {view_name}"
        self.spark.sql(insert_sql)
        try:
            self.spark.catalog.dropTempView(view_name)
        except Exception:
            pass
        return 0.0

    def _drain_ailogs(self, rounds: int = 1) -> int:
        import os
        if not self._ailogs_ready:
            return 0
        inserted = 0
        try:
            for _ in range(max(1, rounds)):
                count, _extra = self._drain_spool("ailogs", self._ailogs_spool_path, "_ailogs_offset",
                                                  self._insert_ailogs_chunk, "ailogs_id")
                inserted += count
                if not count or self._ailogs_offset >= os.path.getsize(self._ailogs_spool_path):
                    break
        except Exception as error:
            self.logger.warning(f"AtomicWriter: AI call log flush failed: {get_clean_error_message(error)}")
        return inserted

    def _update_session_ready(self, ts: str, progress_increment_sum: float):
        if self.attached_to_existing_session:
            self._execute_session_write(
                f"UPDATE {self.session_table} SET last_updated = TIMESTAMP('{ts}') WHERE session_id = {self.session_id}",
                tag="ready_progress")
            return
        update_sql = f"""
        UPDATE {self.session_table}
        SET
            completed_percent = LEAST(99.0, CAST(FLOOR(completed_percent + {float(progress_increment_sum)}) AS DOUBLE)),
            processing_status = 'ready',
            last_updated = TIMESTAMP('{ts}'),
            inspire_json = parse_json({inspire_sql_literal(self._safe_json_str({"last_flush": ts}))})
        WHERE session_id = {self.session_id}
        """
        self._execute_session_write(update_sql, tag="ready_progress")

    def _finalize_session(self, ts: str, final_results_json=None):
        if self.attached_to_existing_session:
            if not self._execute_session_write(
                    f"UPDATE {self.session_table} SET last_updated = TIMESTAMP('{ts}') WHERE session_id = {self.session_id}",
                    tag="finalize", wait=True, timeout=300.0):
                self.logger.warning(f"AtomicWriter: last_updated of session {self.session_id} could not be written; the "
                                    "session keeps its Discover completion")
            self.session_finalized = True
            return True
        final_payload = final_results_json if final_results_json is not None else {"completed": True, "completed_on": ts}
        final_payload_sql = inspire_sql_literal(self._safe_json_str(final_payload))
        finalize_sql = f"""
        UPDATE {self.session_table}
        SET
            completed_percent = 100.0,
            processing_status = 'ready',
            last_updated = TIMESTAMP('{ts}'),
            completed_on = TIMESTAMP('{ts}'),
            inspire_json = parse_json({final_payload_sql}),
            results_json = parse_json({final_payload_sql})
        WHERE session_id = {self.session_id}
        """
        finalized = self._execute_session_write(finalize_sql, tag="finalize", wait=True, timeout=300.0)
        if not finalized:
            self.logger.error(
                f"Inspire session {self.session_id} completion was not recorded in {self.session_table}; "
                f"the Inspire App will keep showing this run as in progress")
        self.session_finalized = finalized
        return finalized

    def flush_pending(self, force: bool = False, finalize: bool = False, final_results_json=None):
        if not self.enabled:
            return 0
        with self._flush_lock:
            if not force:
                processing_status = self._get_session_processing_status()
                if processing_status is None:
                    return 0
            ts = self._utc_now_str()
            inserted_count, new_offset, progress_increment_sum = self._consume_pending_events_stream(ts=ts, chunk_size=300)
            self._drain_ailogs(rounds=20 if finalize else 1)
            if inserted_count == 0 and not finalize:
                return 0
            with self._lock:
                self._spool_offset = max(self._spool_offset, new_offset)
            if finalize:
                self._finalize_session(ts, final_results_json=final_results_json)
            elif inserted_count > 0:
                self._update_session_ready(ts, progress_increment_sum)
            return inserted_count

    def finalize_pipeline(self, message: str = "Completed", final_results_json=None, pipeline_start_step_id: int = None,
                          status: str = "ended_success"):
        if not self.enabled:
            self._console_log(f"finalize (console-only): {message}")
            return
        self.emit_step(
            stage_name="Inspire Journey",
            step_name="Inspire pipeline completed",
            sub_step_name="Completed",
            progress_increment=1.0,
            message=message,
            status=status,
            result_json={"session_id": self.session_id},
            step_id=pipeline_start_step_id
        )
        try:
            self.flush_pending(force=True, finalize=True, final_results_json=final_results_json)
            self._console_log("finalize completed with forced flush and session completion")
        finally:
            if self._session_writer is not None:
                self._session_writer.stop(drain_timeout=15.0)
        if getattr(self, "session_finalized", True) is False:
            raise RuntimeError(
                f"Inspire session {self.session_id} completion could not be recorded in {self.session_table}")


BUSINESS_CONTEXT_ASSUMPTION_FIELDS = ("business_context", "value_chain")
BUSINESS_CONTEXT_USER_ONLY_FIELDS = ("strategic_initiative", "revenue_model")
_MONETARY_FIGURE_RE = re.compile(
    r"(?:[$€£¥]\s?|\b(?:usd|eur|gbp|aed|sar|inr|jpy|cny)\s?)\d[\d,.]*(?:\s?(?:k|m|mm|bn|b|million|billion|trillion|thousand)\b)?"
    r"|\b\d[\d,.]*\s?(?:million|billion|trillion)\b",
    re.IGNORECASE)


def label_inferred_business_context(context_data, business_name):
    if not isinstance(context_data, dict):
        return context_data
    label = f"[ASSUMPTION inferred from the business name '{business_name}' only, not evidenced by the metadata] "

    def labelled(value):
        text = _MONETARY_FIGURE_RE.sub(CONTEXT_NOT_PROVIDED_MARKER, value).strip()
        if not text or text.startswith("[ASSUMPTION") or text.upper().startswith(CONTEXT_NOT_PROVIDED_MARKER):
            return text
        return label + text

    labeled = dict(context_data)
    for field_name in BUSINESS_CONTEXT_USER_ONLY_FIELDS:
        labeled[field_name] = CONTEXT_NOT_PROVIDED_MARKER
    for field_name in BUSINESS_CONTEXT_ASSUMPTION_FIELDS:
        value = labeled.get(field_name)
        if isinstance(value, str):
            labeled[field_name] = labelled(value)
    priorities = labeled.get("business_priorities")
    if isinstance(priorities, (list, tuple)):
        priorities = ", ".join(str(item).strip() for item in priorities if str(item or "").strip())
    if isinstance(priorities, str):
        text = labelled(priorities)
        labeled["business_priorities"] = [text] if text.startswith("[ASSUMPTION") else text
    return labeled


WORKSPACE_TREE_ROOTS = ("/Users/", "/Shared/", "/Repos/")


def canonical_workspace_path(path) -> str:
    text = str(path or "").strip()
    if text.startswith(WORKSPACE_TREE_ROOTS) and not os.path.exists(text):
        return "/Workspace" + text
    return text


def is_workspace_path(path) -> bool:
    return canonical_workspace_path(path).startswith("/Workspace/")


def resolve_generation_path(generation_path, notebook_path=None) -> str:
    path = str(generation_path or "").strip()
    if path.startswith("./"):
        path = os.path.normpath(os.path.join(os.path.dirname(str(notebook_path or "")), path[2:]))
    elif '/..' in path or path.startswith('..'):
        path = os.path.normpath(path)
    return canonical_workspace_path(path)


def read_user_text_file(path, w_client=None) -> str:
    text_path = str(path or "").strip()
    if not text_path.startswith("/"):
        raise ValueError(f"{text_path!r} is not an absolute /Workspace or /Volumes path")
    if text_path.startswith("/Volumes/") or text_path.startswith("/Workspace/"):
        try:
            with open(text_path, "r", encoding="utf-8") as handle:
                return handle.read()
        except OSError as open_error:
            if text_path.startswith("/Volumes/"):
                client = w_client if w_client is not None else WorkspaceClient()
                try:
                    return client.files.download(file_path=text_path).contents.read().decode("utf-8")
                except Exception:
                    raise open_error
    client = w_client if w_client is not None else WorkspaceClient()
    candidates = [text_path]
    if text_path.startswith("/Workspace/"):
        candidates.append(text_path[len("/Workspace"):])
    last_error = None
    for candidate in candidates:
        try:
            exported = client.workspace.export(path=candidate, format=workspace.ExportFormat.AUTO)
            return base64.b64decode(exported.content).decode("utf-8")
        except Exception as export_error:
            last_error = export_error
    raise last_error


def _strategy_name(value, what: str) -> str:
    text = str(value if value is not None else "").strip()
    if not text:
        raise ValueError(f"Strategic Context: {what} has an empty name")
    if len(text) > STRATEGY_MAX_NAME_CHARS:
        raise ValueError(f"Strategic Context: {what} {text[:60]!r}... is longer than {STRATEGY_MAX_NAME_CHARS} characters")
    return text


def _strategy_optional_text(value) -> str:
    return str(value).strip() if value is not None and str(value).strip() else ""


def parse_strategic_context(raw, read_file=None):
    if raw is None:
        return None
    if isinstance(raw, dict):
        data = raw
    else:
        text = str(raw).strip()
        if not text:
            return None
        if text.startswith("/"):
            if read_file is None:
                raise ValueError(f"Strategic Context {text!r} is a path, but no file reader is available")
            try:
                text = read_file(text)
            except Exception as read_error:
                raise ValueError(f"Strategic Context file {str(raw).strip()!r} could not be read: "
                                 f"{get_clean_error_message(read_error)}") from read_error
        if len(text.encode("utf-8")) > STRATEGY_MAX_INLINE_BYTES:
            raise ValueError(f"Strategic Context is larger than {STRATEGY_MAX_INLINE_BYTES} bytes")
        try:
            data = json.loads(text)
        except json.JSONDecodeError as json_error:
            raise ValueError(f"Strategic Context is not valid JSON: {json_error.msg} at line "
                             f"{json_error.lineno}, column {json_error.colno}") from json_error
    if not isinstance(data, dict):
        raise ValueError("Strategic Context must be a JSON object")
    if not data:
        return None
    warnings_out = []
    version = data.get("schema_version", STRATEGY_SCHEMA_VERSION)
    if version != STRATEGY_SCHEMA_VERSION:
        raise ValueError(f"Strategic Context schema_version {version!r} is not supported; use {STRATEGY_SCHEMA_VERSION}")
    known_keys = {"schema_version", "levels", "strategy", "priorities", "sponsors", "labels", "source", "warnings"}
    for key in data:
        if key not in known_keys:
            warnings_out.append(f"unknown key {key!r} ignored")
    raw_levels = data.get("levels") or []
    if not isinstance(raw_levels, list):
        raise ValueError("Strategic Context: levels must be a list of level names")
    levels = [_strategy_name(level, f"level {index}") for index, level in enumerate(raw_levels, 1)]
    if len({level.casefold() for level in levels}) != len(levels):
        raise ValueError("Strategic Context: level names must be unique")
    raw_strategy = data.get("strategy") or []
    if not isinstance(raw_strategy, list):
        raise ValueError("Strategic Context: strategy must be a list of entries")
    counters = {"entries": 0, "kpis": 0, "key_results": 0, "depth": 0}

    def _kpis(items, where):
        out = []
        for item in items or []:
            spec = item if isinstance(item, dict) else {"name": item}
            direction = _strategy_optional_text(spec.get("direction")).casefold()
            if direction and direction not in KPI_DIRECTIONS:
                raise ValueError(f"Strategic Context: KPI direction {spec.get('direction')!r} on {where} must be one of "
                                 f"{', '.join(KPI_DIRECTIONS)}")
            counters["kpis"] += 1
            out.append({"id": f"K{counters['kpis']}", "name": _strategy_name(spec.get("name"), f"a KPI on {where}"),
                        "definition": _strategy_optional_text(spec.get("definition")),
                        "unit": _strategy_optional_text(spec.get("unit")), "direction": direction})
        return out

    def _key_results(items, kpis, where):
        out = []
        kpi_names = {kpi["name"].casefold(): kpi["id"] for kpi in kpis}
        for item in items or []:
            spec = item if isinstance(item, dict) else {"statement": item}
            statement = _strategy_name(spec.get("statement") or spec.get("name"), f"a key result on {where}")
            kpi_ref = _strategy_optional_text(spec.get("kpi"))
            kpi_id = kpi_names.get(kpi_ref.casefold()) if kpi_ref else None
            prior_id = _strategy_optional_text(spec.get("kpi_id"))
            if not kpi_ref and prior_id in {kpi["id"] for kpi in kpis}:
                kpi_id = prior_id
            if kpi_ref and not kpi_id:
                warnings_out.append(f"key result {statement[:60]!r} names KPI {kpi_ref!r}, which is not on {where}")
            counters["key_results"] += 1
            out.append({"id": f"R{counters['key_results']}", "statement": statement, "kpi_id": kpi_id,
                        "baseline": _strategy_optional_text(spec.get("baseline")),
                        "target": _strategy_optional_text(spec.get("target")),
                        "due": _strategy_optional_text(spec.get("due"))})
        return out

    def _entries(items, depth, parent_id, parent_name):
        if not isinstance(items, list):
            raise ValueError(f"Strategic Context: children of {parent_name!r} must be a list")
        if items and depth > STRATEGY_MAX_DEPTH:
            raise ValueError(f"Strategic Context: the strategy tree is deeper than {STRATEGY_MAX_DEPTH} levels")
        out, seen = [], set()
        for index, item in enumerate(items, 1):
            spec = item if isinstance(item, dict) else {"name": item}
            entry_id = f"{parent_id}.{index}" if parent_id else f"S{index}"
            name = _strategy_name(spec.get("name"), f"entry {entry_id}")
            if name.casefold() in seen:
                raise ValueError(f"Strategic Context: {name!r} appears twice under {parent_name!r}")
            seen.add(name.casefold())
            counters["entries"] += 1
            if counters["entries"] > STRATEGY_MAX_ENTRIES:
                raise ValueError(f"Strategic Context: more than {STRATEGY_MAX_ENTRIES} strategy entries")
            counters["depth"] = max(counters["depth"], depth)
            kpis = _kpis(spec.get("kpis"), repr(name))
            out.append({"id": entry_id, "name": name, "depth": depth,
                        "description": _strategy_optional_text(spec.get("description")),
                        "kpis": kpis, "key_results": _key_results(spec.get("key_results"), kpis, repr(name)),
                        "children": _entries(spec.get("children") or [], depth + 1, entry_id, name)})
        return out

    strategy = _entries(raw_strategy, 1, "", "the strategy")
    depth = counters["depth"]
    if depth > 1 and not levels:
        raise ValueError("Strategic Context: levels is required when the strategy tree is deeper than one level")
    if depth and levels and len(levels) < depth:
        raise ValueError(f"Strategic Context: the tree is {depth} levels deep but levels names only {len(levels)}")
    if not levels and depth:
        levels = [STRATEGY_DEFAULT_LEVEL]

    def _label(nodes):
        for node in nodes:
            node["level"] = levels[node.pop("depth") - 1]
            _label(node["children"])
    _label(strategy)
    raw_priorities = data.get("priorities") or []
    if not isinstance(raw_priorities, list):
        raise ValueError("Strategic Context: priorities must be a list")
    priorities, seen_priorities = [], set()
    for item in raw_priorities:
        name = _strategy_name(item.get("name") if isinstance(item, dict) else item, "a priority")
        if name.casefold() not in seen_priorities:
            seen_priorities.add(name.casefold())
            priorities.append(name)
    raw_sponsors = data.get("sponsors") or []
    if not isinstance(raw_sponsors, list):
        raise ValueError("Strategic Context: sponsors must be a list")
    sponsors = []
    for index, item in enumerate(raw_sponsors, 1):
        spec = item if isinstance(item, dict) else {"role": item}
        owns = spec.get("owns") or []
        if isinstance(owns, str):
            owns = [owns]
        sponsors.append({"id": f"SP{index}", "role": _strategy_name(spec.get("role"), f"sponsor {index}"),
                         "name": _strategy_optional_text(spec.get("name")),
                         "reports_to": _strategy_optional_text(spec.get("reports_to")),
                         "owns": [str(o).strip() for o in owns if str(o).strip()]})
    raw_labels = data.get("labels") or {}
    if not isinstance(raw_labels, dict):
        raise ValueError("Strategic Context: labels must be an object")
    labels = {}
    for key, value in raw_labels.items():
        if key not in STRATEGY_LABEL_KEYS:
            warnings_out.append(f"label {key!r} ignored; supported labels: {', '.join(STRATEGY_LABEL_KEYS)}")
            continue
        labels[key] = _strategy_name(value, f"label {key}")
    if not strategy and not priorities and not sponsors and not labels:
        return None
    return {"schema_version": STRATEGY_SCHEMA_VERSION, "source": "customer", "levels": levels if strategy else [],
            "strategy": strategy, "priorities": priorities, "sponsors": sponsors, "labels": labels,
            "warnings": warnings_out}


def _split_outside_parentheses(text: str) -> list:
    parts, depth, current = [], 0, []
    for char in str(text or ""):
        if char == "(":
            depth += 1
        elif char == ")":
            depth = max(0, depth - 1)
        if char == "," and depth == 0:
            parts.append("".join(current))
            current = []
        else:
            current.append(char)
    parts.append("".join(current))
    return [part.strip() for part in parts if part.strip()]


def build_builtin_strategy(goals) -> dict:
    if isinstance(goals, (list, tuple)):
        goals = ", ".join(str(goal) for goal in goals if str(goal or "").strip())
    entries, seen = [], set()
    for text in _split_outside_parentheses(goals):
        name, description = text, ""
        match = re.match(r"^(.*?)\s*\((.*)\)\s*$", text)
        if match and match.group(1).strip():
            name, description = match.group(1).strip(), match.group(2).strip()
        name = name[:STRATEGY_MAX_NAME_CHARS]
        if not name or name.casefold() in seen:
            continue
        seen.add(name.casefold())
        entries.append({"id": f"S{len(entries) + 1}", "name": name, "level": STRATEGY_DEFAULT_LEVEL,
                        "description": description, "kpis": [], "key_results": [], "children": []})
    return {"schema_version": STRATEGY_SCHEMA_VERSION, "source": "builtin",
            "levels": [STRATEGY_DEFAULT_LEVEL] if entries else [], "strategy": entries,
            "priorities": [], "sponsors": [], "labels": {}, "warnings": []}


def iter_strategy_entries(tree):
    def _walk(nodes, ancestors):
        for node in nodes or []:
            path = ancestors + [node]
            yield node, path
            yield from _walk(node.get("children"), path)
    yield from _walk((tree or {}).get("strategy"), [])


def strategy_entry_index(tree) -> dict:
    return {node["id"]: path for node, path in iter_strategy_entries(tree)}


def strategy_leaf_ids(tree) -> list:
    return [node["id"] for node, _path in iter_strategy_entries(tree) if not node.get("children")]


def strategy_catalog(tree) -> dict:
    """Entries by path id with their verbatim path and ancestors, and the tree's KPIs and key results
    by id with the entry they sit on."""
    entries, kpis, key_results = {}, {}, {}
    for node, path in iter_strategy_entries(tree):
        entries[node["id"]] = {"id": node["id"], "name": node["name"], "level": node.get("level", ""),
                               "path": " > ".join(item["name"] for item in path),
                               "ancestors": [item["id"] for item in path[:-1]]}
        for kpi in node.get("kpis") or []:
            kpis[kpi["id"]] = dict(kpi, entry_id=node["id"])
        for key_result in node.get("key_results") or []:
            key_results[key_result["id"]] = dict(key_result, entry_id=node["id"])
    return {"entries": entries, "kpis": kpis, "key_results": key_results}


def strategy_prompt_lines(tree) -> str:
    lines = []
    for node, path in iter_strategy_entries(tree):
        indent = "  " * (len(path) - 1)
        line = f"{indent}{node['id']} [{node.get('level', '')}] {node['name']}"
        if node.get("description"):
            line += f": {node['description']}"
        lines.append(line)
        for kpi in node.get("kpis") or []:
            details = ", ".join(part for part in (kpi.get("direction"), kpi.get("unit")) if part)
            lines.append(f"{indent}  {kpi['id']} KPI: {kpi['name']}" + (f" ({details})" if details else "")
                         + (f". {kpi['definition']}" if kpi.get("definition") else ""))
        for key_result in node.get("key_results") or []:
            lines.append(f"{indent}  {key_result['id']} Key result: {key_result['statement']}"
                         + (f" (KPI {key_result['kpi_id']})" if key_result.get("kpi_id") else ""))
    return "\n".join(lines) or "(no strategy entries: every use case is Not aligned)"


def use_case_key(uc) -> str:
    """Stable identity across runs: the normalized title plus the normalized primary table."""
    title = re.sub(r"[^a-z0-9]+", " ", str((uc or {}).get("Name") or "").lower()).strip()
    tables = [t.strip() for t in str((uc or {}).get("Tables Involved") or "").split(",") if t.strip()]
    primary = re.sub(r"\s*\.\s*", ".", tables[0].replace("`", "")).lower() if tables else ""
    return f"{title}|{primary}"[:300]


FEEDBACK_REWORDED_MIN_SUBJECT = 0.6
FEEDBACK_REWORDED_MIN_TITLE = 0.5
_FEEDBACK_TITLE_STOP_WORDS = frozenset({"a", "an", "and", "at", "by", "for", "from", "in", "of", "on", "the", "to", "with"})


def _feedback_stem(token) -> str:
    token = _singular_token(token)
    return token[:-3] if len(token) > 6 and token.endswith("ing") else token


def _feedback_title_tokens(text) -> frozenset:
    return frozenset(_feedback_stem(t) for t in re.split(r"[^a-z0-9]+", str(text or "").lower())
                     if t and t not in _FEEDBACK_TITLE_STOP_WORDS)


def _feedback_jaccard(left, right) -> float:
    return len(left & right) / len(left | right) if left | right else 0.0


def _feedback_subject(text) -> str:
    return str(text or "").lower().split(" with ", 1)[0]


def feedback_rewording_similarity(title, rejected_title) -> tuple:
    subject = _feedback_jaccard(_feedback_title_tokens(_feedback_subject(title)),
                                _feedback_title_tokens(_feedback_subject(rejected_title)))
    whole = _feedback_jaccard(_feedback_title_tokens(title), _feedback_title_tokens(rejected_title))
    return subject, whole


def feedback_is_rewording(title, rejected_title) -> bool:
    subject, whole = feedback_rewording_similarity(title, rejected_title)
    return subject >= FEEDBACK_REWORDED_MIN_SUBJECT and whole >= FEEDBACK_REWORDED_MIN_TITLE


def strategic_context_session_json(tree) -> str:
    if not tree:
        return ""
    payload = {key: value for key, value in tree.items() if key != "warnings"}
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":"))


class DatabricksInspire:
    
    @staticmethod
    def _extract_primary_table(tables_involved: str) -> str:
        """
        Extracts the primary (first) table from the Tables Involved field.
        
        Args:
            tables_involved: Comma-separated string of table names
            
        Returns:
            The first table name, or 'N/A' if empty
        """
        if not tables_involved or not isinstance(tables_involved, str):
            return "N/A"
        
        # Split by comma and get the first table
        tables = [t.strip() for t in tables_involved.split(',') if t.strip()]
        if tables:
            return tables[0]
        return "N/A"

    @staticmethod
    def _get_enriched_field(ctx: dict, field_name: str, fallback: str, min_length: int = 10) -> str:
        raw = ctx.get(field_name, '') if isinstance(ctx, dict) else ''
        if isinstance(raw, list):
            value = ', '.join(str(item) for item in raw if item) if raw else ''
        else:
            value = str(raw).strip() if raw else ''
        if value and len(value) >= min_length:
            return value
        return fallback

    def _build_use_case_step_payload(self, use_case: dict) -> dict:
        uc = use_case
        def _s(key, default=''):
            v = uc.get(key, default)
            return str(v) if v is not None else str(default)
        def _n(key):
            return self._finite_score(uc.get(key))
        assessment = uc.get('elite_assessment')
        return {
            "use_case_id": _s('No'),
            "business_domain": _s('Business Domain'),
            "subdomain": _s('Subdomain'),
            "usecase": self._normalize_usecase(uc.get('usecase', ''), uc.get('Name', '')),
            "name": _s('Name'),
            "generated": _s('generated'),
            "validated": _s('validated'),
            "type": _s('type'),
            "analytics_technique": _s('Analytics Technique'),
            "business_priority_alignment": _s('Business Priority Alignment'),
            "strategic_goals_alignment": _s('Strategic Goals Alignment'),
            "statement": _s('Statement'),
            "solution": _s('Solution'),
            "business_value": _s('Business Value'),
            "beneficiary": _s('Beneficiary'),
            "sponsor": _s('Sponsor'),
            "tables_involved": _s('Tables Involved'),
            "primary_table": _s('Primary Table'),
            "strategic_alignment": _n('Strategic Alignment'),
            "return_on_investment": _n('Return on Investment'),
            "reusability": _n('Reusability'),
            "time_to_value": _n('Time to Value'),
            "data_availability": _n('Data Availability'),
            "data_accessibility": _n('Data Accessibility'),
            "architecture_fitness": _n('Architecture Fitness'),
            "team_skills": _n('Team Skills'),
            "domain_knowledge": _n('Domain Knowledge'),
            "people_allocation": _n('People Allocation'),
            "budget_allocation": _n('Budget Allocation'),
            "time_to_production": _n('Time to Production'),
            "value_score": _n('Value'),
            "feasibility_score": _n('Feasibility'),
            "priority_score": _n('Priority'),
            "quality": _s('Quality'),
            "quality_score": _n('Quality Score'),
            "inspire_score": _n('Inspire Score'),
            "bob_score": _n('bob_score'),
            "quality_mode": canonical_use_cases_quality(uc.get('quality_mode') or getattr(self, 'use_cases_quality', None)),
            "elite_status": uc.get('elite_status') or None,
            "elite_score": _n('elite_score'),
            "elite_assessment": json.loads(self._tracking_json(assessment)) if isinstance(assessment, dict) and assessment else None,
            "coverage_pick_reason": uc.get('coverage_pick_reason') or None,
            "selection_flags": self._selection_flags(uc) or None,
            "justification": _s('Justification'),
            "quality_summary": _s('Quality Summary'),
            "result_table": _s('result_table'),
            "notebook_path": _s('notebook_path'),
            "high_level_design": _s('High Level Design'),
            "genie_instruction": _s('genie_instruction'),
            **self._v130_payload_fields(uc),
        }

    @classmethod
    def _v130_payload_fields(cls, uc: dict) -> dict:
        """The v1.3.0 use case fields under their tracking column names, as JSON values (None when absent)."""
        out = {}
        for column, key, kind in cls._V130_TRACKING_FIELDS:
            value = uc.get(key)
            if value is None or value == "" or value == [] or value == {}:
                out[column] = None
            elif kind in ("list", "lines"):
                items = value if isinstance(value, (list, tuple)) else [value]
                out[column] = [str(item).strip() for item in items if str(item).strip()] or None
            else:
                out[column] = value
        return out

    _SELECTION_FLAG_KEYS = (
        "scoring_status", "scoring_unscored_reason", "bob_status", "bob_unscored_reason",
        "bob_hard_veto_gate", "gate_unverified", "gate_flags", "decision_verb_flag", "clustering_fallback",
        "selection_drop_reason", "coverage_pick_reason", "fallback_reason", "fallback_bar", "fallback_kind",
    )

    @classmethod
    def _selection_flags(cls, uc: dict) -> dict:
        flags = {}
        for key in cls._SELECTION_FLAG_KEYS:
            value = uc.get(key)
            if value in (None, "", "None", False) or (isinstance(value, (list, tuple, set, dict)) and not value):
                continue
            flags[key] = sorted(value) if isinstance(value, set) else value
        return flags

    @staticmethod
    def _finite_score(value):
        return DatabricksInspire._finite_or(value, None)

    @staticmethod
    def _tracking_json(value):
        if value is None:
            return None
        if isinstance(value, str):
            return value.strip() or None
        if isinstance(value, (dict, list, tuple)) and not value:
            return None
        return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)

    def _get_business_context_fallback(self) -> str:
        # Phase M: Grounded fallback only — NEVER invent scale, market position, or
        # firm-specific facts. If enriched context is unavailable, emit the literal
        # UNKNOWN_FROM_METADATA token and a strictly structural description that
        # downstream prompts must treat as a non-anchor (do-not-invent signal).
        ctx = getattr(self, 'merged_business_context', {})
        if isinstance(ctx, dict) and ctx.get('business_context') and len(ctx['business_context'].strip()) >= 80:
            return ctx['business_context']
        industries_str = ", ".join(self.industries) if hasattr(self, 'industries') and self.industries else ""
        tables_hint = ""
        if hasattr(self, 'available_tables') and self.available_tables:
            sample_tables = list(self.available_tables)[:5]
            tables_hint = f" covering tables {', '.join(sample_tables)}"
        base = f"Business name: {getattr(self, 'business_name', 'UNKNOWN_FROM_METADATA')}."
        if industries_str:
            base += f" Industry: {industries_str}."
        else:
            base += " Industry: UNKNOWN_FROM_METADATA."
        base += f" Data scope{tables_hint}."
        base += (
            " UNKNOWN_FROM_METADATA: enriched business context was unavailable from the business-context LLM. "
            "Downstream consumers MUST NOT invent firm-specific figures (revenue, location counts, employee counts, "
            "customer counts, Fortune-rank, ISO certifications, market share) and MUST treat UNKNOWN_FROM_METADATA "
            "as a do-not-invent signal. Ground every claim in the supplied schema."
        )
        return base

    @staticmethod
    def _story_clean_text(value: str) -> str:
        import re
        text = str(value or "")
        text = text.replace("_PROMPT", "")
        text = text.replace("_", " ")
        text = re.sub(r"\s+", " ", text).strip()
        text = text.replace("LLM", "AI")
        text = text.replace("SQL", "implementation")
        return text

    def _story_status_line(self, status: str, message: str, fallback: str) -> str:
        context = self._story_clean_text(fallback or "")
        if status == "started":
            return f"In progress: {context.lower()}" if context else "In progress"
        if status == "ended_success":
            return f"Completed: {context.lower()}" if context else "Completed"
        if status == "ended_error":
            return f"Needs attention: {context.lower()}" if context else "Needs attention"
        if status == "ended_warning":
            return f"Completed with caution: {context.lower()}" if context else "Completed with caution"
        return context if context else "Progress update"

    @staticmethod
    def _is_best_of_best_worker(prompt_name: str, step_name) -> bool:
        return prompt_name == "ULTRA_HIGH_ELITE_JUDGE_PROMPT" and str(step_name or "").startswith(BEST_OF_BEST_WORKER_PREFIX)

    def _story_prompt_labels(self, prompt_name: str, step_name: str, status: str, message: str):
        stage_name, step_title = (BEST_OF_BEST_STORY_TITLE if self._is_best_of_best_worker(prompt_name, step_name)
                                  else PROMPT_STORY_TITLES.get(
                                      prompt_name,
                                      ("Inspire Progress", f"Processing {self._story_clean_text(prompt_name)}")))
        sub_step_name = self._story_status_line(status, message, step_name)
        return stage_name, step_title, sub_step_name

    def _story_prompt_message(self, prompt_name: str, status: str, step_title: str, sub_step_name: str,
                              step_name: str = "") -> str:
        if status == "started":
            if self._is_best_of_best_worker(prompt_name, step_name):
                return BEST_OF_BEST_STORY_STARTED
            return PROMPT_STORY_STARTED_MESSAGES.get(
                prompt_name,
                f"Inspire has started {self._story_clean_text(step_title).lower()}"
            )
        if status == "ended_success":
            return f"Inspire has completed {self._story_clean_text(step_title).lower()}"
        if status == "ended_warning":
            return f"Inspire has completed {self._story_clean_text(step_title).lower()} with caution"
        if status == "ended_error":
            return f"Inspire needs attention while {self._story_clean_text(step_title).lower()}"
        return sub_step_name

    def _story_pipeline_labels(self, stage_name: str, step_name: str, sub_step_name: str, status: str, message: str):
        stage_map = {
            "Pipeline": "Inspire Journey",
            "Initialization": "Setup",
            "Business Context": "Business Understanding",
            "Document Generation": "Executive Readout",
            "Excel Generation": "Executive Readout",
            "Artifact Writing": "Executive Readout",
            "Summary": "Executive Readout",
            "Translation": "Localization",
            "Use Cases Scoring": "Use Case Prioritization",
            "Business Domains Clustering": "Domain Mapping",
            "Genie Code Instructions": "Implementation Planning",
        }
        base_stage = stage_map.get(stage_name, self._story_clean_text(stage_name or "Inspire Progress"))
        if stage_name == "Pipeline" and step_name == "Execution" and status == "started":
            base_step = "Inspire pipeline started"
        elif stage_name == "Pipeline" and step_name == "Execution" and status == "ended_success":
            base_step = "Inspire pipeline completed"
        elif stage_name == "Pipeline" and step_name == "Execution" and status == "ended_error":
            base_step = "Inspire pipeline needs attention"
        else:
            clean_step = self._story_clean_text(step_name)
            base_step = f"Inspire is working on {clean_step.lower()}" if clean_step else "Inspire is processing your request"
        live_text = self._story_status_line(status, message, sub_step_name)
        return base_stage, base_step, live_text

    @staticmethod
    def _story_with_detail(story_message: str, status: str, detail) -> str:
        detail = str(detail or "").strip()
        return f"{story_message}: {detail}" if detail and status in ("ended_warning", "ended_error") else story_message

    def _story_pipeline_message(self, status: str, step_name: str, sub_step_name: str) -> str:
        clean_step = self._story_clean_text(step_name)
        if status == "started":
            return f"Inspire has started {clean_step.lower()}" if clean_step else "Inspire has started processing"
        if status == "ended_success":
            return f"Inspire has completed {clean_step.lower()}" if clean_step else "Inspire has completed processing"
        if status == "ended_warning":
            return f"Inspire has completed {clean_step.lower()} with caution" if clean_step else "Inspire has completed with caution"
        if status == "ended_error":
            return f"Inspire needs attention while {clean_step.lower()}" if clean_step else "Inspire needs attention"
        return sub_step_name

    def _emit_prompt_status(self, prompt_name: str, step_name: str, status: str, message: str = "", result_json=None, step_id: int = None):
        if not getattr(self, "atomic_writer", None):
            return None
        tracking_meta = PROMPT_TRACKING_CONFIG.get(prompt_name, None)
        if tracking_meta:
            total_prompt_budget = max(1, int(round(float(tracking_meta["progress_increment"]))))
        else:
            total_prompt_budget = 1

        payload = result_json if isinstance(result_json, dict) else {}
        entity_id = str(payload.get("entity_id") or f"{prompt_name}:{step_name}")

        progress_increment = 0
        if status == "started":
            with self._prompt_progress_lock:
                self._prompt_entities_seen.setdefault(prompt_name, set()).add(entity_id)
        elif status == "ended_success":
            with self._prompt_progress_lock:
                self._prompt_entities_seen.setdefault(prompt_name, set()).add(entity_id)
                entities_seen = self._prompt_entities_seen.get(prompt_name, set())
                emitted_so_far = int(self._prompt_progress_emitted.get(prompt_name, 0))
                remaining_budget = max(0, total_prompt_budget - emitted_so_far)
                if remaining_budget > 0:
                    if prompt_name in MULTI_ENTITY_PROGRESS_PROMPTS:
                        progress_increment = 1
                    else:
                        denominator = max(1, len(entities_seen))
                        progress_increment = max(1, remaining_budget // denominator)
                        progress_increment = min(progress_increment, remaining_budget)
                self._prompt_progress_emitted[prompt_name] = emitted_so_far + progress_increment

        story_stage_name, story_step_name, story_sub_step_name = self._story_prompt_labels(
            prompt_name=prompt_name,
            step_name=step_name,
            status=status,
            message=message
        )
        story_message = self._story_prompt_message(
            prompt_name=prompt_name,
            status=status,
            step_title=story_step_name,
            sub_step_name=story_sub_step_name,
            step_name=step_name
        )

        return self.atomic_writer.emit_step(
            stage_name=story_stage_name,
            step_name=story_step_name,
            sub_step_name=story_sub_step_name,
            progress_increment=progress_increment,
            message=story_message,
            status=status,
            result_json=result_json if result_json is not None else {},
            step_id=step_id
        )

    def _emit_pipeline_status(self, stage_name: str, step_name: str, sub_step_name: str, message: str, status: str,
                              progress_increment: float = 0.0, result_json=None, step_id: int = None):
        if not getattr(self, "atomic_writer", None):
            return None
        story_stage_name, story_step_name, story_sub_step_name = self._story_pipeline_labels(
            stage_name=stage_name,
            step_name=step_name,
            sub_step_name=sub_step_name,
            status=status,
            message=message
        )
        story_message = self._story_pipeline_message(
            status=status,
            step_name=step_name,
            sub_step_name=story_sub_step_name
        )
        emitted_step_id = self.atomic_writer.emit_step(
            stage_name=story_stage_name,
            step_name=story_step_name,
            sub_step_name=story_sub_step_name,
            progress_increment=progress_increment,
            message=self._story_with_detail(story_message, status, message),
            status=status,
            result_json=result_json if result_json is not None else {},
            step_id=step_id
        )
        open_steps = getattr(self, "_open_pipeline_step_ids", None)
        if open_steps is not None:
            step_key = f"{stage_name}|{step_name}|{sub_step_name}"
            if status == "started":
                open_steps[step_key] = (emitted_step_id, stage_name, step_name, sub_step_name)
            elif status.startswith("ended_"):
                open_steps.pop(step_key, None)
        return emitted_step_id

    def _attach_ailogs(self):
        """Every LLM call attempt of this run goes to <inspire_database>._ailogs, or without a database to
        a local JSONL that the live log streamer copies next to the live log."""
        writer = getattr(self, "atomic_writer", None)
        self._ailogs_jsonl_path = ""
        if writer is not None and getattr(writer, "enabled", False):
            AILOG_RECORDER.attach(self.session_id, writer.emit_ailog)
            self.logger.info(f"📒 AI call log: every LLM call attempt is written to {writer.ailogs_table}")
            return
        path = os.path.join(getattr(self, "local_log_output_dir", "") or tempfile.gettempdir(), "ailogs.jsonl")
        lock = threading.Lock()

        def _jsonl_sink(row):
            line = json.dumps(dict(row, server_ingested_at=_ailog_now()), ensure_ascii=True, default=str)
            with lock:
                with open(path, "a", encoding="utf-8") as handle:
                    handle.write(line + "\n")
        AILOG_RECORDER.attach(self.session_id, _jsonl_sink)
        self._ailogs_jsonl_path = path
        self.logger.info(f"📒 AI call log: no inspire_database, every LLM call attempt is written to {path}")

    def _ailogs_live_copy_source(self) -> str:
        writer = getattr(self, "atomic_writer", None)
        denied = writer.denied_spool_path("ailogs") if hasattr(writer, "denied_spool_path") else ""
        return getattr(self, "_ailogs_jsonl_path", "") or denied

    def _close_ailogs(self) -> dict:
        if getattr(self, "ailogs_check", None) is not None:
            return self.ailogs_check
        expected = {prompt: stats.get("calls", 0) for prompt, stats in AIAgent._step_stats.items()}
        probed = list((LLM_ENDPOINT_HEALTH.snapshot() or {}).get("probed") or [])
        report = AILOG_RECORDER.self_check(expected, bound_endpoints=probed)
        writer = getattr(self, "atomic_writer", None)
        landing = ""
        if getattr(writer, "enabled", False) and hasattr(writer, "denied_spool_path"):
            denied = writer.denied_spool_path("ailogs")
            quarantined = int((getattr(writer, "quarantined_rows", None) or {}).get("ailogs", 0))
            report["checks"]["rows_reach_the_table"] = not denied and not quarantined
            report["passed"] = sum(1 for ok in report["checks"].values() if ok)
            report["total"] = len(report["checks"])
            if denied:
                copy = getattr(self, "_ailogs_live_copy_dst", "")
                landing = (f"; {writer.ailogs_table} refused the rows, which are kept in {denied}"
                           + (f" and copied to {copy}" if copy else ""))
            elif quarantined:
                landing = f"; {quarantined} row(s) quarantined"
        failed = [name for name, ok in report["checks"].items() if not ok]
        (self.logger.info if not failed else self.logger.warning)(
            f"[AILOGS CHECK] {report['passed']}/{report['total']} checks passed; {report['rows']} row(s) for "
            f"{report['requests']} request(s), {report['non_success']} not successful "
            f"({report['probe_non_success']} from endpoint probes), {report['dropped']} dropped" + landing
            + (f"; failed: {failed}" if failed else ""))
        self.ailogs_check = report
        AILOG_RECORDER.detach()
        return report

    def _close_ailogs_safely(self):
        try:
            self._close_ailogs()
        except Exception as _ailog_err:
            self.logger.warning(f"[AILOGS CHECK] could not run: {get_clean_error_message(_ailog_err)}")

    def finalize_atomic_writer(self, success: bool = True, error_message: str = ""):
        self._close_ailogs_safely()
        if not getattr(self, "atomic_writer", None):
            return
        final_results_json = self._merge_run_outcome_into_results()
        pipeline_start_step_id = getattr(self, "_pipeline_start_step_id", None)
        if not success:
            open_steps = getattr(self, "_open_pipeline_step_ids", {})
            for step_key, (open_step_id, stage, step, sub) in list(open_steps.items()):
                if open_step_id == pipeline_start_step_id:
                    continue
                try:
                    self._emit_pipeline_status(
                        stage_name=stage,
                        step_name=step,
                        sub_step_name=sub,
                        message=error_message or "Pipeline failed: closing open step",
                        status="ended_error",
                        progress_increment=0.0,
                        result_json={"error": error_message[:300] if error_message else "Pipeline terminated"},
                        step_id=open_step_id
                    )
                except Exception:
                    pass
            self._emit_pipeline_status(
                stage_name="Pipeline",
                step_name="Execution",
                sub_step_name="Finalize",
                message=error_message or "Pipeline failed",
                status="ended_error",
                progress_increment=1.0,
                result_json={"session_id": self.session_id},
                step_id=pipeline_start_step_id
            )
            self.atomic_writer.flush_pending(force=True, finalize=True, final_results_json=final_results_json)
        else:
            warnings = self._run_outcome_warnings()
            self.atomic_writer.finalize_pipeline(
                message=f"Completed with warnings: {'; '.join(warnings)}" if warnings else "Completed",
                final_results_json=final_results_json,
                pipeline_start_step_id=pipeline_start_step_id,
                status="ended_warning" if warnings else "ended_success"
            )

    def __init__(self, **kwargs):
        self.spark = spark
        self.dbutils = dbutils
        # Initialize workspace client with timeout configuration to prevent indefinite hangs
        from databricks.sdk.config import Config
        config = Config()
        config.retry_timeout_seconds = 300  # 5 minute timeout for individual API calls
        self.workspace = WorkspaceClient(config=config)  # Changed from w_client to workspace
        self.w_client = self.workspace  # Keep w_client as alias for backward compatibility
        self._notebook_object_ids = {}

        # --- Store Widget Values ---
        self.business_name = kwargs.get("business", "gen")
        
        # --- Business Priorities (user-provided, multi-select) ---
        business_priorities_raw = kwargs.get("business_priorities", "")
        self.user_business_priorities = [goal.strip() for goal in business_priorities_raw.split(',') if goal.strip()]
        
        # --- Generation Instructions (user-provided, free-text) ---
        generation_instructions_raw = kwargs.get("generation_instructions", "")
        self.user_generation_instructions = generation_instructions_raw.strip() if generation_instructions_raw else ""
        _generation_instructions_original_length = len(self.user_generation_instructions)
        if _generation_instructions_original_length > 2000:
            self.user_generation_instructions = self.user_generation_instructions[:2000]
        self.user_strategic_goals = []
        self.parsed_generation_instructions = {}
        self._user_provided_session_id = bool(kwargs.get("_user_provided_session_id", False))
        
        # --- Business Domains (user-provided, comma-separated) ---
        business_domains_raw = kwargs.get("business_domains", "")
        self.user_business_domains = [domain.strip() for domain in business_domains_raw.split(',') if domain.strip()]
        
        self.additional_context = ""
        
        self.catalogs_str = kwargs.get("catalogs", "")
        self.schemas_str = kwargs.get("schemas", "")
        self.tables_str = kwargs.get("tables", "")
        self.generate_choices = [opt.strip() for opt in kwargs.get("generate", "use cases").split(',')]
        _generate_choice_aliases = {"PDF": "PDF Catalog", "PPTX": "Presentation", "MD": "Markdown"}
        for _short, _long in _generate_choice_aliases.items():
            if _short in self.generate_choices and _long not in self.generate_choices:
                self.generate_choices.append(_long)
        
        self.generation_path = kwargs.get("generation_path", "./")
        
        self.output_languages = [lang.strip() for lang in kwargs.get("output_language", "English").split(',') if lang.strip()]
        if not self.output_languages:
            self.output_languages = ["English"]
            
        _runtime = TECHNICAL_CONTEXT["runtime"]
        
        raw_max_parallelism = kwargs.get("max_parallelism", None)
        if raw_max_parallelism is None or str(raw_max_parallelism).strip() == "":
            self.max_parallelism = _runtime["default_max_parallelism"]
            self.auto_parallelism = True
        else:
            self.max_parallelism = min(int(raw_max_parallelism), 100)
            self.auto_parallelism = False
        self.scan_parallelism = _runtime["scan_parallelism"]
        self.cluster_memory_gb = int(kwargs.get("cluster_memory_gb", _runtime["cluster_memory_gb"]) or _runtime["cluster_memory_gb"])
        self.cluster_worker_count = int(kwargs.get("cluster_worker_count", _runtime["cluster_worker_count"]) or _runtime["cluster_worker_count"])
        
        self.technical_exclusion_strategy = resolve_technical_exclusion_strategy(kwargs.get("technical_exclusion_strategy"))
        
        _raw_operation = str(kwargs.get("operation") or "Discover Use Cases").strip()
        if _raw_operation not in INSPIRE_OPERATIONS:
            raise ValueError(f"Operation {_raw_operation!r} is not supported. Valid options: "
                             + ", ".join(f"'{op}'" for op in INSPIRE_OPERATIONS))
        self.operation = _raw_operation
        self.operation_mode = self.operation
        
        self.table_election_mode = str(kwargs.get("table_election_mode") or "Let Inspire Decides").strip()
        if self.table_election_mode not in ("Let Inspire Decides", "All Tables", "Transactional Only"):
            raise ValueError(f"Table Election {self.table_election_mode!r} is not supported. Valid options: "
                             f"'Let Inspire Decides', 'All Tables', 'Transactional Only'")
        
        self.use_cases_quality = canonical_use_cases_quality(kwargs.get("use_cases_quality"))
        self.discovery_type = canonical_discovery_type(kwargs.get("discovery_type"))
        self.customer_strategy = parse_strategic_context(
            kwargs.get("strategic_context"), read_file=lambda path: read_user_text_file(path, self.w_client))
        self.strategy_tree = self.customer_strategy
        self._customer_priorities_override = bool(self.customer_strategy and self.customer_strategy.get("priorities"))
        if self._customer_priorities_override:
            self.user_business_priorities = list(self.customer_strategy["priorities"])

        self.generate_genie_instructions = (self.operation == "Generate Use Cases")
        # v0.8.9 DRY: operation banner is emitted ONCE during widget validation
        # (~line 31905) — removed duplicate __init__ emit to keep the boot log clean.
        
        _quality_filter_map = {
            "Good": {'Medium', 'High', 'Very High', 'Ultra High'},
            "High": {'High', 'Very High', 'Ultra High'},
            "Ultra High": {'High', 'Very High', 'Ultra High'},
        }
        self.quality_filter_acceptable = set(_quality_filter_map[self.use_cases_quality])

        self.llm_timeout_seconds = _runtime["llm_timeout_seconds"]
        self.max_retry_attempts = max(0, _runtime["max_retry_attempts"])
        
        # === NEW: Initialize merged business context ===
        self.merged_business_context = {}
        
        # === NEW: JSON file path for docs-only mode ===
        self.json_file_path = kwargs.get("json_file_path", None)
        self.final_inspire_results_json = {}
        self._pipeline_start_step_id = None
        
        # === Inspire Database (user-provided catalog.database for all result tables) ===
        self.inspire_database = kwargs.get("inspire_database", "")
        if self.inspire_database:
            self.inspire_database = self.inspire_database.strip()
        
        # === Auto-Genie scope ("all" or non-negative int as string) ===
        self.generate_genie_code_for = str(kwargs.get("generate_genie_code_for", GENIE_CODE_DEFAULT_COUNT) or GENIE_CODE_DEFAULT_COUNT).strip()
        self.best_of_best_widget_count = parse_best_of_best_count(kwargs.get("best_of_best_count"))
        self.best_of_best_outcome = None

        # === Session ID (unique per inspire session for tracking) ===
        _user_session_id = kwargs.get("session_id", "")
        if _user_session_id:
            try:
                self.session_id = int(_user_session_id)
            except ValueError:
                import hashlib as _hashlib_mod
                _MAX_SIGNED_INT64 = 9223372036854775807
                _hash_int = int(_hashlib_mod.sha256(_user_session_id.encode()).hexdigest(), 16)
                self.session_id = _hash_int & _MAX_SIGNED_INT64
        else:
            import uuid as _uuid_mod
            _MAX_SIGNED_INT64 = 9223372036854775807
            self.session_id = (_uuid_mod.uuid4().int >> 64) & _MAX_SIGNED_INT64

        self._prompt_progress_lock = threading.Lock()
        self._prompt_entities_seen = {}
        self._prompt_progress_emitted = {}

        self.atomic_writer = AtomicWriter(
            spark=self.spark,
            logger=logging.getLogger("AtomicWriterBootstrap"),
            inspire_database=kwargs.get("inspire_database", ""),
            session_id=self.session_id,
            widget_values=dict(kwargs, use_cases_quality=self.use_cases_quality,
                               technical_exclusion_strategy=self.technical_exclusion_strategy,
                               discovery_type=self.discovery_type,
                               strategic_context_json=strategic_context_session_json(self.customer_strategy))
        )
        
        # === Use case tracking table infrastructure ===
        import threading as _threading_mod
        self._tracking_lock = _threading_mod.Lock()
        self._tracking_table_created = False
        self._tracking_table_name = quote_qualified_name(f"{self.inspire_database}.__inspire_usecases") if self.inspire_database else ""

        self._cached_schema_markdown = None

        # --- Setup Logging & Output ---
        self.sanitized_customer_name = self._sanitize_name(self.business_name)
        
        # Keep log in /tmp during execution (always writable), will copy to output dir at end
        self.local_log_output_dir = f"/tmp/{self.sanitized_customer_name}_{self.session_id}"

        resolved_generation_path = self.generation_path
        if resolved_generation_path.startswith("./"):
            try:
                logical_notebook_path = self.dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
                resolved_generation_path = resolve_generation_path(resolved_generation_path, logical_notebook_path)
                log_print(f"Resolved relative generation path to: {resolved_generation_path}")
            except Exception as _e:
                raise ValueError(
                    f"generation_path '{self.generation_path}' is relative and the notebook "
                    f"context could not be resolved to anchor it (root cause: {_e}). "
                    f"Please set generation_path to an absolute workspace path "
                    f"(e.g. '/Workspace/Users/<you>/inspire_output' or '/Volumes/<cat>/<sch>/<vol>/inspire_output')."
                )
        else:
            resolved_generation_path = resolve_generation_path(resolved_generation_path)

        # Folder name = sanitized business_name (the user-supplied widget value).
        # Per CLAUDE.md §3c (User Vibes Are Supreme Authority), business_name is
        # the primary user-supplied identity for the run; the generation path
        # root MUST reflect it. Falls back to a sanitized form of the UC metadata
        # string only when business_name is empty (validation should have caught
        # this earlier, but keep a safe net).
        _folder_name = self.sanitized_customer_name
        if not _folder_name:
            _schema_src = (self.schemas_str or self.catalogs_str or "").strip()
            if "," in _schema_src:
                _schema_src = _schema_src.split(",")[0].strip()
            if "." in _schema_src:
                _schema_src = _schema_src.split(".")[-1].strip()
            _folder_name = re.sub(r'\s+', '_', _schema_src)
            _folder_name = re.sub(r'[^A-Za-z0-9._\-]', '_', _folder_name).strip("._-")
        self.base_output_dir = os.path.join(resolved_generation_path, _folder_name)
        # No session subfolder — each Discover run wipes and rewrites this schema's
        # folder. session_output_dir remains an alias so downstream code that
        # reads it still works; the existing Discover-mode wipe below deletes it.
        self.session_output_dir = self.base_output_dir
        self.notebook_output_dir = os.path.join(self.base_output_dir, "notebooks")
        self.docs_output_dir = os.path.join(self.base_output_dir, "docs")

        self.local_log_output_dir = setup_logging(self.local_log_output_dir)  # B17: capture fallback dir if primary /tmp write was blocked
        self.logger = logging.getLogger(self.__class__.__name__)
        _root_logger = logging.getLogger()
        if not any(isinstance(_f, DuplicateMessageFilter) for _f in _root_logger.filters):
            _root_logger.addFilter(DuplicateMessageFilter(window_seconds=1.5))
        if not any(isinstance(_f, DuplicateMessageFilter) for _f in self.logger.filters):
            self.logger.addFilter(DuplicateMessageFilter(window_seconds=1.5))
        if _generation_instructions_original_length > 2000:
            self.logger.warning(f"Generation Instructions exceeds 2000 char limit ({_generation_instructions_original_length} chars) -- truncated to 2000")
        
        # Initialize memory-efficient storage manager
        self.storage_manager = IntermediateStorageManager(base_path="/tmp", logger=self.logger)
        
        # Initialize honesty tracking for reporting what happened during processing
        self.processing_honesty = {
            'total_tables_discovered': 0,
            'total_tables_processed': 0,
            'total_batches_created': 0,
            'total_batch_splits': 0,
            'tables_with_columns_dropped': [],
            'batch_split_history': [],
            'tables_completely_processed': [],
            'tables_partially_processed': [],
            'tables_skipped': [],
            'tables_metadata_unavailable': []
        }
        self.validation_timeouts_discarded = []
        
        max_parallelism_display = self.max_parallelism if not self.auto_parallelism else "auto"
        self.logger.info(f"DatabricksInspire initialized for business: {self.business_name}. Target Language(s): {self.output_languages}. Base Output Dir: {self.base_output_dir}. Notebooks Dir: {self.notebook_output_dir}. Docs Dir: {self.docs_output_dir}. Scan parallelism: {self.scan_parallelism}. Max parallelism: {max_parallelism_display}")
        
        try:
            if self.operation == "Discover Use Cases":
                self.logger.info(f"Discover mode: cleaning up this session's output directory only (session_id={self.session_id})...")
                try:
                    self.w_client.workspace.delete(self.session_output_dir, recursive=True)
                    self.logger.info(f"Deleted existing session directory: {self.session_output_dir}")
                except Exception as delete_error:
                    self.logger.debug(f"No existing session directory or deletion failed: {delete_error}")
            else:
                self.logger.info(f"Generate mode: preserving existing session directory {self.session_output_dir}.")
            
            self._ensure_output_directories()
        except Exception as e:
            self.logger.error(f"Failed to create workspace directories: {get_clean_error_message(e)}")
        
        if 'PROMPT_TEMPLATES' not in globals():
            self.logger.critical("CRITICAL ERROR: 'PROMPT_TEMPLATES' dictionary is not defined. Please run the cell defining it.")
            raise NameError("PROMPT_TEMPLATES not found. Please run the cell that defines the prompt dictionary.")

        try:
            _apply_auto_detected_endpoints(self.logger)
        except Exception as _endpoint_err:
            self.logger.warning(f"LLM endpoint auto-detection call failed non-fatally: {get_clean_error_message(_endpoint_err)}")

        self.ai_agent = AIAgent(
            spark=self.spark,
            logger=self.logger,
            worker_llm_config=AI_MODEL_NAME,
            judge_llm_config=AI_MODEL_NAME,
            prompt_templates=PROMPT_TEMPLATES,
            default_timeout_seconds=self.llm_timeout_seconds,
            max_retry_attempts=self.max_retry_attempts,
            status_emitter=self._emit_prompt_status
        )
        self.ai_agent.enable_llm_cache(self.session_id)
        
        self.translation_service = TranslationService(ai_agent=self.ai_agent, logger=self.logger)
        
        # === NEW: Column Registry (Bitmap Technique) ===
        self.column_id_map = {}   # FQN -> ID
        self.id_column_map = {}   # ID -> {fqn, description}
        self.next_column_id = 1
        # === NEW: Table Registry (Bitmap Technique) ===
        self.table_id_map = {}    # Table FQN -> ID
        self.id_table_map = {}    # ID -> table FQN
        self.next_table_id = 1
        self.registry_lock = threading.Lock()
        
        self.data_loader = None
        # Skip data loader if we're in docs-only mode (JSON file path provided)
        # Note: "use cases" is always generated (removed from widget options)
        if not self.json_file_path:
            if ("PDF Catalog" in self.generate_choices or 
                "Use Cases Catalog PDF" in self.generate_choices or
                "Presentation" in self.generate_choices or
                "SQL Regeneration" not in self.generate_choices):
                
                if not (self.catalogs_str or self.schemas_str or self.tables_str):
                    self.logger.error("For 'use cases', 'PDF', or 'Presentation' you must provide at least one catalog, schema, or table.")
                else:
                    # === MEMORY-OPTIMIZED DATA LOADER ===
                    # Enable two-pass mode for intelligent batching based on table sizes
                    # Enable column sampling for tables with >250 columns
                    # Use streaming for schemas with >10K tables
                    self.data_loader = DataLoader(
                        catalogs=self.catalogs_str,
                        schemas=self.schemas_str,
                        tables=self.tables_str,
                        logger=self.logger,
                        enable_two_pass=True,           # Enable intelligent batching
                        enable_column_sampling=True,    # Sample wide tables
                        streaming_batch_size=1000,      # Stream in chunks of 1000 tables
                        max_parallelism=self.scan_parallelism,
                        schema_timeout_seconds=900      # 15 min timeout per schema
                    )
        
    def _ensure_inspire_database_exists(self):
        """Create the inspire database if it does not exist. Called once at the start of run()."""
        if not self.inspire_database:
            return
        try:
            self.spark.sql(f"CREATE DATABASE IF NOT EXISTS {quote_qualified_name(self.inspire_database)}")
            self.logger.info(f"✅ Ensured inspire database exists: {self.inspire_database}")
        except Exception as e:
            self.logger.error(f"Failed to create inspire database {self.inspire_database}: {get_clean_error_message(e)}")
            raise

    def _create_tracking_table(self):
        """Create the __inspire_usecases tracking table in the inspire database.
        Schema mirrors Excel catalog columns exactly (same order) plus:
          - session_id (first column)
          - updated_at (second-to-last)
          - notebook_path (stores workspace path of Genie Code notebook)
          - high_level_design (stores the high level design text)
          - genie_instruction (stores the generated Genie Code instruction)
          - generate_genie_code_instruction (Yes/No flag for on-demand Genie regeneration)
          - directly_involved_schema / foreign_key_relationships / available_tables_listing
            / technique_implementation_guidance / code_structure_mandate (per-UC Genie prompt
            inputs cached at Discover time so Generate mode can rebuild the prompt without re-computing)
          - quality (canonical Quality label), quality_mode (canonical run mode), elite_status,
            elite_score, elite_assessment (JSON) and selection_flags (JSON of the selection flags)
        Uses Delta format for MERGE support. Thread-safe via _tracking_lock.
        """
        if not self._tracking_table_name or self._tracking_table_created:
            return
        tracking_table = quote_qualified_name(self._tracking_table_name)
        with self._tracking_lock:
            if self._tracking_table_created:
                return
            try:
                create_sql = f"""
                CREATE TABLE IF NOT EXISTS {tracking_table} (
                    session_id BIGINT,
                    id STRING,
                    business_domain STRING,
                    subdomain STRING,
                    use_case STRING,
                    description STRING,
                    inspire_score DOUBLE,
                    generated STRING,
                    type STRING,
                    analytics_technique STRING,
                    business_priority_alignment STRING,
                    strategic_goals_alignment STRING,
                    statement STRING,
                    solution STRING,
                    business_value STRING,
                    beneficiary STRING,
                    sponsor STRING,
                    tables_involved STRING,
                    primary_table STRING,
                    strategic_alignment DOUBLE,
                    roi DOUBLE,
                    reusability DOUBLE,
                    time_to_value DOUBLE,
                    data_availability DOUBLE,
                    data_accessibility DOUBLE,
                    architecture_fitness DOUBLE,
                    team_skills DOUBLE,
                    domain_knowledge DOUBLE,
                    people_allocation DOUBLE,
                    budget_allocation DOUBLE,
                    time_to_production DOUBLE,
                    value_score DOUBLE,
                    feasibility_score DOUBLE,
                    priority_score DOUBLE,
                    quality_score DOUBLE,
                    priority_reasons STRING,
                    quality_reasons STRING,
                    result_database STRING,
                    updated_at TIMESTAMP,
                    notebook_path STRING,
                    high_level_design STRING,
                    genie_instruction STRING,
                    generate_genie_code_instruction STRING,
                    has_genie_code STRING,
                    idea_theme STRING,
                    bob_score DOUBLE,
                    bob_tier1_score DOUBLE,
                    bob_tier2_score DOUBLE,
                    bob_tier3_score DOUBLE,
                    directly_involved_schema STRING,
                    foreign_key_relationships STRING,
                    available_tables_listing STRING,
                    technique_implementation_guidance STRING,
                    code_structure_mandate STRING,
                    quality STRING,
                    quality_mode STRING,
                    elite_status STRING,
                    elite_score DOUBLE,
                    elite_assessment STRING,
                    selection_flags STRING,
                    {', '.join(f'{column} STRING' for column, _key, _kind in self._V130_TRACKING_FIELDS)}
                ) USING DELTA
                """
                self.spark.sql(create_sql)
            except Exception as e:
                self.logger.error(f"Failed to create tracking table {self._tracking_table_name}: {get_clean_error_message(e)}")
                self._record_failed_step("Tracking Table Setup", "", e)
                return
            try:
                self._migrate_tracking_table_schema()
            except Exception as e:
                self.logger.error(f"Tracking table {self._tracking_table_name} could not be fully migrated; rows are written "
                                  f"without the missing columns: {get_clean_error_message(e)}")
                self._record_failed_step("Tracking Table Migration", "", e)
            self._tracking_table_created = True
            self.logger.info(f"✅ Tracking table created/verified: {self._tracking_table_name}")

    _TRACKING_LEGACY_RENAMES = (("generated_sql", "notebook_path"), ("sql_generated", "generated"))

    _TRACKING_MIGRATION_COLUMNS = (
        ("notebook_path", "STRING"), ("high_level_design", "STRING"), ("genie_instruction", "STRING"),
        ("generated", "STRING"), ("inspire_score", "DOUBLE"), ("quality_score", "DOUBLE"),
        ("generate_genie_code_instruction", "STRING"), ("has_genie_code", "STRING"), ("idea_theme", "STRING"),
        ("bob_score", "DOUBLE"), ("bob_tier1_score", "DOUBLE"), ("bob_tier2_score", "DOUBLE"),
        ("bob_tier3_score", "DOUBLE"), ("directly_involved_schema", "STRING"),
        ("foreign_key_relationships", "STRING"), ("available_tables_listing", "STRING"),
        ("technique_implementation_guidance", "STRING"), ("code_structure_mandate", "STRING"),
        ("quality", "STRING"), ("quality_mode", "STRING"), ("elite_status", "STRING"),
        ("elite_score", "DOUBLE"), ("elite_assessment", "STRING"), ("selection_flags", "STRING"),
    )

    _V130_TRACKING_FIELDS = (
        ("strategy_alignment", "Strategy Alignment", "json"),
        ("strategy_path", "Strategy Path", "text"),
        ("kpis", "KPIs", "json"),
        ("expected_key_results", "Expected Key Results", "json"),
        ("measures_of_success", "Measures of Success", "lines"),
        ("financial_benefits", "Financial Benefits", "lines"),
        ("non_financial_benefits", "Non-Financial Benefits", "lines"),
        ("data_evidence", "Data Evidence", "json"),
        ("foundation_ids", "Foundations", "list"),
        ("use_case_key", "use_case_key", "text"),
        ("best_of_best", "best_of_best", "text"),
        ("best_of_best_rank", "best_of_best_rank", "text"),
        ("best_of_best_score", "best_of_best_score", "text"),
        ("best_of_best_reason", "best_of_best_reason", "text"),
    )

    _TRACKING_MIGRATION_COLUMNS = _TRACKING_MIGRATION_COLUMNS + tuple(
        (column, "STRING") for column, _key, _kind in _V130_TRACKING_FIELDS)

    @staticmethod
    def _v130_tracking_value(uc: dict, key: str, kind: str):
        value = uc.get(key)
        if value is None or value == "" or value == [] or value == {}:
            return None
        if kind == "json":
            return DatabricksInspire._tracking_json(value)
        if kind in ("list", "lines"):
            items = value if isinstance(value, (list, tuple)) else [value]
            separator = ", " if kind == "list" else "\n"
            text = separator.join(str(item).strip() for item in items if str(item).strip())
            return text or None
        text = str(value).strip()
        return text or None

    def _migrate_tracking_table_schema(self) -> list:
        """Bring an existing __inspire_usecases table to the current schema: legacy columns are
        renamed when the new name is absent (a failed rename falls back to adding the new column
        and keeps the legacy one), then every missing column is added. Columns are read with
        DESCRIBE first, so only real changes are attempted; any ALTER that fails raises
        RuntimeError naming each failed change. Returns the columns added."""
        if not self._tracking_table_name:
            return []
        tracking_table = quote_qualified_name(self._tracking_table_name)
        self._tracking_table_columns = None
        try:
            existing = AtomicWriter.described_column_names(self.spark.sql(f"DESCRIBE TABLE {tracking_table}").collect())
        except Exception as e:
            raise RuntimeError(f"Tracking table {self._tracking_table_name} schema could not be read for migration: "
                               f"{get_clean_error_message(e)}") from e
        self._tracking_table_columns = existing
        failed = []
        for legacy, current in self._TRACKING_LEGACY_RENAMES:
            if legacy not in existing or current in existing:
                continue
            try:
                self.spark.sql(f"ALTER TABLE {tracking_table} RENAME COLUMN {legacy} TO {current}")
                existing.discard(legacy)
                existing.add(current)
                self.logger.info(f"   ✅ Renamed tracking column {legacy} -> {current}")
            except Exception as e:
                self.logger.warning(f"Tracking column {legacy} could not be renamed to {current} "
                                    f"({get_clean_error_message(e)[:200]}); {current} is added as a new column instead")
        added = []
        for column, column_type in self._TRACKING_MIGRATION_COLUMNS:
            if column in existing:
                continue
            try:
                self.spark.sql(f"ALTER TABLE {tracking_table} ADD COLUMN {column} {column_type}")
                existing.add(column)
                added.append(column)
            except Exception as e:
                failed.append(f"{column}: {get_clean_error_message(e)[:200]}")
        if added:
            self.logger.info(f"   ✅ Tracking table migration added {len(added)} column(s): {added}")
        if failed:
            raise RuntimeError(f"Tracking table {self._tracking_table_name} migration failed for "
                               f"{len(failed)} column(s): {failed}")
        return added

    def _tracking_unique_view(self, prefix: str) -> str:
        """Generate a session-unique temp view name to avoid any cross-thread collision."""
        import random as _rnd
        return f"_inspire_{prefix}_{self.session_id}_{_rnd.randint(100000, 999999)}"

    @staticmethod
    def _uc_to_row(uc: dict, session_id: int, normalize_fn) -> tuple:
        """Convert a use case dict to a row tuple matching the full tracking table schema.
        Static helper to avoid closure over large objects.
        """
        def _sf(val, default=None):
            if val is None:
                return default
            try:
                return float(val)
            except (ValueError, TypeError):
                return default
        def _ss(val, default=''):
            if val is None:
                return default
            s = str(val).strip()
            return s if s else default

        uc_name = _ss(uc.get('Name'))
        uc_usecase = _ss(uc.get('usecase'))
        normalized = str(normalize_fn(uc_usecase, uc_name)) if (uc_usecase or uc_name) else ''
        return (
            session_id,
            _ss(uc.get('No')),
            _ss(uc.get('Business Domain')),
            _ss(uc.get('Subdomain')),
            normalized,
            uc_name,
            _sf(uc.get('Inspire Score')),
            _ss(uc.get('generated', 'N/A')),
            _ss(uc.get('type')),
            _ss(uc.get('Analytics Technique')),
            _ss(uc.get('Business Priority Alignment')),
            _ss(uc.get('Strategic Goals Alignment')),
            _ss(uc.get('Statement')),
            _ss(uc.get('Solution')),
            _ss(uc.get('Business Value')),
            _ss(uc.get('Beneficiary')),
            _ss(uc.get('Sponsor')),
            _ss(uc.get('Tables Involved')),
            _ss(uc.get('Primary Table')),
            _sf(uc.get('Strategic Alignment')),
            _sf(uc.get('Return on Investment')),
            _sf(uc.get('Reusability')),
            _sf(uc.get('Time to Value')),
            _sf(uc.get('Data Availability')),
            _sf(uc.get('Data Accessibility')),
            _sf(uc.get('Architecture Fitness')),
            _sf(uc.get('Team Skills')),
            _sf(uc.get('Domain Knowledge')),
            _sf(uc.get('People Allocation')),
            _sf(uc.get('Budget Allocation')),
            _sf(uc.get('Time to Production')),
            _sf(uc.get('Value')),
            _sf(uc.get('Feasibility')),
            _sf(uc.get('Priority')),
            _sf(uc.get('Quality Score')),
            _ss(uc.get('Justification')) if uc.get('Justification') else None,
            _ss(uc.get('Quality Summary')) if uc.get('Quality Summary') else None,
            _ss(uc.get('result_table')) if uc.get('result_table') else None,
            _ss(uc.get('notebook_path')) if uc.get('notebook_path') else None,
            _ss(uc.get('High Level Design')) if uc.get('High Level Design') else None,
            _ss(uc.get('genie_instruction')) if uc.get('genie_instruction') else None,
            _ss(uc.get('generate_genie_code_instruction')) if uc.get('generate_genie_code_instruction') else None,
            _ss(uc.get('has_genie_code')) if uc.get('has_genie_code') else None,
            _ss(uc.get('idea_theme')) if uc.get('idea_theme') else None,
            _sf(uc.get('bob_score')),
            _sf(uc.get('bob_tier1_score')),
            _sf(uc.get('bob_tier2_score')),
            _sf(uc.get('bob_tier3_score')),
            _ss(uc.get('directly_involved_schema')) if uc.get('directly_involved_schema') else None,
            _ss(uc.get('foreign_key_relationships')) if uc.get('foreign_key_relationships') else None,
            _ss(uc.get('available_tables_listing')) if uc.get('available_tables_listing') else None,
            _ss(uc.get('technique_implementation_guidance')) if uc.get('technique_implementation_guidance') else None,
            _ss(uc.get('code_structure_mandate')) if uc.get('code_structure_mandate') else None,
            _ss(uc.get('Quality')) if uc.get('Quality') else None,
            _ss(uc.get('quality_mode')) if uc.get('quality_mode') else None,
            _ss(uc.get('elite_status')) if uc.get('elite_status') else None,
            _sf(uc.get('elite_score')),
            DatabricksInspire._tracking_json(uc.get('elite_assessment')),
            DatabricksInspire._tracking_json(DatabricksInspire._selection_flags(uc)),
        ) + tuple(DatabricksInspire._v130_tracking_value(uc, key, kind)
                  for _column, key, kind in DatabricksInspire._V130_TRACKING_FIELDS)

    _TRACKING_FULL_SCHEMA = None

    @classmethod
    def _get_tracking_full_schema(cls):
        if cls._TRACKING_FULL_SCHEMA is None:
            from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType
            cls._TRACKING_FULL_SCHEMA = StructType([
                StructField("session_id", LongType(), False),
                StructField("id", StringType(), True),
                StructField("business_domain", StringType(), True),
                StructField("subdomain", StringType(), True),
                StructField("use_case", StringType(), True),
                StructField("description", StringType(), True),
                StructField("inspire_score", DoubleType(), True),
                StructField("generated", StringType(), True),
                StructField("type", StringType(), True),
                StructField("analytics_technique", StringType(), True),
                StructField("business_priority_alignment", StringType(), True),
                StructField("strategic_goals_alignment", StringType(), True),
                StructField("statement", StringType(), True),
                StructField("solution", StringType(), True),
                StructField("business_value", StringType(), True),
                StructField("beneficiary", StringType(), True),
                StructField("sponsor", StringType(), True),
                StructField("tables_involved", StringType(), True),
                StructField("primary_table", StringType(), True),
                StructField("strategic_alignment", DoubleType(), True),
                StructField("roi", DoubleType(), True),
                StructField("reusability", DoubleType(), True),
                StructField("time_to_value", DoubleType(), True),
                StructField("data_availability", DoubleType(), True),
                StructField("data_accessibility", DoubleType(), True),
                StructField("architecture_fitness", DoubleType(), True),
                StructField("team_skills", DoubleType(), True),
                StructField("domain_knowledge", DoubleType(), True),
                StructField("people_allocation", DoubleType(), True),
                StructField("budget_allocation", DoubleType(), True),
                StructField("time_to_production", DoubleType(), True),
                StructField("value_score", DoubleType(), True),
                StructField("feasibility_score", DoubleType(), True),
                StructField("priority_score", DoubleType(), True),
                StructField("quality_score", DoubleType(), True),
                StructField("priority_reasons", StringType(), True),
                StructField("quality_reasons", StringType(), True),
                StructField("result_database", StringType(), True),
                StructField("notebook_path", StringType(), True),
                StructField("high_level_design", StringType(), True),
                StructField("genie_instruction", StringType(), True),
                StructField("generate_genie_code_instruction", StringType(), True),
                StructField("has_genie_code", StringType(), True),
                StructField("idea_theme", StringType(), True),
                StructField("bob_score", DoubleType(), True),
                StructField("bob_tier1_score", DoubleType(), True),
                StructField("bob_tier2_score", DoubleType(), True),
                StructField("bob_tier3_score", DoubleType(), True),
                StructField("directly_involved_schema", StringType(), True),
                StructField("foreign_key_relationships", StringType(), True),
                StructField("available_tables_listing", StringType(), True),
                StructField("technique_implementation_guidance", StringType(), True),
                StructField("code_structure_mandate", StringType(), True),
                StructField("quality", StringType(), True),
                StructField("quality_mode", StringType(), True),
                StructField("elite_status", StringType(), True),
                StructField("elite_score", DoubleType(), True),
                StructField("elite_assessment", StringType(), True),
                StructField("selection_flags", StringType(), True),
            ] + [StructField(column, StringType(), True) for column, _key, _kind in cls._V130_TRACKING_FIELDS])
        return cls._TRACKING_FULL_SCHEMA

    def _tracking_row_source(self, uc: dict) -> dict:
        row_uc = dict(uc)
        row_uc["quality_mode"] = canonical_use_cases_quality(row_uc.get("quality_mode") or getattr(self, "use_cases_quality", None))
        if not row_uc.get("code_structure_mandate"):
            row_uc["code_structure_mandate"] = self._build_code_structure_mandate(self._genie_notebook_spec(dict(row_uc)))
        return row_uc

    def _tracking_replace_session(self, use_cases: list, chunk_size: int = 500):
        """Replace this session's rows in __inspire_usecases in ONE Delta transaction: rows are
        staged in chunks into an Inspire-owned staging table, then a single replaceWhere overwrite
        swaps them in, so readers see either the previous rows or the new ones, never a gap.
        On failure the previous rows stay unchanged, the failure is recorded in failed_steps and
        RuntimeError is raised. Thread-safe via _tracking_lock.
        """
        if not self._tracking_table_name or not use_cases:
            return
        self._create_tracking_table()
        if not self._tracking_table_created:
            error = RuntimeError(f"Tracking table {self._tracking_table_name} is unavailable (creation or migration failed)")
            self._record_failed_step("Tracking Table", "", error)
            raise error
        import time
        from pyspark.sql.functions import current_timestamp as _cts
        tracking_table = quote_qualified_name(self._tracking_table_name)
        stage_tracking_table = quote_qualified_name(f"{self._tracking_table_name.replace('`', '')}_stage_{int(self.session_id)}")
        schema = self._get_tracking_full_schema()
        with self._tracking_lock:
            try:
                self.spark.sql(f"DROP TABLE IF EXISTS {stage_tracking_table}")
                staged = 0
                for start in range(0, len(use_cases), max(1, int(chunk_size))):
                    rows = [self._uc_to_row(self._tracking_row_source(uc), self.session_id, self._normalize_usecase)
                            for uc in use_cases[start:start + max(1, int(chunk_size))]]
                    (self.spark.createDataFrame(rows, schema=schema).withColumn("updated_at", _cts())
                        .write.format("delta").mode("append").saveAsTable(stage_tracking_table))
                    staged += len(rows)
                columns = [field.name for field in schema.fields] + ["updated_at"]
                table_columns = getattr(self, "_tracking_table_columns", None)
                writable = [c for c in columns if not table_columns or c.lower() in table_columns]
                if len(writable) < len(columns):
                    self.logger.warning(f"Tracking table {self._tracking_table_name} lacks columns "
                                        f"{[c for c in columns if c not in writable]}; this session is written without them")
                column_list = ", ".join(quote_identifier(c) for c in writable)
                for attempt in range(1, 4):
                    try:
                        (self.spark.sql(f"SELECT {column_list} FROM {stage_tracking_table}")
                            .write.format("delta").mode("overwrite")
                            .option("replaceWhere", f"session_id = {int(self.session_id)}")
                            .saveAsTable(tracking_table))
                        break
                    except Exception as swap_error:
                        if attempt == 3 or "concurrent" not in str(swap_error).lower():
                            raise
                        self.logger.warning(f"Tracking table swap hit a concurrent update (attempt {attempt}/3); retrying")
                        time.sleep(0.5 * attempt)
                self.logger.info(f"✅ Tracking table: session {self.session_id} replaced atomically with {staged} use cases")
            except Exception as e:
                self._record_failed_step("Tracking Table", "", e)
                raise RuntimeError(
                    f"Tracking table {self._tracking_table_name} was not updated for session {self.session_id} "
                    f"(its previous rows are unchanged): {get_clean_error_message(e)}") from e
            finally:
                try:
                    self.spark.sql(f"DROP TABLE IF EXISTS {stage_tracking_table}")
                except Exception as drop_error:
                    self.logger.warning(f"Staging table {stage_tracking_table} could not be dropped: "
                                        f"{get_clean_error_message(drop_error)}")

    def _calculate_dynamic_parallelism(self, total_tables, total_schema_chars, safe_context_limit, base_prompt_size):
        memory_pool = max(1, self.cluster_memory_gb * max(self.cluster_worker_count, 1))
        max_by_memory = max(2, min(8, int(memory_pool // 8)))
        if total_tables <= 0:
            return max_by_memory, 0, 0, 0, max_by_memory
        avg_table_chars = max(1, int(total_schema_chars / total_tables))
        available_chars = max(1, safe_context_limit - base_prompt_size)
        tables_per_batch = max(1, int(available_chars // avg_table_chars))
        est_batches = int((total_tables + tables_per_batch - 1) // tables_per_batch) if tables_per_batch else total_tables
        size_factor = 1.0
        if avg_table_chars >= 24000:
            size_factor = 0.5
        elif avg_table_chars >= 18000:
            size_factor = 0.7
        elif avg_table_chars >= 12000:
            size_factor = 0.85
        recommended = int(max_by_memory * size_factor)
        if recommended < 2:
            recommended = 2
        if recommended > max_by_memory:
            recommended = max_by_memory
        if est_batches > 0 and recommended > est_batches:
            recommended = est_batches
        if recommended < 1:
            recommended = 1
        return recommended, tables_per_batch, est_batches, avg_table_chars, max_by_memory
        
    def _get_translations(self, language: str) -> dict:
        return self.translation_service.get_translations(language)

    _RUN_OUTCOME_LOCK = threading.Lock()

    def _workspace_mkdirs_with_retry(self, path: str, attempts: int = 4, base_delay: float = 1.0):
        import time
        last_error = None
        for attempt in range(1, attempts + 1):
            try:
                self.w_client.workspace.mkdirs(path)
                return
            except Exception as mkdir_error:
                last_error = mkdir_error
                if attempt < attempts:
                    time.sleep(base_delay * (2 ** (attempt - 1)))
        raise last_error

    def _import_workspace_file(self, path: str, content, import_format, overwrite: bool = True):
        self._workspace_mkdirs_with_retry(os.path.dirname(path))
        self.w_client.workspace.import_(path=path, content=content, format=import_format, overwrite=overwrite)

    def _ensure_output_directories(self):
        self.logger.info(f"Ensuring workspace directories exist...")
        output_dirs = [self.base_output_dir, self.session_output_dir, self.notebook_output_dir, self.docs_output_dir]
        failed_dirs = []
        for output_dir in output_dirs:
            try:
                self._workspace_mkdirs_with_retry(output_dir)
            except Exception as dir_error:
                failed_dirs.append(output_dir)
                self.logger.error(f"Failed to create workspace directory {output_dir} after retries: {get_clean_error_message(dir_error)}")
        if failed_dirs:
            self.logger.warning(f"{len(failed_dirs)} output director(ies) could not be pre-created; artifact writers will recreate their parent before writing: {failed_dirs}")
        else:
            self.logger.info(f"Successfully ensured all output directories.")

    def _record_failed_step(self, step: str, language: str, error) -> dict:
        if isinstance(error, BaseException):
            error_text = get_clean_error_message(error, max_chars=300)
        else:
            error_text = str(error or "")[:300]
        entry = {"step": str(step), "language": str(language or ""), "error": error_text}
        with self._RUN_OUTCOME_LOCK:
            failed = getattr(self, "failed_steps", None)
            if not isinstance(failed, list):
                failed = []
                self.failed_steps = failed
            known = any(f["step"] == entry["step"] and f["language"] == entry["language"] for f in failed)
            if not known:
                failed.append(entry)
        if not known:
            self.logger.warning(f"[ARTIFACT STEP FAILED] step={entry['step']} language={entry['language'] or '-'}: {error_text}")
        self._merge_run_outcome_into_results()
        return entry

    def _record_translation_gaps(self, language: str, gaps: dict):
        if not gaps:
            return
        with self._RUN_OUTCOME_LOCK:
            recorded = getattr(self, "translation_gaps", None)
            if not isinstance(recorded, dict):
                recorded = {}
                self.translation_gaps = recorded
            labels = dict((recorded.get(str(language)) or {}).get("labels_in_english") or {})
            recorded[str(language)] = {**gaps, "labels_in_english": {**labels, **dict(gaps.get("labels_in_english") or {})}} if labels else gaps
        self._merge_run_outcome_into_results()

    def _merge_run_outcome_into_results(self) -> dict:
        metadata_report = self._metadata_unavailable_report()
        endpoint_health = LLM_ENDPOINT_HEALTH.snapshot()
        with self._RUN_OUTCOME_LOCK:
            results = getattr(self, "final_inspire_results_json", None)
            if not isinstance(results, dict):
                results = {}
                self.final_inspire_results_json = results
            failed = list(getattr(self, "failed_steps", None) or [])
            gaps = dict(getattr(self, "translation_gaps", None) or {})
            font_fallbacks = dict(getattr(self, "document_font_fallbacks", None) or {})
            if failed:
                results["failed_steps"] = failed
            if gaps:
                results["translation_gaps"] = gaps
            if font_fallbacks:
                results["document_font_fallbacks"] = font_fallbacks
            results["metadata_unavailable"] = metadata_report
            results["llm_endpoint_health"] = endpoint_health
            results.update(self._v130_results_keys())
            return results

    def _v130_results_keys(self) -> dict:
        """The v1.3.0 top-level results: shared by results_json and the JSON catalog, and restored from a
        catalog in docs-only mode."""
        loaded = getattr(self, "_loaded_catalog_meta", None) or {}
        out = {}
        discovery_type = (loaded.get("discovery_type") or "Standard") if loaded else getattr(self, "discovery_type", "Standard")
        deep = getattr(self, "deep_evidence", None)
        out["discovery_type"] = discovery_type
        out["data_access"] = loaded.get("data_access") or (deep.summary() if deep is not None else {
            "discovery_type": discovery_type, "tables_read": [], "tables_read_count": 0, "statements": 0,
            "rows_sent_to_ai": 0})
        tree = getattr(self, "strategy_tree", None)
        if tree:
            out["strategic_context"] = {key: value for key, value in tree.items() if key != "warnings"}
        if getattr(self, "strategy_coverage", None) is not None:
            out["strategy_coverage"] = self.strategy_coverage
        if getattr(self, "data_foundations", None) is not None:
            out["foundations"] = self.data_foundations
            out["delivery_order"] = list(getattr(self, "delivery_order", None) or [])
        if getattr(self, "feedback_history", None) is not None:
            out["portfolio_history"] = list(self.feedback_history.values())
        if getattr(self, "excluded_by_feedback", None) is not None:
            out["excluded_by_feedback"] = list(self.excluded_by_feedback)
        if getattr(self, "value_tracking", None) is not None:
            out["value_tracking"] = list(self.value_tracking)
        if getattr(self, "ailogs_check", None) is not None:
            out["ailogs_check"] = dict(self.ailogs_check)
        if getattr(self, "best_of_best_outcome", None) is not None:
            out["best_of_best"] = self.best_of_best_outcome
        evidence = getattr(self, "discovery_evidence", None) or {}
        usage = evidence.get("usage") or {}
        out["usage_signals"] = loaded.get("usage_signals") or {
            "available": bool(usage.get("available")), "reason": usage.get("reason") or "",
            "tables_with_reads": len(usage.get("reads") or {}), "co_used_pairs": len(usage.get("co_used") or []),
            "metric_views": len(evidence.get("metric_views") or [])}
        return out

    _V130_CATALOG_KEYS = ("discovery_type", "data_access", "usage_signals", "strategic_context", "strategy_coverage",
                          "foundations", "delivery_order", "best_of_best")

    def _doc_data_access(self) -> tuple:
        """(discovery type of the run, tables read). The type is "" for a catalog written before v1.3.0."""
        loaded = getattr(self, "_loaded_catalog_meta", None)
        if loaded is not None:
            access = loaded.get("data_access") or {}
            return str(loaded.get("discovery_type") or ""), int(access.get("tables_read_count") or 0)
        deep = getattr(self, "deep_evidence", None)
        return (str(getattr(self, "discovery_type", "Standard") or "Standard"),
                int(deep.summary()["tables_read_count"]) if deep is not None else 0)

    def _metadata_unavailable_report(self) -> dict:
        loader = getattr(self, "data_loader", None)
        report = loader.metadata_unavailable_report() if loader is not None else {"count": 0, "tables": []}
        honesty = getattr(self, "processing_honesty", None)
        if isinstance(honesty, dict):
            honesty["tables_metadata_unavailable"] = [dict(entry) for entry in report["tables"]]
        return report

    @staticmethod
    def _metadata_unavailable_listing(entries: list, limit: int = 20) -> str:
        listing = "; ".join(f"{e['table']} ({e['reason']})" for e in entries[:limit])
        return listing + (f"; +{len(entries) - limit} more" if len(entries) > limit else "")

    def _nothing_to_discover_message(self) -> str:
        scope = "; ".join(f"{label}: {value}" for label, value in (("catalogs", getattr(self, "catalogs_str", "")),
                                                                    ("schemas", getattr(self, "schemas_str", "")),
                                                                    ("tables", getattr(self, "tables_str", ""))) if value)
        lost = self._metadata_unavailable_report()["tables"]
        reason = (f"{len(lost)} table(s) could not be loaded: {self._metadata_unavailable_listing(lost)}" if lost
                  else "no table with columns was found")
        return (f"Discovery found nothing to analyze in the UC metadata ({scope or 'no catalog, schema or table given'}): "
                f"{reason}. Check the UC Metadata widget and that this run's identity can use the catalogs and schemas and "
                "select from the tables, then run again.")

    def _report_metadata_unavailable(self) -> list:
        report = self._metadata_unavailable_report()
        reported = getattr(self, "_metadata_unavailable_reported", None)
        if not isinstance(reported, set):
            reported = set()
            self._metadata_unavailable_reported = reported
        new = [entry for entry in report["tables"] if entry["table"] not in reported]
        if not new:
            return []
        reported.update(entry["table"] for entry in new)
        listing = self._metadata_unavailable_listing(new)
        self.logger.warning(f"[METADATA_UNAVAILABLE] {len(new)} table(s) could not be loaded and are not analysed: "
                            f"{listing}")
        self._emit_pipeline_status(
            stage_name="Data Preparation", step_name="Metadata Load", sub_step_name="Tables without metadata",
            message=f"{len(new)} table(s) could not be loaded and are not analysed: {listing}",
            status="ended_warning", progress_increment=0.0,
            result_json={"count": len(new), "tables": [dict(entry) for entry in new]})
        return new

    @staticmethod
    def _failed_step_label(failed: dict) -> str:
        return f"{failed['step']} ({failed['language']})" if failed.get("language") else str(failed["step"])

    def _run_outcome_warnings(self) -> list:
        failed = list(getattr(self, "failed_steps", None) or [])
        gaps = dict(getattr(self, "translation_gaps", None) or {})
        lost = self._metadata_unavailable_report()["tables"]
        warnings = []
        if lost:
            warnings.append(f"{len(lost)} table(s) without metadata were not analysed: "
                            + self._metadata_unavailable_listing(lost))
        if failed:
            warnings.append(f"{len(failed)} artifact step(s) failed: "
                            + ", ".join(self._failed_step_label(f) for f in failed))
        if gaps:
            warnings.append("translation gaps: " + ", ".join(
                f"{language} ({gap.get('use_cases', 0)} use case(s) with text shown in English"
                + (f"; labels shown in English: {', '.join(sorted(gap['labels_in_english']))}" if gap.get("labels_in_english") else "") + ")"
                for language, gap in gaps.items()))
        fonts = dict(getattr(self, "document_font_fallbacks", None) or {})
        if fonts:
            warnings.append("PDF fonts replaced by installed fonts: " + ", ".join(
                f"{language} ({', '.join(entry['script'] for entry in entries)})" for language, entries in fonts.items()))
        return warnings

    def _report_run_completion(self, success_message: str, warning_message: str) -> bool:
        self._merge_run_outcome_into_results()
        warnings = self._run_outcome_warnings()
        if warnings:
            text = f"{warning_message}: {'; '.join(warnings)}"
            self.logger.warning(text)
            log_print(text, level="WARNING")
            return False
        self.logger.info(success_message)
        log_print(success_message)
        return True

    def _report_processing_honesty(self):
        """
        Generates and displays a detailed report of what happened during processing.
        Reports on batch splits, column drops, and whether all data was processed completely.
        """
        log_print(f"\n{'='*80}")
        log_print(f"📊 PROCESSING HONESTY REPORT")
        log_print(f"{'='*80}\n")
        
        h = self.processing_honesty
        
        log_print(f"📈 Overall Statistics:")
        log_print(f"   • Total tables discovered: {h['total_tables_discovered']}")
        log_print(f"   • Total tables processed: {h['total_tables_processed']}")
        log_print(f"   • Total batches created: {h['total_batches_created']}")
        log_print(f"   • Total batch splits performed: {h['total_batch_splits']}")
        
        if h['batch_split_history']:
            proactive_splits = [s for s in h['batch_split_history'] if s.get('split_type') == 'Proactive']
            reactive_splits = [s for s in h['batch_split_history'] if s.get('split_type') == 'Reactive']
            
            log_print(f"\n⚡ Batch Splitting Details:")
            log_print(f"   {h['total_batch_splits']} batch(es) were split to fit LLM context limits:")
            log_print(f"   • Proactive splits (detected before LLM call): {len(proactive_splits)}")
            log_print(f"   • Reactive splits (after LLM error): {len(reactive_splits)}")
            
            for idx, split in enumerate(h['batch_split_history'], 1):
                split_icon = "🔮" if split.get('split_type') == 'Proactive' else "⚡"
                log_print(f"   {idx}. {split_icon} {split.get('split_type', 'Unknown')} | Batch '{split['batch']}': {split['original_tables']} tables → {split['split_into']} sub-batches")
                log_print(f"      - Sub-batch 1: {split['sub_batch_1_tables']} tables")
                log_print(f"      - Sub-batch 2: {split['sub_batch_2_tables']} tables")
        else:
            log_print(f"\n✅ No batch splitting was required - all batches fit within LLM context limits")
        
        if h['tables_with_columns_dropped']:
            log_print(f"\n⚠️  Column Dropping Details:", level="WARNING")
            log_print(f"   {len(h['tables_with_columns_dropped'])} table(s) had columns dropped to fit LLM limits:")
            for idx, drop_info in enumerate(h['tables_with_columns_dropped'], 1):
                business_tag = "🔵 BUSINESS" if drop_info['is_business'] else "⚪ non-business"
                log_print(f"   {idx}. {business_tag} | {drop_info['table']}")
                log_print(f"      - Original columns: {drop_info['original_columns']}")
                log_print(f"      - Kept columns: {drop_info['kept_columns']} ({drop_info['drop_percentage']:.1f}% dropped)")
                log_print(f"      - Dropped columns: {drop_info['dropped_columns']}")
        else:
            log_print(f"\n✅ No columns were dropped - all tables processed with full schema")

        lost_tables = self._metadata_unavailable_report()["tables"]
        if lost_tables:
            log_print(f"\n⚠️  Tables Without Metadata:", level="WARNING")
            log_print(f"   {len(lost_tables)} table(s) could not be loaded and were not analysed:", level="WARNING")
            for idx, entry in enumerate(lost_tables, 1):
                log_print(f"   {idx}. {entry['table']}: {entry['reason']}", level="WARNING")
        else:
            log_print(f"\n✅ Metadata was loaded for every requested table")
        
        log_print(f"\n{'='*80}")
        log_print(f"🎯 HONESTY ASSESSMENT:")
        log_print(f"{'='*80}")
        
        honesty_percentage = 100.0
        issues = []

        if lost_tables:
            requested_tables = len(lost_tables) + int(h.get('total_tables_discovered') or 0)
            lost_percentage = len(lost_tables) / requested_tables * 100
            issues.append(f"• {len(lost_tables)} table(s) were not analysed because their metadata could not be loaded "
                          f"({lost_percentage:.1f}% of the requested tables)")
            honesty_percentage -= lost_percentage
        
        if h['batch_split_history']:
            issues.append(f"• {h['total_batch_splits']} batch split(s) occurred (but all tables were still processed)")
        
        if h['tables_with_columns_dropped']:
            total_columns_dropped = sum(d['dropped_columns'] for d in h['tables_with_columns_dropped'])
            total_columns_original = sum(d['original_columns'] for d in h['tables_with_columns_dropped'])
            drop_percentage = (total_columns_dropped / total_columns_original * 100) if total_columns_original > 0 else 0
            
            issues.append(f"• {len(h['tables_with_columns_dropped'])} table(s) had columns dropped ({drop_percentage:.1f}% of columns from affected tables)")
            
            honesty_percentage -= min(drop_percentage / 2, 30)
        
        if not issues:
            log_print(f"\n✅ 100% HONEST - All tables processed completely with all columns")
            log_print(f"   No compromises were made during processing.")
        else:
            log_print(f"\n📊 Honesty Score: {max(0.0, honesty_percentage):.1f}%")
            log_print(f"\n   Processing required the following compromises:")
            for issue in issues:
                log_print(f"   {issue}")
            log_print(f"\n   ℹ️  Note: Batch splits don't affect completeness - all tables are still processed.")
            log_print(f"   ⚠️  Column drops DO affect completeness - some schema details were omitted.", level="WARNING")

        failed_steps = list(getattr(self, "failed_steps", None) or [])
        if failed_steps:
            log_print(f"\n⚠️  {len(failed_steps)} artifact step(s) failed; the run completes with warnings:", level="WARNING")
            for failed in failed_steps:
                log_print(f"   • {self._failed_step_label(failed)}: {failed['error']}", level="WARNING")
            self.logger.warning(f"Run completed with {len(failed_steps)} failed artifact step(s): "
                                f"{[self._failed_step_label(f) for f in failed_steps]}")
        translation_gaps = dict(getattr(self, "translation_gaps", None) or {})
        for language, gaps in translation_gaps.items():
            log_print(f"⚠️  {language}: {gaps.get('use_cases', 0)} use case(s) contain text shown in English "
                      f"(flagged in the {language} documents): {gaps.get('fields', {})}", level="WARNING")
        self._merge_run_outcome_into_results()

        log_print(f"\n{'='*80}\n")
    
    _LANGUAGE_ABBREVIATIONS = {
        "english": "en", "arabic": "ar", "french": "fr", "spanish": "es", "german": "de",
        "portuguese": "pt", "italian": "it", "japanese": "ja", "korean": "ko", "chinese": "zh",
        "hindi": "hi", "russian": "ru", "swedish": "sv", "danish": "da", "norwegian": "no",
        "finnish": "fi", "polish": "pl", "romanian": "ro", "ukrainian": "uk", "dutch": "nl",
        "indonesian": "id", "malay": "ms", "tamil": "ta",
    }

    def _get_lang_abbr(self, language: str) -> str:
        """Returns the ISO 639-1 code for a language name; a qualifier such as "(Mandarin)" is ignored."""
        base = re.sub(r"\s*\(.*?\)\s*", " ", str(language or "")).strip().lower()
        if base in self._LANGUAGE_ABBREVIATIONS:
            return self._LANGUAGE_ABBREVIATIONS[base]
        return re.sub(r"[^a-z]", "", base)[:2] or "xx"

    # === BUSINESS CONTEXT AND SPONSOR MAPPING ===
    CONTEXT_INIT_TIMEOUT_S = 1800

    def _await_context_init(self, ctx_future, instr_future, timeout_s=None):
        timeout_s = self.CONTEXT_INIT_TIMEOUT_S if timeout_s is None else timeout_s
        try:
            context = _safe_future_result(ctx_future, timeout=timeout_s)
        except concurrent.futures.TimeoutError:
            ctx_future.cancel()
            self.logger.error(f"[CONTEXT-INIT-TIMEOUT FIRED] the business context call took longer than {timeout_s}s; "
                              "continuing with an empty LLM business context")
            context = {}
        return context, _safe_future_result(instr_future, timeout=timeout_s)

    def _interpret_generation_instructions(self) -> dict:
        """Parse user generation instructions into structured directives using LLM.
        Returns a dict with table_joins, goal_setting, forced_techniques, table_ignores, per_prompt_instructions.
        """
        raw_instructions = getattr(self, "user_generation_instructions", "") or ""
        default_result = {
            "table_joins": [],
            "goal_setting": {"has_goal": False, "goal_description": "", "goal_summary_sentence": "", "strategic_goal_match": ""},
            "forced_techniques": [],
            "table_ignores": [],
            "construct_exclusions": [],
            "change_management_posture": "",
            "data_characteristics": {},
            "best_of_best_count": None,
            "per_prompt_instructions": {name: "" for name in GENERATION_DIRECTIVE_CATEGORIES_BY_PROMPT},
        }
        if not raw_instructions.strip():
            self.logger.info("No generation instructions provided - skipping interpretation")
            self.parsed_generation_instructions = default_result
            return default_result
        
        self.logger.info(f"Interpreting generation instructions ({len(raw_instructions)} chars)...")
        
        def _extract_json_robust(response_text: str) -> dict:
            """Multi-strategy JSON extraction that handles thinker model quirks."""
            if not response_text:
                return None
            
            strategies = []
            
            cleaned = clean_json_response(response_text)
            strategies.append(("clean_json_response", cleaned))
            
            single_to_double = cleaned.replace("'", '"')
            strategies.append(("single_to_double_quotes", single_to_double))
            
            json_block_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', response_text)
            if json_block_match:
                strategies.append(("markdown_fence_extract", json_block_match.group(1).strip()))
            
            brace_match = re.search(r'(\{[\s\S]*"per_prompt_instructions"[\s\S]*\})', response_text)
            if brace_match:
                strategies.append(("per_prompt_instructions_anchor", brace_match.group(1)))
            
            outermost_match = re.search(r'(\{[\s\S]*\})\s*$', response_text)
            if outermost_match:
                strategies.append(("outermost_brace", outermost_match.group(1)))
            
            for strategy_name, candidate in strategies:
                try:
                    result = json.loads(candidate)
                    if isinstance(result, dict):
                        if "data" in result:
                            result = result["data"]
                        self.logger.info(f"JSON parsed successfully using strategy: {strategy_name}")
                        return result
                except (json.JSONDecodeError, TypeError):
                    continue
            
            try:
                fixed = re.sub(r"(?<!\\)'", '"', response_text)
                fixed = re.sub(r',\s*}', '}', fixed)
                fixed = re.sub(r',\s*]', ']', fixed)
                fixed_clean = clean_json_response(fixed)
                result = json.loads(fixed_clean)
                if isinstance(result, dict):
                    if "data" in result:
                        result = result["data"]
                    self.logger.info("JSON parsed with aggressive quote/trailing-comma fix")
                    return result
            except (json.JSONDecodeError, TypeError):
                pass
            
            self.logger.warning(f"All JSON extraction strategies failed. Response preview: {response_text[:500]}")
            return None
        
        max_attempts = 2
        for attempt in range(1, max_attempts + 1):
            try:
                prompt_vars = {
                    "generation_instructions": raw_instructions,
                    "business_name": self.business_name,
                }
                response_raw = self.ai_agent.run_worker(
                    step_name=f"Generation_Instructions_Interpreter_Attempt{attempt}",
                    worker_prompt_path="GENERAL_INSTRUCTION_INTERPRETER_PROMPT",
                    prompt_vars=prompt_vars,
                    response_schema=None,
                    timeout_override=120,
                    skip_cache=(attempt > 1)  # v0.7.11: retries bypass cache
                )
                if not response_raw:
                    self.logger.warning(f"Interpreter attempt {attempt} returned empty response")
                    continue
                
                parsed = _extract_json_robust(response_raw)
                if parsed is None:
                    self.logger.warning(f"Interpreter attempt {attempt} returned unparseable JSON - will retry")
                    continue
                
                for key in default_result:
                    if key not in parsed:
                        parsed[key] = default_result[key]
                
                if not isinstance(parsed.get("per_prompt_instructions"), dict):
                    parsed["per_prompt_instructions"] = default_result["per_prompt_instructions"]
                for pname in default_result["per_prompt_instructions"]:
                    if pname not in parsed["per_prompt_instructions"]:
                        parsed["per_prompt_instructions"][pname] = ""
                
                goal = parsed.get("goal_setting", {})
                if isinstance(goal, dict) and goal.get("has_goal"):
                    goal_match = goal.get("strategic_goal_match", "")
                    goal_desc = str(goal.get("goal_description", "") or "").strip()
                    self.user_strategic_goal_category = goal_match or ""
                    if goal_desc or goal_match:
                        self.user_strategic_goals = [goal_desc or goal_match]
                        self.logger.info(f"✅ Goal extracted from generation instructions, kept in the user's wording: "
                                         f"{self.user_strategic_goals[0]} (closest standard goal: {goal_match or 'none'})")
                
                joins = parsed.get("table_joins", [])
                if joins:
                    self.logger.info(f"✅ Table joins extracted: {len(joins)} join(s)")
                    for j in joins:
                        self.logger.info(f"  JOIN: {j.get('left_table', '?')} <-> {j.get('right_table', '?')} ON {j.get('join_columns', [])}")
                
                ignores = parsed.get("table_ignores", [])
                if ignores:
                    self.logger.info(f"✅ Table ignore patterns extracted: {len(ignores)} pattern(s)")
                    for ig in ignores:
                        self.logger.info(f"  IGNORE: {ig.get('pattern', '?')} (type: {ig.get('type', '?')})")
                
                techniques = parsed.get("forced_techniques", [])
                if techniques:
                    self.logger.info(f"✅ Forced techniques extracted: {len(techniques)} technique(s)")
                    for t in techniques:
                        self.logger.info(f"  TECHNIQUE: {t.get('technique', '?')} ({t.get('context', 'all use cases')})")
                
                has_instructions = any(
                    v.strip() for v in parsed.get("per_prompt_instructions", {}).values() if isinstance(v, str)
                )
                if has_instructions:
                    self.logger.info(f"✅ Per-prompt instructions generated for {sum(1 for v in parsed['per_prompt_instructions'].values() if isinstance(v, str) and v.strip())} prompts")
                else:
                    self.logger.warning("⚠️ Interpreter returned parsed JSON but no per-prompt instructions were generated")
                
                _goal_parsed = parsed.get("goal_setting")
                has_structured = bool(
                    (isinstance(_goal_parsed, dict) and _goal_parsed.get("has_goal"))
                    or any(parsed.get(k) for k in ("table_joins", "forced_techniques", "table_ignores", "construct_exclusions", "change_management_posture", "data_characteristics"))
                    or parsed.get("best_of_best_count") is not None
                )
                parsed["interpretation_status"] = "ok" if (has_instructions or has_structured) else "empty"
                if parsed["interpretation_status"] == "empty":
                    self.logger.warning("⚠️ Interpreter extracted no directives - the raw Generation Instructions will be injected verbatim into every prompt")
                self.parsed_generation_instructions = parsed
                self.logger.info("✅ Generation instructions interpreted successfully")
                return parsed
            except Exception as e:
                self.logger.warning(f"Interpreter attempt {attempt} failed: {get_clean_error_message(e)}")
                if attempt < max_attempts:
                    self.logger.info(f"Retrying interpretation (attempt {attempt + 1}/{max_attempts})...")
        
        self.logger.error("All interpreter attempts failed - the raw Generation Instructions will be injected verbatim into every prompt.")
        failed_result = dict(default_result, interpretation_status="failed")
        self.parsed_generation_instructions = failed_result
        return failed_result
    
    def _get_generation_instruction_for_prompt(self, prompt_name: str) -> str:
        """Get the formatted generation instruction section for a specific prompt."""
        header = "### NON-NEGOTIABLE GENERATION INSTRUCTIONS (MUST FOLLOW STRICTLY)"
        parsed = getattr(self, "parsed_generation_instructions", {}) or {}
        per_prompt = parsed.get("per_prompt_instructions", {}) or {}
        instruction = str(per_prompt.get(prompt_name, "") or "").strip()
        if instruction.startswith(header):
            return instruction

        def _text(value):
            return str(value or "").strip()

        def _entries(value):
            if isinstance(value, list):
                return value
            return [value] if _text(value) else []

        lines = []
        for category in GENERATION_DIRECTIVE_CATEGORIES_BY_PROMPT.get(prompt_name, ()):
            value = parsed.get(category)
            if category == "goal_setting":
                if isinstance(value, dict) and value.get("has_goal") and _text(value.get("goal_description")):
                    lines.append(f"- GOAL: {_text(value.get('goal_description'))}")
            elif category == "table_joins":
                for join in _entries(value):
                    if not (isinstance(join, dict) and _text(join.get("left_table")) and _text(join.get("right_table"))):
                        continue
                    columns = join.get("join_columns")
                    columns = ", ".join(_text(c) for c in columns if _text(c)) if isinstance(columns, list) else _text(columns)
                    on_clause = f" on {columns}" if columns else ""
                    join_type = _text(join.get("join_type")) or "inner"
                    lines.append(f"- TABLE JOIN: `{_text(join.get('left_table'))}` and `{_text(join.get('right_table'))}` are joined{on_clause} ({join_type} join) and must be treated as one unit")
            elif category == "forced_techniques":
                for item in _entries(value):
                    technique = _text(item.get("technique")) if isinstance(item, dict) else _text(item)
                    if technique:
                        context = _text(item.get("context")) if isinstance(item, dict) else ""
                        lines.append(f"- REQUIRED TECHNIQUE: {technique}" + (f" ({context})" if context else ""))
            elif category == "table_ignores":
                for item in _entries(value):
                    pattern = _text(item.get("pattern")) if isinstance(item, dict) else _text(item)
                    if pattern:
                        kind = _text(item.get("type")) if isinstance(item, dict) else ""
                        lines.append(f"- IGNORE TABLES matching `{pattern}`" + (f" ({kind})" if kind else ""))
            elif category == "construct_exclusions":
                items = [_text(x) for x in _entries(value) if _text(x)]
                if items:
                    lines.append("- CONSTRUCTS THAT DO NOT EXIST IN THIS BUSINESS (never frame a use case around them): " + "; ".join(items))
            elif category == "change_management_posture":
                posture = _text(value).lower()
                if posture:
                    mandate = {
                        "conservative": " -- every Solution MUST name a baseline fallback or rollback trigger",
                        "experimental": " -- every Solution MUST name a pilot scope",
                    }.get(posture, "")
                    lines.append(f"- CHANGE-MANAGEMENT POSTURE: {posture}{mandate}")
            elif category == "data_characteristics":
                if isinstance(value, dict):
                    for table_name, note in value.items():
                        if _text(table_name) and _text(note):
                            lines.append(f"- DATA CHARACTERISTIC of `{_text(table_name)}`: {_text(note)} -- technique selection MUST respect it")
                else:
                    for note in _entries(value):
                        if _text(note):
                            lines.append(f"- DATA CHARACTERISTIC: {_text(note)} -- technique selection MUST respect it")

        raw_instructions = _text(getattr(self, "user_generation_instructions", ""))
        verbatim = raw_instructions if parsed.get("interpretation_status") != "ok" else ""
        feedback_lines = []
        if prompt_name == "REVIEW_USE_CASES_PROMPT":
            feedback_lines = [f"- {item['title']} on `{item['primary_table']}`"
                              for item in list((getattr(self, "feedback_rejected", None) or {}).values())[:50]]
        if not (instruction or lines or verbatim or feedback_lines):
            return "*(No additional generation instructions provided)*"
        parts = [header, "", "The following instructions were provided by the user and MUST be followed without exception. Violating any of these instructions is an IMPLEMENTATION FAILURE."]
        if instruction:
            parts += ["", instruction]
        if lines:
            parts += ["", "**Structured user directives (parsed from the user's Generation Instructions):**"] + lines
        if feedback_lines:
            parts += ["", "**Use cases the user rejected in earlier runs (from feedback). DROP any use case that is the "
                          "same decision as one of them:**"] + feedback_lines
        if verbatim:
            parts += ["", "**User Generation Instructions (verbatim; they could not be parsed, so apply every part that is relevant to this task):**", verbatim]
        return "\n".join(parts)

    def _prompt_overhead_chars(self, prompt_name: str, template: str = None) -> int:
        if template is None:
            template = self.ai_agent.prompt_templates.get(prompt_name) or PROMPT_TEMPLATES.get(prompt_name, "")
        instructions = (self._get_generation_instruction_for_prompt(prompt_name)
                        if "{generation_instructions_section}" in template else "")
        return len(USER_DIRECTIVE_SUPREMACY_PREAMBLE) + len(template) + len(instructions)
    
    def _filter_use_cases_by_goal_alignment(self, use_cases: list, stage: str = "goal_alignment") -> list:
        """Hard filter: keep only use cases the LLM classifies as aligned (YES) with the user's goal.
        Only triggered when the interpreter extracted has_goal=True; logs the funnel stage either way.

        Every UC without a YES/NO verdict (empty answer, unknown verdict, failed or timed-out batch)
        is re-classified in smaller batches with the LLM cache bypassed. A UC still unverified after
        the retries is dropped: the goal is a user directive, so an unverified UC is never kept as
        aligned. If the LLM rejects every UC, the result is empty.
        """
        n_in = len(use_cases or [])
        parsed = getattr(self, "parsed_generation_instructions", {}) or {}
        goal = parsed.get("goal_setting", {})
        if not (isinstance(goal, dict) and goal.get("has_goal")):
            self._log_funnel_stage(stage, n_in, n_in)
            return use_cases

        goal_desc = goal.get("goal_description", "") or ""
        if not goal_desc.strip():
            self.logger.warning("has_goal is True but goal_description is empty - skipping goal filter")
            self._log_funnel_stage(stage, n_in, n_in)
            return use_cases

        if not use_cases:
            self._log_funnel_stage(stage, 0, 0)
            return use_cases

        _sep = "=" * 80
        self.logger.info(f"\n{_sep}")
        self.logger.info(f"🎯 GOAL ALIGNMENT FILTER: Removing use cases that do not match user goal")
        self.logger.info(f"{_sep}")
        self.logger.info(f"Goal: {goal_desc}")
        self.logger.info(f"Use cases to classify: {len(use_cases)}")
        _goal_preview = goal_desc[:100] + "..." if len(goal_desc) > 100 else goal_desc
        log_print(f"\n🎯 GOAL ALIGNMENT FILTER: Checking {len(use_cases)} use cases against goal: \"{_goal_preview}\"")

        def _row(uc: dict) -> str:
            _id = str(uc.get("No", "")).replace("|", "/")
            _name = str(uc.get("Name", "")).replace("|", "/")
            _desc = str(uc.get("Description", "") or uc.get("Statement", "") or "").replace("|", "/")[:200]
            return f"| {_id} | {_name} | {_desc} |\n"

        context_limit = get_max_context_chars("English", "GOAL_ALIGNMENT_FILTER_PROMPT")
        base_prompt_size = self._prompt_overhead_chars("GOAL_ALIGNMENT_FILTER_PROMPT") + len(goal_desc) + 500
        avg_row = max(1, sum(len(_row(uc)) for uc in use_cases) // len(use_cases))
        batch_size = max(1, int((context_limit * 0.85 - base_prompt_size) / avg_row))

        def _classify(batch_label: str, batch: list, skip_cache: bool) -> dict:
            if _is_thread_cancelled():
                raise Exception(f"[GoalFilter_{batch_label}] Aborted: parent task cancelled/timed out")
            use_cases_table = "| ID | Name | Description |\n|---|---|---|\n" + "".join(_row(uc) for uc in batch)
            response_raw = self.ai_agent.run_worker(
                step_name=f"Goal_Alignment_Filter_{batch_label}",
                worker_prompt_path="GOAL_ALIGNMENT_FILTER_PROMPT",
                prompt_vars={"goal_description": goal_desc, "use_cases_table": use_cases_table},
                response_schema=None,
                timeout_override=120,
                skip_cache=skip_cache
            )
            batch_ids = {str(uc.get("No", "")) for uc in batch}
            lines = [[p.strip().strip("\"'`*").strip() for p in re.split(r"[,|:\t]", line) if p.strip()]
                     for line in (response_raw or "").strip().split("\n")]
            rows, _ = CSVParser.aligned_rows(
                [parts for parts in lines if batch_ids & set(parts)], self.logger, f"Goal filter {batch_label}",
                header=["ID", "Aligned"], required=("ID", "Aligned"),
                anchors={"ID": CSVParser.one_of(batch_ids), "Aligned": CSVParser.one_of(("YES", "NO"))})
            verdicts = {}
            for parts in rows:
                if parts[0] in batch_ids and parts[1].upper() in ("YES", "NO"):
                    verdicts[parts[0]] = parts[1].upper()
            return verdicts

        per_task_timeout = self._scoring_task_timeout(120, base_prompt_size + avg_row * min(batch_size, len(use_cases)))
        verdicts, unverified = self._score_with_split_retry(use_cases, _classify, batch_size, "GoalFilter", per_task_timeout)

        filtered = [uc for uc in use_cases if verdicts.get(str(uc.get("No", ""))) == "YES"]
        removed = [uc for uc in use_cases if verdicts.get(str(uc.get("No", ""))) == "NO"]
        if removed:
            self.logger.info(f"\n🎯 GOAL FILTER RESULTS:")
            self.logger.info(f"   ✅ ALIGNED (kept): {len(filtered)} use cases")
            self.logger.info(f"   ❌ NOT ALIGNED (removed): {len(removed)} use cases")
            log_print(f"\n🎯 GOAL FILTER RESULTS:")
            log_print(f"   ✅ ALIGNED with goal: {len(filtered)} use cases kept")
            log_print(f"   ❌ NOT ALIGNED with goal: {len(removed)} use cases removed:")
            for uc in removed:
                _uc_no = uc.get('No', '?')
                _uc_name = uc.get('Name', '?')
                self.logger.info(f"      ❌ {_uc_no} - {_uc_name}")
                log_print(f"      ❌ {_uc_no} - {_uc_name}")
        if unverified:
            self.logger.error(
                f"⚠️ GOAL FILTER: {len(unverified)} use case(s) received no YES/NO verdict after retries and were removed "
                f"(unverified against the user goal): {', '.join(list(unverified)[:20])}{' ...' if len(unverified) > 20 else ''}"
            )
        if not removed and not unverified:
            self.logger.info(f"✅ All {len(filtered)} use cases are aligned with the goal")
            log_print(f"✅ All {len(filtered)} use cases are aligned with the goal")
        if not filtered:
            self.logger.error("🚨 GOAL FILTER: no use case is aligned with the user goal; none are kept.")
            log_print("🚨 GOAL FILTER: no use case is aligned with the user goal; none are kept.", level="ERROR")

        self._log_funnel_stage(stage, n_in, len(filtered), {"not_aligned": len(removed), "goal_unverified": len(unverified)})
        self.logger.info(f"{'=' * 80}\n")
        return filtered

    def _get_business_context_from_llm(self) -> dict:
        """
        Calls the BUSINESS_CONTEXT_WORKER_PROMPT to get comprehensive business context.
        Returns a dict with the business context fields. A failed or unparseable answer is retried
        once with the LLM cache bypassed before an empty context is returned (and logged).
        """
        self.logger.info(f"🔍 Calling Business Context Worker for: {self.business_name}")
        
        prompt_vars = {
            "name": self.business_name,
            "industry": CONTEXT_NOT_PROVIDED_MARKER,
            "type_description": "a business entity or organization",
            "type_label": "organization"
        }
        max_attempts = 2
        for attempt in range(1, max_attempts + 1):
            try:
                self.logger.info(f"⏳ Waiting for LLM response (Business Context extraction, attempt {attempt}/{max_attempts})...")
                response_json = self.ai_agent.run_worker(
                    step_name="Business_Context_Extraction",
                    worker_prompt_path="BUSINESS_CONTEXT_WORKER_PROMPT",
                    prompt_vars=prompt_vars,
                    response_schema=None,
                    skip_cache=(attempt > 1)
                ) or ""
                self.logger.info(f"✅ Received LLM response, parsing business context...")
                
                context_data = None
                strategies = [
                    ("clean_json", clean_json_response(response_json)),
                ]
                fence_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', response_json)
                if fence_match:
                    strategies.append(("markdown_fence", fence_match.group(1).strip()))
                last_brace = re.search(r'(\{[\s\S]*\})\s*$', response_json)
                if last_brace:
                    strategies.append(("last_json_object", last_brace.group(1)))
                strategies.append(("single_to_double", clean_json_response(response_json).replace("'", '"')))
                fixed_resp = re.sub(r',\s*}', '}', re.sub(r',\s*]', ']', re.sub(r"(?<!\\\\)'", '"', response_json)))
                strategies.append(("aggressive_fix", clean_json_response(fixed_resp)))
                for strat_name, candidate in strategies:
                    try:
                        result = json.loads(candidate)
                        if isinstance(result, dict):
                            context_data = result.get("data", result) if "data" in result else result
                            self.logger.info(f"Business context JSON parsed via strategy: {strat_name}")
                            break
                    except (json.JSONDecodeError, TypeError):
                        continue
                if isinstance(context_data, dict):
                    self.logger.info(f"✅ Business Context Worker extracted {len(context_data)} fields")
                    return label_inferred_business_context(context_data, self.business_name)
                self.logger.warning(f"All JSON strategies failed for business context (attempt {attempt}/{max_attempts}). Preview: {str(response_json)[:500]}")
            except Exception as e:
                self.logger.warning(f"Failed to get business context from LLM (attempt {attempt}/{max_attempts}): {get_clean_error_message(e)}")
        self.logger.error("Business context extraction failed after retries; continuing with an empty LLM business context.")
        return {}
    
    def _evidence_join_candidates(self, tables) -> list:
        wanted = {(c.lower(), s.lower(), t.lower()) for c, s, t in tables}
        out, seen = [], set()
        graph = getattr(getattr(self, "data_loader", None), "foreign_key_graph", None) or {}
        for source, relations in list(graph.items()):
            for rel in relations or []:
                if len(rel) < 8:
                    continue
                child = (str(rel[0]), str(rel[1]), str(rel[2]))
                parent = (str(rel[4]), str(rel[5]), str(rel[6]))
                key = (child, str(rel[3]).lower(), parent, str(rel[7]).lower())
                if key in seen or tuple(x.lower() for x in child) not in wanted or tuple(x.lower() for x in parent) not in wanted:
                    continue
                seen.add(key)
                out.append((child, str(rel[3]), parent, str(rel[7])))
        return out

    def _collect_discovery_evidence(self, business_details: list) -> dict:
        rt = TECHNICAL_CONTEXT["runtime"]
        tables = sorted({(str(r[0]), str(r[1]), str(r[2])) for r in business_details or []})
        fqns = [f"{c}.{s}.{t}" for c, s, t in tables]
        discovery_type = getattr(self, "discovery_type", "Standard")
        evidence = {"discovery_type": discovery_type, "metric_views": [], "usage": {}, "profiles": {},
                    "join_matches": [], "history_joins": [],
                    "data_access": {"discovery_type": discovery_type, "tables_read": [], "tables_read_count": 0,
                                    "statements": 0, "rows_sent_to_ai": 0}}
        try:
            evidence["metric_views"] = collect_metric_views(self.spark, {(c, s) for c, s, _t in tables}, self.logger)
        except Exception as error:
            self.logger.warning(f"Metric views unavailable: {get_clean_error_message(error, max_chars=200)}")
        evidence["usage"] = collect_usage_signals(self.spark, fqns, self.logger, days=rt["usage_signal_days"])
        self.deep_evidence = None
        if discovery_type == "Deep":
            deep = DeepEvidence(self.spark, discovery_type, self.logger)
            self.deep_evidence = deep
            columns_by_table = defaultdict(list)
            for row in business_details or []:
                columns_by_table[(str(row[0]), str(row[1]), str(row[2]))].append((row[3], row[4]))
            loader = getattr(self, "data_loader", None)
            try:
                for (c, s, t) in tables[:deep.max_tables]:
                    tags = {}
                    if loader is not None:
                        tags = getattr(loader, "_column_tags", {}).get(loader._lkey(c, s, t), {}) or {}
                    profile = deep.profile_table(build_fqn(c, s, t), f"{c}.{s}.{t}", columns_by_table[(c, s, t)], tags)
                    if profile is not None:
                        evidence["profiles"][f"{c}.{s}.{t}".lower()] = profile
                for child, child_col, parent, parent_col in self._evidence_join_candidates(tables)[:deep.max_joins]:
                    label = f"{'.'.join(child)}.{child_col}->{'.'.join(parent)}.{parent_col}"
                    try:
                        rate = deep.join_match_rate(build_fqn(*child), ".".join(child), child_col,
                                                    build_fqn(*parent), ".".join(parent), parent_col)
                    except (DeepEvidenceBudgetSpent, SparkWorkCancelled):
                        raise
                    except Exception as error:
                        self.logger.warning(f"[DEEP-EVIDENCE] join check {label} unavailable: "
                                            f"{get_clean_error_message(error, max_chars=200)}")
                        continue
                    evidence["join_matches"].append({"child_table": ".".join(child).lower(), "child_column": child_col,
                                                     "parent_table": ".".join(parent).lower(), "parent_column": parent_col,
                                                     "rate": rate})
                try:
                    statements = deep.query_history_statements(sorted({c for c, _s, _t in tables}))
                    evidence["history_joins"] = mine_history_join_edges(statements, fqns)[:200]
                    del statements
                except (DeepEvidenceBudgetSpent, SparkWorkCancelled):
                    raise
                except Exception as error:
                    self.logger.info(f"Query history unavailable: {get_clean_error_message(error, max_chars=200)}")
            except DeepEvidenceBudgetSpent as spent:
                self.logger.warning(f"[DEEP-EVIDENCE] {spent}; the remaining tables stay Standard")
            evidence["data_access"] = deep.summary()
        self.discovery_evidence = evidence
        usage = evidence["usage"]
        self.logger.info(
            f"[EVIDENCE FIRED] discovery_type={discovery_type} metric_views={len(evidence['metric_views'])} "
            f"usage={'yes' if usage.get('available') else 'no (' + str(usage.get('reason') or '') + ')'} "
            f"profiles={len(evidence['profiles'])} join_checks={len(evidence['join_matches'])} "
            f"history_joins={len(evidence['history_joins'])} tables_read={evidence['data_access'].get('tables_read_count', 0)}")
        return evidence

    def _usage_kept_master_details(self, master_details: list, business_details: list) -> list:
        usage = (getattr(self, "discovery_evidence", None) or {}).get("usage") or {}
        reads = {str(k).lower(): int(v or 0) for k, v in (usage.get("reads") or {}).items()}
        if not usage.get("available") or not reads:
            return []
        all_tables = sorted({f"{r[0]}.{r[1]}.{r[2]}".lower() for r in business_details or []})
        counts = sorted((reads.get(t, 0) for t in all_tables), reverse=True)
        share = TECHNICAL_CONTEXT["runtime"]["usage_election_keep_share"]
        if not counts or share <= 0:
            return []
        threshold = counts[max(0, min(len(counts) - 1, int(len(counts) * share) - 1))]
        kept_tables = {f"{r[0]}.{r[1]}.{r[2]}".lower() for r in master_details
                       if reads.get(f"{r[0]}.{r[1]}.{r[2]}".lower(), 0) >= max(1, threshold)}
        if not kept_tables:
            return []
        self.logger.info(f"[USAGE-ELECTION FIRED] kept {len(kept_tables)} heavily read master table(s) that "
                         f"transactional-only election would drop: {sorted(kept_tables)[:10]}")
        return [r for r in master_details if f"{r[0]}.{r[1]}.{r[2]}".lower() in kept_tables]

    def _evidence_prompt_block(self, table_fqns, max_chars: int = 4000) -> tuple:
        lines, joins = self._evidence_lines(table_fqns)
        block = ""
        if lines:
            block = ("\n\n### DATA EVIDENCE (facts about the data, not goals; use them to judge feasibility)\n"
                     + "\n".join(lines))[:max_chars]
        return block, joins[:40]

    def _evidence_lines(self, table_fqns) -> tuple:
        evidence = getattr(self, "discovery_evidence", None) or {}
        if not evidence:
            return [], []
        wanted = {str(t).replace("`", "").lower() for t in table_fqns or []}
        usage = evidence.get("usage") or {}
        lines, joins = [], []
        for fqn in sorted(wanted):
            parts = []
            reads = {k.lower(): v for k, v in (usage.get("reads") or {}).items()}.get(fqn)
            if reads:
                readers = {k.lower(): v for k, v in (usage.get("readers") or {}).items()}.get(fqn, 0)
                parts.append(f"read {reads} times by {readers} principals in the last {TECHNICAL_CONTEXT['runtime']['usage_signal_days']} days")
            profile = (evidence.get("profiles") or {}).get(fqn)
            if profile is not None:
                parts.append(f"{profile.get('rows', 0)} rows")
                for column, info in sorted((profile.get("columns") or {}).items()):
                    if info.get("min") and info.get("max"):
                        parts.append(f"`{column}` spans {info['min'][:10]} to {info['max'][:10]}")
                    if info.get("null_share") is not None and info["null_share"] >= TECHNICAL_CONTEXT["runtime"]["deep_mostly_null_share"]:
                        parts.append(f"`{column}` is {round(info['null_share'] * 100)}% null")
                    if info.get("top_values"):
                        values = ", ".join(f"{v['value']} {round(v['share'] * 100)}%" for v in info["top_values"][:6])
                        parts.append(f"`{column}` values: {values}")
            if parts:
                lines.append(f"- `{fqn}`: " + "; ".join(parts))
        for view in evidence.get("metric_views") or []:
            if str(view.get("source") or "").replace("`", "").lower() in wanted and view.get("measures"):
                measures = ", ".join(m["name"] for m in view["measures"][:8])
                lines.append(f"- metric view `{view['fqn']}` on `{view['source']}` measures: {measures}")
        for pair in usage.get("co_used") or []:
            if pair["left"].lower() in wanted and pair["right"].lower() in wanted:
                joins.append(f"[OBSERVED CO-USE] `{pair['left']}` and `{pair['right']}` are read together in {pair['runs']} runs")
        for edge in evidence.get("history_joins") or []:
            left_table = edge["left"].rsplit(".", 1)[0].lower()
            right_table = edge["right"].rsplit(".", 1)[0].lower()
            if left_table in wanted and right_table in wanted:
                joins.append(f"[OBSERVED JOIN] `{edge['left']}` = `{edge['right']}` (in {edge['count']} recent queries)")
        for match in evidence.get("join_matches") or []:
            if match["child_table"] in wanted and match["parent_table"] in wanted and match.get("rate") is not None:
                joins.append(f"[MEASURED JOIN] `{match['child_table']}.{match['child_column']}` -> "
                             f"`{match['parent_table']}.{match['parent_column']}` matches {round(match['rate'] * 100)}% of rows")
        return lines, joins

    def _resolve_strategy_tree(self, merged_business_context: dict) -> dict:
        customer = getattr(self, "customer_strategy", None)
        if customer and customer.get("strategy"):
            self.strategy_tree = customer
            entries = sum(1 for _node in iter_strategy_entries(customer))
            self.logger.info(f"✅ Strategic Context: {entries} strategy entries on levels {customer.get('levels')}; "
                             f"applied after selection only, never used to find or grade use cases")
        else:
            self.strategy_tree = build_builtin_strategy((merged_business_context or {}).get("strategic_goals") or [])
            if customer:
                for key in ("priorities", "sponsors", "labels"):
                    self.strategy_tree[key] = customer.get(key) or self.strategy_tree.get(key)
            self.logger.info(f"ℹ️ Strategy tree: built-in, {len(self.strategy_tree['strategy'])} strategic goal(s); "
                             f"applied after selection only")
        for warning in (customer or {}).get("warnings") or []:
            self.logger.warning(f"Strategic Context: {warning}")
        return self.strategy_tree

    _BASELINE_PLACEHOLDER_RE = re.compile(r"\[BASELINE(?::K(\d+))?\]", re.IGNORECASE)
    _BASELINE_PENDING = "[BASELINE: measured by the KPI query]"

    def _alignment_context(self, tree: dict) -> dict:
        tree = tree or {}
        evidence = getattr(self, "discovery_evidence", None) or {}
        index = self._gate_schema_index()
        priorities = list(tree.get("priorities") or [])
        if not priorities:
            raw = (getattr(self, "merged_business_context", None) or {}).get("business_priorities") or []
            priorities = [p for p in (raw.split(",") if isinstance(raw, str) else raw)]
        priorities = [str(p).strip() for p in priorities if str(p or "").strip()]
        user_text = " ".join((json.dumps(tree, ensure_ascii=False), str(getattr(self, "user_generation_instructions", "") or ""),
                              " ".join(priorities)))
        return {"tree": tree, "catalog": strategy_catalog(tree), "tables": index["tables"], "originals": index["originals"],
                "metric_views": {str(mv.get("fqn", "")).lower(): mv for mv in evidence.get("metric_views") or []},
                "profiles": evidence.get("profiles") or {}, "priorities": priorities,
                "sponsors": {s["id"]: s for s in tree.get("sponsors") or []},
                "user_figures": {m.group(0).strip().lower() for m in NUMERIC_CLAIM_RE.finditer(user_text)},
                "sponsor_picks": {}, "priority_hits": {}}

    def _alignment_use_case_tables(self, uc: dict, context: dict) -> dict:
        return {ref: context["tables"][ref] for ref in self._uc_table_refs(uc) if ref in context["tables"]}

    def _alignment_payload(self, uc: dict, context: dict) -> dict:
        tables = self._alignment_use_case_tables(uc, context)
        lines, joins = self._evidence_lines(list(tables))
        return {"use_case_id": str(uc.get("No", "")), "name": uc.get("Name", ""), "domain": uc.get("Business Domain", ""),
                "technique": uc.get("Analytics Technique", ""), "statement": str(uc.get("Statement") or "")[:600],
                "solution": str(uc.get("Solution") or "")[:800], "business_value": str(uc.get("Business Value") or "")[:600],
                "sponsor": uc.get("Sponsor", ""),
                "tables": {ref: [f"{name} {dtype.lower()}".strip() for name, dtype in list(cols.items())[:60]]
                           for ref, cols in list(tables.items())[:8]},
                "evidence": (lines + joins)[:12]}

    @staticmethod
    def _alignment_metric_views(context: dict, tables) -> str:
        wanted = {str(t).lower() for t in tables}
        rows = []
        for view in context["metric_views"].values():
            if str(view.get("source") or "").replace("`", "").lower() in wanted and view.get("measures"):
                rows.append(f"- `{view['fqn']}` on `{view['source']}`: "
                            + ", ".join(f"{m['name']} = {m['expr']}" for m in view["measures"][:12]))
        return "\n".join(rows) or "(none for these use cases)"

    @staticmethod
    def _sponsor_roster_lines(tree: dict) -> str:
        rows = []
        for sponsor in (tree or {}).get("sponsors") or []:
            extras = ([f"reports to {sponsor['reports_to']}"] if sponsor.get("reports_to") else []) + \
                     (["owns " + ", ".join(sponsor["owns"])] if sponsor.get("owns") else [])
            rows.append(f"- {sponsor['id']} {sponsor['role']}" + (f" ({'; '.join(extras)})" if extras else ""))
        return "\n".join(rows) or "(no roster: set sponsor_id to null)"

    def _resolve_kpi_spec(self, raw, uc_tables: dict, context: dict, notes: list):
        if not isinstance(raw, dict):
            return None
        kpi_id = str(raw.get("kpi_id") or "").strip().upper() or None
        strategy_kpi = context["catalog"]["kpis"].get(kpi_id) if kpi_id else None
        if kpi_id and strategy_kpi is None:
            notes.append(f"unknown KPI id {kpi_id}")
            kpi_id = None
        name = (strategy_kpi or {}).get("name") or str(raw.get("name") or "").strip()
        if not name:
            return None
        direction = str(raw.get("direction") or "").strip().lower()
        if direction not in KPI_DIRECTIONS:
            direction = (strategy_kpi or {}).get("direction") or ""
        spec = {"name": name[:200], "kpi_id": kpi_id, "direction": direction,
                "unit": str(raw.get("unit") or (strategy_kpi or {}).get("unit") or "")[:40],
                "metric_view": None, "measure": None, "aggregation": None, "table": None, "column": None,
                "filter_column": None, "filter_value": None, "time_column": None, "window_days": None,
                "denominator": None, "computable": False, "query": None, "baseline": None}
        view_ref = str(raw.get("metric_view") or "").replace("`", "").strip().lower()
        if view_ref:
            view = context["metric_views"].get(view_ref)
            measures = {str(m.get("name", "")).lower(): m["name"] for m in (view or {}).get("measures") or []}
            measure = str(raw.get("measure") or "").strip().lower()
            if view is not None and measure in measures:
                spec.update(metric_view=view["fqn"], measure=measures[measure], computable=True)
            else:
                notes.append(f"KPI {name!r}: measure {raw.get('measure')!r} of {raw.get('metric_view')!r} does not exist")
        elif raw.get("aggregation"):
            table = self._normalize_table_ref(raw.get("table"))
            columns = uc_tables.get(table)
            column = str(raw.get("column") or "").replace("`", "").strip().split(".")[-1]
            try:
                aggregation = DeepEvidence.canonical_aggregate(raw.get("aggregation"))
            except ValueError as error:
                notes.append(f"KPI {name!r}: {error}")
                aggregation = None
            column_type = self._kpi_non_numeric_type(aggregation, column, columns)
            if columns is None:
                notes.append(f"KPI {name!r}: table {raw.get('table')!r} is not one of the use case's tables")
            elif aggregation in ("MIN", "MAX") and column and self._column_is_personal(context["originals"].get(table), column):
                notes.append(f"KPI {name!r}: {aggregation} of the personal column {column!r} is never measured")
            elif column_type:
                notes.append(f"KPI {name!r}: {aggregation} of the {column_type} column {column!r} is not a number")
            elif aggregation and ((column and column.lower() in columns) or (not column and aggregation == "COUNT")):
                spec.update(aggregation=aggregation, table=".".join(context["originals"].get(table, ())) or table,
                            column=column or None, computable=True)
                spec["filter_column"], spec["filter_value"], filter_ok = self._kpi_filter(raw, table, columns, context, name, notes)
                if not filter_ok:
                    spec["computable"] = False
                if isinstance(raw.get("denominator"), dict):
                    spec["denominator"] = self._kpi_denominator(raw["denominator"], table, columns, context, name, notes)
                    if spec["denominator"] is None:
                        spec["computable"] = False
                    elif self._kpi_ratio_is_constant(spec):
                        notes.append(f"KPI {name!r}: the numerator equals the denominator "
                                     f"({self._kpi_computed_label(dict(spec, denominator=None))}), so the ratio never changes")
                        spec["computable"] = False
                time_column = str(raw.get("time_column") or "").replace("`", "").strip().split(".")[-1]
                if time_column and str(columns.get(time_column.lower(), "")).startswith(("DATE", "TIMESTAMP")):
                    try:
                        window = int(raw.get("window_days"))
                    except (TypeError, ValueError):
                        window = None
                    if window and 1 <= window <= 3650:
                        spec.update(time_column=time_column, window_days=window)
            else:
                notes.append(f"KPI {name!r}: column {raw.get('column')!r} does not exist in {raw.get('table')!r}")
        if spec["kpi_id"] and spec["computable"] and not self._kpi_link_measures(strategy_kpi, spec):
            notes.append(f"KPI {spec['kpi_id']} unlinked: {spec.get('measure') or spec.get('column') or spec.get('table')} "
                         f"cannot measure {strategy_kpi['name']!r}")
            source = spec.get("measure") or spec.get("column") or "rows"
            spec.update(kpi_id=None, name=f"{(spec.get('aggregation') or 'value').lower()} of {source}"[:200], unit="")
        if spec["computable"] and not self._kpi_name_is_honest(spec):
            computed = self._kpi_computed_label(spec)
            if spec["kpi_id"]:
                notes.append(f"KPI {spec['kpi_id']} kept as a proxy: {computed} does not produce {spec['name']!r} itself")
                spec.update(name=f"{spec['name']} (proxy: {computed})"[:200], unit="")
            else:
                notes.append(f"KPI {spec['name']!r} renamed: {computed} does not produce what the name says")
                spec.update(name=computed[:200], unit="")
        if spec["computable"]:
            spec["query"] = DeepEvidence.kpi_sql(spec)
        return spec

    _PHYSICAL_KPI_FAMILIES = (
        (re.compile(r"\b(co2e?|ghg|carbon|emissions?)\b", re.I), {"co2", "co2e", "ghg", "carbon", "emission", "emissions"}),
        (re.compile(r"\b(kwh|mwh|gwh|wh|kj|mj|joules?|btu|therms?|energy)\b", re.I),
         {"kwh", "mwh", "gwh", "wh", "energy", "joule", "joules", "kj", "mj", "btu", "therm", "therms", "power", "fuel"}),
        (re.compile(r"\b(kg|kgs|kilograms?|grams?|tonnes?|tons?|lbs?|weight)\b", re.I),
         {"kg", "kgs", "kilogram", "kilograms", "gram", "grams", "weight", "mass", "tonne", "tonnes", "ton", "tons", "lb", "lbs"}),
        (re.compile(r"\b(liters?|litres?|gallons?|m3|cubic)\b", re.I),
         {"liter", "liters", "litre", "litres", "gallon", "gallons", "m3", "cubic"}),
        (re.compile(r"\b(km|kilomet(er|re)s?|miles?|distance)\b", re.I),
         {"km", "kms", "kilometer", "kilometers", "kilometre", "kilometres", "mile", "miles", "distance"}),
        (re.compile(r"\b(celsius|fahrenheit|kelvin|temperature)\b", re.I), {"temp", "temperature", "celsius", "fahrenheit", "kelvin"}),
    )

    @classmethod
    def _kpi_link_measures(cls, strategy_kpi: dict, spec: dict) -> bool:
        kpi_text = " ".join(str(strategy_kpi.get(key) or "") for key in ("name", "unit", "definition"))
        source = " ".join(str(spec.get(key) or "") for key in ("metric_view", "measure", "column", "filter_column")).lower()
        tokens = set(re.split(r"[^a-z0-9]+", source))
        return all(tokens & words for kpi_re, words in cls._PHYSICAL_KPI_FAMILIES if kpi_re.search(kpi_text))

    def _kpi_filter(self, raw: dict, table: str, columns: dict, context: dict, name: str, notes: list) -> tuple:
        filter_column = str(raw.get("filter_column") or "").replace("`", "").strip().split(".")[-1]
        filter_value = raw.get("filter_value")
        if not filter_column or filter_value is None:
            return None, None, True
        if filter_column.lower() not in columns:
            notes.append(f"KPI {name!r}: filter column {filter_column!r} does not exist in {table!r}")
            return None, None, False
        profile_columns = {k.lower(): v for k, v in ((context["profiles"].get(table) or {}).get("columns") or {}).items()}
        info = profile_columns.get(filter_column.lower()) or {}
        observed = {str(v.get("value")).casefold(): str(v.get("value")) for v in info.get("top_values") or []
                    if v.get("value") is not None}
        if observed and 0 < (info.get("distinct") or 0) <= DEEP_TOP_VALUES:
            match = observed.get(str(filter_value).casefold())
            if match is None:
                notes.append(f"KPI {name!r}: filter {filter_column}={filter_value!r} matches none of the values "
                             f"the data evidence lists for that column")
                return None, None, False
            return filter_column, match, True
        return filter_column, str(filter_value), True

    def _kpi_denominator(self, raw: dict, table: str, columns: dict, context: dict, name: str, notes: list):
        try:
            aggregation = DeepEvidence.canonical_aggregate(raw.get("aggregation"))
        except ValueError as error:
            notes.append(f"KPI {name!r}: denominator {error}")
            return None
        column = str(raw.get("column") or "").replace("`", "").strip().split(".")[-1]
        if not ((column and column.lower() in columns) or (not column and aggregation == "COUNT")):
            notes.append(f"KPI {name!r}: denominator column {raw.get('column')!r} does not exist in {table!r}")
            return None
        if aggregation in ("MIN", "MAX") and column and self._column_is_personal(context["originals"].get(table), column):
            notes.append(f"KPI {name!r}: {aggregation} of the personal column {column!r} is never measured")
            return None
        column_type = self._kpi_non_numeric_type(aggregation, column, columns)
        if column_type:
            notes.append(f"KPI {name!r}: denominator {aggregation} of the {column_type} column {column!r} is not a number")
            return None
        filter_column, filter_value, filter_ok = self._kpi_filter(raw, table, columns, context, name, notes)
        if not filter_ok:
            return None
        return {"aggregation": aggregation, "column": column or None, "filter_column": filter_column, "filter_value": filter_value}

    @staticmethod
    def _kpi_non_numeric_type(aggregation, column, columns) -> str:
        column_type = str((columns or {}).get(str(column or "").lower()) or "")
        if aggregation in DEEP_KPI_VALUE_AGGREGATES and column_type and not _NUMERIC_TYPE_RE.match(column_type):
            return column_type
        return ""

    _KPI_RATE_RE = re.compile(r"\b(rate|ratio|share|percent|percentage|proportion|per)\b|%", re.I)
    _KPI_MODEL_OUTPUT_RE = re.compile(r"\b(flagged|anomalous|anomaly|anomalies|predicted|forecasted|at[- ]risk|likely|detected|suspicious)\b", re.I)
    _KPI_LABEL_TOKENS = {"flag", "flagged", "anomaly", "anomalous", "pred", "prediction", "predicted", "forecast", "risk",
                         "score", "fraud", "label", "status", "churn", "churned", "suspicious", "detected"}

    _KPI_PERCENT_UNITS = ("%", "percent", "pct", "percentage")
    _KPI_SHARE_RE = re.compile(r"\b(share|percent|percentage|proportion)\b|%", re.I)
    _KPI_SHARE_AGGREGATIONS = {("COUNT", "COUNT"), ("COUNT_DISTINCT", "COUNT"), ("COUNT_DISTINCT", "COUNT_DISTINCT"),
                               ("SUM", "SUM")}

    @staticmethod
    def _kpi_ratio_is_constant(spec: dict) -> bool:
        denominator = spec.get("denominator") or {}
        return bool(denominator) and all(spec.get(key) == denominator.get(key)
                                         for key in ("aggregation", "column", "filter_column", "filter_value"))

    @staticmethod
    def _kpi_computed_label(spec: dict) -> str:
        label = f"{spec['aggregation'].lower()} of {spec.get('column') or 'rows'}"
        denominator = spec.get("denominator")
        if denominator:
            label += f" per {denominator['aggregation'].lower()} of {denominator.get('column') or 'rows'}"
        return label

    @classmethod
    def _kpi_name_is_honest(cls, spec: dict) -> bool:
        if spec.get("metric_view"):
            return True
        if cls._KPI_RATE_RE.search(f"{spec['name']} {spec.get('unit') or ''}") and not spec.get("denominator") \
                and spec.get("aggregation") != "AVG":
            return False
        denominator = spec.get("denominator")
        if denominator and (str(spec.get("unit") or "").strip().lower() in cls._KPI_PERCENT_UNITS
                            or cls._KPI_SHARE_RE.search(spec["name"])) \
                and (spec.get("aggregation"), denominator.get("aggregation")) not in cls._KPI_SHARE_AGGREGATIONS:
            return False
        if cls._KPI_MODEL_OUTPUT_RE.search(spec["name"]):
            filters = [spec.get("filter_column"), (spec.get("denominator") or {}).get("filter_column")]
            tokens = {token for value in filters for token in re.split(r"[^a-z0-9]+", str(value or "").lower())}
            return bool(tokens & cls._KPI_LABEL_TOKENS)
        return True

    def _column_is_personal(self, table_parts, column) -> bool:
        """The same PII rule the Deep profiler applies: a personal column name or a PII tag."""
        loader = getattr(self, "data_loader", None)
        tags = None
        if loader is not None and table_parts and len(table_parts) == 3 and hasattr(loader, "_lkey"):
            table_tags = getattr(loader, "_column_tags", {}).get(loader._lkey(*table_parts), {}) or {}
            tags = {str(key).lower(): value for key, value in table_tags.items()}.get(str(column).lower())
        return deep_column_is_pii(column, tags)

    def _scrub_unsourced_figures(self, text: str, context: dict, column: str) -> tuple:
        scrubbed = []

        def _replace(match):
            figure = match.group(0)
            if figure.strip().lower() in context["user_figures"]:
                return figure
            scrubbed.append(figure)
            return f"[ESTIMATE: depends on {column}]"
        return NUMERIC_CLAIM_RE.sub(_replace, str(text or "")), len(scrubbed)

    def _validate_alignment_record(self, item: dict, uc: dict, context: dict):
        """The validated alignment of one use case, or None when the answer misses a required part."""
        catalog = context["catalog"]
        uc_tables = self._alignment_use_case_tables(uc, context)
        uc_columns = {column for columns in uc_tables.values() for column in columns}
        notes = []
        kpis = [spec for spec in (self._resolve_kpi_spec(raw, uc_tables, context, notes)
                                  for raw in (item.get("kpis") or [])[:3]) if spec is not None]
        kpi_names = {spec["name"].lower() for spec in kpis} | {
            catalog["kpis"][spec["kpi_id"]]["name"].lower() for spec in kpis if spec.get("kpi_id") in catalog["kpis"]}
        alignments = []
        for raw in item.get("alignments") or []:
            if not isinstance(raw, dict):
                continue
            entry = catalog["entries"].get(str(raw.get("id") or "").strip().upper())
            evidence = str(raw.get("evidence") or "").strip()
            if entry is None:
                notes.append(f"unknown strategy id {raw.get('id')!r} dropped")
                continue
            if any(a["id"] == entry["id"] for a in alignments):
                continue
            if not self._grounded_column_mentions(evidence, uc_columns) and not any(n in evidence.lower() for n in kpi_names):
                notes.append(f"alignment {entry['id']} dropped: its evidence cites none of the use case's columns or KPIs")
                continue
            alignments.append({"id": entry["id"], "path": entry["path"], "evidence": evidence})
            if len(alignments) >= STRATEGY_MAX_ALIGNMENTS:
                break
        try:
            score = float(item.get("strategic_alignment"))
        except (TypeError, ValueError):
            score = None
        if alignments and (score is None or score != score):
            return None
        lookup = {p.casefold(): p for p in context["priorities"]}
        priorities = []
        for raw in item.get("priority_alignment") or []:
            match = lookup.get(str(raw).strip().casefold())
            if match and match not in priorities:
                priorities.append(match)
        allowed = {a["id"] for a in alignments} | {anc for a in alignments for anc in catalog["entries"][a["id"]]["ancestors"]}
        key_results = []
        for raw in item.get("key_result_ids") or []:
            key_result = catalog["key_results"].get(str(raw).strip().upper())
            if key_result is not None and key_result["entry_id"] in allowed and \
                    not any(kr.get("id") == key_result["id"] for kr in key_results):
                key_results.append({"id": key_result["id"], "statement": key_result["statement"],
                                    "kpi_id": key_result.get("kpi_id"), "source": "strategy"})
        column = next((f"{spec['table']}.{spec['column']}" for spec in kpis if spec.get("table") and spec.get("column")),
                      self._estimate_target(uc))
        scrubbed = 0

        def _texts(values, limit):
            nonlocal scrubbed
            out = []
            for value in (values if isinstance(values, list) else [values])[:limit]:
                text, count = self._scrub_unsourced_figures(str(value or "").strip(), context, column)
                scrubbed += count
                if text:
                    out.append(text)
            return out
        for proposal in _texts(item.get("proposed_key_results") or [], 2):
            key_results.append({"id": None, "statement": proposal, "kpi_id": None, "source": "proposed"})
        for alignment in alignments:
            alignment["evidence"], count = self._scrub_unsourced_figures(alignment["evidence"], context, column)
            scrubbed += count
        measures = _texts(item.get("measures_of_success") or [], 3)
        financial = _texts(item.get("financial_benefits") or [], 3)
        non_financial = _texts(item.get("non_financial_benefits") or [], 3)
        if not kpis or not financial or not non_financial:
            return None
        sponsor_id = str(item.get("sponsor_id") or "").strip().upper() or None
        return {"alignments": alignments, "priorities": priorities,
                "strategic_alignment": round(max(1.0, min(5.0, score)), 2) if alignments else 1.0,
                "kpis": kpis, "key_results": key_results, "measures": measures, "financial": financial,
                "non_financial": non_financial,
                "sponsor_id": sponsor_id if sponsor_id in context["sponsors"] else None,
                "notes": notes, "scrubbed": scrubbed, "fallback": False}

    def _estimate_target(self, uc: dict) -> str:
        primary = str(uc.get("Primary Table") or self._extract_primary_table(uc.get("Tables Involved", "")) or "").strip()
        return primary if primary and primary != "N/A" else "the cited tables"

    def _fallback_alignment(self, uc: dict, reason: str) -> dict:
        estimate = f"[ESTIMATE: depends on {self._estimate_target(uc)}]"
        assessment = uc.get("elite_assessment") if isinstance(uc.get("elite_assessment"), dict) else {}
        kpi = {"name": str(assessment.get("kpi") or estimate)[:200], "kpi_id": None,
               "direction": assessment.get("kpi_direction") if assessment.get("kpi_direction") in KPI_DIRECTIONS else "",
               "unit": "", "metric_view": None, "measure": None, "aggregation": None, "table": None, "column": None,
               "filter_column": None, "filter_value": None, "time_column": None, "window_days": None,
               "computable": False, "query": None, "baseline": None}
        return {"alignments": [], "priorities": [], "strategic_alignment": 1.0, "kpis": [kpi], "key_results": [],
                "measures": [estimate], "financial": [estimate], "non_financial": [estimate], "sponsor_id": None,
                "notes": [reason], "scrubbed": 0, "fallback": True}

    def _parse_alignment_response(self, response_raw: str, uc_batch: list, context: dict) -> dict:
        by_id = {str(uc.get("No", "")): uc for uc in uc_batch}
        records = {}
        for item in self._parse_json_array_items(response_raw, "strategy alignment"):
            if not isinstance(item, dict):
                continue
            uc_id = str(item.get("use_case_id", "") or "").strip()
            if uc_id in by_id and uc_id not in records:
                record = self._validate_alignment_record(item, by_id[uc_id], context)
                if record is not None:
                    records[uc_id] = record
        return records

    @staticmethod
    def _format_baseline(value: float) -> str:
        return f"{int(round(value)):,}" if float(value).is_integer() or abs(value) >= 1000 else f"{value:,.2f}"

    def _measure_kpi_baselines(self, use_cases: list) -> dict:
        deep = getattr(self, "deep_evidence", None)
        per_use_case = [[spec for spec in uc.get("KPIs") or [] if spec.get("computable") and spec.get("query")] for uc in use_cases]
        depth = max((len(specs) for specs in per_use_case), default=0)
        specs = [specs[i] for i in range(depth) for specs in per_use_case if i < len(specs)]
        stats = {"computable": len(specs), "measured": 0, "unmeasured": 0, "skipped": 0}
        if deep is None or not specs:
            return stats
        measured, measured_on = {}, datetime.date.today().isoformat()
        for spec in specs:
            if spec["query"] not in measured:
                if len(measured) >= deep.max_kpi_baselines:
                    stats["skipped"] += 1
                    continue
                try:
                    value = deep.measure_kpi(spec)
                except DeepEvidenceBudgetSpent as spent:
                    self.logger.warning(f"[STRATEGY-ENRICH] {spent}; the remaining KPI baselines stay as queries")
                    break
                except SparkWorkCancelled:
                    raise
                except Exception as error:
                    self.logger.warning(f"[STRATEGY-ENRICH] KPI baseline for {spec['name']!r} unavailable: "
                                        f"{get_clean_error_message(error, max_chars=200)}")
                    value = None
                measured[spec["query"]] = None if value is None else {
                    "value": value, "window_days": spec.get("window_days"), "measured_on": measured_on}
            spec["baseline"] = dict(measured[spec["query"]]) if measured[spec["query"]] else None
            stats["measured" if spec["baseline"] else "unmeasured"] += 1
        if stats["skipped"]:
            self.logger.warning(f"[STRATEGY-ENRICH] {stats['skipped']} KPI baseline(s) were not measured: the cap of {deep.max_kpi_baselines} "
                                "distinct KPI queries was reached (runtime key deep_max_kpi_baselines); they stay as queries")
        return stats

    def _fill_baseline_placeholders(self, uc: dict):
        kpis = uc.get("KPIs") or []

        def _baseline_text(index):
            spec = kpis[index] if 0 <= index < len(kpis) else None
            baseline = (spec or {}).get("baseline")
            if not baseline:
                return self._BASELINE_PENDING
            window = f", last {baseline['window_days']} days" if baseline.get("window_days") else ""
            return f"{self._format_baseline(baseline['value'])} ({spec['name']}{window}, measured {baseline['measured_on']})"

        names = sorted(((str(spec.get("name") or "").strip().lower(), index) for index, spec in enumerate(kpis)
                        if str(spec.get("name") or "").strip()), key=lambda item: -len(item[0]))

        def _named_index(text):
            lowered = str(text).lower()
            return next((index for name, index in names if re.search(r"(?<!\w)" + re.escape(name) + r"(?!\w)", lowered)), None)

        def _fill(text, default_index):
            named = _named_index(text)
            index = named if named is not None else default_index
            return self._BASELINE_PLACEHOLDER_RE.sub(
                lambda m: _baseline_text(int(m.group(1)) - 1 if m.group(1) else (-1 if index is None else index)), str(text))
        uc["Financial Benefits"] = [_fill(text, 0) for text in uc.get("Financial Benefits") or []]
        uc["Non-Financial Benefits"] = [_fill(text, 0) for text in uc.get("Non-Financial Benefits") or []]
        uc["Measures of Success"] = [_fill(text, 0) for text in uc.get("Measures of Success") or []]
        for key_result in uc.get("Expected Key Results") or []:
            if key_result.get("source") == "proposed":
                key_result["statement"] = _fill(key_result["statement"], 0 if len(kpis) == 1 else None)

    def _use_case_data_evidence(self, uc: dict):
        evidence = getattr(self, "discovery_evidence", None) or {}
        profiles = evidence.get("profiles") or {}
        refs = self._uc_table_refs(uc)
        tables = {}
        for ref in refs:
            profile = profiles.get(ref)
            if profile is None:
                continue
            dates = {column: {"min": info["min"], "max": info["max"]} for column, info in (profile.get("columns") or {}).items()
                     if info.get("min") and info.get("max")}
            tables[ref] = {"rows": profile.get("rows"), "sampled": bool(profile.get("sampled")), "date_ranges": dates}
        joins = [dict(m) for m in evidence.get("join_matches") or [] if m["child_table"] in refs and m["parent_table"] in refs]
        return {"tables": tables, "joins": joins} if tables or joins else None

    @staticmethod
    def _unmeasurable_entries(records: dict, context: dict) -> list:
        entry_kpis = defaultdict(set)
        for kpi_id, kpi in context["catalog"]["kpis"].items():
            entry_kpis[kpi["entry_id"]].add(kpi_id)
        aligned, measured = set(), set()
        for record in records.values():
            computable = {spec.get("kpi_id") for spec in record["kpis"] if spec.get("kpi_id") and spec.get("computable")}
            for alignment in record["alignments"]:
                if entry_kpis.get(alignment["id"]):
                    aligned.add(alignment["id"])
                    if entry_kpis[alignment["id"]] & computable:
                        measured.add(alignment["id"])
        return sorted(aligned - measured)

    @staticmethod
    def _strategy_without(tree: dict, entry_ids: set) -> dict:
        def _keep(nodes):
            return [dict(node, children=_keep(node.get("children"))) for node in nodes or [] if node["id"] not in entry_ids]
        return dict(tree, strategy=_keep(tree.get("strategy")))

    @staticmethod
    def _strip_alignments(records: dict, entry_ids: set) -> int:
        removed = 0
        for record in records.values():
            kept = [a for a in record["alignments"] if a["id"] not in entry_ids]
            if len(kept) != len(record["alignments"]):
                removed += len(record["alignments"]) - len(kept)
                record["alignments"] = kept
                record["notes"].append("alignments to entries the data cannot measure were removed")
                if not kept:
                    record["strategic_alignment"] = 1.0
        return removed

    def _apply_alignment_record(self, uc: dict, record: dict, context: dict):
        builtin = context["tree"].get("source") != "customer"
        paths = [a["path"] for a in record["alignments"]]
        uc["Strategy Alignment"] = record["alignments"]
        uc["Strategy Path"] = "; ".join(paths) or ("" if builtin else STRATEGY_NOT_ALIGNED)
        boost = STRATEGY_PRIORITY_BOOST * min(STRATEGY_PRIORITY_BOOST_MAX_MATCHES, len(record["priorities"]))
        uc["Strategic Alignment"] = round(min(5.0, record["strategic_alignment"] + boost), 2)
        uc["Strategic Goals Alignment"] = ", ".join(paths) or (STRATEGY_BUILTIN_NO_MATCH if builtin else STRATEGY_NOT_ALIGNED)
        uc["Business Priority Alignment"] = ", ".join(record["priorities"]) or STRATEGY_BUILTIN_NO_MATCH
        uc["KPIs"] = record["kpis"]
        uc["Expected Key Results"] = record["key_results"]
        uc["Measures of Success"] = record["measures"]
        uc["Financial Benefits"] = record["financial"]
        uc["Non-Financial Benefits"] = record["non_financial"]
        uc["Data Evidence"] = self._use_case_data_evidence(uc) if getattr(self, "deep_evidence", None) is not None else None
        context["priority_hits"][str(uc.get("No", ""))] = list(record["priorities"])
        if record.get("sponsor_id"):
            context["sponsor_picks"][str(uc.get("No", ""))] = record["sponsor_id"]

    def _strategy_coverage_report(self, use_cases: list, context: dict) -> dict:
        catalog = context["catalog"]
        direct, rolled = Counter(), Counter()
        for uc in use_cases:
            reached = set()
            for alignment in uc.get("Strategy Alignment") or []:
                entry = catalog["entries"].get(alignment["id"])
                if entry is not None:
                    direct[entry["id"]] += 1
                    reached.update([entry["id"]] + entry["ancestors"])
            for entry_id in reached:
                rolled[entry_id] += 1
        entries = [{"id": eid, "path": e["path"], "level": e["level"], "use_cases": rolled[eid], "direct": direct[eid]}
                   for eid, e in catalog["entries"].items()]
        gaps = [{"id": eid, "path": e["path"], "reason": STRATEGY_GAP_TEXT} for eid, e in catalog["entries"].items()
                if rolled[eid] == 0 and all(rolled[parent] > 0 for parent in e["ancestors"][-1:])]
        priority_counts = Counter(p for uc in use_cases for p in context["priority_hits"].get(str(uc.get("No", "")), []))
        aligned = sum(1 for uc in use_cases if uc.get("Strategy Alignment"))
        return {"source": context["tree"].get("source") or "builtin", "levels": context["tree"].get("levels") or [],
                "use_cases": len(use_cases), "aligned": aligned, "not_aligned": len(use_cases) - aligned,
                "entries": entries, "gaps": gaps,
                "priorities": [{"priority": p, "use_cases": priority_counts.get(p, 0)} for p in context["priorities"]],
                "priority_gaps": [p for p in context["priorities"] if not priority_counts.get(p)]}

    def _roster_owner(self, uc: dict, context: dict, owners: dict) -> tuple:
        entries = context["catalog"]["entries"]
        for alignment in uc.get("Strategy Alignment") or []:
            entry = entries.get(alignment["id"])
            if entry is None:
                continue
            owner = owners.get(entry["name"].casefold()) or owners.get(entry["id"].casefold())
            if owner is not None:
                return owner, "owns"
            for parent_id in reversed(entry["ancestors"]):
                parent = entries[parent_id]
                owner = owners.get(parent["name"].casefold()) or owners.get(parent_id.casefold())
                if owner is not None:
                    return owner, "parent"
        owner = owners.get(str(uc.get("Business Domain") or "").strip().casefold())
        return (owner, "domain") if owner is not None else (None, None)

    def _apply_sponsor_roster(self, use_cases: list, context: dict) -> dict:
        sponsors = list(context["sponsors"].values())
        if not sponsors:
            return {}
        by_role = {s["role"].casefold(): s for s in sponsors}
        canonical = getattr(self, "_sponsor_role_cache", None) or {}
        owners = {}
        for sponsor in sponsors:
            for owned in sponsor["owns"]:
                owners.setdefault(owned.casefold(), sponsor)
        roots = [s for s in sponsors if not s["reports_to"] or s["reports_to"].casefold() not in by_role]
        root = (roots or sponsors)[0]
        methods = Counter()
        for uc in use_cases:
            current = str(uc.get("Sponsor") or "").strip()
            match = by_role.get(current.casefold()) or by_role.get(str(canonical.get(current.lower(), current)).casefold())
            method = "kept"
            if match is None:
                match, method = self._roster_owner(uc, context, owners)
            if match is None:
                match = context["sponsors"].get(context["sponsor_picks"].get(str(uc.get("No", ""))))
                method = "llm_pick"
            if match is None:
                match, method = root, "root"
            uc["Sponsor"] = match["role"]
            methods[method] += 1
        self.logger.info(f"[SPONSOR_ROSTER FIRED] {len(use_cases)} sponsor(s) taken from the roster of {len(sponsors)} "
                         f"role(s): {dict(methods)}")
        return dict(methods)

    def _feedback_table_name(self) -> str:
        database = getattr(self, "inspire_database", "") or ""
        return quote_qualified_name(f"{database}.{FEEDBACK_TABLE}") if database else ""

    def _ensure_feedback_table(self, feedback_table: str):
        columns = ", ".join(f"`{name}` {kind}" for name, kind in FEEDBACK_COLUMNS)
        self.spark.sql(f"CREATE TABLE IF NOT EXISTS {feedback_table} ({columns}) USING DELTA "
                       f"COMMENT 'Use case feedback written by the Inspire App and read by the agent'")

    def _load_feedback(self) -> dict:
        """Latest feedback per use case key for this business, from Inspire's own feedback table."""
        self.feedback_rejected, self.feedback_history = {}, {}
        feedback_table = self._feedback_table_name()
        if not feedback_table:
            self.logger.info("Feedback not read: no inspire_database is set")
            return {}
        try:
            self._ensure_feedback_table(feedback_table)
            rows = _metadata_rows(
                self.spark,
                f"SELECT use_case_key, use_case_title, status, reason, session_id, updated_at FROM {feedback_table} "
                f"WHERE lower(business_name) = lower(:business) ORDER BY updated_at DESC",
                args={"business": str(getattr(self, "business_name", "") or "")}, logger=self.logger, context="feedback")
        except SparkWorkCancelled:
            raise
        except Exception as error:
            self.logger.warning(f"Feedback table {feedback_table} unreadable: {get_clean_error_message(error, max_chars=200)}; "
                                f"this run uses no feedback")
            return {}
        latest = {}
        for row in rows:
            key = str(row.use_case_key or "").strip()
            if key and key not in latest:
                latest[key] = row
        for key, row in latest.items():
            status = str(row.status or "").strip().lower()
            entry = {"use_case_key": key, "title": str(row.use_case_title or key.split("|")[0]), "status": status,
                     "reason": str(row.reason or ""), "session_id": row.session_id,
                     "updated_at": str(row.updated_at or ""), "primary_table": key.rsplit("|", 1)[-1]}
            if status == "rejected":
                self.feedback_rejected[key] = entry
            elif status in FEEDBACK_STATUSES:
                self.feedback_history[key] = entry
        self.logger.info(f"[FEEDBACK FIRED] {len(latest)} use case(s) with feedback for this business: "
                         f"{len(self.feedback_rejected)} rejected, {len(self.feedback_history)} accepted, planned or implemented")
        return latest

    def _apply_feedback_exclusions(self, use_cases: list) -> list:
        rejected = getattr(self, "feedback_rejected", None) or {}
        history = getattr(self, "feedback_history", None) or {}
        rejected_by_table = defaultdict(list)
        for rejected_key in rejected:
            title, primary = rejected_key.rsplit("|", 1)
            rejected_by_table[primary].append((rejected_key, title))
        kept, excluded = [], []
        for uc in use_cases:
            key = use_case_key(uc)
            uc["use_case_key"] = key
            if key in rejected:
                excluded.append(dict(rejected[key], use_case_id=str(uc.get("No", "")), match="exact"))
                continue
            title, primary = key.rsplit("|", 1)
            matches = sorted(((feedback_rewording_similarity(title, rejected_title), rejected_key)
                              for rejected_key, rejected_title in rejected_by_table.get(primary, [])
                              if feedback_is_rewording(title, rejected_title)), reverse=True)
            if matches:
                (subject, whole), closest = matches[0]
                excluded.append(dict(rejected[closest], use_case_id=str(uc.get("No", "")), match="reworded",
                                     subject_similarity=round(subject, 2), similarity=round(whole, 2),
                                     proposed_title=str(uc.get("Name") or "")))
                continue
            if key in history:
                uc["Feedback Status"] = history[key]["status"]
            kept.append(uc)
        self.excluded_by_feedback = excluded
        if excluded:
            reworded = [e for e in excluded if e["match"] == "reworded"]
            self.logger.info(f"[FEEDBACK-REJECT FIRED] excluded {len(excluded)} use case(s) the user rejected "
                             f"({len(excluded) - len(reworded)} exact, {len(reworded)} reworded): "
                             f"{[e['title'] for e in excluded][:20]}")
            for entry in reworded[:20]:
                self.logger.info(f"[FEEDBACK-REJECT reworded] {entry['proposed_title']!r} on {entry['primary_table']} "
                                 f"matches the rejected {entry['title']!r} (subject {entry['subject_similarity']}, "
                                 f"title {entry['similarity']})")
        self._log_funnel_stage("feedback_exclusion", len(use_cases), len(kept), {"rejected_by_user": len(excluded)})
        return kept

    def _track_implemented_value(self) -> list:
        """For use cases the user marked implemented, the stored KPI specs are rebuilt by the KPI builder;
        Deep measures the current value next to the stored baseline, Standard returns the query only."""
        implemented = {k: v for k, v in (getattr(self, "feedback_history", None) or {}).items() if v["status"] == "implemented"}
        self.value_tracking = []
        tracking_table = getattr(self, "_tracking_table_name", "") or ""
        if not implemented or not tracking_table:
            return self.value_tracking
        in_sql, params = _sql_in_list_params(sorted(implemented)[:200], "k")
        try:
            rows = _metadata_rows(
                self.spark,
                f"SELECT use_case_key, kpis, session_id FROM {tracking_table} WHERE use_case_key IN ({in_sql}) "
                f"AND kpis IS NOT NULL ORDER BY updated_at DESC",
                args=params, logger=self.logger, context="value tracking")
        except SparkWorkCancelled:
            raise
        except Exception as error:
            self.logger.warning(f"Value tracking unavailable: {get_clean_error_message(error, max_chars=200)}")
            return self.value_tracking
        tables = self._gate_schema_index()["tables"]
        deep = getattr(self, "deep_evidence", None)
        seen = set()
        for row in rows:
            key = str(row.use_case_key or "")
            if key in seen:
                continue
            seen.add(key)
            try:
                specs = json.loads(row.kpis) if isinstance(row.kpis, str) else list(row.kpis or [])
            except ValueError:
                continue
            for spec in specs if isinstance(specs, list) else []:
                if not isinstance(spec, dict) or not spec.get("computable"):
                    continue
                table = self._normalize_table_ref(spec.get("table") or "")
                column = str(spec.get("column") or "").lower()
                if not spec.get("metric_view") and (table not in tables or (column and column not in tables[table])):
                    continue
                entry = {"use_case_key": key, "title": implemented[key]["title"], "kpi": spec.get("name"),
                         "baseline": spec.get("baseline"), "current": None, "change": None}
                try:
                    entry["query"] = DeepEvidence.kpi_sql(spec)
                except (ValueError, KeyError):
                    continue
                if deep is not None:
                    try:
                        value = deep.measure_kpi(spec)
                    except DeepEvidenceBudgetSpent as spent:
                        self.logger.warning(f"[VALUE-TRACKING] {spent}; the remaining KPIs are reported as queries only")
                        deep, value = None, None
                    except SparkWorkCancelled:
                        raise
                    except Exception as error:
                        self.logger.warning(f"Value tracking for {entry['title']!r} unavailable: "
                                            f"{get_clean_error_message(error, max_chars=200)}")
                        value = None
                    if value is not None:
                        entry["current"] = {"value": value, "measured_on": datetime.date.today().isoformat()}
                        if (spec.get("baseline") or {}).get("value") is not None:
                            entry["change"] = round(value - float(spec["baseline"]["value"]), 4)
                entry["status"] = "measured" if entry["current"] else ("query_only" if deep is None else "unavailable")
                self.value_tracking.append(entry)
        self.logger.info(f"[VALUE-TRACKING FIRED] {len(self.value_tracking)} KPI(s) of {len(implemented)} implemented "
                         f"use case(s): {sum(1 for e in self.value_tracking if e['current'])} measured now")
        return self.value_tracking

    def _foundation_join_edges(self, tables: set) -> dict:
        edges = {}
        for child, child_col, parent, parent_col in self._evidence_join_candidates(
                [tuple(t.split(".")) for t in tables if t.count(".") == 2]):
            left, right = ".".join(child).lower(), ".".join(parent).lower()
            edges.setdefault(tuple(sorted((left, right))), {"left": f"{left}.{child_col}", "right": f"{right}.{parent_col}",
                                                         "source": "foreign key"})
        evidence = getattr(self, "discovery_evidence", None) or {}
        for edge in evidence.get("history_joins") or []:
            left, right = edge["left"].rsplit(".", 1)[0].lower(), edge["right"].rsplit(".", 1)[0].lower()
            if left in tables and right in tables:
                edges.setdefault(tuple(sorted((left, right))), {"left": edge["left"], "right": edge["right"],
                                                             "source": "observed in queries"})
        return edges

    def _build_data_foundations(self, use_cases: list) -> list:
        """Shared data foundations of the final use cases over shared tables (used by 2 or more use cases)
        and joins 2 or more use cases use together. A fact table (a child of a kept foreign key and
        nobody's parent) anchors one foundation with the dimensions it joins, so shared dimensions never
        chain facts together; the other shared tables group over their joins. Ranked by use cases
        unlocked, then by their summed Inspire Score. No durations or dates."""
        by_table = defaultdict(list)
        uc_tables = {}
        for uc in use_cases:
            refs = [ref for ref in self._uc_table_refs(uc) if ref and ref != "n/a"]
            uc_tables[str(uc.get("No", ""))] = set(refs)
            for ref in refs:
                by_table[ref].append(uc)
        shared = {table for table, members in by_table.items() if len(members) >= 2}
        edges = self._foundation_join_edges(shared)
        kept_edges = [((left, right), edge) for (left, right), edge in sorted(edges.items())
                      if sum(1 for tables in uc_tables.values() if left in tables and right in tables) >= 2]
        parents_of = defaultdict(set)
        for _pair, edge in kept_edges:
            if edge["source"] == "foreign key":
                parents_of[edge["left"].rsplit(".", 1)[0]].add(edge["right"].rsplit(".", 1)[0])
        facts = set(parents_of) - {table for found in parents_of.values() for table in found}
        groups = [({fact} | parents_of[fact], {fact}) for fact in sorted(facts)]
        rest = shared - {table for tables, _anchors in groups for table in tables}
        parent = {table: table for table in rest}

        def _root(table):
            while parent[table] != table:
                parent[table] = parent[parent[table]]
                table = parent[table]
            return table
        for (left, right), _edge in kept_edges:
            if left in rest and right in rest:
                parent[_root(left)] = _root(right)
        components = defaultdict(set)
        for table in rest:
            components[_root(table)].add(table)
        groups += [(tables, tables) for tables in components.values()]
        foundations = []
        for tables, anchors in groups:
            members = [uc for uc in use_cases if uc_tables[str(uc.get("No", ""))] & anchors]
            members.sort(key=lambda u: (-(self._finite_or(u.get("Inspire Score"), 0.0)), str(u.get("No", ""))))
            foundations.append({"tables": sorted(tables),
                                "joins": [edge for (left, right), edge in kept_edges if left in tables and right in tables],
                                "use_cases": [str(u.get("No", "")) for u in members],
                                "use_case_count": len(members),
                                "summed_inspire_score": round(sum(self._finite_or(u.get("Inspire Score"), 0.0) for u in members), 2)})
        foundations.sort(key=lambda f: (-f["use_case_count"], -f["summed_inspire_score"], f["tables"]))
        rank_of = {}
        for rank, foundation in enumerate(foundations, start=1):
            foundation["id"] = f"F{rank}"
            foundation["rank"] = rank
            foundation["name"] = " + ".join(t.rsplit(".", 1)[-1] for t in foundation["tables"])
            for uc_id in foundation["use_cases"]:
                rank_of.setdefault(uc_id, rank)
        for uc in use_cases:
            uc_id = str(uc.get("No", ""))
            uc["Foundations"] = [f["id"] for f in foundations if uc_id in f["use_cases"]]
        order = sorted(use_cases, key=lambda u: (rank_of.get(str(u.get("No", "")), len(foundations) + 1),
                                                 -(self._finite_or(u.get("Inspire Score"), 0.0)), str(u.get("No", ""))))
        self.delivery_order = [str(u.get("No", "")) for u in order]
        self.data_foundations = foundations
        self.logger.info(f"[FOUNDATIONS FIRED] {len(foundations)} shared data foundation(s) over {len(shared)} shared table(s) "
                         f"for {len(use_cases)} use case(s); top: "
                         + ("; ".join(f"{f['id']} {f['name']} unlocks {f['use_case_count']}" for f in foundations[:5]) or "none"))
        return foundations

    def _run_strategy_alignment(self, use_cases: list) -> list:
        """After selection: align every final use case to the strategy, define KPIs, key results,
        measures and benefits, measure Deep baselines and recompute the scores. Returns the same use
        cases it receives: it never adds, keeps or removes one."""
        if not use_cases:
            return use_cases
        members = [id(uc) for uc in use_cases]
        tree = getattr(self, "strategy_tree", None) or build_builtin_strategy(
            (getattr(self, "merged_business_context", None) or {}).get("strategic_goals") or [])
        context = self._alignment_context(tree)
        rt = TECHNICAL_CONTEXT["runtime"]
        batch_size = rt["strategy_alignment_batch_size"]
        merged = getattr(self, "merged_business_context", None) or {}
        base_vars = {
            "business_name": getattr(self, "business_name", "") or "",
            "business_context": merged.get("business_context") or self._get_business_context_fallback(),
            "generation_instructions_section": self._get_generation_instruction_for_prompt("STRATEGY_ALIGNMENT_PROMPT"),
            "strategy_source": "defined by the customer" if tree.get("source") == "customer" else "inferred strategic goals",
            "strategy_levels": ", ".join(tree.get("levels") or []) or STRATEGY_DEFAULT_LEVEL,
            "strategy_entries": strategy_prompt_lines(tree),
            "business_priorities": "\n".join(f"- {p}" for p in context["priorities"]) or "(none)",
            "sponsor_roster": self._sponsor_roster_lines(tree),
        }
        try:
            self._emit_pipeline_status(stage_name="Strategy Alignment", step_name="Strategy Alignment",
                                       sub_step_name="Started", status="started",
                                       message=f"Aligning {len(use_cases)} selected use cases to the strategy")
        except Exception:
            pass

        def _batch_aligner(batch_vars: dict, batch_context: dict):
            def _align_batch(batch_label: str, uc_batch: list, skip_cache: bool) -> dict:
                if _is_thread_cancelled():
                    raise Exception(f"[StrategyAlign_{batch_label}] Aborted: parent task cancelled/timed out")
                payloads = [self._alignment_payload(uc, batch_context) for uc in uc_batch]
                tables = sorted({ref for payload in payloads for ref in payload["tables"]})
                prompt_vars = dict(batch_vars, metric_views=self._alignment_metric_views(batch_context, tables),
                                   use_cases_json=json.dumps(payloads, ensure_ascii=False, indent=1))
                response_raw = self.ai_agent.run_worker(
                    step_name=f"StrategyAlign_{batch_label}", worker_prompt_path="STRATEGY_ALIGNMENT_PROMPT",
                    prompt_vars=prompt_vars, response_schema=None,
                    timeout_override=max(300, self.llm_timeout_seconds + len(uc_batch) * 15), skip_cache=skip_cache)
                return self._parse_alignment_response(response_raw, uc_batch, batch_context)
            return _align_batch

        sample = [self._alignment_payload(uc, context) for uc in use_cases[:batch_size]]
        est_chars = self._prompt_overhead_chars("STRATEGY_ALIGNMENT_PROMPT") + len(json.dumps(sample)) + \
            sum(len(str(v)) for v in base_vars.values())
        per_task_timeout = self._scoring_task_timeout(max(300, self.llm_timeout_seconds + batch_size * 15), est_chars)
        records, failures = self._score_with_split_retry(use_cases, _batch_aligner(base_vars, context), batch_size,
                                                          "StrategyAlign", per_task_timeout)
        unmeasurable = self._unmeasurable_entries(records, context)
        if unmeasurable:
            pruned = self._strategy_without(tree, set(unmeasurable))
            pruned_vars = dict(base_vars, strategy_entries=strategy_prompt_lines(pruned))
            affected = [uc for uc in use_cases if any(a["id"] in unmeasurable for a in
                                                      (records.get(str(uc.get("No", ""))) or {}).get("alignments", []))]
            realigned, _ = self._score_with_split_retry(affected, _batch_aligner(pruned_vars, self._alignment_context(pruned)),
                                                        batch_size, "StrategyRealign", per_task_timeout)
            for uc in affected:
                uc_id = str(uc.get("No", ""))
                if uc_id in realigned:
                    records[uc_id] = realigned[uc_id]
            stripped = self._strip_alignments(records, set(unmeasurable) | set(self._unmeasurable_entries(records, context)))
            self.logger.info(f"[STRATEGY-MEASURABLE FIRED] {len(unmeasurable)} entr(ies) whose KPIs no aligned use case can "
                             f"compute: {[context['catalog']['entries'][e]['path'] for e in unmeasurable]}; re-aligned "
                             f"{sum(1 for uc in affected if str(uc.get('No', '')) in realigned)}/{len(affected)} use case(s) "
                             f"without them; {stripped} alignment(s) removed")
        fallbacks = []
        for uc in use_cases:
            uc_id = str(uc.get("No", ""))
            record = records.get(uc_id)
            if record is None:
                record = self._fallback_alignment(uc, failures.get(uc_id, "no valid alignment answer"))
                fallbacks.append(uc_id)
            for note in record["notes"]:
                self.logger.info(f"[STRATEGY-ALIGN] {uc_id}: {note}")
            self._apply_alignment_record(uc, record, context)
        if fallbacks:
            self._record_failed_step("Strategy Alignment", "",
                                     f"{len(fallbacks)} use case(s) kept placeholder KPIs and benefits after the "
                                     f"retries: {fallbacks[:20]}")
        baseline_stats = self._measure_kpi_baselines(use_cases)
        rescored = 0
        for uc in use_cases:
            self._fill_baseline_placeholders(uc)
            computed = compute_value_priority(uc, with_alignment=True)
            if computed is not None:
                uc["Value"], uc["Feasibility"], uc["Priority"] = computed
                uc["Priority Score"] = uc["Priority"]
                rescored += 1
            score = self._compute_inspire_score(uc)
            if score is not None:
                uc["Inspire Score"] = score
        coverage = self._strategy_coverage_report(use_cases, context)
        self.strategy_coverage = coverage
        roster = self._apply_sponsor_roster(use_cases, context)
        if [id(uc) for uc in use_cases] != members:
            raise RuntimeError("Strategy alignment changed the set of use cases; it may only annotate them")
        aligned = coverage["aligned"]
        by_top = Counter(a["path"].split(" > ")[0] for uc in use_cases for a in uc.get("Strategy Alignment") or [])
        self.logger.info(f"[STRATEGY-ALIGN FIRED] source={coverage['source']} aligned {aligned}/{len(use_cases)} use case(s), "
                         f"not aligned {coverage['not_aligned']}; by top entry {dict(by_top)}; gaps {len(coverage['gaps'])}; "
                         f"scores recomputed for {rescored}/{len(use_cases)}")
        if coverage["source"] == "customer" and len(use_cases) >= 5 and (
                aligned / len(use_cases) > 0.9 or (aligned and max(by_top.values()) / aligned > 0.5
                                                   and len(context["catalog"]["entries"]) > 1)):
            self.logger.warning(f"[STRATEGY-ALIGN WARN] {aligned}/{len(use_cases)} aligned, largest top entry holds "
                                f"{max(by_top.values()) if by_top else 0}: check that the alignments are not stretched")
        kpi_total = sum(len(uc.get("KPIs") or []) for uc in use_cases)
        self.logger.info(
            f"[STRATEGY-ENRICH FIRED] KPIs {kpi_total} (computable {baseline_stats['computable']}, Deep baselines measured "
            f"{baseline_stats['measured']}); key results from the strategy "
            f"{sum(1 for uc in use_cases for kr in uc.get('Expected Key Results') or [] if kr.get('source') == 'strategy')}, "
            f"proposed {sum(1 for uc in use_cases for kr in uc.get('Expected Key Results') or [] if kr.get('source') == 'proposed')}; "
            f"invented figures replaced {sum(r.get('scrubbed', 0) for r in records.values())}; placeholder fallbacks {len(fallbacks)}")
        try:
            self._emit_pipeline_status(stage_name="Strategy Alignment", step_name="Strategy Alignment",
                                       sub_step_name="Complete", status="ended_warning" if fallbacks else "ended_success",
                                       message=f"Strategy alignment: {aligned}/{len(use_cases)} aligned, "
                                               f"{len(coverage['gaps'])} gap(s)",
                                       result_json={"aligned": aligned, "total": len(use_cases), "gaps": len(coverage["gaps"]),
                                                    "fallbacks": len(fallbacks), "sponsor_roster": roster})
        except Exception:
            pass
        return use_cases

    def _merge_business_contexts(self, llm_context: dict, user_context_str: str) -> dict:
        """
        Merges LLM-generated business context with user-provided context.
        User context ALWAYS takes precedence and overrides LLM context.
        
        Args:
            llm_context: Dictionary from Business Context Worker (15 fields)
            user_context_str: Comma-separated string from user (Business Context widget)
            
        Returns:
            Merged dictionary with user values taking precedence. `industry` is the LLM's value
            when present, otherwise the first line of `business_context` (at most 200 chars).
        """
        merged_context = dict(llm_context or {})
        if not str(merged_context.get("industry") or "").strip():
            _first_line = str(merged_context.get("business_context") or "").strip().split("\n")[0].strip()
            merged_context["industry"] = _first_line[:200].strip()
        
        if not user_context_str or not user_context_str.strip():
            self.logger.info("No user-provided business context. Using LLM context only.")
            return merged_context
        
        # Parse user context - assume it's comma-separated values that override specific fields
        # For simplicity, we'll treat the entire user context as additional focus areas
        user_focus_areas = [area.strip() for area in user_context_str.split(',') if area.strip()]
        
        if user_focus_areas:
            self.logger.info(f"✅ User provided {len(user_focus_areas)} business context items - these will take precedence")
            
            # Store user focus areas separately for later use
            # We'll use them to override or extend LLM context
            merged_context['user_focus_areas'] = ', '.join(user_focus_areas)
            
            # Also update relevant fields with user context
            if 'business_units_divisions_domains_subdomains' in merged_context:
                # Prepend user areas to ensure they're used
                existing = merged_context['business_units_divisions_domains_subdomains']
                merged_context['business_units_divisions_domains_subdomains'] = ', '.join(user_focus_areas) + ', ' + existing
            else:
                merged_context['business_units_divisions_domains_subdomains'] = ', '.join(user_focus_areas)
        
        return merged_context
    
    # === USER DOMAIN ASSIGNMENT ===
    def _assign_to_user_domains(self, use_cases: list, user_domains: list, language: str) -> list:
        """
        Assigns use cases to user-provided business domains using LLM, 
        then discovers subdomains within each domain.
        
        This method is called when user provides specific business domains.
        The user provides TOP-LEVEL DOMAINS ONLY - Inspire discovers subdomains automatically.
        
        A UC gets a user domain only when the LLM names one of the user's domains for it (exact or
        case-insensitive, stored with the user's spelling). UCs left without a valid domain are
        re-sent in smaller batches with the cache bypassed; a UC still unassigned after the retries
        goes through the shared `_repair_unassigned_domains` (targeted re-ask, same-primary-table
        evidence, evidence-based placeholder), so no UC is dropped and none is forced into a user
        domain. Subdomains come from the shared `_detect_subdomains_all_domains` dispatcher.
        
        Args:
            use_cases: List of use case dictionaries
            user_domains: List of user-provided domain names (top-level only)
            language: Output language
            
        Returns:
            List of use cases with Business Domain and Subdomain properly assigned
        """
        import io
        import csv
        from collections import Counter
        
        self.logger.info(f"📍 Assigning {len(use_cases)} use cases to {len(user_domains)} user-provided domains...")
        self.logger.info(f"📍 User provided TOP-LEVEL domains only: {', '.join(user_domains)}")
        self.logger.info(f"📍 Inspire will discover SUBDOMAINS within each domain automatically")
        
        if not use_cases or not user_domains:
            return use_cases
        
        domain_lookup = {}
        for d in user_domains:
            if str(d or "").strip():
                domain_lookup.setdefault(str(d).strip().lower(), d)
        domain_list_str = ", ".join([f'"{d}"' for d in user_domains])
        fieldnames = ['No', 'Name', 'type', 'Analytics Technique', 'Statement', 'Solution', 
                     'Business Value', 'Beneficiary', 'Sponsor', 'Tables Involved']

        def _use_cases_csv(ucs: list) -> str:
            output = io.StringIO()
            writer = csv.DictWriter(output, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(ucs)
            return output.getvalue()

        def _assignment_prompt(ucs: list) -> str:
            return f"""You are an expert business analyst. Your task is to assign each use case to ONE of the following EXACT business domains:

**ALLOWED DOMAINS (USE EXACTLY AS PROVIDED)**: {domain_list_str}

**🚨 CRITICAL RULES 🚨**:
1. You MUST assign EVERY use case to EXACTLY ONE domain from the list above
2. You MUST use the domain names EXACTLY as provided - no modifications
3. You MUST NOT create or invent any new domain names
4. Every use case MUST have a domain assigned

**USE CASES TO ASSIGN**:
{_use_cases_csv(ucs)}

**OUTPUT FORMAT** (CSV only, no explanation):
```csv
use_case_id,domain
```

For each use case, output ONE row with the use case ID and the assigned domain.
Start your response with the CSV header line: use_case_id,domain
"""

        def _assign_batch(batch_label: str, batch: list, skip_cache: bool) -> dict:
            if _is_thread_cancelled():
                raise Exception(f"[Assign_User_Domains_{batch_label}] Aborted: parent task cancelled/timed out")
            response_raw = self.ai_agent._call_ai_query(
                prompt=_assignment_prompt(batch),
                prompt_name="Assign_User_Domains",
                response_schema=None,
                display_name=f"Assign_User_Domains_{language}_{batch_label}",
                skip_cache=skip_cache
            )
            csv_text = clean_csv_response(response_raw)
            batch_ids = {str(uc.get('No', '')) for uc in batch}
            csv_rows, _ = CSVParser.aligned_rows(
                CSVParser.parse_csv_string(csv_text, logger=self.logger, context="User domain assignment"),
                self.logger, f"User domain assignment {batch_label}", required=('use_case_id', 'domain'),
                anchors={'use_case_id': CSVParser.one_of(batch_ids), 'domain': CSVParser.one_of(domain_lookup.values())},
                text=csv_text)
            assignments = {}
            for row in csv_rows:
                uc_id = str(row.get('use_case_id', '') or '').strip()
                canonical = domain_lookup.get(str(row.get('domain', '') or '').strip().lower())
                if uc_id in batch_ids and canonical:
                    assignments[uc_id] = canonical
                elif uc_id in batch_ids:
                    self.logger.warning(f"Use case {uc_id} was assigned '{row.get('domain', '')}', which is not a user-specified domain; it will be re-assigned")
            return assignments

        context_limit = get_max_context_chars("English", f"Assign_User_Domains_{language}")
        avg_row = max(1, len(_use_cases_csv(use_cases)) // len(use_cases))
        batch_size = max(1, int((context_limit * 0.8 - len(_assignment_prompt([]))) / avg_row))
        per_task_timeout = self._scoring_task_timeout(self.llm_timeout_seconds, len(_assignment_prompt([])) + avg_row * min(batch_size, len(use_cases)))
        domain_map, unassigned = self._score_with_split_retry(use_cases, _assign_batch, batch_size, "UserDomainAssign", per_task_timeout)

        assigned_use_cases, missing = [], []
        for uc in use_cases:
            uc_id = str(uc.get('No', ''))
            if uc_id in domain_map:
                uc['Business Domain'] = domain_map[uc_id]
                assigned_use_cases.append(uc)
            else:
                uc['Business Domain'] = ''
                missing.append(uc)
        self.logger.info(f"✅ Assigned {len(assigned_use_cases)}/{len(use_cases)} use cases to user domains")
        if missing:
            self.logger.warning(
                f"⚠️ {len(missing)} use case(s) got no valid user-specified domain after retries; they go to the shared "
                f"domain repair instead of being dropped or forced into a user domain: "
                f"{'; '.join(f'{uc_id}: {reason}' for uc_id, reason in list(unassigned.items())[:10])}"
                f"{' ...' if len(unassigned) > 10 else ''}"
            )
            self._repair_unassigned_domains(missing, assigned_use_cases, fieldnames, language, set(domain_lookup.values()))
        self._log_funnel_stage("user_domain_assignment", len(use_cases), len(use_cases))
        
        domain_counts = Counter(uc.get('Business Domain', 'Unknown') for uc in use_cases)
        for domain, count in sorted(domain_counts.items(), key=lambda x: -x[1]):
            self.logger.info(f"   📁 {domain}: {count} use cases")
        
        self.logger.info(f"📍 STEP 2: Discovering subdomains within each domain (user-domain path)...")
        final_use_cases_with_subdomains = self._detect_subdomains_all_domains(use_cases, language, "user-domain")
        self.logger.info(f"✅ Domain assignment and subdomain discovery complete! {len(final_use_cases_with_subdomains)} use cases processed")
        
        self.logger.info(f"📍 STEP 2.5: Cross-domain subdomain deduplication (user-domain path)...")
        final_use_cases_with_subdomains = self._merge_duplicate_subdomains(final_use_cases_with_subdomains)
        final_use_cases_with_subdomains = self._rename_placeholder_domains_from_subdomains(final_use_cases_with_subdomains)
        
        subdomain_counts = Counter(uc.get('Subdomain', 'Unknown') for uc in final_use_cases_with_subdomains)
        self.logger.info(f"📊 Subdomain distribution:")
        for subdomain, count in sorted(subdomain_counts.items(), key=lambda x: -x[1]):
            self.logger.info(f"   📁 {subdomain}: {count} use cases")
        
        return final_use_cases_with_subdomains
    
    # === HELPER FUNCTIONS ===
    def _apply_domain_mapping_flat(self, all_use_cases: list, domain_mapping: dict) -> list:
        """Applies a domain mapping to a flat list[dict] of use cases."""
        modified_use_cases = []
        for uc in all_use_cases:
            original_domain = uc.get('Business Domain', 'Other')
            new_domain = domain_mapping.get(original_domain, original_domain)
            uc['Business Domain'] = new_domain
            modified_use_cases.append(uc)
        return modified_use_cases
    
    def _group_use_cases_by_domain_flat(self, all_use_cases: list) -> dict:
        """
        Groups a flat list[dict] of use cases by their 'Business Domain'.
        Use cases within each domain are sorted by priority descending (Very High first).
        
        NOTE: This sorting is used for PDF/PPT/XLS outputs. Notebooks use ID-based sorting instead.
        """
        # Sort by priority first (for PDF/PPT/XLS outputs)
        all_use_cases = sorted(all_use_cases, key=self._priority_sort_key)
        grouped_by_domain = defaultdict(list)
        for uc in all_use_cases:
            domain = uc.get('Business Domain') or 'Other'
            grouped_by_domain[domain].append(uc)
        return grouped_by_domain

    def _align_translated_data(self, master_grouped_data: dict, translated_flat_list: list) -> dict:
        """
        Creates a grouped dictionary for the target language that strictly follows 
        the ordering and categorization of the master (English) data.
        
        The translated entries are complete use case copies (TranslationService merges the
        translated text onto the original), so every score and identifier is preserved; a use
        case missing from the translation is kept as a flagged English copy. Two domains whose
        translated names collide stay separate.
        """
        from collections import deque
        translated_by_id = defaultdict(deque)
        for uc in translated_flat_list or []:
            translated_by_id[str(uc.get('No') or '').strip()].append(uc)
        aligned_data = {}
        for en_domain, english_cases in (master_grouped_data or {}).items():
            translated_group = []
            for en_uc in english_cases:
                queue = translated_by_id.get(str(en_uc.get('No') or '').strip())
                translated_group.append(queue.popleft() if queue else TranslationService.untranslated_copy(en_uc))
            translated_domain_name = next(
                (str(uc.get('Business Domain')).strip() for uc in translated_group
                 if uc.get('_translation_status') in ('translated', 'partial')
                 and 'Business Domain' not in (uc.get('_untranslated_fields') or [])
                 and str(uc.get('Business Domain') or '').strip()),
                en_domain)
            if translated_domain_name in aligned_data:
                translated_domain_name = f"{translated_domain_name} ({en_domain})"
            aligned_data[translated_domain_name] = translated_group
        return aligned_data
    # === END HELPER FUNCTIONS ===


    def _build_dedup_markdown(self, use_cases: list) -> str:
        """
        Build a markdown table for deduplication including ID, Name, Domain,
        Analytics Technique, Tables Involved (table names only), and a
        1-line Statement. The extra fields are required to activate Layer 4
        (technique swap) and Layer 5 (table + intent overlap) duplicate
        detection. Scores and activation suffixes are intentionally excluded
        to keep the LLM focused on business intent + data grounding.
        Shared by global and per-domain dedup (DRY).
        """
        pipe_esc = r'\|'
        header = (
            "| ID | Name | Domain | Technique | Tables | Statement |\n"
            "|---|---|---|---|---|---|\n"
        )
        def _sanitize(value: str) -> str:
            return str(value or '').replace('|', pipe_esc).replace('\n', ' ').strip()
        def _compact_tables(raw: str) -> str:
            parts = []
            for token in _sanitize(raw).split(','):
                token = token.strip().strip('`')
                if not token:
                    continue
                parts.append(token.split('.')[-1])
            return ', '.join(parts) if parts else '—'
        md_parts = [header]
        for uc in use_cases:
            name = _sanitize(uc.get('Name', ''))
            domain = _sanitize(uc.get('Business Domain', ''))
            technique = _sanitize(uc.get('Analytics Technique', '')) or '—'
            tables = _compact_tables(uc.get('Tables Involved', ''))
            statement = _sanitize(uc.get('Statement', ''))
            if len(statement) > 200:
                statement = statement[:197] + '...'
            if not statement:
                statement = '—'
            md_parts.append(
                f"| {uc['No']} | {name} | {domain} | {technique} | {tables} | {statement} |\n"
            )
        return "".join(md_parts)

    _CSV_ID_HEADERS = ("usecaseid", "id", "no", "ucid")

    @staticmethod
    def _csv_column_key(name) -> str:
        return re.sub(r"[^a-z0-9]", "", str(name or "").lower())

    @classmethod
    def _csv_columns(cls, header_row) -> dict:
        columns = {}
        for idx, cell in enumerate(header_row or []):
            columns.setdefault(cls._csv_column_key(cell), idx)
        return columns

    @classmethod
    def _csv_id_column(cls, columns: dict):
        return next((columns[name] for name in cls._CSV_ID_HEADERS if name in columns), None)

    @staticmethod
    def _csv_cell_text(row, idx) -> str:
        if idx is None or idx >= len(row):
            return ""
        return str(row[idx] or "").replace("\ufeff", "").strip().strip("`\"'").strip()

    @classmethod
    def _dedup_id_key(cls, value) -> str:
        return cls._csv_cell_text([value], 0).lower()

    @classmethod
    def _dedup_response_columns(cls, rows: list, known: dict) -> tuple:
        if not rows:
            return None, 0
        id_col = cls._csv_id_column(cls._csv_columns(rows[0]))
        if id_col is not None:
            return id_col, 1
        counts = {}
        for row in rows:
            for idx, cell in enumerate(row):
                if cls._dedup_id_key(cell) in known:
                    counts[idx] = counts.get(idx, 0) + 1
        if not counts:
            return None, 0
        id_col = max(counts, key=lambda i: (counts[i], -i))
        return id_col, 0 if cls._dedup_id_key(cls._csv_cell_text(rows[0], id_col)) in known else 1

    def _parse_dedup_response(self, response_raw: str, all_use_cases: list, context_label: str, rank=None) -> list:
        """
        Parse LLM dedup response CSV and return filtered use cases.
        Shared by global and per-domain dedup (DRY).
        Returns the filtered list, or raises on failure.
        """
        response_clean = clean_csv_response(response_raw, strip_preamble=False)
        rows = CSVParser.parse_csv_list(response_clean, logger=self.logger, context=context_label)
        known = {self._dedup_id_key(uc['No']): uc['No'] for uc in all_use_cases}
        id_col, first_row = self._dedup_response_columns(rows, known)
        ids_to_keep = set()
        unknown_ids = []
        if id_col is not None:
            header = [str(cell) for cell in rows[0]] if first_row else []
            header += [f"column {i + 1}" for i in range(len(header), id_col + 1)]
            data = rows[first_row:]
            last_id = next((self._dedup_id_key(row[id_col]) for row in reversed(data)
                            if id_col < len(row) and str(row[id_col]).strip()), "")
            if CSVParser.has_open_quote(response_clean) or (
                    last_id and last_id not in known and any(key.startswith(last_id) for key in known)):
                raise ValueError(f"[{context_label}] the keep-list answer was cut off; reading it would drop every "
                                 f"use case after the cut as a duplicate")
            kept, _ = CSVParser.aligned_rows(
                data, self.logger, context_label, header=header, required=(header[id_col],),
                anchors={header[id_col]: lambda value: self._dedup_id_key(value) in known})
            kept_rows = {id(row) for row in kept}
            lost = [row for row in data if id(row) not in kept_rows and any(self._dedup_id_key(cell) in known for cell in row)]
            if lost:
                raise ValueError(f"[{context_label}] {len(lost)} keep-row(s) name a reviewed use case outside the id column; "
                                 f"reading them would drop those use cases as duplicates")
            rows, first_row = kept, 0
        
        for row in (rows[first_row:] if id_col is not None else []):
            if id_col >= len(row) or not self._dedup_id_key(row[id_col]):
                continue
            uc_id = known.get(self._dedup_id_key(row[id_col]))
            if uc_id is None:
                unknown_ids.append(row[id_col].strip())
                continue
            ids_to_keep.add(uc_id)
        
        if unknown_ids:
            self.logger.warning(f"[{context_label}] Ignored {len(unknown_ids)} id(s) that are not in the reviewed set: "
                                f"{unknown_ids[:10]}")
        if not ids_to_keep:
            raise ValueError(f"[{context_label}] CSV contains no use case IDs of the reviewed set")
        
        kept_use_cases = [uc for uc in all_use_cases if uc['No'] in ids_to_keep]
        removed_count = len(all_use_cases) - len(kept_use_cases)
        removal_pct = (removed_count / len(all_use_cases)) * 100 if all_use_cases else 0
        shared_ids = len(all_use_cases) - len(known)
        if shared_ids:
            self.logger.warning(f"[{context_label}] {shared_ids} use case(s) share an id with another reviewed use case; "
                                f"one keep decision applies to every use case of a shared id")
        
        self.logger.info(f"[{context_label}] Retained {len(kept_use_cases)} use cases, removed {removed_count} ({removal_pct:.1f}%)")
        
        if removed_count > 0:
            from collections import defaultdict
            domain_removal_counts = defaultdict(int)
            for uc in all_use_cases:
                if uc['No'] not in ids_to_keep:
                    domain_removal_counts[uc.get('Business Domain', 'Unknown')] += 1
            
            for domain in sorted(domain_removal_counts.keys()):
                self.logger.info(f"  - {domain}: {domain_removal_counts[domain]} use case(s) removed")
        
        if rank is not None:
            dropped = [uc for uc in all_use_cases if uc['No'] not in ids_to_keep]
            kept_use_cases = self._keep_stronger_dedup_twins(kept_use_cases, dropped, rank, context_label)
        return kept_use_cases

    def _keep_stronger_dedup_twins(self, kept: list, dropped: list, rank, context_label: str) -> list:
        if not kept or not dropped:
            return kept

        def _jaccard(a, b):
            a, b = set(a), set(b)
            return len(a & b) / len(a | b) if (a | b) else 0.0

        sig = {id(uc): self._uc_dedup_signature(uc) for uc in list(kept) + list(dropped)}
        winners = {id(uc): uc for uc in kept}
        for loser in dropped:
            s = sig[id(loser)]
            similarity = [_jaccard(s[1], sig[id(k)][1]) + _jaccard(s[2], sig[id(k)][2]) for k in kept]
            best = max(similarity)
            closest = [i for i, value in enumerate(similarity) if value == best]
            if best <= 0 or len(closest) != 1:
                continue
            twin = kept[closest[0]]
            if rank(loser) < rank(winners[id(twin)]):
                winners[id(twin)] = loser
        result = [winners[id(uc)] for uc in kept]
        swaps = [(uc.get('No'), winners[id(uc)].get('No')) for uc in kept if winners[id(uc)] is not uc]
        if swaps:
            self.logger.info(f"[{context_label}] [DEDUP_TWIN] kept the higher-ranked twin for {len(swaps)} duplicate pair(s): "
                             + ", ".join(f"{new} over {old}" for old, new in swaps[:10]))
        return result

    def _dedup_groups(self, all_use_cases: list) -> dict:
        by_domain = defaultdict(list)
        for uc in all_use_cases:
            by_domain[str(uc.get('Business Domain') or '').strip()].append(uc)
        if '' in by_domain:
            by_domain = {"All use cases": list(all_use_cases)}
        limit = get_max_context_chars("English", "REVIEW_USE_CASES_PROMPT")
        overhead = self._prompt_overhead_chars("REVIEW_USE_CASES_PROMPT") + 1000
        header_chars = len(self._build_dedup_markdown([]))
        groups = {}
        for name, ucs in by_domain.items():
            rows = [len(self._build_dedup_markdown([uc])) - header_chars for uc in ucs]
            if overhead + header_chars + sum(rows) <= limit:
                groups[name] = ucs
                continue
            order = sorted(range(len(ucs)), key=lambda i: (tuple(sorted(self._uc_table_refs(ucs[i]))),
                                                           str(ucs[i].get('Name') or '').lower()))
            chunks, current, size = [], [], overhead + header_chars
            for i in order:
                if current and size + rows[i] > limit:
                    chunks.append(current)
                    current, size = [], overhead + header_chars
                current.append(ucs[i])
                size += rows[i]
            if current:
                chunks.append(current)
            self.logger.info(f"🧩 Dedup group '{name}' ({len(ucs)} use cases) exceeds the review prompt context; "
                             f"split into {len(chunks)} chunk(s) ordered by tables")
            for idx, chunk in enumerate(chunks, 1):
                groups[f"{name} chunk {idx} of {len(chunks)}"] = chunk
        return groups

    def _request_dedup(self, step_name: str, prompt_vars: dict, use_cases: list, context_label: str, rank=None,
                       start_from_model: str = None) -> list:
        response_raw = self.ai_agent.run_worker(
            step_name=step_name,
            worker_prompt_path="REVIEW_USE_CASES_PROMPT",
            prompt_vars=prompt_vars,
            response_schema=None,
            start_from_model=start_from_model
        )
        try:
            return self._parse_dedup_response(response_raw, use_cases, context_label, rank=rank)
        except ValueError as e:
            self.logger.warning(f"[DEDUP-REASK FIRED] [{context_label}] the keep-list answer is unusable "
                                f"({get_clean_error_message(e)}); asking the LLM once more with the cache bypassed")
        response_raw = self.ai_agent.run_worker(
            step_name=f"{step_name}_Reask",
            worker_prompt_path="REVIEW_USE_CASES_PROMPT",
            prompt_vars=prompt_vars,
            response_schema=None,
            start_from_model=start_from_model,
            skip_cache=True
        )
        return self._parse_dedup_response(response_raw, use_cases, context_label, rank=rank)

    def _deduplicate_use_cases(self, all_use_cases: list, rank=None) -> list:
        """
        Global deduplication: attempts to deduplicate ALL use cases in one LLM call
        using the thinker model (highest-capability model) for best cross-domain
        duplicate detection.
        
        Sends full Name, Statement, and Tables for each use case so the LLM can
        assess table-based viability when choosing which duplicate to keep.
        
        Falls back to per-domain parallel deduplication on:
        - Context window overflow (too many use cases for one prompt)
        - Any LLM or parsing failure
        """
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Global Semantic Dedup",
                sub_step_name="Started", status="started",
                message=f"Running one-shot dedup for {len(all_use_cases)} UCs"
            )
        except Exception:
            pass
        self.logger.info(f"🌐 Attempting GLOBAL one-shot deduplication for {len(all_use_cases)} use cases (thinker model)...")
        
        if len(all_use_cases) < 2:
            self.logger.debug("Skipping deduplication, not enough use cases to compare.")
            return all_use_cases
        
        try:
            use_case_markdown = self._build_dedup_markdown(all_use_cases)
            self.logger.debug(f"Built global dedup markdown table: {len(use_case_markdown):,} chars, {len(all_use_cases)} use cases")
        except Exception as e:
            self.logger.error(f"Failed to create markdown for global dedup: {get_clean_error_message(e)}")
            self.logger.info("⬇️  Falling back to per-domain parallel deduplication...")
            return self._deduplicate_use_cases_by_domain_parallel(all_use_cases, rank=rank)
        
        try:
            thinker_cascade = get_models_by_type("thinker")
            if not thinker_cascade:
                self.logger.warning("No thinker models configured. Falling back to per-domain dedup...")
                return self._deduplicate_use_cases_by_domain_parallel(all_use_cases, rank=rank)
            
            thinker_model = thinker_cascade[0]
            thinker_endpoint = thinker_model["llm_endpoint_name"]
            thinker_context_limit = get_context_limit_chars_for_model(
                thinker_model, getattr(self, 'current_language', 'English')
            )
            
            estimated_prompt_size = self._prompt_overhead_chars("REVIEW_USE_CASES_PROMPT") + len(use_case_markdown) + 1000
            
            if estimated_prompt_size > thinker_context_limit:
                self.logger.warning(
                    f"Global dedup prompt ({estimated_prompt_size:,} chars) exceeds thinker model limit "
                    f"({thinker_context_limit:,} chars for {thinker_model.get('name', thinker_endpoint)}). "
                    f"Falling back to per-domain parallel deduplication..."
                )
                return self._deduplicate_use_cases_by_domain_parallel(all_use_cases, rank=rank)
            
            prompt_vars = {
                "use_case_markdown": use_case_markdown,
                "total_count": len(all_use_cases),
                "generation_instructions_section": self._get_generation_instruction_for_prompt("REVIEW_USE_CASES_PROMPT"),
            }
            
            self.logger.info(
                f"⏳ Sending {len(all_use_cases)} use cases to thinker model "
                f"({thinker_model.get('name', thinker_endpoint)}, {estimated_prompt_size:,} chars)..."
            )
            
            unique_use_cases = self._request_dedup("Deduplicate_Global", prompt_vars, all_use_cases, "Global Deduplication",
                                                   rank=rank, start_from_model=thinker_endpoint)
            self.logger.info(f"✅ Received and parsed the global dedup response")
            return unique_use_cases
            
        except Exception as e:
            groups = self._dedup_groups(all_use_cases)
            if len(groups) == 1 and len(next(iter(groups.values()))) == len(all_use_cases):
                self.logger.error(
                    f"🌐 Global one-shot deduplication failed: {get_clean_error_message(e)}. Per-domain dedup would "
                    f"resend the same single group, so all {len(all_use_cases)} use cases are kept."
                )
                return all_use_cases
            self.logger.warning(
                f"🌐 Global one-shot deduplication failed: {get_clean_error_message(e)}. "
                f"Falling back to per-domain parallel deduplication..."
            )
            return self._deduplicate_use_cases_by_domain_parallel(all_use_cases, rank=rank)

    def _deduplicate_use_cases_by_domain_parallel(self, all_use_cases: list, rank=None) -> list:
        """
        Deduplicate use cases at domain level in parallel.
        Uses the same REVIEW_USE_CASES_PROMPT and shared markdown builder as global dedup (DRY).
        
        Args:
            all_use_cases: List of all use case dictionaries (should be scored first)
            
        Returns:
            List of deduplicated use cases
        """
        import concurrent.futures
        import time
        
        self.logger.info(f"🔄 Starting domain-level parallel deduplication for {len(all_use_cases)} use cases...")
        
        domain_use_cases = self._dedup_groups(all_use_cases)
        
        self.logger.info(f"📊 Grouped use cases into {len(domain_use_cases)} dedup group(s)")
        
        deduplicated_results = []
        
        def dedupe_domain(domain_name, domain_ucs):
            """Deduplicate a single domain's use cases using the shared prompt."""
            if len(domain_ucs) < 2:
                self.logger.debug(f"[{domain_name}] Only {len(domain_ucs)} use case(s), skipping dedup.")
                return domain_ucs
            try:
                self.logger.info(f"[{domain_name}] Deduplicating {len(domain_ucs)} use cases...")
                
                use_case_markdown = self._build_dedup_markdown(domain_ucs)
                
                review_context_limit = get_max_context_chars("English", "REVIEW_USE_CASES_PROMPT")
                estimated_size = self._prompt_overhead_chars("REVIEW_USE_CASES_PROMPT") + len(use_case_markdown) + 1000
                
                if estimated_size > review_context_limit:
                    self.logger.warning(f"[{domain_name}] Domain too large ({estimated_size:,} chars). Keeping all {len(domain_ucs)} use cases.")
                    return domain_ucs
                
                prompt_vars = {
                    "use_case_markdown": use_case_markdown,
                    "total_count": len(domain_ucs),
                    "generation_instructions_section": self._get_generation_instruction_for_prompt("REVIEW_USE_CASES_PROMPT"),
                }
                
                self.logger.info(f"⏳ [{domain_name}] Waiting for LLM response...")
                
                deduplicated = self._request_dedup(f"Deduplicate_Domain_{domain_name}", prompt_vars, domain_ucs,
                                                   f"Domain:{domain_name}", rank=rank)
                return deduplicated
                
            except Exception as e:
                self.logger.error(f"[{domain_name}] Deduplication failed: {get_clean_error_message(e)}. Keeping all {len(domain_ucs)} use cases.")
                return domain_ucs
        
        # ADAPTIVE PARALLELISM: Calculate based on domains and total use cases
        total_use_cases = sum(len(ucs) for ucs in domain_use_cases.values())
        num_domains = len(domain_use_cases)
        
        dedup_parallelism, reason = calculate_adaptive_parallelism(
            "deduplication", self.max_parallelism,
            num_items=total_use_cases,
            num_domains=num_domains,
            is_llm_operation=True, logger=self.logger
        )
        
        # Calculate timeout: 5 minutes per domain + 5 minutes buffer
        overall_timeout = (num_domains * 300 // dedup_parallelism) + 300
        
        log_print(f"\n{'='*80}")
        log_print(f"🔄 DEDUPLICATION: Processing {num_domains} domains ({total_use_cases} total use cases)")
        log_print(f"{'='*80}")
        log_adaptive_parallelism_decision("deduplication", dedup_parallelism, self.max_parallelism, reason)
        log_print(f"Overall timeout: {overall_timeout}s ({overall_timeout//60} min)")
        log_print(f"{'='*80}\n")
        
        with _SafeExecutorContext(max_workers=dedup_parallelism, thread_name_prefix="DomainDedupe", logger=self.logger, name="DomainDedupe") as ctx:
            future_to_domain = {}
            for domain, domain_ucs in domain_use_cases.items():
                future = ctx.submit(dedupe_domain, domain, domain_ucs)
                future_to_domain[future] = domain
            
            completed_count = 0
            completed_domains = set()
            start_time = time.time()
            
            try:
                for future in ctx.completed(future_to_domain, timeout=overall_timeout):
                    domain = future_to_domain[future]
                    elapsed = time.time() - start_time
                    try:
                        domain_deduplicated = future.result(timeout=30)
                        deduplicated_results.extend(domain_deduplicated)
                        completed_count += 1
                        completed_domains.add(domain)
                        log_print(f"[Deduplication] ✓ Domain {completed_count}/{len(domain_use_cases)} complete: {domain} ({elapsed:.1f}s elapsed)")
                    except concurrent.futures.TimeoutError:
                        self.logger.error(f"[{domain}] Result collection timed out - keeping original use cases")
                        deduplicated_results.extend(domain_use_cases.get(domain, []))
                        completed_domains.add(domain)
                    except Exception as e:
                        self.logger.error(f"[{domain}] Failed to collect results: {get_clean_error_message(e)} - keeping original use cases")
                        deduplicated_results.extend(domain_use_cases.get(domain, []))
                        completed_domains.add(domain)
            except concurrent.futures.TimeoutError:
                ctx.mark_timed_out()
                self.logger.error(f"⚠️  Overall deduplication timeout reached ({overall_timeout}s). {completed_count}/{len(domain_use_cases)} domains completed.")
                log_print(f"[Deduplication] ⚠️  TIMEOUT - keeping original use cases for incomplete domains", level="WARNING")
                for domain, domain_ucs in domain_use_cases.items():
                    if domain not in completed_domains:
                        self.logger.warning(f"[{domain}] Timed out - keeping all {len(domain_ucs)} original use cases")
                        deduplicated_results.extend(domain_ucs)
        
        del domain_use_cases
        del future_to_domain
        _force_gc(self.logger, "after deduplication")
        
        total_removed = len(all_use_cases) - len(deduplicated_results)
        removal_pct = (total_removed / len(all_use_cases)) * 100 if all_use_cases else 0
        
        self.logger.info(f"✅ Domain-level deduplication complete: Retained {len(deduplicated_results)} use cases, removed {total_removed} ({removal_pct:.1f}%)")
        
        return deduplicated_results

    def _translate_and_prepare_language_pack(self, lang: str, flat_english_use_cases: list, english_grouped_data: dict, business_name: str,
                                             batch_parallelism: int = 1) -> tuple:
        """
        A single-function wrapper to run all data-gathering for a language.
        Designed to be run in a ThreadPoolExecutor.
        """
        lang_abbr = self._get_lang_abbr(lang)
        try:
            # Set language for context limit calculations
            self.ai_agent.set_language(lang)
            
            self.logger.info(f"Starting translation & summary pack for {lang}...")
            lang_translations = self.translation_service.get_translations(lang)
            lang_use_cases_translated = self.translation_service.translate_use_case_list(
                list(flat_english_use_cases), lang, max_parallelism=batch_parallelism, enable_parallelization=batch_parallelism > 1)
            lang_grouped_data = self._align_translated_data(english_grouped_data, lang_use_cases_translated)
            self._record_translation_gaps(lang, self._translation_gap_summary(lang_grouped_data))
            (lang_summary_dict, transliterated_name) = self._get_salesy_summary(lang_grouped_data, business_name, lang, lang_translations)
            
            self.logger.debug(f"Successfully processed all data for {lang}.")
            return (lang, lang_abbr, lang_translations, lang_grouped_data, lang_summary_dict, transliterated_name)
        except Exception as e:
            self.logger.error(f"Failed to process translation artifacts for {lang}: {get_clean_error_message(e)}")
            self._record_failed_step("Language Pack", lang, e)
            return (lang, lang_abbr, None, None, None, None) # Return Nones to signal failure

    @staticmethod
    def _translation_gap_summary(grouped_data: dict) -> dict:
        fields = {}
        with_gaps = 0
        untranslated = 0
        for use_cases in (grouped_data or {}).values():
            for uc in use_cases:
                gaps = uc.get("_untranslated_fields") or []
                if uc.get("_translation_status") == "untranslated":
                    untranslated += 1
                if gaps:
                    with_gaps += 1
                    for field in gaps:
                        fields[field] = fields.get(field, 0) + 1
        if not with_gaps:
            return {}
        return {"use_cases": with_gaps, "untranslated_use_cases": untranslated, "fields": dict(sorted(fields.items()))}

    def _generate_documents_for_all_languages(self, final_consolidated_use_cases: list, english_grouped_data: dict = None, summary_dict: dict = None, languages: list = None, skip_excel_langs: list = None):
        """
        Generates PDF/PPTX/Excel documents for all languages.
        Can be called from normal path (after use case generation) or docs-only path (from JSON).
        
        Args:
            final_consolidated_use_cases: List of use cases (flat)
            english_grouped_data: Optional, will be computed if not provided
            summary_dict: Optional, will be computed if not provided
            languages: Optional list of languages to generate (default: self.output_languages)
        """
        target_languages = languages if languages else self.output_languages
        skip_excel_langs = set(skip_excel_langs or [])
        self.logger.info(f"--- Starting Document Generation for Languages: {target_languages} ---")
        
        # CRITICAL: Install all required dependencies BEFORE starting translations
        # This prevents wasting time on translations if dependencies are missing
        self.logger.info("Checking required document packages before starting translations (each is installed at most once per run)...")
        required_packages = []
        if "PDF Catalog" in self.generate_choices or "Use Cases Catalog PDF" in self.generate_choices or ("English" in (target_languages or [])):
            required_packages.append(("weasyprint", "weasyprint", "PDF Catalog"))
        if "Presentation" in self.generate_choices:
            required_packages.append(("pptx", "python-pptx", "Presentation"))
        if "English" in (target_languages or []) and "English" not in skip_excel_langs:
            required_packages.append(("xlsxwriter", "xlsxwriter", "Use Case Excel"))
        for module_name, pip_name, purpose in required_packages:
            try:
                self._require_python_package(module_name, pip_name, purpose)
            except RuntimeError as dependency_error:
                self.logger.warning(f"⚠️ {get_clean_error_message(dependency_error)}; the {purpose} step will be recorded as failed, "
                                    f"the Markdown and CSV catalogs are still generated.")
        
        # Prepare English data if not provided
        flat_english_use_cases = final_consolidated_use_cases
        if english_grouped_data is None:
            english_grouped_data = self._group_use_cases_by_domain_flat(flat_english_use_cases)
        non_english_languages = [lang for lang in target_languages if lang != "English"]
        if non_english_languages:
            self.logger.warning(f"[VIBE-LANG english-only-artifacts] The Excel, CSV and Markdown catalogs are produced in English only; "
                                f"{', '.join(non_english_languages)} receive the PDF Catalog and the Presentation.")
        
        # Get English translations
        english_translations = self.translation_service.get_translations("English")
        
        # Get summary if not provided
        if summary_dict is None:
            (summary_dict, transliterated_name) = self._get_salesy_summary(english_grouped_data, self.business_name, "English", english_translations)
        else:
            transliterated_name = self.business_name
        
        # STEP 1: Run translations in parallel (no nesting)
        # ADAPTIVE PARALLELISM: Calculate based on languages and use cases
        num_languages = len([l for l in target_languages if l != "English"])
        num_use_cases = len(flat_english_use_cases)
        
        translation_parallelism, reason = calculate_adaptive_parallelism(
            "translation", self.max_parallelism,
            num_items=num_use_cases,
            num_domains=num_languages,
            is_llm_operation=True, logger=self.logger,
            estimated_mb_per_worker=15.0
        )
        log_adaptive_parallelism_decision("translation", translation_parallelism, self.max_parallelism, reason)
        
        translation_futures = []
        translation_results = {}
        
        _lang_timed_out = False
        with _SafeExecutorContext(max_workers=translation_parallelism, thread_name_prefix="Translator", logger=self.logger, name="LangPack") as ctx:
            for lang in target_languages:
                if lang == "English":
                    self.logger.info("Preparing English artifacts (no translation needed).")
                    lang_abbr = self._get_lang_abbr("English")
                    translation_results["English"] = ("English", lang_abbr, english_translations, english_grouped_data, summary_dict, transliterated_name)
                    continue

                self.logger.info(f"Submitting translation & summary pack job for {lang}...")
                f = ctx.submit(
                    self._translate_and_prepare_language_pack,
                    lang, flat_english_use_cases, english_grouped_data, self.business_name,
                    batch_parallelism=max(1, translation_parallelism // max(1, num_languages))
                )
                translation_futures.append((f, lang))

            self.logger.info(f"Waiting for {len(translation_futures)} language packs to complete...")
            total_timeout = len(translation_futures) * 1800
            self.logger.info(f"Language pack processing timeout set to {total_timeout}s ({total_timeout//60} minutes)")
            
            for future, lang in translation_futures:
                try:
                    result = _safe_future_result(future, timeout=1800)
                    (lang, lang_abbr, lang_translations, lang_grouped_data, lang_summary_dict, transliterated_name) = result
                    
                    if lang_translations is None:
                        self.logger.warning(f"Skipping artifact generation for {lang} due to translation/summary failure.")
                        continue

                    self.logger.info(f"Translation pack for {lang} complete.")
                    translation_results[lang] = result
                except (concurrent.futures.TimeoutError, TimeoutError):
                    _lang_timed_out = True
                    ctx.cancel(future)
                    self.logger.error(f"Language pack processing timed out after 30 minutes for {lang}; only {lang} is cancelled, "
                                      f"the other languages keep running")
                    self._record_failed_step("Language Pack", lang, "translation and summary timed out after 30 minutes")
                except Exception as e:
                    self.logger.error(f"Language pack processing failed for {lang}: {get_clean_error_message(e)}")
                    self._record_failed_step("Language Pack", lang, e)
        
        # STEP 2: Run artifact writing in parallel (separate, not nested)
        # ADAPTIVE PARALLELISM: Calculate based on number of artifacts to write
        num_artifacts = len(translation_results) * 3  # Roughly: PDF, PPTX, Excel per language
        
        writing_parallelism, reason = calculate_adaptive_parallelism(
            "artifact_writing", self.max_parallelism,
            num_items=num_artifacts,
            is_llm_operation=False, logger=self.logger
        )
        log_adaptive_parallelism_decision("artifact_writing", writing_parallelism, self.max_parallelism, reason)
        self.logger.info(f"Translations complete. Starting artifact generation for {len(translation_results)} languages...")
        
        with _SafeExecutorContext(max_workers=writing_parallelism, thread_name_prefix="Writer", logger=self.logger, name="ArtifactWriter") as writer_ctx:
            writing_futures = []
            
            for lang, result in translation_results.items():
                (lang, lang_abbr, lang_translations, lang_grouped_data, lang_summary_dict, transliterated_name) = result
                
                self.logger.info(f"Submitting writing jobs for {lang}...")
                
                if lang == "English":
                    f = writer_ctx.submit(self._generate_markdown_catalog, lang, lang_abbr, lang_grouped_data, lang_summary_dict, transliterated_name)
                    step_id = self._emit_pipeline_status(
                        stage_name="Excel Generation",
                        step_name="Markdown Catalog",
                        sub_step_name=lang,
                        message=f"{lang} markdown generation started",
                        status="started",
                        progress_increment=1.0
                    )
                    writing_futures.append((f, f"{lang} Markdown", step_id, "Markdown Catalog", lang))
                    f = writer_ctx.submit(self._generate_csv_catalog, lang, lang_abbr, lang_grouped_data)
                    step_id = self._emit_pipeline_status(
                        stage_name="Excel Generation",
                        step_name="CSV Catalog",
                        sub_step_name=lang,
                        message=f"{lang} CSV generation started",
                        status="started",
                        progress_increment=1.0
                    )
                    writing_futures.append((f, f"{lang} CSV", step_id, "CSV Catalog", lang))
                
                _pdf_requested = "PDF Catalog" in self.generate_choices or "Use Cases Catalog PDF" in self.generate_choices
                _force_english_pdf = (lang == "English") and ("English" in target_languages)
                if _pdf_requested or _force_english_pdf:
                    if _force_english_pdf:
                        _pdf_mode = "override" if not _pdf_requested else "reinforce"
                        self.logger.info(f"[VIBE-LANG FORCE-PDF FIRED] language=English reason=english_in_output_languages mode={_pdf_mode}")
                    f = writer_ctx.submit(self.generate_catalog_pdf, lang, lang_abbr, lang_translations, lang_summary_dict, lang_grouped_data, transliterated_name)
                    step_id = self._emit_pipeline_status(
                        stage_name="Document Generation",
                        step_name="PDF Catalog",
                        sub_step_name=lang,
                        message=f"{lang} PDF generation started",
                        status="started",
                        progress_increment=1.0
                    )
                    writing_futures.append((f, f"{lang} PDF", step_id, "PDF Catalog", lang))
                if "Presentation" in self.generate_choices:
                    f = writer_ctx.submit(self.generate_presentation_pptx, lang, lang_abbr, lang_translations, lang_summary_dict, lang_grouped_data, transliterated_name)
                    step_id = self._emit_pipeline_status(
                        stage_name="Document Generation",
                        step_name="Presentation",
                        sub_step_name=lang,
                        message=f"{lang} presentation generation started",
                        status="started",
                        progress_increment=1.0
                    )
                    writing_futures.append((f, f"{lang} PPTX", step_id, "Presentation", lang))
                if lang == "English" and lang not in skip_excel_langs:
                    f = writer_ctx.submit(self._generate_use_case_excel, lang, lang_abbr, lang_grouped_data)
                    step_id = self._emit_pipeline_status(
                        stage_name="Excel Generation",
                        step_name="Use Case Excel",
                        sub_step_name=lang,
                        message=f"{lang} Excel generation started",
                        status="started",
                        progress_increment=1.0
                    )
                    writing_futures.append((f, f"{lang} Excel", step_id, "Use Case Excel", lang))
                elif lang == "English":
                    self.logger.info(f"Skipping Excel generation for {lang} (already generated).")
                else:
                    self.logger.info(f"Skipping Excel generation for {lang} (English only).")
            
            _artifact_successes = set()
            for future, job_name, start_step_id, step_name, lang in writing_futures:
                try:
                    _safe_future_result(future, timeout=600)
                    self.logger.info(f"✓ {job_name} completed")
                    _artifact_successes.add(job_name)
                    self._emit_pipeline_status(
                        stage_name="Artifact Writing",
                        step_name=step_name,
                        sub_step_name=lang,
                        message=f"{job_name} completed",
                        status="ended_success",
                        progress_increment=1.0,
                        result_json={"job_name": job_name, "artifact_type": step_name, "language": lang},
                        step_id=start_step_id
                    )
                except Exception as e:
                    self.logger.error(f"✗ {job_name} failed: {get_clean_error_message(e)}")
                    self._record_failed_step(step_name, lang, e)
                    self._emit_pipeline_status(
                        stage_name="Artifact Writing",
                        step_name=step_name,
                        sub_step_name=lang,
                        message=f"{job_name} failed",
                        status="ended_error",
                        progress_increment=1.0,
                        result_json={"job_name": job_name, "artifact_type": step_name, "language": lang, "error": get_clean_error_message(e, max_chars=300)},
                        step_id=start_step_id
                    )
        
        if "English" in target_languages:
            _english_md_scheduled = any((_l == "English" and _n == "English Markdown") for (_f, _n, _s, _st, _l) in writing_futures)
            _english_pdf_scheduled = any((_l == "English" and _n == "English PDF") for (_f, _n, _s, _st, _l) in writing_futures)
            _english_md_ok = _english_md_scheduled and ("English Markdown" in _artifact_successes)
            _english_pdf_ok = _english_pdf_scheduled and ("English PDF" in _artifact_successes)
            if _english_md_ok and _english_pdf_ok:
                self.logger.info("[VIBE-LANG all-artifacts-produced] language=English md=1 pdf=1")
            else:
                self.logger.warning(
                    f"[VIBE-LANG missing-artifacts] language=English "
                    f"md_scheduled={_english_md_scheduled} md_ok={_english_md_ok} "
                    f"pdf_scheduled={_english_pdf_scheduled} pdf_ok={_english_pdf_ok}"
                )
        
        self.logger.info("All artifact writing jobs completed.")

    def _run_log_name(self, stem: str, extension: str) -> str:
        if getattr(self, "generate_genie_instructions", False):
            kind, stamp = "generate", ""
        elif getattr(self, "operation", None) == IMPLEMENT_OPERATION:
            kind, stamp = "implement", self._implement_param(IMPLEMENT_RUN_ID_PARAM)
        else:
            return f"{stem}.{extension}"
        if not re.fullmatch(r"[0-9]{1,19}", stamp or ""):
            stamp = getattr(self, "_generate_run_stamp", None)
            if not stamp:
                import time
                stamp = self._generate_run_stamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
        return f"{stem}_{kind}_{stamp}.{extension}"

    def _resolve_live_log_target(self):
        """Pick a destination for the live log file. Priority (v0.8.5):
          1. UC Volume derived from inspire_database (best: external `databricks fs cat` tailing)
          2. Fallback: `<base_output_dir>/.inspire_logs/session_<sid>/log.log` under the
             user-provided generation_path (works for both /Volumes/... and /Workspace/...)
          3. None → caller logs a loud warning and bails.

        Returns dict with keys: dst_path, kind ('volume'|'workspace'), parent_dir.
        Returns None when no destination can be resolved (caller logs guidance)."""
        import os, re
        biz = re.sub(r"[^A-Za-z0-9._-]+", "_", str(getattr(self, "business_name", "unknown") or "unknown"))
        sid = str(getattr(self, "session_id", "unknown") or "unknown")

        reasons = []
        self._live_log_unavailable_reasons = reasons
        db = (getattr(self, "inspire_database", None) or "").strip()
        parts = [p.strip("`") for p in db.split(".")] if db else []
        if len(parts) >= 2 and parts[0] and parts[1]:
            catalog, schema = parts[0], parts[1]
            try:
                self.spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.vol_root")
                vol_base = f"/Volumes/{catalog}/{schema}/vol_root/logs/{biz}/session_{sid}"
                os.makedirs(vol_base, exist_ok=True)
                return {"dst_path": os.path.join(vol_base, self._run_log_name("log", "log")), "kind": "volume", "parent_dir": vol_base}
            except Exception as _cve:
                reasons.append(f"volume {catalog}.{schema}.vol_root: {get_clean_error_message(_cve)}")
                self.logger.warning(f"UC volume target unavailable, falling back to generation_path: {get_clean_error_message(_cve)}")
        else:
            reasons.append("02_inspire_database is not set")

        base = canonical_workspace_path(getattr(self, "base_output_dir", None) or "")
        if base:
            parent = os.path.join(base, ".inspire_logs", f"session_{sid}")
            kind = "workspace" if is_workspace_path(base) else "volume"
            if kind == "volume":
                try:
                    os.makedirs(parent, exist_ok=True)
                except Exception as _me:
                    reasons.append(f"{parent}: {get_clean_error_message(_me)}")
                    self.logger.warning(f"Live log fallback parent dir create failed: {get_clean_error_message(_me)}")
                    return None
            else:
                # v0.8.6 fix: workspace.import_ does NOT auto-create parents —
                # without this mkdirs every 10s flush silently fails with
                # "parent folder does not exist" (debug-level log).
                try:
                    self.w_client.workspace.mkdirs(parent)
                except Exception as _wme:
                    reasons.append(f"{parent}: {get_clean_error_message(_wme)}")
                    self.logger.warning(f"Live log workspace mkdirs failed: {get_clean_error_message(_wme)}")
                    return None
            return {"dst_path": os.path.join(parent, self._run_log_name("log", "log")), "kind": kind, "parent_dir": parent}

        reasons.append("11_generation_path is not set")
        return None

    def _flush_live_log_once(self, src_log: str, dst_log: str, kind: str, state: dict) -> str:
        """Bring the live log target up to date with the local log. An unchanged source is not
        re-sent. A changed source is written whole: a UC Volume file takes one sequential write
        from a fresh open (append and seek fail with Illegal seek), and the workspace import API
        has no append.
        Returns 'missing', 'unchanged' or 'rewritten'."""
        import os, base64
        if not os.path.exists(src_log):
            return "missing"
        stat = os.stat(src_log)
        signature = (stat.st_size, stat.st_mtime_ns)
        if state.get("signature") == signature and (kind == "workspace" or os.path.exists(dst_log)):
            return "unchanged"
        if kind == "workspace":
            with open(src_log, "rb") as _f:
                _b64 = base64.b64encode(_f.read()).decode()
            self.w_client.workspace.import_(
                path=dst_log, content=_b64,
                format=workspace.ImportFormat.AUTO, overwrite=True
            )
        else:
            copy_file_sequentially(src_log, dst_log)
        state["signature"] = signature
        return "rewritten"

    def _start_live_log_flush_thread(self):
        """Copy local log.txt to a live-tailable destination every 10s so external
        observers (humans, agents, CI) can read the run line-by-line without
        waiting for `_upload_log_file` at the very end.

        v0.8.5: now ALWAYS fires (was: only when inspire_database widget set).
        Destination resolution order in `_resolve_live_log_target`:
          1. UC Volume from inspire_database (preferred — external SQL/CLI tail)
          2. `<generation_path>/.inspire_logs/session_<sid>/log.log` (workspace OR volume)
        Cadence: 10s (was 60s in v0.8.3).

        Backward-compat: keeps writing to `_volume_log_*` instance attrs so the
        teardown in `_upload_log_file` works unchanged."""
        import threading, os
        try:
            target = self._resolve_live_log_target()
            if not target:
                reasons = "; ".join(getattr(self, "_live_log_unavailable_reasons", None) or ["no target resolved"])
                self.logger.warning(
                    f"🔇 Live log streaming DISABLED ({reasons}). External tailers will see nothing until "
                    "_upload_log_file runs at the very end of the pipeline. To enable live monitoring, let this "
                    "run's identity write to the vol_root volume of 02_inspire_database or to 11_generation_path."
                )
                return
            dst_log = target["dst_path"]
            kind = target["kind"]
            src_log = os.path.join(self.local_log_output_dir, "log.txt")
            self._volume_log_dst = dst_log
            self._volume_log_kind = kind
            self._volume_log_stop = threading.Event()
            flush_state = {}
            ailogs_dst = os.path.join(target["parent_dir"], self._run_log_name("ailogs", "jsonl"))
            self._ailogs_live_copy_dst = ailogs_dst
            ailogs_state = {}

            def _extra_files():
                ailogs_src = self._ailogs_live_copy_source()
                return [(ailogs_src, ailogs_dst, ailogs_state)] if ailogs_src else []

            self._live_log_fail_count = 0
            def _loop():
                while not self._volume_log_stop.is_set():
                    try:
                        self._flush_live_log_once(src_log, dst_log, kind, flush_state)
                        for _src, _dst, _state in _extra_files():
                            self._flush_live_log_once(_src, _dst, kind, _state)
                        if self._live_log_fail_count:
                            self.logger.info(f"Live log flush recovered after {self._live_log_fail_count} consecutive failure(s)")
                        self._live_log_fail_count = 0
                    except Exception as _ce:
                        try:
                            self._live_log_fail_count += 1
                            _msg = f"Live log flush failed (n={self._live_log_fail_count}): {get_clean_error_message(_ce)}"
                            if self._live_log_fail_count == 1 or self._live_log_fail_count % 30 == 0:
                                self.logger.warning(_msg)
                            else:
                                self.logger.debug(_msg)
                        except Exception:
                            pass
                    if self._volume_log_stop.wait(timeout=10.0):
                        break
                for _src, _dst, _state in [(src_log, dst_log, flush_state)] + _extra_files():
                    try:
                        self._flush_live_log_once(_src, _dst, kind, _state)
                    except Exception:
                        pass

            t = threading.Thread(target=_loop, name=f"LiveLogFlush_{getattr(self, 'session_id', 'x')}", daemon=True)
            t.start()
            self._volume_log_thread = t
            self.logger.info(f"📝 Live log streaming enabled (kind={kind}, 10s flush) → {dst_log}")
        except Exception as _e:
            try:
                self.logger.warning(f"Live log streaming setup failed: {get_clean_error_message(_e)}")
            except Exception:
                pass

    # v0.8.4 alias — keep old name for any internal caller that referenced it.
    def _start_volume_log_flush_thread(self):
        return self._start_live_log_flush_thread()

    def _log_funnel_stage(self, stage: str, n_in: int, n_out: int, drops_by_reason: dict = None):
        drops = {str(k): int(v) for k, v in (drops_by_reason or {}).items() if v}
        entry = {"stage": stage, "in": int(n_in), "out": int(n_out), "drops": drops}
        ledger = getattr(self, "funnel_ledger", None)
        if not isinstance(ledger, list):
            ledger = []
            self.funnel_ledger = ledger
        ledger.append(entry)
        self.logger.info(f"[FUNNEL] stage={stage} in={int(n_in)} out={int(n_out)} drops={json.dumps(drops, sort_keys=True)}")
        return entry

    _GRADING_CONTEXT_KEYS = ("business_context", "industry", "strategic_initiative", "value_chain", "revenue_model")

    @classmethod
    def _grading_context(cls, context) -> dict:
        """The business context the graders receive. Goals, priorities and the strategy tree never pass."""
        context = context or {}
        return {key: context.get(key) or "" for key in cls._GRADING_CONTEXT_KEYS}

    def _compute_inspire_score(self, uc: dict):
        if uc.get("scoring_status") != "scored":
            return None
        try:
            quality = float(uc.get("Quality Score"))
            priority = float(uc.get("Priority"))
        except (TypeError, ValueError):
            return None
        if quality != quality or priority != priority:
            return None
        rt = TECHNICAL_CONTEXT["runtime"]
        quality_normalized = min(10.0, max(0.0, quality * 2.0))
        priority_normalized = min(10.0, max(0.0, priority))
        return round(rt["inspire_score_quality_weight"] * quality_normalized
                     + rt["inspire_score_priority_weight"] * priority_normalized, 2)

    def _apply_inspire_scores(self, use_cases: list) -> list:
        """Inspire Score = quality_weight x (Quality Score x 2) + priority_weight x Priority, both
        on 0-10, from the absolute combined-scoring values. A scored UC whose inputs are missing is
        marked unscored instead of receiving a default."""
        rt = TECHNICAL_CONTEXT["runtime"]
        missing = []
        for uc in use_cases:
            if uc.get("scoring_status") != "scored":
                continue
            score = self._compute_inspire_score(uc)
            if score is None:
                self._mark_unscored(uc, "Inspire Score inputs (Quality Score, Priority) missing")
                missing.append(str(uc.get("No", "")))
            else:
                uc["Inspire Score"] = score
        if missing:
            self.logger.warning(f"⚠️ {len(missing)} UC(s) lack Inspire Score inputs and are marked unscored: {missing[:20]}")
        scores = [uc["Inspire Score"] for uc in use_cases if uc.get("scoring_status") == "scored"]
        quality_distribution = {}
        for uc in use_cases:
            label = uc.get("Quality") or "Unscored"
            quality_distribution[label] = quality_distribution.get(label, 0) + 1
        log_print(f"\n📊 DATA QUALITY DISTRIBUTION:")
        for label in self._VALID_QUALITY_LABELS + ("Unscored",):
            if quality_distribution.get(label, 0) > 0:
                log_print(f"   {label}: {quality_distribution[label]} use cases")
        result = {
            "total_use_cases": len(use_cases),
            "scored_use_cases": len(scores),
            "quality_weight": rt["inspire_score_quality_weight"],
            "priority_weight": rt["inspire_score_priority_weight"],
            "quality_distribution": quality_distribution,
            "inspire_score_min": round(min(scores), 2) if scores else 0,
            "inspire_score_max": round(max(scores), 2) if scores else 0,
            "inspire_score_avg": round(sum(scores) / len(scores), 2) if scores else 0,
        }
        log_print(f"\n📊 INSPIRE SCORES COMPUTED (weights: quality={result['quality_weight']}, priority={result['priority_weight']})")
        self.logger.info(f"✅ Inspire Scores computed for {len(scores)}/{len(use_cases)} use cases")
        try:
            self._emit_pipeline_status(
                stage_name="Scoring",
                step_name="Inspire Score",
                sub_step_name="Computation",
                message=f"Inspire Scores computed for {len(scores)} use cases (avg={result['inspire_score_avg']}, min={result['inspire_score_min']}, max={result['inspire_score_max']})",
                status="ended_success",
                progress_increment=1.0,
                result_json=result,
            )
        except Exception:
            pass
        return use_cases

    def _apply_best_of_best_selection(self, selected: list, schema_details: list, context: dict) -> list:
        """Best-of-Best on the label-filtered set when it has at least bob_min_threshold UCs:
        BoB-unscored UCs and hard-vetoed UCs are dropped; a set larger than
        aggressive_prune_threshold is stratified-trimmed by idea theme."""
        from collections import Counter
        rt = TECHNICAL_CONTEXT["runtime"]
        bob_min = rt["bob_min_threshold"]
        trigger = rt["aggressive_prune_threshold"]
        n_in = len(selected)
        if n_in < bob_min:
            self.logger.info(
                f"BoB scoring skipped: portfolio has {n_in} UCs < bob_min_threshold {bob_min}; "
                f"bob_score will be NULL for this run (insufficient UCs to discriminate)."
            )
            return selected
        will_trim = n_in > trigger
        target_max_llm = self._trim_target(rt)
        if will_trim:
            log_print(
                f"\U0001F3C6 BEST-OF-BEST + STRATIFIED TRIM: {n_in} UCs > prune threshold {trigger}. "
                f"Scoring each UC on 19 gates (0-10), naming idea themes across the portfolio, then trimming per theme to ~{target_max_llm}."
            )
        else:
            log_print(
                f"\U0001F3C6 BEST-OF-BEST SCORING ONLY: {n_in} UCs \u2265 bob_min {bob_min} "
                f"but \u2264 prune threshold {trigger} \u2014 scoring without trim so bob_score is populated for every UC."
            )
        try:
            scored = self._score_best_of_best(selected, schema_details, **self._grading_context(context))
        except Exception as e:
            self.logger.error(
                f"BoB scoring failed: {get_clean_error_message(e)}; keeping the {n_in} combined-scored UCs without bob scores"
            )
            return selected
        bob_unscored = [uc for uc in scored if uc.get("bob_status") == "unscored"]
        for uc in bob_unscored:
            uc["selection_drop_reason"] = f"best-of-best unscored: {uc.get('bob_unscored_reason') or 'no valid gate scores'}"
        kept = [uc for uc in scored if uc.get("bob_status") != "unscored"]
        self._log_funnel_stage("bob_scoring", n_in, len(kept), {"unscored": len(bob_unscored)})
        survivors = self._drop_hard_veto_ucs(kept)
        survivor_ids = {id(uc) for uc in survivors}
        vetoed = [uc for uc in kept if id(uc) not in survivor_ids]
        for uc in vetoed:
            uc["selection_drop_reason"] = f"hard veto on gate {uc.get('bob_hard_veto_gate')}"
        self._log_funnel_stage("bob_hard_veto", len(kept), len(survivors),
                               dict(Counter(str(uc.get("bob_hard_veto_gate")) for uc in vetoed)))
        if not will_trim:
            return survivors
        trimmed = self._stratified_trim_by_theme(
            survivors, target_max=target_max_llm, theme_max_share=rt["bob_theme_max_share"]
        )
        trimmed_ids = {id(uc) for uc in trimmed}
        trimmed_out = [uc for uc in survivors if id(uc) not in trimmed_ids]
        for uc in trimmed_out:
            uc["selection_drop_reason"] = "trimmed by the stratified theme cap"
        self._log_funnel_stage("stratified_trim", len(survivors), len(trimmed), {"theme_cap": len(trimmed_out)})
        if trimmed_out:
            try:
                self._emit_pipeline_status(
                    stage_name="Use Case Prioritization",
                    step_name="LLM Aggressive Consolidation",
                    sub_step_name="Complete", status="ended_success",
                    message=f"Stratified trim: {len(survivors)} \u2192 {len(trimmed)}",
                    result_json={"before": len(survivors), "after": len(trimmed), "target_max": target_max_llm}
                )
            except Exception:
                pass
        return trimmed

    @classmethod
    def _post_bob_dedup_rank(cls, uc) -> tuple:
        return cls._score_rank_part(uc.get("bob_score")) + cls._score_rank_part(uc.get("Priority"))

    def _dedup_canonical_titles(self, use_cases: list) -> list:
        """Post-BoB safety net: UCs that share `_uc_dedup_signature` (the duplicate-decision
        definition G2, Q2 and Phase L L9 use) collapse to one. Keeps the higher bob_score
        (tie-break: higher Priority, then generation order). Idempotent on clean portfolios."""
        n_in = len(use_cases)
        try:
            kept, merge_log = self._dedup_by_signature(use_cases, "duplicate decision", rank=self._post_bob_dedup_rank)
            if not merge_log:
                self.logger.info(
                    f"Canonical-title dedup safety net: 0 duplicate decisions found across {n_in} UCs (clean)."
                )
                self._log_funnel_stage("canonical_title_dedup", n_in, n_in)
                return use_cases
            kept_ids = {id(uc) for uc in kept}
            _dedup_dropped = [uc for uc in use_cases if id(uc) not in kept_ids]
            kept_twin = {entry["dropped"]: entry["kept"] for entry in merge_log}
            kept_by_no = {str(uc.get("No", "")): uc for uc in kept}
            for uc in _dedup_dropped:
                title = self._uc_title(kept_by_no.get(str(kept_twin.get(uc.get("No", "UNK"), ""))))
                uc["selection_drop_reason"] = f'duplicate decision of "{title}"' if title else "duplicate decision of a kept use case"
            _dropped_summary = [
                f"{uc.get('No','?')}:{(uc.get('Name') or uc.get('use_case') or '')[:60]}"
                for uc in _dedup_dropped[:5]
            ]
            log_print(
                f"\U0001F9F9 CANONICAL-TITLE DEDUP SAFETY NET: dropped {len(_dedup_dropped)} duplicate-decision UC(s). "
                f"Kept {len(kept)}/{n_in}. Sample drops: {_dropped_summary}"
            )
            self.logger.warning(
                f"Canonical-title dedup safety net dropped {len(_dedup_dropped)} UC(s) the LLM dedup missed: "
                f"{[uc.get('No') for uc in _dedup_dropped]}"
            )
            self._log_funnel_stage("canonical_title_dedup", n_in, len(kept), {"duplicate_decision": len(_dedup_dropped)})
            return kept
        except Exception as _ddp_err:
            self.logger.warning(
                f"Canonical-title dedup safety net failed non-fatally: {get_clean_error_message(_ddp_err)}"
            )
            self._log_funnel_stage("canonical_title_dedup", n_in, n_in)
            return use_cases

    @staticmethod
    def _finite_or(value, default):
        if isinstance(value, bool):
            return default
        try:
            number = float(value)
        except (TypeError, ValueError, OverflowError):
            return default
        return number if number == number and abs(number) != float("inf") else default

    @classmethod
    def _score_rank_part(cls, value) -> tuple:
        number = cls._finite_or(value, None)
        return (1, 0.0) if number is None else (0, -number)

    @classmethod
    def _coverage_pick_rank(cls, uc: dict) -> tuple:
        not_vetoed = uc.get("bob_hard_veto_gate") in (None, "", "None")
        return (not_vetoed, cls._finite_or(uc.get("elite_score"), float("-inf")),
                cls._finite_or(uc.get("Inspire Score"), float("-inf")), cls._finite_or(uc.get("bob_score"), float("-inf")))

    def _apply_domain_coverage_picks(self, selected: list, pool: list, user_domains: list) -> list:
        """CLAUDE.md §3b: every user-specified Business Domain keeps at least one UC. For each such
        domain (case-insensitive exact match on 'Business Domain') with no UC in `selected`, the
        best-scored scored UC of that domain in `pool` is added with
        elite_status='DOMAIN_COVERAGE_PICK' and a short coverage_pick_reason. At most one pick per
        empty domain; unscored UCs are never picked; no-op when user_domains is empty."""
        if not user_domains:
            return selected
        result = list(selected)
        covered = {str(uc.get("Business Domain") or "").strip().lower() for uc in result}
        taken = {id(uc) for uc in result}
        for domain in user_domains:
            key = str(domain or "").strip().lower()
            if not key or key in covered:
                continue
            candidates = [
                uc for uc in (pool or [])
                if id(uc) not in taken
                and str(uc.get("Business Domain") or "").strip().lower() == key
                and uc.get("scoring_status") == "scored"
                and uc.get("bob_status") != "unscored"
            ]
            if not candidates:
                self.logger.warning(
                    f"[DOMAIN_COVERAGE] User-specified domain '{domain}' has no scored candidate UC; it cannot be covered"
                )
                continue
            pick = max(candidates, key=self._coverage_pick_rank)
            pick["elite_status"] = "DOMAIN_COVERAGE_PICK"
            pick["coverage_pick_reason"] = f"Domain coverage pick, below bar: {pick.get('selection_drop_reason') or 'not selected by the quality bar'}"
            result.append(pick)
            taken.add(id(pick))
            covered.add(key)
            self.logger.warning(
                f"[DOMAIN_COVERAGE] Added {pick.get('No')} '{pick.get('Name', '')}' to cover user domain '{domain}' "
                f"({pick['coverage_pick_reason']})"
            )
        return result

    @staticmethod
    def _elite_has_fatal_flaw(assessment: dict) -> bool:
        a = assessment if isinstance(assessment, dict) else {}
        verification = a.get("verification") if isinstance(a.get("verification"), dict) else {}
        return bool(a.get("fatal_flaws")) or not verification.get("passed")

    def _label_fallback_pick(self, uc: dict, mode: str, kind: str, reason: str) -> None:
        uc["elite_status"] = "FALLBACK_BELOW_BAR"
        uc["fallback_bar"] = mode
        uc["fallback_kind"] = kind
        uc["fallback_reason"] = reason

    def _rank_fallback_picks(self, use_cases: list) -> list:
        return sorted(use_cases, key=lambda u: (self._score_rank_part(u.get("elite_score"))
                                                + self._score_rank_part(u.get("Inspire Score"))
                                                + self._score_rank_part(u.get("bob_score"))))

    def _record_fallback_outcome(self, mode: str, kind: str, picks: list) -> dict:
        self._selection_is_fallback = True
        self._selection_fallback_bar = mode
        message = (f"0 use cases met the {mode} bar; no use case was padded in — these {len(picks)} "
                   f"candidate(s) are the closest and ship labelled below the {mode} bar")
        outcome = {
            "accepted": 0, "mode": mode, "fallback": True, "fallback_kind": kind,
            "fallback_count": len(picks), "message": message,
            "picks": [{"No": uc.get("No"), "Name": uc.get("Name"), "Business Domain": uc.get("Business Domain"),
                       "Inspire Score": uc.get("Inspire Score"), "elite_score": uc.get("elite_score"),
                       "Quality": uc.get("Quality"), "reason": uc.get("fallback_reason")} for uc in picks],
        }
        self.fallback_outcome = outcome
        log_print(f"\n{'='*80}")
        log_print(f"🟡 {message}")
        for idx, uc in enumerate(picks, 1):
            log_print(f"   {idx}. {uc.get('No')} | {uc.get('Name')} | {uc.get('fallback_reason')}")
        log_print(f"{'='*80}")
        self.logger.warning(f"🟡 ZERO-AT-BAR FALLBACK [{mode}/{kind}]: {message}")
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Quality Bar", sub_step_name="Zero-At-Bar Fallback",
                message=message, status="ended_warning", progress_increment=0.0, result_json=outcome)
        except Exception:
            pass
        return outcome

    def _fallback_tier_candidates(self, mode: str, candidates: list) -> list:
        if mode == "Good":
            return list(candidates)
        for tier in ("Medium", "Low"):
            picks = [uc for uc in candidates if self._canonical_quality_label(uc.get("Quality")) == tier]
            if picks:
                return picks
        return []

    def _apply_zero_bar_fallback(self, mode: str, candidates: list) -> list:
        picks = self._rank_fallback_picks(self._fallback_tier_candidates(mode, candidates))
        for uc in picks:
            self._label_fallback_pick(uc, mode, "tier",
                                      uc.get("selection_drop_reason") or f"quality '{uc.get('Quality')}' is below the {mode} bar")
        self._log_funnel_stage("zero_bar_fallback", len(candidates), len(picks), {"tier": len(picks)})
        if picks:
            self._record_fallback_outcome(mode, "tier", picks)
        return picks

    def _apply_elite_zero_bar_fallback(self, judged: list, candidates: list) -> list:
        mode = self.use_cases_quality
        near = [uc for uc in judged if uc.get("elite_status") != "ELITE"
                and not self._elite_has_fatal_flaw(uc.get("elite_assessment"))]
        near = self._rank_fallback_picks(near)
        if near:
            for uc in near:
                failed = (uc.get("elite_assessment") or {}).get("failed_gates") or []
                main = failed[0] if failed else "below the bar"
                self._label_fallback_pick(uc, mode, "near_miss", f"failed check: {main}")
            self._log_funnel_stage("zero_bar_fallback", len(judged), len(near), {"near_miss": len(near)})
            self._record_fallback_outcome(mode, "near_miss", near)
            return near
        high_tier = [uc for uc in candidates if self._canonical_quality_label(uc.get("Quality")) in self._DOC_HIGH_TIERS]
        high_tier = self._rank_fallback_picks(high_tier)
        for uc in high_tier:
            self._label_fallback_pick(uc, mode, "tier",
                                      f"quality '{uc.get('Quality')}' is the closest to the {mode} bar")
        self._log_funnel_stage("zero_bar_fallback", len(candidates), len(high_tier), {"high_tier": len(high_tier)})
        if high_tier:
            self._record_fallback_outcome(mode, "high_tier", high_tier)
        return high_tier

    def _select_scored_use_cases(self, scored_use_cases: list, schema_details: list, context: dict) -> tuple:
        """Selection after combined scoring, identical in structure for every quality mode:
        unscored UCs are excluded; the mode's label filter keeps only acceptable labels (no recovery
        from the rejection pool, no minimum top-up, no next-tier fallback); in label-filter modes
        Best-of-Best scores the kept set and drops BoB-unscored and hard-vetoed UCs, trims a large
        set by theme, and canonical-title duplicates are removed; finally each user-specified
        Business Domain left empty receives one coverage pick. Ultra High replaces the label filter,
        Best-of-Best and coverage steps with the elite stage (`_select_elite_use_cases`).
        Returns (selected, rejected) where rejected holds every input UC that was not selected."""
        from collections import Counter
        mode = self.use_cases_quality
        self._selection_is_fallback = False
        self.fallback_outcome = None
        candidates = [uc for uc in scored_use_cases if uc.get("scoring_status") == "scored"]
        unscored = [uc for uc in scored_use_cases if uc.get("scoring_status") != "scored"]
        for uc in unscored:
            uc["selection_drop_reason"] = f"unscored: {uc.get('scoring_unscored_reason') or 'no valid score'}"
        if unscored:
            self.logger.warning(
                f"⚠️ {len(unscored)} unscored UC(s) excluded from selection: "
                f"{[uc.get('No') for uc in unscored][:20]}{' ...' if len(unscored) > 20 else ''}"
            )
        self._log_funnel_stage("combined_scoring", len(scored_use_cases), len(candidates), {"unscored": len(unscored)})

        if self._is_ultra_high_mode():
            selected, universe = self._select_elite_use_cases(candidates, schema_details, context)
            universe = universe + unscored
        else:
            universe = scored_use_cases
            acceptable = self.quality_filter_acceptable
            acceptable_labels_str = ', '.join(sorted(acceptable))
            log_print(f"\n🔴 QUALITY FILTER [{mode}]: Keeping only {acceptable_labels_str}...")
            self.logger.info(f"🔴 Quality filter [{mode}]: Acceptable labels = {acceptable_labels_str}")
            selected = [uc for uc in candidates if uc.get("Quality") in acceptable]
            rejected = [uc for uc in candidates if uc.get("Quality") not in acceptable]
            for uc in rejected:
                uc["selection_drop_reason"] = f"quality '{uc.get('Quality')}' is below the {mode} bar"
            if rejected:
                log_print(f"\n⚠️ REJECTED USE CASES ({len(rejected)} total - below {mode} threshold):")
                self.logger.info(f"⚠️ REJECTED USE CASES ({len(rejected)} total):")
                for uc in rejected:
                    uc_summary = str(uc.get('Quality Summary', 'No summary') or 'No summary')
                    self.logger.info(f"   ⚠️ {uc.get('No', 'Unknown')}: {uc.get('Name', 'Unknown')} | Quality: {uc.get('Quality')} | {uc_summary}")
            self._log_funnel_stage("quality_filter", len(candidates), len(selected),
                                   dict(Counter(str(uc.get("Quality")) for uc in rejected)))
            selected = self._apply_best_of_best_selection(selected, schema_details, context)
            selected = self._dedup_canonical_titles(selected)

            n_before_picks = len(selected)
            selected_ids = {id(uc) for uc in selected}
            pool = [uc for uc in candidates if id(uc) not in selected_ids]
            selected = self._apply_domain_coverage_picks(selected, pool, list(self.user_business_domains or []))
            self._log_funnel_stage("domain_coverage_picks", n_before_picks, len(selected))
            if not selected:
                selected = self._apply_zero_bar_fallback(mode, candidates)

        n_guard_in = len(selected)
        leaked = [uc for uc in selected if uc.get("scoring_status") != "scored" or uc.get("bob_status") == "unscored"]
        if leaked:
            self.logger.error(f"🚨 {len(leaked)} unscored UC(s) reached the selection output and are removed: {[uc.get('No') for uc in leaked]}")
            leaked_ids = {id(uc) for uc in leaked}
            selected = [uc for uc in selected if id(uc) not in leaked_ids]
        self._log_funnel_stage("final_guard", n_guard_in, len(selected), {"unscored_leak": len(leaked)})

        selected_ids = {id(uc) for uc in selected}
        rejected_all = [uc for uc in universe if id(uc) not in selected_ids]
        log_print(f"\n✅ SELECTED USE CASES: {len(selected)} ({mode})")
        log_print(f"⚠️ NOT SELECTED: {len(rejected_all)}")
        self.logger.info(f"✅ Selection [{mode}]: {len(selected)} selected, {len(rejected_all)} not selected")
        return selected, rejected_all

    def _near_miss_candidates(self, rejected: list, limit: int = 10) -> list:
        inf = float("-inf")
        scored = sorted(
            [uc for uc in rejected if uc.get("scoring_status") == "scored"],
            key=lambda u: (self._finite_or(u.get("elite_score"), inf), self._finite_or(u.get("Inspire Score"), inf),
                           self._finite_or(u.get("bob_score"), inf)),
            reverse=True,
        )
        unscored = [uc for uc in rejected if uc.get("scoring_status") != "scored"]
        return (scored + unscored)[:limit]

    def _build_near_miss_list(self, rejected: list, limit: int = 10) -> list:
        near_miss = []
        for uc in self._near_miss_candidates(rejected, limit):
            entry = {
                "No": uc.get("No", ""),
                "Name": uc.get("Name", ""),
                "Business Domain": uc.get("Business Domain", ""),
                "Inspire Score": uc.get("Inspire Score"),
                "Quality": uc.get("Quality", ""),
                "bob_score": uc.get("bob_score"),
                "reason": uc.get("selection_drop_reason") or uc.get("scoring_unscored_reason") or "not selected",
            }
            if isinstance(uc.get("elite_assessment"), dict):
                entry["elite_score"] = uc.get("elite_score")
                entry["failed_gates"] = list(uc["elite_assessment"].get("failed_gates") or [])
            near_miss.append(entry)
        return near_miss

    def _report_zero_accepted(self, rejected: list) -> dict:
        mode = self.use_cases_quality
        message = f"0 use cases met the {mode} bar"
        near_miss = self._build_near_miss_list(rejected)
        self.logger.warning(f"🛑 {message}. No artifacts will be generated. Near misses (top {len(near_miss)} by score):")
        log_print(f"\n{'='*80}")
        log_print(f"🛑 {message}. No artifacts will be generated.")
        for idx, nm in enumerate(near_miss, 1):
            line = (f"   {idx}. {nm['No']} | {nm['Name']} | Inspire Score={nm['Inspire Score']} | "
                    f"Quality={nm['Quality']} | reason: {nm['reason']}")
            self.logger.warning(line)
            log_print(line)
        log_print(f"{'='*80}")
        result = {
            "accepted": 0,
            "mode": mode,
            "message": message,
            "near_miss": near_miss,
            "funnel_ledger": list(getattr(self, "funnel_ledger", None) or []),
        }
        self.zero_accepted_result = result
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization",
                step_name="Quality Bar",
                sub_step_name="Zero Accepted",
                message=message,
                status="ended_warning",
                progress_increment=1.0,
                result_json=result,
            )
        except Exception:
            pass
        return result

    def _publish_funnel_ledger(self) -> list:
        ledger = list(getattr(self, "funnel_ledger", None) or [])
        results = getattr(self, "final_inspire_results_json", None)
        if not isinstance(results, dict):
            results = {}
        results["funnel_ledger"] = ledger
        zero_result = getattr(self, "zero_accepted_result", None)
        if zero_result:
            results["selection_outcome"] = zero_result
        fallback_result = getattr(self, "fallback_outcome", None)
        if fallback_result:
            results["selection_outcome"] = fallback_result
        elite_outcome = getattr(self, "elite_outcome", None)
        if elite_outcome:
            results["elite_outcome"] = elite_outcome
        self.final_inspire_results_json = results
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization",
                step_name="Selection Funnel",
                sub_step_name="Ledger",
                message=f"Selection funnel recorded {len(ledger)} stage(s)",
                status="ended_success",
                progress_increment=0.0,
                result_json={"funnel_ledger": ledger},
            )
        except Exception:
            pass
        return ledger

    _ELITE_FQ_COLUMN_RE = re.compile(r"[A-Za-z_][\w$-]*(?:\.[A-Za-z_][\w$-]*){3}")

    def _is_ultra_high_mode(self, mode=None) -> bool:
        return canonical_use_cases_quality(getattr(self, "use_cases_quality", None) if mode is None else mode) == "Ultra High"

    @classmethod
    def _final_rank_key(cls, uc: dict) -> tuple:
        score = cls._finite_or(uc.get("elite_score"), None)
        if score is None:
            score = uc.get("Inspire Score")
        return ((0 if uc.get("elite_status") in (None, "", "ELITE") else 1,)
                + cls._score_rank_part(score)
                + cls._score_rank_part(uc.get("bob_score"))
                + cls._score_rank_part(uc.get("Priority"))
                + (str(uc.get("No", "")),))

    @staticmethod
    def _number_by_domain(use_cases: list) -> list:
        by_domain = {}
        for uc in use_cases or []:
            by_domain.setdefault(uc.get("Business Domain") or "Other", []).append(uc)
        ordered = []
        for domain_idx, members in enumerate(by_domain.values(), start=1):
            for uc_idx, uc in enumerate(members, start=1):
                uc["No"] = f"N{domain_idx:02d}-U{uc_idx:02d}"
                ordered.append(uc)
        return ordered

    def _assign_provisional_ids(self, use_cases: list) -> list:
        return self._number_by_domain(sorted(use_cases or [], key=lambda uc: str(uc.get("No", ""))))

    def _finalize_portfolio_order(self, use_cases: list) -> list:
        return self._number_by_domain(sorted(use_cases or [], key=self._final_rank_key))

    def _cluster_elite_final_set(self, use_cases: list) -> list:
        n = len(use_cases)
        if self.user_business_domains:
            self.logger.info(f"[ELITE] the final set of {n} use case(s) keeps the user Business Domains assigned before judging")
            self._log_funnel_stage("domain_clustering", n, n)
            return use_cases
        final_ids = {str(uc.get("No", "")) for uc in use_cases}
        near = [uc for uc in getattr(self, "elite_near_miss_ucs", None) or [] if str(uc.get("No", "")) not in final_ids]
        self.logger.info(f"[ELITE] clustering only the final set of {n} use case(s) into domains and subdomains"
                         + (f", with the {len(near)} near miss(es) of the ledger as context" if near else "")
                         + "; small domains are not merged, so every elite keeps the domain its decision belongs to")
        clustered = self._cluster_domains_and_subdomains(list(use_cases) + near, "English", merge_small_domains=False) or []
        by_id = {str(uc.get("No", "")): uc for uc in clustered}
        for entry in getattr(self, "elite_near_miss", None) or []:
            labelled = by_id.get(str(entry.get("No", "")))
            if labelled is not None and not str(entry.get("Business Domain") or "").strip():
                entry["Business Domain"] = labelled.get("Business Domain", "")
        result = [by_id.get(str(uc.get("No", "")), uc) for uc in use_cases]
        self._log_funnel_stage("domain_clustering", n, len(result))
        return result

    @classmethod
    def _elite_value_score(cls, uc: dict):
        value = cls._finite_or(uc.get("Value"), None)
        return None if value is None else min(10.0, max(0.0, value * 2.0))

    def _build_elite_candidate_pool(self, candidates: list, rt: dict) -> tuple:
        from collections import Counter
        acceptable = set(self.quality_filter_acceptable)
        min_value = rt["ultra_high_second_look_min_value"]
        limit = rt["ultra_high_second_look_max"]
        base, second_look, outside, drops = [], [], [], Counter()
        for uc in candidates:
            value = self._elite_value_score(uc)
            if uc.get("gate_unverified"):
                uc["selection_drop_reason"] = (f"gate unverified ({', '.join(str(g) for g in uc['gate_unverified'])}); "
                                               f"the Ultra High bar needs every gate verified")
                drops["gate_unverified"] += 1
                outside.append(uc)
            elif uc.get("Quality") in acceptable:
                base.append(uc)
            elif uc.get("Quality") == "Medium" and value is not None and value >= min_value:
                second_look.append(uc)
            else:
                uc["selection_drop_reason"] = f"quality '{uc.get('Quality')}' is below the Ultra High bar"
                drops["below_quality_bar"] += 1
                outside.append(uc)
        second_look.sort(key=lambda u: (-self._elite_value_score(u),) + self._score_rank_part(u.get("Inspire Score"))
                         + (str(u.get("No", "")),))
        for uc in second_look[limit:]:
            uc["selection_drop_reason"] = (f"second-look limit of {limit} reached for Medium quality with value "
                                           f"{self._elite_value_score(uc):.1f}/10")
            drops["second_look_limit"] += 1
            outside.append(uc)
        second_look = second_look[:limit]
        self.logger.info(f"[ELITE] candidate pool: {len(base)} of quality {sorted(acceptable)} + {len(second_look)} "
                         f"second-look Medium with value >= {min_value}/10, of {len(candidates)} scored use cases"
                         + (f"; second look: {[uc.get('No') for uc in second_look]}" if second_look else ""))
        self._log_funnel_stage("elite_candidate_pool", len(candidates), len(base) + len(second_look), dict(drops))
        return base + second_look, outside

    def _elite_bob_prescreen(self, pool: list, schema_details: list, context: dict, rt: dict) -> list:
        from collections import Counter
        if not pool:
            self._log_funnel_stage("elite_bob_prescreen", 0, 0)
            return []
        try:
            self._score_best_of_best(pool, schema_details, **self._grading_context(context))
        except Exception as e:
            reason = f"Best-of-Best scoring failed: {get_clean_error_message(e, max_chars=160)}"
            self.logger.error(f"[ELITE] {reason}; no candidate can pass the prescreen")
            for uc in pool:
                uc["bob_status"] = "unscored"
                uc["bob_unscored_reason"] = reason
        drops = Counter()
        scored = []
        for uc in pool:
            if uc.get("bob_status") == "unscored":
                uc["selection_drop_reason"] = f"best-of-best unscored: {uc.get('bob_unscored_reason') or 'no valid gate scores'}"
                drops["bob_unscored"] += 1
            else:
                scored.append(uc)
        not_vetoed = self._drop_hard_veto_ucs(scored)
        not_vetoed_ids = {id(uc) for uc in not_vetoed}
        for uc in scored:
            if id(uc) not in not_vetoed_ids:
                uc["selection_drop_reason"] = f"hard veto on gate {uc.get('bob_hard_veto_gate')}"
                drops["hard_veto"] += 1
        minimum = rt["ultra_high_bob_prescreen_min"]
        survivors = []
        for uc in not_vetoed:
            if self._finite_or(uc.get("bob_score"), float("-inf")) < minimum:
                uc["selection_drop_reason"] = (f"bob_score {uc.get('bob_score')} is below the Ultra High prescreen "
                                               f"minimum {minimum}")
                drops["below_prescreen_min"] += 1
            else:
                survivors.append(uc)
        self._log_funnel_stage("elite_bob_prescreen", len(pool), len(survivors), dict(drops))
        return survivors

    def _load_elite_evidence(self, use_cases: list) -> dict:
        tables = sorted({t for uc in use_cases for t in self._uc_table_refs(uc) if not t.lower().startswith("/volumes")})
        if not tables:
            return {}
        try:
            evidence = self.data_loader.get_metadata_evidence(tables) or {}
        except Exception as e:
            self.logger.error(f"[ELITE] metadata evidence unavailable ({get_clean_error_message(e)}); "
                              f"no candidate can pass verification")
            return {}
        evidence = {str(k).lower(): v for k, v in evidence.items() if isinstance(v, dict)}
        missing = [t for t in tables if t not in evidence]
        linked = self._evidence_linked_tables(evidence)
        if linked:
            try:
                more = self.data_loader.get_metadata_evidence(linked) or {}
            except Exception as e:
                self.logger.warning(f"[ELITE] metadata evidence of {len(linked)} joined table(s) unavailable "
                                    f"({get_clean_error_message(e)}); time via join cannot be verified for them")
                more = {}
            evidence.update({str(k).lower(): v for k, v in more.items() if isinstance(v, dict)})
        self.logger.info(f"[ELITE] metadata evidence for {len(tables) - len(missing)}/{len(tables)} cited table(s)"
                         + (f" and {len(evidence) - (len(tables) - len(missing))} joined table(s)" if linked else "")
                         + (f"; not found: {missing[:10]}" if missing else ""))
        return evidence

    @staticmethod
    def _evidence_linked_tables(evidence: dict) -> list:
        return sorted({str(fk.get("ref_table") or "").lower() for entry in evidence.values()
                       for fk in entry.get("foreign_keys") or []} - set(evidence) - {""})

    @staticmethod
    def _elite_key_columns(evidence: dict, tables: list) -> dict:
        keys = {t: set() for t in tables}
        for t in tables:
            entry = evidence.get(t) or {}
            keys[t] |= {str(p).lower() for p in entry.get("primary_key") or []}
            for fk in entry.get("foreign_keys") or []:
                keys[t].add(str(fk.get("column") or "").lower())
                ref = str(fk.get("ref_table") or "").lower()
                if ref in keys:
                    keys[ref].add(str(fk.get("ref_column") or "").lower())
        return keys

    @staticmethod
    def _elite_dated_joins(evidence: dict, tables: list) -> dict:
        dated = {}
        for t in tables:
            for fk in (evidence.get(t) or {}).get("foreign_keys") or []:
                ref = str(fk.get("ref_table") or "").lower()
                temporal = [c for c in ((evidence.get(ref) or {}).get("columns") or {}).values()
                            if isinstance(c, dict) and _metadata_type_family(c.get("type")) == "temporal"]
                if len(temporal) > 1:
                    temporal = [c for c in temporal if "date" in _name_tokens(c.get("name"))] or temporal
                if ref == t or len(temporal) != 1:
                    continue
                column = str(fk.get("column") or "").lower()
                dated[(t, column)] = {"table": ref, "column": f"{ref}.{str(temporal[0].get('name') or '').lower()}",
                                      "join": f"{t}.{column} = {ref}.{str(fk.get('ref_column') or '').lower()}"}
        return dated

    _NATURAL_KEY_NOISE_TOKENS = frozenset({"name", "label"})

    @classmethod
    def _is_natural_entity_key(cls, column_name, column_type, decision) -> bool:
        if _metadata_type_family(column_type) != "string":
            return False
        tokens = [_singular_token(t) for t in _name_tokens(column_name)]
        if not tokens or any(text_token in t for t in tokens for text_token in cls._TEXT_NAME_TOKENS):
            return False
        entity = [t for t in tokens if t not in cls._NATURAL_KEY_NOISE_TOKENS]
        decision_tokens = {_singular_token(t) for t in _name_tokens(decision)}
        return bool(entity) and all(t in decision_tokens for t in entity)

    def _elite_evidence_summary(self, uc: dict, evidence: dict) -> str:
        lines = []
        tables = [t for t in self._uc_table_refs(uc) if not t.lower().startswith("/volumes")]
        keys = self._elite_key_columns(evidence, [t for t in tables if t in evidence])
        for table in self._uc_table_refs(uc):
            if table.lower().startswith("/volumes"):
                lines.append(f"- `{table}`: volume path, no table metadata exists for it")
                continue
            entry = evidence.get(table)
            if not isinstance(entry, dict):
                lines.append(f"- `{table}`: NOT FOUND in the metadata")
                continue
            columns = list((entry.get("columns") or {}).values())
            documented = sum(1 for c in columns if str(c.get("comment") or "").strip())
            temporal = [c.get("name") for c in columns if _metadata_type_family(c.get("type")) == "temporal"]
            measures = [c.get("name") for c in columns
                        if _metadata_type_family(c.get("type")) in ("numeric", "boolean")
                        and str(c.get("name") or "").lower() not in keys.get(table, set())
                        and not _is_key_like_column(c.get("name"))]
            comment = _clip_metadata_text(entry.get("comment"), 200)
            parts = [f"`{table}`" + (f" ({entry['table_type']})" if entry.get("table_type") else ""),
                     f"comment: {comment}" if comment else "no table comment",
                     f"{len(columns)} columns, {documented} with comments"]
            if entry.get("primary_key"):
                parts.append("primary key: " + ", ".join(f"`{p}`" for p in entry["primary_key"]))
            fks = [f"`{fk.get('column')}` -> `{fk.get('ref_table')}.{fk.get('ref_column')}` ({fk.get('source')})"
                   for fk in entry.get("foreign_keys") or []]
            if fks:
                parts.append("foreign keys: " + "; ".join(fks[:8]) + (f" (+{len(fks) - 8} more)" if len(fks) > 8 else ""))
            parts.append("date/timestamp columns: " + (", ".join(f"`{n}`" for n in temporal[:12]) or "none"))
            dated = self._elite_dated_joins(evidence, [table])
            if dated:
                parts.append("date keys joined to a dated table: " + ", ".join(
                    f"`{column}` -> `{info['column']}`" for (_t, column), info in sorted(dated.items())[:8]))
            parts.append("numeric or boolean non-key columns: " + (", ".join(f"`{n}`" for n in measures[:12]) or "none")
                         + (f" (+{len(measures) - 12} more)" if len(measures) > 12 else ""))
            if entry.get("tags"):
                parts.append("tags: " + _clip_metadata_text(", ".join(f"{k}={v}" for k, v in entry["tags"].items()), 160))
            if entry.get("last_altered"):
                parts.append(f"last altered (advisory only): {entry['last_altered']}")
            lines.append("- " + "; ".join(parts))
        return "\n".join(lines) if lines else "- no table is cited"

    def _build_elite_judge_markdown(self, uc_batch: list, evidence: dict, repair_failures: dict = None) -> str:
        parts = [self._build_use_cases_markdown_for_validation(uc_batch), "## Evidence per use case\n\n"]
        for uc in uc_batch:
            gates = uc.get("bob_scores") or {}
            order = sorted(gates, key=lambda g: int(str(g)[1:]) if str(g)[1:].isdigit() else 99)
            parts.append(f"### Evidence for {uc.get('No', 'UNKNOWN')}\n\n")
            parts.append(f"**Stated beneficiary / sponsor:** {uc.get('Beneficiary') or 'N/A'} / {uc.get('Sponsor') or 'N/A'}\n\n")
            parts.append(f"**Best-of-Best gate scores (0-10, earlier opinion):** "
                         f"{', '.join(f'{g}={gates[g]}' for g in order) or 'N/A'}; bob_score {uc.get('bob_score')}\n\n")
            parts.append("**Metadata evidence:**\n" + self._elite_evidence_summary(uc, evidence) + "\n\n")
            failures = (repair_failures or {}).get(str(uc.get("No", "")))
            if failures:
                parts.append("**Verification failures of the earlier assessment:**\n"
                             + "\n".join(f"- {f}" for f in failures) + "\n\n")
                listing = []
                for table in self._uc_table_refs(uc):
                    columns = ((evidence.get(table) or {}).get("columns") or {}).values()
                    listing.append(f"- `{table}`: " + ", ".join(
                        f"`{c.get('name')}` {c.get('type') or 'untyped'}"
                        + (f" ({_clip_metadata_text(c.get('comment'), 80)})" if str(c.get("comment") or "").strip() else "")
                        for c in columns))
                parts.append("**Real columns of the use case's tables:**\n" + "\n".join(listing) + "\n\n")
        return "".join(parts)

    @staticmethod
    def _elite_code(value) -> str:
        return re.sub(r"[\s-]+", "_", ("" if value is None else str(value)).strip()).upper()

    def _normalize_elite_assessment(self, item: dict, problems: list = None, cap_verdict: bool = False):
        def _text(value):
            text = "" if value is None else str(value).strip()
            return "" if text.lower() in ("null", "none", "n/a") else text

        def _texts(value):
            return [t for t in (_text(v) for v in value) if t] if isinstance(value, list) else []

        def _invalid(reason):
            if problems is not None:
                problems.append(reason)
            return None

        raw_scores = item.get("scores") if isinstance(item, dict) else None
        if not isinstance(raw_scores, dict):
            return _invalid("missing scores")
        scores = {}
        for dim in ULTRA_HIGH_DIMENSIONS:
            value = self._bounded_float(raw_scores.get(dim), 1.0, 10.0)
            if value is None:
                return _invalid("missing or out-of-range score")
            scores[dim] = int(value)
        verdict = self._elite_code(_text(item.get("verdict")))
        tier = _text(item.get("value_tier"))
        decision, kpi = _text(item.get("decision")), _text(item.get("kpi"))
        if verdict not in ("ELITE", "NEAR_MISS", "REJECT"):
            return _invalid(f"invalid verdict {_text(item.get('verdict'))[:40]!r}")
        if not tier or not decision or not kpi:
            return _invalid("missing value tier, decision or KPI")
        if not isinstance(item.get("fatal_flaws"), list):
            return _invalid("missing fatal_flaws")
        fatal_flaws = []
        for flaw in item["fatal_flaws"]:
            category = self._elite_code(flaw.get("category")) if isinstance(flaw, dict) else ""
            explanation = _text(flaw.get("explanation")) if isinstance(flaw, dict) else ""
            if category not in ULTRA_HIGH_FATAL_FLAW_CATEGORIES or not explanation:
                return _invalid("fatal flaw outside the fixed categories or without an explanation")
            fatal_flaws.append({"category": category, "explanation": explanation})
        raw_assumptions = item.get("runtime_assumptions")
        if not isinstance(raw_assumptions, list):
            return _invalid("missing runtime_assumptions")
        runtime_assumptions = []
        for assumption in raw_assumptions:
            kind = self._elite_code(assumption.get("kind")) if isinstance(assumption, dict) else ""
            if kind not in INSPIRE_ASSUMPTION_RULES or not _text(assumption.get("table")) or not _text(assumption.get("column")):
                return _invalid("runtime assumption outside the fixed kinds or without a table and column")
            central = assumption.get("central")
            runtime_assumptions.append({"kind": kind, "table": _text(assumption.get("table")),
                                        "column": _text(assumption.get("column")),
                                        "statement": _text(assumption.get("statement")),
                                        "central": central is True or str(central).strip().lower() == "true"})
        red_team_category = self._elite_code(_text(item.get("red_team_category")))
        if red_team_category not in ULTRA_HIGH_FATAL_FLAW_CATEGORIES + (ULTRA_HIGH_RESIDUAL_RISK,):
            return _invalid("red_team_category outside the fixed categories")
        red_team_verdict = "fatal" if fatal_flaws else "survives"
        stated_red_team_verdict = _text(item.get("red_team_verdict")).lower()
        flaw_categories = {f["category"] for f in fatal_flaws}
        if stated_red_team_verdict and stated_red_team_verdict != red_team_verdict:
            return _invalid("self-contradiction: red_team_verdict disagrees with fatal_flaws")
        if verdict in ("ELITE", "NEAR_MISS") and fatal_flaws:
            return _invalid(f"self-contradiction: verdict {verdict} with fatal flaws")
        rt = TECHNICAL_CONTEXT["runtime"]
        minimum, band = rt["ultra_high_min_dimension_score"], rt["ultra_high_near_miss_band"]
        tier_allowed = tier.lower() in {str(t).lower() for t in rt["ultra_high_allowed_value_tiers"]}
        supported = ("ELITE" if tier_allowed and min(scores.values()) >= minimum else
                     "NEAR_MISS" if tier_allowed and min(scores.values()) >= minimum - band else "REJECT")
        rank = {"REJECT": 0, "NEAR_MISS": 1, "ELITE": 2}
        capped_from = None
        if rank[verdict] > rank[supported]:
            if not cap_verdict:
                return _invalid(f"self-contradiction: verdict {verdict} while its own scores and value tier support {supported}")
            capped_from, verdict = verdict, supported
        if (red_team_category in ULTRA_HIGH_FATAL_FLAW_CATEGORIES and red_team_category not in flaw_categories
                and red_team_category not in ULTRA_HIGH_CONDITIONAL_FLAW_CATEGORIES):
            return _invalid(f"self-contradiction: red team names {red_team_category} but fatal_flaws does not")
        if (red_team_category in ULTRA_HIGH_CONDITIONAL_FLAW_CATEGORIES and red_team_category not in flaw_categories
                and not runtime_assumptions):
            return _invalid("self-contradiction: an unverifiable data shape that is neither fatal nor a runtime assumption")
        outcome = item.get("outcome_definition")
        if isinstance(outcome, dict):
            timing = _text(outcome.get("label_timing")).lower()
            if timing not in ULTRA_HIGH_LABEL_TIMINGS:
                return _invalid("outcome_definition without a valid label_timing")
            outcome = {"column": _text(outcome.get("column")) or None,
                       "derived_from": _texts(outcome.get("derived_from")),
                       "rule": _text(outcome.get("rule")),
                       "label_timing": timing}
        else:
            outcome = None
        return {
            "decision": decision,
            "decision_owner": _text(item.get("decision_owner")),
            "cadence": _text(item.get("cadence")),
            "kpi": kpi,
            "kpi_direction": _text(item.get("kpi_direction")).lower(),
            "value_driver": _text(item.get("value_driver")),
            "value_tier": next((t for t in ULTRA_HIGH_VALUE_TIERS if t.lower() == tier.lower()), tier),
            "signal_columns": _texts(item.get("signal_columns")),
            "outcome_definition": outcome,
            "time_column": _text(item.get("time_column")) or None,
            "entity_key_column": _text(item.get("entity_key_column")) or None,
            "join_path": _texts(item.get("join_path")),
            "build_path": _text(item.get("build_path")),
            "blockers": _texts(item.get("blockers")),
            "runtime_assumptions": runtime_assumptions,
            "fatal_flaws": fatal_flaws,
            "red_team": _text(item.get("red_team")),
            "red_team_category": red_team_category,
            "red_team_verdict": red_team_verdict,
            "scores": scores,
            "verdict": verdict,
            "verdict_capped_from": capped_from,
            "verdict_reason": _text(item.get("verdict_reason")),
            "second_opinion": None,
            "verification": None,
            "evidence_repair": None,
        }

    def _parse_elite_judge_response(self, response_raw: str, uc_batch: list, problems_by_id: dict = None,
                                    cap_ids: set = None) -> dict:
        batch_ids = {str(uc.get("No", "")) for uc in uc_batch}
        try:
            parsed = json.loads(clean_json_response(response_raw or ""))
        except (TypeError, ValueError) as e:
            self.logger.warning(f"[ELITE] judge response unparseable ({get_clean_error_message(e)[:160]}); "
                                f"head: {str(response_raw or '')[:200]}")
            return {}
        if isinstance(parsed, dict):
            parsed = parsed["data"] if isinstance(parsed.get("data"), list) else [parsed]
        if not isinstance(parsed, list):
            return {}
        from collections import Counter
        records, problems = {}, []
        for item in parsed:
            uc_id = str(item.get("No") or item.get("use_case_id") or "").strip() if isinstance(item, dict) else ""
            if uc_id not in batch_ids:
                continue
            found = len(problems)
            assessment = self._normalize_elite_assessment(item, problems, cap_verdict=uc_id in (cap_ids or ()))
            if assessment is None:
                if problems_by_id is not None and len(problems) > found:
                    problems_by_id[uc_id] = problems[-1]
                continue
            records[uc_id] = assessment
        if problems:
            self.logger.warning(f"[ELITE] judge response: {len(problems)} item(s) rejected as invalid or self-contradictory: "
                                + "; ".join(f"{reason} x{n}" for reason, n in sorted(Counter(problems).items())))
        capped = [f"{uc_id} {a['verdict_capped_from']} -> {a['verdict']}" for uc_id, a in records.items()
                  if a.get("verdict_capped_from")]
        if capped:
            self.logger.warning(f"[VERDICT-CAP FIRED] [ELITE] {len(capped)} verdict(s) still above their own scores and value "
                                f"tier after the corrective re-judge were capped at what those support: {'; '.join(capped)}")
        return records

    def _run_elite_judge(self, use_cases: list, schema_details: list, evidence: dict, context: dict, rt: dict,
                         label: str, start_from_model: str = None, force_fresh: bool = False,
                         repair_failures: dict = None) -> tuple:
        context = context or {}
        base_vars = {
            "business_context": context.get("business_context") or self._get_business_context_fallback(),
            "industry": prompt_context_or_not_provided(context.get("industry"), "industry"),
            "strategic_initiative": prompt_context_or_not_provided(context.get("strategic_initiative"),
                                                                   "strategic initiative"),
            "value_chain": prompt_context_or_not_provided(context.get("value_chain"), "value chain"),
            "revenue_model": prompt_context_or_not_provided(context.get("revenue_model"), "revenue model"),
            "generation_instructions_section": self._get_generation_instruction_for_prompt("ULTRA_HIGH_ELITE_JUDGE_PROMPT"),
            "min_dimension_score": rt["ultra_high_min_dimension_score"],
            "allowed_value_tiers": " or ".join(f'"{t}"' for t in rt["ultra_high_allowed_value_tiers"]),
            "near_miss_band": rt["ultra_high_near_miss_band"],
        }
        batch_size = rt["ultra_high_judge_batch_size"]
        timeout = max(300, self.llm_timeout_seconds + batch_size * 30)
        discarded, discarded_lock = {}, threading.Lock()

        def _judge_batch(batch_label: str, uc_batch: list, skip_cache: bool) -> dict:
            if _is_thread_cancelled():
                raise Exception(f"[{label}_{batch_label}] Aborted: parent task cancelled/timed out")
            failures = {uc_id: list(items) for uc_id, items in (repair_failures or {}).items()}
            with discarded_lock:
                cap_ids = {str(uc.get("No", "")) for uc in uc_batch if str(uc.get("No", "")) in discarded}
                for uc in uc_batch:
                    reason = discarded.get(str(uc.get("No", "")))
                    if reason:
                        failures.setdefault(str(uc.get("No", "")), []).append(
                            f"The earlier answer for this use case was discarded ({reason}); return every required key "
                            f"and a verdict of exactly ELITE, NEAR_MISS or REJECT")
            prompt_vars = dict(
                base_vars,
                schema_details=(self._grounding_schema_markdown(schema_details, self._uc_batch_tables(uc_batch))
                                or self._build_schema_markdown_for_validation(schema_details)),
                use_cases_to_judge=self._build_elite_judge_markdown(uc_batch, evidence, failures),
            )
            response_raw = self.ai_agent.run_worker(
                step_name=f"{label}_{batch_label}",
                worker_prompt_path="ULTRA_HIGH_ELITE_JUDGE_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None,
                timeout_override=timeout,
                start_from_model=start_from_model or self._pinned_elite_judge(),
                skip_cache=skip_cache or force_fresh,
            )
            problems = {}
            records = self._parse_elite_judge_response(response_raw, uc_batch, problems, cap_ids)
            with discarded_lock:
                discarded.update(problems)
                for uc_id in records:
                    discarded.pop(uc_id, None)
            return records

        first_batch = use_cases[:batch_size]
        est_chars = (self._prompt_overhead_chars("ULTRA_HIGH_ELITE_JUDGE_PROMPT")
                     + len(self._grounding_schema_markdown(schema_details, self._uc_batch_tables(first_batch))
                           or self._build_schema_markdown_for_validation(schema_details))
                     + len(self._build_elite_judge_markdown(first_batch, evidence, repair_failures)))
        per_task_timeout = self._scoring_task_timeout(timeout, est_chars)
        records, unjudged = self._score_with_split_retry(use_cases, _judge_batch, batch_size, label, per_task_timeout)
        self._log_unscored(label, unjudged)
        return records, unjudged

    def _verify_elite_evidence(self, uc: dict, assessment: dict, evidence: dict) -> dict:
        a = assessment if isinstance(assessment, dict) else {}
        evidence = {str(k).lower(): v for k, v in (evidence or {}).items() if isinstance(v, dict)}
        refs = self._uc_table_refs(uc)
        tables = [r for r in refs if not r.lower().startswith("/volumes")]
        failures = [f"volume source has no table metadata: {r}" for r in refs if r.lower().startswith("/volumes")]
        if not tables:
            failures.append("no table is cited")
        failures += [f"table not found in the metadata: {t}" for t in tables if t not in evidence]
        design_failures = len(failures)
        known = [t for t in tables if t in evidence]
        dated = self._elite_dated_joins(evidence, known)
        dated_tables = sorted({info["table"] for info in dated.values()} - set(known))
        columns = {t: {str(k).lower(): v for k, v in (evidence[t].get("columns") or {}).items() if isinstance(v, dict)}
                   for t in known + dated_tables}
        keys = self._elite_key_columns(evidence, known)

        def _resolve(ref, pool=None):
            parts = [p for p in str(ref or "").replace("`", "").strip().lower().split(".") if p]
            if not parts:
                return []
            column, prefix = parts[-1], parts[:-1][-3:]
            return [(t, column) for t in (known if pool is None else pool)
                    if column in columns[t] and (not prefix or t.split(".")[-len(prefix):] == prefix)]

        def _type(hit):
            return columns[hit[0]][hit[1]].get("type")

        def _name(hit):
            return columns[hit[0]][hit[1]].get("name") or hit[1]

        def _is_key(hit):
            return hit[1] in keys.get(hit[0], set()) or _is_key_like_column(_name(hit))

        def _row_key(hit):
            return _is_own_row_key(hit[0], _name(hit), (evidence.get(hit[0]) or {}).get("primary_key") or (),
                                   [c.get("name") or k for k, c in columns[hit[0]].items()])

        resolved = {}

        def _record(role, ref, pool=None):
            hits = _resolve(ref, pool)
            if len(hits) == 1:
                resolved.setdefault(role, []).append(hits[0])
            elif hits:
                failures.append(f"{role} column is ambiguous across the use case's tables: {ref}")
            else:
                failures.append(f"{role} column not found in the use case's tables: {ref}")

        outcome = a.get("outcome_definition") if isinstance(a.get("outcome_definition"), dict) else None
        cited = [("signal", r) for r in a.get("signal_columns") or []]
        if not cited:
            failures.append("no signal column is cited")
        if a.get("time_column"):
            cited.append(("time_column", a["time_column"]))
        if a.get("entity_key_column"):
            cited.append(("entity_key_column", a["entity_key_column"]))
        if outcome and outcome.get("column"):
            cited.append(("outcome", outcome["column"]))
        cited += [("outcome source", r) for r in (outcome or {}).get("derived_from") or []]
        for path in a.get("join_path") or []:
            cited += [("join", r) for r in self._ELITE_FQ_COLUMN_RE.findall(str(path).replace("`", ""))]
        for role, ref in cited:
            _record(role, ref, known + dated_tables if role in ("time_column", "join") else None)

        driver_hits = []
        short_names = {t.rsplit(".", 1)[-1] for t in known}
        for token in re.findall(r"`([^`]+)`", str(a.get("value_driver") or "")):
            hits = _resolve(token)
            if hits:
                driver_hits += hits
            elif self._normalize_table_ref(token) not in known and token.strip().lower() not in short_names:
                failures.append(f"value driver cites an unknown column: {token}")
        if not driver_hits:
            failures.append("value driver cites no existing column")

        family = self._technique_family(uc.get("Analytics Technique"))
        technique = uc.get("Analytics Technique") or "the technique"
        ai_text = self._uc_ai_text_function(uc, a)
        supervised = family in self._SUPERVISED_FAMILIES and not ai_text
        if family in self._TEMPORAL_FAMILIES and not a.get("time_column"):
            failures.append(f"{technique} needs a DATE or TIMESTAMP time column and none is cited")
        time_via_join, temporal_time = None, False
        for hit in resolved.get("time_column", []):
            is_temporal = _metadata_type_family(_type(hit)) == "temporal"
            links = [(src, info) for src, info in sorted(dated.items()) if info["table"] == hit[0]]
            if hit[0] in known and is_temporal:
                temporal_time = True
            elif hit[0] in known and hit in dated:
                time_via_join = {"key": ".".join(hit), "table": dated[hit]["table"], "column": dated[hit]["column"],
                                 "join": dated[hit]["join"]}
                temporal_time = True
            elif hit[0] not in known and is_temporal and len(links) == 1:
                time_via_join = {"key": ".".join(links[0][0]), "table": hit[0], "column": ".".join(hit),
                                 "join": links[0][1]["join"]}
                temporal_time = True
            elif hit[0] not in known and is_temporal:
                failures.append(f"time column {'.'.join(hit)} joins the use case's tables through {len(links)} date keys; "
                                f"cite the date key the decision orders events by")
            else:
                failures.append(f"time column {'.'.join(hit)} is {_type(hit) or 'untyped'}, not DATE or TIMESTAMP")

        entity_key_basis = None
        if not a.get("entity_key_column"):
            failures.append("no entity key column is cited")
        for hit in resolved.get("entity_key_column", []):
            if _is_key(hit):
                entity_key_basis = "key"
            elif self._is_natural_entity_key(_name(hit), _type(hit), a.get("decision")):
                entity_key_basis = "natural key named by the decision"
            else:
                failures.append(f"entity key column {'.'.join(hit)} is not a primary, foreign or key-like column "
                                f"and not a natural key of the entity the decision acts on")

        outcome_ok = False
        if outcome and outcome.get("column"):
            outcome_ok = bool(resolved.get("outcome"))
        elif outcome and outcome.get("derived_from") and outcome.get("rule"):
            outcome_ok = len(resolved.get("outcome source", [])) == len(outcome["derived_from"])
        if supervised and not outcome_ok:
            failures.append(f"{technique} needs an outcome column that exists or an outcome derived from existing columns")
        if supervised and outcome_ok:
            structural = set(resolved.get("entity_key_column", [])) | set(resolved.get("join", []))
            label_hits = set(resolved.get("outcome", []) + resolved.get("outcome source", [])) - structural
            overlap = sorted(label_hits & set(resolved.get("signal", [])))
            future = str(outcome.get("label_timing") or "").lower() == "future_window" and temporal_time
            if overlap and not future:
                failures.append(f"{technique} predicts an outcome computed from its own signal columns in the same period "
                                f"({', '.join('.'.join(h) for h in overlap)}): the label is a rule over the inputs, so this "
                                f"is a ranking or a rule, not a model")
                design_failures += 1

        measure_hits = resolved.get("signal", []) + resolved.get("outcome", []) + resolved.get("outcome source", []) + driver_hits
        cited_hits = {hit for hits in resolved.values() for hit in hits} | set(driver_hits)
        has_measure = any(_metadata_type_family(_type(h)) in ("numeric", "boolean") and not _is_key(h) for h in measure_hits)
        has_duration = len({h for h in cited_hits if _metadata_type_family(_type(h)) == "temporal"}) >= 2
        if not (has_measure or has_duration or outcome_ok):
            failures.append("the KPI has no numeric measure: no numeric or boolean non-key column, no duration "
                            "between two date columns and no outcome definition")
        if (family in self._TEXT_FAMILIES or ai_text) and not self._column_signals(
                {h[1]: _type(h) for h in cited_hits if not _is_key(h)})["text_cols"]:
            failures.append(f"{technique} needs a free-text column and none is cited")

        assumptions = []
        for assumption in a.get("runtime_assumptions") or []:
            if not isinstance(assumption, dict):
                continue
            hits = _resolve(assumption.get("column"))
            if len(hits) != 1:
                failures.append(f"runtime assumption column not found in the use case's tables: {assumption.get('column')}")
                continue
            entry = dict(assumption, table=hits[0][0], column=".".join(hits[0]))
            if entry.get("kind") == "KEY_HAS_REPEATED_ROWS" and entry.get("central") and _row_key(hits[0]):
                failures.append(f"central runtime assumption contradicted by the metadata: {entry['column']} is the row key "
                                f"of {hits[0][0]}, so each value occurs on one row"
                                + (f" ({entry['statement']})" if entry.get("statement") else ""))
                design_failures += 1
            assumptions.append(entry)
        if TECHNIQUE_EXECUTION_FAMILY.get(self._technique_label(family)) == "D" and not ai_text:
            signal_tables = {h[0] for h in resolved.get("signal", []) if not _is_key(h)}
            for hit in resolved.get("entity_key_column", []):
                if hit[0] not in signal_tables:
                    continue
                if _row_key(hit):
                    failures.append(f"{technique} needs several rows per {'.'.join(hit)} to find items that occur together, "
                                    f"but it is the row key of {hit[0]}, so each value occurs on one row")
                    design_failures += 1
                elif not any(x.get("kind") == "KEY_HAS_REPEATED_ROWS" and x.get("column") == ".".join(hit) for x in assumptions):
                    assumptions.append({"kind": "KEY_HAS_REPEATED_ROWS", "table": hit[0], "column": ".".join(hit),
                                        "statement": f"each {_name(hit)} value covers more than one row, so items can occur together",
                                        "central": True})

        join = {"connected": True, "path": [], "strength": "none"}
        if len(tables) >= 2:
            join = self._find_join_path(tables)
            if not join.get("connected"):
                failures.append("no join path connects the cited tables: " + ", ".join(tables))
                design_failures += 1

        documented = sum(1 for h in cited_hits if str(columns[h[0]][h[1]].get("comment") or "").strip())
        canonical = {"signal_columns": [".".join(h) for h in resolved.get("signal", [])],
                     "time_column": ".".join(resolved["time_column"][0]) if resolved.get("time_column") else None,
                     "entity_key_column": ".".join(resolved["entity_key_column"][0]) if resolved.get("entity_key_column") else None,
                     "runtime_assumptions": assumptions, "time_via_join": time_via_join}
        if outcome:
            canonical["outcome_definition"] = dict(
                outcome, column=".".join(resolved["outcome"][0]) if resolved.get("outcome") else outcome.get("column"),
                derived_from=[".".join(h) for h in resolved.get("outcome source", [])] or list(outcome.get("derived_from") or []))
        return {
            "passed": not failures,
            "failures": failures,
            "repairable": bool(failures) and not design_failures,
            "canonical_columns": canonical,
            "comment_coverage": round(documented / len(cited_hits), 2) if cited_hits else 0.0,
            "documented_columns": documented,
            "cited_columns": len(cited_hits),
            "documented_tables": sum(1 for t in known if str(evidence[t].get("comment") or "").strip()),
            "tables": len(tables),
            "join_strength": join.get("strength"),
            "join_path": list(join.get("path") or []),
            "entity_key_basis": entity_key_basis,
            "time_via_join": time_via_join,
        }

    def _verify_and_canonicalize_elite(self, uc: dict, assessment: dict, evidence: dict) -> dict:
        verification = self._verify_elite_evidence(uc, assessment, evidence)
        if verification["passed"]:
            assessment.update(verification["canonical_columns"])
        assessment["verification"] = verification
        return verification

    @staticmethod
    def _elite_decision(assessment: dict, verification: dict, cfg: dict) -> tuple:
        minimum = cfg["ultra_high_min_dimension_score"]
        band = cfg["ultra_high_near_miss_band"]
        allowed = {str(t).lower() for t in cfg["ultra_high_allowed_value_tiers"]}

        def _opinion(a, v):
            a = a if isinstance(a, dict) else {}
            failed, hard = [], False
            if not (v or {}).get("passed"):
                failed.append("verification")
                hard = True
            if str(a.get("red_team_verdict") or "").lower() != "survives" or a.get("fatal_flaws"):
                failed.append("red_team")
                hard = True
            if str(a.get("value_tier") or "").lower() not in allowed:
                failed.append("value_tier")
                hard = True
            scores = a.get("scores") if isinstance(a.get("scores"), dict) else {}
            for dim in ULTRA_HIGH_DIMENSIONS:
                value = scores.get(dim)
                numeric = isinstance(value, (int, float)) and not isinstance(value, bool)
                if not numeric or value < minimum:
                    failed.append(f"score:{dim}")
                    hard = hard or not numeric or value < minimum - band
            verdict = str(a.get("verdict") or "").upper()
            if verdict != "ELITE":
                failed.append("judge_verdict")
                hard = hard or verdict != "NEAR_MISS"
            if not failed:
                return "ELITE", []
            return ("REJECT" if hard else "NEAR_MISS"), failed

        verdict, failed = _opinion(assessment, verification)
        second = (assessment or {}).get("second_opinion")
        if verdict != "NEAR_MISS" or not isinstance(second, dict):
            return verdict, failed
        second_verdict, _ = _opinion(second, second.get("verification"))
        if second_verdict == "REJECT":
            return "NEAR_MISS", failed + ["second_opinion_reject"]
        first_scores, second_scores = assessment["scores"], second["scores"]
        if all((first_scores[d] + second_scores[d]) / 2.0 >= minimum for d in ULTRA_HIGH_DIMENSIONS):
            return "ELITE", []
        return "NEAR_MISS", failed + ["second_opinion_average_below_bar"]

    @staticmethod
    def _elite_dimension_values(assessment: dict):
        def _complete(scores):
            return isinstance(scores, dict) and all(
                isinstance(scores.get(d), (int, float)) and not isinstance(scores.get(d), bool) for d in ULTRA_HIGH_DIMENSIONS)

        scores = (assessment or {}).get("scores")
        if not _complete(scores):
            return None
        second = assessment.get("second_opinion")
        second_scores = second.get("scores") if isinstance(second, dict) else None
        if _complete(second_scores):
            return {d: (scores[d] + second_scores[d]) / 2.0 for d in ULTRA_HIGH_DIMENSIONS}
        return {d: float(scores[d]) for d in ULTRA_HIGH_DIMENSIONS}

    @classmethod
    def _elite_composite(cls, assessment: dict, weights: dict):
        values = cls._elite_dimension_values(assessment)
        if values is None:
            return None
        return round(sum(float(weights[d]) * values[d] for d in ULTRA_HIGH_DIMENSIONS), 2)

    @staticmethod
    def _elite_drop_reason(assessment: dict, prefix: str = "below the Ultra High bar: ") -> str:
        details = list(((assessment.get("verification") or {}).get("failures") or [])[:2])
        flaws = [f"{f.get('category')}: {f.get('explanation')}" for f in assessment.get("fatal_flaws") or [] if isinstance(f, dict)]
        repair = assessment.get("evidence_repair")
        return (prefix + ", ".join(assessment.get("failed_gates") or [])
                + (f" ({'; '.join(details)})" if details else "")
                + (f"; fatal flaws: {'; '.join(flaws[:2])}" if flaws else "")
                + (f"; evidence repair {repair['outcome']}" if isinstance(repair, dict) else ""))

    def _star_drop_reason(self, item: dict) -> str:
        a = item.get("elite_assessment") if isinstance(item.get("elite_assessment"), dict) else None
        if a is None:
            return f"no elite assessment ({item.get('elite_status') or 'not judged'})"
        if a.get("failed_gates"):
            return self._elite_drop_reason(a, prefix="did not pass: ")
        return item.get("selection_drop_reason") or "passed every elite test but was not kept as an elite"

    @staticmethod
    def _elite_worker_label(stage: str, part: str) -> str:
        return f"{BEST_OF_BEST_WORKER_PREFIX if stage == 'best_of_best' else 'Elite'}{part}"

    def _repair_elite_evidence(self, judged: list, schema_details: list, evidence: dict, context: dict, rt: dict,
                               stage: str = "elite") -> None:
        from collections import Counter
        candidates = []
        for uc in judged:
            a = uc["elite_assessment"]
            v = a["verification"]
            if v["passed"] or not v["repairable"] or a.get("evidence_repair") is not None:
                continue
            if self._elite_decision(a, dict(v, passed=True), rt)[0] == "REJECT":
                continue
            candidates.append(uc)
        if not candidates:
            self._log_funnel_stage(f"{stage}_evidence_repair", 0, 0)
            return
        failures = {str(uc.get("No", "")): list(uc["elite_assessment"]["verification"]["failures"]) for uc in candidates}
        self.logger.info(f"[ELITE] evidence repair for {len(candidates)} assessment(s) that failed verification only on "
                         f"their own citations: one re-judge each with the exact failures and the real columns, LLM "
                         f"cache bypassed: {list(failures)}")
        records, unjudged = self._run_elite_judge(candidates, schema_details, evidence, context, rt,
                                                  self._elite_worker_label(stage, "EvidenceRepair"),
                                                  force_fresh=True, repair_failures=failures)
        drops, repaired = Counter(), 0
        for uc in candidates:
            uc_id = str(uc.get("No", ""))
            record = records.get(uc_id)
            if record is None:
                uc["elite_assessment"]["evidence_repair"] = {"failures": failures[uc_id], "outcome": "unjudged",
                                                             "reason": unjudged.get(uc_id, "not judged")}
                drops["unjudged"] += 1
                continue
            verification = self._verify_and_canonicalize_elite(uc, record, evidence)
            record["evidence_repair"] = {"failures": failures[uc_id],
                                         "outcome": "passed" if verification["passed"] else "failed"}
            uc["elite_assessment"] = record
            if verification["passed"]:
                repaired += 1
            else:
                drops["verification"] += 1
                self.logger.info(f"[ELITE] evidence repair of {uc_id} still fails verification: "
                                 f"{'; '.join(verification['failures'][:3])}")
        self._log_funnel_stage(f"{stage}_evidence_repair", len(candidates), repaired, dict(drops))

    _ELITE_JUDGE_PIN_LOCK = threading.Lock()

    def _pinned_elite_judge(self):
        with self._ELITE_JUDGE_PIN_LOCK:
            demoted = set(LLM_ENDPOINT_HEALTH.snapshot()["demoted"])
            healthy = [m.get("llm_endpoint_name") for m in get_full_fallback_cascade("ULTRA_HIGH_ELITE_JUDGE_PROMPT")
                       if m.get("llm_endpoint_name") and m.get("llm_endpoint_name") not in demoted]
            current = getattr(self, "_elite_judge_pin", None)
            if current in healthy:
                return current
            preferred = [e for e in TECHNICAL_CONTEXT["runtime"]["elite_judge_preferred_endpoints"] if e in healthy]
            pick = (preferred or healthy or [None])[0]
            if pick is not None:
                reason = "the calibrated judge" if preferred else "no calibrated judge endpoint is healthy, so the first healthy one"
                if current:
                    self.logger.warning(f"[ELITE-JUDGE-PINNED FIRED] '{current}' is no longer healthy; the remaining elite judge "
                                        f"calls of this run start on '{pick}' ({reason})")
                else:
                    self.logger.info(f"[ELITE-JUDGE-PINNED FIRED] every elite judge call of this run starts on '{pick}' ({reason})")
            self._elite_judge_pin = pick
            return pick

    def _elite_second_opinion_model(self):
        cascade = get_full_fallback_cascade("ULTRA_HIGH_ELITE_JUDGE_PROMPT")
        judge = self._pinned_elite_judge() or (cascade[0].get("llm_endpoint_name") if cascade else None)
        return next((m.get("llm_endpoint_name") for m in cascade
                     if m.get("llm_endpoint_name") and m.get("llm_endpoint_name") != judge), None)

    def _run_elite_second_opinion(self, near_miss: list, schema_details: list, evidence: dict, context: dict,
                                  rt: dict, stage: str = "elite") -> list:
        from collections import Counter
        if not near_miss:
            self._log_funnel_stage(f"{stage}_second_opinion", 0, 0)
            return []
        model = self._elite_second_opinion_model()
        if model is None:
            self.logger.warning(f"[ELITE] the judge cascade has no second model; {len(near_miss)} near miss(es) "
                                f"cannot get an independent second opinion and stay below the bar")
            for uc in near_miss:
                a = uc["elite_assessment"]
                a["failed_gates"] = list(a.get("failed_gates") or []) + ["second_opinion_unavailable"]
                uc["selection_drop_reason"] = self._elite_drop_reason(a)
            self._log_funnel_stage(f"{stage}_second_opinion", len(near_miss), 0, {"second_opinion_unavailable": len(near_miss)})
            return []
        order = list(near_miss)
        random.Random(f"{getattr(self, 'session_id', '')}:elite_second_opinion").shuffle(order)
        self.logger.info(f"[ELITE] second opinion from '{model}' for {len(order)} near miss(es): LLM cache bypassed, "
                         f"order shuffled")
        records, _unjudged = self._run_elite_judge(order, schema_details, evidence, context, rt,
                                                   self._elite_worker_label(stage, "SecondOpinion"),
                                                   start_from_model=model, force_fresh=True)
        weights = rt["ultra_high_score_weights"]
        promoted, drops = [], Counter()
        for uc in near_miss:
            a = uc["elite_assessment"]
            second = records.get(str(uc.get("No", "")))
            if second is None:
                a["failed_gates"] = list(a.get("failed_gates") or []) + ["second_opinion_unjudged"]
                uc["selection_drop_reason"] = self._elite_drop_reason(a)
                drops["second_opinion_unjudged"] += 1
                continue
            self._verify_and_canonicalize_elite(uc, second, evidence)
            second["model"] = model
            a["second_opinion"] = second
            verdict, failed = self._elite_decision(a, a["verification"], rt)
            a["failed_gates"] = failed
            uc["elite_score"] = self._elite_composite(a, weights)
            if verdict == "ELITE":
                promoted.append(uc)
            else:
                uc["selection_drop_reason"] = self._elite_drop_reason(a)
                drops["not_confirmed"] += 1
        self._log_funnel_stage(f"{stage}_second_opinion", len(near_miss), len(promoted), dict(drops))
        return promoted

    def _dedup_elite_overlap(self, elite: list, stage: str = "elite") -> list:
        n_in = len(elite)
        kept = self._deduplicate_use_cases(elite, rank=self._final_rank_key) if n_in >= 2 else list(elite)
        kept, _ = self._dedup_by_signature(kept, "duplicate elite decision", rank=self._final_rank_key)
        n_overlap = len(kept)
        kept, twins = self._collapse_elite_decision_twins(kept)
        kept_ids = {id(uc) for uc in kept}
        dropped = [uc for uc in elite if id(uc) not in kept_ids]
        for uc in dropped:
            uc.pop("elite_status", None)
            winner = twins.get(id(uc))
            uc["selection_drop_reason"] = (
                f"same decision as the stronger elite use case \"{self._uc_title(winner) or 'a kept elite use case'}\" "
                f"(the higher elite score is kept)"
                if winner is not None else "overlaps a stronger elite use case (the higher elite score is kept)")
        if twins:
            self.logger.info(f"[ELITE] decision twins collapsed: "
                             + ", ".join(f"{uc.get('No')} -> {twins[id(uc)].get('No')}" for uc in dropped if id(uc) in twins))
        self._log_funnel_stage(f"{stage}_overlap", n_in, len(kept),
                               {"duplicate": n_in - n_overlap, "decision_twin": n_overlap - len(kept)})
        return kept

    def _apply_elite_coverage(self, elite: list, pool: list, outside: list, user_domains: list) -> tuple:
        n_in = len(elite)

        def _key(domain):
            return str(domain or "").strip().lower()

        if not user_domains:
            self._log_funnel_stage("elite_domain_coverage", n_in, n_in)
            return elite, outside
        elite_ids = {id(uc) for uc in elite}
        rest = [uc for uc in pool if id(uc) not in elite_ids]
        rest_keys = {_key(uc.get("Business Domain")) for uc in rest}
        result = self._apply_domain_coverage_picks(elite, rest, [d for d in user_domains if _key(d) in rest_keys])
        covered = {_key(uc.get("Business Domain")) for uc in result}
        uncovered = [d for d in user_domains if _key(d) and _key(d) not in covered]
        if uncovered and outside:
            self.logger.info(f"[ELITE] user domain(s) {uncovered} have no candidate in the pool; assigning the "
                             f"{len(outside)} remaining scored use case(s) to the user domains for a coverage pick")
            outside = self._assign_to_user_domains(outside, user_domains, "English")
        if uncovered:
            result = self._apply_domain_coverage_picks(result, outside, uncovered)
        self._log_funnel_stage("elite_domain_coverage", n_in, len(result))
        return result, outside

    def _judge_elite_candidates(self, survivors: list, schema_details: list, context: dict, rt: dict,
                                stage: str = "elite") -> tuple:
        from collections import Counter
        weights = rt["ultra_high_score_weights"]
        evidence = self._load_elite_evidence(survivors)
        records, unjudged = (self._run_elite_judge(survivors, schema_details, evidence, context, rt,
                                                   self._elite_worker_label(stage, "Judge"))
                             if survivors else ({}, {}))
        judged = []
        for uc in survivors:
            assessment = records.get(str(uc.get("No", "")))
            if assessment is None:
                uc["selection_drop_reason"] = (f"elite judge returned no valid assessment after retries: "
                                               f"{unjudged.get(str(uc.get('No', '')), 'not judged')}")
                continue
            uc["elite_assessment"] = assessment
            judged.append(uc)
        self._log_funnel_stage(f"{stage}_judge", len(survivors), len(judged), {"unjudged": len(survivors) - len(judged)})

        for uc in judged:
            self._verify_and_canonicalize_elite(uc, uc["elite_assessment"], evidence)
        self._repair_elite_evidence(judged, schema_details, evidence, context, rt, stage=stage)

        elite, near_miss, rejected_by_gate = [], [], Counter()
        for uc in judged:
            a = uc["elite_assessment"]
            verdict, failed = self._elite_decision(a, a["verification"], rt)
            a["failed_gates"] = failed
            uc["elite_score"] = self._elite_composite(a, weights)
            if verdict == "ELITE":
                elite.append(uc)
            elif verdict == "NEAR_MISS":
                near_miss.append(uc)
            else:
                uc["selection_drop_reason"] = self._elite_drop_reason(a)
                rejected_by_gate[failed[0]] += 1
        self.logger.info(f"[ELITE] first opinion: {len(elite)} elite, {len(near_miss)} near miss, "
                         f"{sum(rejected_by_gate.values())} rejected of {len(judged)} judged")
        self._log_funnel_stage(f"{stage}_verification", len(judged), len(elite) + len(near_miss), dict(rejected_by_gate))
        elite += self._run_elite_second_opinion(near_miss, schema_details, evidence, context, rt, stage=stage)
        for uc in elite:
            uc["elite_status"] = "ELITE"
        elite = self._dedup_elite_overlap(elite, stage=stage)
        return elite, judged, near_miss

    def _select_elite_use_cases(self, candidates: list, schema_details: list, context: dict) -> tuple:
        rt = TECHNICAL_CONTEXT["runtime"]
        user_domains = list(self.user_business_domains or [])
        self.elite_near_miss, self.elite_near_miss_ucs, self.elite_outcome = [], [], None
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Elite Selection",
                sub_step_name="Started", status="started",
                message=f"Ultra High elite selection started for {len(candidates)} scored use cases"
            )
        except Exception:
            pass
        log_print(f"\n🏅 ULTRA HIGH ELITE SELECTION: absolute bar over {len(candidates)} scored use cases (no quota, no cap)")

        pool, outside = self._build_elite_candidate_pool(candidates, rt)
        if pool and user_domains:
            pool = self._assign_to_user_domains(pool, user_domains, "English")
        survivors = self._elite_bob_prescreen(pool, schema_details, context, rt)
        elite, judged, near_miss = self._judge_elite_candidates(survivors, schema_details, context, rt)
        final, outside = self._apply_elite_coverage(elite, pool, outside, user_domains)
        self._log_funnel_stage("elite_final", len(candidates), len(final), {"not_selected": len(candidates) - len(final)})
        if not final:
            final = self._apply_elite_zero_bar_fallback(judged, candidates)

        missed = [uc for uc in judged if uc["elite_assessment"].get("failed_gates")]
        self.elite_near_miss = self._build_near_miss_list(missed)
        self.elite_near_miss_ucs = self._near_miss_candidates(missed)
        for idx, nm in enumerate(self.elite_near_miss, 1):
            self.logger.info(f"[ELITE] near miss {idx}: {nm['No']} | {nm['Name']} | elite_score={nm.get('elite_score')} | "
                             f"failed: {', '.join(nm.get('failed_gates') or [])} | {nm['reason']}")
        n_elite = sum(1 for uc in final if uc.get("elite_status") == "ELITE")
        n_cov = sum(1 for uc in final if uc.get("elite_status") == "DOMAIN_COVERAGE_PICK")
        n_fb = sum(1 for uc in final if uc.get("elite_status") == "FALLBACK_BELOW_BAR")
        message = f"{n_elite} use case{'' if n_elite == 1 else 's'} met the Ultra High bar"
        if n_cov:
            message += f"; {n_cov} domain coverage pick(s) ship labelled below bar"
        if n_fb:
            message += (f"; {n_fb} closest candidate(s) ship labelled below the Ultra High bar "
                        f"(no use case met the bar)")
        self.elite_outcome = {
            "mode": self.use_cases_quality,
            "elite": n_elite,
            "coverage_picks": n_cov,
            "fallback_picks": n_fb,
            "message": message,
            "candidate_pool": len(pool),
            "judged": len(judged),
            "unjudged": len(survivors) - len(judged),
            "second_opinion_model": self._elite_second_opinion_model() if near_miss else None,
            "near_miss": list(self.elite_near_miss),
        }
        log_print(f"🏅 {message}")
        self.logger.info(f"[ELITE] {message}")
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Elite Selection",
                sub_step_name="Complete", status="ended_success" if n_elite else "ended_warning",
                message=message, progress_increment=1.0, result_json=dict(self.elite_outcome)
            )
        except Exception:
            pass
        return final, pool + outside

    _BEST_OF_BEST_FIELDS = ("best_of_best", "best_of_best_rank", "best_of_best_score", "best_of_best_reason")
    _ELITE_JUDGE_FIELDS = ("elite_assessment", "elite_score", "elite_status", "selection_drop_reason")

    def _resolve_best_of_best_count(self) -> tuple:
        widget = getattr(self, "best_of_best_widget_count", None)
        if widget is not None:
            return widget, "widget"
        stated = (getattr(self, "parsed_generation_instructions", None) or {}).get("best_of_best_count")
        if stated is not None:
            if isinstance(stated, float) and stated.is_integer():
                stated = int(stated)
            try:
                count = parse_best_of_best_count(stated)
            except ValueError as e:
                self.logger.warning(f"[BEST-OF-BEST] the generation instructions asked for {stated!r} stars, which "
                                    f"is not usable ({e}); the default {BEST_OF_BEST_DEFAULT_COUNT} applies")
            else:
                if count is not None:
                    return count, "generation instructions"
        return BEST_OF_BEST_DEFAULT_COUNT, "default"

    @classmethod
    def _best_of_best_pass_reason(cls, assessment: dict) -> str:
        values = cls._elite_dimension_values(assessment) or {}
        scores = ", ".join(f"{dim.replace('_', ' ')} {values[dim]:g}/10" for dim in ULTRA_HIGH_DIMENSIONS if dim in values)
        return (f"passed every elite test: value tier {assessment.get('value_tier')}; {scores}; its cited columns and "
                f"joins were verified against the table metadata; the red team found no fatal flaw")

    def _pick_best_of_best(self, eligible: list, scores: dict, count: int) -> list:
        import math
        from collections import Counter

        def key(uc):
            return ((-scores[id(uc)],) + self._score_rank_part(uc.get("Inspire Score"))
                    + self._score_rank_part(uc.get("bob_score")) + (str(uc.get("No", "")),))

        def domain(uc):
            return str(uc.get("Business Domain") or "").strip().lower()

        ranked = sorted(eligible, key=key)
        cap = max(2, math.ceil(count * BEST_OF_BEST_DOMAIN_SHARE))
        picked, taken, per_domain = [], set(), Counter()

        def take(uc):
            picked.append(uc)
            taken.add(id(uc))
            per_domain[domain(uc)] += 1

        for wanted in (str(d or "").strip().lower() for d in getattr(self, "user_business_domains", None) or []):
            best = next((uc for uc in ranked if domain(uc) == wanted and id(uc) not in taken), None)
            if best is not None and len(picked) < count:
                take(best)
        for capped in (True, False):
            for uc in ranked:
                if len(picked) >= count:
                    break
                if id(uc) not in taken and (not capped or per_domain[domain(uc)] < cap):
                    take(uc)
        return sorted(picked, key=key)

    def _mark_best_of_best(self, use_cases: list, context: dict) -> list:
        for uc in use_cases:
            for field in self._BEST_OF_BEST_FIELDS:
                uc.pop(field, None)
        self._best_of_best_summary, self._best_of_best_near_miss = None, []
        if not use_cases:
            return []
        count, source = self._resolve_best_of_best_count()
        rt = TECHNICAL_CONTEXT["runtime"]
        assessments, scores, reasons, eligible, judged_n = {}, {}, {}, [], 0
        short_list = len(use_cases) <= count
        if count == 0:
            self.logger.info("[BEST-OF-BEST] the Best of Best count is 0; no use case is starred")
        elif short_list:
            self.logger.info(f"[BEST-OF-BEST] the catalog holds {len(use_cases)} use case(s), not more than the {count} "
                             f"stars asked for; it is already the short list and none is starred")
        elif self._is_ultra_high_mode():
            for uc in use_cases:
                a = uc.get("elite_assessment") if isinstance(uc.get("elite_assessment"), dict) else None
                score = self._finite_or(uc.get("elite_score"), None)
                judged_n += a is not None
                if score is not None:
                    scores[id(uc)] = score
                if uc.get("elite_status") == "ELITE" and a is not None and score is not None:
                    eligible.append(uc)
                    assessments[id(uc)] = a
                else:
                    reasons[id(uc)] = self._star_drop_reason(uc)
        else:
            minimum = rt["ultra_high_bob_prescreen_min"]
            prescreened = []
            for uc in use_cases:
                bob = self._finite_or(uc.get("bob_score"), None)
                if bob is not None and bob >= minimum:
                    prescreened.append(uc)
                else:
                    reasons[id(uc)] = (f"Best-of-Best score {bob:g} is below the elite prescreen minimum {minimum:g}"
                                       if bob is not None else "no Best-of-Best score to prescreen")
            copies = [{k: v for k, v in uc.items() if k not in self._ELITE_JUDGE_FIELDS} for uc in prescreened]
            origin = {id(copy): uc for copy, uc in zip(copies, prescreened)}
            if copies:
                schema_details = self.storage_manager.load_schema_details("all_columns_for_sql") or []
                elite, judged, _near_miss = self._judge_elite_candidates(copies, schema_details, context, rt,
                                                                         stage="best_of_best")
            else:
                elite, judged = [], []
            elite_ids, judged_ids = {id(c) for c in elite}, {id(c) for c in judged}
            judged_n = len(judged)
            for copy in copies:
                uc = origin[id(copy)]
                score = self._finite_or(copy.get("elite_score"), None)
                if score is not None:
                    scores[id(uc)] = score
                if id(copy) in elite_ids:
                    eligible.append(uc)
                    assessments[id(uc)] = copy["elite_assessment"]
                    uc["elite_assessment"] = copy["elite_assessment"]
                elif id(copy) in judged_ids:
                    reasons[id(uc)] = self._star_drop_reason(copy)
                else:
                    reasons[id(uc)] = copy.get("selection_drop_reason") or "the elite judge returned no assessment"
        starred = self._pick_best_of_best(eligible, scores, count) if count and not short_list else []
        rank_of = {id(uc): rank for rank, uc in enumerate(starred, 1)}
        idle = ("the Best of Best count is 0" if count == 0 else
                f"the catalog holds {len(use_cases)} use cases, not more than {count}, so it is already the short list")
        for uc in use_cases:
            rank = rank_of.get(id(uc))
            uc["best_of_best"] = "Y" if rank else "N"
            if id(uc) in scores:
                uc["best_of_best_score"] = round(scores[id(uc)], 2)
            if rank:
                uc["best_of_best_rank"] = rank
                uc["best_of_best_reason"] = self._best_of_best_pass_reason(assessments[id(uc)])
            elif id(uc) in assessments:
                uc["best_of_best_reason"] = f"passed every elite test; outside the top {count} by elite score"
            else:
                uc["best_of_best_reason"] = reasons.get(id(uc)) or idle
        self._best_of_best_near_miss = sorted(
            (uc for uc in use_cases if id(uc) not in assessments and id(uc) in scores),
            key=lambda uc: (-scores[id(uc)], str(uc.get("No", ""))))[:10]
        self._best_of_best_summary = {"target": count, "source": source, "mode": self.use_cases_quality,
                                      "short_list": short_list, "judged": judged_n, "passed": len(eligible)}
        self.logger.info(f"[BEST-OF-BEST FIRED] starred {len(starred)} of a target of {count} ({source}); "
                         f"{len(eligible)} of {judged_n} judged use case(s) passed every elite test")
        return starred

    def _publish_best_of_best(self, use_cases: list):
        summary = getattr(self, "_best_of_best_summary", None)
        if summary is None:
            return None
        starred = sorted((uc for uc in use_cases if uc.get("best_of_best") == "Y"),
                         key=lambda uc: uc.get("best_of_best_rank") or 0)
        near_miss = [{"No": uc.get("No", ""), "Name": uc.get("Name", ""), "Business Domain": uc.get("Business Domain", ""),
                      "best_of_best_score": uc.get("best_of_best_score"), "reason": uc.get("best_of_best_reason")}
                     for uc in getattr(self, "_best_of_best_near_miss", None) or []]
        outcome = dict(summary, starred=len(starred), ids=[uc.get("No", "") for uc in starred], near_miss=near_miss)
        self.best_of_best_outcome = outcome
        message = (f"{len(starred)} use case{'' if len(starred) == 1 else 's'} starred as Best of Best "
                   f"(target {summary['target']}, from {summary['source']}); {summary['passed']} of "
                   f"{summary['judged']} judged passed every elite test")
        log_print(f"⭐ {message}")
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Best of Best", sub_step_name="Complete",
                status="ended_success" if starred or summary["target"] == 0 or summary.get("short_list") else "ended_warning",
                message=message, progress_increment=0.0, result_json=dict(outcome))
        except Exception:
            pass
        return outcome

    def run(self):
        import time
        self._run_start_time = time.time()
        self.logger.info(f"Starting tasks: {self.generate_choices}, Operation Mode: {self.operation_mode}")
        self.logger.info(f"🔑 Session ID: {self.session_id}")
        log_print(f"🔑 Inspire Session ID: {self.session_id}")
        if self.inspire_database:
            self._ensure_inspire_database_exists()
            self._create_tracking_table()
        self._load_feedback()
        self._attach_ailogs()
        try:
            self._start_live_log_flush_thread()
        except Exception as _vle:
            try:
                self.logger.warning(f"Live log streaming failed to start (non-critical): {get_clean_error_message(_vle)}")
            except Exception:
                pass
        if self.atomic_writer:
            self.atomic_writer.logger = self.logger
            self.atomic_writer.initialize_session()
            self._pipeline_start_step_id = self._emit_pipeline_status(
                stage_name="Pipeline",
                step_name="Execution",
                sub_step_name="Start",
                message="Inspire pipeline started",
                status="started",
                progress_increment=1.0,
                result_json={"operation_mode": self.operation_mode, "session_id": self.session_id}
            )
        if self.inspire_database:
            _db_setup_step_id = self._emit_pipeline_status(
                stage_name="Initialization",
                step_name="Database Setup",
                sub_step_name=self.inspire_database,
                message="Setting up Inspire database",
                status="started",
                progress_increment=0.0
            )
            self._emit_pipeline_status(
                stage_name="Initialization",
                step_name="Database Setup",
                sub_step_name=self.inspire_database,
                message="Inspire database and tracking structures are ready",
                status="ended_success",
                progress_increment=1.0,
                result_json={"database": self.inspire_database},
                step_id=_db_setup_step_id
            )
        
        if 'PROMPT_TEMPLATES' not in globals():
            self.logger.critical("CRITICAL ERROR: 'PROMPT_TEMPLATES' dictionary is not defined. Please run the cell defining it.")
            log_print("CRITICAL ERROR: 'PROMPT_TEMPLATES' dictionary is not defined. Please run the cell defining it.", level="CRITICAL")
            raise NameError("PROMPT_TEMPLATES not found. Please run the cell that defines the prompt dictionary.")
        
        if self.auto_parallelism and self.max_parallelism <= 0:
            (recommended, tables_per_batch, est_batches, avg_table_chars, max_by_memory) = self._calculate_dynamic_parallelism(
                0,
                0,
                0,
                0
            )
            self.max_parallelism = recommended
            self.logger.info(f"Dynamic parallelism set to {self.max_parallelism} (memory_cap={max_by_memory})")
            log_print(f"✅ Dynamic parallelism: {self.max_parallelism}")

        english_translations = self.translation_service.get_translations("English")
        final_consolidated_use_cases = []
        
        if self.json_file_path:
            self.logger.info(f"🚀 JSON MODE: Loading use cases from JSON file: {self.json_file_path}")
            log_print(f"\n{'='*80}")
            log_print(f"🚀 JSON MODE ACTIVATED")
            log_print(f"{'='*80}")
            log_print(f"📁 JSON File: {self.json_file_path}")
            log_print(f"📋 Languages: {', '.join(self.output_languages)}")
            log_print(f"\n⚠️  SKIPPING:", level="WARNING")
            log_print(f"   ❌ Use Case Generation (using existing data from JSON)")
            log_print(f"\n✅ GENERATING:")
            log_print(f"   📓 Notebooks (always generated)")
            log_print(f"   📄 PDF Catalogs" if "PDF Catalog" in self.generate_choices or "Use Cases Catalog PDF" in self.generate_choices else "   (PDF generation not selected)")
            log_print(f"   📊 Presentations" if "Presentation" in self.generate_choices else "   (Presentation generation not selected)")
            log_print(f"{'='*80}\n")
            
            try:
                # Load the JSON catalog
                (final_consolidated_use_cases, summary_dict, english_grouped_data) = self._load_usecases_catalog_json(self.json_file_path)
                summary_dict = self._refresh_english_summary(english_grouped_data, summary_dict)
                
                # 1. Generate documents for all languages (Excel deferred until after notebooks)
                if "PDF Catalog" in self.generate_choices or "Presentation" in self.generate_choices or "Use Cases Catalog PDF" in self.generate_choices or "English" in self.output_languages:
                    self._generate_documents_for_all_languages(
                        final_consolidated_use_cases,
                        english_grouped_data=english_grouped_data,
                        summary_dict=summary_dict,
                        skip_excel_langs=["English"]
                    )
                
                # Report table inclusion/exclusion statistics
                if final_consolidated_use_cases:
                    self._report_table_statistics(final_consolidated_use_cases)
                
                # 2. Generate Genie Code instructions (if selected, JSON mode)
                if self.generate_genie_instructions and final_consolidated_use_cases:
                    self.logger.info("🤖 Genie Code Instructions selected - generating from JSON data...")
                    log_print(f"\n{'='*80}")
                    log_print(f"🤖 GENIE CODE INSTRUCTIONS (from JSON)")
                    log_print(f"{'='*80}")
                    _json_schema_details = self._build_schema_details_from_use_cases(final_consolidated_use_cases)
                    try:
                        self._generate_genie_instructions_and_notebooks(
                            final_consolidated_use_cases,
                            _json_schema_details,
                            "",
                            english_translations,
                            summary_dict
                        )
                    except Exception as e:
                        self.logger.error(f"Genie Code instruction generation failed: {get_clean_error_message(e)}")
                        log_print(f"⚠️ Genie Code instruction generation encountered an error: {str(e)[:200]}", level="WARNING")
                    finally:
                        del _json_schema_details
                elif (not self.generate_genie_instructions) and final_consolidated_use_cases and getattr(self, 'operation', '') == 'Discover Use Cases':
                    self.logger.info("🔎 Discover mode (JSON): Capturing per-UC prompt fields + generating skeleton notebooks...")
                    _json_schema_details = self._build_schema_details_from_use_cases(final_consolidated_use_cases)
                    try:
                        final_consolidated_use_cases = self._run_discover_skeleton_phase(
                            final_consolidated_use_cases,
                            _json_schema_details
                        )
                    except Exception as e:
                        self.logger.error(f"Discover skeleton phase failed (JSON): {get_clean_error_message(e)}")
                        log_print(f"⚠️ Discover skeleton phase failed: {str(e)[:200]}", level="WARNING")
                    finally:
                        del _json_schema_details
                
                # 3. Generate Notebooks (batch fallback — only if NOT already streamed inline during genie phase)
                _notebooks_done_inline_json = getattr(self, '_notebooks_generated_inline', False)
                if final_consolidated_use_cases and not _notebooks_done_inline_json:
                    self.logger.info("📓 Generating per-use-case notebooks from JSON data (batch fallback)...")
                    self.assemble_use_case_notebooks(final_consolidated_use_cases, english_translations, summary_dict)
                elif _notebooks_done_inline_json:
                    self.logger.info("📓 Notebooks already generated inline during genie streaming phase — skipping batch generation")
                else:
                    self.logger.warning("No use cases found in JSON, skipping notebook creation.")
                
                # 4. Generate English Excel AFTER notebooks exist so Use Case ID hyperlinks resolve
                if final_consolidated_use_cases:
                    self.logger.info("Generating English Excel (post-notebook, with hyperlinks)...")
                    lang_abbr_en = self._get_lang_abbr("English")
                    try:
                        self._generate_use_case_excel("English", lang_abbr_en, english_grouped_data)
                    except Exception as e:
                        self.logger.error(f"Failed to generate English Excel: {get_clean_error_message(e)}")

                AIAgent.get_summary_report()
                self.ai_agent.get_manager_stats_report()
                log_print(f"\n{'='*80}")
                self._report_run_completion(
                    f"✅ SUCCESS: All artifacts for {self.business_name} generated successfully from JSON",
                    f"⚠️ COMPLETED WITH WARNINGS: artifacts for {self.business_name} generated from JSON")
                log_print(f"{'='*80}\n")
                self.logger.info("Uploading log file...")
                self._upload_log_file()
                return
                
            except Exception as e:
                self.logger.critical(f"Failed to process in docs-only mode: {get_clean_error_message(e)}")
                log_print(f"\n❌ FATAL: Failed to process in docs-only mode: {get_clean_error_message(e, max_chars=200)}")
                AIAgent.get_summary_report()
                self.ai_agent.get_manager_stats_report()
                try:
                    self.logger.warning("[G29 FAILURE_LOG_UPLOAD FIRED] path=docs_only_mode")
                    self._upload_log_file()
                except Exception:
                    pass
                raise
        
        # === GENERATE USE CASES (Operation == 'Generate Use Cases'): short-circuit. ===
        # Scans __inspire_usecases (Route 1) then workspace notebooks (Route 2) for flagged
        # use cases and regenerates their Genie Code Instruction only. Entire use-case
        # discovery pipeline is skipped.
        if getattr(self, 'operation', '') == 'Generate Use Cases':
            try:
                self._run_generate_mode()
            except Exception as _gen_err:
                self.logger.critical(f"Generate Use Cases mode failed: {get_clean_error_message(_gen_err)}")
                log_print(f"\n❌ FATAL: Generate Use Cases mode failed: {get_clean_error_message(_gen_err, max_chars=200)}")
                AIAgent.get_summary_report()
                self.ai_agent.get_manager_stats_report()
                try:
                    self.logger.warning("[G29 FAILURE_LOG_UPLOAD FIRED] path=generate_mode")
                    self._upload_log_file()
                except Exception:
                    pass
                raise
            self._report_run_completion(
                f"✅ SUCCESS: Genie Code regeneration for {self.business_name} completed",
                f"⚠️ COMPLETED WITH WARNINGS: Genie Code regeneration for {self.business_name}")
            AIAgent.get_summary_report()
            self.ai_agent.get_manager_stats_report()
            self._upload_log_file()
            return

        if getattr(self, 'operation', '') == IMPLEMENT_OPERATION:
            try:
                self._run_implement_mode()
            except Exception as _impl_err:
                self.logger.critical(f"Implement Use Cases failed: {get_clean_error_message(_impl_err)}")
                log_print(f"\n❌ FATAL: Implement Use Cases failed: {get_clean_error_message(_impl_err, max_chars=200)}")
                try:
                    self.logger.warning("[G29 FAILURE_LOG_UPLOAD FIRED] path=implement_mode")
                    self._upload_log_file()
                except Exception:
                    pass
                raise
            self._report_run_completion(
                f"✅ SUCCESS: Use case PoC/MVP with Genie for {self.business_name} completed",
                f"⚠️ COMPLETED WITH WARNINGS: Use case PoC/MVP with Genie for {self.business_name}")
            AIAgent.get_summary_report()
            self.ai_agent.get_manager_stats_report()
            self._upload_log_file()
            return
        
        # === NORMAL PATH: Generate use cases from metadata ===
        
        # === NEW: Call Business Context Worker first ===
        self.logger.info("=" * 80)
        self.logger.info("🚀 STEP 1: EXTRACTING BUSINESS CONTEXT, STRATEGIC GOALS, AND PRIORITIES")
        self.logger.info("=" * 80)
        _ctx_extraction_step_id = self._emit_pipeline_status(
            stage_name="Business Context",
            step_name="Context Extraction",
            sub_step_name="LLM",
            message="Business context extraction started",
            status="started",
            progress_increment=1.0
        )
        
        try:
            # Prepare user context string early - now uses business_domains instead of use_cases_focus
            user_domains_str = ', '.join(self.user_business_domains) if self.user_business_domains else ''
            
            # Extract business context AND interpret generation instructions in parallel
            with _SafeExecutorContext(max_workers=2, thread_name_prefix="CtxInit", logger=self.logger, name="ContextInit") as _ctx_executor:
                _ctx_future = _ctx_executor.submit(self._get_business_context_from_llm)
                _instr_future = _ctx_executor.submit(self._interpret_generation_instructions)
                llm_business_context, _parsed_instructions = self._await_context_init(_ctx_future, _instr_future)
            self.logger.info("✅ Business context extraction and generation instructions interpretation completed (parallel)")
            
            # Merge with user-provided domains (user takes precedence)
            merged_business_context = self._merge_business_contexts(llm_business_context, user_domains_str)
            
            # Handle user-provided strategic goals (hard focus)
            if self.user_strategic_goals:
                self.logger.info(f"✅ User provided {len(self.user_strategic_goals)} strategic goals - ONLY these will be used")
                merged_business_context["strategic_goals"] = self.user_strategic_goals
                self.logger.info(f"   Strategic Goals: {', '.join(self.user_strategic_goals)}")
            else:
                self.logger.info("ℹ️ No user strategic goals provided - will generate goals based on business context")
                llm_goals = merged_business_context.get("strategic_goals", [])
                if isinstance(llm_goals, str):
                    llm_goals = [g.strip() for g in llm_goals.split(",") if g.strip()]
                merged_business_context["strategic_goals"] = llm_goals
            
            # Handle user-provided business priorities
            if self.user_business_priorities:
                if getattr(self, "_customer_priorities_override", False):
                    self.logger.info(f"✅ Strategic Context supplies {len(self.user_business_priorities)} business priorities; "
                                     f"they replace 07_business_priorities")
                self.logger.info(f"✅ User provided {len(self.user_business_priorities)} business priorities - these order the final use cases")
                merged_business_context["business_priorities"] = self.user_business_priorities
                self.logger.info(f"   Business Priorities: {', '.join(self.user_business_priorities)}")

            self._resolve_strategy_tree(merged_business_context)
            
            # Handle user-provided business domains
            if self.user_business_domains:
                self.logger.info(f"✅ User provided {len(self.user_business_domains)} business domains - use cases will be aligned to these domains ONLY")
                merged_business_context["user_business_domains"] = self.user_business_domains
                self.logger.info(f"   Business Domains: {', '.join(self.user_business_domains)}")
            else:
                self.logger.info("ℹ️ No user business domains provided - domains will be inferred from data")
            
            self.logger.info("✅ Business context extracted and merged.")
            self.logger.info("=" * 80)
            _ctx_result = {
                "business_domains_count": len(merged_business_context.get("user_business_domains", []) or []),
                "strategic_goals_count": len(merged_business_context.get("strategic_goals", []) or []),
                "business_context": str(merged_business_context.get("business_context", ""))[:500],
                "strategic_goals": merged_business_context.get("strategic_goals", []),
                "business_priorities": merged_business_context.get("business_priorities", []),
                "user_business_domains": merged_business_context.get("user_business_domains", []),
                "strategic_initiative": str(merged_business_context.get("strategic_initiative", ""))[:300],
                "revenue_model": str(merged_business_context.get("revenue_model", ""))[:300],
            }
            self._emit_pipeline_status(
                stage_name="Business Context",
                step_name="Context Extraction",
                sub_step_name="LLM",
                message="Business context extraction completed",
                status="ended_success",
                progress_increment=3.0,
                result_json=_ctx_result,
                step_id=_ctx_extraction_step_id
            )
        except Exception as _ctx_err:
            self._emit_pipeline_status(
                stage_name="Business Context",
                step_name="Context Extraction",
                sub_step_name="LLM",
                message=f"Business context extraction failed: {get_clean_error_message(_ctx_err, max_chars=200)}",
                status="ended_error",
                progress_increment=1.0,
                result_json={"error": get_clean_error_message(_ctx_err, max_chars=300)},
                step_id=_ctx_extraction_step_id
            )
            raise
        
        # Store merged context for use in prompt generation
        self.merged_business_context = merged_business_context
        self._persist_session_prompt_context()
        
        # Extract business context fields for prompts
        ctx_business_context = merged_business_context.get("business_context", "") or ""
        ctx_industry = merged_business_context.get("industry", "") or ""
        ctx_strategic_initiative = merged_business_context.get("strategic_initiative", "") or ""
        ctx_value_chain = merged_business_context.get("value_chain", "") or ""
        ctx_revenue_model = merged_business_context.get("revenue_model", "") or ""
        scoring_context = {
            "business_context": ctx_business_context,
            "industry": ctx_industry,
            "strategic_initiative": ctx_strategic_initiative,
            "value_chain": ctx_value_chain,
            "revenue_model": ctx_revenue_model,
        }
        self.funnel_ledger = []
        
        
        try:
            if self.data_loader:
                self.logger.debug("Data loader found. Starting batched table processing with MAX_CONTEXT_CHARS management...")
                
                document_context_markdown = ""

                # Now process tables in batches with model-specific context limits from TECHNICAL_CONTEXT
                use_case_gen_context_limit = get_max_context_chars("English", "BASE_USE_CASE_GEN_PROMPT")
                safe_context_limit = get_safe_context_limit("English", buffer_percent=0.9, prompt_name="BASE_USE_CASE_GEN_PROMPT")
                batch_num = 1
                accumulated_columns = []
                accumulated_schema_size = 0
                batches_to_process = []  # List of (batch_num, column_details) tuples
                
                base_prompt_size = self._generation_prompt_base_size((
                    ctx_business_context, ctx_industry, ctx_strategic_initiative, ctx_value_chain,
                    ctx_revenue_model)) + 2000
                self.logger.debug(f"Base generation prompt size: {base_prompt_size} chars, context limit: {use_case_gen_context_limit}")
                
                # CRITICAL FIX: Keep pulling tables until we fill the model's context limit
                self.logger.info("Collecting batches for parallel processing (MAXIMIZING context utilization)...")
                while True:
                    # Keep pulling table batches until we hit the context limit
                    while True:
                        batch_columns = self.data_loader.getNextTables(self.scan_parallelism)
                        
                        if batch_columns is None:
                            # No more tables available
                            break

                        batch_columns = self._augment_columns_with_foreign_keys(batch_columns)
                        
                        batch_schema_size = self._estimate_schema_markdown_size(batch_columns)
                        estimated_prompt_size = base_prompt_size + accumulated_schema_size + batch_schema_size
                        
                        self.logger.debug(f"Batch {batch_num}: Got {len(batch_columns)} columns. "
                                       f"Total accumulated: {len(accumulated_columns) + len(batch_columns)} columns. "
                                       f"Estimated prompt size: {estimated_prompt_size}/{safe_context_limit} chars")
                        
                        if estimated_prompt_size > safe_context_limit:
                            if not accumulated_columns:
                                self.logger.warning(f"Initial table pull exceeds context limit ({estimated_prompt_size} chars). Splitting into sub-batches.")
                                split_batches = self._split_columns_to_fit_context(batch_columns, base_prompt_size, safe_context_limit)
                                for idx, split_columns in enumerate(split_batches, start=1):
                                    table_count = len({(c[0], c[1], c[2]) for c in split_columns})
                                    self.logger.warning(f"   Sub-batch {idx}: {table_count} tables ({len(split_columns)} columns)")
                                    batches_to_process.append((batch_num, split_columns))
                                    batch_num += 1
                                accumulated_columns = []
                                accumulated_schema_size = 0
                                break
                            else:
                                # Would exceed limit - save current accumulator as a batch
                                self.logger.info(f"Context limit reached. Saving batch {batch_num} with {len(accumulated_columns)} columns ({base_prompt_size + accumulated_schema_size} chars)")
                                batches_to_process.append((batch_num, accumulated_columns))
                                batch_num += 1
                                
                                # Start new accumulator with the current batch
                                accumulated_columns = batch_columns
                                accumulated_schema_size = batch_schema_size
                                break
                        else:
                            # Fits - keep accumulating and PULL MORE TABLES
                            accumulated_columns.extend(batch_columns)
                            accumulated_schema_size += batch_schema_size
                            # Continue inner loop to pull more tables
                    
                    # Check if we're done (no more tables)
                    if batch_columns is None:
                        # Add any remaining accumulated columns to batches
                        if accumulated_columns:
                            self.logger.info(f"Adding final batch {batch_num} with {len(accumulated_columns)} columns")
                            batches_to_process.append((batch_num, accumulated_columns))
                        break

                self._report_metadata_unavailable()
                
                if not batches_to_process:
                    raise RuntimeError(self._nothing_to_discover_message())
                
                total_tables = len({(c, s, t) for _, cols in batches_to_process for (c, s, t, _, _, _) in cols})
                tables_per_call = self._determine_tables_per_call(total_tables)
                if tables_per_call > 0:
                    adjusted_batches = []
                    next_batch_num = 1
                    for _, batch_columns in batches_to_process:
                        grouped = self._split_by_table_limit(batch_columns, tables_per_call)
                        for group in grouped:
                            estimated_prompt_size = base_prompt_size + self._estimate_schema_markdown_size(group)
                            if estimated_prompt_size > safe_context_limit:
                                singles = self._split_by_table_limit(group, 1)
                                for single in singles:
                                    adjusted_batches.append((next_batch_num, single))
                                    next_batch_num += 1
                            else:
                                adjusted_batches.append((next_batch_num, group))
                                next_batch_num += 1
                    batches_to_process = adjusted_batches
                    self.logger.info(f"Table-based batching: {tables_per_call} tables per call, {len(batches_to_process)} batches")
                

                # === TABLE JOIN MERGE: Combine batches containing user-specified joined tables ===
                table_joins = self.parsed_generation_instructions.get("table_joins", [])
                if table_joins:
                    self.logger.info(f"🔗 Processing {len(table_joins)} table join instruction(s)...")
                    for join_spec in table_joins:
                        left_tbl = join_spec.get("left_table", "").lower().strip()
                        right_tbl = join_spec.get("right_table", "").lower().strip()
                        if not left_tbl or not right_tbl:
                            continue
                        left_batch_idx = None
                        right_batch_idx = None
                        for bidx, (bnum, bcols) in enumerate(batches_to_process):
                            batch_fqns = {f"{c[0]}.{c[1]}.{c[2]}".lower() for c in bcols}
                            if left_tbl in batch_fqns:
                                left_batch_idx = bidx
                            if right_tbl in batch_fqns:
                                right_batch_idx = bidx
                        if left_batch_idx is not None and right_batch_idx is not None and left_batch_idx != right_batch_idx:
                            left_bnum, left_cols = batches_to_process[left_batch_idx]
                            right_bnum, right_cols = batches_to_process[right_batch_idx]
                            merged_cols = left_cols + [c for c in right_cols if (c[0], c[1], c[2]) not in {(lc[0], lc[1], lc[2]) for lc in left_cols}]
                            self.logger.info(f"  🔗 Merged batch {left_bnum} ({left_tbl}) + batch {right_bnum} ({right_tbl}) into single batch")
                            batches_to_process[left_batch_idx] = (left_bnum, merged_cols)
                            batches_to_process.pop(right_batch_idx)
                            for k in range(len(batches_to_process)):
                                batches_to_process[k] = (k + 1, batches_to_process[k][1])
                        elif left_batch_idx is not None and right_batch_idx is not None:
                            self.logger.info(f"  ✅ Tables {left_tbl} and {right_tbl} already in the same batch")
                        else:
                            missing = []
                            if left_batch_idx is None:
                                missing.append(left_tbl)
                            if right_batch_idx is None:
                                missing.append(right_tbl)
                            self.logger.warning(f"  ⚠️ Could not find table(s) {missing} in any batch for join")


                # === POST-MERGE CONTEXT SIZE VALIDATION ===
                if table_joins:
                    overflow_batches = []
                    for bidx, (bnum, bcols) in enumerate(batches_to_process):
                        merged_size = base_prompt_size + self._estimate_schema_markdown_size(bcols)
                        if merged_size > safe_context_limit:
                            overflow_batches.append(bidx)
                    if overflow_batches:
                        self.logger.warning(f"  ⚠️ {len(overflow_batches)} merged batch(es) exceed context limit -- splitting into single-table sub-batches")
                        new_batches = []
                        next_num = 1
                        for bidx, (bnum, bcols) in enumerate(batches_to_process):
                            if bidx in overflow_batches:
                                singles = self._split_by_table_limit(bcols, 1)
                                for single in singles:
                                    new_batches.append((next_num, single))
                                    next_num += 1
                            else:
                                new_batches.append((next_num, bcols))
                                next_num += 1
                        batches_to_process = new_batches
                        self.logger.info(f"  📦 Post-merge re-split: {len(batches_to_process)} batches")

                # === NEW: Filter business vs technical tables ===
                try:
                    self._emit_pipeline_status(
                        stage_name="Data Preparation", step_name="Filter Business Tables",
                        sub_step_name="Started", status="started",
                        message="Classifying tables as BUSINESS vs TECHNICAL"
                    )
                except Exception:
                    pass
                self.logger.info("🔍 Filtering tables into BUSINESS vs TECHNICAL categories...")
                log_print(f"\n{'='*80}")
                log_print(f"🔍 FILTERING TABLES: Business Data vs Technical/Metadata")
                log_print(f"{'='*80}\n")
                
                # Collect all columns from all batches for filtering
                all_batch_columns = []
                for batch_num, batch_columns in batches_to_process:
                    all_batch_columns.extend(batch_columns)

                # Early FK inference (pre-UC-gen): populate foreign_key_graph with
                # column-name-inferred FKs BEFORE UC batches run their declared-FK lookup.
                # Without this, the per-batch fk_relationships_text is always "None" when
                # UC has no explicit FKs, and LLM misses multi-table UC opportunities.
                try:
                    if self.data_loader and all_batch_columns:
                        _pre_inferred = self.data_loader.merge_inferred_foreign_keys(all_batch_columns)
                        if _pre_inferred:
                            self.logger.info(f"🔗 Early FK inference: {_pre_inferred} relationships available for UC generation")
                except Exception as _pre_fk_err:
                    self.logger.debug(f"Early FK inference skipped: {_pre_fk_err}")
                
                if self.auto_parallelism:
                    total_tables = len({(c, s, t) for (c, s, t, _, _, _) in all_batch_columns})
                    total_schema_chars = self._estimate_schema_markdown_size(all_batch_columns)
                    (recommended, tables_per_batch, est_batches, avg_table_chars, max_by_memory) = self._calculate_dynamic_parallelism(
                        total_tables,
                        total_schema_chars,
                        safe_context_limit,
                        base_prompt_size
                    )
                    self._batch_parallelism_override = recommended
                    if recommended < self.max_parallelism:
                        self.logger.info(
                            f"Batch-specific parallelism set to {recommended} (est_batches={est_batches}), "
                            f"global max_parallelism stays at {self.max_parallelism} (memory_cap={max_by_memory})"
                        )
                    else:
                        self.max_parallelism = recommended
                        self.logger.info(f"Dynamic parallelism set to {self.max_parallelism} (tables={total_tables}, avg_table_chars={avg_table_chars}, tables_per_batch={tables_per_batch}, est_batches={est_batches}, memory_cap={max_by_memory})")
                    log_print(f"✅ Dynamic parallelism: {self.max_parallelism} (tables={total_tables}, est_batches={est_batches})")
                
                # Filter tables
                (business_details, technical_details, business_tables, technical_tables, business_scores, data_category_map, master_tables_set, transactional_tables_set, reference_tables_set) = self._filter_business_tables(
                    all_batch_columns,
                    business_context=ctx_business_context,
                    industry=ctx_industry,
                    exclusion_strategy=self.technical_exclusion_strategy
                )
                
                # === MEMORY OPTIMIZATION: Clear all_batch_columns after filtering ===
                del all_batch_columns
                gc.collect()
                self.logger.debug("🧹 Cleared all_batch_columns from memory")
                
                self._set_known_business_universe(business_details)
                self._known_hallucinated_tables = set()
                
                self.business_scores = business_scores
                self.data_category_map = data_category_map
                if business_details:
                    try:
                        self._collect_discovery_evidence(business_details)
                    except SparkWorkCancelled:
                        raise
                    except Exception as _evidence_err:
                        self.discovery_evidence = None
                        self.logger.warning(f"Discovery evidence unavailable; discovery continues on metadata only: "
                                            f"{get_clean_error_message(_evidence_err)}")
                
                if not business_details:
                    self.logger.warning("No business tables found after filtering. All tables were classified as technical/metadata.")
                    log_print("⚠️ No business tables found. All tables appear to be technical/metadata.", level="WARNING")
                    return
                
                self.logger.info(f"✅ Filtering complete: Proceeding with {len(business_tables)} business tables, "
                               f"excluding {len(technical_tables)} technical/metadata tables, "
                               f"including {len(reference_tables_set)} reference tables as joinable dims")
                
                master_details = []
                transactional_details = []
                for detail in business_details:
                    (catalog, schema, table, _, _, _) = detail
                    fqtn = f"{catalog}.{schema}.{table}"
                    if fqtn in transactional_tables_set:
                        transactional_details.append(detail)
                    else:
                        master_details.append(detail)
                
                _rt = TECHNICAL_CONTEXT["runtime"]
                _table_election_threshold = _rt["table_election_threshold"]
                _table_election_tx_min = _rt["table_election_transactional_tables_min"]
                total_business_tables = len(business_tables)
                tx_count = len(transactional_tables_set)
                
                elected_master_details = master_details
                elected_transactional_details = transactional_details
                
                if self.table_election_mode == "Let Inspire Decides":
                    if total_business_tables > _table_election_threshold:
                        if tx_count >= _table_election_tx_min:
                            elected_master_details = []
                            log_print(f"🗳️ Table Election: {total_business_tables} tables > threshold ({_table_election_threshold}), "
                                      f"using TRANSACTIONAL tables only ({tx_count} tables)")
                            self.logger.info(f"Table Election [Let Inspire Decides]: total={total_business_tables} > threshold={_table_election_threshold}, "
                                           f"tx={tx_count} >= min={_table_election_tx_min} → transactional only")
                        else:
                            log_print(f"🗳️ Table Election: {total_business_tables} tables > threshold ({_table_election_threshold}), "
                                      f"but transactional tables ({tx_count}) < min ({_table_election_tx_min}), "
                                      f"including MASTER tables as well")
                            self.logger.info(f"Table Election [Let Inspire Decides]: total={total_business_tables} > threshold={_table_election_threshold}, "
                                           f"tx={tx_count} < min={_table_election_tx_min} → transactional + master")
                    else:
                        log_print(f"🗳️ Table Election: {total_business_tables} tables <= threshold ({_table_election_threshold}), "
                                  f"using ALL business tables (master + transactional)")
                        self.logger.info(f"Table Election [Let Inspire Decides]: total={total_business_tables} <= threshold={_table_election_threshold} → all tables")
                elif self.table_election_mode == "Transactional Only":
                    elected_master_details = []
                    if not elected_transactional_details:
                        self.logger.warning("Table Election [Transactional Only]: No transactional tables found. Falling back to all business tables.")
                        log_print("⚠️ Table Election: No transactional tables found. Falling back to all business tables.", level="WARNING")
                        elected_master_details = master_details
                    else:
                        log_print(f"🗳️ Table Election: TRANSACTIONAL ONLY mode — using {tx_count} transactional tables, "
                                  f"excluding {len(master_tables_set)} master tables")
                        self.logger.info(f"Table Election [Transactional Only]: tx={tx_count}, excluded master={len(master_tables_set)}")
                else:
                    log_print(f"🗳️ Table Election: ALL TABLES mode — using all {total_business_tables} business tables")
                    self.logger.info(f"Table Election [All Tables]: using all {total_business_tables} business tables")

                if self.table_election_mode == "Let Inspire Decides" and master_details and not elected_master_details:
                    elected_master_details = self._usage_kept_master_details(master_details, business_details)
                
                master_details = elected_master_details
                transactional_details = elected_transactional_details
                
                master_tables_count = len({f"{c}.{s}.{t}" for (c, s, t, _, _, _) in master_details}) if master_details else 0
                tables_per_call = self._determine_tables_per_call(master_tables_count)
                
                adjusted_batches = []
                next_batch_num = 1
                
                if master_details:
                    grouped_master = self._split_by_table_limit(master_details, tables_per_call)
                    for group in grouped_master:
                        estimated_prompt_size = base_prompt_size + self._estimate_schema_markdown_size(group)
                        if estimated_prompt_size > safe_context_limit:
                            singles = self._split_by_table_limit(group, 1)
                            for single in singles:
                                adjusted_batches.append((next_batch_num, single))
                                next_batch_num += 1
                        else:
                            adjusted_batches.append((next_batch_num, group))
                            next_batch_num += 1
                
                if transactional_details:
                    grouped_tx = self._split_by_table_limit(transactional_details, 1)
                    for group in grouped_tx:
                        adjusted_batches.append((next_batch_num, group))
                        next_batch_num += 1
                
                augmented_batches = []
                for batch_num, batch_columns in adjusted_batches:
                    augmented = self._augment_columns_with_related_tables(batch_columns)
                    augmented_batches.append((batch_num, augmented))
                batches_to_process = augmented_batches
                self._report_metadata_unavailable()
                elected_master_count = len({f"{c}.{s}.{t}" for (c, s, t, _, _, _) in master_details}) if master_details else 0
                elected_tx_count = len({f"{c}.{s}.{t}" for (c, s, t, _, _, _) in transactional_details}) if transactional_details else 0
                elected_total = elected_master_count + elected_tx_count
                log_print(f"✅ Business tables (discovered): {len(business_tables)}")
                log_print(f"   📊 Master Data tables: {len(master_tables_set)}")
                log_print(f"   📈 Transactional tables: {len(transactional_tables_set)}")
                log_print(f"🗳️ Elected for generation: {elected_total} (master={elected_master_count}, transactional={elected_tx_count}) [{self.table_election_mode}]")
                log_print(f"🟡 Reference tables (excluded): {len(reference_tables_set)}")
                log_print(f"❌ Technical tables (excluded): {len(technical_tables)}")
                log_print(f"{'='*80}\n")

                # Update honesty tracking
                self.processing_honesty['total_tables_discovered'] = len(business_tables) + len(technical_tables) + len(reference_tables_set)
                self.processing_honesty['total_tables_processed'] = elected_total
                self.processing_honesty['total_batches_created'] = len(batches_to_process)

                # Process each batch once (deduplication handles redundancy)
                # MEMORY OPTIMIZATION: Use file-based storage instead of keeping everything in memory
                # ADAPTIVE PARALLELISM: Calculate based on batches, tables and columns
                total_batch_columns = sum(len(cols) for _, cols in batches_to_process)
                avg_prompt_chars = total_batch_columns * 100  # Estimate ~100 chars per column
                
                batch_parallelism, reason = calculate_adaptive_parallelism(
                    "use_case_generation", self.max_parallelism,
                    num_items=len(batches_to_process),
                    total_columns=total_batch_columns,
                    avg_prompt_chars=avg_prompt_chars,
                    is_llm_operation=True, logger=self.logger,
                    estimated_mb_per_worker=10.0
                )
                
                log_print(f"\n{'='*80}")
                log_print(f"🔄 USE CASE GENERATION: SERIAL ENSEMBLE (2 PASSES)")
                log_print(f"{'='*80}")
                log_print(f"📋 PASS 1: Generate initial use cases from {len(batches_to_process)} batch(es)")
                log_print(f"📋 PASS 2: Generate NEW use cases not in PASS 1 (with feedback)")
                log_print("💾 Using file-based intermediate storage to prevent memory explosion")
                log_print(f"{'='*80}\n")
                
                self.storage_manager.initialize()
                
                # === PASS 1: Generate initial use cases (parallel within pass) ===
                self.logger.info("🔄 PASS 1: Generating initial use cases...")
                log_print(f"\n{'='*60}")
                log_print(f"🔄 PASS 1: Initial Use Case Generation")
                log_print(f"{'='*60}")
                
                _bp_timeout = TECHNICAL_CONTEXT["runtime"]["batch_processing_timeout"]
                total_submissions = len(batches_to_process)
                _cascade_depth = max(1, len(TECHNICAL_CONTEXT.get("models") or []))
                _worst_case_per_batch = _bp_timeout * _cascade_depth
                total_timeout = max(_worst_case_per_batch, (total_submissions * _worst_case_per_batch) // max(1, self.max_parallelism)) + 600
                self.logger.info(f"[G31 CASCADE_TIMEOUT_COORDINATION FIRED] wrapper_total={total_timeout}s bp_timeout={_bp_timeout}s cascade_depth={_cascade_depth} batches={total_submissions} parallelism={self.max_parallelism} worst_case_per_batch={_worst_case_per_batch}s")
                
                _p1_max_attempts = TECHNICAL_CONTEXT["runtime"]["batch_processing_max_attempts"]
                
                _pass1_temp = TECHNICAL_CONTEXT["runtime"]["pass1_temperature"]
                _pass2_temp = TECHNICAL_CONTEXT["runtime"]["pass2_temperature"]
                self.logger.info(f"🌡️ Temperature config: Pass 1={_pass1_temp}, Pass 2={_pass2_temp}")
                log_print(f"🌡️ LLM Temperature: Pass 1={_pass1_temp} (focused), Pass 2={_pass2_temp} (table-grounded gap fill)")

                def _pass1_worker(batch_num_inner, column_details_inner):
                    unique_batch_id = f"P1_{batch_num_inner}"
                    _n_cols = len(column_details_inner) if column_details_inner else 0
                    _n_tables = len({(c[0], c[1], c[2]) for c in column_details_inner}) if column_details_inner else 0
                    import time as _t
                    _t_start = _t.time()
                    try:
                        self._emit_pipeline_status(
                            stage_name="Use Cases Generation",
                            step_name=f"UC Generation Batch {batch_num_inner}",
                            sub_step_name="Started",
                            status="started",
                            message=f"Pass 1 batch {batch_num_inner}: {_n_tables} tables / {_n_cols} columns -> LLM"
                        )
                    except Exception:
                        pass
                    use_cases = self._process_batch_with_retry(
                        column_details_inner,
                        unique_batch_id,
                        business_context=ctx_business_context,
                        strategic_initiative=ctx_strategic_initiative,
                        value_chain=ctx_value_chain,
                        revenue_model=ctx_revenue_model,
                        max_attempts=_p1_max_attempts,
                        previous_use_cases_feedback="",
                        temperature_override=_pass1_temp,
                        industry=ctx_industry
                    )
                    if use_cases:
                        self.storage_manager.save_batch(unique_batch_id, use_cases)
                    try:
                        _elapsed = _t.time() - _t_start
                        self._emit_pipeline_status(
                            stage_name="Use Cases Generation",
                            step_name=f"UC Generation Batch {batch_num_inner}",
                            sub_step_name="Complete",
                            status="ended_success" if use_cases else "ended_warning",
                            message=f"Pass 1 batch {batch_num_inner}: emitted {len(use_cases) if use_cases else 0} UCs in {_elapsed:.1f}s",
                            result_json={"batch_num": batch_num_inner, "n_tables": _n_tables,
                                         "n_columns": _n_cols, "n_use_cases": len(use_cases) if use_cases else 0,
                                         "elapsed_sec": round(_elapsed, 1)}
                        )
                    except Exception:
                        pass
                    return (unique_batch_id, use_cases)
                
                tasks = [(_pass1_worker, (bn, cd)) for bn, cd in batches_to_process]
                for bn, _ in batches_to_process:
                    self.logger.info(f"✓ [PASS 1] Submitted batch {bn}")
                
                pass1_results = ParallelExecutor.execute_parallel(
                    tasks=tasks,
                    max_workers=batch_parallelism,
                    task_name="Pass1Batch",
                    logger=self.logger,
                    timeout_per_task=_worst_case_per_batch,
                    total_timeout=total_timeout,
                    thread_name_prefix="Pass1Batch",
                    return_exceptions=True
                )
                
                batches_completed = 0
                for result in pass1_results:
                    if isinstance(result, Exception):
                        self.logger.error(f"❌ [PASS 1] Batch failed: {get_clean_error_message(result)}")
                        continue
                    unique_batch_id, use_cases = result
                    if use_cases:
                        batches_completed += 1
                        self.logger.info(f"✓ [PASS 1] Batch {unique_batch_id}: {len(use_cases)} use cases ({batches_completed}/{total_submissions})")
                        log_print(f"✓ [PASS 1] Batch complete ({batches_completed}/{total_submissions})")
                
                # === MEMORY OPTIMIZATION: Save PASS 1 results to disk immediately ===
                # Count use cases and save IDs without loading all into memory
                pass1_count = self.storage_manager.get_total_count()
                self.logger.info(f"✅ PASS 1 complete: Generated {pass1_count} use cases")
                log_print(f"✅ PASS 1 complete: {pass1_count} use cases generated")
                
                # Save PASS 1 IDs to disk for later comparison (memory-efficient)
                pass1_ids = []
                for batch in self.storage_manager.iter_batches():
                    for uc in batch:
                        pass1_ids.append(uc.get('No', ''))
                self.storage_manager.save_pass1_ids(pass1_ids)
                del pass1_ids  # Free memory immediately
                
                # Force garbage collection after PASS 1
                gc.collect()
                self.logger.debug("🧹 Memory cleanup after PASS 1")
                
                # === PASS 2: ALWAYS run to find gap use cases with higher temperature (creative diversity pass) ===
                transactional_batches = []
                seen_tx_fqtns = set()
                unique_tx_columns_by_fqtn = {}
                for _, column_details in batches_to_process:
                    for col in column_details:
                        fqtn = f"{col[0]}.{col[1]}.{col[2]}"
                        if fqtn not in seen_tx_fqtns:
                            if fqtn not in unique_tx_columns_by_fqtn:
                                unique_tx_columns_by_fqtn[fqtn] = []
                            unique_tx_columns_by_fqtn[fqtn].append(col)
                    for fqtn in unique_tx_columns_by_fqtn:
                        seen_tx_fqtns.add(fqtn)
                p2_batch_num = 1
                _p2_categories = getattr(self, "data_category_map", None) or {}
                _p2_status = getattr(self, "table_classification_status", None) or {}
                _p2_context_only = []
                for fqtn, tx_cols in unique_tx_columns_by_fqtn.items():
                    if _p2_categories.get(fqtn) != 'TRANSACTIONAL' and _p2_status.get(fqtn) != 'unverified':
                        _p2_context_only.append(fqtn)
                        continue
                    augmented_tx = self._augment_columns_with_related_tables(tx_cols)
                    transactional_batches.append((p2_batch_num, augmented_tx))
                    p2_batch_num += 1
                if _p2_context_only:
                    self.logger.info(f"PASS 2 batching: {len(_p2_context_only)} table(s) record no transactional events, so they "
                                     f"seed no Pass 2 batch and stay join context: {_p2_context_only[:10]}"
                                     f"{' ...' if len(_p2_context_only) > 10 else ''}")
                if len(transactional_batches) > 0:
                    self.logger.info(
                        f"PASS 2 batching: {len(unique_tx_columns_by_fqtn)} unique tables "
                        f"→ {len(transactional_batches)} batches (diversity pass with temp={_pass2_temp})"
                    )
                del seen_tx_fqtns, unique_tx_columns_by_fqtn
                
                if pass1_count > 0 and transactional_batches:
                    self.logger.info("🔄 PASS 2: Generating NEW use cases from TRANSACTIONAL TABLES (with PASS 1 feedback)...")
                    log_print(f"\n{'='*60}")
                    log_print(f"🔄 PASS 2: Ensemble - TRANSACTIONAL TABLES ONLY")
                    log_print(f"{'='*60}")
                    log_print(f"📋 Feedback: {pass1_count} use cases from PASS 1")
                    log_print(f"📊 Transactional batches: {len(transactional_batches)} (focusing on event/transaction data)")
                    log_print(f"🎯 Goal: Find NEW use cases from transactional data NOT covered in PASS 1")
                    
                    # === MEMORY OPTIMIZATION: Build feedback iteratively and save to disk ===
                    feedback_lines = ["**🚀 PASS 2: YOUR MISSION IS TO FIND EVEN MORE VALUE! 🚀**\n"]
                    feedback_lines.append("Pass 1 generated some use cases, but there is MUCH MORE VALUE to extract from the transactional data!")
                    feedback_lines.append("Your job is to find DIFFERENT, COMPLEMENTARY use cases that Pass 1 missed.\n")
                    feedback_lines.append("**Already generated (reference only - avoid exact duplicates, but EXPLORE VARIATIONS):**")
                    feedback_lines.append("| No | Name | Tables Involved |")
                    feedback_lines.append("|---|---|---|")
                    
                    # Use memory-efficient iterator (doesn't load all at once)
                    feedback_count = 0
                    for idx, name, tables in self.storage_manager.iter_pass1_use_cases_for_feedback(limit=200):
                        feedback_lines.append(f"| {idx} | {name} | {tables} |")
                        feedback_count = idx
                    
                    if pass1_count > 200:
                        feedback_lines.append(f"\n*... and {pass1_count - 200} more use cases (not shown)*")
                    
                    feedback_lines.append("\n**⚠️ ALREADY-CLAIMED CORE BUSINESS PROBLEMS (normalized: ANY use case with these core phrases is a DUPLICATE):**")
                    feedback_lines.append("The following are the core phrases extracted from Pass 1 use cases by stripping the 'with...' activation suffix.")
                    feedback_lines.append("If your new use case addresses the SAME core business problem as ANY of these phrases, it is a DUPLICATE; DO NOT GENERATE IT.")
                    feedback_lines.append("")
                    import re as _re
                    seen_cores = set()
                    for _, name, _ in self.storage_manager.iter_pass1_use_cases_for_feedback(limit=200):
                        core = _re.sub(r'\s+with\s+.*$', '', name, flags=_re.IGNORECASE).strip()
                        if core and core not in seen_cores:
                            seen_cores.add(core)
                    for core_phrase in sorted(seen_cores):
                        feedback_lines.append(f"- {core_phrase}")
                    feedback_lines.append("")
                    
                    feedback_lines.append("\n**🔥 PASS 2 MISSION: FIND HIGH-VALUE USE CASES THAT PASS 1 MISSED 🔥**")
                    feedback_lines.append("")
                    feedback_lines.append("**VALUE-FOCUSED EXPLORATION:**")
                    feedback_lines.append("Pass 1 covered some ground, but transactional data often hides MASSIVE business value.")
                    feedback_lines.append("Your job is to find HIGH-ROI use cases that Pass 1 missed - NOT to generate filler.")
                    feedback_lines.append("")
                    feedback_lines.append("**WHAT TO LOOK FOR (only if they deliver REAL business value):**")
                    feedback_lines.append("- 💰 **Revenue opportunities**: Patterns that could increase sales, reduce churn, optimize pricing")
                    feedback_lines.append("- 🛡️ **Risk signals**: Fraud patterns, compliance risks, operational anomalies worth preventing")
                    feedback_lines.append("- 📊 **Strategic insights**: Cross-table relationships that reveal hidden business drivers")
                    feedback_lines.append("- ⚡ **Efficiency gains**: Process bottlenecks, resource optimization opportunities")
                    feedback_lines.append("")
                    feedback_lines.append("**🚨 QUALITY RULES (EXTREME, ZERO TOLERANCE: violations cause IMMEDIATE rejection) 🚨**")
                    feedback_lines.append("")
                    feedback_lines.append("**5-LAYER DUPLICATE DETECTION (match on ANY layer = DO NOT GENERATE):**")
                    feedback_lines.append("- Layer 1: Normalize verb (Forecast/Predict→PREDICT, Detect/Identify→DETECT, etc.) + strip 'with...' activation suffix. Core phrase match → DUPLICATE")
                    feedback_lines.append("- Layer 2: Also replace noun synonyms (Revenue=Income=Sales, Churn=Attrition, Cost=Expense, Customer=Client, etc.). Match → DUPLICATE")
                    feedback_lines.append("- Layer 3: Extract entity + metric. Same entity + same metric as ANY Pass 1 use case → DUPLICATE")
                    feedback_lines.append("- Layer 4: Technique swap on SAME QUESTION only → DUPLICATE ('Forecast Churn' vs 'Predict Churn' = duplicate; 'Forecast Revenue' vs 'Detect Revenue Anomalies' = NOT duplicate, different questions)")
                    feedback_lines.append("- Layer 5: Same tables + similar business question → DUPLICATE")
                    feedback_lines.append("")
                    feedback_lines.append("**ALSO CHECK the ALREADY-CLAIMED CORE PHRASES list above; if your normalized core matches ANY entry, it is a DUPLICATE.**")
                    feedback_lines.append("")
                    feedback_lines.append("**ADDITIONAL REJECTION CRITERIA:**")
                    feedback_lines.append("- ❌ STRIP TEST: Remove AI functions mentally. If what remains is a WHERE clause, threshold, GROUP BY, or date check → REJECT (trivial)")
                    feedback_lines.append("- ❌ DELIVERABLE TEST: Does your 'with [...]' suffix describe a business DELIVERABLE (recommendation, action plan, queue, strategy)? If it describes a technical method, data source, or analytical technique → REJECT. Also check: does the action part (before 'with') contain any technical jargon, domain acronyms, or statistical terms? If yes → REJECT.")
                    feedback_lines.append("- ❌ Single-table use cases when FK relationships exist → REJECT unless there's a genuinely unique single-table insight")
                    feedback_lines.append("- ✅ Genuinely DIFFERENT business entities and metrics using DIFFERENT data dimensions ARE encouraged")
                    feedback_lines.append("- ✅ Cross-table joins with 2+ tables unlock the highest value; prioritize these")
                    feedback_lines.append("- ✅ Use UNDER-REPRESENTED techniques (check Pass 1 technique distribution and balance)")
                    feedback_lines.append("")
                    feedback_lines.append("**SELF-CHECK before EACH use case**: (1) 'Would a skeptical CFO fund this?' (2) 'Does this pass ALL 5 duplicate layers against ALL Pass 1 use cases?' (3) 'Does this REQUIRE genuine multi-step analysis, ML modeling, or advanced statistical reasoning, or is it just basic filtering/aggregation?' (4) 'Does my activation suffix describe a business DELIVERABLE (recommendation, queue, strategy, action plan) rather than a technical method?' (5) 'Does my action part contain ANY technical jargon, domain acronyms, or statistical terms? If yes, rewrite in plain business language.' (6) 'Have I chosen the BEST algorithm for this problem: SQL, ML model, AI function, or hybrid?'")
                    
                    # Save feedback to disk and clear from memory
                    self.storage_manager.save_feedback_file(feedback_lines)
                    previous_use_cases_feedback = "\n".join(feedback_lines)
                    del feedback_lines  # Free memory
                    gc.collect()
                    
                    pass2_submissions = len(transactional_batches)
                    
                    _p2_max_attempts = TECHNICAL_CONTEXT["runtime"]["batch_processing_max_attempts"]
                    _p2_feedback = previous_use_cases_feedback
                    
                    def _pass2_worker(batch_num_inner, column_details_inner):
                        unique_batch_id = f"P2_{batch_num_inner}"
                        _n_cols2 = len(column_details_inner) if column_details_inner else 0
                        _n_tabs2 = len({(c[0], c[1], c[2]) for c in column_details_inner}) if column_details_inner else 0
                        import time as _t2
                        _t2_start = _t2.time()
                        try:
                            self._emit_pipeline_status(
                                stage_name="Use Cases Generation",
                                step_name=f"UC Generation Batch P2_{batch_num_inner}",
                                sub_step_name="Started", status="started",
                                message=f"Pass 2 batch {batch_num_inner}: {_n_tabs2} tables / {_n_cols2} columns -> LLM"
                            )
                        except Exception:
                            pass
                        use_cases = self._process_batch_with_retry(
                            column_details_inner,
                            unique_batch_id,
                            business_context=ctx_business_context,
                            strategic_initiative=ctx_strategic_initiative,
                            value_chain=ctx_value_chain,
                            revenue_model=ctx_revenue_model,
                            max_attempts=_p2_max_attempts,
                            previous_use_cases_feedback=_p2_feedback,
                            temperature_override=_pass2_temp,
                            industry=ctx_industry
                        )
                        if use_cases:
                            self.storage_manager.save_batch(unique_batch_id, use_cases)
                        return (unique_batch_id, use_cases)
                    
                    pass2_tasks = [(_pass2_worker, (bn, cd)) for bn, cd in transactional_batches]
                    for bn, _ in transactional_batches:
                        self.logger.info(f"✓ [PASS 2] Submitted transactional batch {bn}")
                    
                    pass2_results = ParallelExecutor.execute_parallel(
                        tasks=pass2_tasks,
                        max_workers=batch_parallelism,
                        task_name="Pass2Batch",
                        logger=self.logger,
                        timeout_per_task=_worst_case_per_batch,
                        total_timeout=total_timeout,
                        thread_name_prefix="Pass2Batch",
                        return_exceptions=True
                    )
                    
                    batches_completed = 0
                    for result in pass2_results:
                        if isinstance(result, Exception):
                            self.logger.error(f"❌ [PASS 2] Batch failed: {get_clean_error_message(result)}")
                            continue
                        unique_batch_id, use_cases = result
                        if use_cases:
                            batches_completed += 1
                            self.logger.info(f"✓ [PASS 2] Batch {unique_batch_id}: {len(use_cases)} NEW use cases ({batches_completed}/{pass2_submissions})")
                            log_print(f"✓ [PASS 2] Batch complete ({batches_completed}/{pass2_submissions})")
                    
                    # === MEMORY OPTIMIZATION: Count PASS 2 results without loading all into memory ===
                    total_after_pass2 = self.storage_manager.get_total_count()
                    pass2_new_count = total_after_pass2 - pass1_count
                    self.logger.info(f"✅ PASS 2 complete: Generated {pass2_new_count} additional NEW use cases from transactional tables")
                    log_print(f"✅ PASS 2 complete: {pass2_new_count} NEW use cases generated")
                    
                    # Clean up PASS 2 variables to free memory
                    del previous_use_cases_feedback
                    del transactional_batches
                    gc.collect()
                    self.logger.debug("🧹 Memory cleanup after PASS 2")
                else:
                    self.logger.info(
                        f"⚠️ PASS 2 skipped: PASS 1 produced {pass1_count} use case(s) and "
                        f"{len(transactional_batches)} table batch(es) are available for the ensemble pass"
                    )
                
                log_print(f"\n{'='*60}")
                log_print(f"✅ SERIAL ENSEMBLE COMPLETE")
                log_print(f"{'='*60}")
                
                # Check if any batches were successfully processed
                storage_stats = self.storage_manager.get_stats()
                if storage_stats['num_batches'] == 0:
                    self.logger.warning("No use cases were generated from any batch. Skipping all generation.")
                    self.storage_manager.cleanup()
                    self.ai_agent.cleanup_llm_cache()
                    return
                
                self.logger.info(f"📊 Batch processing complete. Storage stats: {storage_stats['num_batches']} batches, "
                               f"{storage_stats['use_case_count']} use cases, {storage_stats['total_size_mb']:.2f} MB on disk")
                
                # MEMORY OPTIMIZATION: Load use cases from disk for deduplication only when needed
                gc.collect()
                try:
                    import psutil, os as _os
                    _proc = psutil.Process(_os.getpid())
                    _mem_mb = _proc.memory_info().rss / (1024 * 1024)
                    self.logger.info(f"📊 MEMORY before loading use cases: {_mem_mb:.0f} MB (RSS)")
                    log_print(f"📊 Memory before dedup: {_mem_mb:.0f} MB")
                except ImportError:
                    import resource
                    _mem_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
                    self.logger.info(f"📊 MEMORY before loading use cases: {_mem_mb:.0f} MB (maxrss)")
                    _proc = None
                
                self.logger.info("Loading use cases from disk for deduplication...")
                all_use_cases = self.storage_manager.load_all_use_cases()
                
                # Filter out use cases without valid tables (before deduplication)
                # Keep use cases that have either:
                # 1. Valid table references (non-empty and not just catalog/schema prefix)
                # 2. Volume paths (for document data use cases)
                pre_filter_count = len(all_use_cases)
                all_use_cases = [
                    uc for uc in all_use_cases 
                    if (uc.get('Tables Involved', '').strip() and (
                        uc.get('Tables Involved', '').startswith('/Volumes') or 
                        '.' in uc.get('Tables Involved', '')  # Has at least one dot (catalog.schema.table format)
                    ))
                ]
                filtered_count = pre_filter_count - len(all_use_cases)
                if filtered_count > 0:
                    self.logger.warning(f"⚠️ Filtered out {filtered_count} use cases without valid tables before deduplication")
                self._log_funnel_stage("table_reference_filter", pre_filter_count, len(all_use_cases), {"no_valid_table_reference": filtered_count})

                self.logger.debug(f"Total use cases generated (pre-deduplication): {len(all_use_cases)}")
                unique_use_cases = all_use_cases

                # === GOAL ALIGNMENT HARD FILTER ===
                # Only triggers when the interpreter extracted a user goal (has_goal=True)
                # Removes use cases that do not align with the user's stated goal
                pre_goal_filter_count = len(unique_use_cases)
                unique_use_cases = self._filter_use_cases_by_goal_alignment(unique_use_cases, stage="goal_alignment")
                if len(unique_use_cases) < pre_goal_filter_count:
                    self.logger.info(f"🎯 Goal filter removed {pre_goal_filter_count - len(unique_use_cases)} use cases "
                                     f"({len(unique_use_cases)} remaining)")

                
                all_columns_for_sql = []
                _seen_detail_keys = set()
                def _dedupe_add(_detail, _bucket):
                    try:
                        (_c, _s, _t, _col, _, _) = _detail
                    except Exception:
                        return
                    _key = (str(_c or ""), str(_s or ""), str(_t or ""), str(_col or ""))
                    if _key in _seen_detail_keys:
                        return
                    _seen_detail_keys.add(_key)
                    _bucket.append(_detail)
                for batch_num, batch_columns in batches_to_process:
                    for detail in batch_columns:
                        # Keep REFERENCE tables available for joins so downstream UCs
                        # may cite them (e.g., nation, region, category, calendar dims)
                        # without triggering the hallucinated-table check. REFERENCE
                        # tables remain flagged via reference_tables_set for classification.
                        _dedupe_add(detail, all_columns_for_sql)
                # Per north-star rule: classified-out tables must remain joinable and
                # visible in downstream grounding. Merge TECHNICAL and REFERENCE details
                # so quality validators see the full discovered universe instead of
                # flagging legit joins (ship_mode, reason, household_demographics,
                # income_band) as HALLUCINATION.
                try:
                    for _detail in (technical_details or []):
                        _dedupe_add(_detail, all_columns_for_sql)
                    _ref_details_for_merge = getattr(self, "_reference_details_last", None) or []
                    for _detail in _ref_details_for_merge:
                        _dedupe_add(_detail, all_columns_for_sql)
                    try:
                        self.logger.info(
                            f"📘 Schema-for-validation: {len(all_columns_for_sql)} columns "
                            f"({len(_seen_detail_keys)} unique; business + "
                            f"{len(technical_details or [])} technical + "
                            f"{len(_ref_details_for_merge)} reference merged)"
                        )
                    except Exception:
                        pass
                except Exception as _tech_err:
                    try:
                        self.logger.debug(f"Could not merge technical/reference details into all_columns_for_sql: {_tech_err}")
                    except Exception:
                        pass
                
                # === INFER FK RELATIONSHIPS FROM COLUMN NAMING PATTERNS ===
                # This adds relationships like customer_id -> customers.id when no explicit FK exists
                if self.data_loader and all_columns_for_sql:
                    self.logger.info("🔗 Analyzing column names to infer PK-FK relationships...")
                    inferred_count = self.data_loader.merge_inferred_foreign_keys(all_columns_for_sql)
                    if inferred_count > 0:
                        log_print(f"🔗 Inferred {inferred_count} table relationships from column naming patterns")
                
                # === MEMORY OPTIMIZATION: Save schema to disk, clear from memory ===
                self.storage_manager.save_schema_details(all_columns_for_sql, "all_columns_for_sql")
                del batches_to_process
                gc.collect()
                self.logger.debug("🧹 Cleared batches_to_process from memory")
                
                # === NEW: Catch uncovered tables BEFORE clustering/scoring/SQL ===
                self.logger.info("🔍 Checking table coverage before clustering and scoring...")
                _pre_catchall_count = len(unique_use_cases)
                catchall_rounds = 3
                for round_idx in range(catchall_rounds):
                    pre_scoring_retry = self._retry_missing_table_coverage(
                        unique_use_cases,
                        all_columns_for_sql,
                        include_business_catchall=True
                    )
                    if not pre_scoring_retry:
                        if round_idx == 0:
                            self.logger.info("✅ All tables covered by initial use cases")
                        else:
                            self.logger.info(f"✅ All tables covered after catch-all pass {round_idx}")
                        break
                    self.logger.info(f"✅ Generated {len(pre_scoring_retry)} additional use cases for uncovered tables (pass {round_idx + 1})")
                    pre_scoring_retry = [
                        uc for uc in pre_scoring_retry
                        if uc.get('Tables Involved', '').strip() and not uc.get('Tables Involved', '').startswith('/Volumes')
                    ]
                    self.logger.info(f"✅ After filtering, {len(pre_scoring_retry)} pre-scoring retry use cases have valid tables")
                    if not pre_scoring_retry:
                        continue
                    unique_use_cases.extend(pre_scoring_retry)
                    self.logger.info(f"📊 Total use cases after pre-scoring coverage pass {round_idx + 1}: {len(unique_use_cases)}")
                else:
                    self.logger.info("⚠️ Catch-all reached maximum retries")
                self._log_funnel_stage("coverage_catchall", _pre_catchall_count, len(unique_use_cases))

                del all_columns_for_sql
                gc.collect()
                self.logger.debug("🧹 Freed all_columns_for_sql from memory (saved to disk)")
                
                # === PHASE 1: Cluster domains/subdomains FIRST (before scoring) ===
                self.logger.info(f"📊 Clustering {len(unique_use_cases)} deduplicated use cases into domains and subdomains...")

                # === GOAL ALIGNMENT FILTER (SECOND PASS) ===
                # Re-run after table coverage retry to filter any newly-generated non-aligned use cases
                _pre_2nd_goal = len(unique_use_cases)
                unique_use_cases = self._filter_use_cases_by_goal_alignment(unique_use_cases, stage="goal_alignment_post_catchall")
                if len(unique_use_cases) < _pre_2nd_goal:
                    self.logger.info(f"🎯 Goal filter (2nd pass) removed {_pre_2nd_goal - len(unique_use_cases)} additional use cases")

                _pre_cluster_count = len(unique_use_cases)
                if self._is_ultra_high_mode():
                    self.logger.info("[ELITE] Ultra High: early domain clustering skipped; only the final elite set is clustered")
                    clustered_use_cases = unique_use_cases
                else:
                    clustered_use_cases = self._cluster_domains_and_subdomains(unique_use_cases, "English")
                    self._log_funnel_stage("domain_clustering", _pre_cluster_count, len(clustered_use_cases or []))
                
                del unique_use_cases
                del all_use_cases
                gc.collect()
                if _proc:
                    _mem_mb = _proc.memory_info().rss / (1024 * 1024)
                    self.logger.info(f"📊 MEMORY after clustering: {_mem_mb:.0f} MB (RSS)")
                
                # === PHASE 3: Intelligent Deduplication (Global first, per-domain fallback) ===
                self.logger.info("🔄 Starting INTELLIGENT deduplication (global attempt → per-domain fallback)...")
                final_consolidated_use_cases = self._deduplicate_use_cases(clustered_use_cases)
                self._log_funnel_stage("llm_dedup", len(clustered_use_cases or []), len(final_consolidated_use_cases or []),
                                       {"llm_duplicate": len(clustered_use_cases or []) - len(final_consolidated_use_cases or [])})
                
                del clustered_use_cases
                _force_gc(self.logger, "after dedup")
                
                final_consolidated_use_cases = self._assign_provisional_ids(final_consolidated_use_cases)
                self.logger.debug(f"Assigned provisional IDs to {len(final_consolidated_use_cases)} use cases; "
                                  f"the final rank renumbers the selected set")

                # === PHASE G + Q GATES: quality checks, grounding, canonical-sig dedup ===
                try:
                    self._emit_pipeline_status(
                        stage_name="Quality Gates", step_name="G+Q Integrity Checks",
                        sub_step_name="Started", status="started",
                        message="Running Phase G quality gates + Phase Q integrity gates (grounding, dedup, relevance, recall)"
                    )
                except Exception:
                    pass
                try:
                    _q_schema = self.storage_manager.load_schema_details("all_columns_for_sql") or []
                    if final_consolidated_use_cases:
                        if not _q_schema:
                            self.logger.error(
                                f"Phase G+Q gates run without column evidence: the schema saved for validation "
                                f"('all_columns_for_sql') is empty, so column-grounded checks cannot pass any of the "
                                f"{len(final_consolidated_use_cases)} use case(s) and Q1 grounding fails closed when no "
                                f"table universe is known"
                            )
                        before_n = len(final_consolidated_use_cases)
                        # Phase G first (cheap deterministic filters)
                        final_consolidated_use_cases = self._phaseG_run_quality_gates(
                            final_consolidated_use_cases, _q_schema
                        )
                        self._log_funnel_stage("phase_g_gates", before_n, len(final_consolidated_use_cases),
                                               self._gate_ledger_drops("G"))
                        # Phase Q (grounding + dedup) over the G-filtered portfolio
                        _post_g_n = len(final_consolidated_use_cases)
                        final_consolidated_use_cases = self._phaseQ_run_integrity_gates(
                            final_consolidated_use_cases, _q_schema
                        )
                        self._log_funnel_stage("phase_q_gates", _post_g_n, len(final_consolidated_use_cases),
                                               self._gate_ledger_drops("Q"))
                        self.logger.info(f"G+Q gates complete: {before_n} -> {len(final_consolidated_use_cases)} use cases")
                        try:
                            self._emit_pipeline_status(
                                stage_name="Quality Gates", step_name="G+Q Integrity Checks",
                                sub_step_name="Complete", status="ended_success",
                                message=f"Quality gates complete: {before_n} -> {len(final_consolidated_use_cases)} use cases kept",
                                result_json={"before": before_n, "after": len(final_consolidated_use_cases)}
                            )
                        except Exception:
                            pass
                        del _q_schema
                        import gc as _gc
                        _gc.collect()
                except Exception as _gq_err:
                    self.logger.warning(f"Phase G+Q gates failed non-fatally: {get_clean_error_message(_gq_err)}")

                # === PHASE 4.5: VALUE+QUALITY SCORING ===
                final_consolidated_use_cases = self._apply_feedback_exclusions(final_consolidated_use_cases)
                _schema_for_quality = self.storage_manager.load_schema_details("all_columns_for_sql") or []
                _cache_tag = self._scored_cache_tag("combined_scored_", final_consolidated_use_cases, scoring_context, _schema_for_quality)
                _post_gate_ids = [str(uc.get('No', '')) for uc in final_consolidated_use_cases]
                _cached_scored = self.storage_manager.load_scored_use_cases(_cache_tag) if final_consolidated_use_cases else None
                if isinstance(_cached_scored, list) and [str(uc.get('No', '')) for uc in _cached_scored] == _post_gate_ids:
                    self.logger.info(f"Loaded {len(_cached_scored)} scored use cases from disk cache ({_cache_tag})")
                    final_consolidated_use_cases = _cached_scored
                elif final_consolidated_use_cases:
                    final_consolidated_use_cases = self._score_combined_value_and_quality(
                        use_cases=final_consolidated_use_cases,
                        full_schema_details=_schema_for_quality,
                        **self._grading_context(scoring_context)
                    )
                    self.storage_manager.save_scored_use_cases(final_consolidated_use_cases, _cache_tag)
                gc.collect()

                self._apply_inspire_scores(final_consolidated_use_cases)
                final_consolidated_use_cases, _selection_rejects = self._select_scored_use_cases(
                    final_consolidated_use_cases, _schema_for_quality, scoring_context
                )
                del _schema_for_quality
                if not final_consolidated_use_cases:
                    self._report_zero_accepted(_selection_rejects)
                del _selection_rejects

                quality_distribution = {}
                for uc in final_consolidated_use_cases:
                    label = uc.get('Quality') or 'Unscored'
                    quality_distribution[label] = quality_distribution.get(label, 0) + 1
                log_print(f"\n📊 FINAL QUALITY DISTRIBUTION (after selection):")
                for label in self._VALID_QUALITY_LABELS:
                    count = quality_distribution.get(label, 0)
                    if count > 0:
                        log_print(f"   {label}: {count} use cases")
                
            else:
                 raise RuntimeError(self._nothing_to_discover_message())
        except Exception as e:
            self.logger.critical(f"A critical error occurred during English generation: {get_clean_error_message(e)}")
            log_print(f"\n❌ FATAL: {get_clean_error_message(e)}")
            self.storage_manager.cleanup()
            self.ai_agent.cleanup_llm_cache()
            AIAgent.get_summary_report()
            self.ai_agent.get_manager_stats_report()
            raise

        if final_consolidated_use_cases:
            _missing_inspire = [uc for uc in final_consolidated_use_cases if uc.get('Inspire Score') is None]
            if _missing_inspire:
                self.logger.error(
                    f"⚠️ {len(_missing_inspire)}/{len(final_consolidated_use_cases)} selected use case(s) have no Inspire Score; "
                    f"they are excluded (no default score is computed): {[uc.get('No') for uc in _missing_inspire][:20]}"
                )
                _missing_ids = {id(uc) for uc in _missing_inspire}
                final_consolidated_use_cases = [uc for uc in final_consolidated_use_cases if id(uc) not in _missing_ids]

        if final_consolidated_use_cases:
            if self._is_ultra_high_mode():
                final_consolidated_use_cases = self._cluster_elite_final_set(final_consolidated_use_cases)
            try:
                self._mark_best_of_best(final_consolidated_use_cases, scoring_context)
            except SparkWorkCancelled:
                raise
            except Exception as _star_err:
                self.logger.error(f"❌ Best of Best marking failed: {get_clean_error_message(_star_err)}; no use case "
                                  f"is starred")
                self._record_failed_step("Best of Best", "", _star_err)
            try:
                final_consolidated_use_cases = self._run_strategy_alignment(final_consolidated_use_cases)
            except SparkWorkCancelled:
                raise
            except Exception as _align_err:
                self.logger.error(f"❌ Strategy alignment failed: {get_clean_error_message(_align_err)}; the use cases keep "
                                  f"their data-only scores")
                self._record_failed_step("Strategy Alignment", "", _align_err)
            final_consolidated_use_cases = self._finalize_portfolio_order(final_consolidated_use_cases)
            for uc in final_consolidated_use_cases:
                uc.setdefault("use_case_key", use_case_key(uc))
            self._publish_best_of_best(final_consolidated_use_cases)
            try:
                self._build_data_foundations(final_consolidated_use_cases)
            except Exception as _foundation_err:
                self.logger.warning(f"Shared data foundations unavailable: {get_clean_error_message(_foundation_err)}")
            try:
                self._track_implemented_value()
            except SparkWorkCancelled:
                raise
            except Exception as _value_err:
                self.logger.warning(f"Value tracking unavailable: {get_clean_error_message(_value_err)}")

        english_grouped_data = self._group_use_cases_by_domain_flat(final_consolidated_use_cases) if final_consolidated_use_cases else {}
        summary_dict = None

        if final_consolidated_use_cases:
            for uc in final_consolidated_use_cases:
                uc['quality_mode'] = self.use_cases_quality
                uc['Analytics Technique'] = self._canonical_analytics_technique(
                    uc.get('Analytics Technique'), uc.get('No'), "Selected use case")
                tables_involved = uc.get('Tables Involved', '')
                uc['Primary Table'] = self._extract_primary_table(tables_involved)
                if 'result_table' not in uc or not uc.get('result_table'):
                    uc['result_table'] = compute_result_table_name(
                        uc.get('No', 'unknown'), uc.get('Name', 'unknown'),
                        tables_involved, inspire_database=self.inspire_database
                    )
            self.logger.info(f"✅ Pre-computed Primary Table and Result Database for {len(final_consolidated_use_cases)} use cases")
            english_grouped_data = self._group_use_cases_by_domain_flat(final_consolidated_use_cases)
            # Phase L: advisory final-gate run AFTER Primary Table derivation (UC dicts now complete).
            try:
                self._emit_pipeline_status(
                    stage_name="Final Gate", step_name="Phase L Production Checks",
                    sub_step_name="Started", status="started",
                    message="Running Phase L 19-check production-readiness gate"
                )
            except Exception:
                pass
            try:
                _q_schema_L = self.storage_manager.load_schema_details("all_columns_for_sql") or []
                _L_results = self._phaseL_final_gate(final_consolidated_use_cases, _q_schema_L)
                _L_fails = [k for k, v in _L_results.items() if not v.get("pass")]
                self.logger.info(f"Phase L final-gate: {len(_L_results)-len(_L_fails)}/{len(_L_results)} passed; failures={_L_fails}")
                try:
                    self._emit_pipeline_status(
                        stage_name="Final Gate", step_name="Phase L Production Checks",
                        sub_step_name="Complete",
                        status="ended_success" if not _L_fails else "ended_warning",
                        message=f"Phase L: {len(_L_results)-len(_L_fails)}/{len(_L_results)} checks passed",
                        result_json={"passed": len(_L_results)-len(_L_fails), "total": len(_L_results), "failures": _L_fails}
                    )
                except Exception:
                    pass
                del _q_schema_L
            except Exception as _L_err:
                self.logger.warning(f"Phase L final-gate failed non-fatally: {get_clean_error_message(_L_err)}")
                try:
                    self._emit_pipeline_status(
                        stage_name="Final Gate", step_name="Phase L Production Checks",
                        sub_step_name="Error", status="ended_warning",
                        message=f"Phase L failed non-fatally: {get_clean_error_message(_L_err)}"
                    )
                except Exception:
                    pass

        if final_consolidated_use_cases:
            english_grouped_data = self._group_use_cases_by_domain_flat(final_consolidated_use_cases)

        if final_consolidated_use_cases:
            self.logger.info("Generating English CSV...")
            lang_abbr_en = self._get_lang_abbr("English")
            try:
                self._generate_csv_catalog("English", lang_abbr_en, english_grouped_data)
            except Exception as e:
                self.logger.error(f"Failed to generate English CSV: {get_clean_error_message(e)}")
            try:
                self._tracking_replace_session(final_consolidated_use_cases)
            except Exception as _track_err:
                self.logger.error(f"⚠️ Tracking table UPDATE 1/2 (post-CSV) failed (non-fatal): {_track_err}")

        
        # === PHASE 2: GENERATE SUMMARY & DOCUMENTATION ===
        if final_consolidated_use_cases and not self.json_file_path:
            if summary_dict is None:
                self.logger.info("Generating executive summaries...")
                (summary_dict, _) = self._get_salesy_summary(english_grouped_data, self.business_name, "English", english_translations)
        
        # === START DOCUMENTATION GENERATION ===
        doc_generation_future = None
        doc_generation_executor = None
        remaining_langs = [lang for lang in self.output_languages if lang != "English"]
        target_langs = ["English"] + remaining_langs if "English" in self.output_languages else remaining_langs
        
        if final_consolidated_use_cases and target_langs and ("PDF Catalog" in self.generate_choices or "Presentation" in self.generate_choices or "Use Cases Catalog PDF" in self.generate_choices or "English" in target_langs):
            self.logger.info("🚀 Starting PDF/PPTX documentation generation...")
            log_print(f"📄 Documentation generation starting (languages: {', '.join(target_langs)})")
            doc_generation_executor = _SafeExecutorContext(max_workers=1, thread_name_prefix="DocGen", logger=self.logger, name="DocGen")
            doc_generation_future = doc_generation_executor.submit(
                self._generate_documents_for_all_languages,
                final_consolidated_use_cases,
                english_grouped_data,
                summary_dict,
                target_langs,
                ["English"]  # skip_excel_langs - already generated
            )
        
        _force_gc(self.logger, "pre-Genie-generation phase cleanup")
        
        # === PHASE 2: GENIE CODE INSTRUCTIONS GENERATION ===
        if self.generate_genie_instructions and final_consolidated_use_cases:
            self.logger.info(f"🤖 PHASE 2: Genie Code instructions generation...")
            _schema_for_genie = self.storage_manager.load_schema_details("all_columns_for_sql")
            try:
                final_consolidated_use_cases = self._generate_genie_instructions_and_notebooks(
                    final_consolidated_use_cases,
                    _schema_for_genie,
                    document_context_markdown,
                    english_translations,
                    summary_dict
                )
            except Exception as e:
                self.logger.error(f"❌ Genie Code instruction generation failed: {get_clean_error_message(e)}")
                log_print(f"⚠️ Genie Code instruction generation encountered an error: {str(e)[:200]}", level="WARNING")
                log_print(f"ℹ️ Proceeding with remaining artifacts...")
            finally:
                del _schema_for_genie
            _force_gc(self.logger, "after Genie Code generation phase")
            self.logger.info("✅ Phase 2 complete: Genie Code instructions generated")
        elif (not self.generate_genie_instructions) and final_consolidated_use_cases and getattr(self, 'operation', '') == 'Discover Use Cases':
            self.logger.info(f"🔎 PHASE 2 (Discover mode): Capturing per-UC Genie prompt fields + generating skeleton notebooks...")
            _schema_for_genie = self.storage_manager.load_schema_details("all_columns_for_sql")
            try:
                final_consolidated_use_cases = self._run_discover_skeleton_phase(
                    final_consolidated_use_cases,
                    _schema_for_genie
                )
            except Exception as e:
                self.logger.error(f"❌ Discover skeleton phase failed: {get_clean_error_message(e)}")
                log_print(f"⚠️ Discover skeleton generation encountered an error: {str(e)[:200]}", level="WARNING")
            finally:
                del _schema_for_genie
            _force_gc(self.logger, "after Discover skeleton phase")
            self.logger.info("✅ Phase 2 complete (Discover): skeleton notebooks generated, prompt fields captured")
        
        if final_consolidated_use_cases and self._tracking_table_name:
            try:
                self._tracking_replace_session(final_consolidated_use_cases)
                self.logger.info("✅ [UPDATE 2/2] Tracking table re-synced post-Phase-2 (notebook_path, genie_instruction, discover fields)")
            except Exception as _track_err2:
                self.logger.error(f"⚠️ Tracking table UPDATE 2/2 (post-Phase-2) failed (non-fatal): {_track_err2}")
        
        # === GENERATE USE CASE NOTEBOOKS (batch fallback — only if NOT already streamed inline during genie phase) ===
        _notebooks_done_inline = getattr(self, '_notebooks_generated_inline', False)
        if final_consolidated_use_cases and not self.json_file_path and not _notebooks_done_inline:
            self.logger.info("📓 Generating per-use-case notebooks (batch fallback — no genie streaming)...")
            log_print(f"\n📓 Generating per-use-case notebooks for {len(final_consolidated_use_cases)} use cases...")
            try:
                self.assemble_use_case_notebooks(final_consolidated_use_cases, english_translations, summary_dict)
            except Exception as e:
                self.logger.error(f"❌ Notebook generation failed: {get_clean_error_message(e)}")
                log_print(f"⚠️ Notebook generation failed: {str(e)[:200]}", level="WARNING")
        elif _notebooks_done_inline:
            self.logger.info("📓 Notebooks already generated inline during genie streaming phase — skipping batch generation")

        if final_consolidated_use_cases:
            self.logger.info("Generating English Excel (post-notebook, with hyperlinks)...")
            lang_abbr_en = self._get_lang_abbr("English")
            english_grouped_data = self._group_use_cases_by_domain_flat(final_consolidated_use_cases)
            try:
                self._generate_use_case_excel("English", lang_abbr_en, english_grouped_data)
            except Exception as e:
                self.logger.error(f"Failed to generate English Excel: {get_clean_error_message(e)}")
            try:
                self._generate_csv_catalog("English", lang_abbr_en, english_grouped_data)
            except Exception as e:
                self.logger.error(f"Failed to regenerate English CSV (post-Auto-Genie): {get_clean_error_message(e)}")

        # === WAIT FOR DOCUMENTATION GENERATION TO COMPLETE ===
        _parallel_doc_gen_was_started = doc_generation_future is not None
        if doc_generation_future:
            try:
                self.logger.info("⏳ Waiting for documentation generation to complete...")
                _safe_future_result(doc_generation_future, timeout=1800)
                self.logger.info("✅ Documentation generation completed")
                log_print("✅ Documentation generation completed")
                try:
                    # Phase O persist removed per user request — next_actions.jsonl,
                    # manifest.json, NEXT_ACTIONS_REPORT.md no longer written.
                    pass
                    try:
                        self._emit_pipeline_status(
                            stage_name="Artifact Writing", step_name="Phase O Run-Artifact Persist",
                            sub_step_name="Complete", status="ended_success",
                            message="Run artifacts persisted"
                        )
                    except Exception:
                        pass
                except Exception as _o_err:
                    self.logger.warning(f"Phase O persistence failed non-fatally: {get_clean_error_message(_o_err)}")
            except concurrent.futures.TimeoutError:
                doc_generation_executor.mark_timed_out()
                self.logger.warning("⚠️ Documentation generation timed out after 30 minutes")
                log_print("⚠️ Documentation generation timed out", level="WARNING")
            except Exception as e:
                self.logger.error(f"❌ Documentation generation failed: {get_clean_error_message(e)}")
                log_print(f"❌ Documentation generation failed: {str(e)[:100]}", level="ERROR")
            finally:
                if doc_generation_executor:
                    doc_generation_executor.shutdown()
                del doc_generation_future
                doc_generation_future = None
        _force_gc(self.logger, "after documentation generation")

        # === SAVE JSON CATALOG (POST-SQL) ===
        if final_consolidated_use_cases and not self.json_file_path:
            self.logger.info("Saving JSON Catalog...")
            summary_dict = self._save_usecases_catalog_json(final_consolidated_use_cases, english_translations, summary_dict)
        
        # Note: PDF/PPTX documentation generation was started in parallel earlier
        # Only generate here if parallel generation was NOT started (no PDF/Presentation selected initially)
        if final_consolidated_use_cases and not _parallel_doc_gen_was_started:
            remaining_langs = [lang for lang in self.output_languages if lang != "English"]
            target_langs = ["English"] + remaining_langs if "English" in self.output_languages else remaining_langs
            if target_langs and ("PDF Catalog" in self.generate_choices or "Presentation" in self.generate_choices or "Use Cases Catalog PDF" in self.generate_choices or "English" in target_langs):
                if summary_dict is None:
                    (summary_dict, _) = self._get_salesy_summary(english_grouped_data, self.business_name, "English", english_translations)
                self.logger.info("Generating PDFs/Presentations and translations...")
                self._generate_documents_for_all_languages(
                    final_consolidated_use_cases,
                    english_grouped_data=english_grouped_data,
                    summary_dict=summary_dict,
                    languages=target_langs,
                    skip_excel_langs=["English"]
                )
        del english_grouped_data
        _force_gc(self.logger, "freed english_grouped_data after all doc gen")
        
        # Report table inclusion/exclusion statistics
        if final_consolidated_use_cases and self.data_loader:
            self._report_table_statistics(final_consolidated_use_cases)
        
        self.storage_manager.cleanup()
        self.ai_agent.cleanup_llm_cache()
        self._publish_funnel_ledger()
        
        self._report_processing_honesty()
        self._report_run_completion(
            f"✅ All Use cases for {self.business_name} generated successfully ({len(final_consolidated_use_cases)} use cases)",
            f"⚠️ COMPLETED WITH WARNINGS: use cases for {self.business_name} generated ({len(final_consolidated_use_cases)} use cases)")
        AIAgent.get_summary_report()
        self.ai_agent.get_manager_stats_report()
        self.logger.info("Uploading log file...")
        self._upload_log_file()
            

    def _refresh_english_summary(self, english_grouped_data: dict, stored: dict) -> dict:
        if not english_grouped_data:
            return stored
        fresh, _name = self._get_salesy_summary(english_grouped_data, self.business_name, "English",
                                                self.translation_service.get_translations("English"))
        if any(key != "Executive" and str(value or "").strip() for key, value in (fresh or {}).items()):
            self.logger.info("[DOCS-SUMMARY refreshed] English executive and domain narratives regenerated for the loaded catalog.")
            return fresh
        self.logger.warning("[DOCS-SUMMARY stored] English narrative regeneration fell back; the narratives stored in the catalog are used.")
        return stored

    def _get_salesy_summary(self, grouped_data: dict, business_name: str, language: str, translations: dict) -> tuple:
        self.logger.debug(f"Calling LLM for executive and domain summaries in {language}...")
        t = translations
        summary_dict = {}
        transliterated_name = business_name # Default
        try:
            domain_list_for_prompt = "\n".join([f"- {domain}" for domain in grouped_data.keys()])

            highlights_parts = []
            for domain, cases in grouped_data.items():
                sorted_cases = sorted(cases, key=self._final_rank_key)
                top_cases = sorted_cases[:3]
                case_bullets = []
                for c in top_cases:
                    name = self._doc_use_case_title(c, language)
                    bv = str(c.get('Business Value') or '')
                    bv_snippet = (bv[:120] + '...') if len(bv) > 120 else bv
                    case_bullets.append(f"  - {name}: {bv_snippet}" if bv_snippet else f"  - {name}")
                highlights_parts.append(f"**{domain}**:\n" + "\n".join(case_bullets))
            domain_use_case_highlights = "\n\n".join(highlights_parts)
            
            prompt_vars = {
                "business_name": business_name,
                "domain_list": domain_list_for_prompt, "output_language": language,
                "domain_use_case_highlights": domain_use_case_highlights,
                "generation_instructions_section": self._get_generation_instruction_for_prompt("SUMMARY_GEN_PROMPT"),
            }
            summary_csv_raw = self.ai_agent.run_worker(
                step_name=f"Executive_Summary_{language}",
                worker_prompt_path="SUMMARY_GEN_PROMPT",
                prompt_vars=prompt_vars, response_schema=None
            )
            self.logger.info(f"LLM summaries (CSV) received for {language}.")
            
            # === ROBUST CSV PARSING (Req 2) ===
            # Support both quoted and unquoted headers from LLM
            header_3_col_quoted = '"Type","Summary","TransliteratedBusinessName"'
            header_3_col_unquoted = 'Type,Summary,TransliteratedBusinessName'
            header_2_col_quoted = '"Type","Summary"'
            header_2_col_unquoted = 'Type,Summary'
            
            header_start_index = summary_csv_raw.find(header_3_col_quoted)
            is_3_col = True
            
            if header_start_index == -1:
                header_start_index = summary_csv_raw.find(header_3_col_unquoted)
            
            if header_start_index == -1:
                self.logger.warning(f"Could not find 3-column CSV header in LLM response for {language}. Attempting 2-col parse.")
                is_3_col = False
                header_start_index = summary_csv_raw.find(header_2_col_quoted)
                if header_start_index == -1:
                    header_start_index = summary_csv_raw.find(header_2_col_unquoted)
                if header_start_index == -1:
                    self.logger.error(f"Could not find 2-column or 3-column CSV header. Aborting summary parse. Response: {summary_csv_raw[:200]}")
                    raise ValueError(f"Could not parse summary CSV for {language}")

            self.logger.info(f"Found CSV header at index {header_start_index}. Parsing as {3 if is_3_col else 2}-column.")
            summary_csv_clean = summary_csv_raw[header_start_index:]
            # ==================================

            # Use centralized CSV parser
            csv_rows = CSVParser.parse_csv_list(
                summary_csv_clean,
                logger=self.logger,
                context="Domain summary",
                delimiter=',',
                quotechar='"',
                quoting=csv.QUOTE_ALL,
                skipinitialspace=True
            )
            if csv_rows:
                header = csv_rows[0]  # First row is header
                csv_reader, _ = CSVParser.aligned_rows(
                    csv_rows[1:], self.logger, f"Domain summary {language}", header=[str(cell).strip() for cell in header],
                    required=("Type", "Summary"), anchors={"Type": CSVParser.one_of(["Executive", *grouped_data])},
                    text=summary_csv_clean)
            else:
                csv_reader = []
            
            first_row = csv_reader[0] if csv_reader and csv_reader[0] else []
            if is_3_col:
                for row in csv_reader:
                    # Handle rows that may span multiple lines or have embedded content
                    if len(row) >= 3:
                        row_type = row[0].strip()
                        summary_dict[row_type] = row[1].strip()
                        if row_type == "Executive" and row[2].strip():
                            transliterated_name = row[2].strip()
                    elif len(row) > 0:
                        # Partial row - try to accumulate
                        self.logger.debug(f"Partial 3-col row (len={len(row)}): {row[:1]}")
                        # Skip partial rows gracefully without warning
            else: # 2-col fallback
                for row in csv_reader:
                    if len(row) >= 2:
                        summary_dict[row[0].strip()] = row[1].strip()
                    elif len(row) > 0:
                        self.logger.debug(f"Partial 2-col row (len={len(row)}): {row[:1]}")
            transliterated_name = business_name if not is_3_col else transliterated_name
            raw_summary = {key: str(text) for key, text in summary_dict.items()}
            summary_dict, unmatched = self._canonical_summary(raw_summary, list(grouped_data.keys()))
            first_key = str(first_row[0]).strip() if first_row else ""
            if "Executive" not in summary_dict and first_key in unmatched:
                summary_dict["Executive"] = raw_summary[first_key]
                unmatched.remove(first_key)
                if is_3_col and len(first_row) >= 3 and str(first_row[2]).strip() and transliterated_name == business_name:
                    transliterated_name = str(first_row[2]).strip()
                self.logger.warning(f"[DOCS-SUMMARY executive-recovered] {language}: the executive row was labelled {first_key!r}")
            missing = [domain for domain in grouped_data if not self._domain_summary_paragraphs(summary_dict, domain)]
            self.logger.info(f"[DOCS-SUMMARY] {language}: {len(grouped_data) - len(missing)}/{len(grouped_data)} domain summaries matched"
                             + (f"; unmatched rows {unmatched[:10]}" if unmatched else "")
                             + (f"; missing {missing[:10]}" if missing else ""))
            
            if "Executive" not in summary_dict:
                self.logger.error("Failed to parse 'Executive' summary from LLM response.")
                raise ValueError("Missing 'Executive' summary")
                
            self.logger.info(f"Successfully parsed {len(summary_dict)} summaries for {language}. Transliterated name: {transliterated_name}")
            return summary_dict, transliterated_name
            
        except Exception as e:
            self.logger.error(f"LLM summary generation failed for {language}: {get_clean_error_message(e)}. Using default text.")
            fallback_dict = {}
            total_cases_fallback = sum(len(cases) for cases in grouped_data.values())
            p1 = t['pdf_fallback_summary_p1'].format(total_cases=total_cases_fallback, business_name=business_name)
            p2 = t['pdf_fallback_summary_p2']
            fallback_dict["Executive"] = f"<p>{p1}</p><p>{p2}</p>"
            return fallback_dict, business_name # Return default name

    def _classify_domain_structure(self, domain_usecases: dict, total_use_cases: int) -> tuple:
        hard_violations, soft_warnings, small_domains = [], [], []
        total_domains = len(domain_usecases)
        if total_domains < min(3, total_use_cases):
            hard_violations.append(f"Only {total_domains} domains, minimum required: {min(3, total_use_cases)}")
        if total_domains > 25:
            hard_violations.append(f"🚨 CRITICAL: {total_domains} domains exceeds MAXIMUM 25 (HARD LIMIT) - THIS IS UNACCEPTABLE")
        for domain in domain_usecases.keys():
            word_count = len(str(domain).split())
            if word_count != 1:
                hard_violations.append(f"Domain '{domain}' has {word_count} word(s), must be exactly 1 word")
        protected = self._protected_domains()
        for domain, ucs in domain_usecases.items():
            count = len(ucs)
            if count < 4:
                if str(domain).strip().lower() in protected:
                    continue
                soft_warnings.append(f"Domain '{domain}' has {count} use case(s); small-domain merge will consolidate it")
                small_domains.append((domain, count))
            elif count > 80:
                hard_violations.append(f"Domain '{domain}' has {count} use case(s), maximum allowed: 80")
        return hard_violations, soft_warnings, small_domains

    def _merge_small_domains(self, use_cases: list, min_cases_per_domain: int = 4) -> list:
        """
        Merges business domains that have fewer than min_cases_per_domain use cases.
        Uses adaptive thresholds based on total UC count and enforces anti-mega-domain constraints.
        User-specified Business Domains are immutable: never renamed or merged away, though other
        domains may merge into them.
        """
        if not use_cases:
            return use_cases
        
        total_ucs = len(use_cases)
        
        if total_ucs < 30:
            min_cases_per_domain = 2
        elif total_ucs < 60:
            min_cases_per_domain = 3
        else:
            min_cases_per_domain = 4
        
        max_domain_share = 0.35
        max_ucs_per_domain = max(int(total_ucs * max_domain_share), min_cases_per_domain + 1)
        
        import math
        target_domain_count = max(3, min(round(math.sqrt(total_ucs)), 25))
        protected = self._protected_domains()
        
        self.logger.info(
            f"Domain merge config: {total_ucs} UCs, min_per_domain={min_cases_per_domain}, "
            f"max_per_domain={max_ucs_per_domain} ({max_domain_share:.0%} of total), "
            f"target_domains={target_domain_count}"
            + (f", user-specified (immutable)={sorted(protected.values())}" if protected else "")
        )
        
        domain_counts = defaultdict(int)
        for uc in use_cases:
            domain = uc.get('Business Domain', 'Other')
            domain_counts[domain] += 1
        
        small_domains = {domain: count for domain, count in domain_counts.items()
                         if count < min_cases_per_domain and str(domain).strip().lower() not in protected}
        
        if not small_domains:
            self.logger.info(f"All domains have at least {min_cases_per_domain} use cases (or are user-specified). No merging needed.")
            return use_cases
        
        self.logger.debug(f"Found {len(small_domains)} domains with fewer than {min_cases_per_domain} use cases: {list(small_domains.keys())}")
        
        domain_info_lines = []
        for domain, count in sorted(domain_counts.items(), key=lambda x: x[1], reverse=True):
            pct = count / total_ucs * 100
            if str(domain).strip().lower() in protected:
                status = f"🔒 USER-SPECIFIED ({pct:.0f}% of total — immutable: never rename it or merge it away; small domains MAY merge into it)"
            elif count < min_cases_per_domain:
                status = f"❌ TOO SMALL (needs merging)"
            elif count > max_ucs_per_domain:
                status = f"⚠️ ALREADY LARGE ({pct:.0f}% of total — do NOT merge more into this)"
            else:
                status = f"✓ OK ({pct:.0f}% of total)"
            domain_info_lines.append(f"  - {domain}: {count} use cases {status}")
        
        domain_info_str = self._escape_template_braces("\n".join(domain_info_lines))
        protected_block = ""
        if protected:
            protected_block = self._escape_template_braces(
                "\n**USER-SPECIFIED DOMAINS (SUPREME USER DIRECTIVE — OVERRIDES EVERY RULE BELOW)**: "
                + ", ".join(f'"{d}"' for d in sorted(protected.values()))
                + "\nNever rename these domains and never map them to another domain. Other domains MAY be "
                  "merged INTO them using the exact spelling above.\n"
            )
        
        merge_prompt = f"""You are a business domain expert. Analyze this list of business domains and their use case counts:

Total use cases: {total_ucs}
Target domain count: {target_domain_count}

{domain_info_str}
{protected_block}
**CRITICAL REQUIREMENTS**:
1. Domains with < {min_cases_per_domain} use cases MUST be merged into a semantically related domain
2. **ANTI-MEGA-DOMAIN RULE**: After merging, NO domain should have more than {max_ucs_per_domain} use cases ({max_domain_share:.0%} of total). If merging a small domain into a large one would exceed this limit, find a DIFFERENT target or merge multiple small domains TOGETHER into a new domain.
3. **SEMANTIC RELATEDNESS**: ONLY merge domains that are genuinely related. Do NOT merge semantically unrelated domains just because one is small. Fraud ≠ Engagement. Inventory ≠ Customers. Workforce ≠ Revenue.
4. After merging, you should still have at least {max(3, target_domain_count - 2)} domains. Do NOT over-merge.

**Your Task**:
1. Identify domains marked ❌ (too small)
2. For each, find the most semantically related domain to merge into
3. If no good semantic match exists, merge 2-3 small related domains TOGETHER into a new combined domain
4. NEVER merge a small domain into a domain marked ⚠️ ALREADY LARGE

**Output Format** (honesty-wrapped JSON):
Your response MUST be wrapped: {{{{"honesty_score": XX, "honesty_justification": "...", "data": <your_mapping>}}}}

The "data" field maps old domain names to new domain names.
Only include domains that need renaming/merging.

**Rules**:
1. Every domain name must be EXACTLY ONE WORD (no spaces), unique core word, industry-specific
2. ❌ BANNED generic names: "Operations", "Engagement", "Management", "Services", "Activities", "General"
3. ✅ Use specific business nouns: "Revenue", "Customers", "Inventory", "Fraud", "Workforce", "Pricing", "Franchise", "Logistics"
4. Domains with ≥ {min_cases_per_domain} can stay as-is UNLESS you have a significantly better name

Start your response with: {{{{"honesty_score":
""" + HONESTY_CHECK_JSON
        
        try:
            # Use run_worker with a direct prompt
            merge_prompt_key = "DOMAINS_MERGER_PROMPT"
            _merger_instr = self._get_generation_instruction_for_prompt("DOMAIN_FINDER_PROMPT")
            merge_prompt = f"{merge_prompt}\n\n{self._escape_template_braces(_merger_instr)}\n"
            self.ai_agent.prompt_templates[merge_prompt_key] = merge_prompt
            
            response = self.ai_agent.run_worker(
                step_name=f"Merge_Small_Domains",
                worker_prompt_path=merge_prompt_key,
                prompt_vars={},
                response_schema=None
            )
            
            # Clean up temporary prompt template
            del self.ai_agent.prompt_templates[merge_prompt_key]
            
            cleaned = clean_json_response(response)
            if not cleaned or cleaned.strip() == '':
                self.logger.warning("Domain merge response is empty after cleaning. Skipping merge.")
                return use_cases
            merge_mapping = json.loads(cleaned)
            
            # Extract data from honesty-wrapped response if present
            if isinstance(merge_mapping, dict) and 'data' in merge_mapping:
                merge_mapping = merge_mapping['data']
            
            # Ensure merge_mapping is a dict with string keys/values
            if not isinstance(merge_mapping, dict):
                self.logger.warning(f"Domain merge response is not a dict: {type(merge_mapping)}. Skipping merge.")
                return use_cases
            
            # Sanitize: LLMs sometimes return nested dicts/lists as values instead of plain strings
            sanitized = {}
            for k, v in merge_mapping.items():
                if not (isinstance(k, str) and isinstance(v, str)):
                    self.logger.warning(f"Domain merge: dropping non-string entry {type(k).__name__}={type(v).__name__} for key '{k}'")
                    continue
                old_name, new_name = k.strip(), v.strip()
                if not old_name or not new_name:
                    self.logger.warning(f"Domain merge: dropping blank mapping '{k}' → '{v}'")
                    continue
                if old_name.lower() in protected and new_name.lower() != old_name.lower():
                    self.logger.warning(f"🛡️ BLOCKED merge of user-specified domain '{old_name}' → '{new_name}': user domains are immutable")
                    continue
                sanitized[old_name] = protected.get(new_name.lower(), new_name)
            merge_mapping = sanitized
            
            # Resolve transitive merges (e.g., A->B, B->C  =>  A->C, B->C)
            for _ in range(5):
                updated = False
                for key, val in list(merge_mapping.items()):
                    if val in merge_mapping and merge_mapping[val] != val:
                         merge_mapping[key] = merge_mapping[val]
                         updated = True
                if not updated:
                    break
            
            projected_counts = defaultdict(int, domain_counts)
            rejected_merges = []
            for old_name, new_name in list(merge_mapping.items()):
                if old_name == new_name:
                    continue
                old_count = domain_counts.get(old_name, 0)
                target_current = projected_counts.get(new_name, 0)
                if target_current + old_count > max_ucs_per_domain:
                    self.logger.warning(
                        f"Rejecting merge '{old_name}' ({old_count} UCs) → '{new_name}' ({target_current} UCs): "
                        f"would create mega-domain with {target_current + old_count} UCs (max {max_ucs_per_domain})"
                    )
                    rejected_merges.append(old_name)
                    del merge_mapping[old_name]
                else:
                    projected_counts[new_name] = target_current + old_count
                    if old_name in projected_counts:
                        del projected_counts[old_name]
            
            if rejected_merges:
                self.logger.info(f"Rejected {len(rejected_merges)} merges that would create mega-domains: {rejected_merges}")
            
            merged_count = 0
            for uc in use_cases:
                old_domain = uc.get('Business Domain', 'Other')
                if old_domain in merge_mapping:
                    new_domain = merge_mapping[old_domain]
                    if old_domain != new_domain:
                        self.logger.info(f"Merging '{old_domain}' ({domain_counts[old_domain]} cases) → '{new_domain}'")
                        uc['Business Domain'] = new_domain
                        merged_count += 1
            
            self.logger.debug(f"Domain merging complete. {merged_count} use cases reassigned.")
            
            final_counts = defaultdict(int)
            for uc in use_cases:
                final_counts[uc.get('Business Domain', 'Other')] += 1
            
            remaining_small = [d for d, count in final_counts.items() if count < min_cases_per_domain]
            if remaining_small:
                self.logger.warning(f"WARNING: {len(remaining_small)} domains still have fewer than {min_cases_per_domain} use cases: {remaining_small}")
            
            mega_domains = {d: c for d, c in final_counts.items() if c > max_ucs_per_domain}
            if mega_domains:
                self.logger.warning(
                    f"⚠️ {len(mega_domains)} mega-domain(s) detected after merge: "
                    + ", ".join(f"'{d}' ({c} UCs, {c/total_ucs:.0%})" for d, c in mega_domains.items())
                    + f". Max allowed: {max_ucs_per_domain} UCs ({max_domain_share:.0%}). "
                    f"These will be split in the next step."
                )
            
            domain_count = len(final_counts)
            self.logger.info(
                f"Post-merge: {domain_count} domains (target: {target_domain_count}). "
                f"Distribution: {dict(sorted(final_counts.items(), key=lambda x: -x[1]))}"
            )
            
            return use_cases
            
        except Exception as e:
            self.logger.error(f"Failed to merge domains: {get_clean_error_message(e)}")
            return use_cases

    def _split_mega_domains(self, use_cases: list) -> list:
        """
        Post-merge step: detects domains that are too large (>35% of total UCs) and splits them
        into more specific domains using an LLM. User-specified Business Domains are never split.
        Use cases the split leaves unmapped get one targeted retry, then the new domain of their
        same-primary-table siblings; only without any evidence do they keep the original domain.
        """
        if not use_cases:
            return use_cases
        
        total_ucs = len(use_cases)
        max_domain_share = 0.35
        max_ucs_per_domain = max(int(total_ucs * max_domain_share), 6)
        protected = self._protected_domains()
        
        domain_counts = defaultdict(int)
        for uc in use_cases:
            domain = uc.get('Business Domain', 'Other')
            domain_counts[domain] += 1
        
        mega_domains = {d: c for d, c in domain_counts.items() if c > max_ucs_per_domain}
        for d in [d for d in mega_domains if str(d).strip().lower() in protected]:
            self.logger.info(f"🛡️ BLOCKED split of user-specified domain '{d}' ({mega_domains[d]} UCs): user domains are immutable")
            del mega_domains[d]
        
        if not mega_domains:
            self.logger.info(f"No splittable mega-domains found (max allowed: {max_ucs_per_domain} UCs per domain). Skipping split.")
            return use_cases
        
        import math
        all_domain_names = set(domain_counts.keys())
        
        for mega_name, mega_count in mega_domains.items():
            target_splits = max(2, round(math.sqrt(mega_count)))
            
            self.logger.info(
                f"📍 Splitting mega-domain '{mega_name}' ({mega_count} UCs, {mega_count/total_ucs:.0%} of total) "
                f"into ~{target_splits} new domains..."
            )
            
            mega_ucs = [uc for uc in use_cases if str(uc.get('Business Domain') or '').strip() == str(mega_name).strip()]
            
            uc_lines = []
            for uc in mega_ucs:
                uc_lines.append(
                    f"  - {uc.get('No', '')}: {uc.get('Name', '')} "
                    f"[{uc.get('type', '')}] [{uc.get('Analytics Technique', '')}] "
                    f"Beneficiary: {uc.get('Beneficiary', '')} | Tables: {uc.get('Tables Involved', '')}"
                )
            uc_str = self._escape_template_braces("\n".join(uc_lines))
            
            other_domains = sorted(all_domain_names - {mega_name})
            other_domains_str = self._escape_template_braces(", ".join(f'"{d}"' for d in other_domains) if other_domains else "(none)")
            mega_label = self._escape_template_braces(mega_name)
            
            split_prompt = f"""You are a business domain expert. The domain "{mega_label}" is too large and must be split.

**CURRENT SITUATION**:
- Domain "{mega_label}" has {mega_count} use cases ({mega_count/total_ucs:.0%} of all {total_ucs} UCs)
- Maximum allowed per domain: {max_ucs_per_domain} UCs ({max_domain_share:.0%})
- Other existing domains: {other_domains_str}

**USE CASES IN "{mega_label}"**:
{uc_str}

**YOUR TASK**: Split these {mega_count} use cases into {target_splits}-{target_splits+1} NEW domain names.

**Output Format** (honesty-wrapped JSON):
{{{{"honesty_score": XX, "honesty_justification": "...", "data": <your_mapping>}}}}

The "data" field maps use_case_id → new_domain_name.
Example:
{{{{"honesty_score": 90, "honesty_justification": "Split based on functional areas.", "data": {{{{"N01-U03": "Revenue", "N01-U06": "Inventory", "N01-U01": "Workforce"}}}}}}}}

**Rules**:
1. Every domain name must be EXACTLY ONE WORD (no spaces), unique
2. ❌ BANNED: "Operations", "Engagement", "Management", "Services", "Activities", "General"
3. ✅ Use specific business nouns: "Revenue", "Inventory", "Fraud", "Workforce", "Franchise", "Pricing", "Customers"
4. New domain names MUST NOT match any existing domain: {other_domains_str}
5. Each new domain must have at least 2 use cases
6. Distribute use cases as evenly as possible across new domains
7. Every use case must be assigned to exactly one new domain

Start your response with: {{{{"honesty_score":
""" + HONESTY_CHECK_JSON

            try:
                split_prompt_key = "DOMAIN_SPLITTER_PROMPT"
                _splitter_instr = self._get_generation_instruction_for_prompt("DOMAIN_FINDER_PROMPT")
                split_prompt = f"{split_prompt}\n\n{self._escape_template_braces(_splitter_instr)}\n"
                self.ai_agent.prompt_templates[split_prompt_key] = split_prompt
                
                response = self.ai_agent.run_worker(
                    step_name=f"Split_Mega_Domain_{mega_name}",
                    worker_prompt_path=split_prompt_key,
                    prompt_vars={},
                    response_schema=None
                )
                
                del self.ai_agent.prompt_templates[split_prompt_key]
                
                split_mapping = self._parse_split_mapping(response, mega_ucs)
                if split_mapping is None:
                    self.logger.warning(f"Domain split response is empty or not a dict. Keeping '{mega_name}' as-is.")
                    continue
                
                new_domain_counts = defaultdict(int)
                for new_domain in split_mapping.values():
                    new_domain_counts[new_domain] += 1
                
                if len(new_domain_counts) < 2:
                    self.logger.warning(f"LLM only produced {len(new_domain_counts)} domain(s) for split. Keeping '{mega_name}' as-is.")
                    continue
                
                collisions = set(new_domain_counts.keys()) & (all_domain_names - {mega_name})
                if collisions:
                    self.logger.warning(f"LLM split produced domain names that collide with existing: {collisions}. Keeping '{mega_name}' as-is.")
                    continue
                
                unmapped = [uc for uc in mega_ucs if str(uc.get('No', '')).strip() not in split_mapping]
                if unmapped:
                    split_mapping.update(self._retry_split_assignment(mega_name, unmapped, sorted(new_domain_counts)))
                
                split_count = 0
                mapped_ucs, still_unmapped = [], []
                for uc in mega_ucs:
                    new_domain = split_mapping.get(str(uc.get('No', '')).strip())
                    if new_domain:
                        uc['Business Domain'] = new_domain
                        split_count += 1
                        mapped_ucs.append(uc)
                    else:
                        still_unmapped.append(uc)
                if still_unmapped:
                    still_unmapped = self._fill_by_table_evidence(still_unmapped, mapped_ucs, 'Business Domain', 'domain:table_evidence')
                    for uc in still_unmapped:
                        self._add_clustering_fallback(uc, 'domain:split_unmapped')
                    if still_unmapped:
                        self.logger.warning(
                            f"⚠️ Split of '{mega_name}': {len(still_unmapped)} use case(s) had no LLM assignment and no "
                            f"same-table evidence; they stay in '{mega_name}': {[uc.get('No') for uc in still_unmapped][:10]}"
                        )
                
                all_domain_names = (all_domain_names - {mega_name}) | set(new_domain_counts.keys())
                if still_unmapped:
                    all_domain_names.add(mega_name)
                
                self.logger.info(
                    f"✅ Split '{mega_name}' → {len(new_domain_counts)} new domains ({split_count} LLM-assigned): "
                    + ", ".join(f"'{d}' ({c} UCs)" for d, c in sorted(new_domain_counts.items(), key=lambda x: -x[1]))
                )
                
            except Exception as e:
                self.logger.error(f"Failed to split mega-domain '{mega_name}': {get_clean_error_message(e)}. Keeping as-is.")
                continue
        
        final_counts = defaultdict(int)
        for uc in use_cases:
            final_counts[uc.get('Business Domain', 'Other')] += 1
        self.logger.info(
            f"Post-split: {len(final_counts)} domains. "
            f"Distribution: {dict(sorted(final_counts.items(), key=lambda x: -x[1]))}"
        )
        
        return use_cases

    def _parse_split_mapping(self, response, ucs, allowed=None):
        cleaned = clean_json_response(response) if response else ""
        if not cleaned or not cleaned.strip():
            return None
        data = json.loads(cleaned)
        if isinstance(data, dict) and 'data' in data:
            data = data['data']
        if not isinstance(data, dict):
            return None
        ids = {str(uc.get('No', '')).strip() for uc in ucs}
        canonical = {str(a).lower(): a for a in (allowed or [])}
        mapping = {}
        for k, v in data.items():
            key = str(k).strip()
            if key not in ids or not isinstance(v, str) or not v.strip():
                continue
            value = v.strip()
            if allowed is not None:
                if value.lower() not in canonical:
                    continue
                value = canonical[value.lower()]
            mapping[key] = value
        return mapping

    def _retry_split_assignment(self, mega_name, unmapped, new_domains) -> dict:
        lines = "\n".join(f"  - {uc.get('No', '')}: {uc.get('Name', '')} | Tables: {uc.get('Tables Involved', '')}"
                          for uc in unmapped)
        allowed = ", ".join(f'"{d}"' for d in new_domains)
        prompt = f"""You split the business domain "{self._escape_template_braces(mega_name)}" into these domains: {self._escape_template_braces(allowed)}.
These use cases were left unassigned:
{self._escape_template_braces(lines)}

Assign EVERY use case above to EXACTLY ONE of those domains, using the exact spelling. Do not invent new domains.

**Output Format** (honesty-wrapped JSON):
{{{{"honesty_score": XX, "honesty_justification": "...", "data": {{{{"<use_case_id>": "<domain>"}}}}}}}}

Start your response with: {{{{"honesty_score":
""" + HONESTY_CHECK_JSON
        prompt_key = "DOMAIN_SPLITTER_PROMPT"
        try:
            self.ai_agent.prompt_templates[prompt_key] = (
                f"{prompt}\n\n{self._escape_template_braces(self._get_generation_instruction_for_prompt('DOMAIN_FINDER_PROMPT'))}\n")
            response = self.ai_agent.run_worker(
                step_name=f"Split_Mega_Domain_{mega_name}_Unmapped",
                worker_prompt_path=prompt_key,
                prompt_vars={},
                response_schema=None,
                skip_cache=True
            )
            mapping = self._parse_split_mapping(response, unmapped, allowed=new_domains) or {}
            self.logger.info(f"Split '{mega_name}': retry assigned {len(mapping)}/{len(unmapped)} unmapped use case(s)")
            return mapping
        except Exception as e:
            self.logger.warning(f"Split '{mega_name}': retry for {len(unmapped)} unmapped use case(s) failed: {get_clean_error_message(e)}")
            return {}
        finally:
            self.ai_agent.prompt_templates.pop(prompt_key, None)

    def _merge_duplicate_subdomains(self, use_cases: list) -> list:
        """
        Step 2.5: Cross-domain subdomain deduplication.
        
        After per-domain subdomain detection (Step 2), different domains may have
        independently chosen identical subdomain names (e.g., "Transaction Monitoring"
        under both Fraud and Operations). This method detects such collisions and
        uses an LLM to rename them so each subdomain name is globally unique while
        remaining descriptive of its specific domain context.
        
        Handles edge cases:
        - 2-domain collisions (most common)
        - 3+ domain collisions (e.g., "Risk Management" in Fraud, Operations, and Pricing)
        - LLM renames that collide with existing non-duplicate subdomain names
        - LLM renames that are not exactly 2 words are rejected
        - LLM returning invalid/empty responses → deterministic fallback always resolves
        
        Args:
            use_cases: List of use case dicts with 'Business Domain' and 'Subdomain' set
            
        Returns:
            Same list with duplicate subdomain names resolved (modified in-place)
        """
        if not use_cases:
            return use_cases
        
        from collections import defaultdict
        
        subdomain_to_domains = defaultdict(set)
        for uc in use_cases:
            domain = uc.get('Business Domain', '').strip()
            subdomain = uc.get('Subdomain', '').strip()
            if domain and subdomain:
                subdomain_to_domains[subdomain].add(domain)
        
        duplicates = {sd: sorted(domains) for sd, domains in subdomain_to_domains.items() if len(domains) > 1}
        
        if not duplicates:
            self.logger.info("✅ No cross-domain subdomain duplicates found. Skipping subdomain merger.")
            return use_cases
        
        total_collisions = sum(len(doms) for doms in duplicates.values())
        self.logger.warning(
            f"⚠️ Found {len(duplicates)} subdomain name(s) duplicated across {total_collisions} domain instances: "
            + ", ".join(f"'{sd}' in [{', '.join(doms)}]" for sd, doms in duplicates.items())
        )
        
        domain_subdomain_ucs = defaultdict(lambda: defaultdict(list))
        for uc in use_cases:
            domain = uc.get('Business Domain', '').strip()
            subdomain = uc.get('Subdomain', '').strip()
            if domain and subdomain:
                domain_subdomain_ucs[domain][subdomain].append(
                    uc.get('Name', uc.get('No', ''))
                )
        
        all_existing_names = set()
        for domain_sds in domain_subdomain_ucs.values():
            all_existing_names.update(domain_sds.keys())
        
        dup_details_lines = []
        for sd, domains in duplicates.items():
            dup_details_lines.append(f"\n  DUPLICATE SUBDOMAIN: \"{sd}\" (appears in {len(domains)} domains)")
            for domain in domains:
                uc_names = domain_subdomain_ucs[domain].get(sd, [])
                dup_details_lines.append(
                    f"    - Domain \"{domain}\": {len(uc_names)} use cases"
                    f" → {', '.join(uc_names[:5])}"
                    + (f" ... and {len(uc_names) - 5} more" if len(uc_names) > 5 else "")
                )
        dup_details_str = self._escape_template_braces("\n".join(dup_details_lines))
        
        all_subdomains_lines = []
        for domain in sorted(domain_subdomain_ucs.keys()):
            sds = sorted(domain_subdomain_ucs[domain].keys())
            for sd in sds:
                uc_names = domain_subdomain_ucs[domain][sd]
                is_dup = "⚠️ DUPLICATE" if sd in duplicates else "✓"
                all_subdomains_lines.append(
                    f"  - Domain \"{domain}\" / Subdomain \"{sd}\": "
                    f"{len(uc_names)} use cases {is_dup}"
                    f"  (UCs: {', '.join(uc_names[:3])}{'...' if len(uc_names) > 3 else ''})"
                )
        all_subdomains_str = self._escape_template_braces("\n".join(all_subdomains_lines))
        
        non_dup_names = sorted(all_existing_names - set(duplicates.keys()))
        reserved_names_str = self._escape_template_braces(", ".join(f'"{n}"' for n in non_dup_names) if non_dup_names else "(none)")
        
        merge_prompt = f"""You are a business domain taxonomy expert. You must resolve DUPLICATE subdomain names that appear in multiple business domains.

**CURRENT SUBDOMAIN MAP** (all domains, subdomains, and their use cases):
{all_subdomains_str}

**DUPLICATES TO RESOLVE**:
{dup_details_str}

**RESERVED NAMES** (already used by non-duplicate subdomains — your renames MUST NOT match any of these):
{reserved_names_str}

**YOUR TASK**: For each duplicate subdomain name, rename ONE OR BOTH instances so that every subdomain name is globally unique across all domains. The new names must:
1. Be EXACTLY 2 words (same constraint as original). Not 1, not 3+. EXACTLY 2 words.
2. Clearly reflect the SPECIFIC domain context and the USE CASES within that subdomain
3. NOT match any existing subdomain name (including the reserved names above)
4. NOT match any OTHER renamed subdomain you produce in this response
5. Remain business-focused and professional — no generic names like "Other Services"

**RULES**:
- Only rename subdomains that are duplicated. Do NOT rename non-duplicated subdomains.
- Prefer renaming the subdomain in the domain where it is LESS central. Use the use case names to judge centrality.
- If both are equally central, rename both to be domain-specific.
- If a subdomain appears in 3+ domains, you MUST rename ALL instances to be unique.
- No two subdomain names (across ALL domains) may be identical after your renaming.
- Validate your output: every new name must be exactly 2 words and globally unique.

**Output Format** (honesty-wrapped JSON):
Your response MUST be wrapped: {{{{"honesty_score": XX, "honesty_justification": "...", "data": <your_mapping>}}}}

The "data" field must be a JSON object mapping:
  "OldDomain:::OldSubdomain" → "NewSubdomain"

Example (2-domain collision — rename the less central one):
{{{{
  "honesty_score": 90,
  "honesty_justification": "Renamed Operations instance as Transaction Monitoring is more central to Fraud domain.",
  "data": {{{{
    "Operations:::Transaction Monitoring": "Process Oversight"
  }}}}
}}}}

Example (3-domain collision — rename all):
{{{{
  "honesty_score": 88,
  "honesty_justification": "All three domains equally use this subdomain, renamed all with domain context.",
  "data": {{{{
    "Fraud:::Risk Management": "Fraud Mitigation",
    "Operations:::Risk Management": "Operational Safeguards",
    "Pricing:::Risk Management": "Pricing Controls"
  }}}}
}}}}

Only include entries for subdomains that need renaming. Start your response with: {{{{"honesty_score":
""" + HONESTY_CHECK_JSON

        try:
            merge_prompt_key = "SUBDOMAINS_MERGER_PROMPT"
            _subdomain_instr = self._get_generation_instruction_for_prompt("SUBDOMAIN_DETECTOR_PROMPT")
            merge_prompt = f"{merge_prompt}\n\n{self._escape_template_braces(_subdomain_instr)}\n"
            self.ai_agent.prompt_templates[merge_prompt_key] = merge_prompt
            
            self.logger.info(
                f"📍 STEP 2.5: Resolving {len(duplicates)} cross-domain subdomain name collision(s) "
                f"across {total_collisions} domain instances..."
            )
            
            response = self.ai_agent.run_worker(
                step_name="Merge_Duplicate_Subdomains",
                worker_prompt_path=merge_prompt_key,
                prompt_vars={},
                response_schema=None
            )
            
            del self.ai_agent.prompt_templates[merge_prompt_key]
            
            rename_mapping = json.loads(clean_json_response(response))
            
            if isinstance(rename_mapping, dict) and 'data' in rename_mapping:
                rename_mapping = rename_mapping['data']
            
            if not isinstance(rename_mapping, dict):
                self.logger.warning(f"Subdomain merger response is not a dict: {type(rename_mapping)}. Falling back to deterministic rename.")
                return self._deterministic_subdomain_dedup(use_cases, duplicates, all_existing_names)
            
            sanitized = {}
            for k, v in rename_mapping.items():
                if isinstance(k, str) and isinstance(v, str) and ':::' in k:
                    new_name = " ".join(v.split())
                    if len(new_name.split()) != 2:
                        self.logger.warning(f"Subdomain merger: rejecting rename '{k}' → '{v}' (must be exactly 2 words)")
                        continue
                    sanitized[k] = new_name
                else:
                    self.logger.warning(f"Subdomain merger: dropping invalid entry '{k}' → '{v}' (expected 'Domain:::Subdomain' → 'NewName')")
            rename_mapping = sanitized
            
            if not rename_mapping:
                self.logger.info("Subdomain merger returned no usable renames. Falling back to deterministic rename.")
                return self._deterministic_subdomain_dedup(use_cases, duplicates, all_existing_names)
            
            proposed_new_names = set()
            collision_with_existing = []
            collision_with_proposed = []
            for key, new_name in list(rename_mapping.items()):
                domain_part = key.split(':::')[0] if ':::' in key else ''
                old_sd = key.split(':::')[1] if ':::' in key else ''
                
                if new_name in all_existing_names and new_name != old_sd:
                    collision_with_existing.append((key, new_name))
                    self.logger.warning(
                        f"LLM proposed rename '{old_sd}' → '{new_name}' in domain '{domain_part}' "
                        f"collides with existing subdomain. Will use deterministic fallback for this one."
                    )
                    del rename_mapping[key]
                elif new_name in proposed_new_names:
                    collision_with_proposed.append((key, new_name))
                    self.logger.warning(
                        f"LLM proposed rename '{old_sd}' → '{new_name}' in domain '{domain_part}' "
                        f"collides with another proposed rename. Will use deterministic fallback for this one."
                    )
                    del rename_mapping[key]
                else:
                    proposed_new_names.add(new_name)
            
            llm_rename_count = 0
            for uc in use_cases:
                domain = uc.get('Business Domain', '').strip()
                subdomain = uc.get('Subdomain', '').strip()
                key = f"{domain}:::{subdomain}"
                if key in rename_mapping:
                    new_name = rename_mapping[key]
                    if new_name != subdomain:
                        uc['Subdomain'] = new_name
                        llm_rename_count += 1
            
            unique_renames = len(set(rename_mapping.values()))
            self.logger.info(
                f"✅ LLM subdomain merger: {llm_rename_count} use cases renamed "
                f"across {unique_renames} unique subdomain renames"
            )
            if collision_with_existing:
                self.logger.info(f"   ⚠️ {len(collision_with_existing)} LLM renames rejected (collided with existing names)")
            if collision_with_proposed:
                self.logger.info(f"   ⚠️ {len(collision_with_proposed)} LLM renames rejected (collided with each other)")
            
            updated_existing = set()
            for uc in use_cases:
                s = uc.get('Subdomain', '').strip()
                if s:
                    updated_existing.add(s)
            
            remaining_dups = defaultdict(set)
            for uc in use_cases:
                d = uc.get('Business Domain', '').strip()
                s = uc.get('Subdomain', '').strip()
                if d and s:
                    remaining_dups[s].add(d)
            still_dup = {sd: sorted(doms) for sd, doms in remaining_dups.items() if len(doms) > 1}
            
            if still_dup:
                self.logger.warning(
                    f"⚠️ After LLM merger, {len(still_dup)} subdomain name(s) still duplicated: "
                    + ", ".join(f"'{sd}' in [{', '.join(doms)}]" for sd, doms in still_dup.items())
                    + ". Applying deterministic fallback for remaining collisions..."
                )
                return self._deterministic_subdomain_dedup(use_cases, still_dup, updated_existing)
            
            return use_cases
        
        except Exception as e:
            self.logger.error(f"Failed to merge duplicate subdomains via LLM: {get_clean_error_message(e)}. Applying deterministic fallback...")
            return self._deterministic_subdomain_dedup(use_cases, duplicates, all_existing_names)

    def _deterministic_subdomain_dedup(self, use_cases: list, duplicates: dict, all_existing_names: set) -> list:
        """
        Deterministic fallback for cross-domain subdomain deduplication.
        Guarantees resolution of all collisions without LLM, producing valid 2-word names.
        
        Strategy: For each duplicated subdomain name, iterate through the domains that share it.
        Keep the name as-is for the domain with the MOST use cases in that subdomain (most central).
        For all other domains, generate a domain-prefixed 2-word name.
        If the generated name also collides, append a numeric suffix to the domain word.
        
        Args:
            use_cases: List of use case dicts
            duplicates: Dict mapping subdomain_name → set/list of domain names that share it
            all_existing_names: Set of all currently used subdomain names (for collision avoidance)
        
        Returns:
            Modified use_cases list with all cross-domain subdomain duplicates resolved
        """
        from collections import defaultdict
        
        domain_sd_count = defaultdict(lambda: defaultdict(int))
        for uc in use_cases:
            d = uc.get('Business Domain', '').strip()
            s = uc.get('Subdomain', '').strip()
            if d and s:
                domain_sd_count[d][s] += 1
        
        used_names = set(all_existing_names)
        rename_plan = {}
        
        fallback_rename_count = 0
        for sd_name, domains in duplicates.items():
            domains_list = sorted(domains) if isinstance(domains, set) else list(domains)
            
            domain_uc_counts = [(d, domain_sd_count[d].get(sd_name, 0)) for d in domains_list]
            domain_uc_counts.sort(key=lambda x: -x[1])
            
            keep_domain = domain_uc_counts[0][0]
            rename_domains = [d for d, _ in domain_uc_counts[1:]]
            
            self.logger.info(
                f"Deterministic dedup '{sd_name}': keeping in '{keep_domain}' "
                f"({domain_uc_counts[0][1]} UCs), renaming in: "
                + ", ".join(f"'{d}' ({c} UCs)" for d, c in domain_uc_counts[1:])
            )
            
            for domain in rename_domains:
                sd_words = sd_name.split()
                domain_words = domain.split()
                domain_prefix = domain_words[0] if domain_words else "Alt"
                sd_suffix = sd_words[-1] if sd_words else "Area"
                
                if domain_prefix.lower() == sd_suffix.lower():
                    sd_suffix = sd_words[0] if len(sd_words) > 1 else "Focus"
                
                candidate = f"{domain_prefix} {sd_suffix}"
                
                attempt = 0
                while candidate.lower() in {n.lower() for n in used_names} or candidate.lower() in {v.lower() for v in rename_plan.values()}:
                    attempt += 1
                    if attempt <= len(sd_words) and attempt < len(sd_words):
                        alt_suffix = sd_words[-(attempt + 1)] if len(sd_words) > attempt else sd_suffix
                        candidate = f"{domain_prefix} {alt_suffix}"
                    else:
                        candidate = f"{domain_prefix}{attempt} {sd_suffix}"
                    if attempt > 10:
                        candidate = f"{domain_prefix} Area{attempt}"
                        break
                
                rename_plan[f"{domain}:::{sd_name}"] = candidate
                used_names.add(candidate)
                self.logger.info(f"   → Domain '{domain}': '{sd_name}' → '{candidate}'")
        
        for uc in use_cases:
            d = uc.get('Business Domain', '').strip()
            s = uc.get('Subdomain', '').strip()
            key = f"{d}:::{s}"
            if key in rename_plan:
                uc['Subdomain'] = rename_plan[key]
                fallback_rename_count += 1
        
        self.logger.info(f"✅ Deterministic subdomain dedup: {fallback_rename_count} use cases renamed across {len(rename_plan)} renames")
        
        final_check = defaultdict(set)
        for uc in use_cases:
            d = uc.get('Business Domain', '').strip()
            s = uc.get('Subdomain', '').strip()
            if d and s:
                final_check[s].add(d)
        final_still_dup = {sd: doms for sd, doms in final_check.items() if len(doms) > 1}
        if final_still_dup:
            self.logger.error(
                f"🚨 CRITICAL: After deterministic dedup, {len(final_still_dup)} collision(s) remain: "
                + ", ".join(f"'{sd}' in [{', '.join(sorted(doms))}]" for sd, doms in final_still_dup.items())
                + ". This should never happen — please report this bug."
            )
        else:
            self.logger.info("✅ All cross-domain subdomain name collisions fully resolved.")
        
        return use_cases

    def _cluster_domains_and_subdomains(self, use_cases: list, language: str, merge_small_domains: bool = True) -> list:
        """
        Cluster use cases into domains and subdomains with the TWO-STEP LLM approach:
        Step 1: assign a domain to every use case (one call, or schema-batched when too large)
        Step 2: detect subdomains per domain in parallel
        Every input use case is returned exactly once. A use case the LLM skips is re-asked, then
        takes the domain of its same-primary-table siblings, then an evidence-based placeholder.
        The LLM-call cost of each stage is logged and stored in processing_honesty.
        
        Args:
            use_cases: List of use case dictionaries
            language: Output language
            
        Returns:
            List of use cases with updated domains and subdomains
        """
        import io
        import csv
        
        try:
            self._emit_pipeline_status(
                stage_name="Domain Mapping", step_name="Domain/Subdomain Clustering",
                sub_step_name="Started", status="started",
                message=f"Two-step clustering started for {len(use_cases)} UCs"
            )
        except Exception:
            pass
        self.logger.info(f"🎯 Starting TWO-STEP domain/subdomain clustering for {len(use_cases)} use cases...")
        
        if not use_cases:
            return use_cases
        
        cost = {}
        mark = self._llm_cost_mark()
        # === USER-PROVIDED BUSINESS DOMAINS ENFORCEMENT ===
        # If user provided business domains, we MUST force all use cases to align to those domains only
        if self.user_business_domains and len(self.user_business_domains) > 0:
            self.logger.info(f"🚨 USER-PROVIDED DOMAINS DETECTED: Forcing use cases to align ONLY to: {', '.join(self.user_business_domains)}")
            log_print(f"\n🚨 USER-PROVIDED DOMAINS: Aligning all use cases to: {', '.join(self.user_business_domains)}")
            # Use LLM to intelligently assign use cases to user-provided domains
            result = self._assign_to_user_domains(use_cases, self.user_business_domains, language)
            result = self._ensure_clustering_complete(use_cases, result)
            self._llm_cost_add(cost, "user_domain_assignment", mark)
            self._report_clustering_cost(cost)
            return result
        
        # === NOTE: Removed legacy fallback - always use two-step approach ===
        # The two-step approach (DOMAIN_FINDER_PROMPT + SUBDOMAIN_DETECTOR_PROMPT) 
        # provides better quality and more consistent results than the old single-step approach
        if len(use_cases) > 250:
            self.logger.info(
                f"📊 Large use case set detected ({len(use_cases)} use cases). "
                f"Using two-step clustering with parallel subdomain detection for optimal quality."
            )
        
        # === STEP 1: DOMAIN DETECTION ===
        
        fieldnames = ['No', 'Name', 'type', 'Analytics Technique', 'Statement', 'Solution',
                      'Business Value', 'Beneficiary', 'Sponsor', 'Tables Involved']
        try:
            # Convert use cases to CSV for LLM (without Business Domain and Subdomain since those will be detected)
            output = io.StringIO()
            writer = csv.DictWriter(output, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(use_cases)
            use_cases_csv = output.getvalue()
            
            # Check context size for domain detection (uses model-specific limits from TECHNICAL_CONTEXT)
            prompt_template = self.ai_agent.prompt_templates.get("DOMAIN_FINDER_PROMPT", "")
            estimated_size = self._prompt_overhead_chars("DOMAIN_FINDER_PROMPT", prompt_template) + len(use_cases_csv) + 1000
            MAX_CONTEXT_CHARS = get_max_context_chars(language, "DOMAIN_FINDER_PROMPT")
            path_label = "single-call"
            
            if estimated_size > MAX_CONTEXT_CHARS:
                # === BATCHED DOMAIN DETECTION: Process in smaller chunks ===
                path_label = "batched"
                self.logger.warning(
                    f"Domain detection prompt size ({estimated_size:,} chars) exceeds MAX_CONTEXT_CHARS ({MAX_CONTEXT_CHARS:,}). "
                    f"Using BATCHED domain detection to process {len(use_cases)} use cases in smaller chunks."
                )
                domain_assignments = self._run_domain_detection_batched(
                    use_cases, fieldnames, language, prompt_template, MAX_CONTEXT_CHARS
                )
                if not domain_assignments:
                    domain_assignments = self._assign_default_domains(use_cases)
            else:
                # Warn if prompt is very large (might cause slowness)
                if estimated_size > MAX_CONTEXT_CHARS * 0.7:
                    self.logger.warning(
                        f"⚠️ Domain detection prompt is very large ({estimated_size:,} chars, {len(use_cases)} use cases). "
                        f"This may take 2-5 minutes to process. Please be patient..."
                    )
                    log_print(f"\n⏳ Processing {len(use_cases)} use cases for domain detection...")
                    log_print(f"   Prompt size: {estimated_size:,} characters")
                    log_print(f"   This may take 2-5 minutes. Please wait...\n")
                
                max_attempts = self.max_retry_attempts + 1
                prompt_vars = self._domain_finder_vars(use_cases_csv, language)
                total_use_cases = len(use_cases)
                domain_assignments = None
                attempts = []
                for attempt in range(1, max_attempts + 1):
                    try:
                        self.logger.info(f"📍 STEP 1: Detecting domains for all {len(use_cases)} use cases ({estimated_size:,} chars) - {attempt}/{max_attempts}")
                        
                        import time
                        start_time = time.time()
                        
                        response_raw = self.ai_agent.run_worker(
                            step_name=f"Detect_Domains_{language}_Attempt{attempt}",
                            worker_prompt_path="DOMAIN_FINDER_PROMPT",
                            prompt_vars=prompt_vars,
                            response_schema=None,
                            skip_cache=(attempt > 1)  # v0.7.11: retries bypass cache
                        )
                        self.logger.info(f"✅ LLM response received in {time.time() - start_time:.1f} seconds")
                        
                        # Validate response is not empty
                        if not response_raw or len(str(response_raw).strip()) == 0:
                            raise ValueError("LLM returned empty response")
                        domain_assignment_map = self._parse_assignment_csv(response_raw, "domain", "Domain detection", use_cases)
                        if not domain_assignment_map:
                            self.logger.error(f"CSV parsing failed. Raw response (first 500 chars): {str(response_raw)[:500]}")
                            raise ValueError("No valid domain assignments found in CSV")
                        domain_assignment_map = self._canonicalize_domain_names(domain_assignment_map)
                        
                        domain_assigned_use_cases = []
                        for uc in use_cases:
                            uc_copy = uc.copy()
                            uc_copy['Business Domain'] = domain_assignment_map.get(str(uc_copy.get('No', '')).strip(), '')
                            uc_copy['Subdomain'] = ''
                            domain_assigned_use_cases.append(uc_copy)
                        
                        domain_usecases = defaultdict(list)
                        for uc in domain_assigned_use_cases:
                            if uc['Business Domain']:
                                domain_usecases[uc['Business Domain']].append(uc)
                        unassigned = total_use_cases - sum(len(v) for v in domain_usecases.values())
                        
                        total_domains = len(domain_usecases)
                        hard_violations, soft_warnings, small_domains = self._classify_domain_structure(
                            domain_usecases, total_use_cases)
                        
                        # Log soft warnings (informational only, doesn't block)
                        if soft_warnings:
                            self.logger.info(f"ℹ️ Soft warnings (acceptable for small datasets): {'; '.join(soft_warnings[:3])}")
                        attempts.append((hard_violations, domain_assigned_use_cases))
                        
                        # If no HARD violations, domain assignment is successful!
                        if not hard_violations:
                            self.logger.info(
                                f"✅ Domain detection successful on attempt {attempt}! Created {total_domains} domains"
                                + (f"; {unassigned} use case(s) the LLM skipped go to targeted repair" if unassigned else "")
                            )
                            domain_assignments = domain_assigned_use_cases
                            break
                        self.logger.warning(f"⚠️ Domain detection attempt {attempt} has {len(hard_violations)} HARD violations")
                        if attempt < max_attempts:
                            self.logger.info(f"Retrying domain detection (attempt {attempt + 1}/{max_attempts})...")
                            # Prepare violation summary with actionable suggestions for next attempt
                            violation_summary = "\n\n**🚨 PREVIOUS ATTEMPT VIOLATIONS - YOU MUST FIX THESE 🚨**:\n"
                            violation_summary += "\n".join([f"- {v}" for v in hard_violations])
                            if unassigned:
                                violation_summary += f"\n- {unassigned} use case(s) were not assigned a domain. Assign EVERY use case."
                            # Add special guidance for small domains (only if it's a hard violation)
                            if small_domains and total_use_cases >= 50:
                                violation_summary += "\n\n**💡 SUGGESTION FOR SMALL DOMAINS**:\n"
                                violation_summary += "Domains with fewer than 4 use cases should be merged into larger related domains.\n"
                                for small_domain, count in small_domains:
                                    violation_summary += f"  - '{small_domain}' ({count} use cases) → Merge with a related domain\n"
                            # Emphasize the HARD LIMIT on maximum domains
                            if total_domains > 25:
                                violation_summary += f"\n\n**🚨 CRITICAL: MAXIMUM 25 DOMAINS IS AN ABSOLUTE HARD LIMIT 🚨**\n"
                                violation_summary += f"You created {total_domains} domains. You MUST reduce to 25 or fewer.\n"
                                violation_summary += f"Merge related domains together to stay within the limit.\n"
                            # Update prompt_vars for next attempt
                            prompt_vars['previous_violations'] = violation_summary
                    
                    except CascadeRebatchError as cre:
                        self.logger.warning(
                            f"🔄 Domain detection cascade rebatch needed after '{cre.failed_model_name}' failed. "
                            f"Switching to batched mode for model '{cre.target_model_name}' "
                            f"(limit: {cre.target_context_limit_chars:,} chars)"
                        )
                        path_label = "batched"
                        batched_result = self._run_domain_detection_batched(
                            use_cases, fieldnames, language, prompt_template,
                            cre.target_context_limit_chars, cre.target_model_endpoint
                        )
                        if batched_result:
                            domain_assignments = batched_result
                        break
                    
                    except Exception as e:
                        self.logger.warning(f"Domain detection attempt {attempt} failed: {get_clean_error_message(e)}")
                
                if domain_assignments is None and attempts:
                    best_violations, domain_assignments = min(reversed(attempts), key=lambda a: len(a[0]))
                    self.logger.error(
                        f"❌ Domain detection still has {len(best_violations)} HARD violation(s) after {max_attempts} attempt(s); "
                        f"keeping the attempt with the fewest: {'; '.join(best_violations[:5])}"
                    )
                    self._honesty()["domain_hard_violations"] = list(best_violations)
                    try:
                        NEXT_ACTIONS.add("DOMAIN_HARD_VIOLATIONS_ACCEPTED", severity="HIGH", phase="Clustering",
                                         step="domain_detection", evidence="; ".join(best_violations)[:500],
                                         suggested_user_action="Set the Business Domains widget to pin the domain list.")
                    except Exception:
                        pass
                
                if not domain_assignments:
                    self.logger.error("Domain detection failed. Using evidence-based fallback domains...")
                    domain_assignments = self._assign_default_domains(use_cases)
                
                missing = [uc for uc in domain_assignments if not str(uc.get('Business Domain') or '').strip()]
                if missing:
                    assigned = [uc for uc in domain_assignments if str(uc.get('Business Domain') or '').strip()]
                    self._repair_unassigned_domains(missing, assigned, fieldnames, language,
                                                    {uc['Business Domain'] for uc in assigned})
            mark = self._llm_cost_add(cost, "domain_detection", mark)
            
            # === STEP 1.5: DOMAIN MERGING (MERGE SMALL DOMAINS) ===
            if merge_small_domains:
                self.logger.info("📍 STEP 1.5: Merging small domains (if any)...")
                domain_assignments = self._merge_small_domains(domain_assignments)
                mark = self._llm_cost_add(cost, "merge_small_domains", mark)
            else:
                self.logger.info("📍 STEP 1.5: small-domain merging skipped for this set; every domain keeps its use cases")
            
            # === STEP 1.6: SPLIT MEGA-DOMAINS ===
            self.logger.info("📍 STEP 1.6: Splitting mega-domains (if any)...")
            domain_assignments = self._split_mega_domains(domain_assignments)
            mark = self._llm_cost_add(cost, "split_mega_domains", mark)
            
            # === STEP 2: SUBDOMAIN DETECTION (PARALLEL WITH SERIAL FALLBACK) ===
            self.logger.info(f"📍 STEP 2: Detecting subdomains for each domain ({path_label} path)...")
            final_use_cases_with_subdomains = self._detect_subdomains_all_domains(domain_assignments, language, path_label)
            mark = self._llm_cost_add(cost, "subdomain_detection", mark)
            
            self.logger.info(f"📍 STEP 2.5: Cross-domain subdomain deduplication...")
            final_use_cases_with_subdomains = self._merge_duplicate_subdomains(final_use_cases_with_subdomains)
            mark = self._llm_cost_add(cost, "subdomain_dedup", mark)
            
            final_use_cases_with_subdomains = self._rename_placeholder_domains_from_subdomains(final_use_cases_with_subdomains)
            mark = self._llm_cost_add(cost, "placeholder_rename", mark)
            
            final_use_cases_with_subdomains = self._ensure_clustering_complete(use_cases, final_use_cases_with_subdomains)
            self.logger.info(f"✅ Two-step clustering complete! {len(final_use_cases_with_subdomains)} use cases processed")
            self._report_clustering_cost(cost)
            return final_use_cases_with_subdomains
        
        except Exception as e:
            self.logger.error(f"Domain/subdomain clustering failed with error: {get_clean_error_message(e)}. Using evidence-based fallback domains/subdomains...")
            # FALLBACK: Assign default domains and subdomains on any error
            fallback_use_cases = self._assign_default_domains(use_cases)
            # Also assign default subdomains for each domain
            domain_map = defaultdict(list)
            for uc in fallback_use_cases:
                domain_map[uc.get('Business Domain', '')].append(uc)
            final_use_cases = []
            for domain_name, domain_ucs in domain_map.items():
                final_use_cases.extend(self._assign_default_subdomains(domain_ucs, domain_name))
            try:
                final_use_cases = self._rename_placeholder_domains_from_subdomains(final_use_cases)
            except Exception as rename_err:
                self.logger.warning(f"Placeholder domain rename failed in clustering fallback: {get_clean_error_message(rename_err)}")
            self._llm_cost_add(cost, "fallback", mark)
            self._report_clustering_cost(cost)
            self.logger.warning(f"✅ Complete fallback applied: {len(final_use_cases)} use cases with evidence-based domains and subdomains")
            return final_use_cases

    def _run_domain_detection_batched(self, use_cases, fieldnames, language, prompt_template, context_limit, start_from_model=None):
        """
        Run domain detection in batches mirroring the same grouping used during use case generation.
        
        Strategy: Group use cases by their table schema prefix (catalog.schema of the primary table),
        which mirrors how the DataLoader batches tables during generation. Each schema-group is processed
        as a separate domain detection call, reusing previously discovered domains.
        
        If a schema-group is still too large for the context window, it is further split by individual
        table prefixes (catalog.schema.table) within that schema.
        
        Use cases a group answer skips (or a failed group) go to one targeted repair call, then to
        same-primary-table evidence, then to an evidence-based placeholder. No use case is assigned
        another use case's batch-majority domain.
        
        Returns list of domain-assigned use cases, or None on total failure.
        """
        import io
        import csv

        def _build_batch_csv(batch_ucs):
            buf = io.StringIO()
            w = csv.DictWriter(buf, fieldnames=fieldnames, extrasaction='ignore')
            w.writeheader()
            w.writerows(batch_ucs)
            return buf.getvalue()

        def _estimate_prompt_size(batch_csv_text):
            return self._prompt_overhead_chars("DOMAIN_FINDER_PROMPT", prompt_template) + len(batch_csv_text) + 5000

        def _detect_domains_for_group(group_ucs, group_label, effective_limit_chars, effective_model, discovered_domains):
            batch_csv = _build_batch_csv(group_ucs)
            prompt_size = _estimate_prompt_size(batch_csv)
            
            if prompt_size > effective_limit_chars:
                table_groups = defaultdict(list)
                for uc in group_ucs:
                    table_groups[self._uc_primary_table(uc) or '_unknown_'].append(uc)
                
                if len(table_groups) <= 1:
                    self.logger.warning(f"⚠️ [{group_label}] Single table group still exceeds context ({prompt_size:,} > {effective_limit_chars:,}). Processing anyway.")
                else:
                    self.logger.info(f"📦 [{group_label}] Group too large ({prompt_size:,} chars, {len(group_ucs)} UCs). Splitting into {len(table_groups)} table sub-groups.")
                    sub_results = []
                    for sub_idx, (table_key, sub_ucs) in enumerate(table_groups.items(), 1):
                        sub_label = f"{group_label}_T{sub_idx}"
                        sub_result = _detect_domains_for_group(sub_ucs, sub_label, effective_limit_chars, effective_model, discovered_domains)
                        sub_results.extend(sub_result)
                    return sub_results
            
            batch_prompt_vars = self._domain_finder_vars(batch_csv, language,
                                                         extra_context=self._discovered_domains_hint(discovered_domains))
            batch_domain_map = {}
            try:
                batch_response = self.ai_agent.run_worker(
                    step_name=f"Detect_Domains_{group_label}_{language}",
                    worker_prompt_path="DOMAIN_FINDER_PROMPT",
                    prompt_vars=batch_prompt_vars,
                    response_schema=None,
                    start_from_model=effective_model
                )
                batch_domain_map = self._parse_assignment_csv(batch_response, "domain", f"Domain detection {group_label}", group_ucs)
            except CascadeRebatchError as cre:
                self.logger.warning(
                    f"🔄 [{group_label}] Cascade rebatch → model '{cre.target_model_name}' "
                    f"(limit: {cre.target_context_limit_chars:,} chars). Re-splitting group."
                )
                return _detect_domains_for_group(
                    group_ucs, group_label, cre.target_context_limit_chars,
                    cre.target_model_endpoint, discovered_domains
                )
            except Exception as e:
                self.logger.error(f"❌ [{group_label}] Domain detection failed: {get_clean_error_message(e)}. "
                                  f"{len(group_ucs)} use case(s) go to targeted repair.")
            discovered_domains.update(batch_domain_map.values())
            
            results = []
            for uc in group_ucs:
                uc_copy = uc.copy()
                uc_copy['Business Domain'] = batch_domain_map.get(str(uc_copy.get('No', '')).strip(), '')
                uc_copy['Subdomain'] = ''
                results.append(uc_copy)
            skipped = sum(1 for uc in results if not uc['Business Domain'])
            self.logger.info(
                f"✅ [{group_label}] Assigned domains to {len(group_ucs) - skipped}/{len(group_ucs)} use cases "
                f"({len(set(batch_domain_map.values()))} domains found)"
                + (f"; {skipped} skipped go to targeted repair" if skipped else "")
            )
            return results

        schema_groups = defaultdict(list)
        for uc in use_cases:
            schema_groups[self._uc_schema_prefix(uc)].append(uc)
        
        self.logger.info(
            f"📦 BATCHED DOMAIN DETECTION (generation-aligned): {len(use_cases)} use cases "
            f"across {len(schema_groups)} schema groups"
        )
        for prefix, group_ucs in sorted(schema_groups.items(), key=lambda x: len(x[1]), reverse=True):
            self.logger.info(f"   📁 {prefix}: {len(group_ucs)} use cases")

        all_discovered_domains = set()
        completed_assignments = []
        effective_limit_chars = context_limit
        effective_model = start_from_model
        
        for group_idx, (schema_prefix, group_ucs) in enumerate(sorted(schema_groups.items()), 1):
            group_label = f"Schema{group_idx}_{schema_prefix.replace('.', '_')}"
            self.logger.info(f"📍 GROUP {group_idx}/{len(schema_groups)}: '{schema_prefix}' ({len(group_ucs)} use cases)...")
            
            group_results = _detect_domains_for_group(
                group_ucs, group_label, effective_limit_chars, effective_model, all_discovered_domains
            )
            completed_assignments.extend(group_results)
        
        if not completed_assignments:
            self.logger.error("All schema-group domain detection attempts failed")
            return None
        
        canonical = self._canonicalize_domain_names(
            {str(i): uc['Business Domain'] for i, uc in enumerate(completed_assignments) if uc['Business Domain']})
        for i, uc in enumerate(completed_assignments):
            if str(i) in canonical:
                uc['Business Domain'] = canonical[str(i)]
        missing = [uc for uc in completed_assignments if not uc['Business Domain']]
        if missing:
            assigned = [uc for uc in completed_assignments if uc['Business Domain']]
            self._repair_unassigned_domains(missing, assigned, fieldnames, language,
                                            {uc['Business Domain'] for uc in assigned})
        
        self.logger.info(
            f"📦 BATCHED DOMAIN DETECTION COMPLETE: {len(completed_assignments)} use cases "
            f"assigned to {len({uc['Business Domain'] for uc in completed_assignments})} domains"
        )
        return completed_assignments

    _GATE_SQL_KEYWORDS = frozenset({
        "SELECT", "FROM", "WHERE", "AND", "OR", "NOT", "NULL", "TRUE", "FALSE", "IS",
        "ORDER", "GROUP", "BY", "HAVING", "LIMIT", "CASE", "WHEN", "THEN", "ELSE", "END",
        "JOIN", "INNER", "OUTER", "LEFT", "RIGHT", "ON", "AS", "DISTINCT",
    })

    _GATE_CATCHALL_TECHNIQUES = frozenset({
        "ai analysis", "ml analysis", "general analytics", "data science", "ai",
        "statistical analysis", "advanced analytics", "analytics",
    })

    _TECHNIQUE_FAMILIES = (
        ("text_sentiment", "Sentiment Analysis", ("sentiment",), ("ai_analyze_sentiment",)),
        ("text_topic", "Clustering", ("topic model", "topic modelling", "lda", "bertopic"), ("bertopic", "latent dirichlet")),
        ("text_summarization", "Document Processing", ("summariz", "summaris"), ("ai_summarize",)),
        ("text_extraction", "Extraction", ("information extraction", "entity extraction", "named entity", "entity recognition"), ("ai_extract",)),
        ("text_classification", "Classification", ("nlp classification", "text classification"), ()),
        ("text_analytics", "Document Processing", ("text analytics", "text mining", "nlp"), ()),
        ("anomaly", "Anomaly Detection", ("anomal", "outlier", "isolation forest", "change-point", "change point"), ("isolation forest", "local outlier factor", "one-class svm", "mahalanobis", "z-score", "lof")),
        ("forecasting", "Forecasting", ("forecast", "time-series", "time series", "seasonality decomposition", "seasonal decomposition"), ("prophet", "arima", "sarima", "ai_forecast", "exponential smoothing", "holt-winters", "lstm")),
        ("trend", "Trend Analysis", ("trend", "period-over-period", "period over period", "month-over-month", "year-over-year", "yoy"), ("moving average", "rolling average", "stl")),
        ("survival", "Survival Analysis", ("survival", "time-to-event", "kaplan", "hazard"), ("kaplan-meier", "cox proportional", "cox model", "weibull")),
        ("cohort", "Cohort Analysis", ("cohort", "vintage"), ()),
        ("funnel", "Funnel Analysis", ("funnel", "drop-off", "dropoff", "conversion path"), ()),
        ("association", "Recommendation", ("market basket", "basket analysis", "association rule", "affinity analysis"), ("apriori", "fp-growth", "fpgrowth")),
        ("recommendation", "Recommendation", ("recommend", "collaborative filter", "content-based"), ("alternating least squares", "matrix factorization", "als")),
        ("clustering", "Clustering", ("cluster", "k-means", "kmeans", "segmentation", "rfm"), ("k-means", "kmeans", "dbscan", "hdbscan", "hierarchical clustering", "gaussian mixture", "gmm")),
        ("classification", "Classification", ("classif", "propensity"), ("xgboost", "random forest", "lightgbm", "catboost", "logistic regression", "gradient boost", "decision tree", "ai_classify", "classifier")),
        ("predictive", "Predictive Modeling", ("predictive model", "prediction model", "predictive"), ("xgboost", "lightgbm", "random forest", "gradient boost", "neural network")),
        ("regression", "Regression Analysis", ("regress",), ("ridge", "lasso", "elastic net", "elasticnet", "glm", "linear regression", "poisson regression", "ols")),
        ("correlation", "Correlation Analysis", ("correlat",), ("pearson", "spearman", "covariance")),
        ("simulation", "Simulation", ("simulat", "monte carlo", "scenario", "what-if", "what if"), ("monte carlo",)),
        ("optimization", "Optimization", ("optimi",), ("linear program", "integer program", "milp", "knapsack", "solver", "constraint program")),
        ("network", "Network Analysis", ("network", "graph analy"), ("pagerank", "graphframes", "centrality", "community detection", "louvain")),
        ("geospatial", "Geospatial Analysis", ("geospatial", "spatial", "geographic"), ("h3", "geohash", "haversine", "st_distance")),
        ("pareto", "Pareto Analysis", ("pareto", "80/20", "concentration"), ("lorenz", "gini", "herfindahl", "cumulative share")),
        ("ranking", "Pareto Analysis", ("ranking", "top-n", "top n", "score-based", "rank"), ("ntile", "percent_rank", "dense_rank")),
    )

    _FREE_TEXT_DETECTIONS = ("text_topic", "text_classification")
    _GATE_FAMILY_SETS = _gate_family_sets(_TECHNIQUE_FAMILIES, _FREE_TEXT_DETECTIONS)
    _SUPERVISED_FAMILIES = _GATE_FAMILY_SETS["supervised"]
    _MODEL_FAMILIES = _GATE_FAMILY_SETS["model"]
    _TEXT_FAMILIES = _GATE_FAMILY_SETS["text"]
    _TEMPORAL_FAMILIES = _GATE_FAMILY_SETS["temporal"]

    _DEDUP_VERB_FAMILIES = {
        "forecast": "PREDICT", "forecasting": "PREDICT", "predict": "PREDICT", "prediction": "PREDICT",
        "anticipate": "PREDICT", "estimate": "PREDICT", "estimation": "PREDICT", "project": "PREDICT",
        "projection": "PREDICT", "envision": "PREDICT",
        "detect": "DETECT", "detection": "DETECT", "identify": "DETECT", "identification": "DETECT",
        "reveal": "DETECT", "find": "DETECT", "discover": "DETECT", "uncover": "DETECT", "expose": "DETECT",
        "spot": "DETECT", "flag": "DETECT", "surface": "DETECT", "catch": "DETECT",
        "classify": "CLASSIFY", "classification": "CLASSIFY", "segment": "CLASSIFY", "segmentation": "CLASSIFY",
        "categorize": "CLASSIFY", "cluster": "CLASSIFY", "clustering": "CLASSIFY", "group": "CLASSIFY",
        "tier": "CLASSIFY", "profile": "CLASSIFY",
        "rank": "PRIORITIZE", "ranking": "PRIORITIZE", "prioritize": "PRIORITIZE", "prioritization": "PRIORITIZE",
        "score": "PRIORITIZE", "scoring": "PRIORITIZE", "triage": "PRIORITIZE",
        "reduce": "REDUCE", "reduction": "REDUCE", "lower": "REDUCE", "cut": "REDUCE", "decrease": "REDUCE",
        "minimize": "REDUCE", "prevent": "REDUCE", "prevention": "REDUCE", "mitigate": "REDUCE",
        "avoid": "REDUCE", "eliminate": "REDUCE", "curb": "REDUCE", "protect": "REDUCE",
        "increase": "INCREASE", "grow": "INCREASE", "boost": "INCREASE", "lift": "INCREASE", "raise": "INCREASE",
        "maximize": "INCREASE", "improve": "INCREASE", "accelerate": "INCREASE", "expand": "INCREASE",
        "recover": "INCREASE", "retain": "INCREASE",
        "optimize": "OPTIMIZE", "optimise": "OPTIMIZE", "optimization": "OPTIMIZE", "rebalance": "OPTIMIZE",
        "rationalize": "OPTIMIZE", "tune": "OPTIMIZE", "stabilize": "OPTIMIZE",
        "allocate": "ALLOCATE", "allocation": "ALLOCATE", "assign": "ALLOCATE", "route": "ALLOCATE",
        "routing": "ALLOCATE", "schedule": "ALLOCATE", "scheduling": "ALLOCATE", "dispatch": "ALLOCATE",
        "recommend": "RECOMMEND", "recommendation": "RECOMMEND", "personalize": "RECOMMEND",
        "personalization": "RECOMMEND", "suggest": "RECOMMEND", "match": "RECOMMEND",
        "simulate": "SIMULATE", "simulation": "SIMULATE",
        "assess": "ASSESS", "assessment": "ASSESS", "evaluate": "ASSESS", "evaluation": "ASSESS",
        "measure": "ASSESS", "gauge": "ASSESS", "quantify": "ASSESS", "benchmark": "ASSESS",
        "analyze": "ANALYZE", "analyse": "ANALYZE", "analysis": "ANALYZE", "examine": "ANALYZE",
        "investigate": "ANALYZE", "explore": "ANALYZE", "study": "ANALYZE", "understand": "ANALYZE",
        "map": "ANALYZE", "compare": "ANALYZE", "correlate": "ANALYZE",
        "track": "TRACK", "tracking": "TRACK", "monitor": "TRACK", "monitoring": "TRACK",
        "observe": "TRACK", "watch": "TRACK",
    }

    _DEDUP_STOPWORDS = frozenset({
        "a", "an", "the", "of", "for", "by", "in", "on", "with", "and", "or", "to", "via", "through",
        "across", "per", "from", "at", "into", "using", "based", "driven", "level", "data", "insight",
        "insights", "use", "case", "new", "better", "key", "all", "each", "every", "our", "their", "its",
        "plan", "strategy", "program", "playbook", "action", "actions",
    })

    _TECHNICAL_TABLE_TOKENS = frozenset({"log", "logs", "audit", "system", "metadata", "temp", "tmp", "backup", "changelog"})
    _NUMERIC_TYPE_PREFIXES = ("INT", "BIGINT", "SMALLINT", "TINYINT", "DOUBLE", "FLOAT", "DECIMAL", "NUMERIC",
                              "REAL", "LONG", "SHORT", "BYTE")
    _DATE_NAME_RE = re.compile(r"(^|_)(date|time|timestamp|ts|year|yr|month|mo|day|dt|week|quarter|period)(_|$)")
    _NUMERIC_NAME_RE = re.compile(r"(^|_)(amount|amt|count|cnt|qty|quantity|num|number|price|cost|charge|fee|rate|"
                                  r"total|sum|avg|units|value|score|size|age|year|yr|revenue|sales|spend|balance|"
                                  r"margin|profit|duration|distance|weight|volume|pct|percent|ratio)(_|$)")
    _TEXT_NAME_TOKENS = ("text", "comment", "review", "description", "desc", "note", "feedback", "transcript",
                         "body", "message", "title", "subject", "content", "reason", "summary", "narrative", "remark")
    _GEO_NAME_TOKENS = ("lat", "lon", "lng", "latitude", "longitude", "zip", "postal", "postcode", "city", "state",
                        "country", "region", "address", "geo", "zone", "location", "district", "territory", "county",
                        "province", "borough", "coordinate", "place")
    _STAGE_NAME_TOKENS = ("stage", "status", "state", "step", "phase", "funnel", "milestone", "lifecycle", "event",
                          "pipeline")
    _KW_PATTERN_CACHE = {}

    @classmethod
    def _kw_regex(cls, kw):
        pat = cls._KW_PATTERN_CACHE.get(kw)
        if pat is None:
            pat = re.compile(r"(?<![a-z0-9])" + re.escape(kw) + (r"(?![a-z0-9])" if len(kw) <= 4 else ""))
            cls._KW_PATTERN_CACHE[kw] = pat
        return pat

    @classmethod
    def _technique_family(cls, technique) -> str:
        label = " ".join(str(technique if technique is not None else "").lower().split())
        if not label:
            return ""
        if label in cls._GATE_CATCHALL_TECHNIQUES:
            return "catch_all"
        for key, _label, aliases, _methods in cls._TECHNIQUE_FAMILIES:
            if any(cls._kw_regex(kw).search(label) for kw in aliases):
                return key
        for key, _label, _aliases, methods in cls._TECHNIQUE_FAMILIES:
            if any(cls._kw_regex(kw).search(label) for kw in methods):
                return key
        for key, family_label, _aliases, _methods in cls._TECHNIQUE_FAMILIES:
            if label == family_label.lower():
                return key
        return label

    _AI_TEXT_FUNCTION_RE = re.compile(r"\b(ai_classify|ai_extract|ai_analyze_sentiment|ai_summarize|ai_parse_document|"
                                      r"ai_translate|ai_mask|ai_fix_grammar)\b", re.I)
    _MODEL_TRAINING_RE = re.compile(r"\b(train|trains|trained|training|retrain|retrains|fit|fits|fitted|mlflow)\b", re.I)
    _AI_TEXT_GUIDANCE_TECHNIQUE = {"ai_analyze_sentiment": "Sentiment Analysis", "ai_extract": "Extraction"}

    @classmethod
    def _model_training_mentioned(cls, text) -> bool:
        low = str(text or "").lower()
        if cls._MODEL_TRAINING_RE.search(low):
            return True
        return any(TECHNIQUE_EXECUTION_FAMILY.get(label) in INSPIRE_MODEL_FAMILIES + ("E",)
                   and any(not kw.startswith("ai_") and cls._kw_regex(kw).search(low) for kw in methods)
                   for _key, label, _aliases, methods in cls._TECHNIQUE_FAMILIES)

    def _uc_ai_text_function(self, uc: dict, assessment: dict = None):
        uc = uc if isinstance(uc, dict) else {}
        a = assessment if isinstance(assessment, dict) else (
            uc.get("elite_assessment") if isinstance(uc.get("elite_assessment"), dict) else {})
        text = " ".join(str(a.get("build_path") or "").split()) or " ".join(
            " ".join(str(uc.get(field) or "") for field in ("Solution", "High Level Design")).split())
        if not text or self._model_training_mentioned(text):
            return None
        match = self._AI_TEXT_FUNCTION_RE.search(text)
        if match:
            return match.group(1).lower()
        if re.search(r"\bai_query\b", text, re.I):
            names = {str(c).replace("`", "").split(".")[-1].lower(): "" for c in a.get("signal_columns") or []}
            if names and self._column_signals(names)["text_cols"]:
                return "ai_query"
        return None

    def _uc_execution_family(self, uc: dict, context: str = "") -> str:
        uc = uc if isinstance(uc, dict) else {}
        family = _execution_family(uc.get("Analytics Technique"), self.logger, context or f"Use case {uc.get('No', 'UNKNOWN')}")
        if family in INSPIRE_MODEL_FAMILIES and self._uc_ai_text_function(uc):
            return "G"
        return family

    def _uc_guidance_technique(self, uc: dict) -> str:
        technique = _technique_registry_key((uc or {}).get("Analytics Technique"))
        if TECHNIQUE_EXECUTION_FAMILY.get(technique) not in INSPIRE_MODEL_FAMILIES:
            return technique
        function = self._uc_ai_text_function(uc)
        return self._AI_TEXT_GUIDANCE_TECHNIQUE.get(function, "Document Processing") if function else technique

    @classmethod
    def _technique_label(cls, family) -> str:
        for key, label, _aliases, _methods in cls._TECHNIQUE_FAMILIES:
            if key == family:
                return label
        return str(family or "")

    @classmethod
    def _signature_family(cls, family) -> str:
        return "supervised" if family in cls._SUPERVISED_FAMILIES else family

    @classmethod
    def _technique_mentions(cls, text, methods_only=False) -> dict:
        low = str(text or "").lower()
        found = {}
        if not low.strip():
            return found
        for key, _label, aliases, methods in cls._TECHNIQUE_FAMILIES:
            for kw in (methods if methods_only else aliases + methods):
                m = cls._kw_regex(kw).search(low)
                if m and (key not in found or m.start() < found[key]):
                    found[key] = m.start()
        return found

    @staticmethod
    def _iter_schema_rows(full_schema_details):
        for row in full_schema_details or []:
            try:
                if isinstance(row, dict):
                    c = row.get("table_catalog") or row.get("catalog") or ""
                    s = row.get("table_schema") or row.get("schema") or ""
                    t = row.get("table_name") or row.get("table") or ""
                    col = row.get("column_name") or row.get("column") or ""
                    dtype = row.get("data_type") or row.get("type") or ""
                    cmt = row.get("comment") or ""
                elif isinstance(row, (tuple, list)) and len(row) >= 4:
                    c, s, t, col = row[0], row[1], row[2], row[3]
                    dtype = row[4] if len(row) > 4 else ""
                    cmt = row[5] if len(row) > 5 else ""
                else:
                    continue
            except Exception:
                continue
            c, s, t, col = (str(x if x is not None else "").replace("`", "").strip() for x in (c, s, t, col))
            if c and s and t and col:
                yield (c, s, t, col, str(dtype or ""), str(cmt or ""))

    @staticmethod
    def _normalize_table_ref(ref) -> str:
        r = re.sub(r"\s*\.\s*", ".", str(ref if ref is not None else "").replace("`", "").strip())
        if r.lower().startswith("/volumes"):
            return r
        return r.lower()

    @classmethod
    def _uc_table_refs(cls, uc) -> list:
        raw = str((uc or {}).get("Tables Involved") or "")
        out = []
        for part in re.split(r"[,;\n]", raw):
            ref = cls._normalize_table_ref(part)
            if ref and ref not in out:
                out.append(ref)
        return out

    def _honesty(self) -> dict:
        ph = getattr(self, "processing_honesty", None)
        if not isinstance(ph, dict):
            ph = {}
            self.processing_honesty = ph
        return ph

    def _gate_schema_index(self, full_schema_details=None) -> dict:
        cache = getattr(self, "_gate_schema_cache", None)
        if full_schema_details is None:
            if cache is not None:
                return cache
            try:
                sm = getattr(self, "storage_manager", None)
                full_schema_details = sm.load_schema_details("all_columns_for_sql") if sm is not None else []
            except Exception as e:
                self.logger.warning(f"[JOIN_PATH] schema unavailable for gate index: {get_clean_error_message(e)}")
                full_schema_details = []
        rows = full_schema_details or []
        key = (id(rows), len(rows), repr(rows[0])[:200] if rows else None, repr(rows[-1])[:200] if rows else None)
        if cache is not None and cache.get("key") == key:
            return cache
        tables, originals = {}, {}
        for (c, s, t, col, dtype, _cmt) in self._iter_schema_rows(rows):
            fqn = f"{c}.{s}.{t}".lower()
            tables.setdefault(fqn, {})[col.lower()] = dtype.upper()
            originals.setdefault(fqn, (c, s, t))
        index = {"key": key, "tables": tables, "originals": originals, "join_graph": None}
        if tables:
            self._gate_schema_cache = index
        return index

    @classmethod
    def _column_signals(cls, cols) -> dict:
        names = list(cols or {})
        types = {n: str((cols or {}).get(n) or "").upper() for n in names}
        numeric = [n for n in names if types[n].startswith(cls._NUMERIC_TYPE_PREFIXES) or cls._NUMERIC_NAME_RE.search(n)]
        dates = [n for n in names if types[n].startswith(("DATE", "TIMESTAMP")) or cls._DATE_NAME_RE.search(n)]
        return {
            "typed": any(t and not t.startswith(("STRING", "VARCHAR", "CHAR", "TEXT")) for t in types.values()),
            "has_date": bool(dates),
            "has_numeric": bool(numeric),
            "numeric_cols": numeric,
            "id_cols": [n for n in names if _is_key_like_column(n) or "_id_" in n or "_sk_" in n],
            "text_cols": [n for n in names if (not types[n] or types[n].startswith(("STRING", "VARCHAR", "CHAR", "TEXT")))
                          and not _is_key_like_column(n) and any(tok in n for tok in cls._TEXT_NAME_TOKENS)],
            "geo_cols": [n for n in names if any(tok in n for tok in cls._GEO_NAME_TOKENS)],
            "stage_cols": [n for n in names if any(tok in n for tok in cls._STAGE_NAME_TOKENS)],
        }

    @classmethod
    def _grounded_column_mentions(cls, text, col_names) -> set:
        raw = str(text or "")
        low = raw.lower()
        names = {str(c).lower() for c in (col_names or ()) if c}
        found = set()
        if not names or not low.strip():
            return found
        for tok in re.findall(r"`([A-Za-z_][A-Za-z0-9_.]*)`", raw):
            last = tok.rsplit(".", 1)[-1].lower()
            if last in names and tok.upper() not in cls._GATE_SQL_KEYWORDS:
                found.add(last)
        for name in names:
            if len(name) >= 3 and name not in found and re.search(r"(?<![a-z0-9_])" + re.escape(name) + r"(?![a-z0-9_])", low):
                found.add(name)
        return found

    @staticmethod
    def _add_gate_flag(uc, flag):
        flags = uc.get("gate_flags")
        if not isinstance(flags, list):
            flags = []
            uc["gate_flags"] = flags
        if flag not in flags:
            flags.append(flag)

    @staticmethod
    def _gate_reason_key(reason) -> str:
        parts = [p.strip() for p in str(reason or "").split(":")]
        key = parts[0]
        if len(parts) > 1 and re.match(r"R\d+_", parts[1]):
            key += ":" + parts[1]
        return key

    def _record_gate_ledger(self, gate, n_in, n_out, dropped=None, flagged=0) -> dict:
        by_reason = {}
        for item in dropped or []:
            if isinstance(item, dict):
                reason = item.get("reason")
            elif isinstance(item, (tuple, list)) and len(item) > 1:
                reason = item[1]
            else:
                reason = item
            k = self._gate_reason_key(reason)
            by_reason[k] = by_reason.get(k, 0) + 1
        entry = {"in": int(n_in), "out": int(n_out), "dropped": int(n_in) - int(n_out),
                 "by_reason": by_reason, "flagged": int(flagged)}
        self._honesty().setdefault("gate_ledger", {})[gate] = entry
        self.logger.info(f"[GATE_LEDGER] {gate}: in={n_in} out={n_out} dropped={entry['dropped']} "
                         f"by_reason={by_reason} flagged={flagged}")
        return entry

    def _record_gate_exception(self, gate, err, use_cases, mark=True):
        import traceback as _tb
        frames = _tb.extract_tb(err.__traceback__)[-3:] if getattr(err, "__traceback__", None) else []
        where = " <- ".join(f"{f.name}:{f.lineno}" for f in reversed(frames)) or "unknown"
        message = get_clean_error_message(err)
        ucs = use_cases or []
        if mark:
            for uc in ucs:
                tags = uc.get("gate_unverified")
                if not isinstance(tags, list):
                    tags = []
                    uc["gate_unverified"] = tags
                if gate not in tags:
                    tags.append(gate)
        self.logger.warning(
            f"[GATE_UNVERIFIED FIRED] {gate} raised {type(err).__name__}: {message} (at {where}); "
            f"{len(ucs)} UC(s) kept" + (" and marked gate_unverified" if mark else ""))
        self._honesty().setdefault("gate_exceptions", []).append(
            {"gate": gate, "error": f"{type(err).__name__}: {message}", "where": where,
             "ucs": len(ucs), "marked": bool(mark)})
        try:
            NEXT_ACTIONS.add("GATE_UNVERIFIED", severity="HIGH", phase=gate.split("_")[0], step=gate,
                             evidence=f"{gate} raised {type(err).__name__}: {message}; {len(ucs)} UC(s) unverified")
        except Exception:
            pass

    def _uc_dedup_signature(self, uc: dict) -> tuple:
        """Shared duplicate-decision signature for G2, Q2 and Phase L L9:
        (verb family, decision-object tokens, sorted lowercase tables, technique family).
        Synonym and nominalized verbs, plurals and reordered words collide; a different decision
        object, table set or technique family does not."""
        uc = uc if isinstance(uc, dict) else {}
        text = str(uc.get("Name") or uc.get("usecase") or uc.get("use_case") or "").strip()
        if not text:
            text = re.split(r"[.;\n]", str(uc.get("Statement") or ""), maxsplit=1)[0]
        verb, objects = self._decision_object_key(text)
        return (verb, objects,
                tuple(sorted(set(self._uc_table_refs(uc)))),
                self._signature_family(self._technique_family(uc.get("Analytics Technique"))))

    def _decision_object_key(self, text) -> tuple:
        clause = re.split(r"\s+(?:to|so that|in order to)\s+", str(text or "").lower(), maxsplit=1)[0]
        tokens = re.findall(r"[a-z0-9]+", clause)
        verbs = self._DEDUP_VERB_FAMILIES
        verb_idx = 0 if tokens and tokens[0] in verbs else None
        if verb_idx is None:
            nominal = [i for i, tok in enumerate(tokens) if tok in verbs and tok.endswith(("ion", "ing", "sis"))]
            plain = [i for i, tok in enumerate(tokens) if tok in verbs]
            verb_idx = (nominal or plain or [None])[0]
        objects = set()
        for i, tok in enumerate(tokens):
            if i == verb_idx or tok in self._DEDUP_STOPWORDS:
                continue
            if len(tok) > 4 and tok.endswith("ies"):
                tok = tok[:-3] + "y"
            elif len(tok) > 3 and tok.endswith("s") and not tok.endswith("ss"):
                tok = tok[:-1]
            objects.add(tok)
        return (verbs[tokens[verb_idx]] if verb_idx is not None else "", tuple(sorted(objects)))

    _ELITE_ACTION_CLUSTERS = {
        "REDUCE": "MOVE_KPI", "INCREASE": "MOVE_KPI", "OPTIMIZE": "MOVE_KPI",
        "PREDICT": "PREDICT", "DETECT": "DETECT", "RECOMMEND": "RECOMMEND",
        "ALLOCATE": "ALLOCATE", "CLASSIFY": "CLASSIFY", "PRIORITIZE": "PRIORITIZE",
        "ASSESS": "ASSESS", "ANALYZE": "ANALYZE", "TRACK": "TRACK", "SIMULATE": "SIMULATE",
    }
    _ELITE_DECISION_FILLERS = frozenset({
        "which", "what", "how", "when", "who", "whom", "where", "whether", "much", "many",
        "receive", "receives", "enter", "enters", "get", "gets", "take", "takes", "give", "gives",
        "weekly", "monthly", "daily", "quarterly", "annual", "annually", "week", "month", "day",
        "quarter", "year", "cycle", "ahead", "time", "window", "season", "seasonal", "period",
        "next", "upcoming", "current", "high", "low", "top", "bottom", "best", "worst",
    })
    _ELITE_GENERIC_TOKENS = frozenset({
        "revenue", "sales", "cost", "costs", "margin", "margins", "profit", "profits",
        "profitability", "spend", "spending", "price", "prices", "pricing", "value", "rate",
        "rates", "loss", "losses", "roi", "kpi", "performance", "efficiency", "growth", "budget",
        "cash", "dollar", "dollars", "amount", "amounts", "money",
    })

    def _norm_identity_text(self, text) -> str:
        return " ".join(re.findall(r"[a-z0-9]+", str(text if text is not None else "").lower()))

    def _elite_assessment_dict(self, uc: dict) -> dict:
        a = (uc or {}).get("elite_assessment")
        return a if isinstance(a, dict) else {}

    def _elite_table_tokens(self, uc: dict) -> set:
        a = self._elite_assessment_dict(uc)
        refs = list(self._uc_table_refs(uc))
        for extra in (a.get("tables_involved"), a.get("primary_table"), uc.get("Primary Table")):
            for part in re.split(r"[,;\n]", str(extra if extra is not None else "")):
                ref = self._normalize_table_ref(part)
                if ref and ref not in refs:
                    refs.append(ref)
        tokens = set()
        for ref in refs:
            name = ref.split(".")[-1]
            for tok in re.split(r"[^a-z0-9]+", name.lower()):
                if len(tok) >= 2:
                    tokens.add(tok)
        return tokens

    def _elite_object_tokens(self, text, table_tokens) -> set:
        out = set()
        for tok in re.findall(r"[a-z0-9]+", str(text if text is not None else "").lower()):
            if len(tok) > 4 and tok.endswith("ies"):
                tok = tok[:-3] + "y"
            elif len(tok) > 3 and tok.endswith("s") and not tok.endswith("ss"):
                tok = tok[:-1]
            if (len(tok) < 3 or tok in self._DEDUP_STOPWORDS or tok in self._ELITE_DECISION_FILLERS
                    or tok in self._ELITE_GENERIC_TOKENS or tok in self._DEDUP_VERB_FAMILIES
                    or tok in table_tokens):
                continue
            out.add(tok)
        return out

    def _elite_decision_identity(self, uc: dict) -> dict:
        a = self._elite_assessment_dict(uc)
        title = str(uc.get("usecase") or uc.get("use_case") or uc.get("Name") or "")
        verb, _objects = self._decision_object_key(title)
        tables = self._elite_table_tokens(uc)
        return {
            "owner": self._norm_identity_text(a.get("decision_owner")),
            "action": self._ELITE_ACTION_CLUSTERS.get(verb, ""),
            "title": self._norm_identity_text(title),
            "objects": self._elite_object_tokens(title + " " + str(a.get("decision") or ""), tables),
            "kpi": self._elite_object_tokens(a.get("kpi"), tables),
            "technique": self._technique_family(uc.get("Analytics Technique")),
            "primary_table": self._norm_identity_text(self._uc_primary_table(uc)),
        }

    def _are_elite_decision_twins(self, a: dict, b: dict) -> bool:
        if not a["owner"] or a["owner"] != b["owner"]:
            return False
        if not a["action"] or a["action"] != b["action"]:
            return False
        if a["primary_table"] == b["primary_table"] and a["technique"] == b["technique"]:
            return False
        if not (a["objects"] & b["objects"]):
            return False
        if a["kpi"] and b["kpi"] and not (a["kpi"] & b["kpi"]):
            return False
        return True

    def _elite_twin_keys(self, uc: dict) -> set:
        keys = set()
        family = self._signature_family(self._technique_family(uc.get("Analytics Technique")))
        title = " ".join(re.findall(r"[a-z0-9]+", str(uc.get("usecase") or uc.get("use_case") or "").lower()))
        if title:
            keys.add(("title", title))
        for text in (uc.get("usecase") or uc.get("use_case"), uc.get("Name")):
            verb, objects = self._decision_object_key(text)
            if verb and objects:
                keys.add(("decision", verb, objects, family))
        return keys

    @staticmethod
    def _uc_title(uc) -> str:
        uc = uc if isinstance(uc, dict) else {}
        return str(uc.get("Name") or uc.get("usecase") or uc.get("use_case") or "").strip()

    def _collapse_elite_decision_twins(self, elite: list) -> tuple:
        n = len(elite)
        parent = list(range(n))

        def _root(i):
            while parent[i] != i:
                parent[i] = parent[parent[i]]
                i = parent[i]
            return i

        def _union(i, j):
            parent[_root(i)] = _root(j)

        owner = {}
        for idx, uc in enumerate(elite):
            for key in self._elite_twin_keys(uc):
                if key in owner:
                    _union(idx, owner[key])
                else:
                    owner[key] = idx
        ids = [self._elite_decision_identity(uc) for uc in elite]
        for i in range(n):
            for j in range(i + 1, n):
                if _root(i) != _root(j) and self._are_elite_decision_twins(ids[i], ids[j]):
                    _union(i, j)
        groups = {}
        for idx in range(n):
            groups.setdefault(_root(idx), []).append(idx)
        kept, twins = [], {}
        for members in groups.values():
            ranked = sorted(members, key=lambda i: (self._final_rank_key(elite[i]), i))
            winner = elite[ranked[0]]
            kept.append(ranked[0])
            collapsed = winner.setdefault("decision_twins_collapsed", [])
            for loser_idx in ranked[1:]:
                loser = elite[loser_idx]
                twins[id(loser)] = winner
                collapsed.append({"No": loser.get("No", "UNKNOWN"), "Name": self._uc_title(loser),
                                  "elite_score": loser.get("elite_score")})
        kept_set = set(kept)
        return [uc for idx, uc in enumerate(elite) if idx in kept_set], twins

    def _dedup_by_signature(self, use_cases: list, reason: str, rank=None) -> tuple:
        if not use_cases:
            return use_cases, []

        def _evidence_rank(uc):
            design = str(uc.get("Solution") or "") + " " + str(uc.get("High Level Design") or "")
            return (self._score_rank_part(uc.get("Inspire Score"))
                    + (-len(set(re.findall(r"`([A-Za-z_][A-Za-z0-9_.]*)`", design))),))

        rank_key = rank or _evidence_rank
        groups = {}
        for idx, uc in enumerate(use_cases):
            groups.setdefault(self._uc_dedup_signature(uc), []).append((idx, uc))
        winners, merge_log = set(), []
        for sig, members in groups.items():
            ranked = sorted(members, key=lambda item: (rank_key(item[1]), item[0]))
            winners.add(ranked[0][0])
            for _idx, loser in ranked[1:]:
                merge_log.append({"kept": ranked[0][1].get("No", "UNK"), "dropped": loser.get("No", "UNK"),
                                  "reason": reason, "tables": list(sig[2]), "technique": sig[3]})
        return [uc for idx, uc in enumerate(use_cases) if idx in winners], merge_log

    @staticmethod
    def _is_join_identifier(col) -> bool:
        c = str(col or "").strip().lower()
        return bool(c) and c not in _GENERIC_KEY_NAMES and _is_key_like_column(c)

    def _join_graph(self) -> dict:
        index = self._gate_schema_index()
        loader = getattr(self, "data_loader", None)
        universe = set(index["originals"].values())
        for tup in (getattr(loader, "all_table_tuples", None) or []):
            try:
                c, s, t = tuple(tup)[:3]
                universe.add((str(c), str(s), str(t)))
            except Exception:
                continue
        relations = []
        if loader is not None and hasattr(loader, "get_foreign_key_relations"):
            try:
                relations = list(loader.get_foreign_key_relations(universe) or [])
            except Exception as e:
                self.logger.warning(f"[JOIN_PATH] foreign keys unavailable ({get_clean_error_message(e)}); "
                                    f"join paths use shared identifier columns only")
        fingerprint = (id(loader), len(relations), len(universe), len(index["tables"]))
        cached = index.get("join_graph")
        if cached is not None and cached["fingerprint"] == fingerprint:
            return cached
        provenance = {}
        if relations and hasattr(loader, "get_table_relations"):
            for source_table in sorted({tuple(str(x) for x in tuple(rel)[:3]) for rel in relations}):
                try:
                    for rel, source, _confidence in loader.get_table_relations(*source_table) or []:
                        provenance[tuple(str(x).lower() for x in tuple(rel)[:8])] = source
                except Exception as e:
                    self.logger.debug(f"[JOIN_PATH] FK provenance unavailable for {'.'.join(source_table)}: "
                                      f"{get_clean_error_message(e)}")
        fk = {"declared": {}, "inferred": {}}
        for rel in relations:
            try:
                rel = tuple(rel)
                src = f"{rel[0]}.{rel[1]}.{rel[2]}".lower()
                ref = f"{rel[4]}.{rel[5]}.{rel[6]}".lower()
                edge = f"{src}.{str(rel[3]).lower()} -> {ref}.{str(rel[7]).lower()}"
                rel_key = tuple(str(x).lower() for x in rel[:8])
            except Exception:
                continue
            if src == ref:
                continue
            tier = "declared" if provenance.get(rel_key) == "declared" else "inferred"
            fk[tier].setdefault(src, []).append((ref, edge))
            fk[tier].setdefault(ref, []).append((src, edge))
        id_cols = {}
        for fqn, cols in index["tables"].items():
            for col in cols:
                if self._is_join_identifier(col):
                    id_cols.setdefault(col, set()).add(fqn)
        limit = max(5, len(index["tables"]) // 2)
        shared = {col: tabs for col, tabs in id_cols.items() if 2 <= len(tabs) <= limit}
        shared_by_table = {}
        for col in sorted(shared):
            for fqn in shared[col]:
                shared_by_table.setdefault(fqn, []).append(col)
        graph = {"fingerprint": fingerprint, "fk": fk, "shared": shared, "shared_by_table": shared_by_table,
                 "allowed": set(index["tables"]) | {".".join(u).lower() for u in universe}}
        index["join_graph"] = graph
        return graph

    @staticmethod
    def _join_path_bfs(graph, refs, tier, max_hops=4):
        from collections import deque
        root, targets = refs[0], set(refs[1:])
        fk_tiers = ("declared",) if tier == "declared" else ("declared", "inferred")
        parents, depth = {root: None}, {root: 0}
        queue = deque([root])
        while queue:
            node = queue.popleft()
            if depth[node] >= max_hops:
                continue
            nbrs = []
            for t in fk_tiers:
                nbrs.extend(graph["fk"][t].get(node, ()))
            if tier == "shared_column":
                for col in graph["shared_by_table"].get(node, ()):
                    for other in sorted(graph["shared"][col]):
                        if other != node:
                            nbrs.append((other, f"{node}.{col} = {other}.{col}"))
            for nbr, edge in nbrs:
                if nbr in parents or (nbr not in graph["allowed"] and nbr not in targets):
                    continue
                parents[nbr] = (node, edge)
                depth[nbr] = depth[node] + 1
                queue.append(nbr)
        if not targets.issubset(parents):
            return None
        path = []
        for target in refs[1:]:
            chain, cur = [], target
            while parents[cur] is not None:
                prev, edge = parents[cur]
                chain.append(edge)
                cur = prev
            for edge in reversed(chain):
                if edge not in path:
                    path.append(edge)
        return path

    @staticmethod
    def _join_path_tables(path) -> list:
        tables = []
        for edge in path or []:
            for side in re.split(r"\s*(?:->|=)\s*", str(edge)):
                table = side.strip().rsplit(".", 1)[0].lower()
                if table.count(".") == 2 and table not in tables:
                    tables.append(table)
        return tables

    def _find_join_path(self, tables: list) -> dict:
        """Join-path contract for Q3 R4 and elite verification. BFS over
        `data_loader.get_foreign_key_relations` for every table in the schema universe, through
        intermediate tables present in the schema (max 4 hops per table), preferring declared
        edges, then naming-inferred edges, then identical identifier-like columns shared by two
        tables. Returns {"connected", "path", "strength"}; strength is "declared" | "inferred" |
        "shared_column" | "none". Declared vs inferred is the provenance
        `data_loader.get_table_relations` records; an edge without recorded provenance counts as
        inferred. Safe without a data loader."""
        refs = []
        for t in tables or []:
            ref = self._normalize_table_ref(t)
            if ref and not ref.lower().startswith("/volumes") and ref not in refs:
                refs.append(ref)
        if len(refs) < 2:
            return {"connected": True, "path": [], "strength": "none"}
        graph = self._join_graph()
        for tier in ("declared", "inferred", "shared_column"):
            path = self._join_path_bfs(graph, refs, tier)
            if path is not None:
                return {"connected": True, "path": path, "strength": tier}
        return {"connected": False, "path": [], "strength": "none"}

    def _phaseG_validate_schema_completeness(self, use_cases: list) -> list:
        """Phase G2: drop UCs missing any required field."""
        required = ("Name", "Statement", "Solution", "Tables Involved", "Analytics Technique")
        kept = []
        dropped = 0
        for uc in use_cases:
            missing = [f for f in required if not str(uc.get(f) or "").strip()]
            if missing:
                dropped += 1
                self.logger.info(f"G schema-completeness: dropping UC {uc.get('No','UNK')} — empty {missing}")
                try:
                    NEXT_ACTIONS.add("UC_SCHEMA_INCOMPLETE", severity="HIGH",
                                     phase="G2", step="validate_schema_completeness",
                                     evidence=f"UC {uc.get('No','UNK')} empty: {missing}")
                except Exception:
                    pass
                continue
            kept.append(uc)
        if dropped:
            self.logger.warning(f"G2 schema-completeness: dropped {dropped} incomplete UCs")
        return kept

    def _phaseG_validate_name_uniqueness(self, use_cases: list) -> list:
        """Phase G2: drop exact decision duplicates via the shared `_uc_dedup_signature`; the same
        name on different tables or with a different technique family is not a duplicate."""
        if not use_cases:
            return use_cases
        kept, merged = self._dedup_by_signature(use_cases, "G2_DUPLICATE_DECISION")
        if merged:
            self.logger.info(f"G2 name-uniqueness: dropped {len(merged)} duplicate decision(s): "
                             + ", ".join(f"{m['dropped']}->{m['kept']}" for m in merged[:10]))
        return kept

    def _phaseG_validate_no_catchall_technique(self, use_cases: list) -> list:
        """Phase G2 / Rule 12: a catch-all technique label is replaced by the technique family whose
        concrete method the UC's own Solution / High Level Design names; UCs that name no concrete
        method are dropped."""
        kept, dropped, relabelled = [], 0, 0
        order = [key for key, _l, _a, _m in self._TECHNIQUE_FAMILIES]
        for uc in use_cases:
            t = " ".join(str(uc.get("Analytics Technique") or "").lower().split())
            if t not in self._GATE_CATCHALL_TECHNIQUES:
                kept.append(uc)
                continue
            design = str(uc.get("Solution") or "") + "\n" + str(uc.get("High Level Design") or "")
            found = self._technique_mentions(design, methods_only=True)
            if found:
                family = min(found, key=lambda k: (found[k], order.index(k)))
                uc["Analytics Technique"] = self._technique_label(family)
                relabelled += 1
                try:
                    NEXT_ACTIONS.add("UC_CATCHALL_TECHNIQUE", severity="INFO",
                                     phase="G2", step="validate_no_catchall_technique",
                                     evidence=f"UC {uc.get('No','UNK')} technique '{t}' relabelled to "
                                              f"'{uc['Analytics Technique']}' from its own design text")
                except Exception:
                    pass
                kept.append(uc)
                continue
            dropped += 1
            self.logger.info(f"G2 catch-all: dropping UC {uc.get('No','UNK')} with technique='{t}' and no concrete method")
            try:
                NEXT_ACTIONS.add("UC_CATCHALL_TECHNIQUE", severity="HIGH",
                                 phase="G2", step="validate_no_catchall_technique",
                                 evidence=f"UC {uc.get('No','UNK')} technique={t}; design names no concrete method")
            except Exception:
                pass
        if relabelled:
            self.logger.info(f"G2 catch-all: relabelled {relabelled} UC(s) from the method their design names")
        if dropped:
            self.logger.warning(f"G2 catch-all: dropped {dropped} UCs with umbrella technique names")
        return kept

    @staticmethod
    def _known_table_names(index) -> set:
        return set(index["tables"]) | {fqn.rsplit(".", 1)[-1] for fqn in index["tables"]}

    def _ungrounded_identifiers(self, text, columns, known_tables) -> list:
        ungrounded = []
        for name in sorted(set(re.findall(r"`([A-Za-z_][A-Za-z0-9_.]*)`", str(text or "")))):
            low = name.lower()
            if low in columns or name.upper() in self._GATE_SQL_KEYWORDS or low in known_tables:
                continue
            if "." in low and low.rsplit(".", 1)[-1] in (known_tables | columns):
                continue
            ungrounded.append(name)
        return ungrounded

    def _phaseG_validate_column_grounding(self, use_cases: list, full_schema_details: list) -> list:
        """Phase G2 / B10, advisory in every mode: backticked identifiers in Solution, Statement and
        Business Value that are neither a column of a cited table, a known table name nor a SQL
        keyword are reported to NEXT_ACTIONS as UC_COLUMN_UNGROUNDED. Never drops a UC."""
        if not use_cases or not full_schema_details:
            return use_cases
        index = self._gate_schema_index(full_schema_details)
        known_tables = self._known_table_names(index)
        for uc in use_cases:
            allowed = set()
            for t in self._uc_table_refs(uc):
                allowed |= set(index["tables"].get(t, {}))
            if not allowed:
                continue
            blob = " ".join(str(uc.get(f) or "") for f in ("Solution", "Statement", "Business Value"))
            ungrounded = self._ungrounded_identifiers(blob, allowed, known_tables)
            if ungrounded:
                try:
                    NEXT_ACTIONS.add("UC_COLUMN_UNGROUNDED", severity="HIGH",
                                     phase="G2", step="validate_column_grounding",
                                     evidence=f"UC {uc.get('No','UNK')}: columns not in schema: {ungrounded[:5]}")
                except Exception:
                    pass
        return use_cases

    def _phaseE0_catalog_drift_check(self, session_id):
        """Phase E0 (Gap 4): at Generate entry, flag tracked UCs whose cited tables no longer exist.
        Metadata only: one parameterized information_schema query per referenced catalog. Tables in
        a catalog whose information_schema cannot be read are unverifiable, not drifted."""
        if not self._tracking_table_name:
            return []
        sid = int(session_id)
        try:
            uc_rows = execute_sql(
                self.spark,
                f"SELECT id, use_case, tables_involved FROM {self._tracking_table_name} WHERE session_id = {sid}",
                self.logger)
        except Exception as e:
            self.logger.warning(f"E0 drift check: failed to read UCs: {get_clean_error_message(e)}")
            return []
        refs_by_uc, wanted = [], {}
        for r in uc_rows:
            refs = [t for t in self._uc_table_refs({"Tables Involved": getattr(r, "tables_involved", "") or ""})
                    if t.count(".") == 2 and not t.lower().startswith("/volumes")]
            refs_by_uc.append((r, refs))
            for t in refs:
                cat, sch, _tbl = t.split(".")
                wanted.setdefault(cat, set()).add(sch)
        existing = {}
        for cat in sorted(wanted):
            if "`" in cat:
                existing[cat] = None
                continue
            schemas = sorted(wanted[cat])
            args = {f"s{i}": s for i, s in enumerate(schemas)}
            markers = ", ".join(f":s{i}" for i in range(len(schemas)))
            try:
                rows = execute_sql(
                    self.spark,
                    f"SELECT table_schema, table_name FROM {quote_identifier(cat)}.information_schema.tables "
                    f"WHERE table_schema IN ({markers})",
                    self.logger, args=args)
                existing[cat] = {f"{str(x['table_schema']).lower()}.{str(x['table_name']).lower()}" for x in rows}
            except Exception as e:
                existing[cat] = None
                self.logger.warning(f"E0 drift check: cannot read {cat}.information_schema "
                                    f"({get_clean_error_message(e)}); its tables are unverifiable, not reported as drift")
        drifted, unverifiable = [], 0
        for r, refs in refs_by_uc:
            missing = []
            for t in refs:
                cat, sch, tbl = t.split(".")
                known = existing.get(cat)
                if known is None:
                    unverifiable += 1
                elif f"{sch}.{tbl}" not in known:
                    missing.append(t)
            if missing:
                drifted.append({"id": r.id, "table": missing[0], "missing": missing})
                try:
                    NEXT_ACTIONS.add(
                        "UC_STALE_CATALOG_DRIFT", severity="HIGH",
                        phase="E0", step="catalog_drift_check",
                        evidence=f"UC {r.id} references {', '.join(missing[:5])} which is no longer in schema",
                        suggested_user_action="Rerun Discover to refresh UCs."
                    )
                except Exception:
                    pass
        if drifted:
            self.logger.warning(f"E0 CATALOG DRIFT: {len(drifted)} UC(s) reference tables no longer in schema"
                                + (f"; {unverifiable} table reference(s) unverifiable" if unverifiable else ""))
        elif unverifiable:
            self.logger.warning(f"E0 CATALOG DRIFT: no drift among verifiable tables; "
                                f"{unverifiable} table reference(s) unverifiable")
        else:
            self.logger.info("E0 CATALOG DRIFT: no drift detected")
        return drifted

    def _gate_ledger_drops(self, prefix: str) -> dict:
        ledger = self._honesty().get("gate_ledger") or {}
        return {gate: int(entry.get("dropped") or 0) for gate, entry in ledger.items()
                if str(gate).startswith(prefix) and isinstance(entry, dict) and entry.get("dropped")}

    def _phaseG_run_quality_gates(self, use_cases: list, full_schema_details: list) -> list:
        """Run G2 deterministic quality gates. Idempotent. A gate that raises is logged with a short
        traceback and the UCs it should have checked are kept but marked `gate_unverified`."""
        if not use_cases:
            return use_cases
        before = len(use_cases)
        steps = (
            ("G2_schema_completeness", lambda ucs: self._phaseG_validate_schema_completeness(ucs)),
            ("G2_name_uniqueness", lambda ucs: self._phaseG_validate_name_uniqueness(ucs)),
            ("G2_catchall_technique", lambda ucs: self._phaseG_validate_no_catchall_technique(ucs)),
            ("G2_column_grounding", lambda ucs: self._phaseG_validate_column_grounding(ucs, full_schema_details)),
        )
        for gate, run_gate in steps:
            n_in = len(use_cases)
            try:
                use_cases = run_gate(use_cases)
            except Exception as gate_err:
                self._record_gate_exception(gate, gate_err, use_cases)
                continue
            self._record_gate_ledger(gate, n_in, len(use_cases))
        self.logger.info(f"Phase G gates: {before} -> {len(use_cases)} use cases")
        return use_cases

    def _phaseQ1_validate_grounding(self, use_cases: list, full_schema_details: list) -> tuple:
        """Phase Q1 GROUNDING (deterministic, metadata only). Returns (kept, dropped).

        Every table in Tables Involved must resolve to the schema universe: the schema rows, the
        business tables and every discovered table (technical and reference tables stay joinable).
        Backticks, separators and case are normalized, and a schema.table or bare table name that
        resolves to exactly one table is rewritten to its fully qualified name. Fails closed: with
        no table universe at all, every UC is dropped as unverifiable and none is reported grounded.
        """
        if not use_cases:
            return use_cases, []
        available = set()
        for (c, s, t, _col, _dtype, _cmt) in self._iter_schema_rows(full_schema_details):
            available.add(f"{c}.{s}.{t}".lower())
        for t in (getattr(self, "global_table_names", None) or ()):
            ref = self._normalize_table_ref(t)
            if ref.count(".") == 2:
                available.add(ref)
        # Extend with the FULL discovered table universe (includes tables
        # filtered as TECHNICAL / not-elected for UC generation — such tables
        # are NOT forgotten; they remain joinable/referenceable). A UC that
        # joins against a reference/technical table is valid as long as that
        # table actually exists in the passed schema. Per user 2026-04-19.
        for tup in (getattr(getattr(self, "data_loader", None), "all_table_tuples", None) or []):
            try:
                available.add(".".join(str(x) for x in tuple(tup)[:3]).lower())
            except Exception:
                continue
        if not available:
            self.logger.error(
                f"[Q1 FAIL_CLOSED FIRED] grounding cannot be checked: the schema universe has 0 tables; "
                f"{len(use_cases)} UC(s) dropped as unverifiable, none reported grounded")
            self._honesty()["q1_grounding_unchecked"] = True
            try:
                NEXT_ACTIONS.add("UC_NOT_GROUNDED", severity="BLOCKING", phase="Q1", step="fail_closed",
                                 evidence=f"No schema available to ground {len(use_cases)} UC(s)",
                                 suggested_user_action="Check catalog permissions and table selection, then rerun.")
            except Exception:
                pass
            return [], [(uc, "Q1_UNCHECKABLE:NO_SCHEMA") for uc in use_cases]
        by_suffix = {}
        for fqn in available:
            parts = fqn.split(".")
            if len(parts) == 3:
                by_suffix.setdefault(f"{parts[1]}.{parts[2]}", set()).add(fqn)
                by_suffix.setdefault(parts[2], set()).add(fqn)
        kept, dropped = [], []
        for uc in use_cases:
            refs = self._uc_table_refs(uc)
            if not refs:
                dropped.append((uc, "EMPTY_TABLES_INVOLVED"))
                continue
            resolved, missing = [], []
            for ref in refs:
                # Volume paths (ai_parse_document use cases) pass through
                if ref.lower().startswith("/volumes"):
                    resolved.append(ref)
                elif ref in available:
                    resolved.append(ref)
                elif ref.count(".") < 2 and len(by_suffix.get(ref, ())) == 1:
                    resolved.append(next(iter(by_suffix[ref])))
                else:
                    missing.append(ref)
            if missing:
                dropped.append((uc, f"UC_NOT_GROUNDED: missing {missing[:3]}"))
                continue
            raw_parts = [p.strip().lower() for p in str(uc.get("Tables Involved") or "").split(",") if p.strip()]
            if raw_parts != [r.lower() for r in resolved]:
                uc["Tables Involved"] = ", ".join(resolved)
            kept.append(uc)
        if dropped:
            self.logger.warning(
                f"Q1 GROUNDING: dropped {len(dropped)} of {len(use_cases)} use cases "
                f"with ungrounded tables"
            )
            for uc, reason in dropped[:5]:
                self.logger.info(f"  dropped {uc.get('No', 'UNK')}: {reason}")
        else:
            self.logger.info(
                f"Q1 GROUNDING: all {len(use_cases)} use cases grounded"
            )
        return kept, dropped

    def _phaseQ2_canonical_dedup(self, use_cases: list) -> tuple:
        """Phase Q2 canonical dedup (deterministic). UCs that share `_uc_dedup_signature` are
        reduced to one twin: the higher score when one exists, then more backticked design
        evidence, then generation order. Returns (kept, merge_log)."""
        if not use_cases:
            return use_cases, []
        kept, merge_log = self._dedup_by_signature(use_cases, "EXACT_CANONICAL_DUPLICATE")
        if merge_log:
            rate = len(merge_log) / max(1, len(use_cases))
            level = "warning" if rate > 0.05 else "info"
            getattr(self.logger, level)(
                f"Q2 DEDUP: dropped {len(merge_log)} of {len(use_cases)} "
                f"as canonical duplicates (rate {rate*100:.1f}%)"
            )
            if rate > 0.05:
                self.logger.warning(
                    "Q2 DEDUP_RATE_HIGH (>5%) — indicates prompt regression; "
                    "investigate which technique/tables cluster is over-represented"
                )
        else:
            self.logger.info(
                f"Q2 DEDUP: all {len(use_cases)} use cases have distinct signatures"
            )
        return kept, merge_log

    def _q3_deep_evidence_reason(self, uc: dict, tables: list, family: str, design_text: str, cols: dict):
        evidence = getattr(self, "discovery_evidence", None) or {}
        profiles = evidence.get("profiles") or {}
        if not profiles or not tables:
            return None
        rt = TECHNICAL_CONTEXT["runtime"]
        primary = profiles.get(str(tables[0]).replace("`", "").lower())
        if primary is not None and primary.get("rows") == 0:
            return f"Q3_R20_EMPTY_PRIMARY_TABLE:{tables[0]}"
        design_low = str(design_text or "").lower()
        table_set = {str(t).replace("`", "").lower() for t in tables}
        for match in evidence.get("join_matches") or []:
            if match.get("rate") is None or match["rate"] >= rt["deep_min_join_match"]:
                continue
            if match["child_table"] in table_set and match["parent_table"] in table_set \
                    and match["child_column"].lower() in design_low and match["parent_column"].lower() in design_low:
                return (f"Q3_R21_JOIN_MATCH_TOO_LOW:{match['child_table']}.{match['child_column']}->"
                        f"{match['parent_table']}.{match['parent_column']}={match['rate']}")
        grounded = [str(c).lower() for c in (self._grounded_column_mentions(design_text, cols) or [])]
        shares, filled = {}, {}
        for table in table_set:
            profile = profiles.get(table) or {}
            for column, info in (profile.get("columns") or {}).items():
                if column.lower() in grounded and info.get("null_share") is not None:
                    shares[column.lower()] = max(shares.get(column.lower(), 0.0), info["null_share"])
                    rows_filled = int(round((profile.get("rows") or 0) * (1.0 - info["null_share"])))
                    filled[column.lower()] = max(filled.get(column.lower(), 0), rows_filled)
        empty = sorted(c for c, n in filled.items() if n < rt["deep_min_signal_rows"])
        if filled and len(empty) == len(filled):
            return f"Q3_R22_KEY_SIGNAL_EMPTY:{','.join(f'{c}={filled[c]}' for c in empty[:5])}"
        for column in sorted(c for c, share in shares.items() if share >= rt["deep_mostly_null_share"]):
            self._add_gate_flag(uc, f"Q3_R22_MOSTLY_NULL_SIGNAL:{column}")
        if family == "classification":
            title_stems = {token[:5] for token in re.split(r"[^a-z0-9]+", str(uc.get("Name") or "").lower()) if len(token) >= 3}
            labels, constant = {}, set()
            for table in table_set:
                for column, info in ((profiles.get(table) or {}).get("columns") or {}).items():
                    low = column.lower()
                    if low not in grounded or info.get("type_family") not in ("string", "boolean") or info.get("distinct") is None:
                        continue
                    if info["distinct"] <= 1:
                        constant.add(low)
                    stems = {token[:5] for token in re.split(r"[^a-z0-9]+", low) if len(token) >= 3}
                    if stems and stems <= title_stems:
                        labels[low] = max(labels.get(low, 0), info["distinct"])
            if labels and all(count <= 1 for count in labels.values()):
                return f"Q3_R24_SINGLE_CLASS_LABEL:{','.join(sorted(labels)[:5])}"
            for column in sorted(constant):
                self._add_gate_flag(uc, f"Q3_R24_CONSTANT_COLUMN:{column}")
        if family in self._TEMPORAL_FAMILIES and primary is not None:
            spans = []
            for info in (primary.get("columns") or {}).values():
                lo, hi = info.get("min"), info.get("max")
                if lo and hi:
                    try:
                        spans.append((datetime.datetime.fromisoformat(str(hi)[:19]) -
                                      datetime.datetime.fromisoformat(str(lo)[:19])).days)
                    except ValueError:
                        continue
            if spans and max(spans) < 60:
                self._add_gate_flag(uc, f"Q3_R23_THIN_TIME_COVERAGE:{max(spans)}_days")
        return None

    def _phaseQ3_validate_relevance(self, use_cases: list, full_schema_details: list) -> tuple:
        """Phase Q3 RELEVANCE (deterministic; metadata, plus aggregate evidence when Discovery Type
        is Deep). Returns (kept, dropped).

        Deep only: R20 drops an empty primary table, R21 drops a cited join whose measured match
        rate is below deep_min_join_match, R22 drops when every grounded signal column has fewer
        than deep_min_signal_rows non-null rows and flags mostly-null signal columns, R23 flags
        temporal analysis on less than 60 days of data, R24 drops a classification whose label
        columns (categorical columns named in the title) hold a single value and flags other
        constant categorical columns.

        Drops: R1 only technical tables (the agent's own table classification), R2 technique
        structurally unsupported by the cited tables' columns (numeric needs are waived when the
        cited columns carry no informative types), R4 no join path (`_find_join_path`), R7 three or
        more invented figures in Business Value, R9 invented data inputs outside simulation, R11
        cohorts formed on a biographical attribute, R13 no real column named in Solution / High
        Level Design or a single one beside invented columns, R14 technique-specific structural
        prerequisites.
        Flags (`gate_flags`): R5 temporal analysis of one reference table, R8 target listed among
        its own inputs, R9 assumed configuration parameter, R10 row-hungry technique on one
        reference table, R13 single real column. R6 is NEXT_ACTIONS only.
        """
        if not use_cases:
            return use_cases, []
        index = self._gate_schema_index(full_schema_details)
        cols_by = index["tables"]
        cat_map = {self._normalize_table_ref(k): str(v or "").upper()
                   for k, v in (getattr(self, "data_category_map", None) or {}).items()}
        business = {self._normalize_table_ref(t) for t in (getattr(self, "global_table_names", None) or ())}
        numeric_claim_re = NUMERIC_CLAIM_RE
        invented_input_re = re.compile(
            r"\b(simulated|assumed|synthetic|hypothetical|placeholder|made[-\s]?up|"
            r"fictitious|imaginary|invented|fabricated|mock)\s+([a-z][a-z_-]{2,30})\b")
        invented_nouns = {"capacity", "cost", "volume", "demand", "revenue", "price", "budget", "rate",
                          "ground", "truth", "label", "target", "score"}
        config_nouns = {"constraint", "threshold", "weight", "parameter", "baseline", "benchmark"}
        biographic = r"(?:`?\w*birth\w*`?|dob|date of birth|ages?\b|age_\w+)"
        biographic_anchor_res = (
            re.compile(r"\bcohorts?\s+(?:\w+\s+){0,2}?(?:by|on|based on|formed by|defined by|grouped by|using|from)\s+"
                       r"(?:\S+\s+){0,2}?" + biographic),
            re.compile(r"\b" + biographic + r"[^.]{0,20}\bcohorts?\b"),
        )
        event_anchor_re = re.compile(r"\b(?:signup|sign-up|sign_up|acquisition|acquired|first|onboard|created|creation|"
                                     r"registration|registered|join|enrol|start|activation|activated|install|"
                                     r"subscri|opened)")
        feature_cue_re = re.compile(r"\b(?:features?|inputs?|predictors?|using|based on|from|with)\b")
        subject_patterns = (
            r"\bper[\s-]+([a-z][a-z_-]{2,20})\b",
            r"\bper\s+each\s+([a-z][a-z_-]{2,20})\b",
            r"\btargeted\s+([a-z][a-z_-]{2,20})\b",
            r"\beach\s+([a-z][a-z_-]{2,20})\s+(incentive|targeting|scheduling|assignment|prioritization|routing|intervention)\b",
            r"\b([a-z][a-z_-]{2,20})\s+(incentive|targeting|scheduling|assignment|prioritization|routing|intervention)\s+(plan|strategy|recommendation|program)\b",
        )
        rule_ids = {"R5": "UC_DATA_FIT_WEAK", "R6": "UC_ACTION_SUBJECT_SUSPECT", "R7": "UC_BV_UNGROUNDED_NUMBER",
                    "R10": "UC_DATA_FIT_WEAK", "R13": "UC_SOLUTION_UNDER_GROUNDED", "R14": "UC_DATA_FIT_WEAK"}

        def _note(rule, severity, uc_no, evidence):
            try:
                NEXT_ACTIONS.add(rule_ids.get(rule, "UC_NOT_RELEVANT"), severity=severity, phase="Q3", step=rule,
                                 evidence=f"UC {uc_no}: {evidence}")
            except Exception:
                pass

        def _is_technical(fqn):
            category = cat_map.get(fqn)
            if category:
                return category == "TECHNICAL"
            if fqn in business:
                return False
            return bool(set(re.split(r"[^a-z0-9]+", fqn.rsplit(".", 1)[-1])) & self._TECHNICAL_TABLE_TOKENS)

        def _subject_has_id_column(subject, col_names):
            s = (subject or "").strip().lower()
            if not s:
                return True
            sn = s.rstrip("s")
            for c in col_names:
                if s in c or sn in c:
                    if _is_key_like_column(c) or c in (s, sn):
                        return True
            return False

        kept, dropped = [], []
        for uc in use_cases:
            uc_no = uc.get("No", "UNK")
            refs = self._uc_table_refs(uc)
            tabs = [t for t in refs if not t.lower().startswith("/volumes")]
            if not tabs:
                kept.append(uc)
                continue
            cols = {}
            for t in tabs:
                cols.update(cols_by.get(t, {}))
            sig = self._column_signals(cols)
            tech = str(uc.get("Analytics Technique") or "").strip()
            family = self._technique_family(tech)
            sol = str(uc.get("Solution") or "")
            hld = str(uc.get("High Level Design") or uc.get("high_level_design") or "")
            stmt = str(uc.get("Statement") or "")
            bv = str(uc.get("Business Value") or "")
            sol_low = sol.lower()
            reason = None
            clusters_text = family == "clustering" and bool(sig["text_cols"]) and (
                "text_topic" in self._technique_mentions(sol + "\n" + hld)
                or bool(self._grounded_column_mentions(sol + "\n" + hld, sig["text_cols"])))

            if all(_is_technical(t) for t in tabs):
                reason = "UC_NOT_RELEVANT:R1_TECHNICAL_ONLY"

            if reason is None:
                numeric_ok = sig["has_numeric"] or not sig["typed"]
                supported = {
                    "forecasting": sig["has_date"] and numeric_ok,
                    "trend": sig["has_date"] and numeric_ok,
                    "cohort": sig["has_date"],
                    "survival": sig["has_date"],
                    "funnel": bool(sig["stage_cols"]),
                    "anomaly": numeric_ok or sig["has_date"],
                    "regression": numeric_ok,
                    "clustering": numeric_ok or clusters_text,
                    "predictive": numeric_ok,
                }.get(family, True)
                if not supported:
                    reason = f"UC_NOT_RELEVANT:R2_TECHNIQUE_DATA_MISMATCH:{tech}"

            if reason is None and len(tabs) >= 2 and not self._find_join_path(tabs)["connected"]:
                reason = "Q3_R4_JOIN_PATH_UNEXECUTABLE"

            if reason is None and family in self._TEMPORAL_FAMILIES and len(tabs) == 1 and cat_map.get(tabs[0]) == "REFERENCE":
                self._add_gate_flag(uc, "Q3_R5_REFERENCE_AS_FACT")
                _note("R5", "INFO", uc_no, f"{tech} on REFERENCE-only table {tabs[0]}")

            if reason is None:
                blob_lower = (stmt + " " + sol + " " + bv).lower()
                found_subject = None
                for pat in subject_patterns:
                    m = re.search(pat, blob_lower)
                    if m:
                        candidate = m.group(1).strip("-_").rstrip("s")
                        if candidate and len(candidate) >= 3:
                            found_subject = candidate
                            break
                if found_subject and not _subject_has_id_column(found_subject, set(cols)):
                    _note("R6", "INFO", uc_no, f"text suggests per-{found_subject} action; verify cited tables "
                                               f"expose a {found_subject}-identifying column")

            if reason is None:
                claims = len(numeric_claim_re.findall(bv.lower()))
                if claims and "[ESTIMATE:" not in bv.upper() and not self._grounded_column_mentions(bv, cols):
                    if claims >= 3:
                        reason = f"Q3_R7_BV_UNGROUNDED_QUANTITIES:{claims}"
                    else:
                        _note("R7", "INFO", uc_no, f"Business Value states {claims} figure(s) without a column "
                                                   f"or [ESTIMATE:] tag")

            if reason is None and family not in self._TEMPORAL_FAMILIES:
                m = re.search(r"\b(?:predict|predicting|classify|classifying|estimate|estimating)\s+`?([a-z_][a-z0-9_]*)`?", sol_low)
                if m and m.group(1) in cols:
                    target = m.group(1)
                    for sentence in re.split(r"[.;\n]", sol_low):
                        cue = feature_cue_re.search(sentence)
                        if cue and f"`{target}`" in sentence[cue.end():]:
                            self._add_gate_flag(uc, f"Q3_R8_TARGET_LEAKAGE:{target}")
                            _note("R8", "INFO", uc_no, f"predicts `{target}` while listing `{target}` as an input")
                            break

            if reason is None and family != "simulation":
                m = invented_input_re.search(sol_low)
                if m and m.group(2) in invented_nouns:
                    reason = f"Q3_R9_SIMULATED_INPUT:{m.group(1)}_{m.group(2)}"
                elif m and m.group(2) in config_nouns:
                    self._add_gate_flag(uc, f"Q3_R9_ASSUMED_PARAMETER:{m.group(2)}")

            if reason is None and family in self._MODEL_FAMILIES and len(tabs) == 1 and cat_map.get(tabs[0]) == "REFERENCE":
                self._add_gate_flag(uc, "Q3_R10_REFERENCE_SCALE")
                _note("R10", "INFO", uc_no, f"{tech} on REFERENCE-scale table {tabs[0]}")

            if reason is None and family == "cohort":
                for sentence in re.split(r"[.;\n]", sol_low):
                    if any(r.search(sentence) for r in biographic_anchor_res) and not event_anchor_re.search(sentence):
                        reason = "Q3_R11_COHORT_BIOGRAPHICAL_ANCHOR"
                        break

            if reason is None and not any(t.lower().startswith("/volumes") for t in refs):
                design = sol + "\n" + hld
                grounded = self._grounded_column_mentions(design, cols)
                if len(grounded) < 2:
                    known = set(cols) | set(cols_by) | {fqn.rsplit(".", 1)[-1] for fqn in cols_by}
                    invented = [tok for tok in set(re.findall(r"`([A-Za-z_][A-Za-z0-9_.]*)`(?!\s*\()", design))
                                if tok.lower() not in known and tok.lower().rsplit(".", 1)[-1] not in known
                                and tok.upper() not in self._GATE_SQL_KEYWORDS and not tok.lower().startswith("ai_")]
                    if not grounded or invented:
                        reason = f"Q3_R13_SOLUTION_UNDER_GROUNDED:{len(grounded)}"
                    else:
                        self._add_gate_flag(uc, "Q3_R13_THIN_GROUNDING")

            if reason is None:
                n_cols = len(cols)
                r14 = None
                if family in self._SUPERVISED_FAMILIES and n_cols < 3:
                    r14 = f"R14_CLASSIFICATION_UNDER_FEATURED:{n_cols}_cols"
                elif family == "clustering" and not clusters_text and len(sig["numeric_cols"]) < 3 and n_cols < 3:
                    r14 = f"R14_CLUSTERING_UNDER_FEATURED:{n_cols}_cols"
                elif family == "association" and not sig["id_cols"]:
                    r14 = "R14_BASKET_NO_GROUPING_ID"
                elif family == "ranking" and n_cols < 2:
                    r14 = "R14_RANKING_NEEDS_DIM_AND_MEASURE"
                elif family == "correlation" and len(sig["numeric_cols"]) < 2 and n_cols < 2:
                    r14 = "R14_CORRELATION_UNDER_COLUMNED"
                elif family == "recommendation" and n_cols < 3:
                    r14 = "R14_RECO_UNDER_COLUMNED"
                elif family in self._TEXT_FAMILIES and not sig["text_cols"]:
                    r14 = "R14_TEXT_NO_TEXT_COLUMN"
                elif family == "network" and len(sig["id_cols"]) < 2:
                    r14 = "R14_NETWORK_NEEDS_TWO_IDS"
                elif family == "geospatial" and not sig["geo_cols"]:
                    r14 = "R14_GEO_NO_GEOGRAPHIC_COLUMN"
                if r14:
                    reason = f"Q3_R14_DATA_FIT_SCALE:{r14}"

            if reason is None:
                reason = self._q3_deep_evidence_reason(uc, tabs, family, sol + "\n" + hld, cols)

            if reason is None:
                kept.append(uc)
            else:
                dropped.append((uc, reason))
                rule = re.search(r"R\d+", reason)
                _note(rule.group(0) if rule else "R0", "HIGH", uc_no, reason)

        if dropped:
            self.logger.warning(f"Q3 RELEVANCE: dropped {len(dropped)} of {len(use_cases)} UCs")
            for uc, reason in dropped[:10]:
                self.logger.info(f"  dropped {uc.get('No','UNK')}: {reason}")
        else:
            self.logger.info(f"Q3 RELEVANCE: all {len(use_cases)} UCs pass")
        return kept, dropped

    def _phaseQ4_detect_operational_patterns(self, full_schema_details: list) -> set:
        """Phase Q4 TIER-1 detector. Returns set of patterns the schema
        STRUCTURALLY supports. Industry-agnostic: type + structure only,
        no English keyword matching."""
        patterns = set()
        if not full_schema_details:
            return patterns
        # Aggregate columns per table
        by_table = {}
        for (c, s, t, col, dtype, _cmt) in self._iter_schema_rows(full_schema_details):
            by_table.setdefault(f"{c}.{s}.{t}", []).append({"name": col, "type": dtype.upper()})
        has_date_col_anywhere = False
        has_fact_like = False
        has_dim_like = False
        has_grouping_candidate = False
        has_status_like = False
        for fqn, cols in by_table.items():
            types = {c["type"] for c in cols}
            numeric_count = sum(1 for c in cols if c["type"].startswith(("INT", "BIGINT", "DOUBLE", "FLOAT", "DECIMAL", "SMALLINT", "LONG")))
            date_cols = [c for c in cols if c["type"].startswith(("DATE", "TIMESTAMP"))]
            if date_cols:
                has_date_col_anywhere = True
            # Fact-like: >= 3 numeric columns + some kind of date column
            if numeric_count >= 3 and date_cols:
                has_fact_like = True
            # Dim-like: small to moderate column count, has an id-like column (name ends with _id/_key/id/key)
            if any(c["name"].lower().endswith(("id", "_id", "key", "_key")) for c in cols) and len(cols) >= 3:
                has_dim_like = True
            # Grouping id candidate: a column that could group multiple rows (order_id / transaction_id / session_id pattern)
            if any(any(tok in c["name"].lower() for tok in ("order", "basket", "transaction", "session", "invoice", "visit")) for c in cols):
                has_grouping_candidate = True
            # Status-like: string column with limited cardinality (we can't probe cardinality here — use name heuristic as TIER-1 proxy)
            if any(c["type"].startswith("STRING") and any(tok in c["name"].lower() for tok in ("status", "state", "flag", "category", "type", "kind")) for c in cols):
                has_status_like = True
        if has_date_col_anywhere and has_fact_like:
            patterns.add("TREND_OR_PERIOD")
        if has_dim_like and has_fact_like:
            patterns.add("SUBJECT_360")
            patterns.add("MIX_OR_MARGIN")
        if has_fact_like and has_status_like and has_date_col_anywhere:
            patterns.add("SLA_OR_ANOMALY")
        if has_grouping_candidate:
            patterns.add("BASKET_OR_FUNNEL")
        self.logger.info(f"Q4 detect_operational_patterns: {sorted(patterns) or 'none (thin schema)'}")
        return patterns

    def _phaseQ5_decision_readiness(self, use_cases: list, full_schema_details: list) -> tuple:
        """Phase Q5 decision readiness (deterministic, no LLM). Returns (kept, dropped).

        R15 flags (`uc["decision_verb_flag"] = True`) names that open with an observational verb.
        R18 flags (`Q5_R18_TECHNIQUE_LABEL_DRIFT`) a technique label named nowhere in the UC while
        the Solution / High Level Design names another family's specific method; general-purpose
        learners such as XGBoost serve many techniques and are not counted as contradictions.
        R19 drops a generic title only when the Solution / High Level Design names no technique at
        all; otherwise the UC keeps the `Q5_R19_GENERIC_TITLE` gate flag.
        No score is read: score floors belong after combined scoring. The text-column requirement
        of text techniques is Q3 R14.
        """
        if not use_cases:
            return use_cases, []
        observational = {"map", "analyze", "analyse", "understand", "study", "examine", "investigate", "explore",
                         "measure", "observe", "track", "monitor", "describe", "review", "assess"}
        # R19 generic-title rejection — catches "Increase Customer Value to Grow Revenue" pattern
        # Schema-agnostic: matches generic verb + generic object + preposition structure.
        generic_title_re = re.compile(
            r"^(?:Improve|Increase|Optimi[sz]e|Maximize|Boost|Grow|Drive)\s+"
            r"(?:the\s+)?"
            r"(?:Customer Value|Business Value|Customer Experience|Customer Satisfaction|"
            r"Customer Engagement|Customer Retention|Customer Loyalty|Customer Base|"
            r"Product Sales|Revenue|Profit|Sales|Growth|Performance|Efficiency|Productivity|"
            r"Operations|Margin|Retention|Loyalty|Conversion|Business Outcomes)"
            r"\s+(?:to|by|for|through|with|and)\s+",
            re.IGNORECASE
        )
        # Second check — if name has ANY specific analytical signal (method / phenomenon
        # / specific entity / specific metric), keep it. Signals are universal tokens
        # found in professional analytics language, not domain-specific vocabulary.
        specific_signals = (
            # Methods / techniques
            "clustering", "segment", "prediction", "predict", "forecast", "regression",
            "classification", "pareto", "cohort", "simulation", "monte carlo", "optimization",
            "optimizing", "attribution", "propensity", "anomaly", "ranking", "isolation",
            "kmeans", "k-means", "uplift", "correlation", "survival", "bayesian",
            # Phenomena / patterns
            "erosion", "leakage", "arbitrage", "fraud", "drift", "bottleneck", "volatility",
            "deviation", "spike", "surge", "lapsed", "dormant", "abandonment", "abuse",
            "misuse", "churn", "defection", "underpenetrated", "underserved", "stagnant",
            # Specific metrics / KPIs (composite forms only; single words too generic)
            "return rate", "conversion rate", "retention rate", "basket size", "cross-sell",
            "upsell", "attach rate", "markdown", "mix shift", "share of wallet",
            # Concrete entities / outputs
            "geographic", "pipeline", "territory", "route", "seasonality", "warehouse",
            "queue", "playbook", "action list", "reallocation", "escalation",
        )
        families = {key for key, _l, _a, _m in self._TECHNIQUE_FAMILIES}
        generic_learners = ("xgboost", "lightgbm", "catboost", "random forest", "gradient boost", "decision tree",
                            "neural network", "classifier", "lstm")
        kept, dropped = [], []
        for uc in use_cases:
            name = str(uc.get("Name") or uc.get("use_case") or "").strip()
            tech = str(uc.get("Analytics Technique") or uc.get("analytics_technique") or "").strip()
            sol = str(uc.get("Solution") or uc.get("solution") or "")
            hld = str(uc.get("High Level Design") or uc.get("high_level_design") or "")
            stmt = str(uc.get("Statement") or uc.get("statement") or "")
            design = sol + "\n" + hld
            reason = None

            words = re.findall(r"[a-z]+", name.lower())
            if words and words[0] in observational:
                uc["decision_verb_flag"] = True

            family = self._technique_family(tech)
            named = {self._technique_label(f) for f in self._technique_mentions("\n".join((name, stmt, design)))}
            if family in families and self._technique_label(family) not in named:
                specific_design = design.lower()
                for learner in generic_learners:
                    specific_design = specific_design.replace(learner, " ")
                rivals = {f for f in self._technique_mentions(specific_design, methods_only=True)
                          if self._signature_family(f) != self._signature_family(family)}
                if rivals:
                    self._add_gate_flag(uc, f"Q5_R18_TECHNIQUE_LABEL_DRIFT:{family}")
                try:
                    NEXT_ACTIONS.add("UC_TECHNIQUE_UNEVIDENCED", severity="INFO", phase="Q5", step="R18",
                                     evidence=f"UC {uc.get('No','UNK')}: technique '{tech}' is named nowhere in its design"
                                              + (f"; design names {sorted(rivals)}" if rivals else ""))
                except Exception:
                    pass

            if reason is None and generic_title_re.match(name) and not any(s in name.lower() for s in specific_signals):
                if self._technique_mentions(design):
                    self._add_gate_flag(uc, "Q5_R19_GENERIC_TITLE")
                else:
                    reason = "Q5_R19_GENERIC_TITLE"

            if reason:
                dropped.append((uc, reason))
            else:
                kept.append(uc)
        if dropped:
            self.logger.warning(f"Q5 DECISION_READINESS: dropped {len(dropped)} of {len(use_cases)} UCs")
            for uc, reason in dropped[:10]:
                self.logger.info(f"  dropped {uc.get('No', uc.get('id','UNK'))}: {reason}")
        else:
            self.logger.info(f"Q5 DECISION_READINESS: all {len(use_cases)} UCs passed")
        return kept, dropped

    def _phaseQ4_validate_recall(self, use_cases: list, detected_patterns: set) -> list:
        """Phase Q4 coverage probe. For every detected pattern not covered by
        any UC, emit UC_RECALL_GAP HIGH. Does not drop UCs."""
        if not use_cases or not detected_patterns:
            return use_cases
        pattern_keywords = {
            "TREND_OR_PERIOD":  ["trend", "period", "month-over", "year-over", "forecast", "seasonal"],
            "SUBJECT_360":      ["360", "profile", "customer segment", "lifetime", "subject"],
            "SLA_OR_ANOMALY":   ["anomal", "sla", "outlier", "breach", "detect"],
            "MIX_OR_MARGIN":    ["mix", "margin", "pareto", "portfolio"],
            "BASKET_OR_FUNNEL": ["basket", "funnel", "bundle", "association", "cohort"],
        }
        covered = set()
        for uc in use_cases:
            blob = " ".join((uc.get(f, "") or "").lower() for f in ("Use Case", "Name", "Statement", "Solution", "Analytics Technique"))
            for p, keys in pattern_keywords.items():
                if any(k in blob for k in keys):
                    covered.add(p)
        missing = detected_patterns - covered
        for p in sorted(missing):
            self.logger.warning(f"Q4 UC_RECALL_GAP: pattern {p} supported by schema but not covered by any UC")
            try:
                NEXT_ACTIONS.add("UC_RECALL_GAP", severity="HIGH",
                                 phase="Q4", step="pattern_coverage",
                                 evidence=f"Pattern {p} supported by schema structure but no UC covers it",
                                 suggested_user_action="Add a UC that exploits this pattern OR declare it out-of-scope via widget 13.")
            except Exception:
                pass
        if not missing:
            self.logger.info(f"Q4 RECALL: all {len(detected_patterns)} detected patterns covered by at least one UC")
        return use_cases

    def _phaseQ_run_integrity_gates(self, use_cases: list, full_schema_details: list) -> list:
        """Run Q1..Q5 against the consolidated portfolio and return the survivors. Every gate's
        in/out/drop reasons go to processing_honesty["gate_ledger"]. A gate that raises is logged
        with a short traceback and the UCs it should have checked are kept but marked
        `uc["gate_unverified"]`."""
        if not use_cases:
            return use_cases
        kept = use_cases
        q1_dropped, q2_merged, q3_dropped, q5_dropped = [], [], [], []
        n_in = len(kept)
        try:
            kept, q1_dropped = self._phaseQ1_validate_grounding(kept, full_schema_details)
            self._record_gate_ledger("Q1", n_in, len(kept), q1_dropped)
        except Exception as _q1_err:
            self._record_gate_exception("Q1", _q1_err, kept)
        n_in = len(kept)
        try:
            kept, q2_merged = self._phaseQ2_canonical_dedup(kept)
            self._record_gate_ledger("Q2", n_in, len(kept), q2_merged)
        except Exception as _q2_err:
            self._record_gate_exception("Q2", _q2_err, kept)
        n_in = len(kept)
        try:
            kept, q3_dropped = self._phaseQ3_validate_relevance(kept, full_schema_details)
            self._record_gate_ledger("Q3", n_in, len(kept), q3_dropped, flagged=sum(
                1 for uc in kept if any(str(f).startswith("Q3_") for f in (uc.get("gate_flags") or []))))
        except Exception as _q3_err:
            self._record_gate_exception("Q3", _q3_err, kept)
        try:
            _patterns = self._phaseQ4_detect_operational_patterns(full_schema_details)
            kept = self._phaseQ4_validate_recall(kept, _patterns)
        except Exception as _q4_err:
            self._record_gate_exception("Q4", _q4_err, kept, mark=False)
        n_in = len(kept)
        try:
            kept, q5_dropped = self._phaseQ5_decision_readiness(kept, full_schema_details)
            self._record_gate_ledger("Q5", n_in, len(kept), q5_dropped, flagged=sum(
                1 for uc in kept if uc.get("decision_verb_flag")
                or any(str(f).startswith("Q5_") for f in (uc.get("gate_flags") or []))))
        except Exception as _q5_err:
            self._record_gate_exception("Q5", _q5_err, kept)
        try:
            kept = self._canonicalize_sponsor_beneficiary_roles(kept)
        except Exception as _q6_err:
            self._record_gate_exception("SPONSOR_CANONICAL", _q6_err, kept, mark=False)
        honesty = self._honesty()
        honesty["q1_grounding_dropped"] = len(q1_dropped)
        honesty["q2_dedup_dropped"] = len(q2_merged)
        honesty["q3_relevance_dropped"] = len(q3_dropped)
        honesty["q5_decision_dropped"] = len(q5_dropped)
        q5_reasons = {}
        for _uc, reason in q5_dropped:
            k = self._gate_reason_key(reason)
            q5_reasons[k] = q5_reasons.get(k, 0) + 1
        honesty["q5_drop_reasons"] = q5_reasons
        honesty["q5_decision_verb_flagged"] = sum(1 for uc in kept if uc.get("decision_verb_flag"))
        return kept

    def _canonicalize_sponsor_beneficiary_roles(self, use_cases: list) -> list:
        """LLM-driven canonicalization of Sponsor and Beneficiary field values.
        Root cause: LLM-generated UCs use inconsistent role labels ('CFO' vs
        'Chief Financial Officer'). Industry-agnostic prompt (no corpus tokens).
        Cached per-run by normalized input. Sentinel: [SPONSOR_CANONICAL FIRED].
        """
        if not use_cases:
            return use_cases
        if not hasattr(self, "_sponsor_role_cache"):
            self._sponsor_role_cache = {}
        def _norm(s):
            return (s or "").strip().lower()
        unique_raw = set()
        for uc in use_cases:
            for field in ("Sponsor", "Beneficiary"):
                v = uc.get(field, "")
                if v and _norm(v):
                    unique_raw.add(str(v).strip())
        to_canonicalize = sorted(r for r in unique_raw if _norm(r) not in self._sponsor_role_cache)
        if to_canonicalize:
            try:
                role_list_str = "\n".join(f"- {r}" for r in to_canonicalize)
                prompt = (
                    "You are a role-name normalizer for a business analytics catalog.\n"
                    "For each input role name below, output its canonical form.\n\n"
                    "RULES (industry-agnostic — must work equally across any business\n"
                    "domain without injecting sector-specific words the input didn't contain):\n"
                    "1. Expand common C-suite abbreviations to their full title:\n"
                    "   CFO -> Chief Financial Officer, CIO -> Chief Information Officer,\n"
                    "   CTO -> Chief Technology Officer, COO -> Chief Operating Officer,\n"
                    "   CMO -> Chief Marketing Officer, CEO -> Chief Executive Officer,\n"
                    "   CDO -> Chief Data Officer, CISO -> Chief Information Security Officer,\n"
                    "   CHRO -> Chief Human Resources Officer, CPO -> Chief Product Officer,\n"
                    "   CRO -> Chief Revenue Officer, CCO -> Chief Customer Officer,\n"
                    "   CSO -> Chief Strategy Officer.\n"
                    "2. Normalize functional-head variants to the single form 'Head of X'\n"
                    "   (so 'VP of Operations', 'Director of Operations', 'Operations Lead'\n"
                    "   all become 'Head of Operations'). Preserve the 'X' verbatim.\n"
                    "3. Never rename in a way that changes meaning (Chief Financial Officer\n"
                    "   is NOT the same as Financial Controller; keep both distinct).\n"
                    "4. If the input is not a recognizable business role (e.g. a person name,\n"
                    "   department name, or free-form description), return it VERBATIM.\n"
                    "5. Do NOT inject industry-specific words the input didn't contain.\n\n"
                    "OUTPUT FORMAT (STRICT JSON):\n"
                    "{\"roles\": [{\"input\": \"<verbatim input>\", \"canonical\": \"<canonical form>\"}]}\n\n"
                    f"INPUT ROLES:\n{role_list_str}"
                )
                schema = {
                    "name": "role_canonicalization",
                    "schema": {
                        "type": "object",
                        "properties": {
                            "roles": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "properties": {
                                        "input": {"type": "string"},
                                        "canonical": {"type": "string"}
                                    },
                                    "required": ["input", "canonical"]
                                }
                            }
                        },
                        "required": ["roles"]
                    }
                }
                resp = self.ai_agent._call_ai_query(
                    prompt=prompt,
                    prompt_name="canonicalize_sponsor_beneficiary_roles",
                    response_schema=schema,
                    display_name="Sponsor_Beneficiary_Canonicalizer"
                )
                cleaned = clean_json_response(resp) if resp else ""
                parsed = json.loads(cleaned) if cleaned else {"roles": []}
                for entry in parsed.get("roles", []):
                    _in = entry.get("input", "")
                    _can = entry.get("canonical", "")
                    if _in and _can:
                        self._sponsor_role_cache[_norm(_in)] = str(_can).strip()
                for r in to_canonicalize:
                    self._sponsor_role_cache.setdefault(_norm(r), r)
                self.logger.info(f"[SPONSOR_CANONICAL FIRED] canonicalized_new={len(to_canonicalize)} cache_size={len(self._sponsor_role_cache)}")
            except Exception as _e:
                self.logger.warning(f"Sponsor/Beneficiary canonicalizer LLM step failed non-fatally: {get_clean_error_message(_e)}")
                for r in to_canonicalize:
                    self._sponsor_role_cache.setdefault(_norm(r), r)
        updates = 0
        for uc in use_cases:
            for field in ("Sponsor", "Beneficiary"):
                v = uc.get(field, "")
                k = _norm(v)
                if k and k in self._sponsor_role_cache:
                    canon = self._sponsor_role_cache[k]
                    if canon and canon != v:
                        uc[field] = canon
                        updates += 1
        if updates > 0:
            self.logger.info(f"[SPONSOR_CANONICAL FIRED] applied field_updates={updates} unique_roles={len(unique_raw)}")
        return use_cases

    def _phaseL_final_gate(self, use_cases: list, full_schema_details: list) -> dict:
        """Phase L final gate: 19 advisory binary checks emitted as NEXT_ACTIONS entries. Never
        filters the portfolio. Returns {check_id: {"pass": bool, "severity": str}} and stores the
        same dict on `self.phase_l_results`."""
        results = {}
        def _check(cid: str, passed: bool, severity: str, evidence: str, suggested: str = ""):
            results[cid] = {"pass": bool(passed), "severity": severity}
            if not passed:
                try:
                    NEXT_ACTIONS.add(
                        rule_id=f"L_{cid}", severity=severity, phase="L", step="FinalGate",
                        evidence=evidence[:300], suggested_user_action=suggested[:300] or None,
                    )
                except Exception:
                    pass
        uc_list = use_cases or []
        n = len(uc_list)
        honesty = self._honesty()
        schema_rows = list(self._iter_schema_rows(full_schema_details))
        index = self._gate_schema_index(full_schema_details) if schema_rows else {"tables": {}}
        # L1 first-run-pass: if no UCs, BLOCK
        _check("L1_any_ucs_produced", n > 0, "BLOCKING",
               f"Pipeline produced {n} use cases", "Re-run with broader schema or lower filters.")
        # L2 domain diversity
        _doms = {str(uc.get("Business Domain") or "").strip() for uc in uc_list if uc.get("Business Domain")}
        _check("L2_domain_diversity", len(_doms) >= 2 or n <= 3, "INFO",
               f"{len(_doms)} distinct domain(s)", "Expand schema coverage to unlock more domains.")
        # L3 subdomain diversity
        _subs = {str(uc.get("Subdomain") or "").strip() for uc in uc_list if uc.get("Subdomain")}
        _check("L3_subdomain_diversity", len(_subs) >= max(2, n // 3), "INFO",
               f"{len(_subs)} distinct subdomain(s)", "Encourage subdomain rename in prompt.")
        # L4 priority distribution non-degenerate (not all "Medium")
        _prios = [str(uc.get("Priority") or "").strip() for uc in uc_list]
        _check("L4_priority_nondegenerate", len(set(_prios)) >= 2 or n <= 2, "INFO",
               f"Priority values: {sorted(set(_prios))}", "Loosen priority heuristic in prompt.")
        # L5 every UC has primary table
        _missing_pt = [uc for uc in uc_list if str(uc.get("Primary Table") or "").strip().upper() in ("", "N/A")]
        _check("L5_every_uc_has_primary_table", not _missing_pt, "BLOCKING",
               f"{len(_missing_pt)}/{n} UCs missing Primary Table", "Reject UCs without Primary Table at parse time.")
        # L6 every UC has statement
        _missing_st = [uc for uc in uc_list if len(str(uc.get("Statement") or "").strip()) < 20]
        _check("L6_every_uc_has_statement", not _missing_st, "HIGH",
               f"{len(_missing_st)}/{n} UCs have empty/short Statement", "Raise statement min-length in parser.")
        # L7 every UC has solution
        _missing_sol = [uc for uc in uc_list if len(str(uc.get("Solution") or "").strip()) < 20]
        _check("L7_every_uc_has_solution", not _missing_sol, "HIGH",
               f"{len(_missing_sol)}/{n} UCs have empty/short Solution", "Raise solution min-length in parser.")
        # L8 every UC has technique
        _missing_tech = [uc for uc in uc_list if not str(uc.get("Analytics Technique") or "").strip()]
        _check("L8_every_uc_has_technique", not _missing_tech, "HIGH",
               f"{len(_missing_tech)}/{n} UCs missing technique", "Enforce technique whitelist in parser.")
        _sigs = [self._uc_dedup_signature(uc) for uc in uc_list]
        _dup = len(_sigs) - len(set(_sigs))
        _check("L9_no_duplicate_names", _dup == 0, "HIGH",
               f"{_dup} UC(s) duplicate another UC's decision signature (verb, object, tables, technique)",
               "Tighten Q2 canonical dedup or rename one of the colliding UCs.")
        # L10 Q1 grounding did not drop >50%
        q1d = honesty.get("q1_grounding_dropped", 0)
        _check("L10_q1_drop_under_50pct", (q1d * 2) < max(1, n + q1d), "HIGH",
               f"Q1 dropped {q1d} vs kept {n}", "Inspect prompt for fabricated tables/columns.")
        # L11 Q2 merged reasonable
        q2m = honesty.get("q2_dedup_dropped", 0)
        _check("L11_q2_merge_under_50pct", (q2m * 2) < max(1, n + q2m), "INFO",
               f"Q2 merged {q2m} vs kept {n}", "Diversify generation seeds.")
        # L12 Q3 relevance drop under 50%
        q3d = honesty.get("q3_relevance_dropped", 0)
        _check("L12_q3_drop_under_50pct", (q3d * 2) < max(1, n + q3d), "HIGH",
               f"Q3 dropped {q3d} vs kept {n}", "Review the Q3 drop reasons in the gate ledger.")
        # L13 schema detail non-empty
        _check("L13_schema_details_present", bool(full_schema_details), "BLOCKING",
               f"full_schema_details has {len(full_schema_details or [])} rows",
               "Check catalog permissions / table selection.")
        # L14 no UC references unknown table (schema rows are tuples: catalog, schema, table, col, type, desc)
        _table_names = {t.lower() for (_c, _s, t, _col, _dt, _cm) in schema_rows}
        _unknown = []
        for uc in uc_list:
            pt = str(uc.get("Primary Table") or "").strip().lower()
            if pt and pt.split(".")[-1] not in _table_names:
                _unknown.append(pt)
        _check("L14_all_primary_tables_grounded", not _unknown, "BLOCKING",
               f"{len(_unknown)} UCs reference unknown tables: {list(set(_unknown))[:5]}",
               "Strengthen Q1 grounding.")
        # L15 business_domain != placeholder
        _placeholders = {"", "other", "unknown", "general", "misc", "miscellaneous", "uncategorized", "unassigned"}
        _ph = [uc for uc in uc_list
               if str(uc.get("Business Domain") or "").strip().lower() in _placeholders
               or re.match(r"^domain\d+$", str(uc.get("Business Domain") or "").strip().lower())]
        _check("L15_no_placeholder_domains", not _ph, "HIGH",
               f"{len(_ph)}/{n} UCs have placeholder domain",
               "Domain detection fell back; rerun or set the Business Domains widget.")
        q5d = honesty.get("q5_decision_dropped", 0)
        _check("L16_q5_drop_under_50pct", (q5d * 2) < max(1, n + q5d), "HIGH",
               f"Q5 dropped {q5d} vs kept {n}", "Review the Q5 R18/R19 drop reasons in the gate ledger.")
        # L17 NEXT_ACTIONS has no unexpected BLOCKING beyond L-gates themselves
        try:
            all_blocking = [a for a in NEXT_ACTIONS.snapshot() if a.get("severity") == "BLOCKING"]
            non_L = [a for a in all_blocking if not (a.get("rule_id","") or "").startswith("L_")]
        except Exception:
            non_L = []
        _check("L17_zero_non_L_blocking", not non_L, "INFO",
               f"{len(non_L)} BLOCKING entries outside L-gates",
               "Resolve the BLOCKING NEXT_ACTIONS entries reported in the run log.")
        # L18 multi-table ratio: at least 30% of UCs should span 2+ tables when schema has 2+ tables.
        try:
            _schema_tables = set(_table_names)
            _multi = 0
            for _uc in uc_list:
                _ti = str(_uc.get("Tables Involved", "") or "")
                _n = len([t for t in _ti.split(",") if t.strip()])
                if _n >= 2:
                    _multi += 1
            _ratio = (_multi / n) if n else 0.0
            _passed = (len(_schema_tables) < 2) or (_ratio >= 0.30)
            _check("L18_multi_table_ratio", _passed, "HIGH",
                   f"multi_table_UCs={_multi}/{n} (ratio={_ratio:.2f}); schema_tables={len(_schema_tables)}",
                   "Strengthen prompt FK-exploitation / STRUCTURAL JOIN HEURISTIC language; consider Q6 rebalance pass.")
        except Exception as _mt_err:
            _check("L18_multi_table_ratio", True, "INFO",
                   f"multi-table check skipped: {_mt_err}", "")
        # L19 production_readiness: aggregate Genie-runnable UC check.
        # J1 — Solution has >=2 column citations
        # J2 — Primary Table in known schema (covered by L14)
        # J3 — All backticked cols resolve to known schema cols (not SQL keywords)
        # J4 — Analytics Technique is specific (not umbrella — Rule 12 whitelist)
        # J5 — Result artifact named (result_table OR Solution names output)
        # J7 — Inspire Score set and >0
        # J8 — Beneficiary non-empty and not generic
        import re as _rL
        _all_schema_cols = {col.lower() for (_c, _s, _t, col, _dt, _cm) in schema_rows}
        _known_tables = self._known_table_names(index)
        _generic_bene = {"", "none", "n/a", "team", "everyone", "all", "business"}
        _prod_ready = 0
        _prod_fails = []
        for _uc in uc_list:
            _fails = []
            _sol = str(_uc.get("Solution","") or "")
            _hld = str(_uc.get("High Level Design","") or "")
            _stmt = str(_uc.get("Statement","") or "")
            _bv = str(_uc.get("Business Value","") or "")
            _blob = _sol + " " + _stmt + " " + _bv
            # J1
            _cited_cols = set()
            for _t in self._uc_table_refs(_uc):
                _cited_cols |= set(index["tables"].get(_t, {}))
            if len(self._grounded_column_mentions(_sol + "\n" + _hld, _cited_cols)) < 2:
                _fails.append("J1_solution_<2_cols")
            # J3
            _ungrounded = self._ungrounded_identifiers(_blob, _all_schema_cols, _known_tables)
            if _ungrounded:
                # allow a few — just a smell, not a hard fail. threshold: >=3 ungrounded.
                if len(_ungrounded) >= 3:
                    _fails.append(f"J3_ungrounded_cols:{len(_ungrounded)}")
            # J4
            _tech = " ".join(str(_uc.get("Analytics Technique","") or "").lower().split())
            if _tech in self._GATE_CATCHALL_TECHNIQUES:
                _fails.append("J4_umbrella_technique")
            # J5
            _rt = str(_uc.get("result_table","") or "").strip()
            if not _rt and not _rL.search(r"\b(output|result|target|saves?|writes?|materializes?|persists?)\b", _sol.lower()):
                _fails.append("J5_no_output_named")
            # J6
            if _rL.search(r"\[(MISSING_DATA|PLACEHOLDER|TODO)\s*:", _blob):
                _fails.append("J6_placeholder_tag")
            # J7
            try:
                _is = float(_uc.get("Inspire Score", _uc.get("Priority", 0)) or 0)
            except Exception:
                _is = 0.0
            if _is <= 0:
                _fails.append("J7_no_score")
            # J8
            _bene = str(_uc.get("Beneficiary","") or "").strip().lower()
            if _bene in _generic_bene or len(_bene) < 3:
                _fails.append("J8_generic_beneficiary")
            if not _fails:
                _prod_ready += 1
            else:
                _prod_fails.append((_uc.get("No","?"), _fails))
        _ready_ratio = (_prod_ready / n) if n else 0
        _check("L19_production_readiness", _ready_ratio >= 0.80, "HIGH",
               f"production-ready {_prod_ready}/{n} ({int(_ready_ratio*100)}%); failing checks per-UC sample: {dict(_prod_fails[:5])}",
               "Review UC_PRODUCTION_READINESS NEXT_ACTIONS entries; tighten prompt/Q3 rules.")
        # emit per-UC summary as INFO (first 20)
        for _uc_no, _fails in _prod_fails[:20]:
            try:
                NEXT_ACTIONS.add("UC_PRODUCTION_READINESS", severity="INFO", phase="L", step="J-checks",
                                 evidence=f"UC {_uc_no}: failed production checks {_fails}")
            except Exception:
                pass
        honesty["phase_L_results"] = {k: v["pass"] for k, v in results.items()}
        honesty["phase_L_fail_count"] = sum(1 for v in results.values() if not v["pass"])
        honesty["phase_L_production_ready_count"] = _prod_ready
        honesty["phase_L_production_ready_ratio"] = round(_ready_ratio, 3)
        self.phase_l_results = results
        return results

    def _rename_placeholder_domains_from_subdomains(self, use_cases: list) -> list:
        """When domain detection fell back to placeholder DomainN names, use the detected subdomains
        to ask the LLM for a single-word umbrella name per placeholder. Honors the
        DOMAIN_FINDER_PROMPT one-word rule and the user's generation instructions, and never renames
        a user-specified domain. Fail-safe: retains the placeholder if the LLM fails or returns an
        invalid name."""
        import re as _re
        if not use_cases:
            return use_cases
        protected = self._protected_domains()
        placeholder_re = _re.compile(r"^Domain\d+$")
        groups = {}
        for uc in use_cases:
            d = (uc.get("Business Domain") or "").strip()
            if placeholder_re.match(d) and d.lower() not in protected:
                groups.setdefault(d, set()).add((uc.get("Subdomain") or "").strip())
        if not groups:
            return use_cases
        self.logger.warning(f"🔁 Placeholder domain rename: found {len(groups)} placeholder domain(s) after subdomain detection; attempting rename from subdomains")
        banned = {"operations","engagement","management","services","activities","processes","functions","solutions","systems","general"}
        valid_re = _re.compile(r"^[A-Z][a-z]+$")
        industries_str = ", ".join(self.industries) if hasattr(self, "industries") and self.industries else "UNKNOWN_INDUSTRY"
        instructions = self._get_generation_instruction_for_prompt("DOMAIN_FINDER_PROMPT")
        rename_map = {}
        for placeholder, subs in groups.items():
            subs = sorted(s for s in subs if s)
            if not subs:
                continue
            subdomain_list = "\n".join(f"- {s}" for s in subs[:30])
            prompt = (
                "You are naming ONE business domain based on the subdomains assigned to it.\n\n"
                "HARD RULES (any violation is rejected):\n"
                "- EXACTLY ONE WORD. No spaces. No CamelCase. Only the first letter capitalized.\n"
                "- The word must be a concrete business noun (e.g., Revenue, Customers, Risk, Inventory, Suppliers, Franchise, Procurement, Pricing, Fraud, Maintenance, Logistics).\n"
                "- BANNED generic words: Operations, Engagement, Management, Services, Activities, Processes, Functions, Solutions, Systems, General.\n"
                "- Must fit the business context below.\n\n"
                f"Business Name: {self.business_name}\n"
                f"Industries: {industries_str}\n\n"
                "SUBDOMAINS in this domain:\n"
                f"{subdomain_list}\n\n"
                f"{instructions}\n"
                "If these naming rules conflict with an explicit user directive in the instructions above, the user directive wins.\n\n"
                "Reply with ONLY the single-word domain name. No punctuation. No explanation."
            )
            try:
                raw = self.ai_agent._call_ai_query(
                    prompt=prompt,
                    prompt_name="PlaceholderDomainRename",
                    response_schema=None,
                    display_name=f"PlaceholderDomainRename_{placeholder}"
                )
                if not raw:
                    self.logger.warning(f"  ✗ {placeholder}: empty LLM response; keeping placeholder")
                    continue
                candidate = str(raw).strip().splitlines()[0].strip().strip(".").strip()
                if (
                    candidate
                    and candidate != placeholder
                    and valid_re.match(candidate)
                    and candidate.lower() not in banned
                ):
                    rename_map[placeholder] = protected.get(candidate.lower(), candidate)
                    self.logger.info(f"  ✓ {placeholder} → {rename_map[placeholder]} (from {len(subs)} subdomains)")
                else:
                    self.logger.warning(f"  ✗ {placeholder}: invalid candidate '{candidate}'; keeping placeholder")
            except Exception as e:
                self.logger.warning(f"  ✗ {placeholder}: rename LLM failed ({get_clean_error_message(e)}); keeping placeholder")
        # Collision guard: if two placeholders map to same name, first keeps it, others retain placeholder
        seen = set()
        for k in list(rename_map.keys()):
            if rename_map[k] in seen:
                self.logger.warning(f"  ✗ name collision on '{rename_map[k]}'; keeping placeholder for {k}")
                del rename_map[k]
            else:
                seen.add(rename_map[k])
        if rename_map:
            for uc in use_cases:
                d = (uc.get("Business Domain") or "").strip()
                if d in rename_map:
                    uc["Business Domain"] = rename_map[d]
            self.logger.info(f"🏁 Rename complete: {len(rename_map)} of {len(groups)} placeholder domain(s) renamed")
        else:
            self.logger.warning(f"⚠️  Rename complete: 0 of {len(groups)} placeholder domain(s) could be renamed; placeholders retained")
        return use_cases

    def _assign_default_domains(self, use_cases: list) -> list:
        """
        FALLBACK when LLM domain detection fails: group use cases by the catalog.schema of their
        primary table (the grouping batched detection uses) under DomainN placeholders, largest
        group first. `_rename_placeholder_domains_from_subdomains` later names each placeholder from
        its detected subdomains, and Phase L L15 reports any placeholder that survives.
        
        Args:
            use_cases: List of use case dictionaries
            
        Returns:
            List of use case copies with a placeholder 'Business Domain' and an empty 'Subdomain'
        """
        if not use_cases:
            return use_cases
        groups = defaultdict(list)
        for uc in use_cases:
            groups[self._uc_schema_prefix(uc)].append(uc)
        ordered = sorted(groups.items(), key=lambda kv: (-len(kv[1]), kv[0]))
        names = {prefix: f"Domain{i}" for i, (prefix, _ucs) in enumerate(ordered, 1)}
        result = []
        for uc in use_cases:
            uc_copy = uc.copy()
            uc_copy['Business Domain'] = names[self._uc_schema_prefix(uc)]
            uc_copy['Subdomain'] = ''
            self._add_clustering_fallback(uc_copy, 'domain:placeholder')
            result.append(uc_copy)
        self.logger.warning(
            f"[CLUSTERING_FALLBACK FIRED] {len(result)} use case(s) grouped by source schema into "
            f"{len(names)} placeholder domain(s): "
            + ", ".join(f"{names[p]}={p} ({len(g)})" for p, g in ordered[:10])
        )
        try:
            NEXT_ACTIONS.add("CLUSTERING_FALLBACK", severity="HIGH", phase="Clustering", step="domains",
                             evidence=f"{len(result)} use case(s) placed in {len(names)} schema-based placeholder domain(s)")
        except Exception:
            pass
        return result

    def _assign_default_subdomains(self, use_cases: list, domain_name: str) -> list:
        """
        FALLBACK when LLM subdomain detection fails: one subdomain per primary table, named from the
        table's own words (exactly 2 words, e.g. store_sales -> "Store Sales").
        
        Args:
            use_cases: List of use case dictionaries
            domain_name: Name of the domain (for logging)
            
        Returns:
            List of use case copies with a table-derived 'Subdomain'
        """
        if not use_cases:
            return use_cases
        result = [uc.copy() for uc in use_cases]
        self._apply_table_subdomains(result, domain_name)
        self.logger.info(
            f"📋 [{domain_name}] Evidence-based subdomain fallback: {len(result)} use cases across "
            f"{len({uc['Subdomain'] for uc in result})} table-derived subdomains"
        )
        return result

    def _apply_table_subdomains(self, use_cases: list, domain_name: str):
        for uc in use_cases or []:
            uc['Subdomain'] = self._fallback_subdomain_name(self._uc_primary_table(uc), domain_name)
            self._add_clustering_fallback(uc, 'subdomain:table_evidence')

    @staticmethod
    def _fallback_subdomain_name(table_ref, domain_name="") -> str:
        name = str(table_ref or "").rsplit(".", 1)[-1]
        spaced = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", name)
        technical = {"fact", "fct", "dim", "tbl", "stg", "raw", "vw", "view", "src", "tmp", "temp",
                     "bronze", "silver", "gold", "int"}
        tokens = [t for t in (w.lower() for w in re.findall(r"[A-Za-z]+", spaced)) if len(t) > 1 and t not in technical]
        if len(tokens) >= 2:
            return f"{tokens[0].title()} {tokens[1].title()}"
        if tokens:
            return f"{tokens[0].title()} Analytics"
        base = re.sub(r"[^A-Za-z0-9]+", "", name) or re.sub(r"[^A-Za-z0-9]+", "", str(domain_name or "")) or "Unmapped"
        return f"{base.title()} Analytics"

    def _protected_domains(self) -> dict:
        protected = {}
        for d in (getattr(self, "user_business_domains", None) or []):
            name = str(d or "").strip()
            if name:
                protected.setdefault(name.lower(), name)
        return protected

    @staticmethod
    def _escape_template_braces(text) -> str:
        return str(text if text is not None else "").replace("{", "{{").replace("}", "}}")

    def _uc_primary_table(self, uc) -> str:
        for ref in self._uc_table_refs(uc):
            if not ref.lower().startswith("/volumes"):
                return ref
        return ""

    def _uc_schema_prefix(self, uc) -> str:
        primary = self._uc_primary_table(uc)
        if not primary:
            return "_no_tables_"
        parts = primary.split(".")
        return ".".join(parts[:2]) if len(parts) >= 2 else primary

    @staticmethod
    def _add_clustering_fallback(uc, tag):
        tags = [t for t in str(uc.get("clustering_fallback") or "").split(";") if t]
        if tag not in tags:
            tags.append(tag)
        uc["clustering_fallback"] = ";".join(tags)

    def _fill_by_table_evidence(self, missing, assigned, field, tag) -> list:
        votes = {}
        for uc in assigned or []:
            value = str(uc.get(field) or "").strip()
            table = self._uc_primary_table(uc)
            if value and table:
                votes.setdefault(table, {})
                votes[table][value] = votes[table].get(value, 0) + 1
        still = []
        for uc in missing or []:
            counts = votes.get(self._uc_primary_table(uc))
            if counts:
                uc[field] = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
                self._add_clustering_fallback(uc, tag)
            else:
                still.append(uc)
        return still

    def _parse_assignment_csv(self, response, value_field, context, ucs) -> dict:
        if not response or not str(response).strip():
            return {}
        text = clean_csv_response(str(response))
        rows = CSVParser.parse_csv_list(text, logger=self.logger, context=context)
        if not rows:
            return {}
        columns = self._csv_columns(rows[0])
        id_col = self._csv_id_column(columns)
        value_col = columns.get(self._csv_column_key(value_field))
        if id_col is None or value_col is None:
            self.logger.warning(f"[{context}] response has no use case id or '{value_field}' column: {rows[0][:6]}")
            return {}
        header = [str(cell) for cell in rows[0]]
        records, _ = CSVParser.aligned_rows(
            rows[1:], self.logger, context, header=header, required=(header[id_col], header[value_col]),
            anchors={header[id_col]: CSVParser.one_of(str(uc.get('No', '')) for uc in ucs or [])}, text=text)
        mapping = {}
        for row in records:
            uc_id = self._csv_cell_text(row, id_col)
            value = str(row[value_col]).strip() if value_col < len(row) else ""
            if uc_id and value:
                mapping[uc_id] = value
        return mapping

    def _canonicalize_domain_names(self, mapping) -> dict:
        protected = self._protected_domains()
        variants = {}
        for value in mapping.values():
            spellings = variants.setdefault(value.lower(), {})
            spellings[value] = spellings.get(value, 0) + 1
        canonical = {low: protected.get(low) or sorted(sp.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
                     for low, sp in variants.items()}
        return {k: canonical[v.lower()] for k, v in mapping.items()}

    @staticmethod
    def _discovered_domains_hint(discovered_domains) -> str:
        if not discovered_domains:
            return ""
        return f"\n\n**PREVIOUSLY DISCOVERED DOMAINS (reuse these where appropriate):**\n{', '.join(sorted(discovered_domains))}\n"

    def _domain_finder_vars(self, use_cases_csv, language, extra_context="", violations="") -> dict:
        return {
            "use_cases_csv": use_cases_csv,
            "output_language": language,
            "business_name": self.business_name,
            "industries": ", ".join(self.industries) if hasattr(self, 'industries') and self.industries else "UNKNOWN_INDUSTRY",
            "business_context": self._get_business_context_fallback() + extra_context,
            "previous_violations": violations,
            "generation_instructions_section": self._get_generation_instruction_for_prompt("DOMAIN_FINDER_PROMPT"),
        }

    def _repair_unassigned_domains(self, missing, assigned, fieldnames, language, discovered_domains):
        if not missing:
            return
        import io
        import csv
        repaired = {}
        try:
            buf = io.StringIO()
            w = csv.DictWriter(buf, fieldnames=fieldnames, extrasaction='ignore')
            w.writeheader()
            w.writerows(missing)
            response = self.ai_agent.run_worker(
                step_name=f"Detect_Domains_Repair_{language}",
                worker_prompt_path="DOMAIN_FINDER_PROMPT",
                prompt_vars=self._domain_finder_vars(
                    buf.getvalue(), language,
                    extra_context=self._discovered_domains_hint(discovered_domains),
                    violations=(f"\n\n**🚨 THESE {len(missing)} USE CASES WERE MISSING FROM THE PREVIOUS ANSWER**: "
                                "assign EVERY one of them, reusing a previously discovered domain whenever one fits.")),
                response_schema=None,
                skip_cache=True
            )
            repaired = self._canonicalize_domain_names(
                self._parse_assignment_csv(response, "domain", "Domain detection repair", missing))
        except Exception as e:
            self.logger.warning(f"Domain repair call for {len(missing)} use case(s) failed: {get_clean_error_message(e)}")
        still = []
        for uc in missing:
            domain = repaired.get(str(uc.get('No', '')).strip())
            if domain:
                uc['Business Domain'] = domain
            else:
                still.append(uc)
        from_llm = len(missing) - len(still)
        still = self._fill_by_table_evidence(still, assigned, 'Business Domain', 'domain:table_evidence')
        from_evidence = len(missing) - from_llm - len(still)
        if still:
            for uc, placeholder in zip(still, self._assign_default_domains(still)):
                uc['Business Domain'] = placeholder['Business Domain']
                uc['clustering_fallback'] = placeholder['clustering_fallback']
        self.logger.warning(
            f"[DOMAIN_REPAIR FIRED] {len(missing)} use case(s) the LLM skipped: {from_llm} re-assigned by a "
            f"targeted call, {from_evidence} from same-table evidence, {len(still)} placed in placeholder domains"
        )

    def _detect_subdomains_all_domains(self, domain_assignments: list, language: str, path_label: str) -> list:
        domain_usecases_map = defaultdict(list)
        unassigned = []
        for uc in domain_assignments or []:
            domain = str(uc.get('Business Domain') or '').strip()
            if domain:
                domain_usecases_map[domain].append(uc)
            else:
                unassigned.append(uc)
        for uc in self._assign_default_domains(unassigned) if unassigned else []:
            domain_usecases_map[uc['Business Domain']].append(uc)
        num_domains = len(domain_usecases_map)
        if not num_domains:
            return []
        gc.collect()
        try:
            import psutil, os as _os
            mem_mb = psutil.Process(_os.getpid()).memory_info().rss / (1024 * 1024)
            self.logger.info(f"📊 MEMORY before subdomain detection: {mem_mb:.0f} MB (RSS)")
        except Exception:
            pass
        subdomain_parallelism, reason = calculate_adaptive_parallelism(
            "subdomain_detection", self.max_parallelism,
            num_items=sum(len(v) for v in domain_usecases_map.values()),
            num_domains=num_domains,
            is_llm_operation=True, logger=self.logger
        )
        log_adaptive_parallelism_decision("subdomain_detection", subdomain_parallelism, self.max_parallelism, reason)
        _subdomain_call_budget = self._compute_subdomain_call_budget(num_domains)
        self.logger.info(f"[G34 SUBDOMAIN_BUDGET_ADAPTIVE FIRED] n_domains={num_domains} per_call_budget={_subdomain_call_budget}s ({path_label} path)")
        workers = max(1, min(int(subdomain_parallelism or 1), num_domains))
        total_timeout = _subdomain_call_budget * (self.max_retry_attempts + 1) * (-(-num_domains // workers)) + 120
        results = {}
        if workers >= 2:
            self.logger.info(f"🔄 PARALLEL SUBDOMAIN DETECTION: {workers} workers for {num_domains} domains (total timeout: {total_timeout}s)")
            log_print(f"🔄 Subdomain detection: {num_domains} domains with {workers} workers")
            try:
                with _SafeExecutorContext(max_workers=workers, thread_name_prefix="SubdomainDetect",
                                          logger=self.logger, name="SubdomainDetection") as executor:
                    future_to_domain = {}
                    for domain_name, domain_use_cases in domain_usecases_map.items():
                        future = executor.submit(
                            self._detect_subdomains_for_domain,
                            domain_name,
                            domain_use_cases,
                            language,
                            _subdomain_call_budget
                        )
                        future_to_domain[future] = domain_name
                    try:
                        for future in executor.completed(future_to_domain, timeout=total_timeout):
                            domain_name = future_to_domain[future]
                            try:
                                results[domain_name] = future.result(timeout=0) or []
                            except Exception as e:
                                self.logger.error(f"❌ Domain '{domain_name}': subdomain detection failed: {get_clean_error_message(e)}")
                                results[domain_name] = []
                    except concurrent.futures.TimeoutError:
                        executor.mark_timed_out()
                        timed_out = [d for d in domain_usecases_map if d not in results]
                        self.logger.error(f"⏱️ Subdomain detection timed out after {total_timeout}s; "
                                          f"table-evidence subdomains for: {timed_out}")
                        stuck = [(t.name, t.is_alive()) for t in threading.enumerate()
                                 if 'SubdomainDetect' in t.name or 'LLM_Query' in t.name]
                        self.logger.error(f"🧵 Stuck threads: {stuck}")
                        for d in timed_out:
                            results[d] = []
            except Exception as e:
                self.logger.error(f"❌ Parallel subdomain detection failed: {get_clean_error_message(e)}; "
                                  f"finishing the remaining domains sequentially")
        for idx, (domain_name, domain_use_cases) in enumerate(domain_usecases_map.items(), 1):
            if domain_name in results:
                continue
            self.logger.info(f"🔄 [{idx}/{num_domains}] Sequential subdomain detection: '{domain_name}' ({len(domain_use_cases)} use cases)")
            try:
                results[domain_name] = self._detect_subdomains_for_domain(
                    domain_name, domain_use_cases, language, timeout_seconds_override=_subdomain_call_budget) or []
            except Exception as e:
                self.logger.error(f"❌ [{idx}/{num_domains}] Domain '{domain_name}' failed: {get_clean_error_message(e)}")
                results[domain_name] = []
        final = []
        for domain_name, domain_use_cases in domain_usecases_map.items():
            final.extend(self._complete_subdomain_result(domain_name, domain_use_cases, results.get(domain_name) or []))
        return final

    def _complete_subdomain_result(self, domain_name, domain_use_cases, got) -> list:
        by_id = {}
        for uc in got or []:
            by_id.setdefault(str(uc.get('No', '')).strip(), []).append(uc)
        out, missing = [], []
        for uc in domain_use_cases:
            bucket = by_id.get(str(uc.get('No', '')).strip())
            if bucket:
                out.append(bucket.pop(0))
            else:
                missing.append(uc)
        if missing:
            if got:
                self.logger.warning(f"⚠️ Domain '{domain_name}': {len(missing)} use case(s) missing from the subdomain "
                                    f"result; table-evidence subdomains assigned")
            out.extend(self._assign_default_subdomains(missing, domain_name))
        return out

    def _ensure_clustering_complete(self, original, result) -> list:
        result = list(result or [])
        counts = {}
        for uc in result:
            key = str(uc.get('No', '')).strip()
            counts[key] = counts.get(key, 0) + 1
        missing = []
        for uc in original or []:
            key = str(uc.get('No', '')).strip()
            if counts.get(key, 0) > 0:
                counts[key] -= 1
            else:
                missing.append(uc)
        if not missing:
            return result
        restored = []
        for uc in missing:
            uc_copy = uc.copy()
            if not str(uc_copy.get('Business Domain') or '').strip():
                uc_copy = self._assign_default_domains([uc_copy])[0]
            if not str(uc_copy.get('Subdomain') or '').strip():
                self._apply_table_subdomains([uc_copy], uc_copy['Business Domain'])
            restored.append(uc_copy)
        self.logger.warning(
            f"[CLUSTERING_COMPLETENESS FIRED] {len(missing)} use case(s) were lost during clustering and are "
            f"restored with evidence-based assignments: {[uc.get('No') for uc in missing][:10]}"
        )
        try:
            NEXT_ACTIONS.add("CLUSTERING_FALLBACK", severity="HIGH", phase="Clustering", step="completeness",
                             evidence=f"{len(missing)} use case(s) lost during clustering were restored")
        except Exception:
            pass
        return result + restored

    @staticmethod
    def _llm_cost_mark() -> tuple:
        return (AIAgent._total_ai_calls, AIAgent._total_input_chars, AIAgent._total_output_chars)

    def _llm_cost_add(self, ledger, stage, mark) -> tuple:
        now = self._llm_cost_mark()
        entry = ledger.setdefault(stage, {"calls": 0, "input_chars": 0, "output_chars": 0})
        entry["calls"] += now[0] - mark[0]
        entry["input_chars"] += now[1] - mark[1]
        entry["output_chars"] += now[2] - mark[2]
        return now

    def _report_clustering_cost(self, ledger):
        self._honesty()["clustering_llm_cost"] = ledger
        self.logger.info(
            f"[CLUSTERING_LLM_COST] total_calls={sum(v['calls'] for v in ledger.values())} | "
            + " | ".join(f"{stage}={v['calls']} calls/~{(v['input_chars'] + v['output_chars']) // 4} tokens"
                         for stage, v in ledger.items())
        )

    @staticmethod
    def _compute_subdomain_call_budget(n_domains: int, floor_seconds: int = 300, per_domain_seconds: int = 60) -> int:
        """v1.0.3 B12: scale-adaptive per-call budget for SUBDOMAIN_DETECTOR_PROMPT.

        At 13-domain scale (tpcds), a 300s floor becomes 780s (13*60), giving the
        attempt-1 timeout ceiling `max(60, 780*0.5+30)=420s` in `_call_ai_query`,
        which comfortably covers the observed 180-350s per-call latency at that
        scale. Small corpora (<=5 domains) stay at the 300s floor unchanged.
        """
        try:
            _n = int(n_domains)
        except (TypeError, ValueError):
            _n = 0
        return max(int(floor_seconds), int(per_domain_seconds) * max(0, _n))

    def _detect_subdomains_for_domain(self, domain_name: str, use_cases: list, language: str,
                                       timeout_seconds_override: int = None) -> list:
        """
        Detect subdomains for a single domain's use cases using LLM.
        This method is designed to be called in parallel for each domain.
        A use case the LLM leaves without a subdomain is a violation that triggers a retry; after
        the last attempt it takes the subdomain of its same-primary-table siblings, else a
        table-derived one.
        
        Args:
            domain_name: Name of the domain
            use_cases: List of use case dictionaries belonging to this domain
            language: Output language
            timeout_seconds_override: v1.0.3 B12 — scale-adaptive per-call budget
                                       from `_compute_subdomain_call_budget(n_domains)`.
                                       Falls back to self.llm_timeout_seconds when None.
            
        Returns:
            List of use cases with updated subdomains
        """
        from collections import defaultdict
        import io
        import csv
        
        log_prefix = f"[Domain: {domain_name}]"
        self.logger.info(f"{log_prefix} 🚀 Starting subdomain detection for {len(use_cases)} use cases (thread={threading.current_thread().name})...")
        log_print(f"{log_prefix} Subdomain detection started ({len(use_cases)} use cases)")
        
        if not use_cases:
            return use_cases
        
        try:
            # Convert use cases to CSV for LLM (Business Domain is set, Subdomain will be detected)
            output = io.StringIO()
            fieldnames = ['No', 'Name', 'type', 'Analytics Technique', 'Statement', 'Solution', 
                         'Business Value', 'Beneficiary', 'Sponsor', 
                         'Tables Involved']
            writer = csv.DictWriter(output, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(use_cases)
            use_cases_csv = output.getvalue()
            
            # Check context size (uses model-specific limits from TECHNICAL_CONTEXT)
            prompt_template = self.ai_agent.prompt_templates.get("SUBDOMAIN_DETECTOR_PROMPT", "")
            estimated_size = self._prompt_overhead_chars("SUBDOMAIN_DETECTOR_PROMPT", prompt_template) + len(use_cases_csv) + 1000
            MAX_CONTEXT_CHARS = get_max_context_chars(language, "SUBDOMAIN_DETECTOR_PROMPT")
            
            if estimated_size > MAX_CONTEXT_CHARS:
                self.logger.warning(
                    f"{log_prefix} Subdomain prompt size ({estimated_size:,} chars) exceeds MAX_CONTEXT_CHARS ({MAX_CONTEXT_CHARS:,}). "
                    f"Splitting into smaller batches for subdomain detection."
                )
                # CONTEXT SPLITTING: Split use cases into smaller batches and process each
                return self._detect_subdomains_with_context_splitting(
                    domain_name, use_cases, language, prompt_template, MAX_CONTEXT_CHARS
                )
            
            max_attempts = self.max_retry_attempts + 1
            prompt_vars = {
                "domain_name": domain_name,
                "use_cases_csv": use_cases_csv,
                "output_language": language,
                "business_name": self.business_name,
                "industries": ", ".join(self.industries) if hasattr(self, 'industries') and self.industries else "UNKNOWN_INDUSTRY",
                "business_context": self._get_business_context_fallback(),
                "previous_violations": "",
                "generation_instructions_section": self._get_generation_instruction_for_prompt("SUBDOMAIN_DETECTOR_PROMPT"),
            }
            
            for attempt in range(1, max_attempts + 1):
                try:
                    self.logger.debug(f"{log_prefix} Subdomain detection attempt {attempt}/{max_attempts}...")
                    
                    _effective_timeout = timeout_seconds_override if timeout_seconds_override is not None else self.llm_timeout_seconds
                    response_raw = self.ai_agent.run_worker(
                        step_name=f"Detect_Subdomains_{domain_name}_{language}_Attempt{attempt}",
                        worker_prompt_path="SUBDOMAIN_DETECTOR_PROMPT",
                        prompt_vars=prompt_vars,
                        response_schema=None,
                        timeout_override=_effective_timeout,  # v1.0.3 B12: scale-adaptive when override supplied
                        skip_cache=(attempt > 1)  # v0.7.11: retries bypass cache
                    )
                    
                    # Validate response
                    if not response_raw or len(response_raw.strip()) == 0:
                        raise ValueError("LLM returned empty response")
                    
                    subdomain_assignment_map = self._parse_assignment_csv(response_raw, "subdomain", "Subdomain detection", use_cases)
                    if not subdomain_assignment_map:
                        self.logger.error(f"{log_prefix} CSV parsing failed. Raw response (first 500 chars): {str(response_raw)[:500]}")
                        raise ValueError("No valid subdomain assignments found in CSV")
                    
                    # Apply subdomain assignments
                    subdomain_assigned_use_cases = []
                    missing_ids = []
                    for uc in use_cases:
                        uc_copy = uc.copy()
                        uc_id = str(uc_copy.get('No', '')).strip()
                        uc_copy['Subdomain'] = subdomain_assignment_map.get(uc_id, '')
                        if not uc_copy['Subdomain']:
                            missing_ids.append(uc_id)
                        subdomain_assigned_use_cases.append(uc_copy)
                    
                    # Validate subdomain assignments
                    violations = []
                    if missing_ids:
                        violations.append(f"{len(missing_ids)} use case(s) missing a subdomain assignment: {', '.join(missing_ids[:20])}")
                    
                    # Group by subdomain
                    subdomain_usecases = defaultdict(list)
                    for uc in subdomain_assigned_use_cases:
                        subdomain = uc.get('Subdomain', '').strip()
                        if subdomain:
                            subdomain_usecases[subdomain].append(uc)
                    
                    total_subdomains = len(subdomain_usecases)
                    total_ucs_in_domain = sum(len(ucs) for ucs in subdomain_usecases.values())
                    min_subdomains = 1 if total_ucs_in_domain <= 4 else 2
                    if total_subdomains < min_subdomains:
                        violations.append(f"Only {total_subdomains} subdomain(s), minimum required: {min_subdomains}")
                    if total_subdomains > 10:
                        violations.append(f"Too many subdomains: {total_subdomains}, maximum allowed: 10")
                    
                    # Validate subdomain naming (must be 2 words)
                    for subdomain in subdomain_usecases.keys():
                        word_count = len(subdomain.split())
                        if word_count != 2:
                            violations.append(f"Subdomain '{subdomain}' has {word_count} word(s), must be exactly 2 words")
                    
                    # Validate use cases per subdomain (minimum 2)
                    for subdomain, ucs in subdomain_usecases.items():
                        count = len(ucs)
                        if count < 2:
                            violations.append(f"Subdomain '{subdomain}' has only {count} use case(s), minimum required: 2")
                    
                    # If no violations, success!
                    if not violations:
                        self.logger.debug(f"{log_prefix} ✅ Subdomain detection successful! Created {total_subdomains} subdomains")
                        return subdomain_assigned_use_cases
                    else:
                        self.logger.warning(f"{log_prefix} ⚠️ Subdomain detection attempt {attempt} has {len(violations)} violations")
                        if attempt == max_attempts:
                            self.logger.warning(f"{log_prefix} Max attempts reached. Auto-fixing subdomain violations...")
                            gaps = [uc for uc in subdomain_assigned_use_cases if not uc.get('Subdomain')]
                            if gaps:
                                assigned = [uc for uc in subdomain_assigned_use_cases if uc.get('Subdomain')]
                                rest = self._fill_by_table_evidence(gaps, assigned, 'Subdomain', 'subdomain:table_evidence')
                                self._apply_table_subdomains(rest, domain_name)
                                self.logger.warning(f"{log_prefix} {len(gaps)} use case(s) without an LLM subdomain assigned from table evidence")
                                subdomain_usecases = defaultdict(list)
                                for uc in subdomain_assigned_use_cases:
                                    subdomain_usecases[uc['Subdomain'].strip()].append(uc)
                            small_subdomains = [sd for sd, ucs in subdomain_usecases.items() if len(ucs) < 2]
                            if small_subdomains and len(subdomain_usecases) > 1:
                                largest_subdomain = max(subdomain_usecases.keys(), key=lambda sd: len(subdomain_usecases[sd]))
                                for small_sd in small_subdomains:
                                    if small_sd == largest_subdomain:
                                        continue
                                    self.logger.warning(f"{log_prefix} Merging subdomain '{small_sd}' ({len(subdomain_usecases[small_sd])} use cases) into '{largest_subdomain}'")
                                    for uc in subdomain_assigned_use_cases:
                                        if uc.get('Subdomain', '').strip() == small_sd:
                                            uc['Subdomain'] = largest_subdomain
                                    subdomain_usecases[largest_subdomain].extend(subdomain_usecases.pop(small_sd))
                            
                            max_subdomains = 10
                            
                            def _find_best_merge_target(src_sd, candidates, sd_usecases):
                                src_words = set(src_sd.lower().split())
                                best_target = None
                                best_score = -1
                                for cand in candidates:
                                    if cand == src_sd:
                                        continue
                                    cand_words = set(cand.lower().split())
                                    name_overlap = sum(
                                        1 for uc in sd_usecases[cand]
                                        if any(w in uc.get('Name', '').lower() for w in src_words)
                                    )
                                    word_overlap = len(src_words & cand_words)
                                    score = word_overlap * 10 + name_overlap
                                    if score > best_score:
                                        best_score = score
                                        best_target = cand
                                if best_target is None:
                                    best_target = min(
                                        (c for c in candidates if c != src_sd),
                                        key=lambda c: len(sd_usecases[c])
                                    )
                                return best_target
                            
                            while len(subdomain_usecases) > max_subdomains:
                                sorted_sds = sorted(subdomain_usecases.keys(), key=lambda sd: len(subdomain_usecases[sd]))
                                smallest_sd = sorted_sds[0]
                                merge_target = _find_best_merge_target(smallest_sd, list(subdomain_usecases.keys()), subdomain_usecases)
                                self.logger.warning(
                                    f"{log_prefix} Too many subdomains ({len(subdomain_usecases)}>{max_subdomains}). "
                                    f"Merging '{smallest_sd}' ({len(subdomain_usecases[smallest_sd])} UCs) into '{merge_target}' ({len(subdomain_usecases[merge_target])} UCs)"
                                )
                                for uc in subdomain_assigned_use_cases:
                                    if uc.get('Subdomain', '').strip() == smallest_sd:
                                        uc['Subdomain'] = merge_target
                                subdomain_usecases[merge_target].extend(subdomain_usecases.pop(smallest_sd))
                            
                            remaining_violations = [v for v in violations if "has only" not in v and "Too many subdomains" not in v and "missing a subdomain" not in v]
                            if remaining_violations:
                                self.logger.warning(f"{log_prefix} Remaining violations after auto-fix: {'; '.join(remaining_violations[:3])}")
                            return subdomain_assigned_use_cases
                        else:
                            # Prepare violation summary for retry
                            violation_summary = "\n\n**🚨 PREVIOUS ATTEMPT VIOLATIONS - YOU MUST FIX THESE 🚨**:\n"
                            violation_summary += "\n".join([f"- {v}" for v in violations])
                            prompt_vars['previous_violations'] = violation_summary
                            continue
                
                except CascadeRebatchError as cre:
                    self.logger.warning(
                        f"{log_prefix} Subdomain detection cascade rebatch needed after '{cre.failed_model_name}' failed. "
                        f"Switching to context-splitting for model '{cre.target_model_name}' "
                        f"(limit: {cre.target_context_limit_chars:,} chars)"
                    )
                    return self._detect_subdomains_with_context_splitting(
                        domain_name, use_cases, language, prompt_template,
                        cre.target_context_limit_chars, cre.target_model_endpoint
                    )

                except Exception as e:
                    self.logger.warning(f"{log_prefix} Subdomain detection attempt {attempt} failed: {get_clean_error_message(e)}")
                    if attempt == max_attempts:
                        self.logger.error(f"{log_prefix} Max attempts reached. Using evidence-based subdomains as fallback...")
                        fallback_use_cases = self._assign_default_subdomains(use_cases, domain_name)
                        self.logger.warning(f"{log_prefix} ✅ Fallback complete: Assigned table-derived subdomains to {len(fallback_use_cases)} use cases")
                        return fallback_use_cases
            
            self.logger.warning(f"{log_prefix} No subdomain assignments made. Using evidence-based subdomains as fallback...")
            return self._assign_default_subdomains(use_cases, domain_name)
            
        except Exception as e:
            self.logger.error(f"{log_prefix} Subdomain detection failed with error: {get_clean_error_message(e)}. Using evidence-based subdomains as fallback...")
            return self._assign_default_subdomains(use_cases, domain_name)

    def _detect_subdomains_with_context_splitting(self, domain_name: str, use_cases: list, language: str, prompt_template: str, max_context_chars: int, start_from_model: str = None) -> list:
        """
        Handle subdomain detection when context exceeds max size by splitting into batches.
        Processes each batch separately and merges results. Supports cascade rebatching.
        A use case no batch assigns takes the subdomain of its same-primary-table siblings, else a
        table-derived one.
        
        Args:
            domain_name: Name of the domain
            use_cases: List of use case dictionaries
            language: Output language
            prompt_template: The prompt template string
            max_context_chars: Maximum context size in characters
            start_from_model: Optional model endpoint to start cascade from
            
        Returns:
            List of use cases with assigned subdomains
        """
        import io
        import csv
        from collections import defaultdict
        
        log_prefix = f"[Domain: {domain_name}][Context Split]"
        
        MAX_CASCADE_RETRIES = 3
        effective_limit = max_context_chars
        effective_start_model = start_from_model
        sample_fieldnames = ['No', 'Name', 'type', 'Analytics Technique', 'Statement', 'Solution', 
                            'Business Value', 'Beneficiary', 'Sponsor', 'Tables Involved']

        all_subdomain_assignments = {}
        _sub_lock = threading.Lock()
        remaining_use_cases = list(use_cases)

        for cascade_retry in range(MAX_CASCADE_RETRIES):
            if not remaining_use_cases:
                break

            sample_output = io.StringIO()
            sample_writer = csv.DictWriter(sample_output, fieldnames=sample_fieldnames, extrasaction='ignore')
            sample_writer.writeheader()
            if remaining_use_cases:
                sample_writer.writerow(remaining_use_cases[0])
            avg_use_case_size = max(100, len(sample_output.getvalue()))

            available_chars = effective_limit - self._prompt_overhead_chars("SUBDOMAIN_DETECTOR_PROMPT", prompt_template) - 2000
            batch_size = max(2, available_chars // max(1, avg_use_case_size))

            self.logger.info(
                f"{log_prefix} Splitting {len(remaining_use_cases)} use cases into batches of ~{batch_size} "
                f"(context limit: {effective_limit:,} chars, model: {effective_start_model or 'default cascade'}, "
                f"already assigned: {len(all_subdomain_assignments)})"
            )

            batches = []
            for i in range(0, len(remaining_use_cases), batch_size):
                batches.append(remaining_use_cases[i:i + batch_size])

            self.logger.info(f"{log_prefix} Created {len(batches)} batches for subdomain detection")

            num_batches = len(batches)
            cascade_rebatch_triggered = False
            cascade_rebatch_error = None

            def _process_subdomain_batch(batch_idx, batch, _start_model=effective_start_model):
                self.logger.info(f"{log_prefix} Processing batch {batch_idx}/{num_batches} ({len(batch)} use cases)")
                local_assignments = {}
                try:
                    batch_output = io.StringIO()
                    batch_writer = csv.DictWriter(batch_output, fieldnames=sample_fieldnames, extrasaction='ignore')
                    batch_writer.writeheader()
                    batch_writer.writerows(batch)
                    batch_csv = batch_output.getvalue()

                    max_attempts = self.max_retry_attempts + 1
                    prompt_vars = {
                        "domain_name": domain_name,
                        "use_cases_csv": batch_csv,
                        "output_language": language,
                        "business_name": self.business_name,
                        "industries": ", ".join(self.industries) if hasattr(self, 'industries') and self.industries else "UNKNOWN_INDUSTRY",
                        "business_context": self._get_business_context_fallback(),
                        "previous_violations": "",
                        "generation_instructions_section": self._get_generation_instruction_for_prompt("SUBDOMAIN_DETECTOR_PROMPT"),
                    }

                    for attempt in range(1, max_attempts + 1):
                        try:
                            response_raw = self.ai_agent.run_worker(
                                step_name=f"Detect_Subdomains_{domain_name}_Batch{batch_idx}_{language}_Attempt{attempt}",
                                worker_prompt_path="SUBDOMAIN_DETECTOR_PROMPT",
                                prompt_vars=prompt_vars,
                                response_schema=None,
                                timeout_override=self.llm_timeout_seconds,
                                start_from_model=_start_model,
                                skip_cache=(attempt > 1)  # v0.7.11: retries bypass cache
                            )

                            if not response_raw or len(response_raw.strip()) == 0:
                                raise ValueError("LLM returned empty response")

                            parsed = self._parse_assignment_csv(response_raw, "subdomain", f"Subdomain detection batch {batch_idx}", batch)
                            if not parsed:
                                raise ValueError("No valid subdomain assignments found in CSV")
                            local_assignments.update(parsed)
                            self.logger.info(f"{log_prefix} Batch {batch_idx} completed successfully")
                            break

                        except CascadeRebatchError:
                            raise
                        except Exception as attempt_err:
                            self.logger.warning(f"{log_prefix} Batch {batch_idx} attempt {attempt} failed: {attempt_err}")
                            if attempt == max_attempts:
                                self.logger.error(f"{log_prefix} Batch {batch_idx} failed after {max_attempts} attempts")

                except CascadeRebatchError:
                    raise
                except Exception as batch_err:
                    self.logger.error(f"{log_prefix} Batch {batch_idx} processing error: {batch_err}")

                with _sub_lock:
                    all_subdomain_assignments.update(local_assignments)

            batch_workers = min(4, num_batches)
            try:
                if num_batches <= 1:
                    _process_subdomain_batch(1, batches[0]) if batches else None
                else:
                    _sb_total_timeout = self.llm_timeout_seconds * 3 * len(batches) + 120
                    with _SafeExecutorContext(max_workers=batch_workers, thread_name_prefix="sub_batch", logger=self.logger, name="SubdomainBatch") as ctx:
                        sb_futures = [ctx.submit(_process_subdomain_batch, i, b) for i, b in enumerate(batches, 1)]
                        try:
                            for f in ctx.completed(sb_futures, timeout=_sb_total_timeout):
                                try:
                                    f.result(timeout=self.llm_timeout_seconds * 3)
                                except CascadeRebatchError as cre:
                                    cascade_rebatch_triggered = True
                                    cascade_rebatch_error = cre
                                    ctx.mark_timed_out()
                                    for remaining_f in sb_futures:
                                        if not remaining_f.done():
                                            remaining_f.cancel()
                                    break
                                except Exception as exc:
                                    self.logger.error(f"{log_prefix} Subdomain batch task failed: {get_clean_error_message(exc)}")
                        except concurrent.futures.TimeoutError:
                            ctx.mark_timed_out()
                            self.logger.error(f"{log_prefix} Subdomain batch timed out globally ({_sb_total_timeout}s)")
            except CascadeRebatchError as cre:
                cascade_rebatch_triggered = True
                cascade_rebatch_error = cre

            if cascade_rebatch_triggered and cascade_rebatch_error:
                effective_limit = cascade_rebatch_error.target_context_limit_chars
                effective_start_model = cascade_rebatch_error.target_model_endpoint
                remaining_use_cases = [
                    uc for uc in remaining_use_cases
                    if str(uc.get('No', '')).strip() not in all_subdomain_assignments
                ]
                self.logger.warning(
                    f"{log_prefix} CASCADE RETRY {cascade_retry + 1}/{MAX_CASCADE_RETRIES}: "
                    f"Rebatching {len(remaining_use_cases)} remaining use cases for model "
                    f"'{cascade_rebatch_error.target_model_name}' (limit: {effective_limit:,} chars, "
                    f"already assigned: {len(all_subdomain_assignments)})"
                )
                if not remaining_use_cases:
                    self.logger.info(f"{log_prefix} All use cases assigned despite cascade error")
                    break
                continue

            break

        result = []
        for uc in use_cases:
            uc_copy = uc.copy()
            uc_copy['Subdomain'] = all_subdomain_assignments.get(str(uc.get('No', '')).strip(), '')
            result.append(uc_copy)
        gaps = [uc for uc in result if not uc['Subdomain']]
        if gaps:
            rest = self._fill_by_table_evidence(gaps, [uc for uc in result if uc['Subdomain']], 'Subdomain', 'subdomain:table_evidence')
            self._apply_table_subdomains(rest, domain_name)
            self.logger.warning(f"{log_prefix} {len(gaps)} use case(s) received no LLM subdomain; assigned from table evidence")

        subdomain_counts = defaultdict(int)
        for uc in result:
            subdomain_counts[uc.get('Subdomain', 'Unknown')] += 1

        self.logger.info(f"{log_prefix} Context splitting complete: {len(result)} use cases assigned to {len(subdomain_counts)} subdomains")
        for subdomain, count in sorted(subdomain_counts.items(), key=lambda x: -x[1]):
            self.logger.debug(f"{log_prefix}   - {subdomain}: {count} use cases")

        return result

    def _report_table_statistics(self, use_cases: list):
        """
        Reports statistics on table inclusion/exclusion based on generated use cases.
        Compares the tables used in use cases against every discovered table.
        """
        try:
            all_available_tables = set()
            for tup in (getattr(self.data_loader, 'all_table_tuples', None) or []):
                try:
                    all_available_tables.add(".".join(str(x) for x in tuple(tup)[:3]).lower())
                except Exception:
                    continue
            
            total_available = len(all_available_tables)
            
            if total_available == 0:
                self.logger.warning("No discovered tables available in the data loader. Skipping table statistics report.")
                return
            
            tables_used = set()
            for uc in use_cases:
                for table in self._uc_table_refs(uc):
                    if table in all_available_tables:
                        tables_used.add(table)
            
            total_included = len(tables_used)
            total_excluded = total_available - total_included
            
            # Calculate percentages
            pct_included = (total_included / total_available * 100) if total_available > 0 else 0
            pct_excluded = (total_excluded / total_available * 100) if total_available > 0 else 0
            
            log_print("\n" + "="*80)
            log_print("TABLE UTILIZATION REPORT")
            log_print("="*80)
            log_print(f"Total Available Tables: {total_available}")
            log_print(f"Tables Included in Use Cases: {total_included} ({pct_included:.1f}%)")
            log_print(f"Tables Excluded (No Business Value): {total_excluded} ({pct_excluded:.1f}%)")
            log_print("="*80 + "\n")
            
            # Log it as well
            self.logger.info(f"Table Statistics: {total_included}/{total_available} tables included ({pct_included:.1f}%), {total_excluded} excluded ({pct_excluded:.1f}%)")
            
        except Exception as e:
            self.logger.error(f"Failed to generate table statistics report: {get_clean_error_message(e)}")
    
    def _priority_sort_key(self, use_case: dict) -> tuple:
        return self._final_rank_key(use_case)
    
    def _natural_sort_key(self, use_case: dict) -> tuple:
        """
        Natural sort key for use case IDs to ensure proper ordering.
        
        Handles IDs like: N01-AI01, N01-AI02, ..., N01-AI10, N01-AI11, N01-ST01, etc.
        Standard string sort would incorrectly order: AI1, AI10, AI11, AI2, AI3...
        Natural sort correctly orders: AI1, AI2, AI3, ..., AI10, AI11...
        
        Returns tuple: (domain_num, source_type, sequence_num, original_id)
        Example: "N01-AI05" -> (1, 'AI', 5, 'N01-AI05')
        """
        import re
        use_case_id = str(use_case.get('No') or 'N99-ZZ999').strip()
        
        # Parse ID format: N##-XX## (e.g., N01-AI05, N02-ST10)
        match = re.match(r'N(\d+)-([A-Z]+)(\d+)', use_case_id)
        if match:
            domain_num = int(match.group(1))
            source_type = match.group(2)  # AI or ST
            seq_num = int(match.group(3))
            # Sort AI before ST (alphabetically)
            return (domain_num, source_type, seq_num, use_case_id)
        
        # Fallback: Try to parse older formats like AI-XXX-U## or just extract numbers
        # Pattern: AI-XXX-U## or ST-XXX-U##
        match2 = re.match(r'(AI|ST)-([A-Z0-9_]+)-U(\d+)', use_case_id)
        if match2:
            source_type = match2.group(1)
            seq_num = int(match2.group(3))
            return (99, source_type, seq_num, use_case_id)
        
        # Final fallback: extract any numbers from the ID for some ordering
        numbers = re.findall(r'\d+', use_case_id)
        if numbers:
            # Use first number as domain, last number as sequence
            return (int(numbers[0]) if len(numbers) > 0 else 99, 
                    'ZZ', 
                    int(numbers[-1]) if len(numbers) > 0 else 999, 
                    use_case_id)
        
        # No numbers found - sort at the end
        return (999, 'ZZ', 999, use_case_id)
    
    _DISPLAY_VALUE_KEYS = {
        'Problem': 'value_type_problem', 'Risk': 'value_type_risk',
        'Opportunity': 'value_type_opportunity', 'Improvement': 'value_type_improvement',
        'Ultra High': 'value_priority_ultra_high', 'Very High': 'value_priority_very_high',
        'High': 'value_priority_high', 'Medium': 'value_priority_medium',
        'Low': 'value_priority_low', 'Very Low': 'value_priority_very_low',
        'Ultra Low': 'value_priority_ultra_low'
    }

    _ALIGNMENT_VALUE_KEYS = {
        'General Improvement': 'value_general_improvement',
        'Not aligned': 'value_not_aligned',
        'Reduce Cost': 'value_reduce_cost',
        'Increase Revenue': 'value_increase_revenue',
        'Boost Productivity': 'value_boost_productivity',
        'Mitigate Risk': 'value_mitigate_risk',
        'Protect Revenue': 'value_protect_revenue',
        'Align to Regulations': 'value_align_to_regulations',
        'Improve Customer Experience': 'value_improve_customer_experience',
        'Enable Data-Driven Decisions': 'value_enable_data_driven_decisions',
        'Optimize Operations': 'value_optimize_operations',
        'Empower Talent': 'value_empower_talent',
        'Enhance Experience': 'value_enhance_experience',
        'Drive Innovation': 'value_drive_innovation',
        'Achieve ESG': 'value_achieve_esg',
        'Execute Strategy': 'value_execute_strategy',
    }

    _TECHNIQUE_VALUE_KEYS = {
        'Correlation Analysis': 'value_correlation_analysis', 'Pareto Analysis': 'value_pareto_analysis',
        'Funnel Analysis': 'value_funnel_analysis', 'Document Processing': 'value_document_processing',
        'Extraction': 'value_extraction', 'AI Analysis': 'value_ai_analysis', 'Simulation': 'value_simulation',
        'Geospatial Analysis': 'value_geospatial_analysis',
        'Forecasting': 'value_forecasting',
        'Classification': 'value_classification',
        'Anomaly Detection': 'value_anomaly_detection',
        'Cohort Analysis': 'value_cohort_analysis',
        'Segmentation': 'value_segmentation',
        'Sentiment Analysis': 'value_sentiment_analysis',
        'Trend Analysis': 'value_trend_analysis',
        'Prescriptive Analytics': 'value_prescriptive_analytics',
        'Root Cause Analysis': 'value_root_cause_analysis',
        'Optimization': 'value_optimization',
        'Recommendation': 'value_recommendation',
        'Time Series Analysis': 'value_time_series_analysis',
        'Predictive Analytics': 'value_predictive_analytics',
        'Descriptive Analytics': 'value_descriptive_analytics',
        'Predictive Modeling': 'value_predictive_modeling',
        'Clustering': 'value_clustering',
        'Regression Analysis': 'value_regression_analysis',
        'Survival Analysis': 'value_survival_analysis',
        'Network Analysis': 'value_network_analysis',
    }

    _TECHNIQUE_FALLBACKS = {
        'Chinese (Mandarin)': {'Forecasting': '预测', 'Classification': '分类', 'Anomaly Detection': '异常检测', 'Cohort Analysis': '队列分析', 'Segmentation': '细分', 'Sentiment Analysis': '情感分析', 'Trend Analysis': '趋势分析', 'Prescriptive Analytics': '规范性分析', 'Root Cause Analysis': '根因分析', 'Optimization': '优化', 'Recommendation': '推荐', 'Time Series Analysis': '时间序列分析', 'Predictive Analytics': '预测分析', 'Descriptive Analytics': '描述性分析', 'Predictive Modeling': '预测建模', 'Clustering': '聚类分析', 'Regression Analysis': '回归分析', 'Survival Analysis': '生存分析', 'Network Analysis': '网络分析'},
        'Arabic': {'Forecasting': 'التنبؤ', 'Classification': 'التصنيف', 'Anomaly Detection': 'كشف الشذوذ', 'Cohort Analysis': 'تحليل الأتراب', 'Segmentation': 'التجزئة', 'Sentiment Analysis': 'تحليل المشاعر', 'Trend Analysis': 'تحليل الاتجاهات', 'Prescriptive Analytics': 'التحليلات التوجيهية', 'Root Cause Analysis': 'تحليل السبب الجذري', 'Optimization': 'التحسين', 'Recommendation': 'التوصية', 'Time Series Analysis': 'تحليل السلاسل الزمنية', 'Predictive Analytics': 'التحليلات التنبؤية', 'Descriptive Analytics': 'التحليلات الوصفية', 'Predictive Modeling': 'النمذجة التنبؤية', 'Clustering': 'التجميع', 'Regression Analysis': 'تحليل الانحدار', 'Survival Analysis': 'تحليل البقاء', 'Network Analysis': 'تحليل الشبكات'},
        'Spanish': {'Forecasting': 'Pronóstico', 'Classification': 'Clasificación', 'Anomaly Detection': 'Detección de Anomalías', 'Cohort Analysis': 'Análisis de Cohortes', 'Segmentation': 'Segmentación', 'Sentiment Analysis': 'Análisis de Sentimiento', 'Trend Analysis': 'Análisis de Tendencias', 'Prescriptive Analytics': 'Analítica Prescriptiva', 'Root Cause Analysis': 'Análisis de Causa Raíz', 'Optimization': 'Optimización', 'Recommendation': 'Recomendación', 'Time Series Analysis': 'Análisis de Series Temporales', 'Predictive Analytics': 'Analítica Predictiva', 'Descriptive Analytics': 'Analítica Descriptiva', 'Predictive Modeling': 'Modelado Predictivo', 'Clustering': 'Agrupamiento', 'Regression Analysis': 'Análisis de Regresión', 'Survival Analysis': 'Análisis de Supervivencia', 'Network Analysis': 'Análisis de Redes'},
        'French': {'Forecasting': 'Prévision', 'Classification': 'Classification', 'Anomaly Detection': 'Détection d\'Anomalies', 'Cohort Analysis': 'Analyse de Cohortes', 'Segmentation': 'Segmentation', 'Sentiment Analysis': 'Analyse de Sentiments', 'Trend Analysis': 'Analyse des Tendances', 'Prescriptive Analytics': 'Analytique Prescriptive', 'Root Cause Analysis': 'Analyse des Causes Profondes', 'Optimization': 'Optimisation', 'Recommendation': 'Recommandation', 'Time Series Analysis': 'Analyse de Séries Temporelles', 'Predictive Analytics': 'Analytique Prédictive', 'Descriptive Analytics': 'Analytique Descriptive', 'Predictive Modeling': 'Modélisation Prédictive', 'Clustering': 'Regroupement', 'Regression Analysis': 'Analyse de Régression', 'Survival Analysis': 'Analyse de Survie', 'Network Analysis': 'Analyse de Réseaux'},
        'German': {'Forecasting': 'Vorhersage', 'Classification': 'Klassifikation', 'Anomaly Detection': 'Anomalieerkennung', 'Cohort Analysis': 'Kohortenanalyse', 'Segmentation': 'Segmentierung', 'Sentiment Analysis': 'Stimmungsanalyse', 'Trend Analysis': 'Trendanalyse', 'Prescriptive Analytics': 'Präskriptive Analytik', 'Root Cause Analysis': 'Ursachenanalyse', 'Optimization': 'Optimierung', 'Recommendation': 'Empfehlung', 'Time Series Analysis': 'Zeitreihenanalyse', 'Predictive Analytics': 'Prädiktive Analytik', 'Descriptive Analytics': 'Deskriptive Analytik', 'Predictive Modeling': 'Prädiktive Modellierung', 'Clustering': 'Clusteranalyse', 'Regression Analysis': 'Regressionsanalyse', 'Survival Analysis': 'Überlebensanalyse', 'Network Analysis': 'Netzwerkanalyse'},
        'Portuguese': {'Forecasting': 'Previsão', 'Classification': 'Classificação', 'Anomaly Detection': 'Detecção de Anomalias', 'Cohort Analysis': 'Análise de Coorte', 'Segmentation': 'Segmentação', 'Sentiment Analysis': 'Análise de Sentimento', 'Trend Analysis': 'Análise de Tendências', 'Prescriptive Analytics': 'Análise Prescritiva', 'Root Cause Analysis': 'Análise de Causa Raiz', 'Optimization': 'Otimização', 'Recommendation': 'Recomendação', 'Time Series Analysis': 'Análise de Séries Temporais', 'Predictive Analytics': 'Análise Preditiva', 'Descriptive Analytics': 'Análise Descritiva', 'Predictive Modeling': 'Modelagem Preditiva', 'Clustering': 'Agrupamento', 'Regression Analysis': 'Análise de Regressão', 'Survival Analysis': 'Análise de Sobrevivência', 'Network Analysis': 'Análise de Redes'},
        'Italian': {'Forecasting': 'Previsione', 'Classification': 'Classificazione', 'Anomaly Detection': 'Rilevamento Anomalie', 'Cohort Analysis': 'Analisi di Coorte', 'Segmentation': 'Segmentazione', 'Sentiment Analysis': 'Analisi del Sentimento', 'Trend Analysis': 'Analisi delle Tendenze', 'Prescriptive Analytics': 'Analisi Prescrittiva', 'Root Cause Analysis': 'Analisi delle Cause Profonde', 'Optimization': 'Ottimizzazione', 'Recommendation': 'Raccomandazione', 'Time Series Analysis': 'Analisi delle Serie Temporali', 'Predictive Analytics': 'Analisi Predittiva', 'Descriptive Analytics': 'Analisi Descrittiva', 'Predictive Modeling': 'Modellazione Predittiva', 'Clustering': 'Raggruppamento', 'Regression Analysis': 'Analisi di Regressione', 'Survival Analysis': 'Analisi di Sopravvivenza', 'Network Analysis': 'Analisi di Rete'},
        'Japanese': {'Forecasting': '予測', 'Classification': '分類', 'Anomaly Detection': '異常検出', 'Cohort Analysis': 'コホート分析', 'Segmentation': 'セグメンテーション', 'Sentiment Analysis': '感情分析', 'Trend Analysis': 'トレンド分析', 'Prescriptive Analytics': '処方的分析', 'Root Cause Analysis': '根本原因分析', 'Optimization': '最適化', 'Recommendation': 'レコメンデーション', 'Time Series Analysis': '時系列分析', 'Predictive Analytics': '予測分析', 'Descriptive Analytics': '記述分析', 'Predictive Modeling': '予測モデリング', 'Clustering': 'クラスタリング', 'Regression Analysis': '回帰分析', 'Survival Analysis': '生存分析', 'Network Analysis': 'ネットワーク分析'},
        'Korean': {'Forecasting': '예측', 'Classification': '분류', 'Anomaly Detection': '이상 탐지', 'Cohort Analysis': '코호트 분석', 'Segmentation': '세분화', 'Sentiment Analysis': '감정 분석', 'Trend Analysis': '추세 분석', 'Prescriptive Analytics': '처방적 분석', 'Root Cause Analysis': '근본 원인 분석', 'Optimization': '최적화', 'Recommendation': '추천', 'Time Series Analysis': '시계열 분석', 'Predictive Analytics': '예측 분석', 'Descriptive Analytics': '기술 분석', 'Predictive Modeling': '예측 모델링', 'Clustering': '클러스터링', 'Regression Analysis': '회귀 분석', 'Survival Analysis': '생존 분석', 'Network Analysis': '네트워크 분석'},
        'Hindi': {'Forecasting': 'पूर्वानुमान', 'Classification': 'वर्गीकरण', 'Anomaly Detection': 'विसंगति पता लगाना', 'Cohort Analysis': 'समूह विश्लेषण', 'Segmentation': 'विभाजन', 'Sentiment Analysis': 'भावना विश्लेषण', 'Trend Analysis': 'रुझान विश्लेषण', 'Prescriptive Analytics': 'निर्देशात्मक विश्लेषण', 'Root Cause Analysis': 'मूल कारण विश्लेषण', 'Optimization': 'अनुकूलन', 'Recommendation': 'सिफारिश', 'Time Series Analysis': 'समय श्रृंखला विश्लेषण', 'Predictive Analytics': 'भविष्य कथन विश्लेषण', 'Descriptive Analytics': 'वर्णनात्मक विश्लेषण', 'Predictive Modeling': 'भविष्य कथन मॉडलिंग', 'Clustering': 'क्लस्टरिंग', 'Regression Analysis': 'प्रतिगमन विश्लेषण', 'Survival Analysis': 'अस्तित्व विश्लेषण', 'Network Analysis': 'नेटवर्क विश्लेषण'},
        'Russian': {'Forecasting': 'Прогнозирование', 'Classification': 'Классификация', 'Anomaly Detection': 'Обнаружение Аномалий', 'Cohort Analysis': 'Когортный Анализ', 'Segmentation': 'Сегментация', 'Sentiment Analysis': 'Анализ Настроений', 'Trend Analysis': 'Анализ Трендов', 'Prescriptive Analytics': 'Предписывающая Аналитика', 'Root Cause Analysis': 'Анализ Первопричин', 'Optimization': 'Оптимизация', 'Recommendation': 'Рекомендация', 'Time Series Analysis': 'Анализ Временных Рядов', 'Predictive Analytics': 'Предиктивная Аналитика', 'Descriptive Analytics': 'Описательная Аналитика', 'Predictive Modeling': 'Предиктивное Моделирование', 'Clustering': 'Кластеризация', 'Regression Analysis': 'Регрессионный Анализ', 'Survival Analysis': 'Анализ Выживаемости', 'Network Analysis': 'Сетевой Анализ'},
    }

    def _translate_display_value(self, value, t: dict):
        if not value or value == 'N/A':
            return value
        translation_key = self._DISPLAY_VALUE_KEYS.get(value)
        return t.get(translation_key, value) if translation_key else value

    def _translate_alignment_value(self, value, t: dict):
        if not value or value == 'N/A':
            return value
        if ',' in str(value):
            translated_parts = []
            for part in [p.strip() for p in str(value).split(',')]:
                key = self._ALIGNMENT_VALUE_KEYS.get(part)
                translated_parts.append(t.get(key, part) if key else part)
            return ', '.join(translated_parts)
        translation_key = self._ALIGNMENT_VALUE_KEYS.get(value)
        return t.get(translation_key, value) if translation_key else value

    @staticmethod
    def _technique_key(value) -> str:
        words = re.sub(r"[\s_\-/]+", " ", unicodedata.normalize("NFKC", str(value or "")).casefold()).split()
        words = [word.replace("modelling", "modeling").replace("isation", "ization") for word in words]
        words = [word[:-4] + "ysis" if word.endswith("yses") else
                 word[:-1] if len(word) > 3 and word.endswith("s") and not word.endswith(("ss", "is", "us")) else word
                 for word in words]
        return " ".join(words)

    @classmethod
    def _canonical_technique(cls, value):
        key = cls._technique_key(value)
        if not key:
            return None
        candidates = {key}
        for suffix in (" analysis", " analytic"):
            if key.endswith(suffix) and len(key) > len(suffix):
                candidates.add(key[:-len(suffix)])
        for label in cls._TECHNIQUE_VALUE_KEYS:
            if cls._technique_key(label) in candidates:
                return label
        for _family, label, aliases, _methods in cls._TECHNIQUE_FAMILIES:
            if any(cls._technique_key(alias) in candidates for alias in aliases):
                return label
        return None

    def _translate_technique_value(self, value, t: dict, language: str):
        if not value or value == 'N/A':
            return value
        translation_key = self._TECHNIQUE_VALUE_KEYS.get(value)
        translated = t.get(translation_key, None) if translation_key else None
        if translated and translated != value:
            return translated
        return self._TECHNIQUE_FALLBACKS.get(language, {}).get(value, value)

    def _rank_domain_groups(self, grouped_data: dict) -> list:
        groups = []
        for domain, use_cases in (grouped_data or {}).items():
            ranked = sorted(use_cases or [], key=self._final_rank_key)
            if ranked:
                groups.append((domain, ranked))
        groups.sort(key=lambda group: self._final_rank_key(group[1][0]))
        return groups

    def _ranked_use_cases(self, grouped_data: dict) -> list:
        flat = [uc for use_cases in (grouped_data or {}).values() for uc in (use_cases or [])]
        return sorted(flat, key=self._final_rank_key)

    NOT_SCORED_LABEL = "Not scored"

    def _display_score(self, value) -> str:
        number = self._finite_score(value)
        return self.NOT_SCORED_LABEL if number is None else f"{number:.2f}"

    _KPI_UP = re.compile(r"\b(increase|increasing|up|higher|maximi[sz]e|raise|grow|growth)\b", re.I)
    _KPI_DOWN = re.compile(r"\b(decrease|decreasing|down|lower|minimi[sz]e|reduce|reduction|cut|decline)\b", re.I)

    @classmethod
    def _kpi_direction_arrow(cls, direction) -> str:
        text = str(direction or "").strip()
        if not text:
            return ""
        up, down = bool(cls._KPI_UP.search(text)), bool(cls._KPI_DOWN.search(text))
        if up and not down:
            return "↑"
        if down and not up:
            return "↓"
        return text

    _DOC_FALLBACK_LABEL_KEYS = {"near_miss": "elite_fallback_near_miss", "tier": "elite_fallback_pick"}

    def _elite_presentation(self, uc: dict) -> dict:
        status = uc.get("elite_status")
        if status == "DOMAIN_COVERAGE_PICK":
            return {"kind": "coverage", "reason": inspire_plain_text(uc.get("coverage_pick_reason")),
                    "label_key": "elite_coverage_pick", "mode": None}
        if status == "FALLBACK_BELOW_BAR":
            label_key = self._DOC_FALLBACK_LABEL_KEYS.get(uc.get("fallback_kind"), "elite_fallback_pick")
            return {"kind": "coverage", "reason": inspire_plain_text(uc.get("fallback_reason")),
                    "label_key": label_key, "mode": uc.get("fallback_bar")}
        return {"kind": "elite" if status == "ELITE" else None, "reason": "", "label_key": None, "mode": None}

    _FIELD_LABEL_KEYS = {
        "Name": "description_label", "Business Domain": "domain", "Subdomain": "subdomain", "type": "type",
        "Statement": "statement", "Solution": "solution", "Business Value": "business_value",
        "Beneficiary": "beneficiary", "Sponsor": "sponsor",
        "Business Priority Alignment": "business_priority_alignment",
        "Strategic Goals Alignment": "strategic_goals_alignment",
        "Coverage Pick Reason": "elite_reason",
        "Below Bar Reason": "elite_reason",
        "Headline": "usecase", "Technique Label": "analytics_technique",
        "Measures of Success": "measures_of_success", "Financial Benefits": "financial_benefits",
        "Non-Financial Benefits": "non_financial_benefits", "KPI Names": "kpis",
        "Proposed Key Results": "expected_key_results",
    }

    def _shown_through_labels(self, uc: dict, field: str, t: dict) -> bool:
        value = str(uc.get(field) or "").strip()
        if not self._label_rendered(field, value):
            return False
        if field == "type":
            return bool(str(t.get(self._DISPLAY_VALUE_KEYS[value]) or "").strip())
        return all(str(t.get(self._ALIGNMENT_VALUE_KEYS[part]) or "").strip() for part in (p.strip() for p in value.split(",")) if part)

    def _untranslated_notice(self, uc: dict, t: dict, language: str) -> str:
        if language == "English":
            return ""
        fields = [f for f in (uc.get("_untranslated_fields") or []) if f and not self._shown_through_labels(uc, f, t)]
        if not fields:
            return ""
        notice = t.get("untranslated_notice", TranslationService.ENGLISH_TRANSLATIONS["untranslated_notice"])
        if uc.get("_translation_status") == "untranslated":
            return notice
        labels = [t.get(self._FIELD_LABEL_KEYS.get(f, ""), f) for f in fields]
        return f"{notice}: {', '.join(labels)}"

    def _build_notebook_url(self, object_id: int) -> str:
        try:
            host = self.w_client.config.host
            if not host or not object_id:
                return ''
            host = host.rstrip('/')
            return f"{host}/editor/notebooks/{object_id}"
        except Exception:
            return ''

    def _get_use_case_notebook_url(self, uc: dict) -> str:
        try:
            uc_id = uc.get('No', 'UNKNOWN')
            if uc_id not in self._notebook_object_ids:
                object_id = None
                recorded = str(uc.get('notebook_path') or '').strip()
                notebook_full_path = recorded or self._build_notebook_workspace_paths(uc)[1]
                stem = notebook_full_path[:-len(".ipynb")] if notebook_full_path.endswith(".ipynb") else notebook_full_path
                candidates = [stem, stem + ".ipynb"]
                if recorded and stem.startswith('/Workspace/'):
                    candidates += [candidate[len('/Workspace'):] for candidate in candidates]
                for candidate in candidates:
                    try:
                        object_id = getattr(self.w_client.workspace.get_status(candidate), 'object_id', None)
                    except Exception:
                        object_id = None
                    if object_id:
                        break
                self._notebook_object_ids.setdefault(uc_id, object_id)
            object_id = self._notebook_object_ids.get(uc_id)
            return self._build_notebook_url(object_id) if object_id else ''
        except Exception:
            return ''


    def assemble_use_case_notebooks(self, all_use_cases: list, translations: dict, summary_dict: dict = None):
        self.logger.debug("--- Starting Use Case Notebook Assembly (English) ---")
        if not all_use_cases:
            self.logger.warning("No use cases were provided for notebook assembly.")
            return

        total_use_cases = len(all_use_cases)
        domains_seen = set()
        subdomains_seen = set()
        for uc in all_use_cases:
            domains_seen.add(uc.get('Business Domain', 'Other'))
            subdomains_seen.add((uc.get('Business Domain', 'Other'), uc.get('Subdomain', 'General')))

        self.logger.info(f"📚 Generating {total_use_cases} notebooks (1 per use case) across {len(domains_seen)} domain folders, {len(subdomains_seen)} subdomain folders IN PARALLEL...")
        log_print(f"📚 Generating {total_use_cases} notebooks (1 per use case) in {len(domains_seen)} domains / {len(subdomains_seen)} subdomains (max {self.max_parallelism} concurrent)...")

        sorted_use_cases = sorted(all_use_cases, key=self._natural_sort_key)

        def create_notebook_for_use_case(task_data):
            idx, use_case = task_data
            _, rel_path, _ = self._build_notebook_workspace_paths(use_case)
            self.logger.info(f"📝 [{idx}/{total_use_cases}] Assembling notebook '{rel_path}'...")
            written = self._write_use_case_notebook(use_case, translations)
            if written:
                self.logger.info(f"✅ [{idx}/{total_use_cases}] Notebook '{rel_path}' completed ({use_case.get('notebook_status')})")
            return (idx, rel_path, written)

        notebook_parallelism, reason = calculate_adaptive_parallelism(
            "notebook_generation", self.max_parallelism,
            num_items=total_use_cases,
            num_domains=len(domains_seen),
            is_llm_operation=False, logger=self.logger
        )
        log_adaptive_parallelism_decision("notebook_generation", notebook_parallelism, self.max_parallelism, reason)

        _notebook_total_timeout = max(600, total_use_cases * 60)
        with _SafeExecutorContext(max_workers=notebook_parallelism, thread_name_prefix="NotebookGen", logger=self.logger, name="NotebookGen") as ctx:
            task_data_list = [(i, uc) for i, uc in enumerate(sorted_use_cases, start=1)]
            futures = [ctx.submit(create_notebook_for_use_case, td) for td in task_data_list]

            completed_count = 0
            failed_count = 0
            try:
                for future in ctx.completed(futures, timeout=_notebook_total_timeout):
                    try:
                        i, notebook_path, success = future.result()
                        if success:
                            completed_count += 1
                            log_print(f"   ✅ Notebook {i}/{total_use_cases} completed: {notebook_path}")
                        else:
                            failed_count += 1
                            log_print(f"   ❌ Notebook {i}/{total_use_cases} failed: {notebook_path}")
                    except Exception as e:
                        failed_count += 1
                        self.logger.error(f"Notebook generation future failed: {get_clean_error_message(e)}")
            except concurrent.futures.TimeoutError:
                ctx.mark_timed_out()
                self.logger.error(f"⏱️ Notebook generation total timeout after {_notebook_total_timeout}s.")
                for future in futures:
                    if not future.done():
                        future.cancel()
                        failed_count += 1

        self.logger.info(f"✅ Notebook generation complete: {completed_count}/{total_use_cases} succeeded, {failed_count} failed across {len(domains_seen)} domains")
        log_print(f"✅ All {total_use_cases} use case notebooks processed: {completed_count} succeeded, {failed_count} failed")

    _DOC_COLORS = {
        "ink": "1F2A33", "teal": "0F5E5A", "teal_dark": "0B4642", "teal_mid": "13746E", "mint": "DCEFEC",
        "mint2": "BFE3DF", "mint3": "CFE8E4", "sand": "F6F0E4", "sand2": "EFE7D6", "sand3": "FAF7F0", "coral": "E2725B",
        "coral_soft": "F3C9BD", "gold": "C9A24B", "gold_ink": "3A2B06", "gold_dark": "8A6A1F", "muted": "6A7681",
        "grey": "9AA4AC", "line": "E4DCCB", "green": "1F9D55", "amber": "D9922E", "red": "D1495B", "wine": "9B2C3B",
        "white": "FFFFFF", "night": "102A2A", "body": "3A454E", "body2": "2B353F", "body3": "4A5560", "ice": "EEF6F4",
        "note": "FFF4EC", "note_line": "F6D6C6", "note_ink": "8A3D26", "note_dark": "6E2D18",
    }
    _DOC_BRAND = {"wordmark": "Databricks Inspire AI", "maker": "Databricks", "product": "Inspire AI",
                  "web_body": "Inter", "web_display": "Fraunces", "office_body": "Arial", "office_display": "Georgia"}
    _DOC_TYPE_SCALE = {"numeral": 60, "hero": 44, "display": 36, "title": 28, "card_title": 24, "heading": 16, "lead": 14, "body": 12,
                       "small": 11, "label": 10, "micro": 8.6}
    _DOC_PDF_PX_PER_PT = 0.86
    _DOC_QUALITY_ORDER = ("Ultra High", "Very High", "High", "Medium", "Low", "Very Low", "Ultra Low")
    _DOC_QUALITY_COLORS = {
        "Ultra High": "teal_dark", "Very High": "teal", "High": "green", "Medium": "amber",
        "Low": "coral", "Very Low": "red", "Ultra Low": "wine", "": "grey",
    }
    _USE_CASE_VIEW_TEXT = (("statement", "Statement"), ("solution", "Solution"), ("business_value", "Business Value"))
    _USE_CASE_VIEW_FACTS = (
        ("sponsor", "Sponsor", "text"), ("beneficiary", "Beneficiary", "text"),
        ("business_priority_alignment", "Business Priority Alignment", "alignment"),
        ("strategic_goals_alignment", "Strategic Goals Alignment", "alignment"),
        ("analytics_technique", "Analytics Technique", "technique"), ("type", "type", "display"),
        ("aspect_tables", "Tables Involved", "tables"), ("foundations", "Foundations", "list"),
    )
    _USE_CASE_VIEW_VALUE = (
        ("kpis", "KPIs", "kpis"), ("expected_key_results", "Expected Key Results", "key_results"),
        ("measures_of_success", "Measures of Success", "lines"), ("financial_benefits", "Financial Benefits", "lines"),
        ("non_financial_benefits", "Non-Financial Benefits", "lines"),
    )
    _USE_CASE_VIEW_ITEM_KINDS = ("list", "kpis", "key_results", "lines")
    _STRATEGY_LABEL_FACTS = (("priority", "business_priority_alignment"), ("kpi", "kpis"),
                             ("key_result", "expected_key_results"), ("sponsor", "sponsor"))
    _USE_CASE_VIEW_SCORES = (
        ("inspire_score", "Inspire Score", 10), ("doc_elite_score", "elite_score", 10), ("quality_score_header", "Quality Score", 5),
        ("doc_bob_score", "bob_score", 10), ("priority_score", "Priority", 10), ("value_score", "Value", 5),
        ("feasibility_score", "Feasibility", 5),
    )
    _USE_CASE_VIEW_ELITE = (
        ("elite_verdict", "verdict_reason"), ("elite_decision", "decision"), ("elite_decision_owner", "decision_owner"),
        ("elite_cadence", "cadence"), ("elite_kpi", "kpi"), ("elite_value_driver", "value_driver"),
        ("elite_evidence_columns", "signal_columns"), ("elite_join_path", "join_path"), ("elite_build_path", "build_path"),
    )
    _USE_CASE_VIEW_EXCLUDED = (
        "Primary Table", "Justification", "Quality Summary", "Strategic Alignment", "Return on Investment", "Reusability",
        "Time to Value", "Data Availability", "Data Accessibility", "Architecture Fitness", "Team Skills", "Domain Knowledge",
        "People Allocation", "Budget Allocation", "Time to Production", "bob_tier1_score", "bob_tier2_score", "bob_tier3_score",
        "result_table", "has_genie_code", "genie_instruction", "generate_genie_code_instruction", "High Level Design",
        "technique_implementation_guidance", "code_structure_mandate", "directly_involved_schema", "foreign_key_relationships",
        "available_tables_listing", "idea_theme", "selection_flags", "quality_mode", "elite_status", "best_of_best_reason",
    )
    _DOC_VIEW_TABLE_LIMIT = 6
    _DOC_INDEX_COLUMNS = (("rank", "doc_rank", 0.08), ("id", None, 0.09), ("title", "usecase", 0.41),
                          ("domain", "domain", 0.17), ("quality", "quality", 0.13), ("inspire_score", "inspire_score", 0.12))
    _DOC_DISCLAIMER_KEYS = ("doc_method_1", "doc_disclaimer_ai", "doc_disclaimer_figures", "disclaimer")
    _DOC_CADENCE_KEYS = {
        "hourly": "value_cadence_hourly", "daily": "value_cadence_daily", "weekly": "value_cadence_weekly",
        "biweekly": "value_cadence_biweekly", "bi-weekly": "value_cadence_biweekly", "fortnightly": "value_cadence_biweekly",
        "monthly": "value_cadence_monthly", "quarterly": "value_cadence_quarterly", "yearly": "value_cadence_yearly",
        "annually": "value_cadence_yearly", "annual": "value_cadence_yearly", "real time": "value_cadence_real_time",
        "real-time": "value_cadence_real_time", "realtime": "value_cadence_real_time", "per event": "value_cadence_per_event",
        "event-driven": "value_cadence_per_event", "on event": "value_cadence_per_event",
    }
    _DOC_HIGH_TIERS = ("Ultra High", "Very High", "High")
    _DOC_MODE_KEYS = {"Good": "value_mode_good", "High": "value_mode_high", "Ultra High": "value_mode_ultra_high"}
    _DOC_RTL_LANGUAGES = frozenset({"Arabic"})
    _DOC_NON_LATIN_LANGUAGES = frozenset({"Arabic", "Hindi", "Tamil", "Japanese", "Korean", "Chinese (Mandarin)",
                                          "Russian", "Ukrainian"})
    _DOC_LOCALES = {
        "English": "en-US", "French": "fr-FR", "German": "de-DE", "Spanish": "es-ES", "Hindi": "hi-IN",
        "Chinese (Mandarin)": "zh-CN", "Japanese": "ja-JP", "Arabic": "ar-SA", "Portuguese": "pt-PT",
        "Russian": "ru-RU", "Swedish": "sv-SE", "Danish": "da-DK", "Norwegian": "nb-NO", "Finnish": "fi-FI",
        "Italian": "it-IT", "Polish": "pl-PL", "Romanian": "ro-RO", "Ukrainian": "uk-UA", "Dutch": "nl-NL",
        "Korean": "ko-KR", "Indonesian": "id-ID", "Malay": "ms-MY", "Tamil": "ta-IN",
    }
    _DOC_LIST_SEPARATORS = {"Arabic": "، ", "Japanese": "、", "Chinese (Mandarin)": "、"}
    _DOC_SCRIPT_PATTERNS = {
        "arabic": re.compile("[\u0600-\u06ff\u0750-\u077f\u08a0-\u08ff\ufb50-\ufdff\ufe70-\ufeff]"),
        "devanagari": re.compile("[\u0900-\u097f\ua8e0-\ua8ff]"),
        "tamil": re.compile("[\u0b80-\u0bff]"),
        "thai": re.compile("[\u0e00-\u0e7f]"),
        "hangul": re.compile("[\u1100-\u11ff\u3130-\u318f\uac00-\ud7af]"),
        "kana": re.compile("[\u3040-\u30ff\u31f0-\u31ff]"),
        "han": re.compile("[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff]"),
    }
    _DOC_LANGUAGE_SCRIPTS = {
        "Arabic": ("arabic",), "Hindi": ("devanagari",), "Tamil": ("tamil",), "Japanese": ("kana", "han"),
        "Korean": ("hangul",), "Chinese (Mandarin)": ("han",),
    }
    _DOC_FONT_FAMILIES = {
        "latin": ("Inter", (400, 600, 700)),
        "display": ("Fraunces", (600, 700)),
        "arabic": ("Noto Sans Arabic", (400, 700)),
        "devanagari": ("Noto Sans Devanagari", (400, 700)),
        "tamil": ("Noto Sans Tamil", (400, 700)),
        "thai": ("Noto Sans Thai", (400, 700)),
        "hangul": ("Noto Sans KR", (400, 700)),
        "kana": ("Noto Sans JP", (400, 700)),
        "han_ja": ("Noto Sans JP", (400, 700)),
        "han_ko": ("Noto Sans KR", (400, 700)),
        "han": ("Noto Sans SC", (400, 700)),
    }
    _DOC_FONT_SOURCE = "https://fonts.googleapis.com/css2?family={family}:wght@{weights}"
    _DOC_FONT_FILES = {}
    _DOC_FONT_LOCK = threading.Lock()
    _DOC_FONT_DIR = []
    _DOC_PLACEHOLDER = re.compile(r"\{([a-z_]+)\}")
    _DOC_SUMMARY_PLACEHOLDERS = (
        re.compile(r"^summary generation failed\b", re.I),
        re.compile(r"^domain: .{1,120} with \d+ use cases?\.?$", re.I),
    )

    def _doc_text(self, t: dict, key: str, language: str = "English", **values) -> str:
        english = TranslationService.ENGLISH_TRANSLATIONS.get(key, key)
        required = set(self._DOC_PLACEHOLDER.findall(english))
        text = english
        for candidate in ((t or {}).get(key), TranslationService.TRANSLATION_FALLBACKS.get(language, {}).get(key)):
            if isinstance(candidate, str) and candidate.strip() and required <= set(self._DOC_PLACEHOLDER.findall(candidate)):
                text = candidate
                break
        for name, value in values.items():
            text = text.replace("{" + name + "}", str(value))
        return text

    def _doc_join(self, items, language: str) -> str:
        return self._DOC_LIST_SEPARATORS.get(language, ", ").join(str(item) for item in items if str(item or "").strip())

    def _doc_quality_mode(self, use_cases: list) -> str:
        from collections import Counter
        modes = Counter()
        for uc in use_cases or []:
            if not str(uc.get("quality_mode") or "").strip():
                continue
            try:
                modes[canonical_use_cases_quality(uc.get("quality_mode"))] += 1
            except ValueError:
                continue
        if modes:
            return max(modes, key=lambda mode: (modes[mode], USE_CASES_QUALITY_VALID.index(mode)))
        if any(uc.get("elite_status") == "ELITE" for uc in use_cases or []):
            return "Ultra High"
        return ""

    _DOC_LEGACY_TEXT_FIELDS = ("Name", "usecase", "Business Domain", "Subdomain", "Statement", "Solution", "Business Value",
                               "Beneficiary", "Sponsor", "Business Priority Alignment", "Strategic Goals Alignment",
                               "coverage_pick_reason", "Strategy Path", "Measures of Success", "Financial Benefits",
                               "Non-Financial Benefits")
    _DOC_LEGACY_ELITE_FIELDS = ("verdict_reason", "decision", "decision_owner", "kpi", "value_driver", "build_path", "red_team")

    def _clean_legacy_catalog_text(self, catalog_json: dict) -> int:
        fixed = 0

        def clean(holder, key):
            nonlocal fixed
            value = holder.get(key)
            if isinstance(value, str) and "''" in value:
                fixed += value.count("''")
                holder[key] = value.replace("''", "'")
            elif isinstance(value, list) and any(isinstance(item, str) and "''" in item for item in value):
                fixed += sum(item.count("''") for item in value if isinstance(item, str))
                holder[key] = [item.replace("''", "'") if isinstance(item, str) else item for item in value]
        clean(catalog_json, "executive_summary")
        for domain_obj in catalog_json.get("domains", []) or []:
            if not isinstance(domain_obj, dict):
                continue
            clean(domain_obj, "summary")
            for uc in domain_obj.get("use_cases", []) or []:
                if not isinstance(uc, dict):
                    continue
                for key in self._DOC_LEGACY_TEXT_FIELDS:
                    clean(uc, key)
                if isinstance(uc.get("elite_assessment"), dict):
                    for key in self._DOC_LEGACY_ELITE_FIELDS:
                        clean(uc["elite_assessment"], key)
        if fixed:
            self.logger.warning(f"[DOCS-LEGACY-QUOTES] the JSON catalog carries {fixed} SQL-escaped apostrophe pair(s) in its "
                                f"narrative text (written before the escaping fix); they are shown as single apostrophes")
        return fixed

    @staticmethod
    def _doc_paragraphs(value) -> list:
        text = str(value or "")
        if not text.strip():
            return []
        text = re.sub(r"(?i)<\s*br\s*/?\s*>", "\n", text)
        paragraphs = []
        for part in re.split(r"(?i)</\s*p\s*>|<\s*p(?:\s[^>]*)?>|\n\s*\n", text):
            plain = html.unescape(re.sub(r"<[^>]+>", " ", part))
            plain = re.sub(r"\s+", " ", inspire_no_em_dash(plain.replace("`", ""))).strip()
            if plain:
                paragraphs.append(plain)
        return paragraphs

    @staticmethod
    def _doc_domain_key(name) -> str:
        text = unicodedata.normalize("NFKC", str(name or "")).casefold()
        text = re.sub(r"[\"'`*_#\u201c\u201d\u2018\u2019]", "", text)
        return re.sub(r"\s+", " ", text).strip(" .:;-")

    def _match_summary_domain(self, raw_key, domains) -> str:
        import difflib
        key = self._doc_domain_key(raw_key)
        if not key:
            return None
        keys = {domain: self._doc_domain_key(domain) for domain in domains}
        exact = [domain for domain, value in keys.items() if value == key]
        if exact:
            return exact[0] if len(exact) == 1 else None

        def bare(value):
            return self._doc_domain_key(re.sub(r"\s*\([^)]*\)\s*$", "", str(value or "")))
        stripped = [domain for domain in domains if bare(domain) and bare(domain) == bare(raw_key)]
        if len(stripped) == 1:
            return stripped[0]
        words = set(re.findall(r"\w+", key))
        contained = []
        for domain, value in keys.items():
            domain_words = set(re.findall(r"\w+", value))
            if domain_words and words and (domain_words <= words or words <= domain_words):
                contained.append(domain)
        if len(contained) == 1:
            rivals = [d for d, v in keys.items() if d != contained[0] and difflib.SequenceMatcher(None, key, v).ratio() >= 0.8]
            return None if rivals else contained[0]
        if contained:
            return None
        ranked = sorted(((difflib.SequenceMatcher(None, key, value).ratio(), domain) for domain, value in keys.items()),
                        key=lambda item: item[0], reverse=True)
        if ranked and ranked[0][0] >= 0.85 and (len(ranked) == 1 or ranked[0][0] - ranked[1][0] >= 0.1):
            return ranked[0][1]
        return None

    def _canonical_summary(self, summary_dict: dict, domains) -> tuple:
        domains = list(domains or [])
        canonical, pending, unmatched = {}, [], []
        for raw_key, text in (summary_dict or {}).items():
            if raw_key == "Executive" or raw_key in domains:
                canonical[raw_key] = text
            else:
                pending.append((raw_key, text))
        for raw_key, text in pending:
            if "Executive" not in canonical and self._doc_domain_key(raw_key).startswith("executive"):
                canonical["Executive"] = text
                continue
            target = self._match_summary_domain(raw_key, domains)
            if target and target not in canonical:
                canonical[target] = text
            else:
                unmatched.append(raw_key)
        return canonical, unmatched

    _DOC_NARRATIVE_COUNT = (
        re.compile(r"\b(?:\d+|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|thirteen|fourteen|fifteen|"
                   r"sixteen|seventeen|eighteen|nineteen|twenty|thirty|forty|fifty|a dozen|dozens of|both)\s+"
                   r"(?:[\w-]+\s+){0,2}use[\s-]cases?\b", re.I),
        re.compile(r"[\d\u4e00\u4e8c\u4e09\u56db\u4e94\u516d\u4e03\u516b\u4e5d\u5341\u4e24]+\s*(?:\u3064|\u4ef6|\u4e2a|\u500b|\u9805\u76ee)?\s*"
                   r"(?:\u306e)?\s*(?:\u30e6\u30fc\u30b9\u30b1\u30fc\u30b9|\u7528\u4f8b|\u4f7f\u7528\u6848\u4f8b)"),
        re.compile(r"(?:\d+|\u0627\u062b\u0646\u062a\u0627\u0646|\u0627\u062b\u0646\u0627\u0646|\u062b\u0644\u0627\u062b|\u0623\u0631\u0628\u0639|"
                   r"\u062e\u0645\u0633|\u0633\u062a|\u0633\u0628\u0639|\u062b\u0645\u0627\u0646\u064a|\u062a\u0633\u0639|\u0639\u0634\u0631)\s+"
                   r"\u062d\u0627\u0644\u0627\u062a\s+(?:\u0627\u0644)?\u0627\u0633\u062a\u062e\u062f\u0627\u0645|"
                   r"\u062d\u0627\u0644\u062a\u0627\s+(?:\u0627\u0644)?\u0627\u0633\u062a\u062e\u062f\u0627\u0645"),
    )
    _DOC_SENTENCE = re.compile(r"[^.!?\u3002\uff01\uff1f\u061f]+[.!?\u3002\uff01\uff1f\u061f]*\s*")

    def _doc_narrative(self, value, where: str = "") -> list:
        paragraphs = []
        for paragraph in self._doc_paragraphs(value):
            sentences = self._DOC_SENTENCE.findall(paragraph) or [paragraph]
            kept = [sentence for sentence in sentences if not any(p.search(sentence) for p in self._DOC_NARRATIVE_COUNT)]
            if len(kept) < len(sentences):
                dropped = [sentence.strip()[:120] for sentence in sentences if sentence not in kept]
                self.logger.warning(f"[DOCS-NARRATIVE count-dropped] {where}: removed {len(dropped)} sentence(s) that state a "
                                    f"use-case count the page already shows: {dropped[:3]}")
            text = "".join(kept).strip()
            if text:
                paragraphs.append(text)
        return paragraphs

    def _domain_summary_paragraphs(self, summary_dict: dict, domain: str) -> list:
        paragraphs = self._doc_narrative((summary_dict or {}).get(domain), domain)
        return [p for p in paragraphs if not any(pattern.search(p) for pattern in self._DOC_SUMMARY_PLACEHOLDERS)]

    @classmethod
    def _doc_subdomains(cls, use_cases: list) -> list:
        counts, order = {}, []
        for uc in use_cases or []:
            name = cls._doc_display(uc.get("Subdomain"))
            if not name:
                continue
            if name not in counts:
                order.append(name)
                counts[name] = 0
            counts[name] += 1
        return sorted(((name, counts[name]) for name in order), key=lambda item: (-item[1], order.index(item[0])))

    def _doc_use_case_title(self, uc: dict, language: str) -> str:
        headline = self._doc_display(uc.get("usecase"))
        name = self._doc_display(uc.get("Name"))
        if language == "English" or "Headline" in (uc.get("_untranslated_fields") or []) or not headline:
            return self._normalize_usecase(headline, name) or "N/A"
        return headline

    @staticmethod
    def _doc_tables(uc: dict) -> list:
        raw = uc.get("Tables Involved") or ""
        if isinstance(raw, (list, tuple)):
            items = [str(item) for item in raw]
        else:
            items = re.split(r"[,\n]", str(raw))
        tables = []
        for item in items:
            name = item.strip().strip("`").replace("`", "")
            if name and name.upper() != "N/A" and name not in tables:
                tables.append(name)
        return tables

    @staticmethod
    def _doc_is_ltr_text(text) -> bool:
        text = str(text or "")
        strong = [kind for kind in (unicodedata.bidirectional(ch) for ch in text) if kind in ("L", "R", "AL")]
        if not strong:
            return bool(re.search(r"\d[\s/:]+\S", text))
        return strong[0] == "L" and strong.count("L") >= 0.6 * len(strong)

    def _doc_bidi(self, text, model: dict) -> str:
        return ' dir="ltr"' if model["rtl"] and self._doc_is_ltr_text(text) else ""

    def _doc_bidi_html(self, text, model: dict) -> str:
        escaped = html.escape(str(text if text is not None else ""))
        return f'<span dir="ltr">{escaped}</span>' if model["rtl"] and self._doc_is_ltr_text(text) else escaped

    _DOC_MIN_SHARE = 1.5

    @classmethod
    def _doc_shares(cls, counts) -> list:
        counts = [max(0, int(n or 0)) for n in counts]
        total = sum(counts)
        if total <= 0:
            return [0.0 for _n in counts]
        floor = cls._DOC_MIN_SHARE if cls._DOC_MIN_SHARE * sum(1 for n in counts if n) <= 100.0 else 0.0
        small = [bool(n) and 100.0 * n / total < floor for n in counts]
        rest = sum(n for n, tiny in zip(counts, small) if not tiny)
        room = 100.0 - floor * sum(small)
        raw = [floor if tiny else (room * n / rest if rest else 0.0) for n, tiny in zip(counts, small)]
        hundredths = [int(value * 100) for value in raw]
        order = sorted(range(len(raw)), key=lambda i: raw[i] * 100 - hundredths[i], reverse=True)
        for i in order[:10000 - sum(hundredths)]:
            hundredths[i] += 1
        return [h / 100.0 for h in hundredths]

    def _doc_quality_color(self, label) -> str:
        return self._DOC_COLORS[self._DOC_QUALITY_COLORS.get(label or "", "grey")]

    @staticmethod
    def _doc_display(value) -> str:
        text = inspire_plain_text(value) if isinstance(value, (list, tuple, dict)) else str(value if value is not None else "")
        text = re.sub(r"\s+", " ", inspire_no_em_dash(text.replace("`", ""))).strip()
        return "" if text.upper() in ("N/A", "NA", "NONE", "NULL", "-") else text

    @classmethod
    def _label_rendered(cls, field: str, value) -> bool:
        text = str(value or "").strip()
        if field == "type":
            return text in cls._DISPLAY_VALUE_KEYS
        if field in ("Business Priority Alignment", "Strategic Goals Alignment"):
            parts = [part.strip() for part in text.split(",") if part.strip()]
            return bool(parts) and all(part in cls._ALIGNMENT_VALUE_KEYS for part in parts)
        return False

    def _doc_technique_label(self, value, t: dict, language: str, uc: dict = None) -> tuple:
        text = self._doc_display(value)
        if not text:
            return "", False
        canonical = self._canonical_technique(text)
        if canonical:
            label = self._translate_technique_value(canonical, t, language)
            known = self._TECHNIQUE_VALUE_KEYS.get(canonical) in t or canonical in self._TECHNIQUE_FALLBACKS.get(language, {})
            return label, language != "English" and not known
        if language == "English":
            return text, False
        translated = self._doc_display((uc or {}).get("Analytics Technique Label"))
        if translated and "Technique Label" not in ((uc or {}).get("_untranslated_fields") or []):
            return translated, False
        return text, True

    def _doc_cadence_label(self, value) -> str:
        text = self._doc_display(value)
        key = self._DOC_CADENCE_KEYS.get(text.casefold())
        return self._doc_text({}, key) if key else text

    def _doc_score_value(self, value, scale, t: dict, language: str) -> str:
        number = self._finite_score(value)
        if number is None:
            return self._doc_text(t, "doc_not_scored", language)
        return f"{number:.1f} / {scale}"

    def _strategy_fact_labels(self, label, path_fact: bool) -> dict:
        """Customer terminology from the strategy labels, by fact key. With path_fact, the alignment fact is
        labelled as the strategy path when the customer supplied a tree (documents show the path as one fact)."""
        tree = getattr(self, "strategy_tree", None) or {}
        custom = tree.get("labels") or {}
        labels = {fact: str(custom[key]).strip() for key, fact in self._STRATEGY_LABEL_FACTS if str(custom.get(key) or "").strip()}
        if path_fact and self._customer_strategy_tree():
            labels["strategic_goals_alignment"] = label("strategy_path")
        return labels

    def _customer_strategy_tree(self):
        tree = getattr(self, "strategy_tree", None) or {}
        return tree if tree.get("source") == "customer" and tree.get("strategy") else None

    def _strategy_level_columns(self) -> list:
        """Excel and CSV: one column per customer level with the aligned entries' names at that level."""
        tree = self._customer_strategy_tree()
        if tree is None:
            return []
        index = strategy_entry_index(tree)

        def names_at(level):
            def get(uc, _rank):
                names = []
                for alignment in uc.get("Strategy Alignment") or []:
                    for node in index.get(str((alignment or {}).get("id") or ""), []):
                        if node.get("level") == level and node.get("name") not in names:
                            names.append(node.get("name"))
                return "; ".join(names)
            return get
        return [(str(level), names_at(level), False) for level in tree.get("levels") or []]

    def _has_value_fields(self, uc: dict) -> bool:
        return any(uc.get(source) for _key, source, _kind in self._USE_CASE_VIEW_VALUE) or bool(uc.get("Foundations"))

    def _kpi_text(self, spec: dict, name, t: dict, language: str) -> str:
        name = str(name or inspire_plain_text(spec.get("name"))).strip()
        if not name:
            return ""
        details = " ".join(part for part in (self._kpi_direction_arrow(spec.get("direction")), str(spec.get("unit") or "").strip()) if part)
        text = f"{name} ({details})" if details else name
        baseline = spec.get("baseline") if isinstance(spec.get("baseline"), dict) else {}
        value = self._finite_score(baseline.get("value"))
        if value is None:
            return text
        values = {"value": self._format_baseline(value), "date": str(baseline.get("measured_on") or "")}
        if baseline.get("window_days"):
            return f'{text}: {self._doc_text(t, "kpi_baseline", language, days=baseline["window_days"], **values)}'
        return f'{text}: {self._doc_text(t, "kpi_baseline_undated", language, **values)}'

    def _use_case_items(self, uc: dict, source: str, kind: str, t: dict, language: str) -> tuple:
        """The items of a list-valued use case field, in order, and whether Inspire-written items that should be
        translated are shown in English. Customer KPIs and key results keep the customer's wording."""
        english = language == "English"
        known_gap = uc.get("_translation_status") == "untranslated"
        if kind in ("kpis", "key_results"):
            column, translated_key = (("KPI Names", "_kpi_names_translated") if kind == "kpis"
                                      else ("Proposed Key Results", "_key_results_translated"))
            entries = [entry for entry in uc.get(source) or [] if isinstance(entry, dict)]
            written = (TranslationService._generated_kpis(uc) if kind == "kpis"
                       else TranslationService._proposed_key_results(uc))
            translated = None if english else TranslationService._translated_items(uc, translated_key, len(written))
            renamed = dict(zip(map(id, written), translated or []))
            if kind == "kpis":
                items = [self._kpi_text(spec, renamed.get(id(spec)), t, language) for spec in entries]
            else:
                items = [renamed.get(id(kr)) or inspire_plain_text(kr.get("statement")) for kr in entries]
            known_gap = known_gap or column in (uc.get("_untranslated_fields") or [])
            return [item for item in items if item], bool(written) and not english and translated is None and not known_gap
        value = uc.get(source)
        if isinstance(value, (list, tuple)):
            return [str(item).strip() for item in value if str(item or "").strip()], False
        text = inspire_plain_text(value)
        return ([text] if text else []), False

    def _use_case_value_facts(self, uc: dict, labels: dict, T, t: dict, language: str) -> tuple:
        facts, flagged = [], []
        for key, source, kind in self._USE_CASE_VIEW_VALUE:
            items, gap = self._use_case_items(uc, source, kind, t, language)
            if items:
                facts.append((key, labels.get(key) or T(key), items))
            if gap:
                flagged.append(key)
        return facts, flagged

    def _use_case_view(self, uc: dict, model: dict, t: dict) -> dict:
        language = model["language"]
        T = lambda key, **values: self._doc_text(t, key, language, **{**model["tiers"], **values})
        missing = T("doc_not_provided")
        untranslated = []
        title = self._doc_display(self._doc_use_case_title(uc, language)) or missing
        name = self._doc_display(uc.get("Name"))
        quality = self._canonical_quality_label(uc.get("Quality")) or ""
        status = uc.get("elite_status")
        presentation = self._elite_presentation(uc)
        kind = "elite" if status == "ELITE" else ("coverage" if presentation["kind"] == "coverage" else "")
        metrics = [("quality", T("quality"), self._translate_display_value(quality, t) if quality else T("doc_not_scored"))]
        for label_key, source, scale in self._USE_CASE_VIEW_SCORES:
            if source == "elite_score" and not (model["ultra"] or self._finite_score(uc.get("elite_score")) is not None):
                continue
            metrics.append((label_key, T(label_key), self._doc_score_value(uc.get(source), scale, t, language)))
        narrative = [(key, T(key), self._doc_display(uc.get(source)) or missing) for key, source in self._USE_CASE_VIEW_TEXT]
        tables = self._doc_tables(uc)
        labels = model.get("fact_labels") or {}
        facts = []
        for key, source, kind_of in self._USE_CASE_VIEW_FACTS:
            raw = uc.get(source)
            if kind_of in self._USE_CASE_VIEW_ITEM_KINDS:
                items, gap = self._use_case_items(uc, source, kind_of, t, language)
                if gap:
                    untranslated.append(key)
                if items:
                    facts.append((key, labels.get(key) or T(key), self._doc_join(items, language)))
                continue
            if kind_of == "tables":
                shown = tables[:self._DOC_VIEW_TABLE_LIMIT]
                extra = len(tables) - len(shown)
                value = self._doc_join(shown, language) + (f" (+{extra})" if extra > 0 else "") if shown else ""
            elif kind_of == "alignment":
                value = self._doc_display(self._translate_alignment_value(self._doc_display(raw), t))
            elif kind_of == "display":
                value = self._doc_display(self._translate_display_value(self._doc_display(raw), t))
            elif kind_of == "technique":
                value, flagged = self._doc_technique_label(raw, t, language, uc)
                if flagged:
                    untranslated.append(key)
            else:
                value = self._doc_display(raw)
            facts.append((key, labels.get(key) or T(key), value or missing))
        value_facts, value_gaps = self._use_case_value_facts(uc, labels, T, t, language)
        untranslated += value_gaps
        coverage = None
        if kind == "coverage":
            mode_label = self._doc_text(t, self._DOC_MODE_KEYS[presentation["mode"]], language) if presentation.get("mode") else ""
            coverage = (T(presentation["label_key"], mode=mode_label), T("elite_reason"),
                        self._doc_display(presentation.get("reason")) or missing)
        notice = self._untranslated_notice(uc, t, language)
        if untranslated:
            extra = ", ".join(T(key) for key in untranslated)
            notice = f"{notice}, {extra}" if notice else f'{T("untranslated_notice")}: {extra}'
        star = None
        if uc.get("best_of_best") == "Y":
            star = (f'\u2605 {T("doc_best_of_best")} #{uc.get("best_of_best_rank")} \u00b7 {T("doc_elite_score")} '
                    f'{self._doc_score_value(uc.get("best_of_best_score"), 10, t, language)}')
        return {
            "uc": uc, "id": str(uc.get("No") or "").strip(), "rank": model["rank_of"].get(id(uc), ""),
            "title": title, "name": name if name and name != title else "",
            "domain": self._doc_display(uc.get("Business Domain")) or missing, "subdomain": self._doc_display(uc.get("Subdomain")),
            "kind": kind, "quality": quality, "metrics": metrics, "narrative": narrative, "facts": facts,
            "value_facts": value_facts, "coverage": coverage, "notice": notice, "untranslated": untranslated,
            "star": star,
        }

    def _doc_index_rows(self, use_cases: list, model: dict, t: dict) -> list:
        language = model["language"]
        rows = []
        for uc in use_cases:
            quality = self._canonical_quality_label(uc.get("Quality")) or ""
            quality_text = self._translate_display_value(quality, t) if quality else self._doc_text(t, "doc_not_scored", language)
            flag = (self._doc_text(t, "doc_coverage_pick", language) if uc.get("elite_status") == "DOMAIN_COVERAGE_PICK"
                    else self._doc_text(t, "doc_fallback_pick", language) if uc.get("elite_status") == "FALLBACK_BELOW_BAR" else "")
            rows.append({
                "uc": uc, "rank": str(model["rank_of"].get(id(uc), "")), "id": str(uc.get("No") or "").strip(),
                "title": self._doc_display(self._doc_use_case_title(uc, language)) or self._doc_text(t, "doc_not_provided", language),
                "domain": self._doc_display(uc.get("Business Domain")) or self._doc_text(t, "doc_not_provided", language),
                "quality": quality_text, "quality_color": self._doc_quality_color(quality), "flag": flag,
                "inspire_score": self._doc_score_value(uc.get("Inspire Score"), 10, t, language),
            })
        return rows

    def _doc_index_header(self, model: dict, t: dict) -> list:
        return [(key, self._doc_text(t, label_key, model["language"]) if label_key else "ID", width)
                for key, label_key, width in self._DOC_INDEX_COLUMNS]

    def _doc_disclaimer(self, model: dict, t: dict) -> tuple:
        T = lambda key, **values: self._doc_text(t, key, model["language"], **{**model["tiers"], **values})
        mode = model.get("discovery_type")
        if mode == "Deep":
            access = T("doc_data_access_deep", tables=model.get("tables_read", 0))
        elif mode == "Standard":
            access = T("doc_data_access_standard")
        else:
            access = T("doc_method_1")
        return T("pdf_disclaimer_title"), [access if key == "doc_method_1" else T(key) for key in self._DOC_DISCLAIMER_KEYS]

    def _doc_foundation_rows(self, model: dict) -> list:
        """Rows of the shared data foundations table in build order: tables, joins, use cases unlocked. No dates."""
        language = model["language"]
        short = lambda column: ".".join(str(column).split(".")[-2:])
        rows = []
        for foundation in getattr(self, "data_foundations", None) or []:
            ids = [str(uc_id) for uc_id in foundation.get("use_cases") or []]
            shown = ids[:self._DOC_FOUNDATION_IDS]
            extra = len(ids) - len(shown)
            rows.append({"id": str(foundation.get("id") or ""),
                         "tables": self._doc_join([str(table).rsplit(".", 1)[-1] for table in foundation.get("tables") or []], language),
                         "joins": self._doc_join([f'{short(j.get("left"))} = {short(j.get("right"))}' for j in foundation.get("joins") or []
                                                  if isinstance(j, dict) and j.get("left") and j.get("right")], language),
                         "count": int(foundation.get("use_case_count") or len(ids)), "use_case_ids": shown, "more": max(0, extra),
                         "use_cases": self._doc_join(shown, language) + (f" (+{extra})" if extra > 0 else "")})
        return rows

    _DOC_FOUNDATION_IDS = 12
    _DOC_PPTX_FOUNDATION_ROWS = 6
    _DOC_VALUE_LEFT = ("kpis", "expected_key_results")

    def _doc_generated_by(self, model: dict, t: dict) -> str:
        return self._doc_text(t, "doc_generated_by", model["language"], brand=self._DOC_BRAND["wordmark"],
                              version=__AGENT_VERSION__, date=model["date"])

    def _doc_outline(self, model: dict, t: dict) -> list:
        T = lambda key, **values: self._doc_text(t, key, model["language"], **{**model["tiers"], **values})
        count = f'{model["total"]} {T("doc_use_cases")}'
        foundations = model.get("foundations") or []
        return [("executive", T("pdf_exec_summary"), model["business_name"]),
                ("guide", T("doc_how_to_read"), T("doc_inspire_score_title")),
                ("portfolio", T("doc_portfolio_glance"), f'{count} \u00b7 {len(model["domains"])} {T("doc_domains")}'),
                ("domains", T("doc_domains"), self._doc_join([d["name"] for d in model["domains"][:4]], model["language"])),
                ("index", T("doc_use_case_index"), count),
                ("methodology", T("doc_methodology"), T("doc_glossary")),
                *([("foundations", T("doc_foundations_title"), self._doc_join([row["tables"] for row in foundations[:2]], model["language"]))]
                  if foundations else []),
                ("roadmap", T("doc_roadmap"), T("doc_step_build")),
                ("disclaimer", T("pdf_disclaimer_title"), self._DOC_BRAND["wordmark"])]

    def _document_model(self, language: str, t: dict, summary_dict: dict, grouped_data: dict, business_name: str) -> dict:
        groups = self._rank_domain_groups(dict(grouped_data or {}))
        ranked = sorted([uc for _, members in groups for uc in members], key=self._final_rank_key)
        summary, unmatched = self._canonical_summary(summary_dict, [name for name, _ in groups])
        if unmatched:
            self.logger.warning(f"[DOCS-SUMMARY unmatched-domain] {language}: summary rows {unmatched[:10]} match no domain of "
                                f"the document; those domain pages use the data-derived overview")
        quality_counts = {label: 0 for label in self._DOC_QUALITY_ORDER}
        unscored = 0
        for uc in ranked:
            label = self._canonical_quality_label(uc.get("Quality"))
            if label:
                quality_counts[label] += 1
            else:
                unscored += 1
        domains, derived = [], []
        for index, (name, members) in enumerate(groups, start=1):
            overview = self._domain_summary_paragraphs(summary, name)
            subdomains = self._doc_subdomains(members)
            if not overview:
                derived.append(name)
                listed = self._doc_join([sub for sub, _ in subdomains[:4]], language) or name
                overview = [self._doc_text(t, "doc_domain_fallback", language, count=len(members), subdomains=listed)]
            scores = [s for s in (self._finite_score(uc.get("Inspire Score")) for uc in members) if s is not None]
            domains.append({
                "index": index, "name": self._doc_display(name) or name, "use_cases": members, "overview": overview, "subdomains": subdomains,
                "best_score": max(scores) if scores else None,
                "high_or_better": sum(1 for uc in members if self._canonical_quality_label(uc.get("Quality")) in self._DOC_HIGH_TIERS),
                "elite": sum(1 for uc in members if uc.get("elite_status") == "ELITE"),
                "coverage": sum(1 for uc in members if uc.get("elite_status") == "DOMAIN_COVERAGE_PICK"),
                "top": [uc for uc in members if uc.get("elite_status") != "DOMAIN_COVERAGE_PICK"][:3] or members[:1],
            })
        if derived:
            self.logger.info(f"[DOCS-SUMMARY derived-overview] {language}: {len(derived)}/{len(groups)} domain page(s) use the "
                             f"data-derived overview because no domain summary was produced: {derived[:10]}")
        executive = self._doc_narrative(summary.get("Executive"), "Executive")
        if not executive:
            total = len(ranked)
            executive = [self._doc_text(t, "pdf_fallback_summary_p1", language, total_cases=total, business_name=business_name),
                         self._doc_text(t, "pdf_fallback_summary_p2", language)]
        scores = [s for s in (self._finite_score(uc.get("Inspire Score")) for uc in ranked) if s is not None]
        mode = self._doc_quality_mode(ranked)
        n_fallback = sum(1 for uc in ranked if uc.get("elite_status") == "FALLBACK_BELOW_BAR")
        if n_fallback and n_fallback == len(ranked):
            notice_mode = self._doc_text(t, self._DOC_MODE_KEYS.get(mode, "value_mode_good"), language) if mode else ""
            executive = [self._doc_text(t, "doc_fallback_notice", language, mode=notice_mode)] + list(executive)
        min_dimension = TECHNICAL_CONTEXT["runtime"].get("ultra_high_min_dimension_score", 8)
        tiers = {"lowest": self._translate_display_value("Ultra Low", t), "highest": self._translate_display_value("Ultra High", t),
                 "medium": self._translate_display_value("Medium", t), "high": self._translate_display_value("High", t),
                 "ultra": self._doc_text(t, "value_mode_ultra_high", language), "min_score": min_dimension}
        model = {
            "language": language, "lang": self._get_lang_abbr(language), "locale": self._DOC_LOCALES.get(language, "en-US"),
            "rtl": language in self._DOC_RTL_LANGUAGES, "latin_display": language not in self._DOC_NON_LATIN_LANGUAGES,
            "business_name": str(business_name or self.business_name or "").strip(),
            "date": datetime.datetime.now().strftime("%Y-%m-%d"),
            "mode": mode, "ultra": self._is_ultra_high_mode(mode), "mode_label": self._doc_text(t, self._DOC_MODE_KEYS[mode], language) if mode else "",
            "total": len(ranked), "domains": domains, "ranked": ranked,
            "rank_of": {id(uc): position for position, uc in enumerate(ranked, start=1)},
            "quality_counts": quality_counts, "unscored": unscored, "executive": executive,
            "elite": sum(1 for uc in ranked if uc.get("elite_status") == "ELITE"),
            "coverage": sum(1 for uc in ranked if uc.get("elite_status") == "DOMAIN_COVERAGE_PICK"),
            "fallback": n_fallback,
            "high_or_better": sum(quality_counts[label] for label in self._DOC_HIGH_TIERS),
            "best_score": max(scores) if scores else None,
            "min_dimension": min_dimension, "tiers": tiers,
        }
        self._doc_presentation(model, t)
        return model

    _DOC_GLOSSARY = (("domain", "doc_gl_domain"), ("subdomain", "doc_gl_subdomain"), ("analytics_technique", "doc_gl_technique"),
                     ("inspire_score", "doc_gl_inspire"), ("quality", "doc_gl_quality"), ("doc_elite_score", "doc_gl_elite"),
                     ("doc_coverage_title", "doc_gl_coverage"), ("doc_notebook_term", "doc_gl_notebook"))
    _DOC_METHOD_STEPS = ("doc_method_1", "doc_method_2", "doc_method_3", "doc_method_4", "doc_method_5")
    _DOC_ROADMAP_STEPS = (("doc_step_select", "doc_step_select_body"), ("doc_step_validate", "doc_step_validate_body"),
                          ("doc_step_build", "doc_step_build_body"), ("doc_step_schedule", "doc_step_schedule_body"),
                          ("doc_step_measure", "doc_step_measure_body"))
    _DOC_TOP_LIMIT = {"elite": 12, "default": 10, "deck_all": 15}

    def _doc_presentation(self, model: dict, t: dict) -> None:
        language = model["language"]
        T = lambda key, **values: self._doc_text(t, key, language, **{**model["tiers"], **values})
        best = self._doc_score_value(model["best_score"], 10, t, language)
        third = (model["elite"], T("doc_elite_use_cases")) if model["ultra"] else (model["high_or_better"], T("doc_high_or_better"))
        model["headline"] = [(str(model["total"]), T("doc_use_cases")), (str(len(model["domains"])), T("doc_domains")),
                             (str(third[0]), third[1]), (best, T("doc_best_score"))]
        model["quality_rows"] = [(label, n) for label, n in model["quality_counts"].items() if n]
        model["quality_names"] = [self._translate_display_value(label, t) for label, _n in model["quality_rows"]]
        if model["unscored"]:
            model["quality_rows"].append(("", model["unscored"]))
            model["quality_names"].append(T("doc_not_scored"))
        model["discovery_type"], model["tables_read"] = self._doc_data_access()
        deep = model["discovery_type"] == "Deep" and model["tables_read"] > 0
        model["guide_lead_key"] = "doc_guide_lead_deep" if deep else "doc_guide_lead"
        model["method_steps"] = [T("doc_method_1_deep" if key == "doc_method_1" and deep else key) for key in self._DOC_METHOD_STEPS]
        model["foundations"] = self._doc_foundation_rows(model)
        model["fact_labels"] = self._strategy_fact_labels(T, path_fact=True)
        model["glossary"] = [(T(term), T(body)) for term, body in self._DOC_GLOSSARY]
        model["roadmap"] = [(T(title), T(body)) for title, body in self._DOC_ROADMAP_STEPS]
        summary = f'{model["total"]} {T("doc_use_cases")} \u00b7 {len(model["domains"])} {T("doc_domains")}'
        model["closing_summary"] = summary + (f' \u00b7 {T("doc_quality_mode")}: {model["mode_label"]}' if model["mode"] else "")
        model["generated_by"] = self._doc_generated_by(model, t)
        elite = [uc for uc in model["ranked"] if uc.get("elite_status") == "ELITE"]
        if model["ultra"] and elite:
            model["top"] = elite[:self._DOC_TOP_LIMIT["elite"]]
        else:
            model["top"] = [uc for uc in model["ranked"] if uc.get("elite_status") != "DOMAIN_COVERAGE_PICK"][:self._DOC_TOP_LIMIT["default"]]
        if model["total"] <= self._DOC_TOP_LIMIT["deck_all"]:
            model["deck"] = list(model["ranked"])
        else:
            chosen = {id(uc) for uc in model["top"]}
            model["deck"] = [uc for uc in model["ranked"] if id(uc) in chosen or uc.get("elite_status") == "DOMAIN_COVERAGE_PICK"
                             or uc.get("best_of_best") == "Y"]
        for domain in model["domains"]:
            stats = [(str(len(domain["use_cases"])), T("doc_use_cases")),
                     (self._doc_score_value(domain["best_score"], 10, t, language), T("doc_best_score"))]
            stats.append((str(domain["elite"]), T("doc_elite_use_cases")) if model["ultra"] or domain["elite"]
                         else (str(domain["high_or_better"]), T("doc_high_or_better")))
            if domain["coverage"]:
                stats.append((str(domain["coverage"]), T("doc_coverage_picks")))
            domain["stats"] = stats
        flagged = {}
        for uc in model["ranked"]:
            technique = self._doc_display(uc.get("Analytics Technique"))
            if technique and self._doc_technique_label(technique, t, language, uc)[1]:
                flagged.setdefault("analytics_technique", set()).add(technique)
        model["labels_in_english"] = {field: sorted(values) for field, values in flagged.items()}
        if flagged:
            self.logger.warning(f"[DOCS-LABELS untranslated] {language}: values shown in English because no {language} label exists: "
                                f"{model['labels_in_english']}")
            self._record_label_gaps(language, model["labels_in_english"])

    def _record_label_gaps(self, language: str, labels: dict) -> None:
        with self._RUN_OUTCOME_LOCK:
            entry = dict((getattr(self, "translation_gaps", None) or {}).get(str(language)) or {})
        entry["labels_in_english"] = {**dict(entry.get("labels_in_english") or {}), **labels}
        entry.setdefault("use_cases", 0)
        self._record_translation_gaps(language, entry)

    def _doc_scripts(self, language: str, texts) -> list:
        blob = "\n".join(str(text) for text in texts if text)
        scripts = [name for name, pattern in self._DOC_SCRIPT_PATTERNS.items() if pattern.search(blob)]
        for name in self._DOC_LANGUAGE_SCRIPTS.get(language, ()):
            if name not in scripts:
                scripts.append(name)
        return scripts

    def _doc_font_keys(self, language: str, scripts) -> list:
        keys = ["latin"]
        if language not in self._DOC_NON_LATIN_LANGUAGES:
            keys.append("display")
        han_key = {"Japanese": "han_ja", "Korean": "han_ko"}.get(language, "han")
        for script in scripts:
            key = han_key if script == "han" else script
            if key in self._DOC_FONT_FAMILIES and self._DOC_FONT_FAMILIES[key][0] not in [self._DOC_FONT_FAMILIES[k][0] for k in keys]:
                keys.append(key)
        return keys

    def _doc_http_get(self, url: str, timeout: int = 30) -> bytes:
        import urllib.request
        request = urllib.request.Request(url, headers={"User-Agent": "Inspire-Docs/1.0"})
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return response.read()

    def _doc_font_files(self, family: str, weights) -> dict:
        with self._DOC_FONT_LOCK:
            cached = self._DOC_FONT_FILES.get(family)
            if cached is not None and all(os.path.exists(path) for path in cached.values()):
                return cached
            files = {}
            try:
                if not self._DOC_FONT_DIR or not os.path.isdir(self._DOC_FONT_DIR[0]):
                    self._DOC_FONT_DIR[:] = [tempfile.mkdtemp(prefix="inspire_fonts_")]
                url = self._DOC_FONT_SOURCE.format(family=family.replace(" ", "+"), weights=";".join(str(w) for w in weights))
                css = self._doc_http_get(url).decode("utf-8", "replace")
                for block in re.findall(r"@font-face\s*\{(.*?)\}", css, re.S):
                    weight = re.search(r"font-weight:\s*(\d+)", block)
                    source = re.search(r"url\((https://[^)\s]+)\)", block)
                    style = re.search(r"font-style:\s*(\w+)", block)
                    if not weight or not source or (style and style.group(1) != "normal"):
                        continue
                    path = os.path.join(self._DOC_FONT_DIR[0], f"{re.sub(r'[^A-Za-z0-9]', '', family)}-{weight.group(1)}.ttf")
                    if not os.path.exists(path):
                        data = self._doc_http_get(source.group(1), timeout=90)
                        if len(data) < 1024:
                            raise ValueError(f"font file for {family} {weight.group(1)} is only {len(data)} bytes")
                        with open(path, "wb") as handle:
                            handle.write(data)
                    files[int(weight.group(1))] = path
                if not files:
                    raise ValueError("the font service returned no usable font file")
                self.logger.info(f"[DOCS-FONTS] {family}: {len(files)} weight(s) ready for the PDF")
            except Exception as error:
                self.logger.warning(f"[DOCS-FONTS] {family} could not be fetched ({get_clean_error_message(error, max_chars=200)}); "
                                    f"the PDF falls back to installed fonts for it and may show missing glyphs")
            self._DOC_FONT_FILES[family] = files
            return files

    _DOC_FONT_LOCAL = {
        "latin": ("Noto Sans", "DejaVu Sans", "Liberation Sans"), "display": ("Noto Serif", "DejaVu Serif", "Liberation Serif"),
        "arabic": ("Noto Sans Arabic", "Noto Naskh Arabic", "Noto Kufi Arabic", "DejaVu Sans"),
        "devanagari": ("Noto Sans Devanagari", "Lohit Devanagari"), "tamil": ("Noto Sans Tamil", "Lohit Tamil"),
        "thai": ("Noto Sans Thai",), "hangul": ("Noto Sans KR", "Noto Sans CJK KR", "NanumGothic"),
        "han_ko": ("Noto Sans KR", "Noto Sans CJK KR", "NanumGothic"), "kana": ("Noto Sans JP", "Noto Sans CJK JP"),
        "han_ja": ("Noto Sans JP", "Noto Sans CJK JP"), "han": ("Noto Sans SC", "Noto Sans CJK SC", "WenQuanYi Zen Hei"),
    }
    _DOC_FONT_DIRS = ("/usr/share/fonts", "/usr/local/share/fonts", "~/.fonts", "~/.local/share/fonts", "/Library/Fonts",
                      "/System/Library/Fonts")
    _DOC_INSTALLED_FONTS = []

    def _doc_installed_fonts(self) -> dict:
        with self._DOC_FONT_LOCK:
            if self._DOC_INSTALLED_FONTS:
                return self._DOC_INSTALLED_FONTS[0]
            found = {}
            try:
                import subprocess
                listing = subprocess.run(["fc-list", ":", "family", "file"], capture_output=True, text=True, timeout=30).stdout
                for line in listing.splitlines():
                    path, _, families = line.partition(":")
                    for family in families.split(","):
                        if family.strip():
                            found.setdefault(family.strip().casefold(), path.strip())
            except Exception:
                found = {}
            if not found:
                for root in self._DOC_FONT_DIRS:
                    for folder, _dirs, files in os.walk(os.path.expanduser(root)):
                        for name in files:
                            stem, ext = os.path.splitext(name)
                            if ext.lower() in (".ttf", ".otf", ".ttc"):
                                found.setdefault("file:" + re.sub(r"[-_ ]", "", stem).casefold(), os.path.join(folder, name))
            self._DOC_INSTALLED_FONTS[:] = [found]
            return found

    def _doc_local_font(self, key: str):
        installed = self._doc_installed_fonts()
        for family in self._DOC_FONT_LOCAL.get(key, ()):
            path = installed.get(family.casefold())
            if path is None:
                wanted = "file:" + family.replace(" ", "").casefold()
                path = next((p for name, p in installed.items() if name.startswith(wanted)), None)
            if path:
                return family, path
        return None, None

    def _doc_font_css(self, language: str, scripts) -> tuple:
        faces, families, missing, fallbacks = [], [], [], []
        for key in self._doc_font_keys(language, scripts):
            family, weights = self._DOC_FONT_FAMILIES[key]
            files = self._doc_font_files(family, weights)
            families.append((key, family))
            local_family, local_path = (None, None)
            if not files:
                missing.append(family)
                local_family, local_path = self._doc_local_font(key)
                fallbacks.append({"script": key, "font": family, "fallback": local_family})
            for weight in weights:
                path = files.get(weight) or (files.get(max(files)) if files else None)
                if path:
                    source = f"url('file://{path}') format('truetype'), local('{family}')"
                elif local_family:
                    source = (f"url('file://{local_path}'), " if local_path.lower().endswith((".ttf", ".otf")) else "") + \
                             f"local('{local_family}'), local('{family}')"
                else:
                    source = f"local('{family}')"
                faces.append(f"@font-face {{ font-family: '{family}'; font-style: normal; font-weight: {weight}; src: {source}; }}")
        body = [family for key, family in families if key != "display"]
        display = [family for key, family in families if key == "display"] + body
        tail = ["Noto Sans", "DejaVu Sans", "sans-serif"]
        stack = lambda names: ", ".join(f"'{name}'" if name != "sans-serif" else name for name in names + tail)
        return "\n".join(faces), stack(body), stack(display), missing, fallbacks

    def _record_document_font_fallback(self, language: str, fallbacks: list):
        if not fallbacks:
            return
        with self._RUN_OUTCOME_LOCK:
            recorded = getattr(self, "document_font_fallbacks", None)
            if not isinstance(recorded, dict):
                recorded = {}
                self.document_font_fallbacks = recorded
            recorded[str(language)] = [dict(entry) for entry in fallbacks]
        listing = "; ".join(f'{entry["script"]}: {entry["font"]} -> {entry["fallback"] or "no installed font, glyphs may be missing"}'
                            for entry in fallbacks)
        self.logger.warning(f"[DOCS-FONTS fallback] {language} PDF: the brand fonts could not be downloaded; {listing}")
        self._emit_pipeline_status(
            stage_name="Document Generation", step_name="PDF Catalog", sub_step_name=f"{language} fonts",
            message=f"{language} PDF uses installed fonts because the brand fonts could not be downloaded: {listing}",
            status="ended_warning", progress_increment=0.0,
            result_json={"language": language, "fallbacks": [dict(entry) for entry in fallbacks]})
        self._merge_run_outcome_into_results()

    _PDF_CSS = """
@page { size: A4; margin: 19mm 15mm 19mm 15mm;
  @top-__S__ { content: element(brand); vertical-align: bottom; padding-bottom: 4mm; }
  @top-__E__ { content: "__BUSINESS__"; font-family: __BODY__; font-size: __F_label__; color: #__C_muted__; vertical-align: bottom; padding-bottom: 5mm; }
  @bottom-__S__ { content: "__FOOTER__"; font-family: __BODY__; font-size: __F_label__; color: #__C_grey__; vertical-align: top; padding-top: 5mm; }
  @bottom-__E__ { content: __PAGE__; font-family: __BODY__; font-size: __F_label__; font-weight: 700; color: #__C_muted__; vertical-align: top; padding-top: 5mm; } }
@page cover { margin: 0; background: #__C_teal__; @top-__S__ { content: none; } @top-__E__ { content: none; } @bottom-__S__ { content: none; } @bottom-__E__ { content: none; } }
@page closing { margin: 0; background: #__C_teal__; @top-__S__ { content: none; } @top-__E__ { content: none; } @bottom-__S__ { content: none; } @bottom-__E__ { content: none; } }
@page opener { @top-__S__ { content: none; } @top-__E__ { content: none; } }
@page sand { background: #__C_sand__; }
* { box-sizing: border-box; }
html { font-family: __BODY__; color: #__C_ink__; font-size: __F_body__; line-height: 1.5; }
body { margin: 0; }
* { border-spacing: 0; }
h1, h2, h3, h4, h5 { font-family: __DISPLAY__; font-weight: 700; line-height: 1.16; margin: 0; }
p { margin: 0 0 7px; }
.txt, p, li, .v, .chip, td { overflow-wrap: break-word; hyphens: auto; }
a { color: #__C_teal__; text-decoration: none; }
.runbrand { position: running(brand); }
.mark { display: inline-block; width: 9px; height: 9px; border-radius: 50%; background: #__C_gold__; margin-__E__: 6px; vertical-align: 0; }
.wordmark { font-family: __BODY__; font-size: __F_label__; color: #__C_muted__; letter-spacing: .3px; white-space: nowrap; }
.wordmark b { color: #__C_teal__; font-weight: 700; }
.wordmark.light { color: #__C_mint2__; }
.wordmark.light b { color: #__C_white__; }
.wordmark.big { font-size: __F_lead__; }
.sheet { break-before: page; page: plain; }
.sheet.sand { page: sand; }
.cards { page: plain; }
.eyebrow { color: #__C_coral__; font-weight: 700; letter-spacing: 2px; text-transform: uppercase; font-size: __F_label__; }
.title { font-size: __F_title__; margin: 5px 0 0; }
.rule { height: 3px; width: 54px; background: #__C_gold__; margin: 9px 0 13px; border-radius: 2px; }
.lead { font-size: __F_lead__; color: #__C_body__; margin-bottom: 12px; }
.grid2 { display: table; width: 100%; table-layout: fixed; border-collapse: separate; border-spacing: 12px 0; margin-left: -12px; margin-right: -12px; }
.grid2 > * { display: table-cell; vertical-align: top; }
.grid2.gap { margin-top: 13px; }
.gap { margin-top: 13px; }
.card { background: #__C_white__; border: 1px solid #__C_line__; border-radius: 14px; padding: 12px 14px; break-inside: avoid; }
.card.soft { background: #__C_sand__; }
.sheet.sand .card.soft { background: #__C_sand2__; border-color: #__C_line__; }
.card h4 { font-size: __F_heading__; margin-bottom: 6px; color: #__C_ink__; }
.card p { font-size: __F_body__; }
.muted { color: #__C_muted__; }
table.stats { width: 100%; table-layout: fixed; border-collapse: separate; border-spacing: 8px 0; margin-left: -8px; margin-right: -8px; }
table.stats td { background: #__C_sand__; border-radius: 11px; padding: 8px 6px; text-align: center; vertical-align: middle; }
.sheet.sand table.stats td { background: #__C_white__; border: 1px solid #__C_line__; }
table.stats .v { font-family: __DISPLAY__; font-size: __F_heading__; font-weight: 700; color: #__C_teal__; line-height: 1.15; hyphens: manual; }
table.stats .l { font-size: __F_label__; color: #__C_muted__; margin-top: 2px; line-height: 1.25; }
table.stats.big .v { font-size: __F_title__; }
.pill { display: inline-block; padding: 1px 8px; border-radius: 20px; font-size: __F_label__; font-weight: 700; color: #__C_white__; white-space: nowrap; }
.chips { margin-top: 2px; }
.chip { display: inline-block; background: #__C_sand2__; border-radius: 7px; padding: 2px 8px; margin: 0 0 4px; margin-__E__: 4px; font-size: __F_small__; color: #__C_teal__; }
.chip b { color: #__C_ink__; }
.cover { page: cover; position: relative; height: 297mm; padding: 17mm 17mm; color: #__C_white__; overflow: hidden;
  background: linear-gradient(150deg, #__C_teal_dark__ 0%, #__C_teal__ 46%, #__C_teal_mid__ 100%); }
.cover .art { position: absolute; top: 0; __E__: 0; width: 150mm; height: 150mm; }
.cover .net { position: absolute; bottom: 0; left: 0; width: 210mm; height: 70mm; }
.cover .inner { position: relative; }
.cover .kick { letter-spacing: 3px; text-transform: uppercase; font-size: __F_label__; font-weight: 700; color: #__C_mint2__; margin-top: 16mm; }
.cover h1 { font-size: __F_hero__; color: #__C_white__; margin: 13px 0 0; max-width: 146mm; }
.cover .for { font-size: __F_lead__; color: #__C_mint2__; margin-top: 20px; }
.cover .biz { font-family: __DISPLAY__; font-size: __F_display__; color: #__C_gold__; font-weight: 700; margin-top: 2px; }
.cover .modepill { display: inline-block; margin-top: 14px; background: #__C_gold__; color: #__C_gold_ink__; border-radius: 20px; padding: 4px 13px; font-weight: 700; font-size: __F_small__; }
.cover table.glass { width: 100%; table-layout: fixed; border-collapse: separate; border-spacing: 11px 0; margin: 22px -11px 0; }
.cover table.glass td { background: rgba(255,255,255,.10); border: 1px solid rgba(255,255,255,.26); border-radius: 14px; padding: 12px 14px; vertical-align: top; }
.cover .glass .v { font-family: __DISPLAY__; font-size: __F_display__; color: #__C_white__; font-weight: 700; line-height: 1.1; }
.cover .glass .l { font-size: __F_small__; color: #__C_mint3__; margin-top: 3px; }
.cover .foot { position: absolute; left: 17mm; right: 17mm; bottom: 74mm; border-top: 1px solid rgba(255,255,255,.26); padding-top: 11px; }
.cover .foot .meta { font-size: __F_small__; color: #__C_mint2__; margin-top: 4px; }
.toc { display: table; width: 100%; table-layout: fixed; border-collapse: separate; border-spacing: 12px 10px; margin: -10px -12px 0; }
.toc .tr { display: table-row; }
.toc .it { display: table-cell; background: #__C_sand__; border-radius: 14px; padding: 12px 14px; vertical-align: top; }
.toc .n { display: inline-block; width: 30px; height: 30px; border-radius: 9px; color: #__C_white__; text-align: center; line-height: 30px; font-family: __DISPLAY__; font-weight: 700; font-size: __F_body__; margin-__E__: 10px; vertical-align: top; }
.toc .tx { display: inline-block; width: 68%; vertical-align: top; }
.toc .t { font-weight: 700; font-size: __F_lead__; color: #__C_ink__; }
.toc .s { font-size: __F_small__; color: #__C_muted__; margin-top: 2px; }
.toc .pg::after { content: target-counter(attr(href), page); }
.toc .pg { display: inline-block; width: 12%; text-align: __E__; vertical-align: top; font-weight: 700; color: #__C_teal__; font-size: __F_body__; }
.exec p { font-size: __F_lead__; line-height: 1.62; color: #__C_body2__; margin-bottom: 9px; }
.exec p.first { color: #__C_ink__; }
.legend span { display: inline-block; margin: 0 10px 4px 0; font-size: __F_small__; color: #__C_muted__; }
.dot { display: inline-block; width: 9px; height: 9px; border-radius: 50%; margin-__E__: 4px; vertical-align: -1px; }
.scale { height: 12px; border-radius: 6px; background: linear-gradient(__GDIR__, #__C_coral__, #__C_amber__ 40%, #__C_green__ 70%, #__C_teal_dark__); margin: 8px 0 3px; }
.scale-l { display: table; width: 100%; font-size: __F_small__; color: #__C_muted__; }
.scale-l span { display: table-cell; }
.scale-l span:nth-child(2) { text-align: center; }
.scale-l span:last-child { text-align: __E__; }
.modes .mrow { padding: 6px 0; border-bottom: 1px solid #__C_line__; font-size: __F_body__; }
.modes .mrow:last-child { border-bottom: none; }
.modes .mrow b { color: #__C_teal__; }
.modes .mrow.on { background: #__C_white__; border-radius: 9px; padding: 6px 8px; border: 1px solid #__C_gold__; }
.tag { display: inline-block; background: #__C_gold__; color: #__C_gold_ink__; border-radius: 6px; padding: 0 6px; font-size: __F_label__; font-weight: 700; margin-__S__: 6px; }
table.bars { width: 100%; border-collapse: collapse; font-size: __F_small__; }
table.bars td { padding: 3px 4px; border-bottom: 1px solid #__C_sand2__; vertical-align: middle; }
table.bars tr:last-child td { border-bottom: none; }
table.bars td.nm { width: 36%; }
table.bars td.n { width: 8%; text-align: center; font-weight: 700; color: #__C_teal__; }
table.bars a { color: #__C_ink__; font-weight: 600; }
.track { height: 9px; background: #__C_sand2__; border-radius: 5px; overflow: hidden; }
.fill { height: 9px; border-radius: 5px; background: linear-gradient(__GDIR__, #__C_teal__, #__C_teal_mid__); }
.stack { display: table; width: 100%; table-layout: fixed; height: 16px; border-radius: 8px; overflow: hidden; margin: 6px 0 10px; }
.stack span { display: table-cell; height: 16px; }
table.qrows { width: 100%; border-collapse: collapse; font-size: __F_small__; }
table.qrows td { padding: 3px 0; border-bottom: 1px solid #__C_sand2__; }
table.qrows tr:last-child td { border-bottom: none; }
table.qrows td.n { text-align: __E__; font-weight: 700; }
table.index { width: 100%; border-collapse: collapse; font-size: __F_small__; table-layout: fixed; }
table.index thead { display: table-header-group; }
table.index th { background: #__C_teal__; color: #__C_white__; padding: 5px 6px; text-align: __S__; font-weight: 600; font-size: __F_label__; }
table.index td { padding: 4px 6px; border-bottom: 1px solid #__C_line__; vertical-align: middle; }
table.index tr { break-inside: avoid; }
table.index tbody tr:nth-child(even) td { background: #__C_sand3__; }
table.index td.ix-rank { font-family: __DISPLAY__; font-weight: 700; color: #__C_teal__; text-align: center; }
table.index td.ix-id a { color: #__C_coral__; font-weight: 700; }
table.index td.ix-id { white-space: nowrap; }
table.index td.ix-title a { color: #__C_ink__; }
table.index .ixflag { display: inline-block; background: #__C_note__; color: #__C_note_ink__; border-radius: 6px; padding: 0 5px; font-size: __F_label__; font-weight: 700; margin-__E__: 4px; }
table.index td.ix-quality, table.index td.ix-inspire_score { text-align: center; }
table.index td.ix-inspire_score { white-space: nowrap; font-weight: 700; color: #__C_teal__; }
.opener { page: opener; break-before: page; }
.opener .band { position: relative; overflow: hidden; color: #__C_white__; margin: -19mm -15mm 0; padding: 12mm 15mm 11mm;
  background: linear-gradient(150deg, #__C_teal_dark__ 0%, #__C_teal__ 50%, #__C_teal_mid__ 100%); }
.opener .band .art { position: absolute; top: -6mm; __E__: -6mm; width: 96mm; height: 96mm; }
.opener .band .in { position: relative; }
.opener .band .wordmark { display: block; margin-bottom: 6mm; }
.opener .num { font-family: __DISPLAY__; font-size: __F_numeral__; line-height: .92; color: rgba(255,255,255,.2); font-weight: 700; }
.opener .band .eyebrow { color: #__C_coral_soft__; margin-top: 2px; }
.opener h2 { font-size: __F_display__; color: #__C_white__; margin-top: 3px; }
.opener .band .sub { color: #__C_mint3__; font-size: __F_body__; margin-top: 7px; max-width: 170mm; }
.opener .body { padding: 7mm 0 4mm; }
.opener .cols { display: table; width: 100%; table-layout: fixed; }
.opener .cols > * { display: table-cell; vertical-align: top; }
.opener .cols > :first-child { width: 57%; padding-__E__: 14px; }
.opener .overview p { font-size: __F_body__; line-height: 1.6; color: #__C_body2__; }
.opener .overview p.first { font-size: __F_lead__; color: #__C_ink__; }
.opener h4 { font-size: __F_heading__; margin-bottom: 7px; }
.uc { break-inside: avoid; border: 1px solid #__C_line__; border-radius: 14px; padding: 11px 14px; margin: 0 0 10px; background: #__C_white__; }
.uc .main { break-inside: avoid; }
.uc .head { display: table; width: 100%; }
.uc .head > * { display: table-cell; vertical-align: top; }
.uc .badge { width: 40px; height: 40px; border-radius: 11px; background: #__C_teal__; color: #__C_white__; font-family: __DISPLAY__; font-weight: 700; font-size: __F_heading__; text-align: center; vertical-align: middle; }
.uc .badge.elite { background: #__C_gold__; color: #__C_gold_ink__; }
.uc .badge.coverage { background: #__C_coral__; }
.uc .hd { padding-__S__: 12px; }
.uc .kick { color: #__C_coral__; font-weight: 700; font-size: __F_label__; letter-spacing: 1.2px; text-transform: uppercase; }
.uc h3 { font-size: __F_heading__; margin-top: 2px; color: #__C_ink__; }
.uc .sub { color: #__C_muted__; font-size: __F_small__; margin-top: 2px; }
.uc .band { height: 5px; border-radius: 3px; margin: 7px 0; background: linear-gradient(__GDIR__, #__C_teal__, #__C_teal_mid__ 62%, #__C_gold__); }
.uc table.stats { margin-bottom: 7px; border-spacing: 5px 0; margin-left: -5px; margin-right: -5px; }
.uc table.stats td { padding: 5px 4px; border-radius: 9px; }
.uc table.stats .v { font-size: __F_small__; }
.uc table.stats .l { font-size: __F_micro__; }
.uc .cols { display: table; width: 100%; table-layout: fixed; }
.uc .cols > * { display: table-cell; vertical-align: top; width: 50%; }
.uc .cols > :first-child { padding-__E__: 13px; }
.uc h5 { font-family: __BODY__; font-size: __F_label__; font-weight: 700; text-transform: uppercase; letter-spacing: .8px; color: #__C_teal__; margin: 0 0 2px; }
.uc .blk { margin-bottom: 6px; }
.uc .blk p { font-size: __F_body__; margin: 0; }
table.kv { border-collapse: collapse; width: 100%; font-size: __F_small__; margin-top: 2px; }
table.kv td { padding: 2px 0; vertical-align: top; border-bottom: 1px solid #__C_sand2__; }
table.kv tr:last-child td { border-bottom: none; }
table.kv td.k { color: #__C_teal__; font-weight: 700; width: 44%; padding-__E__: 9px; hyphens: manual; }
.uc .foot { text-align: __E__; margin-top: 5px; }
.btn { display: inline-block; background: #__C_night__; color: #__C_mint__; border-radius: 8px; padding: 4px 11px; font-size: __F_label__; font-weight: 700; }
.ribbon { background: #__C_note__; border: 1px solid #__C_note_line__; border-__S__: 3px solid #__C_coral__; border-radius: 9px; padding: 6px 10px; font-size: __F_small__; color: #__C_note_ink__; margin-bottom: 8px; }
.ribbon b { color: #__C_note_dark__; }
.ribbon.star { border-__S__-color: #__C_gold__; }
.untr { font-size: __F_label__; color: #__C_muted__; font-style: italic; margin-bottom: 6px; }
.uc .val { display: table; width: 100%; table-layout: fixed; margin-top: 6px; padding-top: 6px; border-top: 1px solid #__C_sand2__; }
.uc .val > * { display: table-cell; vertical-align: top; width: 50%; }
.uc .val > :first-child { padding-__E__: 13px; }
.uc .blk ul { margin: 0; padding-__S__: 14px; font-size: __F_small__; }
.uc .blk li { margin: 0 0 2px; }
table.index td .jn { display: block; color: #__C_muted__; font-size: __F_label__; margin-top: 2px; }
table.found td { vertical-align: top; }
.steps { list-style: none; margin: 0; padding: 0; }
.steps li { position: relative; padding: 7px 0; padding-__S__: 36px; border-bottom: 1px solid #__C_line__; font-size: __F_body__; line-height: 1.4; }
.steps li:last-child { border-bottom: none; }
.steps li .k { position: absolute; __S__: 0; top: 6px; width: 24px; height: 24px; background: #__C_teal__; color: #__C_white__; border-radius: 50%; text-align: center; line-height: 24px; font-weight: 700; font-family: __DISPLAY__; font-size: __F_body__; }
.steps li b { color: #__C_ink__; }
.gloss .g { padding: 5px 0; border-bottom: 1px solid #__C_line__; font-size: __F_small__; }
.gloss .g:last-child { border-bottom: none; }
.gloss .g b { color: #__C_teal__; }
table.road { width: 100%; table-layout: fixed; border-collapse: separate; border-spacing: 10px 0; margin: 0 -10px; }
table.road td { background: #__C_sand__; border-radius: 14px; padding: 12px; vertical-align: top; border-top: 4px solid #__C_teal__; }
table.road .k { display: inline-block; width: 26px; height: 26px; border-radius: 50%; color: #__C_white__; text-align: center; line-height: 26px; font-weight: 700; font-family: __DISPLAY__; }
table.road h4 { font-size: __F_body__; margin: 8px 0 4px; }
table.road p { font-size: __F_small__; color: #__C_body__; margin: 0; }
.callout { background: #__C_ice__; border-__S__: 4px solid #__C_teal__; border-radius: 12px; padding: 12px 14px; font-size: __F_lead__; color: #__C_ink__; }
.disc .item { position: relative; padding: 11px 0; padding-__S__: 44px; border-bottom: 1px solid #__C_line__; font-size: __F_lead__; line-height: 1.55; color: #__C_body2__; }
.disc .item:last-child { border-bottom: none; }
.disc .item .k { position: absolute; __S__: 0; top: 10px; width: 30px; height: 30px; border-radius: 9px; background: #__C_teal__; color: #__C_white__; text-align: center; line-height: 30px; font-weight: 700; font-family: __DISPLAY__; }
.closing { page: closing; break-before: page; position: relative; height: 297mm; padding: 0 22mm; color: #__C_white__; overflow: hidden;
  background: linear-gradient(150deg, #__C_teal_dark__ 0%, #__C_teal__ 46%, #__C_teal_mid__ 100%); text-align: center; }
.closing .art { position: absolute; bottom: -10mm; __S__: -10mm; width: 130mm; height: 130mm; }
.closing .in { position: relative; padding-top: 80mm; }
.closing .kick { letter-spacing: 3px; text-transform: uppercase; font-size: __F_label__; font-weight: 700; color: #__C_mint2__; }
.closing h2 { font-size: __F_display__; color: #__C_white__; margin-top: 12px; }
.closing .gold { font-family: __DISPLAY__; font-size: __F_title__; color: #__C_gold__; font-weight: 700; margin-top: 4px; }
.closing .sum { font-size: __F_lead__; color: #__C_mint__; margin-top: 16px; }
.closing .wordmark { display: block; margin-top: 30px; }
.closing .meta { margin-top: 6px; font-size: __F_small__; color: #__C_mint2__; }
html.nocaps .eyebrow, html.nocaps .uc .kick, html.nocaps .uc h5, html.nocaps .elite .k, html.nocaps .cover .kick, html.nocaps .closing .kick { letter-spacing: 0; text-transform: none; }
html.rtl .cover .art, html.rtl .opener .band .art, html.rtl .closing .art { transform: scaleX(-1); }
html.rtl [dir="ltr"] { text-align: right; }
"""

    def _pdf_css(self, model: dict, body_stack: str, display_stack: str) -> str:
        rtl = model["rtl"]
        start, end = ("right", "left") if rtl else ("left", "right")
        quoted = lambda text: str(text).replace("\\", "\\\\").replace('"', '\\"')
        css = (self._PDF_CSS.replace("__S__", start).replace("__E__", end).replace("__GDIR__", "270deg" if rtl else "90deg")
               .replace("__PAGE__", '"\\200E " counter(page) " / " counter(pages) "\\200E "' if rtl else 'counter(page) " / " counter(pages)')
               .replace("__BODY__", body_stack).replace("__DISPLAY__", display_stack if model["latin_display"] else body_stack)
               .replace("__FOOTER__", quoted(model["generated_by"])).replace("__BUSINESS__", quoted(model["business_name"])))
        css = re.sub(r"__C_(\w+?)__", lambda m: self._DOC_COLORS[m.group(1)].lower(), css)
        return re.sub(r"__F_(\w+?)__", lambda m: f"{self._DOC_TYPE_SCALE[m.group(1)] * self._DOC_PDF_PX_PER_PT:.1f}px", css)

    def _doc_art_svg(self, kind: str) -> str:
        white, gold = "#" + self._DOC_COLORS["white"].lower(), "#" + self._DOC_COLORS["gold"].lower()
        if kind == "rings":
            rings = "".join(f'<circle cx="{cx}" cy="{cy}" r="{r}" fill="none" stroke="{white}" stroke-opacity="{op}" stroke-width="1.2"/>'
                            for cx, cy, r, op in ((470, 110, 150, .18), (470, 110, 110, .14), (470, 110, 70, .12), (350, 40, 120, .1),
                                                  (560, 260, 140, .1), (300, 220, 60, .1), (520, 10, 50, .14)))
            return f'<svg class="art" viewBox="0 0 560 560" xmlns="http://www.w3.org/2000/svg">{rings}<circle cx="505" cy="62" r="24" fill="{gold}" fill-opacity=".85"/></svg>'
        nodes = ((30, 210), (90, 150), (150, 190), (210, 120), (270, 175), (330, 95), (390, 160), (450, 110), (510, 170), (570, 125),
                 (630, 185), (690, 140), (750, 200), (120, 240), (240, 230), (360, 225), (480, 235), (600, 228), (720, 238))
        edges = ((0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (1, 13), (2, 13),
                 (4, 14), (3, 14), (6, 15), (5, 15), (8, 16), (7, 16), (10, 17), (9, 17), (12, 18), (11, 18))
        lines = "".join(f'<line x1="{nodes[a][0]}" y1="{nodes[a][1]}" x2="{nodes[b][0]}" y2="{nodes[b][1]}" stroke="{white}" stroke-opacity=".16" stroke-width="1.4"/>'
                        for a, b in edges)
        dots = "".join(f'<circle cx="{x}" cy="{y}" r="{5 if i % 3 else 7}" fill="{gold if i % 5 == 0 else white}" fill-opacity="{.75 if i % 5 == 0 else .28}"/>'
                       for i, (x, y) in enumerate(nodes))
        bars = "".join(f'<rect x="{x}" y="{265 - h}" width="34" height="{h}" rx="4" fill="{white}" fill-opacity=".07"/>'
                       for x, h in ((20, 40), (70, 70), (120, 55), (170, 95), (220, 65), (270, 120), (320, 80), (370, 105), (420, 60),
                                    (470, 130), (520, 85), (570, 110), (620, 70), (670, 125), (720, 90)))
        return f'<svg class="net" viewBox="0 0 780 265" preserveAspectRatio="none" xmlns="http://www.w3.org/2000/svg">{bars}{lines}{dots}</svg>'

    def _pdf_wordmark(self, extra_class: str = "") -> str:
        brand = self._DOC_BRAND
        return (f'<span class="wordmark {extra_class}" dir="ltr"><i class="mark"></i>{html.escape(brand["maker"])} '
                f'<b>{html.escape(brand["product"])}</b></span>')

    def _pdf_index_table(self, rows: list, model: dict, t: dict, link) -> str:
        e = lambda value: html.escape(str(value if value is not None else ""))
        header = self._doc_index_header(model, t)
        cols = "".join(f'<col style="width:{100.0 * width:.1f}%">' for _key, _label, width in header)
        head = "".join(f"<th>{e(label)}</th>" for _key, label, _width in header)
        body = []
        for row in rows:
            cells = []
            for key, _label, _width in header:
                value = row[key]
                if key == "id":
                    url = link(row["uc"])
                    inner = f'<a href="{e(url)}">{e(value)}</a>' if url else e(value)
                elif key == "title":
                    flag = f'<span class="ixflag">{e(row["flag"])}</span> ' if row["flag"] else ""
                    inner = f'{flag}<a href="#uc-{e(row["id"])}">{self._doc_bidi_html(value, model)}</a>'
                elif key == "quality":
                    inner = f'<span class="pill" style="background:#{row["quality_color"].lower()}">{self._doc_bidi_html(value, model)}</span>'
                elif key == "domain":
                    inner = self._doc_bidi_html(value, model)
                elif key == "inspire_score" and model["rtl"]:
                    inner = f'<span dir="ltr">{e(value)}</span>'
                else:
                    inner = e(value)
                cells.append(f'<td class="ix-{key}">{inner}</td>')
            body.append("<tr>" + "".join(cells) + "</tr>")
        return f'<table class="index"><colgroup>{cols}</colgroup><thead><tr>{head}</tr></thead><tbody>{"".join(body)}</tbody></table>'

    def _catalog_pdf_html(self, model: dict, t: dict, fonts: tuple) -> str:
        e = lambda value: html.escape(str(value if value is not None else ""))
        language = model["language"]
        T = lambda key, **values: self._doc_text(t, key, language, **{**model["tiers"], **values})
        face_css, body_stack, display_stack = fonts[0], fonts[1], fonts[2]
        rtl = model["rtl"]
        css = self._pdf_css(model, body_stack, display_stack)
        classes = " ".join(c for c in ("rtl" if rtl else "", "" if model["latin_display"] else "nocaps") if c)
        nb_urls = {}

        def ltr(value):
            return f'<span dir="ltr">{e(value)}</span>'

        def notebook_url(uc):
            key = id(uc)
            if key not in nb_urls:
                nb_urls[key] = self._get_use_case_notebook_url(uc)
            return nb_urls[key]

        def para_html(paragraphs):
            return "".join(f'<p class="{"first" if i == 0 else ""}"{self._doc_bidi(p, model)}>{e(p)}</p>' for i, p in enumerate(paragraphs))

        def stats_table(items, extra=""):
            cells = "".join(f'<td><div class="v">{self._doc_bidi_html(value, model)}</div><div class="l">{e(label)}</div></td>' for value, label in items)
            return f'<table class="stats {extra}"><tr>{cells}</tr></table>'

        def heading(section, eyebrow, title):
            return (f'<div class="eyebrow">{e(eyebrow)}</div><h2 class="title" id="sec-{section}">{e(title)}</h2><div class="rule"></div>')

        accents = [self._DOC_COLORS[name].lower() for name in ("teal", "coral", "gold", "teal_mid", "green", "amber")]
        parts = [f'<!doctype html><html lang="{e(model["lang"])}" dir="{"rtl" if rtl else "ltr"}" class="{classes}"><head><meta charset="utf-8">'
                 f'<title>{e(T("pdf_title"))} \u00b7 {e(model["business_name"])}</title><style>{face_css}\n{css}</style></head><body>'
                 f'<div class="runbrand">{self._pdf_wordmark()}</div>']

        mode_html = f'<div class="modepill">{e(T("doc_quality_mode"))}: {e(model["mode_label"])}</div>' if model["mode"] else ""
        glass = "".join(f'<td><div class="v">{e(value)}</div><div class="l">{e(label)}</div></td>' for value, label in model["headline"][:3])
        parts.append(
            f'<section class="cover">{self._doc_art_svg("rings")}{self._doc_art_svg("net")}<div class="inner">'
            f'{self._pdf_wordmark("light big")}'
            f'<div class="kick">{e(T("doc_catalog_kicker"))} \u00b7 {ltr(model["date"])}</div>'
            f'<h1>{e(T("pdf_title"))}</h1>'
            f'<div class="for">{e(T("doc_prepared_for"))}</div><div class="biz">{self._doc_bidi_html(model["business_name"], model)}</div>{mode_html}'
            f'<table class="glass"><tr>{glass}</tr></table></div>'
            f'<div class="foot"><div class="meta">{self._doc_bidi_html(model["generated_by"], model)}</div></div></section>')

        outline = self._doc_outline(model, t)
        tiles = [f'<div class="it"><span class="n" style="background:#{accents[i % len(accents)]}">{i + 1:02d}</span>'
                 f'<span class="tx"><span class="t">{e(title)}</span>'
                 f'<div class="s"{self._doc_bidi(subtitle, model)}>{e(subtitle)}</div></span><a class="pg" href="#sec-{key}"></a></div>'
                 for i, (key, title, subtitle) in enumerate(outline)]
        rows = "".join(f'<div class="tr">{"".join(tiles[i:i + 2])}</div>' for i in range(0, len(tiles), 2))
        parts.append(f'<section class="sheet">{heading("contents", model["business_name"], T("doc_contents"))}<div class="toc">{rows}</div></section>')

        parts.append(
            f'<section class="sheet exec">{heading("executive", T("doc_exec_kicker"), T("pdf_exec_summary"))}'
            f'{stats_table(model["headline"], "big")}<div class="txt gap">{para_html(model["executive"])}</div></section>')

        legend = "".join(f'<span><i class="dot" style="background:#{self._doc_quality_color(label).lower()}"></i>{e(self._translate_display_value(label, t))}'
                         f' <b>{model["quality_counts"][label]}</b></span>' for label in self._DOC_QUALITY_ORDER)
        mode_rows = []
        for mode, body_key in (("Good", "doc_mode_good_body"), ("High", "doc_mode_high_body"), ("Ultra High", "doc_mode_ultra_body")):
            on = mode == model["mode"]
            tag = f'<span class="tag">{e(T("doc_this_catalog"))}</span>' if on else ""
            mode_rows.append(f'<div class="mrow{" on" if on else ""}"><b>{e(T(self._DOC_MODE_KEYS[mode]))}</b>{tag}<br>{e(T(body_key))}</div>')
        parts.append(
            f'<section class="sheet sand">{heading("guide", T("doc_guide_kicker"), T("doc_how_to_read"))}<p class="lead">{e(T(model.get("guide_lead_key", "doc_guide_lead")))}</p>'
            f'<div class="grid2"><div class="card"><h4>{e(T("doc_inspire_score_title"))}</h4><p>{e(T("doc_inspire_score_body"))}</p>'
            f'<div class="scale"></div><div class="scale-l"><span>0</span><span>5</span><span>10</span></div></div>'
            f'<div class="card"><h4>{e(T("doc_quality_tiers_title"))}</h4><p>{e(T("doc_quality_tiers_body"))}</p><div class="legend">{legend}</div></div></div>'
            f'<div class="grid2 gap"><div class="card soft modes"><h4>{e(T("doc_modes_title"))}</h4>{"".join(mode_rows)}</div>'
            f'<div><div class="card"><h4>{e(T("doc_elite_score_title"))}</h4><p>{e(T("doc_elite_score_body"))}</p></div>'
            f'<div class="card gap"><h4>{e(T("doc_coverage_title"))}</h4><p>{e(T("doc_coverage_body"))}</p></div></div></div></section>')

        max_count = max((len(d["use_cases"]) for d in model["domains"]), default=1) or 1
        bar_rows = "".join(
            f'<tr><td class="nm"><a href="#domain-{d["index"]}">{self._doc_bidi_html(d["name"], model)}</a></td>'
            f'<td><div class="track"><div class="fill" style="width:{max(4.0, 100.0 * len(d["use_cases"]) / max_count):.1f}%"></div></div></td>'
            f'<td class="n">{len(d["use_cases"])}</td></tr>' for d in model["domains"])
        segments = "".join(f'<span style="width:{share:.2f}%;background:#{self._doc_quality_color(label).lower()}"></span>'
                           for (label, _n), share in zip(model["quality_rows"], self._doc_shares([n for _label, n in model["quality_rows"]])))
        quality_rows = "".join(f'<tr><td><i class="dot" style="background:#{self._doc_quality_color(label).lower()}"></i>{e(name)}</td>'
                               f'<td class="n">{n}</td></tr>' for (label, n), name in zip(model["quality_rows"], model["quality_names"]))
        parts.append(
            f'<section class="sheet">{heading("portfolio", T("doc_portfolio_kicker"), T("doc_portfolio_glance"))}'
            f'<div class="grid2"><div class="card"><h4>{e(T("doc_per_domain"))}</h4><table class="bars">{bar_rows}</table></div>'
            f'<div class="card soft"><h4>{e(T("doc_quality_distribution"))}</h4><div class="stack">{segments}</div><table class="qrows">{quality_rows}</table></div></div>'
            f'<div class="card gap"><h4>{e(T("doc_top_use_cases"))}</h4>{self._pdf_index_table(self._doc_index_rows(model["top"], model, t), model, t, notebook_url)}</div>'
            f'</section>')

        for domain in model["domains"]:
            parts.append(self._pdf_domain_opener(domain, model, t, notebook_url))
            parts.append('<section class="cards">' + "".join(
                self._pdf_use_case_card(self._use_case_view(uc, model, t), model, t, notebook_url(uc)) for uc in domain["use_cases"]) + '</section>')

        parts.append(
            f'<section class="sheet">{heading("index", T("doc_appendix"), T("doc_use_case_index"))}'
            f'<p class="lead">{e(T("doc_index_lead", count=model["total"]))}</p>'
            f'{self._pdf_index_table(self._doc_index_rows(model["ranked"], model, t), model, t, notebook_url)}</section>')

        steps = "".join(f'<li><span class="k">{i}</span>{e(text)}</li>' for i, text in enumerate(model["method_steps"], start=1))
        glossary = "".join(f'<div class="g"><b>{e(term)}</b>: {e(body)}</div>' for term, body in model["glossary"])
        parts.append(
            f'<section class="sheet sand">{heading("methodology", T("doc_appendix"), T("doc_methodology"))}'
            f'<div class="card"><ol class="steps">{steps}</ol></div>'
            f'<div class="card gap gloss"><h4>{e(T("doc_glossary"))}</h4>{glossary}</div></section>')

        if model["foundations"]:
            separator = self._DOC_LIST_SEPARATORS.get(model["language"], ", ")
            found_rows = "".join(
                f'<tr><td class="ix-id">{e(row["id"])}</td><td>{self._doc_bidi_html(row["tables"], model)}'
                + (f'<span class="jn">{self._doc_bidi_html(row["joins"], model)}</span>' if row["joins"] else "") + '</td>'
                f'<td><b>{row["count"]}</b> \u00b7 ' + separator.join(f'<a href="#uc-{e(uc_id)}">{e(uc_id)}</a>' for uc_id in row["use_case_ids"])
                + (f' (+{row["more"]})' if row["more"] else "") + '</td></tr>' for row in model["foundations"])
            parts.append(
                f'<section class="sheet">{heading("foundations", T("doc_next_steps"), T("doc_foundations_title"))}'
                f'<p class="lead">{e(T("doc_foundations_lead"))}</p>'
                f'<table class="index found"><colgroup><col style="width:8%"><col style="width:52%"><col style="width:40%"></colgroup>'
                f'<thead><tr><th></th><th>{e(T("doc_foundation_tables"))}</th><th>{e(T("doc_foundation_unlocks"))}</th></tr></thead>'
                f'<tbody>{found_rows}</tbody></table></section>')

        road = "".join(f'<td style="border-top-color:#{accents[i % len(accents)]}"><span class="k" style="background:#{accents[i % len(accents)]}">{i + 1}</span>'
                       f'<h4>{e(title)}</h4><p>{e(body)}</p></td>' for i, (title, body) in enumerate(model["roadmap"]))
        parts.append(
            f'<section class="sheet">{heading("roadmap", T("doc_next_steps"), T("doc_roadmap"))}'
            f'<table class="road"><tr>{road}</tr></table><div class="callout gap">{e(T("doc_closing_body"))}</div></section>')

        disclaimer_title, disclaimer = self._doc_disclaimer(model, t)
        items = "".join(f'<div class="item"><span class="k">{i}</span>{e(text)}</div>' for i, text in enumerate(disclaimer, start=1))
        parts.append(
            f'<section class="sheet sand">{heading("disclaimer", self._DOC_BRAND["wordmark"], disclaimer_title)}'
            f'<div class="card disc">{items}</div></section>')

        parts.append(
            f'<section class="closing">{self._doc_art_svg("rings")}<div class="in"><div class="kick">{e(T("doc_next_steps"))}</div>'
            f'<h2>{e(T("doc_closing_title"))}</h2><div class="gold">{self._doc_bidi_html(model["business_name"], model)}</div>'
            f'<div class="sum">{e(model["closing_summary"])}</div>{self._pdf_wordmark("light big")}'
            f'<div class="meta">{self._doc_bidi_html(model["generated_by"], model)}</div></div></section>')
        parts.append("</body></html>")
        return "".join(parts)

    def _pdf_domain_opener(self, domain: dict, model: dict, t: dict, link) -> str:
        e = lambda value: html.escape(str(value if value is not None else ""))
        T = lambda key, **values: self._doc_text(t, key, model["language"], **{**model["tiers"], **values})
        number = f'{domain["index"]:02d}'
        subtitle = self._doc_join([name for name, _ in domain["subdomains"][:5]], model["language"])
        chips = "".join(f'<span class="chip">{self._doc_bidi_html(name, model)} <b>{count}</b></span>' for name, count in domain["subdomains"][:12])
        cells = "".join(f'<td><div class="v">{self._doc_bidi_html(value, model)}</div><div class="l">{e(label)}</div></td>' for value, label in domain["stats"])
        anchor = ' id="sec-domains"' if domain["index"] == 1 else ""
        return (f'<section class="opener"{anchor}><div class="band">{self._doc_art_svg("rings")}<div class="in">{self._pdf_wordmark("light")}'
                f'<div class="num">{number}</div><div class="eyebrow">{e(T("doc_domain_label"))} {number}</div>'
                f'<h2 id="domain-{domain["index"]}">{self._doc_bidi_html(domain["name"], model)}</h2>'
                + (f'<div class="sub">{self._doc_bidi_html(subtitle, model)}</div>' if subtitle else "") +
                f'</div></div><div class="body"><div class="cols"><div class="overview txt"><h4>{e(T("doc_domain_overview"))}</h4>'
                + "".join(f'<p class="{"first" if i == 0 else ""}"{self._doc_bidi(p, model)}>{e(p)}</p>' for i, p in enumerate(domain["overview"])) +
                f'</div><div><table class="stats"><tr>{cells}</tr></table>'
                + (f'<h4 style="margin-top:12px">{e(T("doc_what_it_covers"))}</h4><div class="chips">{chips}</div>' if chips else "") +
                f'</div></div><div class="card soft gap"><h4>{e(T("doc_top_opportunities"))}</h4>'
                f'{self._pdf_index_table(self._doc_index_rows(domain["top"], model, t), model, t, link)}</div></div></section>')

    def _pdf_use_case_card(self, view: dict, model: dict, t: dict, url: str) -> str:
        e = lambda value: html.escape(str(value if value is not None else ""))
        bidi = lambda value: self._doc_bidi(value, model)
        kicker = " \u00b7 ".join(part for part in (view["id"], view["domain"], view["subdomain"]) if part)
        metrics = "".join(f'<td><div class="v">{self._doc_bidi_html(value, model)}</div><div class="l">{e(label)}</div></td>'
                          for _key, label, value in view["metrics"])
        out = [f'<article class="uc" id="uc-{e(view["id"])}"><div class="main"><div class="head">'
               f'<div class="badge {view["kind"]}">{e(view["rank"])}</div>'
               f'<div class="hd"><div class="kick">{self._doc_bidi_html(kicker, model)}</div><h3{bidi(view["title"])}>{e(view["title"])}</h3>'
               + (f'<div class="sub"{bidi(view["name"])}>{e(view["name"])}</div>' if view["name"] else "") + '</div></div><div class="band"></div>'
               f'<table class="stats"><tr>{metrics}</tr></table>']
        if view["star"]:
            out.append(f'<div class="ribbon star"><b>{self._doc_bidi_html(view["star"], model)}</b></div>')
        if view["coverage"]:
            label, reason_label, reason = view["coverage"]
            out.append(f'<div class="ribbon"><b>{e(label)}</b>. {e(reason_label)}: {self._doc_bidi_html(reason, model)}</div>')
        if view["notice"]:
            out.append(f'<div class="untr"><em>{e(view["notice"])}</em></div>')
        blocks = "".join(f'<div class="blk"><h5>{e(label)}</h5><p{bidi(value)}>{e(value)}</p></div>' for _key, label, value in view["narrative"])
        facts = "".join(f'<tr><td class="k">{e(label)}</td><td>{self._doc_bidi_html(value, model)}</td></tr>' for _key, label, value in view["facts"])
        out.append(f'<div class="cols txt"><div>{blocks}</div><div><table class="kv">{facts}</table></div></div>')
        if view.get("value_facts"):
            def value_block(entries):
                return "".join(f'<div class="blk"><h5>{e(label)}</h5><ul>'
                               + "".join(f'<li{bidi(item)}>{e(item)}</li>' for item in items) + '</ul></div>'
                               for _key, label, items in entries)
            left = [fact for fact in view["value_facts"] if fact[0] in self._DOC_VALUE_LEFT]
            right = [fact for fact in view["value_facts"] if fact[0] not in self._DOC_VALUE_LEFT]
            out.append(f'<div class="val"><div>{value_block(left)}</div><div>{value_block(right)}</div></div>')
        if url:
            out.append(f'<div class="foot"><a class="btn" href="{e(url)}">{e(self._doc_text(t, "doc_open_notebook", model["language"]))} \u2197</a></div>')
        out.append("</div>")
        out.append("</article>")
        return "".join(out)

    def generate_catalog_pdf(self, language: str, lang_abbr: str, translations: dict, summary_dict: dict, grouped_data: dict, transliterated_name: str):
        self.logger.info(f"--- Starting PDF Catalog Generation for {language} ---")
        t = translations or {}
        try:
            if not grouped_data:
                self.logger.warning(f"No use cases provided to generate_catalog_pdf for {language}. Skipping.")
                return
            self._require_python_package("weasyprint", "weasyprint", f"PDF Catalog ({language})")
            import weasyprint
            import logging as _logging
            try:
                from weasyprint.text.fonts import FontConfiguration
            except ImportError:
                try:
                    from weasyprint.fonts import FontConfiguration
                except ImportError:
                    FontConfiguration = None
            model = self._document_model(language, t, summary_dict, grouped_data, transliterated_name)
            texts = [model["business_name"]] + model["executive"] + [p for d in model["domains"] for p in d["overview"]]
            texts += [str(uc.get(f) or "") for uc in model["ranked"] for f in ("Name", "Statement", "Solution", "Business Value", "Business Domain", "Subdomain")]
            texts += [str(v) for k, v in t.items() if isinstance(v, str) and (k.startswith("doc_") or k.startswith("pdf_"))]
            scripts = self._doc_scripts(language, texts)
            fonts = self._doc_font_css(language, scripts)
            self._record_document_font_fallback(language, fonts[4])
            html_content = self._catalog_pdf_html(model, t, fonts)
            pdf_workspace_path = os.path.join(self.docs_output_dir, f"{self.business_name}-dbx_inspire_{lang_abbr}.pdf")
            local_pdf_path = None
            weasy_logger = _logging.getLogger("weasyprint")
            previous_level = weasy_logger.level
            try:
                with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
                    local_pdf_path = tmp_file.name
                weasy_logger.setLevel(_logging.ERROR)
                try:
                    with self._PDF_RENDER_LOCK:
                        weasyprint.HTML(string=html_content).write_pdf(local_pdf_path, font_config=FontConfiguration() if FontConfiguration else None)
                finally:
                    weasy_logger.setLevel(previous_level)
                with open(local_pdf_path, "rb") as handle:
                    pdf_data = handle.read()
                if not pdf_data:
                    raise ValueError("Generated PDF file is empty.")
                self._import_workspace_file(pdf_workspace_path, base64.b64encode(pdf_data).decode(), workspace.ImportFormat.AUTO)
                abs_path = self.w_client.workspace.get_status(pdf_workspace_path).path
                self.logger.info(f"[DOCS-PDF] {language}: {model['total']} use case(s), {len(model['domains'])} domain page(s), "
                                 f"mode={model['mode'] or 'unknown'}, scripts={scripts}, rtl={model['rtl']}, bytes={len(pdf_data)} -> {abs_path}")
                log_print(f"Success! PDF Catalog ({language}) generated: {abs_path}")
            finally:
                if local_pdf_path and os.path.exists(local_pdf_path):
                    os.remove(local_pdf_path)
        except Exception as e:
            self.logger.critical(f"An error occurred during PDF generation for {language}: {get_clean_error_message(e)}")
            self._record_failed_step("PDF Catalog", language, e)
            raise

    _DOC_PPTX_INDEX_ROWS = 12
    _DOC_PPTX_FONTS = {
        "ea": {"Japanese": "Yu Gothic", "Korean": "Malgun Gothic", "Chinese (Mandarin)": "Microsoft YaHei"},
        "cs": {"Arabic": "Arial", "Hindi": "Nirmala UI", "Tamil": "Nirmala UI"},
    }
    _DOC_PPTX_LINE_FACTORS = {"Japanese": 1.32, "Korean": 1.32, "Chinese (Mandarin)": 1.32, "Arabic": 1.38, "Hindi": 1.45, "Tamil": 1.45}
    _DOC_PPTX_SERIES = ("domain", "uc_card", "index")
    _DOC_WIDE_CHARS = re.compile("[\u1100-\u11ff\u2e80-\u9fff\uac00-\ud7af\uf900-\ufaff\ufe30-\ufe4f\uff00-\uffef]")
    _DOC_TOKEN = re.compile("[\u1100-\u11ff\u2e80-\u9fff\uac00-\ud7af\uf900-\ufaff\ufe30-\ufe4f\uff00-\uffef]|[^\\s\u1100-\u11ff\u2e80-\u9fff\uac00-\ud7af\uf900-\ufaff\ufe30-\ufe4f\uff00-\uffef]+|\\s+")

    class _Deck:
        W, H = 13.333, 7.5
        MIN_PT = 12.0

        def __init__(self, owner, model, t):
            from pptx import Presentation
            from pptx.util import Inches
            self.owner, self.model, self.t = owner, model, t
            self.language, self.rtl = model["language"], model["rtl"]
            self.prs = Presentation()
            self.prs.slide_width, self.prs.slide_height = Inches(self.W), Inches(self.H)
            self.layout = self.prs.slide_layouts[6]
            self.records = []
            self.C = owner._DOC_COLORS
            self.S = {step: max(self.MIN_PT, float(size)) for step, size in owner._DOC_TYPE_SCALE.items()}
            self.brand = owner._DOC_BRAND
            self.ea = owner._DOC_PPTX_FONTS["ea"].get(self.language)
            self.cs = owner._DOC_PPTX_FONTS["cs"].get(self.language)
            self.line_factor = owner._DOC_PPTX_LINE_FACTORS.get(self.language, 1.2)
            self.caps = model["latin_display"]
            self._urls = {}

        def T(self, key, **values):
            return self.owner._doc_text(self.t, key, self.language, **{**self.model["tiers"], **values})

        @staticmethod
        def rgb(value):
            from pptx.dml.color import RGBColor
            return RGBColor.from_string(value)

        def mx(self, x, w):
            return self.W - x - w if self.rtl else x

        def new(self, archetype, background=None, dark=False, gradient=None, top_dark=None):
            slide = self.prs.slides.add_slide(self.layout)
            fill = slide.background.fill
            if gradient:
                fill.gradient()
                fill.gradient_angle = gradient[2]
                stops = fill.gradient_stops
                stops[0].color.rgb, stops[0].position = self.rgb(gradient[0]), 0.0
                stops[1].color.rgb, stops[1].position = self.rgb(gradient[1]), 1.0
            else:
                fill.solid()
                fill.fore_color.rgb = self.rgb(background or self.C["white"])
            self.records.append({"slide": slide, "archetype": archetype, "dark": dark,
                                 "top_dark": dark if top_dark is None else top_dark})
            return slide

        def _alpha(self, shape, alpha):
            from pptx.oxml.ns import qn
            solid = shape._element.spPr.find(qn("a:solidFill"))
            if solid is not None and len(solid):
                color = solid[0]
                node = color.makeelement(qn("a:alpha"), {"val": str(int(max(0.0, min(1.0, alpha)) * 100000))})
                color.append(node)

        def rect(self, slide, x, y, w, h, fill, radius=0.0, line=None, alpha=None, oval=False):
            from pptx.util import Inches, Pt
            from pptx.enum.shapes import MSO_SHAPE
            kind = MSO_SHAPE.OVAL if oval else (MSO_SHAPE.ROUNDED_RECTANGLE if radius else MSO_SHAPE.RECTANGLE)
            shape = slide.shapes.add_shape(kind, Inches(self.mx(x, w)), Inches(y), Inches(w), Inches(h))
            if radius and not oval:
                shape.adjustments[0] = max(0.0, min(0.5, radius / max(0.01, min(w, h))))
            if fill is None:
                shape.fill.background()
            else:
                shape.fill.solid()
                shape.fill.fore_color.rgb = self.rgb(fill)
                if alpha is not None:
                    self._alpha(shape, alpha)
            if line:
                shape.line.color.rgb = self.rgb(line[0])
                shape.line.width = Pt(line[1])
            else:
                shape.line.fill.background()
            self._plain(shape)
            return shape

        def _plain(self, shape):
            from pptx.oxml.ns import qn
            shape.shadow.inherit = False
            style = shape._element.find(qn("p:style"))
            if style is not None:
                shape._element.remove(style)

        def arrow(self, slide, x, y, w, h, color):
            from pptx.util import Inches
            from pptx.enum.shapes import MSO_SHAPE
            kind = MSO_SHAPE.LEFT_ARROW if self.rtl else MSO_SHAPE.RIGHT_ARROW
            shape = slide.shapes.add_shape(kind, Inches(self.mx(x, w)), Inches(y), Inches(w), Inches(h))
            shape.fill.solid()
            shape.fill.fore_color.rgb = self.rgb(color)
            shape.line.fill.background()
            self._plain(shape)
            return shape

        @classmethod
        def char_em(cls, ch):
            code = ord(ch)
            if ch == " ":
                return 0.28
            if code < 128:
                if ch.isupper():
                    return 0.68
                if ch.isdigit():
                    return 0.56
                if ch.islower():
                    return 0.52
                return 0.38
            category = unicodedata.category(ch)
            if category in ("Mn", "Me", "Cf"):
                return 0.0
            if category == "Mc":
                return 0.36
            if 0x0400 <= code <= 0x04FF:
                return 0.62
            if 0x0600 <= code <= 0x06FF or 0x0750 <= code <= 0x077F or 0xFB50 <= code <= 0xFEFF:
                return 0.52
            if 0x0900 <= code <= 0x097F:
                return 0.64
            if 0x0B80 <= code <= 0x0BFF:
                return 0.82
            if DatabricksInspire._DOC_WIDE_CHARS.match(ch):
                return 1.0
            return 0.6

        def line_count(self, text, width_in, pt, bold=False):
            limit = width_in * 72.0 * 0.94
            scale = pt * (1.08 if bold else 1.0)
            total = 0
            for part in str(text).split("\n"):
                lines, current = 1, 0.0
                for token in DatabricksInspire._DOC_TOKEN.findall(part):
                    width = sum(self.char_em(c) for c in token) * scale
                    if token.isspace():
                        current += width
                        continue
                    if current + width > limit and current > 0:
                        lines += 1
                        current = 0.0
                    while width > limit:
                        lines += 1
                        width -= limit
                    current += width
                total += lines
            return total

        def height(self, paragraphs, width_in, pt, bold=False, spacing=1.08, gap=0.35):
            line_h = pt * self.line_factor * spacing / 72.0
            total = 0.0
            for i, text in enumerate(paragraphs):
                total += self.line_count(text, width_in, pt, bold) * line_h
                if i < len(paragraphs) - 1:
                    total += pt * gap / 72.0
            return total

        @staticmethod
        def stack(needs, top, bottom, gap=0.0):
            scale = min(1.0, (bottom - top - gap * (len(needs) - 1)) / max(0.01, sum(needs)))
            spots, y = [], top
            for need in needs:
                spots.append((y, need * scale))
                y += need * scale + gap
            return spots

        @staticmethod
        def trim(text):
            body = str(text).rstrip(" …").rstrip()
            ends = list(re.finditer(r"[.!?\u3002\uff01\uff1f](?=\s|$)", body[:-1]))
            if ends and ends[-1].end() > len(body) * 0.35:
                return body[:ends[-1].end()].rstrip()
            if " " in body.strip():
                return body.rsplit(" ", 1)[0].rstrip(" ,.;:") + "…"
            if len(body) > 2:
                return body[:max(1, int(len(body) * 0.88))] + "…"
            return None

        def fit(self, paragraphs, w, h, size, min_size, bold=False, spacing=1.08, gap=0.35):
            paragraphs = [str(p) for p in paragraphs]
            pt = size
            while pt >= min_size - 1e-6:
                if self.height(paragraphs, w, pt, bold, spacing, gap) <= max(h, self.height(["x"], w, pt, bold, spacing, gap)) + 1e-6:
                    return pt, paragraphs
                pt = round(pt - 0.5, 2)
            pt = min_size
            kept = list(paragraphs)
            room = max(h, self.height(["x"], w, pt, bold, spacing, gap)) + 1e-6
            while kept and self.height(kept, w, pt, bold, spacing, gap) > room:
                shorter = self.trim(kept[-1])
                if shorter is None or len(kept[-1]) < 4:
                    kept.pop()
                else:
                    kept[-1] = shorter
            return pt, kept

        def split(self, paragraphs, w, h, size, min_size, spacing=1.08, gap=0.35):
            paragraphs = [str(p) for p in paragraphs]
            pt, kept = self.fit(paragraphs, w, h, size, min_size, False, spacing, gap)
            if kept == paragraphs:
                return pt, kept, []
            whole = []
            for paragraph in paragraphs:
                if self.height(whole + [paragraph], w, min_size, False, spacing, gap) <= h:
                    whole.append(paragraph)
                else:
                    break
            if whole:
                return min_size, whole, paragraphs[len(whole):]
            return pt, kept, paragraphs[1:]

        @staticmethod
        def _cut_runs(segments, kept_text):
            ellipsis = kept_text.endswith("…")
            length = len(kept_text.rstrip("…").rstrip())
            out, used = [], 0
            for text, style in segments:
                if used + len(text) <= length:
                    out.append((text, style))
                    used += len(text)
                else:
                    out.append((text[:max(0, length - used)].rstrip(), style))
                    break
            if ellipsis and out:
                out[-1] = (out[-1][0].rstrip(" ,.;:") + "…", out[-1][1])
            return out

        def text(self, slide, x, y, w, h, paragraphs, size=None, color=None, bold=False, align="start", anchor="top",
                 min_size=None, spacing=1.08, gap=0.35, fit=True, italic=False, link=None, display=False):
            from pptx.util import Inches, Emu
            from pptx.enum.text import MSO_ANCHOR, MSO_AUTO_SIZE
            size = self.S["lead"] if size is None else size
            color = color or self.C["ink"]
            raw = paragraphs if isinstance(paragraphs, list) else [paragraphs]
            paras = [p if isinstance(p, list) else [(str(p), {})] for p in raw]
            paras = [[(str(seg).replace("`", ""), style) for seg, style in p if seg is not None and str(seg).replace("`", "")] for p in paras]
            paras = [p for p in paras if p]
            if not paras:
                return size
            floor = max(self.MIN_PT, min_size or self.MIN_PT)
            size = max(size, floor)
            if fit:
                plain = ["".join(seg for seg, _ in p) for p in paras]
                size, kept = self.fit(plain, w, h, size, floor, bold, spacing, gap)
                rebuilt = []
                for i, text_value in enumerate(kept):
                    rebuilt.append(paras[i] if text_value == plain[i] else self._cut_runs(paras[i], text_value))
                paras = rebuilt or [[("…", {})]]
            box = slide.shapes.add_textbox(Inches(self.mx(x, w)), Inches(y), Inches(w), Inches(h))
            if link:
                box.click_action.hyperlink.address = link
            frame = box.text_frame
            frame.word_wrap = True
            frame.auto_size = MSO_AUTO_SIZE.NONE
            frame.margin_left = frame.margin_right = frame.margin_top = frame.margin_bottom = Emu(0)
            frame.vertical_anchor = {"top": MSO_ANCHOR.TOP, "middle": MSO_ANCHOR.MIDDLE, "bottom": MSO_ANCHOR.BOTTOM}[anchor]
            face = self.brand["office_display"] if display and self.caps else self.brand["office_body"]
            for i, segments in enumerate(paras):
                paragraph = frame.paragraphs[0] if i == 0 else frame.add_paragraph()
                ltr = self.rtl and self.owner._doc_is_ltr_text("".join(seg for seg, _ in segments))
                self._paragraph(paragraph, align, spacing, size * gap if i < len(paras) - 1 else 0, ltr)
                for seg_text, style in segments:
                    run = paragraph.add_run()
                    run.text = seg_text
                    self._run(run, style.get("size", size), style.get("color", color), style.get("bold", bold),
                              style.get("link"), style.get("italic", italic), style.get("face", face))
            return size

        def _paragraph(self, paragraph, align, spacing, space_after, ltr=False):
            from pptx.util import Pt
            from pptx.enum.text import PP_ALIGN
            paragraph.line_spacing = spacing
            if space_after:
                paragraph.space_after = Pt(space_after)
            paragraph.alignment = {"start": PP_ALIGN.RIGHT if self.rtl else PP_ALIGN.LEFT,
                                   "end": PP_ALIGN.LEFT if self.rtl else PP_ALIGN.RIGHT,
                                   "center": PP_ALIGN.CENTER}[align]
            if self.rtl:
                paragraph._p.get_or_add_pPr().set("rtl", "0" if ltr else "1")

        def _run(self, run, size, color, bold, link, italic, face):
            from pptx.util import Pt
            from pptx.oxml.ns import qn
            font = run.font
            font.size = Pt(max(self.MIN_PT, size))
            font.bold = bool(bold)
            font.italic = bool(italic)
            font.color.rgb = self.rgb(color)
            font.name = face
            props = run._r.get_or_add_rPr()
            props.set("lang", self.model["locale"])
            props.set("altLang", "en-US")
            anchor = props.find(qn("a:latin"))
            for tag, east in (("a:ea", self.ea), ("a:cs", self.cs)):
                if east and anchor is not None:
                    node = props.makeelement(qn(tag), {"typeface": east})
                    anchor.addnext(node)
                    anchor = node
            if link:
                run.hyperlink.address = link

        def url(self, uc):
            key = id(uc)
            if key not in self._urls:
                self._urls[key] = self.owner._get_use_case_notebook_url(uc)
            return self._urls[key]

        def kicker(self, value):
            return value.upper() if self.caps else value

        def ltr(self, value):
            return f"\u200e{value}\u200e" if self.rtl else str(value)

        def theme_links(self):
            from pptx.opc.constants import RELATIONSHIP_TYPE as RT
            theme = self.prs.slide_masters[0].part.part_related_by(RT.THEME)
            blob = theme.blob.decode("utf-8")
            for tag, color in (("hlink", self.C["teal"]), ("folHlink", self.C["teal_mid"])):
                blob = re.sub(rf"(<a:{tag}>\s*<a:srgbClr val=\")[0-9A-Fa-f]{{6}}", rf"\g<1>{color}", blob)
            theme._blob = blob.encode("utf-8")

        def wordmark(self, slide, x, y, w, h, light=False, size=None, align="start"):
            C = self.C
            size = size or self.S["body"]
            dot = size / 72.0 * 0.62
            width = (sum(self.char_em(c) for c in self.brand["wordmark"]) * size * 1.04) / 72.0 + dot + 0.12
            start = x if align == "start" else x + w - width
            self.rect(slide, start, y + (h - dot) / 2.0, dot, dot, C["gold"], oval=True)
            runs = [(self.brand["maker"] + " ", {"color": C["mint2"] if light else C["muted"]}),
                    (self.brand["product"], {"color": C["white"] if light else C["teal"], "bold": True})]
            self.text(slide, start + dot + 0.1, y, max(0.5, width - dot - 0.05), h, [runs], size=size, anchor="middle", fit=False)

        def header(self, slide, kicker, title, dark=False):
            C = self.C
            self.text(slide, 0.6, 0.5, 9.0, 0.3, self.kicker(kicker), size=self.S["body"], color=C["mint2"] if dark else C["coral"], bold=True)
            self.text(slide, 0.6, 0.8, 12.13, 0.66, title, size=self.S["title"], color=C["white"] if dark else C["ink"], bold=True,
                      min_size=18, display=True)
            self.rect(slide, 0.6, 1.5, 0.6, 0.06, C["gold"])

        def lead(self, slide, text, italic=False):
            if not text:
                return 1.75
            h = min(0.62, self.height([text], 12.13, self.S["body"], spacing=1.0) + 0.04)
            self.text(slide, 0.6, 1.62, 12.13, h, text, size=self.S["body"], color=self.C["muted"], italic=italic, spacing=1.0)
            return max(1.75, 1.62 + h + 0.06)

        def tile(self, slide, x, y, w, h, value, label, accent=None, fill=None, value_size=None):
            C = self.C
            self.rect(slide, x, y, w, h, fill or C["sand"], radius=0.12)
            label_need = self.height([label], w - 0.2, self.S["body"], spacing=1.0) + 0.02
            value_need = self.height([str(value)], w - 0.2, value_size or self.S["heading"], bold=True, spacing=1.0)
            label_h = min(h - 0.3, max(h * 0.44, label_need), max(label_need, h - value_need - 0.12))
            self.text(slide, x + 0.1, y + 0.04, w - 0.2, h - label_h - 0.08, str(value), size=value_size or self.S["heading"],
                      color=accent or C["teal"], bold=True, min_size=12, align="center", anchor="middle", display=True)
            self.text(slide, x + 0.1, y + h - label_h - 0.02, w - 0.2, label_h, label, size=self.S["body"], color=C["muted"],
                      align="center", anchor="top", spacing=1.0)

        def chip(self, slide, x, y, w, label, fill, color):
            self.rect(slide, x, y, w, 0.34, fill, radius=0.17)
            self.text(slide, x + 0.08, y, w - 0.16, 0.34, label, size=self.S["body"], color=color, bold=True, align="center", anchor="middle")

        def chip_width(self, label):
            return min(4.2, 0.4 + sum(self.char_em(c) for c in str(label)) * self.S["body"] * 1.1 / 72.0)

        def finish(self):
            C = self.C
            total = len(self.records)
            footer = self.model["generated_by"]
            for number, record in enumerate(self.records, start=1):
                if record["archetype"] == "cover":
                    continue
                slide = record["slide"]
                if record["archetype"] != "closing":
                    self.wordmark(slide, 9.13, 0.12, 3.6, 0.3, light=record["top_dark"], align="end")
                self.text(slide, 0.6, 7.06, 9.0, 0.3, footer, size=self.S["body"], color=C["mint2"] if record["dark"] else C["muted"])
                self.text(slide, 10.73, 7.06, 2.0, 0.3, self.ltr(f"{number} / {total}"), size=self.S["body"],
                          color=C["mint2"] if record["dark"] else C["muted"], bold=True, align="end")
            return [record["archetype"] for record in self.records]

    def _deck_index_table(self, deck: "_Deck", model: dict, slide, rows: list, x: float, y: float, w: float, row_h: float = 0.36,
                          bottom: float = 6.98):
        C, S = deck.C, deck.S
        header = self._doc_index_header(model, deck.t)
        offsets, cursor = [], x
        for _key, _label, width in header:
            offsets.append((cursor, w * width))
            cursor += w * width
        head_h = max(0.38, max(deck.height([label], cw - 0.12, S["body"], bold=True, spacing=1.0) for (_k, label, _w), (_cx, cw)
                                in zip(header, offsets)) + 0.06)
        row_h = max(0.3, min(row_h, (bottom - y - head_h - 0.04) / max(1, len(rows)) - 0.02))
        deck.rect(slide, x, y, w, head_h, C["teal"], radius=0.06)
        for (key, label, _width), (cx, cw) in zip(header, offsets):
            deck.text(slide, cx + 0.06, y, cw - 0.12, head_h, label, size=S["body"], color=C["white"], bold=True, anchor="middle", spacing=1.0,
                      align="center" if key in ("rank", "quality", "inspire_score") else "start")
        for i, row in enumerate(rows):
            top = y + head_h + 0.04 + i * (row_h + 0.02)
            if i % 2:
                deck.rect(slide, x, top, w, row_h, C["sand3"])
            for (key, _label, _width), (cx, cw) in zip(header, offsets):
                value = row[key]
                if key == "quality":
                    pill = min(cw - 0.1, deck.chip_width(value))
                    deck.rect(slide, cx + (cw - pill) / 2.0, top + 0.03, pill, row_h - 0.06, row["quality_color"], radius=(row_h - 0.06) / 2.0)
                    deck.text(slide, cx + (cw - pill) / 2.0 + 0.05, top, pill - 0.1, row_h, value, size=S["body"], color=C["white"], bold=True,
                              align="center", anchor="middle")
                    continue
                style = {"rank": (C["teal"], True, "center"), "id": (C["coral"], True, "start"), "title": (C["ink"], False, "start"),
                         "domain": (C["body"], False, "start"), "inspire_score": (C["teal"], True, "center")}[key]
                link = deck.url(row["uc"]) if key == "id" else None
                shown = deck.ltr(value) if key == "inspire_score" else value
                segments = [(shown, {"link": link})]
                if key == "title" and row["flag"]:
                    segments.insert(0, (f'{row["flag"]} \u00b7 ', {"color": C["note_ink"], "bold": True}))
                deck.text(slide, cx + 0.08, top, cw - 0.12, row_h, [segments], size=S["body"], color=style[0], bold=style[1],
                          align=style[2], anchor="middle")
        return y + head_h + 0.04 + len(rows) * (row_h + 0.02)

    def _deck_cover(self, deck: "_Deck", model: dict):
        C, S = deck.C, deck.S
        slide = deck.new("cover", dark=True, gradient=(C["teal_dark"], C["teal_mid"], 30))
        for x, y, size, alpha in ((8.9, 0.25, 4.2, 0.08), (10.3, 0.45, 2.7, 0.1), (8.2, 3.6, 2.3, 0.07), (10.7, 4.6, 2.5, 0.08)):
            deck.rect(slide, x, y, size, size, C["white"], oval=True, alpha=alpha)
        deck.rect(slide, 11.05, 1.25, 0.62, 0.62, C["gold"], oval=True, alpha=0.9)
        deck.wordmark(slide, 0.75, 0.45, 6.0, 0.42, light=True, size=S["heading"])
        deck.text(slide, 0.75, 1.12, 8.6, 0.32, deck.kicker(f'{deck.T("doc_catalog_kicker")} \u00b7 {deck.ltr(model["date"])}'), size=S["body"],
                  color=C["mint2"], bold=True)
        deck.text(slide, 0.75, 1.5, 8.3, 1.75, deck.T("pptx_main_title"), size=S["hero"], color=C["white"], bold=True, min_size=24, spacing=1.0,
                  display=True)
        deck.text(slide, 0.75, 3.35, 8.3, 0.34, deck.T("doc_prepared_for"), size=S["lead"], color=C["mint2"])
        deck.text(slide, 0.75, 3.67, 8.3, 0.72, model["business_name"], size=S["display"], color=C["gold"], bold=True, min_size=18, display=True)
        if model["mode"]:
            label = f'{deck.T("doc_quality_mode")}: {model["mode_label"]}'
            width = min(5.5, 0.6 + sum(deck.char_em(c) for c in label) * 13.0 / 72.0)
            deck.rect(slide, 0.75, 4.5, width, 0.42, C["gold"], radius=0.21)
            deck.text(slide, 0.85, 4.5, width - 0.2, 0.42, label, size=S["body"], color=C["gold_ink"], bold=True, align="center", anchor="middle")
        for i, (value, label) in enumerate(model["headline"][:3]):
            x = 0.75 + i * 2.85
            deck.rect(slide, x, 5.15, 2.65, 1.3, C["white"], radius=0.14, alpha=0.12)
            label_h = min(0.72, max(0.5, deck.height([label], 2.25, S["body"], bold=True, spacing=1.0) + 0.02))
            deck.text(slide, x + 0.22, 5.2, 2.25, 1.22 - label_h, str(value), size=S["display"], color=C["white"], bold=True, min_size=18,
                      anchor="middle", display=True)
            deck.text(slide, x + 0.22, 6.42 - label_h, 2.25, label_h, label, size=S["body"], color=C["mint2"], bold=True, spacing=1.0)
        deck.text(slide, 0.75, 6.75, 11.8, 0.32, model["generated_by"], size=S["body"], color=C["mint2"])

    def _deck_contents(self, deck: "_Deck", model: dict):
        C, S = deck.C, deck.S
        slide = deck.new("contents")
        deck.header(slide, model["business_name"], deck.T("doc_contents"))
        accents = (C["teal"], C["coral"], C["gold"], C["teal_mid"], C["green"], C["amber"])
        outline = self._doc_outline(model, deck.t)
        rows = -(-len(outline) // 2)
        height = min(1.2, (5.1 - 0.16 * (rows - 1)) / max(1, rows))
        for i, (_key, title, subtitle) in enumerate(outline):
            col, row = i % 2, i // 2
            x, y = 0.6 + col * 6.19, 1.8 + row * (height + 0.16)
            deck.rect(slide, x, y, 5.94, height, C["sand"], radius=0.14)
            deck.rect(slide, x + 0.2, y + (height - 0.66) / 2.0, 0.66, 0.66, accents[i % len(accents)], radius=0.14)
            deck.text(slide, x + 0.2, y + (height - 0.66) / 2.0, 0.66, 0.66, f"{i + 1:02d}", size=S["heading"], color=C["white"], bold=True,
                      align="center", anchor="middle", display=True)
            title_h = min(height * 0.55, deck.height([title], 4.7, S["heading"], bold=True) + 0.02)
            deck.text(slide, x + 1.05, y + 0.12, 4.7, title_h, title, size=S["heading"], color=C["ink"], bold=True, min_size=12, display=True)
            deck.text(slide, x + 1.05, y + 0.16 + title_h, 4.7, height - title_h - 0.24, subtitle, size=S["body"], color=C["muted"], spacing=1.0)

    def _deck_summary(self, deck: "_Deck", model: dict) -> list:
        C, S = deck.C, deck.S
        slide = deck.new("summary")
        deck.header(slide, deck.T("doc_exec_kicker"), deck.T("pdf_exec_summary"))
        deck.rect(slide, 0.6, 1.8, 8.35, 4.95, C["sand"], radius=0.14)
        deck.rect(slide, 0.6, 1.8, 0.1, 4.95, C["teal"])
        size, first, rest = deck.split(model["executive"], 7.75, 4.55, 15, 12)
        deck.text(slide, 0.95, 2.0, 7.75, 4.55, first, size=size, color=C["body"], spacing=1.1, fit=True)
        for i, (value, label) in enumerate(model["headline"]):
            deck.tile(slide, 9.2, 1.8 + i * 1.26, 3.53, 1.13, value, label, value_size=S["title"])
        return rest

    def _deck_summary_more(self, deck: "_Deck", model: dict, rest: list):
        C, S = deck.C, deck.S
        slide = deck.new("summary_more")
        deck.header(slide, deck.T("doc_exec_kicker"), f'{deck.T("pdf_exec_summary")} ({deck.T("doc_continued")})')
        half = (len(rest) + 1) // 2
        for col, chunk in enumerate((rest[:half], rest[half:])):
            if chunk:
                x = 0.6 + col * 6.19
                deck.rect(slide, x, 1.8, 5.94, 4.95, C["white"], radius=0.14, line=(C["line"], 1.0))
                deck.rect(slide, x, 1.8, 5.94, 0.09, C["teal"] if col == 0 else C["coral"])
                deck.text(slide, x + 0.3, 2.1, 5.34, 4.45, chunk, size=S["lead"], color=C["body"], spacing=1.1)

    def _deck_guide(self, deck: "_Deck", model: dict):
        C, S = deck.C, deck.S
        slide = deck.new("guide", background=C["sand"])
        deck.header(slide, deck.T("doc_guide_kicker"), deck.T("doc_how_to_read"))
        guide_lead = deck.T(model.get("guide_lead_key", "doc_guide_lead"))
        lead_h = min(0.62, deck.height([guide_lead], 12.13, S["body"], spacing=1.0) + 0.04)
        deck.text(slide, 0.6, 1.62, 12.13, lead_h, guide_lead, size=S["body"], color=C["body"], spacing=1.0)
        top = max(2.0, 1.62 + lead_h + 0.08)
        cards = ((0.6, deck.T("doc_inspire_score_title"), deck.T("doc_inspire_score_body")),
                 (6.79, deck.T("doc_quality_tiers_title"), deck.T("doc_quality_tiers_body")))
        body_h = max(0.95, max(deck.height([body], 5.44, S["body"], spacing=1.0) + 0.04 for _x, _title, body in cards))
        card_h = min(6.8 - top, max(2.05, body_h + 1.1))
        body_h = min(body_h, card_h - 1.1)
        for x, title, body in cards:
            deck.rect(slide, x, top, 5.94, card_h, C["white"], radius=0.14, line=(C["line"], 1.0))
            deck.text(slide, x + 0.25, top + 0.15, 5.44, 0.36, title, size=S["heading"], color=C["ink"], bold=True, display=True)
            deck.text(slide, x + 0.25, top + 0.55, 5.44, body_h, body, size=S["body"], color=C["body"], spacing=1.0)
        marks = top + 0.63 + body_h
        for i, color in enumerate((C["coral"], C["amber"], C["green"], C["teal_dark"])):
            deck.rect(slide, 0.85 + i * 1.36, marks, 1.36, 0.14, color)
        for x, mark, align in ((0.85, "0", "start"), (3.27, "5", "center"), (5.69, "10", "end")):
            deck.text(slide, x, marks + 0.18, 0.6, 0.26, mark, size=S["body"], color=C["muted"], align=align)
        x = 7.04
        for label in self._DOC_QUALITY_ORDER:
            text = f'{self._translate_display_value(label, deck.t)} {model["quality_counts"][label]}'
            width = deck.chip_width(text)
            if x + width > 12.5:
                break
            deck.chip(slide, x, marks + 0.02, width, text, self._doc_quality_color(label), C["white"])
            x += width + 0.08
        low = top + card_h + 0.15
        if not self._deck_guide_modes(deck, model, slide, low, probe=True):
            slide = deck.new("guide_more", background=C["sand"])
            deck.header(slide, deck.T("doc_guide_kicker"), deck.T("doc_how_to_read"))
            low = 1.8
        self._deck_guide_modes(deck, model, slide, low)

    def _deck_guide_modes(self, deck: "_Deck", model: dict, slide, low: float, probe: bool = False) -> bool:
        C, S = deck.C, deck.S
        blocks = [[(deck.T("doc_elite_score_body"), {})],
                  [(deck.T("doc_coverage_title") + ": ", {"bold": True, "color": C["coral"]}), (deck.T("doc_coverage_body"), {})]]
        block_needs = [deck.height(["".join(seg for seg, _ in block)], 5.44, S["body"], spacing=1.0) + 0.04 for block in blocks]
        rows = []
        for mode, body_key in (("Good", "doc_mode_good_body"), ("High", "doc_mode_high_body"), ("Ultra High", "doc_mode_ultra_body")):
            on = mode == model["mode"]
            name = deck.T(self._DOC_MODE_KEYS[mode]) + (f' \u00b7 {deck.T("doc_this_catalog")}' if on else "")
            need = deck.height([f"{name}: {deck.T(body_key)}"], 5.36, S["body"], spacing=1.0) + 0.06
            rows.append((on, name, deck.T(body_key), need))
        if probe:
            room = 6.72 - low - 0.55
            return sum(block_needs) + 0.14 <= room + 1e-6 and sum(row[3] for row in rows) + 0.16 <= room + 1e-6
        deck.rect(slide, 6.79, low, 5.94, 6.8 - low, C["white"], radius=0.14, line=(C["line"], 1.0))
        deck.text(slide, 7.04, low + 0.15, 5.44, 0.36, deck.T("doc_elite_score_title"), size=S["heading"], color=C["ink"], bold=True, display=True)
        for block, (y, h) in zip(blocks, deck.stack(block_needs, low + 0.55, 6.72, 0.14)):
            deck.text(slide, 7.04, y, 5.44, h, [block], size=S["body"], color=C["body"], spacing=1.0)
        deck.rect(slide, 0.6, low, 5.94, 6.8 - low, C["sand2"], radius=0.14)
        deck.text(slide, 0.85, low + 0.12, 5.44, 0.34, deck.T("doc_modes_title"), size=S["heading"], color=C["ink"], bold=True, display=True)
        for (on, name, body, _need), (y, height) in zip(rows, deck.stack([row[3] for row in rows], low + 0.54, 6.72, 0.08)):
            if on:
                deck.rect(slide, 0.75, y - 0.04, 5.64, height + 0.06, C["white"], radius=0.1, line=(C["gold"], 1.0))
            deck.text(slide, 0.9, y, 5.36, height, [[(name + ": ", {"bold": True, "color": C["teal"]}), (body, {})]],
                      size=S["body"], color=C["body"], spacing=1.0)
        return True

    def _deck_portfolio(self, deck: "_Deck", model: dict):
        C, S = deck.C, deck.S
        slide = deck.new("portfolio")
        deck.header(slide, deck.T("doc_portfolio_kicker"), deck.T("doc_portfolio_glance"))
        deck.rect(slide, 0.6, 1.8, 7.0, 4.95, C["white"], radius=0.14, line=(C["line"], 1.0))
        deck.text(slide, 0.85, 1.95, 6.5, 0.36, deck.T("doc_per_domain"), size=S["heading"], color=C["ink"], bold=True, display=True)
        domains = model["domains"]
        limit = 11
        shown = domains[:limit if len(domains) <= limit else limit - 1]
        peak = max((len(d["use_cases"]) for d in domains), default=1) or 1
        step = min(0.4, 4.1 / max(1, len(shown) + (1 if len(domains) > len(shown) else 0)))
        for i, d in enumerate(shown):
            y = 2.45 + i * step
            deck.text(slide, 0.85, y, 2.45, step, d["name"], size=S["body"], color=C["ink"], anchor="middle")
            deck.rect(slide, 3.4, y + step / 2.0 - 0.075, 3.25, 0.15, C["sand2"], radius=0.07)
            deck.rect(slide, 3.4, y + step / 2.0 - 0.075, max(0.08, 3.25 * len(d["use_cases"]) / peak), 0.15, C["teal"], radius=0.07)
            deck.text(slide, 6.75, y, 0.7, step, str(len(d["use_cases"])), size=S["body"], color=C["teal"], bold=True, anchor="middle", align="end")
        if len(domains) > len(shown):
            deck.text(slide, 0.85, 2.45 + len(shown) * step, 6.5, step, f"+{len(domains) - len(shown)} {deck.T('doc_domains')}",
                      size=S["body"], color=C["muted"], anchor="middle")
        deck.rect(slide, 7.85, 1.8, 4.88, 4.95, C["sand"], radius=0.14)
        deck.text(slide, 8.1, 1.95, 4.4, 0.36, deck.T("doc_quality_distribution"), size=S["heading"], color=C["ink"], bold=True, display=True)
        x = 8.1
        for (label, _n), share in zip(model["quality_rows"], self._doc_shares([n for _label, n in model["quality_rows"]])):
            width = 4.38 * share / 100.0
            deck.rect(slide, x, 2.5, width, 0.3, self._doc_quality_color(label))
            x += width
        for i, ((label, n), name) in enumerate(list(zip(model["quality_rows"], model["quality_names"]))[:8]):
            y = 3.0 + i * 0.4
            deck.rect(slide, 8.12, y + 0.11, 0.16, 0.16, self._doc_quality_color(label), oval=True)
            deck.text(slide, 8.4, y, 3.3, 0.38, name, size=S["body"], color=C["ink"], anchor="middle")
            deck.text(slide, 11.75, y, 0.75, 0.38, str(n), size=S["body"], color=C["teal"], bold=True, anchor="middle", align="end")

    def _deck_top_index(self, deck: "_Deck", model: dict):
        C, S = deck.C, deck.S
        slide = deck.new("top_index")
        deck.header(slide, deck.T("doc_portfolio_kicker"), deck.T("doc_top_use_cases"))
        if model["total"] > len(model["deck"]):
            note = deck.T("doc_deck_scope", count=len(model["deck"]), total=model["total"])
        else:
            note = deck.T("doc_deck_scope_all", total=model["total"])
        top = deck.lead(slide, note, italic=True)
        self._deck_index_table(deck, model, slide, self._doc_index_rows(model["top"], model, deck.t), 0.6, top, 12.13)

    def _deck_domain(self, deck: "_Deck", model: dict, domain: dict):
        C, S = deck.C, deck.S
        number = f'{domain["index"]:02d}'
        slide = deck.new("domain", top_dark=True)
        deck.rect(slide, 0, 0, deck.W, 2.6, C["teal"])
        deck.rect(slide, 10.9, 0.15, 2.3, 2.3, C["white"], oval=True, alpha=0.08)
        deck.text(slide, 0.6, 0.45, 2.4, 1.0, number, size=S["numeral"], color=C["mint2"], bold=True, min_size=30, display=True)
        deck.text(slide, 0.6, 1.4, 7.0, 0.3, deck.kicker(f'{deck.T("doc_domain_label")} {number}'), size=S["body"], color=C["coral_soft"], bold=True)
        deck.text(slide, 0.6, 1.72, 7.4, 0.7, domain["name"], size=S["title"], color=C["white"], bold=True, min_size=16, display=True)
        for i, (value, label) in enumerate(domain["stats"][:4]):
            deck.tile(slide, 7.6 + (i % 2) * 2.6, 0.5 + (i // 2) * 1.04, 2.5, 0.96, value, label, fill=C["white"])
        deck.text(slide, 0.6, 2.82, 7.3, 0.32, deck.T("doc_domain_overview"), size=S["heading"], color=C["teal"], bold=True, display=True)
        deck.text(slide, 0.6, 3.2, 7.3, 1.45, domain["overview"], size=S["body"], color=C["body"], spacing=1.06)
        deck.text(slide, 8.13, 2.82, 4.6, 0.32, deck.T("doc_what_it_covers"), size=S["heading"], color=C["teal"], bold=True, display=True)
        x, y = 8.13, 3.22
        for name, count in domain["subdomains"][:12]:
            label = f"{name} {count}"
            width = deck.chip_width(label)
            if x + width > 12.73:
                x, y = 8.13, y + 0.42
            if y > 4.4:
                break
            deck.chip(slide, x, y, width, label, C["sand2"], C["teal"])
            x += width + 0.08
        deck.text(slide, 0.6, 4.78, 12.13, 0.32, deck.T("doc_top_opportunities"), size=S["heading"], color=C["ink"], bold=True, display=True)
        self._deck_index_table(deck, model, slide, self._doc_index_rows(domain["top"], model, deck.t), 0.6, 5.16, 12.13)

    def _deck_card_head(self, deck: "_Deck", model: dict, slide, view: dict):
        C, S = deck.C, deck.S
        accent = {"elite": C["gold"], "coverage": C["coral"]}.get(view["kind"], C["teal"])
        deck.rect(slide, 0.6, 0.5, 0.82, 0.82, accent, radius=0.14)
        deck.text(slide, 0.6, 0.5, 0.82, 0.82, str(view["rank"]), size=S["heading"], color=C["gold_ink"] if view["kind"] == "elite" else C["white"],
                  bold=True, align="center", anchor="middle", display=True)
        kicker = " \u00b7 ".join(part for part in (view["id"], view["domain"], view["subdomain"]) if part)
        deck.text(slide, 1.6, 0.5, 8.0, 0.28, deck.kicker(kicker), size=S["body"], color=C["coral"], bold=True)
        deck.text(slide, 1.6, 0.78, 11.13, 0.6, view["title"], size=S["card_title"], color=C["ink"], bold=True, min_size=16, display=True, spacing=1.0)
        if view["name"]:
            deck.text(slide, 1.6, 1.36, 11.13, 0.28, view["name"], size=S["body"], color=C["muted"])
        deck.rect(slide, 0.6, 1.7, 12.13, 0.05, C["teal"])

    def _deck_note_band(self, deck: "_Deck", slide, top: float, accent, paragraphs: list, plain: str) -> float:
        C, S = deck.C, deck.S
        band_h = min(1.2, max(0.42, deck.height([plain], 11.8, S["body"], spacing=1.0) + 0.14))
        deck.rect(slide, 0.6, top, 12.13, band_h, C["note"], radius=0.08, line=(C["note_line"], 0.75))
        deck.rect(slide, 0.6, top, 0.07, band_h, accent)
        deck.text(slide, 0.8, top, 11.8, band_h, paragraphs, size=S["body"], color=C["note_ink"], anchor="middle")
        return top + band_h + 0.1

    def _deck_use_case(self, deck: "_Deck", model: dict, view: dict):
        C, S = deck.C, deck.S
        slide = deck.new("uc_card")
        self._deck_card_head(deck, model, slide, view)
        metrics = view["metrics"]
        width = (12.13 - 0.1 * (len(metrics) - 1)) / max(1, len(metrics))
        value_need = max(deck.height([value], width - 0.2, S["heading"], bold=True, spacing=1.0) for _key, _label, value in metrics)
        label_need = max(deck.height([label], width - 0.2, S["body"], spacing=1.0) for _key, label, _value in metrics)
        tile_h = min(1.6, max(1.1, value_need + label_need + 0.22))
        for i, (_key, label, value) in enumerate(metrics):
            deck.tile(slide, 0.6 + i * (width + 0.1), 1.84, width, tile_h, value, label)
        top = 1.84 + tile_h + 0.1
        if view["star"]:
            top = self._deck_note_band(deck, slide, top, C["gold"], [[(view["star"], {"bold": True, "color": C["note_dark"]})]],
                                       view["star"])
        if view["coverage"]:
            label, reason_label, reason = view["coverage"]
            top = self._deck_note_band(deck, slide, top, C["coral"],
                                       [[(f"{label}. ", {"bold": True, "color": C["note_dark"]}), (f"{reason_label}: {reason}", {})]],
                                       f"{label}. {reason_label}: {reason}")
        bottom = 6.88
        if view["notice"]:
            deck.text(slide, 0.6, 6.62, 12.13, 0.3, view["notice"], size=S["body"], color=C["muted"], italic=True)
            bottom = 6.58
        room = bottom - top
        needs = [max(0.5, 0.3 + deck.height([value], 7.1, S["body"], spacing=1.04)) for _key, _label, value in view["narrative"]]
        for (_key, label, value), (y, h) in zip(view["narrative"], deck.stack(needs, top, bottom, 0.12)):
            deck.text(slide, 0.6, y, 7.3, 0.28, deck.kicker(label), size=S["body"], color=C["teal"], bold=True)
            deck.text(slide, 0.6, y + 0.3, 7.3, max(0.25, h - 0.3), value, size=S["body"], color=C["body"], spacing=1.04)
        url = deck.url(view["uc"])
        deck.rect(slide, 8.15, top, 4.58, room, C["sand"], radius=0.14)
        facts_room = room - (0.62 if url else 0.12) - 0.15
        fact_needs = [deck.height([f"{label}: {value}"], 4.2, S["body"], bold=False, spacing=1.0) + 0.1 for _key, label, value in view["facts"]]
        for (_key, label, value), (fy, fh) in zip(view["facts"], deck.stack(fact_needs, top + 0.15, top + 0.15 + facts_room)):
            deck.text(slide, 8.35, fy, 4.2, fh, [[(f"{label}: ", {"bold": True, "color": C["teal"]}), (value, {})]],
                      size=S["body"], color=C["ink"], spacing=1.0)
        if url:
            button = deck.rect(slide, 8.35, top + room - 0.55, 4.18, 0.42, C["night"], radius=0.1)
            button.click_action.hyperlink.address = url
            deck.text(slide, 8.45, top + room - 0.55, 3.98, 0.42, f'{deck.T("doc_open_notebook")} \u2197', size=S["body"],
                      color=C["mint"], bold=True, align="center", anchor="middle", link=url)

    def _deck_use_case_value(self, deck: "_Deck", model: dict, view: dict):
        C, S = deck.C, deck.S
        slide = deck.new("uc_value")
        self._deck_card_head(deck, model, slide, view)
        left = [fact for fact in view["value_facts"] if fact[0] in self._DOC_VALUE_LEFT]
        right = [fact for fact in view["value_facts"] if fact[0] not in self._DOC_VALUE_LEFT]
        for x, entries in ((0.6, left), (6.75, right)):
            needs = [0.42 + deck.height([f"\u2022 {item}" for item in items], 5.6, S["body"], spacing=1.04) + 0.12
                     for _key, _label, items in entries]
            for (_key, label, items), (y, h) in zip(entries, deck.stack(needs, 1.9, 6.88, 0.14)):
                deck.rect(slide, x, y, 5.98, h, C["sand"], radius=0.12)
                deck.text(slide, x + 0.18, y + 0.08, 5.6, 0.28, deck.kicker(label), size=S["body"], color=C["teal"], bold=True)
                deck.text(slide, x + 0.18, y + 0.4, 5.6, max(0.25, h - 0.46), [f"\u2022 {item}" for item in items],
                          size=S["body"], color=C["body"], spacing=1.04)

    def _deck_foundations(self, deck: "_Deck", model: dict):
        C, S = deck.C, deck.S
        slide = deck.new("foundations")
        deck.header(slide, deck.T("doc_next_steps"), deck.T("doc_foundations_title"))
        top = deck.lead(slide, deck.T("doc_foundations_lead"))
        rows = model["foundations"][:self._DOC_PPTX_FOUNDATION_ROWS]
        accents = (C["teal"], C["coral"], C["gold"], C["teal_mid"], C["green"], C["amber"])
        needs = [max(0.62, deck.height([row["tables"]], 6.3, S["body"], bold=True, spacing=1.0)
                     + (deck.height([row["joins"]], 6.3, S["body"], spacing=1.0) if row["joins"] else 0.0) + 0.2) for row in rows]
        for i, (row, (y, h)) in enumerate(zip(rows, deck.stack(needs, top + 0.1, 6.9, 0.1))):
            accent = accents[i % len(accents)]
            deck.rect(slide, 0.6, y, 12.13, h, C["sand"], radius=0.12)
            deck.rect(slide, 0.6, y, 0.09, h, accent)
            deck.text(slide, 0.85, y, 0.75, h, row["id"], size=S["heading"], color=accent, bold=True, anchor="middle", display=True)
            deck.text(slide, 1.65, y + 0.06, 6.3, h - 0.12, [[(row["tables"], {"bold": True, "color": C["ink"]})]]
                      + ([[(row["joins"], {"color": C["muted"]})]] if row["joins"] else []), size=S["body"], color=C["ink"],
                      anchor="middle", spacing=1.0)
            deck.text(slide, 8.2, y + 0.06, 4.4, h - 0.12, [[(f'{row["count"]} ', {"bold": True, "color": C["teal"]}),
                                                           (deck.T("doc_foundation_unlocks"), {"color": C["muted"]})],
                                                          [(row["use_cases"], {})]], size=S["body"], color=C["body"], anchor="middle", spacing=1.0)

    def _deck_index(self, deck: "_Deck", model: dict, chunk: list, page: int, pages: int):
        C, S = deck.C, deck.S
        suffix = f" ({page}/{pages})" if pages > 1 else ""
        slide = deck.new("index")
        deck.header(slide, deck.T("doc_appendix"), deck.T("doc_use_case_index") + suffix)
        top = deck.lead(slide, deck.T("doc_index_lead", count=model["total"]))
        self._deck_index_table(deck, model, slide, self._doc_index_rows(chunk, model, deck.t), 0.6, top, 12.13)

    def _deck_methodology(self, deck: "_Deck", model: dict):
        C, S = deck.C, deck.S
        slide = deck.new("methodology", background=C["sand"])
        deck.header(slide, deck.T("doc_appendix"), deck.T("doc_methodology"))
        deck.rect(slide, 0.6, 1.8, 12.13, 5.05, C["white"], radius=0.14, line=(C["line"], 1.0))
        needs = [max(0.5, deck.height([text], 11.1, S["lead"], spacing=1.04) + 0.1) for text in model["method_steps"]]
        gap = max(0.06, min(0.45, (4.75 - sum(needs)) / max(1, len(needs) - 1)))
        for i, (text, (y, h)) in enumerate(zip(model["method_steps"], deck.stack(needs, 2.0, 6.75, gap))):
            deck.rect(slide, 0.8, y + 0.02, 0.44, 0.44, C["teal"], oval=True)
            deck.text(slide, 0.8, y + 0.02, 0.44, 0.44, str(i + 1), size=S["body"], color=C["white"], bold=True, align="center", anchor="middle")
            deck.text(slide, 1.45, y, 11.1, h, text, size=S["lead"], color=C["body"], spacing=1.04)
        self._deck_glossary(deck, model)

    def _deck_glossary(self, deck: "_Deck", model: dict):
        C, S = deck.C, deck.S
        slide = deck.new("glossary")
        deck.header(slide, deck.T("doc_appendix"), deck.T("doc_glossary"))
        entries = model["glossary"]
        needs = [deck.height([f"{term}: {body}"], 5.49, S["lead"], spacing=1.04) + 0.04 for term, body in entries]
        half, running = len(entries), 0.0
        for i, need in enumerate(needs):
            running += need
            if running >= sum(needs) / 2.0:
                half = i + 1
                break
        for x, part, part_needs in ((0.6, entries[:half], needs[:half]), (6.76, entries[half:], needs[half:])):
            deck.rect(slide, x, 1.8, 5.97, 5.05, C["sand"], radius=0.14)
            gap = max(0.04, min(0.3, (4.75 - sum(part_needs)) / max(1, len(part_needs) - 1)))
            for (term, body), (y, h) in zip(part, deck.stack(part_needs, 2.0, 6.75, gap)):
                deck.text(slide, x + 0.24, y, 5.49, h, [[(f"{term}: ", {"bold": True, "color": C["teal"]}), (body, {})]],
                          size=S["lead"], color=C["body"], spacing=1.04)

    def _deck_roadmap(self, deck: "_Deck", model: dict):
        C, S = deck.C, deck.S
        slide = deck.new("roadmap")
        deck.header(slide, deck.T("doc_next_steps"), deck.T("doc_roadmap"))
        accents = (C["teal"], C["coral"], C["gold"], C["teal_mid"], C["green"])
        width = 2.12
        title_h = max(0.4, max(deck.height([title], width - 0.4, S["lead"], bold=True, spacing=1.0) for title, _body in model["roadmap"]) + 0.04)
        title_h = min(title_h, 1.2)
        body_need = max(deck.height([body], width - 0.4, S["body"], spacing=1.0) for _title, body in model["roadmap"])
        vertical = body_need > 5.35 - 2.82 - title_h + 1e-6
        if vertical:
            needs = [max(0.56, deck.height([body], 7.5, S["body"], spacing=1.0), deck.height([title], 3.1, S["lead"], bold=True, spacing=1.0)) + 0.14
                     for title, body in model["roadmap"]]
            for i, ((title, body), (y, h)) in enumerate(zip(model["roadmap"], deck.stack(needs, 1.85, 5.45, 0.08))):
                accent = accents[i % len(accents)]
                deck.rect(slide, 0.6, y, 12.13, h, C["sand"], radius=0.12)
                deck.rect(slide, 0.6, y, 0.09, h, accent)
                deck.rect(slide, 0.85, y + (h - 0.42) / 2.0, 0.42, 0.42, accent, oval=True)
                deck.text(slide, 0.85, y + (h - 0.42) / 2.0, 0.42, 0.42, str(i + 1), size=S["body"], color=C["white"], bold=True, align="center",
                          anchor="middle")
                deck.text(slide, 1.45, y + 0.07, 3.1, h - 0.14, title, size=S["lead"], color=C["ink"], bold=True, min_size=12, spacing=1.0,
                          anchor="middle", display=True)
                deck.text(slide, 4.85, y + 0.07, 7.6, h - 0.14, body, size=S["body"], color=C["body"], spacing=1.0, anchor="middle")
        for i, (title, body) in enumerate([] if vertical else model["roadmap"]):
            accent = accents[i % len(accents)]
            x = 0.6 + i * (width + 0.383)
            deck.rect(slide, x, 1.85, width, 3.6, C["sand"], radius=0.14)
            deck.rect(slide, x, 1.85, width, 0.09, accent)
            deck.rect(slide, x + 0.2, 2.1, 0.5, 0.5, accent, oval=True)
            deck.text(slide, x + 0.2, 2.1, 0.5, 0.5, str(i + 1), size=S["heading"], color=C["white"], bold=True, align="center", anchor="middle")
            deck.text(slide, x + 0.2, 2.74, width - 0.4, title_h, title, size=S["lead"], color=C["ink"], bold=True, min_size=12, spacing=1.0,
                      display=True)
            deck.text(slide, x + 0.2, 2.82 + title_h, width - 0.4, 5.35 - 2.82 - title_h, body, size=S["body"], color=C["body"], spacing=1.0)
            if i < len(model["roadmap"]) - 1:
                deck.arrow(slide, x + width + 0.06, 3.5, 0.26, 0.3, C["muted"])
        deck.rect(slide, 0.6, 5.6, 12.13, 1.15, C["ice"], radius=0.12)
        deck.rect(slide, 0.6, 5.6, 0.1, 1.15, C["teal"])
        deck.text(slide, 0.95, 5.68, 11.55, 1.0, deck.T("doc_closing_body"), size=S["lead"], color=C["ink"], anchor="middle", spacing=1.06)

    def _deck_disclaimer(self, deck: "_Deck", model: dict):
        C, S = deck.C, deck.S
        slide = deck.new("disclaimer", background=C["sand"])
        title, paragraphs = self._doc_disclaimer(model, deck.t)
        deck.header(slide, self._DOC_BRAND["wordmark"], title)
        deck.rect(slide, 0.6, 1.8, 12.13, 4.95, C["white"], radius=0.14, line=(C["line"], 1.0))
        needs = [max(0.66, deck.height([text], 10.9, S["lead"], spacing=1.06) + 0.16) for text in paragraphs]
        for i, (text, (y, h)) in enumerate(zip(paragraphs, deck.stack(needs, 1.9, 6.65, 0.1))):
            deck.rect(slide, 0.85, y + (h - 0.5) / 2.0, 0.5, 0.5, C["teal"], radius=0.1)
            deck.text(slide, 0.85, y + (h - 0.5) / 2.0, 0.5, 0.5, str(i + 1), size=S["lead"], color=C["white"], bold=True, align="center",
                      anchor="middle")
            deck.text(slide, 1.6, y, 10.9, h, text, size=S["lead"], color=C["body2"], anchor="middle", spacing=1.06)

    def _deck_closing(self, deck: "_Deck", model: dict):
        C, S = deck.C, deck.S
        slide = deck.new("closing", dark=True, gradient=(C["teal_dark"], C["teal_mid"], 30))
        for x, y, size, alpha in ((0.15, 3.9, 3.4, 0.08), (0.9, 4.75, 2.6, 0.08), (10.0, 0.15, 3.1, 0.07)):
            deck.rect(slide, x, y, size, size, C["white"], oval=True, alpha=alpha)
        deck.text(slide, 1.5, 1.6, 10.33, 0.34, deck.kicker(deck.T("doc_next_steps")), size=S["body"], color=C["mint2"], bold=True, align="center")
        deck.text(slide, 1.5, 2.05, 10.33, 1.2, deck.T("doc_closing_title"), size=S["hero"], color=C["white"], bold=True, align="center", min_size=22,
                  display=True)
        deck.text(slide, 1.5, 3.3, 10.33, 0.7, model["business_name"], size=S["title"], color=C["gold"], bold=True, align="center", min_size=16,
                  display=True)
        deck.text(slide, 1.5, 4.15, 10.33, 0.4, model["closing_summary"], size=S["lead"], color=C["mint"], align="center")
        deck.wordmark(slide, 4.67, 4.95, 4.0, 0.42, light=True, size=S["heading"], align="start")

    def _build_presentation_deck(self, model: dict, t: dict):
        deck = self._Deck(self, model, t)
        ranked = model["ranked"]
        top_ids = {id(uc) for uc in model["deck"]}
        n_index = max(1, -(-len(ranked) // self._DOC_PPTX_INDEX_ROWS))
        per_index = max(1, -(-len(ranked) // n_index))
        self._deck_cover(deck, model)
        self._deck_contents(deck, model)
        rest = self._deck_summary(deck, model)
        if rest:
            self._deck_summary_more(deck, model, rest)
        self._deck_guide(deck, model)
        self._deck_portfolio(deck, model)
        self._deck_top_index(deck, model)
        for domain in model["domains"]:
            self._deck_domain(deck, model, domain)
            for uc in domain["use_cases"]:
                if id(uc) in top_ids:
                    view = self._use_case_view(uc, model, t)
                    self._deck_use_case(deck, model, view)
                    if view["value_facts"]:
                        self._deck_use_case_value(deck, model, view)
        for page in range(n_index):
            self._deck_index(deck, model, ranked[page * per_index:(page + 1) * per_index], page + 1, n_index)
        self._deck_methodology(deck, model)
        if model["foundations"]:
            self._deck_foundations(deck, model)
        self._deck_roadmap(deck, model)
        self._deck_disclaimer(deck, model)
        self._deck_closing(deck, model)
        deck.theme_links()
        archetypes = deck.finish()
        repeats = [i for i in range(1, len(archetypes)) if archetypes[i] == archetypes[i - 1] and archetypes[i] not in self._DOC_PPTX_SERIES]
        if repeats:
            self.logger.warning(f"[DOCS-PPTX layout-repeat] consecutive slides share a layout at {repeats}: {archetypes}")
        return deck.prs, archetypes

    def generate_presentation_pptx(self, language: str, lang_abbr: str, translations: dict, summary_dict: dict, grouped_data: dict, transliterated_name: str):
        self.logger.info(f"--- Starting PPTX Presentation Generation for {language} ---")
        t = translations or {}
        try:
            if not grouped_data:
                self.logger.warning(f"No use cases provided to generate_presentation_pptx for {language}. Skipping.")
                return
            self._require_python_package("pptx", "python-pptx", f"Presentation ({language})")
            try:
                import pptx
            except ImportError as error:
                raise RuntimeError(f"python-pptx import failed: {get_clean_error_message(error)}") from error
            model = self._document_model(language, t, summary_dict, grouped_data, transliterated_name)
            prs, archetypes = self._build_presentation_deck(model, t)
            pptx_workspace_path = os.path.join(self.docs_output_dir, f"{self.business_name}-dbx_inspire_{lang_abbr}.pptx")
            local_pptx_path = None
            try:
                with tempfile.NamedTemporaryFile(delete=False, suffix=".pptx") as tmp_file:
                    local_pptx_path = tmp_file.name
                prs.save(local_pptx_path)
                with open(local_pptx_path, "rb") as handle:
                    pptx_data = handle.read()
                if not pptx_data:
                    raise ValueError("Generated PPTX file is empty.")
                self._import_workspace_file(pptx_workspace_path, base64.b64encode(pptx_data).decode(), workspace.ImportFormat.AUTO)
                abs_path = self.w_client.workspace.get_status(pptx_workspace_path).path
                self.logger.info(f"[DOCS-PPTX] {language}: {len(archetypes)} slide(s), rtl={model['rtl']}, layouts={archetypes} -> {abs_path}")
                log_print(f"Success! Presentation ({language}) generated: {abs_path}")
            finally:
                if local_pptx_path and os.path.exists(local_pptx_path):
                    os.remove(local_pptx_path)
        except Exception as e:
            self.logger.critical(f"An error occurred during PPTX generation for {language}: {get_clean_error_message(e)}")
            self._record_failed_step("Presentation", language, e)
            raise

    _PYTHON_PACKAGE_LOCK = threading.Lock()

    _PDF_RENDER_LOCK = threading.Lock()

    _PYTHON_PACKAGE_MINIMUMS = {
        "weasyprint": ("70.0", "versions before 70.0 release their font configuration twice and crash the Python process when a PDF is freed"),
    }

    @staticmethod
    def _version_key(text) -> tuple:
        return tuple((list(map(int, re.findall(r"\d+", str(text or ""))[:3])) + [0, 0, 0])[:3])

    @staticmethod
    def _installed_package_version(pip_name: str) -> str:
        from importlib import metadata
        try:
            return metadata.version(pip_name) or ""
        except Exception:
            return ""

    def _package_too_old(self, module_name: str, pip_name: str, module=None) -> str:
        minimum = self._PYTHON_PACKAGE_MINIMUMS.get(module_name)
        if not minimum:
            return ""
        version = self._installed_package_version(pip_name) or str(getattr(module, "__version__", "") or "")
        if version and self._version_key(version) < self._version_key(minimum[0]):
            return version
        return ""

    def _import_or_install_once(self, module_name: str, pip_name: str) -> tuple:
        import importlib
        minimum = self._PYTHON_PACKAGE_MINIMUMS.get(module_name)
        spec = f"{pip_name}>={minimum[0]}" if minimum else pip_name
        outdated = self._package_too_old(module_name, pip_name)
        if outdated and module_name in sys.modules:
            return False, (f"'{pip_name}' {outdated} is already loaded in this Python process and {minimum[1]}; "
                           f"run 'pip install \"{spec}\"' and restart Python")
        if not outdated:
            try:
                module = importlib.import_module(module_name)
            except ImportError:
                module = None
            except Exception as broken:
                return False, (f"'{module_name}' is installed but cannot be loaded ({get_clean_error_message(broken, max_chars=200)}); "
                               f"reinstalling will not help, the environment is missing a library it needs")
            if module is not None:
                loaded_old = self._package_too_old(module_name, pip_name, module)
                if not loaded_old:
                    return True, ""
                return False, (f"'{pip_name}' {loaded_old} is already loaded in this Python process and {minimum[1]}; "
                               f"run 'pip install \"{spec}\"' and restart Python")
        else:
            self.logger.warning(f"⚠️ '{pip_name}' {outdated} is installed but {minimum[1]}; upgrading to '{spec}'")
        self.logger.info(f"Installing Python package '{spec}' (attempted once per run)...")
        try:
            completed = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", spec],
                                       capture_output=True, text=True, timeout=600)
        except Exception as e:
            return False, f"'pip install {spec}' could not run: {get_clean_error_message(e, max_chars=200)}"
        if completed.returncode != 0:
            detail = (completed.stderr or completed.stdout or "").strip()[-300:]
            return False, f"'pip install {spec}' failed with exit code {completed.returncode}: {detail}"
        importlib.invalidate_caches()
        try:
            module = importlib.import_module(module_name)
        except Exception as e:
            return False, f"'{pip_name}' was installed but '{module_name}' still cannot be loaded: {get_clean_error_message(e, max_chars=200)}"
        still_old = self._package_too_old(module_name, pip_name, module)
        if still_old:
            return False, f"'pip install {spec}' finished but '{pip_name}' {still_old} is still installed and {minimum[1]}"
        return True, ""

    def _require_python_package(self, module_name: str, pip_name: str, purpose: str):
        with self._PYTHON_PACKAGE_LOCK:
            status = getattr(self, "_python_package_status", None)
            if not isinstance(status, dict):
                status = {}
                self._python_package_status = status
            if module_name not in status:
                status[module_name] = self._import_or_install_once(module_name, pip_name)
            ok, error = status[module_name]
        if not ok:
            raise RuntimeError(f"{purpose} cannot be generated: {error}")

    def _install_excel_dependencies(self, logger_instance) -> bool:
        self._require_python_package("xlsxwriter", "xlsxwriter", "Use Case Excel")
        return True

    def _save_excel(self, local_excel_path: str, workspace_path: str, logger_instance, language: str):
        """Uploads a locally generated Excel file to the Databricks workspace; raises on failure."""
        with open(local_excel_path, "rb") as f: excel_data = f.read()
        if not excel_data: raise ValueError("Generated Excel file is empty.")
        logger_instance.info(f"Uploading Excel to workspace path: {workspace_path}")
        excel_data_b64 = base64.b64encode(excel_data).decode()
        self._import_workspace_file(workspace_path, excel_data_b64, workspace.ImportFormat.AUTO)
        abs_path = self.w_client.workspace.get_status(workspace_path).path
        logger_instance.info(f"Success! Excel Catalog uploaded to: {abs_path}")
        log_print(f"Success! Excel Catalog ({language}) generated: {abs_path}")

    _EXPORT_SUB_SCORES = (
        ("Strategic Alignment", "Strategic Alignment"), ("Return on Investment", "ROI"), ("Reusability", "Reusability"),
        ("Time to Value", "Time to Value"), ("Data Availability", "Data Availability"), ("Data Accessibility", "Data Accessibility"),
        ("Architecture Fitness", "Architecture Fitness"), ("Team Skills", "Team Skills"), ("Domain Knowledge", "Domain Knowledge"),
        ("People Allocation", "People Allocation"), ("Budget Allocation", "Budget Allocation"), ("Time to Production", "Time to Production"),
    )
    _EXPORT_TAIL = (("Justification", "Priority Reasons"), ("Quality Summary", "Quality Reasons"), ("result_table", "Result Database"))

    @staticmethod
    def _has_elite_fields(uc: dict) -> bool:
        return bool(uc.get("elite_status") or uc.get("coverage_pick_reason") or uc.get("elite_score") is not None)

    def _export_columns(self, show_elite: bool, show_value: bool = False) -> list:
        label = lambda key: self._doc_text({}, key)
        labels = self._strategy_fact_labels(label, path_fact=False)
        def items(source, kind):
            def get(uc, _rank):
                values, _gap = self._use_case_items(uc, source, kind, {}, "English")
                return (", " if kind == "list" else "; ").join(values)
            return get
        def text(source, default="N/A"):
            def get(uc, _rank):
                value = self._doc_display(uc.get(source))
                return value or default
            return get
        def score(source):
            def get(uc, _rank):
                number = self._finite_score(uc.get(source))
                return self.NOT_SCORED_LABEL if number is None else number
            return get
        def assessment(source):
            def get(uc, _rank):
                values = uc.get("elite_assessment") if isinstance(uc.get("elite_assessment"), dict) else {}
                value = self._doc_display(inspire_plain_text(values.get(source), joiner=" \u2192 " if source == "join_path" else ", "))
                if source == "cadence":
                    value = self._doc_cadence_label(value)
                arrow = self._kpi_direction_arrow(values.get("kpi_direction")) if source == "kpi" else ""
                return f"{value} ({arrow})" if value and arrow else value
            return get
        def elite_score(uc, _rank):
            number = self._finite_score(uc.get("elite_score"))
            if number is not None:
                return number
            return self.NOT_SCORED_LABEL if (uc.get("elite_status") or uc.get("elite_assessment")) else ""
        columns = [("ID", text("No"), False), (label("doc_rank"), lambda _uc, rank: rank, True),
                   ("Business Domain", text("Business Domain"), False), ("Subdomain", text("Subdomain"), False),
                   ("Use Case", lambda uc, _rank: self._doc_use_case_title(uc, "English"), False),
                   ("Description", text("Name"), False), (label("quality"), lambda uc, _rank: self._canonical_quality_label(uc.get("Quality")) or self.NOT_SCORED_LABEL, False)]
        for key, source, _scale in self._USE_CASE_VIEW_SCORES:
            if source == "elite_score":
                if show_elite:
                    columns.append((label(key), elite_score, True))
                continue
            columns.append((label(key), score(source), True))
        columns += [(label(key), text(source), False) for key, source in self._USE_CASE_VIEW_TEXT]
        for key, source, kind in self._USE_CASE_VIEW_FACTS:
            header = labels.get(key) or label(key)
            if kind in self._USE_CASE_VIEW_ITEM_KINDS:
                if show_value:
                    columns.append((header, items(source, kind), False))
            elif kind == "tables":
                columns.append((header, lambda uc, _rank: ", ".join(self._doc_tables(uc)) or "N/A", False))
            elif kind == "technique":
                columns.append((header, lambda uc, _rank, source=source: self._doc_technique_label(uc.get(source), {}, "English")[0] or "N/A", False))
            else:
                columns.append((header, text(source), False))
            if key == "strategic_goals_alignment":
                columns += self._strategy_level_columns()
        if show_value:
            columns += [(labels.get(key) or label(key), items(source, kind), False) for key, source, kind in self._USE_CASE_VIEW_VALUE]
        columns.append(("Primary Table", text("Primary Table"), False))
        columns += [(header, score(source), True) for source, header in self._EXPORT_SUB_SCORES]
        columns += [(header, text(source), False) for source, header in self._EXPORT_TAIL]
        columns.append(("Has Genie Code", text("has_genie_code", "N"), False))
        columns.append((label("doc_best_of_best"), lambda uc, _rank: uc.get("best_of_best_rank") or "", True))
        columns.append(("Best of Best Reason", text("best_of_best_reason", ""), False))
        if show_elite:
            columns.append(("Elite Status", text("elite_status", ""), False))
            columns += [(label(key), assessment(source), False) for key, source in self._USE_CASE_VIEW_ELITE]
            columns.append((label("elite_red_team_note"), assessment("red_team"), False))
            columns.append(("Coverage Pick Reason", text("coverage_pick_reason", ""), False))
            columns.append(("Below Bar Reason", text("fallback_reason", ""), False))
        return self._unique_headers(columns)

    @staticmethod
    def _unique_headers(columns: list) -> list:
        """Customer level names and labels may repeat a built-in header; Excel tables need unique headers."""
        from collections import Counter
        seen, unique = Counter(), []
        for header, get, numeric in columns:
            seen[str(header).casefold()] += 1
            count = seen[str(header).casefold()]
            unique.append((header if count == 1 else f"{header} ({count})", get, numeric))
        return unique

    def _foundations_export_table(self) -> tuple:
        foundations = [f for f in getattr(self, "data_foundations", None) or [] if isinstance(f, dict)]
        if not foundations:
            return [], []
        label = lambda key: self._doc_text({}, key)
        headers = ["ID", label("doc_foundation_tables"), "Joins", label("doc_foundation_unlocks"), "Use Case IDs",
                   "Summed Inspire Score"]
        rows = []
        for foundation in foundations:
            joins = "; ".join(f'{j.get("left")} = {j.get("right")} ({j.get("source")})'
                              for j in foundation.get("joins") or [] if isinstance(j, dict) and j.get("left") and j.get("right"))
            rows.append([str(foundation.get("id") or ""), ", ".join(str(t) for t in foundation.get("tables") or []), joins,
                         int(foundation.get("use_case_count") or len(foundation.get("use_cases") or [])),
                         ", ".join(str(uc_id) for uc_id in foundation.get("use_cases") or []),
                         self._finite_or(foundation.get("summed_inspire_score"), 0.0)])
        return headers, rows

    def _catalog_export_table(self, grouped_data: dict) -> tuple:
        use_cases = self._ranked_use_cases(grouped_data)
        columns = self._export_columns(any(self._has_elite_fields(uc) for uc in use_cases),
                                       any(self._has_value_fields(uc) for uc in use_cases))
        headers = [header for header, _get, _numeric in columns]
        numeric_cols = {i for i, (_header, _get, numeric) in enumerate(columns) if numeric}
        rows = [[get(uc, rank) for _header, get, _numeric in columns] for rank, uc in enumerate(use_cases, start=1)]
        return headers, rows, numeric_cols

    def _catalog_file_name(self, lang_abbr: str, extension: str) -> str:
        output_languages = getattr(self, "output_languages", ["English"]) or ["English"]
        if len(output_languages) > 1:
            return f"{self.business_name}-dbx_inspire_{lang_abbr}.{extension}"
        return f"{self.business_name}-dbx_inspire.{extension}"

    def _generate_use_case_excel(self, language: str, lang_abbr: str, grouped_data: dict):
        warnings.filterwarnings('ignore', module='xlsxwriter')
        # Only generate Excel for English
        if language != "English":
            self.logger.info(f"Skipping Excel generation for {language} (only English Excel is generated).")
            return
        
        self.logger.info(f"--- Starting Excel Catalog Generation with XlsxWriter for {language} ---")
        
        local_excel_path = None
        try:
            self._install_excel_dependencies(self.logger)
            import xlsxwriter

            headers, data_rows, numeric_cols = self._catalog_export_table(grouped_data)
            if not data_rows:
                self.logger.warning(f"No data to write to Excel for {language}. Skipping.")
                return
            id_notebook_urls = {}
            for uc in self._ranked_use_cases(grouped_data):
                nb_url = self._get_use_case_notebook_url(uc)
                if nb_url:
                    id_notebook_urls[str(uc.get('No') or 'N/A')] = nb_url

            excel_file_name = self._catalog_file_name(lang_abbr, "xlsx")
            with tempfile.NamedTemporaryFile(delete=False, suffix=".xlsx") as tmp_file:
                local_excel_path = tmp_file.name
            
            self.logger.info(f"Creating Excel file at {local_excel_path}")
            workbook = xlsxwriter.Workbook(local_excel_path, {'strings_to_numbers': False})
            worksheet = workbook.add_worksheet('Use Cases')
            
            # Modern Business Color Palette
            PRIMARY = '#' + self._DOC_COLORS['teal']
            SECONDARY = '#E74C3C'    # Vibrant Coral  
            ACCENT = '#3498DB'       # Bright Blue
            BACKGROUND = '#ECF0F1'   # Soft Grey
            TEXT = '#2C3E50'         # Dark Grey
            
            # Define cell formats
            header_format = workbook.add_format({
                'bold': True,
                'font_color': 'white',
                'bg_color': PRIMARY,
                'border': 1,
                'align': 'center',
                'valign': 'vcenter',
                'text_wrap': True,
                'font_size': 11
            })
            
            cell_format = workbook.add_format({
                'border': 1,
                'align': 'left',
                'valign': 'top',
                'text_wrap': True,
                'font_size': 10
            })
            
            numeric_format = workbook.add_format({
                'border': 1,
                'align': 'center',
                'valign': 'vcenter',
                'num_format': '0.00',
                'font_size': 10
            })
            
            id_col = 0
            
            # Write headers
            for col_num, header in enumerate(headers):
                worksheet.write(0, col_num, header, header_format)
            
            id_link_format = workbook.add_format({
                'border': 1,
                'align': 'left',
                'valign': 'top',
                'text_wrap': True,
                'font_size': 10,
                'font_color': '#0563C1',
                'underline': True,
            })

            for row_num, row_data in enumerate(data_rows, start=1):
                for col_num, cell_data in enumerate(row_data):
                    if col_num == id_col:
                        id_text = str(cell_data)
                        nb_url = id_notebook_urls.get(id_text)
                        if nb_url:
                            worksheet.write_url(row_num, col_num, nb_url, id_link_format, string=id_text)
                        else:
                            worksheet.write(row_num, col_num, id_text, cell_format)
                    elif col_num in numeric_cols and isinstance(cell_data, (int, float)) and not isinstance(cell_data, bool):
                        worksheet.write_number(row_num, col_num, float(cell_data), numeric_format)
                    else:
                        worksheet.write(row_num, col_num, cell_data, cell_format)
            
            # Auto-fit column widths to content
            for col_num in range(len(headers)):
                # Calculate max width for this column
                max_width = len(str(headers[col_num]))
                for row_data in data_rows:
                    if col_num < len(row_data):
                        cell_value = str(row_data[col_num])
                        max_width = max(max_width, len(cell_value))
                # Set width to fit entire text
                column_width = max_width + 2
                worksheet.set_column(col_num, col_num, column_width)
            
            # Freeze top row
            worksheet.freeze_panes(1, 0)
            
            # Convert data range to native Excel Table
            last_row = len(data_rows)
            last_col = len(headers) - 1
            worksheet.add_table(0, 0, last_row, last_col, {
                'name': 'UseCaseTable',
                'style': 'Table Style Medium 9',
                'columns': [{'header': h} for h in headers]
            })
            
            for col_idx in sorted(numeric_cols):
                worksheet.conditional_format(1, col_idx, last_row, col_idx, {
                    'type': '3_color_scale',
                    'min_color': '#E74C3C',
                    'mid_color': '#F4D03F',
                    'max_color': '#27AE60',
                })

            found_headers, found_rows = self._foundations_export_table()
            if found_rows:
                found_sheet = workbook.add_worksheet('Foundations')
                for col_num, header in enumerate(found_headers):
                    found_sheet.write(0, col_num, header, header_format)
                for row_num, row in enumerate(found_rows, start=1):
                    for col_num, value in enumerate(row):
                        if isinstance(value, (int, float)) and not isinstance(value, bool):
                            found_sheet.write_number(row_num, col_num, value, numeric_format)
                        else:
                            found_sheet.write(row_num, col_num, value, cell_format)
                for col_num, header in enumerate(found_headers):
                    width = max([len(str(header))] + [len(str(row[col_num])) for row in found_rows])
                    found_sheet.set_column(col_num, col_num, min(width + 2, 80))
                found_sheet.freeze_panes(1, 0)
                found_sheet.add_table(0, 0, len(found_rows), len(found_headers) - 1, {
                    'name': 'FoundationTable', 'style': 'Table Style Medium 9',
                    'columns': [{'header': h} for h in found_headers]})

            workbook.close()
            self.logger.info(f"Excel file created successfully with native tables")
            
            workspace_excel_path = os.path.join(self.docs_output_dir, excel_file_name)
            self._save_excel(local_excel_path, workspace_excel_path, self.logger, language)
        except Exception as e:
            self.logger.critical(f"An error occurred during Excel generation for {language}: {get_clean_error_message(e)}")
            self._record_failed_step("Use Case Excel", language, e)
            raise
        finally:
            if local_excel_path and os.path.exists(local_excel_path):
                os.remove(local_excel_path)

    def _generate_markdown_catalog(self, language: str, lang_abbr: str, grouped_data: dict, summary_dict: dict, transliterated_name: str):
        """
        Generates a Markdown catalog file containing all use case information.
        This is ALWAYS generated as a fallback when PDF generation may fail.
        """
        if language != "English":
            self.logger.info(f"Skipping Markdown generation for {language} (only English Markdown is generated).")
            return
        
        self.logger.info(f"--- Starting Markdown Catalog Generation for {language} ---")

        try:
            md_model = self._document_model(language, {}, summary_dict or {}, grouped_data, transliterated_name)
            ranked_groups = [(domain["name"], domain["use_cases"]) for domain in md_model["domains"]]
            summary_dict, _unmatched = self._canonical_summary(summary_dict or {}, [domain for domain, _ in ranked_groups])
            md_content = []
            
            # Header
            md_content.append(f"# {self.business_name} - Databricks Inspire AI Use Cases Catalog\n")
            md_content.append(f"**Generated:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            md_content.append(f"**Business Name:** {transliterated_name or self.business_name}\n")
            
            executive_summary = " ".join(self._doc_narrative(summary_dict.get("Executive"), "Executive"))
            if executive_summary:
                md_content.append("\n## Executive Summary\n")
                md_content.append(f"{executive_summary}\n")
            
            # Stats
            total_use_cases = sum(len(ucs) for _, ucs in ranked_groups)
            md_content.append(f"\n## Overview\n")
            md_content.append(f"- **Total Use Cases:** {total_use_cases}\n")
            md_content.append(f"- **Business Domains:** {len(ranked_groups)}\n")
            
            # Use cases by domain
            md_content.append("\n## Use Cases by Business Domain\n")
            
            for domain_name, use_cases in ranked_groups:
                md_content.append(f"\n### {domain_name}\n")
                md_content.append(f"*{len(use_cases)} use case{'' if len(use_cases) == 1 else 's'}*\n")
                domain_summary = " ".join(self._domain_summary_paragraphs(summary_dict, domain_name))
                if domain_summary:
                    md_content.append(f"\n{domain_summary}\n")
                
                for uc in use_cases:
                    view = self._use_case_view(uc, md_model, {})
                    _md_nb_url = self._get_use_case_notebook_url(uc)
                    heading = f"[{view['id']}]({_md_nb_url})" if _md_nb_url else view["id"]
                    md_content.append(f"\n#### #{view['rank']} {heading}: {view['title']}\n")
                    if view["star"]:
                        md_content.append(f"\n> **{view['star']}**\n")
                    if view["name"]:
                        md_content.append(f"\n*{view['name']}*\n")
                    if view["coverage"]:
                        label, reason_label, reason = view["coverage"]
                        md_content.append(f"\n> **{label}**. {reason_label}: {reason}\n")
                    md_content.append("\n")
                    if view["subdomain"]:
                        md_content.append(f"- **{self._doc_text({}, 'subdomain')}:** {view['subdomain']}\n")
                    for _key, label, value in view["metrics"]:
                        md_content.append(f"- **{label}:** {value}\n")
                    for _key, label, value in view["narrative"]:
                        md_content.append(f"\n**{label}:**\n{value}\n")
                    md_content.append("\n")
                    for _key, label, value in view["facts"]:
                        md_content.append(f"- **{label}:** {value}\n")
                    for _key, label, items in view["value_facts"]:
                        md_content.append(f"\n**{label}:**\n" + "".join(f"- {item}\n" for item in items))
                    md_content.append("\n---\n")
            if md_model["foundations"]:
                cell = lambda value: str(value).replace("|", "\\|")
                md_content.append(f"\n## {self._doc_text({}, 'doc_foundations_title')}\n\n{self._doc_text({}, 'doc_foundations_lead')}\n\n"
                                  f"| | {self._doc_text({}, 'doc_foundation_tables')} | {self._doc_text({}, 'doc_foundation_unlocks')} |\n"
                                  "|---|---|---|\n")
                for row in md_model["foundations"]:
                    tables = row["tables"] + (f" ({row['joins']})" if row["joins"] else "")
                    md_content.append(f"| {cell(row['id'])} | {cell(tables)} | {row['count']}: {cell(row['use_cases'])} |\n")
            disclaimer_title, disclaimer = self._doc_disclaimer(md_model, {})
            md_content.append(f"\n## {disclaimer_title}\n\n" + "\n\n".join(disclaimer) + f"\n\n*{md_model['generated_by']}*\n")
            
            # Save to workspace
            # v0.7.2 bug fix: multi-language must NOT share the same filename.
            # Single-language run → readme.md (preserves "only-md-in-folder" convention).
            # Multi-language run → readme_{lang_abbr}.md per language.
            _output_langs = getattr(self, "output_languages", ["English"]) or ["English"]
            if len(_output_langs) > 1:
                md_file_name = f"readme_{lang_abbr}.md"
            else:
                md_file_name = "readme.md"
            workspace_md_path = os.path.join(self.docs_output_dir, md_file_name)
            md_text = ''.join(md_content)
            
            # Upload to workspace
            md_data_b64 = base64.b64encode(md_text.encode('utf-8')).decode()
            self._import_workspace_file(workspace_md_path, md_data_b64, workspace.ImportFormat.AUTO)
            abs_path = self.w_client.workspace.get_status(workspace_md_path).path
            self.logger.info(f"Success! Markdown Catalog uploaded to: {abs_path}")
            log_print(f"Success! Markdown Catalog ({language}) generated: {abs_path}")
        except Exception as e:
            self.logger.error(f"Failed to generate Markdown catalog for {language}: {get_clean_error_message(e)}")
            self._record_failed_step("Markdown Catalog", language, e)
            raise

    def _generate_csv_catalog(self, language: str, lang_abbr: str, grouped_data: dict):
        """
        Generates a CSV catalog file containing all use case information.
        This is ALWAYS generated as a fallback when Excel generation may fail.
        """
        if language != "English":
            self.logger.info(f"Skipping CSV generation for {language} (only English CSV is generated).")
            return
        
        self.logger.info(f"--- Starting CSV Catalog Generation for {language} ---")
        
        try:
            headers, data_rows, _ = self._catalog_export_table(grouped_data)
            if not data_rows:
                self.logger.warning(f"No data to write to CSV for {language}. Skipping.")
                return
            
            # Build CSV content
            output = io.StringIO()
            writer = csv.writer(output, quoting=csv.QUOTE_ALL)
            writer.writerow(headers)
            for row in data_rows:
                writer.writerow(row)
            csv_content = output.getvalue()
            
            # Save to workspace
            # v0.8.1 bug fix (multi-language): CSV writer was missed by the v0.7.3 patch
            # that fixed Excel and Markdown. When multiple languages are selected, each
            # language wrote over the previous one with the same filename. Apply the
            # same suffix convention as Excel/Markdown/PDF/PPTX.
            workspace_csv_path = os.path.join(self.docs_output_dir, self._catalog_file_name(lang_abbr, "csv"))
            
            # Upload to workspace
            csv_data_b64 = base64.b64encode(csv_content.encode('utf-8')).decode()
            self._import_workspace_file(workspace_csv_path, csv_data_b64, workspace.ImportFormat.AUTO)
            abs_path = self.w_client.workspace.get_status(workspace_csv_path).path
            self.logger.info(f"Success! CSV Catalog uploaded to: {abs_path}")
            log_print(f"Success! CSV Catalog ({language}) generated: {abs_path}")
        except Exception as e:
            self.logger.error(f"Failed to generate CSV catalog for {language}: {get_clean_error_message(e)}")
            self._record_failed_step("CSV Catalog", language, e)
            raise

    def _set_known_business_universe(self, business_details: list) -> None:
        details = list(business_details or []) + list(getattr(self, "_reference_details_last", None) or [])
        self._business_column_details_global = details
        self.global_table_names = {f"{c}.{s}.{t}" for (c, s, t, _, _, _) in details}

    def _validate_use_case_tables(self, parsed_rows: list, full_schema_details: list, log_prefix: str) -> tuple:
        """
        Validate that all tables referenced in 'Tables Involved' field actually exist in the schema.
        This catches LLM hallucinations where it invents table names.
        
        Returns:
            tuple: (is_valid: bool, hallucinated_use_cases: list, valid_use_cases: list)
        """
        import re
        
        # Build set of available tables from schema
        available_tables = set()
        for detail in full_schema_details:
            (catalog, schema, table, _, _, _) = detail
            available_tables.add(f"{catalog}.{schema}.{table}")
            available_tables.add(f"`{catalog}`.`{schema}`.`{table}`")
            available_tables.add(f"{catalog}.{schema}.{table}".lower())
        global_tables = getattr(self, "global_table_names", set())
        for tbl in global_tables:
            available_tables.add(tbl)
            available_tables.add(tbl.lower())
        
        hallucinated_use_cases = []
        valid_use_cases = []
        
        for row in parsed_rows:
            tables_involved_str = row.get('Tables Involved', '').strip()
            use_case_id = row.get('No', 'Unknown')
            use_case_name = row.get('Name', 'Unknown')
            
            # Skip volume paths (for ai_parse_document use cases)
            if tables_involved_str.startswith('/Volumes'):
                valid_use_cases.append(row)
                continue
            
            # Skip empty tables involved (will be caught by other validation)
            if not tables_involved_str:
                hallucinated_use_cases.append(row)
                row['hallucination_reason'] = "No tables specified"
                continue
            
            sql_fragment_keywords = {'SELECT', 'FROM', 'WHERE', 'CTE', 'JOIN', 'LIMIT', 'GROUP BY', 'ORDER BY', 'HAVING', 'UNION', 'INSERT', 'UPDATE', 'DELETE'}
            upper_tables = tables_involved_str.upper()
            if any(kw in upper_tables for kw in sql_fragment_keywords):
                hallucinated_use_cases.append(row)
                row['hallucination_reason'] = f"SQL fragment in Tables Involved (CSV parsing error): {tables_involved_str[:120]}"
                self.logger.warning(f"{log_prefix} Use case {use_case_id}: SQL fragment leaked into Tables Involved field")
                continue
            
            table_matches = []
            for table_str in tables_involved_str.split(','):
                table_str = table_str.strip()
                if not table_str:
                    continue
                cat, sch, tbl = parse_three_level_name(table_str)
                if cat and sch and tbl:
                    table_matches.append((cat, sch, tbl))
            
            if not table_matches:
                hallucinated_use_cases.append(row)
                row['hallucination_reason'] = f"Invalid table format: {tables_involved_str[:120]}"
                self.logger.warning(f"{log_prefix} Use case {use_case_id}: Invalid table format: {tables_involved_str[:120]}")
                continue
            
            # Check if all tables exist in schema
            all_tables_found = True
            missing_tables = []
            for match in table_matches:
                catalog, schema, table = match
                # Try multiple formats (normalized names for comparison)
                table_formats = [
                    f"{catalog}.{schema}.{table}",
                    build_fqn(catalog, schema, table),
                    f"{catalog}.{schema}.{table}".lower()
                ]
                
                found = any(fmt in available_tables for fmt in table_formats)
                if not found:
                    all_tables_found = False
                    missing_tables.append(f"{catalog}.{schema}.{table}")
            
            if not all_tables_found:
                hallucinated_use_cases.append(row)
                row['hallucination_reason'] = f"Tables not found in schema: {', '.join(missing_tables)}"
                self.logger.warning(f"{log_prefix} Use case {use_case_id}: Hallucinated tables: {', '.join(missing_tables)}")
            else:
                reference_tables = {k.lower() for k, v in getattr(self, "data_category_map", {}).items() if v == "REFERENCE"}
                if reference_tables:
                    non_reference_found = False
                    for match in table_matches:
                        catalog, schema, table = match
                        fqtn = f"{catalog}.{schema}.{table}".lower()
                        if fqtn not in reference_tables:
                            non_reference_found = True
                            break
                    if not non_reference_found:
                        hallucinated_use_cases.append(row)
                        row['hallucination_reason'] = "Reference-only use case"
                        self.logger.warning(f"{log_prefix} Use case {use_case_id}: Reference-only tables in use case")
                        continue
                valid_use_cases.append(row)
        
        is_valid = len(hallucinated_use_cases) == 0
        
        if not is_valid:
            self.logger.warning(f"{log_prefix} ⚠️ Found {len(hallucinated_use_cases)} use cases with hallucinated/missing tables out of {len(parsed_rows)} total")
            self.logger.warning(f"{log_prefix}    Valid use cases: {len(valid_use_cases)}")
            self.logger.warning(f"{log_prefix}    Hallucinated use cases: {len(hallucinated_use_cases)}")
        else:
            self.logger.info(f"{log_prefix} ✓ Table validation passed: All {len(valid_use_cases)} use cases reference existing tables")
        
        return (is_valid, hallucinated_use_cases, valid_use_cases)
    
    _VALID_QUALITY_LABELS = ("Ultra High", "Very High", "High", "Medium", "Low", "Very Low", "Ultra Low")

    @classmethod
    def _canonical_quality_label(cls, value):
        text = str(value or "").strip().lower()
        for label in cls._VALID_QUALITY_LABELS:
            if label.lower() == text:
                return label
        return None

    @staticmethod
    def _bounded_float(value, lo: float, hi: float):
        try:
            number = float(value)
        except (TypeError, ValueError):
            return None
        if number != number or number < lo or number > hi:
            return None
        return round(number, 2)

    @staticmethod
    def _use_case_set_fingerprint(use_cases: list, context: dict = None) -> str:
        import hashlib
        fields = ("No", "Name", "type", "Analytics Technique", "Statement", "Solution",
                  "Business Value", "Tables Involved", "High Level Design")
        digest = hashlib.sha256()
        for uc in sorted(use_cases or [], key=lambda u: str(u.get("No", ""))):
            digest.update(json.dumps([str(uc.get(f, "") or "") for f in fields], ensure_ascii=False).encode("utf-8"))
            digest.update(b"\x1e")
        digest.update(json.dumps(context or {}, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8"))
        return digest.hexdigest()[:16]

    @staticmethod
    def _schema_fingerprint_payload(schema_details: list) -> list:
        return sorted(json.dumps(list(d), ensure_ascii=False, default=str) for d in (schema_details or []))

    def _scored_cache_tag(self, prefix: str, use_cases: list, context: dict, schema_details: list) -> str:
        return prefix + self._use_case_set_fingerprint(
            use_cases, dict(context or {}, schema_details=self._schema_fingerprint_payload(schema_details)))

    def _scoring_task_timeout(self, base_timeout_seconds: int, prompt_chars: int) -> int:
        attempt_budget, _ = AIAgent.adaptive_timeout_seconds(base_timeout_seconds, prompt_chars)
        return int(attempt_budget * 2 + 120)

    def _mark_unscored(self, uc: dict, reason: str):
        uc['scoring_status'] = 'unscored'
        uc['scoring_unscored_reason'] = reason
        uc['Quality'] = 'Unscored'
        uc.pop('Quality Score', None)
        uc.pop('Inspire Score', None)

    _STRAGGLER_FLOOR_SECONDS = 120

    def _score_with_split_retry(self, use_cases: list, score_batch, batch_size: int, label: str,
                                per_task_timeout: int, already_scored: dict = None) -> tuple:
        """Score every UC with `score_batch(batch_label, batch, skip_cache) -> {uc_id: valid_record}`.

        Round 0 sends `batch_size` UCs per call. The UCs a batch leaves without a valid record, because
        the call failed or returned invalid items, are re-requested once as a single batch of only the
        missing UCs with the LLM cache bypassed, and only split in half on the rounds after that, for up
        to `runtime.scoring_split_retry_rounds` rounds per UC, without waiting for the other batches.
        A round-0 batch running longer than the per-task timeout, or once three batches have finished
        longer than three times their median duration (at least `_STRAGGLER_FLOOR_SECONDS`), is
        cancelled, so no single slow batch holds the stage. A timed-out batch is retried in halves and
        every retry gets the full per-task timeout, because a timeout says the batch was slow, not that
        its answer was unusable. An unusable answer gets one retry per UC; a second unusable answer
        that recovers nothing is not retried again.
        Only valid records are ever stored, so a failed or timed-out call can never overwrite a score
        that was already obtained. Returns (scores_by_id, unscored_reason_by_id).
        """
        import statistics
        import time as _time
        from collections import deque
        rt = TECHNICAL_CONTEXT["runtime"]
        max_rounds = rt["scoring_split_retry_rounds"]
        max_workers = rt["quality_scoring_max_workers"]
        id_set = {str(uc.get("No", "")) for uc in use_cases}
        scores = {k: v for k, v in (already_scored or {}).items() if k in id_set}
        reasons = {}
        unusable = set()
        pending = [uc for uc in use_cases if str(uc.get("No", "")) not in scores]
        if scores:
            self.logger.info(f"[{label}] Reusing {len(scores)} cached score(s); {len(pending)} UC(s) left to score")
        size = max(1, int(batch_size))
        queue = deque((0, pending[i:i + size]) for i in range(0, len(pending), size))
        workers = max(1, min(len(queue) or 1, max_workers))
        waves = -(-len(queue) // workers) if queue else 0
        stage_start = _time.time()
        deadline = stage_start + (per_task_timeout * waves + 60) * (max_rounds + 1)
        running, started, durations = {}, {}, []
        submitted, last_heartbeat, total_batches = 0, stage_start, len(queue)
        last_logged = stage_start

        def _run(batch_label, chunk, skip_cache):
            started[batch_label] = _time.time()
            return score_batch(batch_label, chunk, skip_cache)

        def _uc_id(uc):
            return str(uc.get("No", ""))

        def _settle(round_idx, chunk, batch_label, result, failure, timed_out=False):
            nonlocal total_batches
            recovered = 0
            for uc in chunk:
                record = result.get(_uc_id(uc)) if isinstance(result, dict) else None
                if record is not None and _uc_id(uc) not in scores:
                    scores[_uc_id(uc)] = record
                    reasons.pop(_uc_id(uc), None)
                    recovered += 1
            missing = [uc for uc in chunk if _uc_id(uc) not in scores]
            for uc in missing:
                reasons[_uc_id(uc)] = failure or "missing or invalid in LLM response"
            if not missing:
                return
            repeated = not timed_out and all(_uc_id(uc) in unusable for uc in missing)
            if not timed_out:
                unusable.update(_uc_id(uc) for uc in missing)
            if round_idx >= max_rounds or (round_idx > 0 and recovered == 0 and repeated):
                self.logger.warning(f"[{label}] {batch_label}: {len(missing)} UC(s) still without a valid score after "
                                    f"{round_idx + 1} attempt(s) ({failure or 'missing or invalid in LLM response'}); no further retry")
                return
            retry_size = len(missing) if round_idx == 0 and not timed_out else max(1, -(-len(missing) // 2))
            retries = [missing[i:i + retry_size] for i in range(0, len(missing), retry_size)]
            for retry in reversed(retries):
                queue.appendleft((round_idx + 1, retry))
            total_batches += len(retries)
            self.logger.warning(f"[{label}] {batch_label}: {len(missing)} UC(s) without a valid score "
                                f"({failure or 'missing or invalid in LLM response'}) -> retried at once in "
                                f"{len(retries)} batch(es) of <= {retry_size}, LLM cache bypassed"
                                f"{' [TIMEOUT-RETRY FIRED] full per-task timeout' if timed_out else ''}")

        with _SafeExecutorContext(max_workers=workers * 2, thread_name_prefix=label, logger=self.logger, name=label) as ctx:
            while queue or running:
                while queue and len(running) < workers:
                    round_idx, chunk = queue.popleft()
                    submitted += 1
                    batch_label = f"R{round_idx}B{submitted}"
                    running[ctx.submit(_run, batch_label, chunk, round_idx > 0)] = (round_idx, chunk, batch_label)
                now = _time.time()
                stop = (f"the {label} stage time budget ran out" if now >= deadline
                        else f"the {label} stage was cancelled by its parent task" if _is_thread_cancelled() else None)
                if stop:
                    for future, (round_idx, chunk, batch_label) in list(running.items()):
                        ctx.cancel(future)
                        for uc in chunk:
                            reasons[_uc_id(uc)] = stop
                    for _round, chunk in queue:
                        for uc in chunk:
                            reasons[_uc_id(uc)] = stop
                    self.logger.error(f"[{label}] {stop} with {len(running)} batch(es) running and {len(queue)} waiting; "
                                      f"their UCs stay unscored")
                    running.clear()
                    queue.clear()
                    break
                if now - last_heartbeat >= 15:
                    log_print(f"⏳ [{label}] {submitted - len(running)}/{total_batches} batch(es) done, {len(running)} in progress")
                    last_heartbeat = now
                    sys.stdout.flush()
                    sys.stderr.flush()
                if now - last_logged >= WAIT_HEARTBEAT_LOG_SECONDS:
                    _log_wait_heartbeat(self.logger, label, submitted - len(running), total_batches, len(running),
                                        now - stage_start, deadline - stage_start)
                    last_logged = now
                straggler_after = (max(self._STRAGGLER_FLOOR_SECONDS, 3 * statistics.median(durations))
                                   if len(durations) >= 3 else None)
                for future, (round_idx, chunk, batch_label) in list(running.items()):
                    if future.done():
                        del running[future]
                        if batch_label in started:
                            durations.append(now - started[batch_label])
                        timed_out = False
                        try:
                            result, failure = future.result(), None
                            if not isinstance(result, dict):
                                result, failure = {}, f"unexpected scorer result type {type(result).__name__}"
                        except BaseException as exc:
                            result, failure = {}, f"{type(exc).__name__}: {get_clean_error_message(exc, max_chars=160)}"
                            timed_out = _classify_runtime_error(exc) == "timeout"
                        _settle(round_idx, chunk, batch_label, result, failure, timed_out)
                        continue
                    elapsed = now - started[batch_label] if batch_label in started else 0.0
                    limit = per_task_timeout
                    if straggler_after is not None and round_idx == 0:
                        limit = min(limit, straggler_after)
                    if elapsed >= limit:
                        del running[future]
                        ctx.cancel(future)
                        kind = "per-task timeout" if limit == per_task_timeout else "straggler budget"
                        self.logger.warning(f"[{label}] {batch_label} exceeded its {kind} of {limit:.0f}s; cancelled")
                        _settle(round_idx, chunk, batch_label, {}, f"TimeoutError: exceeded the {kind} of {limit:.0f}s",
                                timed_out=True)
                if running and not any(f.done() for f in running):
                    _time.sleep(0.05)
        unscored = {}
        for uc in use_cases:
            uc_id = str(uc.get("No", ""))
            if uc_id not in scores:
                unscored[uc_id] = reasons.get(uc_id, "not scored")
        return scores, unscored

    def _log_unscored(self, label: str, unscored: dict):
        if not unscored:
            return
        sample = "; ".join(f"{uc_id}: {reason}" for uc_id, reason in list(unscored.items())[:10])
        self.logger.error(
            f"[{label}] {len(unscored)} UC(s) unscored after retries and excluded from selection: {sample}"
            f"{' ...' if len(unscored) > 10 else ''}"
        )

    def _log_quality_distribution(self, use_cases: list, title: str):
        distribution = {}
        for uc in use_cases:
            label = uc.get('Quality') or 'Unscored'
            distribution[label] = distribution.get(label, 0) + 1
        self.logger.info("")
        self.logger.info("=" * 80)
        self.logger.info(title)
        self.logger.info("=" * 80)
        self.logger.info(f"Total: {len(use_cases)} use cases")
        for label in self._VALID_QUALITY_LABELS + ("Unscored",):
            count = distribution.get(label, 0)
            if count > 0:
                self.logger.info(f"   {label}: {count} use cases")
        self.logger.info("=" * 80)
        return distribution

    @staticmethod
    def _apply_combined_score(uc: dict, s: dict) -> bool:
        uc['Strategic Alignment'] = None
        uc['Return on Investment'] = s.get('roi')
        uc['Reusability'] = s.get('reusability')
        uc['Time to Value'] = s.get('time_to_value')
        uc['Data Availability'] = s.get('data_availability')
        uc['Data Accessibility'] = s.get('data_accessibility')
        uc['Architecture Fitness'] = s.get('architecture_fitness')
        uc['Team Skills'] = s.get('team_skills')
        uc['Domain Knowledge'] = s.get('domain_knowledge')
        uc['People Allocation'] = s.get('people_allocation')
        uc['Budget Allocation'] = s.get('budget_allocation')
        uc['Time to Production'] = s.get('time_to_production')
        uc['Value'] = s.get('value_score')
        uc['Feasibility'] = s.get('feasibility_score')
        uc['Priority'] = s['priority_score']
        computed = compute_value_priority(uc)
        if computed is not None:
            uc['Value'], uc['Feasibility'], uc['Priority'] = computed
        uc['Priority Score'] = uc['Priority']
        uc['Business Priority Alignment'] = ''
        uc['Strategic Goals Alignment'] = ''
        uc['Justification'] = s.get('justification', '')
        uc['Quality'] = s['quality_label']
        uc['Quality Score'] = s['quality_score']
        uc['Quality Summary'] = s.get('quality_summary', '')
        uc['scoring_status'] = 'scored'
        uc.pop('scoring_unscored_reason', None)
        return computed is not None

    def _score_combined_value_and_quality(self, use_cases: list, full_schema_details: list,
                                           business_context: str = "", industry: str = "",
                                           strategic_initiative: str = "", value_chain: str = "",
                                           revenue_model: str = "") -> list:
        """Combined value + quality scoring in a single LLM pass.

        Writes the value factors, Value, Feasibility, Priority (= LLM priority_score, absolute 0-10),
        Priority Score, Quality, Quality Score and Quality Summary. A UC without a valid record
        (label, quality_score and priority_score all present) after split retries is marked
        scoring_status='unscored'; no default label or score is ever written.

        The partial-batch disk cache is keyed on the UC ids, their scored content and the context,
        so a cached score can only be reused for the identical UC text it was produced for.
        """
        if not use_cases:
            self.logger.info("No use cases to score (combined)")
            return use_cases

        full_markdown = self._build_schema_markdown_for_validation(full_schema_details)
        batch_size = TECHNICAL_CONTEXT["runtime"]["quality_scoring_batch_size"]
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Combined Value+Quality Scoring",
                sub_step_name="Started", status="started",
                message=f"Phase 4.5 combined scoring started for {len(use_cases)} UCs"
            )
        except Exception:
            pass
        self.logger.info("=" * 80)
        self.logger.info(f"PHASE 4.5: COMBINED VALUE+QUALITY SCORING ({len(use_cases)} use cases, batch size {batch_size})")
        self.logger.info("=" * 80)

        base_vars = {
            "business_context": business_context or self._get_business_context_fallback(),
            "industry": prompt_context_or_not_provided(industry, "industry"),
            "strategic_initiative": prompt_context_or_not_provided(strategic_initiative, "strategic initiative"),
            "value_chain": prompt_context_or_not_provided(value_chain, "value chain"),
            "revenue_model": prompt_context_or_not_provided(revenue_model, "revenue model"),
            "generation_instructions_section": self._get_generation_instruction_for_prompt("COMBINED_VALUE_QUALITY_SCORE_PROMPT"),
        }
        partial_tag = self._scored_cache_tag("combined_batch_partial_", use_cases, base_vars, full_schema_details)
        cached_scores = self.storage_manager.load_scored_use_cases(partial_tag)
        cached_scores = cached_scores if isinstance(cached_scores, dict) else {}

        def _score_batch(batch_label: str, uc_batch: list, skip_cache: bool) -> dict:
            if _is_thread_cancelled():
                raise Exception(f"[CombinedScore_{batch_label}] Aborted: parent task cancelled/timed out")
            prompt_vars = dict(base_vars,
                               schema_details=self._batch_schema_markdown(full_schema_details, uc_batch, full_markdown),
                               use_cases_to_validate=self._build_use_cases_markdown_for_validation(uc_batch))
            response_raw = self.ai_agent.run_worker(
                step_name=f"CombinedScore_{batch_label}",
                worker_prompt_path="COMBINED_VALUE_QUALITY_SCORE_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None,
                timeout_override=max(300, self.llm_timeout_seconds + len(uc_batch) * 10),
                skip_cache=skip_cache
            )
            return self._parse_combined_score_response(response_raw, uc_batch)

        est_chars = (self._prompt_overhead_chars("COMBINED_VALUE_QUALITY_SCORE_PROMPT")
                     + len(self._batch_schema_markdown(full_schema_details, use_cases[:batch_size], full_markdown))
                     + len(self._build_use_cases_markdown_for_validation(use_cases[:batch_size])))
        per_task_timeout = self._scoring_task_timeout(max(300, self.llm_timeout_seconds + batch_size * 10), est_chars)
        scores, unscored = self._score_with_split_retry(
            use_cases, _score_batch, batch_size, "CombinedScore", per_task_timeout, already_scored=cached_scores
        )
        self.storage_manager.save_scored_use_cases(scores, partial_tag)

        scorer_priority_kept = []
        for uc in use_cases:
            uc_id = str(uc.get('No', ''))
            s = scores.get(uc_id)
            if s is None:
                self._mark_unscored(uc, unscored.get(uc_id, "not scored"))
                continue
            if not self._apply_combined_score(uc, s):
                scorer_priority_kept.append(uc_id)

        scored_count = sum(1 for uc in use_cases if str(uc.get('No', '')) in scores)
        self.logger.info(
            f"[SCORE-HELPER FIRED] Value and Priority recomputed from the sub-scores for "
            f"{scored_count - len(scorer_priority_kept)}/{scored_count} scored use case(s)"
            + (f"; {len(scorer_priority_kept)} kept the scorer's own Priority because a sub-score is missing or "
               f"out of range: {scorer_priority_kept[:20]}" if scorer_priority_kept else ""))
        self._log_unscored("CombinedScore", unscored)
        self._log_quality_distribution(use_cases, "PHASE 4.5 COMPLETE: COMBINED VALUE+QUALITY SCORING")
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Combined Value+Quality Scoring",
                sub_step_name="Complete", status="ended_success" if not unscored else "ended_warning",
                message=f"Combined scoring: {len(scores)}/{len(use_cases)} UCs scored, {len(unscored)} unscored",
                result_json={"scored": len(scores), "unscored": len(unscored), "total": len(use_cases)}
            )
        except Exception:
            pass
        _force_gc(self.logger, "after combined scoring")
        return use_cases

    def _score_best_of_best(self, use_cases: list, full_schema_details: list,
                            business_context: str = "", industry: str = "",
                            strategic_initiative: str = "", value_chain: str = "",
                            revenue_model: str = "") -> list:
        """Best-of-Best ranking pass.

        One BEST_OF_BEST_SCORING_PROMPT call per batch, with the merged business context and the
        schema markdown. For every UC the LLM must return all 19 D-gate scores (0-10) and a short
        justification; the idea themes of all scored UCs are then named in one call by
        _assign_idea_themes. Each UC is mutated with bob_scores, idea_theme, bob_tier1/2/3_score,
        bob_score, bob_hard_veto_gate, bob_justification and bob_status.
        A UC with no complete gate set after split retries gets bob_status='unscored' and no
        bob_score; selection excludes it.
        """
        if not use_cases:
            return use_cases

        rt = TECHNICAL_CONTEXT["runtime"]
        batch_size = rt["bob_scoring_batch_size"]
        target_max = rt["target_max_use_cases"]
        num_batches = -(-len(use_cases) // batch_size)
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Best of Best Ranking",
                sub_step_name="Started", status="started",
                message=f"BoB ranking started for {len(use_cases)} UCs",
                result_json={"ucs": len(use_cases), "batches": num_batches, "target_max": target_max}
            )
        except Exception:
            pass
        self.logger.info("=" * 80)
        self.logger.info(f"🏆 BEST-OF-BEST RANKING ({len(use_cases)} UCs across {num_batches} batches)")
        self.logger.info("=" * 80)

        full_markdown = self._build_schema_markdown_for_validation(full_schema_details)
        gen_instr = self._get_generation_instruction_for_prompt("BEST_OF_BEST_SCORING_PROMPT")

        def _score_batch(batch_label: str, uc_batch: list, skip_cache: bool) -> dict:
            if _is_thread_cancelled():
                raise Exception(f"[BoB_{batch_label}] Aborted: parent task cancelled/timed out")
            prompt_vars = {
                "business_context": business_context or self._get_business_context_fallback(),
                "industry": prompt_context_or_not_provided(industry, "industry"),
                "strategic_initiative": prompt_context_or_not_provided(strategic_initiative, "strategic initiative"),
                "value_chain": prompt_context_or_not_provided(value_chain, "value chain"),
                "revenue_model": prompt_context_or_not_provided(revenue_model, "revenue model"),
                "schema_details": self._batch_schema_markdown(full_schema_details, uc_batch, full_markdown),
                "use_cases_to_validate": self._build_use_cases_markdown_for_validation(uc_batch),
                "target_max": target_max,
                "generation_instructions_section": gen_instr,
            }
            response_raw = self.ai_agent.run_worker(
                step_name=f"BoB_{batch_label}",
                worker_prompt_path="BEST_OF_BEST_SCORING_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None,
                timeout_override=max(300, self.llm_timeout_seconds + len(uc_batch) * 10),
                skip_cache=skip_cache
            )
            batch_ids = {str(uc.get("No", "")) for uc in uc_batch}
            return {rec["No"]: rec for rec in self._parse_bob_response(response_raw, uc_batch) if rec["No"] in batch_ids}

        est_chars = (self._prompt_overhead_chars("BEST_OF_BEST_SCORING_PROMPT")
                     + len(self._batch_schema_markdown(full_schema_details, use_cases[:batch_size], full_markdown))
                     + len(self._build_use_cases_markdown_for_validation(use_cases[:batch_size])))
        per_task_timeout = self._scoring_task_timeout(max(300, self.llm_timeout_seconds + batch_size * 10), est_chars)
        records, unscored = self._score_with_split_retry(use_cases, _score_batch, batch_size, "BoB", per_task_timeout)

        _bob_fields = ("bob_scores", "bob_tier1_score", "bob_tier2_score", "bob_tier3_score",
                       "bob_score", "bob_hard_veto_gate", "bob_justification")
        for uc in use_cases:
            uc_id = str(uc.get("No", ""))
            rec = records.get(uc_id)
            if rec is None:
                for key in _bob_fields:
                    uc.pop(key, None)
                uc["bob_status"] = "unscored"
                uc["bob_unscored_reason"] = unscored.get(uc_id, "not scored")
                continue
            scores = rec["bob_scores"]
            composed = self._compose_bob_score(scores)
            uc["bob_scores"] = scores
            uc["bob_justification"] = rec.get("bob_justification_short", "")
            uc["bob_tier1_score"] = composed["tier1"]
            uc["bob_tier2_score"] = composed["tier2"]
            uc["bob_tier3_score"] = composed["tier3"]
            uc["bob_score"] = composed["total"]
            uc["bob_hard_veto_gate"] = composed["hard_veto_gate"]
            uc["bob_status"] = "scored"
            uc.pop("bob_unscored_reason", None)

        self._assign_idea_themes([uc for uc in use_cases if uc.get("bob_status") == "scored"])
        self._log_unscored("BoB", unscored)
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Best of Best Ranking",
                sub_step_name="Complete", status="ended_success" if not unscored else "ended_warning",
                message=f"BoB ranking scored {len(records)}/{len(use_cases)} UCs ({len(unscored)} unscored)",
                result_json={"scored": len(records), "unscored": len(unscored), "total": len(use_cases)}
            )
        except Exception:
            pass
        self.logger.info(f"🏆 BoB ranking complete: {len(records)}/{len(use_cases)} UCs scored, {len(unscored)} unscored")
        return use_cases

    def _parse_bob_response(self, response_raw: str, uc_batch: list) -> list:
        """Parse the BoB JSON array. Returns one record per item that carries an id and all 19
        D-gate scores as numbers in [0, 10]; an item with any missing or invalid gate is dropped so
        its UC stays unscored and is retried. Tolerant to wrappers / fenced code."""
        import json as _json
        if not response_raw:
            return []
        text = response_raw.strip()
        if text.startswith("```"):
            nl = text.find("\n")
            if nl > 0:
                text = text[nl+1:]
            if text.endswith("```"):
                text = text[:-3].rstrip()
        lb = text.find("[")
        rb = text.rfind("]")
        if lb >= 0 and rb > lb:
            text = text[lb:rb+1]
        try:
            arr = _json.loads(text)
        except Exception as e:
            self.logger.warning(f"BoB parse failed: {get_clean_error_message(e)[:200]}; response head: {response_raw[:200]}")
            return []
        if not isinstance(arr, list):
            return []
        out = []
        invalid = 0
        for item in arr:
            if not isinstance(item, dict):
                continue
            uc_id = str(item.get("No") or item.get("id") or "").strip()
            scores_obj = item.get("bob_scores") or item.get("scores") or {}
            if not uc_id or not isinstance(scores_obj, dict):
                invalid += 1
                continue
            coerced = {}
            for gi in range(1, 20):
                value = self._bounded_float(scores_obj.get(f"D{gi}"), 0.0, 10.0)
                if value is None:
                    coerced = None
                    break
                coerced[f"D{gi}"] = int(round(value))
            if coerced is None:
                invalid += 1
                continue
            out.append({
                "No": uc_id,
                "bob_scores": coerced,
                "bob_justification_short": (item.get("bob_justification_short") or "").strip(),
            })
        if invalid:
            self.logger.warning(f"BoB parse: {invalid} item(s) rejected for a missing id or missing/invalid D1-D19 gate scores")
        return out

    def _compose_bob_score(self, scores_by_gate: dict) -> dict:
        """Compose 19 per-gate 0-10 scores into a tier-weighted total with hard-veto floor.

        Tier 1 (technical): D1..D11 — weight 0.30
        Tier 2 (business):  D12..D14 — weight 0.35
        Tier 3 (PBA):       D15..D19 — weight 0.35
        Hard-veto floor: if ANY gate < 3, total capped at 2.0 (Low — will be
          dropped by the Balanced filter if it wasn't already).
        """
        def _avg(gs, lo, hi):
            vals = [float(gs.get(f"D{i}", 0) or 0) for i in range(lo, hi+1)]
            return round(sum(vals) / max(1, len(vals)), 3)
        t1 = _avg(scores_by_gate, 1, 11)
        t2 = _avg(scores_by_gate, 12, 14)
        t3 = _avg(scores_by_gate, 15, 19)
        total = 0.30*t1 + 0.35*t2 + 0.35*t3
        # Hard-veto floor
        hv_gate = None
        for i in range(1, 20):
            v = float(scores_by_gate.get(f"D{i}", 0) or 0)
            if v < 3.0:
                hv_gate = f"D{i}"
                break
        if hv_gate is not None:
            total = min(total, 2.0)
        return {"tier1": t1, "tier2": t2, "tier3": t3, "total": round(total, 3), "hard_veto_gate": hv_gate}

    def _drop_hard_veto_ucs(self, scored_use_cases: list) -> list:
        """v1.0.3 B10: physically drop hard-veto UCs BEFORE any downstream selection.

        Runs unconditionally after BoB scoring (in both the trim-fires and
        trim-skipped paths). Prevents portfolios in the 20-75 UC band from
        shipping UCs with bob_score capped at the 2.0 hard-veto floor.
        """
        if not scored_use_cases:
            return scored_use_cases
        _n_in = len(scored_use_cases)
        survivors = [uc for uc in scored_use_cases if uc.get("bob_hard_veto_gate") in (None, "", "None")]
        _dropped = _n_in - len(survivors)
        if _dropped:
            _gates_dropped = {}
            for uc in scored_use_cases:
                g = uc.get("bob_hard_veto_gate")
                if g not in (None, "", "None"):
                    _gates_dropped[g] = _gates_dropped.get(g, 0) + 1
            self.logger.info(
                f"[G33 HARD_VETO_DROP FIRED] Dropped {_dropped} hard-veto UC(s) ({_n_in} -> {len(survivors)}); "
                f"by_gate={_gates_dropped}"
            )
        return survivors

    def _trim_target(self, rt: dict) -> int:
        draw = random.Random(f"{getattr(self, 'session_id', '')}:target_max")
        return rt["target_max_use_cases"] + draw.randint(1, max(1, rt["target_max_jitter"]))

    _THEME_METHOD_FAMILIES = ("anomaly detection", "prediction", "forecasting", "segmentation", "recommendation", "optimization",
                              "simulation", "attribution", "analysis", "text processing")
    @staticmethod
    def _clean_theme(theme) -> str:
        text = " ".join(str(theme or "").replace("_", " ").split()).strip(" .,:;'\"`")
        return text if text and len(text) <= 60 and len(text.split()) <= 6 else ""

    def _theme_listing(self, use_cases: list) -> str:
        lines = []
        for uc in use_cases:
            table = str(uc.get("Primary Table") or self._extract_primary_table(uc.get("Tables Involved", "")) or "").split(".")[-1]
            technique = " ".join(str(uc.get("Analytics Technique") or "").split())[:90]
            lines.append(f"- {uc.get('No')} | {' '.join(str(uc.get('Name') or '').split())[:120]} | technique: {technique or 'not given'}"
                         f" | main table: {table or 'not given'}")
        return "\n".join(lines)

    def _ask_idea_themes(self, use_cases: list, existing: list, skip_cache: bool) -> dict:
        section = ""
        if existing:
            section = ("\n## THEMES ALREADY USED\nPut each use case in one of these themes when its idea matches; name a new "
                       "theme only when none does:\n" + "\n".join(f"- {theme}" for theme in existing) + "\n")
        prompt_vars = {"use_cases_listing": self._theme_listing(use_cases), "existing_themes_section": section,
                       "method_families": ", ".join(self._THEME_METHOD_FAMILIES),
                       "generation_instructions_section": self._get_generation_instruction_for_prompt("THEME_ASSIGNMENT_PROMPT")}
        raw = self.ai_agent.run_worker(step_name="Idea_Themes", worker_prompt_path="THEME_ASSIGNMENT_PROMPT", prompt_vars=prompt_vars,
                                       response_schema=None, timeout_override=300, skip_cache=skip_cache)
        ids = {str(uc.get("No") or "") for uc in use_cases}
        answer = {}
        for item in self._parse_json_array_items(raw, "idea themes"):
            if not isinstance(item, dict):
                continue
            uc_id = str(item.get("No") or item.get("id") or "").strip()
            theme = self._clean_theme(item.get("idea_theme") or item.get("theme"))
            if uc_id in ids and theme and uc_id not in answer:
                answer[uc_id] = theme
        return answer

    def _assign_idea_themes(self, use_cases: list) -> dict:
        report = {"use_cases": len(use_cases), "themes": 0, "singletons": 0, "second_call": 0, "regrouped": 0, "fallback": 0,
                  "source": "model"}
        if not use_cases:
            return report
        n = len(use_cases)
        floor = min(8, max(1, n // 6))
        answer, reason = {}, ""
        for attempt in (1, 2):
            try:
                answer = self._ask_idea_themes(use_cases, [], skip_cache=attempt > 1)
            except Exception as error:
                answer, reason = {}, f"the theme call failed ({get_clean_error_message(error, max_chars=200)})"
                break
            distinct = len({theme.casefold() for theme in answer.values()})
            if answer and distinct >= floor:
                break
            reason = ("the answer held no usable theme" if not answer
                      else f"the answer put {n} use cases in {distinct} theme(s), fewer than {floor}")
            answer = {}
        missing = [uc for uc in use_cases if str(uc.get("No") or "") not in answer]
        if answer and missing:
            spellings = list(dict.fromkeys(answer.values()))
            try:
                placed = self._ask_idea_themes(missing, spellings, skip_cache=True)
            except Exception as error:
                placed = {}
                self.logger.warning(f"[THEME-ASSIGN] the call for {len(missing)} use case(s) the first answer left out failed "
                                    f"({get_clean_error_message(error, max_chars=200)})")
            answer.update(placed)
            report["second_call"] = len(placed)
        if answer and n >= 20:
            self._regroup_lonely_themes(use_cases, answer, report)
        if not answer:
            report["source"] = "fallback"
            self.logger.warning(f"[THEME-ASSIGN FALLBACK] {reason or 'no theme was returned'}; the {n} use cases take their subdomain as theme")
        spelling = {}
        for uc in use_cases:
            theme = answer.get(str(uc.get("No") or ""))
            if not theme:
                report["fallback"] += 1
                theme = self._clean_theme(uc.get("Subdomain")) or self._clean_theme(uc.get("Business Domain")) or "Unclassified"
            uc["idea_theme"] = spelling.setdefault(theme.casefold(), theme)
        sizes = Counter(uc["idea_theme"] for uc in use_cases)
        largest = sizes.most_common(1)[0]
        report.update(themes=len(sizes), singletons=sum(1 for size in sizes.values() if size == 1))
        if report["source"] == "model":
            self.logger.info(f"[THEME-ASSIGN FIRED] {n} use cases -> {report['themes']} themes ({report['singletons']} with one use case, "
                             f"largest {largest[0]} ({largest[1]}); {report['second_call']} placed by a second call, "
                             f"{report['fallback']} took their subdomain)")
        if n >= 20 and report["singletons"] * 2 > report["themes"]:
            self.logger.warning(f"[THEME-ASSIGN] {report['singletons']} of {report['themes']} themes hold one use case; "
                                f"the portfolio may be over-fragmented")
        return report

    def _regroup_lonely_themes(self, use_cases: list, answer: dict, report: dict) -> None:
        sizes = Counter(theme.casefold() for theme in answer.values())
        lonely = [uc for uc in use_cases if sizes.get(str(answer.get(str(uc.get("No") or "")) or "").casefold()) == 1]
        if len(lonely) * 4 <= len(sizes):
            return
        shared = list(dict.fromkeys(theme for theme in answer.values() if sizes[theme.casefold()] > 1))
        try:
            placed = self._ask_idea_themes(lonely, shared, skip_cache=True)
        except Exception as error:
            self.logger.warning(f"[THEME-REGROUP] the call for the {len(lonely)} use cases alone in their theme failed "
                                f"({get_clean_error_message(error, max_chars=200)}); the first answer is kept")
            return
        trial = {**answer, **placed}
        trial_sizes = Counter(theme.casefold() for theme in trial.values())
        still_alone = sum(1 for size in trial_sizes.values() if size == 1)
        if not placed or still_alone >= len(lonely):
            self.logger.warning(f"[THEME-REGROUP] placing the {len(lonely)} use cases alone in their theme again left {still_alone} "
                                f"alone; the first answer is kept")
            return
        answer.update(placed)
        report["regrouped"] = len(lonely) - still_alone
        self.logger.info(f"[THEME-REGROUP FIRED] {len(lonely)} of {len(sizes)} themes held one use case; placed again with the "
                         f"{len(shared)} shared themes listed, {still_alone} are still alone and the portfolio has "
                         f"{len(trial_sizes)} themes")

    def _stratified_trim_by_theme(self, scored_use_cases: list, target_max: int,
                                   theme_max_share: float = 0.15) -> list:
        """Deterministic stratified trim preserving idea coverage.

        Algorithm:
          1. Group UCs by idea_theme.
          2. FLOOR: every theme keeps its best UC by bob_score (coverage guarantee).
          3. FILL: the remaining target_max - num_themes slots go to the strongest
             remaining UCs by bob_score DESC, across all themes.
          4. CAP: no theme exceeds floor(target_max * theme_max_share) UCs.

        Returns trimmed list. If input count <= target_max, returns input unchanged.

        v0.7.3: Pre-filter hard-veto UCs (bob_hard_veto_gate != None) BEFORE
        grouping by theme. Any theme that loses all UCs is legitimately
        eliminated. Prevents the 1-per-theme coverage floor from saving a UC
        that failed a hard gate.
        """
        if not scored_use_cases:
            return scored_use_cases

        # v0.7.3: hard-veto pre-filter
        _n_in = len(scored_use_cases)
        survivors = [uc for uc in scored_use_cases if uc.get("bob_hard_veto_gate") in (None, "", "None")]
        _dropped_hv = _n_in - len(survivors)
        if _dropped_hv:
            self.logger.info(
                f"Hard-veto pre-filter: dropped {_dropped_hv} UC(s) with bob_hard_veto_gate set "
                f"({_n_in} -> {len(survivors)}) before stratified trim"
            )
        scored_use_cases = survivors

        if not scored_use_cases or len(scored_use_cases) <= target_max:
            return scored_use_cases

        from collections import defaultdict
        groups = defaultdict(list)
        for uc in scored_use_cases:
            theme = uc.get("idea_theme") or "Unclassified"
            groups[theme].append(uc)
        num_themes = len(groups)
        cap_per_theme = max(1, int(target_max * theme_max_share))

        # Handle degenerate case: more themes than target_max
        if num_themes >= target_max:
            self.logger.info(
                f"Stratified trim: num_themes={num_themes} >= target_max={target_max}. "
                f"Picking top-{target_max} themes by total bob_score, 1 UC per theme."
            )
            ranked_themes = sorted(
                groups.items(),
                key=lambda kv: min(self._score_rank_part(uc.get("bob_score")) for uc in kv[1]),
            )[:target_max]
            out = []
            for theme, lst in ranked_themes:
                best = min(lst, key=lambda u: self._score_rank_part(u.get("bob_score")))
                out.append(best)
            return out

        ranked_groups = {theme: sorted(lst, key=lambda u: self._score_rank_part(u.get("bob_score")))
                         for theme, lst in groups.items()}
        keep = {theme: lst[:1] for theme, lst in ranked_groups.items()}
        rest = sorted((uc for lst in ranked_groups.values() for uc in lst[1:]),
                      key=lambda u: self._score_rank_part(u.get("bob_score")))
        slots = target_max - num_themes
        for uc in rest:
            if slots <= 0:
                break
            theme = uc.get("idea_theme") or "Unclassified"
            if len(keep[theme]) < cap_per_theme:
                keep[theme].append(uc)
                slots -= 1
        out = [uc for theme in groups for uc in keep[theme]]
        largest = max(keep.items(), key=lambda kv: len(kv[1]))
        self.logger.info(
            f"Stratified trim: {len(scored_use_cases)} -> {len(out)} UCs across {num_themes} themes "
            f"(cap_per_theme={cap_per_theme}, largest theme={(largest[0], len(largest[1]))})"
        )
        return out

    def _parse_json_array_items(self, response_raw: str, label: str) -> list:
        """The items of a JSON array answer. A malformed or truncated answer keeps every complete
        object it holds; an empty or unreadable one returns [] so the caller retries."""
        if not response_raw:
            self.logger.warning(f"Empty {label} response")
            return []
        cleaned = response_raw.strip()
        if cleaned.startswith("```"):
            cleaned = re.sub(r'^```\w*\n?', '', cleaned)
            cleaned = re.sub(r'\n?```$', '', cleaned)
        candidate = cleaned
        json_match = re.search(r'\[.*\]', candidate, re.DOTALL)
        if json_match:
            candidate = json_match.group(0)
        try:
            parsed = json.loads(candidate)
        except json.JSONDecodeError as e:
            salvaged = []
            for chunk in _iter_balanced_json_objects(cleaned):
                try:
                    salvaged.append(json.loads(chunk))
                except json.JSONDecodeError:
                    continue
            if not salvaged:
                self.logger.warning(
                    f"Failed to parse {label} JSON (response {len(cleaned)} chars, 0 complete "
                    f"object(s) salvaged): {get_clean_error_message(e)}; the whole batch is left for retry")
                return []
            self.logger.warning(
                f"{label[:1].upper() + label[1:]} response was malformed/truncated ({get_clean_error_message(e)}); "
                f"salvaged {len(salvaged)} complete object(s) and left the rest for retry")
            parsed = salvaged
        return parsed if isinstance(parsed, list) else [parsed]

    def _parse_combined_score_response(self, response_raw: str, original_use_cases: list) -> dict:
        """Parse the combined value+quality JSON response into {uc_id: scores_dict}.

        Only items for UCs of this batch that carry a valid quality_label, a numeric quality_score
        (0-5) and a numeric priority_score (0-10) are returned. Anything else is left out so the
        caller retries it; nothing is defaulted.
        """
        result = {}
        parsed = self._parse_json_array_items(response_raw, "combined score")

        def safe_float(val):
            if val is None:
                return None
            try:
                return round(float(val), 2)
            except (ValueError, TypeError):
                return None

        batch_ids = {str(uc.get('No', '')) for uc in original_use_cases}
        rejected = 0
        for item in parsed:
            if not isinstance(item, dict):
                continue
            uc_id = str(item.get('use_case_id', '') or '').strip()
            if uc_id not in batch_ids:
                continue
            quality_label = self._canonical_quality_label(item.get('quality_label'))
            quality_score = self._bounded_float(item.get('quality_score'), 0.0, 5.0)
            priority_score = self._bounded_float(item.get('priority_score'), 0.0, 10.0)
            if quality_label is None or quality_score is None or priority_score is None:
                rejected += 1
                continue
            result[uc_id] = {
                'roi': safe_float(item.get('roi')),
                'reusability': safe_float(item.get('reusability')),
                'time_to_value': safe_float(item.get('time_to_value')),
                'data_availability': safe_float(item.get('data_availability')),
                'data_accessibility': safe_float(item.get('data_accessibility')),
                'architecture_fitness': safe_float(item.get('architecture_fitness')),
                'team_skills': safe_float(item.get('team_skills')),
                'domain_knowledge': safe_float(item.get('domain_knowledge')),
                'people_allocation': safe_float(item.get('people_allocation')),
                'budget_allocation': safe_float(item.get('budget_allocation')),
                'time_to_production': safe_float(item.get('time_to_production')),
                'value_score': safe_float(item.get('value_score')),
                'feasibility_score': safe_float(item.get('feasibility_score')),
                'priority_score': priority_score,
                'justification': item.get('justification', ''),
                'quality_score': quality_score,
                'quality_label': quality_label,
                'quality_summary': self._sanitize_quality_summary(item.get('quality_summary', '')),
                'strengths': item.get('strengths', ''),
                'weaknesses': item.get('weaknesses', ''),
            }

        self.logger.info(
            f"Parsed combined scores for {len(result)}/{len(original_use_cases)} use cases"
            f"{f' ({rejected} item(s) rejected for a missing/invalid quality_label, quality_score or priority_score)' if rejected else ''}"
        )
        return result

    @staticmethod
    def _sanitize_quality_summary(quality_summary: str) -> str:
        """
        Validates and sanitizes quality_summary values from LLM output.
        Detects when the LLM produced a hallucination-check boilerplate instead of
        a proper data-to-value chain explanation, and flags it so users see a
        meaningful indicator rather than misleading filler text.
        """
        if not quality_summary or not isinstance(quality_summary, str):
            return ''
        
        summary = quality_summary.strip()
        
        bad_prefixes = [
            'NO HALLUCINATION',
            'HALLUCINATION DETECTED',
            'All columns verified',
            'Brief summary -',
            'Brief summary:',
        ]
        
        bad_phrases = [
            'all columns verified to exist',
            'strong causal signals present',
            'unique business problem',
            'strong causal signals. unique business problem',
        ]
        
        summary_lower = summary.lower()
        
        is_boilerplate = False
        for prefix in bad_prefixes:
            if summary_lower.startswith(prefix.lower()):
                is_boilerplate = True
                break
        
        if not is_boilerplate:
            for phrase in bad_phrases:
                if summary_lower == phrase:
                    is_boilerplate = True
                    break
        
        if not is_boilerplate and len(summary) < 80:
            has_column_ref = '`' in summary or 'column' in summary_lower
            has_operation = any(op in summary_lower for op in [
                'aggregat', 'join', 'comput', 'calculat', 'compar', 'datediff',
                'avg', 'sum', 'count', 'percentile', 'group by', 'correlat',
                'deriv', 'transform', 'filter', 'partition', 'window',
            ])
            if not has_column_ref and not has_operation:
                is_boilerplate = True
        
        if is_boilerplate:
            return f'[Insufficient detail] {summary}'
        
        return summary

    def _build_schema_markdown_for_validation(self, full_schema_details: list, tables=None) -> str:
        """Build a concise schema markdown for validation prompts. Uses disk cache if available.

        tables: optional iterable of table FQNs; restricts the markdown to those tables plus their
        one-hop FK partners and renders every column (not cached)."""
        if tables is None:
            cached = self.storage_manager.load_schema_markdown()
            if cached:
                return cached
        
        if not full_schema_details:
            return "No schema details available."
        
        if tables is not None:
            wanted = {str(t).replace('`', '').strip().lower() for t in tables if str(t or '').strip()}
            keep = set(wanted)
            loader = getattr(self, "data_loader", None)
            if loader is not None:
                for graph_key, rels in list(loader.foreign_key_graph.items()):
                    source = ".".join(str(part) for part in graph_key).lower()
                    for rel in rels:
                        target = f"{rel[4] or graph_key[0]}.{rel[5] or graph_key[1]}.{rel[6]}".lower()
                        if source in wanted:
                            keep.add(target)
                        if target in wanted:
                            keep.add(source)
            subset = [d for d in full_schema_details if f"{d[0]}.{d[1]}.{d[2]}".lower() in keep]
            if not subset:
                return "No schema details available."
            subset.sort(key=lambda d: f"{d[0]}.{d[1]}.{d[2]}")
            return "## Available Tables and Columns\n\n" + self._format_schema_for_prompt(
                subset, max_column_comment=200, max_table_comment=300)
        
        ordered = sorted(full_schema_details, key=lambda d: f"{d[0]}.{d[1]}.{d[2]}")
        result = "## Available Tables and Columns\n\n" + self._format_schema_for_prompt(
            ordered, max_columns=30, max_column_comment=50, max_table_comment=200)
        self.storage_manager.save_schema_markdown(result)
        return result
    
    def _uc_batch_tables(self, uc_batch: list) -> list:
        return sorted({t for uc in uc_batch or [] for t in self._uc_table_refs(uc) if not t.lower().startswith("/volumes")})

    def _batch_schema_markdown(self, full_schema_details: list, uc_batch: list, full_markdown: str) -> str:
        md = self._grounding_schema_markdown(full_schema_details, self._uc_batch_tables(uc_batch))
        return md if md is not None else full_markdown

    def _resolve_batch_tables(self, full_schema_details: list, table_refs) -> list:
        known = {f"{d[0]}.{d[1]}.{d[2]}".lower() for d in full_schema_details or []}
        tables = set()
        for ref in table_refs or []:
            low = str(ref).replace("`", "").strip().lower()
            if not low or low.startswith("/volumes"):
                continue
            tables.update([low] if low in known else [fqn for fqn in known if fqn.endswith("." + low)])
        return sorted(tables)

    def _grounding_schema_markdown(self, full_schema_details: list, table_refs):
        tables = self._resolve_batch_tables(full_schema_details, table_refs)
        if not tables:
            return None
        markdown = self._build_schema_markdown_for_validation(full_schema_details, tables=tables)
        return self._enforce_schema_superset(full_schema_details, tables, markdown)

    @staticmethod
    def _schema_markdown_rendered_columns(markdown: str) -> dict:
        rendered, current = {}, None
        for line in (markdown or "").splitlines():
            stripped = line.strip()
            if stripped.startswith("### "):
                current = stripped[4:].replace("`", "").strip().lower()
                rendered.setdefault(current, set())
            elif current is not None and stripped.startswith("|"):
                cell = stripped.strip("|").split("|", 1)[0].strip().lower()
                if cell and cell not in ("column", "---", "..."):
                    rendered[current].add(cell)
        return rendered

    def _schema_markdown_missing_columns(self, markdown: str, schema_details: list, tables) -> list:
        wanted = {str(t).replace("`", "").strip().lower() for t in (tables or []) if str(t or "").strip()}
        if not wanted or not schema_details:
            return []
        required = defaultdict(set)
        for d in schema_details:
            fqn = f"{d[0]}.{d[1]}.{d[2]}".lower()
            if fqn in wanted or fqn.rsplit(".", 1)[-1] in wanted:
                required[fqn].add(str(d[3]).lower())
        rendered = self._schema_markdown_rendered_columns(markdown)
        missing = []
        for fqn in sorted(required):
            have = rendered.get(fqn, set())
            missing.extend(f"{fqn}.{col}" for col in sorted(required[fqn] - have))
        return missing

    def _enforce_schema_superset(self, schema_details: list, tables, markdown: str,
                                 max_column_comment: int = 200, max_table_comment: int = 300) -> str:
        missing = self._schema_markdown_missing_columns(markdown, schema_details, tables)
        if not missing:
            return markdown
        wanted = {str(t).replace("`", "").strip().lower() for t in (tables or []) if str(t or "").strip()}
        subset = [d for d in schema_details
                  if f"{d[0]}.{d[1]}.{d[2]}".lower() in wanted
                  or f"{d[0]}.{d[1]}.{d[2]}".lower().rsplit(".", 1)[-1] in wanted]
        subset.sort(key=lambda d: f"{d[0]}.{d[1]}.{d[2]}")
        rebuilt = "## Available Tables and Columns\n\n" + self._format_schema_for_prompt(
            subset, max_column_comment=max_column_comment, max_table_comment=max_table_comment)
        self.logger.warning(
            f"[SCHEMA-SUPERSET FIRED] schema markdown was missing {len(missing)} column(s) of referenced "
            f"table(s): {missing[:5]}{' ...' if len(missing) > 5 else ''}; rebuilt with full columns")
        return rebuilt

    def _build_use_cases_markdown_for_validation(self, use_cases: list) -> str:
        """Build use cases markdown for validation prompt."""
        md_parts = ["## Use Cases to Validate\n\n"]
        
        for uc in use_cases:
            uc_id = uc.get('No', 'UNKNOWN')
            uc_name = uc.get('Name', 'Unknown')
            uc_type = uc.get('type', 'N/A')
            uc_statement = uc.get('Statement', 'N/A')
            uc_solution = uc.get('Solution', 'N/A')
            uc_value = uc.get('Business Value', 'N/A')
            uc_tables = uc.get('Tables Involved', 'N/A')
            uc_tech_design = uc.get('High Level Design', 'N/A')
            uc_analytics = uc.get('Analytics Technique', 'N/A')
            
            md_parts.append(f"### Use Case: {uc_id} - {uc_name}\n\n")
            md_parts.append(f"**Type:** {uc_type}\n\n")
            md_parts.append(f"**Analytics Technique:** {uc_analytics}\n\n")
            md_parts.append(f"**Problem Statement:**\n{uc_statement}\n\n")
            md_parts.append(f"**Proposed Solution:**\n{uc_solution}\n\n")
            md_parts.append(f"**Business Value:**\n{uc_value}\n\n")
            md_parts.append(f"**Tables Involved:**\n`{uc_tables}`\n\n")
            md_parts.append(f"**High Level Design:**\n{uc_tech_design}\n\n")
            md_parts.append("---\n\n")
        
        return ''.join(md_parts)
    
    def _parse_validation_response(self, response_raw: str, original_use_cases: list) -> list:
        """Parse the validation response from LLM."""
        import json
        
        results = []
        
        try:
            json_clean = response_raw.strip()
            if json_clean.startswith('```'):
                json_clean = re.sub(r'^```[a-z]*\n', '', json_clean)
                json_clean = re.sub(r'\n```$', '', json_clean)
            
            json_start = json_clean.find('[')
            json_end = json_clean.rfind(']') + 1
            if json_start >= 0 and json_end > json_start:
                json_clean = json_clean[json_start:json_end]
            
            parsed = json.loads(json_clean)
            
            if isinstance(parsed, list):
                results = parsed
            else:
                self.logger.warning("Validation response is not a list, rejecting all use cases in this batch")
                for uc in original_use_cases:
                    results.append({
                        'use_case_id': uc.get('No', 'UNKNOWN'),
                        'use_case_name': uc.get('Name', 'Unknown'),
                        'validation_status': 'REJECTED',
                        'confidence_score': 0.0,
                        'fatal_flaws_found': ['VALIDATION_PARSE_DEFAULT_REJECT'],
                        'validation_summary': 'Validation parsing failed - defaulting to REJECTED per Phase B1',
                        'recommendation': 'BLOCKING: fix LLM response format and rerun',
                        'validation_error': 'JSON_PARSE_FAILED'
                    })
                    
        except json.JSONDecodeError as e:
            self.logger.error(f"Failed to parse validation JSON: {get_clean_error_message(e)}")
            self.logger.debug(f"Raw response: {response_raw[:500]}...")
            
            for uc in original_use_cases:
                results.append({
                    'use_case_id': uc.get('No', 'UNKNOWN'),
                    'use_case_name': uc.get('Name', 'Unknown'),
                    'validation_status': 'REJECTED',
                    'confidence_score': 0.0,
                    'fatal_flaws_found': ['VALIDATION_PARSE_DEFAULT_REJECT'],
                    'validation_summary': 'Validation parsing failed - defaulting to REJECTED per Phase B1',
                    'recommendation': 'BLOCKING: fix LLM response format and rerun',
                    'validation_error': 'JSON_PARSE_FAILED'
                })
        
        validated_ids = {r.get('use_case_id') for r in results if isinstance(r, dict)}
        for uc in original_use_cases:
            uc_id = uc.get('No', 'UNKNOWN')
            if uc_id not in validated_ids:
                results.append({
                    'use_case_id': uc_id,
                    'use_case_name': uc.get('Name', 'Unknown'),
                    'validation_status': 'REJECTED',
                    'confidence_score': 0.0,
                    'fatal_flaws_found': ['VALIDATION_MISSING_FROM_RESPONSE'],
                    'validation_summary': 'Not included in validation response - not validated, therefore rejected',
                    'recommendation': 'BLOCKING: re-run validation for this use case',
                    'validation_error': 'MISSING_FROM_RESPONSE'
                })
        
        return results

    def _repair_tables_involved(self, tables_str: str, row_dict: dict, log_prefix: str) -> str:
        """
        Detect and repair 'Tables Involved' values corrupted by CSV column shifting.
        When LLM's High Level Design contains unquoted commas, columns shift and 
        Tables Involved receives SQL fragments instead of table FQTNs.
        
        Repair strategy: scan ALL values in the row for catalog.schema.table patterns
        and reconstruct the field from the matches that are real tables of this run
        (self.global_table_names). A match that is not a known table is never used, and
        nothing is repaired while the known-table registry is empty.
        """
        if not tables_str:
            return tables_str
        known_tables = {str(t).replace('`', '').lower() for t in (getattr(self, "global_table_names", None) or set())}
        
        sql_keywords = {'SELECT', 'FROM', 'WHERE', 'CTE', 'JOIN', 'LIMIT', 'GROUP', 'ORDER', 'HAVING', 'UNION'}
        upper_val = tables_str.upper()
        has_sql = any(kw in upper_val for kw in sql_keywords)
        
        if not has_sql:
            return tables_str
        
        fqtn_pattern = re.compile(r'[`]?(\w+)[`]?\.[`]?(\w+)[`]?\.[`]?(\w+)[`]?')
        found_tables = set()
        
        for key, val in row_dict.items():
            if val and isinstance(val, str):
                for match in fqtn_pattern.finditer(val):
                    cat, sch, tbl = match.group(1), match.group(2), match.group(3)
                    if len(cat) > 1 and len(sch) > 1 and len(tbl) > 1:
                        skip_words = {'select', 'from', 'where', 'join', 'left', 'right', 'inner', 'outer', 
                                     'group', 'order', 'having', 'union', 'case', 'when', 'then', 'else', 'end',
                                     'null', 'true', 'false', 'and', 'not', 'between', 'like', 'exists'}
                        if cat.lower() not in skip_words and sch.lower() not in skip_words and tbl.lower() not in skip_words:
                            if f"{cat}.{sch}.{tbl}".lower() in known_tables:
                                found_tables.add(f"{cat}.{sch}.{tbl}")
        
        if found_tables:
            repaired = ", ".join(sorted(found_tables))
            self.logger.debug(f"{log_prefix} Repaired Tables Involved: extracted {len(found_tables)} FQTNs from shifted CSV row")
            return repaired
        
        return tables_str

    def _canonical_analytics_technique(self, raw, uc_id="", context="") -> str:
        label = " ".join(str(raw if raw is not None else "").split())
        if not label or label.lower() in ("n/a", "none", "null"):
            return ""
        for key in TECHNIQUE_IMPLEMENTATION_GUIDANCE:
            if key.lower() == label.lower():
                return key
        self.logger.warning(
            f"[TECHNIQUE_UNKNOWN] {context or 'Use case'} {uc_id or '?'}: Analytics Technique '{label}' is not in the "
            f"technique registry; the label is kept as written"
        )
        return label

    _CSV_PLACEHOLDER_VALUES = frozenset({"placeholder", "none", "null", "n/a", "tbd", "todo", "-"})
    _USE_CASE_TYPES = ("Problem", "Risk", "Opportunity", "Improvement")

    def _generated_row_anchors(self) -> dict:
        anchors = {"type": CSVParser.one_of(self._USE_CASE_TYPES),
                   "Analytics Technique": lambda value: _technique_registry_key(value) is not None}
        known = {self._normalize_table_ref(t) for t in (getattr(self, "global_table_names", None) or ())}

        def known_tables_only(value):
            refs = self._uc_table_refs({"Tables Involved": value})
            return bool(refs) and set(refs) <= known
        if known:
            anchors["Tables Involved"] = known_tables_only
        return anchors

    def _parse_llm_csv_response(self, llm_response: str, log_prefix: str) -> list:
        self.logger.info(f"{log_prefix} Starting robust 12-column CSV parsing (SQL and scoring metrics will be assigned separately)...")
        parsed_rows = []
        _canonical_header = '"No","Name","usecase","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Tables Involved","High Level Design"'
        
        try:
            # Clean response - remove markdown fences if present
            csv_clean = llm_response.strip()
            if csv_clean.startswith('```'):
                csv_clean = re.sub(r'^```[a-z]*\n', '', csv_clean)
                csv_clean = re.sub(r'\n```$', '', csv_clean)
            
            # Find header line (12 columns - Business Domain, Subdomain, SQL, and scoring columns will be calculated in code)
            # Support both quoted and unquoted headers from LLM, with case-insensitive matching
            # Analytics Technique is now generated by LLM as column 5
            # Column 6 MUST be "Statement" (not "Opportunity" or any other name)
            header_pattern_quoted = r'"No","Name","usecase","[Tt]ype","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Tables Involved","High Level Design"(?:\s*,\s*"honesty_score"\s*,\s*"honesty_justification")?'
            header_pattern_unquoted = r'No,Name,usecase,[Tt]ype,Analytics Technique,Statement,Solution,Business Value,Beneficiary,Sponsor,Tables Involved,High Level Design(?:\s*,\s*honesty_score\s*,\s*honesty_justification)?'
            header_match = re.search(header_pattern_quoted, csv_clean, re.IGNORECASE)
            if not header_match:
                header_match = re.search(header_pattern_unquoted, csv_clean, re.IGNORECASE)
            
            # Fallback: Auto-correct common LLM column name variations
            if not header_match:
                _HEADER_CORRECTIONS = [
                    ('"Opportunity"', '"Statement"'),
                    (',Opportunity,', ',Statement,'),
                    ('"Opportunity Statement"', '"Statement"'),
                    (',Opportunity Statement,', ',Statement,'),
                    ('"Problem Statement"', '"Statement"'),
                    (',Problem Statement,', ',Statement,'),
                    ('"Description"', '"Statement"'),
                    (',Description,', ',Statement,'),
                    ('"Use Case Name"', '"Name"'),
                    (',Use Case Name,', ',Name,'),
                    ('"Title"', '"Name"'),
                    (',Title,', ',Name,'),
                    ('"Use Case Type"', '"type"'),
                    (',Use Case Type,', ',type,'),
                    ('"Category"', '"type"'),
                    (',Category,', ',type,'),
                    ('"Type"', '"type"'),
                    ('"Technique"', '"Analytics Technique"'),
                    (',Technique,', ',Analytics Technique,'),
                    ('"Tables"', '"Tables Involved"'),
                    (',Tables,', ',Tables Involved,'),
                    ('"Tables Used"', '"Tables Involved"'),
                    (',Tables Used,', ',Tables Involved,'),
                    ('"Technical Approach"', '"High Level Design"'),
                    (',Technical Approach,', ',High Level Design,'),
                    ('"Design"', '"High Level Design"'),
                ]
                applied_corrections = []
                for old_val, new_val in _HEADER_CORRECTIONS:
                    if old_val in csv_clean:
                        csv_clean = csv_clean.replace(old_val, new_val)
                        applied_corrections.append(f"{old_val} -> {new_val}")
                if applied_corrections:
                    self.logger.info(f"{log_prefix} Auto-corrected {len(applied_corrections)} column name(s): {'; '.join(applied_corrections[:5])}")
                    header_match = re.search(header_pattern_quoted, csv_clean, re.IGNORECASE)
                    if not header_match:
                        header_match = re.search(header_pattern_unquoted, csv_clean, re.IGNORECASE)
            
            # Final fallback: Try simpler pattern if exact match still fails
            # Search ALL matches and validate each one (the first match may be prompt echoes or corrupted data)
            if not header_match:
                simpler_pattern = r'(?:"No"|No)\s*,\s*(?:"Name"|Name)\s*,\s*(?:"[Tt]ype"|[Tt]ype)'
                canonical_col_count = 12
                _CANONICAL_FIELDS = ["No", "Name", "usecase", "type", "Analytics Technique", "Statement",
                                     "Solution", "Business Value", "Beneficiary", "Sponsor",
                                     "Tables Involved", "High Level Design"]
                _REMAP_ALIASES = {
                    "opportunity": "Statement", "problem statement": "Statement",
                    "opportunity statement": "Statement", "description": "Statement",
                    "use case": "Name", "use case name": "Name", "title": "Name",
                    "category": "type", "use case type": "type", "type": "type",
                    "technique": "Analytics Technique", "method": "Analytics Technique",
                    "tables": "Tables Involved", "tables used": "Tables Involved",
                    "technical approach": "High Level Design", "design": "High Level Design",
                    "value": "Business Value", "impact": "Business Value",
                    "improvement": "Statement",
                }
                
                for candidate_match in re.finditer(simpler_pattern, csv_clean, re.IGNORECASE):
                    match_pos = candidate_match.start()
                    line_start = csv_clean.rfind('\n', 0, match_pos) + 1
                    line_end = csv_clean.find('\n', match_pos)
                    if line_end == -1:
                        line_end = len(csv_clean)
                    original_header = csv_clean[line_start:line_end].strip().strip('`')
                    
                    try:
                        parsed_fields = next(csv.reader(io.StringIO(original_header)))
                    except Exception:
                        continue
                    
                    actual_fields = [f.strip().strip('"') for f in parsed_fields]
                    actual_col_count = len(actual_fields)
                    
                    # VALIDATION: Reject lines that are clearly data rows, not headers
                    # Header fields should be short column names, not full sentences
                    max_field_len = max((len(f) for f in actual_fields), default=0)
                    if max_field_len > 80:
                        self.logger.warning(f"{log_prefix} Skipping candidate header match (field too long: {max_field_len} chars, likely a data row): {original_header[:150]}")
                        continue
                    
                    # VALIDATION: At least 6 of 12 canonical fields should be recognizable
                    canonical_lower = {c.lower() for c in _CANONICAL_FIELDS}
                    alias_keys = set(_REMAP_ALIASES.keys())
                    recognized_count = sum(
                        1 for f in actual_fields
                        if f in _CANONICAL_FIELDS or f.lower().strip() in canonical_lower or f.lower().strip() in alias_keys
                    )
                    if recognized_count < 6:
                        self.logger.warning(f"{log_prefix} Skipping candidate header match (only {recognized_count}/12 fields recognized): {original_header[:150]}")
                        continue
                    
                    self.logger.info(f"{log_prefix} Using fallback CSV header detection (found simplified pattern, {recognized_count} canonical fields recognized)")
                    
                    # Accept honesty column count (12 or 14) as matching canonical
                    if actual_col_count == canonical_col_count or actual_col_count == canonical_col_count + 2:
                        self.logger.info(f"{log_prefix} Replacing non-standard header ({actual_col_count} cols) with canonical header. Original: {original_header[:200]}")
                        csv_clean = csv_clean[:line_start] + _canonical_header + csv_clean[line_end:]
                        header_match = re.search(re.escape(_canonical_header), csv_clean)
                        break
                    else:
                        self.logger.info(f"{log_prefix} Column count mismatch: LLM produced {actual_col_count} columns, expected {canonical_col_count}. Attempting positional remap.")
                        try:
                            remapped_fields = []
                            _seen_canonical = set()
                            for field in actual_fields:
                                fl = field.lower().strip()
                                mapped = None
                                if fl in _REMAP_ALIASES:
                                    mapped = _REMAP_ALIASES[fl]
                                elif field in _CANONICAL_FIELDS:
                                    mapped = field
                                else:
                                    for canon in _CANONICAL_FIELDS:
                                        cl = canon.lower()
                                        if len(cl) >= 3 and (cl == fl or (cl in fl and len(cl) / len(fl) > 0.5) or (fl in cl and len(fl) / len(cl) > 0.5)):
                                            mapped = canon
                                            break
                                if mapped and mapped in _seen_canonical:
                                    remapped_fields.append(field)
                                else:
                                    if mapped:
                                        _seen_canonical.add(mapped)
                                    remapped_fields.append(mapped if mapped else field)
                            rebuilt_header = ','.join(f'"{f}"' for f in remapped_fields)
                            
                            # Post-remap validation: check that at least 8/12 canonical fields are in the rebuilt header
                            canonical_in_rebuilt = sum(1 for f in remapped_fields if f in _CANONICAL_FIELDS)
                            if canonical_in_rebuilt < 8:
                                self.logger.warning(f"{log_prefix} Remap produced only {canonical_in_rebuilt}/12 canonical fields. Skipping this match: {rebuilt_header[:200]}")
                                continue
                            
                            extra_cols = [f for f in remapped_fields if f not in _CANONICAL_FIELDS and f.lower() not in ('honesty_score', 'honesty_justification')]
                            if extra_cols:
                                self.logger.info(f"{log_prefix} Remapped header ({canonical_in_rebuilt}/12 canonical, {len(extra_cols)} extra cols ignored: {extra_cols}): {rebuilt_header[:200]}")
                            else:
                                self.logger.info(f"{log_prefix} Remapped header ({canonical_in_rebuilt}/12 canonical): {rebuilt_header[:200]}")
                            csv_clean = csv_clean[:line_start] + rebuilt_header + csv_clean[line_end:]
                            header_match = re.search(re.escape(rebuilt_header), csv_clean)
                            break
                        except Exception as remap_err:
                            self.logger.warning(f"{log_prefix} Header remap failed for this match: {remap_err}. Trying next match.")
                            continue
            
            if not header_match:
                # LAST RESORT: No valid header found. Try injecting canonical header
                # before the first line that looks like a quoted data row (starts with ")
                first_data_line = re.search(r'^"[^"]{0,20}"', csv_clean, re.MULTILINE)
                if first_data_line:
                    inject_pos = first_data_line.start()
                    self.logger.warning(f"{log_prefix} No valid CSV header found. Injecting canonical header before first data row at position {inject_pos}.")
                    csv_clean = csv_clean[:inject_pos] + _canonical_header + '\n' + csv_clean[inject_pos:]
                    header_match = re.search(re.escape(_canonical_header), csv_clean)
                
                if not header_match:
                    preview = csv_clean[:500] if csv_clean else "(empty)"
                    self.logger.error(f"{log_prefix} Could not find CSV header in LLM response. Response preview: {preview}")
                    return []
            
            # Extract CSV starting from header
            csv_data = csv_clean[header_match.start():]
            
            # Use centralized CSV parser for robust parsing
            _expected_csv_fields = ["No", "Name", "usecase", "type", "Analytics Technique", "Statement",
                                    "Solution", "Business Value", "Beneficiary", "Sponsor",
                                    "Tables Involved", "High Level Design"]
            csv_rows = CSVParser.parse_csv_string(
                csv_data,
                logger=self.logger,
                context=log_prefix,
                quoting=csv.QUOTE_ALL,
                expected_fields=_expected_csv_fields
            )
            
            # FALLBACK: If standard parser returns 0 valid rows but CSV data exists,
            # try a tolerant line-by-line parser using "," as field delimiter
            if not csv_rows and csv_data.count('\n') > 1:
                self.logger.warning(f"{log_prefix} Standard CSV parser returned 0 rows. Attempting tolerant field-split parser...")
                _data_section = csv_data.split('\n', 1)[1] if '\n' in csv_data else ''
                _tolerant_rows = []
                for _line_num, _raw_line in enumerate(_data_section.split('\n'), start=1):
                    _raw_line = _raw_line.strip()
                    if not _raw_line:
                        continue
                    # Split on "," (field delimiter in QUOTE_ALL format)
                    _fields = _raw_line.split('","')
                    # Clean leading/trailing quotes from first and last fields
                    if _fields:
                        _fields[0] = _fields[0].lstrip('"')
                        _fields[-1] = _fields[-1].rstrip('"')
                    if len(_fields) >= 5:
                        _tolerant_rows.append(CSVParser.row_dict(_expected_csv_fields, _fields))
                if _tolerant_rows:
                    self.logger.info(f"{log_prefix} Tolerant parser recovered {len(_tolerant_rows)} rows")
                    csv_rows = _tolerant_rows
            
            from collections import Counter
            dropped_rows = Counter()
            seen_ids = set()
            row_anchors = self._generated_row_anchors()
            cut_index = len(csv_rows) - 1 if CSVParser.has_open_quote(csv_data) else None
            for row_index, row_dict in enumerate(csv_rows):
                try:
                    # Defensive helper: case-insensitive key lookup with strip
                    def safe_get(d, key):
                        value = d.get(key)
                        if value is None:
                            key_lower = key.lower()
                            for k, v in d.items():
                                if k and k.lower() == key_lower:
                                    value = v
                                    break
                        if value is None:
                            return ''
                        if isinstance(value, str):
                            return value.strip()
                        return str(value).strip()
                    
                    use_case_no = safe_get(row_dict, 'No')
                    valid_no = bool(use_case_no)
                    if not valid_no:
                        self.logger.warning(f"{log_prefix} Skipping row with invalid No field: {use_case_no}")
                        dropped_rows["empty_id"] += 1
                        continue
                    
                    if use_case_no.lower() == "no" and safe_get(row_dict, 'Name').lower() == "name":
                        self.logger.warning(f"{log_prefix} Skipping a repeated CSV header row inside the response")
                        dropped_rows["repeated_header"] += 1
                        continue
                    
                    _honesty_noise = ('honesty', 'self-assessment', 'score:', 'justification')
                    _no_lower = use_case_no.lower()
                    if any(marker in _no_lower for marker in _honesty_noise):
                        self.logger.warning(f"{log_prefix} Skipping non-data row (honesty/meta text in No field): {use_case_no[:80]}")
                        dropped_rows["meta_row"] += 1
                        continue
                    
                    if len(use_case_no) > 40 or re.search(r"\s", use_case_no) or not re.search(r"\d", use_case_no):
                        self.logger.warning(f"{log_prefix} Skipping non-data row (the No field is not a use case id): {use_case_no[:80]}")
                        dropped_rows["non_data_row"] += 1
                        continue
                    
                    _all_values_concat = ' '.join(
                        str(v).lower() for v in row_dict.values() if v
                    )
                    _meta_line_markers = ('honesty_score:', 'honesty_justification:', 'honesty self-assessment', '**honesty self-assessment')
                    if any(m in _all_values_concat for m in _meta_line_markers):
                        _meta_fields = [
                            f"{k}={str(v)[:40]}" for k, v in row_dict.items()
                            if v and any(m in str(v).lower() for m in _meta_line_markers)
                        ]
                        self.logger.warning(f"{log_prefix} Skipping non-data row (honesty/meta text leaked into fields: {', '.join(_meta_fields[:3])})")
                        dropped_rows["meta_row"] += 1
                        continue
                    
                    _identity_fields = (safe_get(row_dict, 'Name'), safe_get(row_dict, 'usecase'))
                    if any(token in field.lower() for field in _identity_fields for token in ('honesty_score', 'honesty_justification')) \
                            or any(field.strip().strip('.').lower() in self._CSV_PLACEHOLDER_VALUES for field in _identity_fields):
                        self.logger.warning(f"{log_prefix} Skipping filler row (meta text or a placeholder in place of a use case): "
                                            f"{use_case_no} {_identity_fields[0][:60]!r}")
                        dropped_rows["meta_row"] += 1
                        continue
                    
                    reason, detail = CSVParser.misaligned_row(row_dict, _expected_csv_fields, row_anchors)
                    if not reason and row_index == cut_index and CSVParser.cut_in_data(row_dict):
                        reason, detail = "cut_final_row", "ends inside an open quoted field"
                    if reason:
                        self.logger.warning(f"[CSV-ROW misaligned] {log_prefix} Skipping row {use_case_no}: it {detail}; "
                                            f"it is treated as missing")
                        dropped_rows[reason] += 1
                        continue
                    
                    analytics_technique = self._canonical_analytics_technique(
                        safe_get(row_dict, 'Analytics Technique'), use_case_no, log_prefix)
                    
                    # Helper to safely parse float scores
                    def safe_float(d, key, default=3.0):
                        try:
                            value = safe_get(d, key)
                            if not value:
                                return default
                            return float(value)
                        except (ValueError, TypeError):
                            self.logger.warning(f"{log_prefix} Invalid float value for {key}: {value}, using default {default}")
                            return default
                    
                    # Scoring will be added by LLM scoring step after deduplication
                    # Initialize with placeholder values that will be replaced
                    strategic_alignment = 0.0
                    return_on_investment = 0.0
                    reusability = 0.0
                    time_to_value = 0.0
                    data_availability = 0.0
                    data_accessibility = 0.0
                    architecture_fitness = 0.0
                    team_skills = 0.0
                    domain_knowledge = 0.0
                    people_allocation = 0.0
                    budget_allocation = 0.0
                    time_to_production = 0.0
                    value_score = 0.0
                    feasibility_score = 0.0
                    priority_score = 0.0
                    quality_label = "Pending"
                    
                    # Build row dictionary with all fields (SQL, Business Domain, and Subdomain will be added later)
                    # Using safe_get to handle None values and type conversions
                    # Column name MUST be "Statement" (auto-corrected above if LLM used wrong name)
                    statement_value = safe_get(row_dict, 'Statement')
                    
                    row = {
                        "No": use_case_no,
                        "Name": safe_get(row_dict, 'Name'),
                        "usecase": self._normalize_usecase(
                            safe_get(row_dict, 'usecase'),
                            safe_get(row_dict, 'Name')
                        ),
                        "Business Domain": "",  # Will be set during domain clustering
                        "Subdomain": "",  # Will be set during subdomain clustering
                        "type": safe_get(row_dict, 'type'),
                        "Analytics Technique": analytics_technique,  # From LLM response
                        "Statement": statement_value,
                        "Solution": safe_get(row_dict, 'Solution'),
                        "Business Value": safe_get(row_dict, 'Business Value'),
                        "Beneficiary": safe_get(row_dict, 'Beneficiary'),
                        "Sponsor": safe_get(row_dict, 'Sponsor'),
                        "Tables Involved": self._repair_tables_involved(safe_get(row_dict, 'Tables Involved'), row_dict, log_prefix),
                        "High Level Design": safe_get(row_dict, 'High Level Design'),
                        # Scoring columns (only for Excel)
                        "Strategic Alignment": strategic_alignment,
                        "Return on Investment": return_on_investment,
                        "Reusability": reusability,
                        "Time to Value": time_to_value,
                        "Data Availability": data_availability,
                        "Data Accessibility": data_accessibility,
                        "Architecture Fitness": architecture_fitness,
                        "Team Skills": team_skills,
                        "Domain Knowledge": domain_knowledge,
                        "People Allocation": people_allocation,
                        "Budget Allocation": budget_allocation,
                        "Time to Production": time_to_production,
                        # Calculated fields
                        "Value": round(value_score, 2),
                        "Feasibility": round(feasibility_score, 2),
                        "Priority": round(priority_score, 2),
                        "Quality": quality_label
                    }
                    
                    # Validate row has minimum required fields
                    if not row['Name'] or not statement_value:
                        self.logger.warning(f"{log_prefix} Skipping row #{use_case_no}: Missing required fields (Name or Statement)")
                        dropped_rows["missing_fields"] += 1
                        continue
                    
                    if use_case_no.lower() in seen_ids:
                        self.logger.warning(f"{log_prefix} Skipping a second row with id {use_case_no}: the first valid row with "
                                            f"this id is kept (the response repeated its rows)")
                        dropped_rows["duplicate_id"] += 1
                        continue
                    seen_ids.add(use_case_no.lower())
                    
                    self.logger.debug(f"{log_prefix} Parsed Scenario #{row['No']}: {row['Name']} [Analytics Technique: {analytics_technique}]")
                    
                    parsed_rows.append(row)
                    
                except Exception as e:
                    # Log error with sanitized row data (limit length to avoid huge logs)
                    try:
                        row_summary = {k: str(v)[:100] for k, v in row_dict.items()} if isinstance(row_dict, dict) else str(row_dict)[:200]
                        self.logger.error(f"{log_prefix} Error processing CSV row: {get_clean_error_message(e)}. Row summary: {row_summary}")
                    except:
                        self.logger.error(f"{log_prefix} Error processing CSV row: {get_clean_error_message(e)}. Could not serialize row data.")
                    continue
                    
        except Exception as e:
            self.logger.error(f"{log_prefix} Failed to parse LLM CSV response: {get_clean_error_message(e)}")
            # Show snippet for debugging
            snippet = llm_response[:500] if llm_response else "Empty response"
            self.logger.error(f"{log_prefix} Response snippet: {snippet}")
            return []
        
        # NOTE: Post-processing for naming conventions removed since AI Function field no longer exists
        # Genie Code will determine the best implementation approach
        
        self.logger.info(f"{log_prefix} Robust parsing complete. Found {len(parsed_rows)} rows."
                         + (f" Dropped {sum(dropped_rows.values())} non-use-case row(s): "
                            + ", ".join(f"{reason}={n}" for reason, n in sorted(dropped_rows.items())) if dropped_rows else ""))
        return parsed_rows


    def _retry_missing_table_coverage(self, use_cases: list, all_columns: list, include_business_catchall: bool = False) -> list:
        """
        Retry use case generation for tables that have no use cases.
        Each table can be retried up to 2 times maximum.
        
        Args:
            use_cases: List of existing use case dictionaries
            all_columns: List of all column details (catalog, schema, table, column, type, comment)
            include_business_catchall: If True, also include BUSINESS tables that were never involved in any use cases (catch-all mode)
            
        Returns:
            List of newly generated use cases for missing tables
        """
        from collections import defaultdict
        
        def _table_key(name: str) -> str:
            return str(name or "").replace('`', '').strip().lower()

        # Extract all tables from column details
        all_tables = set()
        table_columns = defaultdict(list)
        for col_tuple in all_columns:
            catalog, schema, table, column, col_type, comment = col_tuple
            fq_table = f"{catalog}.{schema}.{table}"
            all_tables.add(fq_table)
            table_columns[fq_table].append(col_tuple)
        
        # Extract tables that have use cases (INCLUDING those with empty tables field)
        tables_with_use_cases = set()
        for uc in use_cases:
            tables_str = uc.get('Tables Involved', '')
            if tables_str and not tables_str.startswith('/Volumes'):
                for table in tables_str.split(','):
                    table = _table_key(table)
                    if table:
                        tables_with_use_cases.add(table)
        
        # Find tables without use cases
        missing_tables = {t for t in all_tables if _table_key(t) not in tables_with_use_cases}
        
        # === CATCH-ALL: Include BUSINESS tables that were never involved in any use cases ===
        if include_business_catchall and hasattr(self, 'business_scores'):
            self.logger.info("🔍 CATCH-ALL MODE: Checking for BUSINESS tables that were never involved in use cases...")
            
            all_business_keys = {_table_key(fqtn) for fqtn, score in self.business_scores.items() if score > 0}
            unused_business_tables = {
                t for t in all_tables
                if _table_key(t) in all_business_keys and _table_key(t) not in tables_with_use_cases
            }
            
            if unused_business_tables:
                self.logger.warning(f"⚠️ Found {len(unused_business_tables)} BUSINESS tables that were never involved in any use cases")
                missing_tables = missing_tables.union(unused_business_tables)
                unused_sample = sorted(list(unused_business_tables))[:10]
                self.logger.info(f"📋 Sample unused BUSINESS tables: {', '.join(unused_sample)}{'...' if len(unused_business_tables) > 10 else ''}")
            else:
                self.logger.info("✅ All BUSINESS tables have been involved in use cases")
        
        # === JOIN-ELIGIBILITY FILTER FOR MASTER/REFERENCE TABLES ===
        _data_cat = {
            _fqtn.lower(): (_cat or '').upper()
            for _fqtn, _cat in (getattr(self, 'data_category_map', {}) or {}).items()
        }
        _tx_col_index = defaultdict(set)
        for _fqtn_key, _col_list in table_columns.items():
            if _data_cat.get(_fqtn_key.lower(), '') == 'TRANSACTIONAL':
                for _col_tuple in _col_list:
                    _cn = _col_tuple[3] if len(_col_tuple) > 3 else None
                    if _cn:
                        _tx_col_index[_cn.lower()].add(_fqtn_key)
        
        _tx_tables = sorted(t for t in table_columns if _data_cat.get(t.lower(), '') == 'TRANSACTIONAL')
        _table_by_lower = {t.lower(): t for t in table_columns}
        _bridge_tables = defaultdict(set)

        def _compute_join_partners(master_fqtn: str) -> set:
            partners = set()
            for _col_tuple in table_columns.get(master_fqtn, []):
                _cn = _col_tuple[3] if len(_col_tuple) > 3 else None
                if _cn and _cn.lower() in _tx_col_index:
                    partners |= _tx_col_index[_cn.lower()]
            _tbl_basename = master_fqtn.split('.')[-1].lower()
            _singular = _tbl_basename[:-1] if _tbl_basename.endswith('s') else _tbl_basename
            for _base in {_tbl_basename, _singular}:
                for _sfx in _KEY_NAME_SUFFIXES:
                    _candidate = _base + _sfx
                    if _candidate in _tx_col_index:
                        partners |= _tx_col_index[_candidate]
            for _tx in _tx_tables:
                if _tx in partners or _tx == master_fqtn:
                    continue
                _joined = self._find_join_path([master_fqtn, _tx])
                if _joined.get("connected") and _joined.get("strength") in ("declared", "inferred"):
                    partners.add(_tx)
                    for _on_path in self._join_path_tables(_joined.get("path")):
                        _bridge = _table_by_lower.get(_on_path)
                        if _bridge and _bridge not in (master_fqtn, _tx):
                            _bridge_tables[master_fqtn].add(_bridge)
            partners.discard(master_fqtn)
            return partners
        
        _master_join_partners = {}
        _dropped_non_joinable = []
        _filtered_missing = set()
        for _fqtn in missing_tables:
            _cat_label = _data_cat.get(_fqtn.lower(), 'TRANSACTIONAL')
            if _cat_label in ('MASTER', 'REFERENCE'):
                _partners = _compute_join_partners(_fqtn)
                if _partners:
                    _master_join_partners[_fqtn] = _partners
                    _filtered_missing.add(_fqtn)
                else:
                    _dropped_non_joinable.append(_fqtn)
            else:
                _filtered_missing.add(_fqtn)
        
        if _dropped_non_joinable:
            _sample_drop = sorted(_dropped_non_joinable)[:10]
            self.logger.info(
                f"🚫 Catch-all JOIN filter: excluded {len(_dropped_non_joinable)} MASTER/REFERENCE table(s) "
                f"with no foreign key path and no column-name overlap to any TRANSACTIONAL table "
                f"(sample: {', '.join(_sample_drop)}{'...' if len(_dropped_non_joinable) > 10 else ''})"
            )
        if _master_join_partners:
            self.logger.info(
                f"🔗 Catch-all JOIN filter: retained {len(_master_join_partners)} MASTER/REFERENCE table(s) "
                f"that CAN join at least one TRANSACTIONAL table — catch-all prompt will enforce JOIN requirement"
            )
        
        missing_tables = _filtered_missing
        
        if not missing_tables:
            self.logger.info("✅ All tables have at least one use case - no retry needed")
            return []
        
        coverage_percentage = ((len(all_tables) - len(missing_tables)) / len(all_tables)) * 100 if all_tables else 0
        self.logger.warning(f"⚠️ Found {len(missing_tables)} tables without use cases (out of {len(all_tables)} total tables - {coverage_percentage:.1f}% coverage)")
        
        # Show sample of missing tables
        missing_sample = sorted(list(missing_tables))[:10]
        self.logger.info(f"📋 Sample missing tables: {', '.join(missing_sample)}{'...' if len(missing_tables) > 10 else ''}")
        
        # Provide actionable insights
        if len(missing_tables) > len(all_tables) * 0.5:
            self.logger.warning(f"⚠️ More than 50% of tables lack use cases. Consider:")
            self.logger.warning(f"   - Checking if LLM is generating use cases for all tables")
            self.logger.warning(f"   - Verifying table names match between schema and use case generation")
            self.logger.warning(f"   - Reviewing business vs technical table filtering")
        
        # Track retry attempts per table (max 2 attempts)
        if not hasattr(self, '_table_retry_counts'):
            self._table_retry_counts = defaultdict(int)
        
        # Filter tables that haven't exceeded retry limit
        tables_to_retry = []
        for table in missing_tables:
            if self._table_retry_counts[table] < 2:
                tables_to_retry.append(table)
                self._table_retry_counts[table] += 1
            else:
                self.logger.warning(f"⚠️ Table {table} has been retried 2 times already - skipping")
        
        if not tables_to_retry:
            self.logger.info("No tables eligible for retry (all have reached 2 attempts)")
            return []
        
        self.logger.info(f"🔄 Retrying use case generation for {len(tables_to_retry)} tables...")
        
        # Group tables into batches (max 50 tables per batch to avoid context overflow)
        max_tables_per_batch = 50
        retry_batches = []
        for i in range(0, len(tables_to_retry), max_tables_per_batch):
            batch_tables = tables_to_retry[i:i+max_tables_per_batch]
            batch_columns = []
            for table in batch_tables:
                batch_columns.extend(table_columns[table])
            retry_batches.append((batch_tables, batch_columns))
        
        self.logger.info(f"📦 Created {len(retry_batches)} retry batch(es) for {len(tables_to_retry)} tables")
        
        # Process retry batches IN PARALLEL using centralized ParallelExecutor
        all_retry_use_cases = []
        
        # ADAPTIVE PARALLELISM: Calculate based on retry batches and columns
        total_retry_columns = sum(len(cols) for _, cols in retry_batches)
        
        retry_parallelism, reason = calculate_adaptive_parallelism(
            "use_case_generation", self.max_parallelism,
            num_items=len(retry_batches),
            total_columns=total_retry_columns,
            avg_prompt_chars=total_retry_columns * 100,
            is_llm_operation=True, logger=self.logger
        )
        log_adaptive_parallelism_decision("use_case_generation", retry_parallelism, self.max_parallelism, reason)
        
        self.logger.info(f"🔄 Processing {len(retry_batches)} retry batch(es) in parallel...")
        
        _ctx = getattr(self, 'merged_business_context', None) or {}
        _retry_bc = _ctx.get("business_context", "") or ""
        _retry_si = _ctx.get("strategic_initiative", "") or ""
        _retry_vc = _ctx.get("value_chain", "") or ""
        _retry_rm = _ctx.get("revenue_model", "") or ""
        _retry_ind = _ctx.get("industry", "") or ""

        tasks = []
        for batch_idx, (batch_tables, batch_columns) in enumerate(retry_batches, 1):
            _batch_master_tables = sorted(
                t for t in batch_tables
                if _data_cat.get(t.lower(), '') in ('MASTER', 'REFERENCE')
            )
            _catchall_instruction = ""
            if _batch_master_tables:
                _needed_tx, _needed_bridges = set(), set()
                for _mt in _batch_master_tables:
                    _needed_tx |= _master_join_partners.get(_mt, set())
                    _needed_bridges |= _bridge_tables.get(_mt, set())
                _needed_bridges -= set(batch_tables)
                _existing_col_keys = {
                    (f"{_c[0]}.{_c[1]}.{_c[2]}", _c[3]) for _c in batch_columns if len(_c) > 3
                }
                _extra_cols = []
                for _tx_fqtn in sorted(_needed_tx | _needed_bridges):
                    for _col_tuple in table_columns.get(_tx_fqtn, []):
                        if len(_col_tuple) <= 3:
                            continue
                        _key = (f"{_col_tuple[0]}.{_col_tuple[1]}.{_col_tuple[2]}", _col_tuple[3])
                        if _key not in _existing_col_keys:
                            _extra_cols.append(_col_tuple)
                            _existing_col_keys.add(_key)
                if _extra_cols:
                    batch_columns = batch_columns + _extra_cols
                _master_list_str = ', '.join(_batch_master_tables)
                _tx_list_str = ', '.join(sorted(_needed_tx)) if _needed_tx else '(none found)'
                _catchall_instruction = (
                    "CATCH-ALL COVERAGE MODE (NON-NEGOTIABLE ADDITIONAL RULES):\n"
                    f"- The following tables in this batch are MASTER/REFERENCE (static entity/lookup data): {_master_list_str}\n"
                    f"- The valid TRANSACTIONAL join partners available in the schema below are: {_tx_list_str}\n"
                    + (f"- These tables connect a MASTER/REFERENCE table above to those partners; join through them where the "
                       f"foreign keys below require it: {', '.join(sorted(_needed_bridges))}\n" if _needed_bridges else "") +
                    "- EVERY use case you generate that references any MASTER/REFERENCE table above MUST explicitly JOIN at least one of those TRANSACTIONAL tables. The core analytical question MUST be about EVENTS over time (what HAPPENED), never about static entity attributes.\n"
                    "- If a MASTER/REFERENCE table has no meaningful JOIN path to the listed TRANSACTIONAL partners for a given analytical question, SKIP that table; do NOT generate a use case for it.\n"
                    "- The TRANSACTIONAL partner tables listed above are provided ONLY as JOIN context; do NOT produce new use cases whose primary table is one of those partners unless the analytical question is materially different from any use case already produced in prior passes.\n"
                )
            def _retry_task(_cols=batch_columns, _bid=f"RETRY_{batch_idx}", _instr=_catchall_instruction):
                return self._process_batch_with_retry(
                    _cols, _bid,
                    business_context=_retry_bc,
                    strategic_initiative=_retry_si,
                    value_chain=_retry_vc,
                    revenue_model=_retry_rm,
                    industry=_retry_ind,
                    max_attempts=2,
                    previous_use_cases_feedback=_instr
                )
            tasks.append(_retry_task)
            _master_suffix = f" (includes {len(_batch_master_tables)} master/ref table(s) with JOIN enforcement)" if _batch_master_tables else ""
            self.logger.info(f"✓ Prepared retry batch {batch_idx}/{len(retry_batches)} ({len(batch_tables)} tables){_master_suffix}")
        
        # Execute in parallel with centralized utility
        results = ParallelExecutor.execute_parallel(
            tasks=tasks,
            max_workers=retry_parallelism,
            task_name="Retry Batch",
            logger=self.logger,
            thread_name_prefix="RetryBatch",
            return_exceptions=True
        )
        
        # Collect successful results
        for batch_idx, result in enumerate(results, 1):
            if isinstance(result, Exception):
                self.logger.error(f"❌ Retry batch {batch_idx} failed: {result}")
                continue
            if result:
                self.logger.info(f"✅ Retry batch {batch_idx}: Generated {len(result)} use cases")
                all_retry_use_cases.extend(result)
            else:
                self.logger.warning(f"⚠️ Retry batch {batch_idx}: No use cases generated")
        
        if all_retry_use_cases:
            self.logger.info(f"✅ Retry complete: Generated {len(all_retry_use_cases)} additional use cases")
        else:
            self.logger.warning("⚠️ Retry complete: No additional use cases generated")
        
        return all_retry_use_cases

    def _build_generation_rejection_feedback(self, rejected_use_cases: list) -> str:
        """Retry feedback for a generation answer that produced no valid use case. Names every
        rejection reason seen (hallucinated table, reference-only, bad table format, empty or
        SQL-polluted Tables Involved) so the retry prompt differs from the rejected one."""
        bad_tables = set()
        reason_classes = set()
        for uc in rejected_use_cases or []:
            reason = str(uc.get('hallucination_reason', '') or '')
            if reason.startswith('Tables not found'):
                reason_classes.add('missing')
                for t in reason.replace('Tables not found in schema: ', '').split(', '):
                    if t.strip():
                        bad_tables.add(t.strip())
            elif reason.startswith('Reference-only'):
                reason_classes.add('reference_only')
            elif reason.startswith('Invalid table format'):
                reason_classes.add('format')
            elif reason.startswith('No tables specified'):
                reason_classes.add('empty')
            elif reason.startswith('SQL fragment'):
                reason_classes.add('sql_fragment')
            else:
                reason_classes.add('other')
        with self.registry_lock:
            cross_batch_tracker = getattr(self, '_known_hallucinated_tables', None)
            if cross_batch_tracker is not None:
                cross_batch_tracker.update(bad_tables)
                bad_tables.update(cross_batch_tracker)
        lines = [
            "\n\n### 🚨 PREVIOUS ATTEMPT REJECTED: NO VALID USE CASES 🚨",
            "Your previous response was REJECTED. Fix every problem listed below in this answer:",
        ]
        if bad_tables:
            lines.append("- These table names DO NOT EXIST in the schema (hallucinations):")
            lines.extend(f"  - ❌ `{t}`: DOES NOT EXIST" for t in sorted(bad_tables))
        if 'reference_only' in reason_classes:
            lines.append("- Use cases were built only on REFERENCE tables. Every use case MUST be centered on at least one PRIMARY (TRANSACTIONAL) table from the registry; reference tables may only be joined for lookups.")
        if 'format' in reason_classes:
            lines.append("- \"Tables Involved\" must list fully-qualified catalog.schema.table names separated by commas.")
        if 'empty' in reason_classes:
            lines.append("- \"Tables Involved\" was empty. Every use case must list the tables it reads.")
        if 'sql_fragment' in reason_classes:
            lines.append("- \"Tables Involved\" contained SQL text. It must contain only comma-separated catalog.schema.table names; put the logic in High Level Design and quote every CSV field.")
        if 'other' in reason_classes:
            lines.append("- Use cases referenced tables that could not be validated against the registry.")
        lines.append("You MUST only use tables from the ALLOWED TABLES REGISTRY above. Double-check EVERY table name against the registry before including it.\n")
        return "\n".join(lines)

    def _generation_prompt_base_size(self, context_texts=()) -> int:
        return (self._prompt_overhead_chars("BASE_USE_CASE_GEN_PROMPT") + len(QUALITY_GATE_HIGH)
                + sum(len(str(text or "")) for text in context_texts))

    def _process_batch_with_retry(self, column_details: list, batch_num, business_context: str = "", strategic_initiative: str = "", value_chain: str = "", revenue_model: str = "", max_attempts: int = 3, previous_use_cases_feedback: str = "", temperature_override: float = None, industry: str = "") -> list:
        """
        Process a batch of column details to generate use cases with retry logic.
        Automatically splits context if input is too long for the model.
        
        STRATEGY: When tables don't fit in context:
        1. Split tables across multiple sub-batches (NEVER drop business tables)
        2. Process ALL sub-batches recursively
        3. Track which columns are kept from each table (saved to disk, not memory)
        4. Column tracking is loaded from disk during Genie Code generation
        
        Every retry re-queries the LLM with the cache bypassed; a retry after a rejected answer
        also carries feedback naming why the previous answer was rejected.
        
        Args:
            column_details: List of column tuples (catalog, schema, table, column, type, comment)
            batch_num: Batch number for logging and prefixing (can be int or str)
            max_attempts: Maximum number of attempts (default 3)
            
        Returns:
            List of use case dictionaries (includes results from sub-batches)
        """
        log_prefix = f"[Batch {batch_num}]"
        business_context = business_context or self._get_business_context_fallback()
        industry = prompt_context_or_not_provided(industry, "industry")
        strategic_initiative = prompt_context_or_not_provided(strategic_initiative, "strategic initiative")
        value_chain = prompt_context_or_not_provided(value_chain, "value chain")
        revenue_model = prompt_context_or_not_provided(revenue_model, "revenue model")
        previous_use_cases_feedback = previous_use_cases_feedback or ""
        sub_batch_results = []
        
        # === NEW: Register columns and tables for Bitmap ID generation ===
        with self.registry_lock:
            for col_tuple in column_details:
                # col_tuple: (catalog, schema, table, column, type, comment)
                fqn = f"{col_tuple[0]}.{col_tuple[1]}.{col_tuple[2]}.{col_tuple[3]}"
                table_fqn = f"{col_tuple[0]}.{col_tuple[1]}.{col_tuple[2]}"
                
                # Register table if not already registered
                if table_fqn not in self.table_id_map:
                    table_id = str(self.next_table_id)
                    self.next_table_id += 1
                    self.table_id_map[table_fqn] = table_id
                    self.id_table_map[table_id] = table_fqn
                
                if fqn not in self.column_id_map:
                    col_id = str(self.next_column_id)
                    self.next_column_id += 1
                    self.column_id_map[fqn] = col_id
                    
                    # Create description (Type + Comment)
                    desc = f"{col_tuple[4]}"
                    if col_tuple[5]:
                        desc += f" - {col_tuple[5]}"
                    
                    self.id_column_map[col_id] = {
                        "fqn": fqn,
                        "description": desc
                    }
        
        current_column_details = column_details

        safe_limit = get_safe_context_limit(language="English", buffer_percent=0.9, prompt_name="BASE_USE_CASE_GEN_PROMPT")
        if self.user_strategic_goals:
            goals_text = "\n".join([f"- {goal}" for goal in self.user_strategic_goals])
            additional_context_section = f"""**USER GOAL FROM THE GENERATION INSTRUCTIONS (HIGHEST PRIORITY)**:

The user's Generation Instructions set this goal as the scope of the portfolio. It MUST be followed during generation.

**USER GOAL:**
{goals_text}

**REQUIREMENTS**:
- Generate ONLY use cases that serve this goal.
- Generate EVERY possible use case that serves this goal. Do not omit any valid use case.
- Do NOT cap the number of use cases; completeness is mandatory.
- Use semantic understanding of the goal; do NOT apply rigid keyword rules.
- Do NOT generate use cases outside this goal."""
        else:
            additional_context_section = "*(No goal set in the Generation Instructions - proceed with standard business analysis)*"
        if self.user_business_domains:
            domains_list = ", ".join([f'"{domain}"' for domain in self.user_business_domains])
            focus_areas_instruction = f"""  - **🚨 CRITICAL - USER-SPECIFIED BUSINESS DOMAINS 🚨**: You MUST assign use cases ONLY to the following business domains: {domains_list}. 
   * These are the ONLY valid Business Domain values - DO NOT invent new domains.
   * ALL use cases MUST be categorized into one of these exact domains.
   * DO NOT create any domain that is not in this list.
   * DO NOT modify, abbreviate, or expand these domain names - use them EXACTLY as provided.
   * The Business Domain field MUST exactly match one of these domains."""
        else:
            focus_areas_instruction = ""
        base_prompt_size = self._generation_prompt_base_size((
            business_context, industry, strategic_initiative, value_chain, revenue_model,
            additional_context_section, focus_areas_instruction, previous_use_cases_feedback)) + 1000
        
        self.logger.info(f"{log_prefix} Starting batch processing with {len(column_details)} columns from {len(set([c[2] for c in column_details]))} tables")
        all_tables_in_call = sorted({f"{c[0]}.{c[1]}.{c[2]}" for c in column_details})
        _dc_map = getattr(self, "data_category_map", {})
        _tx_tables = [t for t in all_tables_in_call if _dc_map.get(t) == 'TRANSACTIONAL']
        _ctx_tables = [t for t in all_tables_in_call if t not in set(_tx_tables)]
        self.logger.info(f"{log_prefix} PRIMARY (transactional) tables ({len(_tx_tables)}): {', '.join(_tx_tables) if _tx_tables else 'NONE'}")
        self.logger.info(f"{log_prefix} CONTEXT (master/ref) tables ({len(_ctx_tables)}): {', '.join(_ctx_tables) if _ctx_tables else 'NONE'}")
        log_print(f"{log_prefix} PRIMARY tables ({len(_tx_tables)}): {', '.join(_tx_tables) if _tx_tables else 'NONE'}")
        if _ctx_tables:
            log_print(f"{log_prefix} CONTEXT tables ({len(_ctx_tables)}): {', '.join(_ctx_tables)}")
        
        _hallucination_feedback = ""
        for attempt in range(1, max_attempts + 1):
            try:
                if attempt > 1:
                    self.logger.info(f"{log_prefix} Retry attempt {attempt}/{max_attempts}...")

                estimated_schema_size = self._estimate_schema_markdown_size(current_column_details)
                estimated_prompt_size = base_prompt_size + estimated_schema_size
                if estimated_prompt_size > safe_limit:
                    raise InputTooLongError(
                        f"Proactive split: Input length {estimated_prompt_size:,} characters exceeds "
                        f"safe limit of {safe_limit:,} (with 10% buffer)"
                    )

                self.logger.debug(f"{log_prefix} Formatting schema for prompt...")
                schema_markdown = self._format_schema_for_prompt(current_column_details)
                if not schema_markdown:
                    self.logger.warning(f"{log_prefix} Produced no schema markdown. Skipping.")
                    return []
                
                fk_relationships_text = "None"
                related_tables_added = 0
                additional_related_columns = []
                _fk_source = "unknown"
                try:
                    if self.data_loader and getattr(self.data_loader, "foreign_key_graph", None):
                        batch_tables = {(c[0], c[1], c[2]) for c in current_column_details}
                        fk_relations = self.data_loader.get_foreign_key_relations(batch_tables)
                        if fk_relations:
                            rel_lines = []
                            related_tables_to_load = set()
                            
                            for rel in fk_relations:
                                src = f"{rel[0]}.{rel[1]}.{rel[2]}.{rel[3]}"
                                ref_catalog = rel[4] or rel[0]
                                ref_schema = rel[5] or rel[1]
                                ref_table = rel[6]
                                tgt = f"{ref_catalog}.{ref_schema}.{ref_table}.{rel[7]}"
                                rel_lines.append(f"{src} -> {tgt}")
                                
                                ref_tuple = (ref_catalog, ref_schema, ref_table)
                                if ref_tuple not in batch_tables:
                                    related_tables_to_load.add(ref_tuple)
                            
                            if rel_lines:
                                fk_relationships_text = "\n".join(sorted(set(rel_lines)))
                            
                            if related_tables_to_load:
                                self.logger.info(f"{log_prefix} Loading {len(related_tables_to_load)} FK-related tables to include in schema context...")
                                additional_related_columns = []
                                for (ref_cat, ref_sch, ref_tbl) in list(related_tables_to_load)[:10]:
                                    try:
                                        ref_cols = self.data_loader._get_table_details(
                                            ref_cat, ref_sch, ref_tbl,
                                            apply_sampling=self.data_loader.enable_column_sampling if hasattr(self.data_loader, 'enable_column_sampling') else False
                                        )
                                        if ref_cols:
                                            additional_related_columns.extend(ref_cols)
                                            related_tables_added += 1
                                            self.logger.debug(f"{log_prefix} Added FK-related table: {ref_cat}.{ref_sch}.{ref_tbl} ({len(ref_cols)} columns)")
                                    except Exception as e:
                                        self.logger.debug(f"{log_prefix} Could not load FK-related table {ref_cat}.{ref_sch}.{ref_tbl}: {str(e)[:80]}")
                                
                                if additional_related_columns:
                                    related_schema = self._format_schema_for_prompt(additional_related_columns)
                                    if related_schema:
                                        schema_markdown = schema_markdown + "\n\n<!-- FK-Related Tables -->\n" + related_schema
                                        self.logger.info(f"{log_prefix} Added {related_tables_added} FK-related tables to schema context ({len(additional_related_columns)} columns)")
                except Exception as fk_err:
                    self.logger.debug(f"{log_prefix} Failed to gather FK relationships: {str(fk_err)[:100]}")
                
                data_cat_map = getattr(self, "data_category_map", {})
                all_batch_tables = sorted({f"{c[0]}.{c[1]}.{c[2]}" for c in current_column_details})
                
                primary_tables = [t for t in all_batch_tables if data_cat_map.get(t) == 'TRANSACTIONAL']
                if not primary_tables:
                    primary_tables = all_batch_tables
                primary_set = set(primary_tables)
                
                context_tables = sorted(t for t in all_batch_tables if t not in primary_set)
                
                reference_tables_in_cat = {k for k, v in data_cat_map.items() if v == "REFERENCE"}
                master_tables_in_cat = {k for k, v in data_cat_map.items() if v == "MASTER"}
                
                fk_only_tables = set()
                if additional_related_columns:
                    fk_only_tables = {f"{c[0]}.{c[1]}.{c[2]}" for c in additional_related_columns} - set(all_batch_tables)
                
                table_lines = []
                for t in primary_tables:
                    table_lines.append(f"- `{t}` (PRIMARY: TRANSACTIONAL, generate use cases FROM this table)")
                for t in context_tables:
                    if t in reference_tables_in_cat:
                        table_lines.append(f"- `{t}` (CONTEXT: REFERENCE DATA, use ONLY for JOINs and lookups, NEVER as the main analytical subject)")
                    elif t in master_tables_in_cat:
                        table_lines.append(f"- `{t}` (CONTEXT: MASTER DATA, use for JOINs and enrichment, but analytical question must come from transactional events)")
                    else:
                        table_lines.append(f"- `{t}` (CONTEXT: available for JOINs and enrichment)")
                for t in sorted(fk_only_tables):
                    if t in reference_tables_in_cat:
                        table_lines.append(f"- `{t}` (FK-RELATED, REFERENCE: use ONLY for JOINs, NEVER as sole table)")
                    elif t in master_tables_in_cat:
                        table_lines.append(f"- `{t}` (FK-RELATED, MASTER: use for JOINs and enrichment only)")
                    else:
                        table_lines.append(f"- `{t}` (FK-RELATED: available for JOINs)")
                
                all_allowed_tables = sorted(set(all_batch_tables) | fk_only_tables)
                with self.registry_lock:
                    _known_bad = set(getattr(self, '_known_hallucinated_tables', None) or set())
                _known_bad_block = ""
                if _known_bad:
                    _known_bad_block = (
                        "\n\n**⛔ KNOWN HALLUCINATED TABLES (from other batches; DO NOT USE):**\n"
                        + "\n".join(f"- ❌ `{t}`: DOES NOT EXIST" for t in sorted(_known_bad))
                        + "\n"
                    )
                allowed_tables_block = (
                    "\n\n### 🚨 ALLOWED TABLES REGISTRY (EXHAUSTIVE LIST) 🚨\n"
                    "**You may ONLY use tables from this list. ANY other table name is HALLUCINATION and will be REJECTED.**\n"
                    "**Every use case MUST be centered on at least one PRIMARY (TRANSACTIONAL) table.**\n"
                    "**CONTEXT and FK-RELATED tables provide enrichment via JOINs but must NEVER be the sole analytical subject.**\n"
                    "**The analytical question must always derive from EVENTS in transactional data, not from static entity attributes.**\n\n"
                    + "\n".join(table_lines)
                    + "\n\n**Total allowed tables: " + str(len(all_allowed_tables)) + ". NO OTHER TABLES EXIST.**\n"
                    + _known_bad_block
                )
                schema_markdown = schema_markdown + allowed_tables_block
                _evidence_block, _evidence_joins = self._evidence_prompt_block(all_allowed_tables)
                if _evidence_block:
                    schema_markdown = schema_markdown + _evidence_block
                
                _effective_additional_context = additional_context_section
                if _hallucination_feedback:
                    _effective_additional_context = additional_context_section + _hallucination_feedback
                
                # Structural-name FK fallback (industry-agnostic): when Unity Catalog has no declared FKs,
                # scan current_column_details for shared column names across tables. If the SAME column
                # name appears in >=2 tables AND looks like an identifier (ends in _id / _code / _key / Id
                # / starts with id_, or contains "_id_"), treat as inferred join key and emit a hint line.
                try:
                    if fk_relationships_text == "None" and current_column_details:
                        _by_col = {}
                        for _row in current_column_details:
                            try:
                                if isinstance(_row, (tuple, list)) and len(_row) >= 4:
                                    _cat, _sch, _tbl, _col = _row[0], _row[1], _row[2], _row[3]
                                else:
                                    continue
                                _cn = str(_col or "").strip()
                                if not _cn:
                                    continue
                                _key = _cn.lower()
                                _fqt = f"{_cat}.{_sch}.{_tbl}"
                                _by_col.setdefault(_key, set()).add(_fqt)
                            except Exception:
                                continue
                        _lines = []
                        def _looks_like_id(name: str) -> bool:
                            n = name.lower()
                            return (n.endswith("_id") or n.endswith("id") or n.endswith("_code")
                                    or n.endswith("_key") or n.endswith("_sk") or n.endswith("_nk")
                                    or n.startswith("id_") or n.startswith("sk_") or "_id_" in n
                                    or n in {"id", "key", "code"})
                        # Pass 1: ID-like columns (prefer these — highest-confidence joins)
                        for _cname, _tables in _by_col.items():
                            if len(_tables) >= 2 and _looks_like_id(_cname):
                                _t_list = sorted(_tables)
                                _lines.append(f"[INFERRED JOIN KEY] `{_cname}` appears in: {', '.join(_t_list)}")
                        # Pass 2: if Pass 1 produced nothing, fall back to ANY column shared across 2+
                        # tables. Accuweather-style schemas share value columns (city_name,
                        # datetime_valid_local, etc.) that legitimately act as join keys without
                        # matching ID-suffix conventions. Structural — no hardcoded column names.
                        if not _lines:
                            _shared = [(n, sorted(t)) for n, t in _by_col.items() if len(t) >= 2 and n]
                            # sort by (fewer tables first then name) so we emit the most specific joins first
                            _shared.sort(key=lambda x: (len(x[1]), x[0]))
                            for _cname, _t_list in _shared[:40]:
                                _lines.append(f"[INFERRED JOIN KEY] `{_cname}` appears in: {', '.join(_t_list)}")
                        if _lines:
                            fk_relationships_text = "\n".join(_lines[:40])
                            _fk_source = "structural-name FK fallback"
                            try:
                                self.logger.info(f"{log_prefix} No declared FKs; inferred {len(_lines)} shared-column join key(s) structurally")
                            except Exception:
                                pass
                except Exception as _fk_err:
                    try:
                        self.logger.debug(f"{log_prefix} structural FK fallback skipped: {_fk_err}")
                    except Exception:
                        pass
                if _evidence_joins:
                    _declared = "" if fk_relationships_text in (None, "", "None") else fk_relationships_text
                    fk_relationships_text = "\n".join(part for part in (_declared, "\n".join(_evidence_joins)) if part)

                prompt_vars = {
                    "schema_markdown": schema_markdown,
                    "foreign_key_relationships": fk_relationships_text,
                    "business_context": business_context,
                    "industry": industry,
                    "strategic_initiative": strategic_initiative,
                    "value_chain": value_chain,
                    "revenue_model": revenue_model,
                    "additional_context_section": _effective_additional_context,
                    "focus_areas_instruction": focus_areas_instruction,
                    "previous_use_cases_feedback": previous_use_cases_feedback,
                    "quality_gate_block": QUALITY_GATE_HIGH,
                    "generation_instructions_section": self._get_generation_instruction_for_prompt("BASE_USE_CASE_GEN_PROMPT"),
                }
                
                # PROACTIVE CHECK: Estimate prompt size and split BEFORE attempting LLM call
                # This saves time by not waiting for LLM failures
                try:
                    estimated_prompt_size = base_prompt_size + len(schema_markdown) + len(fk_relationships_text)
                    
                    if estimated_prompt_size > safe_limit:
                        # Prompt exceeds safe limit - proactively split WITHOUT attempting LLM call
                        self.logger.warning(
                            f"{log_prefix} PROACTIVE SPLIT: Prompt size ({estimated_prompt_size:,} chars) exceeds "
                            f"safe limit ({safe_limit:,} chars with 10% buffer). Splitting batch without attempting LLM call."
                        )
                        
                        # Raise InputTooLongError to trigger the split logic below
                        raise InputTooLongError(
                            f"Proactive split: Input length {estimated_prompt_size:,} characters exceeds "
                            f"safe limit of {safe_limit:,} (with 10% buffer)"
                        )
                    else:
                        # Safe to proceed - log if we're approaching the limit (>80% of safe limit)
                        if estimated_prompt_size > (safe_limit * 0.8):
                            self.logger.info(
                                f"{log_prefix} Prompt size: {estimated_prompt_size:,} chars "
                                f"({(estimated_prompt_size/safe_limit)*100:.1f}% of safe limit)"
                            )
                
                except InputTooLongError:
                    # Re-raise to trigger split logic
                    raise
                except Exception as e:
                    # Any other error in estimation - log and continue to actual LLM call
                    self.logger.debug(f"{log_prefix} Proactive size check failed: {get_clean_error_message(e, max_chars=2000)}")
                
                self.logger.info(f"⏳ {log_prefix} Sending batch to use case generation prompt...")
                self.logger.info(f"⏳ {log_prefix} Waiting for LLM response (may take 3-5 min)...")
                response_raw = self.ai_agent.run_worker(
                    step_name=f"Batch_{batch_num}",
                    worker_prompt_path="BASE_USE_CASE_GEN_PROMPT",
                    prompt_vars=prompt_vars,
                    response_schema=None,
                    temperature_override=temperature_override,
                    skip_cache=(attempt > 1)
                )
                self.logger.info(f"✅ {log_prefix} Received LLM response, parsing CSV...")
                
                response_clean = clean_csv_response(response_raw) if response_raw else ""
                parsed_rows = self._parse_llm_csv_response(response_clean, log_prefix) if response_clean else []
                
                del response_raw, response_clean
                del prompt_vars, schema_markdown
                
                self.logger.info(f"✅ {log_prefix} Parsed {len(parsed_rows)} use cases")
                
                if not parsed_rows:
                    if attempt < max_attempts:
                        self.logger.warning(f"{log_prefix} Attempt {attempt}: the model returned no use case for this batch. Retrying...")
                        continue
                    self.logger.warning(f"{log_prefix} The model returned no use case for this batch in {max_attempts} attempt(s); "
                                        f"no table was rejected, so the batch adds no use case")
                    return sub_batch_results
                
                validation_schema = list(getattr(self, "_business_column_details_global", None) or []) + list(current_column_details)
                if additional_related_columns:
                    validation_schema = validation_schema + additional_related_columns
                is_tables_valid, hallucinated_use_cases, valid_use_cases = self._validate_use_case_tables(
                    parsed_rows, validation_schema, log_prefix
                )
                
                if not is_tables_valid:
                    hallucinated_count = len(hallucinated_use_cases)
                    hallucination_rate = hallucinated_count / len(parsed_rows) * 100
                    valid_count = len(valid_use_cases)
                    
                    if valid_count == 0:
                        self.logger.warning(f"{log_prefix} ⚠️ Table hallucination detected: {hallucinated_count}/{len(parsed_rows)} use cases ({hallucination_rate:.1f}%)")
                        for i, uc in enumerate(hallucinated_use_cases[:3]):
                            self.logger.warning(f"{log_prefix}    Example {i+1}: {uc.get('No')}: {uc.get('Name')} - {uc.get('hallucination_reason')}")
                        if attempt < max_attempts:
                            _hallucination_feedback = self._build_generation_rejection_feedback(hallucinated_use_cases)
                            self.logger.info(f"{log_prefix} Injecting rejection feedback for retry ({len(hallucinated_use_cases)} rejected use case(s))")
                            self.logger.warning(f"{log_prefix}    Retrying batch (attempt {attempt + 1}/{max_attempts}) because no valid use cases were returned")
                            continue
                        self.logger.error(f"{log_prefix} ❌ No valid use cases after {max_attempts} attempts due to hallucinated tables")
                        return sub_batch_results
                    
                    self.logger.warning(f"{log_prefix} ⚠️ Table hallucination detected: {hallucinated_count}/{len(parsed_rows)} use cases ({hallucination_rate:.1f}%). Dropping hallucinated use cases and continuing with {valid_count} valid use cases.")
                    for i, uc in enumerate(hallucinated_use_cases[:3]):
                        self.logger.warning(f"{log_prefix}    Example {i+1}: {uc.get('No')}: {uc.get('Name')} - {uc.get('hallucination_reason')}")
                    with self.registry_lock:
                        _cross_batch_tracker = getattr(self, '_known_hallucinated_tables', None)
                        if _cross_batch_tracker is not None:
                            for uc in hallucinated_use_cases:
                                reason = uc.get('hallucination_reason', '')
                                if 'Tables not found' in reason:
                                    for t in reason.replace('Tables not found in schema: ', '').split(', '):
                                        t = t.strip()
                                        if t:
                                            _cross_batch_tracker.add(t)
                    parsed_rows = valid_use_cases
                
                # Re-number use cases with batch prefix
                # Handle both int and string batch_num (for retry batches)
                if isinstance(batch_num, int):
                    batch_prefix = f"{batch_num:02d}"  # Changed from B{batch_num:03d} to just 2-digit number
                else:
                    batch_prefix = str(batch_num)  # Already formatted (e.g., "RETRY_1")
                
                used_ids = set()
                for row in parsed_rows:
                    try:
                        original_id = row['No']
                        use_case_num = original_id.split('-')[-1]
                        new_id, suffix = f"AI-{batch_prefix}-U{use_case_num}", 2
                        while new_id in used_ids:
                            new_id, suffix = f"AI-{batch_prefix}-U{use_case_num}_{suffix}", suffix + 1
                        used_ids.add(new_id)
                        row['No'] = new_id
                        row['batch'] = batch_num
                    except Exception as e:
                        self.logger.warning(f"{log_prefix} Failed to re-number row: {get_clean_error_message(e)}")
                        row['batch'] = batch_num
                
                self.logger.debug(f"{log_prefix} Successfully processed {len(parsed_rows)} use cases on attempt {attempt}")
                
                log_print(f"\n{'='*80}")
                log_print(f"📊 TOP USE CASES FROM {log_prefix} (for early quality review):")
                log_print(f"{'='*80}\n")
                
                for use_case in parsed_rows[:10]:
                    use_case_name = self._normalize_usecase(use_case.get('usecase', ''), use_case.get('Name', ''))
                    log_print(f"   {use_case.get('No', 'N/A')}: {use_case_name or 'N/A'}")
                
                log_print(f"\n{'='*80}\n")
                
                return parsed_rows + sub_batch_results
                
            except InputTooLongError as e:
                # Handle "input too long" by intelligently splitting the batch
                # NEVER DROP BUSINESS TABLES - keep splitting until they fit
                
                # Group columns by table
                table_to_columns = {}
                for col in current_column_details:
                    table_key = (col[0], col[1], col[2])  # (catalog, schema, table)
                    if table_key not in table_to_columns:
                        table_to_columns[table_key] = []
                    table_to_columns[table_key].append(col)
                
                num_tables = len(table_to_columns)
                
                # Get business scores to check if tables are marked as business
                business_scores = getattr(self, 'business_scores', {})
                
                def is_business_table(table_key):
                    """Check if a table is marked as business (score > 0)."""
                    fqtn = f"{table_key[0]}.{table_key[1]}.{table_key[2]}"
                    return business_scores.get(fqtn, 0) > 0
                
                if num_tables > 1:
                    tables_list = list(table_to_columns.keys())
                    reference_tables_set = {k for k, v in getattr(self, "data_category_map", {}).items() if v == "REFERENCE"}
                    filtered_tables = []
                    for table_key in tables_list:
                        fqtn = f"{table_key[0]}.{table_key[1]}.{table_key[2]}"
                        if reference_tables_set and fqtn in reference_tables_set:
                            continue
                        filtered_tables.append(table_key)
                    if not filtered_tables:
                        self.logger.warning(f"{log_prefix} Input too long ({str(e)}). Only reference tables present; skipping use case generation.")
                        return sub_batch_results
                    self.logger.warning(f"{log_prefix} Input too long ({str(e)}). Falling back to single-table calls for {num_tables} tables.")
                    split_type = "Proactive" if "Proactive split" in str(e) else "Reactive"
                    split_info = {
                        'batch': batch_num,
                        'original_tables': num_tables,
                        'split_into': num_tables,
                        'sub_batch_1_tables': 1,
                        'sub_batch_2_tables': 1,
                        'reason': 'Input too long for LLM',
                        'split_type': split_type
                    }
                    with self.registry_lock:
                        self.processing_honesty['total_batch_splits'] += 1
                        self.processing_honesty['batch_split_history'].append(split_info)
                    current_column_details = table_to_columns[filtered_tables[0]]
                    current_column_details = self._augment_columns_with_related_tables(current_column_details)
                    for idx, table_key in enumerate(filtered_tables[1:], start=2):
                        single_cols = table_to_columns[table_key]
                        single_cols = self._augment_columns_with_related_tables(single_cols)
                        single_batch_id = f"{batch_num}_T{idx}"
                        single_use_cases = self._process_batch_with_retry(
                            single_cols,
                            single_batch_id,
                            business_context=business_context,
                            strategic_initiative=strategic_initiative,
                            value_chain=value_chain,
                            revenue_model=revenue_model,
                            max_attempts=max_attempts,
                            previous_use_cases_feedback=previous_use_cases_feedback,
                            temperature_override=temperature_override,
                            industry=industry
                        )
                        if single_use_cases:
                            sub_batch_results.extend(single_use_cases)
                            self.logger.info(f"{log_prefix} Single-table sub-batch {single_batch_id} generated {len(single_use_cases)} use cases")
                    continue
                    
                elif num_tables == 1:
                    # Single table is too big: try dropping columns
                    table_key = list(table_to_columns.keys())[0]
                    table_columns = table_to_columns[table_key]
                    table_is_business = is_business_table(table_key)
                    fqtn = f"{table_key[0]}.{table_key[1]}.{table_key[2]}"
                    
                    if len(table_columns) > 500:
                        keep_count = 500
                    else:
                        keep_count = len(table_columns) - 100
                    if keep_count < 5:
                        if table_is_business:
                            keep_count = 5
                        else:
                            self.logger.error(f"{log_prefix} Input too long even with minimal columns ({len(table_columns)} columns from non-business table {table_key[2]}). Dropping this table.")
                            return sub_batch_results
                    current_column_details = table_columns[:keep_count]
                    
                    kept_columns = [col[3] for col in current_column_details]  # col[3] is column name
                    self.storage_manager.save_column_tracking(fqtn, kept_columns)
                    
                    dropped_count = len(table_columns) - keep_count
                    drop_info = {
                        'table': fqtn,
                        'original_columns': len(table_columns),
                        'kept_columns': keep_count,
                        'dropped_columns': dropped_count,
                        'drop_percentage': (dropped_count / len(table_columns)) * 100,
                        'is_business': table_is_business
                    }
                    with self.registry_lock:
                        if fqtn not in [t['table'] for t in self.processing_honesty['tables_with_columns_dropped']]:
                            self.processing_honesty['tables_with_columns_dropped'].append(drop_info)
                        else:
                            # Update existing entry with new drop info
                            for idx, existing in enumerate(self.processing_honesty['tables_with_columns_dropped']):
                                if existing['table'] == fqtn:
                                    self.processing_honesty['tables_with_columns_dropped'][idx] = drop_info
                                    break
                    
                    table_type = "BUSINESS" if table_is_business else "non-business"
                    self.logger.warning(f"{log_prefix} Input too long ({str(e)}). Single {table_type} table {table_key[2]} is too large. Dropping columns from {len(table_columns)} to {keep_count} columns and retrying...")
                    self.logger.info(f"{log_prefix} Saved column tracking for {fqtn}: {len(kept_columns)} columns ({', '.join(kept_columns[:5])}{'...' if len(kept_columns) > 5 else ''})")
                    
                    continue
                    
                else:
                    # No tables? This shouldn't happen
                    self.logger.error(f"{log_prefix} Input too long but no tables found. Cannot process.")
                    return sub_batch_results
                
            except Exception as e:
                if attempt < max_attempts:
                    self.logger.warning(f"{log_prefix} Attempt {attempt} failed: {get_clean_error_message(e)}. Retrying...")
                else:
                    self.logger.error(f"{log_prefix} All {max_attempts} attempts failed: {get_clean_error_message(e)}")
                    return sub_batch_results
        
        # If we exhaust all attempts without success, still return any pending results
        return sub_batch_results

    _JOB_LAUNCHER_CLASS_SOURCE = r'''
import re as _jl_re, os as _jl_os, hashlib as _jl_hashlib

class JobLauncher:
    _TAG_SAFE_RE = None
    _TERMINAL_STATES = ("TERMINATED", "INTERNAL_ERROR", "SKIPPED")
    TASK_KEY = "inspire_task"
    @staticmethod
    def _sanitize_tag(value):
        if JobLauncher._TAG_SAFE_RE is None:
            JobLauncher._TAG_SAFE_RE = _jl_re.compile(r'[^A-Za-z0-9._-]')
        s = JobLauncher._TAG_SAFE_RE.sub('_', str(value))
        s = _jl_re.sub(r'_+', '_', s)
        s = s.strip('_').strip('.').strip('-')
        return s
    def __init__(self, notebook_path, widget_key_values, job_tags=None):
        self.notebook_path = str(notebook_path)
        self.widget_key_values = {str(k): str(v) for k, v in widget_key_values.items()}
        self.job_tags = {self._sanitize_tag(k): self._sanitize_tag(v) for k, v in (job_tags or {}).items()}
    @staticmethod
    def persistent_job_name(slug, notebook_path):
        _slug = JobLauncher._sanitize_tag(str(slug or "").lower())[:60].strip("_.-") or "use_case"
        _hash = _jl_hashlib.sha256(str(notebook_path or "").encode("utf-8")).hexdigest()[:8]
        return f"dbx_inspire_ai_{_slug}_{_hash}"
    @staticmethod
    def find_job_id(w, job_name):
        for _candidate in w.jobs.list(name=job_name):
            if _candidate.settings and _candidate.settings.name == job_name:
                return _candidate.job_id
        return None
    @staticmethod
    def _run_once_task_class(task_cls):
        class RunOnceTask(task_cls):
            def as_dict(self):
                return {**super().as_dict(), "disable_auto_optimization": True}
        return RunOnceTask
    def _task(self, base_parameters):
        from databricks.sdk.service import jobs as _jl_jobs
        is_serverless, cluster_id = self._detect_compute_type()
        classic = bool(cluster_id) and not is_serverless
        task_cls = _jl_jobs.Task if classic else JobLauncher._run_once_task_class(_jl_jobs.Task)
        task = task_cls(task_key=self.TASK_KEY, notebook_task=_jl_jobs.NotebookTask(notebook_path=self.notebook_path, base_parameters=dict(base_parameters)), timeout_seconds=14400)
        if classic:
            task.existing_cluster_id = cluster_id
        return task
    def _recent_run_count(self, w, job_id, window_minutes=60, ceiling=None):
        import time as _rc_time
        _since_ms = int((_rc_time.time() - int(window_minutes) * 60) * 1000)
        _count = 0
        try:
            for _r in w.jobs.list_runs(job_id=job_id, start_time_from=_since_ms):
                _count += 1
                if ceiling is not None and _count > ceiling:
                    break
        except Exception:
            return None
        return _count
    def launch(self, job_name=None, run_name=None, run_parameters=None, max_concurrent_runs=None, queue=False, schedule_cron=None, timezone_id="UTC", launch_cap=20, only_tasks=None):
        _fail = {"success": False, "job_id": None, "run_id": None, "job_name": job_name or "", "run_name": run_name or "", "job_url": "", "error": None}
        try:
            import time as _jl_time
            from databricks.sdk import WorkspaceClient as _JL_WC
            from databricks.sdk.service import jobs as _jl_jobs
            w = _JL_WC()
            _job_name = job_name or f"dbx_inspire_job_{int(_jl_time.time())}"
            _run_name = run_name or _job_name
            task = self._task(self.widget_key_values)
            _extra = {}
            if max_concurrent_runs:
                _extra["max_concurrent_runs"] = int(max_concurrent_runs)
            if queue:
                _extra["queue"] = _jl_jobs.QueueSettings(enabled=True)
            _schedule = None
            if schedule_cron:
                _schedule = _jl_jobs.CronSchedule(quartz_cron_expression=schedule_cron, timezone_id=timezone_id, pause_status=_jl_jobs.PauseStatus.PAUSED)
            _existing_job_id = None
            try:
                _existing_job_id = JobLauncher.find_job_id(w, _job_name)
            except Exception as _list_err:
                print(f"Could not look up existing job {_job_name} ({_list_err}); creating it.")
            if _existing_job_id:
                if _schedule is not None and not getattr(getattr(w.jobs.get(job_id=_existing_job_id), "settings", None), "schedule", None):
                    _extra["schedule"] = _schedule
                w.jobs.update(job_id=_existing_job_id, new_settings=_jl_jobs.JobSettings(name=_job_name, tags=self.job_tags, tasks=[task], **_extra))
                _job_id = _existing_job_id
            else:
                if _schedule is not None:
                    _extra["schedule"] = _schedule
                _created = w.jobs.create(name=_job_name, tags=self.job_tags, tasks=[task], **_extra)
                _job_id = _created.job_id
            _cap = int(launch_cap) if launch_cap is not None else 20
            if _existing_job_id and _cap >= 0:
                _recent = self._recent_run_count(w, _job_id, window_minutes=60, ceiling=_cap)
                if _recent is not None and _recent >= _cap:
                    print(f"[LAUNCH-CAP FIRED] job {_job_name} (id {_job_id}) already had {_recent} run(s) start in the last 60 minutes; the cap is {_cap} per hour, so this run is refused.")
                    _fail["job_id"] = _job_id
                    _fail["error"] = f"launch cap reached: {_recent} runs started for this job in the last 60 minutes (cap {_cap})"
                    return _fail
            _run_now = {"job_id": _job_id}
            if run_parameters:
                _run_now["notebook_params"] = {str(k): str(v) for k, v in run_parameters.items()}
            if only_tasks:
                _run_now["only"] = [str(t) for t in only_tasks]
            _started = w.api_client.do("POST", "/api/2.1/jobs/run-now", body=_run_now) or {}
            _run_id = _started.get("run_id")
            if not _run_id:
                raise RuntimeError(f"the Jobs API started no run for job {_job_id}: {_started}")
            host, org_id = self._get_workspace_context()
            job_url = ""
            if host:
                _org_param = f"?o={org_id}" if org_id else ""
                job_url = f"{host}/jobs/{_job_id}/runs/{_run_id}{_org_param}"
            result = {"success": True, "job_id": _job_id, "run_id": _run_id, "job_name": _job_name, "run_name": _run_name, "job_url": job_url, "error": None}
            self._print_success_banner(result)
            return result
        except Exception as _launch_exc:
            _fail["error"] = str(_launch_exc)
            return _fail
    def plan_next_scheduled_run(self, job_name):
        from databricks.sdk import WorkspaceClient as _PN_WC
        from databricks.sdk.service import jobs as _pn_jobs
        w = _PN_WC()
        job_id = JobLauncher.find_job_id(w, job_name)
        if not job_id:
            return None
        w.jobs.update(job_id=job_id, new_settings=_pn_jobs.JobSettings(tags=self.job_tags, tasks=[self._task(self.widget_key_values)]))
        return job_id
    def wait_for_run(self, run_id, timeout_s=1800, poll_s=15, out=print):
        import time as _wr_time
        from databricks.sdk import WorkspaceClient as _WR_WC
        w = _WR_WC()
        started = _wr_time.time()
        deadline = started + max(0, int(timeout_s))
        last, errors = None, 0
        while True:
            try:
                run = w.jobs.get_run(run_id=run_id)
                errors = 0
            except Exception as _get_err:
                errors += 1
                out(f"  could not read job run {run_id} ({_get_err})")
                if errors >= 5 or _wr_time.time() >= deadline:
                    return {"status": "RUNNING", "note": f"run state unreadable: {_get_err}"}
                _wr_time.sleep(poll_s)
                continue
            life = self._enum_str(getattr(run.state, "life_cycle_state", None))
            result = self._enum_str(getattr(run.state, "result_state", None))
            if (life, result) != last:
                out(f"  [{int(_wr_time.time() - started)}s] job run {run_id}: {life or 'PENDING'}{(' / ' + result) if result else ''}")
                last = (life, result)
            if life in self._TERMINAL_STATES:
                if result == "SUCCESS":
                    return {"status": "SUCCESS", "output": self._run_output(w, run)}
                return {"status": result or life, "error": self.describe_run_failure(w, run)}
            if _wr_time.time() >= deadline:
                return {"status": "RUNNING"}
            _wr_time.sleep(poll_s)
    @staticmethod
    def _task_run_id(run, task_key=None):
        tasks = getattr(run, "tasks", None) or []
        chosen = next((t for t in tasks if getattr(t, "task_key", None) == (task_key or JobLauncher.TASK_KEY)), tasks[0] if tasks else None)
        return getattr(chosen, "run_id", None) if chosen is not None else getattr(run, "run_id", None)
    def _run_output(self, w, run):
        try:
            output = w.jobs.get_run_output(run_id=self._task_run_id(run))
        except Exception as _out_err:
            return f"(run output unavailable: {_out_err})"
        return getattr(getattr(output, "notebook_output", None), "result", None)
    def describe_run_failure(self, w, run):
        parts = []
        message = getattr(getattr(run, "state", None), "state_message", None)
        if message:
            parts.append(str(message))
        try:
            output = w.jobs.get_run_output(run_id=self._task_run_id(run))
            if getattr(output, "error", None):
                parts.append(str(output.error))
            if getattr(output, "error_trace", None):
                parts.append(str(output.error_trace)[-2000:])
        except Exception as _out_err:
            parts.append(f"(run output unavailable: {_out_err})")
        return " | ".join(parts) or "no error message"
    @staticmethod
    def get_current_notebook_path():
        _nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        try:
            _path = _nb_ctx.notebookPath().get()
            if _path: return _path
        except Exception: pass
        try:
            import json as _jl_json
            _ctx = _jl_json.loads(_nb_ctx.toJson())
            for _key in ("notebook_path", "notebookPath"):
                _val = _ctx.get("extraContext", {}).get(_key, "")
                if _val: return _val
                _val = _ctx.get("tags", {}).get(_key, "")
                if _val: return _val
        except Exception: pass
        return ""
    def _detect_compute_type(self):
        if _jl_os.environ.get("IS_SERVERLESS", "").upper() == "TRUE": return True, None
        try:
            if "connect" in type(spark).__module__: return True, None
        except Exception: return True, None
        try:
            _ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            _cid = _ctx.clusterId().get() if _ctx.clusterId().isDefined() else ""
            if _cid: return False, _cid
        except Exception: pass
        return True, None
    def _get_workspace_context(self):
        try:
            import json as _jl_json
            _ctx_json = dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson()
            _ctx = _jl_json.loads(_ctx_json)
            _host = _ctx.get("extraContext", {}).get("api_url", "")
            _org = _ctx.get("extraContext", {}).get("orgId", "")
            if _host: return _host.rstrip("/"), _org
        except Exception: pass
        try:
            from databricks.sdk import WorkspaceClient as _JL_WC3
            _w = _JL_WC3()
            return str(_w.config.host).rstrip("/"), ""
        except Exception: pass
        return "", ""
    def _print_success_banner(self, result):
        _banner = self._build_banner_str(result)
        print(_banner)
        return _banner
    def _build_banner_str(self, result):
        _jn = result.get("job_name", "N/A")
        _rn = result.get("run_name", "N/A")
        _rid = str(result.get("run_id", "N/A"))
        _url = result.get("job_url", "")
        from datetime import datetime as _jl_dt
        _now = _jl_dt.now().strftime("%Y-%m-%d %H:%M:%S")
        _nb = self.notebook_path.rsplit("/", 1)[-1] if self.notebook_path else "N/A"
        _content_lines = [f"  Notebook:     {_nb}", f"  Job Name:     {_jn}", f"  Job Run Name: {_rn}", f"  Job Run ID:   {_rid}", f"  Launched At:  {_now}"]
        _footer_lines = ["  Go to Jobs & Pipelines to follow the progress", "  of the run, or click the link below:"]
        if _url:
            _footer_lines.append("")
            _footer_lines.append(f"  {_url}")
        _all = ["  JOB LAUNCHED SUCCESSFULLY"] + _content_lines + _footer_lines
        _iw = max(len(l) for l in _all) + 2
        def _row(text): return f"  {text}"
        lines = ["", "=" * (_iw + 4), "  JOB LAUNCHED SUCCESSFULLY", "=" * (_iw + 4)]
        for cl in _content_lines: lines.append(_row(cl))
        lines.append("-" * (_iw + 4))
        for fl in _footer_lines: lines.append(_row(fl))
        lines.append("=" * (_iw + 4))
        return "\n".join(lines)
    @staticmethod
    def _enum_str(val):
        if val is None:
            return ""
        if hasattr(val, 'value'):
            return str(val.value).upper()
        return str(val).upper()
    def verify_run_alive(self, run_id, pings=10, interval_s=10):
        import time as _vra_time
        try:
            from databricks.sdk import WorkspaceClient as _VRA_WC
            _w = _VRA_WC()
        except Exception:
            print(f"  Could not create WorkspaceClient for verification. Assuming job is running.")
            return True
        _definitive_failure_states = {"INTERNAL_ERROR", "SKIPPED"}
        _dead_results = {"FAILED", "TIMEDOUT", "CANCELED"}
        for _i in range(pings):
            _vra_time.sleep(interval_s)
            try:
                _run = _w.jobs.get_run(run_id=run_id)
                _state = self._enum_str(getattr(_run.state, 'life_cycle_state', None))
                _result = self._enum_str(getattr(_run.state, 'result_state', None))
                print(f"  Ping {_i + 1}/{pings}: life_cycle_state={_state or 'N/A'}, result_state={_result or 'N/A'}")
                if _state in _definitive_failure_states:
                    print(f"  Job run definitively failed ({_state}). Falling back to local execution.")
                    return False
                if _state in ("TERMINATED", "TERMINATING"):
                    if _result in _dead_results:
                        print(f"  Job run definitively failed ({_state}/{_result}). Falling back to local execution.")
                        return False
                    if _result == "SUCCESS":
                        print(f"  Job run already completed successfully during verification.")
                        return True
            except Exception as _ping_err:
                print(f"  Ping {_i + 1}/{pings}: status check error (assuming still starting): {_ping_err}")
        print(f"  Job not confirmed failed after {pings} pings. Assuming job is running (may be waiting for compute).")
        return True
'''

    _GENIE_DECISION_KEYS = ("decision", "decision_owner", "cadence", "kpi", "kpi_direction", "value_tier", "outcome_definition",
                            "time_column", "entity_key_column", "join_path", "signal_columns", "build_path", "blockers")
    _GENIE_NOT_ASSESSED = "not assessed"
    _GENIE_CADENCES = (
        ("hour", "0 0 * * * ?", 1, 24, 48),
        ("daily", "0 0 6 * * ?", 1, 7, 14),
        ("day", "0 0 6 * * ?", 1, 7, 14),
        ("week", "0 0 6 ? * MON", 7, 52, 8),
        ("month", "0 0 6 1 * ?", 30, 12, 6),
        ("quarter", "0 0 6 1 1,4,7,10 ?", 91, 4, 4),
        ("annual", "0 0 6 1 1 ?", 365, 1, 2),
        ("year", "0 0 6 1 1 ?", 365, 1, 2),
    )
    _GENIE_ACCEPTANCE_TEXT = {
        "A": "out-of-time AUC of at least {min_auc}, PR-AUC of at least {min_pr_auc_lift} times the class rate, and at least {min_auc_gain} AUC above the simple baseline (for survival, a concordance index of at least {min_c_index})",
        "B": ("at most {max_alerts_per_run} alerts per run and at least {min_topk_stability} agreement on the top alerts between two "
              "fits: the share of the first fit's top k alerts that are also among the second fit's top k, with k the smallest of "
              "{max_alerts_per_run} and the two fits' alert counts, so the agreement lies between 0 and 1"),
        "C": "a silhouette of at least {min_silhouette}, every segment holding at least {min_segment_share} of entities, and a bootstrap adjusted Rand index of at least {min_bootstrap_ari}",
        "D": "out-of-time precision@K of at least {min_precision_lift} times the popularity baseline",
        "E": "MASE below {max_mase} against the seasonal naive forecast over at least {min_backtest_folds} backtest folds; otherwise the seasonal naive forecast is published, flagged",
        "F": "groups with fewer than {min_group_rows} rows are left out of the group findings and counted in the summary, while each entity inside them is still scored and, when the entity has its own test, tested on its own per-unit minimum; every tested finding reports its effect size and its Benjamini-Hochberg q-value, and a tested finding with a q-value above {fdr_alpha} is still written, with the tier its value_at_stake gives, 'not significant' in its top reasons, a confidence_level that is not 'High' and data_quality_flag 'ok' unless its own inputs are incomplete, while an entity below its per-unit minimum has no q-value and its top reasons give its count and the minimum",
        "G": "an AI function error rate of at most {max_ai_error_rate}",
        "H": "a feasible plan that is better than the current plan; simulations use a fixed seed and at least {min_simulation_draws} draws",
    }

    _GENIE_ACCEPTANCE_STOP = {
        "model": ('When no candidate passes the acceptance rule, the notebook prints "No approved model yet" with the metrics of the '
                  "candidates and the baseline, registers no model, and writes the baseline's rows{tiers}. "),
        "E": ("When the candidate does not pass the acceptance rule, the notebook publishes the seasonal naive forecast instead, with method "
              "`seasonal_naive`, and its summary says the candidate did not beat it; acceptance never stops this run."),
        "F": ("Acceptance never stops this run: every tested finding is written with its effect size and q-value, and a tested finding "
              "that is not significant keeps the tier its value_at_stake gives and says 'not significant' in its top reasons; an "
              "entity below its per-unit minimum was not tested, so its top reasons give its count and the minimum instead."),
        "G": ('When the AI error rate is above `max_ai_error_rate`, the notebook prints "AI processing failed the error-rate check" with the '
              "rate and writes nothing."),
        "H": ('When no feasible plan beats the current plan, the notebook prints "No better plan found" with the metrics of both plans and '
              "writes the current plan's rows with method `current_plan`, every row Monitor."),
    }

    def _genie_acceptance_stop_rule(self, spec: dict) -> str:
        family, measured = spec["family"], spec["decision_basis"] == "measured"
        if family in INSPIRE_MODEL_FAMILIES:
            return (self._GENIE_ACCEPTANCE_STOP["model"].format(tiers=", each with the tier its value_at_stake gives" if measured else "")
                    + GENIE_BASELINE_VALUE_RULE + " " + (GENIE_MEASURED_TIER_RULE if measured else GENIE_BASELINE_MONITOR_RULE))
        return self._GENIE_ACCEPTANCE_STOP[family] + (" " + GENIE_MEASURED_TIER_RULE if measured else "")

    def _genie_acceptance_section(self, spec: dict) -> str:
        family = spec["family"]
        flag_only = any(INSPIRE_FLAG_ONLY_READINESS.get(rule[1]) for rule in INSPIRE_READINESS_RULES["common"] + INSPIRE_READINESS_RULES[family])
        named = {key: f"`{key}` ({_genie_number_text(value) if isinstance(value, (int, float)) else value})" for key, value in spec["thresholds"].items()}
        return "\n".join([
            "Inspire writes this section. Where any other section of this instruction words acceptance or a stop differently, this section wins.",
            "- Readiness: " + GENIE_READINESS_STOP_RULE
            + (" A readiness check marked as never stopping the run only sets data_quality_flag." if flag_only else ""),
            "- Acceptance: " + self._GENIE_ACCEPTANCE_TEXT[family].format(**named) + ", each threshold read from the widget it names. "
            "This is the whole acceptance rule: a criterion it does not name is printed in the summary and never decides acceptance.",
            "- Acceptance outcome: " + self._genie_acceptance_stop_rule(spec),
            "- " + GENIE_NO_FORCED_PASS_RULE])

    _GENIE_CRITERION_TERMS = (
        ("precision", r"\bprecision\b"), ("recall", r"\brecall\b"), ("F1", r"\bf1\b"), ("accuracy", r"\baccuracy\b"),
        ("AUC", r"\bauc\b"), ("MASE", r"\bmase\b"), ("MAPE", r"\bs?mape\b"), ("RMSE", r"\brmse\b"), ("MAE", r"\bmae\b"),
        ("silhouette", r"\bsilhouette\b"), ("adjusted Rand index", r"\badjusted rand\b|\bari\b"),
        ("concordance index", r"\bconcordance\b|\bc-index\b"), ("q-value", r"\bq-?values?\b"),
        ("a comparison with the baseline", r"\b(?:beat\w*|outperform\w*|better than|worse than|above|below|exceed\w*|at least|"
                                           r"no lower than|not lower than)\b[^.;\n]{0,60}?\b(?:baseline|naive|current plan)\b"),
    )
    _GENIE_LIST_MARKER = r"[ \t]*(?:(?:[-*]|\d+\.)[ \t]+)?(?:\[[ xX]\][ \t]+)?"
    _GENIE_ACCEPTANCE_SCOPE_RE = re.compile(r"\baccept", re.I)
    _GENIE_ACCEPTANCE_LABEL_RE = re.compile("^" + _GENIE_LIST_MARKER + r"(?:\*\*)?\s*accept", re.I)
    _GENIE_CRITERION_CUE_RE = re.compile(r"\b(?:must|requir\w*|at least|at most|not (?:fall )?below|no lower|no worse|beat\w*|exceed\w*|"
                                         r"above|below|better than|higher than|lower than|minimum|maximum|threshold)\b|[<>\u2264\u2265]", re.I)
    _GENIE_REPORT_RE = re.compile(r"\b(?:report\w*|print\w*|summary|logs?|logged|show\w*|display\w*)\b", re.I)
    _GENIE_NOT_CRITERION_RE = re.compile(r"\b(?:not|never|no)\b[^.;]{0,40}\b(?:criteri\w*|decid\w*|part of)\b|\bdoes not use\b|\bnever uses?\b", re.I)
    _GENIE_NOT_FOR_ACCEPTANCE_RE = re.compile(r"\b(?:without using (?:it|them|this|these)|(?:not|never) used|(?:not|never) part of|"
                                              r"(?:does not|never) decides?|not counted)\b[^.;]{0,25}?\bacceptance\b", re.I)
    _GENIE_NEGATED_TERM_RE = re.compile(r"\b(?:rather than|instead of|nor|without)\s+(?:[\w-]+\s+)?$", re.I)
    _GENIE_FAILED_ACCEPTANCE_RE = re.compile(r"\b(?:no candidate (?:passes|passed|is accepted|meets)|(?:fails?|failed|failing|misses|missed|"
                                             r"does not (?:pass|meet)) (?:the )?acceptance|acceptance (?:fails|failed|is not met|stops?|check fails)|"
                                             r"not accepted|no approved model|or (?:the )?acceptance (?:check |rule )?(?:fails?|failed|stops?)|"
                                             r"fail\w* (?:\w+ ){0,3}or (?:the )?acceptance|stop at (?:\w+ or )?acceptance)\b", re.I)
    _GENIE_STOP_OUTCOME_RE = re.compile(r"\b(?:writes? nothing|no rows|stops?|skips?|skipped|exits?|ends the run|halts?)\b", re.I)
    _GENIE_NEVER_STOPS_RE = re.compile(r"\b(?:never|not)\s+(?:\w+\s+){0,2}(?:stops?|stopping|ends?|halts?)\b|\bno (?:acceptance )?stop\b|"
                                       r"\b(?:writ\w*|publish\w*|uses?|falls? back to|keeps?)\s+(?:[\w-]+\s+){0,2}"
                                       r"(?:baseline|current plan|seasonal naive|fallback)", re.I)

    @staticmethod
    def _genie_clause_at(sentence: str, position: int) -> str:
        start = max(sentence.rfind(mark, 0, position) for mark in ";,:()") + 1
        ends = [index for index in (sentence.find(mark, position) for mark in ";,:()") if index >= 0]
        return sentence[start:min(ends) if ends else len(sentence)]

    def _genie_contract_conflicts(self, instruction: str, spec: dict) -> list:
        family = spec["family"]
        text = self._GENIE_ACCEPTANCE_SECTION_RE.sub("\n\n", self._GENIE_PLATFORM_SECTION_RE.sub("\n\n", str(instruction or "")))
        text = self._GENIE_STOP_RULE_LINE_RE.sub("", text)
        stop_rule = self._genie_acceptance_stop_rule(spec)
        stop_first = stop_rule.split(". ")[0].rstrip(".") + "."
        banned = [(label, pattern) for label, pattern in self._GENIE_CRITERION_TERMS
                  if not re.search(pattern, self._GENIE_ACCEPTANCE_TEXT[family], re.I)]
        conflicts = []
        for line in text.splitlines():
            if line.lstrip().startswith("|"):
                continue
            labeled = self._GENIE_ACCEPTANCE_LABEL_RE.match(line)
            for sentence in re.split(r"(?<=[.!?])\s+", re.sub("^" + self._GENIE_LIST_MARKER, "", line).strip()):
                quoted = sentence if len(sentence) <= 300 else sentence[:297] + "..."
                segments = sentence.split(";")
                if "writes nothing" not in stop_rule and any(
                        self._GENIE_FAILED_ACCEPTANCE_RE.search(segment) and self._GENIE_STOP_OUTCOME_RE.search(segment)
                        and not self._GENIE_NEVER_STOPS_RE.search(segment) for segment in segments):
                    conflicts.append(f'"{quoted}" says a failed acceptance stops the run, skips steps or writes nothing, but for this use case: {stop_first}')
                if not (labeled or self._GENIE_ACCEPTANCE_SCOPE_RE.search(sentence)) or not self._GENIE_CRITERION_CUE_RE.search(sentence):
                    continue
                found = []
                for label, pattern in banned:
                    for match in re.finditer(pattern, sentence, re.I):
                        clause = self._genie_clause_at(sentence, match.start())
                        segment = segments[sentence.count(";", 0, match.start())]
                        if (not self._GENIE_REPORT_RE.search(clause) and not self._GENIE_NOT_CRITERION_RE.search(clause)
                                and not self._GENIE_NOT_FOR_ACCEPTANCE_RE.search(segment)
                                and not self._GENIE_NEGATED_TERM_RE.search(sentence[:match.start()])):
                            found.append(label)
                            break
                if found:
                    acceptance = self._genie_acceptance_text(family, spec["thresholds"])
                    conflicts.append(f'"{quoted}" makes {" and ".join(found)} part of acceptance, but the acceptance rule is only: {acceptance}; '
                                     f'report {"it" if len(found) == 1 else "them"} in the summary instead')
        return list(dict.fromkeys(conflicts))

    def _genie_decision_contract(self, use_case: dict) -> list:
        assessment = use_case.get('elite_assessment') if isinstance(use_case.get('elite_assessment'), dict) else {}
        via = assessment.get('time_via_join') if isinstance(assessment.get('time_via_join'), dict) else {}
        labels = dict(self._ELITE_CARD_FIELDS)
        rows = []
        for key in self._GENIE_DECISION_KEYS:
            value = assessment.get(key)
            text = self._card_text(value, separator=" -> " if key == "join_path" else ", ")
            if key == "time_column" and via.get('column'):
                text = f"{via['column']} (reached through {via.get('table')}" + (f" along {via['join']})" if via.get('join') else ")")
            if key == "join_path" and via.get('join') and via['join'] not in text:
                text = f"{text} -> {via['join']}" if text else str(via['join'])
            if not text:
                text = "none recorded" if key == "blockers" and isinstance(value, (list, tuple)) else self._GENIE_NOT_ASSESSED
            rows.append((key, labels[key], text))
        return rows

    def _genie_cadence(self, cadence_text) -> dict:
        text = str(cadence_text or "").lower()
        for token, cron, days, season, horizon in self._GENIE_CADENCES:
            if token in text:
                return {"cron": cron, "days": days, "season": season, "horizon": horizon}
        return {"cron": None, "days": None, "season": None, "horizon": None}

    @staticmethod
    def _genie_ai_endpoint_default() -> str:
        workers = get_models_by_type("worker")
        if workers:
            return str(workers[0].get("llm_endpoint_name") or "")
        fast = [name for name in TECHNICAL_CONTEXT.get("fast_chat_endpoints") or [] if LLM_ENDPOINT_HEALTH.is_usable(name)]
        if fast:
            return fast[0]
        pool = sorted([m for m in TECHNICAL_CONTEXT["models"] if _model_is_selectable(m)], key=lambda m: m.get("order", 999))
        return str(pool[0].get("llm_endpoint_name") or "") if pool else ""

    @staticmethod
    def _genie_action_target(use_case: dict) -> str:
        name = " ".join(str(use_case.get('Name') or '').split())
        parts = re.split(r"\s+with\s+", name, flags=re.I)
        deliverable = parts[-1].strip(" .;:,") if len(parts) > 1 else ""
        if deliverable:
            return f"the {deliverable}"
        return f"the {name} results" if name else "the ranked rows of the business output table"

    @staticmethod
    def _genie_slug(text) -> str:
        slug = re.sub(r'[^a-z0-9_]', '_', str(text or '').lower().strip())
        return re.sub(r'_+', '_', slug).strip('_') or "unknown"

    def _genie_result_table(self, use_case: dict) -> str:
        result_table = use_case.get('result_table') or ''
        if all(parse_three_level_name(str(result_table))):
            return result_table
        return compute_result_table_name(use_case.get('No', 'UNKNOWN'), use_case.get('Name', 'unknown'), use_case.get('Tables Involved', ''),
                                         inspire_database=getattr(self, 'inspire_database', ''))

    def _genie_notebook_spec(self, use_case: dict) -> dict:
        uc_id = use_case.get('No', 'UNKNOWN')
        technique = _technique_registry_key(use_case.get('Analytics Technique')) or str(use_case.get('Analytics Technique') or '')
        family = self._uc_execution_family(use_case, f"Genie notebook {uc_id}")
        decision = {key: text for key, _label, text in self._genie_decision_contract(use_case)}
        cadence = self._genie_cadence(decision["cadence"])
        result_table = use_case['result_table'] = self._genie_result_table(use_case)
        names = inspire_output_names(result_table)
        decision_text = decision["decision"] if decision["decision"] != self._GENIE_NOT_ASSESSED else self._genie_action_target(use_case)
        cycle = decision["cadence"] if decision["cadence"] != self._GENIE_NOT_ASSESSED else ""
        assessment = use_case.get('elite_assessment') if isinstance(use_case.get('elite_assessment'), dict) else {}
        runtime_assumptions = [{key: item.get(key) for key in ("kind", "table", "column", "statement")}
                               for item in assessment.get('runtime_assumptions') or [] if isinstance(item, dict)]
        return {
            "family": family, "decision_basis": GENIE_FAMILY_DECISION_BASIS[family], "technique": technique,
            "use_case_id": str(uc_id), "runtime_assumptions": runtime_assumptions,
            "result_table": names["result_table"], "model_name": names["model_name"] if family in INSPIRE_MODEL_FAMILIES else None,
            "season_length": cadence["season"], "horizon": cadence["horizon"],
            "thresholds": dict(GENIE_READINESS_THRESHOLDS[family]), "ai_endpoint": self._genie_ai_endpoint_default(),
            "value_type": "risk score" if technique == "Survival Analysis" else GENIE_FAMILY_SPECS[family]["value_type"],
            "job_slug": self._genie_slug(use_case.get('Name', '')),
            "job_tags": {
                "dbx_inspire_ai_business": JobLauncher._sanitize_tag(getattr(self, 'business_name', '')),
                "dbx_inspire_ai_usecase": self._genie_slug(use_case.get('Name', '')),
                "dbx_inspire_ai_usecase_id": str(uc_id),
                "dbx_inspire_ai_session": str(getattr(self, 'session_id', '')),
                "dbx_inspire_ai_session_id": str(getattr(self, 'session_id', '')),
                "inspire_version": __AGENT_VERSION__,
                "dbx_inspire_ai_launcher_source": "Inspire App" if getattr(self, '_user_provided_session_id', False) else "Inspire Notebook",
                "dbx_inspire_ai_run_type": "training" if family in INSPIRE_MODEL_FAMILIES else "analysis",
            },
            "schedule_cron": cadence["cron"], "wait_minutes": 30,
            "action_by_tier": {
                "Act now": f"Act in this {cycle + ' ' if cycle else ''}cycle",
                "Review": "Review before the next cycle",
                "Monitor": "No action this cycle; keep monitoring",
            },
            "action_target": decision_text,
        }

    @staticmethod
    def _genie_cell(cell_type: str, source: str, title: str = None) -> dict:
        meta = {"nuid": str(uuid.uuid4())}
        if title:
            meta["title"] = title
        cell = {"cell_type": cell_type, "metadata": {"application/vnd.databricks.v1+cell": meta},
                "source": str(source).splitlines(keepends=True)}
        if cell_type == "code":
            cell.update({"execution_count": 0, "outputs": []})
        return cell

    def _genie_acceptance_text(self, family: str, thresholds: dict) -> str:
        return self._GENIE_ACCEPTANCE_TEXT[family].format(**{k: (_genie_number_text(v) if isinstance(v, (int, float)) else v) for k, v in thresholds.items()})

    @staticmethod
    def _genie_stop_message(message: str, required) -> str:
        text = re.sub(r"\{observed[^}]*\}", "<measured>", str(message))
        try:
            return text.format(required=required)
        except (ValueError, KeyError, IndexError):
            return re.sub(r"\{required[^}]*\}", str(required), text)

    def _genie_threshold_text(self, spec: dict) -> str:
        family, th = spec["family"], spec["thresholds"]
        lines, used = [], set()
        for name, key, op, th_key, message in INSPIRE_READINESS_RULES["common"] + INSPIRE_READINESS_RULES[family]:
            if op == "is_true":
                rule, required = f"- readiness `{key}` ({name}): must be True", None
            elif th_key == "min_entities_per_k*k":
                used.add("min_entities_per_k")
                rule, required = (f"- readiness `{key}` and `k` ({name}): entities >= {th['min_entities_per_k']} x k (widget `min_entities_per_k`)",
                                  f"{th['min_entities_per_k']} x k")
            else:
                used.add(th_key)
                rule, required = f"- readiness `{key}` ({name}): {op} {th[th_key]} (widget `{th_key}`)", th[th_key]
            printed = self._genie_stop_message(message, required)
            flag = INSPIRE_FLAG_ONLY_READINESS.get(key)
            if flag:
                lines.append(f"{rule}, a check that never stops the run: when it fails, the notebook keeps running as of the newest data "
                             f"and sets every row's data_quality_flag to '{flag}'; when it fails, print: \"{printed}\"")
            else:
                lines.append(f'{rule}; when it fails, print: "{printed}"')
        if family == "F":
            lines.append("- a group is the unit the findings compare and report, such as a store, a category, a segment, a tier or a "
                         "period; the entities a Pareto, a network or a ranking scores are rows inside those groups, never groups "
                         "themselves, so a group holds many rows")
        for item in spec.get("runtime_assumptions") or []:
            rule = INSPIRE_ASSUMPTION_RULES.get(str(item.get("kind") or "").strip().upper())
            if rule:
                what, minimum, _message = rule
                lines.append(f"- readiness assumption: the number of {what} of `{item.get('column')}` in `{item.get('table')}` must be at least "
                             f"{minimum}; measure it on the rows its statement is about: when the statement is about order lines, "
                             f"transactions or events, count them in the fact table that holds them, through the join path, never in a "
                             f"dimension table where each key appears once; otherwise stop and say: {item.get('statement')}")
        lines.append(f"- acceptance: {self._genie_acceptance_text(family, th)} (every threshold is read from the widget of the same name)")
        policy = {k: v for k, v in th.items() if k not in used}
        lines.append("- all non-readiness thresholds, one widget each: " + ", ".join(f"{k} = {v}" for k, v in policy.items()))
        return "\n".join(lines)

    def _genie_core_widgets(self, spec: dict) -> list:
        rows = [("result_table", ".".join(inspire_table_parts(spec["result_table"])))]
        if spec.get("model_name"):
            rows.append(("model_name", spec["model_name"]))
        rows.append(("ai_endpoint", spec["ai_endpoint"]))
        rows.append(("as_of_date", "latest"))
        if spec["family"] == "E":
            rows += [("horizon", spec["horizon"]), ("season_length", spec["season_length"])]
        rows.append(("currency", ""))
        if spec["decision_basis"] == "measured":
            rows += [(name, "") for name in INSPIRE_TIER_AMOUNTS]
        else:
            rows += [("act_now_share", INSPIRE_TIER_SHARES["act_now"]), ("review_share", INSPIRE_TIER_SHARES["review"])]
        rows += list(spec["thresholds"].items())
        return [{"name": name, "type": "text", "default": "" if value in (None, "") else str(value),
                 "allowed": genie_allowed_text(GENIE_WIDGET_ALLOWED.get(name)), "label": GENIE_WIDGET_LABELS[name]} for name, value in rows]

    @staticmethod
    def _genie_widget_table(widgets: list) -> str:
        lines = ["| widget | type | default | allowed | used for |", "|---|---|---|---|---|"]
        def cell(value):
            return f"`{value}`" if value and not re.search(r"[\s`|]", value) else value or GENIE_WIDGET_CHOOSE

        lines += [f"| `{w['name']}` | {w['type']} | {cell(w['default'])} | {w.get('allowed') or 'any'} | {w['label']} |" for w in widgets]
        return "\n".join(lines)

    @staticmethod
    def _genie_instruction_section(instruction, title: str):
        return re.search(rf"^## (?:\d+\. )?{re.escape(title)}[^\n]*\n(.*?)(?=^## |\Z)", str(instruction or ""), re.M | re.S)

    _GENIE_WIDGET_ROW_RE = re.compile(r"^\|\s*`?([A-Za-z][A-Za-z0-9_]{0,40})`?\s*\|\s*(text|dropdown)\s*\|\s*(.*?)\s*\|"
                                      r"(?:\s*(.*?)\s*\|)?\s*(.*?)\s*\|\s*$")
    _GENIE_PARAMETERS_LEAD = ("Build step 1 creates these widgets in one cell titled Parameters, with these defaults and the used-for text "
                              "as each widget's label, and reads them into typed variables; every later step uses those variables, never a "
                              "literal value. A value outside the widget's allowed range stops the run before any data is read, and the "
                              "stop reason names the widget, the value and the allowed range.")

    def _apply_genie_instruction_contract(self, use_case_id: str, instruction: str, spec: dict) -> str:
        core = self._genie_core_widgets(spec)
        core_defaults = {w["name"]: w["default"] for w in core}
        match = self._genie_instruction_section(instruction, "Parameters")
        body = match.group(1) if match else ""
        extras, dropped, repaired, others, chosen = [], [], [], [], {}
        for line in body.splitlines():
            row = self._GENIE_WIDGET_ROW_RE.match(line.strip())
            if line.strip().startswith("|"):
                if not row or row.group(1).lower() == "widget":
                    continue
                name, kind, default, label = row.group(1).lower(), row.group(2), row.group(3).strip(), row.group(5)
                rule = genie_allowed_rule(row.group(4))
                if re.fullmatch(r"`[^`]*`", default):
                    default = default[1:-1].strip()
                usable = bool(default) and default.lower() not in ("none", "null", "-", "n/a", "''", '""', GENIE_WIDGET_CHOOSE)
                if name in core_defaults:
                    if not core_defaults[name]:
                        allowed = next(w["allowed"] for w in core if w["name"] == name)
                        if usable and not genie_widget_value_problem(name, default, allowed) and name not in chosen:
                            chosen[name] = default
                    elif default != core_defaults[name]:
                        repaired.append(name)
                elif not usable:
                    dropped.append(name)
                elif len(extras) >= GENIE_MAX_USE_CASE_WIDGETS or name in {w["name"] for w in extras}:
                    dropped.append(name)
                else:
                    allowed = "" if kind == "dropdown" else genie_allowed_text(rule)
                    if genie_widget_value_problem(name, default, allowed):
                        allowed = ""
                    extras.append({"name": name, "type": kind, "default": default, "allowed": allowed,
                                   "label": label.strip() or name})
            elif line.strip() != self._GENIE_PARAMETERS_LEAD:
                others.append(line)
        fallback = [w["name"] for w in core if not w["default"] and w["name"] not in chosen and w["name"] in GENIE_WIDGET_FALLBACKS]
        core = [dict(w, default=chosen.get(w["name"]) or w["default"] or GENIE_WIDGET_FALLBACKS.get(w["name"], "")) for w in core]
        text = "\n".join(others).strip()
        section = (self._GENIE_PARAMETERS_LEAD + "\n\n" + self._genie_widget_table(core + extras) + "\n"
                   + (f"\n{text}\n" if text else "") + "\n")
        if match:
            instruction = instruction[:match.start(1)] + section + instruction[match.end(1):]
        else:
            readiness = re.search(r"^## (?:\d+\. )?Readiness Check", instruction or "", re.M)
            at = readiness.start() if readiness else len(instruction or "")
            head = (instruction or "")[:at].rstrip("\n")
            instruction = (head + "\n\n" if head else "") + "## Parameters\n" + section + (instruction or "")[at:]
        self.logger.info(f"[GENIE-WIDGETS FIRED] {use_case_id}: {len(core) + len(extras)} widgets ({len(core)} core, {len(extras)} for "
                         f"the use case)" + ("" if match else "; the Parameters section was missing and was added")
                         + (f"; core defaults restored: {repaired}" if repaired else "")
                         + (f"; defaults the code left open, chosen in the instruction: {chosen}" if chosen else "")
                         + (f"; no usable choice for {fallback}, so {', '.join(f'{n} = {GENIE_WIDGET_FALLBACKS[n]}' for n in fallback)}"
                            if fallback else "")
                         + (f"; dropped without a usable default or over the limit of {GENIE_MAX_USE_CASE_WIDGETS}: {dropped}" if dropped else ""))
        return self._genie_with_platform_rules(instruction, spec)

    _GENIE_PLATFORM_SECTION_RE = re.compile(r"\n*^## Platform Rules\n.*?(?=^## |\Z)", re.M | re.S)
    _GENIE_ACCEPTANCE_SECTION_RE = re.compile(r"\n*^## " + re.escape(GENIE_ACCEPTANCE_SECTION_TITLE) + r"\n.*?(?=^## |\Z)", re.M | re.S)
    _GENIE_STOP_RULE_LINE_RE = re.compile("^(" + _GENIE_LIST_MARKER + r")(?:\*\*)?STOP RULE\b[^\n]*$", re.M)

    def _genie_with_platform_rules(self, instruction: str, spec: dict) -> str:
        instruction = self._GENIE_ACCEPTANCE_SECTION_RE.sub("\n\n", self._GENIE_PLATFORM_SECTION_RE.sub("\n\n", instruction))
        instruction = self._GENIE_STOP_RULE_LINE_RE.sub(lambda match: match.group(1) + GENIE_STOP_RULE_POINTER, instruction).strip()
        return (instruction + f"\n\n## {GENIE_ACCEPTANCE_SECTION_TITLE}\n" + self._genie_acceptance_section(spec)
                + "\n\n## Platform Rules\n" + self._genie_platform_rules(spec) + "\n")

    @classmethod
    def _genie_family_pins(cls, family: str) -> list:
        return [f"{name}=={GENIE_ENV_PACKAGE_VERSIONS[name]}" for name in GENIE_FAMILY_PACKAGES[family]]

    @classmethod
    def _genie_platform_rules(cls, spec: dict) -> str:
        rules = ([GENIE_RUN_CONTROLS_RULE, GENIE_SERVERLESS_RULE, GENIE_COMPUTE_ONCE_RULE, GENIE_DECIMAL_RULE, GENIE_FRAME_NAMES_RULE,
                  GENIE_SQL_NESTING_RULE, GENIE_JOIN_RULE, GENIE_DATE_FRAME_RULE, GENIE_PERIOD_COMPLETE_RULE]
                 + ([GENIE_CLOSED_WINDOW_RULE] if spec["family"] in GENIE_OUTCOME_WINDOW_FAMILIES else [])
                 + [GENIE_EMPTY_INPUT_RULE]
                 + ([GENIE_SPARSE_MATRIX_RULE] if spec["technique"] in GENIE_SPARSE_MATRIX_TECHNIQUES else [])
                 + ([GENIE_TRAINING_CLASSES_RULE] if "min_positives" in spec["thresholds"] else [])
                 + [GENIE_WEIGHT_CHECK_RULE, GENIE_SIGNIFICANCE_RULE, GENIE_PER_UNIT_SAMPLE_RULE, GENIE_NUMBER_FORMAT_RULE,
                    GENIE_VALUE_AT_STAKE_RULE.format(basis=GENIE_VALUE_AT_STAKE_BASIS[spec["decision_basis"]][0],
                                                     amounts=GENIE_VALUE_AT_STAKE_BASIS[spec["decision_basis"]][1]),
                    GENIE_EXPLANATION_FACTS_RULE, GENIE_RANK_ORDER_RULE]
                 + ([GENIE_FEATURE_COLUMNS_RULE, GENIE_BASELINE_VALUE_RULE
                     + (" " + GENIE_BASELINE_MONITOR_RULE if spec["decision_basis"] == "predicted" else ""), GENIE_MLFLOW_RULE]
                    if spec["family"] in INSPIRE_MODEL_FAMILIES else [])
                 + ([GENIE_EVALUATION_PARITY_RULE] if spec["family"] in GENIE_HOLDOUT_FAMILIES else [])
                 + ([GENIE_PINNED_API_RULE.format(pins=", ".join(cls._genie_family_pins(spec["family"])))]
                    if GENIE_FAMILY_PACKAGES[spec["family"]] else []))
        rules.append("The business output columns, each with the type the write casts it to and what it holds; section 7 says how "
                     "this use case fills each column, and where it says otherwise, these meanings win:\n"
                     + "\n".join(f"  - {line}" for line in cls._genie_output_column_lines(spec)))
        rules.append("Before the write, cast every business output column to these types, the ones the column list above gives. "
                     "When `result_table` already exists with one of these columns in another type, inspire_write_result converts "
                       "that column's existing values to the type the run writes (a struct explanation becomes its result text, a "
                       "string top_reasons becomes a one-element array, a number becomes its type with `cast`), adds a column the "
                       "table lacks (NULL for older rows), keeps a column the contract does not list, NULL in the new rows, and "
                       "prints which columns it changed, so every run writes the same schema and you never rewrite the table. "
                       "Before the write, check that no two rows share both entity_id and segment_or_item and stop the run on the "
                       "first pair that does, printing both rows: a repeat is a row of another grain, or an item or period left out "
                       "of segment_or_item, such as order lines written under their order with only the item's label telling them "
                       "apart, which is resolved by putting that item or period first in segment_or_item, never by aggregating the "
                       "rows to the entity key or leaving rows of the population out. "
                       "A missing value is NULL: build every business output column on the working frame, then replace pandas NaN with "
                       "None in a copy made only for the write, so the frame the summary step uses keeps its columns and types; a cast "
                       "turns NaN into the text 'nan'. Make `w = w.astype(object).where(pd.notna(w), None)` the last change to that "
                       "copy before `spark.createDataFrame(w.to_dict('records'), schema)`, since it also turns numpy numbers into "
                       "Python numbers: a column assigned after it, such as from a list, turns None back into NaN, which Spark writes "
                       "as NaN and sorts above every number.")
        return "\n".join(f"- {rule}" for rule in rules)

    _GENIE_OUTPUT_COLUMN_MEANINGS = {
        "as_of_date": "the date the result applies to",
        "entity_id": ("the entity key, from the use case's entity key column: every row is one entity at that key's grain, or one "
                      "entity with one of its items or periods when the method writes several for each entity, that item or "
                      "period then leading segment_or_item so that entity_id with segment_or_item tells the rows apart, never a "
                      "row of another grain, such as a segment's finding next to customer rows, which the summary step reports instead; "
                      "every entity of the population the decision acts on keeps its row, and one the method cannot score, such "
                      "as a customer with a single order for a model of the gaps between orders, keeps it with its reason in "
                      "data_quality_flag and the Monitor tier, never left out by defining the population as the entities the "
                      "method can score; a row whose key misses a part, such as a sale without a store for a store-item key, "
                      "names no entity and gets no row, so the key never holds '<NA>', 'None' or 'nan'"),
        "entity_label": ("a readable name for the entity, from a name or description column of its entity or reference table when the "
                         "tables read have one (the entity key when there is none), with a period written as its name (2002Q1, "
                         "2002-03) rather than a sequence number, never contact details such as an email address, phone number or "
                         "street address, and a part that is missing on a row, such as the carrier of an order line without a ship "
                         "mode, is left out of that row's label, never written as None"),
        "value": ("the result for this row's own entity, never a value of the group it belongs to, which gives every entity of "
                  "the group the same value and rank, and NULL on a row the method computes no result for, such as an entity "
                  "with no unit left to test, whose shrunken estimate is only its baseline, or one whose outcome window is still "
                  "open, never 0 or the baseline in its place; section 7 says what it means and its unit"),
        "value_type": "always '{value_type}'",
        INSPIRE_VALUE_AT_STAKE: ("the money this record puts at stake in the `currency` widget's currency, from the formula the "
                                 "value_at_stake rule above describes: 0 when nothing is at stake, NULL when the formula cannot "
                                 "price the record, never negative"),
        "segment_or_item": ("the item or period that tells apart the rows of an entity the method writes several rows for, such as "
                            "the line item of an order line, before any group the row belongs to, otherwise the segment, item or "
                            "group the row belongs to ('' when none), a part that is missing on a row left out together with its "
                            "separator, never kept as an empty part such as ' |  | '"),
        "rank": ("1 = the most urgent row: the rows that pass {decision_test} first, then the other rows whose "
                 "data_quality_flag is 'ok', then the flagged rows, each group by value_at_stake, highest first, then by value, "
                 "so a row without enough evidence never outranks a row with it"),
        "priority_tier": "{tier_rule}",
        "top_reasons": ("up to 3 plain reasons for the row's value, each a distinct fact about this row with its number, one fact per "
                        "reason, never several joined in one: a fact may carry the reference it is compared with after ', against' "
                        "('134 days since the last catalog order, against a typical 162 for its group'), and a test's result is "
                        "its own reason ('not significant (q-value 0.674)'), never joined to the value it tested; fewer when "
                        "fewer facts apply, never a repeated reason or one that restates a count another reason gives, filler text "
                        "written to reach 3, or a reason about a value the row lacks, which its data_quality_flag names instead, "
                        "and never a reason with the same text on every row, such as a rate that is 0% for every item; a score "
                        "whose scale a reader cannot judge, such as a PageRank or an anomaly score, is given as its rank by that "
                        "score alone among every row that has the score, flagged rows among them, whose count is the total the "
                        "reason states ('12th of 4,000 senders by influence'), from `rank(method='min')` on "
                        "the score with 1 for the strongest, never the row's own rank, which also sorts on the decision test, the "
                        "flag and the expected impact, and never bare; a row without a score has no such rank: keep the default "
                        "`na_option='keep'`, never 'bottom' or 'top', which give every unscored row a rank past the last scored "
                        "one ('2,790th of 2,789'), so keep_reasons leaves the fact out, and the step asserts that no score rank "
                        "exceeds the total its reason states; a unit below the test's minimum states its count "
                        "and the minimum, never a test result, and a row the test skipped for another reason, such as a missing "
                        "input its data_quality_flag names, never states them, so no row reads '294 peer lines, the median "
                        "needs 5': the step adds that fact only to the rows below the minimum, never to the facts of every row, "
                        "since keep_reasons drops only a fact whose number is missing and so moves a fact listed for every row "
                        "into the reasons of a row that lacks an earlier fact, as when lines missing a sales price read '1.13K "
                        "peer lines, the median needs 5', puts it first among a below-minimum row's facts, and asserts before "
                        "the write that no row whose count reaches the minimum gives it; that count reason is the only one of "
                        "its row that states the count, so the row's rate reads "
                        "'repeat rate 100%, against a baseline of 35%', never 'repeat rate 100% over 3 customers'; the row's "
                        "own value is never a reason by itself ('lapse risk 50.3%' on a row whose value is that risk), since the "
                        "value column holds it, and a reason states it only with the reference it is compared against, as in "
                        "that rate; and a reference names the rows it covers: pandas `merge` matches a missing key to "
                        "every other missing key, so a row whose group key is missing is compared with a group it belongs to, such "
                        "as its carrier's median for a line with no known lane, or reads 'against a median of 15 for lines with no "
                        "known lane', never 'against a lane peer median of 15'"),
        "recommended_action": ("the action text of the row's priority tier (below); an 'Act now' or 'Review' row continues with ': ' "
                               "and its own next step for {action_target}, from a fixed template in section 7 filled with the row's "
                               "computed facts (its entity, main driver or reason, planned change); never generic text and never "
                               "written by a model"),
        INSPIRE_AI_FACTS: ("the facts Inspire's AI explains the record from, on every row, as the AI facts rule below says; never "
                           "written by a model"),
        INSPIRE_CONFIDENCE: ("'High', 'Medium' or 'Low' on every row, set by one rule in code from the record's evidence, as the AI "
                             "facts rule below says; never written by a model"),
        "data_quality_flag": ("'ok', or the reason this row's own inputs are incomplete or too sparse, or the flag these rules set on "
                              "every row when a check of the whole run fails; never a property every row of the population has by its "
                              "definition, such as the outcome window still open on every row a model scores, which the summary "
                              "step states once; never NULL: set it on the final frame, after its last merge "
                              "and before the rank step, so a row a merge left without a flag gets the reason its inputs are missing; "
                              "a result is never a data quality flag: a finding that is "
                              "not significant, below a cutoff or without a planned change keeps 'ok' and says so in its top reasons; "
                              "other run-level notes, such as rows excluded before a fit, go to the summary step, not to every row"),
        "method": "the accepted method's name",
        "model_version": "the registered model version ('' when nothing is registered)",
        "run_id": ("one id per run of the use case: a uuid4 made once by the first step of `run_use_case()` and written on every "
                   "row, never the MLflow run id, which exists only when a model is logged"),
        "use_case_id": "always '{use_case_id}'",
    }

    _GENIE_TIER_RULES = {
        "predicted": ("the use case's decision test",
                      "'Act now' for the top {act_now} by rank of the rows that pass the use case's decision test (significant, past "
                      "the cutoff, a planned change, enough evidence, an expected impact that is not a loss), 'Review' for the next "
                      "{review} of those rows, 'Monitor' for every other row, including every row whose data_quality_flag is not 'ok'"),
        "measured": ("a value_at_stake of at least `review_amount`",
                     "'Act now' for every row whose value_at_stake is at least `act_now_amount`, 'Review' for every other row whose "
                     "value_at_stake is at least `review_amount`, 'Monitor' for every other row, including every row whose "
                     "value_at_stake is NULL or whose data_quality_flag is not 'ok'; inspire_write_result refuses rows whose tier "
                     "does not follow this rule"),
    }

    @classmethod
    def _genie_output_column_lines(cls, spec: dict) -> list:
        decision_test, tier_rule = cls._GENIE_TIER_RULES[spec["decision_basis"]]
        values = {"value_type": spec["value_type"], "use_case_id": spec["use_case_id"], "decision_test": decision_test,
                  "tier_rule": tier_rule.format(act_now=f"{INSPIRE_TIER_SHARES['act_now']:.0%}",
                                                review=f"{INSPIRE_TIER_SHARES['review']:.0%}"),
                  "action_target": spec["action_target"]}
        lines = [f"{name} {sql_type}: {cls._GENIE_OUTPUT_COLUMN_MEANINGS[name].format(**values)}" for name, sql_type in INSPIRE_CORE_COLUMNS]
        return lines + ["Action text per priority tier: " + "; ".join(f"'{tier}' = \"{text}\"" for tier, text in spec["action_by_tier"].items()) + "."]

    def _build_code_structure_mandate(self, spec: dict) -> str:
        family = spec["family"]
        lines = [f"Execution pattern: family {family}, {GENIE_FAMILY_SPECS[family]['label']}.",
                 "Run pattern: " + GENIE_FAMILY_SPECS[family]["pattern"].format(model_name=spec.get("model_name") or "") + "."]
        if family == "E":
            horizon = f"{spec['horizon']} periods" if spec.get('horizon') else "choose it from the decision cadence and say why"
            lines.append(f"Forecast horizon: {horizon}; season length: {spec.get('season_length') or 'choose it from the time column and say why'}.")
        packages = self._genie_family_pins(family)
        lines.append(f"Environment: serverless environment version {GENIE_NOTEBOOK_ENV_VERSION}; "
                     + (f"install exactly {', '.join(packages)} with one %pip line in the first cell you add, alone in that cell, "
                        "because it restarts Python and drops every variable defined before it; that cell goes directly below the "
                        "instruction cell, never above Inspire's cells." if packages else "no extra packages."))
        own_calls = ("It is the default of the `ai_endpoint` widget, which inspire_write_result and inspire_write_summary use; a step "
                     "of the method that calls ai_query itself calls it with failOnError => false and does not set a temperature.")
        if spec.get("ai_endpoint"):
            lines.append(f"AI endpoint: `{spec['ai_endpoint']}`. {own_calls}")
        else:
            lines.append("AI endpoint: a Foundation Model API chat endpoint that is available in this workspace; name it in the instruction. "
                         + own_calls)
        lines.append(f"Business output table {spec['result_table']} (Delta, written only with inspire_write_result, which keeps every "
                     "earlier version of each record as change history and never touches another use case's rows), columns:")
        *columns, actions = self._genie_output_column_lines(spec)
        lines.extend(f"- {line}" for line in columns)
        lines.append(actions)
        lines.append("Column names: " + GENIE_COLUMN_NAMING_RULE)
        lines.append("Date windows: " + GENIE_DATE_FRAME_RULE)
        if family in GENIE_OUTCOME_WINDOW_FAMILIES:
            lines.append("Outcome windows: " + GENIE_CLOSED_WINDOW_RULE)
        lines.append("Significance tests: " + GENIE_SIGNIFICANCE_RULE)
        lines.append("Per-unit statistics: " + GENIE_PER_UNIT_SAMPLE_RULE)
        lines.append("Weights and signed measures: " + GENIE_WEIGHT_CHECK_RULE)
        lines.append("Numbers in text: " + GENIE_NUMBER_FORMAT_RULE)
        lines.append("AI facts: " + GENIE_EXPLANATION_FACTS_RULE)
        return "\n".join(lines)

    @staticmethod
    def _genie_notebook_object(cells: list, notebook_filename: str) -> dict:
        return {
            "cells": cells,
            "metadata": {
                "application/vnd.databricks.v1+notebook": {
                    "computePreferences": None, "dashboards": [],
                    "environmentMetadata": {"base_environment": "", "environment_version": GENIE_NOTEBOOK_ENV_VERSION},
                    "inputWidgetPreferences": None, "language": "python",
                    "notebookMetadata": {"pythonIndentUnit": 4}, "notebookName": notebook_filename, "widgets": {},
                },
                "language_info": {"name": "python"},
            },
            "nbformat": 4, "nbformat_minor": 0,
        }

    def _build_schema_details_from_use_cases(self, use_cases: list) -> list:
        self._ensure_genie_metadata_loader()
        tables = []
        for uc in use_cases or []:
            for name in self._genie_involved_tables(uc):
                if name.lower() not in {t.lower() for t in tables}:
                    tables.append(name)
        evidence = self._genie_table_evidence(tables)
        schema_details = []
        for fqn in sorted(evidence):
            catalog, schema_name, table_name = fqn.split('.')
            for col in (evidence[fqn].get('columns') or {}).values():
                schema_details.append((catalog, schema_name, table_name, col.get('name'), col.get('type') or 'unknown', col.get('comment') or ''))
        missing = [t for t in tables if t.lower() not in evidence]
        if missing:
            self.logger.warning(f"[GENIE-METADATA] {len(missing)}/{len(tables)} table(s) cited by the use cases are not in live Unity Catalog metadata and are left out of the Genie schema: {missing[:20]}")
        return schema_details

    def _genie_metadata_loader(self):
        return getattr(self, "data_loader", None) or getattr(self, "_genie_metadata_only_loader", None)

    def _ensure_genie_metadata_loader(self):
        loader = self._genie_metadata_loader()
        if loader is not None:
            return loader
        try:
            loader = DataLoader(catalogs="", schemas="", tables="", logger=self.logger,
                                enable_two_pass=False, enable_column_sampling=False)
        except Exception as e:
            self.logger.warning(f"[GENIE-METADATA] Live metadata loader unavailable ({get_clean_error_message(e)}); "
                                f"Genie references cannot be checked against Unity Catalog metadata")
            return None
        self._genie_metadata_only_loader = loader
        self.logger.info("[GENIE-METADATA] This run has no scan loader; Genie grounding reads Unity Catalog metadata through a metadata-only loader")
        return loader

    @staticmethod
    def _genie_table_names(text) -> list:
        names = []
        for part in re.split(r'[,\s]+', str(text or '')):
            name = part.replace('`', '').strip().rstrip('.;:)').lstrip('(')
            if name.count('.') == 2 and all(name.split('.')) and name.lower() not in {n.lower() for n in names}:
                names.append(name)
        return names

    def _genie_involved_tables(self, use_case: dict) -> list:
        names = self._genie_table_names(use_case.get('Tables Involved'))
        for extra in (use_case.get('Primary Table'), use_case.get('primary_table')):
            for name in self._genie_table_names(extra):
                if name.lower() not in {n.lower() for n in names}:
                    names.append(name)
        return names

    def _genie_table_evidence(self, tables) -> dict:
        loader = self._genie_metadata_loader()
        wanted = []
        for table in tables or ():
            name = str(table or '').replace('`', '').strip()
            if name.count('.') == 2 and name.lower() not in {w.lower() for w in wanted}:
                wanted.append(name)
        if loader is None or not wanted:
            return {}
        try:
            loader.infer_scope_joins(wanted)
        except SparkWorkCancelled:
            raise
        except Exception as e:
            self.logger.warning(f"[SCOPE-JOINS] join inference failed for {len(wanted)} table(s) ({get_clean_error_message(e)}); "
                                f"the Genie instruction lists declared joins only")
        try:
            evidence = loader.get_metadata_evidence(wanted) or {}
        except Exception as e:
            self.logger.warning(f"[GENIE-METADATA] Live metadata lookup failed for {len(wanted)} table(s): {get_clean_error_message(e)}")
            return {}
        return {str(fqn).lower(): entry for fqn, entry in evidence.items()}

    def _genie_date_tables(self, evidence: dict) -> dict:
        undated = [t for t, entry in evidence.items()
                   if not any(isinstance(c, dict) and _metadata_type_family(c.get("type")) == "temporal"
                              for c in (entry.get("columns") or {}).values())]
        linked = self._evidence_linked_tables({t: evidence[t] for t in undated})
        linked = [t for t in linked if t not in evidence]
        if not linked:
            return {}
        linked_evidence = self._genie_table_evidence(linked)
        dated = self._elite_dated_joins({**evidence, **linked_evidence}, undated)
        return {info["table"]: linked_evidence[info["table"]] for info in dated.values() if info["table"] in linked_evidence}

    def _build_genie_schema_index(self, full_schema_details: list) -> dict:
        schema_by_table = defaultdict(list)
        for detail in full_schema_details or []:
            try:
                (catalog, schema, table, column_name, data_type, comment) = detail
            except (TypeError, ValueError):
                continue
            fqtn = f"{catalog}.{schema}.{table}"
            schema_by_table[fqtn].append(detail)
            schema_by_table[f"`{catalog}`.`{schema}`.`{table}`"] = schema_by_table[fqtn]
        self._genie_schema_index_cache = dict(schema_by_table)
        self.logger.info(f"   ✓ Genie schema index built with {len(schema_by_table)} table entries")
        return self._genie_schema_index_cache

    @staticmethod
    def _genie_index_rows(schema_index: dict, full_schema_details: list = None) -> dict:
        rows = {}
        if schema_index is None:
            schema_index = defaultdict(list)
            for detail in full_schema_details or []:
                if len(detail) >= 6:
                    schema_index[f"{detail[0]}.{detail[1]}.{detail[2]}"].append(detail)
        for key, details in (schema_index or {}).items():
            fqn = str(key).replace('`', '').strip().lower()
            if fqn.count('.') == 2:
                rows.setdefault(fqn, details or [])
        return rows

    def _compute_per_uc_genie_fields(self, use_case: dict, full_schema_details: list, schema_index: dict = None) -> dict:
        """Compute the four per-use-case Genie-prompt fields from live Unity Catalog metadata.

        Single source of truth used by:
          - Discover mode (persists fields on the use case + tracking table)
          - Generate mode (recomputed whenever live metadata is reachable)

        Only tables that exist in live metadata (the metadata loader, else the run's schema index)
        are listed; their full column lists (types and descriptions) are rendered with the shared
        schema formatter, trimmed to the evidence-ranked columns for very wide tables.

        Returns a dict with keys:
          directly_involved_schema, foreign_key_relationships, available_tables_listing,
          technique_implementation_guidance, verified_tables
        """
        involved = self._genie_involved_tables(use_case)
        index_rows = self._genie_index_rows(schema_index, full_schema_details)
        evidence = self._genie_table_evidence(involved)
        date_tables = self._genie_date_tables(evidence)
        if date_tables:
            evidence.update(date_tables)
            involved = involved + sorted(date_tables)
            self.logger.info(f"[GENIE-DATE-TABLES FIRED] {use_case.get('No', 'UNKNOWN')}: {sorted(date_tables)} added to TABLES, "
                             f"since a listed table holds its time only as a key to them")
        validated, details = [], []
        for name in involved:
            fqn = name.lower()
            if fqn in evidence:
                catalog, schema_name, table_name = fqn.split('.')
                validated.append(fqn)
                details.extend((catalog, schema_name, table_name, col.get('name'), col.get('type') or 'unknown', col.get('comment') or '')
                               for col in (evidence[fqn].get('columns') or {}).values())
            elif index_rows.get(fqn):
                validated.append(fqn)
                details.extend(index_rows[fqn])
        dropped = [name for name in involved if name.lower() not in validated]
        if dropped:
            self.logger.info(f"[GENIE-METADATA] {use_case.get('No', 'UNKNOWN')}: {len(dropped)} declared table(s) not found in live metadata, left out of the Genie schema: {dropped}")

        if details:
            directly_involved_schema = self._enforce_schema_superset(
                details, sorted(set(validated)),
                self._format_schema_for_prompt(details, max_column_comment=160, max_table_comment=300),
                max_column_comment=160, max_table_comment=300,
            ) + f"\n({len(validated)} table(s); columns, types and descriptions come from live Unity Catalog metadata.)"
        else:
            directly_involved_schema = "No schema available"

        fk_lines = set()
        for fqn in validated:
            for fk in (evidence.get(fqn) or {}).get('foreign_keys') or []:
                line = f"{fqn}.{fk.get('column')} → {fk.get('ref_table')}.{fk.get('ref_column')}"
                fk_lines.add(line + (" (inferred)" if fk.get('source') == 'inferred' else ""))
        fk_relationships_md = "\n".join(f"- {rel}" for rel in sorted(fk_lines)) if fk_lines else "None"

        primary_listing = ', '.join(validated) if validated else '(none verified in live metadata)'
        other_tables = sorted((set(index_rows) | set(evidence)) - set(validated))
        if other_tables:
            available_tables_listing = (
                f"Primary tables (listed above): {primary_listing}\n\n"
                f"Additional available tables ({len(other_tables)} tables -- you may use any for enrichment):\n"
                + "\n".join(f"- {tbl}" for tbl in other_tables)
            )
        else:
            available_tables_listing = f"Only the primary tables are available: {primary_listing}"

        self._canonical_analytics_technique(use_case.get('Analytics Technique'), use_case.get('No'), "Genie instruction")
        technique_guidance = TECHNIQUE_IMPLEMENTATION_GUIDANCE.get(self._uc_guidance_technique(use_case), "")

        return {
            "directly_involved_schema": directly_involved_schema,
            "foreign_key_relationships": fk_relationships_md,
            "available_tables_listing": available_tables_listing,
            "technique_implementation_guidance": technique_guidance,
            "verified_tables": list(validated),
        }
    
    def _get_enriched_context_for_prompt(self) -> dict:
        """Collect the 6 enriched business-context fields (with defaults) used by the Genie prompt.
        Reads from self.merged_business_context (populated during Discover) or from a pre-loaded
        dict (used by Generate mode when replaying from the session table)."""
        enriched_ctx = getattr(self, 'merged_business_context', {}) or {}
        enriched_business_context = enriched_ctx.get('business_context', '') or self._get_business_context_fallback()
        
        raw_goals = enriched_ctx.get('strategic_goals', '')
        if isinstance(raw_goals, list):
            enriched_strategic_goals = ', '.join(raw_goals) if raw_goals else ''
        else:
            enriched_strategic_goals = str(raw_goals).strip() if raw_goals else ''
        if not enriched_strategic_goals or len(enriched_strategic_goals) < 10:
            enriched_strategic_goals = 'UNKNOWN_FROM_METADATA (strategic goals): downstream prompts MUST NOT invent goals; rely on schema + widget inputs only'
        
        raw_priorities = enriched_ctx.get('business_priorities', '')
        if isinstance(raw_priorities, list):
            enriched_business_priorities = ', '.join(raw_priorities) if raw_priorities else ''
        else:
            enriched_business_priorities = str(raw_priorities).strip() if raw_priorities else ''
        if not enriched_business_priorities or len(enriched_business_priorities) < 10:
            enriched_business_priorities = 'UNKNOWN_FROM_METADATA (business priorities): downstream prompts MUST NOT invent priorities; rely on schema + widget inputs only'
        
        raw_initiative = enriched_ctx.get('strategic_initiative', '')
        enriched_strategic_initiative = str(raw_initiative).strip() if raw_initiative else 'UNKNOWN_FROM_METADATA (strategic initiative)'
        
        raw_value_chain = enriched_ctx.get('value_chain', '')
        enriched_value_chain = str(raw_value_chain).strip() if raw_value_chain else 'UNKNOWN_FROM_METADATA (value chain)'
        
        raw_revenue_model = enriched_ctx.get('revenue_model', '')
        enriched_revenue_model = str(raw_revenue_model).strip() if raw_revenue_model else 'UNKNOWN_FROM_METADATA (revenue model)'
        
        return {
            "enriched_business_context": enriched_business_context,
            "enriched_strategic_goals": enriched_strategic_goals,
            "enriched_business_priorities": enriched_business_priorities,
            "enriched_strategic_initiative": enriched_strategic_initiative,
            "enriched_value_chain": enriched_value_chain,
            "enriched_revenue_model": enriched_revenue_model,
        }
    
    def _persist_session_prompt_context(self):
        """Persist the 7 per-session Genie-prompt context fields to __inspire_session so
        Generate mode can rebuild prompts without re-running business-context extraction.
        Called once after `merged_business_context` is populated. Best-effort (won't fail run)."""
        if not self.atomic_writer or not self.atomic_writer.enabled:
            return
        try:
            ctx = self._get_enriched_context_for_prompt()
            ctx["generation_instructions_section"] = self._get_generation_instruction_for_prompt(
                "USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT"
            ) or ""
            self.atomic_writer.update_session_prompt_context(ctx)
        except Exception as e:
            self.logger.debug(f"Persisting session prompt context failed (non-critical): {get_clean_error_message(e)}")

    _GENIE_PROMPT_FIELD_DEFAULTS = {
        "directly_involved_schema": "No schema available",
        "foreign_key_relationships": "None",
        "available_tables_listing": "",
        "technique_implementation_guidance": "",
    }

    def _genie_prompt_fields(self, use_case: dict, full_schema_details: list, schema_index: dict = None) -> dict:
        if use_case.get('_genie_fields_cached') and self._genie_metadata_loader() is None and not schema_index and not full_schema_details:
            self.logger.warning(f"[GENIE-METADATA] [{use_case.get('No', 'UNKNOWN')}] no live metadata in this run; replaying the Genie prompt fields persisted at Discover time")
            fields = {key: use_case.get(key) or default for key, default in self._GENIE_PROMPT_FIELD_DEFAULTS.items()}
        else:
            fields = self._compute_per_uc_genie_fields(use_case, full_schema_details, schema_index)
            use_case.update(fields)
        fields["technique_implementation_guidance"] = TECHNIQUE_IMPLEMENTATION_GUIDANCE.get(self._uc_guidance_technique(use_case), "")
        use_case["technique_implementation_guidance"] = fields["technique_implementation_guidance"]
        return fields

    def _genie_validation_mode(self) -> str:
        if self._is_ultra_high_mode():
            return "strict"
        mode = os.environ.get('INSPIRE_GENIE_HALLUCINATION_MODE', 'soft').strip().lower()
        if mode in ('off', 'disabled'):
            return "off"
        return "strict" if mode == "strict" else "soft"

    @staticmethod
    def _genie_repair_feedback(problems: list, claims: list = None, length: int = 0, conflicts: list = None, draft: str = "") -> str:
        parts = ["\n\n### REPAIR REQUIRED: THE PREVIOUS DRAFT FAILED VALIDATION"]
        if length > GENIE_INSTRUCTION_MAX_CHARS:
            parts.append(f"The previous draft was {length} characters long; keep the rewritten instruction under {GENIE_INSTRUCTION_MAX_CHARS} characters.")
        if problems:
            parts.append("The previous draft cited these identifiers, which do not exist in the Unity Catalog metadata of this workspace:\n"
                         + "\n".join(f"- `{problem}`" for problem in problems))
        if claims:
            parts.append("The previous draft stated numbers that are not in the inputs (invented impact, accuracy or money figures):\n"
                         + "\n".join(f"- {claim}" for claim in claims))
        if conflicts:
            parts.append(f"The previous draft words acceptance or a stop differently from Inspire's {GENIE_ACCEPTANCE_SECTION_TITLE} section, "
                         "which Inspire appends word for word and which wins over every other section. Rewrite or delete each of these "
                         "sentences so it agrees with that section:\n" + "\n".join(f"- {conflict}" for conflict in conflicts))
        parts.append("Rewrite the complete 8-section instruction. Cite only the tables and columns listed above, the tables under OTHER TABLES IN SCOPE, "
                     "and the notebook's own output objects. Delete every invented number or write it as [ESTIMATE: depends on <table.column>]. "
                     "When data the approach needs is missing, describe it by category in prose instead of naming a table or column that does not exist.")
        if draft:
            parts.append("Your previous draft follows. Keep everything in it that the list above does not name, and change what it names:\n"
                         "<<<PREVIOUS DRAFT\n" + draft.strip() + "\nPREVIOUS DRAFT>>>")
        return "\n".join(parts)

    def _call_genie_instruction_llm(self, use_case_id: str, prompt_vars: dict, skip_cache: bool, step_suffix: str = "") -> str:
        response = self.ai_agent.run_worker(
            step_name=f"Genie_Instruction_{use_case_id}{step_suffix}",
            worker_prompt_path="USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT",
            prompt_vars=prompt_vars,
            response_schema=None,
            timeout_override=420,
            skip_cache=skip_cache
        )
        text = str(response or '').strip()
        return sanitize_llm_special_chars(text) if len(text) > 100 else ''

    def _generate_genie_instruction_for_use_case(self, use_case: dict, full_schema_details: list, document_context: str, schema_index: dict = None) -> dict:
        import time
        use_case_id = use_case.get('No', 'UNKNOWN')

        try:
            start_time = time.time()
            self.logger.info(f"🤖 [{use_case_id}] Starting Genie Code instruction generation...")

            tables_involved_str = use_case.get('Tables Involved', '')
            technique = self._canonical_analytics_technique(use_case.get('Analytics Technique'), use_case_id, "Genie instruction")
            use_case['Analytics Technique'] = technique
            fields = self._genie_prompt_fields(use_case, full_schema_details, schema_index)
            directly_involved_schema = fields["directly_involved_schema"]
            fk_relationships_md = fields["foreign_key_relationships"]
            available_tables_listing = fields["available_tables_listing"]

            _enriched = self._get_enriched_context_for_prompt()
            spec = self._genie_notebook_spec(use_case)
            decision = {key: text for key, _label, text in self._genie_decision_contract(use_case)}
            family = spec["family"]

            prompt_vars = {
                "use_case_id": use_case_id,
                "use_case_name": use_case.get('Name', ''),
                "business_domain": use_case.get('Business Domain', ''),
                "subdomain": use_case.get('Subdomain', ''),
                "use_case_type": use_case.get('type', ''),
                "analytics_technique": technique,
                "technique_implementation_guidance": fields["technique_implementation_guidance"],
                "statement": use_case.get('Statement', ''),
                "solution": use_case.get('Solution', ''),
                "business_value": use_case.get('Business Value', ''),
                "high_level_design": use_case.get('High Level Design', ''),
                "tables_involved": tables_involved_str,
                "primary_table": use_case.get('Primary Table', ''),
                "beneficiary": self._card_text(use_case.get('Beneficiary')) or self._GENIE_NOT_ASSESSED,
                "sponsor": self._card_text(use_case.get('Sponsor')) or self._GENIE_NOT_ASSESSED,
                "directly_involved_schema": directly_involved_schema,
                "foreign_key_relationships": fk_relationships_md,
                "available_tables_listing": available_tables_listing,
                "business_name": self.business_name,
                "enriched_business_context": _enriched["enriched_business_context"],
                "enriched_strategic_goals": _enriched["enriched_strategic_goals"],
                "enriched_business_priorities": _enriched["enriched_business_priorities"],
                "enriched_strategic_initiative": _enriched["enriched_strategic_initiative"],
                "enriched_value_chain": _enriched["enriched_value_chain"],
                "enriched_revenue_model": _enriched["enriched_revenue_model"],
                "result_table": spec["result_table"],
                "model_name": spec["model_name"] or "none (this execution pattern registers no model)",
                "execution_family": f"family {family}, {GENIE_FAMILY_SPECS[family]['label']}",
                "code_structure_mandate": self._build_code_structure_mandate(spec),
                "readiness_thresholds": self._genie_threshold_text(spec),
                "widget_table": self._genie_widget_table(self._genie_core_widgets(spec)),
                "acceptance_stop_rule": self._genie_acceptance_stop_rule(spec),
                "unreached_quantile_rule": "\n- " + GENIE_UNREACHED_QUANTILE_RULE if family in GENIE_OUTCOME_WINDOW_FAMILIES else "",
                "model_feature_rule": "\n- " + GENIE_MODEL_FEATURE_RULE if family in INSPIRE_MODEL_FAMILIES else "",
                "generation_instructions_section": self._get_generation_instruction_for_prompt("USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT"),
            }
            prompt_vars.update(decision)

            prompt_vars_size = sum(len(str(v)) for v in prompt_vars.values())
            self.logger.info(
                f"📏 [{use_case_id}] Genie prompt vars: ~{prompt_vars_size:,} chars total "
                f"(schema={len(directly_involved_schema):,}, fk={len(fk_relationships_md):,}, avail_tables={len(available_tables_listing):,}, family={family})"
            )

            validation_mode = self._genie_validation_mode()
            draft = self._call_genie_instruction_llm(use_case_id, prompt_vars, bool(use_case.get('_skip_llm_cache', False)))
            genie_instruction = self._apply_genie_instruction_contract(use_case_id, draft, spec) if draft else draft
            if not genie_instruction:
                use_case['genie_instruction'] = ''
                use_case['genie_instruction_status'] = 'failed'
                self.logger.error(f"❌ [{use_case_id}] Genie instruction generation returned empty/short response")
                return use_case

            strict = validation_mode == 'strict'
            for round_no in range(GENIE_INSTRUCTION_REPAIR_ROUNDS + 1):
                ok, problems, claims, live = True, [], [], False
                if validation_mode != 'off':
                    ok, problems = self._validate_genie_instruction_references(use_case, genie_instruction, schema_index=schema_index)
                    claims = self._find_unsupported_numeric_claims(genie_instruction)
                    live = use_case.get('genie_validation_basis') == 'live_metadata'
                conflicts = self._genie_contract_conflicts(genie_instruction, spec)
                too_long = validation_mode != 'off' and len(draft) > GENIE_INSTRUCTION_MAX_CHARS
                fix_refs = not ok and live
                fix_text = bool(claims or too_long) and (live or not strict)
                if not (fix_refs or fix_text or conflicts):
                    break
                reasons = ((problems if fix_refs else []) + (claims if fix_text else []) + conflicts
                           + ([f"draft over the {GENIE_INSTRUCTION_MAX_CHARS:,}-character target"] if too_long and fix_text else []))
                if round_no == GENIE_INSTRUCTION_REPAIR_ROUNDS:
                    self.logger.warning(f"[GENIE-REPAIR] [{use_case_id}] still failing after {GENIE_INSTRUCTION_REPAIR_ROUNDS} repair round(s): {reasons}")
                    break
                self.logger.warning(f"[GENIE-REPAIR] [{use_case_id}] round {round_no + 1}/{GENIE_INSTRUCTION_REPAIR_ROUNDS}: {len(problems)} unresolved "
                                    f"reference(s), {len(claims)} unsupported numeric claim(s), {len(conflicts)} sentence(s) against Inspire's "
                                    f"{GENIE_ACCEPTANCE_SECTION_TITLE}, draft {len(draft):,} chars; repair without cache, reading back the "
                                    f"previous draft: {reasons}")
                repair_vars = dict(prompt_vars)
                repair_vars["directly_involved_schema"] = directly_involved_schema + self._genie_repair_feedback(
                    problems if fix_refs else [], claims if fix_text else [], len(draft) if fix_text else 0, conflicts, draft)
                repaired = self._call_genie_instruction_llm(use_case_id, repair_vars, True,
                                                            step_suffix="_repair" if round_no == 0 else f"_repair{round_no + 1}")
                if not repaired:
                    self.logger.warning(f"[GENIE-REPAIR] [{use_case_id}] repair round {round_no + 1} returned no draft; the last draft is kept")
                    break
                draft = repaired
                genie_instruction = self._apply_genie_instruction_contract(use_case_id, repaired, spec)
            if validation_mode == 'off':
                use_case['genie_validation_basis'] = 'disabled'
                status = 'validation_errors' if conflicts else 'unverified'
            else:
                clean = ok and not claims
                if clean and live:
                    status = 'validation_errors' if conflicts else 'success'
                elif strict:
                    status = 'failed_validation'
                    if clean:
                        problems = [f"no live Unity Catalog metadata to validate against (basis: {use_case.get('genie_validation_basis')})"]
                else:
                    status = 'validation_errors' if not clean or conflicts else 'unverified'
            use_case['genie_validation_errors'] = list(problems)
            use_case['genie_numeric_claims'] = list(claims)
            use_case['genie_contract_conflicts'] = list(conflicts)
            if len(draft) > GENIE_INSTRUCTION_MAX_CHARS:
                self.logger.warning(f"[GENIE-SIZE] [{use_case_id}] the drafted instruction is {len(draft):,} characters ({len(genie_instruction):,} "
                                    f"with Inspire's additions), above the {GENIE_INSTRUCTION_MAX_CHARS:,} target; it is kept because length "
                                    f"alone is not a correctness failure")

            elapsed = time.time() - start_time
            if status == 'failed_validation':
                use_case['genie_instruction'] = ''
                use_case['genie_instruction_status'] = status
                self.logger.error(f"[GENIE-VALIDATE STRICT] [{use_case_id}] instruction NOT emitted -- unresolved references: {problems}; "
                                  f"unsupported numeric claims: {claims}")
                return use_case

            use_case['genie_instruction'] = genie_instruction
            use_case['genie_instruction_status'] = status
            if status == 'success':
                self.logger.info(f"✅ [{use_case_id}] Genie instruction generated and validated against live metadata ({len(use_case['genie_instruction']):,} chars, {elapsed:.1f}s)")
            else:
                self.logger.warning(f"⚠️ [{use_case_id}] Genie instruction emitted with status '{status}' ({len(problems)} unresolved reference(s), "
                                    f"{len(claims)} unsupported numeric claim(s), {len(conflicts)} sentence(s) against Inspire's "
                                    f"{GENIE_ACCEPTANCE_SECTION_TITLE}, {elapsed:.1f}s)")
            return use_case

        except Exception as e:
            err_msg = get_clean_error_message(e)
            is_timeout = 'timeout' in err_msg.lower() or 'timed out' in err_msg.lower()
            if is_timeout:
                self.logger.error(f"⏱️ [{use_case_id}] Genie instruction generation timed out: {err_msg}")
                use_case['genie_instruction'] = ''
                use_case['genie_instruction_status'] = 'timeout'
            else:
                self.logger.error(f"❌ [{use_case_id}] Genie instruction generation failed: {err_msg}")
                use_case['genie_instruction'] = ''
                use_case['genie_instruction_status'] = 'failed'
            return use_case

    def _build_notebook_workspace_paths(self, use_case: dict) -> tuple:
        """Return (notebook_dir, notebook_full_path, notebook_filename) for a use case.
        Single source of truth for every per-use-case notebook (skeleton, Genie, batch fallback)
        and for the notebook URL resolver, so one use case can never own two notebooks."""
        use_case_id = use_case.get('No', 'UNKNOWN')
        use_case_name = use_case.get('Name', '')
        domain = use_case.get('Business Domain', 'Other')
        subdomain = use_case.get('Subdomain', 'General')
        sanitized_domain = self._sanitize_name(domain)
        sanitized_subdomain = self._sanitize_name(subdomain)
        notebook_filename = f"{use_case_id} - {self._sanitize_name(use_case_name)}"
        notebook_dir = os.path.join(self.notebook_output_dir, sanitized_domain, sanitized_subdomain)
        notebook_full_path = os.path.join(notebook_dir, f"{notebook_filename}.ipynb")
        return notebook_dir, notebook_full_path, notebook_filename
    
    _GENIE_FINGERPRINT_RE = re.compile(r"(?:<!--\s*)?[*]?Inspire fingerprint: ([0-9a-f]{16})[^\n]*\n?")
    _GENIE_MARKER_FLAG_RE = re.compile(r"^(\s*generate_genie_code_instruction\s*=\s*)(True|False)\s*$", re.M)

    @classmethod
    def _notebook_fingerprint(cls, cells: list) -> str:
        import hashlib
        digest = hashlib.sha256()
        for cell in cells or []:
            source = cell.get("source", "")
            text = "".join(source) if isinstance(source, list) else str(source or "")
            text = cls._GENIE_MARKER_FLAG_RE.sub(r"\1False", cls._GENIE_FINGERPRINT_RE.sub("", text))
            lines = [line.rstrip() for line in text.replace("\r\n", "\n").split("\n")]
            while lines and not lines[-1]:
                lines.pop()
            while lines and not lines[0]:
                lines.pop(0)
            digest.update(f"{cell.get('cell_type')}\x00" .encode("utf-8") + "\n".join(lines).encode("utf-8") + b"\x01")
        return digest.hexdigest()[:16]

    @classmethod
    def _stored_fingerprint(cls, cells: list):
        for cell in cells or []:
            source = cell.get("source", "")
            match = cls._GENIE_FINGERPRINT_RE.search("".join(source) if isinstance(source, list) else str(source or ""))
            if match:
                return match.group(1)
        return None

    @classmethod
    def _stamp_notebook_fingerprint(cls, notebook_obj: dict) -> str:
        cells = notebook_obj.get("cells") or []
        fingerprint = cls._notebook_fingerprint(cells)
        markdown = [c for c in cells if c.get("cell_type") == "markdown"]
        target = next((c for c in markdown if "Generated by Databricks Inspire AI" in "".join(c.get("source") or [])), markdown[0] if markdown else None)
        if target is not None:
            target["source"] = list(target.get("source") or []) + [
                f"\n<!-- Inspire fingerprint: {fingerprint}. Inspire regenerates this notebook in place only while its cells still match this fingerprint. -->\n"]
        return fingerprint

    @staticmethod
    def _is_missing_workspace_object(exc) -> bool:
        text = f"{type(exc).__name__} {exc}"
        return any(token in text for token in ("NotFound", "ResourceDoesNotExist", "RESOURCE_DOES_NOT_EXIST")) or "does not exist" in text.lower()

    def _existing_notebook_state(self, notebook_full_path: str) -> tuple:
        from databricks.sdk.service import workspace
        candidates = [notebook_full_path[:-len(".ipynb")], notebook_full_path] if notebook_full_path.endswith(".ipynb") else [notebook_full_path]
        for candidate in candidates:
            try:
                exported = self.w_client.workspace.export(path=candidate, format=workspace.ExportFormat.JUPYTER)
            except Exception as err:
                if self._is_missing_workspace_object(err):
                    continue
                return "unreadable", get_clean_error_message(err)
            try:
                cells = json.loads(base64.b64decode(exported.content).decode("utf-8")).get("cells") or []
            except (ValueError, TypeError, AttributeError) as err:
                return "unreadable", str(err)
            stored = self._stored_fingerprint(cells)
            written_this_run = (getattr(self, "_genie_written_fingerprints", None) or {}).get(notebook_full_path)
            if stored and (stored == written_this_run or stored == self._notebook_fingerprint(cells)):
                return "untouched", None
            return "edited", None
        return "absent", None

    def _regenerated_notebook_path(self, notebook_dir: str, notebook_filename: str) -> str:
        prefix = f"{notebook_filename} (regenerated "
        try:
            siblings = sorted(str(getattr(entry, 'path', '') or '') for entry in self.w_client.workspace.list(notebook_dir)
                              if str(getattr(entry, 'path', '') or '').rsplit('/', 1)[-1].startswith(prefix))
        except Exception as err:
            self.logger.debug(f"Listing {notebook_dir} for regenerated notebooks failed: {get_clean_error_message(err)}")
            siblings = []
        if siblings:
            latest = siblings[-1] if siblings[-1].endswith(".ipynb") else siblings[-1] + ".ipynb"
            if self._existing_notebook_state(latest)[0] == "untouched":
                return latest
        stamp = datetime.datetime.now().strftime("%Y-%m-%d %H%M%S")
        return os.path.join(notebook_dir, f"{prefix}{stamp}).ipynb")

    def _write_notebook_to_workspace(self, notebook_dir: str, notebook_full_path: str, notebook_obj: dict, use_case_id: str) -> str:
        """Serialize a notebook dict and import it into the workspace with retry.
        Shared by every per-use-case notebook writer; raises after the final failed attempt. A notebook whose cells no
        longer match the fingerprint Inspire stamped at write time (edited by the user or Genie Code) is never
        overwritten: the new notebook is written beside it as '<name> (regenerated <timestamp>)'."""
        from databricks.sdk.service import workspace
        fingerprint = self._stamp_notebook_fingerprint(notebook_obj)
        state, detail = self._existing_notebook_state(notebook_full_path)
        if state in ("edited", "unreadable"):
            original = notebook_full_path
            filename = os.path.basename(notebook_full_path)[:-len(".ipynb")] if notebook_full_path.endswith(".ipynb") else os.path.basename(notebook_full_path)
            notebook_full_path = self._regenerated_notebook_path(notebook_dir, filename)
            notebook_obj.setdefault("metadata", {}).setdefault("application/vnd.databricks.v1+notebook", {})["notebookName"] = os.path.basename(notebook_full_path)[:-len(".ipynb")]
            self.logger.warning(f"[NOTEBOOK-PRESERVED] [{use_case_id}] {original} was edited after Inspire wrote it"
                                f"{' or could not be read (' + detail + ')' if detail else ''}; it is kept unchanged and the new notebook is written beside it: {notebook_full_path}. "
                                "If the kept notebook sets generate_genie_code_instruction = True, set it back to False so Route 2 stops regenerating it.")
        notebook_content_str = json.dumps(notebook_obj, indent=2)
        max_retries = self.max_retry_attempts + 1
        retry_delay = 2
        for attempt in range(1, max_retries + 1):
            try:
                self.w_client.workspace.mkdirs(notebook_dir)
                self.w_client.workspace.import_(
                    path=notebook_full_path, overwrite=True, format=workspace.ImportFormat.JUPYTER,
                    content=base64.b64encode(notebook_content_str.encode('utf-8')).decode(),
                )
                _status = self.w_client.workspace.get_status(notebook_full_path)
                if getattr(_status, 'object_id', None):
                    self._notebook_object_ids[use_case_id] = _status.object_id
                if getattr(self, "_genie_written_fingerprints", None) is None:
                    self._genie_written_fingerprints = {}
                self._genie_written_fingerprints[notebook_full_path] = fingerprint
                self.logger.info(f"✅ [{use_case_id}] Notebook saved: {_status.path}")
                return _status.path
            except Exception as err:
                self.logger.warning(f"[{use_case_id}] Notebook import attempt {attempt}/{max_retries} failed: {err}")
                if attempt < max_retries:
                    import time
                    time.sleep(retry_delay * (2 ** (attempt - 1)))
                else:
                    self.logger.error(f"❌ [{use_case_id}] Failed to import notebook after {max_retries} attempts")
                    raise

    def _write_genie_agents_file(self) -> str:
        if getattr(self, "_genie_agents_file_path", None):
            return self._genie_agents_file_path
        root = getattr(self, "notebook_output_dir", None)
        if not root or getattr(self, "w_client", None) is None:
            return ""
        from databricks.sdk.service import workspace
        path = os.path.join(root, "AGENTS.md")
        body = GENIE_AGENTS_FILE_TEMPLATE
        content = body + f"\n<!-- Inspire fingerprint: {self._notebook_fingerprint([{'cell_type': 'file', 'source': body}])} -->\n"
        try:
            exported = self.w_client.workspace.export(path=path, format=workspace.ExportFormat.AUTO)
            existing = base64.b64decode(exported.content).decode("utf-8", errors="replace")
            stored = self._stored_fingerprint([{"source": existing}])
            if not stored or stored != self._notebook_fingerprint([{"cell_type": "file", "source": existing}]):
                self.logger.warning(f"[GENIE-AGENTS] {path} was edited after Inspire wrote it; it is kept unchanged")
                self._genie_agents_file_path = path
                return path
        except Exception as err:
            if not self._is_missing_workspace_object(err):
                self.logger.warning(f"[GENIE-AGENTS] {path} could not be read ({get_clean_error_message(err)}); it is not overwritten")
                return ""
        try:
            self._import_workspace_file(path, base64.b64encode(content.encode("utf-8")).decode(), workspace.ImportFormat.AUTO)
        except Exception as err:
            self.logger.warning(f"[GENIE-AGENTS] writing {path} failed: {get_clean_error_message(err)}")
            return ""
        self._genie_agents_file_path = path
        self.logger.info(f"[GENIE-AGENTS] Genie Code instructions written to {path} ({len(content)} chars)")
        return path

    _GENIE_ESTIMATE_RE = re.compile(r"\[ESTIMATE:[^\]]*\]", re.IGNORECASE)
    _GENIE_CLAIM_NUMBER = (r"(?:~\s*|about |around |approximately |roughly |up to |over |more than |nearly |almost )?"
                           r"[$€£]?\d+(?:[.,]\d+)?(?:\s*(?:-|to)\s*[$€£]?\d+(?:[.,]\d+)?)?")
    _GENIE_NUMERIC_CLAIM_RES = (
        re.compile(r"\b(?:improv|increas|reduc|decreas|boost|lift|cut|lower|rais|grow|sav|shrink|accelerat)\w*\s+(?:[\w-]+\s+){0,6}?by\s+"
                   + _GENIE_CLAIM_NUMBER + r"\s*(?:%|percent\b|x\b|times\b|percentage points?\b|pp\b|k\b|m\b|million\b|billion\b)", re.IGNORECASE),
        re.compile(_GENIE_CLAIM_NUMBER + r"\s*(?:%|percent\b|x\b|times\b)\+?\)?\s*(?:[\w-]+\s+){0,2}?(?:improvement|increase|reduction|decrease|"
                   r"uplift|lift|gain|boost|savings?|growth|faster|better|fewer|more|less|lower|higher|accuracy|roi)\b", re.IGNORECASE),
        re.compile(r"[$€£]\s?\d[\d,.]*(?:\s*(?:k|m|mm|bn|b|million|billion|thousand)\b)?", re.IGNORECASE),
        re.compile(r"\broi\s+(?:of\s+)?(?:~\s*)?\d", re.IGNORECASE),
        re.compile(r"\b(?:accuracy|precision|recall|auc|f1|mape|rmse|error rate)\s+(?:[\w-]+\s+){0,2}?(?:of|by|to|is|at|reaches|reach)\s+"
                   r"(?:about\s+|around\s+|~\s*)?\d", re.IGNORECASE),
        re.compile(r"\b(?:improvement|increase|reduction|decrease|uplift|lift|gain|savings?|growth)\s*(?:of\s+|by\s+|\(|:)\s*"
                   r"(?:about\s+|around\s+|~\s*)?[$€£]?\d", re.IGNORECASE),
    )
    _GENIE_ALLOWED_NUMBER_CONTEXT_RE = re.compile(
        r"(?:\binterval\b|\bconfidence\b|\bcredible\b|\bquantiles?\b|\bpercentiles?\b|\bP\d{1,2}\b|\bthresholds?\b|\bat least\b|\bat most\b|"
        r"\bminimum\b|\bmaximum\b|>=|<=|\baccept\w*\b|\bmust\b|\brequire\w*\b)", re.IGNORECASE)

    @classmethod
    def _find_unsupported_numeric_claims(cls, text) -> list:
        cleaned = cls._GENIE_ESTIMATE_RE.sub(" ", str(text or ""))
        claims = []
        for clause in re.split(r"(?<=[.!?])\s+|\n+|;\s*", cleaned):
            clause = clause.strip()
            if not clause or cls._GENIE_ALLOWED_NUMBER_CONTEXT_RE.search(clause):
                continue
            if any(pattern.search(clause) for pattern in cls._GENIE_NUMERIC_CLAIM_RES):
                claims.append(clause[:200])
        return list(dict.fromkeys(claims))

    def _write_skeleton_notebook(self, use_case: dict) -> bool:
        use_case['generate_genie_code_instruction'] = 'No'
        use_case['has_genie_code'] = 'N'
        try:
            use_case['notebook_path'] = self._assemble_skeleton_notebook(use_case)
        except Exception as e:
            use_case['notebook_status'] = 'write_failed'
            use_case['notebook_error'] = get_clean_error_message(e)
            self.logger.error(f"❌ [{use_case.get('No', 'UNKNOWN')}] Skeleton notebook write failed: {use_case['notebook_error']}")
            return False
        use_case['notebook_status'] = 'skeleton'
        return True

    def _write_use_case_notebook(self, use_case: dict, translations: dict) -> bool:
        if not str(use_case.get('genie_instruction') or '').strip():
            return self._write_skeleton_notebook(use_case)
        try:
            notebook_path = self._assemble_genie_code_notebook(use_case, translations)
        except Exception as e:
            use_case['notebook_status'] = 'write_failed'
            use_case['notebook_error'] = get_clean_error_message(e)
            self.logger.error(f"❌ [{use_case.get('No', 'UNKNOWN')}] Genie notebook write failed; the existing notebook stays in place: {use_case['notebook_error']}")
            return False
        use_case['notebook_path'] = notebook_path
        use_case['notebook_status'] = 'written'
        use_case['has_genie_code'] = 'Y'
        use_case['generate_genie_code_instruction'] = 'No'
        return True

    def _deliver_genie_notebook(self, use_case: dict, translations: dict) -> bool:
        if use_case.get('genie_instruction_status') not in self._GENIE_EMITTED_STATUSES or not use_case.get('genie_instruction'):
            return False
        return self._write_use_case_notebook(use_case, translations)

    _ELITE_STATUS_LABELS = {
        "ELITE": "Elite: met the Ultra High bar",
        "DOMAIN_COVERAGE_PICK": "Domain coverage pick, below bar",
        "FALLBACK_BELOW_BAR": "Below the bar: closest candidate (no use case met the bar)",
    }
    _ELITE_CARD_FIELDS = (
        ("decision", "Decision Changed"),
        ("decision_owner", "Decision Owner"),
        ("cadence", "Decision Cadence"),
        ("kpi", "KPI"),
        ("kpi_direction", "KPI Direction"),
        ("value_driver", "Value Driver"),
        ("value_tier", "Value Tier"),
        ("signal_columns", "Signal Columns"),
        ("outcome_definition", "Outcome Definition"),
        ("time_column", "Time Column"),
        ("entity_key_column", "Entity Key Column"),
        ("join_path", "Join Path"),
        ("build_path", "Build Path"),
        ("blockers", "Blockers"),
        ("red_team", "Red Team"),
        ("red_team_verdict", "Red Team Verdict"),
        ("scores", "Elite Dimension Scores"),
        ("verdict", "Elite Verdict"),
        ("verdict_reason", "Verdict Reason"),
        ("verification", "Metadata Verification"),
    )

    @staticmethod
    def _card_text(value, separator=", ") -> str:
        if value is None:
            return ''
        if isinstance(value, dict):
            return "; ".join(f"{k}: {DatabricksInspire._card_text(v)}" for k, v in value.items() if DatabricksInspire._card_text(v))
        if isinstance(value, (list, tuple, set)):
            return separator.join(t for t in (DatabricksInspire._card_text(v) for v in value) if t)
        return " ".join(str(value).split()).replace("|", "/")

    def _build_use_case_details_markdown_source(self, use_case: dict) -> list:
        """Build the 'Use Case Details' markdown source list of a skeleton notebook.
        Elite selection fields (status, score, assessment) are added when the use case carries them."""
        use_case_id = use_case.get('No', 'UNKNOWN')
        use_case_name = use_case.get('Name', '')
        domain = use_case.get('Business Domain', 'Other')
        subdomain = use_case.get('Subdomain', 'General')
        def safe_val(val):
            return self._card_text(val) or 'N/A'
        rows = [
            f"## Use Case Details\n\n",
            f"| Aspect | Description |\n",
            f"|---|---|\n",
            f"| **ID** | {safe_val(use_case_id)} |\n",
            f"| **Name** | {safe_val(use_case_name)} |\n",
            f"| **Business Domain** | {safe_val(domain)} |\n",
            f"| **Subdomain** | {safe_val(subdomain)} |\n",
            f"| **Type** | {safe_val(use_case.get('type'))} |\n",
            f"| **Analytics Technique** | {safe_val(use_case.get('Analytics Technique'))} |\n",
            f"| **Priority** | {safe_val(use_case.get('Priority'))} |\n",
            f"| **Primary Table** | {safe_val(use_case.get('Primary Table'))} |\n",
            f"| **Tables Involved** | {safe_val(use_case.get('Tables Involved'))} |\n",
            f"| **Statement** | {safe_val(use_case.get('Statement'))} |\n",
            f"| **Solution** | {safe_val(use_case.get('Solution'))} |\n",
            f"| **Business Value** | {safe_val(use_case.get('Business Value'))} |\n",
            f"| **Quality** | {safe_val(use_case.get('Quality'))} |\n",
            f"| **Quality Reasons** | {safe_val(use_case.get('Quality Summary'))} |\n",
            f"| **Result Table** | {safe_val(use_case.get('result_table'))} |\n",
        ]
        if use_case.get('quality_mode'):
            rows.append(f"| **Quality Mode** | {safe_val(use_case.get('quality_mode'))} |\n")
        elite_status = use_case.get('elite_status')
        if elite_status:
            rows.append(f"| **Elite Status** | {safe_val(self._ELITE_STATUS_LABELS.get(elite_status, elite_status))} |\n")
        if use_case.get('elite_score') is not None:
            try:
                rows.append(f"| **Elite Score** | {float(use_case.get('elite_score')):.1f} / 10 |\n")
            except (TypeError, ValueError):
                rows.append(f"| **Elite Score** | {safe_val(use_case.get('elite_score'))} |\n")
        if use_case.get('coverage_pick_reason'):
            rows.append(f"| **Coverage Pick Reason** | {safe_val(use_case.get('coverage_pick_reason'))} |\n")
        if use_case.get('fallback_reason'):
            rows.append(f"| **Below Bar Reason** | {safe_val(use_case.get('fallback_reason'))} |\n")
        assessment = use_case.get('elite_assessment')
        if isinstance(assessment, dict):
            for key, label in self._ELITE_CARD_FIELDS:
                value = assessment.get(key)
                if key == "verification" and isinstance(value, dict):
                    failures = self._card_text(value.get('failures'))
                    text = "passed" if value.get('passed') else ("failed: " + failures if failures else "failed")
                elif key == "join_path":
                    text = self._card_text(value, separator=" -> ")
                else:
                    text = self._card_text(value)
                if text:
                    rows.append(f"| **{label}** | {text} |\n")
        return rows
    
    def _assemble_skeleton_notebook(self, use_case: dict) -> str:
        """Discover-mode notebook writer.
        
        Produces a 4-cell notebook:
          (1) Title + generation timestamp (markdown)
          (2) Use Case Details table (markdown, shared with full Genie notebook)
          (3) Marker code cell with INSPIRE_USE_CASE_ID, INSPIRE_SESSION_ID, and
              generate_genie_code_instruction = False. Setting the variable to True and
              re-running the notebook tells Route 2 of Generate mode to regenerate the
              full Genie Code Instruction for this use case.
          (4) Placeholder markdown explaining how to trigger Genie regeneration (via
              Inspire UI/SQL UPDATE for Route 1, or by flipping the marker variable for
              Route 2).
        """
        use_case_id = use_case.get('No', 'UNKNOWN')
        notebook_dir, notebook_full_path, notebook_filename = self._build_notebook_workspace_paths(use_case)
        
        final_cells = []
        how_to_run_source = [
            "> ## ⚠️ This notebook is a Genie Code stub, not a runnable pipeline\n",
            ">\n",
            "> No Genie Code instruction has been generated for this use case yet. To make it runnable:\n",
            ">\n",
            "> 1. Mark this use case for generation using one of the two routes in the marker cell below\n",
            "> 2. Run the Inspire notebook with **Operation = Generate Use Cases**; it rewrites this notebook with the Genie Code instruction\n",
            "> 3. Open the rewritten notebook, copy its instruction cell into Genie Code (**⌘K** on macOS, **Ctrl-K** on Windows/Linux) and let it run\n",
        ]
        final_cells.append({
            "cell_type": "markdown",
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": how_to_run_source
        })
        generation_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        title_source = [
            f"# {use_case_id}: {use_case.get('Name', '')}\n\n",
            f"*Generated by Databricks Inspire AI on {generation_timestamp}*\n\n",
            "**Mode:** Discover Use Cases (skeleton notebook, no Genie Code Instruction yet)\n\n",
            "**Disclaimer:** All code instructions are AI-generated examples and should be reviewed by a qualified engineer before being used in production. Databricks is not liable for any issues arising from this code.\n\n",
            "---\n"
        ]
        final_cells.append({
            "cell_type": "markdown",
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": title_source
        })
        final_cells.append({
            "cell_type": "markdown",
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": self._build_use_case_details_markdown_source(use_case)
        })
        
        escaped_id = str(use_case_id).replace('"', '\\"')
        marker_source = [
            "# =============================================================================\n",
            "# INSPIRE GENIE CODE INSTRUCTION: GENERATION MARKER\n",
            "# =============================================================================\n",
            "# This notebook was created by Inspire AI in \"Discover Use Cases\" mode.\n",
            "# The Genie Code Instruction has NOT been generated yet.\n",
            "#\n",
            "# TWO WAYS TO REGENERATE THIS USE CASE IN \"Generate Use Cases\" MODE:\n",
            "#\n",
            "#   (1) Database flag route (preferred, driven by the Inspire UI):\n",
            "#       UPDATE <inspire_database>.__inspire_usecases\n",
            "#          SET generate_genie_code_instruction = 'Yes'\n",
            "#        WHERE id = <this use_case_id> AND session_id = <this session_id>;\n",
            "#       Then re-run the Inspire notebook with Operation = 'Generate Use Cases'.\n",
            "#\n",
            "#   (2) Notebook flag route (fallback when no rows are flagged in the DB):\n",
            "#       Flip the variable below from False -> True, save the notebook, then\n",
            "#       re-run the Inspire notebook with Operation = 'Generate Use Cases'.\n",
            "#\n",
            "# After generation, Inspire rewrites this notebook in place (unless you edited it,\n",
            "# then the new notebook is written beside it) and the flag no longer exists.\n",
            "# =============================================================================\n",
            "\n",
            f'INSPIRE_USE_CASE_ID = "{escaped_id}"\n',
            f"INSPIRE_SESSION_ID = {self.session_id}\n",
            "generate_genie_code_instruction = False\n",
        ]
        final_cells.append({
            "cell_type": "code",
            "execution_count": 0,
            "outputs": [],
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": marker_source
        })
        
        placeholder_source = [
            "## Next Steps\n\n",
            "This is a **skeleton notebook**. To produce the full Genie Code Instruction:\n\n",
            "1. Review the use case details above and confirm this is a use case you want to implement.\n",
            "2. Mark it for generation using **one** of the routes documented in the marker cell.\n",
            "3. Re-run the Inspire notebook with `Operation = 'Generate Use Cases'`.\n\n",
            "Inspire then rewrites this notebook with the Genie Code instruction. Copy that instruction into\n",
            "Genie Code and let it run: Genie Code builds and runs the rest of the notebook.\n"
        ]
        final_cells.append({
            "cell_type": "markdown",
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": placeholder_source
        })
        
        return self._write_notebook_to_workspace(notebook_dir, notebook_full_path,
                                                 self._genie_notebook_object(final_cells, notebook_filename), use_case_id)

    _GENIE_EMITTED_STATUSES = ("success", "validation_errors", "unverified")
    _GENIE_NON_TABLE_NAMESPACES = frozenset({
        "catalog", "catboost", "client", "ctx", "cvxpy", "databricks", "datetime", "dbutils", "delta", "df",
        "geopandas", "h3", "hyperopt", "imblearn", "implicit", "json", "keras", "lifelines", "lightgbm", "math",
        "matplotlib", "mlflow", "mlxtend", "model", "networkx", "nltk", "np", "numpy", "optuna", "ortools", "os",
        "output_df", "pandas", "pd", "pipeline", "plotly", "plt", "pmdarima", "prophet", "pulp", "pyspark", "re",
        "requests", "sc", "scipy", "seaborn", "self", "shap", "shapely", "sklearn", "sksurv", "sns", "spacy",
        "spark", "statsmodels", "sys", "tensorflow", "textblob", "torch", "transformers", "w", "x_test",
        "x_train", "xgboost", "y_test", "y_train",
    })
    _GENIE_NON_TABLE_SUFFIXES = frozenset({
        "ai", "com", "csv", "gz", "htm", "html", "io", "ipynb", "json", "md", "net", "org", "parquet",
        "pdf", "pkl", "py", "txt", "xml", "yaml", "yml", "zip",
    })
    _GENIE_REFERENCE_RE = re.compile(
        r"(?<![A-Za-z0-9_.`-])(`?[A-Za-z_][A-Za-z0-9_-]*`?(?:\.`?[A-Za-z_][A-Za-z0-9_-]*`?)+)(?![A-Za-z0-9_`(-])"
    )

    def _genie_is_non_table_shape(self, parts: list, catalogs: set) -> bool:
        if parts[0] in catalogs:
            return False
        return (parts[0] in self._GENIE_NON_TABLE_NAMESPACES
                or parts[-1] in self._GENIE_NON_TABLE_SUFFIXES
                or parts[min(2, len(parts) - 1)] in self._GENIE_NON_TABLE_SUFFIXES
                or any(len(p) == 1 for p in parts))

    def _validate_genie_instruction_references(self, use_case: dict, genie_instruction: str, schema_index: dict = None) -> tuple:
        """Cross-check every table and column a generated Genie instruction cites against live
        Unity Catalog metadata.

        Returns (ok, unresolved_refs); records the verification basis on
        `use_case['genie_validation_basis']`:
          - 'live_metadata': the run's schema index (`schema_index`, else
            `self._genie_schema_index_cache`) plus metadata fetched through the metadata loader for
            every cited table. The use case's own `Tables Involved`, `Primary Table` and
            `available_tables_listing` are never trusted on this basis.
          - 'uc_declared': no live metadata source exists, so only references outside the use case's
            declared tables can be detected (a weak check, never reported as validated).
          - 'none': no baseline at all; nothing is flagged.

        Checked shapes: `catalog.schema.table`, `catalog.schema.table.column`, `schema.table.column`,
        `table.column[.field]` for tables the metadata knows, and backticked `table.column`. Columns are
        checked whenever the table's column list is known. `system.*`, `*.information_schema.*` and the
        objects the implementation itself creates (the use case's `result_table` and names that extend it,
        such as `<result_table>_model`) are always accepted; dotted names whose
        first segment is a code namespace (sklearn, mlflow, spark, ...) or that look like host or file
        names are not table references unless their first segment is a known catalog.
        """
        uc_id = use_case.get('No', 'UNKNOWN')
        if not genie_instruction:
            return True, []
        if schema_index is None:
            schema_index = getattr(self, '_genie_schema_index_cache', None)
        result_table = str(use_case.get('result_table') or '').replace('`', '').strip().lower()
        chains = []
        for match in self._GENIE_REFERENCE_RE.finditer(genie_instruction):
            raw = match.group(1)
            chains.append(([p.lower() for p in raw.replace('`', '').split('.')], '`' in raw))

        loader = self._genie_metadata_loader()
        involved = self._genie_involved_tables(use_case)
        index_rows = self._genie_index_rows(schema_index)
        evidence = self._genie_table_evidence(involved)
        known = set(index_rows) | set(evidence)
        if known or loader is not None:
            basis = 'live_metadata'
        else:
            known = set(t.lower() for t in involved)
            known.update(t.lower() for t in self._genie_table_names(use_case.get('available_tables_listing')))
            if not known:
                use_case['genie_validation_basis'] = 'none'
                self.logger.info(f"[GENIE-VALIDATE] {uc_id}: no live metadata and no declared tables; references not checked")
                return True, []
            basis = 'uc_declared'
        use_case['genie_validation_basis'] = basis

        def _model(tables):
            catalogs, by_short, by_pair = set(), defaultdict(list), defaultdict(list)
            for fqn in tables:
                cat, sch, tbl = fqn.split('.')
                catalogs.add(cat)
                by_short[tbl].append(fqn)
                by_pair[(sch, tbl)].append(fqn)
            return catalogs, by_short, by_pair

        def _pipeline_output(table3):
            return bool(result_table) and table3.startswith(result_table)

        catalogs, by_short, by_pair = _model(known)
        if loader is not None:
            pending = set()
            for parts, _ in chains:
                if len(parts) not in (3, 4) or parts[0] == 'system' or 'information_schema' in parts:
                    continue
                table3 = '.'.join(parts[:3])
                if _pipeline_output(table3) or table3 in known or (parts[0], parts[1]) in by_pair or parts[0] in by_short:
                    continue
                if not self._genie_is_non_table_shape(parts, catalogs):
                    pending.add(table3)
            if pending:
                evidence.update(self._genie_table_evidence(sorted(pending)))
                known |= set(evidence)
                catalogs, by_short, by_pair = _model(known)

        column_sets = {}

        def _table_columns(fqn):
            if fqn not in column_sets:
                if fqn in evidence:
                    cols = {str(col).lower() for col in (evidence[fqn].get('columns') or {})}
                else:
                    cols = {str(row[3]).lower() for row in index_rows.get(fqn) or [] if len(row) > 3 and row[3]}
                column_sets[fqn] = cols or None
            return column_sets[fqn]

        def _columns(fqns):
            cols = set()
            for fqn in fqns:
                if _table_columns(fqn) is None:
                    return None
                cols |= _table_columns(fqn)
            return cols

        unresolved = []
        for parts, quoted in chains:
            n = len(parts)
            if n > 4 or parts[0] == 'system' or 'information_schema' in parts:
                continue
            if n == 2:
                if (parts[0], parts[1]) in by_pair or not quoted or parts[0] not in by_short:
                    continue
                cols = _columns(by_short[parts[0]])
                if cols is not None and parts[1] not in cols:
                    unresolved.append('.'.join(parts))
                continue
            table3 = '.'.join(parts[:3])
            if _pipeline_output(table3):
                continue
            if table3 in known:
                cols = _table_columns(table3)
                if n == 4 and cols is not None and parts[3] not in cols:
                    unresolved.append('.'.join(parts))
                continue
            if n == 3 and (parts[0], parts[1]) in by_pair:
                cols = _columns(by_pair[(parts[0], parts[1])])
                if cols is not None and parts[2] not in cols:
                    unresolved.append(table3)
                continue
            if parts[0] in by_short:
                cols = _columns(by_short[parts[0]])
                if cols is not None and parts[1] not in cols:
                    unresolved.append('.'.join(parts[:2]))
                continue
            if self._genie_is_non_table_shape(parts, catalogs):
                continue
            unresolved.append(table3)

        unresolved = list(dict.fromkeys(unresolved))
        if unresolved:
            for ref in unresolved:
                self.logger.warning(f"[GENIE-VALIDATE] Unresolved reference: {ref}  (UC: {uc_id}, basis: {basis})")
        else:
            self.logger.info(f"[GENIE-VALIDATE] {uc_id}: 0 unresolved references across {len(chains)} dotted identifiers (basis: {basis})")
        return len(unresolved) == 0, unresolved

    _GENIE_NOTEBOOK_DISCLAIMER = ("**Disclaimer:** AI-generated. Have a qualified engineer review the notebook Genie Code builds before it is used "
                                  "in production; Databricks is not liable for issues arising from this code.")

    def _genie_instruction_cell_source(self, use_case: dict, instruction: str) -> str:
        notes = []
        status = use_case.get('genie_instruction_status')
        unresolved = [str(ref) for ref in (use_case.get('genie_validation_errors') or [])]
        claims = [str(claim) for claim in (use_case.get('genie_numeric_claims') or [])]
        if status == 'validation_errors' and unresolved:
            notes.append("Before you build: Inspire could not find these tables or columns in the Unity Catalog metadata of this workspace: "
                         + ", ".join(unresolved) + ". Check each one with DESCRIBE TABLE and use the real name, or leave it out.")
        if status == 'validation_errors' and claims:
            notes.append("These statements cite figures that are not in the data, so treat them as unproven: " + " | ".join(claims))
        conflicts = [str(conflict) for conflict in (use_case.get('genie_contract_conflicts') or [])]
        if conflicts:
            notes.append(f"Inspire's {GENIE_ACCEPTANCE_SECTION_TITLE} section below wins over these sentences of the instruction: " + " | ".join(conflicts))
        if status == 'unverified':
            notes.append("Before you build: Inspire could not check the tables and columns in this instruction against Unity Catalog metadata. "
                         "Check each one with DESCRIBE TABLE before you use it.")
        kpis = self._genie_kpi_source(use_case)
        parts = notes + [instruction.strip()] + (["## KPIs and Key Results\n\n" + kpis.strip()] if kpis else [])
        text = "\n\n".join(parts).replace('"""', "'''")
        return 'r"""\n' + text + '\n"""\n'

    @staticmethod
    def _genie_kpi_query(spec: dict) -> str:
        try:
            return DeepEvidence.kpi_sql(spec)
        except (ValueError, KeyError, TypeError, AttributeError):
            return ""

    def _genie_kpi_source(self, use_case: dict) -> str:
        """KPIs with their baselines, expected key results and the read-only KPI queries of the use case."""
        kpis, _gap = self._use_case_items(use_case, "KPIs", "kpis", {}, "English")
        results, _gap = self._use_case_items(use_case, "Expected Key Results", "key_results", {}, "English")
        queries = []
        for spec in use_case.get("KPIs") or []:
            query = self._genie_kpi_query(spec) if isinstance(spec, dict) and spec.get("computable") else ""
            if query:
                queries.append((inspire_plain_text(spec.get("name")), query))
        out = ""
        if kpis:
            out += "- **KPIs:**\n" + "".join(f"  - {text}\n" for text in kpis)
        if results:
            out += "- **Expected key results:**\n" + "".join(f"  - {text}\n" for text in results)
        if queries:
            out += ("\n### KPI queries\n\nRead-only aggregates. The summary step runs each one and reports its current value "
                    "against the baseline above.\n\n" + "".join(f"{name}:\n\n```sql\n{query}\n```\n\n" for name, query in queries))
        return out

    def _genie_instruction_widgets(self, instruction: str) -> list:
        match = self._genie_instruction_section(instruction, "Parameters")
        widgets = []
        for line in (match.group(1) if match else "").splitlines():
            row = self._GENIE_WIDGET_ROW_RE.match(line.strip())
            if row and row.group(1).lower() != "widget" and row.group(1) not in {w["name"] for w in widgets}:
                default = row.group(3).strip()
                widgets.append({"name": row.group(1), "type": row.group(2),
                                "default": default[1:-1].strip() if re.fullmatch(r"`[^`]*`", default) else default,
                                "allowed": genie_allowed_text(genie_allowed_rule(row.group(4))), "label": row.group(5).strip()})
        return widgets

    def _genie_widget_names(self, instruction: str, spec: dict) -> list:
        names = [w["name"] for w in self._genie_instruction_widgets(instruction)]
        return names + [w["name"] for w in self._genie_core_widgets(spec) if w["name"] not in names]

    _GENIE_COPY_LINE = ('Copy the cell below into Genie Code without the surrounding quotes (the `r"""` and `"""` lines), and let '
                        'Genie Code implement it.')

    def _genie_card_markdown(self, use_case: dict, spec: dict) -> str:
        contract = {key: text for key, _label, text in self._genie_decision_contract(use_case)}
        unknown = self._GENIE_NOT_ASSESSED
        lines = []
        if use_case.get("best_of_best") == "Y":
            score = use_case.get("best_of_best_score")
            lines.append(f"**\u2605 Best of Best #{use_case.get('best_of_best_rank')}"
                         + (f" \u00b7 Elite Score {float(score):.1f} / 10" if score not in (None, "") else "") + "**\n\n")
        rows = [("What it does", self._card_text(use_case.get("Solution") or use_case.get("Statement")))]
        decision = [contract[key] for key in ("decision", "decision_owner", "cadence") if contract[key] != unknown]
        if decision:
            rows.append(("Decision", " \u00b7 ".join(decision)))
        if contract["kpi"] != unknown:
            rows.append(("KPI", contract["kpi"] + (f" ({contract['kpi_direction']})" if contract["kpi_direction"] != unknown else "")))
        tables = [str(t).strip().strip("`").split(".")[-1] for t in str(use_case.get("Tables Involved") or "").split(",")]
        rows.append(("Data", ", ".join(f"`{t}`" for t in tables if t)))
        rows.append(("Output", f"`{'.'.join(inspire_table_parts(spec['result_table']))}`"))
        return "".join(lines) + "| | |\n|---|---|\n" + "".join(f"| **{label}** | {text} |\n" for label, text in rows if text)

    def _genie_ai_context(self, use_case: dict, spec: dict) -> str:
        lines = [f"Business: {inspire_plain_text(getattr(self, 'business_name', ''))}",
                 f"Use case {spec['use_case_id']}: {inspire_plain_text(use_case.get('Name'))}",
                 f"Why it matters: {inspire_plain_text(use_case.get('Business Value'))}",
                 f"Each record is one {spec['value_type']} to act on for {spec['action_target']}."]
        notes = str(use_case.get("_implement_notes") or "").strip()
        if notes:
            lines.append(f'The user\'s directives for this use case, word for word: """{notes}"""')
        return "\n".join(line for line in lines if not line.endswith(": "))

    def _genie_run_controls_source(self, use_case: dict, spec: dict, instruction: str) -> str:
        import pprint
        run = {"marker": INSPIRE_CHILD_MARKER, "job_slug": spec["job_slug"], "job_tags": dict(spec["job_tags"]),
               "schedule_cron": spec["schedule_cron"], "wait_minutes": spec["wait_minutes"],
               "result_table": ".".join(inspire_table_parts(spec["result_table"])),
               "use_case_id": spec["use_case_id"], "ai_context": self._genie_ai_context(use_case, spec),
               "decision_basis": spec["decision_basis"], "widget_names": self._genie_widget_names(instruction, spec),
               "scheduled_overrides": {"as_of_date": "latest", IMPLEMENT_REFRESH_PARAMETER: IMPLEMENT_REFRESH_DEFAULT}}
        return (GENIE_RUN_CONTROLS_TEMPLATE.replace("@@JOB_LAUNCHER@@", self._JOB_LAUNCHER_CLASS_SOURCE.strip())
                .replace("@@RESULT_HISTORY@@", IMPLEMENT_RESULT_HISTORY_SOURCE.strip())
                .replace("@@RUN@@", pprint.pformat(run, width=120, sort_dicts=False)))

    def _assemble_genie_code_notebook(self, use_case: dict, translations: dict):
        use_case_id = use_case.get('No', 'UNKNOWN')
        use_case_name = use_case.get('Name', '')
        instruction = sanitize_llm_special_chars(str(use_case.get('genie_instruction') or '').strip())
        if not instruction:
            self.logger.warning(f"[{use_case_id}] No Genie instruction to write -- skipping notebook")
            return None
        notebook_dir, notebook_full_path, notebook_filename = self._build_notebook_workspace_paths(use_case)
        final_cells, _spec = self._genie_notebook_cells(use_case, instruction)
        return self._write_notebook_to_workspace(notebook_dir, notebook_full_path,
                                                 self._genie_notebook_object(final_cells, notebook_filename), use_case_id)

    def _genie_notebook_cells(self, use_case: dict, instruction: str) -> tuple:
        use_case_id, use_case_name = use_case.get('No', 'UNKNOWN'), use_case.get('Name', '')
        spec = self._genie_notebook_spec(use_case)
        cells = [
            self._genie_cell("markdown", f"# {use_case_id}: {use_case_name}\n\n{self._genie_card_markdown(use_case, spec)}\n"
                                         f"{self._GENIE_NOTEBOOK_DISCLAIMER}\n\n{self._GENIE_COPY_LINE}\n"),
            self._genie_cell("code", self._genie_instruction_cell_source(use_case, instruction), "Genie Code instruction"),
            self._genie_cell("code", self._genie_run_controls_source(use_case, spec, instruction), "Run controls (Inspire, do not edit)"),
        ]
        return cells, spec

    def _genie_parallelism(self, num_use_cases: int) -> int:
        configured = TECHNICAL_CONTEXT["runtime"].get("genie_regen_parallelism", 10)
        return max(1, min(int(configured), int(num_use_cases or 1)))

    def _run_genie_wave(self, wave_id: int, backlog: list, full_schema_details: list,
                        document_context: str, schema_by_table: dict, translations: dict,
                        wave_workers: int) -> tuple:
        succeeded = []
        not_emitted = []
        _processed_futures = set()
        
        def _process_use_case_genie(uc):
            uc_id = uc.get('No', 'UNKNOWN')
            try:
                uc = self._generate_genie_instruction_for_use_case(uc, full_schema_details, document_context, schema_index=schema_by_table)
                emitted = uc.get('genie_instruction_status') in self._GENIE_EMITTED_STATUSES
                if emitted:
                    self._deliver_genie_notebook(uc, translations)
                return (uc_id, uc, emitted, None)
            except Exception as e:
                self.logger.error(f"❌ [{uc_id}] Genie pipeline failed: {get_clean_error_message(e)}")
                uc['genie_instruction'] = ''
                uc['genie_instruction_status'] = 'failed'
                return (uc_id, uc, False, None)
        
        def _handle_completed_future(future, uc_ref):
            uc_id = uc_ref.get('No', 'UNKNOWN')
            try:
                result_id, result_uc, emitted, _ = future.result(timeout=10)
                if emitted:
                    succeeded.append(result_uc)
                else:
                    is_timeout = 'timeout' in str(result_uc.get('genie_instruction_status', '')).lower() or \
                                 'timed out' in str(result_uc.get('genie_instruction', '')).lower()
                    if is_timeout:
                        result_uc['genie_instruction_status'] = 'timeout'
                    not_emitted.append(result_uc)
            except Exception as e:
                msg = str(e)
                is_timeout = 'timeout' in msg.lower() or 'timed out' in msg.lower()
                if is_timeout:
                    self.logger.warning(f"⏱️ [{uc_id}] Genie instruction timed out in wave {wave_id}")
                    uc_ref['genie_instruction_status'] = 'timeout'
                else:
                    self.logger.error(f"❌ [{uc_id}] Genie instruction failed in wave {wave_id}: {msg[:100]}")
                    uc_ref['genie_instruction_status'] = 'failed'
                uc_ref['genie_instruction'] = ''
                not_emitted.append(uc_ref)
        
        _llm_calls_per_uc = 1 if self._genie_validation_mode() == 'off' else 2
        _wave_timeout = max(600, len(backlog) * 400 * _llm_calls_per_uc)
        
        with _SafeExecutorContext(max_workers=wave_workers, thread_name_prefix=f"GenieWave{wave_id}", logger=self.logger, name=f"GenieWave{wave_id}") as ctx:
            future_to_uc = {}
            for uc in backlog:
                future = ctx.submit(_process_use_case_genie, uc)
                future_to_uc[future] = uc
            
            try:
                for future in ctx.completed(list(future_to_uc.keys()), timeout=_wave_timeout):
                    _processed_futures.add(future)
                    _handle_completed_future(future, future_to_uc[future])
            except concurrent.futures.TimeoutError:
                ctx.mark_timed_out()
                self.logger.error(f"⏱️ Genie wave {wave_id} total timeout after {_wave_timeout}s")
                _salvaged = 0
                for future, uc_ref in future_to_uc.items():
                    if future in _processed_futures:
                        continue
                    if future.done():
                        _salvaged += 1
                        self.logger.info(f"🔧 [{uc_ref.get('No', '?')}] Salvaging done-but-unyielded future after wave timeout")
                        _handle_completed_future(future, uc_ref)
                    else:
                        future.cancel()
                        uc_ref['genie_instruction'] = ''
                        uc_ref['genie_instruction_status'] = 'timeout'
                        not_emitted.append(uc_ref)
                if _salvaged:
                    self.logger.info(f"🔧 Salvaged {_salvaged} completed futures that were missed by timeout race")
        
        return succeeded, not_emitted

    def _run_genie_waves(self, use_cases: list, full_schema_details: list, document_context: str,
                         schema_index: dict, translations: dict, write_skeleton_on_failure: bool = False,
                         per_uc_progress: float = 0.0) -> dict:
        import time
        base_workers = self._genie_parallelism(len(use_cases))
        wave_parallelism = [
            (1, base_workers),
            (2, base_workers),
            (3, max(1, (base_workers + 1) // 2)),
            (4, max(1, (base_workers + 2) // 3)),
            (5, max(1, (base_workers + 2) // 3)),
        ]
        self.logger.info(f"🔧 Genie parallelism: {base_workers} worker(s) for {len(use_cases)} use case(s) (runtime genie_regen_parallelism)")
        emitted, failed = [], []
        backlog = list(use_cases)
        for wave_id, wave_workers in wave_parallelism:
            if not backlog:
                break
            wave_start = time.time()
            log_print(f"\n   ▶️ Genie wave {wave_id}: {len(backlog)} use case(s), parallelism {wave_workers}")
            succeeded, not_emitted = self._run_genie_wave(
                wave_id, backlog, full_schema_details, document_context, schema_index, translations, wave_workers
            )
            for uc in succeeded:
                emitted.append(uc)
                self._emit_pipeline_status(
                    stage_name="Genie Code Instructions",
                    step_name=f"Wave {wave_id} - {uc.get('No', 'UNKNOWN')}",
                    sub_step_name=str(uc.get('Name', ''))[:120],
                    message=f"Genie instruction {uc.get('genie_instruction_status')} for {uc.get('No', 'UNKNOWN')}",
                    status="ended_success" if uc.get('genie_instruction_status') == 'success' else "ended_warning",
                    progress_increment=per_uc_progress,
                    result_json=self._build_use_case_step_payload(uc)
                )
            retry, timeout_ids, hard_fail_ids = [], [], []
            for uc in not_emitted:
                uc_id = uc.get('No', 'UNKNOWN')
                if uc.get('genie_instruction_status') == 'timeout' and wave_id < len(wave_parallelism):
                    uc['genie_instruction_status'] = 'pending_retry'
                    uc['_genie_retry_attempt'] = uc.get('_genie_retry_attempt', 0) + 1
                    uc['_skip_llm_cache'] = True
                    retry.append(uc)
                    timeout_ids.append(uc_id)
                    continue
                failed.append(uc)
                hard_fail_ids.append(uc_id)
                if write_skeleton_on_failure:
                    self._write_skeleton_notebook(uc)
                self._emit_pipeline_status(
                    stage_name="Genie Code Instructions",
                    step_name=f"Wave {wave_id} - {uc_id}",
                    sub_step_name=str(uc.get('Name', ''))[:120],
                    message=f"Genie instruction {uc.get('genie_instruction_status')} for {uc_id}",
                    status="ended_error",
                    progress_increment=per_uc_progress,
                    result_json=self._build_use_case_step_payload(uc)
                )
            log_print(
                f"   📊 Genie wave {wave_id} ({time.time() - wave_start:.1f}s): {len(succeeded)} emitted, "
                f"{len(timeout_ids)} timed out (retry in wave {wave_id + 1}), {len(hard_fail_ids)} final failure(s)"
                + (f" {hard_fail_ids}" if hard_fail_ids else "")
            )
            backlog = retry
            if backlog and wave_id < len(wave_parallelism):
                cooldown_secs = 15 * wave_id
                log_print(f"   ⏳ Cooling down {cooldown_secs}s before Genie wave {wave_id + 1} (reducing API congestion)...")
                time.sleep(cooldown_secs)
        if any(uc.get('notebook_status') == 'written' for uc in emitted):
            self._write_genie_agents_file()
        return {"emitted": emitted, "failed": failed}

    def _genie_code_limit(self):
        raw = str(getattr(self, "generate_genie_code_for", GENIE_CODE_DEFAULT_COUNT) or GENIE_CODE_DEFAULT_COUNT).strip().lower()
        if raw == "all":
            return None
        try:
            return max(0, int(raw))
        except ValueError:
            self.logger.warning(f"13_generate_genie_code_for='{raw}' is not 'all' or an integer; using {GENIE_CODE_DEFAULT_COUNT}")
            return int(GENIE_CODE_DEFAULT_COUNT)

    def _top_n_genie_targets(self, ranked: list, limit, during_discover: bool = True) -> tuple:
        if limit is None:
            return ranked, "all use cases (widget 13 = all)" if during_discover else "all use cases (Generate Use Cases)"
        return ranked[:limit], f"top {limit} by final rank (widget 13)"

    def _select_genie_targets(self, use_cases: list, during_discover: bool = True) -> tuple:
        ranked = sorted(use_cases, key=self._final_rank_key)
        limit = self._genie_code_limit() if during_discover else None
        if limit == 0:
            return [], "no use case (widget 13 = 0); every use case keeps its skeleton notebook"
        if self._is_ultra_high_mode():
            elite = [uc for uc in ranked if uc.get('elite_status') == "ELITE"]
            if elite or not any(uc.get('elite_status') == "FALLBACK_BELOW_BAR" for uc in ranked):
                return elite, f"the {len(elite)} elite use case(s) (Ultra High; domain coverage picks keep skeleton notebooks)"
            fallback = [uc for uc in ranked if uc.get('elite_status') == "FALLBACK_BELOW_BAR"]
            targets, label = self._top_n_genie_targets(fallback, limit, during_discover)
            return targets, (f"{label}; no use case met the Ultra High bar, so the closest below-bar candidates "
                             f"get Genie Code and the rest keep skeleton notebooks")
        return self._top_n_genie_targets(ranked, limit, during_discover)

    def _report_genie_outcome(self, label: str, use_cases: list) -> dict:
        statuses = {}
        for uc in use_cases:
            key = uc.get('genie_instruction_status') or 'not_run'
            statuses[key] = statuses.get(key, 0) + 1
        failed_validation = {str(uc.get('No')): (list(uc.get('genie_validation_errors') or []) + list(uc.get('genie_numeric_claims') or []))[:10]
                             for uc in use_cases if uc.get('genie_instruction_status') == 'failed_validation'}
        with_errors = {str(uc.get('No')): (list(uc.get('genie_validation_errors') or []) + list(uc.get('genie_numeric_claims') or [])
                                           + list(uc.get('genie_contract_conflicts') or []))[:10]
                       for uc in use_cases if uc.get('genie_instruction_status') == 'validation_errors'}
        written = [str(uc.get('No')) for uc in use_cases if uc.get('notebook_status') == 'written']
        validated = [str(uc.get('No')) for uc in use_cases
                     if uc.get('notebook_status') == 'written' and uc.get('genie_instruction_status') == 'success']
        write_failed = [str(uc.get('No')) for uc in use_cases if uc.get('notebook_status') == 'write_failed']
        log_print(f"   🧾 {label}: statuses {dict(sorted(statuses.items()))}; Genie notebooks written {len(written)}/{len(use_cases)} "
                  f"({len(validated)} validated against live metadata)")
        for uc_id, refs in failed_validation.items():
            log_print(f"   ❌ {label} {uc_id}: instruction NOT emitted, failed reference validation: {refs}", level="ERROR")
        for uc_id, refs in with_errors.items():
            log_print(f"   ⚠️ {label} {uc_id}: instruction emitted with problems left after repair: {refs}", level="WARNING")
        for uc_id in write_failed:
            log_print(f"   ❌ {label} {uc_id}: Genie notebook write failed; the existing notebook stays in place", level="ERROR")
        return {
            "total": len(use_cases),
            "statuses": statuses,
            "validated_ids": validated,
            "notebooks_written_ids": written,
            "failed_validation": failed_validation,
            "validation_errors": with_errors,
            "notebook_write_failed": write_failed,
        }

    def _run_discover_skeleton_phase(self, all_use_cases: list, full_schema_details: list) -> list:
        """Discover-mode replacement for Phase 5b.
        
        For each use case (in parallel):
          1. Compute the 4 per-UC Genie prompt fields (directly_involved_schema with real column
             lists from live Unity Catalog metadata, foreign_key_relationships,
             available_tables_listing, technique_implementation_guidance) and attach them to the
             use_case dict so the tracking-table MERGE persists them for later Generate-mode runs.
          2. Set generate_genie_code_instruction = 'No'.
          3. Write the skeleton notebook into the workspace (the Genie notebook instead when the use
             case already carries an instruction, e.g. a JSON reload) and attach notebook_path.
        
        Then generate full Genie Code instructions: every elite use case in Ultra High (strict
        reference and numeric-claim validation with one repair attempt; domain coverage picks keep
        their skeleton), the top-N by final rank (widget 13) in Good and High. A Genie notebook
        replaces the skeleton only when the instruction is emitted; otherwise the skeleton stays and
        the outcome is reported.
        """
        import time
        
        if not all_use_cases:
            self.logger.warning("No use cases provided for Discover skeleton phase")
            return []
        
        log_print(f"\n{'='*80}")
        log_print(f"🔎 DISCOVER MODE — SKELETON NOTEBOOK GENERATION")
        log_print(f"{'='*80}")
        log_print(f"📊 Total use cases: {len(all_use_cases)}")
        log_print(f"📝 Strategy: capture Genie prompt fields from live metadata + write the skeleton notebook for every use case")
        log_print(f"{'='*80}\n")
        
        _step_id = self._emit_pipeline_status(
            stage_name="Discover Skeletons",
            step_name="Generation",
            sub_step_name="Started",
            message=f"Generating {len(all_use_cases)} skeleton notebooks + capturing per-UC Genie prompt fields",
            status="started",
            progress_increment=1.0
        )
        
        start_time = time.time()
        self._ensure_genie_metadata_loader()
        schema_by_table = self._build_genie_schema_index(full_schema_details)
        self._genie_table_evidence([t for uc in all_use_cases for t in self._genie_involved_tables(uc)])
        try:
            _translations = self.translation_service.get_translations("English")
        except Exception as _te:
            self.logger.warning(f"Discover notebooks: translations unavailable: {get_clean_error_message(_te)}; empty dict")
            _translations = {}
        
        total = len(all_use_cases)
        completed = 0
        failed = 0
        _lock = threading.Lock()
        
        def _process(idx_uc):
            nonlocal completed, failed
            idx, uc = idx_uc
            uc_id = uc.get('No', 'UNKNOWN')
            try:
                uc.update(self._compute_per_uc_genie_fields(uc, full_schema_details, schema_by_table))
            except Exception as e:
                self.logger.error(f"❌ [{uc_id}] Genie prompt fields failed: {get_clean_error_message(e)}")
            written = self._write_use_case_notebook(uc, _translations)
            with _lock:
                if written:
                    completed += 1
                else:
                    failed += 1
                _cnt = completed + failed
            self.logger.info(f"🔎 [{_cnt}/{total}] Discover notebook {uc.get('notebook_status')} for {uc_id}: {uc.get('notebook_path') or uc.get('notebook_error')}")
            return written
        
        nb_parallelism, reason = calculate_adaptive_parallelism(
            "discover_skeleton", self.max_parallelism,
            num_items=total,
            is_llm_operation=False, logger=self.logger,
            estimated_mb_per_worker=2.0
        )
        log_adaptive_parallelism_decision("discover_skeleton", nb_parallelism, self.max_parallelism, reason)
        
        self._notebooks_generated_inline = True
        _skeleton_timeout = max(600, total * 60)
        with _SafeExecutorContext(max_workers=nb_parallelism, thread_name_prefix="DiscoverSkel", logger=self.logger, name="DiscoverSkel") as ctx:
            futures = {ctx.submit(_process, item): item[1] for item in enumerate(all_use_cases, start=1)}
            try:
                for future in ctx.completed(futures, timeout=_skeleton_timeout):
                    try:
                        future.result()
                    except Exception as e:
                        self.logger.error(f"❌ Discover skeleton task failed: {get_clean_error_message(e)}")
            except concurrent.futures.TimeoutError:
                ctx.mark_timed_out()
                self.logger.error(f"⏱️ Discover skeleton writes timed out after {_skeleton_timeout}s")
                for future, uc in futures.items():
                    if not future.done():
                        future.cancel()
                        uc['notebook_status'] = 'write_failed'
                        uc['notebook_error'] = f"skeleton write timed out after {_skeleton_timeout}s"
                        with _lock:
                            failed += 1
        
        elapsed = time.time() - start_time
        log_print(f"\n{'='*80}")
        log_print(f"✅ DISCOVER MODE — SKELETON GENERATION COMPLETE")
        log_print(f"{'='*80}")
        log_print(f"   ✅ Notebooks OK:  {completed}/{total} (skeletons, or Genie notebooks for use cases that already carry an instruction)")
        if failed:
            log_print(f"   ❌ Notebooks Fail: {failed}/{total}")
        log_print(f"   ⏱️  Total time: {elapsed:.1f}s")
        log_print(f"{'='*80}\n")
        
        self._emit_pipeline_status(
            stage_name="Discover Skeletons",
            step_name="Generation",
            sub_step_name="Complete",
            message=f"Discover skeletons: {completed} success, {failed} failed",
            status="ended_success" if failed == 0 else "ended_warning",
            progress_increment=2.0,
            result_json={"success": completed, "failed": failed, "total_use_cases": total, "elapsed_seconds": round(elapsed, 1),
                         "write_failed_ids": [str(uc.get('No')) for uc in all_use_cases if uc.get('notebook_status') == 'write_failed']},
            step_id=_step_id
        )
        
        targets, selection = self._select_genie_targets(all_use_cases)
        if not targets:
            log_print(f"   ℹ️ Discover Auto-Genie: Genie Code for {selection}")
            return all_use_cases
        ultra_high = self._is_ultra_high_mode()
        log_print(f"\n{'='*80}")
        log_print(f"🚀 DISCOVER AUTO-GENIE — {len(targets)} use case(s): {selection}")
        log_print(f"{'='*80}")
        for _i, _uc in enumerate(targets, start=1):
            _score = f"elite_score={_uc.get('elite_score')}" if _uc.get('elite_score') is not None else f"Inspire Score={_uc.get('Inspire Score')}"
            log_print(f"   {_i}. {_uc.get('No','?')} ({_score}): {_uc.get('Name','')[:80]}")
        for _uc in targets:
            _uc['_source_session_id'] = self.session_id
        self._run_genie_waves(targets, full_schema_details, '', schema_by_table, _translations)
        outcome = self._report_genie_outcome("Discover Auto-Genie", targets)
        log_print(f"{'='*80}\n")
        _clean = len(outcome["validated_ids"]) == len(targets)
        self._emit_pipeline_status(
            stage_name="Discover Skeletons",
            step_name="Auto-Genie (Elite Use Cases)" if ultra_high else "Auto-Genie (Top-N)",
            sub_step_name="Complete",
            message=f"Auto-Genie {len(targets)} use case(s): {len(outcome['validated_ids'])} validated, "
                    f"{len(outcome['notebooks_written_ids'])} Genie notebooks written, {len(outcome['failed_validation'])} failed validation",
            status="ended_success" if _clean else "ended_warning",
            progress_increment=1.0,
            result_json={"selection": selection, "top_n": len(targets), "success": len(outcome["validated_ids"]),
                         "failed": len(targets) - len(outcome["notebooks_written_ids"]), "ids": outcome["notebooks_written_ids"],
                         **outcome}
        )
        return all_use_cases

    # ==== Generate Use Cases mode ====
    
    _UC_COLUMN_TO_FIELD = {
        'id': 'No',
        'business_domain': 'Business Domain',
        'subdomain': 'Subdomain',
        'use_case': 'usecase',
        'description': 'Name',
        'inspire_score': 'Inspire Score',
        'generated': 'generated',
        'type': 'type',
        'analytics_technique': 'Analytics Technique',
        'business_priority_alignment': 'Business Priority Alignment',
        'strategic_goals_alignment': 'Strategic Goals Alignment',
        'statement': 'Statement',
        'solution': 'Solution',
        'business_value': 'Business Value',
        'beneficiary': 'Beneficiary',
        'sponsor': 'Sponsor',
        'tables_involved': 'Tables Involved',
        'primary_table': 'Primary Table',
        'strategic_alignment': 'Strategic Alignment',
        'roi': 'Return on Investment',
        'reusability': 'Reusability',
        'time_to_value': 'Time to Value',
        'data_availability': 'Data Availability',
        'data_accessibility': 'Data Accessibility',
        'architecture_fitness': 'Architecture Fitness',
        'team_skills': 'Team Skills',
        'domain_knowledge': 'Domain Knowledge',
        'people_allocation': 'People Allocation',
        'budget_allocation': 'Budget Allocation',
        'time_to_production': 'Time to Production',
        'value_score': 'Value',
        'feasibility_score': 'Feasibility',
        'priority_score': 'Priority',
        'quality_score': 'Quality Score',
        'priority_reasons': 'Justification',
        'quality_reasons': 'Quality Summary',
        'result_database': 'result_table',
        'notebook_path': 'notebook_path',
        'high_level_design': 'High Level Design',
        'genie_instruction': 'genie_instruction',
        'generate_genie_code_instruction': 'generate_genie_code_instruction',
        'directly_involved_schema': 'directly_involved_schema',
        'foreign_key_relationships': 'foreign_key_relationships',
        'available_tables_listing': 'available_tables_listing',
        'technique_implementation_guidance': 'technique_implementation_guidance',
        'code_structure_mandate': 'code_structure_mandate',
        'has_genie_code': 'has_genie_code',
        'idea_theme': 'idea_theme',
        'bob_score': 'bob_score',
        'bob_tier1_score': 'bob_tier1_score',
        'bob_tier2_score': 'bob_tier2_score',
        'bob_tier3_score': 'bob_tier3_score',
        'quality': 'Quality',
        'quality_mode': 'quality_mode',
        'elite_status': 'elite_status',
        'elite_score': 'elite_score',
        'elite_assessment': 'elite_assessment',
        'selection_flags': 'selection_flags',
    }
    
    def _row_to_uc(self, row) -> dict:
        """Inverse of _uc_to_row: reconstruct a use_case dict from a __inspire_usecases Row.
        Marks the dict with _genie_fields_cached=True: the persisted per-UC Genie prompt fields are
        replayed only when no live metadata is reachable (see _genie_prompt_fields). JSON columns
        (elite_assessment, selection_flags) are decoded so the notebook card shows the elite fields,
        and each selection flag is restored under its own key, where _selection_flags reads it."""
        row_d = row.asDict() if hasattr(row, 'asDict') else dict(row)
        uc = {}
        for col, field in self._UC_COLUMN_TO_FIELD.items():
            val = row_d.get(col)
            if val is not None:
                uc[field] = val
        for key in ('elite_assessment', 'selection_flags'):
            if isinstance(uc.get(key), str):
                try:
                    uc[key] = json.loads(uc[key])
                except ValueError:
                    self.logger.warning(f"[{uc.get('No', 'UNKNOWN')}] tracking column {key} is not valid JSON; left out of the notebook card")
                    uc.pop(key)
        for column, key, kind in self._V130_TRACKING_FIELDS:
            value = row_d.get(column)
            if value in (None, ""):
                continue
            if kind == "json":
                try:
                    uc[key] = json.loads(value) if isinstance(value, str) else value
                except ValueError:
                    self.logger.warning(f"[{uc.get('No', 'UNKNOWN')}] tracking column {column} is not valid JSON; left out")
            elif kind in ("list", "lines"):
                separator = "," if kind == "list" else "\n"
                uc[key] = [part.strip() for part in str(value).split(separator) if part.strip()]
            else:
                uc[key] = value
        flags = uc.pop('selection_flags', None)
        if isinstance(flags, dict):
            for key in self._SELECTION_FLAG_KEYS:
                if key in flags and uc.get(key) in (None, ""):
                    uc[key] = flags[key]
        uc['_genie_fields_cached'] = True
        uc['_source_session_id'] = row_d.get('session_id')
        return uc
    
    def _load_session_context(self, session_id) -> dict:
        """Load the per-session Genie-prompt context from __inspire_session for replay."""
        if not self.atomic_writer or not self.atomic_writer.enabled:
            return {}
        session_table = self.atomic_writer.session_table
        if not session_table:
            return {}
        try:
            df = self.spark.sql(
                f"SELECT business_name, enriched_business_context, enriched_strategic_goals, "
                f"enriched_business_priorities, enriched_strategic_initiative, enriched_value_chain, "
                f"enriched_revenue_model, generation_instructions_section "
                f"FROM {session_table} WHERE session_id = {int(session_id)} LIMIT 1"
            )
            rows = df.collect()
            if not rows:
                self.logger.warning(f"No session row found for session_id={session_id}")
                return {}
            return rows[0].asDict()
        except Exception as e:
            self.logger.warning(f"Failed to load session context for session_id={session_id}: {get_clean_error_message(e)}")
            return {}
    
    def _apply_session_context(self, session_ctx: dict):
        """Apply a loaded session context onto self so _get_enriched_context_for_prompt works.
        In Generate mode we overwrite merged_business_context, business_name and generation_instructions."""
        if not session_ctx:
            return
        business_name = session_ctx.get('business_name')
        if business_name:
            self.business_name = business_name
        self.merged_business_context = {
            'business_context': session_ctx.get('enriched_business_context', '') or '',
            'strategic_goals': session_ctx.get('enriched_strategic_goals', '') or '',
            'business_priorities': session_ctx.get('enriched_business_priorities', '') or '',
            'strategic_initiative': session_ctx.get('enriched_strategic_initiative', '') or '',
            'value_chain': session_ctx.get('enriched_value_chain', '') or '',
            'revenue_model': session_ctx.get('enriched_revenue_model', '') or '',
        }
        gen_instr_section = session_ctx.get('generation_instructions_section', '') or ''
        if gen_instr_section and gen_instr_section.strip() and gen_instr_section != '*(No additional generation instructions provided)*':
            self.parsed_generation_instructions = {
                'per_prompt_instructions': {
                    'USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT': gen_instr_section
                }
            }
        else:
            self.parsed_generation_instructions = getattr(self, 'parsed_generation_instructions', {}) or {}
    
    def _fetch_flagged_use_cases_route1(self) -> list:
        """Route 1: rows flagged generate_genie_code_instruction='Yes' in this run's session, or, when the caller gave
        no session, in the sessions of this run's business. Rows of other sessions and businesses are never read."""
        if not self._tracking_table_name:
            return []
        flagged = f"SELECT * FROM {self._tracking_table_name} WHERE UPPER(COALESCE(generate_genie_code_instruction, 'No')) = 'YES'"
        if getattr(self, "_user_provided_session_id", False) and str(getattr(self, "session_id", "") or "").strip():
            sql, args, scope = f"{flagged} AND session_id = :sid", {"sid": int(self.session_id)}, f"session {self.session_id}"
        else:
            session_table = quote_qualified_name(f"{self.inspire_database}.__inspire_session")
            sql = f"{flagged} AND session_id IN (SELECT session_id FROM {session_table} WHERE business_name = :biz)"
            args, scope = {"biz": str(self.business_name)}, f"business {self.business_name}"
        self._route1_scope, self._route1_scan_error = scope, ""
        try:
            rows = self.spark.sql(sql, args=args).collect()
        except Exception as e:
            self._route1_scan_error = get_clean_error_message(e)
            self.logger.error(f"Route 1 DB scan failed: {self._route1_scan_error}")
            self._record_failed_step("Route 1 scan", "", e)
            return []
        self.logger.info(f"[ROUTE1-SCOPE] {len(rows)} flagged use case(s) in {scope}")
        return rows
    
    def _scan_notebooks_for_generation_flag(self) -> list:
        """Route 2: walk the generation_path tree, parse each notebook, and return entries
        whose marker cell has generate_genie_code_instruction = True.
        Returns list of dicts: {'notebook_path', 'use_case_id', 'session_id'}."""
        from databricks.sdk.service import workspace as _ws_svc
        import ast
        
        root = getattr(self, 'notebook_output_dir', None)
        self._route2_errors = []
        if not root:
            return []
        found = []
        
        def _walk(path: str):
            try:
                entries = list(self.w_client.workspace.list(path))
            except Exception as e:
                self.logger.debug(f"Route 2 list({path}) failed: {get_clean_error_message(e)}")
                self._route2_errors.append(f"list {path}: {get_clean_error_message(e)}")
                return
            for entry in entries:
                obj_type = getattr(entry, 'object_type', None)
                entry_path = getattr(entry, 'path', None)
                if not entry_path:
                    continue
                if obj_type == _ws_svc.ObjectType.DIRECTORY:
                    _walk(entry_path)
                elif obj_type == _ws_svc.ObjectType.NOTEBOOK:
                    info = _parse_notebook(entry_path)
                    if info:
                        found.append(info)
        
        def _parse_notebook(nb_path: str):
            try:
                exported = self.w_client.workspace.export(path=nb_path, format=_ws_svc.ExportFormat.SOURCE)
                content_b64 = getattr(exported, 'content', None)
                if not content_b64:
                    return None
                source = base64.b64decode(content_b64).decode('utf-8', errors='ignore')
            except Exception as e:
                self.logger.debug(f"Route 2 export({nb_path}) failed: {get_clean_error_message(e)}")
                self._route2_errors.append(f"export {nb_path}: {get_clean_error_message(e)}")
                return None
            if 'generate_genie_code_instruction' not in source:
                return None
            flag_val, uc_id, sess_id = None, None, None
            try:
                tree = ast.parse(source)
                for node in ast.walk(tree):
                    if not isinstance(node, ast.Assign):
                        continue
                    if not node.targets or not isinstance(node.targets[0], ast.Name):
                        continue
                    name = node.targets[0].id
                    val_node = node.value
                    if name == 'generate_genie_code_instruction' and isinstance(val_node, ast.Constant):
                        flag_val = val_node.value
                    elif name == 'INSPIRE_USE_CASE_ID' and isinstance(val_node, ast.Constant):
                        uc_id = val_node.value
                    elif name == 'INSPIRE_SESSION_ID' and isinstance(val_node, ast.Constant):
                        sess_id = val_node.value
            except SyntaxError:
                import re
                m_flag = re.search(r"^\s*generate_genie_code_instruction\s*=\s*(True|False)\s*$", source, re.M)
                m_id = re.search(r"^\s*INSPIRE_USE_CASE_ID\s*=\s*['\"]([^'\"]+)['\"]\s*$", source, re.M)
                m_sess = re.search(r"^\s*INSPIRE_SESSION_ID\s*=\s*(\d+)\s*$", source, re.M)
                if m_flag:
                    flag_val = (m_flag.group(1) == 'True')
                if m_id:
                    uc_id = m_id.group(1)
                if m_sess:
                    sess_id = int(m_sess.group(1))
            if flag_val is True and uc_id:
                return {'notebook_path': nb_path, 'use_case_id': uc_id, 'session_id': sess_id}
            return None
        
        _walk(root)
        return found
    
    def _fetch_flagged_use_cases_route2(self) -> list:
        """Route 2 wrapper: scan notebooks, cross-reference DB by (id, session_id) to reload full row.

        Phase K: batched SQL. Instead of N queries (one per flagged entry), we
        issue a single query using an IN-list over (id, session_id) tuples plus
        a fallback single-query for entries without a session_id. Result: 1-2
        Spark round-trips regardless of flagged entry count."""
        entries = self._scan_notebooks_for_generation_flag()
        self._route2_marked_count = len(entries)
        errors = getattr(self, "_route2_errors", None)
        if errors is None:
            errors = self._route2_errors = []
        if not entries:
            return []
        if not self._tracking_table_name:
            self.logger.warning("Route 2 found flagged notebooks but tracking table not configured — cannot enrich.")
            return []

        # Split entries by whether they have a session_id (tuple-match) or not (id-only latest).
        with_sess = []  # list of (uc_id, sess_id)
        without_sess = []  # list of uc_id
        for e in entries:
            uc_id = e.get('use_case_id')
            sess_id = e.get('session_id')
            if not uc_id:
                continue
            if sess_id is None:
                without_sess.append(str(uc_id))
            else:
                with_sess.append((str(uc_id), int(sess_id)))

        rows = []
        if with_sess:
            try:
                pair_params = {}
                pair_clauses = []
                for idx, (uid, sid) in enumerate(with_sess):
                    pair_params[f"i{idx}"] = uid
                    pair_params[f"s{idx}"] = int(sid)
                    pair_clauses.append(f"(id = :i{idx} AND session_id = :s{idx})")
                df = self.spark.sql(
                    f"SELECT * FROM {self._tracking_table_name} WHERE {' OR '.join(pair_clauses)}",
                    args=pair_params
                )
                rows.extend(df.collect())
            except Exception as ex:
                self.logger.warning(
                    f"Route 2 batched lookup failed ({len(with_sess)} entries): "
                    f"{get_clean_error_message(ex)}; falling back to per-entry queries"
                )
                for uid, sid in with_sess:
                    try:
                        df = self.spark.sql(
                            f"SELECT * FROM {self._tracking_table_name} WHERE id = :uid AND session_id = :sid LIMIT 1",
                            args={"uid": uid, "sid": int(sid)}
                        )
                        fetched = df.collect()
                        if fetched:
                            rows.append(fetched[0])
                    except Exception as ex2:
                        self.logger.warning(f"Route 2 per-entry fallback failed for {uid}/{sid}: {get_clean_error_message(ex2)}")
                        errors.append(f"lookup {uid} in session {sid}: {get_clean_error_message(ex2)}")

        # Batched id-only (latest session) via window function
        if without_sess:
            try:
                id_in_sql, id_params = _sql_in_list_params(without_sess, "u")
                df = self.spark.sql(
                    f"SELECT * FROM ("
                    f"  SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY session_id DESC) AS _rn "
                    f"  FROM {self._tracking_table_name} WHERE id IN ({id_in_sql})"
                    f") WHERE _rn = 1",
                    args=id_params
                )
                rows.extend([r for r in df.collect() if r])
            except Exception as ex:
                self.logger.warning(
                    f"Route 2 id-only batched lookup failed ({len(without_sess)} entries): "
                    f"{get_clean_error_message(ex)}"
                )
                errors.append(f"lookup of {len(without_sess)} id(s) without a session: {get_clean_error_message(ex)}")

        # Log any entries we couldn't match (for operator visibility)
        try:
            fetched_ids = {str(r["id"]) for r in rows if r and "id" in r.asDict()}
            missing = [uid for uid, _ in with_sess if uid not in fetched_ids] +                       [uid for uid in without_sess if uid not in fetched_ids]
            for miss in missing[:20]:
                self.logger.warning(f"Route 2: flagged id={miss} not found in tracking table")
        except Exception:
            pass

        return rows
    
    def _merge_regenerated_ucs_into_tracking(self, regenerated_ucs: list) -> int:
        """Persist regenerated Genie instructions in one MERGE (preserving each row's original
        session_id): genie_instruction, notebook_path, has_genie_code='Y' and
        generate_genie_code_instruction='No' land in the same transaction. Values travel through a
        DataFrame temp view, never through SQL text. Raises when the MERGE fails."""
        if not self._tracking_table_name or not regenerated_ucs:
            return 0
        from pyspark.sql.types import StructType, StructField, LongType, StringType
        rows_by_key = {}
        for uc in regenerated_ucs:
            sess_id = uc.get('_source_session_id')
            uc_id = uc.get('No')
            if sess_id is None or not uc_id:
                continue
            rows_by_key[(int(sess_id), str(uc_id))] = (int(sess_id), str(uc_id), uc.get('genie_instruction') or '', uc.get('notebook_path') or '')
        rows = list(rows_by_key.values())
        if not rows:
            return 0
        schema = StructType([
            StructField("session_id", LongType(), False),
            StructField("id", StringType(), False),
            StructField("genie_instruction", StringType(), True),
            StructField("notebook_path", StringType(), True),
        ])
        view_name = self._tracking_unique_view("genie_regen")
        with self._tracking_lock:
            self.spark.createDataFrame(rows, schema=schema).createOrReplaceTempView(view_name)
            try:
                self.spark.sql(
                    f"MERGE INTO {self._tracking_table_name} AS t USING {view_name} AS s "
                    f"ON t.session_id = s.session_id AND t.id = s.id "
                    f"WHEN MATCHED THEN UPDATE SET t.genie_instruction = s.genie_instruction, "
                    f"t.notebook_path = s.notebook_path, t.has_genie_code = 'Y', "
                    f"t.generate_genie_code_instruction = 'No', t.updated_at = current_timestamp()"
                )
            finally:
                self.spark.catalog.dropTempView(view_name)
        self.logger.info(f"✅ Regenerated Genie instruction persisted for {len(rows)}/{len(regenerated_ucs)} use cases (one MERGE)")
        return len(rows)
    
    def _regenerate_genie_for_flagged_rows(self, flagged_rows: list, route_label: str, skip_cache: bool = False) -> list:
        """Group flagged rows by session_id, load per-session context, regenerate the Genie
        instruction against live Unity Catalog metadata (the same validation and wave runner as
        Discover), and write the Genie notebook over the skeleton when the instruction is emitted.
        Returns the use cases whose Genie notebook was written."""
        import time
        if not flagged_rows:
            return []
        
        by_session = defaultdict(list)
        seen_keys = set()
        for row in flagged_rows:
            uc = self._row_to_uc(row)
            if skip_cache:
                uc['_skip_llm_cache'] = True
            key = (uc.get('_source_session_id'), uc.get('No'))
            if key in seen_keys:
                self.logger.warning(f"[{route_label}] duplicate tracking row for use case {key[1]} in session {key[0]}; regenerated once")
                continue
            seen_keys.add(key)
            by_session[uc.get('_source_session_id')].append(uc)
        
        total = sum(len(v) for v in by_session.values())
        log_print(f"\n{'='*80}")
        log_print(f"🚀 GENERATE MODE ({route_label}) — regenerating {total} use case(s) across {len(by_session)} session(s)")
        log_print(f"{'='*80}")
        
        _step_id = self._emit_pipeline_status(
            stage_name=f"Generate {route_label}",
            step_name="Regeneration",
            sub_step_name="Started",
            message=f"Regenerating Genie instructions for {total} flagged use cases",
            status="started",
            progress_increment=1.0
        )
        
        try:
            translations = self.translation_service.get_translations("English")
        except Exception as _te:
            self.logger.warning(f"Translation service unavailable in Generate mode: {get_clean_error_message(_te)}; falling back to empty dict")
            translations = {}
        
        self._ensure_genie_metadata_loader()
        processed = []
        start_time = time.time()
        
        own_context = (self.business_name, getattr(self, 'merged_business_context', None), getattr(self, 'parsed_generation_instructions', None))
        try:
            for sess_id, ucs in by_session.items():
                log_print(f"\n🔑 Session {sess_id}: {len(ucs)} use case(s)")
                session_ctx = self._load_session_context(sess_id) if sess_id is not None else {}
                if not session_ctx:
                    self.logger.warning(f"No session context loaded for session_id={sess_id}; falling back to current self attributes")
                self._apply_session_context(session_ctx)
                schema_details = self._build_schema_details_from_use_cases(ucs)
                schema_index = self._build_genie_schema_index(schema_details)
                self.logger.info(f"🤖 [{route_label}] Session {sess_id}: running {len(ucs)} Genie regens with parallelism={self._genie_parallelism(len(ucs))}")
                self._run_genie_waves(ucs, schema_details, '', schema_index, translations)
                processed.extend(ucs)
        finally:
            self.business_name, self.merged_business_context, self.parsed_generation_instructions = own_context
        
        succeeded = [uc for uc in processed if uc.get('notebook_status') == 'written']
        outcome = self._report_genie_outcome(f"Generate {route_label}", processed)
        failed_count = total - len(succeeded)
        elapsed = time.time() - start_time
        write_error = next((str(uc.get('notebook_error')) for uc in processed
                            if uc.get('notebook_status') == 'write_failed' and uc.get('notebook_error')), "")
        nothing_written = bool(total) and not succeeded
        log_print(f"\n{'='*80}")
        log_print(f"{'❌' if nothing_written else '✅'} GENERATE MODE ({route_label}) {'FAILED' if nothing_written else 'COMPLETE'}")
        log_print(f"   ✅ Regenerated OK: {len(succeeded)}/{total}")
        if failed_count:
            log_print(f"   ❌ Failed:         {failed_count}/{total}")
        log_print(f"   ⏱️  Total time:    {elapsed:.1f}s")
        log_print(f"{'='*80}\n")
        if failed_count:
            self._record_failed_step(f"Genie notebooks ({route_label})", "",
                                     f"{failed_count} of {total} not written" + (f": {write_error}" if write_error else ""))

        self._emit_pipeline_status(
            stage_name=f"Generate {route_label}",
            step_name="Regeneration",
            sub_step_name="Complete",
            message=f"Regeneration done: {len(succeeded)} ok, {failed_count} failed",
            status=("ended_error" if nothing_written
                    else "ended_success" if not failed_count and len(outcome["validated_ids"]) == total else "ended_warning"),
            progress_increment=2.0,
            result_json={"success": len(succeeded), "failed": failed_count, "total": total, "elapsed_seconds": round(elapsed, 1), **outcome},
            step_id=_step_id
        )
        if nothing_written:
            raise RuntimeError(f"Genie regeneration wrote 0 of {total} notebooks"
                               + (f": {write_error}" if write_error else ""))

        return succeeded
    
    def _implement_table_sql(self) -> str:
        return f"{quote_qualified_name(self.inspire_database)}.{IMPLEMENT_TABLE}" if self.inspire_database else ""

    def _implement_param(self, name: str) -> str:
        override = (getattr(self, "_implement_params", None) or {}).get(name)
        if override is not None:
            return str(override).strip()
        try:
            return str(self.dbutils.widgets.get(name) or "").strip()
        except Exception:
            return ""

    def _implement_emit(self, uc_id: str, step: str, message: str, status: str, payload: dict = None, step_id: int = None):
        write = self.logger.error if status == "ended_error" else self.logger.warning if status == "ended_warning" else self.logger.info
        write(f"[{IMPLEMENT_STAGE}] {uc_id} · {step}: {message}")
        return self._emit_pipeline_status(stage_name=IMPLEMENT_STAGE, step_name=step, sub_step_name=str(uc_id), message=message,
                                          status=status, progress_increment=1.0,
                                          result_json=dict(payload or {}, use_case_id=str(uc_id), phase=step), step_id=step_id)

    def _implement_update(self, key: tuple, **fields):
        implement_table = self._implement_table_sql()
        if not implement_table or not fields:
            return
        kinds = dict(IMPLEMENT_COLUMNS)
        sets, args = [], {"sid": int(key[0]), "uc": str(key[1]), "v": int(key[2])}
        for index, (column, value) in enumerate(fields.items()):
            kind = kinds[column]
            name = f"p{index}"
            if value is None:
                sets.append(f"`{column}` = NULL")
            elif kind == "VARIANT":
                sets.append(f"`{column}` = parse_json(:{name})")
                args[name] = json.dumps(value, default=str)
            elif kind in ("BIGINT", "INT"):
                sets.append(f"`{column}` = CAST(:{name} AS {kind})")
                args[name] = int(value)
            else:
                sets.append(f"`{column}` = :{name}")
                args[name] = str(value)
        sets.append("`updated_at` = current_timestamp()")
        sql = f"UPDATE {implement_table} SET {', '.join(sets)} WHERE session_id = :sid AND use_case_id = :uc AND spec_version = :v"
        RetryHandler.execute_with_retry(lambda: self.spark.sql(sql, args=args), max_attempts=IMPLEMENT_UPDATE_ATTEMPTS, logger=self.logger,
                                        context=f"updating {key[1]} spec {key[2]}", base_delay=0.5, retryable_errors=["concurrent"])

    _IMPLEMENT_READ_COLUMNS = ("session_id", "use_case_id", "spec_version", "based_on_version", "status", "notebook_path", "job_id",
                               "last_preview_run_id", "result_rows", "dashboard_id", "dashboard_url", "genie_space_id",
                               "genie_space_url", "app_name", "app_url", "alert_id", "alert_url", "created_by", "status_message")

    def _implement_select(self, where: str, args: dict, order: str = "use_case_id, spec_version") -> list:
        implement_table = self._implement_table_sql()
        columns = ", ".join(f"`{c}`" for c in self._IMPLEMENT_READ_COLUMNS)
        rows = self.spark.sql(f"SELECT {columns}, to_json(spec_json) AS spec_json, to_json(asset_status) AS asset_status, "
                              f"to_json(genie_widgets) AS genie_widgets, to_json(data_profiles) AS data_profiles, "
                              f"to_json(suggested_schedule) AS suggested_schedule FROM {implement_table} WHERE {where} ORDER BY {order}",
                              args=args).collect()
        out = []
        for row in rows:
            item = row.asDict() if hasattr(row, "asDict") else dict(row)
            for key, empty in (("spec_json", {}), ("asset_status", {}), ("genie_widgets", []), ("data_profiles", {}),
                               ("suggested_schedule", {})):
                try:
                    value = json.loads(item.get(key) or "null")
                except ValueError:
                    value = None
                item[key] = value if isinstance(value, type(empty)) else empty
            out.append(item)
        return out

    def _implement_targets(self) -> list:
        args = {"sid": int(self.session_id)}
        where = "session_id = :sid AND status = 'queued'"
        use_case = self._implement_param(IMPLEMENT_USE_CASE_PARAM)
        if use_case:
            where += " AND use_case_id = :uc"
            args["uc"] = use_case
        version = self._implement_param(IMPLEMENT_SPEC_VERSION_PARAM)
        if version:
            try:
                args["v"] = int(version)
                where += " AND spec_version = :v"
            except ValueError:
                raise ValueError(f"{IMPLEMENT_SPEC_VERSION_PARAM} must be an integer (got {version!r})")
        rows = self._implement_select(where, args)
        newest = {}
        for row in rows:
            current = newest.get(row["use_case_id"])
            if current is None or int(row["spec_version"]) > int(current["spec_version"]):
                if current is not None:
                    self._implement_update((current["session_id"], current["use_case_id"], current["spec_version"]), status="failed",
                                           status_message=f"superseded by spec version {row['spec_version']} before it ran")
                newest[row["use_case_id"]] = row
            else:
                self._implement_update((row["session_id"], row["use_case_id"], row["spec_version"]), status="failed",
                                       status_message=f"superseded by spec version {current['spec_version']} before it ran")
        return [newest[k] for k in sorted(newest)]

    def _implement_requested_row(self):
        use_case = self._implement_param(IMPLEMENT_USE_CASE_PARAM)
        version = self._implement_param(IMPLEMENT_SPEC_VERSION_PARAM)
        if not use_case or not version:
            return None
        rows = self._implement_select("session_id = :sid AND use_case_id = :uc AND spec_version = :v",
                                      {"sid": int(self.session_id), "uc": use_case, "v": int(version)})
        return rows[0] if rows else None

    def _implement_previous(self, target: dict) -> dict:
        rows = self._implement_select(
            "session_id = :sid AND use_case_id = :uc AND spec_version < :v AND status IN ('ready', 'ready_with_warnings')",
            {"sid": int(target["session_id"]), "uc": str(target["use_case_id"]), "v": int(target["spec_version"])},
            order="spec_version DESC")
        return rows[0] if rows else {}

    _IMPLEMENT_CARRIED = {"notebook_path": None, "job_id": None, "last_preview_run_id": None, "result_rows": None,
                          "dashboard_id": "dashboard", "dashboard_url": "dashboard", "genie_space_id": "genie_space",
                          "genie_space_url": "genie_space", "app_name": "app", "app_url": "app", "alert_id": "alert",
                          "alert_url": "alert", "genie_widgets": None, "data_profiles": None, "suggested_schedule": None}

    def _implement_carried(self, target: dict) -> tuple:
        rows = self._implement_select("session_id = :sid AND use_case_id = :uc AND spec_version < :v",
                                      {"sid": int(target["session_id"]), "uc": str(target["use_case_id"]),
                                       "v": int(target["spec_version"])}, order="spec_version DESC")
        carried, status, stopped = {}, {}, set()
        for row in rows:
            row_status = row.get("asset_status") or {}
            for name, entry in row_status.items():
                status.setdefault(name, entry)
            stopped |= {name for name, entry in row_status.items() if (entry or {}).get("state") in IMPLEMENT_STOPPED_STATES}
            for column, output in self._IMPLEMENT_CARRIED.items():
                if column in carried or (output and output in stopped) or row.get(column) in (None, "", [], {}):
                    continue
                carried[column] = row[column]
        return carried, status

    def _implement_use_case_row(self, session_id: int, use_case_id: str):
        if not self._tracking_table_name:
            return None
        rows = self.spark.sql(f"SELECT * FROM {self._tracking_table_name} WHERE session_id = :sid AND id = :uc LIMIT 1",
                              args={"sid": int(session_id), "uc": str(use_case_id)}).collect()
        return rows[0] if rows else None

    def _run_implement_mode(self):
        if not self.inspire_database:
            raise ValueError("Implement Use Cases needs the Inspire Database widget: the wizard specs live in its __inspire_implement table")
        log_print(f"\n{'='*80}\n🚀 OPERATION = '{IMPLEMENT_OPERATION}' — discovery pipeline skipped\n{'='*80}")
        self.atomic_writer._ensure_implement_table()
        targets = self._implement_targets()
        ran = None if targets else self._implement_requested_row()
        if ran and ran["status"] in IMPLEMENT_DONE_STATUSES:
            log_print(f"🔁 [IMPLEMENT-ALREADY-RAN] {ran['use_case_id']} spec {ran['spec_version']} is already {ran['status']}; "
                      "nothing to build")
            self._implement_outcomes = {ran["use_case_id"]: ran["status"]}
            return
        if ran:
            raise RuntimeError(implement_settled_message(ran))
        if not targets:
            raise RuntimeError(f"No queued use case PoC/MVP spec for session {self.session_id}"
                               + (f" and use case {self._implement_param(IMPLEMENT_USE_CASE_PARAM)}"
                                  if self._implement_param(IMPLEMENT_USE_CASE_PARAM) else "")
                               + f" in {self._implement_table_sql()}; submit the wizard first.")
        log_print(f"🔧 Implementing {len(targets)} use case(s): {', '.join(t['use_case_id'] for t in targets)}")
        outcomes = {}
        for target in targets:
            outcomes[target["use_case_id"]] = self._implement_one(target)
        self._implement_outcomes = outcomes
        failed = [uc for uc, status in outcomes.items() if status == "failed"]
        log_print(f"\n{'='*80}\n🧾 Implement summary: " + ", ".join(f"{uc}={status}" for uc, status in outcomes.items()) + f"\n{'='*80}")
        if failed and len(failed) == len(outcomes):
            raise RuntimeError(f"The use case PoC/MVP failed for {', '.join(failed)}; the Build page shows each step and error.")
        if failed:
            self._record_failed_step("Implement Use Cases", "", f"{len(failed)} of {len(outcomes)} failed: {', '.join(failed)}")

    def _implement_one(self, target: dict) -> str:
        uc_id = str(target["use_case_id"])
        key = (int(target["session_id"]), uc_id, int(target["spec_version"]))
        lock = implement_lock_key(self.business_name, uc_id)
        try:
            held = self.atomic_writer.acquire_lock(lock, ttl_seconds=IMPLEMENT_LOCK_TTL_SECONDS,
                                                   holder_note=f"operation={IMPLEMENT_OPERATION} spec={key[2]}")
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-LOCK UNVERIFIED] {lock}: {get_clean_error_message(error)}; proceeding without it")
            held = True
        if not held:
            message = f"another Implement run is already building use case {uc_id}; this spec stays queued for the next run"
            self._implement_emit(uc_id, "Lock", message, "ended_warning", {"spec_version": key[2]})
            return "queued"
        inflight = {}
        try:
            return self._implement_one_locked(target, key, inflight)
        except Exception as error:
            message = get_clean_error_message(error)
            self.logger.error(f"[IMPLEMENT-FAILED] {uc_id} spec {key[2]}: {message}")
            self._implement_emit(uc_id, "Implement", message[:1500], "ended_error", {"spec_version": key[2]})
            self._implement_notebook_undo(uc_id, inflight)
            try:
                self._implement_update(key, status="failed", status_message=message[:4000], **self._implement_stopped_fields(key, inflight))
            except Exception as write_error:
                self.logger.error(f"[IMPLEMENT-STATUS-WRITE FAILED] {uc_id}: {get_clean_error_message(write_error)}")
            return "failed"
        finally:
            try:
                self.atomic_writer.release_lock(lock)
            except Exception as error:
                self.logger.warning(f"[IMPLEMENT-LOCK RELEASE FAILED] {lock}: {get_clean_error_message(error)}")

    def _implement_stopped_fields(self, key: tuple, inflight: dict) -> dict:
        asset_status, path = inflight.get("asset_status"), inflight.get("notebook_path")
        if asset_status is None or not path:
            return {}
        modified_at = self._implement_notebook_modified_at(path)
        if modified_at is None:
            return {}
        restored = inflight.get("notebook_restored")
        asset_status["notebook"] = (dict(restored, modified_at=modified_at) if restored is not None else
                                    {"modified_at": modified_at, "spec_version": key[2], "result_history": IMPLEMENT_RESULT_HISTORY})
        return {"asset_status": asset_status}

    def _implement_notebook_before(self, path: str):
        if self._implement_notebook_modified_at(path) is None:
            return None
        try:
            return self._implement_export_text(path)
        except Exception as error:
            if implement_is_not_found(error):
                return None
            raise ImplementBlocked(f"the notebook {path} could not be read before Inspire rebuilt it, so Inspire did not change it: "
                                   f"{get_clean_error_message(error)[:500]}")

    def _implement_notebook_undo(self, uc_id: str, inflight: dict) -> None:
        text, recorded = inflight.get("notebook_before") or (None, None)
        path = inflight.get("notebook_path")
        if text is None or not path:
            return
        try:
            self._implement_import_notebook(path, text)
        except Exception as error:
            self.logger.error(f"[IMPLEMENT-UNDO FAILED] {uc_id}: this version did not build, and Inspire could not put the notebook "
                              f"{path} back as it was, so the job runs the notebook as this version left it: "
                              f"{get_clean_error_message(error)[:500]}")
            return
        inflight["notebook_restored"] = {k: v for k, v in recorded.items() if k not in ("state", "edited_at", "rewriting")}
        self.logger.warning(f"[IMPLEMENT-UNDO] {uc_id}: this version did not build, so Inspire put the notebook {path} back as it was; "
                            "the job keeps running the last built version")

    def _implement_trains_model(self, use_case: dict) -> bool:
        return self._uc_execution_family(use_case, f"Implement {use_case.get('No', 'UNKNOWN')}") in INSPIRE_MODEL_FAMILIES

    def _implement_role_problems(self, spec: dict, use_case: dict) -> list:
        roles = ((spec or {}).get("data") or {}).get("column_roles") or {}
        tables = sorted({".".join(str(ref).split(".")[:3]).lower() for ref in roles.values() if str(ref).count(".") == 3})
        if not tables:
            return []
        evidence = self._genie_table_evidence(tables)
        linked = self._evidence_linked_tables(evidence)
        if linked:
            evidence.update(self._genie_table_evidence(linked))
        return implement_column_role_problems(roles, evidence, self._elite_dated_joins(evidence, tables),
                                              trains_model=self._implement_trains_model(use_case),
                                              free_text_tokens=self._TEXT_NAME_TOKENS)

    def _implement_one_locked(self, target: dict, key: tuple, inflight: dict = None) -> str:
        inflight = {} if inflight is None else inflight
        uc_id = key[1]
        carried, asset_status = self._implement_carried(target)
        asset_status.pop("data", None)
        inflight["asset_status"] = asset_status
        spec, errors = validate_implement_spec(implement_stored_spec(target.get("spec_json") or {}))
        if spec.get("use_case_id") and spec["use_case_id"] != uc_id:
            errors.append(f"the spec names use case {spec['use_case_id']} but its row is for {uc_id}")
        if errors:
            self._implement_update(key, asset_status=asset_status, **carried)
            raise ValueError("the wizard spec is invalid: " + "; ".join(errors))
        previous = self._implement_previous(target)
        previous_spec = validate_implement_spec(implement_stored_spec(previous.get("spec_json")))[0] if previous else None
        based_on = spec.get("based_on_version")
        if based_on is not None and previous and int(previous["spec_version"]) > based_on:
            self._implement_update(key, asset_status=asset_status, **carried)
            raise ValueError(f"spec v{key[2]} was edited from v{based_on}, but v{previous['spec_version']} has been built since; "
                             f"reopen Edit to see v{previous['spec_version']} and submit again")
        assets = {name: carried.get(column) for name, column in IMPLEMENT_OUTPUT_ID_COLUMNS.items()}
        assets["preview_run_id"] = carried.get("last_preview_run_id")
        reusable = implement_reusable_notebook(previous, asset_status.get("notebook"))
        history_upgrade = (bool(carried.get("notebook_path")) and reusable
                           and not implement_notebook_keeps_history(asset_status.get("notebook")))
        basis = previous_spec if reusable else None
        plan = implement_plan(previous_spec, spec, assets,
                              has_built_notebook=bool(carried.get("notebook_path")) and reusable and not history_upgrade)
        self._implement_update(key, status="filling", change_classes=",".join(plan["classes"]),
                               status_message=f"plan: {json.dumps({k: v for k, v in plan.items() if k != 'classes'})}",
                               asset_status=asset_status, **carried)
        self._implement_emit(uc_id, "Plan", f"spec v{key[2]}: changes {', '.join(plan['classes'])}; "
                             f"{'rebuild the notebook' if plan['refill'] else 'keep the built notebook'}"
                             + ("; the notebook writes its results without a change history, so Genie updates its write step"
                                if history_upgrade else "")
                             + (f"; build {', '.join(plan['build'])}" if plan["build"] else "")
                             + (f"; refresh {', '.join(plan['refresh'])}" if plan["refresh"] else "")
                             + (f"; roll back {', '.join(f'{n} to v{v}' for n, v in plan['rollback'].items())}" if plan["rollback"] else ""),
                             "ended_success", {"spec_version": key[2], "plan": plan})
        row = self._implement_use_case_row(key[0], uc_id)
        if row is None:
            raise RuntimeError(f"use case {uc_id} is not in {self._tracking_table_name} for session {key[0]}")
        use_case = self._row_to_uc(row)
        use_case["_source_session_id"] = key[0]
        primary = (self._genie_table_names(use_case.get("Primary Table") or use_case.get("primary_table")) or [""])[0]
        removal = implement_removed_table_problems(spec, self._genie_involved_tables(use_case), primary)
        if removal:
            raise ImplementBlocked("; ".join(removal))
        if plan["preview"] or plan["refill"]:
            missing = self._implement_missing_tables(implement_source_tables(spec, self._genie_involved_tables(use_case)))
            if missing:
                asset_status["data"] = {"state": "failed", "missing_tables": missing, "spec_version": key[2]}
                self._implement_update(key, asset_status=asset_status)
                raise ImplementBlocked(implement_missing_tables_message(missing))
            misfits = self._implement_role_problems(spec, use_case)
            if misfits:
                raise ImplementBlocked("the column roles set in the wizard do not fit this use case: " + "; ".join(misfits)
                                       + ". Set them back with Reset roles to Inspire's choice in the Data step, or pick columns of the right kind")
        warnings = []
        for name in plan["keep_unmanaged"]:
            asset_status[name] = {"state": "unmanaged", "note": f"{IMPLEMENT_OUTPUT_LABELS[name]} was unticked in spec v{key[2]}; "
                                                               "it is kept and no longer refreshed"}
        for name in plan["delete"]:
            refused = self._implement_delete_output(name, assets[name], key, carried.get("job_id"))
            if refused:
                warnings.append(refused)
                asset_status[name] = {"state": "released", "note": refused}
            else:
                asset_status[name] = {"state": "deleted", "note": f"deleted at the user's request in spec v{key[2]}"}
        if not str(use_case.get("genie_instruction") or "").strip():
            use_case = self._implement_write_instruction(row, uc_id)
        elif plan["refill"]:
            use_case = self._implement_recover_instruction(row, uc_id, use_case, warnings)
        default_table = self._genie_result_table(use_case)
        if spec["config"]["result_table"]:
            use_case["genie_instruction"] = implement_retarget_instruction(use_case.get("genie_instruction"), default_table,
                                                                           spec["config"]["result_table"])
            use_case["result_table"] = spec["config"]["result_table"]
        moved = implement_result_table_move(previous_spec, spec, default_table)
        spec_nb = self._genie_notebook_spec(use_case)
        known = []
        for widget in (self._genie_core_widgets(spec_nb) + list(carried.get("genie_widgets") or [])
                       + self._genie_instruction_widgets(str(use_case.get("genie_instruction") or ""))):
            if isinstance(widget, dict) and widget.get("name") not in {k["name"] for k in known}:
                known.append(widget)
        widget_problems = implement_widget_problems(known, spec["config"].get("widgets"))
        if widget_problems:
            raise ImplementBlocked("widget values are out of range: " + "; ".join(widget_problems))
        notebook_path, job_id = carried.get("notebook_path"), carried.get("job_id")
        runner = None
        widget_names = self._genie_widget_names(str(use_case.get("genie_instruction") or ""), spec_nb) + ["run_profile"]
        result_table = ".".join(inspire_table_parts(use_case["result_table"]))
        use_case["_implement_notes"] = (spec.get("data") or {}).get("notes") or ""
        if moved:
            self._implement_move_result_table(uc_id, key, moved[0], moved[1], warnings)
        if plan["refill"]:
            runner = self._implement_runner(uc_id)
            recorded = dict(asset_status.get("notebook") or {})
            inflight["notebook_path"] = self._implement_claim_notebook(use_case, spec, key, asset_status)
            inflight["notebook_before"] = (self._implement_notebook_before(inflight["notebook_path"]), recorded)
            use_case["_history_upgrade"] = history_upgrade
            notebook_path = self._implement_write_notebook(use_case, spec, basis)
            self._implement_update(key, notebook_path=notebook_path)
        elif notebook_path:
            self._implement_refresh_run_controls(uc_id, key, use_case, spec_nb, notebook_path, asset_status, warnings)
        outcome = {}
        if plan["preview"] or plan["refill"]:
            self._implement_keep_edits_setting(uc_id, result_table, spec, warnings)
            outcome, job_id, fill = self._implement_fill_and_preview(uc_id, key, use_case, spec, basis, notebook_path,
                                                                     widget_names, runner, plan["refill"])
            if fill:
                self._implement_update(key, genie_widgets=fill.get("widgets"), data_profiles=fill.get("profiles"),
                                       suggested_schedule=fill.get("schedule"))
            if plan["refill"] and outcome.get("status") == "written":
                asset_status["notebook"] = {"modified_at": self._implement_notebook_modified_at(notebook_path), "spec_version": key[2],
                                            "result_history": IMPLEMENT_RESULT_HISTORY}
                self._implement_update(key, asset_status=asset_status)
                inflight.pop("notebook_path", None)
                inflight.pop("notebook_before", None)
            if outcome.get("status") != "written":
                raise RuntimeError(implement_stop_message(outcome, spec, trains_model=self._implement_trains_model(use_case)))
            if outcome.get("build_problems"):
                warnings.append(f"Notebook: Genie's last build round left these unfixed: {'; '.join(outcome['build_problems'])}"[:1500])
        self._implement_result_history(uc_id, result_table, spec, warnings)
        gone = {t.lower() for t in spec["data"].get("removed_tables") or []}
        allowed = ([result_table] + [t for t in self._genie_involved_tables(use_case) if t.lower() not in gone]
                   + list((spec["data"]["tables"] or {}).values()))
        reasons_view = (self._implement_reasons_view(result_table, warnings)
                        if plan["build"] or plan["refresh"] or plan["rollback"] or plan["sharing"] else "")
        if reasons_view:
            allowed.append(reasons_view)
        context = {"uc_id": uc_id, "key": key, "use_case": use_case, "spec": spec, "result_table": result_table, "allowed": allowed,
                   "reasons_view": reasons_view, "restyle": plan["restyle"],
                   "notebook_spec": spec_nb, "notebook_path": notebook_path, "assets": assets, "job_id": job_id, "asset_status": asset_status,
                   "urls": {c: carried[c] for c in ("dashboard_url", "genie_space_url", "app_url", "alert_url") if carried.get(c)}}
        fallback_warning = (self._implement_fallback_warning(uc_id, result_table, use_case, spec_nb)
                            if outcome.get("status") == "written" else "")
        if fallback_warning:
            warnings.append(fallback_warning)
        tier_warning = self._implement_tier_warning(uc_id, result_table, outcome.get("job_url") or "", outcome.get("run_id"))
        if tier_warning:
            warnings.append(tier_warning)
        if plan["build"] or plan["refresh"] or plan["rollback"]:
            self._implement_update(key, status="building_outputs")
        builders = {"dashboard": self._implement_dashboard, "genie_space": self._implement_genie_space, "app": self._implement_app,
                    "alert": self._implement_alert}
        runner_error, snapshots = None, {}
        for name in IMPLEMENT_OUTPUTS:
            if name not in plan["build"] and name not in plan["refresh"] and name not in plan["rollback"]:
                continue
            try:
                (context.get("artifacts") or {}).pop(name, None)
                vibes = implement_output_vibes(spec["outputs"], name)
                if name in plan["rollback"]:
                    version = plan["rollback"][name]
                    log_print(f"⏪ [IMPLEMENT-ROLLBACK] {uc_id} {name} v{version} -> v{key[2]}")
                    saved = self._implement_snapshot(key, name, version)
                    context.setdefault("replay", {})[name] = (version, saved)
                    vibes = str(saved.get("vibes") or "")
                elif name != "alert" and name not in plan["restyle"] and runner is None:
                    if runner_error is None:
                        try:
                            runner = self._implement_runner(uc_id)
                        except Exception as error:
                            runner_error = f"Genie Code could not be installed: {get_clean_error_message(error)}"
                    if runner_error:
                        raise RuntimeError(runner_error)
                fields = builders[name](context, runner)
                health = fields.pop("app_health", None)
                extra = fields.pop("_asset", None) or {}
                own = fields.pop("_warnings", None) or []
                artifact = (context.get("artifacts") or {}).pop(name, None)
                if artifact is not None:
                    snapshots[name] = {"artifact": artifact, "vibes": vibes, "spec_version": key[2],
                                       "merged_into": str(extra.get("merged_into") or "")}
                    fields["asset_snapshots"] = json.dumps(snapshots, ensure_ascii=False, default=str)
                self._implement_update(key, **fields)
                assets[name] = fields.get(IMPLEMENT_OUTPUT_ID_COLUMNS[name])
                context["urls"].update({k: v for k, v in fields.items() if k.endswith("_url")})
                asset_status[name] = dict(extra, state="ready", spec_version=key[2], **({"snapshot": True} if artifact is not None else {}))
                if health:
                    asset_status[name]["health"] = health
                    if health not in IMPLEMENT_APP_HEALTH_OK:
                        own.append(f"{IMPLEMENT_OUTPUT_LABELS[name]}: health check {health}")
                if own:
                    asset_status[name]["warnings"] = [w.removeprefix(f"{IMPLEMENT_OUTPUT_LABELS[name]}: ") for w in own]
                warnings += own
            except Exception as error:
                message = get_clean_error_message(error)
                warnings.append(f"{IMPLEMENT_OUTPUT_LABELS[name]}: {message[:500]}")
                asset_status[name] = {"state": "failed", "error": message[:2000], "spec_version": key[2]}
                self._implement_emit(uc_id, IMPLEMENT_OUTPUT_LABELS[name], message[:1500], "ended_warning", {"spec_version": key[2]})
        if plan["schedule"] and job_id:
            try:
                self._implement_schedule(uc_id, int(job_id), spec["schedule"])
            except Exception as error:
                warnings.append(f"Schedule: {get_clean_error_message(error)[:500]}")
        if plan["sharing"]:
            warnings += self._implement_share(uc_id, result_table, spec, assets, job_id, reasons_view)
        status = "ready_with_warnings" if warnings else "ready"
        self._implement_update(key, status=status, asset_status=asset_status, job_id=job_id,
                               status_message=("; ".join(warnings) if warnings else
                                               f"ready: preview wrote {outcome['rows_written']} row(s)" if outcome.get("status") == "written"
                                               else f"ready: no new preview was needed; the last preview wrote "
                                                    f"{carried.get('result_rows')} row(s)"))
        self._implement_emit(uc_id, "Done", "; ".join(warnings) if warnings else "the use case is implemented",
                             "ended_warning" if warnings else "ended_success", {"spec_version": key[2], "assets": asset_status})
        return status

    def _implement_missing_tables(self, tables: list) -> list:
        missing = []
        for table in tables:
            try:
                self.spark.sql(f"SELECT * FROM {inspire_table_name(table)} LIMIT 0").collect()
            except Exception as error:
                message = get_clean_error_message(error)
                reason = implement_table_access_reason(table, message)
                if reason:
                    missing.append({"table": table, "reason": reason, "error": message[:300]})
                else:
                    self.logger.warning(f"[IMPLEMENT-DATA] could not check {table}, so the preview goes ahead: {message[:300]}")
        return missing

    def _implement_write_instruction(self, row, uc_id: str, reason: str = "") -> dict:
        step = self._implement_emit(uc_id, "Instruction", reason or "this use case has no Genie Code instruction yet; writing it first", "started")
        regenerated = self._regenerate_genie_for_flagged_rows([row], "Implement", skip_cache=bool(reason))
        if not regenerated:
            raise RuntimeError("the Genie Code instruction could not be written, so there is nothing for Genie to build")
        self._merge_regenerated_ucs_into_tracking(regenerated)
        self._implement_emit(uc_id, "Instruction", "Genie Code instruction written", "ended_success", step_id=step)
        return regenerated[0]

    def _implement_recover_instruction(self, row, uc_id: str, use_case: dict, warnings: list) -> dict:
        conflicts = self._genie_contract_conflicts(str(use_case.get("genie_instruction") or ""), self._genie_notebook_spec(use_case))
        if not conflicts:
            return use_case
        reason = (f"the saved Genie Code instruction words acceptance or a stop differently from Inspire's {GENIE_ACCEPTANCE_SECTION_TITLE} "
                  f"in {len(conflicts)} sentence(s), so Inspire writes it again: " + "; ".join(conflicts))[:1500]
        try:
            fresh = self._implement_write_instruction(row, uc_id, reason=reason)
        except Exception as error:
            warnings.append(f"Instruction: {len(conflicts)} sentence(s) of the saved instruction word acceptance or a stop differently from "
                            f"Inspire's {GENIE_ACCEPTANCE_SECTION_TITLE}, and writing it again failed "
                            f"({get_clean_error_message(error)[:300]}); that section wins over them")
            return use_case
        left = self._genie_contract_conflicts(str(fresh.get("genie_instruction") or ""), self._genie_notebook_spec(fresh))
        if left:
            warnings.append(f"Instruction: {len(left)} sentence(s) still word acceptance or a stop differently after "
                            f"{GENIE_INSTRUCTION_REPAIR_ROUNDS} repair round(s); Inspire's {GENIE_ACCEPTANCE_SECTION_TITLE} section wins over them")
        return fresh

    def _implement_runner(self, uc_id: str):
        factory = getattr(self, "_genie_runner_factory", None)
        if factory:
            return factory(uc_id)
        ctx = self.dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        host = ctx.apiUrl().get()
        token = ctx.apiToken().get()
        runner = GenieCodeRunner(host, token, self.logger, on_line=lambda line: self.logger.info(f"[GENIE {uc_id}] {line}"))
        step = self._implement_emit(uc_id, "Genie Code", "installing the Genie Code CLI and the Databricks CLI", "started")
        try:
            installed = runner.install()
        except Exception as error:
            self._implement_emit(uc_id, "Genie Code", f"Genie Code could not be installed: {get_clean_error_message(error)[:1500]}",
                                 "ended_error", step_id=step)
            raise
        self._implement_emit(uc_id, "Genie Code", f"Genie Code {GENIE_CODE_CLI_VERSION} ready at {installed['genie']}", "ended_success",
                             step_id=step)
        return runner

    def _implement_notebook_paths(self, use_case: dict) -> tuple:
        notebook_dir, _full, filename = self._build_notebook_workspace_paths(use_case)
        relative = os.path.relpath(notebook_dir, self.notebook_output_dir)
        target_dir = os.path.normpath(os.path.join(self.base_output_dir + "_implemented", relative))
        return target_dir, os.path.join(target_dir, filename)

    def _implement_notebook_modified_at(self, path: str):
        try:
            return getattr(self.w_client.workspace.get_status(path), "modified_at", None)
        except Exception:
            return None

    def _implement_notebook_edit_state(self, path: str, recorded: dict) -> tuple:
        rewriting = recorded.get("rewriting") or {}
        ended = (job_run_ended(rewriting["run_id"]) or {}).get("end_time") if rewriting.get("run_id") else None
        modified_at = self._implement_notebook_modified_at(path)
        return modified_at, implement_notebook_edited(modified_at, recorded, ended)

    def _implement_import_notebook(self, path: str, text: str) -> None:
        from databricks.sdk.service import workspace as _ws
        self.w_client.workspace.import_(path=path, overwrite=True, format=_ws.ImportFormat.JUPYTER,
                                        content=base64.b64encode(text.encode("utf-8")).decode())

    def _implement_refresh_run_controls(self, uc_id: str, key: tuple, use_case: dict, spec_nb: dict, path: str,
                                        asset_status: dict, warnings: list) -> None:
        instruction_title, controls_title = self._IMPLEMENT_INSPIRE_CELLS
        recorded = dict(asset_status.get("notebook") or {})
        before, edited = self._implement_notebook_edit_state(path, recorded)
        if before is None:
            return
        try:
            notebook = json.loads(self._implement_export_text(path))
            if not isinstance(notebook, dict) or not isinstance(notebook.get("cells"), list):
                raise ValueError("the export is not a notebook")
        except Exception as error:
            warnings.append(f"Notebook: Inspire could not read {path} to bring its Run controls up to date, so its runs may not carry "
                            f"forward what people did in the App: {get_clean_error_message(error)[:300]}")
            return
        found = [cell for cell in notebook["cells"] if implement_cell_title(cell) == instruction_title]
        try:
            if len(found) != 1:
                raise ValueError(f"it holds {len(found)} '{instruction_title}' cells, not one")
            instruction = ast.literal_eval(implement_cell_text(found[0]).strip())
            if not isinstance(instruction, str):
                raise ValueError(f"its '{instruction_title}' cell is not the instruction text")
        except (ValueError, TypeError, SyntaxError, MemoryError, RecursionError) as error:
            warnings.append(f"Notebook: Inspire could not bring the Run controls of {path} up to date, because {error}, so its runs may "
                            "not carry forward what people did in the App; rebuild the notebook with Edit the PoC/MVP on the Build page")
            return
        controls = self._genie_cell("code", self._genie_run_controls_source(use_case, spec_nb, instruction), controls_title)
        cells, changes = implement_restore_owned_cells(notebook["cells"], found[0], controls)
        if not changes:
            return
        try:
            self._implement_import_notebook(path, json.dumps(dict(notebook, cells=cells)))
        except Exception as error:
            warnings.append(f"Notebook: Inspire could not write the up-to-date Run controls into {path}, so its runs may not carry "
                            f"forward what people did in the App: {get_clean_error_message(error)[:300]}")
            return
        self.logger.info(f"[IMPLEMENT-NOTEBOOK] {uc_id}: brought the Run controls of {path} up to date ({'; '.join(changes)})")
        after = self._implement_notebook_modified_at(path)
        if recorded.get("modified_at") is not None and not edited and after is not None:
            asset_status["notebook"] = dict(recorded, modified_at=after)
            self._implement_update(key, asset_status=asset_status)

    def _implement_claim_notebook(self, use_case: dict, spec: dict, key: tuple, asset_status: dict) -> str:
        path = self._implement_notebook_paths(use_case)[1]
        recorded = dict(asset_status.get("notebook") or {})
        modified_at, edited = self._implement_notebook_edit_state(path, recorded)
        if edited and not spec.get("confirm_notebook_overwrite"):
            asset_status["notebook"] = dict(recorded, state=IMPLEMENT_NOTEBOOK_EDITED, edited_at=int(modified_at))
            self._implement_update(key, asset_status=asset_status)
            raise ImplementBlocked(implement_notebook_edited_message(path))
        asset_status["notebook"] = dict({k: v for k, v in recorded.items() if k not in ("state", "edited_at", "rewriting")},
                                        rewriting={"spec_version": key[2], "run_id": current_job_run_id()})
        self._implement_update(key, asset_status=asset_status)
        return path

    def _implement_write_notebook(self, use_case: dict, spec: dict, previous_spec) -> str:
        target_dir, path = self._implement_notebook_paths(use_case)
        instruction = sanitize_llm_special_chars(str(use_case.get("genie_instruction") or "").strip())
        instruction = self._genie_with_platform_rules(instruction, self._genie_notebook_spec(use_case))
        trains = self._implement_trains_model(use_case)
        dismissed = self._implement_dismissals(str(use_case.get("No") or ""), str(use_case.get("result_table") or ""))
        instruction = (instruction + "\n\n" + implement_choices_markdown(spec, previous_spec, trains_model=trains, dismissed=dismissed)
                       + (IMPLEMENT_HISTORY_UPGRADE_CHOICE if use_case.get("_history_upgrade") else ""))
        built = dict(use_case, genie_instruction=instruction)
        cells, _spec = self._genie_notebook_cells(built, instruction)
        self._implement_owned_cells = {path: (cells[1], cells[2])}
        modified_at = self._implement_notebook_modified_at(path)
        notebook = None
        if modified_at is not None and previous_spec:
            try:
                notebook = json.loads(self._implement_export_text(path))
                titles = [implement_cell_title(c) for c in notebook.get("cells") or []]
                replaced = 0
                for index, title in enumerate(titles):
                    if title == "Genie Code instruction":
                        notebook["cells"][index] = cells[1]
                        replaced += 1
                    elif title == "Run controls (Inspire, do not edit)":
                        notebook["cells"][index] = cells[2]
                        replaced += 1
                if replaced != 2:
                    notebook = None
            except Exception as error:
                self.logger.warning(f"[IMPLEMENT-NOTEBOOK] {path} could not be read back ({get_clean_error_message(error)}); "
                                    "writing it from the instruction")
                notebook = None
        if notebook is None:
            notebook = self._genie_notebook_object(cells, os.path.basename(path))
        self.w_client.workspace.mkdirs(target_dir)
        self._implement_import_notebook(path, json.dumps(notebook))
        self.logger.info(f"[IMPLEMENT-NOTEBOOK] {use_case.get('No')}: wrote {path}")
        return path

    def _implement_dismissals(self, uc_id: str, result_table: str):
        if not uc_id or not result_table:
            return None
        try:
            have = {str(name).lower() for name, _kind in inspire_table_schema(self.spark, result_table) or []}
            if not {"action_status", "dismiss_reason", "is_current", "change_source", "use_case_id"} <= have:
                return None
            rows = [row.asDict() if hasattr(row, "asDict") else dict(row)
                    for row in self.spark.sql(implement_dismissals_sql(result_table), args={"uc": uc_id}).collect()]
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-FEEDBACK] {uc_id}: the records people dismissed in {result_table} could not be read, so "
                                f"Genie does not see them: {get_clean_error_message(error)[:300]}")
            return None
        reasons = [(" ".join(str(row["reason"]).split())[:200], int(row.get("records") or 0)) for row in rows
                   if row.get("reason") is not None]
        found = {"dismissed": int(rows[0].get("dismissed") or 0) if rows else 0, "reasons": reasons[:IMPLEMENT_DISMISS_REASONS]}
        if found["dismissed"]:
            self.logger.info(f"[IMPLEMENT-FEEDBACK] {uc_id}: Genie sees the {found['dismissed']} record(s) people dismissed in "
                             f"{result_table}, with {len(found['reasons'])} reason(s)")
        return found

    _IMPLEMENT_FILL_PROMPT = (
        "You are working in the Databricks notebook {path} in this workspace. The notebook's second cell holds a "
        "Genie Code instruction for one use case, followed by the Implementation Choices the user made in the Start Use "
        "Case PoC/MVP with Genie wizard; those choices win over the instruction. {task} Add your cells between the instruction cell and the "
        "'Run controls' cell, which stays the last cell. Those two cells are Inspire's: never edit, move, copy or delete them, "
        "and Inspire puts them back if you do. Numbers in text: {number_rule} "
        "Run the notebook with a one-time serverless "
        "run (databricks jobs submit) with the Execution widget set to Here and run_profile set to preview, and keep fixing it "
        "until it runs from top to bottom or the instruction's stop rule applies. The stop rule counts only for a run of cells "
        "that do what the instruction and its Implementation Choices say: cells that read other columns or tables than those "
        "name are a defect to fix before the run counts. A run denied a privilege on the result table, the model the widgets "
        "name or a source table is not a defect to code around: never catch the error or skip the step that needs it, such as "
        "registering the model; set status to stopped and put the denial in stop_reason. When the stop rule applies, set status to "
        "stopped and put the message the notebook printed in stop_reason. Never create a persistent job: Inspire runs "
        "the notebook as its tagged job after you finish. When you finish, reply with only the JSON object your output schema "
        "describes: every widget of the Parameters cell including run_profile (name, type, default, the choices of a dropdown, "
        "the allowed range written as '0 to 1', 'at least 1', 'at most 5', 'whole number, 1 to 10' or empty when any value works, "
        "label), the training and full-load data profiles, and the Quartz cron schedule you suggest for production runs.{feedback}")

    def _implement_fill_and_preview(self, uc_id, key, use_case, spec, previous_spec, notebook_path, widget_names, runner, refill):
        spec_nb = self._genie_notebook_spec(use_case)
        feedback, fill, outcome, job_id, step = "", {}, {}, None, None
        rounds = IMPLEMENT_MAX_GENIE_ROUNDS if refill else 1
        for round_no in range(1, rounds + 1):
            problems = []
            if refill:
                task = ("Build the notebook it describes in that same notebook." if not previous_spec else
                        "The notebook was built before for earlier choices; update it for the current choices.")
                prompt = self._IMPLEMENT_FILL_PROMPT.format(path=notebook_path, task=task, feedback=feedback,
                                                            number_rule=GENIE_NUMBER_FORMAT_RULE)
                step = self._implement_emit(uc_id, "Genie build", f"round {round_no}/{rounds}: Genie is building {notebook_path}",
                                            "started", {"spec_version": key[2], "round": round_no, "rounds": rounds}, step_id=step)
                self._implement_update(key, status="filling", status_message=f"Genie build round {round_no}/{rounds}")
                work = self._implement_work_dir(uc_id, f"fill_{round_no}")
                shutil.rmtree(work, ignore_errors=True)
                result = runner.run(prompt, work, GENIE_FILL_RESULT_SCHEMA, label="build")
                fill = result.get("final") or {}
                restored = self._implement_restore_owned_cells(notebook_path)
                problems = ([f"Genie exited with code {result.get('exit')}" + (" after the time limit" if result.get("timed_out") else "")]
                            if result.get("exit") else []) + normalize_genie_fill_widgets(fill, self._genie_core_widgets(spec_nb))
                problems += validate_genie_fill_result(fill, widget_names[:-1])
                if restored:
                    problems.append(f"you {'; '.join(restored)}. Those cells are Inspire's, so Inspire put them back as it wrote them, "
                                    "with the Run controls cell last: never edit, move, copy or delete them")
                stopped = genie_fill_stopped(fill)
                cell_problems = [] if stopped else self._implement_cell_problems(notebook_path)
                problems += cell_problems
                if result.get("secret_files"):
                    problems.append(f"{', '.join(result['secret_files'])} held the access token (Inspire removed it); never write "
                                    "credentials into a file")
                if self._implement_redact_notebook_secret(notebook_path, getattr(runner, "token", "")):
                    problems.append("the notebook held the access token (Inspire removed it); read credentials from the runtime, "
                                    "never write them into a cell")
                if stopped:
                    reason = str(fill.get("stop_reason") or "").strip() or "Genie gave no reason"
                    not_a_stop = self._implement_acceptance_stop_problem(spec_nb, reason)
                    if not_a_stop and round_no < rounds:
                        feedback = "\n\nInspire checked your last attempt and found: " + "; ".join(problems + [not_a_stop]) + ". Fix these, then reply again."
                        self._implement_emit(uc_id, "Genie build", f"round {round_no}/{rounds}: {not_a_stop}"[:1500], "started",
                                             {"spec_version": key[2], "round": round_no, "rounds": rounds, "failure": not_a_stop[:1500]},
                                             step_id=step)
                        continue
                    self._implement_emit(uc_id, "Genie build", f"the instruction's stop rule applied: {reason}",
                                         "ended_error", {"spec_version": key[2], "round": round_no}, step_id=step)
                    stop = {"status": "stopped", "stop_reason": reason, "stopped_by": "genie"}
                    return (dict(stop, not_a_stop=not_a_stop, sent_back=True) if not_a_stop else stop), job_id, fill
                if problems:
                    found = "; ".join(problems)[:1500]
                    feedback = ("\n\nInspire checked your last attempt and found: " + "; ".join(problems)
                                + ". Fix these, then reply again.")
                    if (result.get("exit") or cell_problems) and round_no < rounds:
                        self._implement_emit(uc_id, "Genie build", f"round {round_no}/{rounds} failed, sent back to Genie: {found}",
                                             "started", {"spec_version": key[2], "round": round_no, "rounds": rounds,
                                                         "failure": found}, step_id=step)
                        continue
                    self._implement_emit(uc_id, "Genie build", found, "ended_warning",
                                         {"spec_version": key[2], "round": round_no, "rounds": rounds}, step_id=step)
                else:
                    self._implement_emit(uc_id, "Genie build", fill.get("summary") or "built", "ended_success",
                                         {"spec_version": key[2], "round": round_no, "widgets": len(fill.get("widgets") or [])},
                                         step_id=step)
            self._implement_update(key, status="previewing")
            outcome, job_id, run_id = self._implement_preview(uc_id, key, use_case, spec, spec_nb, notebook_path, widget_names)
            self._implement_update(key, job_id=job_id, last_preview_run_id=run_id,
                                   result_rows=outcome.get("rows_written") if outcome.get("status") == "written" else None)
            not_a_stop = self._implement_acceptance_stop_problem(spec_nb, outcome.get("stop_reason")) if outcome.get("status") == "stopped" else ""
            if not_a_stop and round_no < rounds:
                feedback = ("\n\nInspire ran the notebook as its tagged job with run_profile=preview after your last attempt, and "
                            + not_a_stop + "." + (f" Inspire also found: {'; '.join(problems)}." if problems else ""))
                self._implement_emit(uc_id, "Genie build", f"the preview run stopped, sent back to Genie: {not_a_stop}"[:1500], "started",
                                     {"spec_version": key[2], "round": round_no, "rounds": rounds, "failure": not_a_stop[:1500]}, step_id=step)
                continue
            if not_a_stop:
                outcome = dict(outcome, not_a_stop=not_a_stop, sent_back=refill)
            if outcome.get("status") == "written" and problems:
                return dict(outcome, build_problems=problems), job_id, fill
            if outcome.get("status") in ("written", "stopped") or outcome.get("launched") is False:
                return outcome, job_id, fill
            error = str(outcome.get("error") or outcome)
            denied = implement_privilege_denial(error, implement_owned_objects(spec, self._genie_involved_tables(use_case)))
            if denied:
                self.logger.warning(f"[IMPLEMENT-DENIED] {uc_id}: not sent back to Genie: {denied}")
                return dict(outcome, denied=denied), job_id, fill
            feedback = ("\n\nInspire ran the notebook as its tagged job with run_profile=preview after your last attempt, and the "
                        f"run failed: {error[:3000]}. Fix the notebook so that run succeeds."
                        + (" The run was denied a privilege on an object this use case does not own: write only to the result "
                           "table, register only the model the widgets name, and never catch a permission error."
                           if _IMPLEMENT_TABLE_DENIED_RE.search(error) else "")
                        + (f" Inspire also found: {'; '.join(problems)}." if problems else ""))
        return outcome, job_id, fill

    _GENIE_STOP_PHRASE_RE = re.compile(r'prints "([^"]+)"')
    _GENIE_FALLBACK_METHOD_RE = re.compile(r"with method `([^`]+)`")

    def _implement_acceptance_stop_problem(self, spec_nb: dict, reason) -> str:
        rule = self._genie_acceptance_stop_rule(spec_nb)
        phrase = self._GENIE_STOP_PHRASE_RE.search(rule)
        if "writes nothing" in rule or not phrase or phrase.group(1).lower() not in str(reason or "").lower():
            return ""
        return (f"the run stopped with \"{phrase.group(1)}\", but that is not a stop for this use case: {rule} Make the notebook write "
                "those rows and return status 'written', then run it again")

    def _implement_fallback_warning(self, uc_id: str, result_table: str, use_case: dict, spec_nb: dict) -> str:
        fallback = self._GENIE_FALLBACK_METHOD_RE.search(self._genie_acceptance_stop_rule(spec_nb))
        trains = self._implement_trains_model(use_case)
        if not fallback and not trains:
            return ""
        try:
            methods = self._implement_latest_run_methods(result_table)
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-ACCEPTANCE] {uc_id}: could not read the methods of the latest run of {result_table}: "
                                f"{get_clean_error_message(error)[:300]}")
            return ""
        named = [name for name, _rows in (methods or {}).get("methods") or []]
        if not named:
            return ""
        if trains and not methods.get("versioned_rows"):
            warning = (f"Notebook: no model passed acceptance on the latest run, so no model is registered and its rows are the "
                       f"baseline's ({', '.join(named)}); the run's summary gives the candidates' metrics against the acceptance rule")
        elif fallback and named == [fallback.group(1)]:
            warning = (f"Notebook: no candidate passed acceptance on the latest run, so every row comes from the fallback "
                       f"{fallback.group(1)}; the run's summary gives the candidates' metrics against the acceptance rule")
        else:
            return ""
        self.logger.warning(f"[IMPLEMENT-ACCEPTANCE] {uc_id}: {warning}")
        return warning

    def _implement_work_dir(self, uc_id: str, name: str) -> str:
        return os.path.join(self.local_log_output_dir, "implement", self._sanitize_name(uc_id), name)

    def _implement_export_text(self, path: str) -> str:
        from databricks.sdk.service import workspace as _ws
        exported = self.w_client.workspace.export(path, format=_ws.ExportFormat.JUPYTER)
        return base64.b64decode(exported.content).decode("utf-8")

    _IMPLEMENT_INSPIRE_CELLS = ("Genie Code instruction", "Run controls (Inspire, do not edit)")

    def _implement_restore_owned_cells(self, path: str) -> list:
        owned = (getattr(self, "_implement_owned_cells", None) or {}).get(path)
        if not owned:
            return []
        try:
            notebook = json.loads(self._implement_export_text(path))
            if not isinstance(notebook, dict) or not isinstance(notebook.get("cells"), list):
                raise ValueError("the export is not a notebook")
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-CELLS] {path} could not be read to check Inspire's cells: "
                                f"{get_clean_error_message(error)}")
            return []
        cells, changes = implement_restore_owned_cells(notebook["cells"], *owned)
        if changes:
            notebook["cells"] = cells
            self._implement_import_notebook(path, json.dumps(notebook))
            self.logger.warning(f"[IMPLEMENT-CELLS] {path}: Genie {'; '.join(changes)}; Inspire put them back")
        return changes

    def _implement_cell_problems(self, path: str) -> list:
        try:
            notebook = json.loads(self._implement_export_text(path))
            if not isinstance(notebook, dict):
                raise ValueError("the export is not a notebook")
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-CELLS] {path} could not be read to check how its cells write numbers and handle "
                                f"errors: {get_clean_error_message(error)}")
            return []
        cells = []
        for index, cell in enumerate(notebook.get("cells") or [], 1):
            if cell.get("cell_type") == "code" and implement_cell_title(cell) not in self._IMPLEMENT_INSPIRE_CELLS:
                cells.append((f"cell {index}", implement_cell_text(cell)))
        return implement_number_text_problems(cells) + implement_denial_catch_problems(cells)

    def _implement_redact_notebook_secret(self, path: str, token: str) -> bool:
        if not token:
            return False
        try:
            text = self._implement_export_text(path)
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-SCRUB] {path} could not be read to check for the access token: "
                                f"{get_clean_error_message(error)}")
            return False
        if token not in text:
            return False
        self._implement_import_notebook(path, text.replace(token, "<redacted>"))
        self.logger.warning(f"[IMPLEMENT-SCRUB] {path} held the access token: redacted it")
        return True

    def _implement_tags(self, spec_nb: dict, key: tuple, notebook_path: str) -> dict:
        launcher_cls = getattr(self, "_job_launcher_cls", None) or JobLauncher
        tags = dict(spec_nb["job_tags"], dbx_inspire_ai_run_type="implement", dbx_inspire_ai_spec_version=str(key[2]),
                    dbx_inspire_ai_notebook=str(notebook_path or "").rsplit("/", 1)[-1])
        return {launcher_cls._sanitize_tag(k): launcher_cls._sanitize_tag(v) for k, v in tags.items()}

    def _implement_tag_app(self, context: dict, app_name: str) -> None:
        base = "/api/2.0/entity-tag-assignments"
        try:
            wanted = self._implement_tags(context["notebook_spec"], context["key"], context["notebook_path"])
            listed = self._implement_api("GET", f"{base}/apps/{app_name}/tags").get("tag_assignments") or []
            have = {str(t.get("tag_key")): str(t.get("tag_value") or "") for t in listed}
            for tag_key, tag_value in wanted.items():
                body = {"entity_type": "apps", "entity_id": app_name, "tag_key": tag_key, "tag_value": tag_value}
                if tag_key not in have:
                    self._implement_api("POST", base, body)
                elif have[tag_key] != tag_value:
                    self._implement_api("PATCH", f"{base}/apps/{app_name}/tags/{tag_key}", body, query={"update_mask": "tag_value"})
        except Exception as error:
            self._implement_emit(context["uc_id"], "App", f"the App {app_name} was not tagged, so its usage cannot be traced to this use case: "
                                 f"{get_clean_error_message(error)[:300]}", "ended_warning", {"spec_version": context["key"][2]})

    def _implement_preview(self, uc_id, key, use_case, spec, spec_nb, notebook_path, widget_names) -> tuple:
        launcher_cls = getattr(self, "_job_launcher_cls", None) or JobLauncher
        tags = self._implement_tags(spec_nb, key, notebook_path)
        scheduled = implement_job_parameters(widget_names, spec, "full")
        preview = implement_job_parameters(widget_names, spec, "preview")
        launcher = launcher_cls(notebook_path, scheduled, tags)
        job_name = launcher_cls.persistent_job_name(spec_nb["job_slug"], notebook_path)
        step = self._implement_emit(uc_id, "Preview run", f"running {job_name} with run_profile=preview", "started",
                                    {"spec_version": key[2]})
        result = launcher.launch(job_name=job_name, run_name=job_name, run_parameters=preview, max_concurrent_runs=1, queue=True,
                                 schedule_cron=spec["schedule"]["cron"] or spec_nb["schedule_cron"],
                                 timezone_id=spec["schedule"]["timezone"], only_tasks=[launcher_cls.TASK_KEY])
        if not result.get("success"):
            self._implement_emit(uc_id, "Preview run", f"the job could not start: {result.get('error')}", "ended_error",
                                 {"spec_version": key[2]}, step_id=step)
            return ({"status": "failed", "launched": False, "error": f"the job could not start: {result.get('error')}"},
                    result.get("job_id"), None)
        self._implement_update(key, job_id=result["job_id"], last_preview_run_id=result["run_id"])
        waited = launcher.wait_for_run(result["run_id"], timeout_s=90 * 60, out=lambda m: self.logger.info(f"[IMPLEMENT {uc_id}] {m}"))
        if waited.get("status") == "SUCCESS":
            outcome = parse_use_case_outcome(waited.get("output"))
            if outcome.get("status") == "written":
                outcome = self._implement_checked_outcome(uc_id, use_case, outcome)
        else:
            outcome = {"status": "failed", "error": waited.get("error") or waited.get("status")}
        outcome["job_url"] = result.get("job_url") or ""
        ok = outcome.get("status") == "written"
        self._implement_emit(uc_id, "Preview run",
                             f"wrote {outcome.get('rows_written')} row(s)" if ok else str(outcome.get("stop_reason") or outcome.get("error"))[:1500],
                             "ended_success" if ok else "ended_error", dict(outcome, spec_version=key[2]), step_id=step)
        return outcome, result["job_id"], result["run_id"]

    def _implement_written_key_counts(self, uc_id: str, result_table: str, run_id) -> dict:
        schema = inspire_table_schema(self.spark, result_table) or []
        current = "0" if inspire_history_missing([name for name, _kind in schema]) else "COUNT_IF(is_current AND change_source = 'run')"
        rows = self.spark.sql(f"SELECT COUNT(*) AS row_count, COUNT(as_of_date) AS dated_rows, COUNT_IF(use_case_id = :uc) AS own_rows, "
                              f"{current} AS current_rows FROM {inspire_table_name(result_table)} WHERE run_id = :rid",
                              args={"uc": uc_id, "rid": str(run_id or "")}).collect()
        row = rows[0] if rows else {}
        row = row.asDict() if hasattr(row, "asDict") else dict(row)
        return {name: int(row.get(name) or 0) for name in ("row_count", "dated_rows", "own_rows", "current_rows")}

    def _implement_checked_outcome(self, uc_id: str, use_case: dict, outcome: dict) -> dict:
        table = str(use_case.get("result_table") or "")
        try:
            problems = implement_written_key_problems(outcome, uc_id, self._implement_written_key_counts(uc_id, table, outcome.get("run_id")),
                                                      table)
        except Exception as error:
            problems = [f"Inspire could not read run {outcome.get('run_id')} back from {table or 'the business output table'}: "
                        f"{get_clean_error_message(error)[:300]}"]
        if not problems:
            self._implement_drop_undated_rows(uc_id, table)
            return outcome
        self.logger.warning(f"[IMPLEMENT-KEYS] {uc_id}: {'; '.join(problems)}")
        return {"status": "failed", "error": f"the run wrote {outcome.get('rows_written')} row(s) that Inspire cannot accept: "
                                             + "; ".join(problems)}

    def _implement_drop_undated_rows(self, uc_id: str, result_table: str):
        try:
            rows = self.spark.sql(f"DELETE FROM {inspire_table_name(result_table)} WHERE use_case_id = :uc AND as_of_date IS NULL",
                                  args={"uc": uc_id}).collect()
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-KEYS] {uc_id}: could not remove the rows without an as_of_date from {result_table}: "
                                f"{get_clean_error_message(error)[:300]}")
            return
        row = rows[0] if rows else {}
        removed = int((row.asDict() if hasattr(row, "asDict") else dict(row)).get("num_affected_rows") or 0)
        if removed:
            self.logger.warning(f"[IMPLEMENT-KEYS] {uc_id}: removed {removed} row(s) without an as_of_date from {result_table}: "
                                "notebook versions that did not write the date left them, and no later run can replace them")

    def _implement_api(self, method: str, path: str, body: dict = None, query: dict = None) -> dict:
        if query:
            return self.w_client.api_client.do(method, path, query=query, body=body) or {}
        return self.w_client.api_client.do(method, path, body=body) or {}

    def _implement_host(self) -> str:
        return str(self.w_client.config.host or "").rstrip("/")

    def _implement_genie_loop(self, context: dict, runner, label: str, prompt: str, deploy, tools: dict = None) -> dict:
        uc_id, key = context["uc_id"], context["key"]
        rounds = IMPLEMENT_MAX_GENIE_ROUNDS
        feedback, last_error, previous, step = "", "", None, None
        for round_no in range(1, rounds + 1):
            work = self._implement_work_dir(uc_id, f"{label.lower().replace(' ', '_')}_{round_no}")
            shutil.rmtree(work, ignore_errors=True)
            carried = implement_carry_files(previous, work) if previous else []
            implement_write_files(work, tools or {})
            self._implement_update(key, status_message=f"{label}: Genie round {round_no}/{rounds}")
            step = self._implement_emit(uc_id, label, f"round {round_no}/{rounds}: Genie is writing the {label}"
                                        + (f" from its last attempt ({len(carried)} file(s))" if carried else ""),
                                        "started", {"spec_version": key[2], "round": round_no, "rounds": rounds}, step_id=step)
            result = runner.run(prompt + feedback, work, GENIE_ARTIFACT_RESULT_SCHEMA, label=label)
            previous = work
            try:
                if result.get("exit"):
                    raise RuntimeError(f"Genie exited with code {result.get('exit')}"
                                       + (" after the time limit" if result.get("timed_out") else "")
                                       + f": {str(result.get('stderr'))[-500:]}")
                if result.get("secret_files"):
                    raise RuntimeError(f"{', '.join(result['secret_files'])} held the access token (Inspire removed it); never "
                                       "write credentials into a file, the runtime provides them")
                fields = deploy(work)
                self._implement_emit(uc_id, label, f"{label} ready", "ended_success",
                                     dict({k: v for k, v in fields.items() if not k.startswith("_")}, spec_version=key[2],
                                          round=round_no), step_id=step)
                return fields
            except ImplementBlocked as error:
                self._implement_emit(uc_id, label, f"round {round_no}/{rounds} failed for a reason Genie cannot fix: "
                                     f"{get_clean_error_message(error)[:1500]}", "ended_error",
                                     {"spec_version": key[2], "round": round_no}, step_id=step)
                raise
            except Exception as error:
                last_error = get_clean_error_message(error)
                retry = round_no < rounds
                self._implement_emit(uc_id, label, f"round {round_no}/{rounds} failed"
                                     + (", sent back to Genie: " if retry else ", no rounds left: ") + last_error[:1500],
                                     "started" if retry else "ended_error",
                                     {"spec_version": key[2], "round": round_no, "rounds": rounds, "failure": last_error[:1500]},
                                     step_id=step)
                feedback = (f"\n\nInspire tried to use what you wrote and it failed: {last_error[:3000]}. Your files from that attempt "
                            "are in your working folder: fix them rather than starting over, test the fix the way Inspire uses it, "
                            "and write them again.")
        raise RuntimeError(f"the {label} still failed after {rounds} Genie rounds: {last_error}")

    def _implement_checks_passed(self, context: dict, name: str) -> None:
        context.setdefault("checked", set()).add(name)

    def _implement_snapshot(self, key: tuple, name: str, version: int) -> dict:
        missing = ImplementBlocked(f"v{version} has no saved {IMPLEMENT_OUTPUT_LABELS[name]} to roll back to")
        if int(version) >= int(key[2]):
            raise missing
        implement_table = self._implement_table_sql()
        rows = self.spark.sql(f"SELECT asset_snapshots FROM {implement_table} WHERE session_id = :sid AND use_case_id = :uc AND "
                              f"spec_version = :v", args={"sid": int(key[0]), "uc": str(key[1]), "v": int(version)}).collect()
        row = (rows[0].asDict() if hasattr(rows[0], "asDict") else dict(rows[0])) if rows else {}
        try:
            saved = json.loads(row.get("asset_snapshots") or "null")
        except (TypeError, ValueError):
            saved = None
        entry = saved.get(name) if isinstance(saved, dict) else None
        artifact = entry.get("artifact") if isinstance(entry, dict) else None
        if not isinstance(artifact, dict) or (name == "alert" and not isinstance(artifact.get("condition_sql"), str)):
            raise missing
        return entry

    def _implement_replay(self, context: dict, name: str, deploy) -> dict:
        uc_id, key, label = context["uc_id"], context["key"], IMPLEMENT_OUTPUT_LABELS[name]
        version, saved = context["replay"][name]
        work = self._implement_work_dir(uc_id, f"{name}_rollback")
        shutil.rmtree(work, ignore_errors=True)
        implement_write_files(work, implement_artifact_files(name, saved["artifact"]))
        context.setdefault("checked", set()).discard(name)
        self._implement_update(key, status_message=f"{label}: rolling back to v{version}")
        step = self._implement_emit(uc_id, label, f"rolling the {label} back to the one Genie wrote in v{version}, with no Genie round; "
                                    "its checks run against the current tables first", "started",
                                    {"spec_version": key[2], "rollback_to": version})
        try:
            fields = deploy(work)
        except Exception as error:
            failure = error if name in context["checked"] else ImplementBlocked(
                f"the v{version} {label} no longer passes its checks against the current tables: "
                f"{get_clean_error_message(error, max_chars=1500)}")
            self._implement_emit(uc_id, label, f"rollback to v{version} failed: {get_clean_error_message(failure, max_chars=1500)}",
                                 "ended_error", {"spec_version": key[2], "rollback_to": version}, step_id=step)
            raise failure
        self._implement_emit(uc_id, label, f"{label} rolled back to v{version}", "ended_success",
                             dict({k: v for k, v in fields.items() if not k.startswith("_")}, spec_version=key[2], rollback_to=version),
                             step_id=step)
        return fields

    def _implement_use_case_brief(self, context: dict) -> str:
        uc = context["use_case"]
        contract = {k: text for k, _label, text in self._genie_decision_contract(uc)}
        output = self._genie_instruction_section(uc.get("genie_instruction"), "Business Output")
        *columns, _actions = self._genie_output_column_lines(context["notebook_spec"])
        view = context.get("reasons_view") or ""
        return (f"Use case {context['uc_id']}: {uc.get('Name')}\nWhat it does: {uc.get('Solution') or uc.get('Statement')}\n"
                f"Decision: {contract.get('decision')} (owner {contract.get('decision_owner')}, cadence {contract.get('cadence')})\n"
                f"KPI: {contract.get('kpi')}\nBusiness output table: `{context['result_table']}`, with these columns:\n"
                + "\n".join(f"- {line}" for line in columns)
                + f"\nChange history: the table keeps every version of each record in {', '.join(INSPIRE_HISTORY_NAMES)} "
                  f"(change_source is 'run' for the rows a run wrote, and 'edit', 'add' or 'delete' for a change people made in the "
                  f"App). The current rows, which people act on, are WHERE {INSPIRE_CURRENT_ROWS}: the latest run with people's "
                  f"changes. Never take the newest as_of_date for the latest run."
                + (f"\nSection 7, Business Output, of the notebook's instruction:\n{output.group(1).strip()}" if output else "")
                + (f"\nReasons view: `{view}`, one row per reason in top_reasons, with the columns {IMPLEMENT_REASONS_VIEW_COLUMNS}; "
                   f"read it to filter, count or chart by reason." if view else "")
                + f"\nNo raw JSON: {IMPLEMENT_NO_RAW_JSON_RULE}"
                + f"\nInput tables: {', '.join(f'`{t}`' for t in context['allowed'][1:] if t != view)}\n")

    def _implement_keep_edits_setting(self, uc_id: str, result_table: str, spec: dict, warnings: list, exists=None) -> None:
        replace = implement_replace_edits(spec)
        try:
            if exists is None:
                exists = inspire_table_schema(self.spark, result_table) is not None
            if exists:
                self.spark.sql(implement_replace_edits_sql(result_table, replace))
        except Exception as error:
            message = (f"runs may not {'replace' if replace else 'keep'} the records people changed in the App, as this use case "
                       f"asks: Inspire could not record that choice on {result_table}: {get_clean_error_message(error)[:300]}")
            self.logger.warning(f"[IMPLEMENT-HISTORY] {uc_id}: {message}")
            warnings.append(f"Result table: {message}")

    def _implement_result_history(self, uc_id: str, result_table: str, spec: dict, warnings: list) -> None:
        try:
            done = inspire_ensure_history(self.spark, result_table, out=lambda m: self.logger.info(f"[IMPLEMENT-HISTORY] {uc_id}: {m}"))
        except Exception as error:
            raise RuntimeError(f"the business output table {result_table} could not be given its change history, so no output could "
                               f"tell its current rows: {get_clean_error_message(error)[:800]}")
        if done is None:
            return
        try:
            inspire_ensure_actions(self.spark, result_table, out=lambda m: self.logger.info(f"[IMPLEMENT-HISTORY] {uc_id}: {m}"))
        except Exception as error:
            raise RuntimeError(f"the business output table {result_table} could not be given the columns {', '.join(INSPIRE_ACTION_NAMES)}, "
                               f"where the App records what people do about each record: {get_clean_error_message(error)[:800]}")
        self._implement_keep_edits_setting(uc_id, result_table, spec, warnings, exists=True)

    def _implement_move_result_table(self, uc_id: str, key: tuple, old: str, new: str, warnings: list) -> None:
        step = self._implement_emit(uc_id, "Result table", f"the results move from {old} to {new}", "started", {"spec_version": key[2]})
        status = "ended_success"
        try:
            if inspire_table_schema(self.spark, old) is None:
                message = f"{old} holds no results yet, so there is no change history to copy to {new}"
            elif inspire_table_schema(self.spark, new) is not None:
                message = (f"{new} already exists, so Inspire left it as it is and copied no change history from {old} into it; "
                           f"{old} is kept as it was")
                status = "ended_warning"
            else:
                self.spark.sql(implement_result_copy_sql(old, new))
                message = f"copied the full change history of {old} to {new}; {old} is kept as it was"
                old_summary, new_summary = inspire_ai_summary_table(old), inspire_ai_summary_table(new)
                try:
                    if inspire_table_schema(self.spark, old_summary) is not None and inspire_table_schema(self.spark, new_summary) is None:
                        self.spark.sql(implement_result_copy_sql(old_summary, new_summary))
                        message += f"; copied the AI summaries of its runs to {new_summary}"
                except Exception as error:
                    message += (f"; the AI summaries of its earlier runs stay in {old_summary}, and {new_summary} starts with the "
                                f"next run: {get_clean_error_message(error)[:300]}")
                    status = "ended_warning"
        except Exception as error:
            message = (f"the change history of {old} could not be copied to {new}, so {new} starts with the next run; {old} is kept "
                       f"as it was: {get_clean_error_message(error)[:300]}")
            status = "ended_warning"
        self.logger.warning(f"[IMPLEMENT-RESULT-MOVE] {uc_id}: {message}")
        if status == "ended_warning":
            warnings.append(f"Result table: {message}")
        self._implement_emit(uc_id, "Result table", message, status, {"spec_version": key[2], "from": old, "to": new}, step_id=step)

    def _implement_reasons_view(self, result_table: str, warnings: list) -> str:
        try:
            self.spark.sql(implement_reasons_view_sql(result_table))
            view = {name.lower() for name, _kind in inspire_table_schema(self.spark, implement_reasons_view(result_table)) or []}
            if not set(IMPLEMENT_REASONS_HISTORY_COLUMNS) <= view:
                self.spark.sql(implement_reasons_view_alter_sql(result_table))
            return implement_reasons_view(result_table)
        except Exception as error:
            warnings.append(f"The one-row-per-reason view {implement_reasons_view(result_table)} was not created, so the outputs show "
                            f"reasons only as joined text: {get_clean_error_message(error)[:300]}")
            return ""

    def _implement_run_query_check(self, sql: str, require_rows: bool = False, limit: int = 20) -> list:
        rows = self.spark.sql(f"SELECT * FROM ({sql.strip().rstrip(';')}) AS _inspire_check LIMIT {int(limit)}").collect()
        if require_rows and not rows:
            raise RuntimeError("the query returned no rows")
        return [row.asDict() if hasattr(row, "asDict") else dict(row) for row in rows]

    def _implement_latest_run_rows(self, result_table: str, limit: int = 1000) -> list:
        rows = self.spark.sql(f"SELECT entity_id, entity_label, priority_tier, rank FROM {inspire_table_name(result_table)} WHERE "
                              f"{implement_latest_run_filter(result_table)} ORDER BY rank LIMIT {int(limit)}").collect()
        return [row.asDict() if hasattr(row, "asDict") else dict(row) for row in rows]

    def _implement_tier_counts(self, result_table: str) -> dict:
        rows = self.spark.sql(f"SELECT priority_tier, COUNT(*) AS tier_rows FROM {inspire_table_name(result_table)} "
                              f"WHERE {implement_latest_run_filter(result_table)} GROUP BY priority_tier").collect()
        counts = {}
        for row in rows:
            row = row.asDict() if hasattr(row, "asDict") else dict(row)
            tier = "no tier" if row.get("priority_tier") is None else str(row["priority_tier"])
            counts[tier] = counts.get(tier, 0) + int(row.get("tier_rows") or 0)
        return counts

    def _implement_latest_run_profile(self, result_table: str) -> dict:
        rows = self.spark.sql(f"SELECT data_quality_flag, COUNT(*) AS flag_rows, MIN(value) AS low, MAX(value) AS high, "
                              f"MAX(value_type) AS value_type FROM {inspire_table_name(result_table)} "
                              f"WHERE {implement_latest_run_filter(result_table)} GROUP BY data_quality_flag").collect()
        groups = [row.asDict() if hasattr(row, "asDict") else dict(row) for row in rows]
        if not groups:
            return None
        lows = [g["low"] for g in groups if g.get("low") is not None]
        highs = [g["high"] for g in groups if g.get("high") is not None]
        flags = sorted(((str(g["data_quality_flag"]), int(g.get("flag_rows") or 0)) for g in groups
                        if g.get("data_quality_flag") is not None and str(g["data_quality_flag"]).strip().lower() != "ok"),
                       key=lambda item: (-item[1], item[0]))
        return {"value_type": next((str(g["value_type"]) for g in groups if g.get("value_type")), ""),
                "low": min(lows) if lows else None, "high": max(highs) if highs else None, "flags": flags[:3]}

    def _implement_latest_run_methods(self, result_table: str) -> dict:
        rows = self.spark.sql(f"SELECT method, COUNT(*) AS method_rows, COUNT_IF(COALESCE(TRIM(CAST(model_version AS STRING)), '') "
                              f"<> '') AS versioned_rows FROM {inspire_table_name(result_table)} "
                              f"WHERE {implement_latest_run_filter(result_table)} GROUP BY method").collect()
        groups = [row.asDict() if hasattr(row, "asDict") else dict(row) for row in rows]
        if not groups:
            return None
        named = {}
        for g in groups:
            name = str(g.get("method") or "").strip() or "no method"
            named[name] = named.get(name, 0) + int(g.get("method_rows") or 0)
        return {"methods": sorted(named.items(), key=lambda item: (-item[1], item[0]))[:3],
                "versioned_rows": sum(int(g.get("versioned_rows") or 0) for g in groups)}

    def _implement_run_reason(self, uc_id: str, result_table: str, run_id) -> str:
        if not run_id:
            return ""
        try:
            rows = self.spark.sql(f"SELECT nothing_to_act FROM {inspire_table_name(inspire_ai_summary_table(result_table))} "
                                  f"WHERE use_case_id = {inspire_sql_literal(uc_id)} AND run_id = {inspire_sql_literal(run_id)}").collect()
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-TIERS] {uc_id}: could not read the latest run's reason for having nothing to act on "
                                f"from {inspire_ai_summary_table(result_table)}: {get_clean_error_message(error)[:300]}")
            return ""
        values = [row.asDict() if hasattr(row, "asDict") else dict(row) for row in rows]
        return str(values[0].get("nothing_to_act") or "") if values else ""

    def _implement_tier_warning(self, uc_id: str, result_table: str, run_url: str, run_id=None) -> str:
        try:
            counts = self._implement_tier_counts(result_table)
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-TIERS] {uc_id}: could not count the priority tiers of the latest run of {result_table}: "
                                f"{get_clean_error_message(error)[:300]}")
            return ""
        if not implement_reached_no_action_tier(counts):
            return ""
        try:
            profile = self._implement_latest_run_profile(result_table)
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-TIERS] {uc_id}: could not read the value range and flags of the latest run of "
                                f"{result_table}: {get_clean_error_message(error)[:300]}")
            profile = None
        try:
            methods = self._implement_latest_run_methods(result_table)
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-TIERS] {uc_id}: could not read the methods of the latest run of {result_table}: "
                                f"{get_clean_error_message(error)[:300]}")
            methods = None
        warning = implement_tier_mix_warning(counts, run_url, profile, methods, self._implement_run_reason(uc_id, result_table, run_id))
        self.logger.warning(f"[IMPLEMENT-TIERS] {uc_id}: {warning}")
        return warning

    def _implement_dashboard(self, context: dict, runner) -> dict:
        spec, assets = context["spec"], context["assets"]
        warehouse = spec["config"]["warehouse_id"]
        payload, modules = self._implement_app_shell()
        shell_sql = modules["shell_sql"]
        target = self._implement_merge_target(context, "dashboard")

        def deploy(work, page):
            genie, note = self._implement_app_genie(context)
            self._implement_page_check(context, "Dashboard", work,
                                       self._implement_app_files(context, payload, modules, page, genie, note,
                                                                 IMPLEMENT_APP_CHECK_CHOICES),
                                       modules, page)
            shapes = {}
            for chart in page.get("charts") or []:
                try:
                    shapes[chart["id"]] = self._implement_run_query_check(shell_sql.chart_query(page, chart["id"], {})[0])
                except Exception as error:
                    raise RuntimeError(f"chart {chart['id']} failed: {get_clean_error_message(error)[:800]}")
            if genie:
                genie = dict(genie, url=f"{self._implement_host()}/genie/rooms/{genie['space_id']}")
            look = spec["outputs"].get("app_look") or {}
            theme = implement_dashboard_theme(payload.get("palettes"), look.get("palette"), page.get("hue"))
            dashboard, errors = check_dashboard_definition(
                implement_dashboard_from_page(page, shell_sql, context["result_table"], shapes, genie, theme), context["allowed"],
                shell_sql.RECORDS_DATASET)
            if errors:
                raise RuntimeError("; ".join(errors))
            shown, problems = implement_dashboard_shown_columns(dashboard), []
            for dataset in dashboard["datasets"]:
                name = dataset["name"]
                try:
                    rows = self._implement_run_query_check("".join(dataset["queryLines"]),
                                                           limit=1000 if name == shell_sql.RECORDS_DATASET else 20)
                except Exception as error:
                    raise RuntimeError(f"dataset {name} failed: {get_clean_error_message(error)[:800]}")
                columns = shown.get(name) or set()
                problems += implement_raw_json_problems(f"dataset {name}", [{k: v for k, v in row.items() if k in columns} for row in rows])
            if problems:
                raise RuntimeError("; ".join(problems))
            self._implement_checks_passed(context, "dashboard")
            body = {"display_name": (spec["outputs"].get("dashboard_name") or f"{context['uc_id']} {context['use_case'].get('Name')}")[:250],
                    "serialized_dashboard": json.dumps(dashboard), "warehouse_id": warehouse,
                    "parent_path": self._implement_notebook_paths(context["use_case"])[0]}
            existing, before = self._implement_own_asset(context, "dashboard"), None
            if target:
                saved = self._implement_merge_dashboard(context, target, dashboard)
                before = saved.pop("_before")
            elif existing:
                before = self._implement_api("GET", self._IMPLEMENT_ASSET_PATHS["dashboard"].format(existing))
                saved = self._implement_api("PATCH", f"/api/2.0/lakeview/dashboards/{existing}",
                                            {k: v for k, v in body.items() if k != "parent_path"})
            else:
                self.w_client.workspace.mkdirs(body["parent_path"])
                saved = self._implement_api("POST", "/api/2.0/lakeview/dashboards", body)
            dashboard_id = saved.get("dashboard_id") or existing
            restore = None if before is None else {"body": {k: before[k] for k in ("display_name", "serialized_dashboard", "warehouse_id")
                                                            if before.get(k)}}
            try:
                self._implement_api("POST", f"/api/2.0/lakeview/dashboards/{dashboard_id}/published",
                                    {"warehouse_id": saved.get("warehouse_id") or warehouse, "embed_credentials": False})
            except Exception as error:
                self._implement_undo_save(context, "dashboard", error,
                                          lambda: self._implement_restore_asset("dashboard", dashboard_id, restore))
                raise
            moved = self._implement_merge_moved(context, "dashboard", dashboard_id)
            return dict({"dashboard_id": dashboard_id, "dashboard_url": f"{self._implement_host()}/dashboardsv3/{dashboard_id}/published"},
                        **{k: v for k, v in saved.items() if k.startswith("_")}, **({"_warnings": moved} if moved else {}))

        return self._implement_page_spec(context, runner, "dashboard", deploy)

    _IMPLEMENT_ASSET_PATHS = {"dashboard": "/api/2.0/lakeview/dashboards/{}", "genie_space": "/api/2.0/genie/spaces/{}",
                              "app": "/api/2.0/apps/{}", "alert": "/api/2.0/alerts/{}"}

    def _implement_undo_save(self, context: dict, name: str, error: Exception, undo) -> None:
        label = IMPLEMENT_OUTPUT_LABELS[name]
        try:
            done = undo()
        except Exception as undo_error:
            raise ImplementBlocked(f"the {label} build failed after Inspire had saved changes ({get_clean_error_message(error)[:600]}), "
                                   f"and Inspire could not undo them: {get_clean_error_message(undo_error)[:600]}") from error
        if not done:
            return
        self.logger.warning(f"[IMPLEMENT-UNDO] {context['uc_id']}: the {label} build failed after Inspire had saved changes, so "
                            f"Inspire {done}: {get_clean_error_message(error)[:300]}")

    def _implement_restore_asset(self, name: str, asset_id: str, restore) -> str:
        path, label = self._IMPLEMENT_ASSET_PATHS[name].format(asset_id), IMPLEMENT_OUTPUT_LABELS[name]
        if restore is None:
            self._implement_api("DELETE", path)
            return f"moved the new {label} {asset_id} to the trash"
        self._implement_api("PATCH", path, restore["body"], **({"query": restore["query"]} if restore.get("query") else {}))
        return f"put the {label} {asset_id} back as it was"

    def _implement_merge_target(self, context: dict, name: str) -> str:
        target = implement_merge_target(context["spec"]["outputs"], name)
        recorded = (context.get("asset_status") or {}).get(name) or {}
        if not target or (target == str(context["assets"].get(name) or "") and not recorded.get("merged_into")):
            return ""
        label = IMPLEMENT_MERGE_LABELS[name]
        if not self._implement_select(f"`{IMPLEMENT_OUTPUT_ID_COLUMNS[name]}` = :asset", {"asset": target}):
            raise ImplementBlocked(f"the {label} {target} you chose to add this use case to was not built by Inspire, so Inspire does "
                                   f"not change it; pick another Inspire {label} or build a new one")
        try:
            found = self._implement_api("GET", self._IMPLEMENT_ASSET_PATHS[name].format(target))
        except Exception as error:
            raise ImplementBlocked(f"the {label} {target} you chose to add this use case to "
                                   + ("no longer exists" if implement_is_not_found(error) else
                                      f"could not be read ({get_clean_error_message(error)[:300]})")
                                   + f"; pick another Inspire {label} or build a new one")
        if str(found.get("lifecycle_state") or "") == "TRASHED":
            raise ImplementBlocked(f"the {label} {target} you chose to add this use case to is in the trash; restore it or pick "
                                   f"another Inspire {label}")
        return target

    def _implement_own_asset(self, context: dict, name: str) -> str:
        asset = str(context["assets"].get(name) or "")
        return "" if asset and ((context.get("asset_status") or {}).get(name) or {}).get("merged_into") == asset else asset

    def _implement_merge_moved(self, context: dict, name: str, target: str) -> list:
        before = str(context["assets"].get(name) or "")
        merging = bool(implement_merge_target(context["spec"]["outputs"], name)) or not self._implement_own_asset(context, name)
        if not before or before == target or not merging:
            return []
        label = IMPLEMENT_MERGE_LABELS[name]
        return [f"{label[:1].upper()}{label[1:]}: this use case is now part of the {label} {target}; the {label} {before} keeps what "
                "Inspire built there before and is no longer refreshed for it"]

    def _implement_merge_dashboard(self, context: dict, target: str, dashboard: dict, attempts: int = 2) -> dict:
        uc_id = context["uc_id"]
        prefix = f"s{self.session_id}_{implement_app_page_id(uc_id)}_"
        for attempt in range(1, attempts + 1):
            current = self._implement_api("GET", self._IMPLEMENT_ASSET_PATHS["dashboard"].format(target))
            try:
                board = json.loads(current.get("serialized_dashboard") or "{}")
            except ValueError as error:
                raise ImplementBlocked(f"the dashboard {target} could not be read: its definition is not valid JSON ({error})")
            merged, problems = implement_merge_dashboard(board, dashboard, prefix, uc_id)
            if problems:
                raise ImplementBlocked(f"this use case was not added to the dashboard {current.get('display_name') or target}: "
                                       + "; ".join(problems))
            body = dict({"serialized_dashboard": json.dumps(merged)}, **({"etag": current["etag"]} if current.get("etag") else {}))
            try:
                self._implement_api("PATCH", self._IMPLEMENT_ASSET_PATHS["dashboard"].format(target), body)
                break
            except Exception as error:
                if "etag" not in str(error).lower():
                    raise
                if attempt == attempts:
                    raise ImplementBlocked(f"the dashboard {current.get('display_name') or target} kept changing while Inspire added "
                                           "this use case's pages; submit again when nobody is editing it")
                self.logger.warning(f"[IMPLEMENT-DASHBOARD-MERGE] {uc_id}: dashboard {target} changed while Inspire merged; reading it again")
        self.logger.info(f"[IMPLEMENT-DASHBOARD-MERGE] {uc_id}: dashboard {target} now has {len(merged.get('pages') or [])} page(s), "
                         f"{sum(str(p.get('name') or '').startswith(prefix) for p in merged.get('pages') or [])} of them this use case's")
        return {"dashboard_id": target, "warehouse_id": str(current.get("warehouse_id") or ""), "_asset": {"merged_into": target},
                "_before": current}

    _GENIE_SPACE_FORMAT = (
        '{"title": "...", "description": "...", "serialized_space": {"version": 2, "config": {"sample_questions": [{"question": '
        '["..."]}]}, "data_sources": {"tables": [{"identifier": "catalog.schema.table", "description": ["..."], "column_configs": '
        '[{"column_name": "...", "description": ["..."], "synonyms": ["..."]}]}]}, "instructions": {"text_instructions": '
        '[{"content": ["..."]}], "example_question_sqls": [{"question": ["..."], "sql": ["SELECT ..."]}], "join_specs": [{"left": '
        '{"identifier": "...", "alias": "..."}, "right": {"identifier": "...", "alias": "..."}, "sql": ["`a`.`k` = `b`.`k`", '
        '"--rt=FROM_RELATIONSHIP_TYPE_MANY_TO_ONE--"]}], "sql_snippets": {"measures": [{"alias": "...", "display_name": "...", '
        '"sql": ["COUNT_IF(`table`.`column` = ...)"], "synonyms": ["..."], "instruction": ["..."]}], "filters": [{"display_name": '
        '"...", "sql": ["`table`.`column` = ..."], "synonyms": ["..."], "instruction": ["..."]}], "expressions": [{"alias": "...", '
        '"display_name": "...", "sql": ["DATE_TRUNC(\'MONTH\', `table`.`column`)"], "synonyms": ["..."], "instruction": ["..."]}]}}, '
        '"benchmarks": {"questions": [{"question": ["..."], "answer": [{"format": "SQL", "content": ["SELECT ..."]}]}]}}}')

    def _implement_genie_rules(self, context: dict) -> str:
        aliases = implement_genie_aliases(context["allowed"])
        tier = IMPLEMENT_ACTION_TIERS[0]
        return (f"- Column configs: give every column of the business output table a description in business terms, and each input "
                f"table column the questions use; list the other words the business uses for a column as its synonyms. Configure "
                f"only real columns: Inspire reads each table's columns and rejects a config for a column the table does not have, "
                f"and it sets example values and entity matching itself.\n"
                f"- SQL expressions in instructions.sql_snippets: measures (aggregates the decision owner asks for, such as the "
                f"number of {tier} rows), filters (conditions, such as the {tier} tier) and expressions (row-level facts, such as "
                f"the month of as_of_date). Each reads one table and names each column as `table`.`column` with these names: "
                + ", ".join(f"`{alias}` for {_implement_plain_table(t)}" for t, alias in aliases.items())
                + ". A measure or an expression has an alias of letters, digits and underscores; a filter has none.\n"
                f"- Benchmarks: at least {IMPLEMENT_GENIE_MIN_BENCHMARKS} more questions a decision owner asks, worded differently "
                f"from the sample questions, each with one answer of format SQL holding its gold query.\n"
                f"- The latest run of `{context['result_table']}` is its rows where "
                f"{implement_latest_run_filter(context['result_table'])}; questions about the current results read those rows.\n"
                f"- Where these rules conflict with what the user chose (the questions or the title), the user's choice wins.\n")

    def _implement_space_columns(self, tables) -> dict:
        columns = {}
        for table in tables or []:
            try:
                rows = self.spark.sql(f"DESCRIBE TABLE {inspire_table_name(table)}").collect()
            except Exception as error:
                raise RuntimeError(f"the columns of {table} could not be read: {get_clean_error_message(error)[:500]}")
            columns[_implement_plain_table(table).lower()] = _parse_describe_rows(rows)
        return columns

    def _implement_genie_space_checks(self, space: dict, allowed, result_table: str) -> list:
        tables = [_implement_plain_table(t.get("identifier")) for t in (space.get("data_sources") or {}).get("tables") or []]
        aliases = implement_genie_aliases(list(allowed or ()) + tables)
        problems = []
        for check in implement_genie_space_checks(space, {t.lower(): aliases[t.lower()] for t in tables}, result_table):
            try:
                rows = self._implement_run_query_check(check["sql"], limit=check.get("limit", 1 if check.get("expect") else 20))
            except Exception as error:
                problems.append(f"{check['label']} failed: {get_clean_error_message(error)[:800]}")
                continue
            if check.get("expect") == "none" and rows:
                problems.append(check["problem"])
            if check.get("raw_json"):
                problems += implement_raw_json_problems(check["label"], rows)
        return problems

    def _implement_genie_space(self, context: dict, runner) -> dict:
        spec, assets = context["spec"], context["assets"]
        warehouse = spec["config"]["warehouse_id"]
        prompt = (f"Write the file genie_space.json in your working folder: a Databricks Genie space for the use case below, so its "
                  f"decision owner can ask questions about the results in plain language.\n{self._implement_use_case_brief(context)}\n"
                  f"Use this JSON shape (the serialized_space format of the Genie Spaces API, version 2; leave every id out):\n"
                  f"{self._GENIE_SPACE_FORMAT}\nRules:\n- Tables: the business output table first, "
                  + (f"then the reasons view `{context['reasons_view']}`, " if context.get("reasons_view") else "")
                  + f"then the input tables the "
                  f"questions need; describe each table and its key columns in business terms.\n- Exactly 5 sample questions a "
                  f"decision owner asks each cycle, in the business's words and about what the business output table's rows are"
                  + (", including these, word for word: " + "; ".join(repr(q) for q in spec["outputs"]["genie_questions"])
                     if spec["outputs"].get("genie_questions") else "")
                  + f", and an example question SQL that answers each one.\n"
                  + ("- A question of the user's that asks about something the business output table does not hold, such as people "
                     "when its rows are items, still goes in word for word: answer it with what the table holds, and list it in a "
                     'top-level "question_notes" array of genie_space.json as {"question": "<the question>", "note": "<one sentence '
                     'on what its answer shows instead>"}. Leave every other question out of question_notes.\n'
                     if spec["outputs"].get("genie_questions") else "")
                  + (f"- Title: {spec['outputs']['genie_space_name']}\n" if spec["outputs"].get("genie_space_name") else "")
                  + f"- Join specs for the verified join path. Each join's sql is exactly two items: the join condition, then the "
                  f"relationship tag that matches the key's cardinality, one of {', '.join(IMPLEMENT_JOIN_TAGS)}; check the "
                  f"cardinality with a COUNT(DISTINCT key) query before you choose it.\n- Test every example SQL, benchmark SQL and SQL "
                  f"expression on SQL warehouse {warehouse} with `databricks api post /api/2.0/sql/statements`; each must succeed: "
                  f"Inspire compiles each one and each join spec, and checks a join tag that calls the business output table's key "
                  f"unique against its rows, before it saves the space.\n"
                  + self._implement_genie_rules(context)
                  + "- Never write data.\n"
                  + implement_vibes_block(spec["outputs"], "genie_space", f'a top-level "{IMPLEMENT_VIBES_NOT_DONE}" array of genie_space.json')
                  + "Then reply with only the JSON object your output schema describes.")
        target = self._implement_merge_target(context, "genie_space")

        def deploy(work):
            path = os.path.join(work, "genie_space.json")
            if not os.path.exists(path):
                raise RuntimeError("genie_space.json was not written")
            text = open(path, encoding="utf-8").read()
            try:
                data = json.loads(text)
            except ValueError:
                data = None
            carried = isinstance(data, dict) and IMPLEMENT_VIBES_NOT_DONE in data
            not_done, errors = implement_vibes_not_done(data, spec["outputs"], "genie_space")
            text = json.dumps(data) if carried else text
            title, description, space, normalize_errors = normalize_genie_space(text, context["allowed"])
            errors += normalize_errors
            questions = ["".join(q["question"]) for q in (space.get("config") or {}).get("sample_questions") or []]
            notes, question_errors = implement_genie_question_problems(text, questions, spec["outputs"].get("genie_questions"))
            errors += question_errors
            if errors:
                raise RuntimeError("; ".join(errors))
            columns = self._implement_space_columns([t["identifier"] for t in space["data_sources"]["tables"]])
            problems = implement_genie_space_grounding(space, columns, context["result_table"], context["allowed"],
                                                       self._TEXT_NAME_TOKENS, implement_latest_run_filter(context["result_table"]))
            if problems:
                raise RuntimeError("; ".join(problems))
            problems = self._implement_genie_space_checks(space, context["allowed"], context["result_table"])
            if problems:
                raise RuntimeError("; ".join(problems))
            self._implement_checks_passed(context, "genie_space")
            artifact = dict({"title": title, "description": description, "serialized_space": json.loads(json.dumps(space))},
                            **({"question_notes": notes} if notes else {}))
            body = {"title": title[:250], "description": description, "warehouse_id": warehouse,
                    "serialized_space": json.dumps(space)}
            existing = self._implement_own_asset(context, "genie_space")
            merged = {}
            if target:
                before = self._implement_genie_read(target)
                merged = self._implement_merge_genie(context, target, space,
                                                     (context["asset_status"].get("genie_space") or {}).get("merged") or {}, before)
                saved = {"space_id": target}
            else:
                before = self._implement_own_genie_before(context, existing)
                if before:
                    saved = self._implement_api("PATCH", f"/api/2.0/genie/spaces/{existing}", body)
                else:
                    existing = ""
                    parent = self._implement_notebook_paths(context["use_case"])[0]
                    self.w_client.workspace.mkdirs(parent)
                    saved = self._implement_api("POST", "/api/2.0/genie/spaces", dict(body, parent_path=parent))
            space_id = saved.get("space_id") or existing
            restore = None if before is None else {"body": dict({k: before[k] for k in ("title", "description", "warehouse_id")
                                                                 if before.get(k)}, serialized_space=json.dumps(before["space"]))}
            try:
                answer = self._implement_genie_smoke(space_id, questions[0])
            except Exception as error:
                self._implement_undo_save(context, "genie_space", error,
                                          lambda: self._implement_restore_asset("genie_space", space_id, restore))
                raise
            sql = next((a["query"].get("query") for a in answer.get("attachments") or [] if isinstance((a or {}).get("query"), dict)), "")
            asset = {"sample_questions": questions, "check_question": questions[0], "check_sql": str(sql or "")[:2000]}
            if notes:
                asset["question_notes"] = notes
            if target:
                asset.update(merged_into=target, merged=merged)
            context.setdefault("artifacts", {})["genie_space"] = artifact
            return {"genie_space_id": space_id, "genie_space_url": f"{self._implement_host()}/genie/rooms/{space_id}",
                    "_asset": asset, "_warnings": [implement_question_note_warning(n) for n in notes]
                    + self._implement_merge_moved(context, "genie_space", space_id) + not_done}

        if "genie_space" in (context.get("replay") or {}):
            return self._implement_replay(context, "genie_space", deploy)
        return self._implement_genie_loop(context, runner, "Genie Agent", prompt, deploy)

    def _implement_genie_read(self, space_id: str) -> dict:
        found = self._implement_api("GET", self._IMPLEMENT_ASSET_PATHS["genie_space"].format(space_id),
                                    query={"include_serialized_space": "true"})
        try:
            space = json.loads(found.get("serialized_space") or "{}")
        except ValueError as error:
            raise ImplementBlocked(f"the Genie Agent {space_id} could not be read: its definition is not valid JSON ({error})")
        return {"title": str(found.get("title") or space_id), "description": str(found.get("description") or ""),
                "warehouse_id": str(found.get("warehouse_id") or ""), "space": space if isinstance(space, dict) else {}}

    def _implement_own_genie_before(self, context: dict, space_id: str):
        if not space_id:
            return None
        try:
            return self._implement_genie_read(space_id)
        except Exception as error:
            if not implement_is_not_found(error):
                raise
        self.logger.warning(f"[IMPLEMENT-GENIE] {context['uc_id']}: the Genie Agent {space_id} is gone; creating a new one")
        return None

    def _implement_merge_genie(self, context: dict, space_id: str, ours: dict, previous: dict, current: dict = None) -> dict:
        uc_id = context["uc_id"]
        current = current or self._implement_genie_read(space_id)
        label = " ".join(f"For use case {uc_id} {context['use_case'].get('Name') or ''}, whose results are in "
                         f"`{context['result_table']}`:".split())
        merged, record, problems = implement_merge_genie_space(current["space"], ours,
                                                              previous if previous.get("space_id") == space_id else {}, label)
        if problems:
            raise ImplementBlocked(f"this use case was not added to the Genie Agent {current['title']}: {'; '.join(problems)}. Pick "
                                   "another Genie Agent or build a new one for this use case")
        self._implement_api("PATCH", self._IMPLEMENT_ASSET_PATHS["genie_space"].format(space_id), {"serialized_space": json.dumps(merged)})
        self.logger.info(f"[IMPLEMENT-GENIE-MERGE] {uc_id}: Genie Agent {space_id} gained {len(record['tables'])} table(s) and "
                         f"{len(record.get('sample_questions') or [])} sample question(s); it holds "
                         f"{len((merged.get('data_sources') or {}).get('tables') or [])} table(s)")
        return dict(record, space_id=space_id)

    def _implement_genie_smoke(self, space_id: str, question: str, timeout_s: int = 300, poll_s: int = 5):
        import time
        started = self._implement_api("POST", f"/api/2.0/genie/spaces/{space_id}/start-conversation", {"content": question})
        cid = started.get("conversation_id") or (started.get("conversation") or {}).get("id")
        mid = started.get("message_id") or (started.get("message") or {}).get("id")
        deadline = time.time() + timeout_s
        while True:
            message = self._implement_api("GET", f"/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}")
            status = str(message.get("status") or "")
            if status == "COMPLETED":
                if not any((a or {}).get("query") for a in message.get("attachments") or []):
                    raise RuntimeError(f"the Genie Agent answered {question!r} without a query; its tables or instructions do not cover it")
                return message
            if status in ("FAILED", "CANCELLED", "QUERY_RESULT_EXPIRED"):
                raise RuntimeError(f"the Genie Agent could not answer {question!r}: {(message.get('error') or {}).get('error') or status}")
            if time.time() > deadline:
                raise RuntimeError(f"the Genie Agent did not answer {question!r} within {timeout_s}s (last status {status})")
            time.sleep(poll_s)

    def _implement_alert(self, context: dict, runner) -> dict:
        spec, uc_id = context["spec"], context["uc_id"]
        outputs = spec["outputs"]
        if not context.get("job_id"):
            raise ImplementBlocked("the use case has no job yet; the alert runs as a task after the notebook task of that job")
        text = outputs.get("alert_condition_text") or ""
        vibes = implement_output_vibes(outputs, "alert")
        before = (context.get("asset_status") or {}).get("alert") or {}
        reuse = (str(before.get("condition_sql") or "") if (text or vibes) and before.get("condition_text", "") == text
                 and before.get("vibes", "") == vibes else "")

        def build(condition_sql: str, condition_text: str = text, condition_vibes: str = vibes, not_done=()) -> dict:
            errors = check_alert_condition_sql(condition_sql, context["allowed"]) if condition_sql else []
            if errors:
                raise RuntimeError("; ".join(errors))
            if condition_sql:
                rows = self._implement_run_query_check(condition_sql, limit=2)
                triggered = [v for k, v in (rows[0] if rows else {}).items() if k.lower() == "triggered"]
                if len(rows) != 1 or not triggered or not isinstance(triggered[0], bool):
                    raise RuntimeError(f"alert_condition.sql must return exactly one row with a BOOLEAN column triggered; it returned "
                                       f"{len(rows)} row(s)" + (f" with triggered = {triggered[0]!r}" if triggered else ""))
            query = implement_alert_query(context["result_table"], outputs, condition_sql)
            try:
                checked = self._implement_run_query_check(query["query"], require_rows=True, limit=1)
            except Exception as error:
                raise (RuntimeError if condition_sql else ImplementBlocked)(
                    f"the alert query failed on the business output table: {get_clean_error_message(error)[:800]}")
            baseline = implement_alert_baseline_now(outputs, checked)
            if baseline:
                self.logger.info(f"[IMPLEMENT-ALERT] {uc_id}: threshold now {baseline['threshold_basis']} at "
                                 f"{baseline['alert_threshold']} from {baseline['history_runs']} earlier run(s)")
            warehouse = spec["config"]["warehouse_id"]
            self._implement_checks_passed(context, "alert")
            before, alert_id = None, ""
            try:
                try:
                    before = self._implement_alert_before(context)
                    alert_id = self._implement_alert_upsert(context, query)
                    subscribers = self._implement_alert_task_upsert(int(context["job_id"]), alert_id, outputs, warehouse)
                except ImplementBlocked:
                    raise
                except Exception as error:
                    raise ImplementBlocked(f"the alert could not be saved: {get_clean_error_message(error)[:800]}")
                state = self._implement_alert_check_run(uc_id, alert_id, warehouse)
            except Exception as error:
                if alert_id:
                    self._implement_undo_save(context, "alert", error, lambda: self._implement_alert_undo(context, alert_id, before))
                raise
            self.logger.info(f"[IMPLEMENT-ALERT] {uc_id}: alert {alert_id} on job {context['job_id']} task "
                             f"{IMPLEMENT_ALERT_TASK_KEY}, {subscribers} subscriber(s), one-time check {state}")
            context.setdefault("artifacts", {})["alert"] = {"condition_text": condition_text, "condition_sql": condition_sql}
            return dict({"alert_id": alert_id, "alert_url": f"{self._implement_host()}/sql/alerts-v2/{alert_id}",
                         "_asset": dict({"condition_text": condition_text, "condition_sql": condition_sql, "check_state": state,
                                         "trigger": implement_alert_trigger_text(outputs), "subscribers": subscribers},
                                        **({"baseline": baseline} if baseline else {}),
                                        **({"vibes": condition_vibes} if condition_vibes else {}))},
                        **({"_warnings": list(not_done)} if not_done else {}))

        def deploy(work, condition_text=text, condition_vibes=vibes):
            path = os.path.join(work, "alert_condition.sql")
            if not os.path.exists(path):
                raise RuntimeError("alert_condition.sql was not written")
            listed = os.path.join(work, IMPLEMENT_ALERT_VIBES_FILE)
            value = None
            if condition_vibes and os.path.exists(listed):
                try:
                    value = json.loads(open(listed, encoding="utf-8").read())
                except ValueError as error:
                    raise RuntimeError(f"{IMPLEMENT_ALERT_VIBES_FILE} is not valid JSON: {error}")
            not_done, errors = implement_vibes_not_done({IMPLEMENT_VIBES_NOT_DONE: value}, {"alert_vibes": condition_vibes}, "alert")
            if errors:
                raise RuntimeError("; ".join(errors))
            return build(open(path, encoding="utf-8").read().strip(), condition_text, condition_vibes, not_done)

        if "alert" in (context.get("replay") or {}):
            saved = context["replay"]["alert"][1]
            return self._implement_replay(context, "alert", lambda work: deploy(
                work, str((saved.get("artifact") or {}).get("condition_text") or ""), str(saved.get("vibes") or "")))
        if (text or vibes) and not reuse:
            runner = runner or self._implement_runner(uc_id)
            fires = (f"true when this holds: {text!r}" if text else
                     f"true when the customer's asks below say the alert should fire; the alert already fires when "
                     f"{implement_alert_trigger_text(outputs)}, and this query adds to that")
            prompt = (f"Write the file alert_condition.sql in your working folder: one Databricks SQL SELECT for the use case below that "
                      f"returns exactly one row with a BOOLEAN column named triggered, {fires}\n"
                      f"{self._implement_use_case_brief(context)}\nRules:\n- Read only the business output table and the input tables "
                      f"above; never write data and use no parameters.\n- The latest run's rows are the current rows, WHERE "
                      f"{implement_latest_run_filter(context['result_table'])}, never the newest as_of_date.\n- Test it on SQL "
                      f"warehouse {spec['config']['warehouse_id']} with `databricks api post /api/2.0/sql/statements`; it must succeed "
                      f"and return one row.\n"
                      + implement_vibes_block(outputs, "alert", f"the file {IMPLEMENT_ALERT_VIBES_FILE} in your working folder, a JSON array")
                      + "Then reply with only the JSON object your output schema describes.")
            return self._implement_genie_loop(context, runner, "Alert", prompt, deploy)
        key = context["key"]
        self._implement_update(key, status_message="Alert: creating it")
        step = self._implement_emit(uc_id, "Alert", f"alert when {implement_alert_trigger_text(outputs)}", "started",
                                    {"spec_version": key[2]})
        try:
            fields = build(reuse)
        except Exception as error:
            self._implement_emit(uc_id, "Alert", get_clean_error_message(error)[:1500], "ended_error", {"spec_version": key[2]},
                                 step_id=step)
            raise
        self._implement_emit(uc_id, "Alert", "Alert ready", "ended_success",
                             dict({k: v for k, v in fields.items() if not k.startswith("_")}, spec_version=key[2]), step_id=step)
        return fields

    def _implement_job_schedule(self, uc_id: str, job_id):
        try:
            return getattr(getattr(self.w_client.jobs.get(job_id=int(job_id)), "settings", None), "schedule", None)
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-ALERT] {uc_id}: could not read the schedule of job {job_id}, so the alert follows the "
                                f"wizard's schedule only: {get_clean_error_message(error)[:300]}")
            return None

    def _implement_alert_upsert(self, context: dict, query: dict) -> str:
        spec, uc_id = context["spec"], context["uc_id"]
        outputs, schedule = spec["outputs"], spec["schedule"]
        cadence = implement_job_cadence(schedule, None if schedule.get("cron") else self._implement_job_schedule(uc_id, context["job_id"]))
        contract = {k: text for k, _label, text in self._genie_decision_contract(context["use_case"])}
        urls = context.get("urls", {})
        summary, description = implement_alert_texts(uc_id, context["use_case"], contract, outputs, urls.get("dashboard_url") or "",
                                                     urls.get("genie_space_url") or "", cadence)
        body = {"display_name": (outputs.get("alert_name") or f"{uc_id} {context['use_case'].get('Name') or ''}".strip())[:250],
                "query_text": query["query"], "warehouse_id": spec["config"]["warehouse_id"],
                "custom_summary": summary, "custom_description": description,
                "evaluation": {"source": {"name": query["column"], "display": query["column"]},
                               "comparison_operator": query["operator"],
                               "threshold": {"value": {"double_value": query["threshold"]}}, "empty_result_state": "OK",
                               "notification": {"notify_on_ok": True, "retrigger_seconds": 60, "subscriptions": []}},
                "schedule": implement_alert_schedule(cadence)}
        existing = context["assets"].get("alert")
        if existing:
            try:
                saved = self._implement_api("PATCH", f"/api/2.0/alerts/{existing}", body,
                                            query={"update_mask": ",".join(IMPLEMENT_ALERT_FIELDS)})
                if str(saved.get("lifecycle_state") or "ACTIVE") != "TRASHED":
                    return str(saved.get("id") or existing)
            except Exception as error:
                if not implement_is_not_found(error):
                    raise
            self.logger.warning(f"[IMPLEMENT-ALERT] {uc_id}: alert {existing} is gone or trashed; creating a new one")
        parent = self._implement_notebook_paths(context["use_case"])[0]
        self.w_client.workspace.mkdirs(parent)
        return str(self._implement_api("POST", "/api/2.0/alerts", dict(body, parent_path=parent))["id"])

    def _implement_alert_task_upsert(self, job_id: int, alert_id: str, outputs: dict, warehouse_id: str) -> int:
        subscribers = implement_alert_subscribers(outputs)
        self._implement_job_after_task(job_id, IMPLEMENT_ALERT_TASK_KEY, {"alert_task": {
            "alert_id": alert_id, "warehouse_id": warehouse_id, "subscribers": subscribers}})
        return len(subscribers)

    def _implement_job_after_task(self, job_id: int, task_key: str, body: dict) -> None:
        launcher_cls = getattr(self, "_job_launcher_cls", None) or JobLauncher
        self._implement_api("POST", "/api/2.2/jobs/update", {"job_id": int(job_id), "new_settings": {"tasks": [dict({
            "task_key": task_key, "depends_on": [{"task_key": launcher_cls.TASK_KEY}], "run_if": "ALL_SUCCESS",
            "timeout_seconds": 1800}, **body)]}})

    def _implement_alert_before(self, context: dict) -> dict:
        existing, alert = str(context["assets"].get("alert") or ""), None
        if existing:
            try:
                found = self._implement_api("GET", self._IMPLEMENT_ASSET_PATHS["alert"].format(existing))
                alert = None if str(found.get("lifecycle_state") or "ACTIVE") == "TRASHED" else found
            except Exception as error:
                if not implement_is_not_found(error):
                    raise
        job = self._implement_api("GET", "/api/2.2/jobs/get", query={"job_id": int(context["job_id"])})
        task = next((t for t in (job.get("settings") or {}).get("tasks") or [] if t.get("task_key") == IMPLEMENT_ALERT_TASK_KEY), None)
        return {"alert_id": existing if alert is not None else "", "alert": alert, "task": task}

    def _implement_alert_undo(self, context: dict, alert_id: str, before: dict) -> str:
        job_id = int(context["job_id"])
        if before["task"]:
            self._implement_api("POST", "/api/2.2/jobs/update", {"job_id": job_id, "new_settings": {"tasks": [before["task"]]}})
        else:
            self._implement_remove_job_task(job_id, IMPLEMENT_ALERT_TASK_KEY)
        kept = before["alert"] if before["alert_id"] == alert_id else None
        fields = [k for k in IMPLEMENT_ALERT_FIELDS if k in (kept or {})]
        restore = None if kept is None else {"body": {k: kept[k] for k in fields}, "query": {"update_mask": ",".join(fields)}}
        return (self._implement_restore_asset("alert", alert_id, restore) + ", and "
                + ("set the job's alert task back" if before["task"] else "took the alert task off the job"))

    def _implement_remove_job_task(self, job_id: int, task_key: str):
        try:
            job = self._implement_api("GET", "/api/2.2/jobs/get", query={"job_id": int(job_id)})
        except Exception as error:
            if implement_is_not_found(error):
                return
            raise
        if any(t.get("task_key") == task_key for t in (job.get("settings") or {}).get("tasks") or []):
            self._implement_api("POST", "/api/2.2/jobs/update", {"job_id": int(job_id), "fields_to_remove": [f"tasks/{task_key}"]})

    def _implement_alert_check_run(self, uc_id: str, alert_id: str, warehouse_id: str, timeout_s: int = 900) -> str:
        launcher_cls = getattr(self, "_job_launcher_cls", None) or JobLauncher
        check_key = f"{IMPLEMENT_ALERT_TASK_KEY}_check"
        try:
            run_id = self._implement_api("POST", "/api/2.2/jobs/runs/submit", {
                "run_name": f"Inspire alert check {uc_id}"[:100],
                "tasks": [{"task_key": check_key, "alert_task": {"alert_id": alert_id, "warehouse_id": warehouse_id, "subscribers": []}}]
            })["run_id"]
        except Exception as error:
            raise ImplementBlocked(f"the one-time alert check could not start: {get_clean_error_message(error)[:800]}")
        waited = launcher_cls("", {}).wait_for_run(run_id, timeout_s=timeout_s, poll_s=10,
                                                    out=lambda m: self.logger.info(f"[IMPLEMENT {uc_id}] alert check {m}"))
        if waited.get("status") == "RUNNING":
            raise ImplementBlocked(f"the one-time alert check run {run_id} did not finish within {timeout_s}s")
        state = "UNKNOWN"
        try:
            run = self._implement_api("GET", "/api/2.2/jobs/runs/get", query={"run_id": int(run_id)})
            task_run = next((t.get("run_id") for t in run.get("tasks") or [] if t.get("task_key") == check_key), run_id)
            output = self._implement_api("GET", "/api/2.2/jobs/runs/get-output", query={"run_id": int(task_run)})
            state = str((output.get("alert_output") or {}).get("alert_state") or "UNKNOWN")
        except Exception as error:
            self.logger.warning(f"[IMPLEMENT-ALERT] {uc_id}: the alert check output of run {run_id} is unreadable "
                                f"({get_clean_error_message(error)})")
        if waited.get("status") != "SUCCESS" or state == "ERROR":
            raise ImplementBlocked(f"the one-time alert check run {run_id} ended {waited.get('status')} with alert state {state}: "
                                   f"{str(waited.get('error') or '')[:800]}")
        return state

    def _implement_app_shell(self) -> tuple:
        if getattr(self, "_implement_app_shell_cache", None) is None:
            payload = implement_app_shell_payload(_INSPIRE_APP_SHELL)
            folder = os.path.join(self.local_log_output_dir, "implement", "_app_shell")
            self._implement_app_shell_cache = (payload, implement_app_shell_modules(folder, payload["files"]))
        return self._implement_app_shell_cache

    def _implement_input_has_dates(self, context: dict) -> bool:
        inputs = [t for t in context["allowed"] if t not in (context["result_table"], context.get("reasons_view"))]
        return any(isinstance(column, dict) and _metadata_type_family(column.get("type")) == "temporal"
                   for entry in self._genie_table_evidence(inputs).values() for column in (entry.get("columns") or {}).values())

    def _implement_app_genie(self, context: dict) -> tuple:
        outputs, status = context["spec"]["outputs"], context["asset_status"].get("genie_space") or {}
        space_id = str(context["assets"].get("genie_space") or "").strip()
        if outputs.get("genie_space") and space_id:
            title = outputs.get("genie_space_name") or f"{context['uc_id']} {context['use_case'].get('Name') or ''}"
            return {"space_id": space_id, "title": " ".join(str(title).split())[:80]}, ""
        reason = " ".join(str(status.get("message") or status.get("error") or "it was not built").split())[:300]
        return None, f"Ask Genie is off because this use case's Genie Agent is not ready: {reason}"

    def _implement_app_prompt(self, context: dict, shell_sql, example: dict, event_date: bool, name: str) -> str:
        spec, result_table = context["spec"], context["result_table"]
        caps = shell_sql.CAPS
        roles = (spec.get("data") or {}).get("column_roles") or {}
        dates = (f"a second date_range on the event date `{roles['event_date']}` of the datasets that carry it" if roles.get("event_date")
                 else "a second date_range on the event date of the input tables (the date the business event happened)"
                 if event_date else "no second date range: the input tables carry no date")
        outputs = spec["outputs"]
        drawn = " and ".join(name for name, chosen in (("the use case's App", outputs.get("app")),
                                                       ("the use case's AI/BI dashboard", outputs.get("dashboard"))) if chosen) \
            or "the use case's App"
        return (f"Write the file {IMPLEMENT_APP_PAGE_FILE} in your working folder: one page spec for the use case below, in the format "
                f"of Inspire's prebuilt Databricks App. Inspire draws it as {drawn} and runs its "
                f"queries on the SQL warehouse{'; the App answers Ask Genie' if outputs.get('app') else ''}; you choose the data, KPIs, "
                f"charts, filters and decision list as JSON. Write no code and no other file.\n"
                + (f"On the dashboard each KPI is a counter, each chart a dashboard chart (a donut with more than "
                   f"{IMPLEMENT_DASHBOARD_PIE_PARTS} parts is drawn as a bar), each filter a dashboard filter and the decision list a "
                   f"table.\n" if outputs.get("dashboard") else "")
                + f"{self._implement_use_case_brief(context)}\n"
                + (f"What the user asked the App to show or let people do (this wins over every rule below): "
                   f"{outputs['app_purpose']}\n" if outputs.get("app_purpose") else "")
                + (f"What the user asked the dashboard to filter on (this wins over every rule below): "
                   f"{', '.join(outputs['dashboard_filters'])}; add a filter on each.\n" if outputs.get("dashboard_filters") else "")
                + f"The page follows this example, which is for another business: keep its structure and write your own content.\n"
                f"```json\n{json.dumps(example, indent=1, ensure_ascii=False)}\n```\nRules:\n"
                f"- version {shell_sql.SPEC_VERSION}. Inspire sets id, use_case_id, result_table, domain, subdomain and sponsor. title (at most "
                f"{shell_sql.TEXT_LIMITS['title']} characters) names the page; question (at most {shell_sql.TEXT_LIMITS['question']}) "
                f"is the decision it serves; story is one sentence with {{kpi_id}} and {{kpi_id:change}} placeholders, where "
                f"{{kpi_id:change}} shows as 'up 4.2%' or 'down 1.3%' and is followed by what it compares against, as in the example; "
                f"when there is no earlier period the App writes 'with no earlier period to compare' in its place and drops that "
                f"comparison; questions are "
                f"2 to 5 questions people would ask Ask Genie about this page; icon is one of {', '.join(shell_sql.ICONS)}; hue is one "
                f"of {', '.join(shell_sql.HUES)}.\n"
                f"- datasets: 1 to {caps['datasets']} named read-only SELECT queries, one statement each with no semicolon, reading "
                f"only these tables: {', '.join(f'`{t}`' for t in context['allowed'])}. The dataset named "
                f"{shell_sql.RECORDS_DATASET} is the decision list: every current row of `{result_table}` "
                f"(WHERE {implement_latest_run_filter(result_table)}), whatever its priority_tier, keeping the table's column names "
                f"entity_id, entity_label, priority_tier, rank, top_reasons and recommended_action unchanged, joined to input tables "
                f"for segment columns. Show each priority_tier exactly as the table stores it.\n"
                f"- filters: 1 to {caps['filters'][1]}. First, one date_range on as_of_date (the run date) of every dataset that has it, "
                f"grain month or day, default one of {', '.join(shell_sql.PRESET_IDS)}; {dates}; at least "
                f"{shell_sql.SEGMENT_FILTERS_MIN} multi_select segment filters people slice by, each naming the column in every dataset that carries it, so every KPI, chart and table "
                f"answers to the filters.\n"
                f"- kpis: {caps['kpis'][0]} to {caps['kpis'][1]}, each one aggregate expression over one dataset's columns (SUM, "
                f"COUNT, AVG, MIN, MAX); never SUM or AVG a ratio, a rate or an average: divide two sums. format is one of "
                f"{', '.join(shell_sql.FORMATS)}; a percent anywhere on the page is written 0 to 100, so multiply a 0 to 1 ratio or "
                f"score by 100 (the App prints the number with a % sign); good is up, down or neutral; compare true shows the change "
                f"on the period before.\n"
                f"- charts: {caps['charts'][0]} to {caps['charts'][1]} of type {', '.join(shell_sql.CHART_TYPES)}, at most "
                f"{caps['y']} measures each, and each chart compares something the others do not, such as another measure, another "
                f"split or another time grain, never the same measure by the same column twice. Only when the customer's asks for "
                f"this {IMPLEMENT_OUTPUT_LABELS[name]} ask for fewer than {caps['charts'][0]} charts, write fewer and set the "
                f"top-level field fewer_charts to the words of those asks that say so, word for word; otherwise leave fewer_charts "
                f"out. Pick the chart by the data: line or area for a trend over time, bar or horizontal_bar to "
                f"rank categories, stacked_bar for a mix over time, pareto for concentration, donut only for a share of a whole with "
                f"6 parts or fewer, scatter for two measures, heatmap for two categories. span 2 makes a chart full width. "
                f"A scatter's x is {{\"label\", \"expression\"}} with an aggregate expression; it has exactly one y, no series, and "
                f"\"group\": the name of the column whose values are its points, as a plain string. A heatmap's x is a category "
                f"column, its series {{\"column\", \"top\"}} is the second category, and it has exactly one y. A donut has exactly "
                f"one y and no series.\n"
                f"- tables: 0 to {caps['tables'][1]} extra tables. records names the decision list's columns: key, label, rank, and "
                f"optional sublabel, tier, reasons, action, columns and detail. The {shell_sql.RECORDS_DATASET} dataset selects "
                f"record_key, the result table's key for one record, so every row of the list is one record with its own "
                f"history. The page has at most {caps['tiles']} tiles.\n"
                f"- A chart's measure differs across its groups: never chart a count or a value that every group shares, such as "
                f"the number of offers per item when every item gets the same number.\n"
                f"- The App pages through the decision list with SQL, one page of rows at a time, so the {shell_sql.RECORDS_DATASET} "
                f"dataset never limits its rows itself. Every KPI and every chart aggregates in SQL on the warehouse, with an aggregate "
                f"expression over its dataset, so no chart selects or plots the raw rows of a table.\n"
                f"- Every title, label, hint and caption is plain business language, never a column name or code.\n"
                f"- Count data_quality_flag issues as rows whose flag is not 'ok'.\n"
                f"- Inspire validates {IMPLEMENT_APP_PAGE_FILE} with the App's own checks, then runs every query at the default "
                f"filters; whatever fails comes back to you with the reason. Before you finish, run "
                f"`python {IMPLEMENT_APP_CHECK_DIR}/{IMPLEMENT_APP_CHECK_SCRIPT}` in your working folder: it runs those same checks "
                f"on {IMPLEMENT_APP_PAGE_FILE} and must print no problems. Then run each dataset's SQL on SQL warehouse "
                f"{spec['config']['warehouse_id']} with `databricks api post /api/2.0/sql/statements`: each must succeed and return "
                f"rows, every chart must draw 2 or more groups, and at least {shell_sql.SEGMENT_FILTERS_MIN} segment filters must offer "
                f"2 or more values at the default filters.\n"
                + implement_vibes_block(outputs, name, f'a top-level "{IMPLEMENT_VIBES_NOT_DONE}" array of {IMPLEMENT_APP_PAGE_FILE}')
                + "Then reply with only the JSON object your output schema describes.")

    def _implement_app_check(self, work: str, files: dict, modules: dict, warehouse: str, lakebase_env=None) -> list:
        core, db = modules["shell_core"], modules["shell_db"]
        root = os.path.join(work, "_app_check")
        shutil.rmtree(root, ignore_errors=True)
        implement_write_files(root, {relative: content for relative, content in files.items()
                                     if relative == core.SETTINGS_FILE or relative.startswith(f"{core.PAGES_DIR}/")})
        host = self._implement_host()
        workspace = lambda *_args: db.Workspace(host, self.w_client.config.authenticate)
        resolved = dict({IMPLEMENT_APP_WAREHOUSE_RESOURCE: warehouse},
                        **({IMPLEMENT_LAKEBASE_RESOURCE: IMPLEMENT_LAKEBASE_ENDPOINT} if lakebase_env else {}))
        injected = dict({"DATABRICKS_HOST": host}, **{k: v for k, v in (lakebase_env or {}).items() if k in IMPLEMENT_APP_POSTGRES_ENV})
        env = implement_app_runtime_env(files.get(IMPLEMENT_APP_YAML), resolved, injected)
        try:
            shell = core.Shell(root, env=env, sp_factory=workspace, user_factory=workspace, strict=bool(lakebase_env))
        except (ValueError, db.QueryError) as error:
            return [str(error)]
        try:
            return shell.check(dict(IMPLEMENT_APP_CHECK_HEADERS))
        except (core.HttpError, db.QueryError) as error:
            return [str(error)]
        finally:
            shell.pool.shutdown(wait=True)
            if shell.lakebase is not None:
                shell.lakebase.close()

    def _implement_lakebase_service(self):
        return self._implement_app_shell()[1]["shell_db"].Workspace(self._implement_host(), self.w_client.config.authenticate)

    def _implement_lakebase_find(self, path: str):
        try:
            return self._implement_api("GET", path)
        except Exception as error:
            if implement_is_not_found(error):
                return None
            raise

    def _implement_lakebase_until(self, read, what: str, deadline: float):
        import time
        while True:
            found = read()
            if found:
                return found
            if time.time() > deadline:
                raise ImplementLakebaseUnavailable(f"{what} was not ready after {IMPLEMENT_LAKEBASE_TIMEOUT_S // 60} minutes")
            time.sleep(IMPLEMENT_LAKEBASE_POLL_S)

    def _implement_lakebase_ready(self, deadline: float) -> dict:
        try:
            self._implement_api("GET", "/api/2.0/postgres/projects", query={"page_size": 1})
        except Exception as error:
            raise ImplementLakebaseUnavailable(implement_lakebase_unavailable(error))
        try:
            return self._implement_lakebase_open(deadline)
        except Exception as error:
            if implement_is_permission_denied(error):
                raise ImplementLakebaseUnavailable(implement_lakebase_unavailable(error, IMPLEMENT_LAKEBASE_PROJECT)) from error
            raise

    def _implement_lakebase_open(self, deadline: float) -> dict:
        if self._implement_lakebase_find(f"/api/2.0/postgres/projects/{IMPLEMENT_LAKEBASE_PROJECT}") is None:
            self.logger.info(f"[IMPLEMENT-LAKEBASE] creating the Lakebase project {IMPLEMENT_LAKEBASE_PROJECT} for Inspire Apps (0.5 to 2 "
                             f"CU, it scales to zero when no one uses it)")
            self._implement_api("POST", "/api/2.0/postgres/projects", {"spec": IMPLEMENT_LAKEBASE_PROJECT_SPEC},
                                query={"project_id": IMPLEMENT_LAKEBASE_PROJECT})

        def host():
            endpoint = self._implement_lakebase_find(f"/api/2.0/postgres/{IMPLEMENT_LAKEBASE_ENDPOINT}") or {}
            return str(((endpoint.get("status") or {}).get("hosts") or {}).get("host") or "")

        pghost = self._implement_lakebase_until(host, f"the database endpoint of Lakebase project {IMPLEMENT_LAKEBASE_PROJECT}", deadline)
        path = f"/api/2.0/postgres/catalogs/{IMPLEMENT_LAKEBASE_CATALOG}"
        catalog = self._implement_lakebase_find(path)
        if catalog is None:
            self._implement_api("POST", "/api/2.0/postgres/catalogs", {"spec": {
                "branch": IMPLEMENT_LAKEBASE_BRANCH, "postgres_database": IMPLEMENT_LAKEBASE_DATABASE}},
                query={"catalog_id": IMPLEMENT_LAKEBASE_CATALOG})
            catalog = self._implement_lakebase_until(lambda: self._implement_lakebase_find(path),
                                                     f"the catalog {IMPLEMENT_LAKEBASE_CATALOG}", deadline)
        held = dict(catalog.get("spec") or {}, **{k: v for k, v in (catalog.get("status") or {}).items() if v})
        if held.get("branch") != IMPLEMENT_LAKEBASE_BRANCH or held.get("postgres_database") != IMPLEMENT_LAKEBASE_DATABASE:
            raise ImplementLakebaseUnavailable(f"the catalog {IMPLEMENT_LAKEBASE_CATALOG} already holds another Lakebase database "
                                               f"({held.get('branch')}, {held.get('postgres_database')})")
        return {"PGHOST": pghost, "PGPORT": "5432", "PGDATABASE": IMPLEMENT_LAKEBASE_DATABASE, "PGUSER": self._implement_builder()}

    def _implement_lakebase_views(self, pages, row: str, schema: str) -> list:
        copies, problems = implement_lakebase_copies(pages, row), []
        for copy in copies:
            self.spark.sql(implement_lakebase_view_sql(copy["result_table"], copy["suffix"], copy["body"], row))
            columns = [implement_lakebase_column(n, t) for n, t in inspire_table_schema(self.spark, copy["view"]) or []]
            problems += implement_lakebase_type_problems(copy, columns)
            copy.update(columns=columns, pg=f"{schema}.{implement_lakebase_pg_table(copy, columns)}")
            copy["synced"] = f"{IMPLEMENT_LAKEBASE_CATALOG}.{copy['pg']}"
        if problems:
            raise ImplementLakebaseUnavailable("; ".join(problems))
        return copies

    def _implement_lakebase_source(self, synced: str) -> str:
        try:
            table = self._implement_lakebase_find(f"/api/2.1/unity-catalog/tables/{synced}") or {}
        except Exception as error:
            if implement_is_permission_denied(error):
                raise ImplementLakebaseUnavailable(f"Inspire cannot tell which table the Lakebase table {synced} copies, "
                                                   f"because the identity Inspire runs as may not read it") from error
            raise
        source = str((table.get("properties") or {}).get("source_table") or "")
        if not source:
            raise ImplementLakebaseUnavailable(f"Inspire cannot tell which table the Lakebase table {synced} copies")
        return source

    def _implement_lakebase_synced(self, copies: list, storage, held_pipeline: str, row: str, deadline: float) -> str:
        pipeline_id = held_pipeline if held_pipeline and self._implement_lakebase_find(f"/api/2.0/pipelines/{held_pipeline}") else ""
        others = set()
        for copy in copies:
            found = self._implement_lakebase_find(f"/api/2.0/postgres/synced_tables/{copy['synced']}")
            if found is not None:
                source = self._implement_lakebase_source(copy["synced"])
                if source.lower() != copy["view"].lower():
                    raise ImplementLakebaseUnavailable(f"the Lakebase table {copy['synced']} already copies {source}")
                held = str((found.get("status") or {}).get("pipeline_id") or "")
                pipeline_id = pipeline_id or held
                others.add(held)
                continue
            spec = {"branch": IMPLEMENT_LAKEBASE_BRANCH, "postgres_database": IMPLEMENT_LAKEBASE_DATABASE,
                    "source_table_full_name": copy["view"], "primary_key_columns": [row], "scheduling_policy": "SNAPSHOT",
                    "create_database_objects_if_missing": True}
            spec.update({"existing_pipeline_id": pipeline_id} if pipeline_id else
                        {"new_pipeline_spec": {"storage_catalog": storage[0], "storage_schema": storage[1]}})
            try:
                self._implement_api("POST", "/api/2.0/postgres/synced_tables", {"spec": spec}, query={"synced_table_id": copy["synced"]})
            except Exception as error:
                if implement_is_permission_denied(error):
                    raise ImplementLakebaseUnavailable(implement_lakebase_schema_denied(copy["synced"], self._implement_builder())) from error
                raise
            if not pipeline_id:
                pipeline_id = self._implement_lakebase_until(lambda: str(((self._implement_lakebase_find(
                    f"/api/2.0/postgres/synced_tables/{copy['synced']}") or {}).get("status") or {}).get("pipeline_id") or ""),
                    f"the sync pipeline of {copy['synced']}", deadline)
        if others - {"", pipeline_id}:
            raise ImplementLakebaseUnavailable(f"the App's Lakebase tables are synced by more than one pipeline "
                                               f"({', '.join(sorted(others - {''} | {pipeline_id}))})")
        return pipeline_id

    def _implement_lakebase_online(self, names, deadline: float, wait: bool = True) -> bool:
        def online():
            waiting = []
            for name in names:
                status = (self._implement_lakebase_find(f"/api/2.0/postgres/synced_tables/{name}") or {}).get("status") or {}
                state = str(status.get("detailed_state") or "")
                if "FAILED" in state:
                    raise ImplementLakebaseUnavailable(f"Lakebase could not copy {name} ({state}): "
                                                       f"{' '.join(str(status.get('message') or '').split())[:400]}")
                if state not in IMPLEMENT_LAKEBASE_ONLINE:
                    waiting.append(name)
            return not waiting

        if not wait:
            return online()
        return self._implement_lakebase_until(online, f"the Lakebase copy of {len(names)} table(s)", deadline)

    def _implement_lakebase_sql(self, env: dict, statements: list) -> None:
        import psycopg
        token = str(self._implement_api("POST", "/api/2.0/postgres/credentials", {"endpoint": IMPLEMENT_LAKEBASE_ENDPOINT}).get("token") or "")
        with psycopg.connect(host=env["PGHOST"], port=int(env["PGPORT"]), dbname=env["PGDATABASE"], user=env["PGUSER"], password=token,
                             sslmode="require", connect_timeout=30, autocommit=True) as conn, conn.cursor() as cur:
            for statement in statements:
                cur.execute(statement)

    def _implement_lakebase_grant(self, lakebase: dict, client_id: str) -> str:
        from psycopg import sql
        path = f"/api/2.0/postgres/{IMPLEMENT_LAKEBASE_BRANCH}/roles"

        def held():
            token = ""
            while True:
                page = self._implement_api("GET", path, query={"page_token": token} if token else None)
                if any(str((r.get("status") or {}).get("postgres_role") or "") == client_id for r in page.get("roles") or []):
                    return True
                token = str(page.get("next_page_token") or "")
                if not token:
                    return False

        try:
            if not held():
                try:
                    self._implement_api("POST", path, {"spec": {"identity_type": "SERVICE_PRINCIPAL", "postgres_role": client_id,
                                                                "auth_method": "LAKEBASE_OAUTH_V1"}}, query={"role_id": client_id})
                except Exception as error:
                    if "already exists" not in str(error).lower() and str(getattr(error, "error_code", "")) != "ALREADY_EXISTS":
                        raise
                import time
                self._implement_lakebase_until(held, f"the Lakebase role of the App's service principal {client_id}",
                                               time.time() + 300)
            schema, who = sql.Identifier(lakebase["schema"]), sql.Identifier(client_id)
            self._implement_lakebase_sql(lakebase["env"], [
                sql.SQL("GRANT USAGE ON SCHEMA {} TO {}").format(schema, who),
                sql.SQL("GRANT SELECT ON ALL TABLES IN SCHEMA {} TO {}").format(schema, who)])
            self._implement_grant("pipelines", lakebase["backend"]["pipeline_id"], [{"type": "service_principal", "name": client_id}],
                                  "CAN_RUN")
        except Exception as error:
            return get_clean_error_message(error)[:500]
        return ""

    def _implement_lakebase_retire(self, gone) -> None:
        for pg_name in sorted(gone):
            try:
                if self._implement_lakebase_find(f"/api/2.0/postgres/synced_tables/{IMPLEMENT_LAKEBASE_CATALOG}.{pg_name}") is not None:
                    self._implement_api("DELETE", f"/api/2.0/postgres/synced_tables/{IMPLEMENT_LAKEBASE_CATALOG}.{pg_name}")
                self.logger.info(f"[IMPLEMENT-LAKEBASE] removed the Lakebase copy {pg_name}, which the App no longer reads")
            except Exception as error:
                self.logger.warning(f"[IMPLEMENT-LAKEBASE] the Lakebase copy {pg_name}, which the App no longer reads, could not be "
                                    f"removed: {get_clean_error_message(error)[:300]}")

    def _implement_app_lakebase(self, context: dict, host: dict, settings: dict, pages, payload: dict, modules: dict, work: str,
                                resync: bool = True) -> dict:
        import time
        core, pg, uc_id, key = modules["shell_core"], modules["shell_pg"], context["uc_id"], context["key"]
        held = ((host.get("settings") or {}).get("backend") or {}) if host.get("settings") else {}
        delta_files = dict(implement_app_source(payload["files"], core.SETTINGS_FILE, core.PAGES_DIR,
                                                dict(settings, backend={"kind": IMPLEMENT_APP_BACKENDS[-1]}), pages),
                           **{"requirements.txt": ""})
        out = {"kind": IMPLEMENT_APP_BACKENDS[-1], "files": delta_files, "delta_files": delta_files, "warnings": [], "held": held}
        kind, note = implement_app_wanted_backend(context["spec"]["outputs"], host)
        if note:
            out["warnings"].append(note)
        if kind != IMPLEMENT_APP_BACKENDS[0]:
            return out
        step = self._implement_emit(uc_id, "App", "copying the App's data into Lakebase, so its pages load instantly", "started",
                                    {"spec_version": key[2]})
        schema = implement_lakebase_schema(host["name"])
        try:
            for module, pip_name in (("psycopg", pg.REQUIREMENTS[0]), ("sqlglot", pg.REQUIREMENTS[1])):
                self._require_python_package(module, pip_name, "The Lakebase copy of the App")
            deadline = time.time() + IMPLEMENT_LAKEBASE_TIMEOUT_S
            env = self._implement_lakebase_ready(deadline)
            app_yaml = implement_app_yaml(payload["files"].get(IMPLEMENT_APP_YAML), True)
            if resync or held.get("kind") != IMPLEMENT_APP_BACKENDS[0]:
                copies = self._implement_lakebase_views(pages, pg.ROW_COLUMN, schema)
                pipeline_id = self._implement_lakebase_synced(copies, inspire_table_parts(context["result_table"])[:2],
                                                              str(held.get("pipeline_id") or ""), pg.ROW_COLUMN, deadline)
                backend = implement_lakebase_backend(pipeline_id, copies)
                runtime = implement_app_runtime_env(app_yaml, {IMPLEMENT_LAKEBASE_RESOURCE: IMPLEMENT_LAKEBASE_ENDPOINT},
                                                    {k: v for k, v in env.items() if k in IMPLEMENT_APP_POSTGRES_ENV})
                reader, names = pg.Lakebase(backend, runtime, self._implement_lakebase_service), [c["synced"] for c in copies]
                for _round in range(2):
                    state, _created_at = reader.sync_now()
                    if state not in pg.SYNC_DONE:
                        raise ImplementLakebaseUnavailable(f"the Lakebase sync ended {state.lower()}")
                    if self._implement_lakebase_online(names, deadline, wait=False):
                        break
                self._implement_lakebase_online(names, deadline)
            else:
                backend = held
            files = dict(implement_app_source(payload["files"], core.SETTINGS_FILE, core.PAGES_DIR, dict(settings, backend=backend), pages),
                         **{"requirements.txt": "\n".join(pg.REQUIREMENTS) + "\n", IMPLEMENT_APP_YAML: app_yaml})
            problems = self._implement_app_check(os.path.join(work, "_lakebase"), files, modules,
                                                 context["spec"]["config"]["warehouse_id"], env)
            if problems:
                raise ImplementLakebaseUnavailable("these App queries did not run on Lakebase: " + "; ".join(problems[:5]))
        except Exception as error:
            warning = implement_lakebase_fallback(str(error) if isinstance(error, ImplementLakebaseUnavailable)
                                                  else get_clean_error_message(error)[:600])
            self.logger.warning(f"[IMPLEMENT-LAKEBASE] {uc_id}: {warning}")
            self._implement_emit(uc_id, "App", warning, "ended_warning", {"spec_version": key[2]}, step_id=step)
            out["warnings"].append(warning)
            return out
        self._implement_emit(uc_id, "App", f"the App reads Lakebase: {len(backend['tables'])} table(s) copied and checked",
                             "ended_success", {"spec_version": key[2]}, step_id=step)
        return dict(out, kind=IMPLEMENT_APP_BACKENDS[0], files=files, backend=backend, env=env, schema=schema,
                    requirements=list(pg.REQUIREMENTS))

    def _implement_lakebase_after(self, context: dict, fields: dict, lakebase: dict) -> None:
        on = (fields.get("_asset") or {}).get("backend") == IMPLEMENT_APP_BACKENDS[0]
        was_on = bool(implement_lakebase_pg_names(lakebase.get("held")))
        job_id = str(context.get("job_id") or "")
        if re.fullmatch(r"[0-9]+", job_id) and (on or was_on):
            try:
                if on:
                    self._implement_job_after_task(int(job_id), IMPLEMENT_LAKEBASE_TASK_KEY,
                                                   {"pipeline_task": {"pipeline_id": lakebase["backend"]["pipeline_id"]}})
                else:
                    self._implement_remove_job_task(int(job_id), IMPLEMENT_LAKEBASE_TASK_KEY)
            except Exception as error:
                fields["_warnings"] = (fields.get("_warnings") or []) + [
                    (f"App: Lakebase copies the result only when someone saves a change in the App, because Inspire could not add the "
                     f"Lakebase sync to job {job_id}" if on else
                     f"App: job {job_id} still syncs Lakebase after each run, because Inspire could not remove that task")
                    + f": {get_clean_error_message(error)[:300]}"]
        kept = implement_lakebase_pg_names(lakebase.get("backend")) if on else set()
        self._implement_lakebase_retire((implement_lakebase_pg_names(lakebase.get("held"))
                                         | implement_lakebase_pg_names(lakebase.get("backend"))) - kept)

    def _implement_app_settings(self, context: dict, modules: dict, page: dict, genie, note: str, outputs: dict) -> dict:
        title = f"{context['uc_id']} {context['use_case'].get('Name') or ''}"
        return implement_app_settings(modules["shell_core"].SETTINGS_VERSION, title, [page["id"]], genie, note,
                                      datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), outputs,
                                      jobs={page["id"]: context.get("job_id")}, admin=self._implement_builder())

    def _implement_builder(self) -> str:
        if getattr(self, "_implement_builder_name", None) is None:
            try:
                rows = self.spark.sql("SELECT current_user() AS me").collect()
                row = rows[0] if rows else {}
                self._implement_builder_name = " ".join(str((row.asDict() if hasattr(row, "asDict") else dict(row)).get("me") or "").split())
            except Exception as error:
                self.logger.warning(f"[IMPLEMENT-APP] the identity Inspire runs as could not be read, so the App names no admin: "
                                    f"{get_clean_error_message(error)[:300]}")
                self._implement_builder_name = ""
        return self._implement_builder_name

    def _implement_cluster_result(self, context: dict, page: dict) -> list:
        table, wanted = context["result_table"], []
        try:
            wanted = implement_cluster_columns(page, inspire_table_schema(self.spark, table))
            if not wanted:
                return []
            rows = self.spark.sql(implement_cluster_detail_sql(table)).collect()
            row = rows[0] if rows else {}
            detail = row.asDict() if hasattr(row, "asDict") else dict(row)
            partitioned = [str(c) for c in detail.get("partitionColumns") or []]
            if partitioned:
                return [f"App: {table} is partitioned by {', '.join(partitioned)}, so Inspire did not cluster it on {', '.join(wanted)} "
                        "for the App's filters; the App's queries may be slow on a large table"]
            if {str(c).lower() for c in detail.get("clusteringColumns") or []} == {c.lower() for c in wanted}:
                return []
            self.spark.sql(implement_cluster_sql(table, wanted))
            self.logger.info(f"[IMPLEMENT-APP] {context['uc_id']}: clustered {table} on {', '.join(wanted)} for the App's filters")
            return []
        except Exception as error:
            return [f"App: Inspire could not cluster {table} on {', '.join(wanted) or 'the columns the App filters on'}, so the App's "
                    f"queries may be slow on a large table: {get_clean_error_message(error)[:300]}"]

    def _implement_app_files(self, context: dict, payload: dict, modules: dict, page: dict, genie, note: str,
                             outputs: dict) -> dict:
        core = modules["shell_core"]
        settings = self._implement_app_settings(context, modules, page, genie, note, outputs)
        return implement_app_source(payload["files"], core.SETTINGS_FILE, core.PAGES_DIR, settings, [page])

    def _implement_page_file(self, context: dict, name: str, work: str, shell_sql, event_date: bool) -> tuple:
        label = IMPLEMENT_OUTPUT_LABELS[name]
        path = os.path.join(work, IMPLEMENT_APP_PAGE_FILE)
        if not os.path.exists(path):
            raise RuntimeError(f"{IMPLEMENT_APP_PAGE_FILE} was not written")
        try:
            raw = json.loads(open(path, encoding="utf-8").read())
        except ValueError as error:
            raise RuntimeError(f"{IMPLEMENT_APP_PAGE_FILE} is not valid JSON: {error}")
        not_done, errors = implement_vibes_not_done(raw, context["spec"]["outputs"], name)
        if errors:
            raise RuntimeError(f"{IMPLEMENT_APP_PAGE_FILE}: " + "; ".join(errors))
        page = implement_app_page(raw, context["uc_id"], context["use_case"], context["result_table"])
        problems = implement_app_page_problems(page, shell_sql, context["allowed"], event_date)
        problems += implement_fewer_charts_problems(page, implement_output_vibes(context["spec"]["outputs"], name), label,
                                                    shell_sql.CAPS["charts"][0])
        problems += implement_page_request_problems(page, context["spec"]["outputs"].get("dashboard_filters"))
        if problems:
            raise RuntimeError(f"{IMPLEMENT_APP_PAGE_FILE} does not fit the {label}: " + "; ".join(problems))
        records = next(d["sql"] for d in page["datasets"] if d["name"] == shell_sql.RECORDS_DATASET)
        try:
            rows = self._implement_run_query_check(records, limit=1000)
        except Exception as error:
            raise RuntimeError(f"dataset {shell_sql.RECORDS_DATASET} failed: {get_clean_error_message(error)[:800]}")
        problems = implement_decision_list_problems(rows, self._implement_latest_run_rows(context["result_table"]),
                                                    dataset=shell_sql.RECORDS_DATASET)
        if problems:
            raise RuntimeError("; ".join(problems))
        return page, not_done

    def _implement_page_check(self, context: dict, label: str, work: str, files: dict, modules: dict, page: dict, pages=None,
                              app_name: str = "") -> None:
        problems = self._implement_app_check(work, files, modules, context["spec"]["config"]["warehouse_id"])
        if problems:
            raise implement_app_check_failure(app_name, problems, pages or [page], page["id"], label)
        context["page_spec"] = page

    def _implement_page_spec(self, context: dict, runner, name: str, deploy) -> dict:
        payload, modules = self._implement_app_shell()
        shell_sql, label = modules["shell_sql"], IMPLEMENT_OUTPUT_LABELS[name]
        vibes = implement_output_vibes(context["spec"]["outputs"], name)

        def kept(work, page, not_done=()):
            saved = json.loads(json.dumps(page))
            fields = deploy(work, page)
            context.setdefault("artifacts", {})[name] = saved
            return dict(fields, _warnings=list(fields.get("_warnings") or []) + list(not_done)) if not_done else fields

        if name in (context.get("replay") or {}):
            event_date = self._implement_input_has_dates(context)
            context["page_spec_vibes"] = None
            return self._implement_replay(context, name, lambda work: kept(
                work, *self._implement_page_file(context, name, work, shell_sql, event_date)))
        page = context.get("page_spec") if not vibes and context.get("page_spec_vibes", "") == "" else None
        if page is None:
            event_date = self._implement_input_has_dates(context)
            prompt = self._implement_app_prompt(context, shell_sql, payload["example"], event_date, name)

            def written(work):
                page, not_done = self._implement_page_file(context, name, work, shell_sql, event_date)
                context["page_spec_vibes"] = vibes
                return kept(work, page, not_done)

            return self._implement_genie_loop(context, runner, label, prompt, written,
                                              tools=implement_app_check_files(payload["files"]["shell_sql.py"], event_date))
        uc_id, key = context["uc_id"], context["key"]
        work = self._implement_work_dir(uc_id, f"{label.lower().replace(' ', '_')}_page")
        shutil.rmtree(work, ignore_errors=True)
        os.makedirs(work, exist_ok=True)
        step = self._implement_emit(uc_id, label, f"building the {label} from the page spec Genie wrote and Inspire checked earlier in "
                                    f"this run, with no second Genie round", "started", {"spec_version": key[2]})
        try:
            fields = kept(work, page)
        except Exception as error:
            self._implement_emit(uc_id, label, f"failed: {get_clean_error_message(error)[:1500]}", "ended_error",
                                 {"spec_version": key[2]}, step_id=step)
            raise
        self._implement_emit(uc_id, label, f"{label} ready", "ended_success",
                             dict({k: v for k, v in fields.items() if not k.startswith("_")}, spec_version=key[2]), step_id=step)
        return fields

    def _implement_app(self, context: dict, runner) -> dict:
        spec, uc_id = context["spec"], context["uc_id"]
        payload, modules = self._implement_app_shell()
        core = modules["shell_core"]
        host = self._implement_app_host(context, core)
        host["notes"] += self._implement_merge_moved(context, "app", host["name"])
        context["app_host"] = host
        if "app" in (context.get("restyle") or ()) and host["settings"] is not None:
            restyled = self._implement_app_restyle(context, host, payload, modules)
            if restyled is not None:
                return restyled
            runner = runner or self._implement_runner(uc_id)
        elif "app" in (context.get("restyle") or ()):
            self.logger.info(f"[IMPLEMENT-APP] {uc_id}: the App {host['name']} has no settings Inspire can read, so Genie rebuilds it")
            runner = runner or self._implement_runner(uc_id)
        host_genie = self._implement_app_host_genie(context, host)

        def deploy(work, page):
            genie, note = self._implement_app_genie(context)
            settings = self._implement_app_settings(context, modules, page, genie, note, spec["outputs"])
            settings, pages, page = implement_app_merge(host, settings, page, self.session_id, spec["outputs"]["app_look"])
            settings = dict(settings, backend={"kind": IMPLEMENT_APP_BACKENDS[-1]})
            genie, note = settings.get("genie"), settings.get("genie_note") or ""
            files = implement_app_source(payload["files"], core.SETTINGS_FILE, core.PAGES_DIR, settings, pages)
            self._implement_page_check(context, "App", work, files, modules, page, pages, host["name"])
            self._implement_checks_passed(context, "app")
            context["app_jobs"] = settings["jobs"]
            read = implement_app_read_tables(page, context["allowed"])
            if genie:
                read += [t for t in context["allowed"] if t not in read]
            read += [t for t in implement_app_page_reads(pages, page["id"]) if t not in read]
            clustered = self._implement_cluster_result(context, page)
            lakebase = self._implement_app_lakebase(context, host, settings, pages, payload, modules, work)
            try:
                fields = self._implement_deploy_app(context, host["name"], lakebase["files"], read, (genie or {}).get("space_id", ""),
                                                    lakebase=lakebase)
            except ImplementBlocked:
                raise
            except Exception as error:
                raise ImplementBlocked(f"the page passed every check, then the App could not be deployed: "
                                       f"{get_clean_error_message(error)[:800]}")
            self._implement_lakebase_after(context, fields, lakebase)
            fields["_warnings"] = ((fields.get("_warnings") or []) + host["notes"] + ([f"App: {note}"] if note else [])
                                   + clustered + implement_app_admin_warnings(settings))
            if not fields["_warnings"]:
                fields.pop("_warnings")
            if host["merged"]:
                fields["_asset"] = dict(fields.get("_asset") or {}, merged_into=host["name"], page=page["id"], genie_merged=host_genie)
            return fields

        try:
            return self._implement_page_spec(context, runner, "app", deploy)
        except Exception as error:
            if host.get("genie_before"):
                space_id, before = host["genie_before"]
                self._implement_undo_save(context, "app", error, lambda: self._implement_restore_asset(
                    "genie_space", space_id, {"body": {"serialized_space": json.dumps(before["space"])}}))
            raise

    def _implement_app_host(self, context: dict, core) -> dict:
        spec, uc_id = context["spec"], context["uc_id"]
        target = implement_merge_target(spec["outputs"], "app")
        host = implement_app_host(target or self._implement_app_name(context, spec["outputs"]["app_name"]))
        name = host["name"]
        try:
            app = self._implement_api("GET", f"/api/2.0/apps/{name}")
        except Exception as error:
            if not implement_is_not_found(error):
                raise ImplementBlocked(f"the App {name} could not be read: {get_clean_error_message(error)[:500]}")
            app = None
        if target and not self._implement_owns_app(None, app):
            raise ImplementBlocked(f"the App {name} you chose to add this use case to "
                                   + ("no longer exists" if app is None else "was not built by Inspire, so Inspire does not change it")
                                   + "; pick another Inspire App or build a new App for this use case")
        if app is None or not self._implement_owns_app(None if target else uc_id, app):
            return host
        host.update(app=app, owner=implement_app_owner(app), merged=bool(target) and not self._implement_owns_app(uc_id, app))
        own_source = os.path.join(self._implement_notebook_paths(context["use_case"])[0], "apps", name)
        host["source"] = (str((app.get("active_deployment") or {}).get("source_code_path") or app.get("default_source_code_path") or "")
                          if host["merged"] else "") or own_source
        try:
            settings = implement_app_held_settings(json.loads(read_user_text_file(f"{host['source']}/{core.SETTINGS_FILE}", self.w_client)))
            pages = [json.loads(read_user_text_file(f"{host['source']}/{core.PAGES_DIR}/{ident}.json", self.w_client))
                     for ident in (settings.get("pages") or [] if isinstance(settings, dict) else [])]
        except Exception as error:
            if not host["merged"] and implement_is_not_found(error):
                self.logger.info(f"[IMPLEMENT-APP] {uc_id}: the App {name} holds no Inspire App pages yet, so it is built afresh")
                return host
            raise ImplementBlocked(f"the pages of the App {name} could not be read from {host['source']}, so Inspire did not change "
                                   f"it: {get_clean_error_message(error)[:500]}")
        host.update(settings=settings, pages=pages)
        if not host["merged"]:
            return host
        problems = core.settings_problems(dict(settings, version=core.SETTINGS_VERSION,
                                               jobs=implement_app_kept_jobs(settings, [p.get("id") for p in pages]),
                                               admin=" ".join(str(settings.get("admin") or "").split()) or self._implement_builder()),
                                          pages)
        if problems:
            raise ImplementBlocked(f"the App {name} does not hold together, so Inspire did not change it: {'; '.join(problems[:10])}")
        wanted = dict(spec["outputs"]["app_look"], auth=spec["outputs"]["app_access"])
        if settings.get("auth") == "sp" and wanted["auth"] == "user":
            raise ImplementBlocked(f"this use case shows each viewer only their own data, but the App {name} shows everyone the same "
                                   "data through its service principal, so every viewer of the App would see all of this use case's "
                                   "data. Add it to an App where each viewer signs in as themselves, or build a new App for it")
        kept = [f"{key} {settings.get(key)}" for key in ("auth", *IMPLEMENT_APP_LOOK_CHOICES) if settings.get(key) != wanted[key]]
        if kept:
            note = (f"App: this use case is a page of the App {name} of use case {host['owner'][0]}, which keeps its own "
                    f"{', '.join(kept)}; the access and look chosen in the wizard apply only to an App this use case builds itself")
            self.logger.warning(f"[IMPLEMENT-APP-MERGE] {uc_id}: {note}")
            host["notes"].append(note)
        return host

    def _implement_app_host_genie(self, context: dict, host: dict) -> dict:
        if not host["merged"]:
            return {}
        previous = (context["asset_status"].get("app") or {}).get("genie_merged") or {}
        genie, own = host["settings"].get("genie"), (self._implement_app_genie(context)[0] or {}).get("space_id", "")
        if not genie:
            if not own:
                return {}
            others = {"data_sources": {"tables": [{"identifier": t} for t in implement_app_page_reads(host["pages"])]}}
            host["genie_before"] = (own, self._implement_genie_read(own))
            return self._implement_merge_genie(context, own, others, previous, host["genie_before"][1])
        if own == str(genie.get("space_id")):
            return {}
        ours = {"data_sources": {"tables": [{"identifier": t} for t in context["allowed"]]}}
        if own:
            try:
                ours = self._implement_genie_read(own)["space"]
            except Exception as error:
                if not implement_is_not_found(error):
                    raise
                self.logger.warning(f"[IMPLEMENT-APP-MERGE] {context['uc_id']}: this use case's Genie Agent {own} is gone, so only its "
                                    f"tables go into the App's Genie Agent {genie.get('space_id')}")
        host["genie_before"] = (str(genie["space_id"]), self._implement_genie_read(str(genie["space_id"])))
        record = self._implement_merge_genie(context, str(genie["space_id"]), ours, previous, host["genie_before"][1])
        questions = [_implement_genie_text(q.get("question")) for q in _implement_genie_items(ours, "sample_questions")]
        try:
            if questions:
                self._implement_genie_smoke(str(genie["space_id"]), questions[0])
        except Exception as error:
            host["notes"].append(f"App: Ask Genie in the App {host['name']} did not answer this use case's question {questions[0]!r} "
                                 f"yet: {get_clean_error_message(error)[:300]}")
        return record

    def _implement_app_restyle(self, context: dict, host: dict, payload: dict, modules: dict) -> dict:
        spec, uc_id, key = context["spec"], context["uc_id"], context["key"]
        core = modules["shell_core"]
        if host["merged"]:
            self._implement_emit(uc_id, "App", f"only the look or data backend changed, and the App {host['name']} keeps its own; "
                                 "nothing to redeploy", "ended_warning", {"spec_version": key[2]})
            return {"app_name": host["name"], "app_url": str(host["app"].get("url") or "").rstrip("/"), "_warnings": host["notes"]}
        step = self._implement_emit(uc_id, "App", "only the look or data backend changed: Inspire rewrites the App settings and "
                                    "redeploys it, without Genie", "started", {"spec_version": key[2]})
        held = host["settings"]
        settings = dict(held, **spec["outputs"]["app_look"], version=core.SETTINGS_VERSION,
                        built_at=datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
                        admin=self._implement_builder() or " ".join(str(held.get("admin") or "").split()),
                        jobs=implement_app_kept_jobs(held, [p.get("id") for p in host["pages"]], implement_app_page_id(uc_id),
                                                     context.get("job_id")), backend={"kind": IMPLEMENT_APP_BACKENDS[-1]})
        context["app_jobs"] = settings["jobs"]
        files = implement_app_source(payload["files"], core.SETTINGS_FILE, core.PAGES_DIR, settings, host["pages"])
        work = self._implement_work_dir(uc_id, "app_restyle")
        try:
            problems = self._implement_app_check(work, files, modules, spec["config"]["warehouse_id"])
            if problems:
                note = (f"App: its pages no longer passed Inspire's checks ({'; '.join(problems[:10])}), so Genie rebuilt them "
                        "with the new look and data backend")
                self._implement_emit(uc_id, "App", note, "ended_warning", {"spec_version": key[2]}, step_id=step)
                self.logger.info(f"[IMPLEMENT-APP] {uc_id}: {note}")
                host["notes"].append(note)
                return None
            read = implement_app_page_reads(host["pages"])
            if settings.get("genie"):
                read += [t for t in context["allowed"] if t not in read]
            lakebase = self._implement_app_lakebase(context, host, settings, host["pages"], payload, modules, work, resync=False)
            fields = self._implement_deploy_app(context, host["name"], lakebase["files"], read,
                                                (settings.get("genie") or {}).get("space_id", ""), lakebase=lakebase)
            self._implement_lakebase_after(context, fields, lakebase)
            unnamed = implement_app_admin_warnings(settings)
            if unnamed:
                fields["_warnings"] = (fields.get("_warnings") or []) + unnamed
        except Exception as error:
            self._implement_emit(uc_id, "App", get_clean_error_message(error)[:1500], "ended_error", {"spec_version": key[2]},
                                 step_id=step)
            raise
        backend_name = "Lakebase" if lakebase["kind"] == IMPLEMENT_APP_BACKENDS[0] else "Delta tables"
        self._implement_emit(uc_id, "App", f"App look set to {', '.join(f'{k} {settings[k]}' for k in IMPLEMENT_APP_LOOK_CHOICES)}; "
                             f"it reads {backend_name}", "ended_success", {"spec_version": key[2], "app_name": host["name"]}, step_id=step)
        return fields

    def _implement_app_owner_tag(self, uc_id: str) -> str:
        return f"Inspire use case {uc_id} (session {self.session_id})"

    def _implement_owns_app(self, uc_id, app) -> bool:
        description = str((app or {}).get("description") or "")
        if uc_id is None:
            return implement_app_owner(app) is not None
        return description.startswith(self._implement_app_owner_tag(uc_id))

    def _implement_app_name(self, context: dict, app_name: str) -> str:
        try:
            existing = self._implement_api("GET", f"/api/2.0/apps/{app_name}")
        except Exception:
            return app_name
        if self._implement_owns_app(context["uc_id"], existing):
            return app_name
        suffix = hashlib.sha256(f"{self.session_id}:{context['uc_id']}".encode()).hexdigest()[:4]
        renamed = f"{app_name[:25].rstrip('-')}-{suffix}"
        self._implement_emit(context["uc_id"], "App", f"an App named {app_name} exists but is not this use case's App (its description "
                             f"does not start with '{self._implement_app_owner_tag(context['uc_id'])}'); deploying as {renamed}",
                             "ended_warning", {"spec_version": context["key"][2]})
        return renamed

    def _implement_set_app_scopes(self, app_name: str, app: dict, scopes: list, resources=None) -> tuple:
        try:
            return self._implement_api("PATCH", f"/api/2.0/apps/{app_name}", {
                "name": app_name, "description": app.get("description") or "",
                "resources": (app.get("resources") or []) if resources is None else resources, "user_api_scopes": scopes}), ""
        except Exception as error:
            return app, get_clean_error_message(error)[:300]

    def _implement_app_job_resources(self, app_name: str, app: dict, jobs, warnings: list, postgres: bool = False) -> tuple:
        held = list(app.get("resources") or [])
        if not implement_app_missing_jobs(held, jobs) and (not postgres or implement_app_holds_postgres(held)):
            return app, False
        app, problem = self._implement_set_app_scopes(app_name, app, list(app.get("user_api_scopes") or []),
                                                      implement_app_resources(held, jobs, postgres))
        missing = implement_app_missing_jobs(app.get("resources") or [], jobs)
        if missing:
            warnings.append(f"App: Refresh now cannot start a run of job {', '.join(missing)}, because Inspire could not give the "
                            "App's service principal CAN MANAGE RUN on it" + (f": {problem}" if problem else ""))
        return app, not problem

    def _implement_app_scopes(self, app_name: str, app: dict, scopes: list, warnings: list) -> dict:
        if sorted(app.get("user_api_scopes") or []) == sorted(scopes):
            return app
        app, problem = self._implement_set_app_scopes(app_name, app, scopes)
        held = sorted(app.get("user_api_scopes") or [])
        if held == sorted(scopes):
            return app
        if scopes:
            raise ImplementBlocked(f"the App {app_name} shows each person only the data their Unity Catalog grants allow, so it must act "
                                   f"as each viewer, but Inspire could not give it the user authorization scopes {', '.join(scopes)}"
                                   + (f": {problem}" if problem else f" (it holds {', '.join(held) or 'none'})")
                                   + ". A workspace admin may limit the scopes Apps can use: ask them to allow these scopes, or choose "
                                   "a shared view for the App")
        warnings.append(f"App: {app_name} reads data as its service principal now, but Inspire could not remove the user authorization "
                        f"scopes {', '.join(held)} it held before" + (f": {problem}" if problem else "")
                        + ". Remove them under User authorization in the App's settings so it no longer asks people to let it act as them")
        return app

    def _implement_deploy_app(self, context: dict, app_name: str, files: dict, read_tables, space_id: str = "",
                              timeout_s: int = 1200, poll_s: int = 10, lakebase=None) -> dict:
        import time
        tables = [context["result_table"]] + [t for t in read_tables or [] if t != context["result_table"]]
        access = {table: self._implement_access_rows(table) for table in tables}
        host = context.get("app_host") or {}
        merged = bool(host.get("merged"))
        held = (host.get("settings") or {}) if merged else {}
        chosen = {"app_access": held.get("auth"), "app_options": held.get("options") or {}} if merged else context["spec"]["outputs"]
        per_user = chosen.get("app_access") == "user"
        scopes = implement_app_user_scopes(chosen)
        lakebase = lakebase or {}
        on_lakebase = lakebase.get("kind") == IMPLEMENT_APP_BACKENDS[0]
        warnings = list(lakebase.get("warnings") or [])

        def to_delta(reason: str):
            nonlocal files, on_lakebase
            files, on_lakebase = lakebase["delta_files"], False
            warnings.append(implement_lakebase_fallback(reason))
            self.logger.warning(f"[IMPLEMENT-LAKEBASE] {context['uc_id']}: {warnings[-1]}")
        try:
            app = self._implement_api("GET", f"/api/2.0/apps/{app_name}")
        except Exception as error:
            if not implement_is_not_found(error):
                raise ImplementBlocked(f"the App {app_name} could not be read: {get_clean_error_message(error)[:500]}")
            app = None
        if app is not None and not self._implement_owns_app(None if merged else context["uc_id"], app):
            raise ImplementBlocked(f"the App {app_name} exists but is not this use case's App, so Inspire neither deploys to it nor "
                                   f"grants it access to {context['result_table']}; pick another App name")
        if app is None and merged:
            raise ImplementBlocked(f"the App {app_name} this use case is added to was deleted while Inspire built its page; pick "
                                   "another Inspire App or build a new App for this use case")
        if app is None and not per_user:
            readable = implement_access_plan(context["result_table"], access[context["result_table"]])
            if readable["gaps"]:
                raise ImplementBlocked(implement_app_access_message(context["result_table"], readable["gaps"], readable["me"]))
        source = ((context["app_host"]["source"] if merged else "")
                  or os.path.join(self._implement_notebook_paths(context["use_case"])[0], "apps", app_name))
        before = self._implement_app_before(app_name, app, source, files)
        jobs = context.get("app_jobs") or {}
        created = touched = started = resourced = False
        try:
            if app is None:
                try:
                    app = self._implement_api("POST", "/api/2.0/apps", {
                        "name": app_name, "description": f"{self._implement_app_owner_tag(context['uc_id'])}: {context['use_case'].get('Name')}"[:250],
                        "resources": implement_app_resources([{"name": IMPLEMENT_APP_WAREHOUSE_RESOURCE, "sql_warehouse": {
                            "id": context["spec"]["config"]["warehouse_id"], "permission": "CAN_USE"}}], jobs, on_lakebase),
                        **({"user_api_scopes": scopes} if scopes else {})})
                except Exception as create_error:
                    raise ImplementBlocked(f"the App {app_name} could not be created: {get_clean_error_message(create_error)[:500]}")
                created = True
            app = self._implement_app_scopes(app_name, app, scopes, warnings)
            app, resourced = self._implement_app_job_resources(app_name, app, jobs, warnings, on_lakebase)
            if on_lakebase and not implement_app_holds_postgres(app.get("resources") or []):
                to_delta("Inspire could not give the App its Lakebase database as a resource")
            if not merged:
                self._implement_tag_app(context, app_name)
            deadline = time.time() + timeout_s
            while True:
                state = str((app.get("compute_status") or {}).get("state") or "")
                if state == "ACTIVE":
                    break
                if state in ("STOPPED", "ERROR") and not started:
                    self._implement_api("POST", f"/api/2.0/apps/{app_name}/start")
                    started = True
                elif state == "ERROR":
                    raise ImplementBlocked(f"the App compute failed to start: {(app.get('compute_status') or {}).get('message')}")
                if time.time() > deadline:
                    raise ImplementBlocked(f"the App compute did not become ACTIVE within {timeout_s}s (state {state})")
                time.sleep(poll_s)
                app = self._implement_api("GET", f"/api/2.0/apps/{app_name}")
            client_id = app.get("service_principal_client_id")
            if client_id and not per_user:
                for table in tables:
                    plan = implement_access_plan(table, access[table], client_id)
                    for need in plan["grants"]:
                        self.spark.sql(implement_grant_sql(need, client_id))
                    if plan["gaps"]:
                        what = "the result table" if table == context["result_table"] else table
                        warnings.append(f"App: {implement_access_gap_message(client_id, plan['gaps'], what)}")
                if space_id:
                    try:
                        self._implement_grant("genie", space_id, [{"type": "service_principal", "name": client_id}], "CAN_RUN")
                    except Exception as error:
                        warnings.append(f"App: Ask Genie may fail because the App's service principal could not get CAN RUN on the "
                                        f"Genie Agent {space_id}: {get_clean_error_message(error)[:300]}")
                if context.get("reasons_view"):
                    self.spark.sql(f"GRANT SELECT ON TABLE {inspire_table_name(context['reasons_view'])} TO `{client_id}`")
            if on_lakebase:
                problem = self._implement_lakebase_grant(lakebase, client_id) if client_id else "the App has no service principal"
                if problem:
                    to_delta(f"the App's service principal could not be given access to its Lakebase tables: {problem}")
            touched = True
            for attempt in (0, 1):
                self._implement_write_app_files(source, files)
                try:
                    deployment_id = self._implement_app_deployment(app_name, source, deadline, timeout_s, poll_s)
                    app = self._implement_api("GET", f"/api/2.0/apps/{app_name}")
                    url = str(app.get("url") or "").rstrip("/")
                    health = self._implement_app_health(app_name, deployment_id, url, "/health", deadline, poll_s)
                    break
                except Exception as deploy_error:
                    if attempt or not on_lakebase:
                        raise
                    to_delta(f"the App did not start with the Lakebase packages {', '.join(lakebase.get('requirements') or [])}: "
                             f"{get_clean_error_message(deploy_error)[:400]}")
                    deadline = time.time() + timeout_s
        except Exception as error:
            if created or (before is not None and (touched or started or resourced or before["scopes"] != sorted(scopes))):
                self._implement_undo_save(context, "app", error,
                                          lambda: self._implement_app_undo(app_name, source, before, touched, started, timeout_s, poll_s))
            raise
        return dict({"app_name": app_name, "app_url": url, "app_health": health,
                     "_asset": {"backend": IMPLEMENT_APP_BACKENDS[0] if on_lakebase else IMPLEMENT_APP_BACKENDS[-1]}},
                    **({"_warnings": warnings} if warnings else {}))

    def _implement_write_app_files(self, source: str, files: dict):
        for relative, text in files.items():
            target = f"{source}/{relative.replace(os.sep, '/')}"
            self.w_client.workspace.mkdirs(target.rsplit("/", 1)[0])
            self._implement_api("POST", "/api/2.0/workspace/import", {"path": target, "format": "RAW", "overwrite": True,
                                                                      "content": base64.b64encode(text.encode("utf-8")).decode()})

    def _implement_app_before(self, app_name: str, app, source: str, files: dict):
        if app is None:
            return None
        kept = {}
        for relative in files:
            try:
                kept[relative] = read_user_text_file(f"{source}/{relative.replace(os.sep, '/')}", self.w_client)
            except Exception as error:
                if not implement_is_not_found(error):
                    raise ImplementBlocked(f"the App {app_name} could not be read before Inspire changed it, so Inspire did "
                                           f"not change it: {get_clean_error_message(error)[:500]}")
                kept[relative] = None
        return {"files": kept, "scopes": sorted(app.get("user_api_scopes") or []), "resources": list(app.get("resources") or []),
                "deployment": str((app.get("active_deployment") or {}).get("deployment_id") or "")}

    def _implement_app_undo(self, app_name: str, source: str, before, touched: bool, started: bool, timeout_s: int, poll_s: int) -> str:
        import time
        if before is None:
            self._implement_api("POST", f"/api/2.0/apps/{app_name}/stop")
            return f"stopped the new App {app_name}"
        done = []
        if touched:
            self._implement_write_app_files(source, {k: v for k, v in before["files"].items() if v is not None})
            for relative in [k for k, v in before["files"].items() if v is None]:
                self._implement_api("POST", "/api/2.0/workspace/delete", {"path": f"{source}/{relative.replace(os.sep, '/')}"})
            done.append(f"put the App {app_name}'s files back as they were")
        app = self._implement_api("GET", f"/api/2.0/apps/{app_name}")
        if sorted(app.get("user_api_scopes") or []) != before["scopes"]:
            app, problem = self._implement_set_app_scopes(app_name, app, before["scopes"])
            held = sorted(app.get("user_api_scopes") or [])
            if held != before["scopes"]:
                raise RuntimeError(f"the App {app_name} still holds the user authorization scopes {', '.join(held) or 'none'}; before "
                                   f"this build it held {', '.join(before['scopes']) or 'none'}" + (f": {problem}" if problem else ""))
            done.append(f"gave the App {app_name} back its user authorization scopes ({', '.join(before['scopes']) or 'none'})")
        if list(app.get("resources") or []) != before.get("resources", list(app.get("resources") or [])):
            app, problem = self._implement_set_app_scopes(app_name, app, list(app.get("user_api_scopes") or []), before["resources"])
            if list(app.get("resources") or []) != before["resources"]:
                raise RuntimeError(f"the App {app_name} still holds the resources this build added"
                                   + (f": {problem}" if problem else ""))
            done.append(f"gave the App {app_name} back the resources it held before")
        active = str((app.get("active_deployment") or {}).get("deployment_id") or "")
        moved = active != before["deployment"]
        if moved and before["deployment"]:
            self._implement_app_deployment(app_name, source, time.time() + timeout_s, timeout_s, poll_s)
            done.append("deployed them again")
        if started or (moved and not before["deployment"]):
            self._implement_api("POST", f"/api/2.0/apps/{app_name}/stop")
            it = "it" if done else f"the App {app_name}"
            done.append(f"stopped {it} again, as it was stopped before" if started else f"stopped {it}, since it had no deployment before")
        if not done:
            return ""
        note = ("" if started or moved or not touched
                else "; its earlier deployment kept running" if active else "; it had no deployment before and still has none")
        return ", ".join(done[:-1]) + (" and " if len(done) > 1 else "") + done[-1] + note

    def _implement_app_deployment(self, app_name: str, source: str, deadline: float, timeout_s: int, poll_s: int) -> str:
        import time
        deployment = self._implement_api("POST", f"/api/2.0/apps/{app_name}/deployments", {"source_code_path": source, "mode": "SNAPSHOT"})
        deployment_id = deployment.get("deployment_id")
        while True:
            state = str((deployment.get("status") or {}).get("state") or "")
            if state == "SUCCEEDED":
                return deployment_id
            if state in ("FAILED", "CANCELLED"):
                raise RuntimeError(f"the App deployment {state}: {(deployment.get('status') or {}).get('message')}")
            if time.time() > deadline:
                raise RuntimeError(f"the App deployment did not finish within {timeout_s}s (state {state})")
            time.sleep(poll_s)
            deployment = self._implement_api("GET", f"/api/2.0/apps/{app_name}/deployments/{deployment_id}")

    def _implement_app_health(self, app_name: str, deployment_id, url: str, path: str, deadline: float, poll_s: int) -> str:
        import time
        if not url:
            raise RuntimeError("the App deployed but has no URL")
        while True:
            app = self._implement_api("GET", f"/api/2.0/apps/{app_name}")
            status = app.get("app_status") or {}
            state = str(status.get("state") or "")
            active = str((app.get("active_deployment") or {}).get("deployment_id") or "")
            if state == "RUNNING" and active == str(deployment_id):
                break
            if state == "CRASHED":
                raise RuntimeError(f"the App crashed after deployment {deployment_id}: {status.get('message') or 'no message'}")
            if time.time() > deadline:
                raise RuntimeError(f"the App is {state or 'not reported'}, not RUNNING on deployment {deployment_id} "
                                   f"(active deployment {active or 'none'}): {status.get('message') or 'no message'}")
            time.sleep(poll_s)
        get = getattr(self, "_implement_http_get", None) or self._implement_default_http_get
        while True:
            code, location = get(f"{url}{path}")
            if code == 200:
                return "passed"
            if code in (401, 403) or (code in (302, 303, 307) and any(w in str(location).lower() for w in ("oidc", "oauth", "login"))):
                return "running"
            if time.time() > deadline:
                raise RuntimeError(f"the App health check {path} answered HTTP {code}, not 200")
            time.sleep(poll_s)

    def _implement_default_http_get(self, url: str) -> tuple:
        import requests
        ctx = self.dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        response = requests.get(url, headers={"Authorization": f"Bearer {ctx.apiToken().get()}"}, allow_redirects=False, timeout=30)
        return response.status_code, response.headers.get("Location", "")

    def _implement_schedule(self, uc_id: str, job_id: int, schedule: dict):
        from databricks.sdk.service import jobs as _jobs
        job = self.w_client.jobs.get(job_id=job_id)
        cadence = implement_job_cadence(schedule, getattr(getattr(job, "settings", None), "schedule", None))
        cron = cadence["cron"]
        if not cron:
            self._implement_emit(uc_id, "Schedule", "no schedule: the use case has no cadence and none was chosen", "ended_success")
            return
        state = _jobs.PauseStatus.UNPAUSED if cadence["enabled"] else _jobs.PauseStatus.PAUSED
        settings = _jobs.JobSettings(schedule=_jobs.CronSchedule(
            quartz_cron_expression=cron, timezone_id=cadence["timezone"], pause_status=state))
        if schedule.get("failure_emails"):
            settings.email_notifications = _jobs.JobEmailNotifications(on_failure=list(schedule["failure_emails"]))
        self.w_client.jobs.update(job_id=job_id, new_settings=settings)
        self._implement_emit(uc_id, "Schedule", f"job {job_id}: {cron} ({schedule.get('timezone') or 'UTC'}), "
                             f"{'enabled' if schedule.get('enabled') else 'paused'}", "ended_success",
                             {"cron": cron, "enabled": bool(schedule.get("enabled"))})

    def _implement_share(self, uc_id: str, result_table: str, spec: dict, assets: dict, job_id, reasons_view: str = "") -> list:
        principals = spec["sharing"]["principals"]
        if not principals:
            return []
        warnings = []
        rows = self._implement_access_rows(result_table, warnings)
        for principal in principals:
            name = str((principal or {}).get("name") or "").strip()
            if not name:
                continue
            plan = implement_access_plan(result_table, rows, name)
            for need in plan["grants"]:
                statement = implement_grant_sql(need, name)
                try:
                    self.spark.sql(statement)
                except Exception as error:
                    warnings.append(f"Sharing: {statement} failed: {get_clean_error_message(error)[:300]}")
            if reasons_view and "`" not in name:
                statement = f"GRANT SELECT ON TABLE {inspire_table_name(reasons_view)} TO `{name}`"
                try:
                    self.spark.sql(statement)
                except Exception as error:
                    warnings.append(f"Sharing: {statement} failed: {get_clean_error_message(error)[:300]}")
            if plan["gaps"]:
                warnings.append(f"Sharing: {implement_access_gap_message(name, plan['gaps'])}")
        targets = [("dashboard", "dashboards", "CAN_RUN"), ("genie_space", "genie", "CAN_RUN"), ("app", "apps", "CAN_USE"),
                   ("alert", "alertsv2", "CAN_RUN")]
        for name, kind, level in targets:
            if spec["outputs"].get(name) and assets.get(name):
                try:
                    app = self._implement_api("GET", f"/api/2.0/apps/{assets[name]}") if name == "app" else None
                    if name == "app" and not self._implement_owns_app(uc_id, app):
                        owner = implement_app_owner(app)
                        warnings.append(f"Sharing {IMPLEMENT_OUTPUT_LABELS[name]}: {assets[name]} "
                                        + (f"belongs to use case {owner[0]}, so Inspire did not share it; that use case's sharing "
                                           "decides who opens it" if owner else
                                           "is no longer this use case's App, so Inspire did not share it"))
                        continue
                    self._implement_grant(kind, assets[name], principals, level)
                except Exception as error:
                    warnings.append(f"Sharing {IMPLEMENT_OUTPUT_LABELS[name]}: {get_clean_error_message(error)[:300]}")
        warehouse = str((spec.get("config") or {}).get("warehouse_id") or "").strip()
        viewed = [label for name, label in (("dashboard", "the dashboard"), ("genie_space", "the Genie Agent"), ("app", "the App"))
                  if spec["outputs"].get(name) and assets.get(name) and (name != "app" or spec["outputs"].get("app_access") == "user")]
        if warehouse and viewed:
            try:
                self._implement_grant("sql/warehouses", warehouse, principals, "CAN_USE")
            except Exception as error:
                warnings.append(f"Sharing the SQL warehouse {warehouse}: {get_clean_error_message(error)[:300]}. The people you "
                                f"share with need CAN USE on it to open {' and '.join(viewed)}")
        if job_id:
            try:
                self._implement_grant("jobs", int(job_id), principals, "CAN_VIEW")
            except Exception as error:
                warnings.append(f"Sharing the job: {get_clean_error_message(error)[:300]}")
        self._implement_emit(uc_id, "Sharing", "; ".join(warnings) if warnings else f"shared with {len(principals)} principal(s)",
                             "ended_warning" if warnings else "ended_success", {"principals": principals})
        return warnings

    def _implement_grant(self, kind: str, object_id, principals, level: str) -> None:
        path = f"/api/2.0/permissions/{kind}/{object_id}"
        acl = implement_permission_acl(implement_principals_below(self._implement_api("GET", path), principals, level), level)
        if acl:
            self._implement_api("PATCH", path, {"access_control_list": acl})

    def _implement_access_rows(self, table: str, warnings=None):
        args = implement_access_args(table)
        if not args:
            return None
        try:
            rows = self.spark.sql(implement_access_sql(), args=args).collect()
        except Exception as error:
            message = f"Inspire could not read who can access {table}, so it grants without checking: {get_clean_error_message(error)[:300]}"
            if warnings is not None:
                warnings.append(f"Sharing: {message}")
            self.logger.warning(f"[IMPLEMENT-ACCESS] {message}")
            return None
        return [row.asDict() if hasattr(row, "asDict") else dict(row) for row in rows]

    def _implement_output_holders(self, name: str, asset_id: str, key: tuple) -> list:
        column = IMPLEMENT_OUTPUT_ID_COLUMNS[name]
        rows = self._implement_select(f"`{column}` = :asset AND NOT (session_id = :sid AND use_case_id = :uc)",
                                      {"asset": str(asset_id), "sid": int(key[0]), "uc": str(key[1])})
        holders = []
        for session_id, use_case_id in sorted({(int(r["session_id"]), str(r["use_case_id"])) for r in rows}):
            carried, _status = self._implement_carried({"session_id": session_id, "use_case_id": use_case_id,
                                                        "spec_version": 2 ** 31 - 1})
            if str(carried.get(column) or "") == str(asset_id) and use_case_id not in holders:
                holders.append(use_case_id)
        return holders

    def _implement_delete_output(self, name: str, asset_id: str, key: tuple, job_id=None) -> str:
        path = self._IMPLEMENT_ASSET_PATHS[name].format(asset_id)
        if name == "alert" and job_id:
            self._implement_remove_job_task(int(job_id), IMPLEMENT_ALERT_TASK_KEY)
        if name == "app" and job_id:
            self._implement_remove_job_task(int(job_id), IMPLEMENT_LAKEBASE_TASK_KEY)
        warning, label = "", IMPLEMENT_MERGE_LABELS.get(name, "")
        try:
            holders = self._implement_output_holders(name, asset_id, key) if name in IMPLEMENT_MERGE_OUTPUTS else []
        except Exception as error:
            holders = None
            warning = (f"{label[:1].upper()}{label[1:]}: Inspire could not check whether other use cases share {asset_id}, so it did "
                       f"not delete it: {get_clean_error_message(error)[:300]}")
        try:
            if holders:
                warning = (f"{label[:1].upper()}{label[1:]}: {asset_id} also holds use case {', '.join(holders)}, so Inspire did not "
                           "delete it; this use case's part stays in it")
            elif holders is not None and name == "app" and not self._implement_owns_app(key[1], self._implement_api("GET", path)):
                warning = f"{IMPLEMENT_OUTPUT_LABELS[name]}: {asset_id} is no longer this use case's App, so Inspire did not delete it"
            elif holders is not None:
                self._implement_api("DELETE", path)
        except Exception as error:
            if not implement_is_not_found(error):
                raise
            self.logger.info(f"[IMPLEMENT-DELETE] {IMPLEMENT_OUTPUT_LABELS[name]} {asset_id} was already gone")
        self._implement_update(key, **{column: None for column, output in self._IMPLEMENT_CARRIED.items() if output == name})
        self._implement_emit(key[1], IMPLEMENT_OUTPUT_LABELS[name], warning or f"deleted {asset_id} at the user's request",
                             "ended_warning" if warning else "ended_success")
        return warning

    def _run_generate_mode(self):
        """Top-level orchestrator for Operation='Generate Use Cases'.
        
        Route 1 (primary): scan __inspire_usecases for rows where
            generate_genie_code_instruction = 'Yes' (flagged via UI/SQL UPDATE).
        Route 2 (fallback, only if Route 1 finds zero rows): walk the generation_path
            workspace tree, AST-parse each notebook's marker cell, pick up any
            generate_genie_code_instruction = True, then cross-reference __inspire_usecases
            by (id, session_id) to rehydrate the UC row.
        
        After successful regeneration, UPDATE the flag back to 'No' for Route 1 entries
        (Route 2 has no persistent flag — overwriting the notebook removes the True marker).
        """
        log_print(f"\n{'='*80}")
        log_print(f"🚀 OPERATION = 'Generate Use Cases' — discovery pipeline skipped")
        # Gap G15 acquire_lock: prevent concurrent Generate runs for the same business.
        # Fail-open: infra failure never blocks the pipeline.
        _lock_contended = False
        lock_business = self.business_name
        try:
            _ok = self.atomic_writer.acquire_lock(
                lock_business, ttl_seconds=7200,
                holder_note=f"operation={self.operation}"
            ) if getattr(self, "atomic_writer", None) else True
            if not _ok:
                _lock_contended = True
                self.logger.warning(f"G15: another run is already generating for business={self.business_name}; this run stops "
                                    f"so the same use cases are not regenerated twice.")
                self._record_failed_step("Generate Lock", "", f"another run holds the Generate lock for business "
                                                              f"{self.business_name}; this run stopped")
                try:
                    NEXT_ACTIONS.add(
                        rule_id="G15_LOCK_CONTENDED", severity="HIGH", phase="Generate", step="Lock",
                        evidence=f"Concurrent Generate run detected for business={self.business_name}",
                        suggested_user_action="Wait for the other run to complete before launching another Generate run for this business."
                    )
                except Exception:
                    pass
        except Exception as _ge:
            self.logger.warning(f"[G15 LOCK_UNVERIFIED] the Generate lock for business={self.business_name} could not be acquired "
                                f"or verified; proceeding without it (fail-open): {get_clean_error_message(_ge)}")
            self._record_failed_step("Generate Lock", "", _ge)
        log_print(f"{'='*80}\n")
        if _lock_contended:
            raise RuntimeError(f"Another Genie regeneration is running for business {self.business_name}; this run stopped so the "
                               f"same use cases are not regenerated twice. Run it again after the other run ends.")

        # v0.8.9 fix: every terminal path (success/early-return/exception) MUST release the
        # G15 lock — previously only the final Route-2 path released it, leaving stale 7200s
        # locks on every Route-1-success / no-DB / no-flagged-rows happy path.
        try:
            # === PHASE E0: CATALOG DRIFT CHECK (Gap 4) ===
            try:
                _drifted = self._phaseE0_catalog_drift_check(self.session_id)
                if _drifted:
                    log_print(f"⚠️ E0 drift: {len(_drifted)} UC(s) reference tables no longer present", level="WARNING")
            except Exception as _drift_err:
                self.logger.warning(f"E0 drift check failed non-fatally: {get_clean_error_message(_drift_err)}")

            if not self._tracking_table_name:
                log_print("⚠️ Inspire Database not configured — Route 1 unavailable. Attempting Route 2 only.")
                self.logger.warning("Generate mode: no inspire_database configured; only Route 2 (notebook scan) can run.")
                self._run_route2_fallback('Route 2 (notebook scan, no DB)', persist_tracking=False)
                return

            log_print("🔎 Route 1: scanning __inspire_usecases for generate_genie_code_instruction = 'Yes'...")
            route1_rows = self._fetch_flagged_use_cases_route1()
            if route1_rows:
                log_print(f"✅ Route 1: found {len(route1_rows)} flagged use case(s) in the database.")
                regen = self._regenerate_genie_for_flagged_rows(route1_rows, 'Route 1 (DB flag)')
                if regen:
                    self._merge_regenerated_ucs_into_tracking(regen)
                return

            log_print("ℹ️ Route 1 found no flagged rows in the database. Falling back to Route 2 (notebook scan)...")
            self._run_route2_fallback('Route 2 (notebook flag)', persist_tracking=True)
        finally:
            try:
                if getattr(self, 'atomic_writer', None):
                    self.atomic_writer.release_lock(lock_business)
            except Exception as _re:
                self.logger.warning(f"[G15 RELEASE_FAILED] the Generate lock for business '{lock_business}' was not released and stays "
                                    f"until it expires: {get_clean_error_message(_re)}")

    def _run_route2_fallback(self, route_label: str, persist_tracking: bool):
        """v0.8.9 DRY: shared Route 2 fallback for both no-DB and Route1-empty paths.
        
        - Scans workspace notebooks for the generate_genie_code_instruction=True marker.
        - Regenerates Genie instructions + notebooks for any flagged entries.
        - When persist_tracking=True (DB available, Route1 empty path), merges regenerated
          UCs back into __inspire_usecases.
        """
        route2_rows = self._fetch_flagged_use_cases_route2()
        route2_errors = getattr(self, "_route2_errors", None) or []
        if route2_errors:
            self._record_failed_step("Route 2 scan", "", f"{len(route2_errors)} read(s) failed, first: {route2_errors[0]}")
        if not route2_rows:
            raise RuntimeError(self._nothing_to_regenerate_message())
        log_print(f"✅ Route 2: found {len(route2_rows)} flagged notebook(s); cross-reference loaded from __inspire_usecases." if persist_tracking else f"✅ Route 2: found {len(route2_rows)} flagged notebook(s).")
        regen = self._regenerate_genie_for_flagged_rows(route2_rows, route_label)
        if regen and persist_tracking:
            self._merge_regenerated_ucs_into_tracking(regen)

    def _nothing_to_regenerate_message(self) -> str:
        root = getattr(self, 'notebook_output_dir', None) or "the generation path"
        reasons = []
        if getattr(self, "_route1_scan_error", ""):
            reasons.append(f"the Route 1 scan of {self._tracking_table_name} failed ({self._route1_scan_error})")
        elif getattr(self, "_route1_scope", ""):
            reasons.append(f"no use case in {self._route1_scope} is flagged generate_genie_code_instruction = 'Yes'")
        marked = getattr(self, "_route2_marked_count", 0)
        if not marked:
            reasons.append(f"no notebook under {root} is marked generate_genie_code_instruction = True")
        elif not self._tracking_table_name:
            reasons.append(f"{marked} marked notebook(s) under {root} need an Inspire database to load their use cases, and none is set")
        else:
            reasons.append(f"{marked} marked notebook(s) under {root} match no use case in {self._tracking_table_name}")
        errors = getattr(self, "_route2_errors", None) or []
        if errors:
            reasons.append(f"the notebook scan could not read {len(errors)} item(s) (first: {errors[0]})")
        return ("Genie regeneration found nothing to regenerate: " + ", and ".join(reasons)
                + ". Flag the use cases to regenerate, as the Inspire App's Genie buttons do, and run again.")

    def _generate_genie_instructions_and_notebooks(self, all_use_cases: list, full_schema_details: list,
                                                    document_context: str, translations: dict,
                                                    summary_dict: dict = None) -> list:
        import time
        
        if not all_use_cases:
            self.logger.warning("No use cases provided for Genie Code instruction generation")
            return []
        
        log_print(f"\n{'='*80}")
        log_print(f"🤖 GENIE CODE INSTRUCTIONS + NOTEBOOK GENERATION (STREAMING)")
        log_print(f"{'='*80}")
        log_print(f"📊 Total use cases: {len(all_use_cases)}")
        log_print(f"📝 Strategy: Wave-based generation with retry for timed-out use cases")
        log_print(f"📓 Notebooks: a Genie notebook the instant an instruction is emitted; a skeleton notebook for use cases whose instruction fails")
        log_print(f"{'='*80}\n")
        
        _genie_gen_step_id = self._emit_pipeline_status(
            stage_name="Genie Code Instructions",
            step_name="Generation",
            sub_step_name="Started",
            message=f"Generating Genie Code instructions for {len(all_use_cases)} use cases",
            status="started",
            progress_increment=1.0
        )
        
        try:
            self._ensure_genie_metadata_loader()
            schema_index = self._build_genie_schema_index(full_schema_details)
            self._notebooks_generated_inline = True
            self.logger.info("📓 Streaming notebook writes claimed; batch fallback disabled for this run")
            overall_start = time.time()
            targets = list(all_use_cases)
            if self._is_ultra_high_mode() or getattr(self, "_selection_is_fallback", False):
                targets, selection = self._select_genie_targets(all_use_cases, during_discover=False)
                target_ids = {id(uc) for uc in targets}
                log_print(f"   🎯 Genie Code for {selection}")
                for uc in all_use_cases:
                    if id(uc) not in target_ids:
                        self._write_use_case_notebook(uc, translations)
            self._run_genie_waves(targets, full_schema_details, document_context, schema_index, translations,
                                  write_skeleton_on_failure=True, per_uc_progress=1.0)
            elapsed = time.time() - overall_start
            outcome = self._report_genie_outcome("Genie Code Instructions", targets)
            skeletons = sum(1 for uc in all_use_cases if uc.get('notebook_status') == 'skeleton')
            
            log_print(f"\n{'='*80}")
            log_print(f"✅ GENIE CODE INSTRUCTIONS + NOTEBOOKS COMPLETE")
            log_print(f"{'='*80}")
            log_print(f"   ✅ Genie validated: {len(outcome['validated_ids'])}/{len(targets)} Genie target(s) of {len(all_use_cases)} use cases")
            log_print(f"   📓 Genie notebooks: {len(outcome['notebooks_written_ids'])}/{len(targets)}, skeleton notebooks: {skeletons}")
            if outcome["notebook_write_failed"]:
                log_print(f"   📓 Notebook write failures: {len(outcome['notebook_write_failed'])}/{len(targets)}")
            log_print(f"   ⏱️  Total time: {elapsed:.1f}s")
            log_print(f"{'='*80}\n")
            
            self._emit_pipeline_status(
                stage_name="Genie Code Instructions",
                step_name="Generation",
                sub_step_name="Complete",
                message=f"Genie Code instructions: {len(outcome['validated_ids'])} of {len(targets)} validated, "
                        f"{len(targets) - len(outcome['notebooks_written_ids'])} Genie target(s) without a Genie notebook",
                status="ended_success" if len(outcome["validated_ids"]) == len(targets) else "ended_warning",
                progress_increment=2.0,
                result_json={"success": len(outcome["validated_ids"]),
                             "failed": len(targets) - len(outcome["notebooks_written_ids"]),
                             "genie_targets": len(targets), "total_use_cases": len(all_use_cases),
                             "elapsed_seconds": round(elapsed, 1), **outcome},
                step_id=_genie_gen_step_id
            )
        except Exception as _genie_err:
            self._emit_pipeline_status(
                stage_name="Genie Code Instructions",
                step_name="Generation",
                sub_step_name="Failed",
                message=f"Genie Code instructions failed: {get_clean_error_message(_genie_err, max_chars=200)}",
                status="ended_error",
                progress_increment=1.0,
                result_json={"error": get_clean_error_message(_genie_err, max_chars=300), "total_use_cases": len(all_use_cases)},
                step_id=_genie_gen_step_id
            )
            raise
        
        return list(all_use_cases)

    def _filter_business_tables(self, db_details: list, business_context: str = "", industry: str = "", exclusion_strategy: str = TECHNICAL_EXCLUSION_STRATEGY) -> tuple:
        """
        Filters tables into business-relevant vs technical/metadata tables using LLM.
        RESPECTS MAX_CONTEXT_CHARS by batching tables if needed.
        Implements RECURSIVE BATCHING: If a batch is too large, splits it and tries again recursively.
        The classifier sees each table's comment and column names/types. Tables matching the user's
        table-ignore directives (Generation Instructions) are excluded deterministically before the LLM;
        the instructions themselves reach the prompt only through `_get_generation_instruction_for_prompt`.
        Failure policy: tables the classifier does not return are retried; tables still unclassified are kept
        as BUSINESS with classification_status="unverified" and a WARNING (never silently dropped or kept).
        
        Args:
            db_details: List of (catalog, schema, table, column, type, comment) tuples
            business_context: Business context description
            industry: Industry classification
            exclusion_strategy: Technical exclusion strategy; only TECHNICAL_EXCLUSION_STRATEGY ("Aggressive")
                is supported and any other value raises ValueError
            
        Returns:
            Tuple of (business_tables_details, technical_tables_details, business_table_names, technical_table_names,
            business_scores_dict, data_category_map, master_tables_set, transactional_tables_set, reference_tables_set)
        """
        exclusion_strategy = resolve_technical_exclusion_strategy(exclusion_strategy)
        if not db_details:
            self.table_classification_status = {}
            return ([], [], set(), set(), {}, {}, set(), set(), set())
        
        parsed_instructions = getattr(self, "parsed_generation_instructions", None) or {}
        ignore_rules = []
        for item in parsed_instructions.get("table_ignores") or []:
            if isinstance(item, dict):
                pattern, kind = item.get("pattern"), item.get("type")
            else:
                pattern, kind = item, None
            pattern = str(pattern or "").strip().strip("`").lower()
            kind = str(kind or "").strip().lower()
            if kind not in ("prefix", "suffix", "exact", "glob"):
                kind = "glob" if any(ch in pattern for ch in "*?[") else "exact"
            core = pattern.rstrip("*") if kind == "prefix" else pattern.lstrip("*") if kind == "suffix" else pattern
            if not core:
                if pattern:
                    self.logger.warning(f"Ignoring table-ignore directive {item!r}: its {kind} pattern is empty after normalization")
                continue
            ignore_rules.append({"pattern": pattern, "type": kind})
        
        legacy_context = (getattr(self, "additional_context", "") or "").strip()
        
        # Call the main recursive processing logic
        return self._filter_business_tables_with_batching(
            db_details=db_details,
            business_context=business_context,
            industry=industry,
            exclusion_strategy=exclusion_strategy,
            additional_context=legacy_context,
            ignore_rules=ignore_rules
        )
    
    def _process_filter_batch_recursive(self, batch_tables: list, batch_idx: int, total_batches: int, 
                                       business_name: str, industry: str, business_context: str,
                                       exclusion_strategy: str, strategy_rule_text: str,
                                       additional_context_section: str = "",
                                       depth: int = 0, max_depth: int = 10, table_lines: dict = None) -> dict:
        """
        Recursively process a batch of tables for filtering with automatic sub-batching if context limit exceeded.
        
        Args:
            batch_tables: List of table names (catalog.schema.table, backticks optional) to classify
            batch_idx: Batch index for logging
            total_batches: Total number of batches
            business_name: Business name
            industry: Industry classification
            business_context: Business context description
            exclusion_strategy: Exclusion strategy
            strategy_rule_text: Strategy-specific rules text
            additional_context_section: User-provided filtering instructions (formatted)
            depth: Current recursion depth
            max_depth: Maximum recursion depth to prevent infinite loops
            table_lines: Optional {table_name: prompt line with comment and columns}
            
        Returns:
            Dict mapping table_name -> (classification, business_score, data_category) for the tables the
            classifier actually returned. Tables it could not classify are absent (the caller retries them).
        """
        plain_tables = [str(t).replace('`', '') for t in batch_tables]
        lines_map = table_lines or {}
        if depth > max_depth:
            self.logger.warning(f"[Batch {batch_idx}] Max recursion depth ({max_depth}) reached; "
                                f"{len(plain_tables)} table(s) left unclassified for retry.")
            return {}
        
        depth_prefix = "  " * depth
        if depth > 0:
            log_prefix = f"[Batch {batch_idx}-Split{depth}]"
        else:
            log_prefix = f"[Batch {batch_idx}]"
        
        log_print(f"{depth_prefix}{log_prefix} Processing {len(plain_tables)} tables (depth={depth})...")
        self.logger.info(f"{depth_prefix}{log_prefix} Processing {len(plain_tables)} tables (depth={depth})...")
        
        tables_markdown = "\n".join(
            lines_map.get(t) or ("- `" + "`.`".join(t.split(".", 2)) + "`") for t in plain_tables)
        
        # Verify this batch's prompt size
        prompt_vars = {
            "business_name": business_name,
            "industry": industry,
            "business_context": business_context,
            "exclusion_strategy": exclusion_strategy,
            "strategy_rules": strategy_rule_text,
            "additional_context_section": additional_context_section,
            "tables_markdown": tables_markdown,
            "generation_instructions_section": self._get_generation_instruction_for_prompt("FILTER_BUSINESS_TABLES_PROMPT"),
        }
        
        # Format the prompt to check actual size (using model-specific limits from TECHNICAL_CONTEXT)
        filter_context_limit = get_max_context_chars("English", "FILTER_BUSINESS_TABLES_PROMPT")
        test_prompt = self.ai_agent._load_and_format_prompt("FILTER_BUSINESS_TABLES_PROMPT", prompt_vars)
        actual_prompt_size = len(test_prompt)
        
        _common_kwargs = dict(
            total_batches=total_batches, business_name=business_name,
            industry=industry, business_context=business_context,
            exclusion_strategy=exclusion_strategy, strategy_rule_text=strategy_rule_text,
            additional_context_section=additional_context_section,
            depth=depth + 1, max_depth=max_depth, table_lines=table_lines
        )
        
        # Check if batch is too large
        if actual_prompt_size > filter_context_limit:
            # Need to split this batch
            if len(plain_tables) <= 1:
                self.logger.warning(
                    f"{depth_prefix}{log_prefix} Prompt for single table {plain_tables[0]} is {actual_prompt_size:,} chars "
                    f"(> {filter_context_limit:,}); left unclassified for retry."
                )
                return {}
            
            # Split into 2 sub-batches
            mid_point = len(plain_tables) // 2
            first_half = plain_tables[:mid_point]
            second_half = plain_tables[mid_point:]
            
            self.logger.warning(
                f"{depth_prefix}{log_prefix} Batch too large ({actual_prompt_size:,} chars > {filter_context_limit:,} limit). "
                f"Splitting {len(plain_tables)} tables into 2 sub-batches: {len(first_half)} + {len(second_half)} tables"
            )
            
            results = {}
            sub_batch_timeout = self.llm_timeout_seconds * 4
            with _SafeExecutorContext(max_workers=2, thread_name_prefix="filter_split", logger=self.logger,
                                      name="FilterSplit") as split_exec:
                f_first = split_exec.submit(
                    self._process_filter_batch_recursive,
                    batch_tables=first_half, batch_idx=f"{batch_idx}a", **_common_kwargs
                )
                f_second = split_exec.submit(
                    self._process_filter_batch_recursive,
                    batch_tables=second_half, batch_idx=f"{batch_idx}b", **_common_kwargs
                )
                try:
                    for f in split_exec.completed([f_first, f_second], timeout=sub_batch_timeout):
                        try:
                            results.update(f.result(timeout=self.llm_timeout_seconds * 3))
                        except Exception as exc:
                            self.logger.warning(f"{depth_prefix}{log_prefix} Sub-batch failed ({get_clean_error_message(exc)}); its tables will be retried")
                except concurrent.futures.TimeoutError:
                    split_exec.mark_timed_out()
                    self.logger.warning(f"{depth_prefix}{log_prefix} Sub-batch splitting timed out after {sub_batch_timeout}s; unfinished tables will be retried")
            
            self.logger.info(f"{depth_prefix}{log_prefix} Sub-batches complete. Total: {len(results)} tables classified")
            return results
        
        exact_names = {t: t for t in plain_tables}
        lower_names = {}
        by_schema_table = defaultdict(list)
        by_table = defaultdict(list)
        for t in plain_tables:
            lower_names.setdefault(t.lower(), t)
            parts = t.lower().split(".")
            by_schema_table[".".join(parts[-2:])].append(t)
            by_table[parts[-1]].append(t)
        
        def resolve_table_name(raw_name):
            name = raw_name.strip().replace('`', '').replace('"', '').replace("'", '')
            hit = exact_names.get(name) or lower_names.get(name.lower())
            if hit:
                return hit
            parts = name.lower().split(".")
            candidates = by_schema_table.get(".".join(parts[-2:]), []) if len(parts) >= 2 else by_table.get(parts[-1], [])
            return candidates[0] if len(candidates) == 1 else None
        
        # Batch size is OK - process it with retry logic
        self.logger.debug(f"{depth_prefix}{log_prefix} Batch size OK ({actual_prompt_size:,} chars)")
        
        try:
            max_retries = self.max_retry_attempts + 1
            for attempt in range(1, max_retries + 1):
                try:
                    # Call LLM to classify tables in this batch
                    attempt_suffix = f" (attempt {attempt}/{max_retries})" if attempt > 1 else ""
                    log_print(f"{depth_prefix}⏳ {log_prefix} Waiting for LLM response{attempt_suffix} (filtering {len(plain_tables)} tables into BUSINESS vs TECHNICAL)...")
                    self.logger.info(f"{depth_prefix}⏳ {log_prefix} Waiting for LLM response{attempt_suffix} (filtering {len(plain_tables)} tables into BUSINESS vs TECHNICAL)...")
                    response_raw = self.ai_agent.run_worker(
                        step_name=f"Filter_Business_Tables_Batch_{batch_idx}_Depth{depth}_Attempt{attempt}",
                        worker_prompt_path="FILTER_BUSINESS_TABLES_PROMPT",
                        prompt_vars=prompt_vars,
                        response_schema=None,
                        skip_cache=(attempt > 1)  # v0.7.11: retries bypass cache
                    )
                    log_print(f"{depth_prefix}✅ {log_prefix} Received LLM response, parsing classifications...")
                    self.logger.info(f"{depth_prefix}✅ {log_prefix} Received LLM response, parsing classifications...")
                    
                    # Parse CSV response using centralized utility
                    response_clean = clean_csv_response(response_raw)
                    
                    csv_rows = CSVParser.parse_csv_string(
                        response_clean,
                        logger=self.logger,
                        context=log_prefix
                    )
                    csv_rows, _ = CSVParser.aligned_rows(
                        csv_rows, self.logger, log_prefix,
                        required=("Table Name", "Classification", "Data Category", "Business Score"),
                        anchors={"Table Name": lambda value: resolve_table_name(str(value or "")) is not None,
                                 "Classification": CSVParser.one_of(("BUSINESS", "TECHNICAL")),
                                 "Data Category": CSVParser.one_of(("MASTER", "TRANSACTIONAL", "REFERENCE", "TECHNICAL"))},
                        numeric=("Business Score",), text=response_clean)
                    
                    results = {}
                    unknown_names = []
                    for row in csv_rows:
                        raw_name = (row.get('Table Name') or '').strip()
                        if not raw_name:
                            continue
                        table_name = resolve_table_name(raw_name)
                        if table_name is None:
                            unknown_names.append(raw_name)
                            continue
                        classification = (row.get('Classification') or '').strip().upper()
                        data_category = (row.get('Data Category') or '').strip().upper()
                        business_score_str = (row.get('Business Score') or '50').strip()
                        
                        # Parse business score
                        try:
                            business_score = int(business_score_str)
                            business_score = max(0, min(100, business_score))  # Clamp to 0-100
                        except (ValueError, TypeError):
                            business_score = 50  # Default to medium score if parsing fails
                            self.logger.warning(f"{depth_prefix}Table {table_name}: Invalid business score '{business_score_str}', using default 50")
                        
                        if classification == 'BUSINESS':
                            if data_category not in ('MASTER', 'REFERENCE', 'TRANSACTIONAL'):
                                self.logger.warning(f"{depth_prefix}Table {table_name}: LLM returned invalid data_category '{data_category}', defaulting to MASTER")
                                data_category = 'MASTER'
                            results[table_name] = ('BUSINESS', business_score, data_category)
                        elif classification == 'TECHNICAL':
                            results[table_name] = ('TECHNICAL', 0, 'TECHNICAL')
                        else:
                            self.logger.warning(f"{depth_prefix}Table {table_name} has unclear classification '{classification}'; left unclassified for retry")
                    
                    if unknown_names:
                        self.logger.warning(f"{depth_prefix}{log_prefix} Ignored {len(unknown_names)} classifier row(s) naming tables that were not requested: {unknown_names[:5]}")
                    # Validate we got results
                    if not results:
                        raise ValueError(f"no requested table could be matched in the classifier output (rows={len(csv_rows)})")
                    
                    if len(results) < len(plain_tables):
                        self.logger.warning(f"{depth_prefix}{log_prefix} Classifier returned {len(results)}/{len(plain_tables)} requested tables; "
                                            f"{len(plain_tables) - len(results)} will be retried")
                    else:
                        self.logger.info(f"{depth_prefix}✅ {log_prefix} Complete: {len(results)} tables classified")
                    return results
                    
                except (ValueError, AttributeError, KeyError) as parse_error:
                    # CSV parsing errors - retry
                    if attempt < max_retries:
                        self.logger.warning(f"{depth_prefix}{log_prefix} CSV parsing error on attempt {attempt}/{max_retries}: {parse_error}. Retrying...")
                        continue
                    self.logger.warning(f"{depth_prefix}{log_prefix} Failed to parse classifier output after {max_retries} attempts: {parse_error}. "
                                        f"{len(plain_tables)} table(s) left unclassified for retry.")
                    return {}
            
        except InputTooLongError as e:
            # Even though we pre-checked, the LLM still rejected it. Split more aggressively.
            if len(plain_tables) <= 1:
                self.logger.warning(
                    f"{depth_prefix}{log_prefix} LLM rejected the prompt for single table {plain_tables[0]} as too long; "
                    f"left unclassified for retry. Error: {str(e)[:200]}"
                )
                return {}
            
            # Split into 2 sub-batches
            mid_point = len(plain_tables) // 2
            first_half = plain_tables[:mid_point]
            second_half = plain_tables[mid_point:]
            
            self.logger.warning(
                f"{depth_prefix}{log_prefix} LLM rejected batch (InputTooLongError). "
                f"Splitting {len(plain_tables)} tables into 2 sub-batches: {len(first_half)} + {len(second_half)} tables"
            )
            
            # Process both halves recursively
            results = {}
            results.update(self._process_filter_batch_recursive(batch_tables=first_half, batch_idx=f"{batch_idx}a", **_common_kwargs))
            results.update(self._process_filter_batch_recursive(batch_tables=second_half, batch_idx=f"{batch_idx}b", **_common_kwargs))
            return results
            
        except Exception as batch_error:
            self.logger.warning(f"{depth_prefix}{log_prefix} Failed to process batch ({get_clean_error_message(batch_error)}); "
                                f"{len(plain_tables)} table(s) left unclassified for retry.")
            return {}
    
    def _filter_business_tables_with_batching(self, db_details: list, business_context: str, 
                                             industry: str, exclusion_strategy: str,
                                             additional_context: str = "", ignore_rules: list = None) -> tuple:
        """
        Main filtering logic with batching support and recursive sub-batching.
        """
        import fnmatch
        industry_value = industry or "UNKNOWN_INDUSTRY"
        business_context_value = business_context or self._get_business_context_fallback()
        table_columns = {}
        for (catalog, schema, table, column_name, data_type, _comment) in db_details:
            table_columns.setdefault((catalog, schema, table), []).append((column_name, data_type))
        all_tables = {f"{c}.{s}.{t}": (c, s, t) for (c, s, t) in table_columns}
        results = {}
        classification_status = {}
        
        try:
            # === LOG USER ADDITIONAL CONTEXT INTERPRETATION ===
            additional_context_section = ""
            if additional_context:
                self.logger.info("=" * 80)
                self.logger.info("🔍 INTERPRETING USER ADDITIONAL CONTEXT FOR TABLE FILTERING")
                self.logger.info("=" * 80)
                self.logger.info(f"📋 User Instructions: {additional_context[:500]}{'...' if len(additional_context) > 500 else ''}")
                
                # Log interpretation of common patterns
                context_lower = additional_context.lower()
                
                # Check for database/catalog exclusions
                if 'ignore' in context_lower or 'exclude' in context_lower or 'skip' in context_lower:
                    self.logger.info("🎯 DETECTED: User wants to EXCLUDE/IGNORE certain data")
                
                # Check for business entity references
                business_entities = ['customer', 'subscriber', 'client', 'user', 'member', 'patient', 'employee', 
                                   'vendor', 'supplier', 'partner', 'order', 'transaction', 'product', 'inventory']
                detected_entities = [e for e in business_entities if e in context_lower]
                if detected_entities:
                    self.logger.info(f"🎯 DETECTED BUSINESS ENTITIES: {', '.join(detected_entities)}")
                    self.logger.info("   → LLM will apply SEMANTIC understanding (e.g., 'subscriber' = 'customer')")
                
                # Check for database/catalog references
                if 'database' in context_lower or 'catalog' in context_lower or 'schema' in context_lower:
                    self.logger.info("🎯 DETECTED: Database/Catalog/Schema level filtering instructions")
                
                self.logger.info("=" * 80)
                
                # Build the additional context section for the prompt
                additional_context_section = f"""
**🚨🚨🚨 HIGHEST PRIORITY: USER-PROVIDED TABLE EXCLUSION DIRECTIVES 🚨🚨🚨**

**⛔ THESE USER DIRECTIVES OVERRIDE EVERY OTHER RULE IN THIS PROMPT ⛔** Apply them with the table-exclusion rules below.

{additional_context}
"""
            
            strategy_rule_text = """**AGGRESSIVE FILTERING**:
- Apply STRICT interpretation of technical patterns
- Err on the side of EXCLUDING borderline tables
- Any table with >50% technical-looking columns → TECHNICAL
- Aggressively filter logs, metrics, configurations, snapshots, audits
- **Exception**: Only include if clearly core to business operations AND business name indicates relevance"""
            
            def matches_ignore(catalog, schema, table):
                full = f"{catalog}.{schema}.{table}".lower()
                schema_table = f"{schema}.{table}".lower()
                name = str(table).lower()
                for rule in ignore_rules or []:
                    pattern, kind = rule["pattern"], rule["type"]
                    for candidate in ((name,) if "." not in pattern else (schema_table, full)):
                        if ((kind == "prefix" and candidate.startswith(pattern.rstrip("*")))
                                or (kind == "suffix" and candidate.endswith(pattern.lstrip("*")))
                                or (kind == "exact" and candidate == pattern)
                                or (kind == "glob" and fnmatch.fnmatchcase(candidate, pattern))):
                            return True
                return False
            
            ignored_tables = sorted(t for t, parts in all_tables.items() if matches_ignore(*parts))
            for table_name in ignored_tables:
                results[table_name] = ('TECHNICAL', 0, 'TECHNICAL')
                classification_status[table_name] = 'user_ignored'
            if ignored_tables:
                self.logger.info(f"🚫 User table-ignore directives excluded {len(ignored_tables)} table(s) before classification: "
                                 f"{', '.join(ignored_tables[:10])}{' ...' if len(ignored_tables) > 10 else ''}")
            
            loader = getattr(self, "data_loader", None)
            
            def table_line(catalog, schema, table):
                columns = table_columns[(catalog, schema, table)]
                comment = _clip_metadata_text(loader.get_table_comment(catalog, schema, table), 240) if loader is not None else ""
                shown = ", ".join(f"{name} {str(dtype or 'unknown').lower()}" for name, dtype in columns[:40])
                more = f", ... (+{len(columns) - 40} more)" if len(columns) > 40 else ""
                line = f"- `{catalog}`.`{schema}`.`{table}`"
                if comment:
                    line += f" | comment: {comment}"
                return f"{line} | columns ({len(columns)}): {shown}{more}"
            
            to_classify = sorted(t for t in all_tables if t not in results)
            table_lines = {t: table_line(*all_tables[t]) for t in to_classify}
            total_tables = len(to_classify)
            
            self.logger.info(f"Filtering {total_tables} tables into business vs technical categories with '{exclusion_strategy}' strategy...")
            
            # Size batches from the real formatted prompt (model-specific limits from TECHNICAL_CONTEXT)
            filter_tables_context_limit = get_max_context_chars("English", "FILTER_BUSINESS_TABLES_PROMPT")
            base_prompt_size = len(self.ai_agent._load_and_format_prompt("FILTER_BUSINESS_TABLES_PROMPT", {
                "business_name": self.business_name,
                "industry": industry_value,
                "business_context": business_context_value,
                "exclusion_strategy": exclusion_strategy,
                "strategy_rules": strategy_rule_text,
                "additional_context_section": additional_context_section,
                "tables_markdown": "",
                "generation_instructions_section": self._get_generation_instruction_for_prompt("FILTER_BUSINESS_TABLES_PROMPT"),
            }))
            
            # Reserve 20% of model's context limit for safety margin
            available_chars = max(1, int(filter_tables_context_limit * 0.8) - base_prompt_size)
            batches = []
            current_batch = []
            current_size = 0
            for table_name in to_classify:
                line_size = len(table_lines[table_name]) + 1
                if current_batch and current_size + line_size > available_chars:
                    batches.append(current_batch)
                    current_batch = []
                    current_size = 0
                current_batch.append(table_name)
                current_size += line_size
            if current_batch:
                batches.append(current_batch)
            
            self.logger.info(f"Base prompt size: {base_prompt_size} chars, Available for tables: {available_chars} chars; "
                             f"{total_tables} tables packed into {len(batches)} batch(es)")
            
            if batches:
                # ADAPTIVE PARALLELISM: Calculate based on batches and tables
                classification_parallelism, reason = calculate_adaptive_parallelism(
                    "domain_clustering", self.max_parallelism,
                    num_items=total_tables,
                    num_domains=len(batches),
                    is_llm_operation=True, logger=self.logger
                )
                log_adaptive_parallelism_decision("domain_clustering", classification_parallelism, self.max_parallelism, reason)
                
                self.logger.info(f"Processing {len(batches)} classification batch(es) in parallel...")
                
                # Prepare tasks for parallel execution
                tasks = []
                for batch_idx, batch_tables in enumerate(batches, 1):
                    tasks.append((
                        self._process_filter_batch_recursive,
                        (batch_tables, batch_idx, len(batches), self.business_name,
                         industry_value, business_context_value,
                         exclusion_strategy, strategy_rule_text, additional_context_section, 0, 10, table_lines)
                    ))
                
                _num_waves = -(-max(len(tasks), 1) // max(classification_parallelism, 1))
                classification_total_timeout = self.llm_timeout_seconds * 4 * _num_waves + 180
                self.logger.info(f"Classification total timeout: {classification_total_timeout}s ({classification_total_timeout // 60} min) [{len(tasks)} tasks, {classification_parallelism} workers, {_num_waves} waves]")
                batch_outputs = ParallelExecutor.execute_parallel(
                    tasks=tasks,
                    max_workers=classification_parallelism,
                    task_name="Classification Batch",
                    logger=self.logger,
                    thread_name_prefix="FilterBatch",
                    return_exceptions=True,
                    total_timeout=classification_total_timeout
                )
                
                for batch_idx, batch_results in enumerate(batch_outputs, 1):
                    if isinstance(batch_results, Exception):
                        self.logger.warning(f"Classification batch {batch_idx}/{len(batches)} failed ({get_clean_error_message(batch_results)}); "
                                            f"its tables will be retried")
                        continue
                    results.update(batch_results)
                    self.logger.info(f"✅ Classification batch {batch_idx}/{len(batches)} completed")
            
            # === RETRY TABLES THE CLASSIFIER DID NOT RETURN (UP TO 2 RETRIES) ===
            for retry_attempt in range(1, 3):
                missing_tables = [t for t in to_classify if t not in results]
                if not missing_tables:
                    break
                self.logger.warning(f"🔄 RETRY {retry_attempt}/2: {len(missing_tables)} table(s) were not classified. Retrying classification...")
                try:
                    results.update(self._process_filter_batch_recursive(
                        missing_tables, f"RETRY_{retry_attempt}", 1, self.business_name,
                        industry_value, business_context_value, exclusion_strategy, strategy_rule_text,
                        additional_context_section, 0, 10, table_lines))
                except Exception as retry_error:
                    self.logger.warning(f"Retry {retry_attempt} classification failed: {get_clean_error_message(retry_error)}")
            
            for table_name in to_classify:
                if table_name in results:
                    classification_status[table_name] = 'verified'
        except Exception as e:
            self.logger.warning(f"Table filter failed ({get_clean_error_message(e)}); "
                                f"unclassified tables are kept as BUSINESS with classification_status=unverified")
        
        unverified_tables = sorted(t for t in all_tables if t not in results)
        if unverified_tables:
            for table_name in unverified_tables:
                results[table_name] = ('BUSINESS', 50, 'MASTER')
                classification_status[table_name] = 'unverified'
            self.logger.warning(f"⚠️ {len(unverified_tables)} table(s) could not be classified after retries; kept as BUSINESS/MASTER "
                                f"with classification_status=unverified: {', '.join(unverified_tables[:10])}"
                                f"{' ...' if len(unverified_tables) > 10 else ''}")
            NEXT_ACTIONS.add("TABLE_FILTER_PARSE_FAILED", severity="BLOCKING", phase="B5", step="filter_business_tables",
                             evidence=f"{len(unverified_tables)} table(s) kept as BUSINESS with classification_status=unverified: {unverified_tables[:10]}",
                             suggested_user_action="Check the table-filter LLM endpoint and re-run; the listed tables were not classified.")
        self.table_classification_status = classification_status
        
        business_tables_set = set()
        technical_tables_set = set()
        master_tables_set = set()
        transactional_tables_set = set()
        reference_tables_set = set()
        data_category_map = {}
        business_scores = {}
        for table_name, (classification, business_score, data_category) in results.items():
            if classification == 'BUSINESS':
                business_tables_set.add(table_name)
                if data_category == 'REFERENCE':
                    reference_tables_set.add(table_name)
                elif data_category == 'TRANSACTIONAL':
                    transactional_tables_set.add(table_name)
                else:
                    data_category = 'MASTER'
                    master_tables_set.add(table_name)
                data_category_map[table_name] = data_category
                business_scores[table_name] = business_score
            else:
                technical_tables_set.add(table_name)
                data_category_map[table_name] = 'TECHNICAL'
                business_scores[table_name] = 0
        
        # Split db_details into business, technical, and reference buckets.
        # REFERENCE rows are captured so downstream validators (Q1 grounding,
        # Combined-Scoring hallucination checker) can see the full discovered
        # schema universe — per north-star rule classified-out tables must
        # remain joinable and present in available_tables_listing.
        business_details = []
        technical_details = []
        reference_details = []
        for detail in db_details:
            fqtn = f"{detail[0]}.{detail[1]}.{detail[2]}"
            if fqtn in reference_tables_set:
                reference_details.append(detail)
            elif fqtn in business_tables_set:
                business_details.append(detail)
            else:
                technical_details.append(detail)
        
        # expose reference_details on self so callers can access without threading
        # another return value through every call site.
        self._reference_details_last = reference_details
        
        status_counts = defaultdict(int)
        for status_value in classification_status.values():
            status_counts[status_value] += 1
        self.logger.info(f"✅ Filtering complete: {len(business_tables_set)} business tables, {len(technical_tables_set)} technical tables, "
                         f"{len(reference_tables_set)} reference tables (classification: {dict(status_counts)})")
        # Only log technical tables (top 10), not business tables
        if technical_tables_set:
            technical_list = sorted(list(technical_tables_set))[:10]
            more_indicator = f" (showing 10 of {len(technical_tables_set)})" if len(technical_tables_set) > 10 else ""
            self.logger.info(f"Technical tables excluded{more_indicator}: {', '.join(technical_list)}")
        else:
            self.logger.debug("No technical tables to exclude.")
        
        # Log sample business scores
        if business_scores:
            sample_scores = sorted(business_scores.items(), key=lambda x: x[1], reverse=True)[:5]
            self.logger.info(f"Top business tables by score: {', '.join([f'{t}({s})' for t, s in sample_scores])}")
        
        return (business_details, technical_details, business_tables_set, technical_tables_set, business_scores, data_category_map, master_tables_set, transactional_tables_set, reference_tables_set)

    def _estimate_schema_markdown_size(self, db_details: list) -> int:
        if not db_details:
            return 0
        return len(self._format_schema_for_prompt(db_details))

    def _split_columns_to_fit_context(self, column_details: list, base_prompt_size: int, context_limit: int) -> list:
        if not column_details:
            return []
        table_columns = {}
        table_order = []
        for col in column_details:
            key = (col[0], col[1], col[2])
            if key not in table_columns:
                table_columns[key] = []
                table_order.append(key)
            table_columns[key].append(col)
        budget = context_limit - base_prompt_size
        units = []
        for key in table_order:
            cols = table_columns[key]
            size = self._estimate_schema_markdown_size(cols)
            if size <= budget or len(cols) < 2:
                units.append((cols, size))
                continue
            first_alone = self._estimate_schema_markdown_size(cols[:1])
            row_sizes = [self._estimate_schema_markdown_size([cols[1], cols[0]]) - self._estimate_schema_markdown_size(cols[1:2])]
            row_sizes.extend(self._estimate_schema_markdown_size([cols[0], col]) - first_alone for col in cols[1:])
            fixed_size = first_alone - row_sizes[0]
            chunk, chunk_size = [], fixed_size
            for col, row_size in zip(cols, row_sizes):
                if chunk and chunk_size + row_size > budget:
                    units.append((chunk, chunk_size))
                    chunk, chunk_size = [], fixed_size
                chunk.append(col)
                chunk_size += row_size
            if chunk:
                units.append((chunk, chunk_size))
            self.logger.warning(f"Table {key[0]}.{key[1]}.{key[2]} ({len(cols)} columns, {size:,} chars) exceeds the "
                                f"{budget:,}-char schema budget on its own; its columns were split across batches")
        batches = []
        current_cols = []
        current_size = base_prompt_size
        for cols, size in units:
            if current_cols and (current_size + size + 1) > context_limit:
                batches.append(current_cols)
                current_cols = []
                current_size = base_prompt_size
            current_cols.extend(cols)
            current_size += size + 1
        if current_cols:
            batches.append(current_cols)
        return batches

    def _determine_tables_per_call(self, total_tables: int) -> int:
        """Tables per UC generation call.

        Policy rationale: LLM needs to see ALL tables together to produce
        cross-table (multi-table) UCs. Splitting small schemas 1-table-per-call
        forces single-table UCs regardless of FK hints. Only split when schema
        size forces us to.
        """
        if total_tables <= 20:
            # Small schemas: pass ALL tables in a single call. LLM sees full
            # cross-table landscape; fk_relationships_text + related-tables
            # context are visible in one prompt.
            return max(1, total_tables)
        if total_tables <= 50:
            # Medium schemas: still prefer one-call for multi-table UC breadth.
            return total_tables
        if total_tables < 100:
            return max(10, total_tables // 5)
        if total_tables < 200:
            return max(15, total_tables // 8)
        if total_tables < 400:
            return max(20, total_tables // 12)
        return max(25, total_tables // 16)

    def _split_by_table_limit(self, column_details: list, max_tables: int) -> list:
        if not column_details or max_tables <= 0:
            return []
        table_columns = {}
        table_order = []
        for col in column_details:
            key = (col[0], col[1], col[2])
            if key not in table_columns:
                table_columns[key] = []
                table_order.append(key)
            table_columns[key].append(col)
        batches = []
        current_tables = []
        for key in table_order:
            current_tables.append(key)
            if len(current_tables) >= max_tables:
                batch_cols = []
                for table_key in current_tables:
                    batch_cols.extend(table_columns[table_key])
                batches.append(batch_cols)
                current_tables = []
        if current_tables:
            batch_cols = []
            for table_key in current_tables:
                batch_cols.extend(table_columns[table_key])
            batches.append(batch_cols)
        return batches

    # === MODIFIED: _format_schema_for_prompt ===
    def _format_schema_for_prompt(self, db_details: list, load_column_tracking: bool = False,
                                  max_columns: int = None, max_column_comment: int = None,
                                  max_table_comment: int = 600) -> str:
        """
        Format schema for prompt, respecting column tracking from context splitting.
        
        If columns were dropped due to context limits, only include the tracked columns.
        Column tracking is loaded from disk on-demand, not kept in memory.
        Each table block carries the metadata evidence that is known for it: table comment,
        data category, primary key, outgoing foreign keys (declared or inferred) and a sampling note.
        
        Args:
            db_details: List of column tuples (catalog, schema, table, column, type, comment)
            load_column_tracking: If True, load column tracking from disk for implementation
            max_columns: Optional per-table column cap; evidence-bearing columns are kept first
            max_column_comment: Optional per-column comment length cap
            max_table_comment: Table comment length cap
        
        Returns:
            Formatted schema markdown
        """
        if not db_details: return ""
        tables = defaultdict(list)
        
        # Load column tracking from disk only when needed
        column_tracking_cache = {}
        if load_column_tracking:
            # Get unique tables from db_details
            unique_tables = set()
            for (catalog, schema, table, _, _, _) in db_details:
                fqtn_plain = f"{catalog}.{schema}.{table}"
                unique_tables.add(fqtn_plain)
            
            # Load column tracking for tables that have it
            for fqtn in unique_tables:
                if self.storage_manager.has_column_tracking(fqtn):
                    tracked_cols = self.storage_manager.load_column_tracking(fqtn)
                    if tracked_cols:
                        column_tracking_cache[fqtn] = tracked_cols
        
        for (catalog, schema, table, column_name, data_type, comment) in db_details:
            fqtn_plain = f"{catalog}.{schema}.{table}"
            
            # Check if this table has column tracking (meaning columns were dropped)
            if fqtn_plain in column_tracking_cache:
                tracked_columns = column_tracking_cache[fqtn_plain]
                # Only include columns that were tracked (kept during splitting)
                if column_name in tracked_columns:
                    tables[(catalog, schema, table)].append((column_name, data_type or 'unknown', comment or ''))
            else:
                # No tracking for this table - include all columns
                tables[(catalog, schema, table)].append((column_name, data_type or 'unknown', comment or ''))
        
        loader = getattr(self, "data_loader", None)
        category_map = getattr(self, "data_category_map", None) or {}
        status_map = getattr(self, "table_classification_status", None) or {}
        markdown_parts = []
        for (catalog, schema, table), columns in tables.items():
            if not columns:
                continue  # Skip tables with no columns (shouldn't happen but safety check)
            fqtn_plain = f"{catalog}.{schema}.{table}"
            lines = [f"### `{catalog}`.`{schema}`.`{table}`"]
            if fqtn_plain in column_tracking_cache:
                lines.append(f"<!-- NOTE: Table has {len(columns)} columns (subset used for context fitting) -->")
            primary_key, relations = [], []
            if loader is not None:
                table_comment = _clip_metadata_text(loader.get_table_comment(catalog, schema, table), max_table_comment)
                if table_comment:
                    lines.append(f"Table comment: {table_comment}")
                primary_key = loader.get_primary_key(catalog, schema, table)
                relations = loader.get_table_relations(catalog, schema, table)
            category = category_map.get(fqtn_plain) or category_map.get(fqtn_plain.lower())
            if category:
                status = status_map.get(fqtn_plain) or status_map.get(fqtn_plain.lower())
                suffix = " (unverified default: the table filter could not classify this table)" if status == "unverified" else ""
                lines.append(f"Data category: {category}{suffix}")
            if primary_key:
                lines.append(f"Primary key: {', '.join(primary_key)}")
            if relations:
                lines.append("Foreign keys: " + "; ".join(
                    f"{rel[3]} -> {rel[4] or catalog}.{rel[5] or schema}.{rel[6]}.{rel[7]}" + (" (inferred)" if source == "inferred" else "")
                    for rel, source, _ in relations))
            if loader is not None and fqtn_plain not in column_tracking_cache:
                sampling = loader.get_sampling_info(catalog, schema, table)
                if sampling and len(columns) == sampling[0]:
                    lines.append(f"Columns shown: {sampling[0]} of {sampling[1]} (wide table sampled; declared keys, commented, "
                                 f"numeric-measure and date/timestamp columns are always kept)")
            shown = columns
            if max_columns is not None and len(columns) > max_columns:
                protected = {str(p).lower() for p in primary_key}
                protected.update(str(rel[3]).lower() for rel, _, _ in relations)
                ranked = sorted(range(len(columns)), key=lambda i: (
                    ColumnSampler.evidence_rank(columns[i][0], columns[i][1], columns[i][2], protected), i))
                shown = [columns[i] for i in sorted(ranked[:max_columns])]
            lines.append("| column | type | column_description |")
            lines.append("| --- | --- | --- |")
            lines.extend(f"| {col} | {dtype} | {_clip_metadata_text(desc, max_column_comment)} |" for col, dtype, desc in shown)
            if len(shown) < len(columns):
                lines.append(f"| ... | ... | ({len(columns) - len(shown)} more columns not shown) |")
            markdown_parts.append("\n".join(lines) + "\n")
        
        return "\n".join(markdown_parts)

    def _augment_columns_with_foreign_keys(self, column_details: list) -> list:
        if not column_details or not self.data_loader or not getattr(self.data_loader, "foreign_key_graph", None):
            return column_details
        tables_present = {(c, s, t) for (c, s, t, _, _, _) in column_details}
        additional = []
        for base_table in list(tables_present):
            relations = self.data_loader.foreign_key_graph.get(base_table, [])
            for rel in relations:
                ref_catalog = rel[4] or base_table[0]
                ref_schema = rel[5] or base_table[1]
                ref_table = rel[6]
                ref_tuple = (ref_catalog, ref_schema, ref_table)
                if ref_tuple in tables_present:
                    continue
                try:
                    ref_cols = self.data_loader._get_table_details(
                        ref_catalog,
                        ref_schema,
                        ref_table,
                        apply_sampling=self.data_loader.enable_column_sampling
                    )
                    if ref_cols:
                        additional.extend(ref_cols)
                        tables_present.add(ref_tuple)
                        self.logger.debug(f"Auto-added foreign key table {ref_tuple} to batch context")
                except Exception as e:
                    self.logger.warning(f"Could not load foreign key table {'.'.join(map(str, ref_tuple))} for batch context: "
                                        f"{get_clean_error_message(e)}")
        if additional:
            return column_details + additional
        return column_details

    def _get_reverse_foreign_key_graph(self):
        if not self.data_loader or not getattr(self.data_loader, "foreign_key_graph", None):
            return {}
        graph_size = (len(self.data_loader.foreign_key_graph),
                      sum(len(rels) for rels in self.data_loader.foreign_key_graph.values()))
        cached_size = getattr(self, "_reverse_foreign_key_graph_size", None)
        if hasattr(self, "_reverse_foreign_key_graph") and cached_size == graph_size:
            return self._reverse_foreign_key_graph
        reverse_graph = defaultdict(list)
        for src_key, rels in self.data_loader.foreign_key_graph.items():
            for rel in rels:
                ref_catalog = rel[4] or src_key[0]
                ref_schema = rel[5] or src_key[1]
                ref_table = rel[6]
                ref_key = (ref_catalog, ref_schema, ref_table)
                reverse_graph[ref_key].append(rel)
        self._reverse_foreign_key_graph = reverse_graph
        self._reverse_foreign_key_graph_size = graph_size
        return reverse_graph

    def _augment_columns_with_related_tables(self, column_details: list, exclude_reference: bool = True) -> list:
        if not column_details or not self.data_loader or not getattr(self.data_loader, "foreign_key_graph", None):
            return column_details
        allowed_tables = None
        if hasattr(self, "data_category_map"):
            excluded_categories = {"TECHNICAL"}
            if exclude_reference:
                excluded_categories.add("REFERENCE")
            allowed_tables = {k for k, v in self.data_category_map.items() if v not in excluded_categories}
        tables_present = {(c, s, t) for (c, s, t, _, _, _) in column_details}
        additional = []
        reverse_graph = self._get_reverse_foreign_key_graph()
        for base_table in list(tables_present):
            relations = self.data_loader.foreign_key_graph.get(base_table, [])
            for rel in relations:
                ref_catalog = rel[4] or base_table[0]
                ref_schema = rel[5] or base_table[1]
                ref_table = rel[6]
                ref_fqtn = f"{ref_catalog}.{ref_schema}.{ref_table}"
                if allowed_tables is not None and ref_fqtn not in allowed_tables:
                    continue
                ref_tuple = (ref_catalog, ref_schema, ref_table)
                if ref_tuple in tables_present:
                    continue
                try:
                    ref_cols = self.data_loader._get_table_details(
                        ref_catalog,
                        ref_schema,
                        ref_table,
                        apply_sampling=self.data_loader.enable_column_sampling
                    )
                    if ref_cols:
                        additional.extend(ref_cols)
                        tables_present.add(ref_tuple)
                except Exception as e:
                    self.logger.warning(f"Could not load FK-related table {'.'.join(map(str, ref_tuple))} for batch context: "
                                        f"{get_clean_error_message(e)}")
            reverse_relations = reverse_graph.get(base_table, [])
            for rel in reverse_relations:
                src_catalog = rel[0]
                src_schema = rel[1]
                src_table = rel[2]
                src_fqtn = f"{src_catalog}.{src_schema}.{src_table}"
                if allowed_tables is not None and src_fqtn not in allowed_tables:
                    continue
                src_tuple = (src_catalog, src_schema, src_table)
                if src_tuple in tables_present:
                    continue
                try:
                    ref_cols = self.data_loader._get_table_details(
                        src_catalog,
                        src_schema,
                        src_table,
                        apply_sampling=self.data_loader.enable_column_sampling
                    )
                    if ref_cols:
                        additional.extend(ref_cols)
                        tables_present.add(src_tuple)
                except Exception as e:
                    self.logger.warning(f"Could not load FK-related table {'.'.join(map(str, src_tuple))} for batch context: "
                                        f"{get_clean_error_message(e)}")
        if additional:
            return column_details + additional
        return column_details

    def _expand_tables_with_foreign_keys(self, tables: set):
        if not tables or not self.data_loader or not getattr(self.data_loader, "foreign_key_graph", None):
            return tables, []
        expanded = set()
        relationships = []
        for tbl in tables:
            expanded.add(tbl)
            cat, sch, tbl_name = parse_three_level_name(tbl)
            if not (cat and sch and tbl_name):
                continue
            key = (cat, sch, tbl_name)
            for rel in self.data_loader.foreign_key_graph.get(key, []):
                ref_catalog = rel[4] or cat
                ref_schema = rel[5] or sch
                ref_table = rel[6]
                ref_str = f"{ref_catalog}.{ref_schema}.{ref_table}"
                expanded.add(ref_str)
                relationships.append(f"{cat}.{sch}.{tbl_name}.{rel[3]} → {ref_catalog}.{ref_schema}.{ref_table}.{rel[7]}")
        return expanded, relationships

    def _sanitize_name(self, name: str) -> str:
        if not name: return "_"
        s = re.sub(r'[^a-z0-9_]', '_', str(name).lower())
        s = re.sub(r'_+', '_', s).strip('_')
        return s or "_"

    @staticmethod
    def _normalize_usecase(usecase: str, full_name: str = "") -> str:
        candidate = (usecase or "").strip()
        source = candidate if candidate else (full_name or "").strip()
        if not source:
            return "N/A"
        source = re.sub(r"\s+", " ", source).strip()
        if " with " in source.lower():
            source = re.split(r"\s+with\s+", source, flags=re.IGNORECASE)[0].strip()
        source = " ".join(w.capitalize() if w.lower() not in ("to", "by", "and", "or", "the", "of", "in", "on", "for", "from", "through", "via", "without") or i == 0 else w.lower() for i, w in enumerate(source.split()))
        return source if source else "N/A"

    def _save_usecases_catalog_json(self, final_consolidated_use_cases: list, english_translations: dict, summary_dict: dict = None) -> dict:
        """
        Saves the usecases_catalog.json file with all the content for later doc generation.
        Returns the summary_dict used (computed if not provided).
        
        JSON Structure:
        {
            "title": "...",
            "executive_summary": "...",
            "domains": "domain1: X use cases, domain2: Y use cases, ...",
            "domains": [
                {
                    "summary": "...",
                    "use_cases": [
                        {id, title, type, statement, etc...}
                    ]
                }
            ]
        }
        """
        try:
            self.logger.info("Generating JSON Catalog...")
            
            # Group use cases by domain
            flat_english_use_cases = final_consolidated_use_cases
            english_grouped_data = dict(self._rank_domain_groups(self._group_use_cases_by_domain_flat(flat_english_use_cases)))
            
            # Get summary if not provided
            if summary_dict is None:
                (summary_dict, transliterated_name) = self._get_salesy_summary(english_grouped_data, self.business_name, "English", english_translations)
            
            # Build domain summary string
            domain_counts = []
            for domain_name, use_cases in english_grouped_data.items():
                domain_counts.append(f"{domain_name}: {len(use_cases)} use cases")
            domains_summary = ", ".join(domain_counts)
            
            # === NEW: Build Column Bitmap and replace names with IDs ===
            import copy
            column_registry = {}
            # Format: ID -> "FQN, Description"
            for col_id, info in self.id_column_map.items():
                column_registry[col_id] = f"{info['fqn']}, {info['description']}"

            # === NEW: Build Table Registry ===
            table_registry = {}
            # Format: ID -> Table FQN
            for table_id, table_fqn in self.id_table_map.items():
                table_registry[table_id] = table_fqn

            # Create deep copy of use cases to modify for JSON output
            json_english_grouped_data = copy.deepcopy(english_grouped_data)

            # Pre-compute table -> column IDs map for faster lookup
            table_to_col_ids = defaultdict(list)
            for fqn, cid in self.column_id_map.items():
                # fqn is catalog.schema.table.column
                parts = fqn.split('.')
                if len(parts) >= 2:
                    table_fqn = ".".join(parts[:-1])
                    table_to_col_ids[table_fqn].append(cid)

            # Replace column names with IDs in the JSON copy
            column_suffix_index = defaultdict(set)
            for fqn, cid in self.column_id_map.items():
                parts = str(fqn).lower().split('.')
                for depth in range(1, len(parts) + 1):
                    column_suffix_index[".".join(parts[-depth:])].add(cid)
            unresolved_column_names = set()

            def _column_names_to_ids(text):
                col_ids = []
                for name in [c.strip() for c in re.split(r'[,\n]', text) if c.strip()]:
                    if name in self.column_id_map:
                        col_ids.append(self.column_id_map[name])
                        continue
                    matches = column_suffix_index.get(name.replace('`', '').lower(), set())
                    if len(matches) == 1:
                        col_ids.append(next(iter(matches)))
                    else:
                        unresolved_column_names.add(name)
                        col_ids.append(name)
                return ", ".join(col_ids)

            for domain in json_english_grouped_data:
                for uc in json_english_grouped_data[domain]:
                    for column_field in ("Columns Involved", "Involved Columns"):
                        if uc.get(column_field):
                            uc[column_field] = _column_names_to_ids(uc[column_field])

                    # 2. Process directly_involved_schema (Full table schemas -> IDs)
                    # Requirement: directly_involved_schema should only have comma separated list of column ids from the registry
                    involved_tables = uc.get('_directly_involved_tables', [])
                    schema_col_ids = set()
                    table_ids = []
                    if involved_tables:
                        for table in involved_tables:
                            # table is usually FQN from discovery
                            if table in table_to_col_ids:
                                schema_col_ids.update(table_to_col_ids[table])
                            # Convert table name to table ID
                            if table in self.table_id_map:
                                table_ids.append(self.table_id_map[table])
                    
                    # Always overwrite directly_involved_schema with IDs string (or empty string)
                    if schema_col_ids:
                         uc['directly_involved_schema'] = ", ".join(sorted(list(schema_col_ids), key=lambda x: int(x) if x.isdigit() else float('inf')))
                    else:
                         uc['directly_involved_schema'] = ""
                    
                    # Convert directly_involved_tables to table IDs string
                    if table_ids:
                        uc['directly_involved_tables'] = ", ".join(sorted(table_ids, key=lambda x: int(x) if x.isdigit() else float('inf')))
                    else:
                        uc['directly_involved_tables'] = ""
                    
                    # Remove old underscore-prefixed keys
                    if '_directly_involved_schema' in uc:
                        del uc['_directly_involved_schema']
                    if '_directly_involved_tables' in uc:
                        del uc['_directly_involved_tables']
                    # Remove generated/validated from JSON - these are now stored in notebook cells
                    if 'generated' in uc:
                        del uc['generated']
                    if 'validated' in uc:
                        del uc['validated']
            
            if unresolved_column_names:
                self.logger.warning(f"JSON catalog: {len(unresolved_column_names)} column name(s) kept as written because they are "
                                    f"ambiguous or not in the column registry: {sorted(unresolved_column_names)[:10]}")
                NEXT_ACTIONS.add("COLUMN_ID_FUZZY_MATCH_REJECTED", severity="SAFE_IGNORE", phase="artifacts",
                                 step="save_usecases_catalog_json",
                                 evidence=f"Unresolved column names left as written: {sorted(unresolved_column_names)[:10]}")
            
            # Build the JSON structure
            catalog_json = {
                "business_name": self.business_name,
                "title": f"{self.business_name} Use Cases Catalog",
                "executive_summary": summary_dict.get("Executive", ""),
                "domains_summary": domains_summary,
                "column_registry": column_registry,
                "table_registry": table_registry,
                "metadata": {
                    "catalogs": [c.strip() for c in self.catalogs_str.split(',') if c.strip()],
                    "schemas": [s.strip() for s in self.schemas_str.split(',') if s.strip()],
                    "generation_path": self.generation_path
                },
                "domains": []
            }
            v130_keys = self._v130_results_keys()
            catalog_json.update({key: v130_keys[key] for key in self._V130_CATALOG_KEYS if key in v130_keys})
            
            # Add each domain from the ID-ified data
            domain_summaries, _unmatched = self._canonical_summary(summary_dict, list(json_english_grouped_data.keys()))
            for domain_name, use_cases in json_english_grouped_data.items():
                domain_obj = {
                    "domain_name": domain_name,
                    "summary": domain_summaries.get(domain_name, f"Domain: {domain_name} with {len(use_cases)} use cases"),
                    "use_cases": use_cases
                }
                catalog_json["domains"].append(domain_obj)

            with self._RUN_OUTCOME_LOCK:
                self.final_inspire_results_json = catalog_json
            
            # Save to workspace
            json_path = os.path.join(self.docs_output_dir, f"{self.business_name}-dbx_inspire.json")
            self.logger.info(f"Saving JSON Catalog to: {json_path}")
            
            json_content = json.dumps(catalog_json, indent=2, ensure_ascii=False, default=str)
            json_data_b64 = base64.b64encode(json_content.encode('utf-8')).decode()
            
            self._import_workspace_file(json_path, json_data_b64, workspace.ImportFormat.AUTO)
            
            self.logger.info(f"✅ JSON Catalog saved successfully to {json_path}")
            log_print(f"✅ JSON Catalog saved to: {json_path}")
            self._merge_run_outcome_into_results()
            
            return summary_dict
            
        except Exception as e:
            error_text = get_clean_error_message(e)
            self.logger.error(f"Failed to save JSON Catalog: {error_text}")
            log_print(f"❌ JSON Catalog was NOT saved: {error_text}", level="ERROR")
            self._record_failed_step("JSON Catalog", "English", e)
            NEXT_ACTIONS.add("JSON_CATALOG_SAVE_FAILED", severity="BLOCKING", phase="artifacts",
                             step="save_usecases_catalog_json", evidence=error_text[:300],
                             suggested_user_action="Check write access and size limits of the docs folder under generation_path, then regenerate documents.")
            try:
                self._emit_pipeline_status(stage_name="Artifact Writing", step_name="JSON Catalog", sub_step_name="Failed",
                                           message=f"JSON catalog was not saved: {error_text[:200]}", status="ended_error",
                                           result_json={"error": error_text[:300]})
            except Exception as status_error:
                self.logger.warning(f"Could not record the JSON catalog failure in the step table: {get_clean_error_message(status_error)}")
            return summary_dict

    def _load_usecases_catalog_json(self, json_file_path: str) -> tuple:
        """
        Loads the JSON Catalog file for docs-only mode.
        
        Returns:
            tuple: (final_consolidated_use_cases, summary_dict, english_grouped_data)
        """
        try:
            self.logger.info(f"Loading JSON Catalog from: {json_file_path}")
            
            json_content = read_user_text_file(json_file_path, self.w_client)
            catalog_json = json.loads(json_content)
            self._clean_legacy_catalog_text(catalog_json)
            self._loaded_catalog_meta = {key: catalog_json.get(key) for key in self._V130_CATALOG_KEYS}
            if catalog_json.get("strategic_context"):
                self.strategy_tree = catalog_json["strategic_context"]
            if catalog_json.get("strategy_coverage") is not None:
                self.strategy_coverage = catalog_json["strategy_coverage"]
            if catalog_json.get("foundations") is not None:
                self.data_foundations = catalog_json["foundations"]
                self.delivery_order = list(catalog_json.get("delivery_order") or [])
            if catalog_json.get("best_of_best") is not None:
                self.best_of_best_outcome = catalog_json["best_of_best"]
            
            self.logger.info(f"✅ Successfully loaded JSON Catalog")
            
            # === NEW: Load business name from JSON, fallback to widget value ===
            json_business_name = catalog_json.get("business_name", None)
            if json_business_name:
                old_business_name = self.business_name
                self.business_name = json_business_name
                self.logger.info(f"Using business name from JSON: '{json_business_name}' (widget value '{old_business_name}' ignored)")
                log_print(f"📌 Using business name from JSON: '{json_business_name}'")
            else:
                self.logger.info(f"Business name not found in JSON. Using widget value: '{self.business_name}'")
                log_print(f"📌 Using business name from widget: '{self.business_name}'")
            
            # Extract data - summary_dict should match _get_salesy_summary format
            # It uses "Executive" as key for executive summary, and domain names for domain summaries
            summary_dict = {
                "Executive": catalog_json.get("executive_summary", "")
            }
            
            # === NEW: Restore Column Names from IDs ===
            column_registry = catalog_json.get("column_registry", {})
            # Parse registry: ID -> FQN
            id_to_fqn = {}
            for cid, val in column_registry.items():
                # Value format: "fqn, description"
                # Split only on first comma to separate FQN from description
                parts = val.split(",", 1)
                id_to_fqn[cid] = parts[0].strip()
            
            id_to_table = {str(tid): str(fqn) for tid, fqn in (catalog_json.get("table_registry", {}) or {}).items()}

            def _ids_to_names(text, registry):
                return ", ".join(registry.get(part, part) for part in (p.strip() for p in str(text).split(",")) if part)

            # Reconstruct grouped data and flat list
            english_grouped_data = {}
            final_consolidated_use_cases = []
            
            for domain_obj in catalog_json.get("domains", []):
                domain_name = domain_obj.get("domain_name", "General Operations")
                use_cases = domain_obj.get("use_cases", [])
                
                # Restore column and table names in use cases
                for uc in use_cases:
                    for column_field in ("Columns Involved", "Involved Columns", "directly_involved_schema"):
                        if uc.get(column_field):
                            uc[column_field] = _ids_to_names(uc[column_field], id_to_fqn)
                    if uc.get("directly_involved_tables"):
                        uc["directly_involved_tables"] = _ids_to_names(uc["directly_involved_tables"], id_to_table)

                domain_summary = domain_obj.get("summary", "")
                
                english_grouped_data[domain_name] = use_cases
                final_consolidated_use_cases.extend(use_cases)
                
                # Add domain summary to summary_dict
                summary_dict[domain_name] = domain_summary
            
            self.logger.info(f"Loaded {len(final_consolidated_use_cases)} use cases from {len(english_grouped_data)} domains")
            
            # === NEW: Filter out any use cases with "Pending" priority (safety check) ===
            pending_use_cases = [uc for uc in final_consolidated_use_cases if uc.get('Priority') == 'Pending']
            if pending_use_cases:
                self.logger.warning(f"⚠️ Found {len(pending_use_cases)} use cases with 'Pending' priority in JSON - these will be filtered out")
                for uc in pending_use_cases[:5]:  # Log first 5 for debugging
                    self.logger.warning(f"  - {uc.get('No', 'N/A')}: {uc.get('Name', 'N/A')}")
                
                # Filter from flat list
                final_consolidated_use_cases = [uc for uc in final_consolidated_use_cases if uc.get('Priority') != 'Pending']
                
                # Also filter from grouped data
                for domain_name in list(english_grouped_data.keys()):
                    english_grouped_data[domain_name] = [uc for uc in english_grouped_data[domain_name] if uc.get('Priority') != 'Pending']
                
                self.logger.info(f"✅ Filtered to {len(final_consolidated_use_cases)} scored use cases (removed {len(pending_use_cases)} pending)")
            
            return (final_consolidated_use_cases, summary_dict, english_grouped_data)
            
        except Exception as e:
            self.logger.error(f"Failed to load JSON Catalog: {get_clean_error_message(e)}")
            raise

    def _stop_live_log_stream(self, tag="finalize"):
        already_stopped = getattr(self, "_volume_log_stream_stopped", False)
        if already_stopped:
            return
        try:
            _vstop = getattr(self, "_volume_log_stop", None)
            _vthread = getattr(self, "_volume_log_thread", None)
            if _vstop is not None:
                _vstop.set()
                if _vthread is not None:
                    _vthread.join(timeout=5.0)
        except Exception:
            pass
        self._volume_log_stream_stopped = True
        try:
            self.logger.info(f"[G30 LIVE_LOG_STREAM_STOPPED FIRED] tag={tag}")
        except Exception:
            pass

    def _upload_log_file(self):
        if getattr(self, "_upload_log_done", False):
            return
        self._upload_log_done = True
        self._close_ailogs_safely()
        try:
            self.logger.info("[G28 END_OF_RUN_LOG_CAPTURE FIRED]")
        except Exception:
            pass
        log_file_path = None
        try:
            for handler in logging.root.handlers:
                if isinstance(handler, logging.FileHandler):
                    log_file_path = handler.baseFilename
                    break
            if not log_file_path:
                self.logger.warning("Could not find FileHandler to upload log file.")
                return
            if not os.path.exists(log_file_path):
                self.logger.warning(f"Log file not found at expected path: {log_file_path}")
                return
            self.logger.info(f"Reading log file from: {log_file_path}")
            with open(log_file_path, "rb") as f: log_data = f.read()
            if not log_data:
                self.logger.warning("Log file is empty. Skipping upload.")
                return
            
            output_log_path = os.path.join(self.base_output_dir, "log.txt")
            if not is_workspace_path(self.base_output_dir):
                try:
                    os.makedirs(os.path.dirname(output_log_path), exist_ok=True)
                    copy_file_sequentially(log_file_path, output_log_path)
                    self.logger.info(f"✅ Log file copied to output directory: {output_log_path}")
                    log_print(f"✅ Log file available at: {output_log_path}")
                except Exception as copy_error:
                    self.logger.warning(f"Failed to copy log file to output directory: {copy_error}")
            else:
                self.logger.debug(f"Skipping local log copy — output is a workspace path: {self.base_output_dir}")
            
            local_backup_log = os.path.join("/tmp", self.sanitized_customer_name, "log.txt")
            if log_file_path != local_backup_log:
                try:
                    local_backup_log = b17_writable_file(local_backup_log, "inspire_log_backup_", self.logger)
                    shutil.copy2(log_file_path, local_backup_log)
                    self.logger.info(f"✅ Log file backup saved to: {local_backup_log}")
                except Exception as backup_err:
                    self.logger.warning(f"Local log backup was not saved: {get_clean_error_message(backup_err)}")
            
            # Also upload to workspace for Databricks UI access
            workspace_log_path = os.path.join(self.docs_output_dir, self._run_log_name("generation_log", "txt"))
            
            self.logger.info(f"Uploading log file to workspace: {workspace_log_path}")
            log_data_b64 = base64.b64encode(log_data).decode()
            self._import_workspace_file(workspace_log_path, log_data_b64, workspace.ImportFormat.AUTO)
            abs_path = self.w_client.workspace.get_status(workspace_log_path).path
            self.logger.info(f"Successfully uploaded log file to workspace: {abs_path}")
            log_print(f"✅ Log file also uploaded to workspace: {abs_path}")
        except Exception as e:
            self.logger.error(f"Failed to upload log file: {get_clean_error_message(e)}")
            if log_file_path:
                self.logger.error(f"Log file was at: {log_file_path}")
        finally:
            self._stop_live_log_stream(tag="upload_log_file")

# COMMAND ----------

# DBTITLE 1,JobLauncher
# ==============================================================================
# JOB LAUNCHER — Self-Launching Databricks Job Pattern
# ==============================================================================

exec(compile(DatabricksInspire._JOB_LAUNCHER_CLASS_SOURCE, "<JobLauncher>", "exec"), globals())

# COMMAND ----------

# DBTITLE 1,Main
# ==============================================================================
# 4. MAIN EXECUTION METHOD (MODIFIED)
# ==============================================================================

def _detect_is_job_context():
    """B15: return True iff running inside a Databricks Job (not interactive notebook).

    Reuses the notebook's established `_nb_ctx.jobId().isDefined()` pattern (see e.g. lines ~27873, 27963, 31934). Interactive notebook runs have no jobId; job-spec runs do.

    B20 (v1.0.8): the v1.0.6 env-var-based rewrite (B16) was reverted after
    live job-runtime verification found that DATABRICKS_JOB_ID / JOB_RUN_ID / TASK_KEY /
    RUN_ID are NOT set on Databricks Serverless job runtime — the env-var variant
    caused the launcher gate to fire on inner-job runs (widget SID passed by
    parent launcher), producing an infinite outer→inner→grandchild spawn until
    MAXIMUM_CONCURRENT_RUNS_REACHED kicked in (two separate job runs each spawned
    MAX-CONCURRENT-blocked children). The original
    crash (interactive UI, jobId() erroneously returning True → in-process →
    EACCES on /tmp/log.txt) is fixed at the crash point by B17+B18 (setup_logging
    tempfile fallback layers) instead of trying to reliably discriminate context.
    No reliable job-vs-interactive signal exists across all Databricks runtimes;
    root-cause fix belongs at the failure surface, not at classification."""
    try:
        _nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        return bool(_nb_ctx.jobId().isDefined())
    except Exception:
        return False

def _should_fire_launcher_gate(is_job_context, widget_session_id, child_marker=None):
    """B15 + v1.2.0 launcher-fix: pure decision for whether the Job Launch Gate should fire.

    Recursion is broken marker-first: when the parent launcher stamped this run with
    INSPIRE_CHILD_MARKER=="1" (a context-independent run parameter) this IS the job run of the
    notebook, so the gate never fires regardless of what jobId() reports. jobId() is only a
    secondary signal, consulted when the marker is absent, so an unreliable or throwing context
    probe on Serverless can no longer make a child relaunch itself.
    Truth table (marker absent):
      interactive + empty  sid → fire (bootstrap)
      interactive + persisted sid → fire (widget-persistence bypass — B15 root cause)
      job + empty sid → fire (outer job, launcher spawns inner)
      job + non-empty sid → skip (inner job — external caller took responsibility)
    Marker present → skip, always.
    """
    if str(child_marker or "").strip() == "1":
        return False
    _widget_sid = (widget_session_id or "").strip()
    _is_inner_job = bool(is_job_context) and bool(_widget_sid)
    return not _is_inner_job

def main():
    """
    Main function to read widget values, validate inputs,
    and run the DatabricksInspire class.
    
    *** IMPORTANT ***
    Run the `create_widgets()` cell first and fill in the UI values
    BEFORE running this main() function.
    """
    
    print_ascii_banner()
    recreate_deferred_widgets(_DEFERRED_WIDGETS)

    # ── Job Launch Gate ──────────────────────────────────────────────────
    _raw_session_id_from_widget = ""
    try:
        _raw_session_id_from_widget = dbutils.widgets.get("14_session_id").strip()
    except Exception:
        pass

    _child_marker_from_widget = ""
    try:
        _child_marker_from_widget = dbutils.widgets.get(INSPIRE_CHILD_MARKER).strip()
    except Exception:
        pass

    # B15: fire the launcher gate for ANY interactive run (regardless of widget-persisted session_id) — Databricks widgets persist their last-set value across kernel restarts, so a prior job-spec run's SID can leak into a subsequent interactive run and bypass the launcher, forcing in-process execution with a stale session_id (B14 amplifier). Only skip the gate when we are an inner job (job context AND widget-provided SID) or when the child marker is present.
    _is_job_context = _detect_is_job_context()
    _should_fire = _should_fire_launcher_gate(_is_job_context, _raw_session_id_from_widget, _child_marker_from_widget)
    _launcher_source = "Inspire App" if _raw_session_id_from_widget else "Inspire Notebook"
    if _should_fire and _raw_session_id_from_widget and not _is_job_context:
        print(f"  [B15] Interactive run detected (no jobId). Widget had persisted session_id='{_raw_session_id_from_widget}' — regenerating fresh SID and launching new job to prevent stale-file collision.")
    if _should_fire:
        _raw_session_id_from_widget = ""  # B15: force fresh SID path; ensures downstream code (line ~32195) sees this as a launcher-driven run, not a user-provided SID.
        _launch_result = None
        _launcher = None
        try:
            import uuid as _jl_uuid
            _generated_sid = str((_jl_uuid.uuid4().int >> 64) & 9223372036854775807)
            _job_widgets = {}
            for _wn in _NOTEBOOK_WIDGET_NAMES:
                try:
                    _job_widgets[_wn] = dbutils.widgets.get(_wn)
                except Exception:
                    pass
            _job_widgets["14_session_id"] = _generated_sid

            _biz = _job_widgets.get("00_business_name", "").strip()
            _nb_path = JobLauncher.get_current_notebook_path()
            _nb_name = _nb_path.rsplit("/", 1)[-1] if _nb_path else "unknown"
            _job_tags = {
                "dbx_inspire_ai_business": _biz,
                "dbx_inspire_ai_run_type": "discovery",
                "dbx_inspire_ai_session": _generated_sid,
                "dbx_inspire_ai_session_id": _generated_sid,
                "dbx_inspire_ai_usecases": "0",
                "dbx_inspire_ai_domains": "0",
                "dbx_inspire_ai_subdomains": "0",
                "dbx_inspire_ai_notebook": _nb_name,
                "dbx_inspire_ai_launcher_source": _launcher_source,
            }

            if _nb_path:
                import re
                _sanitized_biz = re.sub(r'[^a-z0-9_]', '_', _biz.lower().strip())
                _sanitized_biz = re.sub(r'_+', '_', _sanitized_biz).strip('_') or "unknown"
                _job_name = f"dbx_inspire_ai_{_sanitized_biz}"
                _job_widgets[INSPIRE_CHILD_MARKER] = "1"
                _launcher = JobLauncher(_nb_path, _job_widgets, _job_tags)
                _launch_result = _launcher.launch(job_name=_job_name, run_name=_job_name, run_parameters={INSPIRE_CHILD_MARKER: "1"}, max_concurrent_runs=1, queue=True)
                if not _launch_result["success"]:
                    print(f"\n  Job launch failed: {_launch_result['error']}")
                    print("  Continuing with local notebook execution...\n")
            else:
                print("\n  Could not determine current notebook path for job launch.")
                print("  Continuing with local notebook execution...\n")
        except Exception as _jl_gate_err:
            print(f"\n  Job launch gate error: {_jl_gate_err}")
            print("  Continuing with local notebook execution...\n")

        if _launch_result and _launch_result.get("success"):
            _run_id = _launch_result.get("run_id")
            print(f"  Verifying job run {_run_id} is alive (10 pings, 10s each)...")
            _is_alive = True
            if _launcher and _run_id:
                _is_alive = _launcher.verify_run_alive(_run_id, pings=10, interval_s=10)
            if _is_alive:
                _exit_msg = _launcher._build_banner_str(_launch_result) if _launcher else ("Job launched: " + _launch_result.get("job_url", ""))
                dbutils.notebook.exit(_exit_msg)
            else:
                print("  Job launched but definitively failed during verification. Falling back to local execution...")

    _user_provided_session_id = bool(_raw_session_id_from_widget)
    # ── End Job Launch Gate ──────────────────────────────────────────────

    # --- 1. Get Widget Values ---
    
    try:
        operation = dbutils.widgets.get("15_operation")
    except Exception:
        operation = "Discover Use Cases"
    if not operation:
        operation = "Discover Use Cases"
    # v0.8.9 DRY: operation banner is emitted ONCE during widget validation
    # (~line 31905). Removed duplicate raw-extraction emit.
    
    # --- Business Name ---
    business_name = dbutils.widgets.get("00_business_name")
    
    # --- UC Metadata ---
    catalogs_and_schemas_str = dbutils.widgets.get("01_uc_metadata")
    
    # --- Inspire Database (catalog.database format) ---
    inspire_database = dbutils.widgets.get("02_inspire_database")
    if inspire_database:
        inspire_database = inspire_database.strip()

    try:
        discovery_type = dbutils.widgets.get("03_discovery_type")
    except Exception:
        discovery_type = ""
    try:
        strategic_context_str = dbutils.widgets.get("10_strategic_context") or ""
    except Exception:
        strategic_context_str = ""
    try:
        best_of_best_count_str = dbutils.widgets.get("16_best_of_best_count") or ""
    except Exception:
        best_of_best_count_str = ""
    

    # --- Table Election Mode ---
    table_election_mode = dbutils.widgets.get("04_table_election")
    log_print(f"🗳️ Table Election: {table_election_mode}")
    
    # --- Use Cases Quality Filter ---
    use_cases_quality = dbutils.widgets.get("05_use_cases_quality")
    log_print(f"🎚️ Use Cases Quality: {use_cases_quality}")
    
    # --- Business Domains ---
    business_domains_str = dbutils.widgets.get("06_business_domains")
    
    # --- Business Priorities (multi-select) ---
    business_priorities_str = dbutils.widgets.get("07_business_priorities")
    
    # --- Strategic Goals ---
    generation_instructions_str = dbutils.widgets.get("08_generation_instructions")
    
    # Check if this is a JSON file path (docs-only mode)
    json_file_path = None
    catalogs_list = []
    schemas_list = []
    tables_list = []
    
    if catalogs_and_schemas_str:
        catalogs_and_schemas_str = catalogs_and_schemas_str.strip()
        # Check if it's a JSON file path (starts with /)
        if catalogs_and_schemas_str.startswith('/'):
            json_file_path = catalogs_and_schemas_str
            log_print(f"Detected JSON file path: {json_file_path}")
            log_print("Running in DOCS-ONLY mode: Will skip use case generation and notebook generation.")
        else:
            # Parse catalogs, schemas, and tables from the merged widget
            for item in catalogs_and_schemas_str.split(','):
                item = item.strip()
                if not item:
                    continue
                dot_count = item.count('.')
                if dot_count == 2:
                    # Fully qualified table (catalog.schema.table)
                    tables_list.append(item)
                elif dot_count == 1:
                    # Fully qualified schema (catalog.schema)
                    schemas_list.append(item)
                elif dot_count == 0:
                    # Catalog only
                    catalogs_list.append(item)
                else:
                    # Invalid format - log warning
                    log_print(f"Invalid metadata format '{item}' - expected 0, 1, or 2 dots", level="WARNING")
    
    catalogs_str = ','.join(catalogs_list)
    schemas_str = ','.join(schemas_list)
    tables_str = ','.join(tables_list)
    
    # --- Generation Options ---
    generate_str = dbutils.widgets.get("09_generation_options")
    # Force "use cases" to be included always
    if generate_str:
        if "use cases" not in generate_str:
             generate_str += ", use cases"
    else:
        generate_str = "use cases"
    
    # Parse generation options for special flags
    generate_options_list = [opt.strip() for opt in generate_str.split(',') if opt.strip()]
    
    # --- Generation Path ---
    generation_path = dbutils.widgets.get("11_generation_path")
    
    # --- Documents Languages (multiselect) ---
    output_language_str = dbutils.widgets.get("12_documents_languages") 
    
    # --- Inspire Session ID (optional - auto-generated if empty) ---
    # B15: read from the launcher-gate-post-value, not the raw widget, so that when the gate cleared a persisted-widget SID (interactive fallback path) we don't accidentally re-inherit the stale value.
    user_session_id = _raw_session_id_from_widget

    # --- Generate Genie Code Instruction For (text: 'all' or non-negative integer) ---
    try:
        generate_genie_code_for_raw = dbutils.widgets.get("13_generate_genie_code_for")
    except Exception:
        generate_genie_code_for_raw = GENIE_CODE_DEFAULT_COUNT
    generate_genie_code_for_raw = (generate_genie_code_for_raw or GENIE_CODE_DEFAULT_COUNT).strip()
    if not generate_genie_code_for_raw:
        generate_genie_code_for_raw = GENIE_CODE_DEFAULT_COUNT

    # ============================================================================
    # --- 2. VALIDATE ALL WIDGET VALUES (FAIL FAST BEFORE ANY PROCESSING) ---
    log_print("=" * 80)
    log_print("🔍 VALIDATING WIDGET INPUTS...")
    log_print("=" * 80)
    
    validation_errors = []
    
    _valid_operations = set(INSPIRE_OPERATIONS)
    if operation not in _valid_operations:
        validation_errors.append(f"❌ 'Operation' (15_operation) must be one of: {', '.join(sorted(_valid_operations))} (got: '{operation}')")
    else:
        log_print(f"✅ Operation: '{operation}'")
    
    # Validate Business Name first
    if not business_name:
        validation_errors.append("❌ 'Business Name' (00_business_name) is REQUIRED")
    else:
        log_print(f"✅ Business Name: '{business_name}'")
    
    # Validate Inspire Database (OPTIONAL, format: catalog.database when provided)
    if not inspire_database:
        log_print("ℹ️ Inspire Database: Not provided (optional). SQL will use placeholder database 'main._inspire'")
    else:
        inspire_db_parts = inspire_database.split('.')
        if len(inspire_db_parts) != 2 or not inspire_db_parts[0].strip() or not inspire_db_parts[1].strip():
            validation_errors.append(f"❌ 'Inspire Database' (02_inspire_database) must be in 'catalog.database' format (got: '{inspire_database}')")
        else:
            log_print(f"✅ Inspire Database: '{inspire_database}'")
            # B19 (v1.0.7): schema/catalog pre-flight — fail EARLY here instead of 60-90s later
            # at finalize_atomic_writer with a downstream TABLE_OR_VIEW_NOT_FOUND that masks the
            # real root cause. Auto-creates the schema when possible; when the catalog is missing
            # returns an actionable error that names exactly what the user must do.
            try:
                _b19_ok, _b19_err = _ensure_inspire_database_ready(
                    spark, inspire_database,
                    log_print_fn=lambda m: log_print(m),
                    get_error_msg_fn=get_clean_error_message,
                )
                if not _b19_ok:
                    validation_errors.append(_b19_err)
            except Exception as _b19_outer:
                # Never let a pre-flight bug crash widget validation itself — surface as an error entry.
                validation_errors.append(
                    f"❌ 'Inspire Database' ({inspire_database}): pre-flight failed unexpectedly. "
                    f"Root cause: {get_clean_error_message(_b19_outer)[:300]}"
                )
    
    # Validate Table Election mode
    valid_table_elections = ["Let Inspire Decides", "All Tables", "Transactional Only"]
    if table_election_mode not in valid_table_elections:
        validation_errors.append(f"❌ 'Table Election' (04_table_election) must be one of: {', '.join(valid_table_elections)}")
    else:
        log_print(f"✅ Table Election: '{table_election_mode}'")
    
    # Validate Use Cases Quality (accept new canonical labels + legacy aliases)
    try:
        _canonical_uc_quality = canonical_use_cases_quality(use_cases_quality)
    except ValueError as _quality_err:
        validation_errors.append(f"❌ 'Use Cases Quality' (05_use_cases_quality): {_quality_err}")
    else:
        if not str(use_cases_quality or "").strip():
            log_print(f"ℹ️ Use Cases Quality: Not provided, using the default '{_canonical_uc_quality}'")
        elif _canonical_uc_quality != use_cases_quality:
            log_print(f"✅ Use Cases Quality: '{use_cases_quality}' → mapped to '{_canonical_uc_quality}' (legacy label)")
        else:
            log_print(f"✅ Use Cases Quality: '{use_cases_quality}'")
        use_cases_quality = _canonical_uc_quality

    try:
        discovery_type = canonical_discovery_type(discovery_type)
        if discovery_type == "Deep":
            log_print("✅ Discovery Type: 'Deep' (reads aggregates from your tables; no row and no PII value reaches the AI)")
        else:
            log_print("✅ Discovery Type: 'Standard' (never reads a row of your data)")
    except ValueError as _discovery_err:
        validation_errors.append(f"❌ 'Discovery Type' (03_discovery_type): {_discovery_err}")

    if strategic_context_str.strip():
        try:
            _parsed_strategy = parse_strategic_context(strategic_context_str, read_file=read_user_text_file)
            if _parsed_strategy:
                _entry_count = sum(1 for _node in iter_strategy_entries(_parsed_strategy))
                log_print(f"✅ Strategic Context: {_entry_count} strategy entries, levels {_parsed_strategy.get('levels')}, "
                          f"{len(_parsed_strategy.get('priorities') or [])} priorities, "
                          f"{len(_parsed_strategy.get('sponsors') or [])} sponsors (applied after selection only)")
                for _warning in _parsed_strategy.get("warnings") or []:
                    log_print(f"⚠️ Strategic Context: {_warning}", level="WARNING")
            else:
                log_print("ℹ️ Strategic Context: empty, the built-in strategic goals are used")
        except ValueError as _strategy_err:
            validation_errors.append(f"❌ 'Strategic Context' (10_strategic_context): {_strategy_err}")
    else:
        log_print("ℹ️ Strategic Context: Not provided (the built-in strategic goals are used for alignment)")

    try:
        _best_of_best_count = parse_best_of_best_count(best_of_best_count_str)
        if _best_of_best_count is None:
            log_print(f"ℹ️ Best of Best: not set, up to {BEST_OF_BEST_DEFAULT_COUNT} use cases are starred unless the "
                      f"generation instructions give another number")
        elif _best_of_best_count == 0:
            log_print("ℹ️ Best of Best: 0, no use case is starred")
        else:
            log_print(f"✅ Best of Best: up to {_best_of_best_count} use cases are starred")
    except ValueError as _best_of_best_err:
        validation_errors.append(f"❌ 'Best of Best Count' (16_best_of_best_count): {_best_of_best_err}")
    

    # Log Business Priorities (optional)
    if business_priorities_str:
        log_print(f"✅ Business Priorities: '{business_priorities_str}'")
    else:
        log_print(f"ℹ️ Business Priorities: Not provided")
    
    # Log Business Domains (optional)
    if business_domains_str:
        log_print(f"✅ Business Domains: '{business_domains_str}'")
    else:
        log_print(f"ℹ️ Business Domains: Not provided (domains will be inferred from data)")
    
    # Log Generation Instructions (optional but HIGHEST PRIORITY when provided)
    if len(generation_instructions_str) > 2000:
        log_print(f"⚠️ Generation Instructions exceeds 2000 character limit ({len(generation_instructions_str)} chars). Truncating to 2000 chars.")
        generation_instructions_str = generation_instructions_str[:2000]
    
    if generation_instructions_str:
        log_print(f"✅ Generation Instructions: '{generation_instructions_str[:100]}...' (HIGHEST PRIORITY)")
    else:
        log_print(f"ℹ️ Generation Instructions: Not provided")
    
    if operation == IMPLEMENT_OPERATION:
        if not inspire_database:
            validation_errors.append("❌ 'Inspire Database' (02_inspire_database) is REQUIRED for Implement Use Cases: the wizard specs live in its __inspire_implement table")
        if not user_session_id:
            validation_errors.append("❌ 'Inspire Session ID' (14_session_id) is REQUIRED for Implement Use Cases: it names the session whose queued specs are built")

    # UC Metadata validation
    if operation == IMPLEMENT_OPERATION:
        log_print("ℹ️ UC Metadata: not used by Implement Use Cases (each use case names its own tables)")
    elif not json_file_path:
        if not catalogs_str and not schemas_str and not tables_str:
            validation_errors.append("❌ 'UC Metadata' (01_uc_metadata) is REQUIRED when discovering use cases")
        else:
            log_print(f"✅ UC Metadata provided: catalogs={len(catalogs_str.split(',')) if catalogs_str else 0}, schemas={len(schemas_str.split(',')) if schemas_str else 0}, tables={len(tables_str.split(',')) if tables_str else 0}")
    else:
        log_print(f"✅ Docs-only mode: Using JSON file '{json_file_path}'")
    
    if not generate_str:
        validation_errors.append("❌ 'Generation Options' (09_generation_options) is REQUIRED - select at least one option")
    else:
        log_print(f"✅ Generation Options: {generate_str}")
    
    if not generation_path:
        validation_errors.append("❌ 'Generation Path' (11_generation_path) is REQUIRED")
    else:
        log_print(f"✅ Generation Path: '{generation_path}'")
    
    
    # Language is only REQUIRED for PDF/Presentation artifacts, optional for notebooks-only
    requires_language = ("PDF Catalog" in generate_str or 
                        "Presentation" in generate_str or 
                        "Use Cases Catalog PDF" in generate_str)
    
    if requires_language:
        if not output_language_str:
            validation_errors.append("❌ 'Documents Languages' (12_documents_languages) is REQUIRED when generating PDF or Presentation")
        else:
            languages = [lang.strip() for lang in output_language_str.split(',') if lang.strip()]
            log_print(f"✅ Documents Languages: {', '.join(languages)}")
    else:
        # Default to English for notebooks-only mode (no PDF/Presentation)
        if not output_language_str:
            output_language_str = "English"
            languages = ["English"]
            log_print(f"ℹ️ Documents Languages: Not required (no PDF/Presentation selected), defaulting to English")
        else:
            languages = [lang.strip() for lang in output_language_str.split(',') if lang.strip()]
            log_print(f"ℹ️ Documents Languages: {', '.join(languages)} (optional for notebooks-only)")
    
    # Log derived options
    generate_genie_instructions = (operation == "Generate Use Cases")
    if operation == "Generate Use Cases":
        log_print("ℹ️ Genie Code Instructions: ENABLED (Generate Use Cases mode — will regenerate notebooks for flagged use cases only)")
        _ignored_gen_artifacts = [opt for opt in generate_options_list if opt in ("PDF Catalog", "Presentation", "Use Cases Catalog PDF")]
        if _ignored_gen_artifacts:
            log_print(f"ℹ️ Generate Use Cases mode ignores artifact selections: {', '.join(_ignored_gen_artifacts)} (only notebooks are regenerated)")
    elif operation == IMPLEMENT_OPERATION:
        log_print("ℹ️ Operation: Implement Use Cases — Genie builds, previews, publishes and schedules each queued wizard spec of this session")
    else:
        log_print("ℹ️ Operation: Discover Use Cases — skeleton notebooks will be created; Genie Code Instructions deferred until you run Operation='Generate Use Cases'")
    log_print(f"ℹ️ Technical table filtering: {TECHNICAL_EXCLUSION_STRATEGY} (mandatory)")
    
    if user_session_id:
        log_print(f"✅ Inspire Session ID: '{user_session_id}' (user-provided)")
    else:
        log_print(f"ℹ️ Inspire Session ID: Not provided (will be auto-generated)")
    
    _ggc_norm = generate_genie_code_for_raw.strip().lower()
    if _ggc_norm == "all":
        log_print("✅ Generate Genie Code Instruction For: ALL use cases")
    else:
        try:
            _ggc_int = int(_ggc_norm)
            if _ggc_int < 0:
                validation_errors.append(f"❌ 'Generate Genie Code Instruction For' (13_generate_genie_code_for) must be 'all' or a non-negative integer (got: '{generate_genie_code_for_raw}')")
            elif _ggc_int == 0:
                log_print("ℹ️ Generate Genie Code Instruction For: 0 (disabled — skeleton notebooks only)")
            else:
                log_print(f"✅ Generate Genie Code Instruction For: top {_ggc_int} use cases (by Inspire Score desc)")
        except ValueError:
            validation_errors.append(f"❌ 'Generate Genie Code Instruction For' (13_generate_genie_code_for) must be 'all' or a non-negative integer (got: '{generate_genie_code_for_raw}')")
    
    if validation_errors:
        import sys as _sys
        error_count = len(validation_errors)
        error_summary = "\n".join(validation_errors)
        
        log_print("=" * 80, level="ERROR")
        log_print(f"❌ VALIDATION FAILED - {error_count} ERROR(S) FOUND:", level="ERROR")
        log_print("=" * 80, level="ERROR")
        for error in validation_errors:
            log_print(error, level="ERROR")
        log_print("=" * 80, level="ERROR")
        
        print(f"\n{'='*80}\n❌ VALIDATION ERRORS ({error_count}):\n{error_summary}\n{'='*80}\n", file=_sys.stderr, flush=True)
        _sys.stdout.flush()
        _sys.stderr.flush()
        
        exit_msg = f"Validation failed with {error_count} error(s):\n{error_summary}"
        dbutils.notebook.exit(exit_msg)
    
    log_print("=" * 80)
    log_print("✅ ALL VALIDATIONS PASSED - Starting generation...")
    log_print("=" * 80)

    # --- 3. Pack values and Run ---
    
    widget_values = {
        "operation": operation,
        "business": business_name,
        "inspire_database": inspire_database,
        "table_election_mode": table_election_mode,
        "use_cases_quality": use_cases_quality,
        "discovery_type": discovery_type,
        "strategic_context": strategic_context_str,
        "best_of_best_count": best_of_best_count_str,
        "generation_instructions": generation_instructions_str,
            "_user_provided_session_id": _user_provided_session_id,
        "business_priorities": business_priorities_str,
        "business_domains": business_domains_str,
        "catalogs": catalogs_str,
        "schemas": schemas_str,
        "tables": tables_str,
        "generate": generate_str,
        "generation_path": generation_path,
        "output_language": output_language_str,
        "technical_exclusion_strategy": TECHNICAL_EXCLUSION_STRATEGY,
        "json_file_path": json_file_path,
        "session_id": user_session_id,
        "generate_genie_code_for": generate_genie_code_for_raw,
    }
    widget_values["_running_in_job_context"] = _is_job_context

    inspirer = None
    try:
        inspirer = DatabricksInspire(**widget_values)
        inspirer.run()
        inspirer.finalize_atomic_writer(success=True)

        if widget_values.get("_running_in_job_context"):
            try:
                _fin_results = getattr(inspirer, "final_inspire_results_json", {})
                _fin_domains = _fin_results.get("domains", [])
                _fin_uc_count = sum(
                    len(d.get("use_cases", []))
                    for d in _fin_domains
                )
                _fin_domain_count = len(_fin_domains)
                _fin_subdomain_set = set()
                for _d in _fin_domains:
                    for _uc in _d.get("use_cases", []):
                        _sd = _uc.get("subdomain", "")
                        if _sd:
                            _fin_subdomain_set.add((_d.get("domain_name", ""), _sd))
                _final_counts = {
                    "dbx_inspire_ai_usecases": str(_fin_uc_count),
                    "dbx_inspire_ai_domains": str(_fin_domain_count),
                    "dbx_inspire_ai_subdomains": str(len(_fin_subdomain_set)),
                }
                _logger = logging.getLogger("main")
                _logger.info(f"[JobTags] Final generation counts (informational only -- Databricks run tags are IMMUTABLE mid-run; these do NOT retag the current run): {_final_counts}")
            except Exception as _tag_update_err:
                logging.getLogger("main").warning(f"[JobTags] Final-count emit failed: {_tag_update_err}")

    except NameError as ne:
        if inspirer:
            inspirer.finalize_atomic_writer(success=False, error_message=str(ne))
        if ('DataLoader' in str(ne) or 'AIAgent' in str(ne) or 
            'PROMPT_TEMPLATES' in str(ne) or 'DatabricksInspire' in str(ne) or 
            'setup_logging' in str(ne) or 'TranslationService' in str(ne)):
            
            print(f"ERROR: A required class, function, or variable is missing: {ne}", file=sys.stderr)
            print("Please ensure `setup_logging`, `DataLoader`, `AIAgent`, `PROMPT_TEMPLATES`, `TranslationService`, and `DatabricksInspire` are defined in preceding cells.", file=sys.stderr)
        else:
            raise
    except Exception as e:
        if inspirer:
            inspirer.finalize_atomic_writer(success=False, error_message=str(e))
        logging.getLogger("main").critical(f"Main execution failed: {e}")
        raise


_INSPIRE_APP_SHELL = "..."


In [0]:
if __name__ == "__main__":
    logging.getLogger("py4j").setLevel(logging.ERROR)
    main()